In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# =================================================================================================
# STAGE28 — FRESH KAGGLE REPOSITORY BOOTSTRAP
#
# Run BEFORE Stage28-0.
#
# This cell:
#   - loads GitHub token from Kaggle Secrets
#   - clones the repository
#   - verifies exact main HEAD
#   - configures secure credential helper for later git push
#   - configures git identity
#
# Scientific operations:
#   MODEL_FITS          = 0
#   MODEL_INFERENCE     = 0
#   THRESHOLD_SELECTION = 0
#   TARGET_OPENINGS     = 0
# =================================================================================================

from __future__ import annotations

import os
import stat
import subprocess
from pathlib import Path


# =================================================================================================
# CONSTANTS
# =================================================================================================

SEP = "=" * 110

REPO_URL = "https://github.com/themubasshir/ids2018-validation-safe-ablation.git"
REPO = Path("/kaggle/working/ids2018-validation-safe-ablation").resolve()

EXPECTED_BRANCH = "main"

EXPECTED_HEAD = "66597724a692da577efd19c1ca070fb27750f3ee"

SECRET_NAMES = [
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "GITHUB_PAT",
    "github_pat",
    "GH_PAT",
]


# =================================================================================================
# HELPERS
# =================================================================================================

def run(cmd, cwd=None, check=True, env=None):
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        env=env,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}):\n"
            f"{' '.join(cmd)}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    return result


def git(*args, cwd=REPO, check=True):
    r = run(
        ["git", *args],
        cwd=cwd,
        check=check,
    )
    return (r.stdout or "").strip()


# =================================================================================================
# 1. LOAD GITHUB TOKEN FROM KAGGLE SECRETS
# =================================================================================================

print()
print(SEP)
print("STAGE28 — GITHUB SECRET")
print(SEP)
print()

github_token = None
github_secret_name = None


# First try environment variables in case bootstrap already exported it.
for secret_name in SECRET_NAMES:
    value = os.environ.get(secret_name)

    if value and value.strip():
        github_token = value.strip()
        github_secret_name = secret_name
        print(f"[FOUND ENV] {secret_name}")
        break


# Otherwise use Kaggle Secrets.
if github_token is None:
    try:
        from kaggle_secrets import UserSecretsClient

        secrets_client = UserSecretsClient()

        for secret_name in SECRET_NAMES:
            try:
                value = secrets_client.get_secret(secret_name)

                if value and value.strip():
                    github_token = value.strip()
                    github_secret_name = secret_name
                    print(f"[FOUND KAGGLE SECRET] {secret_name}")
                    break

            except Exception:
                pass

    except Exception as e:
        print("[INFO] kaggle_secrets unavailable:", type(e).__name__)


if not github_token:
    raise RuntimeError(
        "GitHub token not found.\n\n"
        "Expected one of these Kaggle Secret labels:\n"
        + "\n".join(f"  - {name}" for name in SECRET_NAMES)
    )


# Export only in-process.
# DO NOT PRINT THE VALUE.
os.environ["GITHUB_TOKEN"] = github_token

print()
print(f"Using secret label: {github_secret_name}")
print("Token value       : [HIDDEN]")


# =================================================================================================
# 2. CLONE / REFRESH REPOSITORY
# =================================================================================================

print()
print(SEP)
print("STAGE28 — CLONE / REFRESH REPOSITORY")
print(SEP)
print()

if not REPO.exists():

    print("Repository not present — fresh clone.")
    print()

    run(
        [
            "git",
            "clone",
            "--branch",
            EXPECTED_BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO),
        ],
        cwd="/kaggle/working",
    )

elif (REPO / ".git").is_dir():

    print("Repository already exists — refreshing.")
    print()

    git("fetch", "--prune", "origin")

    git("checkout", EXPECTED_BRANCH)

    # We expect a clean scientific bootstrap.
    status = git("status", "--porcelain")

    if status:
        raise RuntimeError(
            "Existing repository has local modifications.\n"
            "Refusing to overwrite them:\n\n"
            + status
        )

    git(
        "reset",
        "--hard",
        f"origin/{EXPECTED_BRANCH}",
    )

else:

    raise RuntimeError(
        f"Path exists but is not a git repository:\n{REPO}"
    )


# =================================================================================================
# 3. CONFIGURE SECURE GITHUB CREDENTIAL HELPER
# =================================================================================================

print()
print(SEP)
print("CONFIGURE GITHUB AUTHENTICATION")
print(SEP)
print()

# Avoid storing the token in .git/config or printing it.
#
# Git calls this helper during HTTPS authentication.
# The helper obtains GITHUB_TOKEN from the process environment.

credential_helper = Path(
    "/kaggle/working/stage28_github_credential_helper.sh"
)

credential_helper.write_text(
    """#!/bin/sh
case "$1" in
    get)
        echo "username=x-access-token"
        echo "password=$GITHUB_TOKEN"
        ;;
esac
""",
    encoding="utf-8",
)

credential_helper.chmod(
    credential_helper.stat().st_mode
    | stat.S_IXUSR
)

git(
    "config",
    "--local",
    "credential.helper",
    str(credential_helper),
)

# Ensure origin itself contains NO token.
git(
    "remote",
    "set-url",
    "origin",
    REPO_URL,
)

print("[PASS] Credential helper configured.")
print("[PASS] Git remote URL contains no embedded token.")


# =================================================================================================
# 4. CONFIGURE GIT IDENTITY
# =================================================================================================

print()
print(SEP)
print("CONFIGURE GIT IDENTITY")
print(SEP)
print()

git(
    "config",
    "--local",
    "user.name",
    "themubasshir",
)

git(
    "config",
    "--local",
    "user.email",
    "themubasshir@users.noreply.github.com",
)

print("Git user.name :", git("config", "--get", "user.name"))
print("Git user.email:", git("config", "--get", "user.email"))


# =================================================================================================
# 5. VERIFY EXACT REMOTE / LOCAL HEAD
# =================================================================================================

print()
print(SEP)
print("VERIFY EXACT STAGE28 PARENT")
print(SEP)
print()

git("fetch", "origin", EXPECTED_BRANCH)

branch = git("branch", "--show-current")
local_head = git("rev-parse", "HEAD")
origin_head = git(
    "rev-parse",
    f"origin/{EXPECTED_BRANCH}",
)

remote_ls = git(
    "ls-remote",
    "origin",
    f"refs/heads/{EXPECTED_BRANCH}",
)

remote_head = remote_ls.split()[0]


print("Repository    :", REPO)
print("Branch        :", branch)
print("Local HEAD    :", local_head)
print("origin/main   :", origin_head)
print("Remote main   :", remote_head)
print("Expected HEAD :", EXPECTED_HEAD)
print()


if branch != EXPECTED_BRANCH:
    raise RuntimeError(
        f"Wrong branch: expected {EXPECTED_BRANCH}, got {branch}"
    )


if local_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Local repository HEAD mismatch.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={local_head}"
    )


if origin_head != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main mismatch.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={origin_head}"
    )


if remote_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Remote GitHub main mismatch.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={remote_head}"
    )


status = git("status", "--porcelain")

if status:
    raise RuntimeError(
        "Repository is not clean after bootstrap:\n"
        + status
    )


# =================================================================================================
# 6. FINAL STATUS
# =================================================================================================

print()
print(SEP)
print("STAGE28 REPOSITORY BOOTSTRAP — COMPLETE")
print(SEP)
print()

print("[PASS] Repository cloned")
print("[PASS] main checked out")
print("[PASS] local HEAD == origin/main")
print("[PASS] origin/main == GitHub remote main")
print("[PASS] exact Stage28 parent verified")
print("[PASS] GitHub push authentication configured")
print("[PASS] worktree clean")

print()
print("Stage28 parent:")
print(" ", EXPECTED_HEAD)

print()
print("Scientific operations:")
print("  MODEL_FITS          = 0")
print("  MODEL_INFERENCE     = 0")
print("  THRESHOLD_SELECTION = 0")
print("  TARGET_OPENINGS     = 0")

print()
print("NEXT:")
print("  Run the Stage28-0 protocol-lock cell.")
print()
print(SEP)


STAGE28 — GITHUB SECRET

[FOUND KAGGLE SECRET] GITHUB_TOKEN

Using secret label: GITHUB_TOKEN
Token value       : [HIDDEN]

STAGE28 — CLONE / REFRESH REPOSITORY

Repository not present — fresh clone.


CONFIGURE GITHUB AUTHENTICATION

[PASS] Credential helper configured.
[PASS] Git remote URL contains no embedded token.

CONFIGURE GIT IDENTITY

Git user.name : themubasshir
Git user.email: themubasshir@users.noreply.github.com

VERIFY EXACT STAGE28 PARENT

Repository    : /kaggle/working/ids2018-validation-safe-ablation
Branch        : main
Local HEAD    : 66597724a692da577efd19c1ca070fb27750f3ee
origin/main   : 66597724a692da577efd19c1ca070fb27750f3ee
Remote main   : 66597724a692da577efd19c1ca070fb27750f3ee
Expected HEAD : 66597724a692da577efd19c1ca070fb27750f3ee


STAGE28 REPOSITORY BOOTSTRAP — COMPLETE

[PASS] Repository cloned
[PASS] main checked out
[PASS] local HEAD == origin/main
[PASS] origin/main == GitHub remote main
[PASS] exact Stage28 parent verified
[PASS] GitHub push aut

In [3]:
# =================================================================================================
# STAGE 28-0 — PROTOCOL LOCK
# Stability & Novelty–Chronology Disentanglement
#
# IMPORTANT:
#   - NO DATASET OPENING
#   - NO MODEL FITTING
#   - NO MODEL INFERENCE
#   - NO THRESHOLD SELECTION
#   - NO TARGET OPENING
#
# This cell:
#   1. verifies the exact Stage28 parent
#   2. verifies inherited Stage22R + Stage27 provenance
#   3. freezes Stage28A + Stage28B protocol
#   4. freezes seeds [42,43,44,45,46]
#   5. freezes exact fit budget
#   6. freezes CPU-only primary execution
#   7. freezes random-LOAO construction
#   8. writes checksums
#   9. commits + pushes Stage28-0
#  10. remotely verifies the freeze
#
# Scientific operations in this cell:
#   MODEL_FITS          = 0
#   MODEL_INFERENCE     = 0
#   THRESHOLD_SELECTION = 0
#   TARGET_OPENINGS     = 0
# =================================================================================================

from __future__ import annotations

import os
import json
import hashlib
import subprocess
import sys
from pathlib import Path
from datetime import datetime, timezone


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 110

REPO = Path("/kaggle/working/ids2018-validation-safe-ablation").resolve()

EXPECTED_BRANCH = "main"

# Latest verified Stage27 reproducibility-addendum commit.
EXPECTED_PARENT = "66597724a692da577efd19c1ca070fb27750f3ee"

STAGE28_ROOT = REPO / "results" / "stage28_stability_novelty_control"
LOCK_DIR = STAGE28_ROOT / "stage28_0_protocol_lock"

COMMIT_MESSAGE = "stage28-0: freeze stability and random-LOAO protocol"

FROZEN_SEEDS = [42, 43, 44, 45, 46]
REFERENCE_SEED = 42
NEW_SEEDS = [43, 44, 45, 46]

ELIGIBLE_FAMILIES = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

LEARNERS = [
    "XGBOOST",
    "LIGHTGBM",
]

EXPECTED_STAGE27_TARGET_ATTACK_SUPPORT = {
    "BOT": 1966,
    "DDOS": 128027,
    "INFILTRATION": 36,
    "PORT_SCAN": 158930,
    "WEB_ATTACK": 2180,
}

STAGE27_INFERENTIAL_MIN_POSITIVE_SUPPORT = 50

# Stage28B split membership is frozen ONCE and reused across all five model seeds.
# This keeps model-seed variability separate from random-membership variability.
STAGE28B_SPLIT_SEED = 42
STAGE28B_TARGET_BENIGN_FRACTION = 0.20
STAGE28B_VALIDATION_FRACTION_OF_REMAINING_DEVELOPMENT = 0.20


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def run(cmd, cwd=REPO, check=True, capture=True):
    result = subprocess.run(
        cmd,
        cwd=str(cwd),
        check=False,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
    )

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}):\n"
            f"{' '.join(cmd)}\n\n"
            f"STDOUT:\n{result.stdout or ''}\n\n"
            f"STDERR:\n{result.stderr or ''}"
        )

    return result


def git(*args, check=True):
    r = run(["git", *args], check=check)
    return (r.stdout or "").strip()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    text = json.dumps(
        obj,
        indent=2,
        sort_keys=False,
        ensure_ascii=False,
    ) + "\n"

    path.write_text(text, encoding="utf-8")


def read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def require_file(relative_path: str) -> Path:
    path = REPO / relative_path

    if not path.is_file():
        raise RuntimeError(
            f"Required frozen artifact missing:\n{relative_path}"
        )

    return path


def receipt(relative_path: str) -> dict:
    path = require_file(relative_path)

    return {
        "path": relative_path,
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }


def remote_main_sha() -> str:
    output = git(
        "ls-remote",
        "origin",
        f"refs/heads/{EXPECTED_BRANCH}",
    )

    if not output:
        raise RuntimeError(
            f"Could not resolve remote branch origin/{EXPECTED_BRANCH}"
        )

    return output.split()[0]


# =================================================================================================
# 2. EXACT REPOSITORY STATE
# =================================================================================================

print()
print(SEP)
print("STAGE28-0 — EXACT REPOSITORY STATE")
print(SEP)
print()

if not REPO.is_dir():
    raise RuntimeError(
        f"Repository not found:\n{REPO}"
    )

if not (REPO / ".git").exists():
    raise RuntimeError(
        f"Not a git repository:\n{REPO}"
    )

branch = git("branch", "--show-current")
head = git("rev-parse", "HEAD")
status_before = git("status", "--porcelain")
remote_before = remote_main_sha()

print("Repository :", REPO)
print("Branch     :", branch)
print("HEAD       :", head)
print("Remote main:", remote_before)
print()

if branch != EXPECTED_BRANCH:
    raise RuntimeError(
        "Unexpected branch.\n"
        f"expected={EXPECTED_BRANCH}\n"
        f"actual={branch}"
    )

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected Stage28 scientific/provenance parent.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )

if remote_before != EXPECTED_PARENT:
    raise RuntimeError(
        "Remote main does not match the expected Stage28 parent.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"remote={remote_before}"
    )

if status_before:
    raise RuntimeError(
        "Repository must be clean before Stage28-0 freeze.\n\n"
        + status_before
    )

if LOCK_DIR.exists():
    raise RuntimeError(
        "Stage28-0 destination already exists.\n"
        "Refusing to overwrite a possible protocol freeze:\n"
        f"{LOCK_DIR}"
    )

print("[EXACT] Stage28 parent verified.")
print("[EXACT] Local main == remote main.")
print("[EXACT] Worktree clean.")
print("[EXACT] Stage28-0 destination absent.")


# =================================================================================================
# 3. REQUIRED INHERITED PROVENANCE
# =================================================================================================

print()
print(SEP)
print("VERIFY INHERITED STAGE22R + STAGE27 PROVENANCE")
print(SEP)
print()

INHERITED_PATHS = [

    # ---------------------------------------------------------------------------------------------
    # Stage22R — FULL random natural
    # ---------------------------------------------------------------------------------------------
    "results/stage22r_training/stage22r_2a_random_natural/"
    "stage22r_2a_random_natural_result.json",

    "results/stage22r_training/stage22r_2a_random_natural/"
    "random_natural_lightgbm_model.txt",

    "results/stage22r_training/stage22r_2a_random_natural/"
    "random_natural_xgboost_model.json",

    # ---------------------------------------------------------------------------------------------
    # Stage22R — FULL chronological natural
    # ---------------------------------------------------------------------------------------------
    "results/stage22r_training/stage22r_2c_chronological_natural/"
    "stage22r_2c_chronological_natural_result.json",

    # ---------------------------------------------------------------------------------------------
    # Stage22R memberships / input provenance
    # ---------------------------------------------------------------------------------------------
    "results/stage22r_protocol_recovery/"
    "stage22r_1b1_development_memberships/"
    "stage22r_1b1_membership_summary.json",

    "results/stage22r_protocol_recovery/"
    "stage22r_1c_development_model_inputs/"
    "stage22r_1c_development_model_input_manifest.json",

    # ---------------------------------------------------------------------------------------------
    # Stage22R shared final holdout
    # ---------------------------------------------------------------------------------------------
    "results/stage22r_training/"
    "stage22r_final_single_holdout/"
    "stage22r_final_holdout_report.md",

    # ---------------------------------------------------------------------------------------------
    # Stage27 protocol
    # ---------------------------------------------------------------------------------------------
    "results/stage27_loao_unseen_attack/"
    "stage27_0_protocol_lock/"
    "fold_spec.json",

    "results/stage27_loao_unseen_attack/"
    "stage27_0_protocol_lock/"
    "model_inventory.json",

    "results/stage27_loao_unseen_attack/"
    "stage27_0_protocol_lock/"
    "feature_representation.json",

    "results/stage27_loao_unseen_attack/"
    "stage27_0_protocol_lock/"
    "threshold_policy.json",

    "results/stage27_loao_unseen_attack/"
    "stage27_0_protocol_lock/"
    "target_population_spec.json",

    "results/stage27_loao_unseen_attack/"
    "stage27_0_protocol_lock/"
    "support_rules.json",

    "results/stage27_loao_unseen_attack/"
    "stage27_0_protocol_lock/"
    "metric_spec.json",

    "results/stage27_loao_unseen_attack/"
    "stage27_0_protocol_lock/"
    "class_weight_policy.json",

    "results/stage27_loao_unseen_attack/"
    "stage27_0_protocol_lock/"
    "inherited_receipts.json",

    # ---------------------------------------------------------------------------------------------
    # Stage27 seed-42 model provenance
    # ---------------------------------------------------------------------------------------------
    "results/stage27_loao_unseen_attack/"
    "stage27_2a_preopening_models/"
    "model_artifact_manifest.json",

    "results/stage27_loao_unseen_attack/"
    "stage27_2a_preopening_models/"
    "fit_execution_receipt.json",

    # ---------------------------------------------------------------------------------------------
    # Stage27 frozen final metrics / synthesis
    # ---------------------------------------------------------------------------------------------
    "results/stage27_loao_unseen_attack/"
    "stage27_4a_final_synthesis/"
    "stage27_final_primary_metrics.csv",

    "results/stage27_loao_unseen_attack/"
    "stage27_4a_final_synthesis/"
    "stage27_final_operating_points.csv",

    "results/stage27_loao_unseen_attack/"
    "stage27_4a_final_synthesis/"
    "stage27_final_novelty_gaps.csv",

    "results/stage27_loao_unseen_attack/"
    "stage27_4a_final_synthesis/"
    "stage27_synthesis_receipt.json",

    "results/stage27_loao_unseen_attack/"
    "stage27_4a_final_synthesis/"
    "stage27_synthesis.md",

    # ---------------------------------------------------------------------------------------------
    # Stage27 publication/reproducibility closure
    # ---------------------------------------------------------------------------------------------
    "docs/STAGE27_PUBLICATION_CLOSEOUT.md",
]

inherited_receipt_entries = []

for relative in INHERITED_PATHS:
    r = receipt(relative)
    inherited_receipt_entries.append(r)
    print(
        f"[OK] {relative}\n"
        f"     sha256={r['sha256']}"
    )

print()
print(f"[EXACT] inherited artifacts verified: {len(inherited_receipt_entries)}")


# =================================================================================================
# 4. VERIFY PARENT MODEL SEMANTICS
# =================================================================================================

print()
print(SEP)
print("VERIFY PARENT MODEL SEMANTICS")
print(SEP)
print()

stage22_random_path = (
    REPO
    / "results/stage22r_training/stage22r_2a_random_natural/"
      "stage22r_2a_random_natural_result.json"
)

stage22_chrono_path = (
    REPO
    / "results/stage22r_training/stage22r_2c_chronological_natural/"
      "stage22r_2c_chronological_natural_result.json"
)

stage27_model_path = (
    REPO
    / "results/stage27_loao_unseen_attack/stage27_0_protocol_lock/"
      "model_inventory.json"
)

stage27_threshold_path = (
    REPO
    / "results/stage27_loao_unseen_attack/stage27_0_protocol_lock/"
      "threshold_policy.json"
)

stage22_random = read_json(stage22_random_path)
stage22_chrono = read_json(stage22_chrono_path)
stage27_models = read_json(stage27_model_path)
stage27_thresholds = read_json(stage27_threshold_path)


# ---- Stage22 ensemble assertions ------------------------------------------------------------------

if stage22_random["models"]["strategy"] != "ENS_LGBM_XGB_EQUAL":
    raise RuntimeError(
        "Stage22 RANDOM_NATURAL is not the expected equal-weight ensemble."
    )

if stage22_random["models"]["ensemble_probability"] != (
    "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST"
):
    raise RuntimeError(
        "Unexpected Stage22 RANDOM_NATURAL ensemble probability rule."
    )

if stage22_random["models"]["lightgbm"]["configuration"] != "LGBM_11":
    raise RuntimeError("Unexpected Stage22 LightGBM configuration.")

if stage22_random["models"]["xgboost"]["configuration"] != "XGB_11":
    raise RuntimeError("Unexpected Stage22 XGBoost configuration.")

if (
    stage22_random["models"]["lightgbm"]["executed_parameters"]["random_state"]
    != 42
):
    raise RuntimeError("Stage22 LightGBM reference seed is not 42.")

if (
    stage22_random["models"]["xgboost"]["parameters"]["random_state"]
    != 42
):
    raise RuntimeError("Stage22 XGBoost reference seed is not 42.")


# ---- Stage27 learner assertions -------------------------------------------------------------------

if (
    stage27_models["primary_learner"]["configuration_id"]
    != "XGB_11"
):
    raise RuntimeError("Unexpected Stage27 primary learner.")

if (
    stage27_models["replication_learner"]["configuration_id"]
    != "LGBM_11"
):
    raise RuntimeError("Unexpected Stage27 replication learner.")

if (
    stage27_models["primary_learner"]["stage27_cpu_parameters"]["random_state"]
    != 42
):
    raise RuntimeError("Stage27 XGBoost reference seed is not 42.")

if (
    stage27_models["replication_learner"]["stage27_cpu_parameters"]["random_state"]
    != 42
):
    raise RuntimeError("Stage27 LightGBM reference seed is not 42.")


# ---- Stage27 threshold assertions -----------------------------------------------------------------

grid = stage27_thresholds["grid"]

if (
    grid["integer_start"] != 1
    or grid["integer_stop_inclusive"] != 99
    or grid["integer_step"] != 1
):
    raise RuntimeError(
        "Unexpected Stage27 threshold grid."
    )

if stage27_thresholds["standard"]["threshold"] != 0.5:
    raise RuntimeError(
        "Unexpected Stage27 standard threshold."
    )


print("[EXACT] Stage22 scientific unit = equal-weight LightGBM + XGBoost ensemble.")
print("[EXACT] Stage22 configurations = LGBM_11 + XGB_11.")
print("[EXACT] Stage27 learners = XGB_11 + LGBM_11.")
print("[EXACT] Reference seed = 42.")
print("[EXACT] Parent threshold semantics verified.")


# =================================================================================================
# 5. FREEZE EXACT FIT ACCOUNTING
# =================================================================================================

print()
print(SEP)
print("FREEZE EXACT STAGE28 FIT BUDGET")
print(SEP)
print()

# Stage22 FULL:
# 2 split cells × 2 learner components × 5 seeds = 20 component fits.
#
# Seed 42 already has both learner components for RANDOM_NATURAL and
# CHRONOLOGICAL_NATURAL:
#
# 2 split cells × 2 components = 4 reused fits.
#
# New:
# 2 × 2 × 4 = 16.

STAGE22_TOTAL_COMPONENT_REALIZATIONS = 2 * 2 * 5
STAGE22_REUSED_SEED42_COMPONENT_FITS = 2 * 2
STAGE22_NEW_COMPONENT_FITS = 2 * 2 * 4

# Stage27 chronology-first LOAO:
# 5 families × 2 learners × 5 seeds = 50.
# Existing seed42 = 10.
# New seeds 43–46 = 40.

STAGE27_TOTAL_COMPONENT_REALIZATIONS = 5 * 2 * 5
STAGE27_REUSED_SEED42_FITS = 5 * 2
STAGE27_NEW_FITS = 5 * 2 * 4

# Stage28B random LOAO:
# completely new control:
# 5 families × 2 learners × 5 seeds = 50.

STAGE28B_TOTAL_COMPONENT_REALIZATIONS = 5 * 2 * 5
STAGE28B_REUSED_FITS = 0
STAGE28B_NEW_FITS = 5 * 2 * 5

TOTAL_EXISTING_REUSED = (
    STAGE22_REUSED_SEED42_COMPONENT_FITS
    + STAGE27_REUSED_SEED42_FITS
    + STAGE28B_REUSED_FITS
)

TOTAL_NEW_FITS = (
    STAGE22_NEW_COMPONENT_FITS
    + STAGE27_NEW_FITS
    + STAGE28B_NEW_FITS
)

TOTAL_COMPONENT_REALIZATIONS = (
    STAGE22_TOTAL_COMPONENT_REALIZATIONS
    + STAGE27_TOTAL_COMPONENT_REALIZATIONS
    + STAGE28B_TOTAL_COMPONENT_REALIZATIONS
)

# Scientific evaluation cells:
# Stage22 has 2 ensemble outputs × 5 seeds = 10 ensemble-level evaluations.
# Stage27 = 50 learner-level cells.
# Stage28B = 50 learner-level cells.
TOTAL_SCIENTIFIC_EVALUATION_CELLS = 10 + 50 + 50

assert STAGE22_NEW_COMPONENT_FITS == 16
assert STAGE27_NEW_FITS == 40
assert STAGE28B_NEW_FITS == 50
assert TOTAL_EXISTING_REUSED == 14
assert TOTAL_NEW_FITS == 106
assert TOTAL_COMPONENT_REALIZATIONS == 120
assert TOTAL_SCIENTIFIC_EVALUATION_CELLS == 110

print("Stage22 existing seed42 components reused :", STAGE22_REUSED_SEED42_COMPONENT_FITS)
print("Stage22 NEW component fits               :", STAGE22_NEW_COMPONENT_FITS)
print()
print("Stage27 existing seed42 fits reused      :", STAGE27_REUSED_SEED42_FITS)
print("Stage27 NEW chronology LOAO fits         :", STAGE27_NEW_FITS)
print()
print("Stage28B NEW random LOAO fits            :", STAGE28B_NEW_FITS)
print()
print("TOTAL EXISTING FITS REUSED               :", TOTAL_EXISTING_REUSED)
print("TOTAL NEW FIT BUDGET                     :", TOTAL_NEW_FITS)
print("TOTAL COMPONENT REALIZATIONS             :", TOTAL_COMPONENT_REALIZATIONS)
print("TOTAL SCIENTIFIC EVALUATION CELLS        :", TOTAL_SCIENTIFIC_EVALUATION_CELLS)


# =================================================================================================
# 6. CREATE LOCK DIRECTORY
# =================================================================================================

LOCK_DIR.mkdir(parents=True, exist_ok=False)

timestamp_utc = datetime.now(timezone.utc).isoformat()


# =================================================================================================
# 7. scientific_questions.json
# =================================================================================================

scientific_questions = {
    "stage": "Stage28-0",
    "stage_role": "FINAL_EXPERIMENTAL_STAGE_OF_CURRENT_MANUSCRIPT",
    "arms": {
        "28A": {
            "name": "TRAINING_SEED_STABILITY",
            "question": (
                "Are the paper's headline temporal and unseen-family conclusions "
                "stable across independently seeded training realizations?"
            ),
        },
        "28B": {
            "name": "RANDOM_SPLIT_LOAO_CONTROL",
            "question": (
                "How much of observed LOAO degradation remains when attack-family "
                "novelty is tested without the chronology-first separation used "
                "in Stage27?"
            ),
        },
    },
    "joint_question": (
        "Do the major conclusions survive model realization, and can attack-family "
        "novelty be distinguished more clearly from temporal distribution shift?"
    ),
    "success_does_not_require_stability": True,
    "optimization_stage": False,
    "new_architectures": 0,
    "new_hyperparameter_search": 0,
    "new_feature_search": 0,
    "new_threshold_rules": 0,
    "new_taxonomy": 0,
    "new_attack_families": 0,
}

write_json(
    LOCK_DIR / "scientific_questions.json",
    scientific_questions,
)


# =================================================================================================
# 8. inherited_receipts.json
# =================================================================================================

inherited_receipts = {
    "stage": "Stage28-0",
    "parent_commit": EXPECTED_PARENT,
    "repository": "themubasshir/ids2018-validation-safe-ablation",
    "artifacts": inherited_receipt_entries,
    "inheritance_rule": (
        "These artifacts are frozen provenance inputs. Stage28 may not silently "
        "alter their memberships, taxonomy, feature representation, model "
        "hyperparameters, target semantics, or threshold-selection rules."
    ),
}

write_json(
    LOCK_DIR / "inherited_receipts.json",
    inherited_receipts,
)


# =================================================================================================
# 9. seed_spec.json
# =================================================================================================

seed_spec = {
    "stage": "Stage28-0",
    "training_seeds": FROZEN_SEEDS,
    "reference_seed": REFERENCE_SEED,
    "new_training_seeds": NEW_SEEDS,
    "number_of_training_seeds": len(FROZEN_SEEDS),

    "stage28a": {
        "stage22_membership": "INHERIT_EXACT_STAGE22R_MEMBERSHIPS",
        "stage27_membership": "INHERIT_EXACT_STAGE27_FOLD_MEMBERSHIPS",
        "membership_changes_across_training_seed": False,
        "intended_varying_factor": "TRAINING_REALIZATION_ONLY",
    },

    "stage28b": {
        "random_membership_seed": STAGE28B_SPLIT_SEED,
        "membership_seed_varies_with_model_seed": False,
        "random_membership_materialized_once_per_family": True,
        "same_membership_reused_for_all_five_model_seeds": True,
        "reason": (
            "Training-seed uncertainty must not be confounded with split-membership "
            "uncertainty. Random LOAO geometry is frozen once; only learner training "
            "realization changes across seeds."
        ),
    },

    "per_fit_seed_receipt_required": [
        "model_seed",
        "library_random_state",
        "data_order_seed",
        "thread_count",
        "determinism_controls",
    ],

    "data_order_policy": (
        "Do not deliberately reshuffle parent memberships for Stage28A. "
        "For Stage28B preserve the exact frozen materialized row order after "
        "membership creation. No additional seed-dependent row-order perturbation "
        "is authorized."
    ),

    "forbidden": [
        "ADD_SEED_AFTER_RESULTS",
        "REMOVE_SEED_AFTER_RESULTS",
        "BEST_SEED_SELECTION",
        "SEED_SPECIFIC_HYPERPARAMETER_CHANGE",
        "SEED_SPECIFIC_FEATURE_CHANGE",
        "SEED_SPECIFIC_THRESHOLD_RULE_CHANGE",
    ],
}

write_json(
    LOCK_DIR / "seed_spec.json",
    seed_spec,
)


# =================================================================================================
# 10. fit_budget.json
# =================================================================================================

fit_budget = {
    "stage": "Stage28-0",

    "stage22_full_seed_stability": {
        "scientific_unit": "ENS_LGBM_XGB_EQUAL",
        "split_cells": [
            "RANDOM_NATURAL",
            "CHRONOLOGICAL_NATURAL",
        ],
        "component_learners": [
            "LIGHTGBM",
            "XGBOOST",
        ],
        "seeds": FROZEN_SEEDS,
        "total_component_realizations": STAGE22_TOTAL_COMPONENT_REALIZATIONS,
        "existing_seed42_component_fits_reused": STAGE22_REUSED_SEED42_COMPONENT_FITS,
        "new_component_fits": STAGE22_NEW_COMPONENT_FITS,
    },

    "stage27_chronological_loao_seed_stability": {
        "families": ELIGIBLE_FAMILIES,
        "learners": LEARNERS,
        "seeds": FROZEN_SEEDS,
        "total_component_realizations": STAGE27_TOTAL_COMPONENT_REALIZATIONS,
        "existing_seed42_fits_reused": STAGE27_REUSED_SEED42_FITS,
        "new_fits": STAGE27_NEW_FITS,
    },

    "stage28b_random_loao": {
        "families": ELIGIBLE_FAMILIES,
        "learners": LEARNERS,
        "seeds": FROZEN_SEEDS,
        "total_component_realizations": STAGE28B_TOTAL_COMPONENT_REALIZATIONS,
        "existing_fits_reused": STAGE28B_REUSED_FITS,
        "new_fits": STAGE28B_NEW_FITS,
    },

    "totals": {
        "existing_seed42_fits_reused": TOTAL_EXISTING_REUSED,
        "new_fit_budget": TOTAL_NEW_FITS,
        "total_component_realizations_including_reuse": TOTAL_COMPONENT_REALIZATIONS,
        "scientific_evaluation_cells_including_reuse": TOTAL_SCIENTIFIC_EVALUATION_CELLS,
    },

    "hard_rule": (
        "No model fit outside the frozen new_fit_budget=106 is authorized. "
        "An operational retry of the exact same frozen fit does not create a new "
        "scientific realization but must retain a failed-attempt receipt."
    ),

    "reuse_failure_rule": (
        "If any expected seed-42 artifact cannot establish scientific equivalence, "
        "execution must stop. A PRE-RESULT protocol amendment must account for any "
        "replacement fit before the affected fit is executed."
    ),
}

write_json(
    LOCK_DIR / "fit_budget.json",
    fit_budget,
)


# =================================================================================================
# 11. model_inventory.json
# =================================================================================================

stage22_lgbm_params = dict(
    stage22_random["models"]["lightgbm"]["executed_parameters"]
)

stage22_xgb_params = dict(
    stage22_random["models"]["xgboost"]["parameters"]
)

# Stage28 primary compute policy is CPU.
# Only backend values are normalized. All algorithmic parameters remain inherited.
stage22_lgbm_params["device_type"] = "cpu"
stage22_xgb_params["device"] = "cpu"

stage27_xgb_cpu = dict(
    stage27_models["primary_learner"]["stage27_cpu_parameters"]
)

stage27_lgbm_cpu = dict(
    stage27_models["replication_learner"]["stage27_cpu_parameters"]
)

model_inventory = {
    "stage": "Stage28-0",

    "stage22_full": {
        "scientific_unit": "EQUAL_WEIGHT_CLASSICAL_ENSEMBLE",
        "strategy": "ENS_LGBM_XGB_EQUAL",
        "probability_rule": "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "lightgbm": {
            "configuration_id": "LGBM_11",
            "base_cpu_parameters_seed42": stage22_lgbm_params,
            "seed_change_rule": (
                "For seed s, replace random_state=42 with random_state=s only."
            ),
        },

        "xgboost": {
            "configuration_id": "XGB_11",
            "base_cpu_parameters_seed42": stage22_xgb_params,
            "seed_change_rule": (
                "For seed s, replace random_state=42 with random_state=s only."
            ),
        },

        "new_hyperparameter_search": False,
        "component_weights_changed": False,
    },

    "loao": {
        "xgboost": {
            "configuration_id": "XGB_11",
            "role": "PRIMARY",
            "base_cpu_parameters_seed42": stage27_xgb_cpu,
            "seed_change_rule": (
                "For seed s, replace random_state=42 with random_state=s only."
            ),
        },

        "lightgbm": {
            "configuration_id": "LGBM_11",
            "role": "REPLICATION_SENSITIVITY",
            "base_cpu_parameters_seed42": stage27_lgbm_cpu,
            "seed_change_rule": (
                "For seed s, replace random_state=42 with random_state=s only."
            ),
        },

        "stage27_primary_ensemble": False,
        "stage28_loao_ensemble": False,
    },

    "prohibited": [
        "OPTUNA",
        "HYPERPARAMETER_SEARCH",
        "ARCHITECTURE_SEARCH",
        "EARLY_STOPPING_ADDITION",
        "TARGET_FAMILY_TUNING",
        "SEED_SPECIFIC_TUNING",
        "MODEL_SELECTION_USING_STAGE28_RESULTS",
    ],
}

write_json(
    LOCK_DIR / "model_inventory.json",
    model_inventory,
)


# =================================================================================================
# 12. compute_policy.json
# =================================================================================================

compute_policy = {
    "stage": "Stage28-0",
    "primary_device": "CPU",
    "gpu_required": False,
    "gpu_authorized_for_primary_stage28": False,

    "reason": (
        "Stage28 is a controlled robustness experiment. CPU execution is frozen "
        "to minimize backend heterogeneity relative to the Stage27 CPU execution "
        "and to avoid introducing GPU/CPU backend as an additional factor."
    ),

    "xgboost": {
        "device": "cpu",
        "tree_method": "hist",
    },

    "lightgbm": {
        "device_type": "cpu",
    },

    "backend_only_change_policy": (
        "No further backend substitution is permitted after Stage28-0 unless an "
        "operational failure requires a PRE-RESULT, MINIMAL, DOCUMENTED, COMMITTED, "
        "REMOTELY VERIFIED protocol amendment."
    ),

    "algorithmic_hyperparameter_change_authorized": False,

    "thread_policy": {
        "n_jobs": -1,
        "record_actual_runtime_cpu_count": True,
        "record_actual_thread_configuration_per_fit": True,
    },
}

write_json(
    LOCK_DIR / "compute_policy.json",
    compute_policy,
)


# =================================================================================================
# 13. stage22_cell_spec.json
# =================================================================================================

stage22_cell_spec = {
    "stage": "Stage28-0",
    "arm": "28A",

    "representation": "FULL",
    "feature_count": 70,

    "cells": [
        "RANDOM_NATURAL",
        "CHRONOLOGICAL_NATURAL",
    ],

    "membership": {
        "rule": "INHERIT_EXACT_STAGE22R_TRAIN_VALIDATION_MEMBERSHIPS",
        "new_random_split": False,
        "membership_may_change_by_training_seed": False,
    },

    "scientific_unit": {
        "strategy": "ENS_LGBM_XGB_EQUAL",
        "component_learners": [
            "LIGHTGBM",
            "XGBOOST",
        ],
        "probability_rule": "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",
    },

    "reference_seed": 42,
    "reference_seed_retraining": "FORBIDDEN_IF_DURABLE_ARTIFACT_IS_SCIENTIFICALLY_EQUIVALENT",

    "evaluation_population": {
        "name": "STAGE22R_SHARED_FINAL_SINGLE_HOLDOUT",
        "rule": (
            "All five seed realizations are evaluated on the same frozen Stage22R "
            "final holdout geometry used for the parent random-vs-chronological "
            "scientific comparison."
        ),
        "threshold_selection_on_final_holdout": "FORBIDDEN",
        "model_selection_on_final_holdout": "FORBIDDEN",
        "stage28_reuse_status": (
            "PRE-REGISTERED ROBUSTNESS INFERENCE ONLY; parent holdout is no longer "
            "blind, so Stage28 does not represent a new untouched holdout claim."
        ),
    },

    "forbidden": [
        "RANDOM_REBALANCED",
        "CHRONOLOGICAL_REBALANCED",
        "STAGE23_FEATURE_ABLATIONS",
        "SHAP_RECOMPUTATION",
        "SUBSET_SEARCH",
        "NEW_HOLDOUT",
    ],
}

write_json(
    LOCK_DIR / "stage22_cell_spec.json",
    stage22_cell_spec,
)


# =================================================================================================
# 14. loao_family_spec.json
# =================================================================================================

loao_family_spec = {
    "stage": "Stage28-0",

    "eligible_families": ELIGIBLE_FAMILIES,

    "structurally_ineligible_families_not_resurrected": [
        "DOS",
        "AUTH_BRUTE_FORCE",
    ],

    "learners": LEARNERS,

    "stage28a_chronological": {
        "rule": "INHERIT_EXACT_STAGE27_ELIGIBLE_FOLDS_AND_MEMBERSHIPS",
        "family_train_count": 0,
        "family_validation_count": 0,
        "target_population": "INHERIT_EXACT_STAGE27_TARGET_POPULATION",
    },

    "stage28b_random": {
        "family_train_count": 0,
        "family_validation_count": 0,
        "held_out_family_all_positive_rows_assigned_to_target": True,
    },

    "machine_assertions_required_before_every_loao_fit": [
        "held_out_family_train_count == 0",
        "held_out_family_validation_count == 0",
    ],

    "family_specific_results_primary": True,
    "single_aggregate_zero_day_score": "FORBIDDEN",
}

write_json(
    LOCK_DIR / "loao_family_spec.json",
    loao_family_spec,
)


# =================================================================================================
# 15. random_loao_split_spec.json
# =================================================================================================

random_loao_split_spec = {
    "stage": "Stage28-0",
    "arm": "28B",
    "name": "RANDOM_SPLIT_LOAO_CONTROL",

    "dataset_universe": "STAGE27_FULL_EFFECTIVE_TARGET_POPULATION_CICIDS2017",
    "taxonomy": "INHERIT_EXACT_STAGE27_TAXONOMY",
    "eligible_families": ELIGIBLE_FAMILIES,

    "membership_seed": STAGE28B_SPLIT_SEED,
    "membership_seed_fixed_across_model_seeds": True,

    "canonical_input_order": (
        "Use the exact inherited Stage27 effective-population row order. "
        "No value-based resorting, label-based resorting, or model-seed-dependent "
        "reordering is authorized before membership generation."
    ),

    "construction_per_held_out_family": {

        "step_1_remove_held_out_family": {
            "rule": (
                "Every positive row belonging to held-out family h is excluded from "
                "TRAIN and VALIDATION and assigned to TARGET_POSITIVE."
            ),
            "held_out_family_train_count": 0,
            "held_out_family_validation_count": 0,
        },

        "step_2_random_target_benign": {
            "population": "ALL_ELIGIBLE_BENIGN_ROWS_IN_STAGE27_EFFECTIVE_UNIVERSE",
            "operator": "sklearn.model_selection.train_test_split",
            "test_size": STAGE28B_TARGET_BENIGN_FRACTION,
            "shuffle": True,
            "random_state": STAGE28B_SPLIT_SEED,
            "stratify": None,
            "selected_test_partition_role": "TARGET_BENIGN",
            "remaining_partition_role": "BENIGN_DEVELOPMENT_POOL",
        },

        "step_3_development_pool": {
            "population": (
                "BENIGN_DEVELOPMENT_POOL + ALL_NON_HELD_OUT_ATTACK_ROWS"
            ),
            "target_known_attack_rows": 0,
        },

        "step_4_train_validation": {
            "operator": "sklearn.model_selection.train_test_split",
            "test_size": STAGE28B_VALIDATION_FRACTION_OF_REMAINING_DEVELOPMENT,
            "shuffle": True,
            "random_state": STAGE28B_SPLIT_SEED,
            "stratify": "BINARY_LABEL",
            "train_partition_role": "TRAIN",
            "test_partition_role": "VALIDATION",
        },

        "step_5_target": {
            "composition": [
                "ALL_HELD_OUT_FAMILY_ATTACK_ROWS",
                "RANDOMLY_SELECTED_TARGET_BENIGN_ROWS",
            ],
            "known_attack_families_in_primary_target": 0,
        },
    },

    "approximate_benign_geometry": {
        "train_fraction_of_all_benign": 0.64,
        "validation_fraction_of_all_benign": 0.16,
        "target_fraction_of_all_benign": 0.20,
    },

    "approximate_nonheldout_attack_geometry": {
        "train_fraction": 0.80,
        "validation_fraction": 0.20,
        "target_fraction": 0.0,
    },

    "held_out_attack_geometry": {
        "train_fraction": 0.0,
        "validation_fraction": 0.0,
        "target_fraction": 1.0,
    },

    "reason_for_fixed_membership_seed": (
        "Stage28B is intended to compare random versus chronology-first LOAO "
        "without adding random split membership as a second varying factor across "
        "training seeds."
    ),

    "leakage_safeguards": {
        "inherit_stage27_effective_population_cleaning": True,
        "inherit_available_duplicate_controls": True,
        "inherit_flow_identity_safeguards": True,
        "introduce_new_session_group_definition": False,
        "row_level_random_control": True,
    },

    "interpretation_limit": (
        "Stage28B is a random-evaluation control intended to isolate the effect of "
        "chronology relative to the chronology-first LOAO experiment. It is NOT "
        "treated as a deployment-realistic estimate."
    ),

    "membership_outputs_required_before_first_fit": [
        "per_family_membership_receipt.json",
        "train_membership_sha256",
        "validation_membership_sha256",
        "target_membership_sha256",
        "class_counts",
        "held_out_family_exclusion_assertions",
    ],
}

write_json(
    LOCK_DIR / "random_loao_split_spec.json",
    random_loao_split_spec,
)


# =================================================================================================
# 16. target_population_spec.json
# =================================================================================================

target_population_spec = {
    "stage": "Stage28-0",

    "stage28a_stage22": {
        "target": "STAGE22R_SHARED_FINAL_SINGLE_HOLDOUT",
        "same_target_across_training_seeds": True,
        "target_used_for_training": False,
        "target_used_for_threshold_selection": False,
    },

    "stage28a_stage27": {
        "target": "INHERIT_EXACT_STAGE27_FAMILY_SPECIFIC_TARGET_POPULATIONS",
        "same_target_across_training_seeds": True,
        "target_used_for_training": False,
        "target_used_for_threshold_selection": False,
    },

    "stage28b_random_loao": {
        "target_positive": "ALL_ROWS_OF_HELD_OUT_ATTACK_FAMILY",
        "target_negative": (
            "FIXED_20_PERCENT_RANDOM_SAMPLE_OF_ELIGIBLE_BENIGN_ROWS"
        ),
        "known_attack_rows_in_primary_target": 0,
        "same_target_across_five_model_seeds": True,
        "positive_only_target_forbidden": True,
        "prevalence_and_pr_chance_anchor_must_be_recorded": True,
    },

    "expected_held_out_attack_support": EXPECTED_STAGE27_TARGET_ATTACK_SUPPORT,

    "target_optimization": "FORBIDDEN",
}

write_json(
    LOCK_DIR / "target_population_spec.json",
    target_population_spec,
)


# =================================================================================================
# 17. threshold_policy.json
# =================================================================================================

threshold_policy = {
    "stage": "Stage28-0",

    "stage22_full": {
        "source": (
            "results/stage22r_training/stage22r_2a_random_natural/"
            "stage22r_2a_random_natural_result.json"
        ),

        "selection_population": "EACH_PARENT_CELL_FROZEN_DEVELOPMENT_VALIDATION",

        "standard": {
            "threshold": 0.50,
            "selected": False,
        },

        "balanced": {
            "objective": "MAX_F1",
            "tie_break_order": [
                "LOWER_FPR",
                "HIGHER_RECALL",
                "CLOSER_TO_0_50",
                "LOWER_THRESHOLD",
            ],
        },

        "security": {
            "constraint": "FPR <= 0.05 EXACT",
            "objective": "MAX_F2",
            "tie_break_order": [
                "LOWER_FPR",
                "HIGHER_RECALL",
                "LOWER_THRESHOLD",
            ],
            "no_relaxation": True,
        },

        "grid": {
            "integer_percent_start": 5,
            "integer_percent_stop_inclusive": 95,
            "integer_step": 1,
            "count": 91,
        },

        "selected_separately_per_training_seed": True,
        "final_holdout_threshold_search": "FORBIDDEN",
    },

    "loao_stage27_and_stage28b": {
        "source": (
            "results/stage27_loao_unseen_attack/"
            "stage27_0_protocol_lock/threshold_policy.json"
        ),

        "selection_population": (
            "KNOWN_FAMILY_VALIDATION_ONLY_WITH_ZERO_HELD_OUT_FAMILY_POSITIVES"
        ),

        "standard": {
            "threshold": 0.50,
            "selected": False,
        },

        "balanced": {
            "objective": "MAXIMIZE_F1",
            "tie_break_order": [
                "MINIMIZE_FPR",
                "CHOOSE_HIGHER_THRESHOLD",
            ],
        },

        "security": {
            "constraint": "FPR <= 0.05",
            "objective": "MAXIMIZE_F2",
            "tie_break_order": [
                "MINIMIZE_FPR",
                "CHOOSE_HIGHER_THRESHOLD",
            ],
            "if_no_feasible_threshold": (
                "SECURITY_THRESHOLD_INFEASIBLE_NO_RELAXATION"
            ),
        },

        "grid": {
            "construction": "INTEGER_PERCENT / 100",
            "integer_start": 1,
            "integer_stop_inclusive": 99,
            "integer_step": 1,
            "count": 99,
        },

        "selected_separately_per": [
            "HELD_OUT_FAMILY",
            "LEARNER",
            "TRAINING_SEED",
        ],

        "held_out_family_in_threshold_selection": "FORBIDDEN",
        "target_threshold_search": "FORBIDDEN",
        "post_target_threshold_change": "FORBIDDEN",
    },
}

write_json(
    LOCK_DIR / "threshold_policy.json",
    threshold_policy,
)


# =================================================================================================
# 18. support_policy.json
# =================================================================================================

support_policy = {
    "stage": "Stage28-0",

    "source": (
        "results/stage27_loao_unseen_attack/"
        "stage27_0_protocol_lock/support_rules.json"
    ),

    "minimum_positive_support_for_inferential_family_claim": (
        STAGE27_INFERENTIAL_MIN_POSITIVE_SUPPORT
    ),

    "below_threshold_rule": "DESCRIPTIVE_ONLY_NOT_DROPPED",

    "expected_held_out_positive_support": EXPECTED_STAGE27_TARGET_ATTACK_SUPPORT,

    "family_status": {
        family: (
            "INFERENTIAL_SUPPORT_ELIGIBLE"
            if count >= STAGE27_INFERENTIAL_MIN_POSITIVE_SUPPORT
            else "DESCRIPTIVE_ONLY_SUPPORT_LT_50"
        )
        for family, count in EXPECTED_STAGE27_TARGET_ATTACK_SUPPORT.items()
    },

    "infiltration_rule": (
        "INFILTRATION remains descriptive-only because frozen held-out positive "
        "support is 36 < 50. Random splitting may not be used to manufacture a "
        "stronger inferential status."
    ),
}

write_json(
    LOCK_DIR / "support_policy.json",
    support_policy,
)


# =================================================================================================
# 19. metric_spec.json
# =================================================================================================

metric_spec = {
    "stage": "Stage28-0",

    "stage22": {
        "ranking_metrics": [
            "PR_AUC",
            "ROC_AUC",
        ],
        "operating_metrics": [
            "PRECISION",
            "RECALL",
            "FPR",
            "F1",
            "F2_WHERE_NATIVE",
            "TP",
            "FP",
            "TN",
            "FN",
        ],
        "operating_points": [
            "STANDARD",
            "BALANCED",
            "SECURITY",
        ],
    },

    "loao": {
        "ranking_metrics": [
            "ROC_AUC",
            "PR_AUC",
            "PR_CHANCE_ANCHOR",
            "PR_EXCESS_WHERE_DEFINED",
            "PR_LIFT_WHERE_DEFINED",
        ],

        "operating_metrics": [
            "STANDARD_RECALL",
            "BALANCED_RECALL",
            "SECURITY_RECALL",
            "FPR",
            "PRECISION",
            "F1",
            "TP",
            "FP",
            "TN",
            "FN",
        ],

        "comparative_metrics": [
            "KNOWN_FAMILY_CONTROL_WHERE_NATIVE",
            "NOVELTY_GENERALIZATION_GAP_WHERE_MATHEMATICALLY_COMPATIBLE",
            "RANDOM_VS_CHRONOLOGICAL_LOAO_CONTRAST",
        ],

        "undefined_metric_policy": "PRESERVE_AS_NAN_DO_NOT_COERCE",
    },

    "seed_summary_statistics": {
        "mean": True,
        "median": True,
        "standard_deviation": {
            "enabled": True,
            "definition": "SAMPLE_STANDARD_DEVIATION",
            "ddof": 1,
        },
        "minimum": True,
        "maximum": True,
        "range": "MAX_MINUS_MIN",
        "iqr": {
            "enabled": True,
            "definition": "Q75_MINUS_Q25",
            "percentile_method": "linear",
        },
    },

    "aggregation_rule": (
        "Family-specific LOAO results remain primary. No single aggregate "
        "zero-day score is authorized."
    ),
}

write_json(
    LOCK_DIR / "metric_spec.json",
    metric_spec,
)


# =================================================================================================
# 20. seed_uncertainty_spec.json
# =================================================================================================

seed_uncertainty_spec = {
    "stage": "Stage28-0",

    "bootstrap_uncertainty": {
        "meaning": (
            "Target-sampling uncertainty conditional on an already fitted model."
        ),
    },

    "training_seed_uncertainty": {
        "meaning": (
            "Sensitivity to fitted-model realization under the frozen training "
            "procedure."
        ),
    },

    "combine_into_single_synthetic_ci": False,

    "report_separately": True,

    "five_seed_statistics": [
        "mean",
        "median",
        "sample_standard_deviation_ddof_1",
        "minimum",
        "maximum",
        "range",
        "IQR_Q75_minus_Q25_linear",
    ],

    "best_seed_reporting": "FORBIDDEN",
    "seed42_only_reporting_if_other_seeds_exist": "FORBIDDEN",
}

write_json(
    LOCK_DIR / "seed_uncertainty_spec.json",
    seed_uncertainty_spec,
)


# =================================================================================================
# 21. conclusion_stability_spec.json
# =================================================================================================

conclusion_stability_spec = {
    "stage": "Stage28-0",

    "output_required": (
        "results/stage28_stability_novelty_control/"
        "stage28_3_seed_uncertainty/conclusion_stability.csv"
    ),

    "required_fields": [
        "claim_id",
        "parent_stage",
        "family_if_applicable",
        "learner_if_applicable",
        "seed",
        "claim_condition",
        "condition_met",
    ],

    "stability_rate": (
        "number_of_frozen_seeds_supporting_condition / 5"
    ),

    "stage22_directional_claims": [
        {
            "claim_id": "STAGE22_PR_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT",
            "condition": (
                "PR_AUC_RANDOM_NATURAL < PR_AUC_CHRONOLOGICAL_NATURAL"
            ),
            "reason": (
                "This freezes the direction observed in the already-frozen "
                "Stage22R shared final holdout rather than assuming the common "
                "but unsupported direction that random must outperform chronology."
            ),
        },
        {
            "claim_id": "STAGE22_ROC_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT",
            "condition": (
                "ROC_AUC_RANDOM_NATURAL < ROC_AUC_CHRONOLOGICAL_NATURAL"
            ),
        },
    ],

    "loao_qualitative_conditions": [
        {
            "id": "ROC_ABOVE_CHANCE",
            "condition": "ROC_AUC > 0.5",
        },
        {
            "id": "PR_ABOVE_CHANCE",
            "condition": "PR_AUC > PR_CHANCE_ANCHOR",
        },
        {
            "id": "STANDARD_DETECTION_PRESENT",
            "condition": "STANDARD_RECALL > 0",
        },
        {
            "id": "BALANCED_DETECTION_PRESENT",
            "condition": "BALANCED_RECALL > 0",
        },
        {
            "id": "SECURITY_DETECTION_PRESENT_WHERE_FEASIBLE",
            "condition": (
                "SECURITY_THRESHOLD_FEASIBLE AND SECURITY_RECALL > 0"
            ),
        },
        {
            "id": "LEARNER_ORDER_ROC_STABILITY",
            "condition": (
                "sign(XGB_ROC_AUC - LGBM_ROC_AUC) equals the frozen seed42 sign "
                "for the same family"
            ),
        },
    ],

    "interpretation": (
        "Conclusion-stability analysis is descriptive robustness analysis, "
        "not a newly optimized significance test."
    ),

    "post_result_condition_creation": "FORBIDDEN",
}

write_json(
    LOCK_DIR / "conclusion_stability_spec.json",
    conclusion_stability_spec,
)


# =================================================================================================
# 22. interpretation_matrix.json
# =================================================================================================

interpretation_matrix = {
    "stage": "Stage28-0",

    "rules": [
        {
            "result": "TEMPORAL_GAP_STABLE_ACROSS_ALL_TESTED_SEEDS",
            "permitted_interpretation": (
                "Stage22 temporal conclusion is robust to the five tested "
                "training realizations."
            ),
        },
        {
            "result": "TEMPORAL_GAP_CHANGES_SIGN_ACROSS_SEEDS",
            "permitted_interpretation": (
                "Temporal conclusion is materially realization-sensitive."
            ),
        },
        {
            "result": "LOAO_FAMILY_RESULT_STABLE_ACROSS_SEEDS",
            "permitted_interpretation": (
                "Family-specific novelty result is robust under the five tested seeds."
            ),
        },
        {
            "result": "LOAO_VARIES_STRONGLY_BY_SEED",
            "permitted_interpretation": (
                "Unseen-family performance is realization-sensitive."
            ),
        },
        {
            "result": "LEARNER_GAP_MUCH_GREATER_THAN_SEED_SPREAD",
            "permitted_interpretation": (
                "Stage27 learner dependence is unlikely to be explained merely by "
                "training-seed realization."
            ),
        },
        {
            "result": "LEARNER_GAP_COMPARABLE_TO_SEED_SPREAD",
            "permitted_interpretation": (
                "Stage27 learner dependence should be interpreted cautiously "
                "relative to realization variability."
            ),
        },
        {
            "result": "RANDOM_LOAO_MUCH_GREATER_THAN_CHRONOLOGICAL_LOAO",
            "permitted_interpretation": (
                "Chronology/distribution shift materially compounds unseen-family "
                "difficulty for that family under the frozen benchmark."
            ),
        },
        {
            "result": "RANDOM_AND_CHRONOLOGICAL_LOAO_BOTH_COLLAPSE",
            "permitted_interpretation": (
                "Poor unseen-family transfer persists even without the "
                "chronology-first evaluation geometry."
            ),
        },
        {
            "result": "RANDOM_AND_CHRONOLOGICAL_LOAO_BOTH_SURVIVE",
            "permitted_interpretation": (
                "Transfer for that family survives both tested evaluation geometries."
            ),
        },
        {
            "result": "FAMILY_SPECIFIC_MIXTURE",
            "permitted_interpretation": (
                "The novelty–chronology relationship is family dependent."
            ),
        },
        {
            "result": "RANKING_STABLE_THRESHOLD_RECALL_UNSTABLE",
            "permitted_interpretation": (
                "Ranking robustness does not imply stable deployment operating points."
            ),
        },
        {
            "result": "UNEXPECTED_REVERSAL",
            "permitted_interpretation": (
                "Report the reversal as observed; do not modify the protocol."
            ),
        },
    ],
}

write_json(
    LOCK_DIR / "interpretation_matrix.json",
    interpretation_matrix,
)


# =================================================================================================
# 23. prohibited_claims.json
# =================================================================================================

prohibited_claims = {
    "stage": "Stage28-0",

    "prohibited": [
        "Five seeds prove universal reproducibility.",
        "Random LOAO represents deployment.",
        "The random-vs-chronological difference is purely caused by temporal drift.",
        "Stage28 completely separates novelty and chronology.",
        "Seed stability proves causal behavioral learning.",
        "Seed instability invalidates the model.",
        "LOAO equals real zero-day detection.",
    ],

    "preferred_language": [
        "robust across the five tested seeds",
        "realization-sensitive",
        "random-vs-chronological LOAO contrast",
        "consistent with chronology compounding novelty difficulty",
        "unseen-family proxy",
        "conditional on the benchmark and frozen protocol",
    ],
}

write_json(
    LOCK_DIR / "prohibited_claims.json",
    prohibited_claims,
)


# =================================================================================================
# 24. Freeze policy files first; then build freeze_record
# =================================================================================================

policy_files = sorted(
    p
    for p in LOCK_DIR.glob("*.json")
    if p.name != "freeze_record.json"
)

policy_hashes = {
    p.name: sha256_file(p)
    for p in policy_files
}


# =================================================================================================
# 25. freeze_record.json
# =================================================================================================

freeze_record = {
    "stage": "Stage28-0",
    "status": "PROTOCOL_FROZEN_PRE_FIT",
    "timestamp_utc": timestamp_utc,

    "repository": "themubasshir/ids2018-validation-safe-ablation",
    "branch": EXPECTED_BRANCH,
    "parent_commit": EXPECTED_PARENT,

    "arms": [
        "28A_TRAINING_SEED_STABILITY",
        "28B_RANDOM_SPLIT_LOAO_CONTROL",
    ],

    "frozen_seeds": FROZEN_SEEDS,

    "eligible_loao_families": ELIGIBLE_FAMILIES,

    "fit_budget": {
        "existing_seed42_fits_reused": TOTAL_EXISTING_REUSED,
        "new_fit_budget": TOTAL_NEW_FITS,
        "total_component_realizations": TOTAL_COMPONENT_REALIZATIONS,
    },

    "science_operations_before_freeze": {
        "model_fits": 0,
        "model_inference": 0,
        "threshold_selection": 0,
        "target_openings": 0,
        "bootstrap_recomputation": 0,
        "new_formal_statistical_tests": 0,
    },

    "anti_adaptation_rules": [
        "NO_ADDING_SEEDS_AFTER_RESULTS",
        "NO_REMOVING_SEEDS_AFTER_RESULTS",
        "NO_BEST_SEED_SELECTION",
        "NO_HYPERPARAMETER_CHANGE",
        "NO_LEARNER_CHANGE",
        "NO_FEATURE_REPRESENTATION_CHANGE",
        "NO_FAMILY_ELIGIBILITY_CHANGE",
        "NO_TARGET_POPULATION_CHANGE",
        "NO_THRESHOLD_GRID_CHANGE",
        "NO_TARGET_DERIVED_THRESHOLD_SELECTION",
        "NO_METRIC_ADDITION_OR_REMOVAL_AFTER_RESULTS",
        "NO_SUPPORT_THRESHOLD_CHANGE",
        "NO_NEW_RANDOM_SPLIT_AFTER_RESULTS",
        "NO_NEURAL_ARCHITECTURE_ADDITION",
        "NO_STAGE27_TARGET_REOPENING_FOR_OPTIMIZATION",
    ],

    "failure_handling": {
        "record_failure": True,
        "operational_retry_of_exact_frozen_fit": True,
        "preserve_failed_attempt_receipt": True,
        "parameter_change_to_obtain_success": False,
        "amendment_if_needed": [
            "PRE_RESULT",
            "MINIMAL",
            "DOCUMENTED",
            "COMMITTED",
            "REMOTELY_VERIFIED",
        ],
    },

    "final_stopping_rule": (
        "Stage28 is the final experimental stage authorized for the current "
        "manuscript. After Stage28 closure, no additional model architecture, "
        "dataset, ablation, stress test, seed expansion, threshold experiment, "
        "feature analysis, or evaluation axis may be added to the current "
        "empirical study without opening a separately scoped future research project."
    ),

    "after_stage28": "STOP_EXPERIMENTS_AND_MOVE_TO_MANUSCRIPT_SYNTHESIS",

    "protocol_file_sha256_before_freeze_record": policy_hashes,
}

write_json(
    LOCK_DIR / "freeze_record.json",
    freeze_record,
)


# =================================================================================================
# 26. FINAL CHECKSUM MANIFEST
# =================================================================================================

json_files = sorted(LOCK_DIR.glob("*.json"))

checksum_lines = []

for path in json_files:
    checksum_lines.append(
        f"{sha256_file(path)}  {path.name}"
    )

checksums_path = LOCK_DIR / "checksums.sha256"

checksums_path.write_text(
    "\n".join(checksum_lines) + "\n",
    encoding="utf-8",
)

print()
print(SEP)
print("STAGE28-0 ARTIFACTS MATERIALIZED")
print(SEP)
print()

for path in sorted(LOCK_DIR.iterdir()):
    if path.is_file():
        print(
            f"{path.name:38s} "
            f"{path.stat().st_size:10,d} bytes  "
            f"{sha256_file(path)}"
        )


# =================================================================================================
# 27. MACHINE AUDIT BEFORE COMMIT
# =================================================================================================

print()
print(SEP)
print("PRE-COMMIT MACHINE AUDIT")
print(SEP)
print()

EXPECTED_JSON_FILES = {
    "scientific_questions.json",
    "inherited_receipts.json",
    "seed_spec.json",
    "fit_budget.json",
    "model_inventory.json",
    "compute_policy.json",
    "stage22_cell_spec.json",
    "loao_family_spec.json",
    "random_loao_split_spec.json",
    "target_population_spec.json",
    "threshold_policy.json",
    "support_policy.json",
    "metric_spec.json",
    "seed_uncertainty_spec.json",
    "conclusion_stability_spec.json",
    "interpretation_matrix.json",
    "prohibited_claims.json",
    "freeze_record.json",
}

actual_json_files = {
    p.name
    for p in LOCK_DIR.glob("*.json")
}

if actual_json_files != EXPECTED_JSON_FILES:
    raise RuntimeError(
        "Unexpected Stage28-0 JSON artifact set.\n"
        f"expected={sorted(EXPECTED_JSON_FILES)}\n"
        f"actual={sorted(actual_json_files)}"
    )

# Re-read key files from disk.
budget_check = read_json(LOCK_DIR / "fit_budget.json")
seed_check = read_json(LOCK_DIR / "seed_spec.json")
compute_check = read_json(LOCK_DIR / "compute_policy.json")
random_check = read_json(LOCK_DIR / "random_loao_split_spec.json")
freeze_check = read_json(LOCK_DIR / "freeze_record.json")

assert seed_check["training_seeds"] == [42, 43, 44, 45, 46]
assert budget_check["totals"]["new_fit_budget"] == 106
assert budget_check["totals"]["existing_seed42_fits_reused"] == 14
assert budget_check["totals"]["total_component_realizations_including_reuse"] == 120

assert compute_check["primary_device"] == "CPU"
assert compute_check["gpu_required"] is False
assert compute_check["gpu_authorized_for_primary_stage28"] is False

assert random_check["membership_seed"] == 42
assert random_check["membership_seed_fixed_across_model_seeds"] is True

assert freeze_check["science_operations_before_freeze"]["model_fits"] == 0
assert freeze_check["science_operations_before_freeze"]["model_inference"] == 0
assert freeze_check["science_operations_before_freeze"]["threshold_selection"] == 0
assert freeze_check["science_operations_before_freeze"]["target_openings"] == 0

print("[PASS] exact seed set              :", FROZEN_SEEDS)
print("[PASS] existing seed42 fits reused :", TOTAL_EXISTING_REUSED)
print("[PASS] new fit budget              :", TOTAL_NEW_FITS)
print("[PASS] total component universe     :", TOTAL_COMPONENT_REALIZATIONS)
print("[PASS] primary compute              : CPU")
print("[PASS] Stage28B membership seed     :", STAGE28B_SPLIT_SEED)
print("[PASS] Stage28B membership fixed    : YES")
print("[PASS] held-out family train count  : MUST == 0")
print("[PASS] held-out family valid count  : MUST == 0")
print("[PASS] model fits before freeze     : 0")
print("[PASS] inference before freeze      : 0")
print("[PASS] threshold selection          : 0")
print("[PASS] target openings              : 0")


# =================================================================================================
# 28. GIT DIFF AUDIT
# =================================================================================================

print()
print(SEP)
print("GIT DIFF AUDIT")
print(SEP)
print()

status_after_write = git("status", "--porcelain")

print(status_after_write)

# Only Stage28 root may be newly modified.
unexpected_lines = []

for line in status_after_write.splitlines():
    if not line.strip():
        continue

    path_text = line[3:].strip()

    if not path_text.startswith(
        "results/stage28_stability_novelty_control/"
    ):
        unexpected_lines.append(line)

if unexpected_lines:
    raise RuntimeError(
        "Unexpected repository modifications detected:\n"
        + "\n".join(unexpected_lines)
    )

print()
print("[PASS] Only Stage28 protocol artifacts are modified.")


# =================================================================================================
# 29. STAGE EXACT FILES — DO NOT USE git add .
# =================================================================================================

files_to_stage = sorted(
    p.relative_to(REPO).as_posix()
    for p in LOCK_DIR.iterdir()
    if p.is_file()
)

for relative in files_to_stage:
    git("add", "--", relative)

cached_names = git("diff", "--cached", "--name-only").splitlines()

if sorted(cached_names) != sorted(files_to_stage):
    raise RuntimeError(
        "Staged file set mismatch.\n\n"
        f"Expected:\n{files_to_stage}\n\n"
        f"Actual:\n{cached_names}"
    )

# whitespace sanity
git("diff", "--cached", "--check")

print("[PASS] Exact Stage28-0 file set staged.")
print()
for p in cached_names:
    print("  ", p)


# =================================================================================================
# 30. COMMIT
# =================================================================================================

print()
print(SEP)
print("COMMIT STAGE28-0")
print(SEP)
print()

git("commit", "-m", COMMIT_MESSAGE)

stage28_freeze_commit = git("rev-parse", "HEAD")

if stage28_freeze_commit == EXPECTED_PARENT:
    raise RuntimeError(
        "Commit did not advance HEAD."
    )

commit_parent = git(
    "rev-parse",
    f"{stage28_freeze_commit}^"
)

if commit_parent != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage28-0 commit parent mismatch.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={commit_parent}"
    )

print("Stage28-0 commit:", stage28_freeze_commit)
print("Parent          :", commit_parent)


# =================================================================================================
# 31. PUSH
# =================================================================================================

print()
print(SEP)
print("PUSH STAGE28-0")
print(SEP)
print()

# Bootstrap cell already configured authentication.
# This intentionally does not print or inspect the GitHub token.
push_result = run(
    [
        "git",
        "push",
        "origin",
        f"{EXPECTED_BRANCH}:{EXPECTED_BRANCH}",
    ],
    cwd=REPO,
    check=False,
)

if push_result.returncode != 0:
    raise RuntimeError(
        "Stage28-0 push failed.\n\n"
        f"STDOUT:\n{push_result.stdout or ''}\n\n"
        f"STDERR:\n{push_result.stderr or ''}"
    )

print(push_result.stdout or "")
print(push_result.stderr or "")


# =================================================================================================
# 32. REMOTE VERIFICATION
# =================================================================================================

print()
print(SEP)
print("REMOTE VERIFICATION")
print(SEP)
print()

remote_after = remote_main_sha()

print("Local HEAD :", stage28_freeze_commit)
print("Remote main:", remote_after)

if remote_after != stage28_freeze_commit:
    raise RuntimeError(
        "REMOTE FREEZE VERIFICATION FAILED.\n"
        f"local={stage28_freeze_commit}\n"
        f"remote={remote_after}"
    )

# Refresh remote-tracking ref.
git("fetch", "origin", EXPECTED_BRANCH)

origin_main = git(
    "rev-parse",
    f"origin/{EXPECTED_BRANCH}"
)

if origin_main != stage28_freeze_commit:
    raise RuntimeError(
        "origin/main does not match Stage28-0 freeze commit."
    )

status_final = git("status", "--porcelain")

if status_final:
    raise RuntimeError(
        "Repository is not clean after Stage28-0 commit/push:\n"
        + status_final
    )


# =================================================================================================
# 33. FINAL SEALED STATUS
# =================================================================================================

print()
print(SEP)
print("STAGE28-0 PROTOCOL LOCK — COMPLETE / REMOTELY VERIFIED")
print(SEP)
print()

print("Repository:")
print(" ", REPO)
print()

print("Parent commit:")
print(" ", EXPECTED_PARENT)
print()

print("Stage28-0 freeze commit:")
print(" ", stage28_freeze_commit)
print()

print("Remote origin/main:")
print(" ", remote_after)
print()

print("Frozen Stage28 arms:")
print("  28A — TRAINING-SEED STABILITY")
print("  28B — RANDOM-SPLIT LOAO CONTROL")
print()

print("Frozen seeds:")
print(" ", FROZEN_SEEDS)
print()

print("Eligible LOAO families:")
for family in ELIGIBLE_FAMILIES:
    print(" ", family)

print()
print("Fit accounting:")
print("  existing seed42 component fits reused :", TOTAL_EXISTING_REUSED)
print("  NEW authorized fits                   :", TOTAL_NEW_FITS)
print("  total component realizations          :", TOTAL_COMPONENT_REALIZATIONS)
print("  scientific evaluation cells           :", TOTAL_SCIENTIFIC_EVALUATION_CELLS)

print()
print("Compute:")
print("  PRIMARY_DEVICE = CPU")
print("  GPU_REQUIRED   = FALSE")
print()

print("Science operations consumed by Stage28-0:")
print("  MODEL_FITS          = 0")
print("  MODEL_INFERENCE     = 0")
print("  THRESHOLD_SELECTION = 0")
print("  TARGET_OPENINGS     = 0")
print()

print("Stage28B:")
print("  split membership seed = 42")
print("  memberships frozen once per family")
print("  memberships DO NOT vary across model seeds")
print("  held-out-family TRAIN count      == 0")
print("  held-out-family VALIDATION count == 0")
print()

print("NEXT AUTHORIZED STEP:")
print("  Stage28-1 — materialize/audit frozen memberships and")
print("  execution manifests BEFORE the first new fit.")
print()

print("NO MODEL TRAINING HAS BEEN AUTHORIZED IN THIS CELL.")
print()
print(SEP)


STAGE28-0 — EXACT REPOSITORY STATE

Repository : /kaggle/working/ids2018-validation-safe-ablation
Branch     : main
HEAD       : 66597724a692da577efd19c1ca070fb27750f3ee
Remote main: 66597724a692da577efd19c1ca070fb27750f3ee

[EXACT] Stage28 parent verified.
[EXACT] Local main == remote main.
[EXACT] Worktree clean.
[EXACT] Stage28-0 destination absent.

VERIFY INHERITED STAGE22R + STAGE27 PROVENANCE

[OK] results/stage22r_training/stage22r_2a_random_natural/stage22r_2a_random_natural_result.json
     sha256=dd7951f6d9c92538944a04357bab14fcc28f82ade748d37ad710fff2051abf73
[OK] results/stage22r_training/stage22r_2a_random_natural/random_natural_lightgbm_model.txt
     sha256=9a92f4b8cd26738a470a7ff6fb72f51422a803e10b36bb32ac4bfc91373e6e50
[OK] results/stage22r_training/stage22r_2a_random_natural/random_natural_xgboost_model.json
     sha256=ac9e630c8a479f073953b7f05eccf8bc94d24eaf0d8bc6bf77f5954979235b4e
[OK] results/stage22r_training/stage22r_2c_chronological_natural/stage22r_2c_chrono

In [4]:
# =================================================================================================
# STAGE28-0A — PRE-EXECUTION PROTOCOL AMENDMENT
#
# Purpose:
#   1. remove Stage22 seed/backend confounding
#   2. clarify Stage28B attack-population semantics
#   3. explicitly inherit Stage27 LOAO class-weight semantics
#
# IMPORTANT:
#   - PRE-RESULT
#   - NO DATASET OPENING
#   - NO MODEL FITTING
#   - NO MODEL INFERENCE
#   - NO THRESHOLD SELECTION
#   - NO TARGET OPENING
#
# Base Stage28-0 freeze:
#   55a70a6e8c111339f087b8f5a0f20dc63f8adb13
#
# Revised fit accounting:
#   EXISTING REUSED = 12
#   NEW FITS        = 108
#   TOTAL COMPONENT REALIZATIONS = 120
# =================================================================================================

from __future__ import annotations

import json
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =================================================================================================
# CONSTANTS
# =================================================================================================

SEP = "=" * 110

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

BRANCH = "main"

BASE_STAGE28_FREEZE = (
    "55a70a6e8c111339f087b8f5a0f20dc63f8adb13"
)

ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

BASE_LOCK = (
    ROOT
    / "stage28_0_protocol_lock"
)

AMEND_DIR = (
    ROOT
    / "stage28_0a_preexecution_amendment"
)

COMMIT_MESSAGE = (
    "stage28-0a: remove seed-backend confound and clarify LOAO population"
)

SEEDS = [42, 43, 44, 45, 46]

FAMILIES = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]


# =================================================================================================
# HELPERS
# =================================================================================================

def run(cmd, cwd=REPO, check=True):
    r = subprocess.run(
        cmd,
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed ({r.returncode}):\n"
            f"{' '.join(cmd)}\n\n"
            f"STDOUT:\n{r.stdout}\n\n"
            f"STDERR:\n{r.stderr}"
        )

    return r


def git(*args, check=True):
    r = run(
        ["git", *args],
        check=check,
    )
    return (r.stdout or "").strip()


def read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            sort_keys=False,
        ) + "\n",
        encoding="utf-8",
    )


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            block = f.read(1024 * 1024)
            if not block:
                break
            h.update(block)

    return h.hexdigest()


def require(path: Path):
    if not path.is_file():
        raise RuntimeError(
            f"Required artifact missing:\n{path}"
        )
    return path


def remote_main():
    x = git(
        "ls-remote",
        "origin",
        "refs/heads/main",
    )

    if not x:
        raise RuntimeError(
            "Unable to resolve origin/main."
        )

    return x.split()[0]


# =================================================================================================
# 1. VERIFY EXACT BASE
# =================================================================================================

print()
print(SEP)
print("STAGE28-0A — VERIFY EXACT PRE-AMENDMENT STATE")
print(SEP)
print()

branch = git("branch", "--show-current")
head = git("rev-parse", "HEAD")
remote = remote_main()
status = git("status", "--porcelain")

print("Branch      :", branch)
print("Local HEAD  :", head)
print("Remote main :", remote)
print("Expected    :", BASE_STAGE28_FREEZE)
print()

if branch != BRANCH:
    raise RuntimeError(
        f"Expected branch {BRANCH}; got {branch}"
    )

if head != BASE_STAGE28_FREEZE:
    raise RuntimeError(
        "Unexpected local HEAD.\n"
        f"expected={BASE_STAGE28_FREEZE}\n"
        f"actual={head}"
    )

if remote != BASE_STAGE28_FREEZE:
    raise RuntimeError(
        "Unexpected remote HEAD.\n"
        f"expected={BASE_STAGE28_FREEZE}\n"
        f"actual={remote}"
    )

if status:
    raise RuntimeError(
        "Worktree must be clean before amendment:\n"
        + status
    )

if AMEND_DIR.exists():
    raise RuntimeError(
        "Amendment directory already exists:\n"
        f"{AMEND_DIR}"
    )

print("[PASS] exact Stage28-0 freeze verified")
print("[PASS] local == remote")
print("[PASS] clean worktree")


# =================================================================================================
# 2. LOAD BASE STAGE28-0
# =================================================================================================

base_budget_path = require(
    BASE_LOCK / "fit_budget.json"
)

base_compute_path = require(
    BASE_LOCK / "compute_policy.json"
)

base_random_path = require(
    BASE_LOCK / "random_loao_split_spec.json"
)

base_models_path = require(
    BASE_LOCK / "model_inventory.json"
)

base_freeze_path = require(
    BASE_LOCK / "freeze_record.json"
)

base_budget = read_json(base_budget_path)
base_compute = read_json(base_compute_path)
base_random = read_json(base_random_path)
base_models = read_json(base_models_path)
base_freeze = read_json(base_freeze_path)


# Base assertions
assert (
    base_budget["totals"]["existing_seed42_fits_reused"]
    == 14
)

assert (
    base_budget["totals"]["new_fit_budget"]
    == 106
)

assert (
    base_budget["totals"][
        "total_component_realizations_including_reuse"
    ]
    == 120
)

assert base_compute["primary_device"] == "CPU"

assert (
    base_freeze["science_operations_before_freeze"][
        "model_fits"
    ]
    == 0
)

print()
print("[PASS] base Stage28-0 budget = 106 new fits")
print("[PASS] base compute policy = CPU")


# =================================================================================================
# 3. VERIFY THE STAGE22 BACKEND CONFOUND FROM DURABLE RECEIPTS
# =================================================================================================

print()
print(SEP)
print("VERIFY STAGE22 SEED-42 BACKENDS")
print(SEP)
print()

random_result_path = require(
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
    / "stage22r_2a_random_natural_result.json"
)

chrono_result_path = require(
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
    / "stage22r_2c_chronological_natural_result.json"
)

random_result = read_json(random_result_path)
chrono_result = read_json(chrono_result_path)


def audit_stage22_parent(name, result):
    lgb = result["models"]["lightgbm"]
    xgb = result["models"]["xgboost"]

    lgb_device = (
        lgb["executed_parameters"]["device_type"]
    )

    xgb_device = (
        xgb["parameters"]["device"]
    )

    lgb_seed = (
        lgb["executed_parameters"]["random_state"]
    )

    xgb_seed = (
        xgb["parameters"]["random_state"]
    )

    print(f"{name}:")
    print("  LightGBM backend :", lgb_device)
    print("  XGBoost backend  :", xgb_device)
    print("  LightGBM seed    :", lgb_seed)
    print("  XGBoost seed     :", xgb_seed)

    if lgb_device != "cpu":
        raise RuntimeError(
            f"{name}: expected inherited LightGBM CPU."
        )

    if xgb_device != "cuda":
        raise RuntimeError(
            f"{name}: expected inherited XGBoost CUDA."
        )

    if lgb_seed != 42 or xgb_seed != 42:
        raise RuntimeError(
            f"{name}: expected reference seed 42."
        )


audit_stage22_parent(
    "RANDOM_NATURAL",
    random_result,
)

audit_stage22_parent(
    "CHRONOLOGICAL_NATURAL",
    chrono_result,
)

print()
print(
    "[CONFIRMED] Reusing Stage22 seed42 XGBoost "
    "would confound seed with backend."
)


# =================================================================================================
# 4. VERIFY CICIDS2017 TARGET-ONLY-UNSEEN POPULATION
# =================================================================================================

print()
print(SEP)
print("VERIFY STAGE27 TAXONOMY EDGE CASE")
print(SEP)
print()

census_path = require(
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
    / "canonical_label_census.json"
)

census = read_json(census_path)

family_counts = census["family_counts"]

heartbleed_count = family_counts[
    "TARGET_ONLY_UNSEEN"
]

other_unseen_count = family_counts[
    "OTHER_ATTACK_UNSEEN_LABEL"
]

print(
    "TARGET_ONLY_UNSEEN rows       :",
    heartbleed_count,
)

print(
    "OTHER_ATTACK_UNSEEN_LABEL rows:",
    other_unseen_count,
)

if heartbleed_count != 11:
    raise RuntimeError(
        "Expected exactly 11 TARGET_ONLY_UNSEEN rows."
    )

if other_unseen_count != 0:
    raise RuntimeError(
        "Expected zero OTHER_ATTACK_UNSEEN_LABEL rows."
    )

print()
print(
    "[CONFIRMED] Stage28B attack pool must explicitly "
    "exclude target-only-unseen rows."
)


# =================================================================================================
# 5. REVISED FIT ACCOUNTING
# =================================================================================================

print()
print(SEP)
print("REVISED FIT ACCOUNTING")
print(SEP)
print()

# Stage22:
#
# 2 cells × 2 learners × 5 seeds = 20 total components.
#
# Reusable seed42:
#   LightGBM RANDOM_NATURAL        = 1
#   LightGBM CHRONOLOGICAL_NATURAL = 1
#
# NOT reusable for Stage28 seed-stability:
#   XGBoost RANDOM_NATURAL seed42        = CUDA parent
#   XGBoost CHRONOLOGICAL_NATURAL seed42 = CUDA parent
#
# These two XGB seed42 components must be refitted on CPU.
#
# Thus:
#   reused Stage22 = 2
#   new Stage22    = 18
#
# Stage27:
#   reused = 10
#   new    = 40
#
# Stage28B:
#   reused = 0
#   new    = 50

STAGE22_TOTAL = 20
STAGE22_REUSED = 2
STAGE22_NEW = 18

STAGE27_TOTAL = 50
STAGE27_REUSED = 10
STAGE27_NEW = 40

STAGE28B_TOTAL = 50
STAGE28B_REUSED = 0
STAGE28B_NEW = 50

TOTAL_COMPONENTS = (
    STAGE22_TOTAL
    + STAGE27_TOTAL
    + STAGE28B_TOTAL
)

TOTAL_REUSED = (
    STAGE22_REUSED
    + STAGE27_REUSED
    + STAGE28B_REUSED
)

TOTAL_NEW = (
    STAGE22_NEW
    + STAGE27_NEW
    + STAGE28B_NEW
)

assert TOTAL_COMPONENTS == 120
assert TOTAL_REUSED == 12
assert TOTAL_NEW == 108
assert TOTAL_REUSED + TOTAL_NEW == TOTAL_COMPONENTS

print("Stage22 reused :", STAGE22_REUSED)
print("Stage22 NEW    :", STAGE22_NEW)

print()
print("Stage27 reused :", STAGE27_REUSED)
print("Stage27 NEW    :", STAGE27_NEW)

print()
print("Stage28B NEW   :", STAGE28B_NEW)

print()
print("TOTAL REUSED   :", TOTAL_REUSED)
print("TOTAL NEW      :", TOTAL_NEW)
print("TOTAL UNIVERSE :", TOTAL_COMPONENTS)


# =================================================================================================
# 6. CREATE AMENDMENT DIRECTORY
# =================================================================================================

AMEND_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

timestamp = datetime.now(
    timezone.utc
).isoformat()


# =================================================================================================
# 7. AMENDMENT RECORD
# =================================================================================================

amendment = {
    "stage": "Stage28-0A",

    "type": (
        "PRE_RESULT_MINIMAL_EXECUTION_SEMANTICS_AMENDMENT"
    ),

    "timestamp_utc": timestamp,

    "base_stage28_freeze_commit": (
        BASE_STAGE28_FREEZE
    ),

    "status": (
        "FROZEN_BEFORE_ANY_STAGE28_MODEL_FIT"
    ),

    "reasons": {

        "stage22_seed_backend_confound": {
            "problem": (
                "Stage28-0 required CPU execution for new "
                "Stage28 realizations while proposing reuse "
                "of Stage22 seed42 XGBoost artifacts that "
                "were executed on CUDA."
            ),

            "scientific_risk": (
                "The seed42 versus seeds43-46 comparison "
                "would confound training seed with execution "
                "backend."
            ),

            "observed_parent_backends": {
                "RANDOM_NATURAL": {
                    "LIGHTGBM": "cpu",
                    "XGBOOST": "cuda",
                },
                "CHRONOLOGICAL_NATURAL": {
                    "LIGHTGBM": "cpu",
                    "XGBOOST": "cuda",
                },
            },

            "resolution": (
                "Reuse the two seed42 LightGBM CPU models. "
                "Do NOT use the two inherited seed42 CUDA "
                "XGBoost models in Stage28 seed-stability "
                "statistics. Refit exactly those two XGB_11 "
                "seed42 components on CPU under the frozen "
                "Stage28 compute policy."
            ),

            "parent_stage22_cuda_models": (
                "RETAINED_AS_HISTORICAL_PARENT_ARTIFACTS_"
                "BUT_EXCLUDED_FROM_STAGE28_SEED_STATISTICS"
            ),

            "stage28_seed42_ensemble": (
                "REUSED_SEED42_CPU_LIGHTGBM_PLUS_"
                "NEW_SEED42_CPU_XGBOOST_EQUAL_WEIGHT"
            ),

            "algorithmic_hyperparameter_change": False,

            "seed_change": False,

            "feature_change": False,

            "split_change": False,

            "threshold_rule_change": False,
        },

        "random_loao_attack_population_ambiguity": {
            "problem": (
                "The Stage28-0 phrase "
                "'ALL_NON_HELD_OUT_ATTACK_ROWS' could be "
                "read to include TARGET_ONLY_UNSEEN rows."
            ),

            "observed_target_only_unseen_rows": (
                heartbleed_count
            ),

            "resolution": (
                "For Stage28B TRAIN and VALIDATION, attack "
                "rows are restricted to the Stage27 PRIMARY "
                "SEVEN attack-family taxonomy excluding the "
                "currently held-out family."
            ),

            "target_only_unseen_train_count": 0,
            "target_only_unseen_validation_count": 0,

            "other_unseen_train_count": 0,
            "other_unseen_validation_count": 0,

            "held_out_family_train_count": 0,
            "held_out_family_validation_count": 0,

            "primary_target": (
                "HELD_OUT_FAMILY_ATTACK_ROWS_PLUS_"
                "FROZEN_RANDOM_TARGET_BENIGN_ROWS_ONLY"
            ),
        },
    },

    "fit_budget_revision": {
        "before": {
            "existing_reused": 14,
            "new_fits": 106,
            "total_components": 120,
        },

        "after": {
            "existing_reused": 12,
            "new_fits": 108,
            "total_components": 120,
        },

        "net_new_fit_change": 2,

        "reason": (
            "Exactly two Stage22 seed42 XGBoost CPU "
            "replacement fits are newly authorized."
        ),
    },

    "science_operations_before_amendment": {
        "model_fits": 0,
        "model_inference": 0,
        "threshold_selection": 0,
        "target_openings": 0,
        "bootstrap_recomputation": 0,
        "formal_tests": 0,
    },

    "base_protocol_immutability": (
        "Stage28-0 files are not modified. This amendment "
        "is additive and controls execution whenever it "
        "clarifies or supersedes conflicting execution "
        "semantics in Stage28-0."
    ),

    "post_result_adaptation": False,
}

write_json(
    AMEND_DIR
    / "stage28_0a_amendment.json",
    amendment,
)


# =================================================================================================
# 8. EFFECTIVE FIT BUDGET
# =================================================================================================

effective_budget = {
    "stage": "Stage28-0A",

    "supersedes_for_execution": (
        "stage28_0_protocol_lock/fit_budget.json"
    ),

    "stage22_full_seed_stability": {
        "scientific_unit": (
            "ENS_LGBM_XGB_EQUAL"
        ),

        "cells": [
            "RANDOM_NATURAL",
            "CHRONOLOGICAL_NATURAL",
        ],

        "seeds": SEEDS,

        "total_component_realizations": 20,

        "reused": [
            {
                "cell": "RANDOM_NATURAL",
                "learner": "LIGHTGBM",
                "seed": 42,
                "backend": "cpu",
            },
            {
                "cell": "CHRONOLOGICAL_NATURAL",
                "learner": "LIGHTGBM",
                "seed": 42,
                "backend": "cpu",
            },
        ],

        "historical_not_reused": [
            {
                "cell": "RANDOM_NATURAL",
                "learner": "XGBOOST",
                "seed": 42,
                "historical_backend": "cuda",
                "reason": (
                    "BACKEND_MISMATCH_WITH_STAGE28_CPU_POLICY"
                ),
            },
            {
                "cell": "CHRONOLOGICAL_NATURAL",
                "learner": "XGBOOST",
                "seed": 42,
                "historical_backend": "cuda",
                "reason": (
                    "BACKEND_MISMATCH_WITH_STAGE28_CPU_POLICY"
                ),
            },
        ],

        "new_fits": 18,

        "new_fit_breakdown": {
            "seed42_xgboost_cpu_replacements": 2,
            "seeds43_to_46_all_components": 16,
        },
    },

    "stage27_chronological_loao": {
        "total_component_realizations": 50,
        "existing_seed42_fits_reused": 10,
        "new_fits": 40,
    },

    "stage28b_random_loao": {
        "total_component_realizations": 50,
        "existing_fits_reused": 0,
        "new_fits": 50,
    },

    "totals": {
        "existing_reused": 12,
        "new_fit_budget": 108,
        "total_component_realizations": 120,
        "scientific_evaluation_cells": 110,
    },

    "hard_limit": (
        "No more than 108 new successful scientific "
        "model fits are authorized under Stage28."
    ),
}

write_json(
    AMEND_DIR
    / "effective_fit_budget.json",
    effective_budget,
)


# =================================================================================================
# 9. EFFECTIVE STAGE22 EXECUTION POLICY
# =================================================================================================

stage22_execution = {
    "stage": "Stage28-0A",

    "compute_policy": "CPU",

    "cells": [
        "RANDOM_NATURAL",
        "CHRONOLOGICAL_NATURAL",
    ],

    "seeds": SEEDS,

    "lightgbm": {
        "configuration": "LGBM_11",
        "backend": "cpu",

        "seed42": (
            "REUSE_DURABLE_PARENT_CPU_MODEL"
        ),

        "seeds43_to_46": (
            "NEW_CPU_FIT"
        ),
    },

    "xgboost": {
        "configuration": "XGB_11",
        "backend": "cpu",

        "seed42": (
            "NEW_CPU_REFIT_REQUIRED"
        ),

        "seeds43_to_46": (
            "NEW_CPU_FIT"
        ),

        "historical_seed42_cuda_model": (
            "NOT_USED_IN_STAGE28_SEED_VARIANCE"
        ),
    },

    "ensemble_probability": (
        "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST"
    ),

    "algorithmic_hyperparameters_changed": False,

    "seed42_parent_result_usage": {
        "historical_stage22_result": (
            "RETAIN_FOR_PARENT_STAGE_REPORTING"
        ),

        "stage28_seed_statistics": (
            "USE_CPU_NORMALIZED_STAGE28_SEED42_ENSEMBLE"
        ),

        "do_not_substitute_parent_cuda_xgb_result": True,
    },
}

write_json(
    AMEND_DIR
    / "effective_stage22_execution_policy.json",
    stage22_execution,
)


# =================================================================================================
# 10. EFFECTIVE RANDOM-LOAO POPULATION SPEC
# =================================================================================================

random_loao_effective = {
    "stage": "Stage28-0A",

    "base_spec": (
        "stage28_0_protocol_lock/"
        "random_loao_split_spec.json"
    ),

    "eligible_held_out_families": FAMILIES,

    "primary_seven_attack_families": [
        "BOT",
        "DDOS",
        "DOS",
        "AUTH_BRUTE_FORCE",
        "INFILTRATION",
        "PORT_SCAN",
        "WEB_ATTACK",
    ],

    "per_held_out_family": {

        "train_validation_attack_population": (
            "PRIMARY_SEVEN_ATTACK_FAMILIES_"
            "EXCLUDING_HELD_OUT_FAMILY"
        ),

        "excluded_attack_categories": [
            "TARGET_ONLY_UNSEEN",
            "OTHER_ATTACK_UNSEEN_LABEL",
        ],

        "held_out_family": {
            "train_count": 0,
            "validation_count": 0,
            "all_attack_rows_assigned_to_primary_target": True,
        },

        "target_only_unseen": {
            "train_count": 0,
            "validation_count": 0,
            "primary_target_count": 0,
        },

        "other_attack_unseen_label": {
            "train_count": 0,
            "validation_count": 0,
            "primary_target_count": 0,
        },

        "primary_target": (
            "HELD_OUT_FAMILY_ATTACKS_PLUS_"
            "FROZEN_RANDOM_BENIGN_TARGET"
        ),
    },

    "benign_target_split": {
        "fraction": 0.20,
        "random_state": 42,
        "shuffle": True,
        "stratify": None,
    },

    "development_train_validation_split": {
        "validation_fraction": 0.20,
        "random_state": 42,
        "shuffle": True,
        "stratify": "BINARY_LABEL",
    },

    "membership_fixed_across_training_seeds": True,

    "interpretation": (
        "Random LOAO control; not a deployment estimate."
    ),
}

write_json(
    AMEND_DIR
    / "effective_random_loao_population_spec.json",
    random_loao_effective,
)


# =================================================================================================
# 11. EFFECTIVE LOAO CLASS-WEIGHT POLICY
# =================================================================================================

class_weight_effective = {
    "stage": "Stage28-0A",

    "source": (
        "results/stage27_loao_unseen_attack/"
        "stage27_0_protocol_lock/"
        "class_weight_policy.json"
    ),

    "applies_to": [
        "STAGE28A_STAGE27_CHRONOLOGICAL_LOAO",
        "STAGE28B_RANDOM_LOAO",
    ],

    "negative_class_weight": 1.0,

    "positive_class_weight": (
        "train_benign / train_attack"
    ),

    "implementation": (
        "Construct fit-time sample_weight with BENIGN=1.0 "
        "and ATTACK=(train_benign/train_attack) for both "
        "XGBoost and LightGBM."
    ),

    "stage28a_chronological": (
        "Same frozen Stage27 memberships therefore same "
        "per-fold class-weight values as Stage27."
    ),

    "stage28b_random": (
        "Recompute once from each final frozen random-LOAO "
        "TRAIN membership before fitting and reuse the "
        "result for all five model seeds of that family."
    ),

    "validation_rows_used": 0,
    "target_rows_used": 0,

    "weight_search": False,

    "single_class_train": (
        "HARD_SCIENTIFIC_INFEASIBILITY_BEFORE_MODEL_FIT"
    ),
}

write_json(
    AMEND_DIR
    / "effective_loao_class_weight_policy.json",
    class_weight_effective,
)


# =================================================================================================
# 12. AMENDMENT FREEZE RECEIPT
# =================================================================================================

files_before_receipt = sorted(
    AMEND_DIR.glob("*.json")
)

hashes_before_receipt = {
    p.name: sha256_file(p)
    for p in files_before_receipt
}

receipt = {
    "stage": "Stage28-0A",

    "status": (
        "PRE_EXECUTION_AMENDMENT_FROZEN"
    ),

    "timestamp_utc": timestamp,

    "base_commit": BASE_STAGE28_FREEZE,

    "effective_fit_budget": {
        "reused": 12,
        "new": 108,
        "total": 120,
    },

    "scientific_operations_consumed": {
        "model_fits": 0,
        "model_inference": 0,
        "threshold_selection": 0,
        "target_openings": 0,
    },

    "amendments": [
        "STAGE22_SEED42_XGB_CPU_REFIT",
        "STAGE28B_PRIMARY_SEVEN_ATTACK_POOL_EXPLICIT",
        "LOAO_CLASS_WEIGHT_INHERITANCE_EXPLICIT",
    ],

    "unchanged": [
        "SEED_SET",
        "FEATURE_REPRESENTATION",
        "MODEL_HYPERPARAMETERS",
        "LEARNERS",
        "ENSEMBLE_WEIGHTS",
        "ELIGIBLE_FAMILIES",
        "TARGET_BENIGN_FRACTION",
        "RANDOM_MEMBERSHIP_SEED",
        "THRESHOLD_RULES",
        "METRICS",
        "SUPPORT_THRESHOLD",
        "INTERPRETATION_RULES",
    ],

    "file_sha256": hashes_before_receipt,

    "next_authorized_step": (
        "Stage28-1 membership reconstruction/materialization "
        "and execution-manifest freeze before first new fit."
    ),
}

write_json(
    AMEND_DIR
    / "amendment_freeze_receipt.json",
    receipt,
)


# =================================================================================================
# 13. CHECKSUM MANIFEST
# =================================================================================================

json_files = sorted(
    AMEND_DIR.glob("*.json")
)

lines = []

for p in json_files:
    lines.append(
        f"{sha256_file(p)}  {p.name}"
    )

checksum_path = (
    AMEND_DIR / "checksums.sha256"
)

checksum_path.write_text(
    "\n".join(lines) + "\n",
    encoding="utf-8",
)


# =================================================================================================
# 14. MACHINE AUDIT
# =================================================================================================

print()
print(SEP)
print("STAGE28-0A MACHINE AUDIT")
print(SEP)
print()

budget_check = read_json(
    AMEND_DIR
    / "effective_fit_budget.json"
)

stage22_check = read_json(
    AMEND_DIR
    / "effective_stage22_execution_policy.json"
)

random_check = read_json(
    AMEND_DIR
    / "effective_random_loao_population_spec.json"
)

amend_check = read_json(
    AMEND_DIR
    / "stage28_0a_amendment.json"
)

assert (
    budget_check["totals"]["new_fit_budget"]
    == 108
)

assert (
    budget_check["totals"]["existing_reused"]
    == 12
)

assert (
    budget_check["totals"][
        "total_component_realizations"
    ]
    == 120
)

assert (
    stage22_check["xgboost"]["seed42"]
    == "NEW_CPU_REFIT_REQUIRED"
)

assert (
    stage22_check["lightgbm"]["seed42"]
    == "REUSE_DURABLE_PARENT_CPU_MODEL"
)

assert (
    random_check["per_held_out_family"][
        "target_only_unseen"
    ]["train_count"]
    == 0
)

assert (
    random_check["per_held_out_family"][
        "target_only_unseen"
    ]["validation_count"]
    == 0
)

assert (
    amend_check[
        "science_operations_before_amendment"
    ]["model_fits"]
    == 0
)

print("[PASS] Stage22 seed42 XGB -> CPU refit")
print("[PASS] Stage22 seed42 LGBM -> reuse CPU")
print("[PASS] revised new-fit budget = 108")
print("[PASS] revised reused fits = 12")
print("[PASS] total component universe = 120")
print("[PASS] TARGET_ONLY_UNSEEN train = 0")
print("[PASS] TARGET_ONLY_UNSEEN validation = 0")
print("[PASS] model fits consumed = 0")
print("[PASS] inference consumed = 0")
print("[PASS] threshold selection consumed = 0")
print("[PASS] target openings consumed = 0")


# =================================================================================================
# 15. SHOW ARTIFACTS
# =================================================================================================

print()
print(SEP)
print("AMENDMENT ARTIFACTS")
print(SEP)
print()

for p in sorted(AMEND_DIR.iterdir()):
    if p.is_file():
        print(
            f"{p.name:48s} "
            f"{p.stat().st_size:9,d} bytes  "
            f"{sha256_file(p)}"
        )


# =================================================================================================
# 16. GIT AUDIT
# =================================================================================================

print()
print(SEP)
print("GIT AUDIT")
print(SEP)
print()

status = git("status", "--porcelain")

print(status)

unexpected = []

for line in status.splitlines():
    if not line.strip():
        continue

    path = line[3:].strip()

    if not path.startswith(
        "results/stage28_stability_novelty_control/"
        "stage28_0a_preexecution_amendment/"
    ):
        unexpected.append(line)

if unexpected:
    raise RuntimeError(
        "Unexpected repository modifications:\n"
        + "\n".join(unexpected)
    )

print()
print(
    "[PASS] only Stage28-0A amendment "
    "artifacts modified"
)


# =================================================================================================
# 17. STAGE EXACT FILES
# =================================================================================================

files_to_stage = sorted(
    p.relative_to(REPO).as_posix()
    for p in AMEND_DIR.iterdir()
    if p.is_file()
)

for relative in files_to_stage:
    git(
        "add",
        "--",
        relative,
    )

staged = git(
    "diff",
    "--cached",
    "--name-only",
).splitlines()

if sorted(staged) != sorted(files_to_stage):
    raise RuntimeError(
        "Staged file set mismatch."
    )

git(
    "diff",
    "--cached",
    "--check",
)

print()
print("[PASS] exact file set staged")

for path in staged:
    print(" ", path)


# =================================================================================================
# 18. COMMIT
# =================================================================================================

print()
print(SEP)
print("COMMIT STAGE28-0A")
print(SEP)
print()

git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
)

amend_commit = git(
    "rev-parse",
    "HEAD",
)

parent = git(
    "rev-parse",
    f"{amend_commit}^",
)

if parent != BASE_STAGE28_FREEZE:
    raise RuntimeError(
        "Amendment commit parent mismatch.\n"
        f"expected={BASE_STAGE28_FREEZE}\n"
        f"actual={parent}"
    )

print("Amendment commit:", amend_commit)
print("Parent          :", parent)


# =================================================================================================
# 19. PUSH
# =================================================================================================

print()
print(SEP)
print("PUSH STAGE28-0A")
print(SEP)
print()

push = run(
    [
        "git",
        "push",
        "origin",
        "main:main",
    ],
    check=False,
)

if push.returncode != 0:
    raise RuntimeError(
        "Push failed.\n\n"
        f"STDOUT:\n{push.stdout}\n\n"
        f"STDERR:\n{push.stderr}"
    )

print(push.stdout)
print(push.stderr)


# =================================================================================================
# 20. REMOTE VERIFY
# =================================================================================================

print()
print(SEP)
print("REMOTE VERIFICATION")
print(SEP)
print()

remote_after = remote_main()

print("Local HEAD :", amend_commit)
print("Remote main:", remote_after)

if remote_after != amend_commit:
    raise RuntimeError(
        "Remote verification failed."
    )

git(
    "fetch",
    "origin",
    "main",
)

origin_main = git(
    "rev-parse",
    "origin/main",
)

if origin_main != amend_commit:
    raise RuntimeError(
        "origin/main mismatch after fetch."
    )

final_status = git(
    "status",
    "--porcelain",
)

if final_status:
    raise RuntimeError(
        "Worktree not clean after amendment:\n"
        + final_status
    )


# =================================================================================================
# 21. FINAL STATUS
# =================================================================================================

print()
print(SEP)
print(
    "STAGE28-0A PRE-EXECUTION AMENDMENT — "
    "COMPLETE / REMOTELY VERIFIED"
)
print(SEP)
print()

print("Base Stage28-0:")
print(" ", BASE_STAGE28_FREEZE)

print()
print("Stage28-0A commit:")
print(" ", amend_commit)

print()
print("Effective fit accounting:")
print("  existing reused = 12")
print("  NEW fits        = 108")
print("  total universe  = 120")

print()
print("Stage22 seed42:")
print("  LightGBM RANDOM_NATURAL        -> REUSE CPU")
print("  LightGBM CHRONOLOGICAL_NATURAL -> REUSE CPU")
print("  XGBoost RANDOM_NATURAL         -> REFIT CPU")
print("  XGBoost CHRONOLOGICAL_NATURAL  -> REFIT CPU")

print()
print("Stage28B attack population:")
print(
    "  TRAIN/VALIDATION = primary-seven attacks "
    "excluding held-out family"
)
print("  TARGET_ONLY_UNSEEN = excluded")
print("  OTHER_UNSEEN       = excluded")

print()
print("Science operations:")
print("  MODEL_FITS          = 0")
print("  MODEL_INFERENCE     = 0")
print("  THRESHOLD_SELECTION = 0")
print("  TARGET_OPENINGS     = 0")

print()
print("NEXT AUTHORIZED STEP:")
print(
    "  Stage28-1 — reconstruct/audit inherited "
    "memberships, materialize random-LOAO memberships,"
)
print(
    "  and freeze the 120-component execution manifest "
    "before the first fit."
)

print()
print(SEP)


STAGE28-0A — VERIFY EXACT PRE-AMENDMENT STATE

Branch      : main
Local HEAD  : 55a70a6e8c111339f087b8f5a0f20dc63f8adb13
Remote main : 55a70a6e8c111339f087b8f5a0f20dc63f8adb13
Expected    : 55a70a6e8c111339f087b8f5a0f20dc63f8adb13

[PASS] exact Stage28-0 freeze verified
[PASS] local == remote
[PASS] clean worktree

[PASS] base Stage28-0 budget = 106 new fits
[PASS] base compute policy = CPU

VERIFY STAGE22 SEED-42 BACKENDS

RANDOM_NATURAL:
  LightGBM backend : cpu
  XGBoost backend  : cuda
  LightGBM seed    : 42
  XGBoost seed     : 42
CHRONOLOGICAL_NATURAL:
  LightGBM backend : cpu
  XGBoost backend  : cuda
  LightGBM seed    : 42
  XGBoost seed     : 42

[CONFIRMED] Reusing Stage22 seed42 XGBoost would confound seed with backend.

VERIFY STAGE27 TAXONOMY EDGE CASE

TARGET_ONLY_UNSEEN rows       : 11
OTHER_ATTACK_UNSEEN_LABEL rows: 0

[CONFIRMED] Stage28B attack pool must explicitly exclude target-only-unseen rows.

REVISED FIT ACCOUNTING

Stage22 reused : 2
Stage22 NEW    : 18

Sta

In [6]:
# =================================================================================================
# STAGE28-1A — INHERITED MEMBERSHIP RECONSTRUCTION + BYTE/CONTENT AUDIT
#
# Scientific purpose:
#   A. Audit Stage22R frozen FULL memberships (no reconstruction of scientific split).
#   B. Locate the 8 exact CICIDS2017 Stage27 source assets in /kaggle/input.
#   C. Rebuild Stage27 global family/day/source code arrays from LABELS ONLY.
#   D. Require exact historical content SHA256 reproduction.
#   E. Rebuild the five eligible Stage27 chronology-first LOAO memberships.
#   F. Require exact historical membership content SHA256 reproduction.
#   G. Freeze reconstruction receipts and push them to GitHub.
#
# IMPORTANT:
#   - NO PREDICTOR VALUES READ
#   - NO MODEL FITTING
#   - NO MODEL INFERENCE
#   - NO THRESHOLD SELECTION
#   - NO TARGET PREDICTOR OPENING
#
# Expected parent:
#   0029c28c417d63ffb45ad4096ab7c44cbd875116
#
# Runtime caches are NOT committed to Git.
# Only small reconstruction receipts are committed.
# =================================================================================================

from __future__ import annotations

import hashlib
import json
import math
import re
import subprocess
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 118

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

INPUT_ROOT = Path("/kaggle/input").resolve()

EXPECTED_BRANCH = "main"

EXPECTED_PARENT = (
    "0029c28c417d63ffb45ad4096ab7c44cbd875116"
)

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

OUTPUT_DIR = (
    STAGE28_ROOT
    / "stage28_1a_inherited_membership_reconstruction"
)

RUNTIME_ROOT = Path(
    "/kaggle/working/stage28_1a_runtime_cache"
)

GLOBAL_CACHE_ROOT = (
    RUNTIME_ROOT
    / "stage27_global_cache"
)

CHRONO_CACHE_ROOT = (
    RUNTIME_ROOT
    / "stage27_chronological_memberships"
)

COMMIT_MESSAGE = (
    "stage28-1a: reconstruct and verify inherited memberships"
)

ELIGIBLE_FAMILIES = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

PRIMARY_SEVEN = [
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def run(cmd, cwd=REPO, check=True):
    r = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed ({r.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{r.stdout}\n\n"
            f"STDERR:\n{r.stderr}"
        )

    return r


def git(*args, check=True):
    return (
        run(
            ["git", *args],
            check=check,
        ).stdout
        or ""
    ).strip()


def read_json(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def write_json(path: Path, obj):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            sort_keys=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(
    path: Path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array_content(arr: np.ndarray):
    arr = np.ascontiguousarray(arr)

    h = hashlib.sha256()
    h.update(arr.tobytes(order="C"))

    return h.hexdigest()


def require_file(path: Path):
    if not path.is_file():
        raise RuntimeError(
            f"Required artifact missing:\n{path}"
        )

    return path


def remote_main_sha():
    out = git(
        "ls-remote",
        "origin",
        "refs/heads/main",
    )

    if not out:
        raise RuntimeError(
            "Unable to resolve origin/main."
        )

    return out.split()[0]


def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


# =================================================================================================
# 2. EXACT REPOSITORY GATE
# =================================================================================================

banner(
    "STAGE28-1A — EXACT REPOSITORY GATE"
)

if not REPO.is_dir():
    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )

branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = remote_main_sha()

status = git(
    "status",
    "--porcelain",
)

print("Repository :", REPO)
print("Branch     :", branch)
print("Local HEAD :", head)
print("Remote main:", remote)
print("Expected   :", EXPECTED_PARENT)

if branch != EXPECTED_BRANCH:
    raise RuntimeError(
        f"Expected branch {EXPECTED_BRANCH}; "
        f"got {branch}"
    )

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected local HEAD.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )

if remote != EXPECTED_PARENT:
    raise RuntimeError(
        "Unexpected remote main.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={remote}"
    )

if status:
    raise RuntimeError(
        "Worktree must be clean before Stage28-1A:\n"
        + status
    )

if OUTPUT_DIR.exists():
    raise RuntimeError(
        "Stage28-1A output directory already exists:\n"
        f"{OUTPUT_DIR}"
    )

print()
print("[PASS] exact Stage28-0A parent")
print("[PASS] local == remote")
print("[PASS] worktree clean")


# =================================================================================================
# 3. LOAD FROZEN STAGE28 / STAGE27 PROVENANCE
# =================================================================================================

banner(
    "LOAD FROZEN PROVENANCE"
)

stage28_amendment = require_file(
    STAGE28_ROOT
    / "stage28_0a_preexecution_amendment"
    / "stage28_0a_amendment.json"
)

effective_budget = require_file(
    STAGE28_ROOT
    / "stage28_0a_preexecution_amendment"
    / "effective_fit_budget.json"
)

source_receipt_path = require_file(
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
    / "source_effective_population_receipt.json"
)

family_cache_receipt_path = require_file(
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
    / "family_code_cache_receipt.json"
)

census_path = require_file(
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
    / "canonical_label_census.json"
)

source_receipt = read_json(
    source_receipt_path
)

family_cache_receipt = read_json(
    family_cache_receipt_path
)

census = read_json(
    census_path
)

budget = read_json(
    effective_budget
)

assert (
    budget["totals"]["new_fit_budget"]
    == 108
)

assert (
    budget["totals"]["existing_reused"]
    == 12
)

assert (
    budget["totals"][
        "total_component_realizations"
    ]
    == 120
)

print("[PASS] effective Stage28 new-fit budget = 108")
print("[PASS] effective Stage28 reused fits = 12")
print("[PASS] component universe = 120")

print()
print(
    "Frozen CICIDS2017 effective rows:",
    source_receipt["population"]["effective_rows"],
)

if (
    source_receipt["population"]["effective_rows"]
    != 2_830_743
):
    raise RuntimeError(
        "Unexpected Stage27 effective population."
    )


# =================================================================================================
# 4. STAGE22 MEMBERSHIP AUDIT
# =================================================================================================

banner(
    "STAGE22R FROZEN MEMBERSHIP AUDIT"
)

stage22_membership_dir = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
)

stage22_summary_path = require_file(
    stage22_membership_dir
    / "stage22r_1b1_membership_summary.json"
)

stage22_summary = read_json(
    stage22_summary_path
)

random_bitset_path = require_file(
    stage22_membership_dir
    / "random_validation.packbits"
)

expected_random_sha = (
    stage22_summary["artifacts"][
        "random_validation.packbits"
    ]["sha256"]
)

actual_random_sha = sha256_file(
    random_bitset_path
)

logical_length = (
    stage22_summary["artifacts"][
        "random_validation.packbits"
    ]["logical_length"]
)

expected_population = (
    stage22_summary["artifacts"][
        "random_validation.packbits"
    ]["population"]
)

print("Random membership:")
print("  path       :", random_bitset_path)
print("  expected sha:", expected_random_sha)
print("  actual sha  :", actual_random_sha)

if actual_random_sha != expected_random_sha:
    raise RuntimeError(
        "Stage22 random_validation.packbits "
        "SHA256 mismatch."
    )

packed = np.fromfile(
    random_bitset_path,
    dtype=np.uint8,
)

bits = np.unpackbits(
    packed,
    bitorder="little",
)[:logical_length]

actual_population = int(
    bits.sum()
)

if actual_population != expected_population:
    raise RuntimeError(
        "Stage22 random validation population mismatch.\n"
        f"expected={expected_population}\n"
        f"actual={actual_population}"
    )

if logical_length != 14_412_403:
    raise RuntimeError(
        "Unexpected Stage22 clean development length."
    )

stage22_cells = stage22_summary["cells"]

assert (
    stage22_cells[
        "RANDOM_NATURAL"
    ]["train"]["rows"]
    == 11_529_922
)

assert (
    stage22_cells[
        "RANDOM_NATURAL"
    ]["validation"]["rows"]
    == 2_882_481
)

assert (
    stage22_cells[
        "CHRONOLOGICAL_NATURAL"
    ]["train"]["rows"]
    == 13_818_623
)

assert (
    stage22_cells[
        "CHRONOLOGICAL_NATURAL"
    ]["validation"]["rows"]
    == 593_780
)

print()
print("[PASS] Stage22 random membership SHA exact")
print(
    "[PASS] random validation population:",
    actual_population,
)
print(
    "[PASS] chronological membership counts exact"
)

del bits
del packed


# =================================================================================================
# 5. DISCOVER EXACT CICIDS2017 SOURCE ASSETS
# =================================================================================================

banner(
    "DISCOVER EXACT CICIDS2017 SOURCE ASSETS"
)

if not INPUT_ROOT.exists():
    raise RuntimeError(
        "/kaggle/input does not exist."
    )

all_input_files = [
    p
    for p in INPUT_ROOT.rglob("*")
    if p.is_file()
]

print(
    "Total /kaggle/input files:",
    len(all_input_files),
)

by_name = {}

for p in all_input_files:
    by_name.setdefault(
        p.name.casefold(),
        [],
    ).append(p)

resolved_sources = []

for segment in source_receipt["segments"]:

    basename = segment["basename"]
    expected_sha = segment["sha256"]

    candidates = by_name.get(
        basename.casefold(),
        [],
    )

    print()
    print(
        f"[SOURCE {segment['source_index']}] "
        f"{basename}"
    )

    if not candidates:
        raise RuntimeError(
            "Required Stage27 source basename "
            "not found anywhere under /kaggle/input:\n"
            f"{basename}"
        )

    print(
        "  basename candidates:",
        len(candidates),
    )

    exact = []

    for candidate in candidates:

        actual_sha = sha256_file(
            candidate
        )

        print(
            "   ",
            candidate.relative_to(INPUT_ROOT),
            actual_sha,
        )

        if actual_sha == expected_sha:
            exact.append(candidate)

    if len(exact) != 1:
        raise RuntimeError(
            "Expected exactly one byte-exact "
            f"source for {basename}; found "
            f"{len(exact)}."
        )

    chosen = exact[0]

    resolved_sources.append(
        {
            "segment": segment,
            "runtime_path": chosen,
        }
    )

    print(
        "  [EXACT]",
        chosen,
    )

print()
print(
    "[PASS] all 8 byte-exact CICIDS2017 "
    "source assets located"
)


# =================================================================================================
# 6. LABEL CANONICALIZATION
# =================================================================================================

banner(
    "REBUILD STAGE27 GLOBAL FAMILY / DAY / SOURCE ARRAYS"
)

codebook = family_cache_receipt[
    "family_codebook"
]

expected_codebook = {
    "BENIGN": 0,
    "BOT": 1,
    "DDOS": 2,
    "DOS": 3,
    "AUTH_BRUTE_FORCE": 4,
    "INFILTRATION": 5,
    "PORT_SCAN": 6,
    "WEB_ATTACK": 7,
    "TARGET_ONLY_UNSEEN": 8,
    "OTHER_ATTACK_UNSEEN_LABEL": 9,
}

if codebook != expected_codebook:
    raise RuntimeError(
        "Unexpected Stage27 family codebook."
    )


# Authorized semantic mapping to the frozen family taxonomy.
LABEL_TO_CODE = {

    "benign": 0,

    "bot": 1,

    "ddos": 2,

    "dos goldeneye": 3,
    "dos hulk": 3,
    "dos slowhttptest": 3,
    "dos slowloris": 3,

    "ftp-patator": 4,
    "ssh-patator": 4,

    "infiltration": 5,

    "portscan": 6,

    "web attack - brute force": 7,
    "web attack - sql injection": 7,
    "web attack - xss": 7,

    "heartbleed": 8,
}


# Unicode dash characters normalized by Stage27 canonicalization.
NORMAL_DASHES = [
    "\u2010",
    "\u2011",
    "\u2012",
    "\u2013",
    "\u2014",
    "\u2015",
    "\u2212",
    "\ufe58",
    "\ufe63",
    "\uff0d",
]


# Frozen CP1252/U+0096 Web Attack compatibility aliases.
WEB_ALIASES = {
    "web attack \u0096 brute force":
        "web attack - brute force",

    "web attack \u0096 sql injection":
        "web attack - sql injection",

    "web attack \u0096 xss":
        "web attack - xss",
}


def canonicalize_label_series(
    s: pd.Series,
):
    s = s.astype("string")

    s = s.str.strip()

    for dash in NORMAL_DASHES:
        s = s.str.replace(
            dash,
            "-",
            regex=False,
        )

    s = s.str.replace(
        r"\s+",
        " ",
        regex=True,
    )

    s = s.str.casefold()

    s = s.replace(
        WEB_ALIASES
    )

    return s


total_effective_rows = int(
    source_receipt["population"]["effective_rows"]
)

global_family_codes = np.empty(
    total_effective_rows,
    dtype=np.uint8,
)

global_day_codes = np.empty(
    total_effective_rows,
    dtype=np.uint8,
)

global_source_codes = np.empty(
    total_effective_rows,
    dtype=np.uint8,
)

source_runtime_records = []

membership_label_rows_read = 0


for item in resolved_sources:

    segment = item["segment"]
    path = item["runtime_path"]

    source_index = int(
        segment["source_index"]
    )

    day = segment["day"]
    day_code = int(
        segment["day_code"]
    )

    physical_rows = int(
        segment["physical_rows"]
    )

    effective_rows = int(
        segment["effective_rows"]
    )

    global_start = int(
        segment["global_start_zero_based"]
    )

    global_stop = int(
        segment["global_stop_exclusive"]
    )

    print()
    print(
        f"[{source_index}] {day} — "
        f"{path.name}"
    )

    pf = pq.ParquetFile(path)

    actual_physical_rows = (
        pf.metadata.num_rows
    )

    if (
        actual_physical_rows
        != physical_rows
    ):
        raise RuntimeError(
            f"Physical row mismatch for "
            f"{path.name}.\n"
            f"expected={physical_rows}\n"
            f"actual={actual_physical_rows}"
        )

    schema_names = (
        pf.schema_arrow.names
    )

    label_candidates = [
        col
        for col in schema_names
        if col.strip().casefold()
        == "label"
    ]

    if len(label_candidates) != 1:
        raise RuntimeError(
            f"Could not uniquely resolve Label "
            f"column in {path.name}.\n"
            f"Candidates={label_candidates}\n"
            f"Schema={schema_names}"
        )

    label_col = label_candidates[0]

    table = pq.read_table(
        path,
        columns=[label_col],
    )

    labels = (
        table.column(0)
        .to_pandas()
    )

    membership_label_rows_read += len(
        labels
    )

    if len(labels) != physical_rows:
        raise RuntimeError(
            "Read row count mismatch."
        )

    # Stage27 freezes effective rows as a prefix
    # for the one source with the structural-null suffix.
    if effective_rows < physical_rows:

        rule = segment[
            "inclusion_rule"
        ]

        if not rule.startswith(
            "INCLUDE_PHYSICAL_ORDINALS_1_THROUGH_"
        ):
            raise RuntimeError(
                "Unexpected partial-source "
                "inclusion rule."
            )

        labels = labels.iloc[
            :effective_rows
        ].copy()

    elif effective_rows != physical_rows:
        raise RuntimeError(
            "Unsupported effective-row geometry."
        )

    canonical = (
        canonicalize_label_series(
            labels
        )
    )

    mapped = canonical.map(
        LABEL_TO_CODE
    )

    if mapped.isna().any():

        unknown = (
            canonical[
                mapped.isna()
            ]
            .value_counts(
                dropna=False
            )
        )

        print(
            "\nUnknown canonical labels:"
        )

        print(unknown)

        raise RuntimeError(
            "Encountered label outside the "
            "frozen Stage27 taxonomy."
        )

    codes = mapped.to_numpy(
        dtype=np.uint8,
        copy=True,
    )

    if len(codes) != effective_rows:
        raise RuntimeError(
            "Effective code length mismatch."
        )

    if (
        global_stop - global_start
        != effective_rows
    ):
        raise RuntimeError(
            "Frozen global geometry mismatch."
        )

    global_family_codes[
        global_start:global_stop
    ] = codes

    global_day_codes[
        global_start:global_stop
    ] = day_code

    global_source_codes[
        global_start:global_stop
    ] = source_index

    local_counts = np.bincount(
        codes,
        minlength=10,
    )

    print(
        "  physical rows :",
        physical_rows,
    )

    print(
        "  effective rows:",
        effective_rows,
    )

    print(
        "  Label column  :",
        repr(label_col),
    )

    print(
        "  family counts :",
        {
            name: int(
                local_counts[code]
            )
            for name, code
            in codebook.items()
            if local_counts[code] > 0
        },
    )

    source_runtime_records.append(
        {
            "source_index": source_index,
            "day": day,
            "basename": path.name,
            "runtime_path": str(path),
            "sha256": sha256_file(path),
            "physical_rows": physical_rows,
            "effective_rows": effective_rows,
            "label_column": label_col,
        }
    )

    del table
    del labels
    del canonical
    del mapped
    del codes


# =================================================================================================
# 7. GLOBAL CENSUS AUDIT
# =================================================================================================

banner(
    "GLOBAL FAMILY / DAY CENSUS AUDIT"
)

observed_counts_arr = np.bincount(
    global_family_codes,
    minlength=10,
)

observed_family_counts = {
    family: int(
        observed_counts_arr[code]
    )
    for family, code
    in codebook.items()
}

expected_family_counts = (
    census["family_counts"]
)

print("Expected:")
print(
    json.dumps(
        expected_family_counts,
        indent=2,
    )
)

print()
print("Observed:")
print(
    json.dumps(
        observed_family_counts,
        indent=2,
    )
)

if (
    observed_family_counts
    != expected_family_counts
):
    raise RuntimeError(
        "Global Stage27 family census "
        "does not reproduce exactly."
    )

if int(
    len(global_family_codes)
) != 2_830_743:
    raise RuntimeError(
        "Unexpected global population length."
    )


# Family × day counts.
day_name_to_code = {}

for seg in source_receipt["segments"]:
    day_name_to_code[
        seg["day"]
    ] = int(
        seg["day_code"]
    )


for family, expected_days in (
    census["family_day_counts"].items()
):

    family_code = codebook[
        family
    ]

    for day_name, expected_count in (
        expected_days.items()
    ):

        day_code = (
            day_name_to_code[
                day_name
            ]
        )

        observed = int(
            np.count_nonzero(
                (global_family_codes == family_code)
                & (
                    global_day_codes
                    == day_code
                )
            )
        )

        if observed != expected_count:
            raise RuntimeError(
                f"Family/day mismatch: "
                f"{family} {day_name}\n"
                f"expected={expected_count}\n"
                f"actual={observed}"
            )


for day_name, expected_count in (
    census["benign_day_counts"].items()
):

    day_code = (
        day_name_to_code[
            day_name
        ]
    )

    observed = int(
        np.count_nonzero(
            (global_family_codes == 0)
            & (
                global_day_codes
                == day_code
            )
        )
    )

    if observed != expected_count:
        raise RuntimeError(
            f"Benign/day mismatch: "
            f"{day_name}\n"
            f"expected={expected_count}\n"
            f"actual={observed}"
        )


print()
print("[PASS] exact global family census")
print("[PASS] exact family × day census")
print("[PASS] exact benign × day census")


# =================================================================================================
# 8. REQUIRE EXACT HISTORICAL CONTENT HASHES
# =================================================================================================

banner(
    "GLOBAL ARRAY CONTENT-SHA256 GATE"
)

expected_arrays = (
    family_cache_receipt[
        "arrays"
    ]
)

actual_global_hashes = {
    "global_family_codes":
        sha256_array_content(
            global_family_codes
        ),

    "global_day_codes":
        sha256_array_content(
            global_day_codes
        ),

    "global_source_codes":
        sha256_array_content(
            global_source_codes
        ),
}

for key, actual_sha in (
    actual_global_hashes.items()
):

    expected_sha = (
        expected_arrays[key][
            "content_sha256"
        ]
    )

    print(key)
    print("  expected:", expected_sha)
    print("  actual  :", actual_sha)

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"{key} content SHA mismatch."
        )

print()
print(
    "[PASS] all 3 Stage27 runtime global "
    "arrays reproduced byte-for-byte"
)


# =================================================================================================
# 9. SAVE RECONSTRUCTED GLOBAL RUNTIME CACHE
# =================================================================================================

GLOBAL_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

np.save(
    GLOBAL_CACHE_ROOT
    / "global_family_codes.npy",
    global_family_codes,
)

np.save(
    GLOBAL_CACHE_ROOT
    / "global_day_codes.npy",
    global_day_codes,
)

np.save(
    GLOBAL_CACHE_ROOT
    / "global_source_codes.npy",
    global_source_codes,
)

print()
print(
    "Runtime global cache:",
    GLOBAL_CACHE_ROOT,
)


# =================================================================================================
# 10. RECONSTRUCT FIVE EXACT STAGE27 CHRONOLOGY FOLDS
# =================================================================================================

banner(
    "RECONSTRUCT FIVE EXACT STAGE27 CHRONOLOGY FOLDS"
)

primary_codes = np.array(
    [
        codebook[x]
        for x in PRIMARY_SEVEN
    ],
    dtype=np.uint8,
)

fold_receipts_out = {}

for family in ELIGIBLE_FAMILIES:

    print()
    print("-" * 118)
    print("FOLD:", family)
    print("-" * 118)

    historical_receipt_path = (
        REPO
        / "results"
        / "stage27_loao_unseen_attack"
        / "stage27_1a_fold_membership"
        / f"fold_{family}_membership_receipt.json"
    )

    historical = read_json(
        require_file(
            historical_receipt_path
        )
    )

    if (
        historical["held_out_family"]
        != family
    ):
        raise RuntimeError(
            "Historical fold identity mismatch."
        )

    heldout_code = codebook[
        family
    ]

    train_day_codes = np.array(
        [
            day_name_to_code[x]
            for x
            in historical["train_days"]
        ],
        dtype=np.uint8,
    )

    validation_day_codes = np.array(
        [
            day_name_to_code[x]
            for x
            in historical[
                "validation_days"
            ]
        ],
        dtype=np.uint8,
    )

    target_day_codes = np.array(
        [
            day_name_to_code[x]
            for x
            in historical["target_days"]
        ],
        dtype=np.uint8,
    )


    # ---------------------------------------------------------------------------------------------
    # Attack eligibility:
    # primary-seven families excluding held-out.
    # Codes 8 and 9 are excluded everywhere except neither is part
    # of Stage27 primary/operational target semantics.
    # ---------------------------------------------------------------------------------------------

    primary_attack = np.isin(
        global_family_codes,
        primary_codes,
    )

    known_attack = (
        primary_attack
        & (
            global_family_codes
            != heldout_code
        )
    )

    benign = (
        global_family_codes == 0
    )

    heldout = (
        global_family_codes
        == heldout_code
    )

    train_day_mask = np.isin(
        global_day_codes,
        train_day_codes,
    )

    validation_day_mask = np.isin(
        global_day_codes,
        validation_day_codes,
    )

    target_day_mask = np.isin(
        global_day_codes,
        target_day_codes,
    )


    # ---------------------------------------------------------------------------------------------
    # Exact Stage27 membership semantics.
    # ---------------------------------------------------------------------------------------------

    train_mask = (
        train_day_mask
        & (
            benign
            | known_attack
        )
    )

    validation_mask = (
        validation_day_mask
        & (
            benign
            | known_attack
        )
    )

    primary_target_mask = (
        target_day_mask
        & (
            benign
            | heldout
        )
    )

    operational_target_mask = (
        target_day_mask
        & (
            benign
            | primary_attack
        )
    )


    # ---------------------------------------------------------------------------------------------
    # Indices in frozen global-row order.
    # ---------------------------------------------------------------------------------------------

    train_idx = np.flatnonzero(
        train_mask
    ).astype(
        np.int32,
        copy=False,
    )

    validation_idx = np.flatnonzero(
        validation_mask
    ).astype(
        np.int32,
        copy=False,
    )

    primary_target_idx = (
        np.flatnonzero(
            primary_target_mask
        ).astype(
            np.int32,
            copy=False,
        )
    )

    operational_target_idx = (
        np.flatnonzero(
            operational_target_mask
        ).astype(
            np.int32,
            copy=False,
        )
    )


    # ---------------------------------------------------------------------------------------------
    # Held-out exclusion.
    # ---------------------------------------------------------------------------------------------

    heldout_train_count = int(
        np.count_nonzero(
            global_family_codes[
                train_idx
            ]
            == heldout_code
        )
    )

    heldout_validation_count = int(
        np.count_nonzero(
            global_family_codes[
                validation_idx
            ]
            == heldout_code
        )
    )

    if heldout_train_count != 0:
        raise RuntimeError(
            f"{family}: held-out family "
            "appeared in TRAIN."
        )

    if heldout_validation_count != 0:
        raise RuntimeError(
            f"{family}: held-out family "
            "appeared in VALIDATION."
        )


    # ---------------------------------------------------------------------------------------------
    # Counts.
    # ---------------------------------------------------------------------------------------------

    observed_counts = {

        "train_rows":
            len(train_idx),

        "train_benign":
            int(
                np.count_nonzero(
                    global_family_codes[
                        train_idx
                    ] == 0
                )
            ),

        "train_attack":
            int(
                np.count_nonzero(
                    global_family_codes[
                        train_idx
                    ] != 0
                )
            ),

        "validation_rows":
            len(validation_idx),

        "validation_benign":
            int(
                np.count_nonzero(
                    global_family_codes[
                        validation_idx
                    ] == 0
                )
            ),

        "validation_attack":
            int(
                np.count_nonzero(
                    global_family_codes[
                        validation_idx
                    ] != 0
                )
            ),

        "primary_target_rows":
            len(primary_target_idx),

        "primary_target_benign":
            int(
                np.count_nonzero(
                    global_family_codes[
                        primary_target_idx
                    ] == 0
                )
            ),

        "primary_target_heldout_attack":
            int(
                np.count_nonzero(
                    global_family_codes[
                        primary_target_idx
                    ]
                    == heldout_code
                )
            ),

        "operational_target_rows":
            len(
                operational_target_idx
            ),

        "operational_target_benign":
            int(
                np.count_nonzero(
                    global_family_codes[
                        operational_target_idx
                    ] == 0
                )
            ),

        "operational_target_attack":
            int(
                np.count_nonzero(
                    global_family_codes[
                        operational_target_idx
                    ] != 0
                )
            ),
    }

    expected_counts = historical[
        "counts"
    ]

    print(
        "Expected counts:",
        expected_counts,
    )

    print(
        "Observed counts:",
        observed_counts,
    )

    if observed_counts != expected_counts:
        raise RuntimeError(
            f"{family}: fold counts do "
            "not reproduce exactly."
        )


    # ---------------------------------------------------------------------------------------------
    # Exact historical content hashes.
    # ---------------------------------------------------------------------------------------------

    arrays = {
        "train_global_idx":
            train_idx,

        "validation_global_idx":
            validation_idx,

        "primary_target_global_idx":
            primary_target_idx,

        "operational_target_global_idx":
            operational_target_idx,
    }

    array_receipts = {}

    for key, arr in arrays.items():

        actual_sha = (
            sha256_array_content(arr)
        )

        expected_sha = (
            historical[
                "runtime_membership_arrays"
            ][key]["content_sha256"]
        )

        print()
        print(key)
        print(
            "  rows    :",
            len(arr),
        )
        print(
            "  expected:",
            expected_sha,
        )
        print(
            "  actual  :",
            actual_sha,
        )

        if actual_sha != expected_sha:
            raise RuntimeError(
                f"{family}/{key}: "
                "content SHA mismatch."
            )

        array_receipts[key] = {
            "count": int(
                len(arr)
            ),
            "dtype": str(
                arr.dtype
            ),
            "content_sha256":
                actual_sha,
        }


    # ---------------------------------------------------------------------------------------------
    # Class weight reproduction.
    # ---------------------------------------------------------------------------------------------

    train_benign = observed_counts[
        "train_benign"
    ]

    train_attack = observed_counts[
        "train_attack"
    ]

    realized_weight = (
        train_benign
        / train_attack
    )

    expected_weight = historical[
        "class_weight"
    ]["realized_value"]

    if not math.isclose(
        realized_weight,
        expected_weight,
        rel_tol=0.0,
        abs_tol=1e-15,
    ):
        raise RuntimeError(
            f"{family}: class weight mismatch.\n"
            f"expected={expected_weight}\n"
            f"actual={realized_weight}"
        )


    # ---------------------------------------------------------------------------------------------
    # Save runtime arrays OUTSIDE repo.
    # ---------------------------------------------------------------------------------------------

    family_cache = (
        CHRONO_CACHE_ROOT
        / family
    )

    family_cache.mkdir(
        parents=True,
        exist_ok=True,
    )

    for key, arr in arrays.items():
        np.save(
            family_cache
            / f"{key}.npy",
            arr,
        )


    # ---------------------------------------------------------------------------------------------
    # Durable Stage28 reconstruction receipt.
    # ---------------------------------------------------------------------------------------------

    reconstruction = {
        "stage": "Stage28-1A",

        "type":
            "INHERITED_STAGE27_FOLD_RECONSTRUCTION_RECEIPT",

        "held_out_family":
            family,

        "historical_stage27_receipt": {
            "path": str(
                historical_receipt_path.relative_to(
                    REPO
                )
            ),
            "sha256": sha256_file(
                historical_receipt_path
            ),
        },

        "membership_semantics":
            historical[
                "membership_semantics"
            ],

        "train_days":
            historical["train_days"],

        "validation_days":
            historical[
                "validation_days"
            ],

        "target_days":
            historical["target_days"],

        "counts":
            observed_counts,

        "heldout_exclusion": {
            "train_count":
                heldout_train_count,

            "validation_count":
                heldout_validation_count,

            "status": "PASS",
        },

        "arrays":
            array_receipts,

        "class_weight": {
            "formula":
                "train_benign / train_attack",

            "realized_value":
                realized_weight,

            "historical_value":
                expected_weight,

            "exact_reproduction":
                True,
        },

        "runtime_cache": {
            "path":
                str(family_cache),

            "committed_to_git":
                False,
        },

        "scientific_operations": {
            "model_fits": 0,
            "model_inference": 0,
            "threshold_selection": 0,
            "predictor_values_read": 0,
            "target_predictor_openings": 0,
        },

        "status":
            "PASS_EXACT_CONTENT_REPRODUCTION",
    }

    out_path = (
        OUTPUT_DIR
        / (
            f"fold_{family}_"
            "reconstruction_receipt.json"
        )
    )

    write_json(
        out_path,
        reconstruction,
    )

    fold_receipts_out[
        family
    ] = reconstruction

    print()
    print(
        f"[PASS] {family}: "
        "exact Stage27 membership reproduced"
    )


# =================================================================================================
# 11. SOURCE / GLOBAL CACHE RECONSTRUCTION RECEIPT
# =================================================================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

global_reconstruction_receipt = {
    "stage": "Stage28-1A",

    "type":
        "STAGE27_GLOBAL_CACHE_RECONSTRUCTION_RECEIPT",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage28_parent_commit":
        EXPECTED_PARENT,

    "source_population": {
        "identity":
            source_receipt[
                "population"
            ][
                "frozen_effective_population_identity"
            ],

        "physical_rows":
            source_receipt[
                "population"
            ]["physical_rows"],

        "effective_rows":
            source_receipt[
                "population"
            ]["effective_rows"],

        "structural_null_excluded_rows":
            source_receipt[
                "population"
            ][
                "structural_null_excluded_rows"
            ],
    },

    "resolved_sources":
        source_runtime_records,

    "membership_label_access": {
        "physical_label_rows_read":
            membership_label_rows_read,

        "predictor_columns_read": 0,

        "predictor_values_read": 0,

        "purpose":
            "MEMBERSHIP_RECONSTRUCTION_ONLY",
    },

    "global_arrays": {
        key: {
            "content_sha256": sha,
            "historical_expected_sha256":
                expected_arrays[key][
                    "content_sha256"
                ],
            "exact_match": True,
        }
        for key, sha
        in actual_global_hashes.items()
    },

    "family_counts":
        observed_family_counts,

    "runtime_cache": {
        "path":
            str(
                GLOBAL_CACHE_ROOT
            ),

        "committed_to_git":
            False,
    },

    "status":
        "PASS_EXACT_CONTENT_REPRODUCTION",
}

write_json(
    OUTPUT_DIR
    / "stage27_global_cache_reconstruction_receipt.json",
    global_reconstruction_receipt,
)


# =================================================================================================
# 12. STAGE22 AUDIT RECEIPT
# =================================================================================================

stage22_audit_receipt = {
    "stage": "Stage28-1A",

    "type":
        "STAGE22R_INHERITED_MEMBERSHIP_AUDIT",

    "membership_summary": {
        "path": str(
            stage22_summary_path.relative_to(
                REPO
            )
        ),

        "sha256":
            sha256_file(
                stage22_summary_path
            ),
    },

    "random_validation": {
        "path": str(
            random_bitset_path.relative_to(
                REPO
            )
        ),

        "sha256":
            actual_random_sha,

        "expected_sha256":
            expected_random_sha,

        "logical_length":
            logical_length,

        "validation_population":
            actual_population,

        "status": "PASS_EXACT",
    },

    "random_natural": {
        "train_rows":
            stage22_cells[
                "RANDOM_NATURAL"
            ]["train"]["rows"],

        "validation_rows":
            stage22_cells[
                "RANDOM_NATURAL"
            ]["validation"]["rows"],
    },

    "chronological_natural": {
        "train_rows":
            stage22_cells[
                "CHRONOLOGICAL_NATURAL"
            ]["train"]["rows"],

        "validation_rows":
            stage22_cells[
                "CHRONOLOGICAL_NATURAL"
            ]["validation"]["rows"],
    },

    "membership_changed":
        False,

    "scientific_operations": {
        "model_fits": 0,
        "model_inference": 0,
        "threshold_selection": 0,
        "holdout_openings": 0,
    },

    "status":
        "PASS_EXACT_INHERITED_MEMBERSHIP",
}

write_json(
    OUTPUT_DIR
    / "stage22_membership_audit.json",
    stage22_audit_receipt,
)


# =================================================================================================
# 13. STAGE28-1A FREEZE RECORD
# =================================================================================================

freeze_record = {
    "stage": "Stage28-1A",

    "status":
        "INHERITED_MEMBERSHIPS_RECONSTRUCTED_AND_VERIFIED",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_PARENT,

    "stage22": {
        "random_validation_bitset":
            "PASS_EXACT",

        "random_natural_membership_changed":
            False,

        "chronological_natural_membership_changed":
            False,
    },

    "stage27": {
        "source_assets":
            "8_OF_8_BYTE_EXACT",

        "global_family_codes":
            "PASS_EXACT_CONTENT_SHA256",

        "global_day_codes":
            "PASS_EXACT_CONTENT_SHA256",

        "global_source_codes":
            "PASS_EXACT_CONTENT_SHA256",

        "eligible_folds": {
            family:
                "PASS_EXACT_CONTENT_SHA256"
            for family
            in ELIGIBLE_FAMILIES
        },

        "held_out_train_count":
            0,

        "held_out_validation_count":
            0,
    },

    "fit_budget_after_reconstruction": {
        "new_fits_authorized": 108,
        "new_fits_consumed": 0,
        "new_fits_remaining": 108,
        "existing_reused": 12,
        "component_universe": 120,
    },

    "scientific_operations": {
        "model_fits": 0,
        "model_inference": 0,
        "threshold_selection": 0,
        "target_predictor_openings": 0,
        "predictor_values_read": 0,
    },

    "runtime_cache_policy": {
        "committed_to_git": False,

        "global_cache":
            str(GLOBAL_CACHE_ROOT),

        "chronological_cache":
            str(
                CHRONO_CACHE_ROOT
            ),

        "recovery_rule":
            (
                "Reconstruct again from the same "
                "8 byte-exact Stage27 source assets "
                "and require historical content SHA256."
            ),
    },

    "next_authorized_step": (
        "Stage28-1B — materialize the five frozen "
        "Stage28B random-LOAO memberships, audit "
        "zero held-out-family leakage and excluded "
        "target-only-unseen rows, then freeze the "
        "120-component execution manifest before "
        "the first model fit."
    ),
}

write_json(
    OUTPUT_DIR
    / "stage28_1a_freeze_record.json",
    freeze_record,
)


# =================================================================================================
# 14. CHECKSUMS
# =================================================================================================

json_files = sorted(
    OUTPUT_DIR.glob("*.json")
)

checksum_lines = [
    f"{sha256_file(p)}  {p.name}"
    for p in json_files
]

checksum_path = (
    OUTPUT_DIR
    / "checksums.sha256"
)

checksum_path.write_text(
    "\n".join(
        checksum_lines
    )
    + "\n",
    encoding="utf-8",
)


# =================================================================================================
# 15. FINAL MACHINE AUDIT
# =================================================================================================

banner(
    "STAGE28-1A FINAL MACHINE AUDIT"
)

assert (
    actual_global_hashes[
        "global_family_codes"
    ]
    == expected_arrays[
        "global_family_codes"
    ]["content_sha256"]
)

assert (
    actual_global_hashes[
        "global_day_codes"
    ]
    == expected_arrays[
        "global_day_codes"
    ]["content_sha256"]
)

assert (
    actual_global_hashes[
        "global_source_codes"
    ]
    == expected_arrays[
        "global_source_codes"
    ]["content_sha256"]
)

assert (
    observed_family_counts[
        "TARGET_ONLY_UNSEEN"
    ]
    == 11
)

assert (
    observed_family_counts[
        "OTHER_ATTACK_UNSEEN_LABEL"
    ]
    == 0
)

assert (
    len(fold_receipts_out)
    == 5
)

for family in ELIGIBLE_FAMILIES:

    r = fold_receipts_out[
        family
    ]

    assert (
        r["heldout_exclusion"][
            "train_count"
        ]
        == 0
    )

    assert (
        r["heldout_exclusion"][
            "validation_count"
        ]
        == 0
    )

    assert (
        r["status"]
        == "PASS_EXACT_CONTENT_REPRODUCTION"
    )

print(
    "[PASS] 8/8 exact CICIDS2017 sources"
)

print(
    "[PASS] global family-code content SHA"
)

print(
    "[PASS] global day-code content SHA"
)

print(
    "[PASS] global source-code content SHA"
)

print(
    "[PASS] 5/5 chronology LOAO folds "
    "reproduced exactly"
)

print(
    "[PASS] held-out family TRAIN count = 0"
)

print(
    "[PASS] held-out family VALIDATION count = 0"
)

print(
    "[PASS] Stage22 random membership exact"
)

print(
    "[PASS] predictor values read = 0"
)

print(
    "[PASS] model fits consumed = 0"
)

print(
    "[PASS] inference consumed = 0"
)

print(
    "[PASS] threshold selection consumed = 0"
)

print(
    "[PASS] target predictor openings = 0"
)


# =================================================================================================
# 16. DISPLAY DURABLE ARTIFACTS
# =================================================================================================

banner(
    "STAGE28-1A DURABLE ARTIFACTS"
)

for p in sorted(
    OUTPUT_DIR.iterdir()
):

    if p.is_file():

        print(
            f"{p.name:58s} "
            f"{p.stat().st_size:10,d} bytes  "
            f"{sha256_file(p)}"
        )


# =================================================================================================
# 17. GIT AUDIT
# =================================================================================================

banner(
    "GIT AUDIT"
)

status = git(
    "status",
    "--porcelain",
)

print(status)

expected_prefix = (
    "results/"
    "stage28_stability_novelty_control/"
    "stage28_1a_inherited_membership_reconstruction/"
)

unexpected = []

for line in status.splitlines():

    if not line.strip():
        continue

    path = line[3:].strip()

    if not path.startswith(
        expected_prefix
    ):
        unexpected.append(line)

if unexpected:
    raise RuntimeError(
        "Unexpected repository modifications:\n"
        + "\n".join(unexpected)
    )

print()
print(
    "[PASS] only Stage28-1A durable "
    "receipt artifacts modified"
)


# =================================================================================================
# 18. STAGE EXACT FILE SET
# =================================================================================================

files_to_stage = sorted(
    p.relative_to(REPO).as_posix()
    for p in OUTPUT_DIR.iterdir()
    if p.is_file()
)

for relative in files_to_stage:

    git(
        "add",
        "--",
        relative,
    )

staged = git(
    "diff",
    "--cached",
    "--name-only",
).splitlines()

if sorted(staged) != sorted(
    files_to_stage
):
    raise RuntimeError(
        "Staged file set mismatch."
    )

git(
    "diff",
    "--cached",
    "--check",
)

print()
print(
    "[PASS] exact Stage28-1A file "
    "set staged"
)

for path in staged:
    print(" ", path)


# =================================================================================================
# 19. COMMIT
# =================================================================================================

banner(
    "COMMIT STAGE28-1A"
)

git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
)

commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    f"{commit_sha}^",
)

if (
    commit_parent
    != EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-1A commit parent mismatch.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={commit_parent}"
    )

print(
    "Stage28-1A commit:",
    commit_sha,
)

print(
    "Parent            :",
    commit_parent,
)


# =================================================================================================
# 20. PUSH
# =================================================================================================

banner(
    "PUSH STAGE28-1A"
)

push = run(
    [
        "git",
        "push",
        "origin",
        "main:main",
    ],
    check=False,
)

if push.returncode != 0:

    raise RuntimeError(
        "Stage28-1A push failed.\n\n"
        f"STDOUT:\n{push.stdout}\n\n"
        f"STDERR:\n{push.stderr}"
    )

print(
    push.stdout or ""
)

print(
    push.stderr or ""
)


# =================================================================================================
# 21. REMOTE VERIFY
# =================================================================================================

banner(
    "REMOTE VERIFICATION"
)

remote_after = (
    remote_main_sha()
)

print(
    "Local HEAD :",
    commit_sha,
)

print(
    "Remote main:",
    remote_after,
)

if remote_after != commit_sha:
    raise RuntimeError(
        "Stage28-1A remote "
        "verification failed."
    )

git(
    "fetch",
    "origin",
    "main",
)

origin_main = git(
    "rev-parse",
    "origin/main",
)

if origin_main != commit_sha:
    raise RuntimeError(
        "origin/main mismatch "
        "after Stage28-1A push."
    )

final_status = git(
    "status",
    "--porcelain",
)

if final_status:
    raise RuntimeError(
        "Repository dirty after "
        "Stage28-1A commit:\n"
        + final_status
    )


# =================================================================================================
# 22. FINAL
# =================================================================================================

banner(
    "STAGE28-1A — COMPLETE / REMOTELY VERIFIED"
)

print(
    "Stage28-1A commit:"
)

print(
    " ",
    commit_sha,
)

print()

print(
    "Inherited membership status:"
)

print(
    "  Stage22 RANDOM_NATURAL       : EXACT"
)

print(
    "  Stage22 CHRONOLOGICAL_NATURAL: EXACT"
)

for family in ELIGIBLE_FAMILIES:
    print(
        f"  Stage27 {family:14s}: "
        "EXACT CONTENT REPRODUCTION"
    )

print()

print(
    "CICIDS2017 global cache:"
)

print(
    "  rows                 = 2,830,743"
)

print(
    "  family code SHA      = PASS"
)

print(
    "  day code SHA         = PASS"
)

print(
    "  source code SHA      = PASS"
)

print()

print(
    "Scientific operations:"
)

print(
    "  MODEL_FITS                = 0"
)

print(
    "  MODEL_INFERENCE           = 0"
)

print(
    "  THRESHOLD_SELECTION       = 0"
)

print(
    "  PREDICTOR_VALUES_READ     = 0"
)

print(
    "  TARGET_PREDICTOR_OPENINGS = 0"
)

print()

print(
    "Fit budget:"
)

print(
    "  NEW authorized = 108"
)

print(
    "  consumed       = 0"
)

print(
    "  remaining      = 108"
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage28-1B — materialize the five "
    "frozen RANDOM-LOAO memberships and "
    "freeze the 120-component execution manifest."
)

print()
print(SEP)


STAGE28-1A — EXACT REPOSITORY GATE

Repository : /kaggle/working/ids2018-validation-safe-ablation
Branch     : main
Local HEAD : 0029c28c417d63ffb45ad4096ab7c44cbd875116
Remote main: 0029c28c417d63ffb45ad4096ab7c44cbd875116
Expected   : 0029c28c417d63ffb45ad4096ab7c44cbd875116

[PASS] exact Stage28-0A parent
[PASS] local == remote
[PASS] worktree clean

LOAD FROZEN PROVENANCE

[PASS] effective Stage28 new-fit budget = 108
[PASS] effective Stage28 reused fits = 12
[PASS] component universe = 120

Frozen CICIDS2017 effective rows: 2830743

STAGE22R FROZEN MEMBERSHIP AUDIT

Random membership:
  path       : /kaggle/working/ids2018-validation-safe-ablation/results/stage22r_protocol_recovery/stage22r_1b1_development_memberships/random_validation.packbits
  expected sha: 8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad
  actual sha  : 8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad

[PASS] Stage22 random membership SHA exact
[PASS] random validation populati

RuntimeError: Required Stage27 source basename not found anywhere under /kaggle/input:
Monday-WorkingHours.pcap_ISCX.csv.parquet

In [7]:
# =================================================================================================
# STAGE28-1A-D0 — CICIDS2017 ATTACHED-ASSET RESOLVER
#
# Purpose:
#   Diagnose the failed Stage28-1A source lookup without touching science.
#
# This cell:
#   - verifies exact Stage28-0A parent
#   - verifies clean Git worktree
#   - reads the frozen Stage27 8-source receipt
#   - scans EVERY file under /kaggle/input
#   - resolves sources by exact SIZE + SHA256, ignoring Kaggle filenames
#   - inspects ZIP/TAR containers if necessary
#   - writes ONLY a temporary runtime resolver map under /kaggle/working
#
# ZERO:
#   predictor reads
#   label reads
#   model fits
#   inference
#   threshold selection
#   target openings
#   git modifications
# =================================================================================================

from __future__ import annotations

import hashlib
import json
import subprocess
import tarfile
import zipfile
from pathlib import Path


SEP = "=" * 118

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

INPUT_ROOT = Path("/kaggle/input").resolve()

EXPECTED_HEAD = (
    "0029c28c417d63ffb45ad4096ab7c44cbd875116"
)

SOURCE_RECEIPT_PATH = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
    / "source_effective_population_receipt.json"
)

TEMP_MAP = Path(
    "/kaggle/working/stage28_1a_exact_source_map.json"
)


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, cwd=REPO):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    if p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p.stdout.strip()


def sha256_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# =================================================================================================
# 1. REPOSITORY SAFETY GATE
# =================================================================================================

banner("STAGE28-1A-D0 — REPOSITORY SAFETY GATE")

head = run([
    "git",
    "rev-parse",
    "HEAD",
])

origin = run([
    "git",
    "rev-parse",
    "origin/main",
])

status = run([
    "git",
    "status",
    "--porcelain",
])

print("Local HEAD :", head)
print("origin/main:", origin)
print("Expected   :", EXPECTED_HEAD)
print("Git clean  :", not bool(status))

if head != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={head}"
    )

if origin != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected origin/main.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={origin}"
    )

if status:
    raise RuntimeError(
        "Repository became dirty after the failed "
        "Stage28-1A attempt:\n"
        + status
    )

print()
print("[PASS] failed Stage28-1A left no Git changes")


# =================================================================================================
# 2. LOAD FROZEN SOURCE IDENTITIES
# =================================================================================================

banner("LOAD FROZEN STAGE27 SOURCE IDENTITIES")

if not SOURCE_RECEIPT_PATH.is_file():
    raise RuntimeError(
        f"Missing source receipt:\n{SOURCE_RECEIPT_PATH}"
    )

source_receipt = json.loads(
    SOURCE_RECEIPT_PATH.read_text(
        encoding="utf-8"
    )
)

segments = source_receipt["segments"]

if len(segments) != 8:
    raise RuntimeError(
        f"Expected 8 frozen sources, got {len(segments)}"
    )

expected_by_sha = {}
expected_sizes = set()

for seg in segments:

    record = {
        "source_index": int(seg["source_index"]),
        "day": seg["day"],
        "basename": seg["basename"],
        "sha256": seg["sha256"],
        "size_bytes": int(seg["size_bytes"]),
        "physical_rows": int(seg["physical_rows"]),
        "effective_rows": int(seg["effective_rows"]),
    }

    expected_by_sha[
        record["sha256"]
    ] = record

    expected_sizes.add(
        record["size_bytes"]
    )

    print(
        f"[{record['source_index']}] "
        f"{record['day']:9s}  "
        f"{record['basename']}"
    )
    print(
        f"    bytes  = {record['size_bytes']:,}"
    )
    print(
        f"    sha256 = {record['sha256']}"
    )


# =================================================================================================
# 3. COMPLETE KAGGLE INPUT INVENTORY
# =================================================================================================

banner("COMPLETE /kaggle/input INVENTORY")

if not INPUT_ROOT.exists():
    raise RuntimeError(
        "/kaggle/input is missing."
    )

files = sorted(
    p
    for p in INPUT_ROOT.rglob("*")
    if p.is_file()
)

print("Total files:", len(files))
print()

for i, path in enumerate(files, 1):

    size = path.stat().st_size

    print(
        f"[{i:02d}] "
        f"{path.relative_to(INPUT_ROOT)}"
    )
    print(
        f"     bytes={size:,} "
        f"suffix={path.suffix!r}"
    )


# =================================================================================================
# 4. FAST SIZE FILTER
# =================================================================================================

banner("SIZE-MATCH CANDIDATES")

size_candidates = []

for path in files:

    size = path.stat().st_size

    if size in expected_sizes:

        size_candidates.append(path)

        expected = [
            x
            for x in expected_by_sha.values()
            if x["size_bytes"] == size
        ]

        print(
            "[SIZE MATCH]",
            path.relative_to(INPUT_ROOT),
        )

        for item in expected:
            print(
                "   possible frozen source:",
                item["basename"],
            )

if not size_candidates:
    print(
        "[INFO] No direct files have an exact frozen source size."
    )
else:
    print()
    print(
        "Direct size-match candidates:",
        len(size_candidates),
    )


# =================================================================================================
# 5. EXACT SHA RESOLUTION
# =================================================================================================

banner("DIRECT EXACT-SHA RESOLUTION")

resolved = {}
hashed_files = 0

# First hash size-matched files only.
for path in size_candidates:

    size = path.stat().st_size
    actual_sha = sha256_file(path)
    hashed_files += 1

    print(
        path.relative_to(INPUT_ROOT)
    )
    print(
        "  sha256:",
        actual_sha,
    )

    frozen = expected_by_sha.get(
        actual_sha
    )

    if frozen is not None:

        if size != frozen["size_bytes"]:
            raise RuntimeError(
                "SHA matched but size did not — impossible "
                "under expected provenance."
            )

        key = frozen["basename"]

        if key in resolved:
            raise RuntimeError(
                "Duplicate byte-exact input sources detected "
                f"for {key}."
            )

        resolved[key] = {
            **frozen,
            "runtime_path": str(path),
            "resolution": "DIRECT_FILE_SHA256",
        }

        print(
            "  [EXACT FROZEN MATCH]",
            key,
        )

print()
print(
    f"Direct exact matches: {len(resolved)} / 8"
)


# =================================================================================================
# 6. OPTIONAL FULL SHA SCAN IF SIZE METADATA DID NOT RESOLVE EVERYTHING
# =================================================================================================

if len(resolved) < 8:

    banner("SECONDARY FULL-FILE SHA SCAN")

    already_hashed = set(
        size_candidates
    )

    for path in files:

        if path in already_hashed:
            continue

        # Skip obviously tiny metadata files to avoid noise.
        size = path.stat().st_size

        if size < 1024:
            continue

        print(
            "Hashing:",
            path.relative_to(INPUT_ROOT),
            f"({size:,} bytes)",
        )

        actual_sha = sha256_file(path)
        hashed_files += 1

        frozen = expected_by_sha.get(
            actual_sha
        )

        if frozen is not None:

            key = frozen["basename"]

            if key in resolved:
                raise RuntimeError(
                    "Duplicate byte-exact frozen source "
                    f"detected for {key}."
                )

            resolved[key] = {
                **frozen,
                "runtime_path": str(path),
                "resolution": "DIRECT_FILE_SHA256",
            }

            print(
                "  >>> EXACT MATCH:",
                key,
            )

    print()
    print(
        f"Exact direct matches after full scan: "
        f"{len(resolved)} / 8"
    )


# =================================================================================================
# 7. ARCHIVE INVENTORY IF SOURCES ARE PACKAGED
# =================================================================================================

archive_inventory = []

if len(resolved) < 8:

    banner("ARCHIVE INSPECTION")

    for path in files:

        rel = str(
            path.relative_to(INPUT_ROOT)
        )

        # -----------------------------------------------------------------------------------------
        # ZIP
        # -----------------------------------------------------------------------------------------

        try:
            is_zip = zipfile.is_zipfile(path)
        except Exception:
            is_zip = False

        if is_zip:

            print()
            print("[ZIP]", rel)

            with zipfile.ZipFile(
                path,
                "r",
            ) as zf:

                members = zf.infolist()

                print(
                    "  members:",
                    len(members),
                )

                for member in members:

                    if member.is_dir():
                        continue

                    possible = [
                        x
                        for x in expected_by_sha.values()
                        if (
                            member.file_size
                            == x["size_bytes"]
                        )
                        or (
                            x["basename"].casefold()
                            in member.filename.casefold()
                        )
                    ]

                    if possible:

                        print(
                            "  [CANDIDATE]",
                            member.filename,
                            f"bytes={member.file_size:,}",
                        )

                        for x in possible:
                            print(
                                "       possible:",
                                x["basename"],
                            )

                        archive_inventory.append({
                            "archive": str(path),
                            "archive_type": "ZIP",
                            "member": member.filename,
                            "member_size": member.file_size,
                        })


        # -----------------------------------------------------------------------------------------
        # TAR family
        # -----------------------------------------------------------------------------------------

        try:
            is_tar = tarfile.is_tarfile(path)
        except Exception:
            is_tar = False

        if is_tar:

            print()
            print("[TAR]", rel)

            with tarfile.open(
                path,
                "r:*",
            ) as tf:

                members = [
                    x
                    for x in tf.getmembers()
                    if x.isfile()
                ]

                print(
                    "  members:",
                    len(members),
                )

                for member in members:

                    possible = [
                        x
                        for x in expected_by_sha.values()
                        if (
                            member.size
                            == x["size_bytes"]
                        )
                        or (
                            x["basename"].casefold()
                            in member.name.casefold()
                        )
                    ]

                    if possible:

                        print(
                            "  [CANDIDATE]",
                            member.name,
                            f"bytes={member.size:,}",
                        )

                        for x in possible:
                            print(
                                "       possible:",
                                x["basename"],
                            )

                        archive_inventory.append({
                            "archive": str(path),
                            "archive_type": "TAR",
                            "member": member.name,
                            "member_size": member.size,
                        })


# =================================================================================================
# 8. RESOLUTION REPORT
# =================================================================================================

banner("FROZEN SOURCE RESOLUTION REPORT")

for seg in segments:

    basename = seg["basename"]

    print(
        f"[{seg['source_index']}] "
        f"{basename}"
    )

    if basename in resolved:

        x = resolved[basename]

        print(
            "  STATUS : EXACT DIRECT MATCH"
        )

        print(
            "  PATH   :",
            x["runtime_path"],
        )

        print(
            "  SHA256 :",
            x["sha256"],
        )

    else:

        print(
            "  STATUS : NOT YET RESOLVED"
        )

print()
print(
    "Exact direct matches:",
    len(resolved),
    "/ 8",
)

print(
    "Archive candidates  :",
    len(archive_inventory),
)

print(
    "Files SHA256-hashed :",
    hashed_files,
)


# =================================================================================================
# 9. WRITE TEMPORARY RUNTIME MAP IF POSSIBLE
# =================================================================================================

payload = {
    "stage": "Stage28-1A-D0",
    "scientific_operation": False,
    "expected_parent": EXPECTED_HEAD,
    "frozen_source_count": 8,
    "direct_exact_match_count": len(resolved),
    "resolved": resolved,
    "archive_candidates": archive_inventory,
    "science_counters": {
        "predictor_values_read": 0,
        "label_values_read": 0,
        "model_fits": 0,
        "model_inference": 0,
        "threshold_selection": 0,
        "target_openings": 0,
    },
}

TEMP_MAP.write_text(
    json.dumps(
        payload,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

print()
print(
    "Temporary resolver map:"
)
print(
    " ",
    TEMP_MAP,
)


# =================================================================================================
# 10. FINAL
# =================================================================================================

banner("STAGE28-1A-D0 COMPLETE")

if len(resolved) == 8:

    print(
        "[PASS] All 8 frozen CICIDS2017 sources "
        "exist in the attached datasets."
    )

    print()
    print(
        "The problem was ONLY Kaggle filename/path renaming."
    )

    print()
    print(
        "NEXT:"
    )

    print(
        "  Use this exact resolver map in the corrected "
        "Stage28-1A reconstruction cell."
    )

else:

    print(
        f"[STOP] Only {len(resolved)}/8 frozen sources "
        "were found as direct byte-exact files."
    )

    print()

    if archive_inventory:

        print(
            "One or more unresolved sources appear to be "
            "inside attached archives."
        )

        print(
            "The next repair step will extract only the "
            "frozen candidate members and verify their SHA256."
        )

    else:

        print(
            "No matching archive members were detected."
        )

        print(
            "The inventory above will show what was actually "
            "attached to this fresh notebook."
        )

print()
print(
    "Git modifications      : 0"
)

print(
    "Predictor values read  : 0"
)

print(
    "Label values read      : 0"
)

print(
    "Model fits             : 0"
)

print(
    "Inference              : 0"
)

print(
    "Threshold selection    : 0"
)

print(
    "Target openings        : 0"
)

print()
print(SEP)


STAGE28-1A-D0 — REPOSITORY SAFETY GATE

Local HEAD : 0029c28c417d63ffb45ad4096ab7c44cbd875116
origin/main: 0029c28c417d63ffb45ad4096ab7c44cbd875116
Expected   : 0029c28c417d63ffb45ad4096ab7c44cbd875116
Git clean  : True

[PASS] failed Stage28-1A left no Git changes

LOAD FROZEN STAGE27 SOURCE IDENTITIES

[0] Monday     Monday-WorkingHours.pcap_ISCX.csv.parquet
    bytes  = 65,465,382
    sha256 = dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
[1] Tuesday    Tuesday-WorkingHours.pcap_ISCX.csv.parquet
    bytes  = 52,701,751
    sha256 = 27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4
[2] Wednesday  Wednesday-workingHours.pcap_ISCX.csv.parquet
    bytes  = 76,512,727
    sha256 = d23a259820b16e1ad54f9f3b58d5727c5032d383015f90bc7c07cebbdf8a7140
[3] Thursday   Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet
    bytes  = 27,901,448
    sha256 = 5da010354f0fc1040fd1fe65967096e1063475de8dd30ae4f657c07201d728a7
[4] Thursday   Thursday-Wor

In [8]:
# =================================================================================================
# STAGE28-1A-D1 — RECOVER BYTE-EXACT FROZEN CICIDS2017 ASSETS
#
# Source:
#   Hugging Face dataset: bvsam/cic-ids-2017
#   directory: traffic_labels/
#
# This cell:
#   - verifies exact Stage28-0A parent
#   - downloads the 8 Stage27 frozen parquet assets
#   - requires exact frozen byte size
#   - requires exact frozen SHA256
#   - materializes them under /kaggle/working/stage27_cicids2017_sources
#   - writes a temporary runtime source map
#
# ZERO SCIENTIFIC OPERATIONS:
#   LABEL_VALUES_READ        = 0
#   PREDICTOR_VALUES_READ    = 0
#   MODEL_FITS               = 0
#   MODEL_INFERENCE          = 0
#   THRESHOLD_SELECTION      = 0
#   TARGET_OPENINGS          = 0
#   GIT_MODIFICATIONS        = 0
# =================================================================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone


# =================================================================================================
# CONSTANTS
# =================================================================================================

SEP = "=" * 118

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_HEAD = (
    "0029c28c417d63ffb45ad4096ab7c44cbd875116"
)

SOURCE_RECEIPT_PATH = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
    / "source_effective_population_receipt.json"
)

HF_REPO_ID = "bvsam/cic-ids-2017"

# Pin the normalization commit associated with the frozen Stage27 parquet generation.
# Byte identity remains the ultimate gate; a download is accepted ONLY if SHA256 matches.
HF_PRIMARY_REVISION = (
    "b7e532345512edcd530cb1770dc76636aeb52802"
)

RUNTIME_SOURCE_ROOT = Path(
    "/kaggle/working/stage27_cicids2017_sources"
)

RUNTIME_SOURCE_MAP = Path(
    "/kaggle/working/stage28_1a_exact_source_map.json"
)


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, cwd=REPO):
    r = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if r.returncode != 0:
        raise RuntimeError(
            f"Command failed ({r.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{r.stdout}\n\n"
            f"STDERR:\n{r.stderr}"
        )

    return r.stdout.strip()


def sha256_file(path: Path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# =================================================================================================
# 1. EXACT REPOSITORY SAFETY GATE
# =================================================================================================

banner("STAGE28-1A-D1 — REPOSITORY SAFETY GATE")

head = run([
    "git",
    "rev-parse",
    "HEAD",
])

origin = run([
    "git",
    "rev-parse",
    "origin/main",
])

status = run([
    "git",
    "status",
    "--porcelain",
])

print("Local HEAD :", head)
print("origin/main:", origin)
print("Expected   :", EXPECTED_HEAD)
print("Git clean  :", not bool(status))

if head != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={head}"
    )

if origin != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected origin/main.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={origin}"
    )

if status:
    raise RuntimeError(
        "Git worktree must be clean:\n"
        + status
    )

print()
print("[PASS] exact Stage28-0A parent")
print("[PASS] Git worktree clean")


# =================================================================================================
# 2. LOAD FROZEN STAGE27 SOURCE RECEIPT
# =================================================================================================

banner("LOAD FROZEN STAGE27 SOURCE IDENTITIES")

if not SOURCE_RECEIPT_PATH.is_file():
    raise RuntimeError(
        f"Missing frozen receipt:\n{SOURCE_RECEIPT_PATH}"
    )

receipt = json.loads(
    SOURCE_RECEIPT_PATH.read_text(
        encoding="utf-8"
    )
)

segments = receipt["segments"]

if len(segments) != 8:
    raise RuntimeError(
        f"Expected 8 frozen segments, found {len(segments)}"
    )

for seg in segments:

    print(
        f"[{seg['source_index']}] "
        f"{seg['basename']}"
    )

    print(
        f"    bytes  = {seg['size_bytes']:,}"
    )

    print(
        f"    sha256 = {seg['sha256']}"
    )


# =================================================================================================
# 3. IMPORT / INSTALL HUGGING FACE CLIENT
# =================================================================================================

banner("HUGGING FACE CLIENT")

try:
    from huggingface_hub import hf_hub_download
    import huggingface_hub

except ImportError:

    print(
        "huggingface_hub not installed; installing..."
    )

    subprocess.run(
        [
            "python",
            "-m",
            "pip",
            "install",
            "-q",
            "huggingface_hub",
        ],
        check=True,
    )

    from huggingface_hub import hf_hub_download
    import huggingface_hub


print(
    "huggingface_hub version:",
    huggingface_hub.__version__,
)

print(
    "Dataset repo:",
    HF_REPO_ID,
)

print(
    "Primary pinned revision:",
    HF_PRIMARY_REVISION,
)


# =================================================================================================
# 4. PREPARE RUNTIME SOURCE DIRECTORY
# =================================================================================================

banner("PREPARE RUNTIME SOURCE DIRECTORY")

RUNTIME_SOURCE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    "Runtime source root:",
    RUNTIME_SOURCE_ROOT,
)

# Existing files are allowed ONLY if byte-exact.
# Nothing is silently overwritten until its identity is checked.


# =================================================================================================
# 5. DOWNLOAD + EXACT BYTE GATE
# =================================================================================================

banner("RECOVER 8 BYTE-EXACT CICIDS2017 PARQUETS")

resolved = []

for seg in segments:

    source_index = int(
        seg["source_index"]
    )

    basename = seg["basename"]

    remote_filename = (
        f"traffic_labels/{basename}"
    )

    expected_size = int(
        seg["size_bytes"]
    )

    expected_sha = seg["sha256"]

    destination = (
        RUNTIME_SOURCE_ROOT
        / basename
    )

    print()
    print("-" * 118)

    print(
        f"[{source_index}] {basename}"
    )

    print(
        "Remote:",
        remote_filename,
    )

    print(
        "Expected bytes :",
        f"{expected_size:,}",
    )

    print(
        "Expected SHA256:",
        expected_sha,
    )


    # ---------------------------------------------------------------------------------------------
    # Reuse existing runtime file only if byte-exact.
    # ---------------------------------------------------------------------------------------------

    if destination.is_file():

        current_size = (
            destination.stat().st_size
        )

        current_sha = sha256_file(
            destination
        )

        if (
            current_size == expected_size
            and current_sha == expected_sha
        ):

            print(
                "[REUSE EXACT]",
                destination,
            )

            resolved.append({
                "source_index": source_index,
                "day": seg["day"],
                "basename": basename,
                "remote_filename": remote_filename,
                "runtime_path": str(destination),
                "size_bytes": current_size,
                "sha256": current_sha,
                "resolution": "EXISTING_BYTE_EXACT_RUNTIME_FILE",
            })

            continue

        print(
            "[REMOVE] existing runtime file is not exact"
        )

        destination.unlink()


    # ---------------------------------------------------------------------------------------------
    # Attempt pinned revision first.
    # ---------------------------------------------------------------------------------------------

    download_path = None
    revision_used = None
    errors = []

    revisions_to_try = [
        HF_PRIMARY_REVISION,
        "main",
    ]

    for revision in revisions_to_try:

        print()
        print(
            "Trying revision:",
            revision,
        )

        try:

            candidate = Path(
                hf_hub_download(
                    repo_id=HF_REPO_ID,
                    repo_type="dataset",
                    filename=remote_filename,
                    revision=revision,
                )
            )

        except Exception as exc:

            errors.append({
                "revision": revision,
                "error": repr(exc),
            })

            print(
                "[DOWNLOAD FAILED]",
                type(exc).__name__,
                str(exc)[:400],
            )

            continue


        candidate_size = (
            candidate.stat().st_size
        )

        candidate_sha = (
            sha256_file(candidate)
        )

        print(
            "Downloaded cache path:",
            candidate,
        )

        print(
            "Actual bytes :",
            f"{candidate_size:,}",
        )

        print(
            "Actual SHA256:",
            candidate_sha,
        )


        # -----------------------------------------------------------------------------------------
        # SHA256 + size are the scientific identity.
        # -----------------------------------------------------------------------------------------

        if (
            candidate_size == expected_size
            and candidate_sha == expected_sha
        ):

            download_path = candidate
            revision_used = revision

            print(
                "[EXACT MATCH]"
            )

            break

        print(
            "[REJECT] object does not match "
            "the frozen Stage27 byte identity"
        )


    if download_path is None:

        raise RuntimeError(
            "\nCould not recover the exact frozen asset:\n"
            f"{basename}\n\n"
            f"Expected size : {expected_size}\n"
            f"Expected SHA  : {expected_sha}\n\n"
            f"Attempts:\n"
            f"{json.dumps(errors, indent=2)}"
        )


    # ---------------------------------------------------------------------------------------------
    # Copy out of HF cache into stable Kaggle working location.
    # ---------------------------------------------------------------------------------------------

    shutil.copyfile(
        download_path,
        destination,
    )


    # ---------------------------------------------------------------------------------------------
    # Verify copied bytes independently.
    # ---------------------------------------------------------------------------------------------

    final_size = (
        destination.stat().st_size
    )

    final_sha = sha256_file(
        destination
    )

    if final_size != expected_size:
        raise RuntimeError(
            f"Copied size mismatch for {basename}."
        )

    if final_sha != expected_sha:
        raise RuntimeError(
            f"Copied SHA mismatch for {basename}."
        )


    resolved.append({
        "source_index": source_index,
        "day": seg["day"],
        "basename": basename,
        "remote_filename": remote_filename,
        "hf_repo_id": HF_REPO_ID,
        "hf_revision_used": revision_used,
        "runtime_path": str(destination),
        "size_bytes": final_size,
        "sha256": final_sha,
        "resolution": "HF_DOWNLOAD_BYTE_EXACT",
    })

    print(
        "[PASS] stable runtime copy:",
        destination,
    )


# =================================================================================================
# 6. COMPLETE 8/8 GATE
# =================================================================================================

banner("8/8 FROZEN SOURCE GATE")

if len(resolved) != 8:
    raise RuntimeError(
        f"Expected 8 resolved assets; found {len(resolved)}"
    )

resolved.sort(
    key=lambda x: x["source_index"]
)

for item in resolved:

    destination = Path(
        item["runtime_path"]
    )

    expected = segments[
        item["source_index"]
    ]

    actual_size = (
        destination.stat().st_size
    )

    actual_sha = sha256_file(
        destination
    )

    if (
        actual_size
        != int(expected["size_bytes"])
    ):
        raise RuntimeError(
            f"Final size gate failed: {destination.name}"
        )

    if (
        actual_sha
        != expected["sha256"]
    ):
        raise RuntimeError(
            f"Final SHA gate failed: {destination.name}"
        )

    print(
        f"[PASS {item['source_index']}] "
        f"{destination.name}"
    )

    print(
        f"     bytes={actual_size:,}"
    )

    print(
        f"     sha256={actual_sha}"
    )


# =================================================================================================
# 7. TOTAL SIZE ACCOUNTING
# =================================================================================================

expected_total_bytes = sum(
    int(x["size_bytes"])
    for x in segments
)

actual_total_bytes = sum(
    Path(x["runtime_path"]).stat().st_size
    for x in resolved
)

print()
print(
    "Expected total bytes:",
    f"{expected_total_bytes:,}",
)

print(
    "Actual total bytes  :",
    f"{actual_total_bytes:,}",
)

if actual_total_bytes != expected_total_bytes:
    raise RuntimeError(
        "Total byte count mismatch."
    )


# =================================================================================================
# 8. WRITE TEMPORARY RUNTIME SOURCE MAP
# =================================================================================================

banner("WRITE RUNTIME SOURCE MAP")

source_map = {
    "stage": "Stage28-1A-D1",

    "type": "TEMPORARY_RUNTIME_SOURCE_RESOLUTION",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "scientific_parent": EXPECTED_HEAD,

    "source_dataset": {
        "provider": "HUGGING_FACE",
        "repo_id": HF_REPO_ID,
        "primary_revision": HF_PRIMARY_REVISION,
    },

    "source_count": 8,

    "runtime_root": str(
        RUNTIME_SOURCE_ROOT
    ),

    "resolved": resolved,

    "total_bytes": actual_total_bytes,

    "byte_identity": "8_OF_8_PASS_EXACT_SHA256_AND_SIZE",

    "scientific_operations": {
        "label_values_read": 0,
        "predictor_values_read": 0,
        "model_fits": 0,
        "model_inference": 0,
        "threshold_selection": 0,
        "target_openings": 0,
    },

    "committed_to_git": False,
}

RUNTIME_SOURCE_MAP.write_text(
    json.dumps(
        source_map,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

print(
    "Runtime map:",
    RUNTIME_SOURCE_MAP,
)

print(
    "Runtime source root:",
    RUNTIME_SOURCE_ROOT,
)


# =================================================================================================
# 9. GIT CLEANLINESS GATE
# =================================================================================================

banner("GIT CLEANLINESS GATE")

status_after = run([
    "git",
    "status",
    "--porcelain",
])

if status_after:
    raise RuntimeError(
        "Asset recovery unexpectedly modified Git:\n"
        + status_after
    )

print(
    "[PASS] Git remains clean"
)


# =================================================================================================
# 10. FINAL
# =================================================================================================

banner("STAGE28-1A-D1 — EXACT ASSET RECOVERY COMPLETE")

print(
    "Hugging Face dataset:",
    HF_REPO_ID,
)

print()

print(
    "Recovered assets:",
    len(resolved),
    "/ 8",
)

print()

for item in resolved:

    print(
        f"  [{item['source_index']}] "
        f"{item['basename']}"
    )

    print(
        "      revision:",
        item.get(
            "hf_revision_used",
            "EXISTING_RUNTIME",
        ),
    )

    print(
        "      sha256 :",
        item["sha256"],
    )

print()

print(
    "Runtime root:"
)

print(
    " ",
    RUNTIME_SOURCE_ROOT,
)

print()

print(
    "Scientific operations:"
)

print(
    "  LABEL_VALUES_READ        = 0"
)

print(
    "  PREDICTOR_VALUES_READ    = 0"
)

print(
    "  MODEL_FITS               = 0"
)

print(
    "  MODEL_INFERENCE          = 0"
)

print(
    "  THRESHOLD_SELECTION      = 0"
)

print(
    "  TARGET_OPENINGS          = 0"
)

print(
    "  GIT_MODIFICATIONS        = 0"
)

print()

print(
    "NEXT:"
)

print(
    "  Stage28-1A-R1 — reconstruct the inherited "
    "Stage27 membership cache from these exact assets "
    "and require the historical content SHA256s."
)

print()
print(SEP)


STAGE28-1A-D1 — REPOSITORY SAFETY GATE

Local HEAD : 0029c28c417d63ffb45ad4096ab7c44cbd875116
origin/main: 0029c28c417d63ffb45ad4096ab7c44cbd875116
Expected   : 0029c28c417d63ffb45ad4096ab7c44cbd875116
Git clean  : True

[PASS] exact Stage28-0A parent
[PASS] Git worktree clean

LOAD FROZEN STAGE27 SOURCE IDENTITIES

[0] Monday-WorkingHours.pcap_ISCX.csv.parquet
    bytes  = 65,465,382
    sha256 = dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
[1] Tuesday-WorkingHours.pcap_ISCX.csv.parquet
    bytes  = 52,701,751
    sha256 = 27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4
[2] Wednesday-workingHours.pcap_ISCX.csv.parquet
    bytes  = 76,512,727
    sha256 = d23a259820b16e1ad54f9f3b58d5727c5032d383015f90bc7c07cebbdf8a7140
[3] Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet
    bytes  = 27,901,448
    sha256 = 5da010354f0fc1040fd1fe65967096e1063475de8dd30ae4f657c07201d728a7
[4] Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.

traffic_labels/Monday-WorkingHours.pcap_(…):   0%|          | 0.00/65.5M [00:00<?, ?B/s]

Downloaded cache path: /root/.cache/huggingface/hub/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Monday-WorkingHours.pcap_ISCX.csv.parquet
Actual bytes : 65,465,382
Actual SHA256: dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
[EXACT MATCH]
[PASS] stable runtime copy: /kaggle/working/stage27_cicids2017_sources/Monday-WorkingHours.pcap_ISCX.csv.parquet

----------------------------------------------------------------------------------------------------------------------
[1] Tuesday-WorkingHours.pcap_ISCX.csv.parquet
Remote: traffic_labels/Tuesday-WorkingHours.pcap_ISCX.csv.parquet
Expected bytes : 52,701,751
Expected SHA256: 27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4

Trying revision: b7e532345512edcd530cb1770dc76636aeb52802


traffic_labels/Tuesday-WorkingHours.pcap(…):   0%|          | 0.00/52.7M [00:00<?, ?B/s]

Downloaded cache path: /root/.cache/huggingface/hub/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Tuesday-WorkingHours.pcap_ISCX.csv.parquet
Actual bytes : 52,701,751
Actual SHA256: 27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4
[EXACT MATCH]
[PASS] stable runtime copy: /kaggle/working/stage27_cicids2017_sources/Tuesday-WorkingHours.pcap_ISCX.csv.parquet

----------------------------------------------------------------------------------------------------------------------
[2] Wednesday-workingHours.pcap_ISCX.csv.parquet
Remote: traffic_labels/Wednesday-workingHours.pcap_ISCX.csv.parquet
Expected bytes : 76,512,727
Expected SHA256: d23a259820b16e1ad54f9f3b58d5727c5032d383015f90bc7c07cebbdf8a7140

Trying revision: b7e532345512edcd530cb1770dc76636aeb52802


...

Downloaded cache path: /root/.cache/huggingface/hub/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.parquet
Actual bytes : 23,048,086
Actual SHA256: 7c5876d52189fc01af54bad6cf23afe9f7fbc0e3ca6c3595920754f0c3ba8f66
[EXACT MATCH]
[PASS] stable runtime copy: /kaggle/working/stage27_cicids2017_sources/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.parquet

----------------------------------------------------------------------------------------------------------------------
[6] Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
Remote: traffic_labels/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
Expected bytes : 18,632,427
Expected SHA256: 4d78cee297c27f1a9947b9384793e587a46c7a3ea89db199553dabddc9835d4a

Trying revision: b7e532345512edcd530cb1770dc76636aeb52802


traffic_labels/Friday-WorkingHours-After(…):   0%|          | 0.00/18.6M [00:00<?, ?B/s]

Downloaded cache path: /root/.cache/huggingface/hub/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
Actual bytes : 18,632,427
Actual SHA256: 4d78cee297c27f1a9947b9384793e587a46c7a3ea89db199553dabddc9835d4a
[EXACT MATCH]
[PASS] stable runtime copy: /kaggle/working/stage27_cicids2017_sources/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet

----------------------------------------------------------------------------------------------------------------------
[7] Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet
Remote: traffic_labels/Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet
Expected bytes : 21,999,571
Expected SHA256: 2c00236b13a69f4b1c222b8f4a89451dc2148cb04a8cd0c45c2a87af51471774

Trying revision: b7e532345512edcd530cb1770dc76636aeb52802


traffic_labels/Friday-WorkingHours-Morni(…):   0%|          | 0.00/22.0M [00:00<?, ?B/s]

Downloaded cache path: /root/.cache/huggingface/hub/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet
Actual bytes : 21,999,571
Actual SHA256: 2c00236b13a69f4b1c222b8f4a89451dc2148cb04a8cd0c45c2a87af51471774
[EXACT MATCH]
[PASS] stable runtime copy: /kaggle/working/stage27_cicids2017_sources/Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet

8/8 FROZEN SOURCE GATE

[PASS 0] Monday-WorkingHours.pcap_ISCX.csv.parquet
     bytes=65,465,382
     sha256=dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
[PASS 1] Tuesday-WorkingHours.pcap_ISCX.csv.parquet
     bytes=52,701,751
     sha256=27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4
[PASS 2] Wednesday-workingHours.pcap_ISCX.csv.parquet
     bytes=76,512,727
     sha256=d23a259820b16e1ad54f9f3b58d5727c5032d383015f90bc7c07cebbdf8a7140
[PASS 3] Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet
     

In [9]:
# =================================================================================================
# STAGE28-1A-R1 — RECONSTRUCT + VERIFY INHERITED STAGE27 MEMBERSHIPS
#
# Requires runtime assets recovered by Stage28-1A-D1:
#   /kaggle/working/stage27_cicids2017_sources/
#
# Scientific purpose:
#   1. audit frozen Stage22 FULL memberships
#   2. verify the recovered 8 CICIDS2017 assets again
#   3. read LABELS ONLY and reconstruct:
#        - global_family_codes
#        - global_day_codes
#        - global_source_codes
#   4. require exact historical Stage27 content SHA256 for all 3 arrays
#   5. reconstruct the five eligible chronology-first LOAO memberships
#   6. require exact historical content SHA256 for every membership array
#   7. reproduce frozen Stage27 per-fold class weights
#   8. freeze durable reconstruction receipts
#   9. commit + push + remotely verify
#
# ZERO:
#   PREDICTOR VALUES READ
#   MODEL FITS
#   MODEL INFERENCE
#   THRESHOLD SELECTION
#   TARGET PREDICTOR OPENINGS
#
# Expected scientific parent:
#   0029c28c417d63ffb45ad4096ab7c44cbd875116
#
# NOTE:
#   Reading labels for membership reconstruction is permitted and recorded.
#   Runtime .npy caches live outside Git.
# =================================================================================================

from __future__ import annotations

import hashlib
import json
import math
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 118

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_BRANCH = "main"

EXPECTED_PARENT = (
    "0029c28c417d63ffb45ad4096ab7c44cbd875116"
)

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

OUTPUT_DIR = (
    STAGE28_ROOT
    / "stage28_1a_inherited_membership_reconstruction"
)

SOURCE_ROOT = Path(
    "/kaggle/working/stage27_cicids2017_sources"
).resolve()

SOURCE_MAP_PATH = Path(
    "/kaggle/working/stage28_1a_exact_source_map.json"
)

RUNTIME_ROOT = Path(
    "/kaggle/working/stage28_1a_runtime_cache"
)

GLOBAL_CACHE_ROOT = (
    RUNTIME_ROOT
    / "stage27_global_cache"
)

CHRONO_CACHE_ROOT = (
    RUNTIME_ROOT
    / "stage27_chronological_memberships"
)

COMMIT_MESSAGE = (
    "stage28-1a: reconstruct and verify inherited memberships"
)

ELIGIBLE_FAMILIES = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

PRIMARY_SEVEN = [
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, cwd=REPO, check=True):
    r = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and r.returncode != 0:
        raise RuntimeError(
            f"Command failed ({r.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{r.stdout}\n\n"
            f"STDERR:\n{r.stderr}"
        )

    return r


def git(*args, check=True):
    return (
        run(
            ["git", *args],
            check=check,
        ).stdout
        or ""
    ).strip()


def read_json(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def write_json(path: Path, obj):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            sort_keys=False,
        )
        + "\n",
        encoding="utf-8",
    )


def require_file(path: Path):
    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def sha256_file(
    path: Path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array_content(
    arr: np.ndarray,
):
    arr = np.ascontiguousarray(arr)

    h = hashlib.sha256()
    h.update(
        arr.tobytes(order="C")
    )

    return h.hexdigest()


def remote_main_sha():
    out = git(
        "ls-remote",
        "origin",
        "refs/heads/main",
    )

    if not out:
        raise RuntimeError(
            "Unable to resolve remote main."
        )

    return out.split()[0]


# =================================================================================================
# 2. EXACT REPOSITORY GATE
# =================================================================================================

banner(
    "STAGE28-1A-R1 — EXACT REPOSITORY GATE"
)

branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = remote_main_sha()

status = git(
    "status",
    "--porcelain",
)

print("Repository :", REPO)
print("Branch     :", branch)
print("Local HEAD :", head)
print("Remote main:", remote)
print("Expected   :", EXPECTED_PARENT)
print("Git clean  :", not bool(status))

if branch != EXPECTED_BRANCH:
    raise RuntimeError(
        f"Expected branch {EXPECTED_BRANCH}, "
        f"got {branch}"
    )

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Local HEAD mismatch.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )

if remote != EXPECTED_PARENT:
    raise RuntimeError(
        "Remote main mismatch.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={remote}"
    )

if status:
    raise RuntimeError(
        "Worktree must be clean:\n"
        + status
    )

if OUTPUT_DIR.exists():
    raise RuntimeError(
        "Stage28-1A durable output directory "
        "already exists:\n"
        f"{OUTPUT_DIR}"
    )

print()
print("[PASS] exact Stage28-0A parent")
print("[PASS] local == remote")
print("[PASS] worktree clean")
print("[PASS] Stage28-1A durable destination absent")


# =================================================================================================
# 3. LOAD FROZEN PROVENANCE
# =================================================================================================

banner(
    "LOAD FROZEN STAGE28 / STAGE27 PROVENANCE"
)

effective_budget_path = require_file(
    STAGE28_ROOT
    / "stage28_0a_preexecution_amendment"
    / "effective_fit_budget.json"
)

source_receipt_path = require_file(
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
    / "source_effective_population_receipt.json"
)

family_cache_receipt_path = require_file(
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
    / "family_code_cache_receipt.json"
)

census_path = require_file(
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
    / "canonical_label_census.json"
)

budget = read_json(
    effective_budget_path
)

source_receipt = read_json(
    source_receipt_path
)

family_cache_receipt = read_json(
    family_cache_receipt_path
)

census = read_json(
    census_path
)

assert (
    budget["totals"]["new_fit_budget"]
    == 108
)

assert (
    budget["totals"]["existing_reused"]
    == 12
)

assert (
    budget["totals"][
        "total_component_realizations"
    ]
    == 120
)

assert (
    source_receipt[
        "population"
    ]["effective_rows"]
    == 2_830_743
)

print(
    "[PASS] effective new-fit budget = 108"
)
print(
    "[PASS] existing reused fits = 12"
)
print(
    "[PASS] component universe = 120"
)
print(
    "[PASS] Stage27 effective population = 2,830,743"
)


# =================================================================================================
# 4. STAGE22R FROZEN FULL-MEMBERSHIP AUDIT
# =================================================================================================

banner(
    "STAGE22R FROZEN FULL-MEMBERSHIP AUDIT"
)

stage22_membership_dir = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
)

stage22_summary_path = require_file(
    stage22_membership_dir
    / "stage22r_1b1_membership_summary.json"
)

random_validation_path = require_file(
    stage22_membership_dir
    / "random_validation.packbits"
)

stage22_summary = read_json(
    stage22_summary_path
)

random_artifact = (
    stage22_summary[
        "artifacts"
    ][
        "random_validation.packbits"
    ]
)

expected_random_sha = (
    random_artifact[
        "sha256"
    ]
)

actual_random_sha = sha256_file(
    random_validation_path
)

logical_length = int(
    random_artifact[
        "logical_length"
    ]
)

expected_population = int(
    random_artifact[
        "population"
    ]
)

print(
    "random_validation.packbits"
)
print(
    "  expected SHA:",
    expected_random_sha,
)
print(
    "  actual SHA  :",
    actual_random_sha,
)

if actual_random_sha != expected_random_sha:
    raise RuntimeError(
        "Stage22 random-validation "
        "membership SHA mismatch."
    )

packed = np.fromfile(
    random_validation_path,
    dtype=np.uint8,
)

bits = np.unpackbits(
    packed,
    bitorder="little",
)[:logical_length]

actual_population = int(
    bits.sum()
)

if actual_population != expected_population:
    raise RuntimeError(
        "Stage22 random validation "
        "population mismatch."
    )

cells22 = stage22_summary[
    "cells"
]

expected_stage22_counts = {
    "RANDOM_NATURAL": {
        "train_rows": 11_529_922,
        "validation_rows": 2_882_481,
    },
    "CHRONOLOGICAL_NATURAL": {
        "train_rows": 13_818_623,
        "validation_rows": 593_780,
    },
}

for cell, expected in (
    expected_stage22_counts.items()
):
    observed_train = int(
        cells22[cell][
            "train"
        ]["rows"]
    )

    observed_validation = int(
        cells22[cell][
            "validation"
        ]["rows"]
    )

    if (
        observed_train
        != expected["train_rows"]
        or observed_validation
        != expected["validation_rows"]
    ):
        raise RuntimeError(
            f"Stage22 membership count mismatch "
            f"for {cell}."
        )

print()
print(
    "[PASS] Stage22 RANDOM_NATURAL membership exact"
)
print(
    "[PASS] Stage22 CHRONOLOGICAL_NATURAL counts exact"
)

del packed
del bits


# =================================================================================================
# 5. VERIFY RECOVERED CICIDS2017 RUNTIME ASSETS
# =================================================================================================

banner(
    "VERIFY RECOVERED CICIDS2017 RUNTIME ASSETS"
)

if not SOURCE_ROOT.is_dir():
    raise RuntimeError(
        "Recovered source root missing:\n"
        f"{SOURCE_ROOT}"
    )

if not SOURCE_MAP_PATH.is_file():
    raise RuntimeError(
        "Stage28-1A-D1 runtime source map missing:\n"
        f"{SOURCE_MAP_PATH}"
    )

source_map = read_json(
    SOURCE_MAP_PATH
)

if (
    source_map.get(
        "byte_identity"
    )
    != "8_OF_8_PASS_EXACT_SHA256_AND_SIZE"
):
    raise RuntimeError(
        "Runtime source-map byte identity "
        "is not exact."
    )

segments = source_receipt[
    "segments"
]

if len(segments) != 8:
    raise RuntimeError(
        "Expected exactly 8 Stage27 segments."
    )

resolved_sources = []

for seg in segments:

    source_index = int(
        seg["source_index"]
    )

    basename = seg[
        "basename"
    ]

    expected_size = int(
        seg["size_bytes"]
    )

    expected_sha = seg[
        "sha256"
    ]

    path = (
        SOURCE_ROOT
        / basename
    )

    require_file(path)

    actual_size = (
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )

    print(
        f"[{source_index}] {basename}"
    )
    print(
        f"    bytes : {actual_size:,}"
    )
    print(
        f"    sha256: {actual_sha}"
    )

    if actual_size != expected_size:
        raise RuntimeError(
            f"{basename}: size mismatch."
        )

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"{basename}: SHA mismatch."
        )

    resolved_sources.append(
        {
            "segment": seg,
            "path": path,
        }
    )

print()
print(
    "[PASS] 8/8 recovered assets remain byte-exact"
)


# =================================================================================================
# 6. FROZEN LABEL CANONICALIZATION
# =================================================================================================

banner(
    "FROZEN STAGE27 LABEL TAXONOMY"
)

codebook = (
    family_cache_receipt[
        "family_codebook"
    ]
)

EXPECTED_CODEBOOK = {
    "BENIGN": 0,
    "BOT": 1,
    "DDOS": 2,
    "DOS": 3,
    "AUTH_BRUTE_FORCE": 4,
    "INFILTRATION": 5,
    "PORT_SCAN": 6,
    "WEB_ATTACK": 7,
    "TARGET_ONLY_UNSEEN": 8,
    "OTHER_ATTACK_UNSEEN_LABEL": 9,
}

if codebook != EXPECTED_CODEBOOK:
    raise RuntimeError(
        "Frozen Stage27 codebook mismatch."
    )


LABEL_TO_CODE = {

    "benign": 0,

    "bot": 1,

    "ddos": 2,

    "dos goldeneye": 3,
    "dos hulk": 3,
    "dos slowhttptest": 3,
    "dos slowloris": 3,

    "ftp-patator": 4,
    "ssh-patator": 4,

    "infiltration": 5,

    "portscan": 6,

    "web attack - brute force": 7,
    "web attack - sql injection": 7,
    "web attack - xss": 7,

    "heartbleed": 8,
}


UNICODE_DASHES = [
    "\u2010",
    "\u2011",
    "\u2012",
    "\u2013",
    "\u2014",
    "\u2015",
    "\u2212",
    "\ufe58",
    "\ufe63",
    "\uff0d",
]


WEB_ATTACK_CP1252_ALIASES = {
    "web attack \u0096 brute force":
        "web attack - brute force",

    "web attack \u0096 sql injection":
        "web attack - sql injection",

    "web attack \u0096 xss":
        "web attack - xss",
}


def canonicalize_labels(
    series: pd.Series,
):
    s = series.astype(
        "string"
    )

    s = s.str.strip()

    for dash in UNICODE_DASHES:
        s = s.str.replace(
            dash,
            "-",
            regex=False,
        )

    s = s.str.replace(
        r"\s+",
        " ",
        regex=True,
    )

    s = s.str.casefold()

    s = s.replace(
        WEB_ATTACK_CP1252_ALIASES
    )

    return s


print(
    "[PASS] frozen Stage27 codebook loaded"
)
print(
    "[PASS] Web-Attack U+0096 compatibility aliases frozen"
)


# =================================================================================================
# 7. RECONSTRUCT GLOBAL ARRAYS FROM LABELS ONLY
# =================================================================================================

banner(
    "RECONSTRUCT STAGE27 GLOBAL ARRAYS — LABELS ONLY"
)

total_effective_rows = int(
    source_receipt[
        "population"
    ]["effective_rows"]
)

global_family_codes = np.empty(
    total_effective_rows,
    dtype=np.uint8,
)

global_day_codes = np.empty(
    total_effective_rows,
    dtype=np.uint8,
)

global_source_codes = np.empty(
    total_effective_rows,
    dtype=np.uint8,
)

physical_label_rows_read = 0
effective_label_rows_used = 0

source_execution_records = []


for item in resolved_sources:

    seg = item["segment"]
    path = item["path"]

    source_index = int(
        seg["source_index"]
    )

    day_name = seg[
        "day"
    ]

    day_code = int(
        seg["day_code"]
    )

    expected_physical_rows = int(
        seg["physical_rows"]
    )

    effective_rows = int(
        seg["effective_rows"]
    )

    global_start = int(
        seg["global_start_zero_based"]
    )

    global_stop = int(
        seg["global_stop_exclusive"]
    )

    print()
    print("-" * 118)
    print(
        f"[SOURCE {source_index}] "
        f"{day_name} — {path.name}"
    )

    pf = pq.ParquetFile(
        path
    )

    physical_rows = int(
        pf.metadata.num_rows
    )

    if (
        physical_rows
        != expected_physical_rows
    ):
        raise RuntimeError(
            f"{path.name}: physical-row mismatch.\n"
            f"expected={expected_physical_rows}\n"
            f"actual={physical_rows}"
        )

    schema_names = (
        pf.schema_arrow.names
    )

    label_candidates = [
        col
        for col in schema_names
        if col.strip().casefold()
        == "label"
    ]

    if len(label_candidates) != 1:
        raise RuntimeError(
            f"{path.name}: Label column could "
            f"not be uniquely identified.\n"
            f"Candidates={label_candidates}\n"
            f"Schema={schema_names}"
        )

    label_col = (
        label_candidates[0]
    )

    # LABEL COLUMN ONLY.
    table = pq.read_table(
        path,
        columns=[
            label_col,
        ],
    )

    labels = (
        table
        .column(0)
        .to_pandas()
    )

    physical_label_rows_read += int(
        len(labels)
    )

    if len(labels) != physical_rows:
        raise RuntimeError(
            f"{path.name}: label-row read mismatch."
        )

    # Frozen effective-population rule.
    if effective_rows < physical_rows:

        rule = seg[
            "inclusion_rule"
        ]

        expected_prefix = (
            "INCLUDE_PHYSICAL_ORDINALS_1_THROUGH_"
        )

        if not rule.startswith(
            expected_prefix
        ):
            raise RuntimeError(
                f"{path.name}: unsupported "
                "partial-source inclusion rule."
            )

        labels = (
            labels
            .iloc[:effective_rows]
            .copy()
        )

    elif effective_rows != physical_rows:
        raise RuntimeError(
            f"{path.name}: invalid effective-row geometry."
        )

    effective_label_rows_used += int(
        len(labels)
    )

    canonical = (
        canonicalize_labels(
            labels
        )
    )

    mapped = canonical.map(
        LABEL_TO_CODE
    )

    if mapped.isna().any():

        unknown = (
            canonical[
                mapped.isna()
            ]
            .value_counts(
                dropna=False
            )
        )

        print()
        print(
            "UNKNOWN CANONICAL LABELS:"
        )
        print(
            unknown.to_string()
        )

        raise RuntimeError(
            f"{path.name}: encountered label "
            "outside frozen Stage27 taxonomy."
        )

    codes = mapped.to_numpy(
        dtype=np.uint8,
        copy=True,
    )

    if len(codes) != effective_rows:
        raise RuntimeError(
            f"{path.name}: effective code "
            "length mismatch."
        )

    if (
        global_stop - global_start
        != effective_rows
    ):
        raise RuntimeError(
            f"{path.name}: global geometry mismatch."
        )

    global_family_codes[
        global_start:global_stop
    ] = codes

    global_day_codes[
        global_start:global_stop
    ] = day_code

    global_source_codes[
        global_start:global_stop
    ] = source_index

    local_counts = np.bincount(
        codes,
        minlength=10,
    )

    nonzero_counts = {
        family: int(
            local_counts[code]
        )
        for family, code
        in codebook.items()
        if local_counts[code] > 0
    }

    print(
        "Physical rows :",
        f"{physical_rows:,}",
    )
    print(
        "Effective rows:",
        f"{effective_rows:,}",
    )
    print(
        "Label column  :",
        repr(label_col),
    )
    print(
        "Family counts :",
        nonzero_counts,
    )

    source_execution_records.append(
        {
            "source_index":
                source_index,

            "day":
                day_name,

            "basename":
                path.name,

            "runtime_path":
                str(path),

            "size_bytes":
                path.stat().st_size,

            "sha256":
                sha256_file(path),

            "physical_rows":
                physical_rows,

            "effective_rows":
                effective_rows,

            "global_start_zero_based":
                global_start,

            "global_stop_exclusive":
                global_stop,

            "label_column":
                label_col,

            "family_counts":
                nonzero_counts,
        }
    )

    del table
    del labels
    del canonical
    del mapped
    del codes


if (
    effective_label_rows_used
    != total_effective_rows
):
    raise RuntimeError(
        "Effective label-row total mismatch."
    )

print()
print(
    "Physical label rows read :",
    f"{physical_label_rows_read:,}",
)

print(
    "Effective label rows used:",
    f"{effective_label_rows_used:,}",
)

print(
    "Predictor values read    : 0"
)


# =================================================================================================
# 8. GLOBAL CENSUS REPRODUCTION
# =================================================================================================

banner(
    "GLOBAL FAMILY / DAY CENSUS REPRODUCTION"
)

observed_count_array = np.bincount(
    global_family_codes,
    minlength=10,
)

observed_family_counts = {
    family: int(
        observed_count_array[
            code
        ]
    )
    for family, code
    in codebook.items()
}

expected_family_counts = (
    census[
        "family_counts"
    ]
)

print(
    "Expected family census:"
)
print(
    json.dumps(
        expected_family_counts,
        indent=2,
    )
)

print()
print(
    "Observed family census:"
)
print(
    json.dumps(
        observed_family_counts,
        indent=2,
    )
)

if (
    observed_family_counts
    != expected_family_counts
):
    raise RuntimeError(
        "Global family census "
        "does not reproduce exactly."
    )


day_name_to_code = {}

for seg in segments:
    day_name_to_code[
        seg["day"]
    ] = int(
        seg["day_code"]
    )


# Family × day exact audit.
for family, expected_days in (
    census[
        "family_day_counts"
    ].items()
):

    family_code = (
        codebook[
            family
        ]
    )

    for day_name, expected_count in (
        expected_days.items()
    ):

        day_code = (
            day_name_to_code[
                day_name
            ]
        )

        observed = int(
            np.count_nonzero(
                (
                    global_family_codes
                    == family_code
                )
                &
                (
                    global_day_codes
                    == day_code
                )
            )
        )

        if observed != int(
            expected_count
        ):
            raise RuntimeError(
                "Family/day census mismatch:\n"
                f"family={family}\n"
                f"day={day_name}\n"
                f"expected={expected_count}\n"
                f"actual={observed}"
            )


# Benign × day exact audit.
for day_name, expected_count in (
    census[
        "benign_day_counts"
    ].items()
):

    day_code = (
        day_name_to_code[
            day_name
        ]
    )

    observed = int(
        np.count_nonzero(
            (
                global_family_codes
                == codebook["BENIGN"]
            )
            &
            (
                global_day_codes
                == day_code
            )
        )
    )

    if observed != int(
        expected_count
    ):
        raise RuntimeError(
            "Benign/day census mismatch:\n"
            f"day={day_name}\n"
            f"expected={expected_count}\n"
            f"actual={observed}"
        )

print()
print(
    "[PASS] global family census exact"
)
print(
    "[PASS] family × day census exact"
)
print(
    "[PASS] benign × day census exact"
)


# =================================================================================================
# 9. HISTORICAL GLOBAL CONTENT-SHA256 GATE
# =================================================================================================

banner(
    "HISTORICAL GLOBAL ARRAY CONTENT-SHA256 GATE"
)

expected_arrays = (
    family_cache_receipt[
        "arrays"
    ]
)

actual_global_hashes = {
    "global_family_codes":
        sha256_array_content(
            global_family_codes
        ),

    "global_day_codes":
        sha256_array_content(
            global_day_codes
        ),

    "global_source_codes":
        sha256_array_content(
            global_source_codes
        ),
}

for name, actual_sha in (
    actual_global_hashes.items()
):

    expected_sha = (
        expected_arrays[
            name
        ][
            "content_sha256"
        ]
    )

    expected_count = int(
        expected_arrays[
            name
        ][
            "count"
        ]
    )

    actual_count = int(
        len(
            {
                "global_family_codes":
                    global_family_codes,

                "global_day_codes":
                    global_day_codes,

                "global_source_codes":
                    global_source_codes,
            }[name]
        )
    )

    print(name)
    print(
        "  rows    :",
        f"{actual_count:,}",
    )
    print(
        "  expected:",
        expected_sha,
    )
    print(
        "  actual  :",
        actual_sha,
    )

    if actual_count != expected_count:
        raise RuntimeError(
            f"{name}: count mismatch."
        )

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"{name}: historical "
            "content SHA256 mismatch."
        )

print()
print(
    "[PASS] all 3 global runtime arrays "
    "reproduced byte-for-byte"
)


# =================================================================================================
# 10. SAVE GLOBAL RUNTIME CACHE OUTSIDE GIT
# =================================================================================================

banner(
    "SAVE VERIFIED GLOBAL RUNTIME CACHE"
)

GLOBAL_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

global_runtime_paths = {}

for name, arr in {
    "global_family_codes":
        global_family_codes,

    "global_day_codes":
        global_day_codes,

    "global_source_codes":
        global_source_codes,
}.items():

    path = (
        GLOBAL_CACHE_ROOT
        / f"{name}.npy"
    )

    np.save(
        path,
        arr,
        allow_pickle=False,
    )

    global_runtime_paths[
        name
    ] = str(path)

    print(
        name,
        "->",
        path,
    )

print()
print(
    "[PASS] runtime cache materialized outside Git"
)


# =================================================================================================
# 11. RECONSTRUCT FIVE EXACT CHRONOLOGY-FIRST LOAO FOLDS
# =================================================================================================

banner(
    "RECONSTRUCT FIVE EXACT STAGE27 CHRONOLOGY-FIRST LOAO FOLDS"
)

primary_codes = np.array(
    [
        codebook[
            family
        ]
        for family
        in PRIMARY_SEVEN
    ],
    dtype=np.uint8,
)

benign_mask_global = (
    global_family_codes
    == codebook["BENIGN"]
)

primary_attack_mask_global = (
    np.isin(
        global_family_codes,
        primary_codes,
    )
)

fold_receipts = {}


for family in ELIGIBLE_FAMILIES:

    print()
    print("=" * 118)
    print(
        "FOLD:",
        family,
    )
    print("=" * 118)

    historical_path = require_file(
        REPO
        / "results"
        / "stage27_loao_unseen_attack"
        / "stage27_1a_fold_membership"
        / (
            f"fold_{family}_"
            "membership_receipt.json"
        )
    )

    historical = read_json(
        historical_path
    )

    if (
        historical[
            "held_out_family"
        ]
        != family
    ):
        raise RuntimeError(
            f"{family}: historical fold "
            "identity mismatch."
        )

    heldout_code = (
        codebook[
            family
        ]
    )

    train_day_codes = np.array(
        [
            day_name_to_code[
                day
            ]
            for day
            in historical[
                "train_days"
            ]
        ],
        dtype=np.uint8,
    )

    validation_day_codes = np.array(
        [
            day_name_to_code[
                day
            ]
            for day
            in historical[
                "validation_days"
            ]
        ],
        dtype=np.uint8,
    )

    target_day_codes = np.array(
        [
            day_name_to_code[
                day
            ]
            for day
            in historical[
                "target_days"
            ]
        ],
        dtype=np.uint8,
    )

    heldout_global = (
        global_family_codes
        == heldout_code
    )

    known_attack_global = (
        primary_attack_mask_global
        &
        (
            global_family_codes
            != heldout_code
        )
    )

    train_day_mask = np.isin(
        global_day_codes,
        train_day_codes,
    )

    validation_day_mask = np.isin(
        global_day_codes,
        validation_day_codes,
    )

    target_day_mask = np.isin(
        global_day_codes,
        target_day_codes,
    )


    # ---------------------------------------------------------------------------------------------
    # Frozen Stage27 semantics.
    # ---------------------------------------------------------------------------------------------

    train_mask = (
        train_day_mask
        &
        (
            benign_mask_global
            |
            known_attack_global
        )
    )

    validation_mask = (
        validation_day_mask
        &
        (
            benign_mask_global
            |
            known_attack_global
        )
    )

    primary_target_mask = (
        target_day_mask
        &
        (
            benign_mask_global
            |
            heldout_global
        )
    )

    operational_target_mask = (
        target_day_mask
        &
        (
            benign_mask_global
            |
            primary_attack_mask_global
        )
    )


    # ---------------------------------------------------------------------------------------------
    # Frozen zero-based global row IDs.
    # ---------------------------------------------------------------------------------------------

    train_idx = np.flatnonzero(
        train_mask
    ).astype(
        np.int32,
        copy=False,
    )

    validation_idx = np.flatnonzero(
        validation_mask
    ).astype(
        np.int32,
        copy=False,
    )

    primary_target_idx = (
        np.flatnonzero(
            primary_target_mask
        ).astype(
            np.int32,
            copy=False,
        )
    )

    operational_target_idx = (
        np.flatnonzero(
            operational_target_mask
        ).astype(
            np.int32,
            copy=False,
        )
    )


    # ---------------------------------------------------------------------------------------------
    # Disjointness.
    # ---------------------------------------------------------------------------------------------

    if np.intersect1d(
        train_idx,
        validation_idx,
        assume_unique=True,
    ).size:
        raise RuntimeError(
            f"{family}: TRAIN/VALIDATION overlap."
        )

    if np.intersect1d(
        train_idx,
        primary_target_idx,
        assume_unique=True,
    ).size:
        raise RuntimeError(
            f"{family}: TRAIN/TARGET overlap."
        )

    if np.intersect1d(
        validation_idx,
        primary_target_idx,
        assume_unique=True,
    ).size:
        raise RuntimeError(
            f"{family}: VALIDATION/TARGET overlap."
        )


    # ---------------------------------------------------------------------------------------------
    # Held-out exclusion.
    # ---------------------------------------------------------------------------------------------

    heldout_train_count = int(
        np.count_nonzero(
            global_family_codes[
                train_idx
            ]
            == heldout_code
        )
    )

    heldout_validation_count = int(
        np.count_nonzero(
            global_family_codes[
                validation_idx
            ]
            == heldout_code
        )
    )

    if heldout_train_count != 0:
        raise RuntimeError(
            f"{family}: held-out family "
            "leaked into TRAIN."
        )

    if heldout_validation_count != 0:
        raise RuntimeError(
            f"{family}: held-out family "
            "leaked into VALIDATION."
        )


    # ---------------------------------------------------------------------------------------------
    # Explicit unseen-category exclusion.
    # ---------------------------------------------------------------------------------------------

    excluded_codes = np.array(
        [
            codebook[
                "TARGET_ONLY_UNSEEN"
            ],
            codebook[
                "OTHER_ATTACK_UNSEEN_LABEL"
            ],
        ],
        dtype=np.uint8,
    )

    excluded_train = int(
        np.count_nonzero(
            np.isin(
                global_family_codes[
                    train_idx
                ],
                excluded_codes,
            )
        )
    )

    excluded_validation = int(
        np.count_nonzero(
            np.isin(
                global_family_codes[
                    validation_idx
                ],
                excluded_codes,
            )
        )
    )

    if excluded_train != 0:
        raise RuntimeError(
            f"{family}: target-only/other unseen "
            "rows appeared in TRAIN."
        )

    if excluded_validation != 0:
        raise RuntimeError(
            f"{family}: target-only/other unseen "
            "rows appeared in VALIDATION."
        )


    # ---------------------------------------------------------------------------------------------
    # Counts.
    # ---------------------------------------------------------------------------------------------

    observed_counts = {

        "train_rows":
            int(
                len(
                    train_idx
                )
            ),

        "train_benign":
            int(
                np.count_nonzero(
                    global_family_codes[
                        train_idx
                    ]
                    == codebook[
                        "BENIGN"
                    ]
                )
            ),

        "train_attack":
            int(
                np.count_nonzero(
                    global_family_codes[
                        train_idx
                    ]
                    != codebook[
                        "BENIGN"
                    ]
                )
            ),

        "validation_rows":
            int(
                len(
                    validation_idx
                )
            ),

        "validation_benign":
            int(
                np.count_nonzero(
                    global_family_codes[
                        validation_idx
                    ]
                    == codebook[
                        "BENIGN"
                    ]
                )
            ),

        "validation_attack":
            int(
                np.count_nonzero(
                    global_family_codes[
                        validation_idx
                    ]
                    != codebook[
                        "BENIGN"
                    ]
                )
            ),

        "primary_target_rows":
            int(
                len(
                    primary_target_idx
                )
            ),

        "primary_target_benign":
            int(
                np.count_nonzero(
                    global_family_codes[
                        primary_target_idx
                    ]
                    == codebook[
                        "BENIGN"
                    ]
                )
            ),

        "primary_target_heldout_attack":
            int(
                np.count_nonzero(
                    global_family_codes[
                        primary_target_idx
                    ]
                    == heldout_code
                )
            ),

        "operational_target_rows":
            int(
                len(
                    operational_target_idx
                )
            ),

        "operational_target_benign":
            int(
                np.count_nonzero(
                    global_family_codes[
                        operational_target_idx
                    ]
                    == codebook[
                        "BENIGN"
                    ]
                )
            ),

        "operational_target_attack":
            int(
                np.count_nonzero(
                    global_family_codes[
                        operational_target_idx
                    ]
                    != codebook[
                        "BENIGN"
                    ]
                )
            ),
    }

    expected_counts = (
        historical[
            "counts"
        ]
    )

    print(
        "Expected counts:"
    )
    print(
        json.dumps(
            expected_counts,
            indent=2,
        )
    )

    print()
    print(
        "Observed counts:"
    )
    print(
        json.dumps(
            observed_counts,
            indent=2,
        )
    )

    if observed_counts != expected_counts:
        raise RuntimeError(
            f"{family}: historical fold "
            "counts do not reproduce exactly."
        )


    # ---------------------------------------------------------------------------------------------
    # Historical content-SHA gate.
    # ---------------------------------------------------------------------------------------------

    arrays = {
        "train_global_idx":
            train_idx,

        "validation_global_idx":
            validation_idx,

        "primary_target_global_idx":
            primary_target_idx,

        "operational_target_global_idx":
            operational_target_idx,
    }

    array_receipts = {}

    for key, arr in arrays.items():

        actual_sha = (
            sha256_array_content(
                arr
            )
        )

        expected_info = (
            historical[
                "runtime_membership_arrays"
            ][
                key
            ]
        )

        expected_sha = (
            expected_info[
                "content_sha256"
            ]
        )

        expected_count = int(
            expected_info[
                "count"
            ]
        )

        print()
        print(key)
        print(
            "  rows    :",
            f"{len(arr):,}",
        )
        print(
            "  expected:",
            expected_sha,
        )
        print(
            "  actual  :",
            actual_sha,
        )

        if len(arr) != expected_count:
            raise RuntimeError(
                f"{family}/{key}: "
                "count mismatch."
            )

        if actual_sha != expected_sha:
            raise RuntimeError(
                f"{family}/{key}: historical "
                "content SHA mismatch."
            )

        array_receipts[
            key
        ] = {
            "dtype":
                str(
                    arr.dtype
                ),

            "count":
                int(
                    len(arr)
                ),

            "content_sha256":
                actual_sha,

            "historical_content_sha256":
                expected_sha,

            "exact_match":
                True,
        }


    # ---------------------------------------------------------------------------------------------
    # Class-weight reproduction.
    # ---------------------------------------------------------------------------------------------

    train_benign = (
        observed_counts[
            "train_benign"
        ]
    )

    train_attack = (
        observed_counts[
            "train_attack"
        ]
    )

    if train_attack <= 0:
        raise RuntimeError(
            f"{family}: zero training attacks."
        )

    class_weight = (
        train_benign
        / train_attack
    )

    historical_weight = float(
        historical[
            "class_weight"
        ][
            "realized_value"
        ]
    )

    print()
    print(
        "Class weight:"
    )
    print(
        "  recomputed :",
        class_weight,
    )
    print(
        "  historical :",
        historical_weight,
    )

    if not math.isclose(
        class_weight,
        historical_weight,
        rel_tol=0.0,
        abs_tol=1e-15,
    ):
        raise RuntimeError(
            f"{family}: class-weight mismatch."
        )


    # ---------------------------------------------------------------------------------------------
    # Save runtime arrays outside Git.
    # ---------------------------------------------------------------------------------------------

    family_runtime_dir = (
        CHRONO_CACHE_ROOT
        / family
    )

    family_runtime_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    runtime_paths = {}

    for key, arr in (
        arrays.items()
    ):

        path = (
            family_runtime_dir
            / f"{key}.npy"
        )

        np.save(
            path,
            arr,
            allow_pickle=False,
        )

        runtime_paths[
            key
        ] = str(
            path
        )


    # ---------------------------------------------------------------------------------------------
    # Durable Stage28 reconstruction receipt.
    # ---------------------------------------------------------------------------------------------

    receipt_out = {

        "stage":
            "Stage28-1A-R1",

        "type":
            "INHERITED_STAGE27_CHRONOLOGY_FOLD_RECONSTRUCTION_RECEIPT",

        "created_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "stage28_parent_commit":
            EXPECTED_PARENT,

        "held_out_family":
            family,

        "historical_receipt": {
            "path":
                str(
                    historical_path.relative_to(
                        REPO
                    )
                ),

            "sha256":
                sha256_file(
                    historical_path
                ),
        },

        "train_days":
            historical[
                "train_days"
            ],

        "validation_days":
            historical[
                "validation_days"
            ],

        "target_days":
            historical[
                "target_days"
            ],

        "membership_semantics":
            historical[
                "membership_semantics"
            ],

        "counts":
            observed_counts,

        "heldout_exclusion": {
            "train_count":
                heldout_train_count,

            "validation_count":
                heldout_validation_count,

            "status":
                "PASS",
        },

        "target_only_and_other_unseen_exclusion": {
            "train_count":
                excluded_train,

            "validation_count":
                excluded_validation,

            "status":
                "PASS",
        },

        "arrays":
            array_receipts,

        "class_weight": {
            "formula":
                "train_benign / train_attack",

            "recomputed_value":
                class_weight,

            "historical_value":
                historical_weight,

            "exact_reproduction":
                True,

            "validation_rows_used":
                0,

            "target_rows_used":
                0,
        },

        "runtime_cache": {
            "directory":
                str(
                    family_runtime_dir
                ),

            "paths":
                runtime_paths,

            "committed_to_git":
                False,
        },

        "scientific_operations": {
            "predictor_values_read":
                0,

            "model_fits":
                0,

            "model_inference":
                0,

            "threshold_selection":
                0,

            "target_predictor_openings":
                0,
        },

        "status":
            "PASS_EXACT_HISTORICAL_CONTENT_REPRODUCTION",
    }

    fold_receipts[
        family
    ] = receipt_out

    write_json(
        OUTPUT_DIR
        / (
            f"fold_{family}_"
            "reconstruction_receipt.json"
        ),
        receipt_out,
    )

    print()
    print(
        f"[PASS] {family}: exact historical "
        "membership reproduced"
    )


# =================================================================================================
# 12. GLOBAL RECONSTRUCTION RECEIPT
# =================================================================================================

banner(
    "WRITE GLOBAL RECONSTRUCTION RECEIPT"
)

global_receipt = {

    "stage":
        "Stage28-1A-R1",

    "type":
        "STAGE27_GLOBAL_MEMBERSHIP_CACHE_RECONSTRUCTION_RECEIPT",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage28_parent_commit":
        EXPECTED_PARENT,

    "recovered_source_map": {
        "runtime_path":
            str(
                SOURCE_MAP_PATH
            ),

        "sha256":
            sha256_file(
                SOURCE_MAP_PATH
            ),

        "committed_to_git":
            False,
    },

    "source_population": {
        "identity":
            source_receipt[
                "population"
            ][
                "frozen_effective_population_identity"
            ],

        "physical_rows":
            int(
                source_receipt[
                    "population"
                ][
                    "physical_rows"
                ]
            ),

        "effective_rows":
            total_effective_rows,

        "structural_null_excluded_rows":
            int(
                source_receipt[
                    "population"
                ][
                    "structural_null_excluded_rows"
                ]
            ),
    },

    "source_assets":
        source_execution_records,

    "membership_label_access": {
        "physical_label_rows_read":
            physical_label_rows_read,

        "effective_label_rows_used":
            effective_label_rows_used,

        "predictor_columns_read":
            0,

        "predictor_values_read":
            0,

        "purpose":
            "MEMBERSHIP_RECONSTRUCTION_ONLY",
    },

    "global_arrays": {
        name: {
            "count":
                int(
                    expected_arrays[
                        name
                    ][
                        "count"
                    ]
                ),

            "dtype":
                expected_arrays[
                    name
                ][
                    "dtype"
                ],

            "historical_content_sha256":
                expected_arrays[
                    name
                ][
                    "content_sha256"
                ],

            "reconstructed_content_sha256":
                actual_global_hashes[
                    name
                ],

            "exact_match":
                True,

            "runtime_path":
                global_runtime_paths[
                    name
                ],
        }
        for name
        in actual_global_hashes
    },

    "family_counts":
        observed_family_counts,

    "runtime_cache": {
        "directory":
            str(
                GLOBAL_CACHE_ROOT
            ),

        "committed_to_git":
            False,
    },

    "scientific_operations": {
        "model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "predictor_values_read":
            0,

        "target_predictor_openings":
            0,
    },

    "status":
        "PASS_EXACT_HISTORICAL_CONTENT_REPRODUCTION",
}

write_json(
    OUTPUT_DIR
    / "stage27_global_cache_reconstruction_receipt.json",
    global_receipt,
)


# =================================================================================================
# 13. STAGE22 AUDIT RECEIPT
# =================================================================================================

stage22_receipt = {

    "stage":
        "Stage28-1A-R1",

    "type":
        "STAGE22R_INHERITED_FULL_MEMBERSHIP_AUDIT",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "stage28_parent_commit":
        EXPECTED_PARENT,

    "membership_summary": {
        "path":
            str(
                stage22_summary_path.relative_to(
                    REPO
                )
            ),

        "sha256":
            sha256_file(
                stage22_summary_path
            ),
    },

    "random_validation_membership": {
        "path":
            str(
                random_validation_path.relative_to(
                    REPO
                )
            ),

        "expected_sha256":
            expected_random_sha,

        "actual_sha256":
            actual_random_sha,

        "logical_length":
            logical_length,

        "population":
            actual_population,

        "exact_match":
            True,
    },

    "RANDOM_NATURAL": {
        "train_rows":
            int(
                cells22[
                    "RANDOM_NATURAL"
                ][
                    "train"
                ][
                    "rows"
                ]
            ),

        "validation_rows":
            int(
                cells22[
                    "RANDOM_NATURAL"
                ][
                    "validation"
                ][
                    "rows"
                ]
            ),

        "membership_changed":
            False,
    },

    "CHRONOLOGICAL_NATURAL": {
        "train_rows":
            int(
                cells22[
                    "CHRONOLOGICAL_NATURAL"
                ][
                    "train"
                ][
                    "rows"
                ]
            ),

        "validation_rows":
            int(
                cells22[
                    "CHRONOLOGICAL_NATURAL"
                ][
                    "validation"
                ][
                    "rows"
                ]
            ),

        "membership_changed":
            False,
    },

    "scientific_operations": {
        "model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "holdout_predictor_openings":
            0,
    },

    "status":
        "PASS_EXACT_INHERITED_MEMBERSHIP",
}

write_json(
    OUTPUT_DIR
    / "stage22_full_membership_audit.json",
    stage22_receipt,
)


# =================================================================================================
# 14. STAGE28-1A FREEZE RECORD
# =================================================================================================

freeze_record = {

    "stage":
        "Stage28-1A-R1",

    "type":
        "INHERITED_MEMBERSHIP_RECONSTRUCTION_FREEZE",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_PARENT,

    "status":
        "INHERITED_MEMBERSHIPS_RECONSTRUCTED_AND_VERIFIED",

    "stage22": {
        "RANDOM_NATURAL":
            "PASS_EXACT",

        "CHRONOLOGICAL_NATURAL":
            "PASS_EXACT",

        "membership_change":
            False,
    },

    "stage27": {
        "source_assets":
            "8_OF_8_BYTE_EXACT",

        "global_family_codes":
            "PASS_EXACT_CONTENT_SHA256",

        "global_day_codes":
            "PASS_EXACT_CONTENT_SHA256",

        "global_source_codes":
            "PASS_EXACT_CONTENT_SHA256",

        "chronology_folds": {
            family:
                "PASS_EXACT_CONTENT_SHA256"
            for family
            in ELIGIBLE_FAMILIES
        },

        "eligible_family_count":
            5,
    },

    "fit_budget": {
        "new_fits_authorized":
            108,

        "new_fits_consumed":
            0,

        "new_fits_remaining":
            108,

        "existing_reused":
            12,

        "component_universe":
            120,

        "scientific_evaluation_cells":
            110,
    },

    "scientific_access": {
        "membership_label_rows_read_physical":
            physical_label_rows_read,

        "membership_label_rows_used_effective":
            effective_label_rows_used,

        "predictor_values_read":
            0,

        "model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "target_predictor_openings":
            0,
    },

    "runtime_cache_policy": {
        "global_cache":
            str(
                GLOBAL_CACHE_ROOT
            ),

        "chronology_memberships":
            str(
                CHRONO_CACHE_ROOT
            ),

        "committed_to_git":
            False,

        "recovery_rule":
            (
                "Recover the same 8 byte-exact frozen "
                "CICIDS2017 source assets and require "
                "historical global-array and fold-membership "
                "content SHA256 before use."
            ),
    },

    "next_authorized_step":
        (
            "Stage28-1B — materialize and freeze the five "
            "Stage28B random-LOAO memberships using fixed "
            "membership seed 42; audit zero held-out-family "
            "and target-only-unseen leakage; freeze the "
            "120-component execution manifest before fit #1."
        ),
}

write_json(
    OUTPUT_DIR
    / "stage28_1a_freeze_record.json",
    freeze_record,
)


# =================================================================================================
# 15. DURABLE CHECKSUM MANIFEST
# =================================================================================================

json_files = sorted(
    OUTPUT_DIR.glob(
        "*.json"
    )
)

checksum_lines = [
    f"{sha256_file(path)}  {path.name}"
    for path
    in json_files
]

checksums_path = (
    OUTPUT_DIR
    / "checksums.sha256"
)

checksums_path.write_text(
    "\n".join(
        checksum_lines
    )
    + "\n",
    encoding="utf-8",
)


# =================================================================================================
# 16. FINAL PRE-COMMIT MACHINE AUDIT
# =================================================================================================

banner(
    "STAGE28-1A-R1 FINAL MACHINE AUDIT"
)

assert (
    observed_family_counts[
        "BENIGN"
    ]
    == 2_273_097
)

assert (
    observed_family_counts[
        "TARGET_ONLY_UNSEEN"
    ]
    == 11
)

assert (
    observed_family_counts[
        "OTHER_ATTACK_UNSEEN_LABEL"
    ]
    == 0
)

assert (
    len(
        fold_receipts
    )
    == 5
)

for family in ELIGIBLE_FAMILIES:

    receipt = (
        fold_receipts[
            family
        ]
    )

    assert (
        receipt[
            "heldout_exclusion"
        ][
            "train_count"
        ]
        == 0
    )

    assert (
        receipt[
            "heldout_exclusion"
        ][
            "validation_count"
        ]
        == 0
    )

    assert (
        receipt[
            "target_only_and_other_unseen_exclusion"
        ][
            "train_count"
        ]
        == 0
    )

    assert (
        receipt[
            "target_only_and_other_unseen_exclusion"
        ][
            "validation_count"
        ]
        == 0
    )

    assert (
        receipt[
            "status"
        ]
        == "PASS_EXACT_HISTORICAL_CONTENT_REPRODUCTION"
    )

print(
    "[PASS] 8/8 frozen CICIDS2017 source assets"
)
print(
    "[PASS] 2,830,743 effective rows"
)
print(
    "[PASS] exact Stage27 family census"
)
print(
    "[PASS] global_family_codes historical SHA"
)
print(
    "[PASS] global_day_codes historical SHA"
)
print(
    "[PASS] global_source_codes historical SHA"
)
print(
    "[PASS] 5/5 chronology LOAO fold contents"
)
print(
    "[PASS] 5/5 class-weight reproductions"
)
print(
    "[PASS] held-out TRAIN count = 0"
)
print(
    "[PASS] held-out VALIDATION count = 0"
)
print(
    "[PASS] target-only/other unseen TRAIN = 0"
)
print(
    "[PASS] target-only/other unseen VALIDATION = 0"
)
print(
    "[PASS] Stage22 FULL memberships unchanged"
)
print(
    "[PASS] predictor values read = 0"
)
print(
    "[PASS] model fits consumed = 0"
)
print(
    "[PASS] model inference consumed = 0"
)
print(
    "[PASS] threshold selection consumed = 0"
)
print(
    "[PASS] target predictor openings = 0"
)


# =================================================================================================
# 17. SHOW DURABLE ARTIFACTS
# =================================================================================================

banner(
    "STAGE28-1A-R1 DURABLE ARTIFACTS"
)

for path in sorted(
    OUTPUT_DIR.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:62s} "
            f"{path.stat().st_size:10,d} bytes  "
            f"{sha256_file(path)}"
        )


# =================================================================================================
# 18. GIT DIFF AUDIT
# =================================================================================================

banner(
    "GIT DIFF AUDIT"
)

status = git(
    "status",
    "--porcelain",
)

print(
    status
)

expected_prefix = (
    "results/"
    "stage28_stability_novelty_control/"
    "stage28_1a_inherited_membership_reconstruction/"
)

unexpected = []

for line in (
    status.splitlines()
):

    if not line.strip():
        continue

    path = (
        line[3:]
        .strip()
    )

    if not path.startswith(
        expected_prefix
    ):
        unexpected.append(
            line
        )

if unexpected:
    raise RuntimeError(
        "Unexpected repository modifications:\n"
        + "\n".join(
            unexpected
        )
    )

print()
print(
    "[PASS] only Stage28-1A durable "
    "receipts are modified"
)


# =================================================================================================
# 19. STAGE EXACT FILE SET
# =================================================================================================

files_to_stage = sorted(
    path
    .relative_to(
        REPO
    )
    .as_posix()

    for path
    in OUTPUT_DIR.iterdir()

    if path.is_file()
)

for relative_path in (
    files_to_stage
):

    git(
        "add",
        "--",
        relative_path,
    )

staged = git(
    "diff",
    "--cached",
    "--name-only",
).splitlines()

if sorted(staged) != sorted(
    files_to_stage
):
    raise RuntimeError(
        "Staged Stage28-1A file set mismatch."
    )

git(
    "diff",
    "--cached",
    "--check",
)

print()
print(
    "[PASS] exact Stage28-1A file set staged"
)

for path in staged:
    print(
        " ",
        path,
    )


# =================================================================================================
# 20. COMMIT
# =================================================================================================

banner(
    "COMMIT STAGE28-1A"
)

git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
)

commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    f"{commit_sha}^",
)

if commit_parent != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage28-1A commit parent mismatch.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={commit_parent}"
    )

print(
    "Stage28-1A commit:",
    commit_sha,
)

print(
    "Parent            :",
    commit_parent,
)


# =================================================================================================
# 21. PUSH
# =================================================================================================

banner(
    "PUSH STAGE28-1A"
)

push = run(
    [
        "git",
        "push",
        "origin",
        "main:main",
    ],
    check=False,
)

if push.returncode != 0:
    raise RuntimeError(
        "Stage28-1A push failed.\n\n"
        f"STDOUT:\n{push.stdout}\n\n"
        f"STDERR:\n{push.stderr}"
    )

print(
    push.stdout or ""
)

print(
    push.stderr or ""
)


# =================================================================================================
# 22. REMOTE VERIFICATION
# =================================================================================================

banner(
    "REMOTE VERIFICATION"
)

remote_after = (
    remote_main_sha()
)

print(
    "Local HEAD :",
    commit_sha,
)

print(
    "Remote main:",
    remote_after,
)

if remote_after != commit_sha:
    raise RuntimeError(
        "Remote main does not equal "
        "Stage28-1A commit."
    )

git(
    "fetch",
    "origin",
    "main",
)

origin_after = git(
    "rev-parse",
    "origin/main",
)

if origin_after != commit_sha:
    raise RuntimeError(
        "origin/main mismatch after fetch."
    )

final_status = git(
    "status",
    "--porcelain",
)

if final_status:
    raise RuntimeError(
        "Repository not clean after Stage28-1A:\n"
        + final_status
    )


# =================================================================================================
# 23. FINAL
# =================================================================================================

banner(
    "STAGE28-1A — COMPLETE / REMOTELY VERIFIED"
)

print(
    "Stage28-1A commit:"
)
print(
    " ",
    commit_sha,
)

print()

print(
    "Inherited Stage22:"
)
print(
    "  RANDOM_NATURAL        = EXACT"
)
print(
    "  CHRONOLOGICAL_NATURAL = EXACT"
)

print()

print(
    "Inherited Stage27 chronology LOAO:"
)

for family in ELIGIBLE_FAMILIES:
    print(
        f"  {family:14s} = EXACT CONTENT REPRODUCTION"
    )

print()

print(
    "Global CICIDS2017 cache:"
)
print(
    "  effective rows       = 2,830,743"
)
print(
    "  global_family_codes  = SHA PASS"
)
print(
    "  global_day_codes     = SHA PASS"
)
print(
    "  global_source_codes  = SHA PASS"
)

print()

print(
    "Fit budget:"
)
print(
    "  NEW authorized = 108"
)
print(
    "  consumed       = 0"
)
print(
    "  remaining      = 108"
)
print(
    "  existing reused= 12"
)
print(
    "  component total= 120"
)

print()

print(
    "Scientific operations:"
)
print(
    "  PREDICTOR_VALUES_READ     = 0"
)
print(
    "  MODEL_FITS                = 0"
)
print(
    "  MODEL_INFERENCE           = 0"
)
print(
    "  THRESHOLD_SELECTION       = 0"
)
print(
    "  TARGET_PREDICTOR_OPENINGS = 0"
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage28-1B — create the five frozen "
    "random-LOAO memberships using membership seed 42,"
)

print(
    "  audit zero held-out/unseen leakage, "
    "and freeze the 120-component execution manifest "
    "BEFORE FIT #1."
)

print()
print(SEP)


STAGE28-1A-R1 — EXACT REPOSITORY GATE

Repository : /kaggle/working/ids2018-validation-safe-ablation
Branch     : main
Local HEAD : 0029c28c417d63ffb45ad4096ab7c44cbd875116
Remote main: 0029c28c417d63ffb45ad4096ab7c44cbd875116
Expected   : 0029c28c417d63ffb45ad4096ab7c44cbd875116
Git clean  : True

[PASS] exact Stage28-0A parent
[PASS] local == remote
[PASS] worktree clean
[PASS] Stage28-1A durable destination absent

LOAD FROZEN STAGE28 / STAGE27 PROVENANCE

[PASS] effective new-fit budget = 108
[PASS] existing reused fits = 12
[PASS] component universe = 120
[PASS] Stage27 effective population = 2,830,743

STAGE22R FROZEN FULL-MEMBERSHIP AUDIT

random_validation.packbits
  expected SHA: 8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad
  actual SHA  : 8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad

[PASS] Stage22 RANDOM_NATURAL membership exact
[PASS] Stage22 CHRONOLOGICAL_NATURAL counts exact

VERIFY RECOVERED CICIDS2017 RUNTIME ASSETS

[0] Monday-W

In [1]:
# =================================================================================================
# STAGE28 — FRESH KAGGLE RECOVERY AFTER SESSION RESET
#
# Recover exact durable scientific parent:
#   e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
#
# This cell does ONLY:
#   - GitHub credential recovery
#   - clone/reset repository
#   - exact local/remote HEAD verification
#   - clean-worktree verification
#
# ZERO scientific operations.
# =================================================================================================

from __future__ import annotations

import os
import stat
import subprocess
from pathlib import Path


SEP = "=" * 118

REPO_URL = (
    "https://github.com/"
    "themubasshir/"
    "ids2018-validation-safe-ablation.git"
)

REPO = Path(
    "/kaggle/working/"
    "ids2018-validation-safe-ablation"
)

EXPECTED_BRANCH = "main"

EXPECTED_HEAD = (
    "e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b"
)

SECRET_ALIASES = [
    "GITHUB_TOKEN",
    "github_token",
    "GH_TOKEN",
    "GITHUB_PAT",
    "github_pat",
    "GH_PAT",
]


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, cwd=None, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args, cwd=REPO, check=True):
    return (
        run(
            ["git", *args],
            cwd=cwd,
            check=check,
        ).stdout
        or ""
    ).strip()


# =================================================================================================
# 1. RECOVER GITHUB TOKEN
# =================================================================================================

banner("GITHUB SECRET RECOVERY")

github_token = None
token_source = None


# First try environment.
for name in SECRET_ALIASES:

    value = os.environ.get(name)

    if value and value.strip():

        github_token = value.strip()
        token_source = f"environment:{name}"
        break


# Then Kaggle Secrets.
if github_token is None:

    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()

        for name in SECRET_ALIASES:

            try:
                value = client.get_secret(name)

            except Exception:
                value = None

            if value and value.strip():

                github_token = value.strip()
                token_source = f"kaggle_secret:{name}"
                break

    except Exception as exc:
        print(
            "[INFO] Kaggle Secrets client unavailable:",
            repr(exc),
        )


if github_token is None:

    raise RuntimeError(
        "\nGitHub token not found.\n\n"
        "Expected one of these Kaggle Secret labels:\n"
        + "\n".join(
            f"  - {x}"
            for x in SECRET_ALIASES
        )
    )


os.environ["GITHUB_TOKEN"] = github_token

print(
    "[PASS] GitHub credential recovered from",
    token_source,
)

print(
    "[PASS] token value intentionally not displayed"
)


# =================================================================================================
# 2. CLONE / RESET
# =================================================================================================

banner("REPOSITORY RECOVERY")

if REPO.exists():

    if not (
        REPO / ".git"
    ).is_dir():

        raise RuntimeError(
            f"{REPO} exists but is not a Git repository."
        )

    print(
        "Existing repository found — resetting to origin/main."
    )

    run(
        [
            "git",
            "fetch",
            "--prune",
            "origin",
        ],
        cwd=REPO,
    )

    run(
        [
            "git",
            "reset",
            "--hard",
            "origin/main",
        ],
        cwd=REPO,
    )

    run(
        [
            "git",
            "clean",
            "-fd",
        ],
        cwd=REPO,
    )

else:

    print(
        "Fresh Kaggle runtime — cloning repository."
    )

    run(
        [
            "git",
            "clone",
            "--branch",
            EXPECTED_BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO),
        ]
    )


# =================================================================================================
# 3. TOKEN-FREE REMOTE + CREDENTIAL HELPER
# =================================================================================================

banner("CONFIGURE SAFE GIT AUTHENTICATION")

helper = Path(
    "/kaggle/working/"
    "stage28_github_credential_helper.sh"
)

helper.write_text(
    """#!/bin/sh
case "$1" in
    get)
        echo "username=x-access-token"
        echo "password=$GITHUB_TOKEN"
        ;;
esac
""",
    encoding="utf-8",
)

helper.chmod(
    helper.stat().st_mode
    | stat.S_IXUSR
)

git(
    "remote",
    "set-url",
    "origin",
    REPO_URL,
)

git(
    "config",
    "--local",
    "credential.helper",
    str(helper),
)

git(
    "config",
    "--local",
    "user.name",
    "themubasshir",
)

git(
    "config",
    "--local",
    "user.email",
    "themubasshir@users.noreply.github.com",
)

print(
    "Remote:",
    git(
        "remote",
        "get-url",
        "origin",
    ),
)

print(
    "[PASS] remote URL contains no token"
)


# =================================================================================================
# 4. EXACT SCIENTIFIC-PARENT VERIFICATION
# =================================================================================================

banner("STAGE28 SCIENTIFIC-PARENT GATE")

git(
    "fetch",
    "origin",
    "main",
)

branch = git(
    "branch",
    "--show-current",
)

local_head = git(
    "rev-parse",
    "HEAD",
)

origin_main = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote_main = (
    remote_line.split()[0]
)

status = git(
    "status",
    "--porcelain",
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)

print(
    "Expected HEAD:",
    EXPECTED_HEAD,
)

print(
    "Local HEAD   :",
    local_head,
)

print(
    "origin/main  :",
    origin_main,
)

print(
    "Remote main  :",
    remote_main,
)

print(
    "Branch       :",
    branch,
)

print(
    "Subject      :",
    subject,
)

print(
    "Git clean    :",
    not bool(status),
)


if branch != EXPECTED_BRANCH:

    raise RuntimeError(
        f"Expected branch {EXPECTED_BRANCH}; "
        f"got {branch}"
    )


if local_head != EXPECTED_HEAD:

    raise RuntimeError(
        "Local HEAD mismatch.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={local_head}"
    )


if origin_main != EXPECTED_HEAD:

    raise RuntimeError(
        "origin/main mismatch.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={origin_main}"
    )


if remote_main != EXPECTED_HEAD:

    raise RuntimeError(
        "Remote main mismatch.\n"
        f"expected={EXPECTED_HEAD}\n"
        f"actual={remote_main}"
    )


if status:

    raise RuntimeError(
        "Recovered repository is not clean:\n"
        + status
    )


print()
print(
    "[PASS] exact Stage28-1A durable state recovered"
)


# =================================================================================================
# 5. VERIFY CRITICAL STAGE28 FILES
# =================================================================================================

banner("CRITICAL DURABLE ARTIFACT GATE")

required = [

    # Stage28-0
    (
        REPO
        / "results"
        / "stage28_stability_novelty_control"
        / "stage28_0_protocol_lock"
        / "freeze_record.json"
    ),

    # Stage28-0A
    (
        REPO
        / "results"
        / "stage28_stability_novelty_control"
        / "stage28_0a_preexecution_amendment"
        / "effective_fit_budget.json"
    ),

    # Stage28-1A
    (
        REPO
        / "results"
        / "stage28_stability_novelty_control"
        / "stage28_1a_inherited_membership_reconstruction"
        / "stage28_1a_freeze_record.json"
    ),

    (
        REPO
        / "results"
        / "stage28_stability_novelty_control"
        / "stage28_1a_inherited_membership_reconstruction"
        / "stage27_global_cache_reconstruction_receipt.json"
    ),
]


for path in required:

    if not path.is_file():

        raise RuntimeError(
            f"Missing required Stage28 artifact:\n{path}"
        )

    print(
        "[PASS]",
        path.relative_to(REPO),
    )


# =================================================================================================
# 6. READ BACK FROZEN FIT LEDGER
# =================================================================================================

banner("FROZEN FIT LEDGER")

budget_path = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_0a_preexecution_amendment"
    / "effective_fit_budget.json"
)

freeze_1a_path = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_1a_inherited_membership_reconstruction"
    / "stage28_1a_freeze_record.json"
)

import json

budget = json.loads(
    budget_path.read_text(
        encoding="utf-8"
    )
)

freeze_1a = json.loads(
    freeze_1a_path.read_text(
        encoding="utf-8"
    )
)


assert (
    budget[
        "totals"
    ][
        "new_fit_budget"
    ]
    == 108
)

assert (
    budget[
        "totals"
    ][
        "existing_reused"
    ]
    == 12
)

assert (
    budget[
        "totals"
    ][
        "total_component_realizations"
    ]
    == 120
)

assert (
    freeze_1a[
        "fit_budget"
    ][
        "new_fits_consumed"
    ]
    == 0
)


print(
    "NEW fits authorized:",
    108,
)

print(
    "NEW fits consumed  :",
    0,
)

print(
    "NEW fits remaining :",
    108,
)

print(
    "Existing reused    :",
    12,
)

print(
    "Component universe :",
    120,
)


# =================================================================================================
# 7. FINAL
# =================================================================================================

banner(
    "FRESH STAGE28 RECOVERY COMPLETE"
)

print(
    "Repository:"
)
print(
    " ",
    REPO,
)

print()

print(
    "HEAD:"
)
print(
    " ",
    local_head,
)

print()

print(
    "Durable state:"
)

print(
    "  Stage28-0  protocol lock : REMOTELY FROZEN"
)

print(
    "  Stage28-0A amendment     : REMOTELY FROZEN"
)

print(
    "  Stage28-1A reconstruction: REMOTELY FROZEN"
)

print()

print(
    "Runtime state lost by Kaggle reset:"
)

print(
    "  CICIDS2017 parquet cache : NEEDS RECOVERY"
)

print(
    "  Stage27 global .npy cache: NEEDS RECONSTRUCTION"
)

print(
    "  Stage28B memberships     : NOT YET CREATED"
)

print()

print(
    "Scientific operations:"
)

print(
    "  MODEL_FITS                = 0"
)

print(
    "  MODEL_INFERENCE           = 0"
)

print(
    "  THRESHOLD_SELECTION       = 0"
)

print(
    "  TARGET_PREDICTOR_OPENINGS = 0"
)

print()

print(
    "NEXT RECOVERY STEP:"
)

print(
    "  Recover the exact 8 frozen CICIDS2017 "
    "parquet assets, then rebuild only the "
    "Stage28-1A runtime global cache."
)

print()
print(SEP)


GITHUB SECRET RECOVERY

[PASS] GitHub credential recovered from kaggle_secret:GITHUB_TOKEN
[PASS] token value intentionally not displayed

REPOSITORY RECOVERY

Fresh Kaggle runtime — cloning repository.

CONFIGURE SAFE GIT AUTHENTICATION

Remote: https://github.com/themubasshir/ids2018-validation-safe-ablation.git
[PASS] remote URL contains no token

STAGE28 SCIENTIFIC-PARENT GATE

Expected HEAD: e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
Local HEAD   : e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
origin/main  : e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
Remote main  : e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
Branch       : main
Subject      : stage28-1a: reconstruct and verify inherited memberships
Git clean    : True

[PASS] exact Stage28-1A durable state recovered

CRITICAL DURABLE ARTIFACT GATE

[PASS] results/stage28_stability_novelty_control/stage28_0_protocol_lock/freeze_record.json
[PASS] results/stage28_stability_novelty_control/stage28_0a_preexecution_amendment/effective_fit_budget.

In [2]:
# =================================================================================================
# STAGE28-RUNTIME-R1 — RESTORE STAGE28-1A GLOBAL RUNTIME STATE AFTER KAGGLE RESET
#
# Durable scientific parent:
#   e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
#
# This is RUNTIME RECOVERY ONLY.
# It does NOT create or amend a scientific stage.
#
# Operations:
#   1. recover the 8 exact Stage27 CICIDS2017 parquet assets
#   2. verify frozen SIZE + SHA256
#   3. read LABEL column only
#   4. reconstruct:
#        global_family_codes.npy
#        global_day_codes.npy
#        global_source_codes.npy
#   5. require exact Stage28-1A / historical content SHA256
#
# ZERO:
#   predictor values
#   model fits
#   model inference
#   threshold selection
#   target predictor openings
#   Git modifications
# =================================================================================================

from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# =================================================================================================
# CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_HEAD = (
    "e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b"
)

HF_REPO_ID = "bvsam/cic-ids-2017"

HF_REVISION = (
    "b7e532345512edcd530cb1770dc76636aeb52802"
)

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

STAGE28_1A_DIR = (
    STAGE28_ROOT
    / "stage28_1a_inherited_membership_reconstruction"
)

STAGE27_MEMBERSHIP_DIR = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
)

SOURCE_RECEIPT_PATH = (
    STAGE27_MEMBERSHIP_DIR
    / "source_effective_population_receipt.json"
)

FAMILY_CACHE_RECEIPT_PATH = (
    STAGE27_MEMBERSHIP_DIR
    / "family_code_cache_receipt.json"
)

CENSUS_PATH = (
    STAGE27_MEMBERSHIP_DIR
    / "canonical_label_census.json"
)

STAGE28_1A_GLOBAL_RECEIPT_PATH = (
    STAGE28_1A_DIR
    / "stage27_global_cache_reconstruction_receipt.json"
)

SOURCE_ROOT = Path(
    "/kaggle/working/stage27_cicids2017_sources"
)

GLOBAL_CACHE_ROOT = Path(
    "/kaggle/working/stage28_1a_runtime_cache/"
    "stage27_global_cache"
)

RUNTIME_SOURCE_MAP = Path(
    "/kaggle/working/stage28_1a_exact_source_map.json"
)

RUNTIME_RECOVERY_RECEIPT = Path(
    "/kaggle/working/stage28_runtime_r1_receipt.json"
)


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return (
        run(
            ["git", *args],
            cwd=REPO,
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array_content(arr):
    arr = np.ascontiguousarray(
        arr
    )

    h = hashlib.sha256()

    h.update(
        arr.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


# =================================================================================================
# 1. EXACT DURABLE-PARENT GATE
# =================================================================================================

banner(
    "STAGE28-RUNTIME-R1 — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

status = git(
    "status",
    "--porcelain",
)

branch = git(
    "branch",
    "--show-current",
)

print(
    "Local HEAD :",
    head,
)

print(
    "origin/main:",
    origin,
)

print(
    "Expected   :",
    EXPECTED_HEAD,
)

print(
    "Branch     :",
    branch,
)

print(
    "Git clean  :",
    not bool(
        status
    ),
)

if head != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected local HEAD."
    )

if origin != EXPECTED_HEAD:
    raise RuntimeError(
        "Unexpected origin/main."
    )

if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )

if status:
    raise RuntimeError(
        "Git worktree must be clean:\n"
        + status
    )

print()
print(
    "[PASS] exact remotely frozen Stage28-1A state"
)


# =================================================================================================
# 2. LOAD DURABLE FROZEN RECEIPTS
# =================================================================================================

banner(
    "LOAD DURABLE FROZEN RECEIPTS"
)

source_receipt = read_json(
    require_file(
        SOURCE_RECEIPT_PATH
    )
)

family_cache_receipt = read_json(
    require_file(
        FAMILY_CACHE_RECEIPT_PATH
    )
)

census = read_json(
    require_file(
        CENSUS_PATH
    )
)

stage28_1a_global_receipt = read_json(
    require_file(
        STAGE28_1A_GLOBAL_RECEIPT_PATH
    )
)

segments = (
    source_receipt[
        "segments"
    ]
)

if len(segments) != 8:
    raise RuntimeError(
        "Frozen Stage27 source count != 8."
    )

if (
    source_receipt[
        "population"
    ][
        "effective_rows"
    ]
    != 2_830_743
):
    raise RuntimeError(
        "Unexpected frozen effective population."
    )

print(
    "[PASS] Stage27 source receipt"
)

print(
    "[PASS] Stage27 family-cache receipt"
)

print(
    "[PASS] Stage27 canonical census"
)

print(
    "[PASS] Stage28-1A global reconstruction receipt"
)


# =================================================================================================
# 3. HUGGING FACE CLIENT
# =================================================================================================

banner(
    "HUGGING FACE CLIENT"
)

try:

    from huggingface_hub import (
        hf_hub_download,
    )

    import huggingface_hub

except ImportError:

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "huggingface_hub",
        ],
        check=True,
    )

    from huggingface_hub import (
        hf_hub_download,
    )

    import huggingface_hub


print(
    "huggingface_hub:",
    huggingface_hub.__version__,
)

print(
    "dataset:",
    HF_REPO_ID,
)

print(
    "revision:",
    HF_REVISION,
)


# =================================================================================================
# 4. RECOVER ALL 8 BYTE-EXACT ASSETS
# =================================================================================================

banner(
    "RECOVER 8 BYTE-EXACT CICIDS2017 ASSETS"
)

SOURCE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

resolved = []


for seg in segments:

    source_index = int(
        seg[
            "source_index"
        ]
    )

    basename = (
        seg[
            "basename"
        ]
    )

    expected_size = int(
        seg[
            "size_bytes"
        ]
    )

    expected_sha = (
        seg[
            "sha256"
        ]
    )

    remote_filename = (
        f"traffic_labels/{basename}"
    )

    destination = (
        SOURCE_ROOT
        / basename
    )

    print()
    print("-" * 120)

    print(
        f"[{source_index}] {basename}"
    )

    print(
        "Expected bytes :",
        f"{expected_size:,}",
    )

    print(
        "Expected SHA256:",
        expected_sha,
    )


    # ----------------------------------------------------------------------------------------------
    # Reuse only if byte-exact.
    # ----------------------------------------------------------------------------------------------

    if destination.is_file():

        current_size = (
            destination
            .stat()
            .st_size
        )

        current_sha = sha256_file(
            destination
        )

        if (
            current_size
            == expected_size
            and
            current_sha
            == expected_sha
        ):

            print(
                "[REUSE EXACT]",
                destination,
            )

            resolved.append({
                "source_index":
                    source_index,

                "day":
                    seg["day"],

                "basename":
                    basename,

                "runtime_path":
                    str(
                        destination
                    ),

                "size_bytes":
                    current_size,

                "sha256":
                    current_sha,

                "resolution":
                    "EXISTING_BYTE_EXACT",
            })

            continue

        print(
            "[REMOVE] stale/non-exact runtime file"
        )

        destination.unlink()


    # ----------------------------------------------------------------------------------------------
    # Pinned Hugging Face revision.
    # ----------------------------------------------------------------------------------------------

    cached = Path(
        hf_hub_download(
            repo_id=
                HF_REPO_ID,

            repo_type=
                "dataset",

            filename=
                remote_filename,

            revision=
                HF_REVISION,
        )
    )

    actual_size = (
        cached
        .stat()
        .st_size
    )

    actual_sha = sha256_file(
        cached
    )

    print(
        "Downloaded:",
        cached,
    )

    print(
        "Actual bytes :",
        f"{actual_size:,}",
    )

    print(
        "Actual SHA256:",
        actual_sha,
    )


    if actual_size != expected_size:
        raise RuntimeError(
            f"{basename}: frozen size mismatch."
        )

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"{basename}: frozen SHA256 mismatch."
        )

    shutil.copyfile(
        cached,
        destination,
    )

    copied_size = (
        destination
        .stat()
        .st_size
    )

    copied_sha = sha256_file(
        destination
    )

    if copied_size != expected_size:
        raise RuntimeError(
            f"{basename}: copied size mismatch."
        )

    if copied_sha != expected_sha:
        raise RuntimeError(
            f"{basename}: copied SHA mismatch."
        )

    print(
        "[PASS] exact runtime copy"
    )

    resolved.append({
        "source_index":
            source_index,

        "day":
            seg["day"],

        "basename":
            basename,

        "hf_repo_id":
            HF_REPO_ID,

        "hf_revision":
            HF_REVISION,

        "runtime_path":
            str(
                destination
            ),

        "size_bytes":
            copied_size,

        "sha256":
            copied_sha,

        "resolution":
            "HF_PINNED_REVISION_BYTE_EXACT",
    })


if len(resolved) != 8:
    raise RuntimeError(
        "Did not recover exactly 8 source assets."
    )


expected_total_bytes = sum(
    int(
        x[
            "size_bytes"
        ]
    )
    for x in segments
)

actual_total_bytes = sum(
    Path(
        x[
            "runtime_path"
        ]
    )
    .stat()
    .st_size

    for x in resolved
)

if (
    actual_total_bytes
    != expected_total_bytes
):
    raise RuntimeError(
        "Recovered total-byte mismatch."
    )

print()
print(
    "[PASS] 8 / 8 exact source assets"
)

print(
    "Total bytes:",
    f"{actual_total_bytes:,}",
)


# =================================================================================================
# 5. WRITE TEMPORARY SOURCE MAP
# =================================================================================================

RUNTIME_SOURCE_MAP.write_text(
    json.dumps(
        {
            "stage":
                "Stage28-RUNTIME-R1",

            "scientific_stage":
                False,

            "durable_parent":
                EXPECTED_HEAD,

            "hf_repo_id":
                HF_REPO_ID,

            "hf_revision":
                HF_REVISION,

            "resolved":
                resolved,

            "byte_identity":
                "8_OF_8_PASS_EXACT_SHA256_AND_SIZE",

            "committed_to_git":
                False,
        },
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)


# =================================================================================================
# 6. FROZEN CODEBOOK / CANONICALIZATION
# =================================================================================================

banner(
    "RECONSTRUCT GLOBAL MEMBERSHIP ARRAYS — LABELS ONLY"
)

codebook = (
    family_cache_receipt[
        "family_codebook"
    ]
)

EXPECTED_CODEBOOK = {
    "BENIGN": 0,
    "BOT": 1,
    "DDOS": 2,
    "DOS": 3,
    "AUTH_BRUTE_FORCE": 4,
    "INFILTRATION": 5,
    "PORT_SCAN": 6,
    "WEB_ATTACK": 7,
    "TARGET_ONLY_UNSEEN": 8,
    "OTHER_ATTACK_UNSEEN_LABEL": 9,
}

if codebook != EXPECTED_CODEBOOK:
    raise RuntimeError(
        "Frozen family codebook mismatch."
    )


LABEL_TO_CODE = {

    "benign": 0,

    "bot": 1,

    "ddos": 2,

    "dos goldeneye": 3,
    "dos hulk": 3,
    "dos slowhttptest": 3,
    "dos slowloris": 3,

    "ftp-patator": 4,
    "ssh-patator": 4,

    "infiltration": 5,

    "portscan": 6,

    "web attack - brute force": 7,
    "web attack - sql injection": 7,
    "web attack - xss": 7,

    "heartbleed": 8,
}


UNICODE_DASHES = [
    "\u2010",
    "\u2011",
    "\u2012",
    "\u2013",
    "\u2014",
    "\u2015",
    "\u2212",
    "\ufe58",
    "\ufe63",
    "\uff0d",
]


WEB_ATTACK_ALIASES = {

    "web attack \u0096 brute force":
        "web attack - brute force",

    "web attack \u0096 sql injection":
        "web attack - sql injection",

    "web attack \u0096 xss":
        "web attack - xss",
}


def canonicalize_labels(series):

    s = series.astype(
        "string"
    )

    s = s.str.strip()

    for dash in UNICODE_DASHES:

        s = s.str.replace(
            dash,
            "-",
            regex=False,
        )

    s = s.str.replace(
        r"\s+",
        " ",
        regex=True,
    )

    s = s.str.casefold()

    s = s.replace(
        WEB_ATTACK_ALIASES
    )

    return s


# =================================================================================================
# 7. BUILD GLOBAL ARRAYS
# =================================================================================================

TOTAL_ROWS = int(
    source_receipt[
        "population"
    ][
        "effective_rows"
    ]
)

global_family_codes = np.empty(
    TOTAL_ROWS,
    dtype=np.uint8,
)

global_day_codes = np.empty(
    TOTAL_ROWS,
    dtype=np.uint8,
)

global_source_codes = np.empty(
    TOTAL_ROWS,
    dtype=np.uint8,
)

physical_label_rows_read = 0
effective_label_rows_used = 0


for seg in segments:

    source_index = int(
        seg[
            "source_index"
        ]
    )

    basename = (
        seg[
            "basename"
        ]
    )

    path = (
        SOURCE_ROOT
        / basename
    )

    physical_expected = int(
        seg[
            "physical_rows"
        ]
    )

    effective_rows = int(
        seg[
            "effective_rows"
        ]
    )

    global_start = int(
        seg[
            "global_start_zero_based"
        ]
    )

    global_stop = int(
        seg[
            "global_stop_exclusive"
        ]
    )

    day_code = int(
        seg[
            "day_code"
        ]
    )

    print()
    print(
        f"[{source_index}] "
        f"{seg['day']} — {basename}"
    )

    pf = pq.ParquetFile(
        path
    )

    physical_actual = int(
        pf.metadata.num_rows
    )

    if (
        physical_actual
        != physical_expected
    ):
        raise RuntimeError(
            f"{basename}: physical row mismatch."
        )

    label_candidates = [
        col
        for col
        in pf.schema_arrow.names
        if (
            col.strip().casefold()
            == "label"
        )
    ]

    if len(
        label_candidates
    ) != 1:
        raise RuntimeError(
            f"{basename}: could not uniquely "
            "resolve Label column."
        )

    label_col = (
        label_candidates[0]
    )

    # LABEL ONLY. No predictors.
    table = pq.read_table(
        path,
        columns=[
            label_col,
        ],
    )

    labels = (
        table
        .column(0)
        .to_pandas()
    )

    if len(
        labels
    ) != physical_actual:
        raise RuntimeError(
            f"{basename}: label row count mismatch."
        )

    physical_label_rows_read += int(
        len(
            labels
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Frozen Stage27 effective-row rule.
    # ----------------------------------------------------------------------------------------------

    if effective_rows < physical_actual:

        rule = (
            seg[
                "inclusion_rule"
            ]
        )

        if not rule.startswith(
            "INCLUDE_PHYSICAL_ORDINALS_1_THROUGH_"
        ):
            raise RuntimeError(
                f"{basename}: unexpected "
                "effective-row rule."
            )

        labels = (
            labels
            .iloc[
                :effective_rows
            ]
            .copy()
        )

    elif (
        effective_rows
        != physical_actual
    ):
        raise RuntimeError(
            f"{basename}: unsupported row geometry."
        )


    effective_label_rows_used += int(
        len(
            labels
        )
    )

    canonical = canonicalize_labels(
        labels
    )

    mapped = canonical.map(
        LABEL_TO_CODE
    )

    if mapped.isna().any():

        unknown = (
            canonical[
                mapped.isna()
            ]
            .value_counts(
                dropna=False
            )
        )

        print()
        print(
            "UNKNOWN LABELS:"
        )

        print(
            unknown.to_string()
        )

        raise RuntimeError(
            f"{basename}: frozen taxonomy mismatch."
        )


    codes = mapped.to_numpy(
        dtype=np.uint8,
        copy=True,
    )

    if (
        len(
            codes
        )
        != effective_rows
    ):
        raise RuntimeError(
            f"{basename}: effective rows mismatch."
        )

    if (
        global_stop
        - global_start
        != effective_rows
    ):
        raise RuntimeError(
            f"{basename}: global geometry mismatch."
        )


    global_family_codes[
        global_start:global_stop
    ] = codes

    global_day_codes[
        global_start:global_stop
    ] = day_code

    global_source_codes[
        global_start:global_stop
    ] = source_index


    local_counts = np.bincount(
        codes,
        minlength=10,
    )

    print(
        "  physical :",
        f"{physical_actual:,}",
    )

    print(
        "  effective:",
        f"{effective_rows:,}",
    )

    print(
        "  families :",
        {
            name: int(
                local_counts[
                    code
                ]
            )
            for name, code
            in codebook.items()
            if local_counts[
                code
            ] > 0
        },
    )

    del table
    del labels
    del canonical
    del mapped
    del codes


if (
    effective_label_rows_used
    != TOTAL_ROWS
):
    raise RuntimeError(
        "Total effective label rows mismatch."
    )


# =================================================================================================
# 8. CENSUS GATE
# =================================================================================================

banner(
    "FROZEN CENSUS GATE"
)

observed_arr = np.bincount(
    global_family_codes,
    minlength=10,
)

observed_counts = {
    family: int(
        observed_arr[
            code
        ]
    )
    for family, code
    in codebook.items()
}

expected_counts = (
    census[
        "family_counts"
    ]
)

print(
    "Observed:"
)

print(
    json.dumps(
        observed_counts,
        indent=2,
    )
)

if (
    observed_counts
    != expected_counts
):
    raise RuntimeError(
        "Stage27 family census mismatch."
    )

print()
print(
    "[PASS] frozen family census reproduced exactly"
)


# =================================================================================================
# 9. EXACT CONTENT-SHA GATE
# =================================================================================================

banner(
    "EXACT HISTORICAL CONTENT-SHA256 GATE"
)

historical_arrays = (
    family_cache_receipt[
        "arrays"
    ]
)

stage28_arrays = (
    stage28_1a_global_receipt[
        "global_arrays"
    ]
)

arrays = {

    "global_family_codes":
        global_family_codes,

    "global_day_codes":
        global_day_codes,

    "global_source_codes":
        global_source_codes,
}

array_hashes = {}


for name, arr in (
    arrays.items()
):

    actual_sha = (
        sha256_array_content(
            arr
        )
    )

    historical_sha = (
        historical_arrays[
            name
        ][
            "content_sha256"
        ]
    )

    stage28_1a_historical = (
        stage28_arrays[
            name
        ][
            "historical_content_sha256"
        ]
    )

    stage28_1a_reconstructed = (
        stage28_arrays[
            name
        ][
            "reconstructed_content_sha256"
        ]
    )

    expected_count = int(
        historical_arrays[
            name
        ][
            "count"
        ]
    )

    print(
        name
    )

    print(
        "  rows                 :",
        f"{len(arr):,}",
    )

    print(
        "  historical           :",
        historical_sha,
    )

    print(
        "  Stage28-1A historical:",
        stage28_1a_historical,
    )

    print(
        "  Stage28-1A rebuilt   :",
        stage28_1a_reconstructed,
    )

    print(
        "  current recovery     :",
        actual_sha,
    )


    if len(
        arr
    ) != expected_count:
        raise RuntimeError(
            f"{name}: row-count mismatch."
        )


    if not (
        actual_sha
        == historical_sha
        == stage28_1a_historical
        == stage28_1a_reconstructed
    ):
        raise RuntimeError(
            f"{name}: runtime recovery "
            "content SHA mismatch."
        )


    array_hashes[
        name
    ] = actual_sha


print()
print(
    "[PASS] all 3 global arrays reproduce "
    "the frozen historical byte content"
)


# =================================================================================================
# 10. MATERIALIZE GLOBAL .NPY CACHE
# =================================================================================================

banner(
    "MATERIALIZE STAGE28-1A GLOBAL RUNTIME CACHE"
)

GLOBAL_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

for name, arr in (
    arrays.items()
):

    path = (
        GLOBAL_CACHE_ROOT
        / f"{name}.npy"
    )

    np.save(
        path,
        arr,
        allow_pickle=False,
    )

    reloaded = np.load(
        path,
        mmap_mode="r",
        allow_pickle=False,
    )

    reload_sha = (
        sha256_array_content(
            np.asarray(
                reloaded
            )
        )
    )

    if (
        reload_sha
        != array_hashes[
            name
        ]
    ):
        raise RuntimeError(
            f"{name}: saved runtime .npy "
            "content verification failed."
        )

    print(
        "[PASS]",
        name,
    )

    print(
        "       path:",
        path,
    )

    print(
        "       content SHA:",
        reload_sha,
    )


# =================================================================================================
# 11. RUNTIME RECOVERY RECEIPT — OUTSIDE GIT
# =================================================================================================

runtime_receipt = {

    "stage":
        "Stage28-RUNTIME-R1",

    "scientific_stage":
        False,

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "durable_scientific_parent":
        EXPECTED_HEAD,

    "purpose":
        "RECOVER_DISPOSABLE_STAGE28_1A_RUNTIME_STATE_AFTER_KAGGLE_SESSION_RESET",

    "source_assets": {
        "count":
            8,

        "hf_repo_id":
            HF_REPO_ID,

        "hf_revision":
            HF_REVISION,

        "total_bytes":
            actual_total_bytes,

        "status":
            "8_OF_8_EXACT_SIZE_AND_SHA256",
    },

    "global_arrays": {
        name: {
            "content_sha256":
                array_hashes[
                    name
                ],

            "historical_exact_match":
                True,

            "runtime_path":
                str(
                    GLOBAL_CACHE_ROOT
                    / f"{name}.npy"
                ),
        }
        for name
        in arrays
    },

    "data_access": {
        "physical_label_rows_read":
            physical_label_rows_read,

        "effective_label_rows_used":
            effective_label_rows_used,

        "predictor_values_read":
            0,

        "purpose":
            "DURABLE_MEMBERSHIP_CACHE_RECOVERY_ONLY",
    },

    "scientific_operations": {
        "model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "target_predictor_openings":
            0,
    },

    "git_modifications":
        0,

    "status":
        "RUNTIME_STATE_RECOVERED_EXACTLY",
}

RUNTIME_RECOVERY_RECEIPT.write_text(
    json.dumps(
        runtime_receipt,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)


# =================================================================================================
# 12. FINAL GIT CLEANNESS GATE
# =================================================================================================

banner(
    "FINAL GIT CLEANLINESS GATE"
)

status_after = git(
    "status",
    "--porcelain",
)

if status_after:
    raise RuntimeError(
        "Runtime recovery unexpectedly modified Git:\n"
        + status_after
    )

head_after = git(
    "rev-parse",
    "HEAD",
)

if (
    head_after
    != EXPECTED_HEAD
):
    raise RuntimeError(
        "HEAD changed during runtime recovery."
    )

print(
    "[PASS] Git remains completely clean"
)

print(
    "[PASS] durable HEAD unchanged:",
    head_after,
)


# =================================================================================================
# 13. FINAL
# =================================================================================================

banner(
    "STAGE28-RUNTIME-R1 — RECOVERY COMPLETE"
)

print(
    "Durable scientific state:"
)
print(
    "  HEAD =",
    EXPECTED_HEAD,
)

print()

print(
    "Recovered CICIDS2017:"
)
print(
    "  assets       = 8 / 8 EXACT"
)
print(
    "  total bytes  =",
    f"{actual_total_bytes:,}",
)

print()

print(
    "Recovered global cache:"
)
print(
    "  rows                = 2,830,743"
)
print(
    "  global_family_codes = EXACT SHA PASS"
)
print(
    "  global_day_codes    = EXACT SHA PASS"
)
print(
    "  global_source_codes = EXACT SHA PASS"
)

print()

print(
    "Data access during recovery:"
)
print(
    "  physical labels read =",
    f"{physical_label_rows_read:,}",
)
print(
    "  effective labels used=",
    f"{effective_label_rows_used:,}",
)
print(
    "  predictor values     = 0"
)

print()

print(
    "Scientific operations:"
)
print(
    "  MODEL_FITS                = 0"
)
print(
    "  MODEL_INFERENCE           = 0"
)
print(
    "  THRESHOLD_SELECTION       = 0"
)
print(
    "  TARGET_PREDICTOR_OPENINGS = 0"
)

print()

print(
    "Fit ledger remains:"
)
print(
    "  NEW authorized = 108"
)
print(
    "  consumed       = 0"
)
print(
    "  remaining      = 108"
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)
print(
    "  Stage28-1B — materialize the five random-LOAO "
    "memberships and freeze the 120-component execution "
    "manifest BEFORE fit #1."
)

print()
print(SEP)


STAGE28-RUNTIME-R1 — EXACT DURABLE-PARENT GATE

Local HEAD : e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
origin/main: e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
Expected   : e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
Branch     : main
Git clean  : True

[PASS] exact remotely frozen Stage28-1A state

LOAD DURABLE FROZEN RECEIPTS

[PASS] Stage27 source receipt
[PASS] Stage27 family-cache receipt
[PASS] Stage27 canonical census
[PASS] Stage28-1A global reconstruction receipt

HUGGING FACE CLIENT

huggingface_hub: 1.11.0
dataset: bvsam/cic-ids-2017
revision: b7e532345512edcd530cb1770dc76636aeb52802

RECOVER 8 BYTE-EXACT CICIDS2017 ASSETS


------------------------------------------------------------------------------------------------------------------------
[0] Monday-WorkingHours.pcap_ISCX.csv.parquet
Expected bytes : 65,465,382
Expected SHA256: dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02


traffic_labels/Monday-WorkingHours.pcap_(…):   0%|          | 0.00/65.5M [00:00<?, ?B/s]

Downloaded: /root/.cache/huggingface/hub/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Monday-WorkingHours.pcap_ISCX.csv.parquet
Actual bytes : 65,465,382
Actual SHA256: dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02
[PASS] exact runtime copy

------------------------------------------------------------------------------------------------------------------------
[1] Tuesday-WorkingHours.pcap_ISCX.csv.parquet
Expected bytes : 52,701,751
Expected SHA256: 27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4


traffic_labels/Tuesday-WorkingHours.pcap(…):   0%|          | 0.00/52.7M [00:00<?, ?B/s]

Downloaded: /root/.cache/huggingface/hub/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Tuesday-WorkingHours.pcap_ISCX.csv.parquet
Actual bytes : 52,701,751
Actual SHA256: 27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4
[PASS] exact runtime copy

------------------------------------------------------------------------------------------------------------------------
[2] Wednesday-workingHours.pcap_ISCX.csv.parquet
Expected bytes : 76,512,727
Expected SHA256: d23a259820b16e1ad54f9f3b58d5727c5032d383015f90bc7c07cebbdf8a7140


...

Downloaded: /root/.cache/huggingface/hub/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.parquet
Actual bytes : 23,048,086
Actual SHA256: 7c5876d52189fc01af54bad6cf23afe9f7fbc0e3ca6c3595920754f0c3ba8f66
[PASS] exact runtime copy

------------------------------------------------------------------------------------------------------------------------
[6] Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
Expected bytes : 18,632,427
Expected SHA256: 4d78cee297c27f1a9947b9384793e587a46c7a3ea89db199553dabddc9835d4a


traffic_labels/Friday-WorkingHours-After(…):   0%|          | 0.00/18.6M [00:00<?, ?B/s]

Downloaded: /root/.cache/huggingface/hub/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
Actual bytes : 18,632,427
Actual SHA256: 4d78cee297c27f1a9947b9384793e587a46c7a3ea89db199553dabddc9835d4a
[PASS] exact runtime copy

------------------------------------------------------------------------------------------------------------------------
[7] Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet
Expected bytes : 21,999,571
Expected SHA256: 2c00236b13a69f4b1c222b8f4a89451dc2148cb04a8cd0c45c2a87af51471774


traffic_labels/Friday-WorkingHours-Morni(…):   0%|          | 0.00/22.0M [00:00<?, ?B/s]

Downloaded: /root/.cache/huggingface/hub/datasets--bvsam--cic-ids-2017/snapshots/b7e532345512edcd530cb1770dc76636aeb52802/traffic_labels/Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet
Actual bytes : 21,999,571
Actual SHA256: 2c00236b13a69f4b1c222b8f4a89451dc2148cb04a8cd0c45c2a87af51471774
[PASS] exact runtime copy

[PASS] 8 / 8 exact source assets
Total bytes: 305,935,672

RECONSTRUCT GLOBAL MEMBERSHIP ARRAYS — LABELS ONLY


[0] Monday — Monday-WorkingHours.pcap_ISCX.csv.parquet
  physical : 529,918
  effective: 529,918
  families : {'BENIGN': 529918}

[1] Tuesday — Tuesday-WorkingHours.pcap_ISCX.csv.parquet
  physical : 445,909
  effective: 445,909
  families : {'BENIGN': 432074, 'AUTH_BRUTE_FORCE': 13835}

[2] Wednesday — Wednesday-workingHours.pcap_ISCX.csv.parquet
  physical : 692,703
  effective: 692,703
  families : {'BENIGN': 440031, 'DOS': 252661, 'TARGET_ONLY_UNSEEN': 11}

[3] Thursday — Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet
  physical : 288,60

In [3]:
# =================================================================================================
# STAGE28-1B — RANDOM-LOAO MEMBERSHIP FREEZE + 120-COMPONENT EXECUTION MANIFEST
#
# Expected durable parent:
#   e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
#
# Requires recovered runtime cache:
#   /kaggle/working/stage28_1a_runtime_cache/stage27_global_cache/
#
# PURPOSE
# -------
# 1. Verify exact Stage28-1A global runtime cache.
# 2. Materialize five Stage28B RANDOM-LOAO memberships.
# 3. Membership seed = 42, fixed across model seeds 42..46.
# 4. Freeze exact TRAIN / VALIDATION / PRIMARY TARGET memberships.
# 5. Freeze per-family TRAIN-only positive class weights.
# 6. Audit all held-out/unseen leakage constraints.
# 7. Verify the 12 pre-existing reusable model components.
# 8. Freeze complete 120-component execution manifest:
#
#       Stage22 FULL:
#           20 components = 2 reused + 18 new
#
#       Stage27 chronology LOAO:
#           50 components = 10 reused + 40 new
#
#       Stage28B random LOAO:
#           50 components = 0 reused + 50 new
#
#       TOTAL:
#           120 component realizations
#            12 reused
#           108 new fits
#           110 scientific evaluation cells
#
# ZERO:
#   PREDICTOR VALUES READ
#   MODEL FITS
#   MODEL INFERENCE
#   THRESHOLD SELECTION
#   TARGET PREDICTOR OPENINGS
#
# IMPORTANT RANDOM MEMBERSHIP ORDER
# ---------------------------------
# A. Target benign:
#      input = all benign global IDs in canonical ascending order
#      train_test_split(test_size=.20, random_state=42, shuffle=True, stratify=None)
#
# B. Development:
#      remaining benign + primary-seven attacks excluding held-out family
#      development input rebuilt in canonical ascending global-row order
#
# C. Train / validation:
#      train_test_split(test_size=.20, random_state=42,
#                       shuffle=True, stratify=binary_label)
#
# D. Returned TRAIN / VALIDATION row order is frozen exactly.
# E. PRIMARY TARGET is stored in canonical ascending global-row order.
#
# Runtime .npy memberships stay outside Git.
# Only durable receipts/manifests are committed.
# =================================================================================================

from __future__ import annotations

import csv
import hashlib
import json
import math
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import sklearn
from sklearn.model_selection import train_test_split


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_BRANCH = "main"

EXPECTED_PARENT = (
    "e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b"
)

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

LOCK_DIR = (
    STAGE28_ROOT
    / "stage28_0_protocol_lock"
)

AMEND_DIR = (
    STAGE28_ROOT
    / "stage28_0a_preexecution_amendment"
)

STAGE28_1A_DIR = (
    STAGE28_ROOT
    / "stage28_1a_inherited_membership_reconstruction"
)

OUTPUT_DIR = (
    STAGE28_ROOT
    / "stage28_1b_random_loao_membership_and_execution_lock"
)

GLOBAL_RUNTIME_ROOT = Path(
    "/kaggle/working/"
    "stage28_1a_runtime_cache/"
    "stage27_global_cache"
)

RANDOM_RUNTIME_ROOT = Path(
    "/kaggle/working/"
    "stage28_1b_runtime_cache/"
    "random_loao"
)

MEMBERSHIP_SEED = 42

MODEL_SEEDS = [
    42,
    43,
    44,
    45,
    46,
]

ELIGIBLE_FAMILIES = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

PRIMARY_SEVEN = [
    "BOT",
    "DDOS",
    "DOS",
    "AUTH_BRUTE_FORCE",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

LEARNERS = [
    "XGBOOST",
    "LIGHTGBM",
]

STAGE22_CELLS = [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]

COMMIT_MESSAGE = (
    "stage28-1b: freeze random LOAO memberships and execution manifest"
)


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args, check=True):
    return (
        run(
            ["git", *args],
            check=check,
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def write_json(path, obj):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            sort_keys=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array_content(arr):
    arr = np.ascontiguousarray(
        arr
    )

    h = hashlib.sha256()

    h.update(
        arr.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


def membership_bitset_sha(mask):
    packed = np.packbits(
        np.asarray(
            mask,
            dtype=np.uint8,
        ),
        bitorder="little",
    )

    return sha256_array_content(
        packed
    )


def canonical_json_sha(obj):
    blob = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        blob
    ).hexdigest()


def remote_main_sha():
    out = git(
        "ls-remote",
        "origin",
        "refs/heads/main",
    )

    if not out:
        raise RuntimeError(
            "Unable to resolve remote main."
        )

    return out.split()[0]


def parameter_set(
    base,
    seed,
):
    params = dict(
        base
    )

    if int(
        params.get(
            "random_state",
            -1,
        )
    ) != 42:
        raise RuntimeError(
            "Frozen base parameters do not "
            "use random_state=42."
        )

    params[
        "random_state"
    ] = int(
        seed
    )

    return params


# =================================================================================================
# 2. EXACT REPOSITORY GATE
# =================================================================================================

banner(
    "STAGE28-1B — EXACT REPOSITORY GATE"
)

branch = git(
    "branch",
    "--show-current",
)

head = git(
    "rev-parse",
    "HEAD",
)

remote = remote_main_sha()

status = git(
    "status",
    "--porcelain",
)

print(
    "Repository :",
    REPO,
)

print(
    "Branch     :",
    branch,
)

print(
    "Local HEAD :",
    head,
)

print(
    "Remote main:",
    remote,
)

print(
    "Expected   :",
    EXPECTED_PARENT,
)

print(
    "Git clean  :",
    not bool(
        status
    ),
)

if branch != EXPECTED_BRANCH:
    raise RuntimeError(
        f"Expected branch {EXPECTED_BRANCH}; "
        f"got {branch}"
    )

if head != EXPECTED_PARENT:
    raise RuntimeError(
        "Local HEAD mismatch.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={head}"
    )

if remote != EXPECTED_PARENT:
    raise RuntimeError(
        "Remote main mismatch.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={remote}"
    )

if status:
    raise RuntimeError(
        "Worktree must be clean:\n"
        + status
    )

if OUTPUT_DIR.exists():
    raise RuntimeError(
        "Stage28-1B output directory "
        "already exists:\n"
        f"{OUTPUT_DIR}"
    )

print()
print(
    "[PASS] exact Stage28-1A durable parent"
)

print(
    "[PASS] local == remote"
)

print(
    "[PASS] clean worktree"
)

print(
    "[PASS] Stage28-1B destination absent"
)


# =================================================================================================
# 3. LOAD / VERIFY EFFECTIVE FROZEN SPECIFICATIONS
# =================================================================================================

banner(
    "LOAD EFFECTIVE STAGE28 SPECIFICATIONS"
)

random_spec_path = require_file(
    LOCK_DIR
    / "random_loao_split_spec.json"
)

model_inventory_path = require_file(
    LOCK_DIR
    / "model_inventory.json"
)

seed_spec_path = require_file(
    LOCK_DIR
    / "seed_spec.json"
)

effective_random_path = require_file(
    AMEND_DIR
    / "effective_random_loao_population_spec.json"
)

effective_budget_path = require_file(
    AMEND_DIR
    / "effective_fit_budget.json"
)

effective_stage22_path = require_file(
    AMEND_DIR
    / "effective_stage22_execution_policy.json"
)

effective_weight_path = require_file(
    AMEND_DIR
    / "effective_loao_class_weight_policy.json"
)

stage28_1a_freeze_path = require_file(
    STAGE28_1A_DIR
    / "stage28_1a_freeze_record.json"
)


random_spec = read_json(
    random_spec_path
)

model_inventory = read_json(
    model_inventory_path
)

seed_spec = read_json(
    seed_spec_path
)

effective_random = read_json(
    effective_random_path
)

effective_budget = read_json(
    effective_budget_path
)

effective_stage22 = read_json(
    effective_stage22_path
)

effective_weight = read_json(
    effective_weight_path
)

stage28_1a_freeze = read_json(
    stage28_1a_freeze_path
)


if (
    seed_spec[
        "training_seeds"
    ]
    != MODEL_SEEDS
):
    raise RuntimeError(
        "Frozen training seed list mismatch."
    )

if (
    seed_spec[
        "stage28b"
    ][
        "random_membership_seed"
    ]
    != MEMBERSHIP_SEED
):
    raise RuntimeError(
        "Frozen random membership seed mismatch."
    )

if (
    seed_spec[
        "stage28b"
    ][
        "membership_seed_varies_with_model_seed"
    ]
    is not False
):
    raise RuntimeError(
        "Stage28B membership is not frozen "
        "across model seeds."
    )

if (
    random_spec[
        "membership_seed"
    ]
    != MEMBERSHIP_SEED
):
    raise RuntimeError(
        "Random-LOAO base membership seed mismatch."
    )

if (
    random_spec[
        "construction_per_held_out_family"
    ][
        "step_2_random_target_benign"
    ][
        "test_size"
    ]
    != 0.2
):
    raise RuntimeError(
        "Frozen target-benign fraction != 0.20."
    )

if (
    random_spec[
        "construction_per_held_out_family"
    ][
        "step_4_train_validation"
    ][
        "test_size"
    ]
    != 0.2
):
    raise RuntimeError(
        "Frozen validation fraction != 0.20."
    )

if (
    effective_random[
        "eligible_held_out_families"
    ]
    != ELIGIBLE_FAMILIES
):
    raise RuntimeError(
        "Eligible family list mismatch."
    )

if (
    effective_random[
        "primary_seven_attack_families"
    ]
    != PRIMARY_SEVEN
):
    raise RuntimeError(
        "Primary-seven family list mismatch."
    )

if (
    effective_budget[
        "totals"
    ][
        "existing_reused"
    ]
    != 12
):
    raise RuntimeError(
        "Reused component budget != 12."
    )

if (
    effective_budget[
        "totals"
    ][
        "new_fit_budget"
    ]
    != 108
):
    raise RuntimeError(
        "New-fit budget != 108."
    )

if (
    effective_budget[
        "totals"
    ][
        "total_component_realizations"
    ]
    != 120
):
    raise RuntimeError(
        "Component universe != 120."
    )

if (
    effective_budget[
        "totals"
    ][
        "scientific_evaluation_cells"
    ]
    != 110
):
    raise RuntimeError(
        "Scientific evaluation cells != 110."
    )

if (
    effective_stage22[
        "lightgbm"
    ][
        "seed42"
    ]
    != "REUSE_DURABLE_PARENT_CPU_MODEL"
):
    raise RuntimeError(
        "Stage22 LightGBM seed42 reuse "
        "policy mismatch."
    )

if (
    effective_stage22[
        "xgboost"
    ][
        "seed42"
    ]
    != "NEW_CPU_REFIT_REQUIRED"
):
    raise RuntimeError(
        "Stage22 XGBoost seed42 CPU-refit "
        "policy mismatch."
    )

if (
    effective_stage22[
        "xgboost"
    ][
        "historical_seed42_cuda_model"
    ]
    != "NOT_USED_IN_STAGE28_SEED_VARIANCE"
):
    raise RuntimeError(
        "Historical CUDA-XGB exclusion "
        "policy mismatch."
    )

if (
    stage28_1a_freeze[
        "fit_budget"
    ][
        "new_fits_consumed"
    ]
    != 0
):
    raise RuntimeError(
        "Stage28-1A unexpectedly consumed fits."
    )

print(
    "[PASS] model seeds = [42,43,44,45,46]"
)

print(
    "[PASS] membership seed = 42"
)

print(
    "[PASS] memberships fixed across model seeds"
)

print(
    "[PASS] target benign fraction = 0.20"
)

print(
    "[PASS] development validation fraction = 0.20"
)

print(
    "[PASS] development attacks = primary-seven excluding held-out"
)

print(
    "[PASS] TARGET_ONLY_UNSEEN excluded"
)

print(
    "[PASS] OTHER_ATTACK_UNSEEN_LABEL excluded"
)

print(
    "[PASS] effective fit budget = 108 NEW / 12 REUSED"
)

print(
    "[PASS] 120 component realizations / 110 evaluation cells"
)


# =================================================================================================
# 4. VERIFY RECOVERED STAGE28-1A GLOBAL RUNTIME CACHE
# =================================================================================================

banner(
    "VERIFY RECOVERED STAGE28-1A GLOBAL RUNTIME CACHE"
)

family_codes_path = require_file(
    GLOBAL_RUNTIME_ROOT
    / "global_family_codes.npy"
)

day_codes_path = require_file(
    GLOBAL_RUNTIME_ROOT
    / "global_day_codes.npy"
)

source_codes_path = require_file(
    GLOBAL_RUNTIME_ROOT
    / "global_source_codes.npy"
)

global_family_codes = np.load(
    family_codes_path,
    mmap_mode="r",
    allow_pickle=False,
)

global_day_codes = np.load(
    day_codes_path,
    mmap_mode="r",
    allow_pickle=False,
)

global_source_codes = np.load(
    source_codes_path,
    mmap_mode="r",
    allow_pickle=False,
)

N = int(
    len(
        global_family_codes
    )
)

if N != 2_830_743:
    raise RuntimeError(
        f"Unexpected population N={N}"
    )

if (
    len(global_day_codes) != N
    or
    len(global_source_codes) != N
):
    raise RuntimeError(
        "Recovered global-array lengths disagree."
    )

global_receipt = read_json(
    require_file(
        STAGE28_1A_DIR
        / "stage27_global_cache_reconstruction_receipt.json"
    )
)

for name, arr in {
    "global_family_codes":
        global_family_codes,

    "global_day_codes":
        global_day_codes,

    "global_source_codes":
        global_source_codes,
}.items():

    actual_sha = sha256_array_content(
        np.asarray(
            arr
        )
    )

    r = (
        global_receipt[
            "global_arrays"
        ][
            name
        ]
    )

    stage28_sha = (
        r[
            "reconstructed_content_sha256"
        ]
    )

    historical_sha = (
        r[
            "historical_content_sha256"
        ]
    )

    print(
        name
    )

    print(
        "  current   :",
        actual_sha,
    )

    print(
        "  Stage28-1A:",
        stage28_sha,
    )

    print(
        "  historical:",
        historical_sha,
    )

    if not (
        actual_sha
        == stage28_sha
        == historical_sha
    ):
        raise RuntimeError(
            f"{name}: recovered runtime "
            "content SHA mismatch."
        )

print()
print(
    "[PASS] Stage28-1A runtime cache is exact"
)


# =================================================================================================
# 5. LOAD FROZEN STAGE27 TAXONOMY / CENSUS
# =================================================================================================

banner(
    "FROZEN STAGE27 TAXONOMY / CENSUS"
)

family_receipt = read_json(
    require_file(
        REPO
        / "results"
        / "stage27_loao_unseen_attack"
        / "stage27_1a_fold_membership"
        / "family_code_cache_receipt.json"
    )
)

census = read_json(
    require_file(
        REPO
        / "results"
        / "stage27_loao_unseen_attack"
        / "stage27_1a_fold_membership"
        / "canonical_label_census.json"
    )
)

CODEBOOK = (
    family_receipt[
        "family_codebook"
    ]
)

EXPECTED_CODEBOOK = {
    "BENIGN": 0,
    "BOT": 1,
    "DDOS": 2,
    "DOS": 3,
    "AUTH_BRUTE_FORCE": 4,
    "INFILTRATION": 5,
    "PORT_SCAN": 6,
    "WEB_ATTACK": 7,
    "TARGET_ONLY_UNSEEN": 8,
    "OTHER_ATTACK_UNSEEN_LABEL": 9,
}

if CODEBOOK != EXPECTED_CODEBOOK:
    raise RuntimeError(
        "Frozen family codebook mismatch."
    )

BENIGN_CODE = (
    CODEBOOK[
        "BENIGN"
    ]
)

TARGET_ONLY_CODE = (
    CODEBOOK[
        "TARGET_ONLY_UNSEEN"
    ]
)

OTHER_UNSEEN_CODE = (
    CODEBOOK[
        "OTHER_ATTACK_UNSEEN_LABEL"
    ]
)

PRIMARY_CODES = np.asarray(
    [
        CODEBOOK[
            x
        ]
        for x
        in PRIMARY_SEVEN
    ],
    dtype=np.uint8,
)

expected_benign = int(
    census[
        "global_arithmetic"
    ][
        "benign"
    ]
)

expected_primary_attack = int(
    census[
        "global_arithmetic"
    ][
        "primary_seven_family_attack"
    ]
)

expected_target_only = int(
    census[
        "global_arithmetic"
    ][
        "target_only_unseen"
    ]
)

expected_other_unseen = int(
    census[
        "global_arithmetic"
    ][
        "other_attack_unseen_label"
    ]
)

EXPECTED_ELIGIBLE_ROWS = (
    expected_benign
    +
    expected_primary_attack
)

if EXPECTED_ELIGIBLE_ROWS != 2_830_732:
    raise RuntimeError(
        "Unexpected eligible Stage28B population."
    )

if expected_target_only != 11:
    raise RuntimeError(
        "TARGET_ONLY_UNSEEN count != 11."
    )

if expected_other_unseen != 0:
    raise RuntimeError(
        "OTHER_ATTACK_UNSEEN_LABEL count != 0."
    )

print(
    "Benign                 :",
    f"{expected_benign:,}",
)

print(
    "Primary-seven attacks  :",
    f"{expected_primary_attack:,}",
)

print(
    "Eligible Stage28B rows :",
    f"{EXPECTED_ELIGIBLE_ROWS:,}",
)

print(
    "Target-only unseen     :",
    expected_target_only,
)

print(
    "Other unseen           :",
    expected_other_unseen,
)


# =================================================================================================
# 6. FREEZE SHARED TARGET-BENIGN MEMBERSHIP
# =================================================================================================

banner(
    "FREEZE SHARED TARGET-BENIGN MEMBERSHIP"
)

benign_idx = np.flatnonzero(
    global_family_codes
    == BENIGN_CODE
).astype(
    np.int32,
    copy=False,
)

if len(
    benign_idx
) != expected_benign:
    raise RuntimeError(
        "Benign population mismatch."
    )

# Canonical ascending input order is guaranteed by np.flatnonzero.
if (
    len(benign_idx) > 1
    and
    not np.all(
        benign_idx[:-1]
        <
        benign_idx[1:]
    )
):
    raise RuntimeError(
        "Benign input order is not canonical ascending."
    )


benign_dev_returned, target_benign_returned = (
    train_test_split(
        benign_idx,
        test_size=0.20,
        random_state=MEMBERSHIP_SEED,
        shuffle=True,
        stratify=None,
    )
)

benign_dev_returned = np.asarray(
    benign_dev_returned,
    dtype=np.int32,
)

target_benign_returned = np.asarray(
    target_benign_returned,
    dtype=np.int32,
)


target_benign_mask = np.zeros(
    N,
    dtype=np.bool_,
)

target_benign_mask[
    target_benign_returned
] = True


benign_dev_mask = (
    (
        global_family_codes
        == BENIGN_CODE
    )
    &
    (
        ~target_benign_mask
    )
)

target_benign_count = int(
    target_benign_mask.sum()
)

benign_dev_count = int(
    benign_dev_mask.sum()
)

# sklearn uses ceil(n * test_size) for float test_size here.
if target_benign_count != 454_620:
    raise RuntimeError(
        "Unexpected target-benign realization.\n"
        f"expected=454620\n"
        f"actual={target_benign_count}"
    )

if benign_dev_count != 1_818_477:
    raise RuntimeError(
        "Unexpected benign-development count."
    )

if (
    target_benign_count
    +
    benign_dev_count
    != expected_benign
):
    raise RuntimeError(
        "Benign partition coverage mismatch."
    )


target_benign_canonical_idx = np.flatnonzero(
    target_benign_mask
).astype(
    np.int32,
    copy=False,
)


target_benign_return_order_sha = (
    sha256_array_content(
        target_benign_returned
    )
)

target_benign_canonical_sha = (
    sha256_array_content(
        target_benign_canonical_idx
    )
)

target_benign_bitset_sha = (
    membership_bitset_sha(
        target_benign_mask
    )
)


print(
    "All benign rows            :",
    f"{expected_benign:,}",
)

print(
    "Target benign rows         :",
    f"{target_benign_count:,}",
)

print(
    "Benign development rows    :",
    f"{benign_dev_count:,}",
)

print(
    "Operator-return SHA256     :",
    target_benign_return_order_sha,
)

print(
    "Canonical target SHA256    :",
    target_benign_canonical_sha,
)

print(
    "Membership bitset SHA256   :",
    target_benign_bitset_sha,
)

print()
print(
    "[PASS] shared target-benign membership "
    "materialized exactly once"
)


RANDOM_RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

shared_target_runtime_path = (
    RANDOM_RUNTIME_ROOT
    / "shared_target_benign_global_idx.npy"
)

np.save(
    shared_target_runtime_path,
    target_benign_canonical_idx,
    allow_pickle=False,
)


# =================================================================================================
# 7. MATERIALIZE FIVE RANDOM-LOAO MEMBERSHIPS
# =================================================================================================

banner(
    "MATERIALIZE FIVE RANDOM-LOAO MEMBERSHIPS"
)

primary_attack_global = np.isin(
    global_family_codes,
    PRIMARY_CODES,
)

eligible_global = (
    (
        global_family_codes
        == BENIGN_CODE
    )
    |
    primary_attack_global
)

excluded_unseen_global = (
    (
        global_family_codes
        == TARGET_ONLY_CODE
    )
    |
    (
        global_family_codes
        == OTHER_UNSEEN_CODE
    )
)

if int(
    excluded_unseen_global.sum()
) != 11:
    raise RuntimeError(
        "Excluded unseen population != 11."
    )


fold_receipts = {}
fold_class_weights = {}


for family in ELIGIBLE_FAMILIES:

    print()
    print("=" * 120)
    print(
        "RANDOM-LOAO:",
        family,
    )
    print("=" * 120)

    heldout_code = (
        CODEBOOK[
            family
        ]
    )

    heldout_global = (
        global_family_codes
        == heldout_code
    )

    known_attack_global = (
        primary_attack_global
        &
        (
            ~heldout_global
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Development population:
    # remaining benign + primary-seven non-held-out attacks.
    #
    # Rebuild in canonical ascending Stage27 global-row order.
    # ----------------------------------------------------------------------------------------------

    development_mask = (
        benign_dev_mask
        |
        known_attack_global
    )

    development_idx = np.flatnonzero(
        development_mask
    ).astype(
        np.int32,
        copy=False,
    )

    if (
        len(development_idx) > 1
        and
        not np.all(
            development_idx[:-1]
            <
            development_idx[1:]
        )
    ):
        raise RuntimeError(
            f"{family}: development input not "
            "canonical ascending."
        )


    development_y = (
        global_family_codes[
            development_idx
        ]
        != BENIGN_CODE
    ).astype(
        np.uint8,
        copy=False,
    )


    dev_benign = int(
        np.count_nonzero(
            development_y == 0
        )
    )

    dev_attack = int(
        np.count_nonzero(
            development_y == 1
        )
    )

    expected_heldout = int(
        census[
            "family_counts"
        ][
            family
        ]
    )

    expected_known_attack = (
        expected_primary_attack
        -
        expected_heldout
    )

    if dev_benign != benign_dev_count:
        raise RuntimeError(
            f"{family}: development benign mismatch."
        )

    if dev_attack != expected_known_attack:
        raise RuntimeError(
            f"{family}: development known-attack mismatch."
        )


    # ----------------------------------------------------------------------------------------------
    # Frozen development train/validation split.
    # Returned row order is scientifically frozen.
    # ----------------------------------------------------------------------------------------------

    train_idx, validation_idx = (
        train_test_split(
            development_idx,
            test_size=0.20,
            random_state=MEMBERSHIP_SEED,
            shuffle=True,
            stratify=development_y,
        )
    )

    train_idx = np.asarray(
        train_idx,
        dtype=np.int32,
    )

    validation_idx = np.asarray(
        validation_idx,
        dtype=np.int32,
    )


    # ----------------------------------------------------------------------------------------------
    # Primary target:
    # shared target benign + ALL held-out-family positives.
    # Canonical ascending global row order.
    # ----------------------------------------------------------------------------------------------

    primary_target_mask = (
        target_benign_mask
        |
        heldout_global
    )

    primary_target_idx = np.flatnonzero(
        primary_target_mask
    ).astype(
        np.int32,
        copy=False,
    )


    # ----------------------------------------------------------------------------------------------
    # Strong membership role audit.
    # 0 = unassigned/excluded
    # 1 = TRAIN
    # 2 = VALIDATION
    # 3 = PRIMARY TARGET
    # ----------------------------------------------------------------------------------------------

    role = np.zeros(
        N,
        dtype=np.uint8,
    )

    role[
        train_idx
    ] = 1


    if np.any(
        role[
            validation_idx
        ] != 0
    ):
        raise RuntimeError(
            f"{family}: TRAIN/VALIDATION overlap."
        )

    role[
        validation_idx
    ] = 2


    if np.any(
        role[
            primary_target_idx
        ] != 0
    ):
        raise RuntimeError(
            f"{family}: development/TARGET overlap."
        )

    role[
        primary_target_idx
    ] = 3


    # ----------------------------------------------------------------------------------------------
    # Exact eligible-population coverage.
    # ----------------------------------------------------------------------------------------------

    assigned_rows = int(
        np.count_nonzero(
            role
        )
    )

    if assigned_rows != EXPECTED_ELIGIBLE_ROWS:
        raise RuntimeError(
            f"{family}: assigned-row count mismatch.\n"
            f"expected={EXPECTED_ELIGIBLE_ROWS}\n"
            f"actual={assigned_rows}"
        )

    if np.any(
        role[
            eligible_global
        ] == 0
    ):
        raise RuntimeError(
            f"{family}: eligible population not "
            "fully assigned."
        )

    if np.any(
        role[
            excluded_unseen_global
        ] != 0
    ):
        raise RuntimeError(
            f"{family}: excluded unseen category assigned."
        )


    # ----------------------------------------------------------------------------------------------
    # Held-out / unseen leakage audit.
    # ----------------------------------------------------------------------------------------------

    heldout_train = int(
        np.count_nonzero(
            global_family_codes[
                train_idx
            ]
            == heldout_code
        )
    )

    heldout_validation = int(
        np.count_nonzero(
            global_family_codes[
                validation_idx
            ]
            == heldout_code
        )
    )

    heldout_target = int(
        np.count_nonzero(
            global_family_codes[
                primary_target_idx
            ]
            == heldout_code
        )
    )


    target_codes = global_family_codes[
        primary_target_idx
    ]

    target_known_attack = int(
        np.count_nonzero(
            np.isin(
                target_codes,
                PRIMARY_CODES,
            )
            &
            (
                target_codes
                != heldout_code
            )
        )
    )


    target_only_train = int(
        np.count_nonzero(
            global_family_codes[
                train_idx
            ]
            == TARGET_ONLY_CODE
        )
    )

    target_only_validation = int(
        np.count_nonzero(
            global_family_codes[
                validation_idx
            ]
            == TARGET_ONLY_CODE
        )
    )

    target_only_target = int(
        np.count_nonzero(
            target_codes
            == TARGET_ONLY_CODE
        )
    )


    other_unseen_train = int(
        np.count_nonzero(
            global_family_codes[
                train_idx
            ]
            == OTHER_UNSEEN_CODE
        )
    )

    other_unseen_validation = int(
        np.count_nonzero(
            global_family_codes[
                validation_idx
            ]
            == OTHER_UNSEEN_CODE
        )
    )

    other_unseen_target = int(
        np.count_nonzero(
            target_codes
            == OTHER_UNSEEN_CODE
        )
    )


    if heldout_train != 0:
        raise RuntimeError(
            f"{family}: held-out family in TRAIN."
        )

    if heldout_validation != 0:
        raise RuntimeError(
            f"{family}: held-out family in VALIDATION."
        )

    if heldout_target != expected_heldout:
        raise RuntimeError(
            f"{family}: TARGET does not contain "
            "all held-out attacks."
        )

    if target_known_attack != 0:
        raise RuntimeError(
            f"{family}: known attack in PRIMARY TARGET."
        )

    if any([
        target_only_train,
        target_only_validation,
        target_only_target,
        other_unseen_train,
        other_unseen_validation,
        other_unseen_target,
    ]):
        raise RuntimeError(
            f"{family}: excluded unseen category leaked."
        )


    # ----------------------------------------------------------------------------------------------
    # Counts.
    # ----------------------------------------------------------------------------------------------

    train_codes = (
        global_family_codes[
            train_idx
        ]
    )

    validation_codes = (
        global_family_codes[
            validation_idx
        ]
    )

    train_benign = int(
        np.count_nonzero(
            train_codes
            == BENIGN_CODE
        )
    )

    train_attack = int(
        len(train_codes)
        -
        train_benign
    )

    validation_benign = int(
        np.count_nonzero(
            validation_codes
            == BENIGN_CODE
        )
    )

    validation_attack = int(
        len(validation_codes)
        -
        validation_benign
    )

    target_benign = int(
        np.count_nonzero(
            target_codes
            == BENIGN_CODE
        )
    )

    target_attack = int(
        len(target_codes)
        -
        target_benign
    )

    if target_benign != target_benign_count:
        raise RuntimeError(
            f"{family}: target benign count mismatch."
        )

    if target_attack != expected_heldout:
        raise RuntimeError(
            f"{family}: target attack count mismatch."
        )


    # ----------------------------------------------------------------------------------------------
    # Frozen TRAIN-only positive-class weight.
    # ----------------------------------------------------------------------------------------------

    if train_attack <= 0:
        raise RuntimeError(
            f"{family}: zero TRAIN attack rows."
        )

    positive_class_weight = (
        train_benign
        /
        train_attack
    )

    fold_class_weights[
        family
    ] = (
        positive_class_weight
    )


    # ----------------------------------------------------------------------------------------------
    # Freeze ordered-array and membership-set hashes.
    # ----------------------------------------------------------------------------------------------

    train_mask = (
        role == 1
    )

    validation_mask = (
        role == 2
    )

    target_mask = (
        role == 3
    )


    hashes = {

        "train": {
            "ordered_global_idx_content_sha256":
                sha256_array_content(
                    train_idx
                ),

            "membership_bitset_sha256":
                membership_bitset_sha(
                    train_mask
                ),
        },

        "validation": {
            "ordered_global_idx_content_sha256":
                sha256_array_content(
                    validation_idx
                ),

            "membership_bitset_sha256":
                membership_bitset_sha(
                    validation_mask
                ),
        },

        "primary_target": {
            "ordered_global_idx_content_sha256":
                sha256_array_content(
                    primary_target_idx
                ),

            "membership_bitset_sha256":
                membership_bitset_sha(
                    target_mask
                ),
        },
    }


    # ----------------------------------------------------------------------------------------------
    # Runtime arrays outside Git.
    # ----------------------------------------------------------------------------------------------

    family_runtime_dir = (
        RANDOM_RUNTIME_ROOT
        / family
    )

    family_runtime_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    runtime_paths = {}

    for name, arr in {
        "train_global_idx":
            train_idx,

        "validation_global_idx":
            validation_idx,

        "primary_target_global_idx":
            primary_target_idx,
    }.items():

        path = (
            family_runtime_dir
            / f"{name}.npy"
        )

        np.save(
            path,
            arr,
            allow_pickle=False,
        )

        # Reload + verify ordered content immediately.
        check_arr = np.load(
            path,
            mmap_mode="r",
            allow_pickle=False,
        )

        actual_saved_sha = (
            sha256_array_content(
                np.asarray(
                    check_arr
                )
            )
        )

        expected_saved_sha = {
            "train_global_idx":
                hashes[
                    "train"
                ][
                    "ordered_global_idx_content_sha256"
                ],

            "validation_global_idx":
                hashes[
                    "validation"
                ][
                    "ordered_global_idx_content_sha256"
                ],

            "primary_target_global_idx":
                hashes[
                    "primary_target"
                ][
                    "ordered_global_idx_content_sha256"
                ],
        }[
            name
        ]

        if (
            actual_saved_sha
            != expected_saved_sha
        ):
            raise RuntimeError(
                f"{family}/{name}: runtime-save "
                "verification failed."
            )

        runtime_paths[
            name
        ] = str(
            path
        )


    receipt = {

        "stage":
            "Stage28-1B",

        "arm":
            "28B_RANDOM_SPLIT_LOAO_CONTROL",

        "type":
            "RANDOM_LOAO_MEMBERSHIP_RECEIPT",

        "created_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "scientific_parent_commit":
            EXPECTED_PARENT,

        "held_out_family":
            family,

        "membership_seed":
            MEMBERSHIP_SEED,

        "membership_fixed_across_model_seeds":
            True,

        "model_seeds":
            MODEL_SEEDS,

        "software": {
            "python":
                sys.version.split()[0],

            "numpy":
                np.__version__,

            "scikit_learn":
                sklearn.__version__,

            "platform":
                platform.platform(),
        },

        "construction": {

            "canonical_population_order":
                "STAGE27_ZERO_BASED_GLOBAL_ROW_ID_ASCENDING",

            "shared_target_benign": {
                "operator":
                    "sklearn.model_selection.train_test_split",

                "input":
                    "ALL_BENIGN_GLOBAL_IDS_CANONICAL_ASCENDING",

                "test_size":
                    0.20,

                "shuffle":
                    True,

                "random_state":
                    42,

                "stratify":
                    None,

                "role":
                    "TARGET_BENIGN",
            },

            "development_population": (
                "REMAINING_BENIGN_PLUS_PRIMARY_SEVEN_"
                "ATTACKS_EXCLUDING_HELD_OUT_FAMILY"
            ),

            "development_input_order":
                "CANONICAL_GLOBAL_ROW_ID_ASCENDING",

            "train_validation": {
                "operator":
                    "sklearn.model_selection.train_test_split",

                "test_size":
                    0.20,

                "shuffle":
                    True,

                "random_state":
                    42,

                "stratify":
                    "BINARY_LABEL",

                "train_order":
                    "EXACT_OPERATOR_RETURN_ORDER",

                "validation_order":
                    "EXACT_OPERATOR_RETURN_ORDER",
            },

            "primary_target": (
                "SHARED_RANDOM_TARGET_BENIGN_PLUS_"
                "ALL_HELD_OUT_FAMILY_ATTACKS"
            ),

            "primary_target_order":
                "CANONICAL_GLOBAL_ROW_ID_ASCENDING",
        },

        "counts": {

            "all_benign":
                expected_benign,

            "target_benign":
                target_benign_count,

            "benign_development":
                benign_dev_count,

            "known_attack_development":
                expected_known_attack,

            "development_rows":
                int(
                    len(
                        development_idx
                    )
                ),

            "train_rows":
                int(
                    len(
                        train_idx
                    )
                ),

            "train_benign":
                train_benign,

            "train_attack":
                train_attack,

            "validation_rows":
                int(
                    len(
                        validation_idx
                    )
                ),

            "validation_benign":
                validation_benign,

            "validation_attack":
                validation_attack,

            "primary_target_rows":
                int(
                    len(
                        primary_target_idx
                    )
                ),

            "primary_target_benign":
                target_benign,

            "primary_target_heldout_attack":
                target_attack,

            "eligible_population_rows":
                EXPECTED_ELIGIBLE_ROWS,

            "excluded_target_only_unseen":
                expected_target_only,

            "excluded_other_unseen":
                expected_other_unseen,
        },

        "leakage_audit": {

            "heldout_train":
                heldout_train,

            "heldout_validation":
                heldout_validation,

            "heldout_primary_target":
                heldout_target,

            "known_attack_primary_target":
                target_known_attack,

            "target_only_unseen_train":
                target_only_train,

            "target_only_unseen_validation":
                target_only_validation,

            "target_only_unseen_primary_target":
                target_only_target,

            "other_unseen_train":
                other_unseen_train,

            "other_unseen_validation":
                other_unseen_validation,

            "other_unseen_primary_target":
                other_unseen_target,

            "train_validation_disjoint":
                True,

            "train_target_disjoint":
                True,

            "validation_target_disjoint":
                True,

            "eligible_population_exactly_once":
                True,

            "status":
                "PASS",
        },

        "class_weight": {

            "formula":
                "train_benign / train_attack",

            "benign_weight":
                1.0,

            "attack_weight":
                positive_class_weight,

            "train_benign":
                train_benign,

            "train_attack":
                train_attack,

            "validation_rows_used":
                0,

            "target_rows_used":
                0,

            "fixed_across_model_seeds":
                True,
        },

        "hashes":
            hashes,

        "runtime_paths":
            runtime_paths,

        "runtime_artifacts_committed_to_git":
            False,

        "scientific_operations": {
            "predictor_values_read":
                0,

            "model_fits":
                0,

            "model_inference":
                0,

            "threshold_selection":
                0,

            "target_predictor_openings":
                0,
        },

        "status":
            "FROZEN_BEFORE_FIRST_MODEL_FIT",
    }


    fold_receipts[
        family
    ] = receipt


    write_json(
        OUTPUT_DIR
        / (
            f"random_loao_{family}_"
            "membership_receipt.json"
        ),
        receipt,
    )


    print(
        "Development:"
    )

    print(
        "  rows         :",
        f"{len(development_idx):,}",
    )

    print(
        "  benign       :",
        f"{dev_benign:,}",
    )

    print(
        "  known attack :",
        f"{dev_attack:,}",
    )

    print()

    print(
        "TRAIN:"
    )

    print(
        "  rows         :",
        f"{len(train_idx):,}",
    )

    print(
        "  benign       :",
        f"{train_benign:,}",
    )

    print(
        "  attack       :",
        f"{train_attack:,}",
    )

    print()

    print(
        "VALIDATION:"
    )

    print(
        "  rows         :",
        f"{len(validation_idx):,}",
    )

    print(
        "  benign       :",
        f"{validation_benign:,}",
    )

    print(
        "  known attack :",
        f"{validation_attack:,}",
    )

    print()

    print(
        "PRIMARY TARGET:"
    )

    print(
        "  rows         :",
        f"{len(primary_target_idx):,}",
    )

    print(
        "  benign       :",
        f"{target_benign:,}",
    )

    print(
        "  held-out     :",
        f"{target_attack:,}",
    )

    print(
        "  known attack :",
        target_known_attack,
    )

    print()

    print(
        "Positive class weight:",
        positive_class_weight,
    )

    print()

    print(
        "TRAIN SHA      :",
        hashes[
            "train"
        ][
            "ordered_global_idx_content_sha256"
        ],
    )

    print(
        "VALIDATION SHA :",
        hashes[
            "validation"
        ][
            "ordered_global_idx_content_sha256"
        ],
    )

    print(
        "TARGET SHA     :",
        hashes[
            "primary_target"
        ][
            "ordered_global_idx_content_sha256"
        ],
    )

    print()

    print(
        f"[PASS] {family}: frozen with zero leakage"
    )


# =================================================================================================
# 8. CROSS-FAMILY SHARED TARGET-BENIGN AUDIT
# =================================================================================================

banner(
    "CROSS-FAMILY SHARED TARGET-BENIGN AUDIT"
)

for family in ELIGIBLE_FAMILIES:

    observed = (
        fold_receipts[
            family
        ][
            "counts"
        ][
            "primary_target_benign"
        ]
    )

    if observed != target_benign_count:
        raise RuntimeError(
            f"{family}: target-benign population differs."
        )

print(
    "[PASS] identical target-benign membership "
    "used for all 5 random-LOAO folds"
)

print(
    "Rows  :",
    f"{target_benign_count:,}",
)

print(
    "SHA256:",
    target_benign_bitset_sha,
)


# =================================================================================================
# 9. SHARED TARGET-BENIGN RECEIPT
# =================================================================================================

write_json(
    OUTPUT_DIR
    / "shared_target_benign_membership_receipt.json",
    {

        "stage":
            "Stage28-1B",

        "type":
            "SHARED_RANDOM_TARGET_BENIGN_MEMBERSHIP_RECEIPT",

        "created_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "scientific_parent_commit":
            EXPECTED_PARENT,

        "operator":
            "sklearn.model_selection.train_test_split",

        "input_order":
            "ALL_BENIGN_GLOBAL_IDS_CANONICAL_ASCENDING",

        "test_size":
            0.20,

        "random_state":
            42,

        "shuffle":
            True,

        "stratify":
            None,

        "all_benign_rows":
            expected_benign,

        "target_benign_rows":
            target_benign_count,

        "benign_development_rows":
            benign_dev_count,

        "operator_return_order_content_sha256":
            target_benign_return_order_sha,

        "canonical_target_idx_content_sha256":
            target_benign_canonical_sha,

        "membership_bitset_sha256":
            target_benign_bitset_sha,

        "scikit_learn_version":
            sklearn.__version__,

        "numpy_version":
            np.__version__,

        "runtime_path":
            str(
                shared_target_runtime_path
            ),

        "runtime_artifact_committed_to_git":
            False,

        "shared_across_all_families":
            True,

        "membership_varies_across_model_seeds":
            False,

        "status":
            "FROZEN_BEFORE_FIRST_MODEL_FIT",
    },
)


# =================================================================================================
# 10. FREEZE PARAMETER SETS
# =================================================================================================

banner(
    "FREEZE MODEL PARAMETER SETS"
)

stage22_spec = (
    model_inventory[
        "stage22_full"
    ]
)

loao_spec = (
    model_inventory[
        "loao"
    ]
)

parameter_sets = {}


for context, spec in [
    (
        "STAGE22_FULL",
        stage22_spec,
    ),
    (
        "LOAO",
        loao_spec,
    ),
]:

    for learner, model_key in [
        (
            "XGBOOST",
            "xgboost",
        ),
        (
            "LIGHTGBM",
            "lightgbm",
        ),
    ]:

        config_id = (
            spec[
                model_key
            ][
                "configuration_id"
            ]
        )

        base = (
            spec[
                model_key
            ][
                "base_cpu_parameters_seed42"
            ]
        )

        for seed in MODEL_SEEDS:

            params = parameter_set(
                base,
                seed,
            )

            parameter_id = (
                f"{context}::{learner}::SEED{seed}"
            )

            parameter_sets[
                parameter_id
            ] = {

                "context":
                    context,

                "learner":
                    learner,

                "configuration_id":
                    config_id,

                "seed":
                    seed,

                "parameters":
                    params,

                "canonical_json_sha256":
                    canonical_json_sha(
                        params
                    ),

                "allowed_change_from_seed42":
                    "random_state ONLY",
            }


# Hard drift audit.
for context in [
    "STAGE22_FULL",
    "LOAO",
]:

    for learner in LEARNERS:

        reference = dict(
            parameter_sets[
                f"{context}::{learner}::SEED42"
            ][
                "parameters"
            ]
        )

        reference.pop(
            "random_state"
        )

        for seed in MODEL_SEEDS:

            candidate = dict(
                parameter_sets[
                    f"{context}::{learner}::SEED{seed}"
                ][
                    "parameters"
                ]
            )

            observed_seed = candidate.pop(
                "random_state"
            )

            if observed_seed != seed:
                raise RuntimeError(
                    "random_state drift."
                )

            if candidate != reference:
                raise RuntimeError(
                    f"Parameter drift beyond seed: "
                    f"{context}/{learner}/seed{seed}"
                )


write_json(
    OUTPUT_DIR
    / "execution_parameter_sets.json",
    {

        "stage":
            "Stage28-1B",

        "scientific_parent_commit":
            EXPECTED_PARENT,

        "primary_compute_backend":
            "CPU",

        "sets":
            parameter_sets,

        "prohibited":
            [
                "HYPERPARAMETER_SEARCH",
                "EARLY_STOPPING_ADDITION",
                "SEED_SPECIFIC_TUNING",
                "BACKEND_CHANGE",
                "FEATURE_CHANGE",
            ],

        "status":
            "FROZEN_BEFORE_FIRST_MODEL_FIT",
    },
)

print(
    "[PASS] parameter sets frozen"
)

print(
    "[PASS] only random_state varies across seeds"
)


# =================================================================================================
# 11. VERIFY 12 REUSABLE MODEL COMPONENTS
# =================================================================================================

banner(
    "VERIFY 12 REUSABLE MODEL COMPONENTS"
)

reusable_models = {}


# --------------------------------------------------------------------------------------------------
# 11A. Stage22 seed42 CPU LightGBM only.
# --------------------------------------------------------------------------------------------------

stage22_result_specs = {

    "RANDOM_NATURAL": {
        "result":
            (
                REPO
                / "results"
                / "stage22r_training"
                / "stage22r_2a_random_natural"
                / "stage22r_2a_random_natural_result.json"
            ),

        "model":
            (
                REPO
                / "results"
                / "stage22r_training"
                / "stage22r_2a_random_natural"
                / "random_natural_lightgbm_model.txt"
            ),

        "artifact_key":
            "random_natural_lightgbm_model.txt",
    },

    "CHRONOLOGICAL_NATURAL": {
        "result":
            (
                REPO
                / "results"
                / "stage22r_training"
                / "stage22r_2c_chronological_natural"
                / "stage22r_2c_chronological_natural_result.json"
            ),

        "model":
            (
                REPO
                / "results"
                / "stage22r_training"
                / "stage22r_2c_chronological_natural"
                / "chronological_natural_lightgbm_model.txt"
            ),

        "artifact_key":
            "chronological_natural_lightgbm_model.txt",
    },
}


for cell, spec in (
    stage22_result_specs.items()
):

    result_path = require_file(
        spec[
            "result"
        ]
    )

    model_path = require_file(
        spec[
            "model"
        ]
    )

    result = read_json(
        result_path
    )

    expected_sha = (
        result[
            "artifacts"
        ][
            "hashes_before_result_json"
        ][
            spec[
                "artifact_key"
            ]
        ]
    )

    actual_sha = sha256_file(
        model_path
    )

    backend = (
        result[
            "models"
        ][
            "lightgbm"
        ][
            "executed_parameters"
        ][
            "device_type"
        ]
    )

    model_seed = (
        result[
            "models"
        ][
            "lightgbm"
        ][
            "executed_parameters"
        ][
            "random_state"
        ]
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"{cell}: reusable Stage22 "
            "LightGBM SHA mismatch."
        )

    if backend != "cpu":
        raise RuntimeError(
            f"{cell}: reusable LightGBM "
            "is not CPU."
        )

    if model_seed != 42:
        raise RuntimeError(
            f"{cell}: reusable LightGBM "
            "is not seed42."
        )


    # Also hard-check that historical XGB was CUDA,
    # therefore NOT reused in Stage28 seed stability.
    xgb_backend = (
        result[
            "models"
        ][
            "xgboost"
        ][
            "backend"
        ]
    )

    if xgb_backend != "cuda":
        raise RuntimeError(
            f"{cell}: historical Stage22 XGB "
            "backend no longer matches frozen evidence."
        )


    reuse_key = (
        f"STAGE22::{cell}::LIGHTGBM::SEED42"
    )

    reusable_models[
        reuse_key
    ] = {

        "path":
            str(
                model_path.relative_to(
                    REPO
                )
            ),

        "sha256":
            actual_sha,

        "backend":
            "cpu",

        "seed":
            42,

        "configuration_id":
            "LGBM_11",
    }

    print(
        "[PASS]",
        reuse_key,
        actual_sha,
    )


# --------------------------------------------------------------------------------------------------
# 11B. Stage27 seed42: five families × two CPU learners.
# --------------------------------------------------------------------------------------------------

stage27_model_manifest_path = require_file(
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_2a_preopening_models"
    / "model_artifact_manifest.json"
)

stage27_model_manifest = read_json(
    stage27_model_manifest_path
)

stage27_models = (
    stage27_model_manifest[
        "models"
    ]
)


for family in ELIGIBLE_FAMILIES:

    for learner in LEARNERS:

        source_key = (
            f"{family}::{learner}"
        )

        if (
            source_key
            not in stage27_models
        ):
            raise RuntimeError(
                f"Missing Stage27 reusable model: "
                f"{source_key}"
            )

        source = (
            stage27_models[
                source_key
            ]
        )

        model_path = require_file(
            REPO
            / source[
                "durable_path"
            ]
        )

        actual_sha = sha256_file(
            model_path
        )

        expected_sha = (
            source[
                "sha256"
            ]
        )

        if actual_sha != expected_sha:
            raise RuntimeError(
                f"{source_key}: reusable "
                "Stage27 model SHA mismatch."
            )

        expected_config = (
            "XGB_11"
            if learner == "XGBOOST"
            else "LGBM_11"
        )

        if (
            source[
                "configuration_id"
            ]
            != expected_config
        ):
            raise RuntimeError(
                f"{source_key}: configuration mismatch."
            )


        # Cross-check reused Stage27 class weight against
        # Stage28-1A reconstruction receipt.
        chronology_receipt = read_json(
            require_file(
                STAGE28_1A_DIR
                / (
                    f"fold_{family}_"
                    "reconstruction_receipt.json"
                )
            )
        )

        expected_weight = float(
            chronology_receipt[
                "class_weight"
            ][
                "recomputed_value"
            ]
        )

        source_weight = float(
            source[
                "positive_class_weight"
            ]
        )

        if not math.isclose(
            source_weight,
            expected_weight,
            rel_tol=0.0,
            abs_tol=1e-15,
        ):
            raise RuntimeError(
                f"{source_key}: Stage27 reused "
                "class weight mismatch."
            )


        reuse_key = (
            f"STAGE27::{family}::{learner}::SEED42"
        )

        reusable_models[
            reuse_key
        ] = {

            "path":
                source[
                    "durable_path"
                ],

            "sha256":
                actual_sha,

            "backend":
                "cpu",

            "seed":
                42,

            "configuration_id":
                source[
                    "configuration_id"
                ],

            "positive_class_weight":
                source_weight,
        }

        print(
            "[PASS]",
            reuse_key,
            actual_sha,
        )


if len(
    reusable_models
) != 12:
    raise RuntimeError(
        "Reusable component count != 12.\n"
        f"actual={len(reusable_models)}"
    )

print()
print(
    "[PASS] exactly 12 reusable model "
    "components verified"
)


# =================================================================================================
# 12. BUILD COMPLETE 120-COMPONENT EXECUTION MANIFEST
# =================================================================================================

banner(
    "BUILD 120-COMPONENT EXECUTION MANIFEST"
)

manifest_rows = []

component_ordinal = 0


def append_component(
    *,
    arm,
    experiment,
    unit,
    evaluation_cell_id,
    learner,
    seed,
    parameter_context,
    membership_reference,
    fit_action,
    class_weight=None,
    reuse_key=None,
):
    global component_ordinal

    component_ordinal += 1

    parameter_id = (
        f"{parameter_context}::{learner}::SEED{seed}"
    )

    pset = (
        parameter_sets[
            parameter_id
        ]
    )

    row = {

        "component_ordinal":
            component_ordinal,

        "component_id":
            f"C{component_ordinal:03d}",

        "arm":
            arm,

        "experiment":
            experiment,

        "unit":
            unit,

        "evaluation_cell_id":
            evaluation_cell_id,

        "learner":
            learner,

        "configuration_id":
            pset[
                "configuration_id"
            ],

        "model_seed":
            seed,

        "compute_backend":
            "CPU",

        "parameter_set_id":
            parameter_id,

        "parameter_sha256":
            pset[
                "canonical_json_sha256"
            ],

        "membership_reference":
            membership_reference,

        "positive_class_weight":
            (
                ""
                if class_weight is None
                else repr(
                    float(
                        class_weight
                    )
                )
            ),

        "fit_action":
            fit_action,

        "reused_model_path":
            "",

        "reused_model_sha256":
            "",

        "new_fit_budget_units":
            (
                1
                if fit_action
                == "NEW_FIT_AUTHORIZED"
                else 0
            ),

        "reuse_budget_units":
            (
                1
                if fit_action
                == "REUSE_EXISTING"
                else 0
            ),
    }


    if fit_action == "REUSE_EXISTING":

        if reuse_key is None:
            raise RuntimeError(
                "Reuse component missing reuse_key."
            )

        source = (
            reusable_models[
                reuse_key
            ]
        )

        row[
            "reused_model_path"
        ] = source[
            "path"
        ]

        row[
            "reused_model_sha256"
        ] = source[
            "sha256"
        ]


    manifest_rows.append(
        row
    )


# --------------------------------------------------------------------------------------------------
# A. Stage22 FULL
# 2 cells × 5 seeds × 2 learners = 20 component realizations.
# Ensemble evaluation = 2 cells × 5 seeds = 10 evaluation cells.
# --------------------------------------------------------------------------------------------------

for cell in STAGE22_CELLS:

    for seed in MODEL_SEEDS:

        evaluation_cell_id = (
            f"28A_STAGE22::{cell}::SEED{seed}"
        )

        for learner in LEARNERS:

            if (
                seed == 42
                and learner
                == "LIGHTGBM"
            ):

                fit_action = (
                    "REUSE_EXISTING"
                )

                reuse_key = (
                    f"STAGE22::{cell}::LIGHTGBM::SEED42"
                )

            else:

                fit_action = (
                    "NEW_FIT_AUTHORIZED"
                )

                reuse_key = None


            append_component(

                arm=
                    "28A_TRAINING_SEED_STABILITY",

                experiment=
                    "STAGE22_FULL",

                unit=
                    cell,

                evaluation_cell_id=
                    evaluation_cell_id,

                learner=
                    learner,

                seed=
                    seed,

                parameter_context=
                    "STAGE22_FULL",

                membership_reference=(
                    "STAGE22R_FROZEN_"
                    f"{cell}_MEMBERSHIP"
                ),

                fit_action=
                    fit_action,

                class_weight=
                    None,

                reuse_key=
                    reuse_key,
            )


# --------------------------------------------------------------------------------------------------
# B. Stage27 chronology-first LOAO
# 5 × 5 × 2 = 50 components/evaluation cells.
# Seed42 = ten exact existing reusable models.
# --------------------------------------------------------------------------------------------------

for family in ELIGIBLE_FAMILIES:

    chronology_receipt_rel = (
        "results/"
        "stage28_stability_novelty_control/"
        "stage28_1a_inherited_membership_reconstruction/"
        f"fold_{family}_reconstruction_receipt.json"
    )

    chronology_receipt = read_json(
        REPO
        / chronology_receipt_rel
    )

    chronology_weight = float(
        chronology_receipt[
            "class_weight"
        ][
            "recomputed_value"
        ]
    )


    for seed in MODEL_SEEDS:

        for learner in LEARNERS:

            evaluation_cell_id = (
                f"28A_STAGE27_CHRONO::"
                f"{family}::{learner}::SEED{seed}"
            )

            if seed == 42:

                fit_action = (
                    "REUSE_EXISTING"
                )

                reuse_key = (
                    f"STAGE27::{family}::{learner}::SEED42"
                )

            else:

                fit_action = (
                    "NEW_FIT_AUTHORIZED"
                )

                reuse_key = None


            append_component(

                arm=
                    "28A_TRAINING_SEED_STABILITY",

                experiment=
                    "STAGE27_CHRONOLOGY_LOAO",

                unit=
                    family,

                evaluation_cell_id=
                    evaluation_cell_id,

                learner=
                    learner,

                seed=
                    seed,

                parameter_context=
                    "LOAO",

                membership_reference=
                    chronology_receipt_rel,

                fit_action=
                    fit_action,

                class_weight=
                    chronology_weight,

                reuse_key=
                    reuse_key,
            )


# --------------------------------------------------------------------------------------------------
# C. Stage28B random LOAO
# 5 × 5 × 2 = 50 new component/evaluation cells.
# --------------------------------------------------------------------------------------------------

for family in ELIGIBLE_FAMILIES:

    random_receipt_rel = (
        "results/"
        "stage28_stability_novelty_control/"
        "stage28_1b_random_loao_membership_and_execution_lock/"
        f"random_loao_{family}_membership_receipt.json"
    )

    random_weight = (
        fold_class_weights[
            family
        ]
    )


    for seed in MODEL_SEEDS:

        for learner in LEARNERS:

            evaluation_cell_id = (
                f"28B_RANDOM_LOAO::"
                f"{family}::{learner}::SEED{seed}"
            )

            append_component(

                arm=
                    "28B_RANDOM_SPLIT_LOAO_CONTROL",

                experiment=
                    "STAGE28B_RANDOM_LOAO",

                unit=
                    family,

                evaluation_cell_id=
                    evaluation_cell_id,

                learner=
                    learner,

                seed=
                    seed,

                parameter_context=
                    "LOAO",

                membership_reference=
                    random_receipt_rel,

                fit_action=
                    "NEW_FIT_AUTHORIZED",

                class_weight=
                    random_weight,

                reuse_key=
                    None,
            )


# =================================================================================================
# 13. HARD MANIFEST ACCOUNTING
# =================================================================================================

if len(
    manifest_rows
) != 120:
    raise RuntimeError(
        f"Manifest component rows != 120: "
        f"{len(manifest_rows)}"
    )


new_fit_count = sum(
    row[
        "new_fit_budget_units"
    ]
    for row
    in manifest_rows
)

reuse_count = sum(
    row[
        "reuse_budget_units"
    ]
    for row
    in manifest_rows
)

evaluation_cells = sorted(
    set(
        row[
            "evaluation_cell_id"
        ]
        for row
        in manifest_rows
    )
)


if new_fit_count != 108:
    raise RuntimeError(
        f"New fit budget != 108: "
        f"{new_fit_count}"
    )

if reuse_count != 12:
    raise RuntimeError(
        f"Reuse count != 12: "
        f"{reuse_count}"
    )

if len(
    evaluation_cells
) != 110:
    raise RuntimeError(
        f"Evaluation cell count != 110: "
        f"{len(evaluation_cells)}"
    )


def count_manifest(
    experiment,
    fit_action=None,
):
    rows = [
        x
        for x
        in manifest_rows
        if x[
            "experiment"
        ]
        == experiment
    ]

    if fit_action is not None:
        rows = [
            x
            for x
            in rows
            if x[
                "fit_action"
            ]
            == fit_action
        ]

    return len(
        rows
    )


assert count_manifest(
    "STAGE22_FULL"
) == 20

assert count_manifest(
    "STAGE22_FULL",
    "REUSE_EXISTING",
) == 2

assert count_manifest(
    "STAGE22_FULL",
    "NEW_FIT_AUTHORIZED",
) == 18


assert count_manifest(
    "STAGE27_CHRONOLOGY_LOAO"
) == 50

assert count_manifest(
    "STAGE27_CHRONOLOGY_LOAO",
    "REUSE_EXISTING",
) == 10

assert count_manifest(
    "STAGE27_CHRONOLOGY_LOAO",
    "NEW_FIT_AUTHORIZED",
) == 40


assert count_manifest(
    "STAGE28B_RANDOM_LOAO"
) == 50

assert count_manifest(
    "STAGE28B_RANDOM_LOAO",
    "NEW_FIT_AUTHORIZED",
) == 50


print(
    "[PASS] component realizations = 120"
)

print(
    "[PASS] evaluation cells = 110"
)

print(
    "[PASS] reused components = 12"
)

print(
    "[PASS] NEW authorized fits = 108"
)

print(
    "[PASS] Stage22 FULL = 2 reused + 18 new"
)

print(
    "[PASS] Stage27 chronology = 10 reused + 40 new"
)

print(
    "[PASS] Stage28B random = 50 new"
)


# =================================================================================================
# 14. WRITE COMPONENT EXECUTION MANIFEST
# =================================================================================================

manifest_path = (
    OUTPUT_DIR
    / "stage28_component_execution_manifest.csv"
)

manifest_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

fieldnames = list(
    manifest_rows[
        0
    ].keys()
)

with manifest_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames,
        lineterminator="\n",
    )

    writer.writeheader()

    writer.writerows(
        manifest_rows
    )


write_json(
    OUTPUT_DIR
    / "stage28_component_execution_manifest_receipt.json",
    {

        "stage":
            "Stage28-1B",

        "type":
            "COMPLETE_COMPONENT_EXECUTION_MANIFEST_RECEIPT",

        "created_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "scientific_parent_commit":
            EXPECTED_PARENT,

        "manifest": {
            "path":
                str(
                    manifest_path.relative_to(
                        REPO
                    )
                ),

            "rows":
                120,

            "sha256":
                sha256_file(
                    manifest_path
                ),

            "component_realizations":
                120,

            "scientific_evaluation_cells":
                110,
        },

        "budget": {
            "existing_reused":
                12,

            "new_fit_budget":
                108,

            "new_fits_consumed_before_freeze":
                0,

            "new_fits_remaining":
                108,

            "hard_limit":
                108,
        },

        "breakdown": {

            "stage22_full": {
                "components":
                    20,

                "reused":
                    2,

                "new":
                    18,

                "scientific_evaluation_cells":
                    10,
            },

            "stage27_chronology_loao": {
                "components":
                    50,

                "reused":
                    10,

                "new":
                    40,

                "scientific_evaluation_cells":
                    50,
            },

            "stage28b_random_loao": {
                "components":
                    50,

                "reused":
                    0,

                "new":
                    50,

                "scientific_evaluation_cells":
                    50,
            },
        },

        "seed_policy": {
            "model_seeds":
                MODEL_SEEDS,

            "membership_seed":
                MEMBERSHIP_SEED,

            "random_loao_membership_fixed_across_model_seeds":
                True,

            "only_model_parameter_changed_across_seeds":
                "random_state",
        },

        "compute": {
            "primary_backend":
                "CPU",

            "gpu_authorized":
                False,
        },

        "status":
            "FROZEN_BEFORE_FIRST_MODEL_FIT",
    },
)


# =================================================================================================
# 15. RANDOM-LOAO SUMMARY
# =================================================================================================

write_json(
    OUTPUT_DIR
    / "stage28_random_loao_membership_summary.json",
    {

        "stage":
            "Stage28-1B",

        "arm":
            "28B_RANDOM_SPLIT_LOAO_CONTROL",

        "created_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "scientific_parent_commit":
            EXPECTED_PARENT,

        "membership_seed":
            42,

        "model_seeds":
            MODEL_SEEDS,

        "membership_fixed_across_model_seeds":
            True,

        "shared_target_benign": {
            "rows":
                target_benign_count,

            "all_benign":
                expected_benign,

            "operator_return_order_sha256":
                target_benign_return_order_sha,

            "canonical_idx_sha256":
                target_benign_canonical_sha,

            "membership_bitset_sha256":
                target_benign_bitset_sha,
        },

        "folds": {
            family: {

                "train_rows":
                    fold_receipts[
                        family
                    ][
                        "counts"
                    ][
                        "train_rows"
                    ],

                "validation_rows":
                    fold_receipts[
                        family
                    ][
                        "counts"
                    ][
                        "validation_rows"
                    ],

                "target_rows":
                    fold_receipts[
                        family
                    ][
                        "counts"
                    ][
                        "primary_target_rows"
                    ],

                "target_positive":
                    fold_receipts[
                        family
                    ][
                        "counts"
                    ][
                        "primary_target_heldout_attack"
                    ],

                "positive_class_weight":
                    fold_class_weights[
                        family
                    ],

                "train_ordered_sha256":
                    fold_receipts[
                        family
                    ][
                        "hashes"
                    ][
                        "train"
                    ][
                        "ordered_global_idx_content_sha256"
                    ],

                "validation_ordered_sha256":
                    fold_receipts[
                        family
                    ][
                        "hashes"
                    ][
                        "validation"
                    ][
                        "ordered_global_idx_content_sha256"
                    ],

                "target_ordered_sha256":
                    fold_receipts[
                        family
                    ][
                        "hashes"
                    ][
                        "primary_target"
                    ][
                        "ordered_global_idx_content_sha256"
                    ],

                "leakage_status":
                    "PASS",
            }

            for family
            in ELIGIBLE_FAMILIES
        },

        "interpretation_limit": (
            "Random-evaluation control only; "
            "not a deployment-realistic estimate."
        ),

        "status":
            "FROZEN_BEFORE_FIRST_MODEL_FIT",
    },
)


# =================================================================================================
# 16. STAGE28-1B FREEZE RECORD
# =================================================================================================

write_json(
    OUTPUT_DIR
    / "stage28_1b_freeze_record.json",
    {

        "stage":
            "Stage28-1B",

        "type":
            "RANDOM_LOAO_AND_EXECUTION_PRETRAINING_FREEZE",

        "created_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "parent_commit":
            EXPECTED_PARENT,

        "status":
            "FROZEN_BEFORE_FIRST_STAGE28_MODEL_FIT",

        "random_loao": {

            "families":
                ELIGIBLE_FAMILIES,

            "membership_seed":
                42,

            "model_seeds":
                MODEL_SEEDS,

            "memberships_fixed_across_model_seeds":
                True,

            "shared_target_benign_rows":
                target_benign_count,

            "heldout_train_count":
                0,

            "heldout_validation_count":
                0,

            "known_attack_primary_target_count":
                0,

            "target_only_unseen_assigned_count":
                0,

            "other_unseen_assigned_count":
                0,

            "eligible_population_rows_per_fold":
                EXPECTED_ELIGIBLE_ROWS,

            "leakage_audit":
                "PASS_ALL_FIVE_FOLDS",
        },

        "execution_manifest": {

            "component_rows":
                120,

            "scientific_evaluation_cells":
                110,

            "existing_reused":
                12,

            "new_fit_budget":
                108,

            "new_fits_consumed":
                0,

            "new_fits_remaining":
                108,

            "compute":
                "CPU",
        },

        "scientific_operations": {

            "predictor_values_read":
                0,

            "model_fits":
                0,

            "model_inference":
                0,

            "threshold_selection":
                0,

            "target_predictor_openings":
                0,
        },

        "runtime_membership_cache": {

            "path":
                str(
                    RANDOM_RUNTIME_ROOT
                ),

            "committed_to_git":
                False,

            "recovery_rule": (
                "Reconstruct from the exact Stage27 "
                "global-family content identity and frozen "
                "Stage28B sklearn split operators with "
                "membership seed 42; require the membership "
                "hashes frozen by Stage28-1B."
            ),
        },

        "next_authorized_step": (
            "Stage28-1C — recover/materialize and verify "
            "the frozen 70-feature model inputs required "
            "by the execution manifest, still before "
            "the first Stage28 model fit."
        ),
    },
)


# =================================================================================================
# 17. CHECKSUM MANIFEST
# =================================================================================================

artifact_candidates = sorted(
    [
        p
        for p
        in OUTPUT_DIR.iterdir()
        if p.is_file()
        and p.suffix.lower()
        in {
            ".json",
            ".csv",
        }
    ]
)

checksums_path = (
    OUTPUT_DIR
    / "checksums.sha256"
)

checksums_path.write_text(
    "\n".join(
        f"{sha256_file(p)}  {p.name}"
        for p
        in artifact_candidates
    )
    + "\n",
    encoding="utf-8",
)


# =================================================================================================
# 18. PRE-COMMIT HARD MACHINE AUDIT
# =================================================================================================

banner(
    "STAGE28-1B PRE-COMMIT MACHINE AUDIT"
)

if len(
    fold_receipts
) != 5:
    raise RuntimeError(
        "Random-LOAO fold count != 5."
    )


for family in ELIGIBLE_FAMILIES:

    leak = (
        fold_receipts[
            family
        ][
            "leakage_audit"
        ]
    )

    required_zero = [
        "heldout_train",
        "heldout_validation",
        "known_attack_primary_target",
        "target_only_unseen_train",
        "target_only_unseen_validation",
        "target_only_unseen_primary_target",
        "other_unseen_train",
        "other_unseen_validation",
        "other_unseen_primary_target",
    ]

    for field in required_zero:

        if leak[
            field
        ] != 0:
            raise RuntimeError(
                f"{family}: {field} != 0"
            )

    if (
        leak[
            "eligible_population_exactly_once"
        ]
        is not True
    ):
        raise RuntimeError(
            f"{family}: eligible population "
            "coverage failure."
        )


print(
    "[PASS] 5 random-LOAO memberships frozen"
)

print(
    "[PASS] membership seed = 42"
)

print(
    "[PASS] memberships fixed across seeds 42..46"
)

print(
    "[PASS] shared target benign = 454,620"
)

print(
    "[PASS] held-out TRAIN = 0 for 5/5"
)

print(
    "[PASS] held-out VALIDATION = 0 for 5/5"
)

print(
    "[PASS] known attacks PRIMARY TARGET = 0 for 5/5"
)

print(
    "[PASS] TARGET_ONLY_UNSEEN assigned = 0"
)

print(
    "[PASS] OTHER_ATTACK_UNSEEN_LABEL assigned = 0"
)

print(
    "[PASS] eligible population exact coverage"
)

print(
    "[PASS] class weights frozen from TRAIN only"
)

print(
    "[PASS] 12 reusable models verified"
)

print(
    "[PASS] 120 component realizations frozen"
)

print(
    "[PASS] 110 scientific evaluation cells frozen"
)

print(
    "[PASS] 108 NEW fits authorized"
)

print(
    "[PASS] NEW fits consumed = 0"
)

print(
    "[PASS] predictor values read = 0"
)

print(
    "[PASS] model inference = 0"
)

print(
    "[PASS] threshold selection = 0"
)

print(
    "[PASS] target predictor openings = 0"
)


# =================================================================================================
# 19. DISPLAY DURABLE ARTIFACTS
# =================================================================================================

banner(
    "STAGE28-1B DURABLE ARTIFACTS"
)

for path in sorted(
    OUTPUT_DIR.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:66s} "
            f"{path.stat().st_size:10,d} bytes  "
            f"{sha256_file(path)}"
        )


# =================================================================================================
# 20. GIT DIFF AUDIT
# =================================================================================================

banner(
    "GIT DIFF AUDIT"
)

status = git(
    "status",
    "--porcelain",
)

print(
    status
)

expected_prefix = (
    "results/"
    "stage28_stability_novelty_control/"
    "stage28_1b_random_loao_membership_and_execution_lock/"
)

unexpected = []

for line in (
    status.splitlines()
):

    if not line.strip():
        continue

    path = (
        line[3:]
        .strip()
    )

    if not path.startswith(
        expected_prefix
    ):
        unexpected.append(
            line
        )


if unexpected:
    raise RuntimeError(
        "Unexpected repository modifications:\n"
        + "\n".join(
            unexpected
        )
    )

print()
print(
    "[PASS] only Stage28-1B durable artifacts modified"
)


# =================================================================================================
# 21. EXACT STAGED FILE SET
# =================================================================================================

files_to_stage = sorted(
    p.relative_to(
        REPO
    ).as_posix()

    for p
    in OUTPUT_DIR.iterdir()

    if p.is_file()
)


for relative_path in (
    files_to_stage
):

    git(
        "add",
        "--",
        relative_path,
    )


staged = git(
    "diff",
    "--cached",
    "--name-only",
).splitlines()


if sorted(
    staged
) != sorted(
    files_to_stage
):
    raise RuntimeError(
        "Stage28-1B staged file set mismatch."
    )


git(
    "diff",
    "--cached",
    "--check",
)


print()
print(
    "[PASS] exact Stage28-1B file set staged"
)

for path in staged:
    print(
        " ",
        path,
    )


# =================================================================================================
# 22. COMMIT
# =================================================================================================

banner(
    "COMMIT STAGE28-1B"
)

git(
    "commit",
    "-m",
    COMMIT_MESSAGE,
)

commit_sha = git(
    "rev-parse",
    "HEAD",
)

commit_parent = git(
    "rev-parse",
    f"{commit_sha}^",
)


if commit_parent != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage28-1B commit parent mismatch.\n"
        f"expected={EXPECTED_PARENT}\n"
        f"actual={commit_parent}"
    )


print(
    "Stage28-1B commit:",
    commit_sha,
)

print(
    "Parent            :",
    commit_parent,
)


# =================================================================================================
# 23. PUSH
# =================================================================================================

banner(
    "PUSH STAGE28-1B"
)

push = run(
    [
        "git",
        "push",
        "origin",
        "main:main",
    ],
    check=False,
)


if push.returncode != 0:
    raise RuntimeError(
        "Stage28-1B push failed.\n\n"
        f"STDOUT:\n{push.stdout}\n\n"
        f"STDERR:\n{push.stderr}"
    )


print(
    push.stdout or ""
)

print(
    push.stderr or ""
)


# =================================================================================================
# 24. REMOTE VERIFICATION
# =================================================================================================

banner(
    "REMOTE VERIFICATION"
)

remote_after = (
    remote_main_sha()
)

print(
    "Local HEAD :",
    commit_sha,
)

print(
    "Remote main:",
    remote_after,
)


if (
    remote_after
    != commit_sha
):
    raise RuntimeError(
        "Remote verification failed."
    )


git(
    "fetch",
    "origin",
    "main",
)

origin_after = git(
    "rev-parse",
    "origin/main",
)


if (
    origin_after
    != commit_sha
):
    raise RuntimeError(
        "origin/main mismatch after push."
    )


final_status = git(
    "status",
    "--porcelain",
)


if final_status:
    raise RuntimeError(
        "Repository dirty after Stage28-1B:\n"
        + final_status
    )


# =================================================================================================
# 25. FINAL
# =================================================================================================

banner(
    "STAGE28-1B — COMPLETE / REMOTELY VERIFIED"
)

print(
    "Stage28-1B commit:"
)

print(
    " ",
    commit_sha,
)

print()

print(
    "Random-LOAO membership:"
)

print(
    "  membership seed       = 42"
)

print(
    "  target benign rows    = 454,620"
)

print(
    "  membership varies by model seed = NO"
)

print(
    "  eligible families     = 5"
)

print(
    "  held-out TRAIN        = 0 for 5/5"
)

print(
    "  held-out VALIDATION   = 0 for 5/5"
)

print(
    "  known attack TARGET   = 0 for 5/5"
)

print(
    "  TARGET_ONLY_UNSEEN    = EXCLUDED"
)

print(
    "  OTHER_UNSEEN          = EXCLUDED"
)

print()

print(
    "Execution universe:"
)

print(
    "  component realizations      = 120"
)

print(
    "  scientific evaluation cells = 110"
)

print(
    "  existing reused             = 12"
)

print(
    "  NEW authorized fits         = 108"
)

print(
    "  NEW fits consumed           = 0"
)

print(
    "  NEW fits remaining          = 108"
)

print()

print(
    "Compute:"
)

print(
    "  primary backend = CPU"
)

print(
    "  GPU authorized  = FALSE"
)

print()

print(
    "Scientific operations:"
)

print(
    "  PREDICTOR_VALUES_READ     = 0"
)

print(
    "  MODEL_FITS                = 0"
)

print(
    "  MODEL_INFERENCE           = 0"
)

print(
    "  THRESHOLD_SELECTION       = 0"
)

print(
    "  TARGET_PREDICTOR_OPENINGS = 0"
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage28-1C — recover/materialize and verify "
    "the frozen 70-feature model inputs required by "
    "the execution manifest, still BEFORE FIT #1."
)

print()
print(SEP)


STAGE28-1B — EXACT REPOSITORY GATE

Repository : /kaggle/working/ids2018-validation-safe-ablation
Branch     : main
Local HEAD : e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
Remote main: e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
Expected   : e7ad0283cbe1290a1a1b8ecf3c495ae142e6247b
Git clean  : True

[PASS] exact Stage28-1A durable parent
[PASS] local == remote
[PASS] clean worktree
[PASS] Stage28-1B destination absent

LOAD EFFECTIVE STAGE28 SPECIFICATIONS

[PASS] model seeds = [42,43,44,45,46]
[PASS] membership seed = 42
[PASS] memberships fixed across model seeds
[PASS] target benign fraction = 0.20
[PASS] development validation fraction = 0.20
[PASS] development attacks = primary-seven excluding held-out
[PASS] TARGET_ONLY_UNSEEN excluded
[PASS] OTHER_ATTACK_UNSEEN_LABEL excluded
[PASS] effective fit budget = 108 NEW / 12 REUSED
[PASS] 120 component realizations / 110 evaluation cells

VERIFY RECOVERED STAGE28-1A GLOBAL RUNTIME CACHE

global_family_codes
  current   : b0e9fb8b0eba74fdb

In [4]:
# =================================================================================================
# STAGE28-1C-A — MODEL-INPUT RECOVERY PREFLIGHT / ENVIRONMENT + SOURCE CONTRACT AUDIT
#
# Durable parent:
#   ba011ec01f1399939111b24664ebd5d66c630f95
#
# PURPOSE
# -------
# Verify everything needed before reconstructing model-input caches:
#
#   1. exact Stage28-1B durable parent / clean Git
#   2. 120-component manifest + 108-new-fit ledger
#   3. XGBoost / LightGBM / sklearn / pandas / pyarrow runtime
#   4. eight exact IDS2018 development source files are attached
#   5. frozen Stage22 K79 exclusion artifact identity + schema
#   6. frozen Stage22 70-feature cache specification
#   7. recovered CICIDS2017 eight-source runtime remains exact
#   8. Stage24 full-population 70-feature FLOAT64 source hashes exist for all 8 sources
#   9. Stage27 FLAG_CORRECTED representation / numeric policy
#  10. Stage22 and Stage27 ordered 70-feature lists are identical
#  11. five Stage28B runtime membership arrays still match their durable Stage28-1B hashes
#
# ZERO:
#   predictor values read
#   model fits
#   model inference
#   threshold selection
#   target predictor openings
#   Git modifications
#
# This is a PRE-FLIGHT ONLY. No durable Stage28-1C artifact is committed here.
# =================================================================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import platform
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

INPUT_ROOT = Path(
    "/kaggle/input"
).resolve()

EXPECTED_HEAD = (
    "ba011ec01f1399939111b24664ebd5d66c630f95"
)

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

STAGE28_1B_DIR = (
    STAGE28_ROOT
    / "stage28_1b_random_loao_membership_and_execution_lock"
)

STAGE28_RANDOM_RUNTIME = Path(
    "/kaggle/working/"
    "stage28_1b_runtime_cache/"
    "random_loao"
)

STAGE27_SOURCE_ROOT = Path(
    "/kaggle/working/stage27_cicids2017_sources"
)

STAGE22_MANIFEST_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
    / "stage22r_1c_development_model_input_manifest.json"
)

STAGE22_NONFINITE_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
    / "stage22r_1c_nonfinite_conversion_counts.csv"
)

K79_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1a3_k79_development_freeze"
    / "stage22r_k79_development_exclusions.parquet"
)

STAGE24_SCHEMA_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_0_protocol_lock"
    / "stage24_0a_schema_provenance_audit.json"
)

STAGE24_FLAG70_RESULT_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_2_primary_target_openings"
    / "stage24_2d_bridge70_flag_corrected"
    / "stage24_2d_bridge70_flag_corrected_result.json"
)

STAGE27_PREFIT_PATH = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1b_prefit_feature_materialization"
    / "prefit_source_feature_receipt.json"
)

STAGE27_SOURCE_RECEIPT_PATH = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
    / "source_effective_population_receipt.json"
)

ELIGIBLE_FAMILIES = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

IDS2018_DEVELOPMENT_FILES = [
    "02-14-2018.csv",
    "02-15-2018.csv",
    "02-16-2018.csv",
    "02-20-2018.csv",
    "02-21-2018.csv",
    "02-22-2018.csv",
    "02-23-2018.csv",
    "02-28-2018.csv",
]


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return (
        run(
            ["git", *args]
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array_content(arr):
    arr = np.ascontiguousarray(
        arr
    )

    h = hashlib.sha256()

    h.update(
        arr.tobytes(
            order="C"
        )
    )

    return h.hexdigest()


def find_json_value_paths(
    obj,
    target,
    path=(),
):
    found = []

    if isinstance(obj, dict):

        for key, value in obj.items():

            child_path = (
                path
                + (str(key),)
            )

            if value == target:
                found.append(
                    child_path
                )

            found.extend(
                find_json_value_paths(
                    value,
                    target,
                    child_path,
                )
            )

    elif isinstance(obj, list):

        for i, value in enumerate(obj):

            child_path = (
                path
                + (f"[{i}]",)
            )

            if value == target:
                found.append(
                    child_path
                )

            found.extend(
                find_json_value_paths(
                    value,
                    target,
                    child_path,
                )
            )

    return found


# =================================================================================================
# 2. EXACT STAGE28-1B DURABLE-PARENT GATE
# =================================================================================================

banner(
    "STAGE28-1C-A — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

remote = (
    remote_line.split()[0]
)

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)

print(
    "Expected HEAD:",
    EXPECTED_HEAD,
)

print(
    "Local HEAD   :",
    head,
)

print(
    "origin/main  :",
    origin,
)

print(
    "Remote main  :",
    remote,
)

print(
    "Branch       :",
    branch,
)

print(
    "Subject      :",
    subject,
)

print(
    "Git clean    :",
    not bool(status),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_HEAD
):
    raise RuntimeError(
        "Stage28-1B durable-parent mismatch."
    )

if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean:\n"
        + status
    )

print()
print(
    "[PASS] exact Stage28-1B durable state"
)


# =================================================================================================
# 3. STAGE28-1B MANIFEST / FIT LEDGER GATE
# =================================================================================================

banner(
    "STAGE28-1B EXECUTION-UNIVERSE GATE"
)

manifest_path = require_file(
    STAGE28_1B_DIR
    / "stage28_component_execution_manifest.csv"
)

manifest_receipt_path = require_file(
    STAGE28_1B_DIR
    / "stage28_component_execution_manifest_receipt.json"
)

freeze_1b_path = require_file(
    STAGE28_1B_DIR
    / "stage28_1b_freeze_record.json"
)

manifest_receipt = read_json(
    manifest_receipt_path
)

freeze_1b = read_json(
    freeze_1b_path
)

actual_manifest_sha = sha256_file(
    manifest_path
)

expected_manifest_sha = (
    manifest_receipt[
        "manifest"
    ][
        "sha256"
    ]
)

if actual_manifest_sha != expected_manifest_sha:
    raise RuntimeError(
        "Stage28 component-manifest SHA mismatch."
    )


with manifest_path.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:

    manifest_rows = list(
        csv.DictReader(
            f
        )
    )


if len(
    manifest_rows
) != 120:
    raise RuntimeError(
        "Component manifest row count != 120."
    )


new_rows = [
    row
    for row
    in manifest_rows
    if (
        row[
            "fit_action"
        ]
        == "NEW_FIT_AUTHORIZED"
    )
]

reuse_rows = [
    row
    for row
    in manifest_rows
    if (
        row[
            "fit_action"
        ]
        == "REUSE_EXISTING"
    )
]

evaluation_cells = {
    row[
        "evaluation_cell_id"
    ]
    for row
    in manifest_rows
}


if len(
    new_rows
) != 108:
    raise RuntimeError(
        "NEW component count != 108."
    )

if len(
    reuse_rows
) != 12:
    raise RuntimeError(
        "REUSE component count != 12."
    )

if len(
    evaluation_cells
) != 110:
    raise RuntimeError(
        "Evaluation-cell count != 110."
    )

if (
    freeze_1b[
        "execution_manifest"
    ][
        "new_fits_consumed"
    ]
    != 0
):
    raise RuntimeError(
        "Fit ledger no longer zero."
    )

print(
    "[PASS] manifest SHA exact"
)

print(
    "[PASS] component rows = 120"
)

print(
    "[PASS] evaluation cells = 110"
)

print(
    "[PASS] NEW fits = 108"
)

print(
    "[PASS] reused = 12"
)

print(
    "[PASS] NEW fits consumed = 0"
)


# =================================================================================================
# 4. LIBRARY / HARDWARE ENVIRONMENT
# =================================================================================================

banner(
    "MODEL RUNTIME ENVIRONMENT"
)

try:
    import xgboost
except Exception as exc:
    raise RuntimeError(
        "Unable to import XGBoost."
    ) from exc

try:
    import lightgbm
except Exception as exc:
    raise RuntimeError(
        "Unable to import LightGBM."
    ) from exc


print(
    "Python       :",
    sys.version.split()[0],
)

print(
    "Platform     :",
    platform.platform(),
)

print(
    "NumPy        :",
    np.__version__,
)

print(
    "pandas       :",
    pd.__version__,
)

print(
    "scikit-learn :",
    sklearn.__version__,
)

print(
    "PyArrow      :",
    pyarrow.__version__,
)

print(
    "XGBoost      :",
    xgboost.__version__,
)

print(
    "LightGBM     :",
    lightgbm.__version__,
)

print(
    "CPU count    :",
    os.cpu_count(),
)


# Historical Stage22/27 model builds.
EXPECTED_XGB = "3.2.0"
EXPECTED_LGBM = "4.6.0"

if (
    xgboost.__version__
    != EXPECTED_XGB
):
    raise RuntimeError(
        "XGBoost version drift detected BEFORE FIT #1.\n"
        f"expected={EXPECTED_XGB}\n"
        f"actual={xgboost.__version__}"
    )

if (
    lightgbm.__version__
    != EXPECTED_LGBM
):
    raise RuntimeError(
        "LightGBM version drift detected BEFORE FIT #1.\n"
        f"expected={EXPECTED_LGBM}\n"
        f"actual={lightgbm.__version__}"
    )

print()
print(
    "[PASS] XGBoost version = 3.2.0"
)

print(
    "[PASS] LightGBM version = 4.6.0"
)


# =================================================================================================
# 5. LOAD STAGE22 FROZEN MODEL-INPUT MANIFEST
# =================================================================================================

banner(
    "STAGE22 FROZEN 70-FEATURE INPUT CONTRACT"
)

stage22_manifest = read_json(
    require_file(
        STAGE22_MANIFEST_PATH
    )
)

if (
    stage22_manifest[
        "stage"
    ]
    != "Stage22R-1C"
):
    raise RuntimeError(
        "Unexpected Stage22 model-input manifest."
    )

feature22 = (
    stage22_manifest[
        "row_schema"
    ][
        "predictor_columns"
    ]
)

if len(
    feature22
) != 70:
    raise RuntimeError(
        "Stage22 feature count != 70."
    )

cache22 = (
    stage22_manifest[
        "cache"
    ]
)

if (
    cache22[
        "file_count"
    ]
    != 8
):
    raise RuntimeError(
        "Stage22 cache file count != 8."
    )

if (
    cache22[
        "total_rows"
    ]
    != 14_412_403
):
    raise RuntimeError(
        "Stage22 clean development rows mismatch."
    )

if (
    cache22[
        "total_bytes"
    ]
    != 1_541_208_291
):
    raise RuntimeError(
        "Stage22 historical cache byte total mismatch."
    )


print(
    "Feature count       :",
    len(
        feature22
    ),
)

print(
    "Feature dtype       :",
    stage22_manifest[
        "feature_configuration"
    ][
        "dtype"
    ],
)

print(
    "Clean rows          :",
    f"{cache22['total_rows']:,}",
)

print(
    "Cache files         :",
    cache22[
        "file_count"
    ],
)

print(
    "Historical bytes    :",
    f"{cache22['total_bytes']:,}",
)

print(
    "Parse dtype         :",
    stage22_manifest[
        "numeric_policy"
    ][
        "parse_dtype"
    ],
)

print(
    "Inf handling        :",
    stage22_manifest[
        "numeric_policy"
    ][
        "positive_infinity"
    ],
)

print(
    "Explicit imputation :",
    stage22_manifest[
        "numeric_policy"
    ][
        "explicit_imputation"
    ],
)

print(
    "Scaling             :",
    stage22_manifest[
        "numeric_policy"
    ][
        "scaling"
    ],
)

print()
print(
    "[PASS] Stage22 frozen 70-feature input contract"
)


# =================================================================================================
# 6. IDS2018 RAW-SOURCE RESOLUTION — HEADER ONLY
# =================================================================================================

banner(
    "RESOLVE 8 IDS2018 DEVELOPMENT SOURCE FILES"
)

if not INPUT_ROOT.exists():
    raise RuntimeError(
        "/kaggle/input missing."
    )


# Recover historical source sizes from Stage24's schema-only audit.
schema24 = read_json(
    require_file(
        STAGE24_SCHEMA_PATH
    )
)

ids18_records = (
    schema24[
        "schema_records"
    ][
        "ids2018"
    ]
)

historical_raw_size = {}

for record in ids18_records:

    record_path = str(
        record[
            "path"
        ]
    )

    if (
        "solarmainframe/ids-intrusion-csv/"
        in record_path
    ):

        basename = Path(
            record_path
        ).name

        historical_raw_size[
            basename
        ] = int(
            record[
                "size_bytes"
            ]
        )


input_files = [
    p
    for p in INPUT_ROOT.rglob(
        "*"
    )
    if p.is_file()
]

resolved_ids18 = {}


for basename in IDS2018_DEVELOPMENT_FILES:

    candidates = [
        p
        for p in input_files
        if p.name == basename
    ]

    expected_size = (
        historical_raw_size.get(
            basename
        )
    )

    print()
    print(
        basename
    )

    print(
        "  candidates:",
        len(
            candidates
        ),
    )

    if expected_size is not None:

        print(
            "  frozen size:",
            f"{expected_size:,}",
        )

        candidates = [
            p
            for p in candidates
            if (
                p.stat().st_size
                == expected_size
            )
        ]

        print(
            "  size-exact:",
            len(
                candidates
            ),
        )


    if len(
        candidates
    ) != 1:
        raise RuntimeError(
            f"{basename}: expected exactly one "
            "historically size-compatible source; "
            f"found {len(candidates)}."
        )


    path = candidates[0]

    # HEADER ONLY — no predictor rows.
    header = list(
        pd.read_csv(
            path,
            nrows=0,
        ).columns
    )

    stripped_header = [
        str(
            x
        ).strip()
        for x in header
    ]


    missing = [
        feature
        for feature
        in feature22
        if feature
        not in stripped_header
    ]

    if missing:
        raise RuntimeError(
            f"{basename}: missing Stage22 features:\n"
            + "\n".join(
                missing
            )
        )


    label_candidates = [
        col
        for col in stripped_header
        if (
            col.casefold()
            == "label"
        )
    ]

    if len(
        label_candidates
    ) != 1:
        raise RuntimeError(
            f"{basename}: Label column resolution failed."
        )


    resolved_ids18[
        basename
    ] = {
        "path":
            str(
                path
            ),

        "bytes":
            int(
                path.stat().st_size
            ),

        "column_count":
            len(
                stripped_header
            ),

        "label_column":
            label_candidates[
                0
            ],
    }


    print(
        "  [PASS]",
        path,
    )

    print(
        "  bytes:",
        f"{path.stat().st_size:,}",
    )

    print(
        "  columns:",
        len(
            stripped_header
        ),
    )


if len(
    resolved_ids18
) != 8:
    raise RuntimeError(
        "Did not resolve 8 IDS2018 development files."
    )

print()
print(
    "[PASS] 8/8 IDS2018 development sources resolved"
)

print(
    "[PASS] only CSV headers were read"
)


# =================================================================================================
# 7. K79 EXCLUSION ARTIFACT IDENTITY + SCHEMA
# =================================================================================================

banner(
    "STAGE22 K79 EXCLUSION MEMBERSHIP"
)

require_file(
    K79_PATH
)

expected_k79_sha = (
    stage22_manifest[
        "clean_development_parent"
    ][
        "k79_exclusion_membership_sha256"
    ]
)

actual_k79_sha = sha256_file(
    K79_PATH
)

print(
    "Path        :",
    K79_PATH,
)

print(
    "Expected SHA:",
    expected_k79_sha,
)

print(
    "Actual SHA  :",
    actual_k79_sha,
)


if (
    actual_k79_sha
    != expected_k79_sha
):
    raise RuntimeError(
        "K79 exclusion artifact SHA mismatch."
    )


k79_pf = pq.ParquetFile(
    K79_PATH
)

k79_rows = int(
    k79_pf.metadata.num_rows
)

k79_columns = (
    k79_pf.schema_arrow.names
)

print(
    "Rows        :",
    f"{k79_rows:,}",
)

print(
    "Columns     :",
    k79_columns,
)

print(
    "Row groups  :",
    k79_pf.num_row_groups,
)

print()
print(
    "[PASS] exact K79 artifact"
)


# =================================================================================================
# 8. HISTORICAL STAGE22 CACHE FILE CONTRACT
# =================================================================================================

banner(
    "STAGE22 HISTORICAL CACHE FILES"
)

historical_cache_files = {}


for record in (
    cache22[
        "files"
    ]
):

    day_id = int(
        record[
            "day_id"
        ]
    )

    historical_cache_files[
        record[
            "source_file"
        ]
    ] = record

    print(
        f"[{day_id}] "
        f"{record['source_file']:15s} "
        f"rows={record['rows']:9,d} "
        f"attack={record['attack']:7,d} "
        f"benign={record['benign']:9,d}"
    )

    print(
        "    cache:",
        record[
            "cache_file"
        ],
    )

    print(
        "    SHA  :",
        record[
            "sha256"
        ],
    )


if set(
    historical_cache_files
) != set(
    IDS2018_DEVELOPMENT_FILES
):
    raise RuntimeError(
        "Stage22 source/cache file universe mismatch."
    )


print()
print(
    "[PASS] 8 historical Stage22 cache identities loaded"
)


# =================================================================================================
# 9. STAGE22 NONFINITE EXPECTATION TABLE
# =================================================================================================

banner(
    "STAGE22 NUMERIC-AUDIT CONTRACT"
)

require_file(
    STAGE22_NONFINITE_PATH
)

nonfinite_df = pd.read_csv(
    STAGE22_NONFINITE_PATH
)

print(
    nonfinite_df.to_string(
        index=False
    )
)

print()

print(
    "Global expected positive inf -> NaN:",
    stage22_manifest[
        "nonfinite_conversion"
    ][
        "positive_infinity_to_nan"
    ],
)

print(
    "Global expected negative inf -> NaN:",
    stage22_manifest[
        "nonfinite_conversion"
    ][
        "negative_infinity_to_nan"
    ],
)

print(
    "Global expected output NaNs        :",
    stage22_manifest[
        "nonfinite_conversion"
    ][
        "output_nan_values"
    ],
)


# =================================================================================================
# 10. CICIDS2017 SOURCE + STAGE24 FULL-POPULATION FEATURE HASH CONTRACT
# =================================================================================================

banner(
    "CICIDS2017 FULL 70-FEATURE REPRODUCTION CONTRACT"
)

source27 = read_json(
    require_file(
        STAGE27_SOURCE_RECEIPT_PATH
    )
)

stage24_flag70 = read_json(
    require_file(
        STAGE24_FLAG70_RESULT_PATH
    )
)

stage27_prefit = read_json(
    require_file(
        STAGE27_PREFIT_PATH
    )
)


segments27 = (
    source27[
        "segments"
    ]
)

feature_audit24 = (
    stage24_flag70[
        "feature_matrix_audit"
    ]
)


if len(
    segments27
) != 8:
    raise RuntimeError(
        "Stage27 source segment count != 8."
    )

if len(
    feature_audit24
) != 8:
    raise RuntimeError(
        "Stage24 feature-matrix audit count != 8."
    )


audit24_by_id = {
    int(
        row[
            "file_id"
        ]
    ):
        row

    for row
    in feature_audit24
}


for seg in segments27:

    file_id = int(
        seg[
            "source_index"
        ]
    )

    basename = (
        seg[
            "basename"
        ]
    )

    runtime_path = (
        STAGE27_SOURCE_ROOT
        / basename
    )

    require_file(
        runtime_path
    )

    actual_sha = sha256_file(
        runtime_path
    )

    if actual_sha != seg[
        "sha256"
    ]:
        raise RuntimeError(
            f"{basename}: recovered CICIDS source SHA mismatch."
        )


    audit = (
        audit24_by_id[
            file_id
        ]
    )

    if int(
        audit[
            "rows"
        ]
    ) != int(
        seg[
            "effective_rows"
        ]
    ):
        raise RuntimeError(
            f"{basename}: Stage24 feature-audit "
            "row count mismatch."
        )


    print(
        f"[{file_id}] {basename}"
    )

    print(
        "    effective rows:",
        f"{seg['effective_rows']:,}",
    )

    print(
        "    source SHA    :",
        actual_sha,
    )

    print(
        "    FLOAT64 70f SHA:",
        audit[
            "flag_corrected_feature_matrix_float64_sha256"
        ],
    )


print()
print(
    "[PASS] 8/8 CICIDS source bytes exact"
)

print(
    "[PASS] 8/8 historical Stage24 FLOAT64 "
    "feature-matrix hashes available"
)


# =================================================================================================
# 11. STAGE27 FEATURE REPRESENTATION
# =================================================================================================

banner(
    "STAGE27 FEATURE REPRESENTATION"
)

rep27 = (
    stage27_prefit[
        "feature_representation"
    ]
)

feature27 = (
    rep27[
        "feature_order"
    ]
)


print(
    "Identity          :",
    rep27[
        "identity"
    ],
)

print(
    "Feature count     :",
    rep27[
        "feature_count"
    ],
)

print(
    "Adapter variant   :",
    rep27[
        "adapter_variant"
    ],
)

print(
    "Adapter SHA       :",
    rep27[
        "adapter_mapping_sha256"
    ],
)

print(
    "Parse dtype       :",
    rep27[
        "parse_dtype"
    ],
)

print(
    "Final matrix dtype:",
    rep27[
        "final_model_matrix_dtype"
    ],
)

print(
    "Positive infinity:",
    rep27[
        "positive_infinity"
    ],
)

print(
    "Negative infinity:",
    rep27[
        "negative_infinity"
    ],
)

print(
    "Imputation        :",
    rep27[
        "explicit_imputation"
    ],
)

print(
    "Scaling           :",
    rep27[
        "scaling"
    ],
)


if (
    rep27[
        "adapter_variant"
    ]
    != "FLAG_CORRECTED_mapping"
):
    raise RuntimeError(
        "Stage27 adapter variant mismatch."
    )

if (
    rep27[
        "adapter_mapping_sha256"
    ]
    != "88d7f3e133e7d20ee05dc4618c6102f0c420936e0fe72a9a094d951a9c4dad7a"
):
    raise RuntimeError(
        "Stage27 adapter mapping SHA mismatch."
    )


# =================================================================================================
# 12. STAGE22 <-> STAGE27 ORDERED FEATURE IDENTITY
# =================================================================================================

banner(
    "STAGE22 ↔ STAGE27 ORDERED FEATURE IDENTITY"
)

if feature22 != feature27:

    print(
        "Stage22-only / ordering differences:"
    )

    for i, (
        a,
        b,
    ) in enumerate(
        zip(
            feature22,
            feature27,
        )
    ):

        if a != b:
            print(
                i,
                repr(
                    a
                ),
                "!=",
                repr(
                    b
                ),
            )

    raise RuntimeError(
        "Stage22 and Stage27 70-feature "
        "ordered lists differ."
    )


print(
    "[PASS] exact ordered 70-feature semantic identity"
)

print()

for i, feature in enumerate(
    feature22,
    1,
):

    print(
        f"{i:02d}. {feature}"
    )


# =================================================================================================
# 13. LOCATE FLAG-CORRECTED MAPPING PROVENANCE IN DURABLE JSON
# =================================================================================================

banner(
    "FLAG-CORRECTED MAPPING PROVENANCE SEARCH"
)

TARGET_MAPPING_SHA = (
    "88d7f3e133e7d20ee05dc4618c6102f0c420936e0fe72a9a094d951a9c4dad7a"
)

mapping_hits = []


for json_path in (
    REPO
    / "results"
    / "stage24_cross_dataset"
).rglob(
    "*.json"
):

    try:

        text = json_path.read_text(
            encoding="utf-8"
        )

    except Exception:
        continue


    if (
        TARGET_MAPPING_SHA in text
        or
        "FLAG_CORRECTED_mapping" in text
    ):

        mapping_hits.append(
            json_path
        )


print(
    "Matching durable JSON files:",
    len(
        mapping_hits
    ),
)

for path in mapping_hits:

    print(
        " ",
        path.relative_to(
            REPO
        ),
    )


if not mapping_hits:

    raise RuntimeError(
        "Unable to locate durable flag-corrected "
        "mapping provenance."
    )


# Additionally show exact JSON value paths to the adapter hash.
for path in mapping_hits:

    try:
        obj = read_json(
            path
        )

    except Exception:
        continue

    paths = find_json_value_paths(
        obj,
        TARGET_MAPPING_SHA,
    )

    for p in paths:

        print(
            "  hash path:",
            path.relative_to(
                REPO
            ),
            "::",
            " / ".join(
                p
            ),
        )


print()
print(
    "[PASS] flag-corrected mapping provenance located"
)


# =================================================================================================
# 14. VERIFY STAGE28B RANDOM MEMBERSHIP RUNTIME ARRAYS SURVIVED
# =================================================================================================

banner(
    "VERIFY STAGE28B RUNTIME MEMBERSHIP ARRAYS"
)

for family in ELIGIBLE_FAMILIES:

    receipt_path = require_file(
        STAGE28_1B_DIR
        / (
            f"random_loao_{family}_"
            "membership_receipt.json"
        )
    )

    receipt = read_json(
        receipt_path
    )

    runtime_paths = (
        receipt[
            "runtime_paths"
        ]
    )

    checks = [
        (
            "train_global_idx",
            "train",
        ),
        (
            "validation_global_idx",
            "validation",
        ),
        (
            "primary_target_global_idx",
            "primary_target",
        ),
    ]


    print()
    print(
        family
    )


    for runtime_key, hash_key in checks:

        path = Path(
            runtime_paths[
                runtime_key
            ]
        )

        require_file(
            path
        )

        arr = np.load(
            path,
            mmap_mode="r",
            allow_pickle=False,
        )

        actual_sha = (
            sha256_array_content(
                np.asarray(
                    arr
                )
            )
        )

        expected_sha = (
            receipt[
                "hashes"
            ][
                hash_key
            ][
                "ordered_global_idx_content_sha256"
            ]
        )


        print(
            f"  {runtime_key:29s} "
            f"rows={len(arr):9,d}"
        )

        print(
            "    expected:",
            expected_sha,
        )

        print(
            "    actual  :",
            actual_sha,
        )


        if (
            actual_sha
            != expected_sha
        ):
            raise RuntimeError(
                f"{family}/{runtime_key}: "
                "runtime membership hash mismatch."
            )


    print(
        f"  [PASS] {family}"
    )


print()
print(
    "[PASS] all 15 Stage28B runtime membership "
    "arrays remain exact"
)


# =================================================================================================
# 15. FINAL GIT CLEANLINESS / SCIENCE COUNTERS
# =================================================================================================

banner(
    "STAGE28-1C-A FINAL PREFLIGHT AUDIT"
)

final_status = git(
    "status",
    "--porcelain",
)

final_head = git(
    "rev-parse",
    "HEAD",
)


if final_status:
    raise RuntimeError(
        "Preflight unexpectedly modified Git:\n"
        + final_status
    )

if (
    final_head
    != EXPECTED_HEAD
):
    raise RuntimeError(
        "HEAD changed during preflight."
    )


print(
    "[PASS] durable HEAD unchanged"
)

print(
    "[PASS] Git clean"
)

print(
    "[PASS] 8 IDS2018 development sources resolved"
)

print(
    "[PASS] K79 membership artifact exact"
)

print(
    "[PASS] Stage22 70-feature contract loaded"
)

print(
    "[PASS] 8 CICIDS2017 sources exact"
)

print(
    "[PASS] 8 historical Stage24 feature-matrix hashes loaded"
)

print(
    "[PASS] Stage27 FLAG_CORRECTED representation exact"
)

print(
    "[PASS] Stage22 == Stage27 ordered 70-feature list"
)

print(
    "[PASS] 15 random-LOAO runtime membership arrays exact"
)

print()

print(
    "Scientific operations:"
)

print(
    "  PREDICTOR_VALUES_READ     = 0"
)

print(
    "  MODEL_FITS                = 0"
)

print(
    "  MODEL_INFERENCE           = 0"
)

print(
    "  THRESHOLD_SELECTION       = 0"
)

print(
    "  TARGET_PREDICTOR_OPENINGS = 0"
)

print()

print(
    "Fit ledger:"
)

print(
    "  NEW authorized = 108"
)

print(
    "  consumed       = 0"
)

print(
    "  remaining      = 108"
)

print()

print(
    "NEXT:"
)

print(
    "  Stage28-1C-B — reconstruct the Stage22 "
    "70-feature development cache from the exact "
    "8 IDS2018 sources + frozen K79 exclusions."
)

print(
    "  It will require the historical per-day "
    "row/count/numeric/cache identity gates before "
    "we touch model fit #1."
)

print()
print(SEP)


STAGE28-1C-A — EXACT DURABLE-PARENT GATE

Expected HEAD: ba011ec01f1399939111b24664ebd5d66c630f95
Local HEAD   : ba011ec01f1399939111b24664ebd5d66c630f95
origin/main  : ba011ec01f1399939111b24664ebd5d66c630f95
Remote main  : ba011ec01f1399939111b24664ebd5d66c630f95
Branch       : main
Subject      : stage28-1b: freeze random LOAO memberships and execution manifest
Git clean    : True

[PASS] exact Stage28-1B durable state

STAGE28-1B EXECUTION-UNIVERSE GATE

[PASS] manifest SHA exact
[PASS] component rows = 120
[PASS] evaluation cells = 110
[PASS] NEW fits = 108
[PASS] reused = 12
[PASS] NEW fits consumed = 0

MODEL RUNTIME ENVIRONMENT

Python       : 3.12.13
Platform     : Linux-6.12.90+-x86_64-with-glibc2.35
NumPy        : 2.0.2
pandas       : 2.3.3
scikit-learn : 1.6.1
PyArrow      : 24.0.0
XGBoost      : 3.2.0
LightGBM     : 4.6.0
CPU count    : 4

[PASS] XGBoost version = 3.2.0
[PASS] LightGBM version = 4.6.0

STAGE22 FROZEN 70-FEATURE INPUT CONTRACT

Feature count       : 70
Fea

In [5]:
# =================================================================================================
# STAGE28-1C-B — BYTE-EXACT STAGE22R 70-FEATURE CACHE RECOVERY
#
# Durable scientific parent:
#   ba011ec01f1399939111b24664ebd5d66c630f95
#
# IMPORTANT
# ---------
# Stage22R already created a reset-safe PRIVATE Kaggle dataset containing
# the exact eight historical Stage22R-1C model-input Parquets:
#
#   jmmubasshirrahman/stage22r-1c-70f-cache-3cd41c5f
#
# Therefore Stage28 does NOT reconstruct these predictors from raw IDS2018.
# It restores the already-frozen byte-exact cache and requires all historical
# SHA256 / size / row / schema identities.
#
# RECOVERY ORDER
# --------------
# 1. Look for exact cache files already attached under /kaggle/input.
# 2. If not all eight are attached, non-mutating download of the frozen
#    private Kaggle dataset is attempted with kagglehub.
# 3. Every candidate must pass:
#       filename
#       historical byte size
#       SHA256
#       Parquet row count
#       exact 74-column ordered schema
# 4. Verified files are symlinked into:
#       /kaggle/working/stage22r_1c_70f_development_cache
#
# ZERO:
#   raw IDS2018 source rows opened
#   predictor values recomputed
#   predictor values read
#   labels recomputed/read
#   final-holdout openings
#   model fits
#   model inference
#   threshold selection
#   Git modifications
#
# No durable Stage28 commit in this cell.
# =================================================================================================

from __future__ import annotations

import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

INPUT_ROOT = Path(
    "/kaggle/input"
).resolve()

EXPECTED_HEAD = (
    "ba011ec01f1399939111b24664ebd5d66c630f95"
)

PRIVATE_DATASET_ID = (
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

EXPECTED_RUNTIME_CACHE = Path(
    "/kaggle/working/"
    "stage22r_1c_70f_development_cache"
)

RUNTIME_RECEIPT_PATH = Path(
    "/kaggle/working/"
    "stage28_1c_b_stage22_cache_recovery_receipt.json"
)

REMOTE_RECEIPT_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_remote_cache_checkpoint"
    / "stage22r_1c_private_kaggle_cache_receipt.json"
)

MODEL_INPUT_MANIFEST_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
    / "stage22r_1c_development_model_input_manifest.json"
)

K79_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1a3_k79_development_freeze"
    / "stage22r_k79_development_exclusions.parquet"
)


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return (
        run(
            ["git", *args]
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


# =================================================================================================
# 2. EXACT DURABLE-PARENT GATE
# =================================================================================================

banner(
    "STAGE28-1C-B — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote = (
    remote_line
    .split()[0]
)

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD:",
    EXPECTED_HEAD,
)

print(
    "Local HEAD   :",
    head,
)

print(
    "origin/main  :",
    origin,
)

print(
    "Remote main  :",
    remote,
)

print(
    "Branch       :",
    branch,
)

print(
    "Git clean    :",
    not bool(
        status
    ),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_HEAD
):
    raise RuntimeError(
        "Stage28-1B durable parent mismatch."
    )

if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean:\n"
        + status
    )


print()
print(
    "[PASS] exact Stage28-1B durable parent"
)


# =================================================================================================
# 3. LOAD FROZEN RESET-SAFE CACHE CHECKPOINT
# =================================================================================================

banner(
    "LOAD STAGE22R RESET-SAFE CACHE CHECKPOINT"
)

remote_receipt = read_json(
    require_file(
        REMOTE_RECEIPT_PATH
    )
)

manifest = read_json(
    require_file(
        MODEL_INPUT_MANIFEST_PATH
    )
)


if (
    remote_receipt[
        "stage"
    ]
    != "Stage22R-1C-RC1"
):
    raise RuntimeError(
        "Unexpected remote-cache checkpoint stage."
    )

if (
    remote_receipt[
        "status"
    ]
    != "PRIVATE_KAGGLE_RESET_SAFE_CACHE_CHECKPOINT_VERIFIED"
):
    raise RuntimeError(
        "Remote cache checkpoint is not frozen/verified."
    )

if (
    remote_receipt[
        "kaggle_dataset"
    ][
        "dataset_id"
    ]
    != PRIVATE_DATASET_ID
):
    raise RuntimeError(
        "Private cache dataset identity mismatch."
    )

if (
    remote_receipt[
        "gpu_reset_recovery"
    ][
        "expected_runtime_cache_path"
    ]
    != str(
        EXPECTED_RUNTIME_CACHE
    )
):
    raise RuntimeError(
        "Frozen runtime-cache path mismatch."
    )


expected_manifest_sha = (
    remote_receipt[
        "scientific_input_manifest"
    ][
        "sha256"
    ]
)

actual_manifest_sha = sha256_file(
    MODEL_INPUT_MANIFEST_PATH
)

print(
    "Private dataset :",
    PRIVATE_DATASET_ID,
)

print(
    "Frozen manifest :",
    MODEL_INPUT_MANIFEST_PATH.relative_to(
        REPO
    ),
)

print(
    "Expected SHA    :",
    expected_manifest_sha,
)

print(
    "Actual SHA      :",
    actual_manifest_sha,
)


if (
    actual_manifest_sha
    != expected_manifest_sha
):
    raise RuntimeError(
        "Stage22R scientific-input manifest SHA mismatch."
    )


print()
print(
    "[PASS] exact Stage22R reset-safe recovery contract"
)


# =================================================================================================
# 4. CROSS-CHECK PRIVATE RECEIPT AGAINST SCIENTIFIC INPUT MANIFEST
# =================================================================================================

banner(
    "CROSS-CHECK CACHE IDENTITIES"
)

remote_files = {
    x[
        "filename"
    ]:
    x

    for x
    in remote_receipt[
        "cache"
    ][
        "files"
    ]
}

manifest_files = {
    x[
        "cache_file"
    ]:
    x

    for x
    in manifest[
        "cache"
    ][
        "files"
    ]
}


if set(
    remote_files
) != set(
    manifest_files
):
    raise RuntimeError(
        "Remote-cache file universe differs "
        "from scientific-input manifest."
    )


if len(
    remote_files
) != 8:
    raise RuntimeError(
        "Expected exactly 8 Stage22 cache files."
    )


for filename in sorted(
    remote_files
):

    r = remote_files[
        filename
    ]

    m = manifest_files[
        filename
    ]

    checks = {

        "rows":
            (
                int(r["rows"]),
                int(m["rows"]),
            ),

        "bytes":
            (
                int(r["bytes"]),
                int(m["bytes"]),
            ),

        "sha256":
            (
                r["sha256"],
                m["sha256"],
            ),

        "attack":
            (
                int(r["attack"]),
                int(m["attack"]),
            ),

        "benign":
            (
                int(r["benign"]),
                int(m["benign"]),
            ),
    }


    for field, (
        a,
        b,
    ) in checks.items():

        if a != b:

            raise RuntimeError(
                f"{filename}: receipt/manifest "
                f"mismatch in {field}.\n"
                f"remote={a}\n"
                f"manifest={b}"
            )


    print(
        f"[PASS] {filename}"
    )

    print(
        f"       rows={r['rows']:,} "
        f"bytes={r['bytes']:,}"
    )

    print(
        "       SHA256:",
        r[
            "sha256"
        ],
    )


if (
    int(
        remote_receipt[
            "cache"
        ][
            "rows"
        ]
    )
    != 14_412_403
):
    raise RuntimeError(
        "Remote cache total rows mismatch."
    )

if (
    int(
        remote_receipt[
            "cache"
        ][
            "bytes"
        ]
    )
    != 1_541_208_291
):
    raise RuntimeError(
        "Remote cache total bytes mismatch."
    )


print()
print(
    "[PASS] remote reset-safe checkpoint "
    "== frozen Stage22 scientific cache manifest"
)


# =================================================================================================
# 5. VERIFY K79 PARENT IS STILL EXACT
# =================================================================================================

banner(
    "K79 PARENT IDENTITY GATE"
)

expected_k79_sha = (
    manifest[
        "clean_development_parent"
    ][
        "k79_exclusion_membership_sha256"
    ]
)

actual_k79_sha = sha256_file(
    require_file(
        K79_PATH
    )
)

print(
    "Expected:",
    expected_k79_sha,
)

print(
    "Actual  :",
    actual_k79_sha,
)


if (
    actual_k79_sha
    != expected_k79_sha
):
    raise RuntimeError(
        "Frozen K79 membership artifact changed."
    )


print(
    "[PASS] exact K79 parent"
)


# =================================================================================================
# 6. EXPECTED PARQUET SCHEMA — METADATA + 70 FEATURES
# =================================================================================================

banner(
    "FROZEN PARQUET SCHEMA CONTRACT"
)

metadata_columns = (
    manifest[
        "row_schema"
    ][
        "metadata_columns"
    ]
)

predictor_columns = (
    manifest[
        "row_schema"
    ][
        "predictor_columns"
    ]
)

expected_columns = (
    metadata_columns
    +
    predictor_columns
)


if metadata_columns != [
    "clean_position",
    "day_id",
    "original_row_index",
    "binary_label",
]:
    raise RuntimeError(
        "Unexpected Stage22 metadata-column order."
    )

if len(
    predictor_columns
) != 70:
    raise RuntimeError(
        "Expected 70 predictor columns."
    )

if len(
    expected_columns
) != 74:
    raise RuntimeError(
        "Expected total 74 cache columns."
    )


print(
    "Metadata columns:",
    metadata_columns,
)

print(
    "Predictors      :",
    len(
        predictor_columns
    ),
)

print(
    "Total columns   :",
    len(
        expected_columns
    ),
)

print(
    "Cache format    :",
    manifest[
        "cache"
    ][
        "format"
    ],
)

print(
    "Compression     :",
    manifest[
        "cache"
    ][
        "compression"
    ],
)

print(
    "Row-group size  :",
    manifest[
        "cache"
    ][
        "row_group_size"
    ],
)


# =================================================================================================
# 7. DISCOVER EXACT CACHE FILES ALREADY ATTACHED
# =================================================================================================

banner(
    "DISCOVER ATTACHED BYTE-EXACT STAGE22 CACHE"
)

input_files = []

if INPUT_ROOT.exists():

    input_files = [
        p
        for p
        in INPUT_ROOT.rglob(
            "*"
        )
        if p.is_file()
    ]


resolved = {}


def resolve_from_candidates(
    search_files,
    resolution_name,
):
    """
    Resolve as many missing expected cache files as possible.

    Gate order:
      basename
      exact byte size
      exact SHA256

    Multiple byte-identical candidates are harmless;
    deterministic lexical-first is used.
    """

    for filename in sorted(
        remote_files
    ):

        if filename in resolved:
            continue

        expected = (
            remote_files[
                filename
            ]
        )

        candidates = [
            p
            for p
            in search_files
            if (
                p.name
                == filename
            )
        ]

        candidates = [
            p
            for p
            in candidates
            if (
                p.stat().st_size
                == int(
                    expected[
                        "bytes"
                    ]
                )
            )
        ]


        exact = []

        for path in sorted(
            candidates,
            key=lambda x: str(
                x
            ),
        ):

            actual_sha = sha256_file(
                path
            )

            if (
                actual_sha
                == expected[
                    "sha256"
                ]
            ):

                exact.append(
                    path
                )


        if exact:

            resolved[
                filename
            ] = {
                "path":
                    exact[
                        0
                    ],

                "resolution":
                    resolution_name,

                "duplicate_exact_candidates":
                    len(
                        exact
                    ),
            }


resolve_from_candidates(
    input_files,
    "ATTACHED_KAGGLE_INPUT",
)


print(
    "Exact files already attached:",
    len(
        resolved
    ),
    "/ 8",
)

for filename in sorted(
    resolved
):

    print(
        "[FOUND]",
        filename,
    )

    print(
        "       ",
        resolved[
            filename
        ][
            "path"
        ],
    )


# =================================================================================================
# 8. NON-MUTATING PRIVATE DATASET RECOVERY IF NEEDED
# =================================================================================================

download_root = None
download_error = None


if len(
    resolved
) < 8:

    banner(
        "RECOVER PRIVATE RESET-SAFE KAGGLE DATASET"
    )

    print(
        "Dataset:",
        PRIVATE_DATASET_ID,
    )

    print(
        "Attached exact files:",
        len(
            resolved
        ),
        "/ 8",
    )

    print()
    print(
        "Attempting authenticated KaggleHub recovery..."
    )


    try:

        try:
            import kagglehub

        except ImportError:

            print(
                "[INFO] kagglehub not imported; "
                "installing client package."
            )

            install = subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "kagglehub",
                ],
                text=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                check=False,
            )

            if install.returncode != 0:

                raise RuntimeError(
                    "Unable to install kagglehub:\n"
                    + install.stderr
                )

            import kagglehub


        print(
            "kagglehub:",
            getattr(
                kagglehub,
                "__version__",
                "UNKNOWN",
            ),
        )


        download_root = Path(
            kagglehub.dataset_download(
                PRIVATE_DATASET_ID
            )
        ).resolve()


        print(
            "Recovered dataset root:"
        )

        print(
            " ",
            download_root,
        )


        downloaded_files = [
            p
            for p
            in download_root.rglob(
                "*"
            )
            if p.is_file()
        ]


        resolve_from_candidates(
            downloaded_files,
            "PRIVATE_KAGGLE_RESET_SAFE_DATASET",
        )


    except Exception as exc:

        download_error = repr(
            exc
        )

        print()
        print(
            "[RECOVERY ATTEMPT FAILED]"
        )

        print(
            download_error
        )


# =================================================================================================
# 9. REQUIRE 8 / 8 BYTE-EXACT SOURCE CACHE FILES
# =================================================================================================

banner(
    "8/8 BYTE-EXACT STAGE22 CACHE SOURCE GATE"
)

if len(
    resolved
) != 8:

    missing = sorted(
        set(
            remote_files
        )
        -
        set(
            resolved
        )
    )

    print(
        "Resolved:",
        len(
            resolved
        ),
        "/ 8",
    )

    print()

    print(
        "Missing:"
    )

    for filename in missing:

        print(
            " ",
            filename,
        )


    print()

    print(
        "The exact frozen reset-safe Kaggle "
        "dataset is:"
    )

    print(
        " ",
        PRIVATE_DATASET_ID,
    )

    print()

    print(
        "Attach that PRIVATE dataset to this "
        "Kaggle notebook, then rerun THIS SAME CELL."
    )


    if download_error:

        print()

        print(
            "KaggleHub recovery error:"
        )

        print(
            " ",
            download_error,
        )


    raise RuntimeError(
        "Stage22 exact reset-safe cache is not "
        "currently accessible. No raw-source "
        "reconstruction was attempted."
    )


for filename in sorted(
    resolved
):

    expected = (
        remote_files[
            filename
        ]
    )

    path = (
        resolved[
            filename
        ][
            "path"
        ]
    )

    actual_bytes = (
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )


    if (
        actual_bytes
        != int(
            expected[
                "bytes"
            ]
        )
    ):
        raise RuntimeError(
            f"{filename}: byte-size mismatch."
        )

    if (
        actual_sha
        != expected[
            "sha256"
        ]
    ):
        raise RuntimeError(
            f"{filename}: SHA256 mismatch."
        )


    print(
        "[PASS]",
        filename,
    )

    print(
        "       source:",
        path,
    )

    print(
        "       mode  :",
        resolved[
            filename
        ][
            "resolution"
        ],
    )

    print(
        "       bytes :",
        f"{actual_bytes:,}",
    )

    print(
        "       SHA256:",
        actual_sha,
    )


# =================================================================================================
# 10. PARQUET METADATA / ORDERED SCHEMA GATE
#
# IMPORTANT:
# Only Parquet METADATA is inspected.
# No data column values are read.
# =================================================================================================

banner(
    "PARQUET METADATA / ORDERED SCHEMA GATE"
)

metadata_audit = {}


for filename in sorted(
    remote_files
):

    expected = (
        remote_files[
            filename
        ]
    )

    path = (
        resolved[
            filename
        ][
            "path"
        ]
    )

    pf = pq.ParquetFile(
        path
    )

    rows = int(
        pf.metadata.num_rows
    )

    row_groups = int(
        pf.metadata.num_row_groups
    )

    columns = list(
        pf.schema_arrow.names
    )


    print(
        filename
    )

    print(
        "  rows      :",
        f"{rows:,}",
    )

    print(
        "  row groups:",
        row_groups,
    )

    print(
        "  columns   :",
        len(
            columns
        ),
    )


    if rows != int(
        expected[
            "rows"
        ]
    ):
        raise RuntimeError(
            f"{filename}: Parquet row-count mismatch."
        )

    if columns != expected_columns:

        print()
        print(
            "Expected columns:"
        )

        print(
            expected_columns
        )

        print()

        print(
            "Actual columns:"
        )

        print(
            columns
        )

        raise RuntimeError(
            f"{filename}: ordered Parquet "
            "schema mismatch."
        )


    metadata_audit[
        filename
    ] = {
        "rows":
            rows,

        "row_groups":
            row_groups,

        "columns":
            len(
                columns
            ),

        "ordered_schema_exact":
            True,
    }


    print(
        "  [PASS] exact metadata/schema"
    )


print()
print(
    "[PASS] 8/8 historical Parquet "
    "row/schema identities exact"
)


# =================================================================================================
# 11. RESTORE EXPECTED RUNTIME CACHE PATH
# =================================================================================================

banner(
    "RESTORE HISTORICAL RUNTIME CACHE PATH"
)

EXPECTED_RUNTIME_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)


expected_names = set(
    remote_files
)


# The runtime directory is disposable.
# Remove only stale files/symlinks inside this exact runtime path.
for child in list(
    EXPECTED_RUNTIME_CACHE.iterdir()
):

    if child.name not in expected_names:

        if child.is_dir() and not child.is_symlink():

            shutil.rmtree(
                child
            )

        else:

            child.unlink()


for filename in sorted(
    remote_files
):

    source = (
        resolved[
            filename
        ][
            "path"
        ]
        .resolve()
    )

    target = (
        EXPECTED_RUNTIME_CACHE
        / filename
    )


    if target.exists() or target.is_symlink():

        target.unlink()


    # Symlink is explicitly compatible with the historical
    # reset-recovery receipt and avoids another 1.54 GB copy.
    target.symlink_to(
        source
    )


    if not target.is_file():

        raise RuntimeError(
            f"Unable to restore runtime file:\n"
            f"{target}"
        )


    target_sha = sha256_file(
        target
    )

    if (
        target_sha
        != remote_files[
            filename
        ][
            "sha256"
        ]
    ):
        raise RuntimeError(
            f"{filename}: runtime symlink "
            "content verification failed."
        )


    print(
        "[PASS]",
        target,
    )

    print(
        "       ->",
        source,
    )


# =================================================================================================
# 12. COMPLETE RUNTIME CACHE GATE
# =================================================================================================

banner(
    "COMPLETE STAGE22R RUNTIME CACHE GATE"
)

runtime_files = sorted(
    p.name
    for p
    in EXPECTED_RUNTIME_CACHE.iterdir()
    if p.is_file()
)

if (
    runtime_files
    != sorted(
        expected_names
    )
):
    raise RuntimeError(
        "Runtime cache file universe mismatch."
    )


total_rows = 0
total_bytes = 0


for filename in sorted(
    expected_names
):

    target = (
        EXPECTED_RUNTIME_CACHE
        / filename
    )

    expected = (
        remote_files[
            filename
        ]
    )

    actual_sha = sha256_file(
        target
    )

    actual_bytes = int(
        target.stat().st_size
    )

    rows = int(
        pq.ParquetFile(
            target
        ).metadata.num_rows
    )


    if (
        actual_sha
        != expected[
            "sha256"
        ]
    ):
        raise RuntimeError(
            f"{filename}: final SHA gate failed."
        )

    if (
        actual_bytes
        != int(
            expected[
                "bytes"
            ]
        )
    ):
        raise RuntimeError(
            f"{filename}: final byte gate failed."
        )

    if (
        rows
        != int(
            expected[
                "rows"
            ]
        )
    ):
        raise RuntimeError(
            f"{filename}: final row gate failed."
        )


    total_rows += rows
    total_bytes += actual_bytes


if total_rows != 14_412_403:
    raise RuntimeError(
        "Recovered Stage22 cache total rows mismatch."
    )

if total_bytes != 1_541_208_291:
    raise RuntimeError(
        "Recovered Stage22 cache total bytes mismatch."
    )


print(
    "Runtime root:",
    EXPECTED_RUNTIME_CACHE,
)

print(
    "Files       :",
    len(
        runtime_files
    ),
)

print(
    "Rows        :",
    f"{total_rows:,}",
)

print(
    "Bytes       :",
    f"{total_bytes:,}",
)

print()
print(
    "[PASS] historical Stage22R-1C "
    "70-feature cache restored byte-for-byte"
)


# =================================================================================================
# 13. WRITE DISPOSABLE RUNTIME RECOVERY RECEIPT
#
# Outside Git. This does NOT create Stage28 durable scientific state.
# =================================================================================================

runtime_receipt = {

    "stage":
        "Stage28-1C-B",

    "type":
        "RUNTIME_ONLY_STAGE22_BYTE_EXACT_CACHE_RECOVERY",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "durable_scientific_parent":
        EXPECTED_HEAD,

    "historical_stage22_checkpoint": {
        "stage":
            remote_receipt[
                "stage"
            ],

        "scientific_parent_commit":
            remote_receipt[
                "scientific_parent_commit"
            ],

        "dataset_id":
            PRIVATE_DATASET_ID,

        "checkpoint_status":
            remote_receipt[
                "status"
            ],
    },

    "recovery_method": {
        "raw_ids2018_source_files_opened":
            0,

        "raw_predictor_values_recomputed":
            0,

        "source":
            (
                "BYTE_EXACT_RESET_SAFE_CACHE_"
                "ATTACHED_OR_PRIVATE_KAGGLE_DATASET"
            ),

        "runtime_binding":
            "SYMLINK_TO_VERIFIED_BYTE_EXACT_FILES",
    },

    "cache": {
        "runtime_path":
            str(
                EXPECTED_RUNTIME_CACHE
            ),

        "file_count":
            8,

        "rows":
            total_rows,

        "bytes":
            total_bytes,

        "files": {
            filename: {
                "sha256":
                    remote_files[
                        filename
                    ][
                        "sha256"
                    ],

                "rows":
                    int(
                        remote_files[
                            filename
                        ][
                            "rows"
                        ]
                    ),

                "bytes":
                    int(
                        remote_files[
                            filename
                        ][
                            "bytes"
                        ]
                    ),

                "source_path":
                    str(
                        resolved[
                            filename
                        ][
                            "path"
                        ]
                    ),

                "resolution":
                    resolved[
                        filename
                    ][
                        "resolution"
                    ],

                "parquet_schema_exact":
                    True,
            }

            for filename
            in sorted(
                remote_files
            )
        },
    },

    "scientific_access": {
        "raw_ids2018_rows_read":
            0,

        "predictor_values_read":
            0,

        "predictor_values_recomputed":
            0,

        "labels_read":
            0,

        "labels_recomputed":
            0,

        "final_holdout_openings":
            0,

        "model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,
    },

    "fit_ledger": {
        "new_authorized":
            108,

        "new_consumed":
            0,

        "new_remaining":
            108,
    },

    "status":
        "BYTE_EXACT_STAGE22_RUNTIME_CACHE_RECOVERED",
}


RUNTIME_RECEIPT_PATH.write_text(
    json.dumps(
        runtime_receipt,
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)


print()
print(
    "Runtime receipt:"
)

print(
    " ",
    RUNTIME_RECEIPT_PATH,
)


# =================================================================================================
# 14. FINAL GIT CLEANLINESS GATE
# =================================================================================================

banner(
    "FINAL GIT CLEANLINESS GATE"
)

final_head = git(
    "rev-parse",
    "HEAD",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD     :",
    final_head,
)

print(
    "Git clean:",
    not bool(
        final_status
    ),
)


if final_head != EXPECTED_HEAD:

    raise RuntimeError(
        "HEAD changed during Stage28-1C-B."
    )


if final_status:

    raise RuntimeError(
        "Stage28-1C-B unexpectedly changed Git:\n"
        + final_status
    )


print()
print(
    "[PASS] durable Git state untouched"
)


# =================================================================================================
# 15. FINAL
# =================================================================================================

banner(
    "STAGE28-1C-B — BYTE-EXACT STAGE22 CACHE RECOVERY COMPLETE"
)

print(
    "Durable parent:"
)

print(
    " ",
    EXPECTED_HEAD,
)

print()

print(
    "Historical Stage22 cache:"
)

print(
    "  dataset =",
    PRIVATE_DATASET_ID,
)

print(
    "  files   = 8 / 8 EXACT"
)

print(
    "  rows    =",
    f"{total_rows:,}",
)

print(
    "  bytes   =",
    f"{total_bytes:,}",
)

print(
    "  SHA256  = 8 / 8 HISTORICAL EXACT"
)

print(
    "  schema  = 8 / 8 ORDERED EXACT"
)

print()

print(
    "Runtime cache:"
)

print(
    " ",
    EXPECTED_RUNTIME_CACHE,
)

print()

print(
    "Scientific access:"
)

print(
    "  RAW_IDS2018_SOURCE_ROWS_READ = 0"
)

print(
    "  PREDICTOR_VALUES_READ        = 0"
)

print(
    "  PREDICTOR_VALUES_RECOMPUTED  = 0"
)

print(
    "  LABEL_VALUES_READ            = 0"
)

print(
    "  FINAL_HOLDOUT_OPENINGS       = 0"
)

print(
    "  MODEL_FITS                   = 0"
)

print(
    "  MODEL_INFERENCE              = 0"
)

print(
    "  THRESHOLD_SELECTION          = 0"
)

print()

print(
    "Fit ledger:"
)

print(
    "  NEW authorized = 108"
)

print(
    "  consumed       = 0"
)

print(
    "  remaining      = 108"
)

print()

print(
    "NEXT:"
)

print(
    "  Stage28-1C-C — reproduce the already-opened "
    "CICIDS2017 FLAG_CORRECTED 70-feature matrices "
    "and require all eight historical Stage24 "
    "FLOAT64 feature-matrix SHA256 identities."
)

print()

print(
    "  No model fit is authorized yet."
)

print()
print(SEP)


STAGE28-1C-B — EXACT DURABLE-PARENT GATE

Expected HEAD: ba011ec01f1399939111b24664ebd5d66c630f95
Local HEAD   : ba011ec01f1399939111b24664ebd5d66c630f95
origin/main  : ba011ec01f1399939111b24664ebd5d66c630f95
Remote main  : ba011ec01f1399939111b24664ebd5d66c630f95
Branch       : main
Git clean    : True

[PASS] exact Stage28-1B durable parent

LOAD STAGE22R RESET-SAFE CACHE CHECKPOINT

Private dataset : jmmubasshirrahman/stage22r-1c-70f-cache-3cd41c5f
Frozen manifest : results/stage22r_protocol_recovery/stage22r_1c_development_model_inputs/stage22r_1c_development_model_input_manifest.json
Expected SHA    : 05fe6226afd37c3ab434140e1164643eecc7abc75c295a2d6fb0280d4f42127e
Actual SHA      : 05fe6226afd37c3ab434140e1164643eecc7abc75c295a2d6fb0280d4f42127e

[PASS] exact Stage22R reset-safe recovery contract

CROSS-CHECK CACHE IDENTITIES

[PASS] day_00_02-14-2018.parquet
       rows=822,947 bytes=87,229,572
       SHA256: a542ad551f5aab59c9ad27958d6bb065e3336b3ca4f040da21e49ad71936f990
[PA

In [6]:
# =================================================================================================
# STAGE28-1C-C0 — STAGE24 FLAG_CORRECTED MATRIX-HASH SEMANTICS PREFLIGHT
#
# Durable scientific parent:
#   ba011ec01f1399939111b24664ebd5d66c630f95
#
# PURPOSE
# -------
# Before reopening any CICIDS2017 predictor values, recover from the frozen
# Stage24 repository evidence:
#
#   1. exact Stage24-2D result/checksum identity
#   2. exact 70-feature FLAG_CORRECTED mapping metadata
#   3. all historical per-source FLOAT64 matrix hashes and their JSON paths
#   4. the exact hashing implementation / byte convention used by Stage24
#   5. whether Stage24 left any reset-safe private Kaggle/cache checkpoint
#
# THIS CELL READS ONLY:
#   - Git metadata
#   - committed JSON / SHA256 / Python / Markdown / text metadata
#   - Stage28-1C-B runtime receipt
#   - filesystem names/stat metadata
#
# ZERO:
#   CICIDS2017 predictor values read
#   CICIDS2017 labels read
#   IDS2018 predictor values read
#   final-holdout openings
#   model fits
#   model inference
#   threshold selection
#   Git modifications
#
# No durable Stage28 commit is created.
# =================================================================================================

from __future__ import annotations

import hashlib
import json
import os
import re
import subprocess
from pathlib import Path
from typing import Any


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_HEAD = (
    "ba011ec01f1399939111b24664ebd5d66c630f95"
)

STAGE24_ROOT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
)

STAGE24_2D_DIR = (
    STAGE24_ROOT
    / "stage24_2_primary_target_openings"
    / "stage24_2d_bridge70_flag_corrected"
)

STAGE24_RESULT = (
    STAGE24_2D_DIR
    / "stage24_2d_bridge70_flag_corrected_result.json"
)

STAGE24_CHECKSUMS = (
    STAGE24_2D_DIR
    / "checksums.sha256"
)

STAGE24_SCRIPT = (
    REPO
    / "scripts"
    / "stage24"
    / "stage24_cross_dataset_generalization.py"
)

STAGE22_RUNTIME_RECEIPT = Path(
    "/kaggle/working/"
    "stage28_1c_b_stage22_cache_recovery_receipt.json"
)

STAGE22_RUNTIME_CACHE = Path(
    "/kaggle/working/"
    "stage22r_1c_70f_development_cache"
)

DIAGNOSTIC_RECEIPT = Path(
    "/kaggle/working/"
    "stage28_1c_c0_stage24_hash_semantics_preflight.json"
)

EXPECTED_STAGE22_CACHE_FILES = 8
EXPECTED_STAGE22_ROWS = 14_412_403
EXPECTED_NEW_FITS = 108


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title: str) -> None:
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args) -> str:
    return (
        run(
            ["git", *args]
        ).stdout
        or ""
    ).strip()


def require_file(path: Path) -> Path:
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path: Path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path: Path,
    chunk_size: int = 4 * 1024 * 1024,
) -> str:

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def is_sha256(value: Any) -> bool:
    return (
        isinstance(value, str)
        and re.fullmatch(
            r"[0-9a-fA-F]{64}",
            value.strip(),
        )
        is not None
    )


def walk_json(
    obj: Any,
    path: str = "$",
):
    """
    Yield:
        (path, key, value)

    for every dictionary key and list element recursively.
    """

    if isinstance(obj, dict):

        for key, value in obj.items():

            child_path = (
                f"{path}.{key}"
            )

            yield (
                child_path,
                str(key),
                value,
            )

            yield from walk_json(
                value,
                child_path,
            )

    elif isinstance(obj, list):

        for i, value in enumerate(obj):

            child_path = (
                f"{path}[{i}]"
            )

            yield (
                child_path,
                f"[{i}]",
                value,
            )

            yield from walk_json(
                value,
                child_path,
            )


def scalar_summary_dict(
    d: dict,
) -> dict:
    """
    Keep useful scalar metadata while avoiding dumping large
    feature-order/mapping collections.
    """

    out = {}

    for key, value in d.items():

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ) or value is None:

            out[key] = value

        elif (
            isinstance(value, list)
            and len(value) <= 10
            and all(
                isinstance(
                    x,
                    (
                        str,
                        int,
                        float,
                        bool,
                        type(None),
                    ),
                )
                for x in value
            )
        ):

            out[key] = value

    return out


def print_context_blocks(
    text: str,
    terms: list[str],
    context: int = 10,
    max_blocks: int = 40,
) -> dict:

    lines = text.splitlines()

    matched_lines = set()

    per_term = {}

    for term in terms:

        hits = [
            i
            for i, line
            in enumerate(lines)
            if term.lower()
            in line.lower()
        ]

        per_term[term] = [
            i + 1
            for i in hits
        ]

        matched_lines.update(
            hits
        )

    if not matched_lines:

        print(
            "[INFO] No requested source-code "
            "search terms were found."
        )

        return per_term

    ranges = []

    for idx in sorted(
        matched_lines
    ):

        start = max(
            0,
            idx - context,
        )

        stop = min(
            len(lines),
            idx + context + 1,
        )

        if (
            ranges
            and start
            <= ranges[-1][1]
        ):

            ranges[-1] = (
                ranges[-1][0],
                max(
                    ranges[-1][1],
                    stop,
                ),
            )

        else:

            ranges.append(
                (
                    start,
                    stop,
                )
            )

    ranges = ranges[
        :max_blocks
    ]

    for block_no, (
        start,
        stop,
    ) in enumerate(
        ranges,
        start=1,
    ):

        print()
        print(
            "-" * 100
        )

        print(
            f"SOURCE CONTEXT BLOCK {block_no}"
            f" — lines {start + 1}:{stop}"
        )

        print(
            "-" * 100
        )

        for i in range(
            start,
            stop,
        ):

            marker = (
                ">>>"
                if i in matched_lines
                else "   "
            )

            print(
                f"{marker} {i + 1:6d}: "
                f"{lines[i]}"
            )

    return per_term


# =================================================================================================
# 2. DURABLE-PARENT / CLEAN-WORKTREE GATE
# =================================================================================================

banner(
    "STAGE28-1C-C0 — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote = (
    remote_line
    .split()[0]
)

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD:",
    EXPECTED_HEAD,
)

print(
    "Local HEAD   :",
    head,
)

print(
    "origin/main  :",
    origin,
)

print(
    "Remote main  :",
    remote,
)

print(
    "Branch       :",
    branch,
)

print(
    "Git clean    :",
    not bool(status),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_HEAD
):
    raise RuntimeError(
        "Durable Stage28 parent mismatch."
    )

if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean:\n"
        + status
    )


print()
print(
    "[PASS] exact Stage28 durable parent"
)


# =================================================================================================
# 3. REQUIRE STAGE28-1C-B RECOVERY STATE
# =================================================================================================

banner(
    "STAGE28-1C-B RUNTIME RECOVERY GATE"
)

c_b = read_json(
    require_file(
        STAGE22_RUNTIME_RECEIPT
    )
)


print(
    "Receipt:",
    STAGE22_RUNTIME_RECEIPT,
)

print(
    "Status :",
    c_b.get(
        "status"
    ),
)

print(
    "Parent :",
    c_b.get(
        "durable_scientific_parent"
    ),
)


if (
    c_b.get(
        "durable_scientific_parent"
    )
    != EXPECTED_HEAD
):
    raise RuntimeError(
        "Stage28-1C-B parent mismatch."
    )

if (
    c_b.get(
        "status"
    )
    != "BYTE_EXACT_STAGE22_RUNTIME_CACHE_RECOVERED"
):
    raise RuntimeError(
        "Stage28-1C-B cache recovery is not verified."
    )

cache_meta = (
    c_b.get(
        "cache",
        {},
    )
)

if int(
    cache_meta.get(
        "file_count",
        -1,
    )
) != EXPECTED_STAGE22_CACHE_FILES:
    raise RuntimeError(
        "Stage22 runtime-cache file count mismatch."
    )

if int(
    cache_meta.get(
        "rows",
        -1,
    )
) != EXPECTED_STAGE22_ROWS:
    raise RuntimeError(
        "Stage22 runtime-cache row count mismatch."
    )


scientific_access = (
    c_b.get(
        "scientific_access",
        {}
    )
)

for key, value in (
    scientific_access.items()
):

    if int(value) != 0:

        raise RuntimeError(
            "Stage28-1C-B contains non-zero "
            f"scientific-access counter: "
            f"{key}={value}"
        )


fit_ledger = (
    c_b.get(
        "fit_ledger",
        {}
    )
)

if int(
    fit_ledger.get(
        "new_consumed",
        -1,
    )
) != 0:
    raise RuntimeError(
        "A Stage28 fit was consumed before C0."
    )

if int(
    fit_ledger.get(
        "new_remaining",
        -1,
    )
) != EXPECTED_NEW_FITS:
    raise RuntimeError(
        "Stage28 remaining-fit ledger mismatch."
    )


# Filesystem existence only.
# DO NOT open the cache Parquets here.
runtime_entries = []

if STAGE22_RUNTIME_CACHE.is_dir():

    runtime_entries = sorted(
        p.name
        for p
        in STAGE22_RUNTIME_CACHE.iterdir()
        if p.is_file()
    )


print(
    "Runtime cache files:",
    len(
        runtime_entries
    ),
)

print(
    "New fits consumed   :",
    fit_ledger[
        "new_consumed"
    ],
)

print(
    "New fits remaining  :",
    fit_ledger[
        "new_remaining"
    ],
)


if len(
    runtime_entries
) != EXPECTED_STAGE22_CACHE_FILES:
    raise RuntimeError(
        "Stage22 runtime cache is no longer "
        "present with all 8 files."
    )


print()
print(
    "[PASS] Stage28-1C-B runtime recovery intact"
)


# =================================================================================================
# 4. VERIFY STAGE24-2D COMMITTED CHECKSUMS
# =================================================================================================

banner(
    "STAGE24-2D COMMITTED CHECKSUM GATE"
)

require_file(
    STAGE24_RESULT
)

require_file(
    STAGE24_CHECKSUMS
)

checksum_text = (
    STAGE24_CHECKSUMS.read_text(
        encoding="utf-8"
    )
)

print(
    "Checksum file:"
)

print(
    " ",
    STAGE24_CHECKSUMS.relative_to(
        REPO
    )
)

print()

print(
    checksum_text.rstrip()
)


checksum_entries = []

for raw_line in (
    checksum_text.splitlines()
):

    line = raw_line.strip()

    if not line:
        continue

    m = re.match(
        r"^([0-9a-fA-F]{64})\s+\*?(.+?)\s*$",
        line,
    )

    if not m:
        raise RuntimeError(
            "Unrecognized checksums.sha256 line:\n"
            + raw_line
        )

    sha = (
        m.group(1)
        .lower()
    )

    rel_name = (
        m.group(2)
        .strip()
    )

    candidate = (
        STAGE24_2D_DIR
        / rel_name
    )

    require_file(
        candidate
    )

    actual = sha256_file(
        candidate
    )

    print()
    print(
        "[CHECK]",
        rel_name,
    )

    print(
        "  expected:",
        sha,
    )

    print(
        "  actual  :",
        actual,
    )

    if actual != sha:
        raise RuntimeError(
            f"Stage24 checksum mismatch: "
            f"{rel_name}"
        )

    checksum_entries.append(
        {
            "file":
                rel_name,

            "sha256":
                sha,
        }
    )


if not checksum_entries:
    raise RuntimeError(
        "Stage24 checksum file contained no entries."
    )


print()
print(
    "[PASS] all Stage24-2D committed "
    "checksum entries exact"
)


# =================================================================================================
# 5. LOAD STAGE24-2D RESULT METADATA ONLY
# =================================================================================================

banner(
    "STAGE24-2D RESULT STRUCTURE"
)

stage24 = read_json(
    STAGE24_RESULT
)


print(
    "Result:",
    STAGE24_RESULT.relative_to(
        REPO
    ),
)

print()

print(
    "Top-level keys:"
)

for key in stage24.keys():
    print(
        " ",
        key,
    )


representation = stage24.get(
    "representation"
)

if not isinstance(
    representation,
    dict,
):
    raise RuntimeError(
        "Stage24 result has no representation object."
    )


print()
print(
    "representation keys:"
)

for key in (
    representation.keys()
):
    print(
        " ",
        key,
    )


bridge_dimension = (
    representation.get(
        "bridge_dimension"
    )
)

feature_order = (
    representation.get(
        "feature_order"
    )
)

flag_mapping = (
    representation.get(
        "flag_corrected_mapping"
    )
)


print()
print(
    "Bridge dimension:",
    bridge_dimension,
)

print(
    "Feature count    :",
    (
        len(feature_order)
        if isinstance(
            feature_order,
            list,
        )
        else "NOT_A_LIST"
    ),
)

print(
    "Mapping entries  :",
    (
        len(flag_mapping)
        if isinstance(
            flag_mapping,
            dict,
        )
        else "NOT_A_DICT"
    ),
)

print(
    "Mapping SHA256   :",
    representation.get(
        "mapping_sha256"
    ),
)

print(
    "Source schema SHA:",
    representation.get(
        "source_schema_sha256"
    ),
)

print(
    "Parse dtype      :",
    representation.get(
        "parse_dtype"
    ),
)

print(
    "Positive infinity:",
    representation.get(
        "positive_infinity"
    ),
)

print(
    "Negative infinity:",
    representation.get(
        "negative_infinity"
    ),
)

print(
    "NaN handling     :",
    representation.get(
        "model_input_nan_handling"
    ),
)

print(
    "Input semantics  :",
    representation.get(
        "input_semantics"
    ),
)


if bridge_dimension != 70:
    raise RuntimeError(
        "Stage24 bridge dimension is not 70."
    )

if not (
    isinstance(
        feature_order,
        list,
    )
    and len(
        feature_order
    ) == 70
):
    raise RuntimeError(
        "Stage24 frozen feature order is not 70."
    )

if not (
    isinstance(
        flag_mapping,
        dict,
    )
    and len(
        flag_mapping
    ) == 70
):
    raise RuntimeError(
        "Stage24 FLAG_CORRECTED mapping "
        "does not contain 70 entries."
    )


print()
print(
    "[PASS] frozen Stage24 FLAG_CORRECTED "
    "70-feature representation recovered"
)


# =================================================================================================
# 6. PRINT EXACT FEATURE ORDER + MAPPING
#
# Metadata only; no target dataset values are accessed.
# =================================================================================================

banner(
    "FROZEN STAGE24 70-FEATURE FLAG_CORRECTED MAPPING"
)

for i, feature in enumerate(
    feature_order
):

    mapped = (
        flag_mapping.get(
            feature,
            "<MISSING>"
        )
    )

    print(
        f"{i:02d}. "
        f"{feature!r}"
        f"  ->  "
        f"{mapped!r}"
    )

    if mapped == "<MISSING>":
        raise RuntimeError(
            f"Missing Stage24 mapping for "
            f"{feature!r}"
        )


# =================================================================================================
# 7. DISCOVER EVERY HASH/MATRIX/DTYPE/SHAPE FIELD IN RESULT JSON
# =================================================================================================

banner(
    "STAGE24 RESULT — HASH / MATRIX / DTYPE / SHAPE METADATA"
)

interesting_key_re = re.compile(
    r"(sha|hash|matrix|float64|dtype|shape|row_count|rows|mapping)",
    flags=re.IGNORECASE,
)

interesting_scalars = []

all_sha256_values = []


for path, key, value in walk_json(
    stage24
):

    if (
        isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
                type(None),
            ),
        )
        and interesting_key_re.search(
            key
        )
    ):

        interesting_scalars.append(
            (
                path,
                value,
            )
        )

    if is_sha256(
        value
    ):

        all_sha256_values.append(
            (
                path,
                value.lower(),
            )
        )


for path, value in (
    interesting_scalars
):

    print(
        f"{path} = {value!r}"
    )


print()
print(
    "Total SHA256-looking scalar values:",
    len(
        all_sha256_values
    ),
)


# =================================================================================================
# 8. LOCATE DICTIONARIES CONTAINING MATRIX/HASH SEMANTICS
# =================================================================================================

banner(
    "LIKELY PER-SOURCE MATRIX-HASH RECORDS"
)

candidate_records = []


for path, key, value in walk_json(
    stage24
):

    if not isinstance(
        value,
        dict,
    ):
        continue

    keys_lower = [
        str(k).lower()
        for k
        in value.keys()
    ]

    joined = " ".join(
        keys_lower
    )

    has_matrix = (
        "matrix"
        in joined
    )

    has_sha = (
        "sha"
        in joined
        or "hash"
        in joined
    )

    has_float64 = (
        "float64"
        in joined
        or any(
            isinstance(v, str)
            and "float64"
            in v.lower()
            for v
            in value.values()
        )
    )


    if (
        (
            has_matrix
            and has_sha
        )
        or (
            has_float64
            and has_sha
        )
    ):

        summary = (
            scalar_summary_dict(
                value
            )
        )

        candidate_records.append(
            {
                "path":
                    path,

                "summary":
                    summary,
            }
        )


if candidate_records:

    for i, record in enumerate(
        candidate_records,
        start=1,
    ):

        print()
        print(
            f"[CANDIDATE {i}]"
        )

        print(
            "Path:",
            record[
                "path"
            ],
        )

        print(
            json.dumps(
                record[
                    "summary"
                ],
                indent=2,
                ensure_ascii=False,
            )
        )

else:

    print(
        "[INFO] No dictionary with explicit "
        "matrix+hash keys was found."
    )


# =================================================================================================
# 9. SEARCH SPECIFIC HISTORICAL MATRIX-HASH FIELD NAMES
# =================================================================================================

banner(
    "EXPLICIT HISTORICAL MATRIX-HASH FIELD SEARCH"
)

target_key_terms = [
    "flag_corrected_feature_matrix_float64_sha256",
    "feature_matrix_float64_sha256",
    "feature_matrix_sha256",
    "matrix_float64_sha256",
    "matrix_sha256",
]

explicit_matrix_hash_hits = []


for path, key, value in walk_json(
    stage24
):

    key_lower = (
        key.lower()
    )

    if any(
        term
        in key_lower
        for term
        in target_key_terms
    ):

        explicit_matrix_hash_hits.append(
            (
                path,
                value,
            )
        )


if explicit_matrix_hash_hits:

    for path, value in (
        explicit_matrix_hash_hits
    ):

        print(
            f"[FOUND] {path}"
        )

        print(
            "        ",
            value,
        )

else:

    print(
        "[INFO] No exact target field name "
        "found in Stage24 result JSON."
    )


# =================================================================================================
# 10. GROUP SHA256 VALUES BY LIKELY SCIENTIFIC PURPOSE
# =================================================================================================

banner(
    "ALL STAGE24 SHA256 VALUES WITH JSON PATHS"
)

for i, (
    path,
    value,
) in enumerate(
    all_sha256_values,
    start=1,
):

    print(
        f"{i:03d}. {path}"
    )

    print(
        f"     {value}"
    )


# =================================================================================================
# 11. INSPECT FROZEN STAGE24 IMPLEMENTATION AS TEXT ONLY
# =================================================================================================

banner(
    "STAGE24 SOURCE-CODE HASH IMPLEMENTATION SEARCH"
)

require_file(
    STAGE24_SCRIPT
)

script_text = (
    STAGE24_SCRIPT.read_text(
        encoding="utf-8",
        errors="replace",
    )
)


print(
    "Script:",
    STAGE24_SCRIPT.relative_to(
        REPO
    ),
)

print(
    "Bytes :",
    STAGE24_SCRIPT.stat().st_size,
)

print(
    "Lines :",
    len(
        script_text.splitlines()
    ),
)


SOURCE_SEARCH_TERMS = [
    "flag_corrected_feature_matrix_float64_sha256",
    "feature_matrix_float64_sha256",
    "matrix_float64",
    "float64_sha256",
    ".tobytes(",
    "tobytes(",
    "ascontiguousarray",
    "hashlib.sha256",
    "sha256(",
    "digest",
]


source_hits = print_context_blocks(
    script_text,
    SOURCE_SEARCH_TERMS,
    context=12,
    max_blocks=50,
)


print()
print(
    "Search-term hit counts:"
)

for term in (
    SOURCE_SEARCH_TERMS
):

    lines = (
        source_hits.get(
            term,
            []
        )
    )

    print(
        f"  {term!r}: "
        f"{len(lines)} hit(s)"
        + (
            f" @ {lines[:20]}"
            if lines
            else ""
        )
    )


# =================================================================================================
# 12. SEARCH ALL STAGE24 COMMITTED TEXT METADATA FOR RESET-SAFE CACHE EVIDENCE
# =================================================================================================

banner(
    "STAGE24 RESET-SAFE CACHE / KAGGLE CHECKPOINT SEARCH"
)

tracked_stage24 = (
    git(
        "ls-files",
        "results/stage24_cross_dataset",
        "scripts/stage24",
    )
    .splitlines()
)

safe_text_suffixes = {
    ".json",
    ".md",
    ".txt",
    ".sha256",
    ".py",
    ".yaml",
    ".yml",
    ".toml",
}

CACHE_SEARCH_TERMS = [
    "dataset_id",
    "kaggle_dataset",
    "kagglehub",
    "private_kaggle",
    "reset_safe",
    "reset-safe",
    "runtime_cache",
    "cache_path",
    "cache_checkpoint",
    "checkpoint",
]

cache_text_hits = []

candidate_filenames = []


for rel in tracked_stage24:

    path = (
        REPO
        / rel
    )

    name_lower = (
        path.name.lower()
    )

    if any(
        token in name_lower
        for token in (
            "cache",
            "checkpoint",
            "flag_corrected",
            "matrix",
        )
    ):

        candidate_filenames.append(
            rel
        )

    if (
        not path.is_file()
        or path.suffix.lower()
        not in safe_text_suffixes
    ):
        continue

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    for line_no, line in enumerate(
        text.splitlines(),
        start=1,
    ):

        if any(
            term.lower()
            in line.lower()
            for term
            in CACHE_SEARCH_TERMS
        ):

            cache_text_hits.append(
                {
                    "file":
                        rel,

                    "line":
                        line_no,

                    "text":
                        line.strip()[
                            :1200
                        ],
                }
            )


print(
    "Candidate filenames containing "
    "cache/checkpoint/matrix/flag_corrected:"
)

if candidate_filenames:

    for rel in (
        candidate_filenames
    ):

        print(
            " ",
            rel,
        )

else:

    print(
        "  <NONE>"
    )


print()
print(
    "Text hits for reset-safe/Kaggle/cache terms:"
)


if cache_text_hits:

    for hit in (
        cache_text_hits[
            :250
        ]
    ):

        print(
            f"{hit['file']}:{hit['line']}: "
            f"{hit['text']}"
        )

    if len(
        cache_text_hits
    ) > 250:

        print(
            f"... truncated "
            f"{len(cache_text_hits) - 250} "
            f"additional hits"
        )

else:

    print(
        "  <NONE>"
    )


# =================================================================================================
# 13. DETERMINE WHAT THE PREFLIGHT HAS RESOLVED
# =================================================================================================

banner(
    "HASH-SEMANTICS RESOLUTION SUMMARY"
)

script_term_counts = {
    term:
        len(
            source_hits.get(
                term,
                []
            )
        )

    for term
    in SOURCE_SEARCH_TERMS
}


has_explicit_matrix_hash_json = bool(
    explicit_matrix_hash_hits
)

has_matrix_hash_candidate = bool(
    candidate_records
)

has_tobytes_impl = (
    script_term_counts.get(
        ".tobytes(",
        0,
    )
    > 0
    or script_term_counts.get(
        "tobytes(",
        0,
    )
    > 0
)

has_sha_impl = (
    script_term_counts.get(
        "hashlib.sha256",
        0,
    )
    > 0
    or script_term_counts.get(
        "sha256(",
        0,
    )
    > 0
)

has_contiguous_impl = (
    script_term_counts.get(
        "ascontiguousarray",
        0,
    )
    > 0
)

has_cache_evidence = bool(
    cache_text_hits
)


print(
    "Explicit matrix-hash JSON field(s):",
    len(
        explicit_matrix_hash_hits
    ),
)

print(
    "Matrix/hash candidate records       :",
    len(
        candidate_records
    ),
)

print(
    "Source uses tobytes                 :",
    has_tobytes_impl,
)

print(
    "Source uses SHA256                  :",
    has_sha_impl,
)

print(
    "Source uses ascontiguousarray       :",
    has_contiguous_impl,
)

print(
    "Stage24 cache/checkpoint text hits  :",
    len(
        cache_text_hits
    ),
)


# This is deliberately conservative:
# do not invent semantics if the source does not expose them.
hash_semantics_evidence_present = bool(
    has_sha_impl
    and (
        has_tobytes_impl
        or has_explicit_matrix_hash_json
        or has_matrix_hash_candidate
    )
)


if hash_semantics_evidence_present:

    print()
    print(
        "[PASS] Stage24 contains explicit "
        "matrix-hash implementation evidence."
    )

else:

    print()
    print(
        "[NOTICE] Exact matrix byte convention "
        "is not yet mechanically proven by the "
        "simple marker tests above."
    )

    print(
        "         Use the printed source context "
        "before authorizing predictor access."
    )


# =================================================================================================
# 14. RUNTIME-ONLY DIAGNOSTIC RECEIPT
# =================================================================================================

diagnostic_receipt = {

    "stage":
        "Stage28-1C-C0",

    "type":
        "METADATA_ONLY_STAGE24_HASH_SEMANTICS_PREFLIGHT",

    "durable_scientific_parent":
        EXPECTED_HEAD,

    "stage22_runtime_recovery": {
        "status":
            c_b.get(
                "status"
            ),

        "files_present":
            len(
                runtime_entries
            ),

        "rows":
            cache_meta.get(
                "rows"
            ),
    },

    "stage24_2d": {
        "result":
            str(
                STAGE24_RESULT.relative_to(
                    REPO
                )
            ),

        "checksums":
            checksum_entries,

        "bridge_dimension":
            bridge_dimension,

        "feature_count":
            len(
                feature_order
            ),

        "mapping_count":
            len(
                flag_mapping
            ),

        "mapping_sha256":
            representation.get(
                "mapping_sha256"
            ),

        "source_schema_sha256":
            representation.get(
                "source_schema_sha256"
            ),

        "parse_dtype":
            representation.get(
                "parse_dtype"
            ),

        "positive_infinity":
            representation.get(
                "positive_infinity"
            ),

        "negative_infinity":
            representation.get(
                "negative_infinity"
            ),

        "model_input_nan_handling":
            representation.get(
                "model_input_nan_handling"
            ),

        "input_semantics":
            representation.get(
                "input_semantics"
            ),

        "sha256_scalar_paths":
            [
                {
                    "path":
                        path,

                    "sha256":
                        value,
                }

                for path, value
                in all_sha256_values
            ],

        "explicit_matrix_hash_hits":
            [
                {
                    "path":
                        path,

                    "value":
                        value,
                }

                for path, value
                in explicit_matrix_hash_hits
            ],

        "candidate_matrix_hash_records":
            candidate_records,

        "script_search_hits":
            source_hits,

        "script_term_counts":
            script_term_counts,

        "hash_semantics_evidence_present":
            hash_semantics_evidence_present,
    },

    "stage24_reset_safe_cache_search": {
        "candidate_filenames":
            candidate_filenames,

        "text_hit_count":
            len(
                cache_text_hits
            ),

        "text_hits":
            cache_text_hits[
                :250
            ],
    },

    "scientific_access": {
        "cicids2017_predictor_values_read":
            0,

        "cicids2017_labels_read":
            0,

        "ids2018_predictor_values_read":
            0,

        "final_holdout_openings":
            0,

        "model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,
    },

    "fit_ledger": {
        "new_authorized":
            EXPECTED_NEW_FITS,

        "new_consumed":
            0,

        "new_remaining":
            EXPECTED_NEW_FITS,
    },

    "git_modified":
        False,
}


DIAGNOSTIC_RECEIPT.write_text(
    json.dumps(
        diagnostic_receipt,
        indent=2,
        ensure_ascii=False,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)


print()
print(
    "Runtime-only diagnostic receipt:"
)

print(
    " ",
    DIAGNOSTIC_RECEIPT,
)


# =================================================================================================
# 15. FINAL GIT CLEANLINESS GATE
# =================================================================================================

banner(
    "FINAL GIT CLEANLINESS GATE"
)

final_head = git(
    "rev-parse",
    "HEAD",
)

final_origin = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head,
)

print(
    "origin/main:",
    final_origin,
)

print(
    "Git clean  :",
    not bool(
        final_status
    ),
)


if (
    final_head
    != EXPECTED_HEAD
    or final_origin
    != EXPECTED_HEAD
):
    raise RuntimeError(
        "Git parent changed during C0."
    )

if final_status:
    raise RuntimeError(
        "Stage28-1C-C0 changed Git:\n"
        + final_status
    )


print()
print(
    "[PASS] Git state untouched"
)


# =================================================================================================
# 16. FINAL
# =================================================================================================

banner(
    "STAGE28-1C-C0 — PREFLIGHT COMPLETE"
)

print(
    "Durable parent:"
)

print(
    " ",
    EXPECTED_HEAD,
)

print()

print(
    "Stage22 historical cache:"
)

print(
    "  8 / 8 runtime files present"
)

print(
    "  predictor values opened here = 0"
)

print()

print(
    "Stage24 FLAG_CORRECTED metadata:"
)

print(
    "  bridge dimension =",
    bridge_dimension,
)

print(
    "  feature count    =",
    len(
        feature_order
    ),
)

print(
    "  mapping entries  =",
    len(
        flag_mapping
    ),
)

print(
    "  mapping SHA256   =",
    representation.get(
        "mapping_sha256"
    ),
)

print(
    "  source schema SHA=",
    representation.get(
        "source_schema_sha256"
    ),
)

print()

print(
    "Historical SHA256-looking fields found =",
    len(
        all_sha256_values
    ),
)

print(
    "Explicit matrix-hash field hits        =",
    len(
        explicit_matrix_hash_hits
    ),
)

print(
    "Matrix/hash candidate records          =",
    len(
        candidate_records
    ),
)

print(
    "Hash implementation evidence present   =",
    hash_semantics_evidence_present,
)

print(
    "Stage24 cache/checkpoint text hits      =",
    len(
        cache_text_hits
    ),
)

print()

print(
    "Scientific access:"
)

print(
    "  CICIDS2017 PREDICTORS = 0"
)

print(
    "  CICIDS2017 LABELS     = 0"
)

print(
    "  IDS2018 PREDICTORS    = 0"
)

print(
    "  FINAL HOLDOUT         = 0"
)

print(
    "  MODEL FITS            = 0"
)

print(
    "  INFERENCE             = 0"
)

print(
    "  THRESHOLD SELECTION   = 0"
)

print()

print(
    "Fit ledger:"
)

print(
    "  authorized = 108"
)

print(
    "  consumed   = 0"
)

print(
    "  remaining  = 108"
)

print()

print(
    "NEXT:"
)

print(
    "  Inspect this C0 output."
)

print(
    "  Only after the exact Stage24 FLOAT64 "
    "matrix-hash convention and historical "
    "per-source identities are resolved will "
    "Stage28-1C-C open CICIDS2017 predictors."
)

print()
print(SEP)


STAGE28-1C-C0 — EXACT DURABLE-PARENT GATE

Expected HEAD: ba011ec01f1399939111b24664ebd5d66c630f95
Local HEAD   : ba011ec01f1399939111b24664ebd5d66c630f95
origin/main  : ba011ec01f1399939111b24664ebd5d66c630f95
Remote main  : ba011ec01f1399939111b24664ebd5d66c630f95
Branch       : main
Git clean    : True

[PASS] exact Stage28 durable parent

STAGE28-1C-B RUNTIME RECOVERY GATE

Receipt: /kaggle/working/stage28_1c_b_stage22_cache_recovery_receipt.json
Status : BYTE_EXACT_STAGE22_RUNTIME_CACHE_RECOVERED
Parent : ba011ec01f1399939111b24664ebd5d66c630f95
Runtime cache files: 8
New fits consumed   : 0
New fits remaining  : 108

[PASS] Stage28-1C-B runtime recovery intact

STAGE24-2D COMMITTED CHECKSUM GATE

Checksum file:
  results/stage24_cross_dataset/stage24_2_primary_target_openings/stage24_2d_bridge70_flag_corrected/checksums.sha256

5b819f03d2f336dde1415cf08f48f1963fba15d725661dcebe3c37f5f5aaf53c  stage24_2d_bridge70_flag_corrected_result.json
09e9d90ef1adef44c0a50f80d3e1771b566ed0ad

RuntimeError: Stage24 bridge dimension is not 70.

In [7]:
# =================================================================================================
# STAGE28-1C-C0-R1 — STAGE24 FLAG_CORRECTED HASH-SEMANTICS PREFLIGHT (SCHEMA-CORRECTED)
#
# Durable scientific parent:
#   ba011ec01f1399939111b24664ebd5d66c630f95
#
# CORRECTION FROM C0
# ------------------
# The frozen Stage24-2D result does NOT define:
#
#     representation.bridge_dimension
#
# The committed representation instead defines its dimensionality through:
#
#     representation.feature_count
#     len(representation.feature_order)
#     len(representation.flag_corrected_mapping)
#
# All three must equal 70.
#
# This is a diagnostic-code correction only.
# It changes no frozen scientific protocol, population, mapping, model, threshold,
# membership, target, fit budget, or statistical decision.
#
# PURPOSE
# -------
# Recover, using committed metadata/source only:
#
#   1. exact Stage24-2D artifact identity
#   2. exact 70-feature FLAG_CORRECTED mapping metadata
#   3. all historical SHA256/matrix audit fields
#   4. exact Stage24 matrix-hashing implementation
#   5. any reset-safe Stage24 cache/checkpoint evidence
#
# ZERO:
#   CICIDS2017 predictor values read
#   CICIDS2017 labels read
#   IDS2018 predictor values read
#   final-holdout openings
#   model fits
#   model inference
#   threshold selection
#   Git modifications
#
# No durable Stage28 commit is created.
# =================================================================================================

from __future__ import annotations

import hashlib
import json
import re
import subprocess
from pathlib import Path
from typing import Any


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_HEAD = (
    "ba011ec01f1399939111b24664ebd5d66c630f95"
)

STAGE24_ROOT = (
    REPO
    / "results"
    / "stage24_cross_dataset"
)

STAGE24_2D_DIR = (
    STAGE24_ROOT
    / "stage24_2_primary_target_openings"
    / "stage24_2d_bridge70_flag_corrected"
)

STAGE24_RESULT = (
    STAGE24_2D_DIR
    / "stage24_2d_bridge70_flag_corrected_result.json"
)

STAGE24_CHECKSUMS = (
    STAGE24_2D_DIR
    / "checksums.sha256"
)

STAGE24_SCRIPT = (
    REPO
    / "scripts"
    / "stage24"
    / "stage24_cross_dataset_generalization.py"
)

STAGE22_RUNTIME_RECEIPT = Path(
    "/kaggle/working/"
    "stage28_1c_b_stage22_cache_recovery_receipt.json"
)

STAGE22_RUNTIME_CACHE = Path(
    "/kaggle/working/"
    "stage22r_1c_70f_development_cache"
)

DIAGNOSTIC_RECEIPT = Path(
    "/kaggle/working/"
    "stage28_1c_c0_r1_stage24_hash_semantics_preflight.json"
)

EXPECTED_FEATURE_COUNT = 70
EXPECTED_STAGE22_CACHE_FILES = 8
EXPECTED_STAGE22_ROWS = 14_412_403
EXPECTED_NEW_FITS = 108


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title: str) -> None:
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args) -> str:
    return (
        run(
            ["git", *args]
        ).stdout
        or ""
    ).strip()


def require_file(path: Path) -> Path:
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path: Path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path: Path,
    chunk_size: int = 4 * 1024 * 1024,
) -> str:

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def is_sha256(value: Any) -> bool:
    return (
        isinstance(value, str)
        and re.fullmatch(
            r"[0-9a-fA-F]{64}",
            value.strip(),
        )
        is not None
    )


def walk_json(
    obj: Any,
    path: str = "$",
):
    if isinstance(obj, dict):

        for key, value in obj.items():

            child_path = (
                f"{path}.{key}"
            )

            yield (
                child_path,
                str(key),
                value,
            )

            yield from walk_json(
                value,
                child_path,
            )

    elif isinstance(obj, list):

        for i, value in enumerate(obj):

            child_path = (
                f"{path}[{i}]"
            )

            yield (
                child_path,
                f"[{i}]",
                value,
            )

            yield from walk_json(
                value,
                child_path,
            )


def scalar_summary_dict(
    d: dict,
) -> dict:

    out = {}

    for key, value in d.items():

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        ) or value is None:

            out[key] = value

        elif (
            isinstance(value, list)
            and len(value) <= 12
            and all(
                isinstance(
                    x,
                    (
                        str,
                        int,
                        float,
                        bool,
                        type(None),
                    ),
                )
                for x in value
            )
        ):

            out[key] = value

    return out


def print_context_blocks(
    text: str,
    terms: list[str],
    context: int = 12,
    max_blocks: int = 60,
) -> dict:

    lines = text.splitlines()

    matched_lines = set()

    per_term = {}

    for term in terms:

        hits = [
            i
            for i, line
            in enumerate(lines)
            if term.lower()
            in line.lower()
        ]

        per_term[term] = [
            i + 1
            for i in hits
        ]

        matched_lines.update(
            hits
        )

    if not matched_lines:

        print(
            "[INFO] No requested source-code "
            "search terms were found."
        )

        return per_term

    ranges = []

    for idx in sorted(
        matched_lines
    ):

        start = max(
            0,
            idx - context,
        )

        stop = min(
            len(lines),
            idx + context + 1,
        )

        if (
            ranges
            and start
            <= ranges[-1][1]
        ):

            ranges[-1] = (
                ranges[-1][0],
                max(
                    ranges[-1][1],
                    stop,
                ),
            )

        else:

            ranges.append(
                (
                    start,
                    stop,
                )
            )

    ranges = ranges[
        :max_blocks
    ]

    for block_no, (
        start,
        stop,
    ) in enumerate(
        ranges,
        start=1,
    ):

        print()
        print(
            "-" * 100
        )

        print(
            f"SOURCE CONTEXT BLOCK {block_no}"
            f" — lines {start + 1}:{stop}"
        )

        print(
            "-" * 100
        )

        for i in range(
            start,
            stop,
        ):

            marker = (
                ">>>"
                if i in matched_lines
                else "   "
            )

            print(
                f"{marker} {i + 1:6d}: "
                f"{lines[i]}"
            )

    return per_term


# =================================================================================================
# 2. DURABLE-PARENT / CLEAN-WORKTREE GATE
# =================================================================================================

banner(
    "STAGE28-1C-C0-R1 — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote = (
    remote_line
    .split()[0]
)

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD:",
    EXPECTED_HEAD,
)

print(
    "Local HEAD   :",
    head,
)

print(
    "origin/main  :",
    origin,
)

print(
    "Remote main  :",
    remote,
)

print(
    "Branch       :",
    branch,
)

print(
    "Git clean    :",
    not bool(status),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_HEAD
):
    raise RuntimeError(
        "Durable Stage28 parent mismatch."
    )

if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean:\n"
        + status
    )


print()
print(
    "[PASS] exact Stage28 durable parent"
)


# =================================================================================================
# 3. REQUIRE STAGE28-1C-B RECOVERY STATE
# =================================================================================================

banner(
    "STAGE28-1C-B RUNTIME RECOVERY GATE"
)

c_b = read_json(
    require_file(
        STAGE22_RUNTIME_RECEIPT
    )
)


print(
    "Receipt:",
    STAGE22_RUNTIME_RECEIPT,
)

print(
    "Status :",
    c_b.get(
        "status"
    ),
)

print(
    "Parent :",
    c_b.get(
        "durable_scientific_parent"
    ),
)


if (
    c_b.get(
        "durable_scientific_parent"
    )
    != EXPECTED_HEAD
):
    raise RuntimeError(
        "Stage28-1C-B parent mismatch."
    )

if (
    c_b.get(
        "status"
    )
    != "BYTE_EXACT_STAGE22_RUNTIME_CACHE_RECOVERED"
):
    raise RuntimeError(
        "Stage28-1C-B cache recovery is not verified."
    )

cache_meta = (
    c_b.get(
        "cache",
        {},
    )
)

if int(
    cache_meta.get(
        "file_count",
        -1,
    )
) != EXPECTED_STAGE22_CACHE_FILES:
    raise RuntimeError(
        "Stage22 runtime-cache file count mismatch."
    )

if int(
    cache_meta.get(
        "rows",
        -1,
    )
) != EXPECTED_STAGE22_ROWS:
    raise RuntimeError(
        "Stage22 runtime-cache row count mismatch."
    )


scientific_access = (
    c_b.get(
        "scientific_access",
        {}
    )
)

for key, value in scientific_access.items():

    if int(value) != 0:

        raise RuntimeError(
            "Stage28-1C-B contains non-zero "
            f"scientific-access counter: "
            f"{key}={value}"
        )


fit_ledger = (
    c_b.get(
        "fit_ledger",
        {}
    )
)

if int(
    fit_ledger.get(
        "new_consumed",
        -1,
    )
) != 0:
    raise RuntimeError(
        "A Stage28 fit was consumed before C0-R1."
    )

if int(
    fit_ledger.get(
        "new_remaining",
        -1,
    )
) != EXPECTED_NEW_FITS:
    raise RuntimeError(
        "Stage28 remaining-fit ledger mismatch."
    )


runtime_entries = []

if STAGE22_RUNTIME_CACHE.is_dir():

    runtime_entries = sorted(
        p.name
        for p
        in STAGE22_RUNTIME_CACHE.iterdir()
        if p.is_file()
    )


print(
    "Runtime cache files:",
    len(
        runtime_entries
    ),
)

print(
    "New fits consumed   :",
    fit_ledger[
        "new_consumed"
    ],
)

print(
    "New fits remaining  :",
    fit_ledger[
        "new_remaining"
    ],
)


if len(
    runtime_entries
) != EXPECTED_STAGE22_CACHE_FILES:
    raise RuntimeError(
        "Stage22 runtime cache is no longer "
        "present with all 8 files."
    )


print()
print(
    "[PASS] Stage28-1C-B runtime recovery intact"
)


# =================================================================================================
# 4. VERIFY STAGE24-2D COMMITTED CHECKSUMS
# =================================================================================================

banner(
    "STAGE24-2D COMMITTED CHECKSUM GATE"
)

require_file(
    STAGE24_RESULT
)

require_file(
    STAGE24_CHECKSUMS
)

checksum_text = (
    STAGE24_CHECKSUMS.read_text(
        encoding="utf-8"
    )
)

print(
    "Checksum file:"
)

print(
    " ",
    STAGE24_CHECKSUMS.relative_to(
        REPO
    )
)

print()

print(
    checksum_text.rstrip()
)


checksum_entries = []

for raw_line in checksum_text.splitlines():

    line = raw_line.strip()

    if not line:
        continue

    m = re.match(
        r"^([0-9a-fA-F]{64})\s+\*?(.+?)\s*$",
        line,
    )

    if not m:
        raise RuntimeError(
            "Unrecognized checksums.sha256 line:\n"
            + raw_line
        )

    sha = (
        m.group(1)
        .lower()
    )

    rel_name = (
        m.group(2)
        .strip()
    )

    candidate = (
        STAGE24_2D_DIR
        / rel_name
    )

    require_file(
        candidate
    )

    actual = sha256_file(
        candidate
    )

    print()
    print(
        "[CHECK]",
        rel_name,
    )

    print(
        "  expected:",
        sha,
    )

    print(
        "  actual  :",
        actual,
    )

    if actual != sha:
        raise RuntimeError(
            f"Stage24 checksum mismatch: "
            f"{rel_name}"
        )

    checksum_entries.append(
        {
            "file":
                rel_name,

            "sha256":
                sha,
        }
    )


if not checksum_entries:
    raise RuntimeError(
        "Stage24 checksum file contained no entries."
    )


print()
print(
    "[PASS] all Stage24-2D committed "
    "checksum entries exact"
)


# =================================================================================================
# 5. LOAD FROZEN STAGE24 METADATA
# =================================================================================================

banner(
    "STAGE24-2D RESULT STRUCTURE"
)

stage24 = read_json(
    STAGE24_RESULT
)


print(
    "Result:",
    STAGE24_RESULT.relative_to(
        REPO
    ),
)

print()

print(
    "Top-level keys:"
)

for key in stage24.keys():
    print(
        " ",
        key,
    )


representation = stage24.get(
    "representation"
)

if not isinstance(
    representation,
    dict,
):
    raise RuntimeError(
        "Stage24 result has no representation object."
    )


bridge_meta = stage24.get(
    "bridge",
    {}
)

if not isinstance(
    bridge_meta,
    dict,
):
    bridge_meta = {}


print()
print(
    "representation keys:"
)

for key in representation.keys():
    print(
        " ",
        key,
    )


print()
print(
    "bridge keys:"
)

for key in bridge_meta.keys():
    print(
        " ",
        key,
    )


feature_count_declared = (
    representation.get(
        "feature_count"
    )
)

feature_order = (
    representation.get(
        "feature_order"
    )
)

flag_mapping = (
    representation.get(
        "flag_corrected_mapping"
    )
)

feature_order_count = (
    len(feature_order)
    if isinstance(
        feature_order,
        list,
    )
    else None
)

mapping_count = (
    len(flag_mapping)
    if isinstance(
        flag_mapping,
        dict,
    )
    else None
)


print()
print(
    "Declared feature_count:",
    feature_count_declared,
)

print(
    "Feature-order count   :",
    feature_order_count,
)

print(
    "Mapping-entry count   :",
    mapping_count,
)

print(
    "Parse dtype           :",
    representation.get(
        "parse_dtype"
    ),
)

print(
    "Positive infinity     :",
    representation.get(
        "positive_infinity"
    ),
)

print(
    "Negative infinity     :",
    representation.get(
        "negative_infinity"
    ),
)

print(
    "Input semantics       :",
    representation.get(
        "input_semantics"
    ),
)

print(
    "Scaling               :",
    representation.get(
        "scaling"
    ),
)

print(
    "Explicit imputation   :",
    representation.get(
        "explicit_imputation"
    ),
)

print(
    "Changed mapping count :",
    representation.get(
        "changed_physical_mapping_count"
    ),
)

print(
    "Changed source feats  :",
    representation.get(
        "changed_source_features"
    ),
)

print(
    "ACK mapping invariant :",
    representation.get(
        "ack_mapping_invariant"
    ),
)


# =================================================================================================
# 6. CORRECTED 70-DIMENSIONALITY GATE
# =================================================================================================

banner(
    "CORRECTED FROZEN REPRESENTATION DIMENSIONALITY GATE"
)

if feature_count_declared is None:
    raise RuntimeError(
        "representation.feature_count is missing."
    )

if int(
    feature_count_declared
) != EXPECTED_FEATURE_COUNT:
    raise RuntimeError(
        "Frozen representation.feature_count "
        f"is {feature_count_declared}, expected 70."
    )

if not isinstance(
    feature_order,
    list,
):
    raise RuntimeError(
        "representation.feature_order is not a list."
    )

if feature_order_count != EXPECTED_FEATURE_COUNT:
    raise RuntimeError(
        "Frozen feature_order does not contain 70 features."
    )

if not isinstance(
    flag_mapping,
    dict,
):
    raise RuntimeError(
        "representation.flag_corrected_mapping "
        "is not a dictionary."
    )

if mapping_count != EXPECTED_FEATURE_COUNT:
    raise RuntimeError(
        "Frozen FLAG_CORRECTED mapping "
        "does not contain 70 entries."
    )

missing_mapping_features = [
    feature
    for feature in feature_order
    if feature not in flag_mapping
]

if missing_mapping_features:
    raise RuntimeError(
        "Frozen mapping is missing feature-order entries:\n"
        + "\n".join(
            repr(x)
            for x
            in missing_mapping_features
        )
    )


if (
    representation.get(
        "parse_dtype"
    )
    != "float64"
):
    raise RuntimeError(
        "Stage24 parse_dtype is not float64."
    )

if (
    representation.get(
        "input_semantics"
    )
    != "FROZEN_POSITIONAL_ORDER"
):
    raise RuntimeError(
        "Unexpected Stage24 input semantics."
    )


print(
    "representation.feature_count =",
    feature_count_declared,
)

print(
    "len(feature_order)           =",
    feature_order_count,
)

print(
    "len(flag_corrected_mapping) =",
    mapping_count,
)

print()

print(
    "[PASS] frozen Stage24 representation "
    "is exactly 70-dimensional"
)


# =================================================================================================
# 7. PRINT EXACT FEATURE ORDER + PHYSICAL MAPPING
# =================================================================================================

banner(
    "FROZEN STAGE24 70-FEATURE FLAG_CORRECTED MAPPING"
)

for i, feature in enumerate(
    feature_order
):

    mapped = flag_mapping[
        feature
    ]

    print(
        f"{i:02d}. "
        f"{feature!r}"
        f"  ->  "
        f"{mapped!r}"
    )


# =================================================================================================
# 8. PRINT BRIDGE METADATA
# =================================================================================================

banner(
    "STAGE24 BRIDGE METADATA"
)

if bridge_meta:

    print(
        json.dumps(
            bridge_meta,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    )

else:

    print(
        "<EMPTY OR ABSENT>"
    )


# =================================================================================================
# 9. EXTRACT FEATURE_MATRIX_AUDIT + NUMERIC_AUDIT VERBATIM
# =================================================================================================

banner(
    "STAGE24 FEATURE_MATRIX_AUDIT"
)

feature_matrix_audit = stage24.get(
    "feature_matrix_audit"
)

print(
    json.dumps(
        feature_matrix_audit,
        indent=2,
        ensure_ascii=False,
        default=str,
    )
)


banner(
    "STAGE24 NUMERIC_AUDIT"
)

numeric_audit = stage24.get(
    "numeric_audit"
)

print(
    json.dumps(
        numeric_audit,
        indent=2,
        ensure_ascii=False,
        default=str,
    )
)


# =================================================================================================
# 10. DISCOVER EVERY HASH/MATRIX/DTYPE/SHAPE/ROW FIELD
# =================================================================================================

banner(
    "STAGE24 RESULT — HASH / MATRIX / DTYPE / SHAPE / ROW METADATA"
)

interesting_key_re = re.compile(
    r"(sha|hash|matrix|float64|dtype|shape|row|mapping|schema|column|feature)",
    flags=re.IGNORECASE,
)

interesting_scalars = []

all_sha256_values = []


for path, key, value in walk_json(
    stage24
):

    if (
        isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
                type(None),
            ),
        )
        and interesting_key_re.search(
            key
        )
    ):

        interesting_scalars.append(
            (
                path,
                value,
            )
        )

    if is_sha256(
        value
    ):

        all_sha256_values.append(
            (
                path,
                value.lower(),
            )
        )


for path, value in interesting_scalars:

    print(
        f"{path} = {value!r}"
    )


print()
print(
    "Total SHA256-looking scalar values:",
    len(
        all_sha256_values
    ),
)


# =================================================================================================
# 11. LIKELY MATRIX/HASH RECORDS
# =================================================================================================

banner(
    "LIKELY PER-SOURCE / PER-MATRIX HASH RECORDS"
)

candidate_records = []


for path, key, value in walk_json(
    stage24
):

    if not isinstance(
        value,
        dict,
    ):
        continue

    keys_lower = [
        str(k).lower()
        for k
        in value.keys()
    ]

    joined = " ".join(
        keys_lower
    )

    has_matrix = (
        "matrix"
        in joined
    )

    has_sha = (
        "sha"
        in joined
        or "hash"
        in joined
    )

    has_float64 = (
        "float64"
        in joined
        or any(
            isinstance(v, str)
            and "float64"
            in v.lower()
            for v
            in value.values()
        )
    )

    has_rows = (
        "row"
        in joined
    )

    if (
        (
            has_matrix
            and has_sha
        )
        or (
            has_float64
            and has_sha
        )
        or (
            has_rows
            and has_sha
            and "feature"
            in joined
        )
    ):

        candidate_records.append(
            {
                "path":
                    path,

                "summary":
                    scalar_summary_dict(
                        value
                    ),
            }
        )


if candidate_records:

    for i, record in enumerate(
        candidate_records,
        start=1,
    ):

        print()
        print(
            f"[CANDIDATE {i}]"
        )

        print(
            "Path:",
            record[
                "path"
            ],
        )

        print(
            json.dumps(
                record[
                    "summary"
                ],
                indent=2,
                ensure_ascii=False,
                default=str,
            )
        )

else:

    print(
        "[INFO] No explicit matrix/hash "
        "dictionary record found."
    )


# =================================================================================================
# 12. EXPLICIT MATRIX-HASH FIELD SEARCH
# =================================================================================================

banner(
    "EXPLICIT HISTORICAL MATRIX-HASH FIELD SEARCH"
)

target_key_terms = [
    "flag_corrected_feature_matrix_float64_sha256",
    "feature_matrix_float64_sha256",
    "feature_matrix_sha256",
    "matrix_float64_sha256",
    "matrix_sha256",
    "feature_sha256",
    "float64_sha256",
]

explicit_matrix_hash_hits = []


for path, key, value in walk_json(
    stage24
):

    key_lower = (
        key.lower()
    )

    if any(
        term in key_lower
        for term
        in target_key_terms
    ):

        explicit_matrix_hash_hits.append(
            (
                path,
                value,
            )
        )


if explicit_matrix_hash_hits:

    for path, value in explicit_matrix_hash_hits:

        print(
            f"[FOUND] {path}"
        )

        print(
            "        ",
            value,
        )

else:

    print(
        "[INFO] No exact target-style matrix-hash "
        "field name found."
    )


# =================================================================================================
# 13. ALL SHA256 VALUES
# =================================================================================================

banner(
    "ALL STAGE24 SHA256 VALUES WITH JSON PATHS"
)

if all_sha256_values:

    for i, (
        path,
        value,
    ) in enumerate(
        all_sha256_values,
        start=1,
    ):

        print(
            f"{i:03d}. {path}"
        )

        print(
            f"     {value}"
        )

else:

    print(
        "<NONE>"
    )


# =================================================================================================
# 14. INSPECT FROZEN STAGE24 IMPLEMENTATION AS TEXT ONLY
# =================================================================================================

banner(
    "STAGE24 SOURCE-CODE MATRIX-HASH IMPLEMENTATION SEARCH"
)

require_file(
    STAGE24_SCRIPT
)

script_text = (
    STAGE24_SCRIPT.read_text(
        encoding="utf-8",
        errors="replace",
    )
)


print(
    "Script:",
    STAGE24_SCRIPT.relative_to(
        REPO
    ),
)

print(
    "Bytes :",
    STAGE24_SCRIPT.stat().st_size,
)

print(
    "Lines :",
    len(
        script_text.splitlines()
    ),
)


SOURCE_SEARCH_TERMS = [
    "feature_matrix_audit",
    "flag_corrected_feature_matrix_float64_sha256",
    "feature_matrix_float64_sha256",
    "matrix_float64_sha256",
    "feature_matrix_sha256",
    "matrix_sha256",
    "float64_sha256",
    "tobytes",
    "ascontiguousarray",
    "np.ascontiguousarray",
    "hashlib.sha256",
    "sha256",
    "C_CONTIGUOUS",
    "dtype",
    "float64",
]


source_hits = print_context_blocks(
    script_text,
    SOURCE_SEARCH_TERMS,
    context=14,
    max_blocks=80,
)


print()
print(
    "Search-term hit counts:"
)

for term in SOURCE_SEARCH_TERMS:

    lines = source_hits.get(
        term,
        [],
    )

    print(
        f"  {term!r}: "
        f"{len(lines)} hit(s)"
        + (
            f" @ {lines[:30]}"
            if lines
            else ""
        )
    )


# =================================================================================================
# 15. SEARCH ALL COMMITTED STAGE24 TEXT FOR MATRIX HASH SEMANTICS
# =================================================================================================

banner(
    "ALL STAGE24 COMMITTED TEXT — MATRIX/HASH SEMANTICS SEARCH"
)

tracked_stage24 = (
    git(
        "ls-files",
        "results/stage24_cross_dataset",
        "scripts/stage24",
        "docs",
    )
    .splitlines()
)

safe_text_suffixes = {
    ".json",
    ".md",
    ".txt",
    ".sha256",
    ".py",
    ".yaml",
    ".yml",
    ".toml",
}

MATRIX_TEXT_TERMS = [
    "feature_matrix_audit",
    "feature_matrix_float64",
    "matrix_float64",
    "matrix sha",
    "matrix_sha",
    "tobytes",
    "ascontiguousarray",
    "flag_corrected_mapping",
]

matrix_text_hits = []


for rel in tracked_stage24:

    path = (
        REPO
        / rel
    )

    if (
        not path.is_file()
        or path.suffix.lower()
        not in safe_text_suffixes
    ):
        continue

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    for line_no, line in enumerate(
        text.splitlines(),
        start=1,
    ):

        if any(
            term.lower()
            in line.lower()
            for term
            in MATRIX_TEXT_TERMS
        ):

            matrix_text_hits.append(
                {
                    "file":
                        rel,

                    "line":
                        line_no,

                    "text":
                        line.strip()[
                            :1400
                        ],
                }
            )


if matrix_text_hits:

    for hit in matrix_text_hits[
        :400
    ]:

        print(
            f"{hit['file']}:{hit['line']}: "
            f"{hit['text']}"
        )

    if len(
        matrix_text_hits
    ) > 400:

        print(
            f"... truncated "
            f"{len(matrix_text_hits) - 400} "
            f"additional hits"
        )

else:

    print(
        "<NONE>"
    )


# =================================================================================================
# 16. SEARCH RESET-SAFE CACHE / CHECKPOINT EVIDENCE
# =================================================================================================

banner(
    "STAGE24 RESET-SAFE CACHE / KAGGLE CHECKPOINT SEARCH"
)

CACHE_SEARCH_TERMS = [
    "dataset_id",
    "kaggle_dataset",
    "kagglehub",
    "private_kaggle",
    "reset_safe",
    "reset-safe",
    "runtime_cache",
    "cache_path",
    "cache_checkpoint",
    "checkpoint",
    "stage24_cache",
]

cache_text_hits = []

candidate_filenames = []


for rel in tracked_stage24:

    path = (
        REPO
        / rel
    )

    name_lower = (
        path.name.lower()
    )

    if any(
        token in name_lower
        for token in (
            "cache",
            "checkpoint",
            "flag_corrected",
            "matrix",
        )
    ):

        candidate_filenames.append(
            rel
        )

    if (
        not path.is_file()
        or path.suffix.lower()
        not in safe_text_suffixes
    ):
        continue

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    for line_no, line in enumerate(
        text.splitlines(),
        start=1,
    ):

        if any(
            term.lower()
            in line.lower()
            for term
            in CACHE_SEARCH_TERMS
        ):

            cache_text_hits.append(
                {
                    "file":
                        rel,

                    "line":
                        line_no,

                    "text":
                        line.strip()[
                            :1400
                        ],
                }
            )


print(
    "Candidate filenames:"
)

if candidate_filenames:

    for rel in sorted(
        set(
            candidate_filenames
        )
    ):

        print(
            " ",
            rel,
        )

else:

    print(
        "  <NONE>"
    )


print()
print(
    "Cache/checkpoint text hits:"
)


if cache_text_hits:

    for hit in cache_text_hits[
        :300
    ]:

        print(
            f"{hit['file']}:{hit['line']}: "
            f"{hit['text']}"
        )

    if len(
        cache_text_hits
    ) > 300:

        print(
            f"... truncated "
            f"{len(cache_text_hits) - 300} "
            f"additional hits"
        )

else:

    print(
        "  <NONE>"
    )


# =================================================================================================
# 17. SEMANTICS EVIDENCE SUMMARY
# =================================================================================================

banner(
    "HASH-SEMANTICS RESOLUTION SUMMARY"
)

script_term_counts = {
    term:
        len(
            source_hits.get(
                term,
                []
            )
        )

    for term in SOURCE_SEARCH_TERMS
}


has_explicit_matrix_hash_json = bool(
    explicit_matrix_hash_hits
)

has_matrix_hash_candidate = bool(
    candidate_records
)

has_tobytes_impl = (
    script_term_counts.get(
        "tobytes",
        0,
    )
    > 0
)

has_sha_impl = (
    script_term_counts.get(
        "hashlib.sha256",
        0,
    )
    > 0
    or script_term_counts.get(
        "sha256",
        0,
    )
    > 0
)

has_contiguous_impl = (
    script_term_counts.get(
        "ascontiguousarray",
        0,
    )
    > 0
    or script_term_counts.get(
        "np.ascontiguousarray",
        0,
    )
    > 0
)

has_feature_matrix_audit = (
    isinstance(
        feature_matrix_audit,
        (
            dict,
            list,
        ),
    )
)

hash_semantics_evidence_present = bool(
    has_feature_matrix_audit
    or has_explicit_matrix_hash_json
    or has_matrix_hash_candidate
    or (
        has_sha_impl
        and has_tobytes_impl
    )
)


print(
    "feature_matrix_audit present       :",
    has_feature_matrix_audit,
)

print(
    "Explicit matrix-hash JSON hit(s)   :",
    len(
        explicit_matrix_hash_hits
    ),
)

print(
    "Matrix/hash candidate records      :",
    len(
        candidate_records
    ),
)

print(
    "Source uses tobytes                :",
    has_tobytes_impl,
)

print(
    "Source uses SHA256                 :",
    has_sha_impl,
)

print(
    "Source uses contiguous conversion  :",
    has_contiguous_impl,
)

print(
    "Committed matrix/hash text hits    :",
    len(
        matrix_text_hits
    ),
)

print(
    "Stage24 cache/checkpoint text hits :",
    len(
        cache_text_hits
    ),
)

print(
    "Hash-semantics evidence present    :",
    hash_semantics_evidence_present,
)


# =================================================================================================
# 18. RUNTIME-ONLY DIAGNOSTIC RECEIPT
# =================================================================================================

diagnostic_receipt = {

    "stage":
        "Stage28-1C-C0-R1",

    "type":
        "METADATA_ONLY_STAGE24_HASH_SEMANTICS_PREFLIGHT_SCHEMA_CORRECTED",

    "correction": {
        "previous_invalid_assumption":
            "representation.bridge_dimension",

        "authoritative_dimension_fields": [
            "representation.feature_count",
            "len(representation.feature_order)",
            "len(representation.flag_corrected_mapping)",
        ],

        "scientific_protocol_changed":
            False,
    },

    "durable_scientific_parent":
        EXPECTED_HEAD,

    "stage22_runtime_recovery": {
        "status":
            c_b.get(
                "status"
            ),

        "files_present":
            len(
                runtime_entries
            ),

        "rows":
            cache_meta.get(
                "rows"
            ),
    },

    "stage24_2d": {
        "result":
            str(
                STAGE24_RESULT.relative_to(
                    REPO
                )
            ),

        "checksums":
            checksum_entries,

        "representation_feature_count":
            feature_count_declared,

        "feature_order_count":
            feature_order_count,

        "mapping_count":
            mapping_count,

        "feature_order":
            feature_order,

        "flag_corrected_mapping":
            flag_mapping,

        "parse_dtype":
            representation.get(
                "parse_dtype"
            ),

        "positive_infinity":
            representation.get(
                "positive_infinity"
            ),

        "negative_infinity":
            representation.get(
                "negative_infinity"
            ),

        "input_semantics":
            representation.get(
                "input_semantics"
            ),

        "scaling":
            representation.get(
                "scaling"
            ),

        "explicit_imputation":
            representation.get(
                "explicit_imputation"
            ),

        "changed_physical_mapping_count":
            representation.get(
                "changed_physical_mapping_count"
            ),

        "changed_source_features":
            representation.get(
                "changed_source_features"
            ),

        "ack_mapping_invariant":
            representation.get(
                "ack_mapping_invariant"
            ),

        "bridge_metadata":
            bridge_meta,

        "feature_matrix_audit":
            feature_matrix_audit,

        "numeric_audit":
            numeric_audit,

        "sha256_scalar_paths":
            [
                {
                    "path":
                        path,

                    "sha256":
                        value,
                }

                for path, value
                in all_sha256_values
            ],

        "explicit_matrix_hash_hits":
            [
                {
                    "path":
                        path,

                    "value":
                        value,
                }

                for path, value
                in explicit_matrix_hash_hits
            ],

        "candidate_matrix_hash_records":
            candidate_records,

        "script_search_hits":
            source_hits,

        "script_term_counts":
            script_term_counts,

        "matrix_text_hit_count":
            len(
                matrix_text_hits
            ),

        "hash_semantics_evidence_present":
            hash_semantics_evidence_present,
    },

    "stage24_reset_safe_cache_search": {
        "candidate_filenames":
            sorted(
                set(
                    candidate_filenames
                )
            ),

        "text_hit_count":
            len(
                cache_text_hits
            ),

        "text_hits":
            cache_text_hits[
                :300
            ],
    },

    "scientific_access": {
        "cicids2017_predictor_values_read":
            0,

        "cicids2017_labels_read":
            0,

        "ids2018_predictor_values_read":
            0,

        "final_holdout_openings":
            0,

        "model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,
    },

    "fit_ledger": {
        "new_authorized":
            EXPECTED_NEW_FITS,

        "new_consumed":
            0,

        "new_remaining":
            EXPECTED_NEW_FITS,
    },

    "git_modified":
        False,
}


DIAGNOSTIC_RECEIPT.write_text(
    json.dumps(
        diagnostic_receipt,
        indent=2,
        ensure_ascii=False,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)


print()
print(
    "Runtime-only diagnostic receipt:"
)

print(
    " ",
    DIAGNOSTIC_RECEIPT,
)


# =================================================================================================
# 19. FINAL GIT CLEANLINESS GATE
# =================================================================================================

banner(
    "FINAL GIT CLEANLINESS GATE"
)

final_head = git(
    "rev-parse",
    "HEAD",
)

final_origin = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head,
)

print(
    "origin/main:",
    final_origin,
)

print(
    "Git clean  :",
    not bool(
        final_status
    ),
)


if (
    final_head
    != EXPECTED_HEAD
    or final_origin
    != EXPECTED_HEAD
):
    raise RuntimeError(
        "Git parent changed during C0-R1."
    )

if final_status:
    raise RuntimeError(
        "Stage28-1C-C0-R1 changed Git:\n"
        + final_status
    )


print()
print(
    "[PASS] Git state untouched"
)


# =================================================================================================
# 20. FINAL
# =================================================================================================

banner(
    "STAGE28-1C-C0-R1 — PREFLIGHT COMPLETE"
)

print(
    "Durable parent:"
)

print(
    " ",
    EXPECTED_HEAD,
)

print()

print(
    "Corrected frozen dimensionality:"
)

print(
    "  representation.feature_count =",
    feature_count_declared,
)

print(
    "  len(feature_order)            =",
    feature_order_count,
)

print(
    "  len(flag_corrected_mapping)   =",
    mapping_count,
)

print()

print(
    "Stage24 numeric representation:"
)

print(
    "  parse dtype       =",
    representation.get(
        "parse_dtype"
    ),
)

print(
    "  input semantics   =",
    representation.get(
        "input_semantics"
    ),
)

print(
    "  +inf handling     =",
    representation.get(
        "positive_infinity"
    ),
)

print(
    "  -inf handling     =",
    representation.get(
        "negative_infinity"
    ),
)

print(
    "  scaling           =",
    representation.get(
        "scaling"
    ),
)

print(
    "  explicit impute   =",
    representation.get(
        "explicit_imputation"
    ),
)

print()

print(
    "Historical SHA256-looking fields =",
    len(
        all_sha256_values
    ),
)

print(
    "Explicit matrix-hash hits        =",
    len(
        explicit_matrix_hash_hits
    ),
)

print(
    "Matrix/hash candidate records    =",
    len(
        candidate_records
    ),
)

print(
    "Matrix/hash committed text hits  =",
    len(
        matrix_text_hits
    ),
)

print(
    "Hash implementation evidence     =",
    hash_semantics_evidence_present,
)

print(
    "Stage24 cache/checkpoint hits     =",
    len(
        cache_text_hits
    ),
)

print()

print(
    "Scientific access:"
)

print(
    "  CICIDS2017 PREDICTORS = 0"
)

print(
    "  CICIDS2017 LABELS     = 0"
)

print(
    "  IDS2018 PREDICTORS    = 0"
)

print(
    "  FINAL HOLDOUT         = 0"
)

print(
    "  MODEL FITS            = 0"
)

print(
    "  INFERENCE             = 0"
)

print(
    "  THRESHOLD SELECTION   = 0"
)

print()

print(
    "Fit ledger:"
)

print(
    "  authorized = 108"
)

print(
    "  consumed   = 0"
)

print(
    "  remaining  = 108"
)

print()

print(
    "NEXT:"
)

print(
    "  Return this C0-R1 output."
)

print(
    "  We will use the frozen feature_matrix_audit "
    "and source-code hash implementation to build "
    "Stage28-1C-C without guessing any byte convention."
)

print()
print(SEP)


STAGE28-1C-C0-R1 — EXACT DURABLE-PARENT GATE

Expected HEAD: ba011ec01f1399939111b24664ebd5d66c630f95
Local HEAD   : ba011ec01f1399939111b24664ebd5d66c630f95
origin/main  : ba011ec01f1399939111b24664ebd5d66c630f95
Remote main  : ba011ec01f1399939111b24664ebd5d66c630f95
Branch       : main
Git clean    : True

[PASS] exact Stage28 durable parent

STAGE28-1C-B RUNTIME RECOVERY GATE

Receipt: /kaggle/working/stage28_1c_b_stage22_cache_recovery_receipt.json
Status : BYTE_EXACT_STAGE22_RUNTIME_CACHE_RECOVERED
Parent : ba011ec01f1399939111b24664ebd5d66c630f95
Runtime cache files: 8
New fits consumed   : 0
New fits remaining  : 108

[PASS] Stage28-1C-B runtime recovery intact

STAGE24-2D COMMITTED CHECKSUM GATE

Checksum file:
  results/stage24_cross_dataset/stage24_2_primary_target_openings/stage24_2d_bridge70_flag_corrected/checksums.sha256

5b819f03d2f336dde1415cf08f48f1963fba15d725661dcebe3c37f5f5aaf53c  stage24_2d_bridge70_flag_corrected_result.json
09e9d90ef1adef44c0a50f80d3e1771b566ed

In [8]:
# =================================================================================================
# STAGE28-1C-C — CICIDS2017 FLAG_CORRECTED 70F MODEL-INPUT RECOVERY
#
# Durable scientific parent:
#   ba011ec01f1399939111b24664ebd5d66c630f95
#
# PURPOSE
# -------
# Reproduce the already-opened CICIDS2017 Stage24 FLAG_CORRECTED representation
# required by the frozen Stage28 execution manifest.
#
# SCIENTIFIC STATUS
# -----------------
# This is NOT a new target analysis.
#
# Stage24 already opened the complete 2,830,743-row effective CICIDS2017
# population and froze:
#
#   - exact source bytes
#   - exact 70-feature semantic mapping
#   - exact FLOAT64 matrix SHA256 for each of 8 source segments
#   - exact infinity->NaN policy
#
# Stage27 additionally froze:
#
#   - same exact 70-feature semantic mapping
#   - FLOAT64 parse -> FLOAT32 model representation
#   - exact Mon-Wed known-population FLOAT32 content SHA256
#
# This cell therefore performs a deterministic REPRODUCTION / CACHE RECOVERY.
#
# HARD GATES
# ----------
# 1. exact Git parent / clean repository
# 2. Stage22 byte-exact runtime cache still recovered
# 3. Stage28 fit ledger still 0 / 108 consumed
# 4. exact Stage24-2D result SHA
# 5. Stage24 feature order == Stage27 feature order
# 6. Stage24 FLAG_CORRECTED mapping == Stage27 adapter mapping
# 7. Stage27 adapter mapping SHA = frozen historical value
# 8. all 8 CICIDS2017 source bytes exact
# 9. reproduce all 8 historical Stage24 FLOAT64 matrix SHA256 values
# 10. reproduce all 8 Stage24 numeric-audit counts
# 11. materialize one global FLOAT32 matrix in Stage27 global-row order
# 12. reproduce Stage27 historical Mon-Wed:
#       global-index content SHA256
#       binary-label content SHA256
#       FLOAT32 feature content SHA256
#
# RUNTIME OUTPUT ONLY
# -------------------
# /kaggle/working/stage28_1c_runtime_cache/cicids2017/
#     global_features_float32.npy
#
# /kaggle/working/stage28_1c_c_cicids2017_70f_recovery_receipt.json
#
# NO GIT WRITE / COMMIT IN THIS CELL.
#
# ZERO:
#   model fits
#   model inference
#   threshold selection
#   bootstrap
#   target-driven feature selection
#   target-driven model selection
#   target-driven threshold selection
#
# IMPORTANT COUNTER SEMANTICS
# ---------------------------
# CICIDS2017 predictor rows WILL be read here:
#
#       2,830,743
#
# because that is the purpose of this deterministic runtime-cache recovery.
#
# This does NOT consume a NEW scientific target opening because the exact same
# full population was already opened and frozen in Stage24, and the recovered
# matrices are required to match those historical Stage24 hashes exactly.
# =================================================================================================

from __future__ import annotations

import gc
import hashlib
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_HEAD = (
    "ba011ec01f1399939111b24664ebd5d66c630f95"
)

EXPECTED_TOTAL_ROWS = 2_830_743

EXPECTED_FEATURE_COUNT = 70

EXPECTED_STAGE27_ADAPTER_SHA = (
    "88d7f3e133e7d20ee05dc4618c6102f0c420936e0fe72a9a094d951a9c4dad7a"
)

EXPECTED_STAGE24_RESULT_SHA = (
    "5b819f03d2f336dde1415cf08f48f1963fba15d725661dcebe3c37f5f5aaf53c"
)

EXPECTED_STAGE27_MONWED_ROWS = 1_668_519

EXPECTED_STAGE27_MONWED_FEATURE_SHA = (
    "fc3137b10bbb2542240df3d380286b88f6ea282aba200fd893e6445391855ccf"
)

EXPECTED_STAGE27_MONWED_LABEL_SHA = (
    "cc0d2a0d3b8d57461a0374e808dbbf942832a81fc9fcb5541eb8769cb207235b"
)

EXPECTED_STAGE27_MONWED_INDEX_SHA = (
    "b001047298984aa5db11b80ebbdb12fe48a2277ee7ed62ad1a27817ef67adc7e"
)

MONWED_GLOBAL_STOP_EXCLUSIVE = 1_668_530

EXPECTED_HEARTBLEED_EXCLUDED = 11

STAGE24_RESULT_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_2_primary_target_openings"
    / "stage24_2d_bridge70_flag_corrected"
    / "stage24_2d_bridge70_flag_corrected_result.json"
)

STAGE27_FEATURE_SPEC_PATH = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_0_protocol_lock"
    / "feature_representation.json"
)

STAGE27_SOURCE_RECEIPT_PATH = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
    / "source_effective_population_receipt.json"
)

STAGE27_MONWED_RECEIPT_PATH = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1b_prefit_feature_materialization"
    / "monwed_known_feature_cache_receipt.json"
)

STAGE28_1B_FREEZE_PATH = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_1b_random_loao_membership_and_execution_lock"
    / "stage28_1b_freeze_record.json"
)

STAGE22_RECOVERY_RECEIPT = Path(
    "/kaggle/working/"
    "stage28_1c_b_stage22_cache_recovery_receipt.json"
)

STAGE22_RUNTIME_CACHE = Path(
    "/kaggle/working/"
    "stage22r_1c_70f_development_cache"
)

GLOBAL_FAMILY_CODES_PATH = Path(
    "/kaggle/working/"
    "stage28_1a_runtime_cache/"
    "stage27_global_cache/"
    "global_family_codes.npy"
)

SOURCE_ROOT = Path(
    "/kaggle/working/"
    "stage27_cicids2017_sources"
)

CACHE_ROOT = Path(
    "/kaggle/working/"
    "stage28_1c_runtime_cache/"
    "cicids2017"
)

GLOBAL_FEATURE_PATH = (
    CACHE_ROOT
    / "global_features_float32.npy"
)

RUNTIME_RECEIPT_PATH = Path(
    "/kaggle/working/"
    "stage28_1c_c_cicids2017_70f_recovery_receipt.json"
)

EXPECTED_NEW_FITS = 108

os.environ[
    "CUDA_VISIBLE_DEVICES"
] = ""


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return (
        run(
            ["git", *args]
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(
        path
    )

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(
            path
        ).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


# -------------------------------------------------------------------------------------------------
# Exact Stage24 array-hash semantics:
#
#     arr = np.ascontiguousarray(array)
#     hashlib.sha256(arr.view(np.uint8)).hexdigest()
#
# -------------------------------------------------------------------------------------------------

def sha256_array_stage24(
    array,
):
    arr = np.ascontiguousarray(
        array
    )

    return hashlib.sha256(
        arr.view(
            np.uint8
        )
    ).hexdigest()


def sha256_array_content_blocked(
    array,
    block_rows=65_536,
):
    """
    Same raw C-order content semantics as Stage24 / Stage27,
    but bounded-memory for large memmaps.
    """

    h = hashlib.sha256()

    if array.ndim == 1:

        for start in range(
            0,
            array.shape[0],
            block_rows,
        ):

            stop = min(
                start
                + block_rows,
                array.shape[0],
            )

            block = np.ascontiguousarray(
                array[
                    start:stop
                ]
            )

            h.update(
                block.view(
                    np.uint8
                )
            )

    elif array.ndim == 2:

        for start in range(
            0,
            array.shape[0],
            block_rows,
        ):

            stop = min(
                start
                + block_rows,
                array.shape[0],
            )

            block = np.ascontiguousarray(
                array[
                    start:stop,
                    :
                ]
            )

            h.update(
                block.view(
                    np.uint8
                )
            )

    else:

        raise RuntimeError(
            f"Unsupported ndim={array.ndim}"
        )

    return h.hexdigest()


def sha256_selected_matrix_rows(
    matrix,
    global_idx,
    block_rows=65_536,
):
    """
    Hash selected matrix rows in the supplied index order.
    This reproduces the content identity of a standalone
    selected-row matrix without materializing the whole subset.
    """

    h = hashlib.sha256()

    for start in range(
        0,
        len(
            global_idx
        ),
        block_rows,
    ):

        stop = min(
            start
            + block_rows,
            len(
                global_idx
            ),
        )

        idx = np.asarray(
            global_idx[
                start:stop
            ],
            dtype=np.int64,
        )

        block = np.ascontiguousarray(
            matrix[
                idx,
                :
            ]
        )

        h.update(
            block.view(
                np.uint8
            )
        )

    return h.hexdigest()


def qident(
    value,
):
    """
    DuckDB identifier quoting.
    """

    return (
        '"'
        + str(
            value
        ).replace(
            '"',
            '""',
        )
        + '"'
    )


def sql_path(
    path,
):
    return str(
        Path(
            path
        ).resolve()
    ).replace(
        "'",
        "''",
    )


def clean_runtime_cache(
    path,
):
    expected = Path(
        "/kaggle/working/"
        "stage28_1c_runtime_cache/"
        "cicids2017"
    ).resolve()

    path = Path(
        path
    ).resolve()

    if path != expected:
        raise RuntimeError(
            f"Refusing to clean unexpected path:\n{path}"
        )

    if path.exists():
        shutil.rmtree(
            path
        )

    path.mkdir(
        parents=True,
        exist_ok=False,
    )


# =================================================================================================
# 2. EXACT DURABLE-PARENT GATE
# =================================================================================================

banner(
    "STAGE28-1C-C — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote = (
    remote_line
    .split()[0]
)

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD:",
    EXPECTED_HEAD,
)

print(
    "Local HEAD   :",
    head,
)

print(
    "origin/main  :",
    origin,
)

print(
    "Remote main  :",
    remote,
)

print(
    "Branch       :",
    branch,
)

print(
    "Git clean    :",
    not bool(
        status
    ),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_HEAD
):
    raise RuntimeError(
        "Durable Stage28 parent mismatch."
    )

if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean:\n"
        + status
    )


print()
print(
    "[PASS] exact Stage28-1B durable parent"
)


# =================================================================================================
# 3. PRIOR RUNTIME RECOVERY / FIT-LEDGER GATES
# =================================================================================================

banner(
    "PRIOR RECOVERY + FIT-LEDGER GATES"
)

stage22_recovery = read_json(
    require_file(
        STAGE22_RECOVERY_RECEIPT
    )
)

if (
    stage22_recovery[
        "status"
    ]
    != "BYTE_EXACT_STAGE22_RUNTIME_CACHE_RECOVERED"
):
    raise RuntimeError(
        "Stage22 byte-exact cache recovery not intact."
    )

if (
    stage22_recovery[
        "durable_scientific_parent"
    ]
    != EXPECTED_HEAD
):
    raise RuntimeError(
        "Stage22 recovery parent mismatch."
    )

if not STAGE22_RUNTIME_CACHE.is_dir():
    raise RuntimeError(
        "Recovered Stage22 runtime cache missing."
    )

stage28_1b = read_json(
    require_file(
        STAGE28_1B_FREEZE_PATH
    )
)

execution_state = (
    stage28_1b[
        "execution_manifest"
    ]
)

if int(
    execution_state[
        "new_fit_budget"
    ]
) != EXPECTED_NEW_FITS:
    raise RuntimeError(
        "Stage28 new-fit budget mismatch."
    )

if int(
    execution_state[
        "new_fits_consumed"
    ]
) != 0:
    raise RuntimeError(
        "Stage28 fit ledger is no longer zero."
    )

if int(
    execution_state[
        "new_fits_remaining"
    ]
) != EXPECTED_NEW_FITS:
    raise RuntimeError(
        "Stage28 remaining-fit budget mismatch."
    )


print(
    "[PASS] Stage22 byte-exact cache remains recovered"
)

print(
    "[PASS] Stage28 NEW fits consumed = 0"
)

print(
    "[PASS] Stage28 NEW fits remaining = 108"
)


# =================================================================================================
# 4. DUCKDB GATE
# =================================================================================================

banner(
    "DUCKDB RUNTIME"
)

try:
    import duckdb

except Exception as exc:
    raise RuntimeError(
        "DuckDB is unavailable. "
        "Do NOT install/change versions inside this scientific cell."
    ) from exc


print(
    "DuckDB:",
    duckdb.__version__,
)

print(
    "pandas:",
    pd.__version__,
)

print(
    "NumPy :",
    np.__version__,
)


# =================================================================================================
# 5. LOAD FROZEN STAGE24 / STAGE27 REPRESENTATIONS
# =================================================================================================

banner(
    "LOAD FROZEN REPRESENTATION CONTRACTS"
)

require_file(
    STAGE24_RESULT_PATH
)

actual_stage24_sha = sha256_file(
    STAGE24_RESULT_PATH
)

print(
    "Stage24 result expected:",
    EXPECTED_STAGE24_RESULT_SHA,
)

print(
    "Stage24 result actual  :",
    actual_stage24_sha,
)


if (
    actual_stage24_sha
    != EXPECTED_STAGE24_RESULT_SHA
):
    raise RuntimeError(
        "Stage24-2D result SHA mismatch."
    )


stage24 = read_json(
    STAGE24_RESULT_PATH
)

feature27 = read_json(
    require_file(
        STAGE27_FEATURE_SPEC_PATH
    )
)

source_receipt = read_json(
    require_file(
        STAGE27_SOURCE_RECEIPT_PATH
    )
)

monwed_receipt = read_json(
    require_file(
        STAGE27_MONWED_RECEIPT_PATH
    )
)


representation24 = (
    stage24[
        "representation"
    ]
)

feature_order24 = list(
    representation24[
        "feature_order"
    ]
)

mapping24 = dict(
    representation24[
        "flag_corrected_mapping"
    ]
)

feature_order27 = list(
    feature27[
        "feature_order"
    ]
)

adapter27 = (
    feature27[
        "cicids2017_semantic_adapter"
    ]
)

mapping27 = dict(
    adapter27[
        "mapping"
    ]
)


if len(
    feature_order24
) != EXPECTED_FEATURE_COUNT:
    raise RuntimeError(
        "Stage24 feature count != 70."
    )

if feature_order24 != feature_order27:
    raise RuntimeError(
        "Stage24 / Stage27 ordered feature lists differ."
    )

if mapping24 != mapping27:
    raise RuntimeError(
        "Stage24 FLAG_CORRECTED mapping differs "
        "from frozen Stage27 adapter."
    )

if list(
    mapping27.keys()
) != feature_order27:
    raise RuntimeError(
        "Stage27 adapter key order differs "
        "from frozen feature order."
    )

if (
    adapter27[
        "variant"
    ]
    != "FLAG_CORRECTED_mapping"
):
    raise RuntimeError(
        "Stage27 adapter variant mismatch."
    )

if (
    adapter27[
        "mapping_sha256"
    ]
    != EXPECTED_STAGE27_ADAPTER_SHA
):
    raise RuntimeError(
        "Stage27 adapter SHA mismatch."
    )

if (
    representation24[
        "parse_dtype"
    ]
    != "float64"
):
    raise RuntimeError(
        "Stage24 parse dtype changed."
    )

if (
    representation24[
        "positive_infinity"
    ]
    != "CONVERT_TO_NAN"
):
    raise RuntimeError(
        "Stage24 +inf policy changed."
    )

if (
    representation24[
        "negative_infinity"
    ]
    != "CONVERT_TO_NAN"
):
    raise RuntimeError(
        "Stage24 -inf policy changed."
    )

if (
    representation24[
        "scaling"
    ]
    != "NONE"
):
    raise RuntimeError(
        "Stage24 scaling policy changed."
    )

if (
    representation24[
        "explicit_imputation"
    ]
    != "NONE"
):
    raise RuntimeError(
        "Stage24 imputation policy changed."
    )


unique_physical_columns = []

for model_feature in (
    feature_order27
):

    physical = mapping27[
        model_feature
    ]

    if physical not in (
        unique_physical_columns
    ):
        unique_physical_columns.append(
            physical
        )


print(
    "Feature count          :",
    len(
        feature_order27
    ),
)

print(
    "Unique physical columns:",
    len(
        unique_physical_columns
    ),
)

print(
    "Adapter SHA256         :",
    adapter27[
        "mapping_sha256"
    ],
)

print(
    "Parse dtype            :",
    representation24[
        "parse_dtype"
    ],
)

print(
    "Final Stage27 dtype    :",
    feature27[
        "numeric_policy"
    ][
        "final_model_matrix_dtype"
    ],
)

print()
print(
    "[PASS] Stage24 FLAG_CORRECTED "
    "== Stage27 frozen adapter"
)


# =================================================================================================
# 6. LOAD HISTORICAL PER-SOURCE FLOAT64 HASH + NUMERIC AUDITS
# =================================================================================================

banner(
    "HISTORICAL STAGE24 PER-SOURCE IDENTITIES"
)

feature_audit = {
    int(
        row[
            "file_id"
        ]
    ):
        row

    for row
    in stage24[
        "feature_matrix_audit"
    ]
}

numeric_audit = {
    int(
        row[
            "file_id"
        ]
    ):
        row

    for row
    in stage24[
        "numeric_audit"
    ]
}


if set(
    feature_audit.keys()
) != set(
    range(
        8
    )
):
    raise RuntimeError(
        "Stage24 feature-matrix audit does not "
        "contain file IDs 0..7."
    )

if set(
    numeric_audit.keys()
) != set(
    range(
        8
    )
):
    raise RuntimeError(
        "Stage24 numeric audit does not "
        "contain file IDs 0..7."
    )


for file_id in range(
    8
):

    f = feature_audit[
        file_id
    ]

    n = numeric_audit[
        file_id
    ]

    if int(
        f[
            "rows"
        ]
    ) != int(
        n[
            "rows"
        ]
    ):
        raise RuntimeError(
            f"Stage24 file {file_id}: "
            "feature/numeric row mismatch."
        )

    print(
        f"[{file_id}] "
        f"{f['day']:10s} "
        f"rows={f['rows']:9,d}"
    )

    print(
        "    FLOAT64 SHA:",
        f[
            "flag_corrected_feature_matrix_float64_sha256"
        ],
    )

    print(
        "    +inf -> NaN:",
        n[
            "positive_inf_to_nan"
        ],
    )

    print(
        "    -inf -> NaN:",
        n[
            "negative_inf_to_nan"
        ],
    )

    print(
        "    final NaN  :",
        n[
            "nan_cells_after_inf_conversion"
        ],
    )


# =================================================================================================
# 7. SOURCE SEGMENT / BYTE-IDENTITY GATE
# =================================================================================================

banner(
    "8/8 CICIDS2017 SOURCE BYTE-IDENTITY GATE"
)

segments = sorted(
    source_receipt[
        "segments"
    ],
    key=lambda x: int(
        x[
            "source_index"
        ]
    ),
)


if len(
    segments
) != 8:
    raise RuntimeError(
        "Expected exactly eight CICIDS2017 source segments."
    )


running_global_stop = 0
source_paths = {}


for expected_id, segment in enumerate(
    segments
):

    source_id = int(
        segment[
            "source_index"
        ]
    )

    if source_id != expected_id:
        raise RuntimeError(
            "Source indices are not exactly 0..7."
        )

    global_start = int(
        segment[
            "global_start_zero_based"
        ]
    )

    global_stop = int(
        segment[
            "global_stop_exclusive"
        ]
    )

    effective_rows = int(
        segment[
            "effective_rows"
        ]
    )


    if global_start != running_global_stop:
        raise RuntimeError(
            f"Source {source_id}: global geometry gap/overlap."
        )

    if (
        global_stop
        - global_start
        != effective_rows
    ):
        raise RuntimeError(
            f"Source {source_id}: effective/global row mismatch."
        )


    basename = (
        segment[
            "basename"
        ]
    )

    path = (
        SOURCE_ROOT
        / basename
    )

    require_file(
        path
    )

    actual_sha = sha256_file(
        path
    )

    expected_sha = (
        segment[
            "sha256"
        ]
    )

    if (
        actual_sha
        != expected_sha
    ):
        raise RuntimeError(
            f"{basename}: source SHA mismatch."
        )


    parquet = pq.ParquetFile(
        path
    )

    physical_rows = int(
        parquet.metadata.num_rows
    )

    if (
        physical_rows
        != int(
            segment[
                "physical_rows"
            ]
        )
    ):
        raise RuntimeError(
            f"{basename}: physical-row mismatch."
        )


    schema = set(
        parquet.schema_arrow.names
    )

    missing = [
        column
        for column
        in unique_physical_columns
        if column not in schema
    ]

    if missing:
        raise RuntimeError(
            f"{basename}: missing frozen physical columns:\n"
            + "\n".join(
                missing
            )
        )

    if "Label" not in schema:
        raise RuntimeError(
            f"{basename}: Label column missing; "
            "cannot reproduce Stage24 effective-row filter."
        )


    if (
        effective_rows
        != int(
            feature_audit[
                source_id
            ][
                "rows"
            ]
        )
    ):
        raise RuntimeError(
            f"{basename}: Stage24 audit row mismatch."
        )


    source_paths[
        source_id
    ] = path

    running_global_stop = (
        global_stop
    )


    print(
        f"[PASS] [{source_id}] "
        f"{basename}"
    )

    print(
        "       physical :",
        f"{physical_rows:,}",
    )

    print(
        "       effective:",
        f"{effective_rows:,}",
    )

    print(
        "       SHA256   :",
        actual_sha,
    )


if (
    running_global_stop
    != EXPECTED_TOTAL_ROWS
):
    raise RuntimeError(
        "Final global source stop != 2,830,743."
    )


print()
print(
    "[PASS] 8 / 8 exact source bytes"
)

print(
    "[PASS] global row geometry = 2,830,743"
)


# =================================================================================================
# 8. VERIFY GLOBAL FAMILY-CODE CACHE
# =================================================================================================

banner(
    "GLOBAL FAMILY-CODE CACHE GATE"
)

require_file(
    GLOBAL_FAMILY_CODES_PATH
)

family_codes = np.load(
    GLOBAL_FAMILY_CODES_PATH,
    allow_pickle=False,
    mmap_mode="r",
)

if family_codes.shape != (
    EXPECTED_TOTAL_ROWS,
):
    raise RuntimeError(
        "Global family-code shape mismatch."
    )

if str(
    family_codes.dtype
) != "uint8":
    raise RuntimeError(
        "Global family-code dtype mismatch."
    )


print(
    "Rows :",
    f"{family_codes.shape[0]:,}",
)

print(
    "Dtype:",
    family_codes.dtype,
)

print(
    "[PASS] global row-identity cache ready"
)


# =================================================================================================
# 9. ALLOCATE GLOBAL FLOAT32 MODEL MATRIX
# =================================================================================================

banner(
    "ALLOCATE GLOBAL FLOAT32 MODEL MATRIX"
)

clean_runtime_cache(
    CACHE_ROOT
)

disk = shutil.disk_usage(
    "/kaggle/working"
)

required_payload = (
    EXPECTED_TOTAL_ROWS
    * EXPECTED_FEATURE_COUNT
    * np.dtype(
        np.float32
    ).itemsize
)


print(
    "Free working space:",
    f"{disk.free / (1024**3):.3f} GiB",
)

print(
    "Matrix payload    :",
    f"{required_payload / (1024**3):.3f} GiB",
)


if disk.free < (
    required_payload
    + 1_000_000_000
):
    raise RuntimeError(
        "Insufficient working space for "
        "global FLOAT32 matrix plus safety margin."
    )


global_X = np.lib.format.open_memmap(
    GLOBAL_FEATURE_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(
        EXPECTED_TOTAL_ROWS,
        EXPECTED_FEATURE_COUNT,
    ),
)


print(
    "Path :",
    GLOBAL_FEATURE_PATH,
)

print(
    "Shape:",
    global_X.shape,
)

print(
    "Dtype:",
    global_X.dtype,
)

print(
    "[PASS] global runtime matrix allocated"
)


# =================================================================================================
# 10. EXACT STAGE24 DUCKDB PROJECTION
#
# Stage24 constructed its target projection as:
#
#   CAST(physical_column AS DOUBLE) AS model_feature
#
# and retained rows for which:
#
#   "Label" IS NOT NULL
#
# This is reproduced directly.
#
# We DO NOT materialize Label values into Python.
# The Label column is used only for the already-frozen structural-null row filter.
# =================================================================================================

banner(
    "REPRODUCE 8 STAGE24 FLAG_CORRECTED FLOAT64 MATRICES"
)

conn = duckdb.connect(
    database=":memory:"
)

try:
    conn.execute(
        "SET preserve_insertion_order = true"
    )
except Exception:
    # Older DuckDB builds may already preserve insertion order by default.
    # The historical matrix hashes below are the definitive row-order gate.
    pass


per_source_recovery = []

total_predictor_rows = 0

total_positive_inf = 0
total_negative_inf = 0
total_final_nan = 0

global_float32_hasher = hashlib.sha256()


for segment in segments:

    source_id = int(
        segment[
            "source_index"
        ]
    )

    path = (
        source_paths[
            source_id
        ]
    )

    basename = (
        segment[
            "basename"
        ]
    )

    global_start = int(
        segment[
            "global_start_zero_based"
        ]
    )

    global_stop = int(
        segment[
            "global_stop_exclusive"
        ]
    )

    expected_rows = int(
        segment[
            "effective_rows"
        ]
    )

    expected_feature_sha = (
        feature_audit[
            source_id
        ][
            "flag_corrected_feature_matrix_float64_sha256"
        ]
    )

    expected_numeric = (
        numeric_audit[
            source_id
        ]
    )


    print()
    print(
        "-" * 120
    )

    print(
        f"[{source_id}] "
        f"{segment['day']} — {basename}"
    )

    print(
        "-" * 120
    )


    projection_sql = ",\n".join(
        (
            f"CAST("
            f"{qident(mapping27[model_feature])}"
            f" AS DOUBLE) "
            f"AS {qident(model_feature)}"
        )
        for model_feature
        in feature_order27
    )


    query = f"""
        SELECT
            {projection_sql}
        FROM read_parquet('{sql_path(path)}')
        WHERE "Label" IS NOT NULL
    """


    df = conn.execute(
        query
    ).fetchdf()


    if list(
        df.columns
    ) != feature_order27:
        raise RuntimeError(
            f"{basename}: DuckDB projection column order mismatch."
        )


    X64 = df[
        feature_order27
    ].to_numpy(
        dtype=np.float64,
        copy=True,
    )


    del df
    gc.collect()


    if X64.shape != (
        expected_rows,
        EXPECTED_FEATURE_COUNT,
    ):
        raise RuntimeError(
            f"{basename}: reproduced matrix shape mismatch.\n"
            f"expected={(expected_rows, EXPECTED_FEATURE_COUNT)}\n"
            f"actual={X64.shape}"
        )


    # ----------------------------------------------------------------------------------------------
    # Exact frozen numeric policy.
    # ----------------------------------------------------------------------------------------------

    positive_inf = int(
        np.isposinf(
            X64
        ).sum()
    )

    negative_inf = int(
        np.isneginf(
            X64
        ).sum()
    )

    inf_mask = np.isinf(
        X64
    )

    if inf_mask.any():

        X64[
            inf_mask
        ] = np.nan


    final_nan = int(
        np.isnan(
            X64
        ).sum()
    )


    if (
        positive_inf
        != int(
            expected_numeric[
                "positive_inf_to_nan"
            ]
        )
    ):
        raise RuntimeError(
            f"{basename}: +inf audit mismatch.\n"
            f"expected={expected_numeric['positive_inf_to_nan']}\n"
            f"actual={positive_inf}"
        )

    if (
        negative_inf
        != int(
            expected_numeric[
                "negative_inf_to_nan"
            ]
        )
    ):
        raise RuntimeError(
            f"{basename}: -inf audit mismatch.\n"
            f"expected={expected_numeric['negative_inf_to_nan']}\n"
            f"actual={negative_inf}"
        )

    if (
        final_nan
        != int(
            expected_numeric[
                "nan_cells_after_inf_conversion"
            ]
        )
    ):
        raise RuntimeError(
            f"{basename}: final NaN audit mismatch.\n"
            f"expected={expected_numeric['nan_cells_after_inf_conversion']}\n"
            f"actual={final_nan}"
        )


    # ----------------------------------------------------------------------------------------------
    # Exact Stage24 FLOAT64 matrix fingerprint.
    # ----------------------------------------------------------------------------------------------

    actual_feature_sha = (
        sha256_array_stage24(
            X64
        )
    )


    print(
        "Rows              :",
        f"{X64.shape[0]:,}",
    )

    print(
        "Shape             :",
        X64.shape,
    )

    print(
        "Dtype             :",
        X64.dtype,
    )

    print(
        "+inf -> NaN       :",
        f"{positive_inf:,}",
    )

    print(
        "-inf -> NaN       :",
        f"{negative_inf:,}",
    )

    print(
        "Final NaN cells   :",
        f"{final_nan:,}",
    )

    print(
        "Historical SHA256 :",
        expected_feature_sha,
    )

    print(
        "Reproduced SHA256 :",
        actual_feature_sha,
    )


    if (
        actual_feature_sha
        != expected_feature_sha
    ):
        raise RuntimeError(
            f"{basename}: Stage24 FLOAT64 matrix SHA mismatch.\n"
            "STOP. Do not fit any model."
        )


    print(
        "[PASS] exact historical Stage24 FLOAT64 matrix"
    )


    # ----------------------------------------------------------------------------------------------
    # Stage27 frozen final model representation = FLOAT32.
    # ----------------------------------------------------------------------------------------------

    X32 = X64.astype(
        np.float32,
        copy=False,
    )


    float32_inf = int(
        np.isinf(
            X32
        ).sum()
    )

    float32_nan = int(
        np.isnan(
            X32
        ).sum()
    )


    if float32_inf != 0:
        raise RuntimeError(
            f"{basename}: FLOAT64 -> FLOAT32 "
            f"created {float32_inf:,} infinite cells."
        )


    global_X[
        global_start:global_stop,
        :
    ] = X32


    # Hash the logical full global FLOAT32 matrix in exact source/global order.
    contiguous32 = np.ascontiguousarray(
        X32
    )

    global_float32_hasher.update(
        contiguous32.view(
            np.uint8
        )
    )


    total_predictor_rows += int(
        X32.shape[0]
    )

    total_positive_inf += (
        positive_inf
    )

    total_negative_inf += (
        negative_inf
    )

    total_final_nan += (
        final_nan
    )


    per_source_recovery.append(
        {
            "source_index":
                source_id,

            "day":
                segment[
                    "day"
                ],

            "basename":
                basename,

            "source_sha256":
                segment[
                    "sha256"
                ],

            "physical_rows":
                int(
                    segment[
                        "physical_rows"
                    ]
                ),

            "effective_rows":
                expected_rows,

            "global_start_zero_based":
                global_start,

            "global_stop_exclusive":
                global_stop,

            "stage24_float64_matrix_sha256_expected":
                expected_feature_sha,

            "stage28_reproduced_float64_matrix_sha256":
                actual_feature_sha,

            "stage24_float64_identity_exact":
                True,

            "positive_inf_to_nan":
                positive_inf,

            "negative_inf_to_nan":
                negative_inf,

            "nan_cells_after_inf_conversion":
                final_nan,

            "float32_inf_after_conversion":
                float32_inf,

            "float32_nan_cells":
                float32_nan,
        }
    )


    del contiguous32
    del X32
    del X64
    del inf_mask

    gc.collect()


conn.close()


if (
    total_predictor_rows
    != EXPECTED_TOTAL_ROWS
):
    raise RuntimeError(
        "Total materialized predictor rows mismatch."
    )


global_X.flush()


print()
print(
    "[PASS] 8 / 8 Stage24 FLOAT64 matrix hashes reproduced exactly"
)

print(
    "[PASS] 8 / 8 Stage24 numeric audits reproduced exactly"
)

print(
    "[PASS] global FLOAT32 matrix populated"
)


# =================================================================================================
# 11. RELOAD GLOBAL FLOAT32 CACHE / FULL CONTENT HASH
# =================================================================================================

banner(
    "GLOBAL FLOAT32 MODEL-MATRIX CONTENT GATE"
)

del global_X
gc.collect()


global_X = np.load(
    GLOBAL_FEATURE_PATH,
    allow_pickle=False,
    mmap_mode="r",
)


if global_X.shape != (
    EXPECTED_TOTAL_ROWS,
    EXPECTED_FEATURE_COUNT,
):
    raise RuntimeError(
        "Recovered global FLOAT32 matrix shape mismatch."
    )

if global_X.dtype != np.float32:
    raise RuntimeError(
        "Recovered global model matrix dtype != float32."
    )


streamed_global_sha = (
    global_float32_hasher.hexdigest()
)

reloaded_global_sha = (
    sha256_array_content_blocked(
        global_X
    )
)


print(
    "Shape                :",
    global_X.shape,
)

print(
    "Dtype                :",
    global_X.dtype,
)

print(
    "NPY bytes            :",
    f"{GLOBAL_FEATURE_PATH.stat().st_size:,}",
)

print(
    "Stream content SHA256:",
    streamed_global_sha,
)

print(
    "Reload content SHA256:",
    reloaded_global_sha,
)


if (
    streamed_global_sha
    != reloaded_global_sha
):
    raise RuntimeError(
        "Global FLOAT32 content hash changed "
        "between construction and reload."
    )


global_inf = 0
global_nan = 0

for start in range(
    0,
    EXPECTED_TOTAL_ROWS,
    65_536,
):

    stop = min(
        start
        + 65_536,
        EXPECTED_TOTAL_ROWS,
    )

    block = np.asarray(
        global_X[
            start:stop,
            :
        ]
    )

    global_inf += int(
        np.isinf(
            block
        ).sum()
    )

    global_nan += int(
        np.isnan(
            block
        ).sum()
    )


if global_inf != 0:
    raise RuntimeError(
        "Infinity exists in final global FLOAT32 matrix."
    )


print(
    "Global FLOAT32 inf   :",
    global_inf,
)

print(
    "Global FLOAT32 NaN   :",
    f"{global_nan:,}",
)

print()
print(
    "[PASS] global FLOAT32 model-input cache is stable"
)


# =================================================================================================
# 12. STAGE27 MON-WED KNOWN-POPULATION REPRODUCTION
#
# Independent historical cross-check:
#
# Stage27-1B froze:
#
#   rows              = 1,668,519
#   global index SHA  = b0010472...
#   binary label SHA  = cc0d2a0d...
#   feature FLOAT32   = fc3137b1...
#
# This subset is reconstructed from the current Stage28 global identities.
# =================================================================================================

banner(
    "INDEPENDENT STAGE27 MON-WED FLOAT32 IDENTITY GATE"
)

historical_population = (
    monwed_receipt[
        "population"
    ]
)

historical_artifacts = (
    monwed_receipt[
        "artifacts"
    ]
)


if int(
    historical_population[
        "rows"
    ]
) != EXPECTED_STAGE27_MONWED_ROWS:
    raise RuntimeError(
        "Historical Stage27 Mon-Wed row count mismatch."
    )


monwed_family = np.asarray(
    family_codes[
        :MONWED_GLOBAL_STOP_EXCLUSIVE
    ],
    dtype=np.uint8,
)


# Known population = BENIGN + primary seven.
# Code 8 Heartbleed and code 9 OTHER are excluded.
known_mask = (
    monwed_family
    <= 7
)

known_idx = np.flatnonzero(
    known_mask
).astype(
    np.int32,
    copy=False,
)


if known_idx.shape != (
    EXPECTED_STAGE27_MONWED_ROWS,
):
    raise RuntimeError(
        "Reconstructed Stage27 Mon-Wed "
        "known-population row count mismatch."
    )


excluded_monwed = (
    MONWED_GLOBAL_STOP_EXCLUSIVE
    -
    len(
        known_idx
    )
)

if (
    excluded_monwed
    != EXPECTED_HEARTBLEED_EXCLUDED
):
    raise RuntimeError(
        "Expected exactly 11 Mon-Wed "
        "target-only Heartbleed exclusions."
    )


if not np.all(
    known_idx[
        1:
    ]
    >
    known_idx[
        :-1
    ]
):
    raise RuntimeError(
        "Stage27 Mon-Wed known indices are not "
        "strictly increasing."
    )


known_index_sha = (
    sha256_array_stage24(
        known_idx
    )
)


known_binary = (
    family_codes[
        known_idx
    ]
    != 0
).astype(
    np.uint8,
    copy=False,
)


known_label_sha = (
    sha256_array_stage24(
        known_binary
    )
)


print(
    "Known rows          :",
    f"{len(known_idx):,}",
)

print(
    "Excluded Heartbleed :",
    excluded_monwed,
)

print()

print(
    "Historical index SHA:",
    historical_artifacts[
        "global_idx_int32"
    ][
        "content_sha256"
    ],
)

print(
    "Current index SHA   :",
    known_index_sha,
)

print()

print(
    "Historical label SHA:",
    historical_artifacts[
        "binary_labels_uint8"
    ][
        "content_sha256"
    ],
)

print(
    "Current label SHA   :",
    known_label_sha,
)


if (
    known_index_sha
    != EXPECTED_STAGE27_MONWED_INDEX_SHA
):
    raise RuntimeError(
        "Stage27 Mon-Wed global-index "
        "content SHA mismatch."
    )

if (
    known_index_sha
    != historical_artifacts[
        "global_idx_int32"
    ][
        "content_sha256"
    ]
):
    raise RuntimeError(
        "Current Mon-Wed index SHA differs "
        "from durable Stage27 receipt."
    )


if (
    known_label_sha
    != EXPECTED_STAGE27_MONWED_LABEL_SHA
):
    raise RuntimeError(
        "Stage27 Mon-Wed binary-label "
        "content SHA mismatch."
    )

if (
    known_label_sha
    != historical_artifacts[
        "binary_labels_uint8"
    ][
        "content_sha256"
    ]
):
    raise RuntimeError(
        "Current Mon-Wed label SHA differs "
        "from durable Stage27 receipt."
    )


print()
print(
    "[PASS] Stage27 Mon-Wed membership identity reproduced"
)


# =================================================================================================
# 13. STAGE27 HISTORICAL FLOAT32 FEATURE HASH
# =================================================================================================

print()
print(
    "Hashing reconstructed Stage27 "
    "Mon-Wed known FLOAT32 feature population..."
)

monwed_feature_sha = (
    sha256_selected_matrix_rows(
        global_X,
        known_idx,
        block_rows=65_536,
    )
)


historical_feature_sha = (
    historical_artifacts[
        "features_float32"
    ][
        "content_sha256"
    ]
)


print(
    "Historical feature SHA:",
    historical_feature_sha,
)

print(
    "Current feature SHA   :",
    monwed_feature_sha,
)


if (
    historical_feature_sha
    != EXPECTED_STAGE27_MONWED_FEATURE_SHA
):
    raise RuntimeError(
        "Durable Stage27 Mon-Wed feature "
        "receipt changed unexpectedly."
    )

if (
    monwed_feature_sha
    != EXPECTED_STAGE27_MONWED_FEATURE_SHA
):
    raise RuntimeError(
        "Reconstructed global FLOAT32 matrix "
        "does not reproduce Stage27 historical "
        "Mon-Wed model input."
    )


print()
print(
    "[PASS] EXACT Stage27 historical "
    "Mon-Wed FLOAT32 feature content reproduced"
)


# =================================================================================================
# 14. FINAL NUMERIC / GEOMETRY AUDIT
# =================================================================================================

banner(
    "FINAL CICIDS2017 MODEL-INPUT AUDIT"
)

expected_total_stage24_pos_inf = sum(
    int(
        row[
            "positive_inf_to_nan"
        ]
    )
    for row
    in stage24[
        "numeric_audit"
    ]
)

expected_total_stage24_neg_inf = sum(
    int(
        row[
            "negative_inf_to_nan"
        ]
    )
    for row
    in stage24[
        "numeric_audit"
    ]
)

expected_total_stage24_nan = sum(
    int(
        row[
            "nan_cells_after_inf_conversion"
        ]
    )
    for row
    in stage24[
        "numeric_audit"
    ]
)


if (
    total_positive_inf
    != expected_total_stage24_pos_inf
):
    raise RuntimeError(
        "Global +inf audit mismatch."
    )

if (
    total_negative_inf
    != expected_total_stage24_neg_inf
):
    raise RuntimeError(
        "Global -inf audit mismatch."
    )

if (
    total_final_nan
    != expected_total_stage24_nan
):
    raise RuntimeError(
        "Global FLOAT64 NaN audit mismatch."
    )

if (
    global_nan
    != total_final_nan
):
    raise RuntimeError(
        "FLOAT64 -> FLOAT32 conversion changed "
        "the number of NaN cells."
    )


print(
    "Rows                    :",
    f"{EXPECTED_TOTAL_ROWS:,}",
)

print(
    "Features                :",
    EXPECTED_FEATURE_COUNT,
)

print(
    "FLOAT64 +inf -> NaN     :",
    f"{total_positive_inf:,}",
)

print(
    "FLOAT64 -inf -> NaN     :",
    f"{total_negative_inf:,}",
)

print(
    "Final NaN cells         :",
    f"{total_final_nan:,}",
)

print(
    "Final FLOAT32 inf cells :",
    global_inf,
)

print(
    "Global FLOAT32 SHA256   :",
    reloaded_global_sha,
)

print()
print(
    "[PASS] complete CICIDS2017 model-input "
    "representation recovered"
)


# =================================================================================================
# 15. RUNTIME-ONLY RECOVERY RECEIPT
# =================================================================================================

banner(
    "WRITE RUNTIME-ONLY RECOVERY RECEIPT"
)

runtime_receipt = {

    "stage":
        "Stage28-1C-C",

    "type":
        "RUNTIME_ONLY_CICIDS2017_FLAG_CORRECTED_70F_RECOVERY",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "durable_scientific_parent":
        EXPECTED_HEAD,

    "scientific_interpretation": {
        "action":
            "REPRODUCE_ALREADY_OPENED_STAGE24_EVIDENCE",

        "new_target_opening_consumed":
            False,

        "target_adaptive_choice_performed":
            False,

        "feature_selection_performed":
            False,

        "mapping_selection_performed":
            False,

        "threshold_selection_performed":
            False,

        "model_selection_performed":
            False,
    },

    "representation": {
        "feature_count":
            EXPECTED_FEATURE_COUNT,

        "feature_order":
            feature_order27,

        "adapter_variant":
            adapter27[
                "variant"
            ],

        "adapter_mapping_sha256":
            adapter27[
                "mapping_sha256"
            ],

        "parse_dtype":
            "float64",

        "stage24_hash_semantics":
            (
                "SHA256("
                "NP_ASCONTIGUOUSARRAY_FLOAT64"
                ".VIEW_UINT8)"
            ),

        "positive_infinity":
            "CONVERT_TO_NAN",

        "negative_infinity":
            "CONVERT_TO_NAN",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "final_model_matrix_dtype":
            "float32",

        "global_row_order":
            "FROZEN_STAGE27_ZERO_BASED_GLOBAL_ROW_ID",
    },

    "stage24_reproduction": {
        "stage24_result_path":
            str(
                STAGE24_RESULT_PATH.relative_to(
                    REPO
                )
            ),

        "stage24_result_sha256":
            actual_stage24_sha,

        "source_count":
            8,

        "all_source_byte_identities_exact":
            True,

        "all_float64_matrix_hashes_exact":
            True,

        "all_numeric_audits_exact":
            True,

        "per_source":
            per_source_recovery,
    },

    "global_float32_cache": {
        "path":
            str(
                GLOBAL_FEATURE_PATH
            ),

        "shape": [
            EXPECTED_TOTAL_ROWS,
            EXPECTED_FEATURE_COUNT,
        ],

        "dtype":
            "float32",

        "npy_bytes":
            int(
                GLOBAL_FEATURE_PATH.stat().st_size
            ),

        "content_sha256":
            reloaded_global_sha,

        "infinite_cells":
            global_inf,

        "nan_cells":
            global_nan,

        "committed_to_git":
            False,
    },

    "stage27_independent_crosscheck": {
        "population":
            "MONDAY_WEDNESDAY_KNOWN",

        "rows":
            int(
                len(
                    known_idx
                )
            ),

        "excluded_target_only_unseen":
            excluded_monwed,

        "global_index_content_sha256":
            known_index_sha,

        "historical_global_index_content_sha256":
            EXPECTED_STAGE27_MONWED_INDEX_SHA,

        "binary_label_content_sha256":
            known_label_sha,

        "historical_binary_label_content_sha256":
            EXPECTED_STAGE27_MONWED_LABEL_SHA,

        "feature_float32_content_sha256":
            monwed_feature_sha,

        "historical_feature_float32_content_sha256":
            EXPECTED_STAGE27_MONWED_FEATURE_SHA,

        "status":
            "EXACT_MATCH",
    },

    "scientific_access": {
        "cicids2017_effective_predictor_rows_read":
            EXPECTED_TOTAL_ROWS,

        "cicids2017_predictor_columns_per_row":
            EXPECTED_FEATURE_COUNT,

        "label_values_materialized_to_python":
            0,

        "label_column_used_only_for_stage24_structural_null_filter":
            True,

        "new_scientific_target_openings":
            0,

        "model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "bootstrap_replicates":
            0,
    },

    "fit_ledger": {
        "new_authorized":
            EXPECTED_NEW_FITS,

        "new_consumed":
            0,

        "new_remaining":
            EXPECTED_NEW_FITS,
    },

    "git_modified":
        False,

    "status":
        "CICIDS2017_MODEL_INPUT_CACHE_RECOVERED_AND_HISTORICALLY_VERIFIED",
}


RUNTIME_RECEIPT_PATH.write_text(
    json.dumps(
        runtime_receipt,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )
    + "\n",
    encoding="utf-8",
)


print(
    "Receipt:"
)

print(
    " ",
    RUNTIME_RECEIPT_PATH,
)

print()

print(
    "[PASS] runtime recovery receipt written outside Git"
)


# =================================================================================================
# 16. FINAL GIT CLEANLINESS GATE
# =================================================================================================

banner(
    "FINAL GIT CLEANLINESS GATE"
)

final_head = git(
    "rev-parse",
    "HEAD",
)

final_origin = git(
    "rev-parse",
    "origin/main",
)

final_remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not final_remote_line:
    raise RuntimeError(
        "Unable to resolve final remote main."
    )

final_remote = (
    final_remote_line
    .split()[0]
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Expected   :",
    EXPECTED_HEAD,
)

print(
    "HEAD       :",
    final_head,
)

print(
    "origin/main:",
    final_origin,
)

print(
    "Remote main:",
    final_remote,
)

print(
    "Git clean  :",
    not bool(
        final_status
    ),
)


if not (
    final_head
    == final_origin
    == final_remote
    == EXPECTED_HEAD
):
    raise RuntimeError(
        "Git parent changed during Stage28-1C-C."
    )

if final_status:
    raise RuntimeError(
        "Stage28-1C-C unexpectedly modified Git:\n"
        + final_status
    )


print()
print(
    "[PASS] durable repository completely untouched"
)


# =================================================================================================
# 17. FINAL
# =================================================================================================

banner(
    "STAGE28-1C-C — CICIDS2017 MODEL-INPUT RECOVERY COMPLETE"
)

print(
    "Durable parent:"
)

print(
    " ",
    EXPECTED_HEAD,
)

print()

print(
    "Stage24 reproduction:"
)

print(
    "  sources                     = 8 / 8 BYTE EXACT"
)

print(
    "  FLOAT64 matrices            = 8 / 8 HISTORICAL SHA EXACT"
)

print(
    "  numeric audits              = 8 / 8 EXACT"
)

print(
    "  effective rows              =",
    f"{EXPECTED_TOTAL_ROWS:,}",
)

print(
    "  feature count               =",
    EXPECTED_FEATURE_COUNT,
)

print()

print(
    "Stage27 model representation:"
)

print(
    "  final dtype                 = float32"
)

print(
    "  global matrix shape         =",
    global_X.shape,
)

print(
    "  global FLOAT32 content SHA  =",
    reloaded_global_sha,
)

print()

print(
    "Independent Stage27 cross-check:"
)

print(
    "  Mon-Wed known rows          =",
    f"{len(known_idx):,}",
)

print(
    "  global-index SHA            = EXACT"
)

print(
    "  binary-label SHA            = EXACT"
)

print(
    "  FLOAT32 feature SHA         = EXACT"
)

print(
    "  expected feature SHA        =",
    EXPECTED_STAGE27_MONWED_FEATURE_SHA,
)

print()

print(
    "Scientific accounting:"
)

print(
    "  CICIDS2017 predictor rows read      =",
    f"{EXPECTED_TOTAL_ROWS:,}",
)

print(
    "  new scientific target openings      = 0"
)

print(
    "  target-adaptive choices              = 0"
)

print(
    "  model fits                           = 0"
)

print(
    "  model inference                      = 0"
)

print(
    "  threshold selection                  = 0"
)

print(
    "  bootstrap                            = 0"
)

print()

print(
    "Fit ledger:"
)

print(
    "  NEW authorized = 108"
)

print(
    "  consumed       = 0"
)

print(
    "  remaining      = 108"
)

print()

print(
    "Runtime model-input assets now available:"
)

print(
    "  Stage22:"
)

print(
    "   ",
    STAGE22_RUNTIME_CACHE,
)

print(
    "  CICIDS2017:"
)

print(
    "   ",
    GLOBAL_FEATURE_PATH,
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage28-1C-D — freeze the recovered "
    "Stage22 + CICIDS2017 model-input identities "
    "durably in Git, bind them to the 120-component "
    "execution manifest, commit/push, and verify that "
    "FIT #1 is still unconsumed."
)

print()

print(
    "  NO MODEL FIT YET."
)

print()
print(SEP)


STAGE28-1C-C — EXACT DURABLE-PARENT GATE

Expected HEAD: ba011ec01f1399939111b24664ebd5d66c630f95
Local HEAD   : ba011ec01f1399939111b24664ebd5d66c630f95
origin/main  : ba011ec01f1399939111b24664ebd5d66c630f95
Remote main  : ba011ec01f1399939111b24664ebd5d66c630f95
Branch       : main
Git clean    : True

[PASS] exact Stage28-1B durable parent

PRIOR RECOVERY + FIT-LEDGER GATES

[PASS] Stage22 byte-exact cache remains recovered
[PASS] Stage28 NEW fits consumed = 0
[PASS] Stage28 NEW fits remaining = 108

DUCKDB RUNTIME

DuckDB: 1.3.2
pandas: 2.3.3
NumPy : 2.0.2

LOAD FROZEN REPRESENTATION CONTRACTS

Stage24 result expected: 5b819f03d2f336dde1415cf08f48f1963fba15d725661dcebe3c37f5f5aaf53c
Stage24 result actual  : 5b819f03d2f336dde1415cf08f48f1963fba15d725661dcebe3c37f5f5aaf53c
Feature count          : 70
Unique physical columns: 70
Adapter SHA256         : 88d7f3e133e7d20ee05dc4618c6102f0c420936e0fe72a9a094d951a9c4dad7a
Parse dtype            : float64
Final Stage27 dtype    : float32


In [9]:
# =================================================================================================
# STAGE28-1C-D — FINAL PRE-FIT MODEL-INPUT IDENTITY FREEZE + MANIFEST BINDING
#
# Expected durable parent:
#   ba011ec01f1399939111b24664ebd5d66c630f95
#
# PURPOSE
# -------
# Durably freeze the identities of the two already-recovered Stage28 model-input universes:
#
#   A. Stage22 FULL
#      exact historical Stage22R-1C byte-identical 70-feature Parquet cache
#
#   B. Stage27 chronology LOAO + Stage28B random LOAO
#      exact Stage24 FLAG_CORRECTED CICIDS2017 population converted to the
#      frozen Stage27 float32 model representation
#
# Then bind every one of the already-frozen 120 Stage28 model components to
# exactly one of those input contracts.
#
# IMPORTANT
# ---------
# This cell commits ONLY small provenance / binding files.
#
# It NEVER commits:
#   - Stage22 cache Parquets
#   - CICIDS2017 source Parquets
#   - global_features_float32.npy
#   - runtime membership .npy files
#
# SCIENCE
# -------
# NO model fit.
# NO inference.
# NO threshold selection.
# NO bootstrap.
# NO new target opening.
# NO new feature mapping.
# NO new split.
# NO change to the 120-component execution universe.
#
# If this succeeds, FIT #1 becomes authorized.
# =================================================================================================

from __future__ import annotations

import base64
import csv
import hashlib
import json
import os
import subprocess
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "ba011ec01f1399939111b24664ebd5d66c630f95"
)

COMMIT_MESSAGE = (
    "stage28-1c: freeze recovered model input identities"
)

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

STAGE28_1B_DIR = (
    STAGE28_ROOT
    / "stage28_1b_random_loao_membership_and_execution_lock"
)

MANIFEST_PATH = (
    STAGE28_1B_DIR
    / "stage28_component_execution_manifest.csv"
)

MANIFEST_RECEIPT_PATH = (
    STAGE28_1B_DIR
    / "stage28_component_execution_manifest_receipt.json"
)

STAGE28_1B_FREEZE_PATH = (
    STAGE28_1B_DIR
    / "stage28_1b_freeze_record.json"
)

EXPECTED_MANIFEST_SHA = (
    "47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505"
)


# -------------------------------------------------------------------------------------------------
# Stage22 historical durable contract.
# -------------------------------------------------------------------------------------------------

STAGE22_MANIFEST_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
    / "stage22r_1c_development_model_input_manifest.json"
)

STAGE22_REMOTE_RECEIPT_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_remote_cache_checkpoint"
    / "stage22r_1c_private_kaggle_cache_receipt.json"
)

STAGE22_RUNTIME_RECEIPT_PATH = Path(
    "/kaggle/working/"
    "stage28_1c_b_stage22_cache_recovery_receipt.json"
)

STAGE22_RUNTIME_CACHE = Path(
    "/kaggle/working/"
    "stage22r_1c_70f_development_cache"
)


# -------------------------------------------------------------------------------------------------
# CICIDS2017 recovered model-input contract.
# -------------------------------------------------------------------------------------------------

CICIDS_RUNTIME_RECEIPT_PATH = Path(
    "/kaggle/working/"
    "stage28_1c_c_cicids2017_70f_recovery_receipt.json"
)

CICIDS_GLOBAL_MATRIX_PATH = Path(
    "/kaggle/working/"
    "stage28_1c_runtime_cache/"
    "cicids2017/"
    "global_features_float32.npy"
)

STAGE24_RESULT_PATH = (
    REPO
    / "results"
    / "stage24_cross_dataset"
    / "stage24_2_primary_target_openings"
    / "stage24_2d_bridge70_flag_corrected"
    / "stage24_2d_bridge70_flag_corrected_result.json"
)

STAGE27_FEATURE_SPEC_PATH = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_0_protocol_lock"
    / "feature_representation.json"
)

STAGE27_MONWED_RECEIPT_PATH = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1b_prefit_feature_materialization"
    / "monwed_known_feature_cache_receipt.json"
)


EXPECTED_CICIDS_ROWS = 2_830_743
EXPECTED_FEATURES = 70

EXPECTED_CICIDS_GLOBAL_CONTENT_SHA = (
    "c303a01a12657284cbc7269a34464b9f037b5a6feb9fac8d790452a504b76344"
)

EXPECTED_STAGE27_MONWED_SHA = (
    "fc3137b10bbb2542240df3d380286b88f6ea282aba200fd893e6445391855ccf"
)

EXPECTED_STAGE27_ADAPTER_SHA = (
    "88d7f3e133e7d20ee05dc4618c6102f0c420936e0fe72a9a094d951a9c4dad7a"
)


# -------------------------------------------------------------------------------------------------
# Durable Stage28-1C output.
# -------------------------------------------------------------------------------------------------

OUT = (
    STAGE28_ROOT
    / "stage28_1c_model_input_identity_freeze"
)

CONTRACTS_PATH = (
    OUT
    / "stage28_model_input_contracts.json"
)

BINDING_PATH = (
    OUT
    / "stage28_component_input_binding.csv"
)

FREEZE_PATH = (
    OUT
    / "stage28_1c_freeze_record.json"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


EXPECTED_COMPONENTS = 120
EXPECTED_EVALUATION_CELLS = 110
EXPECTED_NEW = 108
EXPECTED_REUSED = 12

EXPECTED_EXPERIMENT_COUNTS = {
    "STAGE22_FULL":
        20,

    "STAGE27_CHRONOLOGY_LOAO":
        50,

    "STAGE28B_RANDOM_LOAO":
        50,
}

CONTRACT_STAGE22 = (
    "STAGE22R_1C_BYTE_EXACT_70F_DEVELOPMENT_CACHE"
)

CONTRACT_CICIDS = (
    "CICIDS2017_STAGE24_FLAG_CORRECTED_STAGE27_FLOAT32_GLOBAL"
)


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return (
        run(
            ["git", *args]
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(
        path
    )

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(
            path
        ).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def canonical_json_sha256(
    obj,
):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
        ensure_ascii=False,
        allow_nan=False,
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        payload
    ).hexdigest()


def global_matrix_audit(
    array,
    block_rows=65_536,
):
    h = hashlib.sha256()

    inf_count = 0
    nan_count = 0

    for start in range(
        0,
        array.shape[0],
        block_rows,
    ):

        stop = min(
            start
            + block_rows,
            array.shape[0],
        )

        block = np.ascontiguousarray(
            array[
                start:stop,
                :
            ]
        )

        h.update(
            block.view(
                np.uint8
            )
        )

        inf_count += int(
            np.isinf(
                block
            ).sum()
        )

        nan_count += int(
            np.isnan(
                block
            ).sum()
        )

    return (
        h.hexdigest(),
        inf_count,
        nan_count,
    )


def write_json(
    path,
    obj,
):
    Path(
        path
    ).write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def recover_github_token():
    """
    Recover credential without printing it.
    """

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]

    # Kaggle Secrets first.
    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()

        for label in labels:

            try:
                value = client.get_secret(
                    label
                )

            except Exception:
                value = None

            if (
                isinstance(
                    value,
                    str,
                )
                and value.strip()
            ):

                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass


    # Environment fallback.
    for label in labels:

        value = os.environ.get(
            label
        )

        if (
            isinstance(
                value,
                str,
            )
            and value.strip()
        ):

            return (
                value.strip(),
                f"environment:{label}",
            )


    raise RuntimeError(
        "GitHub credential unavailable. "
        "No Stage28-1C durable files have been written."
    )


def authenticated_push(
    token,
):
    """
    Push without storing token in the remote URL
    and without printing the authorization header.
    """

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode(
            "utf-8"
        )
    ).decode(
        "ascii"
    )

    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(
            REPO
        ),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:

        raise RuntimeError(
            "Authenticated git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return (
        p.stdout
        + p.stderr
    ).strip()


# =================================================================================================
# 2. EXACT PARENT / CLEANNESS GATE
# =================================================================================================

banner(
    "STAGE28-1C-D — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote = (
    remote_line
    .split()[0]
)

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)


print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    head,
)

print(
    "origin/main    :",
    origin,
)

print(
    "Remote main    :",
    remote,
)

print(
    "Branch         :",
    branch,
)

print(
    "Subject        :",
    subject,
)

print(
    "Git clean      :",
    not bool(
        status
    ),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-1C-D parent mismatch."
    )

if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean before freeze:\n"
        + status
    )

if OUT.exists():
    raise RuntimeError(
        "Stage28-1C durable output directory "
        "already exists unexpectedly:\n"
        f"{OUT}"
    )


print()
print(
    "[PASS] exact clean Stage28-1B parent"
)


# =================================================================================================
# 3. RECOVER GITHUB CREDENTIAL BEFORE ANY DURABLE WRITE
# =================================================================================================

banner(
    "GITHUB WRITE AUTHORIZATION"
)

github_token, token_source = (
    recover_github_token()
)

print(
    "[PASS] GitHub credential recovered from",
    token_source,
)

print(
    "[PASS] token value intentionally not displayed"
)


remote_url = git(
    "remote",
    "get-url",
    "origin",
)


print(
    "Remote:",
    remote_url,
)


if (
    "github.com"
    not in remote_url
):
    raise RuntimeError(
        "Unexpected GitHub remote."
    )


# Refuse tokenized HTTPS remotes.
if (
    remote_url.startswith(
        "https://"
    )
    and "@github.com" in remote_url
):
    raise RuntimeError(
        "Remote URL appears to contain embedded credentials."
    )


print(
    "[PASS] remote URL contains no embedded token"
)


# =================================================================================================
# 4. RE-VERIFY STAGE28-1B EXECUTION UNIVERSE
# =================================================================================================

banner(
    "STAGE28-1B EXECUTION-UNIVERSE GATE"
)

manifest_receipt = read_json(
    require_file(
        MANIFEST_RECEIPT_PATH
    )
)

freeze_1b = read_json(
    require_file(
        STAGE28_1B_FREEZE_PATH
    )
)

actual_manifest_sha = sha256_file(
    require_file(
        MANIFEST_PATH
    )
)


print(
    "Frozen manifest SHA:",
    EXPECTED_MANIFEST_SHA,
)

print(
    "Actual manifest SHA:",
    actual_manifest_sha,
)


if (
    actual_manifest_sha
    != EXPECTED_MANIFEST_SHA
):
    raise RuntimeError(
        "Stage28 component manifest SHA mismatch."
    )

if (
    manifest_receipt[
        "manifest"
    ][
        "sha256"
    ]
    != EXPECTED_MANIFEST_SHA
):
    raise RuntimeError(
        "Manifest receipt SHA mismatch."
    )


with MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:

    manifest_rows = list(
        csv.DictReader(
            f
        )
    )


if len(
    manifest_rows
) != EXPECTED_COMPONENTS:
    raise RuntimeError(
        "Component manifest row count != 120."
    )


evaluation_cells = {
    row[
        "evaluation_cell_id"
    ]
    for row
    in manifest_rows
}

if len(
    evaluation_cells
) != EXPECTED_EVALUATION_CELLS:
    raise RuntimeError(
        "Evaluation-cell count != 110."
    )


fit_action_counts = Counter(
    row[
        "fit_action"
    ]
    for row
    in manifest_rows
)

if (
    fit_action_counts[
        "NEW_FIT_AUTHORIZED"
    ]
    != EXPECTED_NEW
):
    raise RuntimeError(
        "Manifest NEW-fit count != 108."
    )

if (
    fit_action_counts[
        "REUSE_EXISTING"
    ]
    != EXPECTED_REUSED
):
    raise RuntimeError(
        "Manifest reuse count != 12."
    )


experiment_counts = Counter(
    row[
        "experiment"
    ]
    for row
    in manifest_rows
)


print(
    "Components      :",
    len(
        manifest_rows
    ),
)

print(
    "Evaluation cells:",
    len(
        evaluation_cells
    ),
)

print(
    "NEW fits        :",
    fit_action_counts[
        "NEW_FIT_AUTHORIZED"
    ],
)

print(
    "Reused          :",
    fit_action_counts[
        "REUSE_EXISTING"
    ],
)

print(
    "Experiments     :",
    dict(
        experiment_counts
    ),
)


if dict(
    experiment_counts
) != EXPECTED_EXPERIMENT_COUNTS:
    raise RuntimeError(
        "Frozen experiment component counts changed."
    )


execution = (
    freeze_1b[
        "execution_manifest"
    ]
)

if (
    int(
        execution[
            "component_rows"
        ]
    )
    != EXPECTED_COMPONENTS
):
    raise RuntimeError(
        "Stage28-1B component_rows changed."
    )

if (
    int(
        execution[
            "scientific_evaluation_cells"
        ]
    )
    != EXPECTED_EVALUATION_CELLS
):
    raise RuntimeError(
        "Stage28-1B evaluation-cell count changed."
    )

if (
    int(
        execution[
            "new_fit_budget"
        ]
    )
    != EXPECTED_NEW
):
    raise RuntimeError(
        "Stage28-1B new-fit budget changed."
    )

if (
    int(
        execution[
            "new_fits_consumed"
        ]
    )
    != 0
):
    raise RuntimeError(
        "A Stage28 new fit was consumed before "
        "the model-input freeze."
    )

if (
    execution[
        "compute"
    ]
    != "CPU"
):
    raise RuntimeError(
        "Stage28 compute backend is no longer CPU."
    )


print()
print(
    "[PASS] 120-component execution universe unchanged"
)

print(
    "[PASS] 108/108 NEW fits remain unconsumed"
)

print(
    "[PASS] CPU-only execution remains frozen"
)


# =================================================================================================
# 5. BYTE-EXACT STAGE22 MODEL-INPUT CONTRACT
# =================================================================================================

banner(
    "STAGE22 BYTE-EXACT MODEL-INPUT IDENTITY"
)

stage22_manifest = read_json(
    require_file(
        STAGE22_MANIFEST_PATH
    )
)

stage22_remote = read_json(
    require_file(
        STAGE22_REMOTE_RECEIPT_PATH
    )
)

stage22_runtime = read_json(
    require_file(
        STAGE22_RUNTIME_RECEIPT_PATH
    )
)


if (
    stage22_runtime[
        "status"
    ]
    != "BYTE_EXACT_STAGE22_RUNTIME_CACHE_RECOVERED"
):
    raise RuntimeError(
        "Stage22 runtime recovery status invalid."
    )

if (
    stage22_runtime[
        "durable_scientific_parent"
    ]
    != EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage22 runtime recovery parent mismatch."
    )


stage22_manifest_sha = sha256_file(
    STAGE22_MANIFEST_PATH
)

expected_stage22_manifest_sha = (
    stage22_remote[
        "scientific_input_manifest"
    ][
        "sha256"
    ]
)


print(
    "Historical manifest SHA:",
    expected_stage22_manifest_sha,
)

print(
    "Current manifest SHA   :",
    stage22_manifest_sha,
)


if (
    stage22_manifest_sha
    != expected_stage22_manifest_sha
):
    raise RuntimeError(
        "Stage22 frozen model-input manifest SHA mismatch."
    )


historical_stage22_files = {
    row[
        "cache_file"
    ]:
    row

    for row
    in stage22_manifest[
        "cache"
    ][
        "files"
    ]
}

runtime_stage22_files = (
    stage22_runtime[
        "cache"
    ][
        "files"
    ]
)


if set(
    historical_stage22_files
) != set(
    runtime_stage22_files
):
    raise RuntimeError(
        "Stage22 runtime/historical cache file universe differs."
    )


if len(
    historical_stage22_files
) != 8:
    raise RuntimeError(
        "Stage22 cache does not contain exactly 8 files."
    )


stage22_file_contracts = []

stage22_total_rows = 0
stage22_total_bytes = 0


for filename in sorted(
    historical_stage22_files
):

    historical = (
        historical_stage22_files[
            filename
        ]
    )

    runtime_meta = (
        runtime_stage22_files[
            filename
        ]
    )

    runtime_path = (
        STAGE22_RUNTIME_CACHE
        / filename
    )

    require_file(
        runtime_path
    )


    actual_bytes = int(
        runtime_path.stat().st_size
    )

    actual_sha = sha256_file(
        runtime_path
    )

    expected_bytes = int(
        historical[
            "bytes"
        ]
    )

    expected_rows = int(
        historical[
            "rows"
        ]
    )

    expected_sha = (
        historical[
            "sha256"
        ]
    )


    if (
        actual_bytes
        != expected_bytes
    ):
        raise RuntimeError(
            f"{filename}: Stage22 byte-size mismatch."
        )

    if (
        actual_sha
        != expected_sha
    ):
        raise RuntimeError(
            f"{filename}: Stage22 SHA256 mismatch."
        )

    if (
        runtime_meta[
            "sha256"
        ]
        != expected_sha
    ):
        raise RuntimeError(
            f"{filename}: Stage28 runtime receipt SHA mismatch."
        )

    if (
        int(
            runtime_meta[
                "rows"
            ]
        )
        != expected_rows
    ):
        raise RuntimeError(
            f"{filename}: Stage28 runtime receipt row mismatch."
        )


    stage22_total_rows += (
        expected_rows
    )

    stage22_total_bytes += (
        expected_bytes
    )


    stage22_file_contracts.append(
        {
            "cache_file":
                filename,

            "source_file":
                historical[
                    "source_file"
                ],

            "rows":
                expected_rows,

            "attack":
                int(
                    historical[
                        "attack"
                    ]
                ),

            "benign":
                int(
                    historical[
                        "benign"
                    ]
                ),

            "bytes":
                expected_bytes,

            "sha256":
                expected_sha,
        }
    )


    print(
        "[PASS]",
        filename,
    )

    print(
        "       rows  :",
        f"{expected_rows:,}",
    )

    print(
        "       bytes :",
        f"{expected_bytes:,}",
    )

    print(
        "       SHA256:",
        expected_sha,
    )


if (
    stage22_total_rows
    != 14_412_403
):
    raise RuntimeError(
        "Stage22 total row count mismatch."
    )

if (
    stage22_total_bytes
    != 1_541_208_291
):
    raise RuntimeError(
        "Stage22 total byte count mismatch."
    )


stage22_contract_core = {

    "contract_id":
        CONTRACT_STAGE22,

    "scientific_source":
        "FROZEN_STAGE22R_1C_MODEL_INPUTS",

    "historical_manifest": {
        "path":
            str(
                STAGE22_MANIFEST_PATH.relative_to(
                    REPO
                )
            ),

        "sha256":
            stage22_manifest_sha,
    },

    "reset_safe_checkpoint": {
        "dataset_id":
            stage22_remote[
                "kaggle_dataset"
            ][
                "dataset_id"
            ],

        "status":
            stage22_remote[
                "status"
            ],
    },

    "runtime_path":
        str(
            STAGE22_RUNTIME_CACHE
        ),

    "runtime_committed_to_git":
        False,

    "feature_count":
        70,

    "feature_dtype":
        stage22_manifest[
            "feature_configuration"
        ][
            "dtype"
        ],

    "rows":
        stage22_total_rows,

    "bytes":
        stage22_total_bytes,

    "files":
        stage22_file_contracts,

    "numeric_policy":
        stage22_manifest[
            "numeric_policy"
        ],

    "scientific_identity_semantics":
        (
            "EIGHT_BYTE_EXACT_HISTORICAL_PARQUET_SHA256_IDENTITIES"
        ),
}


stage22_contract_sha = (
    canonical_json_sha256(
        stage22_contract_core
    )
)


print()
print(
    "Stage22 Stage28 contract SHA256:"
)

print(
    " ",
    stage22_contract_sha,
)

print()
print(
    "[PASS] Stage22 runtime cache is still "
    "byte-identical to historical Stage22R-1C"
)


# =================================================================================================
# 6. CICIDS2017 GLOBAL FLOAT32 MODEL-INPUT CONTRACT
# =================================================================================================

banner(
    "CICIDS2017 GLOBAL FLOAT32 MODEL-INPUT IDENTITY"
)

cicids_runtime = read_json(
    require_file(
        CICIDS_RUNTIME_RECEIPT_PATH
    )
)


if (
    cicids_runtime[
        "status"
    ]
    !=
    "CICIDS2017_MODEL_INPUT_CACHE_RECOVERED_AND_HISTORICALLY_VERIFIED"
):
    raise RuntimeError(
        "Stage28-1C-C runtime recovery status invalid."
    )

if (
    cicids_runtime[
        "durable_scientific_parent"
    ]
    != EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-1C-C parent mismatch."
    )


if (
    cicids_runtime[
        "fit_ledger"
    ][
        "new_consumed"
    ]
    != 0
):
    raise RuntimeError(
        "Stage28-1C-C reports a consumed fit."
    )


global_meta = (
    cicids_runtime[
        "global_float32_cache"
    ]
)


if (
    global_meta[
        "content_sha256"
    ]
    != EXPECTED_CICIDS_GLOBAL_CONTENT_SHA
):
    raise RuntimeError(
        "Stage28-1C-C global FLOAT32 "
        "receipt SHA mismatch."
    )


require_file(
    CICIDS_GLOBAL_MATRIX_PATH
)

global_X = np.load(
    CICIDS_GLOBAL_MATRIX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)


print(
    "Shape:",
    global_X.shape,
)

print(
    "Dtype:",
    global_X.dtype,
)

print(
    "Hashing complete global FLOAT32 logical content..."
)


if global_X.shape != (
    EXPECTED_CICIDS_ROWS,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        "CICIDS2017 global matrix shape mismatch."
    )

if global_X.dtype != np.float32:
    raise RuntimeError(
        "CICIDS2017 global matrix dtype != float32."
    )


(
    global_content_sha,
    global_inf,
    global_nan,
) = global_matrix_audit(
    global_X
)


print(
    "Expected content SHA:",
    EXPECTED_CICIDS_GLOBAL_CONTENT_SHA,
)

print(
    "Actual content SHA  :",
    global_content_sha,
)

print(
    "Infinite cells      :",
    f"{global_inf:,}",
)

print(
    "NaN cells           :",
    f"{global_nan:,}",
)


if (
    global_content_sha
    != EXPECTED_CICIDS_GLOBAL_CONTENT_SHA
):
    raise RuntimeError(
        "CICIDS2017 global FLOAT32 logical "
        "content SHA mismatch."
    )

if global_inf != 0:
    raise RuntimeError(
        "Infinity exists in frozen CICIDS2017 "
        "FLOAT32 model matrix."
    )

if global_nan != 5_734:
    raise RuntimeError(
        "CICIDS2017 global NaN count mismatch."
    )


crosscheck = (
    cicids_runtime[
        "stage27_independent_crosscheck"
    ]
)


if (
    crosscheck[
        "status"
    ]
    != "EXACT_MATCH"
):
    raise RuntimeError(
        "Stage27 independent cross-check is not exact."
    )

if (
    crosscheck[
        "feature_float32_content_sha256"
    ]
    != EXPECTED_STAGE27_MONWED_SHA
):
    raise RuntimeError(
        "Stage27 Mon-Wed independent feature SHA changed."
    )


stage27_historical = read_json(
    require_file(
        STAGE27_MONWED_RECEIPT_PATH
    )
)

if (
    stage27_historical[
        "artifacts"
    ][
        "features_float32"
    ][
        "content_sha256"
    ]
    != EXPECTED_STAGE27_MONWED_SHA
):
    raise RuntimeError(
        "Historical Stage27 Mon-Wed receipt changed."
    )


stage27_feature_spec = read_json(
    require_file(
        STAGE27_FEATURE_SPEC_PATH
    )
)

adapter = (
    stage27_feature_spec[
        "cicids2017_semantic_adapter"
    ]
)

if (
    adapter[
        "mapping_sha256"
    ]
    != EXPECTED_STAGE27_ADAPTER_SHA
):
    raise RuntimeError(
        "Stage27 adapter SHA mismatch."
    )


stage24_result_sha = sha256_file(
    require_file(
        STAGE24_RESULT_PATH
    )
)

if (
    stage24_result_sha
    !=
    cicids_runtime[
        "stage24_reproduction"
    ][
        "stage24_result_sha256"
    ]
):
    raise RuntimeError(
        "Stage24 result identity differs "
        "from Stage28-1C-C receipt."
    )


per_source = (
    cicids_runtime[
        "stage24_reproduction"
    ][
        "per_source"
    ]
)

if len(
    per_source
) != 8:
    raise RuntimeError(
        "Stage28-1C-C per-source audit count != 8."
    )


for row in per_source:

    if not row[
        "stage24_float64_identity_exact"
    ]:
        raise RuntimeError(
            "A Stage24 FLOAT64 source matrix "
            "was not reproduced exactly."
        )

    if (
        row[
            "stage24_float64_matrix_sha256_expected"
        ]
        !=
        row[
            "stage28_reproduced_float64_matrix_sha256"
        ]
    ):
        raise RuntimeError(
            "Stage24 expected/reproduced matrix SHA differs."
        )


cicids_contract_core = {

    "contract_id":
        CONTRACT_CICIDS,

    "scientific_source":
        (
            "STAGE24_ALREADY_OPENED_FULL_EFFECTIVE_CICIDS2017_"
            "FLAG_CORRECTED_REPRESENTATION"
        ),

    "runtime_path":
        str(
            CICIDS_GLOBAL_MATRIX_PATH
        ),

    "runtime_committed_to_git":
        False,

    "rows":
        EXPECTED_CICIDS_ROWS,

    "features":
        EXPECTED_FEATURES,

    "dtype":
        "float32",

    "logical_content_sha256":
        global_content_sha,

    "npy_bytes":
        int(
            CICIDS_GLOBAL_MATRIX_PATH.stat().st_size
        ),

    "nan_cells":
        global_nan,

    "infinite_cells":
        global_inf,

    "stage24": {
        "result_path":
            str(
                STAGE24_RESULT_PATH.relative_to(
                    REPO
                )
            ),

        "result_sha256":
            stage24_result_sha,

        "source_count":
            8,

        "all_source_bytes_exact":
            True,

        "all_float64_matrix_hashes_exact":
            True,

        "all_numeric_audits_exact":
            True,

        "float64_hash_semantics":
            (
                "SHA256("
                "NP_ASCONTIGUOUSARRAY_MATRIX"
                ".VIEW_UINT8)"
            ),

        "per_source":
            per_source,
    },

    "stage27": {
        "feature_spec_path":
            str(
                STAGE27_FEATURE_SPEC_PATH.relative_to(
                    REPO
                )
            ),

        "adapter_variant":
            adapter[
                "variant"
            ],

        "adapter_mapping_sha256":
            adapter[
                "mapping_sha256"
            ],

        "final_model_matrix_dtype":
            "float32",

        "historical_monwed_known_rows":
            1_668_519,

        "historical_monwed_float32_content_sha256":
            EXPECTED_STAGE27_MONWED_SHA,

        "reproduced_monwed_float32_content_sha256":
            crosscheck[
                "feature_float32_content_sha256"
            ],

        "independent_crosscheck":
            "EXACT_MATCH",
    },

    "scientific_identity_semantics":
        (
            "GLOBAL_STAGE27_ROW_ORDER_FLOAT32_LOGICAL_CONTENT_SHA256"
        ),
}


cicids_contract_sha = (
    canonical_json_sha256(
        cicids_contract_core
    )
)


print()
print(
    "CICIDS2017 global content SHA256:"
)

print(
    " ",
    global_content_sha,
)

print()

print(
    "Stage28 CICIDS input-contract SHA256:"
)

print(
    " ",
    cicids_contract_sha,
)

print()

print(
    "[PASS] full CICIDS2017 Stage27-format "
    "model-input identity exact"
)

print(
    "[PASS] historical Stage27 Mon-Wed "
    "FLOAT32 cross-check remains exact"
)


# =================================================================================================
# 7. BIND ALL 120 COMPONENTS TO FROZEN INPUT CONTRACTS
# =================================================================================================

banner(
    "BIND 120 COMPONENTS TO MODEL-INPUT CONTRACTS"
)

binding_rows = []


for row in manifest_rows:

    experiment = (
        row[
            "experiment"
        ]
    )


    if experiment == "STAGE22_FULL":

        contract_id = (
            CONTRACT_STAGE22
        )

        contract_sha = (
            stage22_contract_sha
        )

        input_runtime_asset = str(
            STAGE22_RUNTIME_CACHE
        )

        input_identity = (
            stage22_contract_sha
        )


    elif experiment in {
        "STAGE27_CHRONOLOGY_LOAO",
        "STAGE28B_RANDOM_LOAO",
    }:

        contract_id = (
            CONTRACT_CICIDS
        )

        contract_sha = (
            cicids_contract_sha
        )

        input_runtime_asset = str(
            CICIDS_GLOBAL_MATRIX_PATH
        )

        input_identity = (
            global_content_sha
        )


    else:

        raise RuntimeError(
            "Unexpected Stage28 experiment "
            f"in execution manifest: {experiment}"
        )


    if (
        row[
            "compute_backend"
        ]
        != "CPU"
    ):
        raise RuntimeError(
            f"{row['component_id']}: "
            "non-CPU component found."
        )


    binding_rows.append(
        {
            "component_ordinal":
                row[
                    "component_ordinal"
                ],

            "component_id":
                row[
                    "component_id"
                ],

            "arm":
                row[
                    "arm"
                ],

            "experiment":
                experiment,

            "unit":
                row[
                    "unit"
                ],

            "evaluation_cell_id":
                row[
                    "evaluation_cell_id"
                ],

            "learner":
                row[
                    "learner"
                ],

            "model_seed":
                row[
                    "model_seed"
                ],

            "compute_backend":
                row[
                    "compute_backend"
                ],

            "membership_reference":
                row[
                    "membership_reference"
                ],

            "fit_action":
                row[
                    "fit_action"
                ],

            "input_contract_id":
                contract_id,

            "input_contract_sha256":
                contract_sha,

            "input_scientific_identity_sha256":
                input_identity,

            "runtime_asset":
                input_runtime_asset,
        }
    )


if len(
    binding_rows
) != EXPECTED_COMPONENTS:
    raise RuntimeError(
        "Component-input binding count != 120."
    )


binding_contract_counts = Counter(
    row[
        "input_contract_id"
    ]
    for row
    in binding_rows
)


expected_binding_counts = {
    CONTRACT_STAGE22:
        20,

    CONTRACT_CICIDS:
        100,
}


print(
    "Binding counts:"
)

for key, value in (
    binding_contract_counts.items()
):

    print(
        " ",
        key,
        "=",
        value,
    )


if dict(
    binding_contract_counts
) != expected_binding_counts:
    raise RuntimeError(
        "Model-input binding counts are incorrect."
    )


if {
    row[
        "component_id"
    ]
    for row
    in binding_rows
} != {
    row[
        "component_id"
    ]
    for row
    in manifest_rows
}:
    raise RuntimeError(
        "Component binding universe differs "
        "from frozen execution manifest."
    )


print()
print(
    "[PASS] all 120 components have exactly one input contract"
)

print(
    "[PASS] Stage22 FULL -> Stage22 byte-exact cache"
)

print(
    "[PASS] Stage27 chronology + Stage28B random -> "
    "CICIDS2017 global float32 cache"
)


# =================================================================================================
# 8. CREATE DURABLE OUTPUT DIRECTORY
# =================================================================================================

banner(
    "CREATE STAGE28-1C DURABLE PROVENANCE"
)

OUT.mkdir(
    parents=True,
    exist_ok=False,
)


# =================================================================================================
# 9. WRITE MODEL-INPUT CONTRACTS
# =================================================================================================

contracts_document = {

    "stage":
        "Stage28-1C",

    "type":
        "FROZEN_MODEL_INPUT_CONTRACTS",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent_commit":
        EXPECTED_PARENT,

    "execution_manifest": {
        "path":
            str(
                MANIFEST_PATH.relative_to(
                    REPO
                )
            ),

        "sha256":
            EXPECTED_MANIFEST_SHA,

        "component_rows":
            EXPECTED_COMPONENTS,

        "scientific_evaluation_cells":
            EXPECTED_EVALUATION_CELLS,
    },

    "contracts": {

        CONTRACT_STAGE22: {
            "contract_sha256":
                stage22_contract_sha,

            **stage22_contract_core,
        },

        CONTRACT_CICIDS: {
            "contract_sha256":
                cicids_contract_sha,

            **cicids_contract_core,
        },
    },

    "binding_rule": {
        "STAGE22_FULL":
            CONTRACT_STAGE22,

        "STAGE27_CHRONOLOGY_LOAO":
            CONTRACT_CICIDS,

        "STAGE28B_RANDOM_LOAO":
            CONTRACT_CICIDS,
    },

    "runtime_artifact_policy": {
        "runtime_model_inputs_committed_to_git":
            False,

        "reason":
            (
                "Large runtime caches are reproducible from "
                "already-frozen historical source identities; "
                "Git stores only their scientific identities."
            ),
    },

    "status":
        "FROZEN_BEFORE_FIRST_STAGE28_MODEL_FIT",
}


write_json(
    CONTRACTS_PATH,
    contracts_document,
)


print(
    "[WRITE]",
    CONTRACTS_PATH.relative_to(
        REPO
    ),
)


# =================================================================================================
# 10. WRITE 120-COMPONENT INPUT BINDING
# =================================================================================================

binding_fields = [
    "component_ordinal",
    "component_id",
    "arm",
    "experiment",
    "unit",
    "evaluation_cell_id",
    "learner",
    "model_seed",
    "compute_backend",
    "membership_reference",
    "fit_action",
    "input_contract_id",
    "input_contract_sha256",
    "input_scientific_identity_sha256",
    "runtime_asset",
]


with BINDING_PATH.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=binding_fields,
        lineterminator="\n",
    )

    writer.writeheader()

    writer.writerows(
        binding_rows
    )


binding_sha = sha256_file(
    BINDING_PATH
)


print(
    "[WRITE]",
    BINDING_PATH.relative_to(
        REPO
    ),
)

print(
    "        rows  :",
    len(
        binding_rows
    ),
)

print(
    "        SHA256:",
    binding_sha,
)


# =================================================================================================
# 11. WRITE FINAL STAGE28-1C FREEZE RECORD
# =================================================================================================

contracts_file_sha = sha256_file(
    CONTRACTS_PATH
)

stage22_runtime_receipt_sha = sha256_file(
    STAGE22_RUNTIME_RECEIPT_PATH
)

cicids_runtime_receipt_sha = sha256_file(
    CICIDS_RUNTIME_RECEIPT_PATH
)


freeze_record = {

    "stage":
        "Stage28-1C",

    "type":
        "MODEL_INPUT_IDENTITY_AND_COMPONENT_BINDING_FREEZE",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_PARENT,

    "status":
        "FROZEN_BEFORE_FIRST_STAGE28_MODEL_FIT",

    "execution_manifest": {
        "path":
            str(
                MANIFEST_PATH.relative_to(
                    REPO
                )
            ),

        "sha256":
            EXPECTED_MANIFEST_SHA,

        "components":
            EXPECTED_COMPONENTS,

        "scientific_evaluation_cells":
            EXPECTED_EVALUATION_CELLS,

        "existing_reused":
            EXPECTED_REUSED,

        "new_fit_budget":
            EXPECTED_NEW,

        "new_fits_consumed":
            0,

        "new_fits_remaining":
            EXPECTED_NEW,

        "compute":
            "CPU",

        "gpu_authorized":
            False,
    },

    "model_input_contracts": {
        "document":
            CONTRACTS_PATH.name,

        "document_sha256":
            contracts_file_sha,

        "stage22": {
            "contract_id":
                CONTRACT_STAGE22,

            "contract_sha256":
                stage22_contract_sha,

            "rows":
                stage22_total_rows,

            "files":
                8,

            "byte_exact_historical_cache":
                True,
        },

        "cicids2017": {
            "contract_id":
                CONTRACT_CICIDS,

            "contract_sha256":
                cicids_contract_sha,

            "rows":
                EXPECTED_CICIDS_ROWS,

            "features":
                EXPECTED_FEATURES,

            "dtype":
                "float32",

            "logical_content_sha256":
                global_content_sha,

            "historical_stage27_monwed_float32_sha256":
                EXPECTED_STAGE27_MONWED_SHA,

            "stage27_monwed_crosscheck":
                "EXACT_MATCH",

            "all_eight_stage24_float64_hashes":
                "EXACT_MATCH",
        },
    },

    "component_input_binding": {
        "path":
            BINDING_PATH.name,

        "rows":
            EXPECTED_COMPONENTS,

        "sha256":
            binding_sha,

        "stage22_bound_components":
            20,

        "cicids2017_bound_components":
            100,
    },

    "runtime_recovery_provenance": {
        "stage22_runtime_receipt": {
            "path":
                str(
                    STAGE22_RUNTIME_RECEIPT_PATH
                ),

            "sha256":
                stage22_runtime_receipt_sha,

            "committed_to_git":
                False,
        },

        "cicids2017_runtime_receipt": {
            "path":
                str(
                    CICIDS_RUNTIME_RECEIPT_PATH
                ),

            "sha256":
                cicids_runtime_receipt_sha,

            "committed_to_git":
                False,
        },
    },

    "scientific_accounting": {
        "stage22_raw_predictor_recomputation":
            0,

        "stage22_predictor_rows_reopened_during_recovery":
            0,

        "cicids2017_predictor_rows_reproduced_from_already_opened_stage24_population":
            EXPECTED_CICIDS_ROWS,

        "new_scientific_target_openings":
            0,

        "target_adaptive_choices":
            0,

        "model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "bootstrap":
            0,
    },

    "invariants": {
        "protocol_changed":
            False,

        "model_configuration_changed":
            False,

        "membership_rule_changed":
            False,

        "threshold_rule_changed":
            False,

        "feature_mapping_changed":
            False,

        "population_changed":
            False,

        "fit_budget_changed":
            False,

        "execution_manifest_changed":
            False,

        "runtime_large_files_committed":
            False,
    },

    "next_authorized_step":
        (
            "Stage28-2A — begin the frozen Stage22 FULL "
            "CPU training-seed stability execution. "
            "FIT #1 may occur only after re-verifying "
            "this Stage28-1C freeze and the 120-component manifest."
        ),
}


write_json(
    FREEZE_PATH,
    freeze_record,
)


print(
    "[WRITE]",
    FREEZE_PATH.relative_to(
        REPO
    ),
)


# =================================================================================================
# 12. WRITE CHECKSUM MANIFEST
# =================================================================================================

durable_payloads = [
    CONTRACTS_PATH,
    BINDING_PATH,
    FREEZE_PATH,
]


checksum_lines = []

for path in durable_payloads:

    digest = sha256_file(
        path
    )

    checksum_lines.append(
        f"{digest}  {path.name}"
    )


CHECKSUMS_PATH.write_text(
    "\n".join(
        checksum_lines
    )
    + "\n",
    encoding="utf-8",
)


print(
    "[WRITE]",
    CHECKSUMS_PATH.relative_to(
        REPO
    ),
)

print()

print(
    CHECKSUMS_PATH.read_text(
        encoding="utf-8"
    ).rstrip()
)


# =================================================================================================
# 13. SELF-VERIFY DURABLE ARTIFACTS
# =================================================================================================

banner(
    "SELF-VERIFY STAGE28-1C DURABLE ARTIFACTS"
)

expected_files = {
    CONTRACTS_PATH.name,
    BINDING_PATH.name,
    FREEZE_PATH.name,
    CHECKSUMS_PATH.name,
}

actual_files = {
    p.name
    for p
    in OUT.iterdir()
    if p.is_file()
}


if (
    actual_files
    != expected_files
):
    raise RuntimeError(
        "Unexpected Stage28-1C output file universe.\n"
        f"Expected: {sorted(expected_files)}\n"
        f"Actual:   {sorted(actual_files)}"
    )


for line in (
    CHECKSUMS_PATH
    .read_text(
        encoding="utf-8"
    )
    .splitlines()
):

    digest, filename = (
        line.split(
            None,
            1,
        )
    )

    filename = (
        filename.strip()
        .lstrip("*")
    )

    artifact = (
        OUT
        / filename
    )

    actual = sha256_file(
        artifact
    )

    print(
        "[CHECK]",
        filename,
    )

    print(
        "        expected:",
        digest,
    )

    print(
        "        actual  :",
        actual,
    )

    if actual != digest:
        raise RuntimeError(
            f"Durable artifact checksum mismatch: {filename}"
        )


print()
print(
    "[PASS] all Stage28-1C durable checksums exact"
)


# =================================================================================================
# 14. VERIFY ONLY AUTHORIZED GIT CHANGES EXIST
# =================================================================================================

banner(
    "AUTHORIZED GIT-CHANGE GATE"
)

out_rel = str(
    OUT.relative_to(
        REPO
    )
)

tracked_modifications = (
    git(
        "diff",
        "--name-only",
    )
    .splitlines()
)

untracked = (
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
        "--",
        out_rel,
    )
    .splitlines()
)


if tracked_modifications:
    raise RuntimeError(
        "Tracked files changed unexpectedly before commit:\n"
        + "\n".join(
            tracked_modifications
        )
    )


expected_untracked = {
    str(
        path.relative_to(
            REPO
        )
    )
    for path
    in [
        CONTRACTS_PATH,
        BINDING_PATH,
        FREEZE_PATH,
        CHECKSUMS_PATH,
    ]
}


if set(
    untracked
) != expected_untracked:
    raise RuntimeError(
        "Unexpected untracked Stage28-1C files.\n"
        f"Expected:\n"
        + "\n".join(
            sorted(
                expected_untracked
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                untracked
            )
        )
    )


# Also ensure there are no unrelated untracked files inside Git.
all_repo_untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    )
    .splitlines()
)


if (
    all_repo_untracked
    != expected_untracked
):
    unexpected = (
        all_repo_untracked
        -
        expected_untracked
    )

    raise RuntimeError(
        "Unrelated untracked files exist in repository:\n"
        + "\n".join(
            sorted(
                unexpected
            )
        )
    )


print(
    "[PASS] exactly four authorized durable files exist"
)

print(
    "[PASS] no runtime matrix/cache file is inside Git"
)


# =================================================================================================
# 15. FINAL PRE-COMMIT FIT LEDGER ASSERTION
# =================================================================================================

banner(
    "FINAL PRE-COMMIT SCIENTIFIC LEDGER"
)

# Re-read the immutable Stage28-1B freeze immediately before commit.
freeze_1b_final = read_json(
    STAGE28_1B_FREEZE_PATH
)

if (
    freeze_1b_final[
        "execution_manifest"
    ][
        "new_fits_consumed"
    ]
    != 0
):
    raise RuntimeError(
        "Fit ledger changed before Stage28-1C commit."
    )


print(
    "NEW fits authorized:",
    EXPECTED_NEW,
)

print(
    "NEW fits consumed  : 0"
)

print(
    "NEW fits remaining :",
    EXPECTED_NEW,
)

print()

print(
    "MODEL_FITS                = 0"
)

print(
    "MODEL_INFERENCE           = 0"
)

print(
    "THRESHOLD_SELECTION       = 0"
)

print(
    "NEW_SCIENTIFIC_OPENINGS   = 0"
)

print()

print(
    "[PASS] FIT #1 remains unconsumed"
)


# =================================================================================================
# 16. STAGE ONLY THE FOUR AUTHORIZED FILES
# =================================================================================================

banner(
    "STAGE STAGE28-1C DURABLE PROVENANCE"
)

for path in [
    CONTRACTS_PATH,
    BINDING_PATH,
    FREEZE_PATH,
    CHECKSUMS_PATH,
]:

    run(
        [
            "git",
            "add",
            "--",
            str(
                path.relative_to(
                    REPO
                )
            ),
        ]
    )


staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    )
    .splitlines()
)


if (
    staged
    != expected_untracked
):
    raise RuntimeError(
        "Staged file universe is not exactly "
        "the four authorized Stage28-1C files.\n"
        + "\n".join(
            sorted(
                staged
            )
        )
    )


print(
    "[PASS] exactly four Stage28-1C files staged"
)

for path in sorted(
    staged
):
    print(
        " ",
        path,
    )


# =================================================================================================
# 17. COMMIT
# =================================================================================================

banner(
    "COMMIT STAGE28-1C FREEZE"
)

# Preserve existing repository identity when configured.
git_name = git(
    "config",
    "--get",
    "user.name",
)

git_email = git(
    "config",
    "--get",
    "user.email",
)


if not git_name:
    run(
        [
            "git",
            "config",
            "user.name",
            "Stage28 Kaggle",
        ]
    )

if not git_email:
    run(
        [
            "git",
            "config",
            "user.email",
            "stage28-kaggle@users.noreply.github.com",
        ]
    )


commit_result = run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ]
)


print(
    commit_result.stdout.strip()
)


new_head = git(
    "rev-parse",
    "HEAD",
)

new_parent = git(
    "rev-parse",
    "HEAD^",
)

new_subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)


print()
print(
    "Parent :",
    new_parent,
)

print(
    "Commit :",
    new_head,
)

print(
    "Subject:",
    new_subject,
)


if (
    new_parent
    != EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-1C commit parent mismatch."
    )

if (
    new_subject
    != COMMIT_MESSAGE
):
    raise RuntimeError(
        "Unexpected Stage28-1C commit subject."
    )


print()
print(
    "[PASS] Stage28-1C provenance committed"
)


# =================================================================================================
# 18. PUSH WITH SAFE AUTH
# =================================================================================================

banner(
    "PUSH STAGE28-1C FREEZE"
)

push_output = authenticated_push(
    github_token
)

if push_output:
    print(
        push_output
    )


# Token no longer needed.
github_token = None


# =================================================================================================
# 19. REMOTE DURABILITY VERIFICATION
# =================================================================================================

banner(
    "REMOTE DURABILITY VERIFICATION"
)

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_final = git(
    "rev-parse",
    "HEAD",
)

origin_final = git(
    "rev-parse",
    "origin/main",
)

remote_final_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_final_line:
    raise RuntimeError(
        "Unable to resolve pushed remote main."
    )

remote_final = (
    remote_final_line
    .split()[0]
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD :",
    local_final,
)

print(
    "origin/main:",
    origin_final,
)

print(
    "Remote main:",
    remote_final,
)

print(
    "Git clean  :",
    not bool(
        final_status
    ),
)


if not (
    local_final
    == origin_final
    == remote_final
    == new_head
):
    raise RuntimeError(
        "Stage28-1C remote durability gate failed."
    )

if final_status:
    raise RuntimeError(
        "Repository dirty after Stage28-1C push:\n"
        + final_status
    )


print()
print(
    "[PASS] Stage28-1C commit remotely durable"
)

print(
    "[PASS] worktree clean"
)


# =================================================================================================
# 20. RE-VERIFY COMMITTED ARTIFACTS FROM HEAD
# =================================================================================================

banner(
    "POST-COMMIT ARTIFACT GATE"
)

for relative in sorted(
    expected_untracked
):

    tracked = git(
        "ls-files",
        "--error-unmatch",
        relative,
    )

    if tracked != relative:
        raise RuntimeError(
            f"Committed artifact missing from Git: {relative}"
        )

    print(
        "[PASS]",
        relative,
    )


# Re-open committed freeze record.
committed_freeze = read_json(
    FREEZE_PATH
)


if (
    committed_freeze[
        "status"
    ]
    != "FROZEN_BEFORE_FIRST_STAGE28_MODEL_FIT"
):
    raise RuntimeError(
        "Committed Stage28-1C freeze status invalid."
    )

if (
    committed_freeze[
        "execution_manifest"
    ][
        "new_fits_consumed"
    ]
    != 0
):
    raise RuntimeError(
        "Committed Stage28-1C ledger is not zero."
    )

if (
    committed_freeze[
        "model_input_contracts"
    ][
        "cicids2017"
    ][
        "logical_content_sha256"
    ]
    != EXPECTED_CICIDS_GLOBAL_CONTENT_SHA
):
    raise RuntimeError(
        "Committed CICIDS model-input identity changed."
    )


print()
print(
    "[PASS] committed freeze record scientifically exact"
)


# =================================================================================================
# 21. FINAL
# =================================================================================================

banner(
    "STAGE28-1C — MODEL-INPUT IDENTITY FREEZE COMPLETE"
)

print(
    "Durable Stage28-1C commit:"
)

print(
    " ",
    new_head,
)

print()

print(
    "Parent:"
)

print(
    " ",
    EXPECTED_PARENT,
)

print()

print(
    "Execution universe:"
)

print(
    "  components       = 120"
)

print(
    "  evaluation cells = 110"
)

print(
    "  reused models    = 12"
)

print(
    "  NEW fits         = 108"
)

print(
    "  CPU only         = YES"
)

print()

print(
    "Stage22 input contract:"
)

print(
    "  files            = 8 / 8 BYTE EXACT"
)

print(
    "  rows             =",
    f"{stage22_total_rows:,}",
)

print(
    "  bytes            =",
    f"{stage22_total_bytes:,}",
)

print(
    "  contract SHA256  =",
    stage22_contract_sha,
)

print()

print(
    "CICIDS2017 input contract:"
)

print(
    "  rows             =",
    f"{EXPECTED_CICIDS_ROWS:,}",
)

print(
    "  features         = 70"
)

print(
    "  dtype            = float32"
)

print(
    "  content SHA256   =",
    global_content_sha,
)

print(
    "  Stage27 MonWed   = EXACT",
    EXPECTED_STAGE27_MONWED_SHA,
)

print(
    "  contract SHA256  =",
    cicids_contract_sha,
)

print()

print(
    "Component binding:"
)

print(
    "  Stage22 FULL components       = 20"
)

print(
    "  CICIDS chronology components  = 50"
)

print(
    "  CICIDS random-LOAO components = 50"
)

print(
    "  total bound                    = 120 / 120"
)

print()

print(
    "Scientific operations:"
)

print(
    "  MODEL_FITS              = 0"
)

print(
    "  MODEL_INFERENCE         = 0"
)

print(
    "  THRESHOLD_SELECTION     = 0"
)

print(
    "  NEW_TARGET_OPENINGS     = 0"
)

print(
    "  TARGET_ADAPTIVE_CHOICES = 0"
)

print()

print(
    "Fit ledger:"
)

print(
    "  NEW authorized = 108"
)

print(
    "  consumed       = 0"
)

print(
    "  remaining      = 108"
)

print()

print(
    "STATUS:"
)

print(
    "  ALL MODEL INPUTS + MEMBERSHIP REFERENCES + "
    "EXECUTION COMPONENTS ARE NOW FROZEN BEFORE FIT #1."
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage28-2A — Stage22 FULL CPU training-seed stability."
)

print(
    "  This is where the first NEW Stage28 model fit may occur."
)

print()

print(
    "  Do not begin Stage27 chronology or Stage28B random LOAO "
    "until the Stage22 Stage28-2A execution checkpoint is handled."
)

print()
print(SEP)


STAGE28-1C-D — EXACT DURABLE-PARENT GATE

Expected parent: ba011ec01f1399939111b24664ebd5d66c630f95
Local HEAD     : ba011ec01f1399939111b24664ebd5d66c630f95
origin/main    : ba011ec01f1399939111b24664ebd5d66c630f95
Remote main    : ba011ec01f1399939111b24664ebd5d66c630f95
Branch         : main
Subject        : stage28-1b: freeze random LOAO memberships and execution manifest
Git clean      : True

[PASS] exact clean Stage28-1B parent

GITHUB WRITE AUTHORIZATION

[PASS] GitHub credential recovered from kaggle_secret:GITHUB_TOKEN
[PASS] token value intentionally not displayed
Remote: https://github.com/themubasshir/ids2018-validation-safe-ablation.git
[PASS] remote URL contains no embedded token

STAGE28-1B EXECUTION-UNIVERSE GATE

Frozen manifest SHA: 47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505
Actual manifest SHA: 47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505
Components      : 120
Evaluation cells: 110
NEW fits        : 108
Reused          : 12

In [10]:
# =================================================================================================
# STAGE28-2A0 — STAGE22 FULL EXECUTION MATRIX + MEMBERSHIP MATERIALIZATION
#
# Expected durable parent:
#   738e3ee29d098d4132828aeaaacfa12a8cfd7b52
#
# PURPOSE
# -------
# Prepare the already-frozen Stage22 FULL development population for the
# Stage28-2A seed-stability fits.
#
# This cell:
#
#   1. verifies the Stage28-1C durable freeze
#   2. re-verifies all 8 byte-exact Stage22R-1C cache Parquets
#   3. materializes ONE canonical 14,412,403 x 70 float64 execution matrix
#   4. materializes the exact binary labels in the same canonical order
#   5. verifies clean_position == 0..14,412,402 exactly
#   6. verifies day geometry and class counts
#   7. reconstructs the inherited RANDOM_NATURAL membership from the
#      historical little-endian random_validation.packbits
#   8. reconstructs CHRONOLOGICAL_NATURAL as day_id 0..6 train / day_id 7 validation
#   9. saves disposable runtime membership arrays
#
# ZERO:
#   model fits
#   model inference
#   threshold selection
#   final-holdout opening
#   Git modification
#
# After this passes:
#
#   Stage28-2A1 = RANDOM_NATURAL seed42
#                  - FIT #1 = CPU XGBoost
#                  - LightGBM seed42 = exact historical CPU reuse
#                  - validation inference
#                  - frozen per-seed thresholds
#
# =================================================================================================

from __future__ import annotations

import gc
import hashlib
import json
import os
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_HEAD = (
    "738e3ee29d098d4132828aeaaacfa12a8cfd7b52"
)

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

STAGE28_1C_DIR = (
    STAGE28_ROOT
    / "stage28_1c_model_input_identity_freeze"
)

STAGE28_1C_FREEZE = (
    STAGE28_1C_DIR
    / "stage28_1c_freeze_record.json"
)

STAGE28_INPUT_CONTRACTS = (
    STAGE28_1C_DIR
    / "stage28_model_input_contracts.json"
)

STAGE28_BINDING = (
    STAGE28_1C_DIR
    / "stage28_component_input_binding.csv"
)

STAGE28_2A_DURABLE_ROOT = (
    STAGE28_ROOT
    / "stage28_2a_stage22_seed_stability"
)

STAGE22_MANIFEST_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
    / "stage22r_1c_development_model_input_manifest.json"
)

STAGE22_MEMBERSHIP_SUMMARY_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
    / "stage22r_1b1_membership_summary.json"
)

RANDOM_VALIDATION_PACKBITS_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
    / "random_validation.packbits"
)

STAGE22_CACHE_ROOT = Path(
    "/kaggle/working/"
    "stage22r_1c_70f_development_cache"
)

RUNTIME_ROOT = Path(
    "/kaggle/working/"
    "stage28_2a_runtime_cache/"
    "stage22_full"
)

X_PATH = (
    RUNTIME_ROOT
    / "development_features_float64.npy"
)

Y_PATH = (
    RUNTIME_ROOT
    / "development_binary_labels_uint8.npy"
)

DAY_PATH = (
    RUNTIME_ROOT
    / "development_day_ids_uint8.npy"
)

RANDOM_TRAIN_IDX_PATH = (
    RUNTIME_ROOT
    / "random_natural_train_global_idx_int32.npy"
)

RANDOM_VALIDATION_IDX_PATH = (
    RUNTIME_ROOT
    / "random_natural_validation_global_idx_int32.npy"
)

RUNTIME_RECEIPT_PATH = Path(
    "/kaggle/working/"
    "stage28_2a0_stage22_execution_matrix_receipt.json"
)


EXPECTED_ROWS = 14_412_403
EXPECTED_FEATURES = 70

EXPECTED_ATTACK = 1_972_299
EXPECTED_BENIGN = 12_440_104

EXPECTED_RANDOM_TRAIN_ROWS = 11_529_922
EXPECTED_RANDOM_TRAIN_ATTACK = 1_577_839
EXPECTED_RANDOM_TRAIN_BENIGN = 9_952_083

EXPECTED_RANDOM_VAL_ROWS = 2_882_481
EXPECTED_RANDOM_VAL_ATTACK = 394_460
EXPECTED_RANDOM_VAL_BENIGN = 2_488_021

EXPECTED_CHRONO_TRAIN_ROWS = 13_818_623
EXPECTED_CHRONO_TRAIN_ATTACK = 1_910_043
EXPECTED_CHRONO_TRAIN_BENIGN = 11_908_580

EXPECTED_CHRONO_VAL_ROWS = 593_780
EXPECTED_CHRONO_VAL_ATTACK = 62_256
EXPECTED_CHRONO_VAL_BENIGN = 531_524

EXPECTED_RANDOM_PACKBITS_SHA = (
    "8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad"
)

EXPECTED_STAGE22_CONTRACT_SHA = (
    "975113373af098bb12b274248dfa012a536c3e6a8270d7b37dd4f3d9bf79b9f2"
)

EXPECTED_NEW_FITS = 108


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return (
        run(
            ["git", *args]
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array_blocked(
    arr,
    block_rows=65_536,
):
    """
    Logical raw C-order content SHA256.
    Works on memmaps without materializing the entire array.
    """

    h = hashlib.sha256()

    if arr.ndim == 1:

        for start in range(
            0,
            arr.shape[0],
            block_rows,
        ):

            stop = min(
                start + block_rows,
                arr.shape[0],
            )

            block = np.ascontiguousarray(
                arr[start:stop]
            )

            h.update(
                block.view(np.uint8)
            )

    elif arr.ndim == 2:

        for start in range(
            0,
            arr.shape[0],
            block_rows,
        ):

            stop = min(
                start + block_rows,
                arr.shape[0],
            )

            block = np.ascontiguousarray(
                arr[
                    start:stop,
                    :
                ]
            )

            h.update(
                block.view(np.uint8)
            )

    else:
        raise RuntimeError(
            f"Unsupported array ndim={arr.ndim}"
        )

    return h.hexdigest()


def safe_reset_runtime_root():
    expected = Path(
        "/kaggle/working/"
        "stage28_2a_runtime_cache/"
        "stage22_full"
    ).resolve()

    actual = RUNTIME_ROOT.resolve()

    if actual != expected:
        raise RuntimeError(
            "Refusing to clean unexpected runtime path."
        )

    if actual.exists():
        shutil.rmtree(actual)

    actual.mkdir(
        parents=True,
        exist_ok=False,
    )


# =================================================================================================
# 2. EXACT DURABLE-PARENT GATE
# =================================================================================================

banner(
    "STAGE28-2A0 — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote = remote_line.split()[0]

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD:",
    EXPECTED_HEAD,
)

print(
    "Local HEAD   :",
    head,
)

print(
    "origin/main  :",
    origin,
)

print(
    "Remote main  :",
    remote,
)

print(
    "Branch       :",
    branch,
)

print(
    "Git clean    :",
    not bool(status),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_HEAD
):
    raise RuntimeError(
        "Stage28-1C durable parent mismatch."
    )

if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean:\n"
        + status
    )


if STAGE28_2A_DURABLE_ROOT.exists():
    raise RuntimeError(
        "Stage28-2A durable execution output already exists.\n"
        "Do not overwrite an existing execution checkpoint."
    )


print()
print(
    "[PASS] exact remotely durable Stage28-1C parent"
)


# =================================================================================================
# 3. VERIFY STAGE28-1C FREEZE
# =================================================================================================

banner(
    "STAGE28-1C MODEL-INPUT FREEZE GATE"
)

freeze_1c = read_json(
    require_file(
        STAGE28_1C_FREEZE
    )
)

contracts = read_json(
    require_file(
        STAGE28_INPUT_CONTRACTS
    )
)

require_file(
    STAGE28_BINDING
)


if (
    freeze_1c[
        "status"
    ]
    != "FROZEN_BEFORE_FIRST_STAGE28_MODEL_FIT"
):
    raise RuntimeError(
        "Stage28-1C freeze status invalid."
    )


execution = (
    freeze_1c[
        "execution_manifest"
    ]
)


if int(
    execution[
        "new_fit_budget"
    ]
) != EXPECTED_NEW_FITS:
    raise RuntimeError(
        "Stage28 fit budget != 108."
    )

if int(
    execution[
        "new_fits_consumed"
    ]
) != 0:
    raise RuntimeError(
        "FIT #1 was already consumed."
    )

if int(
    execution[
        "new_fits_remaining"
    ]
) != EXPECTED_NEW_FITS:
    raise RuntimeError(
        "Stage28 remaining fit ledger != 108."
    )

if (
    execution[
        "compute"
    ]
    != "CPU"
):
    raise RuntimeError(
        "Stage28 compute backend changed."
    )


stage22_contract = (
    freeze_1c[
        "model_input_contracts"
    ][
        "stage22"
    ]
)


print(
    "Stage22 contract SHA:",
    stage22_contract[
        "contract_sha256"
    ],
)

print(
    "Stage22 rows        :",
    f"{stage22_contract['rows']:,}",
)

print(
    "Stage22 files       :",
    stage22_contract[
        "files"
    ],
)

print(
    "NEW fits consumed   :",
    execution[
        "new_fits_consumed"
    ],
)

print(
    "NEW fits remaining  :",
    execution[
        "new_fits_remaining"
    ],
)


if (
    stage22_contract[
        "contract_sha256"
    ]
    != EXPECTED_STAGE22_CONTRACT_SHA
):
    raise RuntimeError(
        "Stage22 input-contract SHA mismatch."
    )

if int(
    stage22_contract[
        "rows"
    ]
) != EXPECTED_ROWS:
    raise RuntimeError(
        "Stage22 contract row count mismatch."
    )

if int(
    stage22_contract[
        "files"
    ]
) != 8:
    raise RuntimeError(
        "Stage22 contract file count mismatch."
    )


print()
print(
    "[PASS] Stage28-1C Stage22 contract exact"
)

print(
    "[PASS] FIT ledger still 0 / 108"
)


# =================================================================================================
# 4. LOAD FROZEN STAGE22 CONTRACTS
# =================================================================================================

banner(
    "LOAD FROZEN STAGE22 MODEL-INPUT / MEMBERSHIP CONTRACTS"
)

manifest = read_json(
    require_file(
        STAGE22_MANIFEST_PATH
    )
)

membership = read_json(
    require_file(
        STAGE22_MEMBERSHIP_SUMMARY_PATH
    )
)


feature_order = list(
    manifest[
        "row_schema"
    ][
        "predictor_columns"
    ]
)


if len(
    feature_order
) != EXPECTED_FEATURES:
    raise RuntimeError(
        "Stage22 feature count != 70."
    )


if (
    membership[
        "clean_development"
    ][
        "canonical_order"
    ]
    !=
    "(day_id ASC, original_zero_based_row_index ASC) after frozen K79 exclusions"
):
    raise RuntimeError(
        "Stage22 canonical row order changed."
    )


if (
    membership[
        "membership_derivation"
    ][
        "RANDOM_NATURAL_train"
    ]
    != "logical complement of random_validation bitset"
):
    raise RuntimeError(
        "Random train derivation changed."
    )

if (
    membership[
        "membership_derivation"
    ][
        "RANDOM_NATURAL_validation"
    ]
    != "random_validation bitset"
):
    raise RuntimeError(
        "Random validation derivation changed."
    )

if (
    membership[
        "membership_derivation"
    ][
        "CHRONOLOGICAL_NATURAL_train"
    ]
    != "day_id 0..6"
):
    raise RuntimeError(
        "Chronological train derivation changed."
    )

if (
    membership[
        "membership_derivation"
    ][
        "CHRONOLOGICAL_NATURAL_validation"
    ]
    != "day_id 7"
):
    raise RuntimeError(
        "Chronological validation derivation changed."
    )


print(
    "Rows        :",
    f"{EXPECTED_ROWS:,}",
)

print(
    "Features    :",
    EXPECTED_FEATURES,
)

print(
    "Dtype       :",
    manifest[
        "feature_configuration"
    ][
        "dtype"
    ],
)

print(
    "Feature[0]  :",
    feature_order[0],
)

print(
    "Feature[-1] :",
    feature_order[-1],
)

print()
print(
    "[PASS] Stage22 model-input + membership semantics exact"
)


# =================================================================================================
# 5. RE-VERIFY 8 BYTE-EXACT CACHE PARQUETS
# =================================================================================================

banner(
    "8/8 BYTE-EXACT STAGE22 CACHE GATE"
)

cache_records = sorted(
    manifest[
        "cache"
    ][
        "files"
    ],
    key=lambda x: int(
        x[
            "day_id"
        ]
    ),
)


if len(
    cache_records
) != 8:
    raise RuntimeError(
        "Expected exactly 8 Stage22 cache files."
    )


expected_cursor = 0


for expected_day, record in enumerate(
    cache_records
):

    day_id = int(
        record[
            "day_id"
        ]
    )

    if day_id != expected_day:
        raise RuntimeError(
            "Stage22 cache day IDs are not 0..7."
        )


    path = (
        STAGE22_CACHE_ROOT
        / record[
            "cache_file"
        ]
    )

    require_file(path)


    actual_bytes = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(path)


    if actual_bytes != int(
        record[
            "bytes"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: byte-size mismatch."
        )

    if actual_sha != record[
        "sha256"
    ]:
        raise RuntimeError(
            f"{path.name}: SHA256 mismatch."
        )


    pf = pq.ParquetFile(path)

    parquet_rows = int(
        pf.metadata.num_rows
    )


    if parquet_rows != int(
        record[
            "rows"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: Parquet row count mismatch."
        )


    print(
        f"[PASS] day {day_id} "
        f"{path.name}"
    )

    print(
        "       rows  :",
        f"{parquet_rows:,}",
    )

    print(
        "       SHA256:",
        actual_sha,
    )


    expected_cursor += parquet_rows


if expected_cursor != EXPECTED_ROWS:
    raise RuntimeError(
        "Stage22 cache total rows mismatch."
    )


print()
print(
    "[PASS] all eight historical cache files byte-exact"
)


# =================================================================================================
# 6. DISK SAFETY GATE
# =================================================================================================

banner(
    "RUNTIME STORAGE GATE"
)

disk = shutil.disk_usage(
    "/kaggle/working"
)

x_payload = (
    EXPECTED_ROWS
    * EXPECTED_FEATURES
    * np.dtype(
        np.float64
    ).itemsize
)

metadata_payload = (
    EXPECTED_ROWS
    * (
        np.dtype(
            np.uint8
        ).itemsize
        * 2
    )
)

idx_payload_approx = (
    (
        EXPECTED_RANDOM_TRAIN_ROWS
        +
        EXPECTED_RANDOM_VAL_ROWS
    )
    * np.dtype(
        np.int32
    ).itemsize
)

required = (
    x_payload
    +
    metadata_payload
    +
    idx_payload_approx
)


print(
    "Free space             :",
    f"{disk.free / (1024**3):.3f} GiB",
)

print(
    "Feature matrix payload :",
    f"{x_payload / (1024**3):.3f} GiB",
)

print(
    "Metadata + indices     :",
    f"{(metadata_payload + idx_payload_approx) / (1024**3):.3f} GiB",
)


# Keep > 1.5 GiB free after this runtime cache.
if disk.free < (
    required
    + int(
        1.5
        * 1024**3
    )
):
    raise RuntimeError(
        "Insufficient /kaggle/working disk space "
        "for the Stage22 execution cache plus safety margin."
    )


print()
print(
    "[PASS] sufficient runtime disk space"
)


# =================================================================================================
# 7. CLEAN / ALLOCATE DISPOSABLE STAGE28-2A RUNTIME CACHE
# =================================================================================================

banner(
    "ALLOCATE STAGE22 EXECUTION MATRICES"
)

safe_reset_runtime_root()


X = np.lib.format.open_memmap(
    X_PATH,
    mode="w+",
    dtype=np.float64,
    shape=(
        EXPECTED_ROWS,
        EXPECTED_FEATURES,
    ),
)

y = np.lib.format.open_memmap(
    Y_PATH,
    mode="w+",
    dtype=np.uint8,
    shape=(
        EXPECTED_ROWS,
    ),
)

day_ids = np.lib.format.open_memmap(
    DAY_PATH,
    mode="w+",
    dtype=np.uint8,
    shape=(
        EXPECTED_ROWS,
    ),
)


print(
    "X:",
    X_PATH,
)

print(
    "  shape:",
    X.shape,
)

print(
    "  dtype:",
    X.dtype,
)

print()

print(
    "y:",
    Y_PATH,
)

print()

print(
    "day:",
    DAY_PATH,
)


# =================================================================================================
# 8. MATERIALIZE CANONICAL STAGE22 DEVELOPMENT MATRIX
# =================================================================================================

banner(
    "MATERIALIZE CANONICAL STAGE22 DEVELOPMENT POPULATION"
)

required_columns = [
    "clean_position",
    "day_id",
    "binary_label",
    *feature_order,
]


cursor = 0

total_attack = 0
total_benign = 0
total_nan = 0
total_inf = 0

day_audit = []


for record in cache_records:

    day_id = int(
        record[
            "day_id"
        ]
    )

    path = (
        STAGE22_CACHE_ROOT
        / record[
            "cache_file"
        ]
    )

    expected_day_rows = int(
        record[
            "rows"
        ]
    )

    expected_day_attack = int(
        record[
            "attack"
        ]
    )

    expected_day_benign = int(
        record[
            "benign"
        ]
    )


    pf = pq.ParquetFile(path)


    if list(
        pf.schema_arrow.names
    ) != (
        [
            "clean_position",
            "day_id",
            "original_row_index",
            "binary_label",
            *feature_order,
        ]
    ):
        raise RuntimeError(
            f"{path.name}: ordered 74-column schema changed."
        )


    day_start = cursor

    day_attack = 0
    day_benign = 0
    day_nan = 0
    day_inf = 0


    print()
    print(
        "-" * 120
    )

    print(
        f"Day {day_id} — {path.name}"
    )

    print(
        "-" * 120
    )


    for batch in pf.iter_batches(
        batch_size=65_536,
        columns=required_columns,
        use_threads=True,
    ):

        n = int(
            batch.num_rows
        )

        if n <= 0:
            continue


        clean_pos = (
            batch
            .column(0)
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.int64,
                copy=False,
            )
        )

        block_day = (
            batch
            .column(1)
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.uint8,
                copy=False,
            )
        )

        block_y = (
            batch
            .column(2)
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.uint8,
                copy=False,
            )
        )


        expected_positions = np.arange(
            cursor,
            cursor + n,
            dtype=np.int64,
        )


        if not np.array_equal(
            clean_pos,
            expected_positions,
        ):
            raise RuntimeError(
                f"{path.name}: clean_position "
                f"sequence mismatch at global cursor {cursor:,}."
            )


        if not np.all(
            block_day
            == day_id
        ):
            raise RuntimeError(
                f"{path.name}: day_id mismatch."
            )


        if not np.all(
            (
                block_y == 0
            )
            |
            (
                block_y == 1
            )
        ):
            raise RuntimeError(
                f"{path.name}: binary_label outside {{0,1}}."
            )


        # Construct exact float64 feature block from the
        # already-frozen Stage22 Parquet cache.
        feature_arrays = []

        for j in range(
            EXPECTED_FEATURES
        ):

            arr = (
                batch
                .column(
                    3 + j
                )
                .to_numpy(
                    zero_copy_only=False
                )
                .astype(
                    np.float64,
                    copy=False,
                )
            )

            feature_arrays.append(
                arr
            )


        X_block = np.column_stack(
            feature_arrays
        )


        if X_block.shape != (
            n,
            EXPECTED_FEATURES,
        ):
            raise RuntimeError(
                "Unexpected Stage22 feature-block shape."
            )


        inf_count = int(
            np.isinf(
                X_block
            ).sum()
        )

        nan_count = int(
            np.isnan(
                X_block
            ).sum()
        )


        # Stage22R-1C cache already froze inf -> NaN.
        if inf_count != 0:
            raise RuntimeError(
                f"{path.name}: infinity survived "
                "the historical Stage22 cache."
            )


        stop = (
            cursor
            + n
        )


        X[
            cursor:stop,
            :
        ] = X_block

        y[
            cursor:stop
        ] = block_y

        day_ids[
            cursor:stop
        ] = block_day


        attack = int(
            block_y.sum()
        )

        benign = (
            n
            - attack
        )


        day_attack += attack
        day_benign += benign

        day_nan += nan_count
        day_inf += inf_count

        cursor = stop


        del clean_pos
        del block_day
        del block_y
        del expected_positions
        del feature_arrays
        del X_block

        gc.collect()


    day_rows = (
        cursor
        - day_start
    )


    if day_rows != expected_day_rows:
        raise RuntimeError(
            f"{path.name}: materialized day row count mismatch."
        )

    if day_attack != expected_day_attack:
        raise RuntimeError(
            f"{path.name}: attack count mismatch."
        )

    if day_benign != expected_day_benign:
        raise RuntimeError(
            f"{path.name}: benign count mismatch."
        )


    print(
        "Rows   :",
        f"{day_rows:,}",
    )

    print(
        "Attack :",
        f"{day_attack:,}",
    )

    print(
        "Benign :",
        f"{day_benign:,}",
    )

    print(
        "NaNs   :",
        f"{day_nan:,}",
    )

    print(
        "[PASS] canonical day materialized"
    )


    total_attack += day_attack
    total_benign += day_benign
    total_nan += day_nan
    total_inf += day_inf


    day_audit.append(
        {
            "day_id":
                day_id,

            "cache_file":
                path.name,

            "global_start":
                day_start,

            "global_stop_exclusive":
                cursor,

            "rows":
                day_rows,

            "attack":
                day_attack,

            "benign":
                day_benign,

            "nan_cells":
                day_nan,

            "inf_cells":
                day_inf,
        }
    )


if cursor != EXPECTED_ROWS:
    raise RuntimeError(
        "Canonical Stage22 materialization "
        "did not reach 14,412,403 rows."
    )

if total_attack != EXPECTED_ATTACK:
    raise RuntimeError(
        "Global Stage22 attack count mismatch."
    )

if total_benign != EXPECTED_BENIGN:
    raise RuntimeError(
        "Global Stage22 benign count mismatch."
    )


expected_output_nan = int(
    manifest[
        "nonfinite_conversion"
    ][
        "output_nan_values"
    ]
)


if total_nan != expected_output_nan:
    raise RuntimeError(
        "Global Stage22 NaN count mismatch.\n"
        f"expected={expected_output_nan:,}\n"
        f"actual={total_nan:,}"
    )

if total_inf != 0:
    raise RuntimeError(
        "Global Stage22 execution matrix contains infinity."
    )


X.flush()
y.flush()
day_ids.flush()


print()
print(
    "[PASS] complete canonical development population materialized"
)

print(
    "Rows   :",
    f"{cursor:,}",
)

print(
    "Attack :",
    f"{total_attack:,}",
)

print(
    "Benign :",
    f"{total_benign:,}",
)

print(
    "NaNs   :",
    f"{total_nan:,}",
)

print(
    "Inf    :",
    total_inf,
)


# =================================================================================================
# 9. RELOAD MEMMAPS
# =================================================================================================

banner(
    "RELOAD EXECUTION MEMMAPS"
)

del X
del y
del day_ids

gc.collect()


X = np.load(
    X_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

y = np.load(
    Y_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

day_ids = np.load(
    DAY_PATH,
    mmap_mode="r",
    allow_pickle=False,
)


if X.shape != (
    EXPECTED_ROWS,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        "Reloaded Stage22 X shape mismatch."
    )

if X.dtype != np.float64:
    raise RuntimeError(
        "Reloaded Stage22 X dtype != float64."
    )

if y.shape != (
    EXPECTED_ROWS,
):
    raise RuntimeError(
        "Reloaded Stage22 y shape mismatch."
    )

if y.dtype != np.uint8:
    raise RuntimeError(
        "Reloaded Stage22 y dtype != uint8."
    )


print(
    "[PASS] Stage22 execution memmaps reload exactly"
)


# =================================================================================================
# 10. RANDOM_NATURAL MEMBERSHIP
# =================================================================================================

banner(
    "RANDOM_NATURAL — EXACT INHERITED MEMBERSHIP"
)

require_file(
    RANDOM_VALIDATION_PACKBITS_PATH
)

actual_pack_sha = sha256_file(
    RANDOM_VALIDATION_PACKBITS_PATH
)

print(
    "Expected packbits SHA:",
    EXPECTED_RANDOM_PACKBITS_SHA,
)

print(
    "Actual packbits SHA  :",
    actual_pack_sha,
)


if (
    actual_pack_sha
    != EXPECTED_RANDOM_PACKBITS_SHA
):
    raise RuntimeError(
        "Frozen random-validation packbits SHA mismatch."
    )


packed = np.fromfile(
    RANDOM_VALIDATION_PACKBITS_PATH,
    dtype=np.uint8,
)

random_val_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:EXPECTED_ROWS].astype(
    np.bool_,
    copy=False,
)


if int(
    random_val_mask.sum()
) != EXPECTED_RANDOM_VAL_ROWS:
    raise RuntimeError(
        "Random validation population mismatch."
    )


random_validation_idx = np.flatnonzero(
    random_val_mask
).astype(
    np.int32,
    copy=False,
)

random_train_idx = np.flatnonzero(
    ~random_val_mask
).astype(
    np.int32,
    copy=False,
)


if len(
    random_train_idx
) != EXPECTED_RANDOM_TRAIN_ROWS:
    raise RuntimeError(
        "Random train row count mismatch."
    )

if len(
    random_validation_idx
) != EXPECTED_RANDOM_VAL_ROWS:
    raise RuntimeError(
        "Random validation row count mismatch."
    )


random_train_attack = int(
    np.asarray(
        y[
            random_train_idx
        ]
    ).sum()
)

random_val_attack = int(
    np.asarray(
        y[
            random_validation_idx
        ]
    ).sum()
)


random_train_benign = (
    len(
        random_train_idx
    )
    -
    random_train_attack
)

random_val_benign = (
    len(
        random_validation_idx
    )
    -
    random_val_attack
)


if (
    random_train_attack
    != EXPECTED_RANDOM_TRAIN_ATTACK
    or random_train_benign
    != EXPECTED_RANDOM_TRAIN_BENIGN
):
    raise RuntimeError(
        "Random train class counts mismatch."
    )

if (
    random_val_attack
    != EXPECTED_RANDOM_VAL_ATTACK
    or random_val_benign
    != EXPECTED_RANDOM_VAL_BENIGN
):
    raise RuntimeError(
        "Random validation class counts mismatch."
    )


np.save(
    RANDOM_TRAIN_IDX_PATH,
    random_train_idx,
    allow_pickle=False,
)

np.save(
    RANDOM_VALIDATION_IDX_PATH,
    random_validation_idx,
    allow_pickle=False,
)


random_train_idx_sha = (
    sha256_array_blocked(
        random_train_idx
    )
)

random_val_idx_sha = (
    sha256_array_blocked(
        random_validation_idx
    )
)


print(
    "TRAIN rows   :",
    f"{len(random_train_idx):,}",
)

print(
    "  attack     :",
    f"{random_train_attack:,}",
)

print(
    "  benign     :",
    f"{random_train_benign:,}",
)

print(
    "  index SHA  :",
    random_train_idx_sha,
)

print()

print(
    "VALID rows   :",
    f"{len(random_validation_idx):,}",
)

print(
    "  attack     :",
    f"{random_val_attack:,}",
)

print(
    "  benign     :",
    f"{random_val_benign:,}",
)

print(
    "  index SHA  :",
    random_val_idx_sha,
)

print()

print(
    "[PASS] RANDOM_NATURAL inherited membership exact"
)


# =================================================================================================
# 11. CHRONOLOGICAL_NATURAL MEMBERSHIP
# =================================================================================================

banner(
    "CHRONOLOGICAL_NATURAL — EXACT INHERITED MEMBERSHIP"
)

chrono_train_stop = (
    EXPECTED_CHRONO_TRAIN_ROWS
)


if not np.all(
    day_ids[
        :chrono_train_stop
    ]
    <= 6
):
    raise RuntimeError(
        "Chronological train contains day 7."
    )

if not np.all(
    day_ids[
        chrono_train_stop:
    ]
    == 7
):
    raise RuntimeError(
        "Chronological validation is not exactly day 7."
    )


chrono_train_y = np.asarray(
    y[
        :chrono_train_stop
    ]
)

chrono_val_y = np.asarray(
    y[
        chrono_train_stop:
    ]
)


chrono_train_attack = int(
    chrono_train_y.sum()
)

chrono_val_attack = int(
    chrono_val_y.sum()
)


chrono_train_benign = (
    len(
        chrono_train_y
    )
    -
    chrono_train_attack
)

chrono_val_benign = (
    len(
        chrono_val_y
    )
    -
    chrono_val_attack
)


if len(
    chrono_train_y
) != EXPECTED_CHRONO_TRAIN_ROWS:
    raise RuntimeError(
        "Chronological train rows mismatch."
    )

if len(
    chrono_val_y
) != EXPECTED_CHRONO_VAL_ROWS:
    raise RuntimeError(
        "Chronological validation rows mismatch."
    )


if (
    chrono_train_attack
    != EXPECTED_CHRONO_TRAIN_ATTACK
    or chrono_train_benign
    != EXPECTED_CHRONO_TRAIN_BENIGN
):
    raise RuntimeError(
        "Chronological train class counts mismatch."
    )

if (
    chrono_val_attack
    != EXPECTED_CHRONO_VAL_ATTACK
    or chrono_val_benign
    != EXPECTED_CHRONO_VAL_BENIGN
):
    raise RuntimeError(
        "Chronological validation class counts mismatch."
    )


print(
    "TRAIN rows:",
    f"{len(chrono_train_y):,}",
)

print(
    "  attack  :",
    f"{chrono_train_attack:,}",
)

print(
    "  benign  :",
    f"{chrono_train_benign:,}",
)

print()

print(
    "VALID rows:",
    f"{len(chrono_val_y):,}",
)

print(
    "  attack  :",
    f"{chrono_val_attack:,}",
)

print(
    "  benign  :",
    f"{chrono_val_benign:,}",
)

print()

print(
    "[PASS] CHRONOLOGICAL_NATURAL inherited membership exact"
)


# =================================================================================================
# 12. EXECUTION-CACHE CONTENT IDENTITIES
# =================================================================================================

banner(
    "STAGE22 RUNTIME EXECUTION-CACHE IDENTITIES"
)

print(
    "Hashing binary-label vector..."
)

y_sha = sha256_array_blocked(
    y
)

print(
    "Label content SHA256:",
    y_sha,
)

print()

print(
    "Hashing day-id vector..."
)

day_sha = sha256_array_blocked(
    day_ids
)

print(
    "Day content SHA256:",
    day_sha,
)

print()

print(
    "The 8+ GiB feature matrix is NOT re-hashed here "
    "because its scientific identity is already the "
    "eight byte-exact historical Stage22 Parquet hashes."
)

print(
    "The runtime .npy matrix is only an execution layout."
)


# =================================================================================================
# 13. WRITE RUNTIME-ONLY EXECUTION RECEIPT
# =================================================================================================

runtime_receipt = {

    "stage":
        "Stage28-2A0",

    "type":
        "RUNTIME_ONLY_STAGE22_FULL_EXECUTION_MATRIX_AND_MEMBERSHIP_MATERIALIZATION",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "durable_scientific_parent":
        EXPECTED_HEAD,

    "scientific_contract": {
        "stage22_contract_sha256":
            EXPECTED_STAGE22_CONTRACT_SHA,

        "source":
            (
                "EIGHT_BYTE_EXACT_HISTORICAL_"
                "STAGE22R_1C_PARQUET_CACHES"
            ),

        "feature_count":
            EXPECTED_FEATURES,

        "feature_dtype":
            "float64",

        "canonical_order":
            (
                "(day_id ASC, original_zero_based_row_index ASC) "
                "after frozen K79 exclusions"
            ),
    },

    "runtime_matrix": {
        "features_path":
            str(
                X_PATH
            ),

        "labels_path":
            str(
                Y_PATH
            ),

        "day_ids_path":
            str(
                DAY_PATH
            ),

        "shape": [
            EXPECTED_ROWS,
            EXPECTED_FEATURES,
        ],

        "dtype":
            "float64",

        "rows":
            EXPECTED_ROWS,

        "attack":
            total_attack,

        "benign":
            total_benign,

        "nan_cells":
            total_nan,

        "inf_cells":
            total_inf,

        "binary_labels_content_sha256":
            y_sha,

        "day_ids_content_sha256":
            day_sha,

        "runtime_files_committed_to_git":
            False,
    },

    "day_audit":
        day_audit,

    "random_natural": {
        "validation_packbits_path":
            str(
                RANDOM_VALIDATION_PACKBITS_PATH.relative_to(
                    REPO
                )
            ),

        "validation_packbits_sha256":
            actual_pack_sha,

        "train_index_path":
            str(
                RANDOM_TRAIN_IDX_PATH
            ),

        "validation_index_path":
            str(
                RANDOM_VALIDATION_IDX_PATH
            ),

        "train_index_content_sha256":
            random_train_idx_sha,

        "validation_index_content_sha256":
            random_val_idx_sha,

        "train": {
            "rows":
                len(
                    random_train_idx
                ),

            "attack":
                random_train_attack,

            "benign":
                random_train_benign,
        },

        "validation": {
            "rows":
                len(
                    random_validation_idx
                ),

            "attack":
                random_val_attack,

            "benign":
                random_val_benign,
        },
    },

    "chronological_natural": {
        "train_slice": [
            0,
            EXPECTED_CHRONO_TRAIN_ROWS,
        ],

        "validation_slice": [
            EXPECTED_CHRONO_TRAIN_ROWS,
            EXPECTED_ROWS,
        ],

        "train": {
            "rows":
                EXPECTED_CHRONO_TRAIN_ROWS,

            "attack":
                chrono_train_attack,

            "benign":
                chrono_train_benign,
        },

        "validation": {
            "rows":
                EXPECTED_CHRONO_VAL_ROWS,

            "attack":
                chrono_val_attack,

            "benign":
                chrono_val_benign,
        },
    },

    "scientific_operations": {
        "stage22_development_predictor_rows_read_for_execution":
            EXPECTED_ROWS,

        "model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "final_holdout_openings":
            0,

        "new_scientific_target_openings":
            0,
    },

    "fit_ledger": {
        "new_authorized":
            EXPECTED_NEW_FITS,

        "new_consumed":
            0,

        "new_remaining":
            EXPECTED_NEW_FITS,
    },

    "status":
        "READY_FOR_STAGE28_FIT_001",
}


RUNTIME_RECEIPT_PATH.write_text(
    json.dumps(
        runtime_receipt,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )
    + "\n",
    encoding="utf-8",
)


print()
print(
    "Runtime receipt:"
)

print(
    " ",
    RUNTIME_RECEIPT_PATH,
)


# =================================================================================================
# 14. FINAL GIT CLEANLINESS GATE
# =================================================================================================

banner(
    "FINAL GIT CLEANLINESS GATE"
)

final_head = git(
    "rev-parse",
    "HEAD",
)

final_origin = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head,
)

print(
    "origin/main:",
    final_origin,
)

print(
    "Git clean  :",
    not bool(
        final_status
    ),
)


if (
    final_head
    != EXPECTED_HEAD
    or final_origin
    != EXPECTED_HEAD
):
    raise RuntimeError(
        "Git parent changed during Stage28-2A0."
    )

if final_status:
    raise RuntimeError(
        "Stage28-2A0 unexpectedly changed Git:\n"
        + final_status
    )


print()
print(
    "[PASS] durable repository untouched"
)


# =================================================================================================
# 15. FINAL
# =================================================================================================

banner(
    "STAGE28-2A0 — STAGE22 EXECUTION MATRIX READY"
)

print(
    "Durable parent:"
)

print(
    " ",
    EXPECTED_HEAD,
)

print()

print(
    "Stage22 execution population:"
)

print(
    "  rows       =",
    f"{EXPECTED_ROWS:,}",
)

print(
    "  features   =",
    EXPECTED_FEATURES,
)

print(
    "  dtype      = float64"
)

print(
    "  attack     =",
    f"{total_attack:,}",
)

print(
    "  benign     =",
    f"{total_benign:,}",
)

print(
    "  NaN cells  =",
    f"{total_nan:,}",
)

print(
    "  inf cells  =",
    total_inf,
)

print()

print(
    "RANDOM_NATURAL:"
)

print(
    "  train      =",
    f"{len(random_train_idx):,}",
)

print(
    "  validation =",
    f"{len(random_validation_idx):,}",
)

print()

print(
    "CHRONOLOGICAL_NATURAL:"
)

print(
    "  train      =",
    f"{EXPECTED_CHRONO_TRAIN_ROWS:,}",
)

print(
    "  validation =",
    f"{EXPECTED_CHRONO_VAL_ROWS:,}",
)

print()

print(
    "Scientific operations:"
)

print(
    "  MODEL_FITS            = 0"
)

print(
    "  MODEL_INFERENCE       = 0"
)

print(
    "  THRESHOLD_SELECTION   = 0"
)

print(
    "  FINAL_HOLDOUT_OPENING = 0"
)

print()

print(
    "Fit ledger:"
)

print(
    "  authorized = 108"
)

print(
    "  consumed   = 0"
)

print(
    "  remaining  = 108"
)

print()

print(
    "STATUS:"
)

print(
    "  READY FOR FIT #1."
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage28-2A1 — RANDOM_NATURAL / seed42."
)

print(
    "  FIT #1: CPU XGBoost XGB_11 seed42."
)

print(
    "  LightGBM LGBM_11 seed42 will be REUSED "
    "from its exact historical CPU artifact."
)

print(
    "  Then validation probabilities and the "
    "three frozen operating thresholds will be produced."
)

print()

print(
    "  The shared final holdout remains untouched."
)

print()
print(SEP)


STAGE28-2A0 — EXACT DURABLE-PARENT GATE

Expected HEAD: 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Local HEAD   : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
origin/main  : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Remote main  : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Branch       : main
Git clean    : True

[PASS] exact remotely durable Stage28-1C parent

STAGE28-1C MODEL-INPUT FREEZE GATE

Stage22 contract SHA: 975113373af098bb12b274248dfa012a536c3e6a8270d7b37dd4f3d9bf79b9f2
Stage22 rows        : 14,412,403
Stage22 files       : 8
NEW fits consumed   : 0
NEW fits remaining  : 108

[PASS] Stage28-1C Stage22 contract exact
[PASS] FIT ledger still 0 / 108

LOAD FROZEN STAGE22 MODEL-INPUT / MEMBERSHIP CONTRACTS

Rows        : 14,412,403
Features    : 70
Dtype       : float64
Feature[0]  : Dst Port
Feature[-1] : Idle Min

[PASS] Stage22 model-input + membership semantics exact

8/8 BYTE-EXACT STAGE22 CACHE GATE

[PASS] day 0 day_00_02-14-2018.parquet
       rows  : 822,947
       SHA256: a54

In [1]:
# =================================================================================================
# STAGE28-RUNTIME-R2 — RECOVER AFTER KAGGLE RESET BEFORE STAGE28-2A1
#
# Last durable scientific commit:
#   738e3ee29d098d4132828aeaaacfa12a8cfd7b52
#
# LAST COMPLETED WORK BEFORE RESET
# --------------------------------
# Stage28-2A0 completed successfully, but it was RUNTIME ONLY.
#
# Therefore the reset lost:
#   - /kaggle/working/stage22r_1c_70f_development_cache
#   - /kaggle/working/stage28_2a_runtime_cache/stage22_full/*
#   - /kaggle/working/stage28_2a0_stage22_execution_matrix_receipt.json
#
# It DID NOT lose:
#   - any durable scientific state
#   - Stage28-1C freeze
#   - execution manifest
#   - fit budget
#
# FIT #1 WAS NEVER RUN.
#
# This recovery cell:
#   1. recovers exact Git repo at 738e3ee...
#   2. verifies Stage28-1C is the durable pre-fit state
#   3. verifies fit ledger is still 0 / 108
#   4. restores the byte-exact Stage22R historical 70F cache
#      from the frozen private Kaggle checkpoint
#   5. verifies all 8 SHA256 / byte identities
#
# ZERO:
#   predictor values read
#   model fits
#   model inference
#   threshold selection
#   final holdout opening
#   Git modifications
#
# NEXT AFTER SUCCESS:
#   rerun Stage28-2A0 to reconstruct the disposable execution
#   matrix + memberships, then proceed to Stage28-2A1 / FIT #1.
# =================================================================================================

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

EXPECTED_HEAD = (
    "738e3ee29d098d4132828aeaaacfa12a8cfd7b52"
)

REPO_URL = (
    "https://github.com/themubasshir/"
    "ids2018-validation-safe-ablation.git"
)

REPO = Path(
    "/kaggle/working/"
    "ids2018-validation-safe-ablation"
)

PRIVATE_DATASET_ID = (
    "jmmubasshirrahman/"
    "stage22r-1c-70f-cache-3cd41c5f"
)

STAGE22_RUNTIME_CACHE = Path(
    "/kaggle/working/"
    "stage22r_1c_70f_development_cache"
)

STAGE28_2A_RUNTIME_ROOT = Path(
    "/kaggle/working/"
    "stage28_2a_runtime_cache"
)

STAGE28_2A0_RUNTIME_RECEIPT = Path(
    "/kaggle/working/"
    "stage28_2a0_stage22_execution_matrix_receipt.json"
)

STAGE28_ROOT_REL = Path(
    "results/"
    "stage28_stability_novelty_control"
)

STAGE28_1C_REL = (
    STAGE28_ROOT_REL
    / "stage28_1c_model_input_identity_freeze"
)

STAGE28_1C_FREEZE_REL = (
    STAGE28_1C_REL
    / "stage28_1c_freeze_record.json"
)

STAGE28_INPUT_CONTRACTS_REL = (
    STAGE28_1C_REL
    / "stage28_model_input_contracts.json"
)

STAGE28_BINDING_REL = (
    STAGE28_1C_REL
    / "stage28_component_input_binding.csv"
)

STAGE28_1B_MANIFEST_REL = (
    STAGE28_ROOT_REL
    / "stage28_1b_random_loao_membership_and_execution_lock"
    / "stage28_component_execution_manifest.csv"
)

STAGE22_MANIFEST_REL = Path(
    "results/"
    "stage22r_protocol_recovery/"
    "stage22r_1c_development_model_inputs/"
    "stage22r_1c_development_model_input_manifest.json"
)

STAGE22_REMOTE_RECEIPT_REL = Path(
    "results/"
    "stage22r_protocol_recovery/"
    "stage22r_1c_remote_cache_checkpoint/"
    "stage22r_1c_private_kaggle_cache_receipt.json"
)

EXPECTED_STAGE22_ROWS = 14_412_403
EXPECTED_STAGE22_BYTES = 1_541_208_291

EXPECTED_COMPONENTS = 120
EXPECTED_EVALUATION_CELLS = 110
EXPECTED_NEW_FITS = 108
EXPECTED_REUSED = 12

EXPECTED_STAGE22_CONTRACT_SHA = (
    "975113373af098bb12b274248dfa012a536c3e6a8270d7b37dd4f3d9bf79b9f2"
)


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def command(
    args,
    *,
    cwd=None,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in args],
        cwd=(
            str(cwd)
            if cwd is not None
            else None
        ),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, args))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return (
        command(
            ["git", *args],
            cwd=REPO,
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def recover_github_token():

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]

    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()

        for label in labels:

            try:
                value = client.get_secret(
                    label
                )

            except Exception:
                value = None

            if (
                isinstance(value, str)
                and value.strip()
            ):

                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass


    for label in labels:

        value = os.environ.get(
            label
        )

        if (
            isinstance(value, str)
            and value.strip()
        ):

            return (
                value.strip(),
                f"environment:{label}",
            )


    return (
        None,
        None,
    )


# =================================================================================================
# 2. GITHUB SECRET RECOVERY
# =================================================================================================

banner(
    "GITHUB SECRET RECOVERY"
)

github_token, token_source = (
    recover_github_token()
)

if github_token:

    print(
        "[PASS] GitHub credential recovered from",
        token_source,
    )

    print(
        "[PASS] token value intentionally not displayed"
    )

else:

    print(
        "[NOTICE] GitHub credential was not recovered."
    )

    print(
        "         This does not block read-only recovery,"
    )

    print(
        "         but it must be available before the next"
    )

    print(
        "         scientific checkpoint is pushed."
    )


# =================================================================================================
# 3. REPOSITORY RECOVERY
# =================================================================================================

banner(
    "REPOSITORY RECOVERY"
)

if not REPO.exists():

    print(
        "Fresh Kaggle runtime — cloning repository."
    )

    clone = command(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO),
        ]
    )

    if clone.stdout.strip():
        print(
            clone.stdout.strip()
        )

else:

    if not (
        REPO
        / ".git"
    ).exists():

        raise RuntimeError(
            "Repository path exists but is not a Git repository:\n"
            f"{REPO}"
        )

    print(
        "Existing repository found."
    )

    print(
        "Fetching origin/main..."
    )

    command(
        [
            "git",
            "fetch",
            "origin",
            "main",
        ],
        cwd=REPO,
    )


# =================================================================================================
# 4. EXACT DURABLE SCIENTIFIC PARENT GATE
# =================================================================================================

banner(
    "STAGE28 DURABLE SCIENTIFIC-PARENT GATE"
)

local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote_head = (
    remote_line.split()[0]
)

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)


print(
    "Expected HEAD:",
    EXPECTED_HEAD,
)

print(
    "Local HEAD   :",
    local_head,
)

print(
    "origin/main  :",
    origin_head,
)

print(
    "Remote main  :",
    remote_head,
)

print(
    "Branch       :",
    branch,
)

print(
    "Subject      :",
    subject,
)

print(
    "Git clean    :",
    not bool(status),
)


if (
    origin_head
    != EXPECTED_HEAD
    or remote_head
    != EXPECTED_HEAD
):
    raise RuntimeError(
        "Remote scientific state is not the expected "
        "Stage28-1C commit.\n\n"
        "STOP instead of silently advancing or rewinding."
    )


if local_head != EXPECTED_HEAD:

    if status:
        raise RuntimeError(
            "Local repository is not at the expected commit "
            "and contains changes. Refusing automatic reset."
        )

    print()
    print(
        "Local clone is clean but at another commit."
    )

    print(
        "Resetting read-only runtime checkout to exact "
        "origin/main..."
    )

    command(
        [
            "git",
            "checkout",
            "main",
        ],
        cwd=REPO,
    )

    command(
        [
            "git",
            "reset",
            "--hard",
            EXPECTED_HEAD,
        ],
        cwd=REPO,
    )


local_head = git(
    "rev-parse",
    "HEAD",
)

status = git(
    "status",
    "--porcelain",
)


if local_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Unable to recover exact Stage28-1C HEAD."
    )

if status:
    raise RuntimeError(
        "Repository is not clean after recovery:\n"
        + status
    )


print()
print(
    "[PASS] exact Stage28-1C durable state recovered"
)


# =================================================================================================
# 5. CRITICAL DURABLE ARTIFACT GATE
# =================================================================================================

banner(
    "CRITICAL STAGE28 DURABLE ARTIFACT GATE"
)

critical_paths = [
    STAGE28_1C_FREEZE_REL,
    STAGE28_INPUT_CONTRACTS_REL,
    STAGE28_BINDING_REL,
    STAGE28_1B_MANIFEST_REL,
    STAGE22_MANIFEST_REL,
    STAGE22_REMOTE_RECEIPT_REL,
]


for rel in critical_paths:

    require_file(
        REPO
        / rel
    )

    print(
        "[PASS]",
        rel,
    )


# =================================================================================================
# 6. PRE-FIT SCIENTIFIC LEDGER GATE
# =================================================================================================

banner(
    "PRE-FIT SCIENTIFIC LEDGER"
)

freeze_1c = read_json(
    REPO
    / STAGE28_1C_FREEZE_REL
)

contracts = read_json(
    REPO
    / STAGE28_INPUT_CONTRACTS_REL
)


if (
    freeze_1c[
        "status"
    ]
    != "FROZEN_BEFORE_FIRST_STAGE28_MODEL_FIT"
):
    raise RuntimeError(
        "Stage28-1C freeze status changed."
    )


execution = (
    freeze_1c[
        "execution_manifest"
    ]
)


checks = {
    "components":
        (
            int(
                execution[
                    "components"
                ]
            ),
            EXPECTED_COMPONENTS,
        ),

    "evaluation_cells":
        (
            int(
                execution[
                    "scientific_evaluation_cells"
                ]
            ),
            EXPECTED_EVALUATION_CELLS,
        ),

    "existing_reused":
        (
            int(
                execution[
                    "existing_reused"
                ]
            ),
            EXPECTED_REUSED,
        ),

    "new_fit_budget":
        (
            int(
                execution[
                    "new_fit_budget"
                ]
            ),
            EXPECTED_NEW_FITS,
        ),

    "new_fits_consumed":
        (
            int(
                execution[
                    "new_fits_consumed"
                ]
            ),
            0,
        ),

    "new_fits_remaining":
        (
            int(
                execution[
                    "new_fits_remaining"
                ]
            ),
            EXPECTED_NEW_FITS,
        ),
}


for name, (
    actual,
    expected,
) in checks.items():

    print(
        f"{name:24s}:",
        actual,
    )

    if actual != expected:

        raise RuntimeError(
            f"Stage28 ledger mismatch: {name}\n"
            f"expected={expected}\n"
            f"actual={actual}"
        )


if (
    execution[
        "compute"
    ]
    != "CPU"
):
    raise RuntimeError(
        "Stage28 CPU compute policy changed."
    )


stage22_freeze = (
    freeze_1c[
        "model_input_contracts"
    ][
        "stage22"
    ]
)


if (
    stage22_freeze[
        "contract_sha256"
    ]
    != EXPECTED_STAGE22_CONTRACT_SHA
):
    raise RuntimeError(
        "Stage22 input-contract SHA changed."
    )


print()
print(
    "[PASS] fit #1 is still UNCONSUMED"
)

print(
    "[PASS] 108 / 108 new fits remain"
)

print(
    "[PASS] Stage28 CPU-only policy remains frozen"
)


# =================================================================================================
# 7. LOAD HISTORICAL STAGE22 RESET-SAFE CACHE CONTRACT
# =================================================================================================

banner(
    "STAGE22 RESET-SAFE CACHE CONTRACT"
)

manifest = read_json(
    REPO
    / STAGE22_MANIFEST_REL
)

remote_receipt = read_json(
    REPO
    / STAGE22_REMOTE_RECEIPT_REL
)


if (
    remote_receipt[
        "status"
    ]
    != "PRIVATE_KAGGLE_RESET_SAFE_CACHE_CHECKPOINT_VERIFIED"
):
    raise RuntimeError(
        "Stage22 reset-safe cache checkpoint status invalid."
    )


if (
    remote_receipt[
        "kaggle_dataset"
    ][
        "dataset_id"
    ]
    != PRIVATE_DATASET_ID
):
    raise RuntimeError(
        "Unexpected Stage22 private dataset identity."
    )


historical_files = {
    row[
        "cache_file"
    ]:
    row

    for row
    in manifest[
        "cache"
    ][
        "files"
    ]
}


if len(
    historical_files
) != 8:
    raise RuntimeError(
        "Stage22 historical cache file count != 8."
    )


if int(
    manifest[
        "cache"
    ][
        "total_rows"
    ]
) != EXPECTED_STAGE22_ROWS:
    raise RuntimeError(
        "Historical Stage22 total rows changed."
    )


if int(
    manifest[
        "cache"
    ][
        "total_bytes"
    ]
) != EXPECTED_STAGE22_BYTES:
    raise RuntimeError(
        "Historical Stage22 total bytes changed."
    )


print(
    "Private dataset:",
    PRIVATE_DATASET_ID,
)

print(
    "Files          : 8"
)

print(
    "Rows           :",
    f"{EXPECTED_STAGE22_ROWS:,}",
)

print(
    "Bytes          :",
    f"{EXPECTED_STAGE22_BYTES:,}",
)

print()
print(
    "[PASS] historical reset-safe contract exact"
)


# =================================================================================================
# 8. SEARCH ATTACHED /kaggle/input FIRST
# =================================================================================================

banner(
    "DISCOVER BYTE-EXACT STAGE22 CACHE"
)

input_root = Path(
    "/kaggle/input"
)

all_input_files = []

if input_root.exists():

    all_input_files = [
        p
        for p
        in input_root.rglob(
            "*"
        )
        if p.is_file()
    ]


resolved = {}


def resolve_exact_candidates(
    candidates,
    source_label,
):

    for filename, expected in (
        historical_files.items()
    ):

        if filename in resolved:
            continue


        matching_name = [
            p
            for p
            in candidates
            if p.name == filename
        ]


        matching_size = [
            p
            for p
            in matching_name
            if int(
                p.stat().st_size
            ) == int(
                expected[
                    "bytes"
                ]
            )
        ]


        exact = []

        for path in sorted(
            matching_size,
            key=lambda p: str(p),
        ):

            actual_sha = sha256_file(
                path
            )

            if (
                actual_sha
                == expected[
                    "sha256"
                ]
            ):

                exact.append(
                    path
                )


        if exact:

            resolved[
                filename
            ] = {
                "path":
                    exact[0],

                "source":
                    source_label,

                "exact_candidates":
                    len(
                        exact
                    ),
            }


resolve_exact_candidates(
    all_input_files,
    "ATTACHED_KAGGLE_INPUT",
)


print(
    "Exact attached files:",
    len(
        resolved
    ),
    "/ 8",
)


# =================================================================================================
# 9. PRIVATE KAGGLE CHECKPOINT RECOVERY IF NEEDED
# =================================================================================================

download_root = None
download_error = None


if len(
    resolved
) < 8:

    banner(
        "DOWNLOAD FROZEN PRIVATE KAGGLE CHECKPOINT"
    )

    print(
        "Dataset:",
        PRIVATE_DATASET_ID,
    )


    try:

        try:
            import kagglehub

        except ImportError:

            print(
                "[INFO] Installing kagglehub client..."
            )

            install = command(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "kagglehub",
                ],
                cwd="/kaggle/working",
            )

            import kagglehub


        print(
            "kagglehub version:",
            getattr(
                kagglehub,
                "__version__",
                "UNKNOWN",
            ),
        )


        download_root = Path(
            kagglehub.dataset_download(
                PRIVATE_DATASET_ID
            )
        ).resolve()


        print(
            "Recovered dataset root:"
        )

        print(
            " ",
            download_root,
        )


        downloaded_files = [
            p
            for p
            in download_root.rglob(
                "*"
            )
            if p.is_file()
        ]


        resolve_exact_candidates(
            downloaded_files,
            "PRIVATE_KAGGLE_RESET_SAFE_DATASET",
        )


    except Exception as exc:

        download_error = repr(
            exc
        )


# =================================================================================================
# 10. REQUIRE ALL 8 EXACT CACHE FILES
# =================================================================================================

banner(
    "8/8 BYTE-EXACT STAGE22 CACHE GATE"
)

if len(
    resolved
) != 8:

    missing = sorted(
        set(
            historical_files
        )
        -
        set(
            resolved
        )
    )


    print(
        "Resolved:",
        len(
            resolved
        ),
        "/ 8",
    )

    print()
    print(
        "Missing:"
    )

    for name in missing:
        print(
            " ",
            name,
        )

    print()
    print(
        "Attach this private Kaggle dataset:"
    )

    print(
        " ",
        PRIVATE_DATASET_ID,
    )


    if download_error:

        print()
        print(
            "Automatic recovery error:"
        )

        print(
            " ",
            download_error,
        )


    raise RuntimeError(
        "Exact Stage22 runtime cache unavailable. "
        "No scientific operation was performed."
    )


total_rows = 0
total_bytes = 0


for filename in sorted(
    resolved
):

    expected = historical_files[
        filename
    ]

    path = resolved[
        filename
    ][
        "path"
    ]


    actual_bytes = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(
        path
    )


    if (
        actual_bytes
        != int(
            expected[
                "bytes"
            ]
        )
    ):
        raise RuntimeError(
            f"{filename}: byte-size mismatch."
        )

    if (
        actual_sha
        != expected[
            "sha256"
        ]
    ):
        raise RuntimeError(
            f"{filename}: SHA256 mismatch."
        )


    total_rows += int(
        expected[
            "rows"
        ]
    )

    total_bytes += (
        actual_bytes
    )


    print(
        "[PASS]",
        filename,
    )

    print(
        "       source:",
        resolved[
            filename
        ][
            "source"
        ],
    )

    print(
        "       rows  :",
        f"{int(expected['rows']):,}",
    )

    print(
        "       bytes :",
        f"{actual_bytes:,}",
    )

    print(
        "       SHA256:",
        actual_sha,
    )


if total_rows != EXPECTED_STAGE22_ROWS:
    raise RuntimeError(
        "Recovered Stage22 total rows mismatch."
    )

if total_bytes != EXPECTED_STAGE22_BYTES:
    raise RuntimeError(
        "Recovered Stage22 total bytes mismatch."
    )


print()
print(
    "[PASS] 8 / 8 exact historical cache files recovered"
)


# =================================================================================================
# 11. RESTORE HISTORICAL RUNTIME PATH
# =================================================================================================

banner(
    "RESTORE STAGE22 RUNTIME CACHE PATH"
)

if STAGE22_RUNTIME_CACHE.exists():

    shutil.rmtree(
        STAGE22_RUNTIME_CACHE
    )


STAGE22_RUNTIME_CACHE.mkdir(
    parents=True,
    exist_ok=False,
)


for filename in sorted(
    historical_files
):

    source = (
        resolved[
            filename
        ][
            "path"
        ]
        .resolve()
    )

    target = (
        STAGE22_RUNTIME_CACHE
        / filename
    )


    target.symlink_to(
        source
    )


    if not target.is_file():

        raise RuntimeError(
            f"Failed to bind runtime cache file:\n{target}"
        )


    actual_sha = sha256_file(
        target
    )


    if (
        actual_sha
        != historical_files[
            filename
        ][
            "sha256"
        ]
    ):

        raise RuntimeError(
            f"{filename}: runtime-binding SHA mismatch."
        )


    print(
        "[PASS]",
        target,
    )

    print(
        "       ->",
        source,
    )


print()
print(
    "[PASS] Stage22 byte-exact runtime cache restored"
)


# =================================================================================================
# 12. CONFIRM RESET DESTROYED 2A0 DISPOSABLE STATE
# =================================================================================================

banner(
    "STAGE28-2A0 DISPOSABLE STATE"
)

if STAGE28_2A_RUNTIME_ROOT.exists():

    print(
        "[NOTICE] Stage28-2A runtime directory exists:"
    )

    print(
        " ",
        STAGE28_2A_RUNTIME_ROOT,
    )

    print(
        "It will be safely recreated by Stage28-2A0."
    )

else:

    print(
        "[EXPECTED] Stage28-2A runtime matrix is absent."
    )


if STAGE28_2A0_RUNTIME_RECEIPT.exists():

    print(
        "[NOTICE] Old Stage28-2A0 runtime receipt exists:"
    )

    print(
        " ",
        STAGE28_2A0_RUNTIME_RECEIPT,
    )

    print(
        "Stage28-2A0 will recreate/overwrite its "
        "disposable runtime receipt."
    )

else:

    print(
        "[EXPECTED] Stage28-2A0 runtime receipt is absent."
    )


print()
print(
    "This loss is operational only."
)

print(
    "The successful pre-reset Stage28-2A0 run "
    "consumed no scientific fit budget."
)


# =================================================================================================
# 13. FINAL GIT / LEDGER GATE
# =================================================================================================

banner(
    "FINAL RECOVERY GATE"
)

final_head = git(
    "rev-parse",
    "HEAD",
)

final_origin = git(
    "rev-parse",
    "origin/main",
)

final_remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not final_remote_line:
    raise RuntimeError(
        "Unable to resolve final remote main."
    )

final_remote = (
    final_remote_line.split()[0]
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Expected   :",
    EXPECTED_HEAD,
)

print(
    "HEAD       :",
    final_head,
)

print(
    "origin/main:",
    final_origin,
)

print(
    "Remote main:",
    final_remote,
)

print(
    "Git clean  :",
    not bool(
        final_status
    ),
)


if not (
    final_head
    == final_origin
    == final_remote
    == EXPECTED_HEAD
):
    raise RuntimeError(
        "Final durable-state recovery gate failed."
    )

if final_status:
    raise RuntimeError(
        "Repository dirty after runtime recovery:\n"
        + final_status
    )


print()
print(
    "[PASS] durable scientific state unchanged"
)


# =================================================================================================
# 14. FINAL
# =================================================================================================

banner(
    "STAGE28-RUNTIME-R2 — RESET RECOVERY COMPLETE"
)

print(
    "Durable scientific commit:"
)

print(
    " ",
    EXPECTED_HEAD,
)

print()

print(
    "Recovered Stage22 cache:"
)

print(
    "  files = 8 / 8 BYTE EXACT"
)

print(
    "  rows  =",
    f"{EXPECTED_STAGE22_ROWS:,}",
)

print(
    "  bytes =",
    f"{EXPECTED_STAGE22_BYTES:,}",
)

print(
    "  runtime path:"
)

print(
    "   ",
    STAGE22_RUNTIME_CACHE,
)

print()

print(
    "Scientific operations during recovery:"
)

print(
    "  PREDICTOR_VALUES_READ   = 0"
)

print(
    "  MODEL_FITS              = 0"
)

print(
    "  MODEL_INFERENCE         = 0"
)

print(
    "  THRESHOLD_SELECTION     = 0"
)

print(
    "  FINAL_HOLDOUT_OPENINGS  = 0"
)

print()

print(
    "Fit ledger:"
)

print(
    "  NEW authorized = 108"
)

print(
    "  consumed       = 0"
)

print(
    "  remaining      = 108"
)

print()

print(
    "RECOVERY STATUS:"
)

print(
    "  No scientific state was lost."
)

print(
    "  FIT #1 remains unconsumed."
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Re-run Stage28-2A0 to reconstruct the "
    "disposable 14,412,403 x 70 Stage22 execution "
    "matrix and exact RANDOM/CHRONO memberships."
)

print()

print(
    "  After Stage28-2A0 passes again:"
)

print(
    "  Stage28-2A1 = RANDOM_NATURAL seed42"
)

print(
    "  FIT #1 = C001 CPU XGBoost."
)

print()
print(SEP)


GITHUB SECRET RECOVERY

[PASS] GitHub credential recovered from kaggle_secret:GITHUB_TOKEN
[PASS] token value intentionally not displayed

REPOSITORY RECOVERY

Fresh Kaggle runtime — cloning repository.

STAGE28 DURABLE SCIENTIFIC-PARENT GATE

Expected HEAD: 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Local HEAD   : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
origin/main  : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Remote main  : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Branch       : main
Subject      : stage28-1c: freeze recovered model input identities
Git clean    : True

[PASS] exact Stage28-1C durable state recovered

CRITICAL STAGE28 DURABLE ARTIFACT GATE

[PASS] results/stage28_stability_novelty_control/stage28_1c_model_input_identity_freeze/stage28_1c_freeze_record.json
[PASS] results/stage28_stability_novelty_control/stage28_1c_model_input_identity_freeze/stage28_model_input_contracts.json
[PASS] results/stage28_stability_novelty_control/stage28_1c_model_input_identity_freeze/stage

In [2]:
# =================================================================================================
# STAGE28-2A0 — STAGE22 FULL EXECUTION MATRIX + MEMBERSHIP MATERIALIZATION
#
# Expected durable parent:
#   738e3ee29d098d4132828aeaaacfa12a8cfd7b52
#
# PURPOSE
# -------
# Reconstruct disposable runtime state lost in the Kaggle reset:
#
#   - canonical 14,412,403 x 70 Stage22 float64 execution matrix
#   - binary labels
#   - day IDs
#   - RANDOM_NATURAL train/validation indices
#   - CHRONOLOGICAL_NATURAL train/validation geometry
#
# NO:
#   model fits
#   inference
#   threshold selection
#   final holdout opening
#   Git modification
#
# After success:
#   Stage28-2A1 = FIT #1 (C001 CPU XGBoost seed42)
# =================================================================================================

from __future__ import annotations

import gc
import hashlib
import json
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_HEAD = (
    "738e3ee29d098d4132828aeaaacfa12a8cfd7b52"
)

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

STAGE28_1C_DIR = (
    STAGE28_ROOT
    / "stage28_1c_model_input_identity_freeze"
)

STAGE28_1C_FREEZE = (
    STAGE28_1C_DIR
    / "stage28_1c_freeze_record.json"
)

STAGE28_INPUT_CONTRACTS = (
    STAGE28_1C_DIR
    / "stage28_model_input_contracts.json"
)

STAGE28_BINDING = (
    STAGE28_1C_DIR
    / "stage28_component_input_binding.csv"
)

STAGE28_2A_DURABLE_ROOT = (
    STAGE28_ROOT
    / "stage28_2a_stage22_seed_stability"
)

STAGE22_MANIFEST_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
    / "stage22r_1c_development_model_input_manifest.json"
)

STAGE22_MEMBERSHIP_SUMMARY_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
    / "stage22r_1b1_membership_summary.json"
)

RANDOM_VALIDATION_PACKBITS_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
    / "random_validation.packbits"
)

STAGE22_CACHE_ROOT = Path(
    "/kaggle/working/"
    "stage22r_1c_70f_development_cache"
)

RUNTIME_ROOT = Path(
    "/kaggle/working/"
    "stage28_2a_runtime_cache/"
    "stage22_full"
)

X_PATH = (
    RUNTIME_ROOT
    / "development_features_float64.npy"
)

Y_PATH = (
    RUNTIME_ROOT
    / "development_binary_labels_uint8.npy"
)

DAY_PATH = (
    RUNTIME_ROOT
    / "development_day_ids_uint8.npy"
)

RANDOM_TRAIN_IDX_PATH = (
    RUNTIME_ROOT
    / "random_natural_train_global_idx_int32.npy"
)

RANDOM_VALIDATION_IDX_PATH = (
    RUNTIME_ROOT
    / "random_natural_validation_global_idx_int32.npy"
)

RUNTIME_RECEIPT_PATH = Path(
    "/kaggle/working/"
    "stage28_2a0_stage22_execution_matrix_receipt.json"
)


EXPECTED_ROWS = 14_412_403
EXPECTED_FEATURES = 70

EXPECTED_ATTACK = 1_972_299
EXPECTED_BENIGN = 12_440_104

EXPECTED_RANDOM_TRAIN_ROWS = 11_529_922
EXPECTED_RANDOM_TRAIN_ATTACK = 1_577_839
EXPECTED_RANDOM_TRAIN_BENIGN = 9_952_083

EXPECTED_RANDOM_VAL_ROWS = 2_882_481
EXPECTED_RANDOM_VAL_ATTACK = 394_460
EXPECTED_RANDOM_VAL_BENIGN = 2_488_021

EXPECTED_CHRONO_TRAIN_ROWS = 13_818_623
EXPECTED_CHRONO_TRAIN_ATTACK = 1_910_043
EXPECTED_CHRONO_TRAIN_BENIGN = 11_908_580

EXPECTED_CHRONO_VAL_ROWS = 593_780
EXPECTED_CHRONO_VAL_ATTACK = 62_256
EXPECTED_CHRONO_VAL_BENIGN = 531_524

EXPECTED_RANDOM_PACKBITS_SHA = (
    "8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad"
)

EXPECTED_STAGE22_CONTRACT_SHA = (
    "975113373af098bb12b274248dfa012a536c3e6a8270d7b37dd4f3d9bf79b9f2"
)

EXPECTED_NEW_FITS = 108


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return (
        run(
            ["git", *args]
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array_blocked(
    arr,
    block_rows=65_536,
):
    h = hashlib.sha256()

    if arr.ndim == 1:

        for start in range(
            0,
            arr.shape[0],
            block_rows,
        ):

            stop = min(
                start + block_rows,
                arr.shape[0],
            )

            block = np.ascontiguousarray(
                arr[start:stop]
            )

            h.update(
                block.view(np.uint8)
            )

    elif arr.ndim == 2:

        for start in range(
            0,
            arr.shape[0],
            block_rows,
        ):

            stop = min(
                start + block_rows,
                arr.shape[0],
            )

            block = np.ascontiguousarray(
                arr[
                    start:stop,
                    :
                ]
            )

            h.update(
                block.view(np.uint8)
            )

    else:

        raise RuntimeError(
            f"Unsupported array ndim={arr.ndim}"
        )

    return h.hexdigest()


def safe_reset_runtime_root():

    expected = Path(
        "/kaggle/working/"
        "stage28_2a_runtime_cache/"
        "stage22_full"
    ).resolve()

    actual = RUNTIME_ROOT.resolve()

    if actual != expected:
        raise RuntimeError(
            "Refusing to clean unexpected runtime path."
        )

    if actual.exists():
        shutil.rmtree(actual)

    actual.mkdir(
        parents=True,
        exist_ok=False,
    )


# =================================================================================================
# 2. EXACT DURABLE-PARENT GATE
# =================================================================================================

banner(
    "STAGE28-2A0 — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote = remote_line.split()[0]

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD:",
    EXPECTED_HEAD,
)

print(
    "Local HEAD   :",
    head,
)

print(
    "origin/main  :",
    origin,
)

print(
    "Remote main  :",
    remote,
)

print(
    "Branch       :",
    branch,
)

print(
    "Git clean    :",
    not bool(status),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_HEAD
):
    raise RuntimeError(
        "Stage28-1C durable parent mismatch."
    )

if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean:\n"
        + status
    )


if STAGE28_2A_DURABLE_ROOT.exists():
    raise RuntimeError(
        "Stage28-2A durable execution output already exists.\n"
        "Do not overwrite an existing scientific checkpoint."
    )


print()
print(
    "[PASS] exact remotely durable Stage28-1C parent"
)


# =================================================================================================
# 3. VERIFY STAGE28-1C FREEZE
# =================================================================================================

banner(
    "STAGE28-1C MODEL-INPUT FREEZE GATE"
)

freeze_1c = read_json(
    require_file(
        STAGE28_1C_FREEZE
    )
)

contracts = read_json(
    require_file(
        STAGE28_INPUT_CONTRACTS
    )
)

require_file(
    STAGE28_BINDING
)


if (
    freeze_1c[
        "status"
    ]
    != "FROZEN_BEFORE_FIRST_STAGE28_MODEL_FIT"
):
    raise RuntimeError(
        "Stage28-1C freeze status invalid."
    )


execution = (
    freeze_1c[
        "execution_manifest"
    ]
)


if int(
    execution[
        "new_fit_budget"
    ]
) != EXPECTED_NEW_FITS:
    raise RuntimeError(
        "Stage28 fit budget != 108."
    )

if int(
    execution[
        "new_fits_consumed"
    ]
) != 0:
    raise RuntimeError(
        "FIT #1 was already consumed."
    )

if int(
    execution[
        "new_fits_remaining"
    ]
) != EXPECTED_NEW_FITS:
    raise RuntimeError(
        "Stage28 remaining fit ledger != 108."
    )

if (
    execution[
        "compute"
    ]
    != "CPU"
):
    raise RuntimeError(
        "Stage28 compute backend changed."
    )


stage22_contract = (
    freeze_1c[
        "model_input_contracts"
    ][
        "stage22"
    ]
)


print(
    "Stage22 contract SHA:",
    stage22_contract[
        "contract_sha256"
    ],
)

print(
    "Stage22 rows        :",
    f"{stage22_contract['rows']:,}",
)

print(
    "Stage22 files       :",
    stage22_contract[
        "files"
    ],
)

print(
    "NEW fits consumed   :",
    execution[
        "new_fits_consumed"
    ],
)

print(
    "NEW fits remaining  :",
    execution[
        "new_fits_remaining"
    ],
)


if (
    stage22_contract[
        "contract_sha256"
    ]
    != EXPECTED_STAGE22_CONTRACT_SHA
):
    raise RuntimeError(
        "Stage22 input-contract SHA mismatch."
    )

if int(
    stage22_contract[
        "rows"
    ]
) != EXPECTED_ROWS:
    raise RuntimeError(
        "Stage22 contract row count mismatch."
    )

if int(
    stage22_contract[
        "files"
    ]
) != 8:
    raise RuntimeError(
        "Stage22 contract file count mismatch."
    )


print()
print(
    "[PASS] Stage28-1C Stage22 contract exact"
)

print(
    "[PASS] FIT ledger still 0 / 108"
)


# =================================================================================================
# 4. LOAD FROZEN STAGE22 MODEL-INPUT / MEMBERSHIP CONTRACTS
# =================================================================================================

banner(
    "LOAD FROZEN STAGE22 MODEL-INPUT / MEMBERSHIP CONTRACTS"
)

manifest = read_json(
    require_file(
        STAGE22_MANIFEST_PATH
    )
)

membership = read_json(
    require_file(
        STAGE22_MEMBERSHIP_SUMMARY_PATH
    )
)


feature_order = list(
    manifest[
        "row_schema"
    ][
        "predictor_columns"
    ]
)


if len(feature_order) != EXPECTED_FEATURES:
    raise RuntimeError(
        "Stage22 feature count != 70."
    )


if (
    membership[
        "clean_development"
    ][
        "canonical_order"
    ]
    !=
    "(day_id ASC, original_zero_based_row_index ASC) after frozen K79 exclusions"
):
    raise RuntimeError(
        "Stage22 canonical row order changed."
    )


if (
    membership[
        "membership_derivation"
    ][
        "RANDOM_NATURAL_train"
    ]
    != "logical complement of random_validation bitset"
):
    raise RuntimeError(
        "Random train derivation changed."
    )

if (
    membership[
        "membership_derivation"
    ][
        "RANDOM_NATURAL_validation"
    ]
    != "random_validation bitset"
):
    raise RuntimeError(
        "Random validation derivation changed."
    )

if (
    membership[
        "membership_derivation"
    ][
        "CHRONOLOGICAL_NATURAL_train"
    ]
    != "day_id 0..6"
):
    raise RuntimeError(
        "Chronological train derivation changed."
    )

if (
    membership[
        "membership_derivation"
    ][
        "CHRONOLOGICAL_NATURAL_validation"
    ]
    != "day_id 7"
):
    raise RuntimeError(
        "Chronological validation derivation changed."
    )


print(
    "Rows        :",
    f"{EXPECTED_ROWS:,}",
)

print(
    "Features    :",
    EXPECTED_FEATURES,
)

print(
    "Dtype       :",
    manifest[
        "feature_configuration"
    ][
        "dtype"
    ],
)

print(
    "Feature[0]  :",
    feature_order[0],
)

print(
    "Feature[-1] :",
    feature_order[-1],
)

print()
print(
    "[PASS] Stage22 model-input + membership semantics exact"
)


# =================================================================================================
# 5. RE-VERIFY 8 BYTE-EXACT CACHE PARQUETS
# =================================================================================================

banner(
    "8/8 BYTE-EXACT STAGE22 CACHE GATE"
)

cache_records = sorted(
    manifest[
        "cache"
    ][
        "files"
    ],
    key=lambda x: int(
        x[
            "day_id"
        ]
    ),
)


if len(cache_records) != 8:
    raise RuntimeError(
        "Expected exactly 8 Stage22 cache files."
    )


expected_cursor = 0


for expected_day, record in enumerate(
    cache_records
):

    day_id = int(
        record[
            "day_id"
        ]
    )

    if day_id != expected_day:
        raise RuntimeError(
            "Stage22 cache day IDs are not 0..7."
        )


    path = (
        STAGE22_CACHE_ROOT
        / record[
            "cache_file"
        ]
    )

    require_file(path)


    actual_bytes = int(
        path.stat().st_size
    )

    actual_sha = sha256_file(path)


    if actual_bytes != int(
        record[
            "bytes"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: byte-size mismatch."
        )

    if actual_sha != record[
        "sha256"
    ]:
        raise RuntimeError(
            f"{path.name}: SHA256 mismatch."
        )


    pf = pq.ParquetFile(path)

    parquet_rows = int(
        pf.metadata.num_rows
    )


    if parquet_rows != int(
        record[
            "rows"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: Parquet row count mismatch."
        )


    print(
        f"[PASS] day {day_id} "
        f"{path.name}"
    )

    print(
        "       rows  :",
        f"{parquet_rows:,}",
    )

    print(
        "       SHA256:",
        actual_sha,
    )


    expected_cursor += parquet_rows


if expected_cursor != EXPECTED_ROWS:
    raise RuntimeError(
        "Stage22 cache total rows mismatch."
    )


print()
print(
    "[PASS] all eight historical cache files byte-exact"
)


# =================================================================================================
# 6. DISK SAFETY GATE
# =================================================================================================

banner(
    "RUNTIME STORAGE GATE"
)

disk = shutil.disk_usage(
    "/kaggle/working"
)

x_payload = (
    EXPECTED_ROWS
    * EXPECTED_FEATURES
    * np.dtype(
        np.float64
    ).itemsize
)

metadata_payload = (
    EXPECTED_ROWS
    * (
        np.dtype(
            np.uint8
        ).itemsize
        * 2
    )
)

idx_payload_approx = (
    (
        EXPECTED_RANDOM_TRAIN_ROWS
        +
        EXPECTED_RANDOM_VAL_ROWS
    )
    * np.dtype(
        np.int32
    ).itemsize
)

required = (
    x_payload
    +
    metadata_payload
    +
    idx_payload_approx
)


print(
    "Free space             :",
    f"{disk.free / (1024**3):.3f} GiB",
)

print(
    "Feature matrix payload :",
    f"{x_payload / (1024**3):.3f} GiB",
)

print(
    "Metadata + indices     :",
    f"{(metadata_payload + idx_payload_approx) / (1024**3):.3f} GiB",
)


if disk.free < (
    required
    + int(
        1.5
        * 1024**3
    )
):
    raise RuntimeError(
        "Insufficient /kaggle/working disk space."
    )


print()
print(
    "[PASS] sufficient runtime disk space"
)


# =================================================================================================
# 7. ALLOCATE DISPOSABLE RUNTIME CACHE
# =================================================================================================

banner(
    "ALLOCATE STAGE22 EXECUTION MATRICES"
)

safe_reset_runtime_root()


X = np.lib.format.open_memmap(
    X_PATH,
    mode="w+",
    dtype=np.float64,
    shape=(
        EXPECTED_ROWS,
        EXPECTED_FEATURES,
    ),
)

y = np.lib.format.open_memmap(
    Y_PATH,
    mode="w+",
    dtype=np.uint8,
    shape=(
        EXPECTED_ROWS,
    ),
)

day_ids = np.lib.format.open_memmap(
    DAY_PATH,
    mode="w+",
    dtype=np.uint8,
    shape=(
        EXPECTED_ROWS,
    ),
)


print(
    "X:",
    X_PATH,
)

print(
    "  shape:",
    X.shape,
)

print(
    "  dtype:",
    X.dtype,
)

print()

print(
    "y:",
    Y_PATH,
)

print()

print(
    "day:",
    DAY_PATH,
)


# =================================================================================================
# 8. MATERIALIZE CANONICAL STAGE22 DEVELOPMENT MATRIX
# =================================================================================================

banner(
    "MATERIALIZE CANONICAL STAGE22 DEVELOPMENT POPULATION"
)

required_columns = [
    "clean_position",
    "day_id",
    "binary_label",
    *feature_order,
]


cursor = 0

total_attack = 0
total_benign = 0
total_nan = 0
total_inf = 0

day_audit = []


for record in cache_records:

    day_id = int(
        record[
            "day_id"
        ]
    )

    path = (
        STAGE22_CACHE_ROOT
        / record[
            "cache_file"
        ]
    )

    expected_day_rows = int(
        record[
            "rows"
        ]
    )

    expected_day_attack = int(
        record[
            "attack"
        ]
    )

    expected_day_benign = int(
        record[
            "benign"
        ]
    )


    pf = pq.ParquetFile(path)


    if list(
        pf.schema_arrow.names
    ) != (
        [
            "clean_position",
            "day_id",
            "original_row_index",
            "binary_label",
            *feature_order,
        ]
    ):
        raise RuntimeError(
            f"{path.name}: ordered 74-column schema changed."
        )


    day_start = cursor

    day_attack = 0
    day_benign = 0
    day_nan = 0
    day_inf = 0


    print()
    print(
        "-" * 120
    )

    print(
        f"Day {day_id} — {path.name}"
    )

    print(
        "-" * 120
    )


    for batch in pf.iter_batches(
        batch_size=65_536,
        columns=required_columns,
        use_threads=True,
    ):

        n = int(
            batch.num_rows
        )

        if n <= 0:
            continue


        clean_pos = (
            batch
            .column(0)
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.int64,
                copy=False,
            )
        )

        block_day = (
            batch
            .column(1)
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.uint8,
                copy=False,
            )
        )

        block_y = (
            batch
            .column(2)
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.uint8,
                copy=False,
            )
        )


        expected_positions = np.arange(
            cursor,
            cursor + n,
            dtype=np.int64,
        )


        if not np.array_equal(
            clean_pos,
            expected_positions,
        ):
            raise RuntimeError(
                f"{path.name}: clean_position sequence mismatch "
                f"at global cursor {cursor:,}."
            )


        if not np.all(
            block_day == day_id
        ):
            raise RuntimeError(
                f"{path.name}: day_id mismatch."
            )


        if not np.all(
            (block_y == 0)
            |
            (block_y == 1)
        ):
            raise RuntimeError(
                f"{path.name}: binary_label outside {{0,1}}."
            )


        feature_arrays = []

        for j in range(
            EXPECTED_FEATURES
        ):

            arr = (
                batch
                .column(
                    3 + j
                )
                .to_numpy(
                    zero_copy_only=False
                )
                .astype(
                    np.float64,
                    copy=False,
                )
            )

            feature_arrays.append(
                arr
            )


        X_block = np.column_stack(
            feature_arrays
        )


        if X_block.shape != (
            n,
            EXPECTED_FEATURES,
        ):
            raise RuntimeError(
                "Unexpected Stage22 feature-block shape."
            )


        inf_count = int(
            np.isinf(
                X_block
            ).sum()
        )

        nan_count = int(
            np.isnan(
                X_block
            ).sum()
        )


        if inf_count != 0:
            raise RuntimeError(
                f"{path.name}: infinity survived historical cache."
            )


        stop = cursor + n


        X[
            cursor:stop,
            :
        ] = X_block

        y[
            cursor:stop
        ] = block_y

        day_ids[
            cursor:stop
        ] = block_day


        attack = int(
            block_y.sum()
        )

        benign = n - attack


        day_attack += attack
        day_benign += benign
        day_nan += nan_count
        day_inf += inf_count

        cursor = stop


        del clean_pos
        del block_day
        del block_y
        del expected_positions
        del feature_arrays
        del X_block

        gc.collect()


    day_rows = (
        cursor
        - day_start
    )


    if day_rows != expected_day_rows:
        raise RuntimeError(
            f"{path.name}: materialized row count mismatch."
        )

    if day_attack != expected_day_attack:
        raise RuntimeError(
            f"{path.name}: attack count mismatch."
        )

    if day_benign != expected_day_benign:
        raise RuntimeError(
            f"{path.name}: benign count mismatch."
        )


    print(
        "Rows   :",
        f"{day_rows:,}",
    )

    print(
        "Attack :",
        f"{day_attack:,}",
    )

    print(
        "Benign :",
        f"{day_benign:,}",
    )

    print(
        "NaNs   :",
        f"{day_nan:,}",
    )

    print(
        "[PASS] canonical day materialized"
    )


    total_attack += day_attack
    total_benign += day_benign
    total_nan += day_nan
    total_inf += day_inf


    day_audit.append(
        {
            "day_id":
                day_id,

            "cache_file":
                path.name,

            "global_start":
                day_start,

            "global_stop_exclusive":
                cursor,

            "rows":
                day_rows,

            "attack":
                day_attack,

            "benign":
                day_benign,

            "nan_cells":
                day_nan,

            "inf_cells":
                day_inf,
        }
    )


if cursor != EXPECTED_ROWS:
    raise RuntimeError(
        "Canonical Stage22 materialization "
        "did not reach 14,412,403 rows."
    )

if total_attack != EXPECTED_ATTACK:
    raise RuntimeError(
        "Global Stage22 attack count mismatch."
    )

if total_benign != EXPECTED_BENIGN:
    raise RuntimeError(
        "Global Stage22 benign count mismatch."
    )


expected_output_nan = int(
    manifest[
        "nonfinite_conversion"
    ][
        "output_nan_values"
    ]
)


if total_nan != expected_output_nan:
    raise RuntimeError(
        "Global Stage22 NaN count mismatch.\n"
        f"expected={expected_output_nan:,}\n"
        f"actual={total_nan:,}"
    )

if total_inf != 0:
    raise RuntimeError(
        "Global execution matrix contains infinity."
    )


X.flush()
y.flush()
day_ids.flush()


print()
print(
    "[PASS] complete canonical development population materialized"
)

print(
    "Rows   :",
    f"{cursor:,}",
)

print(
    "Attack :",
    f"{total_attack:,}",
)

print(
    "Benign :",
    f"{total_benign:,}",
)

print(
    "NaNs   :",
    f"{total_nan:,}",
)

print(
    "Inf    :",
    total_inf,
)


# =================================================================================================
# 9. RELOAD MEMMAPS
# =================================================================================================

banner(
    "RELOAD EXECUTION MEMMAPS"
)

del X
del y
del day_ids

gc.collect()


X = np.load(
    X_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

y = np.load(
    Y_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

day_ids = np.load(
    DAY_PATH,
    mmap_mode="r",
    allow_pickle=False,
)


if X.shape != (
    EXPECTED_ROWS,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        "Reloaded Stage22 X shape mismatch."
    )

if X.dtype != np.float64:
    raise RuntimeError(
        "Reloaded Stage22 X dtype != float64."
    )

if y.shape != (
    EXPECTED_ROWS,
):
    raise RuntimeError(
        "Reloaded Stage22 y shape mismatch."
    )

if y.dtype != np.uint8:
    raise RuntimeError(
        "Reloaded Stage22 y dtype != uint8."
    )


print(
    "[PASS] Stage22 execution memmaps reload exactly"
)


# =================================================================================================
# 10. RANDOM_NATURAL MEMBERSHIP
# =================================================================================================

banner(
    "RANDOM_NATURAL — EXACT INHERITED MEMBERSHIP"
)

require_file(
    RANDOM_VALIDATION_PACKBITS_PATH
)

actual_pack_sha = sha256_file(
    RANDOM_VALIDATION_PACKBITS_PATH
)


print(
    "Expected packbits SHA:",
    EXPECTED_RANDOM_PACKBITS_SHA,
)

print(
    "Actual packbits SHA  :",
    actual_pack_sha,
)


if (
    actual_pack_sha
    != EXPECTED_RANDOM_PACKBITS_SHA
):
    raise RuntimeError(
        "Frozen random-validation packbits SHA mismatch."
    )


packed = np.fromfile(
    RANDOM_VALIDATION_PACKBITS_PATH,
    dtype=np.uint8,
)

random_val_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:EXPECTED_ROWS].astype(
    np.bool_,
    copy=False,
)


if int(
    random_val_mask.sum()
) != EXPECTED_RANDOM_VAL_ROWS:
    raise RuntimeError(
        "Random validation population mismatch."
    )


random_validation_idx = np.flatnonzero(
    random_val_mask
).astype(
    np.int32,
    copy=False,
)

random_train_idx = np.flatnonzero(
    ~random_val_mask
).astype(
    np.int32,
    copy=False,
)


if len(
    random_train_idx
) != EXPECTED_RANDOM_TRAIN_ROWS:
    raise RuntimeError(
        "Random train row count mismatch."
    )

if len(
    random_validation_idx
) != EXPECTED_RANDOM_VAL_ROWS:
    raise RuntimeError(
        "Random validation row count mismatch."
    )


random_train_attack = int(
    np.asarray(
        y[
            random_train_idx
        ]
    ).sum()
)

random_val_attack = int(
    np.asarray(
        y[
            random_validation_idx
        ]
    ).sum()
)


random_train_benign = (
    len(random_train_idx)
    -
    random_train_attack
)

random_val_benign = (
    len(random_validation_idx)
    -
    random_val_attack
)


if (
    random_train_attack
    != EXPECTED_RANDOM_TRAIN_ATTACK
    or random_train_benign
    != EXPECTED_RANDOM_TRAIN_BENIGN
):
    raise RuntimeError(
        "Random train class counts mismatch."
    )

if (
    random_val_attack
    != EXPECTED_RANDOM_VAL_ATTACK
    or random_val_benign
    != EXPECTED_RANDOM_VAL_BENIGN
):
    raise RuntimeError(
        "Random validation class counts mismatch."
    )


np.save(
    RANDOM_TRAIN_IDX_PATH,
    random_train_idx,
    allow_pickle=False,
)

np.save(
    RANDOM_VALIDATION_IDX_PATH,
    random_validation_idx,
    allow_pickle=False,
)


random_train_idx_sha = (
    sha256_array_blocked(
        random_train_idx
    )
)

random_val_idx_sha = (
    sha256_array_blocked(
        random_validation_idx
    )
)


print(
    "TRAIN rows   :",
    f"{len(random_train_idx):,}",
)

print(
    "  attack     :",
    f"{random_train_attack:,}",
)

print(
    "  benign     :",
    f"{random_train_benign:,}",
)

print(
    "  index SHA  :",
    random_train_idx_sha,
)

print()

print(
    "VALID rows   :",
    f"{len(random_validation_idx):,}",
)

print(
    "  attack     :",
    f"{random_val_attack:,}",
)

print(
    "  benign     :",
    f"{random_val_benign:,}",
)

print(
    "  index SHA  :",
    random_val_idx_sha,
)

print()

print(
    "[PASS] RANDOM_NATURAL inherited membership exact"
)


# =================================================================================================
# 11. CHRONOLOGICAL_NATURAL MEMBERSHIP
# =================================================================================================

banner(
    "CHRONOLOGICAL_NATURAL — EXACT INHERITED MEMBERSHIP"
)

chrono_train_stop = (
    EXPECTED_CHRONO_TRAIN_ROWS
)


if not np.all(
    day_ids[
        :chrono_train_stop
    ]
    <= 6
):
    raise RuntimeError(
        "Chronological train contains day 7."
    )

if not np.all(
    day_ids[
        chrono_train_stop:
    ]
    == 7
):
    raise RuntimeError(
        "Chronological validation is not exactly day 7."
    )


chrono_train_y = np.asarray(
    y[
        :chrono_train_stop
    ]
)

chrono_val_y = np.asarray(
    y[
        chrono_train_stop:
    ]
)


chrono_train_attack = int(
    chrono_train_y.sum()
)

chrono_val_attack = int(
    chrono_val_y.sum()
)


chrono_train_benign = (
    len(chrono_train_y)
    -
    chrono_train_attack
)

chrono_val_benign = (
    len(chrono_val_y)
    -
    chrono_val_attack
)


if len(
    chrono_train_y
) != EXPECTED_CHRONO_TRAIN_ROWS:
    raise RuntimeError(
        "Chronological train rows mismatch."
    )

if len(
    chrono_val_y
) != EXPECTED_CHRONO_VAL_ROWS:
    raise RuntimeError(
        "Chronological validation rows mismatch."
    )


if (
    chrono_train_attack
    != EXPECTED_CHRONO_TRAIN_ATTACK
    or chrono_train_benign
    != EXPECTED_CHRONO_TRAIN_BENIGN
):
    raise RuntimeError(
        "Chronological train class counts mismatch."
    )

if (
    chrono_val_attack
    != EXPECTED_CHRONO_VAL_ATTACK
    or chrono_val_benign
    != EXPECTED_CHRONO_VAL_BENIGN
):
    raise RuntimeError(
        "Chronological validation class counts mismatch."
    )


print(
    "TRAIN rows:",
    f"{len(chrono_train_y):,}",
)

print(
    "  attack  :",
    f"{chrono_train_attack:,}",
)

print(
    "  benign  :",
    f"{chrono_train_benign:,}",
)

print()

print(
    "VALID rows:",
    f"{len(chrono_val_y):,}",
)

print(
    "  attack  :",
    f"{chrono_val_attack:,}",
)

print(
    "  benign  :",
    f"{chrono_val_benign:,}",
)

print()

print(
    "[PASS] CHRONOLOGICAL_NATURAL inherited membership exact"
)


# =================================================================================================
# 12. EXECUTION-CACHE CONTENT IDENTITIES
# =================================================================================================

banner(
    "STAGE22 RUNTIME EXECUTION-CACHE IDENTITIES"
)

print(
    "Hashing binary-label vector..."
)

y_sha = sha256_array_blocked(
    y
)

print(
    "Label content SHA256:",
    y_sha,
)

print()

print(
    "Hashing day-id vector..."
)

day_sha = sha256_array_blocked(
    day_ids
)

print(
    "Day content SHA256:",
    day_sha,
)

print()

print(
    "Feature matrix runtime .npy is execution layout only."
)

print(
    "Its scientific identity remains the eight "
    "byte-exact historical Stage22 Parquet SHA256 values."
)


# =================================================================================================
# 13. WRITE RUNTIME-ONLY EXECUTION RECEIPT
# =================================================================================================

runtime_receipt = {

    "stage":
        "Stage28-2A0",

    "type":
        "RUNTIME_ONLY_STAGE22_FULL_EXECUTION_MATRIX_AND_MEMBERSHIP_MATERIALIZATION",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "durable_scientific_parent":
        EXPECTED_HEAD,

    "scientific_contract": {
        "stage22_contract_sha256":
            EXPECTED_STAGE22_CONTRACT_SHA,

        "source":
            (
                "EIGHT_BYTE_EXACT_HISTORICAL_"
                "STAGE22R_1C_PARQUET_CACHES"
            ),

        "feature_count":
            EXPECTED_FEATURES,

        "feature_dtype":
            "float64",

        "canonical_order":
            (
                "(day_id ASC, original_zero_based_row_index ASC) "
                "after frozen K79 exclusions"
            ),
    },

    "runtime_matrix": {
        "features_path":
            str(X_PATH),

        "labels_path":
            str(Y_PATH),

        "day_ids_path":
            str(DAY_PATH),

        "shape": [
            EXPECTED_ROWS,
            EXPECTED_FEATURES,
        ],

        "dtype":
            "float64",

        "rows":
            EXPECTED_ROWS,

        "attack":
            total_attack,

        "benign":
            total_benign,

        "nan_cells":
            total_nan,

        "inf_cells":
            total_inf,

        "binary_labels_content_sha256":
            y_sha,

        "day_ids_content_sha256":
            day_sha,

        "runtime_files_committed_to_git":
            False,
    },

    "day_audit":
        day_audit,

    "random_natural": {
        "validation_packbits_path":
            str(
                RANDOM_VALIDATION_PACKBITS_PATH.relative_to(
                    REPO
                )
            ),

        "validation_packbits_sha256":
            actual_pack_sha,

        "train_index_path":
            str(
                RANDOM_TRAIN_IDX_PATH
            ),

        "validation_index_path":
            str(
                RANDOM_VALIDATION_IDX_PATH
            ),

        "train_index_content_sha256":
            random_train_idx_sha,

        "validation_index_content_sha256":
            random_val_idx_sha,

        "train": {
            "rows":
                len(random_train_idx),

            "attack":
                random_train_attack,

            "benign":
                random_train_benign,
        },

        "validation": {
            "rows":
                len(random_validation_idx),

            "attack":
                random_val_attack,

            "benign":
                random_val_benign,
        },
    },

    "chronological_natural": {
        "train_slice": [
            0,
            EXPECTED_CHRONO_TRAIN_ROWS,
        ],

        "validation_slice": [
            EXPECTED_CHRONO_TRAIN_ROWS,
            EXPECTED_ROWS,
        ],

        "train": {
            "rows":
                EXPECTED_CHRONO_TRAIN_ROWS,

            "attack":
                chrono_train_attack,

            "benign":
                chrono_train_benign,
        },

        "validation": {
            "rows":
                EXPECTED_CHRONO_VAL_ROWS,

            "attack":
                chrono_val_attack,

            "benign":
                chrono_val_benign,
        },
    },

    "scientific_operations": {
        "stage22_development_predictor_rows_read_for_execution":
            EXPECTED_ROWS,

        "model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "final_holdout_openings":
            0,

        "new_scientific_target_openings":
            0,
    },

    "fit_ledger": {
        "new_authorized":
            EXPECTED_NEW_FITS,

        "new_consumed":
            0,

        "new_remaining":
            EXPECTED_NEW_FITS,
    },

    "status":
        "READY_FOR_STAGE28_FIT_001",
}


RUNTIME_RECEIPT_PATH.write_text(
    json.dumps(
        runtime_receipt,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )
    + "\n",
    encoding="utf-8",
)


print()
print(
    "Runtime receipt:"
)

print(
    " ",
    RUNTIME_RECEIPT_PATH,
)


# =================================================================================================
# 14. FINAL GIT CLEANLINESS GATE
# =================================================================================================

banner(
    "FINAL GIT CLEANLINESS GATE"
)

final_head = git(
    "rev-parse",
    "HEAD",
)

final_origin = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head,
)

print(
    "origin/main:",
    final_origin,
)

print(
    "Git clean  :",
    not bool(final_status),
)


if (
    final_head != EXPECTED_HEAD
    or final_origin != EXPECTED_HEAD
):
    raise RuntimeError(
        "Git parent changed during Stage28-2A0."
    )

if final_status:
    raise RuntimeError(
        "Stage28-2A0 unexpectedly changed Git:\n"
        + final_status
    )


print()
print(
    "[PASS] durable repository untouched"
)


# =================================================================================================
# 15. FINAL
# =================================================================================================

banner(
    "STAGE28-2A0 — STAGE22 EXECUTION MATRIX READY"
)

print(
    "Durable parent:"
)

print(
    " ",
    EXPECTED_HEAD,
)

print()

print(
    "Stage22 execution population:"
)

print(
    "  rows       =",
    f"{EXPECTED_ROWS:,}",
)

print(
    "  features   =",
    EXPECTED_FEATURES,
)

print(
    "  dtype      = float64"
)

print(
    "  attack     =",
    f"{total_attack:,}",
)

print(
    "  benign     =",
    f"{total_benign:,}",
)

print(
    "  NaN cells  =",
    f"{total_nan:,}",
)

print(
    "  inf cells  =",
    total_inf,
)

print()

print(
    "RANDOM_NATURAL:"
)

print(
    "  train      =",
    f"{len(random_train_idx):,}",
)

print(
    "  validation =",
    f"{len(random_validation_idx):,}",
)

print()

print(
    "CHRONOLOGICAL_NATURAL:"
)

print(
    "  train      =",
    f"{EXPECTED_CHRONO_TRAIN_ROWS:,}",
)

print(
    "  validation =",
    f"{EXPECTED_CHRONO_VAL_ROWS:,}",
)

print()

print(
    "Scientific operations:"
)

print(
    "  MODEL_FITS            = 0"
)

print(
    "  MODEL_INFERENCE       = 0"
)

print(
    "  THRESHOLD_SELECTION   = 0"
)

print(
    "  FINAL_HOLDOUT_OPENING = 0"
)

print()

print(
    "Fit ledger:"
)

print(
    "  authorized = 108"
)

print(
    "  consumed   = 0"
)

print(
    "  remaining  = 108"
)

print()

print(
    "STATUS:"
)

print(
    "  READY FOR FIT #1."
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage28-2A1 — RANDOM_NATURAL / seed42."
)

print(
    "  FIT #1: C001 CPU XGBoost XGB_11 seed42."
)

print(
    "  C002 LightGBM seed42 = exact historical CPU reuse."
)

print(
    "  Shared final holdout remains untouched."
)

print()
print(SEP)


STAGE28-2A0 — EXACT DURABLE-PARENT GATE

Expected HEAD: 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Local HEAD   : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
origin/main  : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Remote main  : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Branch       : main
Git clean    : True

[PASS] exact remotely durable Stage28-1C parent

STAGE28-1C MODEL-INPUT FREEZE GATE

Stage22 contract SHA: 975113373af098bb12b274248dfa012a536c3e6a8270d7b37dd4f3d9bf79b9f2
Stage22 rows        : 14,412,403
Stage22 files       : 8
NEW fits consumed   : 0
NEW fits remaining  : 108

[PASS] Stage28-1C Stage22 contract exact
[PASS] FIT ledger still 0 / 108

LOAD FROZEN STAGE22 MODEL-INPUT / MEMBERSHIP CONTRACTS

Rows        : 14,412,403
Features    : 70
Dtype       : float64
Feature[0]  : Dst Port
Feature[-1] : Idle Min

[PASS] Stage22 model-input + membership semantics exact

8/8 BYTE-EXACT STAGE22 CACHE GATE

[PASS] day 0 day_00_02-14-2018.parquet
       rows  : 822,947
       SHA256: a54

In [3]:
# =================================================================================================
# STAGE28-2A1 — RANDOM_NATURAL / SEED42
#
# FIT #1:
#   C001 — XGBOOST / XGB_11 / seed42 / CPU / NEW_FIT_AUTHORIZED
#
# REUSE:
#   C002 — LIGHTGBM / LGBM_11 / seed42 / CPU / REUSE_EXISTING
#
# Durable parent:
#   738e3ee29d098d4132828aeaaacfa12a8cfd7b52
#
# IMPORTANT
# ---------
# This is the FIRST successful-new-fit slot authorized for Stage28.
#
# Before fitting, this cell proves that our inherited Stage22 metric and
# threshold mechanics reproduce the historical seed42 RANDOM_NATURAL result.
#
# Then:
#   - train CPU XGBoost C001
#   - reuse exact CPU LightGBM C002
#   - infer on frozen DEVELOPMENT validation only
#   - equal-weight ensemble
#   - select frozen standard/balanced/security thresholds
#   - persist the checkpoint
#   - commit and push immediately
#
# SHARED FINAL HOLDOUT REMAINS COMPLETELY CLOSED.
#
# Expected post-success ledger:
#   new fits consumed = 1
#   new fits remaining = 107
# =================================================================================================

from __future__ import annotations

import base64
import csv
import gc
import hashlib
import json
import math
import os
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    auc,
    roc_auc_score,
)

import xgboost as xgb
import lightgbm as lgb


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "738e3ee29d098d4132828aeaaacfa12a8cfd7b52"
)

COMMIT_MESSAGE = (
    "stage28-2a1: execute random natural seed42 checkpoint"
)

EXPECTED_XGB_VERSION = "3.2.0"
EXPECTED_LGBM_VERSION = "4.6.0"

EXPECTED_COMPONENT_MANIFEST_SHA = (
    "47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505"
)

EXPECTED_XGB_PARAMETER_SHA = (
    "1eb12cc88f3e1b455da9882d8439938fda175c66435940a5e2a6850cbcda63ff"
)

EXPECTED_LGBM_PARAMETER_SHA = (
    "dbb7ac185b3ac6aa8e3ac915cfa2b2080ee54bec8e7a38cab83537faad2f2754"
)

EXPECTED_REUSED_LGBM_SHA = (
    "9a92f4b8cd26738a470a7ff6fb72f51422a803e10b36bb32ac4bfc91373e6e50"
)

EXPECTED_HISTORICAL_VALIDATION_PROB_SHA = (
    "9dc64ccdb6580dad727668ff3e511e0a8572cbb1de49b9ffe8999b0c3d190394"
)

EXPECTED_RANDOM_TRAIN_ROWS = 11_529_922
EXPECTED_RANDOM_TRAIN_ATTACK = 1_577_839
EXPECTED_RANDOM_TRAIN_BENIGN = 9_952_083

EXPECTED_RANDOM_VAL_ROWS = 2_882_481
EXPECTED_RANDOM_VAL_ATTACK = 394_460
EXPECTED_RANDOM_VAL_BENIGN = 2_488_021

EXPECTED_FEATURES = 70

EXPECTED_STAGE28_NEW_FITS = 108


# =================================================================================================
# 1. PATHS
# =================================================================================================

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

STAGE28_1C_FREEZE = (
    STAGE28_ROOT
    / "stage28_1c_model_input_identity_freeze"
    / "stage28_1c_freeze_record.json"
)

STAGE28_1B_DIR = (
    STAGE28_ROOT
    / "stage28_1b_random_loao_membership_and_execution_lock"
)

COMPONENT_MANIFEST_PATH = (
    STAGE28_1B_DIR
    / "stage28_component_execution_manifest.csv"
)

PARAMETER_SETS_PATH = (
    STAGE28_1B_DIR
    / "execution_parameter_sets.json"
)

THRESHOLD_POLICY_PATH = (
    STAGE28_ROOT
    / "stage28_0_protocol_lock"
    / "threshold_policy.json"
)


# -------------------------------------------------------------------------------------------------
# Stage28-2A0 runtime state.
# -------------------------------------------------------------------------------------------------

STAGE28_2A0_RECEIPT = Path(
    "/kaggle/working/"
    "stage28_2a0_stage22_execution_matrix_receipt.json"
)

RUNTIME_ROOT = Path(
    "/kaggle/working/"
    "stage28_2a_runtime_cache/"
    "stage22_full"
)

X_PATH = (
    RUNTIME_ROOT
    / "development_features_float64.npy"
)

Y_PATH = (
    RUNTIME_ROOT
    / "development_binary_labels_uint8.npy"
)

RANDOM_TRAIN_IDX_PATH = (
    RUNTIME_ROOT
    / "random_natural_train_global_idx_int32.npy"
)

RANDOM_VALIDATION_IDX_PATH = (
    RUNTIME_ROOT
    / "random_natural_validation_global_idx_int32.npy"
)


# -------------------------------------------------------------------------------------------------
# Historical seed42 Stage22 artifacts.
# -------------------------------------------------------------------------------------------------

HISTORICAL_RANDOM_DIR = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2a_random_natural"
)

HISTORICAL_RESULT_PATH = (
    HISTORICAL_RANDOM_DIR
    / "stage22r_2a_random_natural_result.json"
)

HISTORICAL_LGBM_MODEL_PATH = (
    HISTORICAL_RANDOM_DIR
    / "random_natural_lightgbm_model.txt"
)

HISTORICAL_VALIDATION_PROB_PATH = (
    HISTORICAL_RANDOM_DIR
    / "random_natural_validation_ensemble_probabilities.npz"
)


# -------------------------------------------------------------------------------------------------
# Stage28-2A1 durable output.
# -------------------------------------------------------------------------------------------------

OUT = (
    STAGE28_ROOT
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a1_random_natural_seed42"
)

XGB_MODEL_PATH = (
    OUT
    / "random_natural_seed42_xgboost_cpu_model.json"
)

VALIDATION_PROB_PATH = (
    OUT
    / "random_natural_seed42_validation_ensemble_probabilities.npz"
)

THRESHOLD_GRID_PATH = (
    OUT
    / "random_natural_seed42_validation_threshold_grid.csv"
)

REUSE_RECEIPT_PATH = (
    OUT
    / "random_natural_seed42_lightgbm_reuse_receipt.json"
)

FIT_LEDGER_PATH = (
    OUT
    / "stage28_2a1_fit_ledger.json"
)

RESULT_PATH = (
    OUT
    / "stage28_2a1_random_natural_seed42_result.json"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


# =================================================================================================
# 2. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return (
        run(
            ["git", *args]
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path,
    obj,
):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def canonical_json_sha256(obj):
    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(",", ":"),
            ensure_ascii=False,
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


def safe_div(a, b):
    return (
        float(a) / float(b)
        if b
        else 0.0
    )


def count_metrics(
    tp,
    fp,
    tn,
    fn,
):
    total = (
        tp
        + fp
        + tn
        + fn
    )

    precision = safe_div(
        tp,
        tp + fp,
    )

    recall = safe_div(
        tp,
        tp + fn,
    )

    fpr = safe_div(
        fp,
        fp + tn,
    )

    return {
        "accuracy":
            safe_div(
                tp + tn,
                total,
            ),

        "precision":
            precision,

        "recall":
            recall,

        "fpr":
            fpr,

        "f1":
            safe_div(
                2 * tp,
                (2 * tp) + fp + fn,
            ),

        "f2":
            safe_div(
                5 * tp,
                (5 * tp) + (4 * fn) + fp,
            ),

        "tp":
            int(tp),

        "fp":
            int(fp),

        "tn":
            int(tn),

        "fn":
            int(fn),
    }


def evaluate_threshold(
    y_true,
    probability_float32,
    integer_percent,
):
    threshold = (
        integer_percent
        / 100.0
    )

    runtime_threshold = np.float32(
        threshold
    )

    pred = (
        probability_float32
        >= runtime_threshold
    )

    positive = (
        y_true
        == 1
    )

    negative = ~positive


    tp = int(
        np.count_nonzero(
            pred & positive
        )
    )

    fp = int(
        np.count_nonzero(
            pred & negative
        )
    )

    fn = int(
        np.count_nonzero(
            (~pred) & positive
        )
    )

    tn = int(
        np.count_nonzero(
            (~pred) & negative
        )
    )


    row = count_metrics(
        tp,
        fp,
        tn,
        fn,
    )

    row.update(
        {
            "threshold_integer_percent":
                int(
                    integer_percent
                ),

            "threshold":
                float(
                    threshold
                ),

            "threshold_float32_runtime":
                float(
                    runtime_threshold
                ),
        }
    )

    return row


def build_grid(
    y_true,
    probability_float32,
):
    return [
        evaluate_threshold(
            y_true,
            probability_float32,
            pct,
        )
        for pct
        in range(
            5,
            96,
        )
    ]


def choose_balanced(grid):
    return max(
        grid,
        key=lambda r: (
            r["f1"],
            -r["fpr"],
            r["recall"],
            -abs(
                r["threshold"]
                - 0.50
            ),
            -r["threshold"],
        ),
    )


def choose_security(grid):
    feasible = [
        r
        for r
        in grid
        if r["fpr"] <= 0.05
    ]

    if not feasible:
        return None

    return max(
        feasible,
        key=lambda r: (
            r["f2"],
            -r["fpr"],
            r["recall"],
            -r["threshold"],
        ),
    )


def assert_close(
    actual,
    expected,
    label,
    atol=1e-12,
):
    if not math.isclose(
        float(actual),
        float(expected),
        rel_tol=0.0,
        abs_tol=atol,
    ):
        raise RuntimeError(
            f"{label} mismatch.\n"
            f"expected={expected!r}\n"
            f"actual={actual!r}"
        )


def assert_operating_point(
    actual,
    expected,
    label,
):
    integer_fields = [
        "threshold_integer_percent",
        "tp",
        "fp",
        "tn",
        "fn",
    ]

    float_fields = [
        "threshold",
        "threshold_float32_runtime",
        "accuracy",
        "precision",
        "recall",
        "fpr",
        "f1",
        "f2",
    ]


    for field in integer_fields:

        if int(
            actual[field]
        ) != int(
            expected[field]
        ):

            raise RuntimeError(
                f"{label}/{field} mismatch.\n"
                f"expected={expected[field]}\n"
                f"actual={actual[field]}"
            )


    for field in float_fields:

        assert_close(
            actual[field],
            expected[field],
            f"{label}/{field}",
        )


def recover_github_token():

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]


    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()

        for label in labels:

            try:
                value = client.get_secret(
                    label
                )

            except Exception:
                value = None

            if (
                isinstance(value, str)
                and value.strip()
            ):
                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass


    for label in labels:

        value = os.environ.get(
            label
        )

        if (
            isinstance(value, str)
            and value.strip()
        ):
            return (
                value.strip(),
                f"environment:{label}",
            )


    raise RuntimeError(
        "GitHub credential unavailable."
    )


def authenticated_push(token):

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode("utf-8")
    ).decode("ascii")


    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )


    if p.returncode != 0:

        raise RuntimeError(
            "Authenticated git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )


    return (
        p.stdout
        + p.stderr
    ).strip()


# =================================================================================================
# 3. DURABLE-PARENT GATE
# =================================================================================================

banner(
    "STAGE28-2A1 — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote = (
    remote_line.split()[0]
)

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    head,
)

print(
    "origin/main    :",
    origin,
)

print(
    "Remote main    :",
    remote,
)

print(
    "Branch         :",
    branch,
)

print(
    "Git clean      :",
    not bool(status),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-2A1 parent mismatch."
    )

if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean:\n"
        + status
    )

if OUT.exists():
    raise RuntimeError(
        "Stage28-2A1 durable output already exists.\n"
        "Do not overwrite a scientific execution checkpoint."
    )


print()
print(
    "[PASS] exact clean Stage28-1C parent"
)


# =================================================================================================
# 4. GITHUB DURABILITY AUTH BEFORE FIT
# =================================================================================================

banner(
    "GITHUB DURABILITY AUTHORIZATION"
)

github_token, token_source = (
    recover_github_token()
)


print(
    "[PASS] credential recovered from",
    token_source,
)

print(
    "[PASS] token intentionally not displayed"
)


remote_url = git(
    "remote",
    "get-url",
    "origin",
)


if (
    remote_url.startswith("https://")
    and "@github.com"
    in remote_url
):
    raise RuntimeError(
        "Remote URL contains embedded credentials."
    )


print(
    "[PASS] clean GitHub remote"
)


# =================================================================================================
# 5. LIBRARY / CPU GATE
# =================================================================================================

banner(
    "FROZEN MODEL RUNTIME GATE"
)

print(
    "XGBoost :",
    xgb.__version__,
)

print(
    "LightGBM:",
    lgb.__version__,
)


if xgb.__version__ != EXPECTED_XGB_VERSION:
    raise RuntimeError(
        "XGBoost version drift."
    )

if lgb.__version__ != EXPECTED_LGBM_VERSION:
    raise RuntimeError(
        "LightGBM version drift."
    )


# Frozen Stage28 policy overrides any generic GPU preference.
os.environ[
    "CUDA_VISIBLE_DEVICES"
] = ""


print(
    "[PASS] Stage28 CPU-only execution policy active"
)


# =================================================================================================
# 6. PRE-FIT SCIENTIFIC LEDGER
# =================================================================================================

banner(
    "PRE-FIT SCIENTIFIC LEDGER"
)

freeze_1c = read_json(
    require_file(
        STAGE28_1C_FREEZE
    )
)

receipt_2a0 = read_json(
    require_file(
        STAGE28_2A0_RECEIPT
    )
)


if (
    freeze_1c[
        "status"
    ]
    != "FROZEN_BEFORE_FIRST_STAGE28_MODEL_FIT"
):
    raise RuntimeError(
        "Stage28-1C freeze status invalid."
    )


if (
    int(
        freeze_1c[
            "execution_manifest"
        ][
            "new_fits_consumed"
        ]
    )
    != 0
):
    raise RuntimeError(
        "Stage28 durable ledger is no longer pre-fit."
    )


if (
    receipt_2a0[
        "status"
    ]
    != "READY_FOR_STAGE28_FIT_001"
):
    raise RuntimeError(
        "Stage28-2A0 is not ready for fit #1."
    )


if (
    receipt_2a0[
        "durable_scientific_parent"
    ]
    != EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-2A0 parent mismatch."
    )


if (
    int(
        receipt_2a0[
            "fit_ledger"
        ][
            "new_consumed"
        ]
    )
    != 0
):
    raise RuntimeError(
        "Stage28-2A0 fit ledger is not zero."
    )


print(
    "Authorized : 108"
)

print(
    "Consumed   : 0"
)

print(
    "Remaining  : 108"
)

print()

print(
    "[PASS] FIT #1 remains unconsumed"
)


# =================================================================================================
# 7. LOAD STAGE22 EXECUTION ASSETS
# =================================================================================================

banner(
    "LOAD STAGE22 RANDOM_NATURAL EXECUTION ASSETS"
)

X = np.load(
    require_file(
        X_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)

y = np.load(
    require_file(
        Y_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)

train_idx = np.load(
    require_file(
        RANDOM_TRAIN_IDX_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)

validation_idx = np.load(
    require_file(
        RANDOM_VALIDATION_IDX_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)


if X.shape != (
    14_412_403,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        "Execution matrix shape mismatch."
    )

if X.dtype != np.float64:
    raise RuntimeError(
        "Execution matrix dtype != float64."
    )

if len(train_idx) != EXPECTED_RANDOM_TRAIN_ROWS:
    raise RuntimeError(
        "Random training membership size mismatch."
    )

if len(validation_idx) != EXPECTED_RANDOM_VAL_ROWS:
    raise RuntimeError(
        "Random validation membership size mismatch."
    )


y_validation = np.asarray(
    y[
        validation_idx
    ],
    dtype=np.uint8,
)


train_attack = int(
    np.asarray(
        y[
            train_idx
        ]
    ).sum()
)

validation_attack = int(
    y_validation.sum()
)


if train_attack != EXPECTED_RANDOM_TRAIN_ATTACK:
    raise RuntimeError(
        "Random training attack count mismatch."
    )

if validation_attack != EXPECTED_RANDOM_VAL_ATTACK:
    raise RuntimeError(
        "Random validation attack count mismatch."
    )


print(
    "TRAIN:",
    f"{len(train_idx):,}",
)

print(
    " attack:",
    f"{train_attack:,}",
)

print(
    " benign:",
    f"{len(train_idx) - train_attack:,}",
)

print()

print(
    "VALIDATION:",
    f"{len(validation_idx):,}",
)

print(
    " attack:",
    f"{validation_attack:,}",
)

print(
    " benign:",
    f"{len(validation_idx) - validation_attack:,}",
)

print()

print(
    "[PASS] exact RANDOM_NATURAL memberships loaded"
)


# =================================================================================================
# 8. C001 / C002 MANIFEST GATE
# =================================================================================================

banner(
    "C001 / C002 FROZEN COMPONENT GATE"
)

manifest_sha = sha256_file(
    require_file(
        COMPONENT_MANIFEST_PATH
    )
)


print(
    "Expected manifest SHA:",
    EXPECTED_COMPONENT_MANIFEST_SHA,
)

print(
    "Actual manifest SHA  :",
    manifest_sha,
)


if manifest_sha != EXPECTED_COMPONENT_MANIFEST_SHA:
    raise RuntimeError(
        "Stage28 component manifest changed."
    )


with COMPONENT_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:

    rows = list(
        csv.DictReader(f)
    )


by_component = {
    row[
        "component_id"
    ]:
    row
    for row
    in rows
}


c001 = by_component["C001"]
c002 = by_component["C002"]


expected_c001 = {
    "arm":
        "28A_TRAINING_SEED_STABILITY",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "RANDOM_NATURAL",

    "evaluation_cell_id":
        "28A_STAGE22::RANDOM_NATURAL::SEED42",

    "learner":
        "XGBOOST",

    "configuration_id":
        "XGB_11",

    "model_seed":
        "42",

    "compute_backend":
        "CPU",

    "parameter_set_id":
        "STAGE22_FULL::XGBOOST::SEED42",

    "parameter_sha256":
        EXPECTED_XGB_PARAMETER_SHA,

    "membership_reference":
        "STAGE22R_FROZEN_RANDOM_NATURAL_MEMBERSHIP",

    "fit_action":
        "NEW_FIT_AUTHORIZED",

    "new_fit_budget_units":
        "1",
}


expected_c002 = {
    "arm":
        "28A_TRAINING_SEED_STABILITY",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "RANDOM_NATURAL",

    "evaluation_cell_id":
        "28A_STAGE22::RANDOM_NATURAL::SEED42",

    "learner":
        "LIGHTGBM",

    "configuration_id":
        "LGBM_11",

    "model_seed":
        "42",

    "compute_backend":
        "CPU",

    "parameter_set_id":
        "STAGE22_FULL::LIGHTGBM::SEED42",

    "parameter_sha256":
        EXPECTED_LGBM_PARAMETER_SHA,

    "membership_reference":
        "STAGE22R_FROZEN_RANDOM_NATURAL_MEMBERSHIP",

    "fit_action":
        "REUSE_EXISTING",

    "reused_model_sha256":
        EXPECTED_REUSED_LGBM_SHA,

    "reuse_budget_units":
        "1",
}


for field, expected in expected_c001.items():

    if c001[field] != expected:

        raise RuntimeError(
            f"C001 {field} mismatch.\n"
            f"expected={expected}\n"
            f"actual={c001[field]}"
        )


for field, expected in expected_c002.items():

    if c002[field] != expected:

        raise RuntimeError(
            f"C002 {field} mismatch.\n"
            f"expected={expected}\n"
            f"actual={c002[field]}"
        )


print(
    "[PASS] C001 = CPU XGBoost seed42 NEW FIT"
)

print(
    "[PASS] C002 = CPU LightGBM seed42 REUSE"
)


# =================================================================================================
# 9. PARAMETER SET GATE
# =================================================================================================

banner(
    "FROZEN PARAMETER-SET GATE"
)

parameter_doc = read_json(
    require_file(
        PARAMETER_SETS_PATH
    )
)

parameter_sets = (
    parameter_doc[
        "sets"
    ]
)


xgb_record = parameter_sets[
    c001[
        "parameter_set_id"
    ]
]

lgbm_record = parameter_sets[
    c002[
        "parameter_set_id"
    ]
]


xgb_params = dict(
    xgb_record[
        "parameters"
    ]
)

lgbm_params = dict(
    lgbm_record[
        "parameters"
    ]
)


xgb_param_sha = canonical_json_sha256(
    xgb_params
)

lgbm_param_sha = canonical_json_sha256(
    lgbm_params
)


print(
    "XGB SHA expected:",
    EXPECTED_XGB_PARAMETER_SHA,
)

print(
    "XGB SHA actual  :",
    xgb_param_sha,
)

print()

print(
    "LGBM SHA expected:",
    EXPECTED_LGBM_PARAMETER_SHA,
)

print(
    "LGBM SHA actual  :",
    lgbm_param_sha,
)


if xgb_param_sha != EXPECTED_XGB_PARAMETER_SHA:
    raise RuntimeError(
        "XGBoost parameter-set SHA mismatch."
    )

if lgbm_param_sha != EXPECTED_LGBM_PARAMETER_SHA:
    raise RuntimeError(
        "LightGBM parameter-set SHA mismatch."
    )


if (
    xgb_params[
        "device"
    ]
    != "cpu"
    or xgb_params[
        "tree_method"
    ]
    != "hist"
):
    raise RuntimeError(
        "C001 is not frozen to CPU/hist."
    )


if (
    lgbm_params[
        "device_type"
    ]
    != "cpu"
):
    raise RuntimeError(
        "C002 is not frozen to CPU."
    )


print()
print(
    "[PASS] exact frozen CPU parameter sets"
)


# =================================================================================================
# 10. C002 HISTORICAL LIGHTGBM REUSE GATE
# =================================================================================================

banner(
    "C002 — BYTE-EXACT HISTORICAL LIGHTGBM REUSE"
)

require_file(
    HISTORICAL_LGBM_MODEL_PATH
)

lgbm_model_sha = sha256_file(
    HISTORICAL_LGBM_MODEL_PATH
)


print(
    "Expected:",
    EXPECTED_REUSED_LGBM_SHA,
)

print(
    "Actual  :",
    lgbm_model_sha,
)


if lgbm_model_sha != EXPECTED_REUSED_LGBM_SHA:
    raise RuntimeError(
        "Historical LightGBM model SHA mismatch."
    )


lgb_booster = lgb.Booster(
    model_file=str(
        HISTORICAL_LGBM_MODEL_PATH
    )
)


lgb_iterations = int(
    lgb_booster.current_iteration()
)

lgb_trees = int(
    lgb_booster.num_trees()
)

lgb_features = int(
    lgb_booster.num_feature()
)


print(
    "Iterations:",
    lgb_iterations,
)

print(
    "Trees     :",
    lgb_trees,
)

print(
    "Features  :",
    lgb_features,
)


if lgb_iterations != 400:
    raise RuntimeError(
        "Historical LightGBM iteration count != 400."
    )

if lgb_trees != 400:
    raise RuntimeError(
        "Historical LightGBM tree count != 400."
    )

if lgb_features != EXPECTED_FEATURES:
    raise RuntimeError(
        "Historical LightGBM feature count != 70."
    )


print()
print(
    "[PASS] exact historical CPU LightGBM loaded"
)

print(
    "[PASS] reuse consumes zero new-fit budget"
)


# =================================================================================================
# 11. THRESHOLD POLICY GATE
# =================================================================================================

banner(
    "FROZEN STAGE22 THRESHOLD POLICY"
)

threshold_policy = read_json(
    require_file(
        THRESHOLD_POLICY_PATH
    )
)


policy = (
    threshold_policy[
        "stage22_full"
    ]
)


if (
    policy[
        "selection_population"
    ]
    != "EACH_PARENT_CELL_FROZEN_DEVELOPMENT_VALIDATION"
):
    raise RuntimeError(
        "Stage22 threshold-selection population changed."
    )


if not (
    policy[
        "grid"
    ][
        "integer_percent_start"
    ] == 5
    and
    policy[
        "grid"
    ][
        "integer_percent_stop_inclusive"
    ] == 95
    and
    policy[
        "grid"
    ][
        "integer_step"
    ] == 1
    and
    policy[
        "grid"
    ][
        "count"
    ] == 91
):
    raise RuntimeError(
        "Stage22 threshold grid changed."
    )


if (
    policy[
        "final_holdout_threshold_search"
    ]
    != "FORBIDDEN"
):
    raise RuntimeError(
        "Final-holdout threshold rule changed."
    )


if (
    policy[
        "balanced"
    ][
        "tie_break_order"
    ]
    != [
        "LOWER_FPR",
        "HIGHER_RECALL",
        "CLOSER_TO_0_50",
        "LOWER_THRESHOLD",
    ]
):
    raise RuntimeError(
        "Balanced tie-break order changed."
    )


if (
    policy[
        "security"
    ][
        "tie_break_order"
    ]
    != [
        "LOWER_FPR",
        "HIGHER_RECALL",
        "LOWER_THRESHOLD",
    ]
):
    raise RuntimeError(
        "Security tie-break order changed."
    )


print(
    "[PASS] exact Stage22 threshold semantics frozen"
)


# =================================================================================================
# 12. HISTORICAL METRIC / THRESHOLD SELF-TEST
#
# Must pass BEFORE FIT #1.
# =================================================================================================

banner(
    "HISTORICAL STAGE22 RANDOM_NATURAL SELF-TEST BEFORE FIT #1"
)

historical_result = read_json(
    require_file(
        HISTORICAL_RESULT_PATH
    )
)


require_file(
    HISTORICAL_VALIDATION_PROB_PATH
)

historical_probability_file_sha = sha256_file(
    HISTORICAL_VALIDATION_PROB_PATH
)


print(
    "Historical NPZ expected SHA:",
    EXPECTED_HISTORICAL_VALIDATION_PROB_SHA,
)

print(
    "Historical NPZ actual SHA  :",
    historical_probability_file_sha,
)


if (
    historical_probability_file_sha
    != EXPECTED_HISTORICAL_VALIDATION_PROB_SHA
):
    raise RuntimeError(
        "Historical probability artifact SHA mismatch."
    )


historical_expected_roc = float(
    historical_result[
        "validation_probability"
    ][
        "roc_auc"
    ]
)

historical_expected_pr = float(
    historical_result[
        "validation_probability"
    ][
        "pr_auc"
    ]
)


historical_candidates = []


with np.load(
    HISTORICAL_VALIDATION_PROB_PATH,
    allow_pickle=False,
) as npz:

    print(
        "NPZ keys:",
        list(npz.files),
    )


    for key in npz.files:

        arr = npz[key]


        if not (
            arr.ndim == 1
            and arr.shape[0]
            == EXPECTED_RANDOM_VAL_ROWS
            and np.issubdtype(
                arr.dtype,
                np.floating,
            )
        ):
            continue


        p = np.asarray(
            arr,
            dtype=np.float32,
        )


        roc_value = float(
            roc_auc_score(
                y_validation,
                p,
            )
        )


        ap_value = float(
            average_precision_score(
                y_validation,
                p,
            )
        )


        precision_curve, recall_curve, _ = (
            precision_recall_curve(
                y_validation,
                p,
            )
        )


        trapezoidal_value = float(
            auc(
                recall_curve,
                precision_curve,
            )
        )


        roc_match = math.isclose(
            roc_value,
            historical_expected_roc,
            rel_tol=0.0,
            abs_tol=1e-12,
        )

        ap_match = math.isclose(
            ap_value,
            historical_expected_pr,
            rel_tol=0.0,
            abs_tol=1e-12,
        )

        trap_match = math.isclose(
            trapezoidal_value,
            historical_expected_pr,
            rel_tol=0.0,
            abs_tol=1e-12,
        )


        print()
        print(
            "Candidate:",
            key,
        )

        print(
            " ROC-AUC        :",
            roc_value,
        )

        print(
            " AveragePrecision:",
            ap_value,
        )

        print(
            " Trapezoidal PR :",
            trapezoidal_value,
        )

        print(
            " ROC match       :",
            roc_match,
        )

        print(
            " AP match        :",
            ap_match,
        )

        print(
            " Trap match      :",
            trap_match,
        )


        if (
            roc_match
            and (
                ap_match
                or trap_match
            )
        ):

            historical_candidates.append(
                {
                    "key":
                        key,

                    "probability":
                        p,

                    "roc":
                        roc_value,

                    "ap":
                        ap_value,

                    "trapezoidal":
                        trapezoidal_value,

                    "ap_match":
                        ap_match,

                    "trap_match":
                        trap_match,
                }
            )


if len(
    historical_candidates
) != 1:
    raise RuntimeError(
        "Could not uniquely resolve the historical "
        "Stage22 ensemble probability vector."
    )


historical_match = (
    historical_candidates[
        0
    ]
)

historical_probability_key = (
    historical_match[
        "key"
    ]
)

historical_probability = (
    historical_match[
        "probability"
    ]
)


if (
    historical_match[
        "ap_match"
    ]
    and
    not historical_match[
        "trap_match"
    ]
):

    PR_AUC_DEFINITION = (
        "SKLEARN_AVERAGE_PRECISION_SCORE"
    )

elif (
    historical_match[
        "trap_match"
    ]
    and
    not historical_match[
        "ap_match"
    ]
):

    PR_AUC_DEFINITION = (
        "SKLEARN_PRECISION_RECALL_CURVE_TRAPEZOIDAL_AUC"
    )

elif (
    historical_match[
        "ap_match"
    ]
    and
    historical_match[
        "trap_match"
    ]
):

    PR_AUC_DEFINITION = (
        "AP_AND_TRAPEZOIDAL_IDENTICAL_ON_PARENT_VECTOR;"
        "USE_SKLEARN_AVERAGE_PRECISION_SCORE"
    )

else:

    raise RuntimeError(
        "Unable to resolve historical PR-AUC definition."
    )


historical_grid = build_grid(
    y_validation,
    historical_probability,
)

historical_standard = next(
    r
    for r
    in historical_grid
    if r[
        "threshold_integer_percent"
    ] == 50
)

historical_balanced = choose_balanced(
    historical_grid
)

historical_security = choose_security(
    historical_grid
)


if historical_security is None:
    raise RuntimeError(
        "Historical security threshold unexpectedly infeasible."
    )


assert_operating_point(
    historical_standard,
    historical_result[
        "operating_points"
    ][
        "standard"
    ],
    "HISTORICAL_STANDARD",
)

assert_operating_point(
    historical_balanced,
    historical_result[
        "operating_points"
    ][
        "balanced"
    ],
    "HISTORICAL_BALANCED",
)

assert_operating_point(
    historical_security,
    historical_result[
        "operating_points"
    ][
        "security"
    ][
        "result"
    ],
    "HISTORICAL_SECURITY",
)


if (
    historical_standard[
        "threshold"
    ]
    != 0.50
):
    raise RuntimeError(
        "Historical standard threshold != 0.50."
    )

if (
    historical_balanced[
        "threshold"
    ]
    != 0.46
):
    raise RuntimeError(
        "Historical balanced threshold != 0.46."
    )

if (
    historical_security[
        "threshold"
    ]
    != 0.10
):
    raise RuntimeError(
        "Historical security threshold != 0.10."
    )


print()
print(
    "[PASS] historical ROC-AUC reproduced:",
    historical_match[
        "roc"
    ],
)

print(
    "[PASS] historical PR-AUC reproduced:",
    historical_expected_pr,
)

print(
    "[PASS] PR-AUC semantics:",
    PR_AUC_DEFINITION,
)

print(
    "[PASS] STANDARD = 0.50"
)

print(
    "[PASS] BALANCED = 0.46"
)

print(
    "[PASS] SECURITY = 0.10"
)

print()

print(
    "All historical evaluation semantics reproduced."
)

print(
    "FIT #1 is now authorized."
)


del historical_probability
del historical_match
del historical_candidates
del historical_grid

gc.collect()


# =================================================================================================
# 13. CREATE DURABLE OUTPUT DIRECTORY
# =================================================================================================

OUT.mkdir(
    parents=True,
    exist_ok=False,
)


# =================================================================================================
# 14. MATERIALIZE EXACT RANDOM_NATURAL TRAINING POPULATION
# =================================================================================================

banner(
    "MATERIALIZE RANDOM_NATURAL TRAINING MATRIX"
)

print(
    "Rows    :",
    f"{EXPECTED_RANDOM_TRAIN_ROWS:,}",
)

print(
    "Features:",
    EXPECTED_FEATURES,
)

print(
    "Dtype   : float64"
)

print()

print(
    "Materializing exact inherited training membership..."
)


materialize_start = time.perf_counter()


X_train = np.asarray(
    X[
        train_idx,
        :
    ],
    dtype=np.float64,
    order="C",
)

y_train = np.asarray(
    y[
        train_idx
    ],
    dtype=np.uint8,
    order="C",
)


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


if X_train.shape != (
    EXPECTED_RANDOM_TRAIN_ROWS,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        "X_train shape mismatch."
    )

if y_train.shape != (
    EXPECTED_RANDOM_TRAIN_ROWS,
):
    raise RuntimeError(
        "y_train shape mismatch."
    )

if int(
    y_train.sum()
) != EXPECTED_RANDOM_TRAIN_ATTACK:
    raise RuntimeError(
        "y_train attack count mismatch."
    )

if np.isinf(
    X_train
).any():
    raise RuntimeError(
        "Infinity exists in training matrix."
    )


print(
    "Materialization seconds:",
    materialize_seconds,
)

print(
    "Matrix payload:",
    f"{X_train.nbytes / (1024**3):.3f} GiB",
)

print()

print(
    "[PASS] exact training matrix ready"
)


# =================================================================================================
# 15. FIT #1 — C001 CPU XGBOOST
# =================================================================================================

banner(
    "FIT #1 — C001 XGBOOST XGB_11 SEED42 CPU"
)

print(
    "Component   : C001"
)

print(
    "Experiment  : STAGE22_FULL"
)

print(
    "Unit        : RANDOM_NATURAL"
)

print(
    "Seed        : 42"
)

print(
    "Backend     : CPU"
)

print(
    "tree_method :",
    xgb_params[
        "tree_method"
    ],
)

print(
    "device      :",
    xgb_params[
        "device"
    ],
)

print(
    "Estimators  :",
    xgb_params[
        "n_estimators"
    ],
)

print()

print(
    "Starting Stage28 scientific FIT #1..."
)


fit_start = time.perf_counter()


xgb_model = xgb.XGBClassifier(
    **xgb_params
)


xgb_model.fit(
    X_train,
    y_train,
)


fit_seconds = (
    time.perf_counter()
    - fit_start
)


booster = (
    xgb_model.get_booster()
)

boosted_rounds = int(
    booster.num_boosted_rounds()
)


print()
print(
    "FIT #1 completed."
)

print(
    "Fit seconds:",
    fit_seconds,
)

print(
    "Boosted rounds:",
    boosted_rounds,
)


if boosted_rounds != 400:
    raise RuntimeError(
        "XGBoost boosted-round count != 400."
    )

if int(
    xgb_model.n_features_in_
) != EXPECTED_FEATURES:
    raise RuntimeError(
        "XGBoost model feature count != 70."
    )


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_model_sha = sha256_file(
    XGB_MODEL_PATH
)


print(
    "Model SHA256:",
    xgb_model_sha,
)

print()

print(
    "[PASS] C001 FIT #1 successful"
)


# Training material no longer required.
del X_train
del y_train

gc.collect()


# =================================================================================================
# 16. DEVELOPMENT VALIDATION INFERENCE
# =================================================================================================

banner(
    "RANDOM_NATURAL SEED42 — DEVELOPMENT VALIDATION INFERENCE"
)

n_validation = len(
    validation_idx
)


ensemble_probability = np.empty(
    n_validation,
    dtype=np.float32,
)


chunk_size = 65_536

inference_start = time.perf_counter()


for start in range(
    0,
    n_validation,
    chunk_size,
):

    stop = min(
        start
        + chunk_size,
        n_validation,
    )


    idx_chunk = np.asarray(
        validation_idx[
            start:stop
        ],
        dtype=np.int64,
    )


    X_chunk = np.asarray(
        X[
            idx_chunk,
            :
        ],
        dtype=np.float64,
        order="C",
    )


    p_xgb = np.asarray(
        xgb_model.predict_proba(
            X_chunk
        )[
            :,
            1
        ],
        dtype=np.float64,
    )


    p_lgbm = np.asarray(
        lgb_booster.predict(
            X_chunk,
            num_iteration=lgb_iterations,
        ),
        dtype=np.float64,
    )


    expected_chunk = (
        stop
        - start
    )


    if len(p_xgb) != expected_chunk:
        raise RuntimeError(
            "XGBoost inference length mismatch."
        )

    if len(p_lgbm) != expected_chunk:
        raise RuntimeError(
            "LightGBM inference length mismatch."
        )


    # Exact inherited Stage22 ensemble semantics:
    #
    #   component combination dtype = float64
    #   ensemble persisted dtype     = float32

    combined64 = (
        (0.5 * p_lgbm)
        +
        (0.5 * p_xgb)
    )


    ensemble_probability[
        start:stop
    ] = combined64.astype(
        np.float32,
        copy=False,
    )


    del idx_chunk
    del X_chunk
    del p_xgb
    del p_lgbm
    del combined64


    if (
        start == 0
        or stop == n_validation
        or (
            stop
            // chunk_size
        ) % 10 == 0
    ):
        print(
            f"  inferred "
            f"{stop:,} / {n_validation:,}"
        )


    gc.collect()


inference_seconds = (
    time.perf_counter()
    - inference_start
)


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):
    raise RuntimeError(
        "Non-finite ensemble probability detected."
    )


if (
    np.any(
        ensemble_probability < 0
    )
    or
    np.any(
        ensemble_probability > 1
    )
):
    raise RuntimeError(
        "Ensemble probability outside [0,1]."
    )


print()
print(
    "Inference seconds:",
    inference_seconds,
)

print()

print(
    "[PASS] C001 validation inference complete"
)

print(
    "[PASS] C002 reused-model validation inference complete"
)

print(
    "[PASS] equal-weight float32 ensemble complete"
)


# =================================================================================================
# 17. VALIDATION RANKING METRICS
# =================================================================================================

banner(
    "RANDOM_NATURAL SEED42 — VALIDATION RANKING METRICS"
)

validation_roc_auc = float(
    roc_auc_score(
        y_validation,
        ensemble_probability,
    )
)


if (
    PR_AUC_DEFINITION
    == "SKLEARN_PRECISION_RECALL_CURVE_TRAPEZOIDAL_AUC"
):

    precision_curve, recall_curve, _ = (
        precision_recall_curve(
            y_validation,
            ensemble_probability,
        )
    )

    validation_pr_auc = float(
        auc(
            recall_curve,
            precision_curve,
        )
    )

else:

    validation_pr_auc = float(
        average_precision_score(
            y_validation,
            ensemble_probability,
        )
    )


print(
    "ROC-AUC:",
    validation_roc_auc,
)

print(
    "PR-AUC :",
    validation_pr_auc,
)

print(
    "PR definition:",
    PR_AUC_DEFINITION,
)


# =================================================================================================
# 18. FROZEN THRESHOLD SELECTION
# =================================================================================================

banner(
    "RANDOM_NATURAL SEED42 — FROZEN VALIDATION THRESHOLDS"
)

grid_rows = build_grid(
    y_validation,
    ensemble_probability,
)


standard = next(
    r
    for r
    in grid_rows
    if r[
        "threshold_integer_percent"
    ] == 50
)

balanced = choose_balanced(
    grid_rows
)

security = choose_security(
    grid_rows
)


if security is None:
    raise RuntimeError(
        "Security threshold infeasible under frozen FPR<=0.05 rule."
    )


print(
    "STANDARD:"
)

print(
    " threshold:",
    standard[
        "threshold"
    ],
)

print(
    " F1       :",
    standard[
        "f1"
    ],
)

print(
    " recall   :",
    standard[
        "recall"
    ],
)

print(
    " FPR      :",
    standard[
        "fpr"
    ],
)

print()

print(
    "BALANCED:"
)

print(
    " threshold:",
    balanced[
        "threshold"
    ],
)

print(
    " F1       :",
    balanced[
        "f1"
    ],
)

print(
    " recall   :",
    balanced[
        "recall"
    ],
)

print(
    " FPR      :",
    balanced[
        "fpr"
    ],
)

print()

print(
    "SECURITY:"
)

print(
    " threshold:",
    security[
        "threshold"
    ],
)

print(
    " F2       :",
    security[
        "f2"
    ],
)

print(
    " recall   :",
    security[
        "recall"
    ],
)

print(
    " FPR      :",
    security[
        "fpr"
    ],
)


# =================================================================================================
# 19. WRITE VALIDATION PROBABILITY + THRESHOLD GRID
# =================================================================================================

banner(
    "FREEZE STAGE28-2A1 VALIDATION ARTIFACTS"
)

np.savez_compressed(
    VALIDATION_PROB_PATH,

    ensemble_probability_float32=
        ensemble_probability,

    validation_global_idx_int32=
        np.asarray(
            validation_idx,
            dtype=np.int32,
        ),

    binary_label_uint8=
        y_validation,
)


grid_df = pd.DataFrame(
    grid_rows
)[
    [
        "threshold_integer_percent",
        "threshold",
        "threshold_float32_runtime",
        "accuracy",
        "precision",
        "recall",
        "fpr",
        "f1",
        "f2",
        "tp",
        "fp",
        "tn",
        "fn",
    ]
]


grid_df.to_csv(
    THRESHOLD_GRID_PATH,
    index=False,
    lineterminator="\n",
)


validation_probability_sha = sha256_file(
    VALIDATION_PROB_PATH
)

threshold_grid_sha = sha256_file(
    THRESHOLD_GRID_PATH
)


print(
    "Validation probability SHA:",
    validation_probability_sha,
)

print(
    "Threshold grid SHA       :",
    threshold_grid_sha,
)


# =================================================================================================
# 20. LIGHTGBM REUSE RECEIPT
# =================================================================================================

reuse_receipt = {

    "stage":
        "Stage28-2A1",

    "component_id":
        "C002",

    "evaluation_cell_id":
        "28A_STAGE22::RANDOM_NATURAL::SEED42",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "RANDOM_NATURAL",

    "learner":
        "LIGHTGBM",

    "configuration_id":
        "LGBM_11",

    "seed":
        42,

    "compute_backend":
        "CPU",

    "fit_action":
        "REUSE_EXISTING",

    "historical_model": {
        "path":
            str(
                HISTORICAL_LGBM_MODEL_PATH.relative_to(
                    REPO
                )
            ),

        "sha256":
            lgbm_model_sha,

        "expected_sha256":
            EXPECTED_REUSED_LGBM_SHA,

        "iterations":
            lgb_iterations,

        "trees":
            lgb_trees,

        "features":
            lgb_features,

        "library_version":
            lgb.__version__,
    },

    "parameter_set_id":
        c002[
            "parameter_set_id"
        ],

    "parameter_sha256":
        lgbm_param_sha,

    "new_fit_budget_consumed":
        0,

    "validation_inference_performed":
        True,

    "status":
        "EXACT_HISTORICAL_CPU_LIGHTGBM_REUSED",
}


write_json(
    REUSE_RECEIPT_PATH,
    reuse_receipt,
)


reuse_receipt_sha = sha256_file(
    REUSE_RECEIPT_PATH
)


# =================================================================================================
# 21. FIT LEDGER
# =================================================================================================

fit_ledger = {

    "stage":
        "Stage28-2A1",

    "durable_parent":
        EXPECTED_PARENT,

    "evaluation_cell_id":
        "28A_STAGE22::RANDOM_NATURAL::SEED42",

    "authorized_stage28_new_fit_budget":
        108,

    "pre_cell_new_fits_consumed":
        0,

    "this_cell": {
        "successful_new_fits":
            1,

        "successful_reused_components":
            1,

        "new_fit_components": [
            "C001",
        ],

        "reused_components": [
            "C002",
        ],
    },

    "cumulative_new_fits_consumed":
        1,

    "new_fits_remaining":
        107,

    "stage22_new_fits_consumed":
        1,

    "stage22_new_fits_remaining":
        17,

    "model_fits_attempted":
        1,

    "model_fits_successful":
        1,

    "status":
        "FIT_001_SUCCESSFULLY_CONSUMED",
}


write_json(
    FIT_LEDGER_PATH,
    fit_ledger,
)


fit_ledger_sha = sha256_file(
    FIT_LEDGER_PATH
)


# =================================================================================================
# 22. RESULT RECEIPT
# =================================================================================================

result = {

    "stage":
        "Stage28-2A1",

    "status":
        "RANDOM_NATURAL_SEED42_VALIDATION_CHECKPOINT_FROZEN",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_PARENT,

    "arm":
        "28A_TRAINING_SEED_STABILITY",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "RANDOM_NATURAL",

    "training_seed":
        42,

    "evaluation_cell_id":
        "28A_STAGE22::RANDOM_NATURAL::SEED42",

    "membership": {
        "train_rows":
            EXPECTED_RANDOM_TRAIN_ROWS,

        "train_attack":
            EXPECTED_RANDOM_TRAIN_ATTACK,

        "train_benign":
            EXPECTED_RANDOM_TRAIN_BENIGN,

        "validation_rows":
            EXPECTED_RANDOM_VAL_ROWS,

        "validation_attack":
            EXPECTED_RANDOM_VAL_ATTACK,

        "validation_benign":
            EXPECTED_RANDOM_VAL_BENIGN,

        "membership_changed_by_training_seed":
            False,
    },

    "models": {
        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "xgboost": {
            "component_id":
                "C001",

            "fit_action":
                "NEW_FIT_AUTHORIZED",

            "configuration":
                "XGB_11",

            "seed":
                42,

            "backend":
                "cpu",

            "parameters":
                xgb_params,

            "parameter_sha256":
                xgb_param_sha,

            "library_version":
                xgb.__version__,

            "boosted_rounds":
                boosted_rounds,

            "fit_seconds":
                float(
                    fit_seconds
                ),

            "training_materialization_seconds":
                float(
                    materialize_seconds
                ),

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                xgb_model_sha,
        },

        "lightgbm": {
            "component_id":
                "C002",

            "fit_action":
                "REUSE_EXISTING",

            "configuration":
                "LGBM_11",

            "seed":
                42,

            "backend":
                "cpu",

            "parameter_sha256":
                lgbm_param_sha,

            "historical_model_path":
                str(
                    HISTORICAL_LGBM_MODEL_PATH.relative_to(
                        REPO
                    )
                ),

            "historical_model_sha256":
                lgbm_model_sha,

            "library_version":
                lgb.__version__,

            "iterations":
                lgb_iterations,

            "trees":
                lgb_trees,

            "new_fit_budget_consumed":
                0,
        },
    },

    "historical_semantics_self_test": {
        "historical_probability_artifact":
            str(
                HISTORICAL_VALIDATION_PROB_PATH.relative_to(
                    REPO
                )
            ),

        "historical_probability_sha256":
            historical_probability_file_sha,

        "historical_probability_key":
            historical_probability_key,

        "historical_roc_auc":
            historical_expected_roc,

        "historical_pr_auc":
            historical_expected_pr,

        "pr_auc_definition":
            PR_AUC_DEFINITION,

        "historical_standard_threshold":
            0.50,

        "historical_balanced_threshold":
            0.46,

        "historical_security_threshold":
            0.10,

        "status":
            "EXACT_REPRODUCTION",
    },

    "validation_probability": {
        "rows":
            EXPECTED_RANDOM_VAL_ROWS,

        "roc_auc":
            validation_roc_auc,

        "pr_auc":
            validation_pr_auc,

        "pr_auc_definition":
            PR_AUC_DEFINITION,

        "artifact":
            VALIDATION_PROB_PATH.name,

        "artifact_sha256":
            validation_probability_sha,

        "inference_seconds":
            float(
                inference_seconds
            ),
    },

    "threshold_selection": {
        "selection_population":
            "FROZEN_RANDOM_NATURAL_DEVELOPMENT_VALIDATION",

        "final_holdout_threshold_search":
            "FORBIDDEN",

        "prediction_rule":
            (
                "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_"
                "PROBABILITY_GTE_THRESHOLD"
            ),

        "grid_integer_percent": [
            5,
            95,
        ],

        "grid_points":
            91,

        "balanced":
            (
                "MAX_F1; TIES LOWER_FPR, HIGHER_RECALL, "
                "CLOSER_TO_0_50, LOWER_THRESHOLD"
            ),

        "security":
            (
                "FPR<=0.05 EXACT; MAX_F2; TIES LOWER_FPR, "
                "HIGHER_RECALL, LOWER_THRESHOLD; NO_RELAXATION"
            ),

        "grid_artifact":
            THRESHOLD_GRID_PATH.name,

        "grid_sha256":
            threshold_grid_sha,
    },

    "operating_points": {
        "standard":
            standard,

        "balanced":
            balanced,

        "security": {
            "status":
                "AVAILABLE",

            "result":
                security,
        },
    },

    "scientific_accounting": {
        "new_model_fits_this_cell":
            1,

        "new_model_fits_cumulative":
            1,

        "new_model_fits_remaining":
            107,

        "existing_models_reused_this_cell":
            1,

        "development_validation_model_inference":
            True,

        "threshold_selection_completed":
            True,

        "shared_final_holdout_openings":
            0,

        "shared_final_holdout_predictor_rows_read":
            0,

        "shared_final_holdout_labels_read":
            0,

        "new_target_openings":
            0,

        "target_adaptive_choices":
            0,
    },

    "next_authorized_step":
        (
            "Stage28-2A2 — CHRONOLOGICAL_NATURAL seed42. "
            "FIT #2 = C011 CPU XGBoost seed42. "
            "C012 = exact historical CPU LightGBM reuse. "
            "Shared final holdout remains closed."
        ),
}


write_json(
    RESULT_PATH,
    result,
)


result_sha = sha256_file(
    RESULT_PATH
)


# =================================================================================================
# 23. CHECKSUM MANIFEST
# =================================================================================================

artifact_paths = [
    XGB_MODEL_PATH,
    VALIDATION_PROB_PATH,
    THRESHOLD_GRID_PATH,
    REUSE_RECEIPT_PATH,
    FIT_LEDGER_PATH,
    RESULT_PATH,
]


checksum_lines = []


for artifact in artifact_paths:

    checksum_lines.append(
        f"{sha256_file(artifact)}  {artifact.name}"
    )


CHECKSUMS_PATH.write_text(
    "\n".join(
        checksum_lines
    )
    + "\n",
    encoding="utf-8",
)


print()
print(
    "Durable artifact checksums:"
)

print()

print(
    CHECKSUMS_PATH.read_text(
        encoding="utf-8"
    ).rstrip()
)


# =================================================================================================
# 24. SELF-VERIFY ARTIFACT UNIVERSE
# =================================================================================================

banner(
    "SELF-VERIFY STAGE28-2A1 ARTIFACTS"
)

expected_files = {
    XGB_MODEL_PATH.name,
    VALIDATION_PROB_PATH.name,
    THRESHOLD_GRID_PATH.name,
    REUSE_RECEIPT_PATH.name,
    FIT_LEDGER_PATH.name,
    RESULT_PATH.name,
    CHECKSUMS_PATH.name,
}


actual_files = {
    p.name
    for p
    in OUT.iterdir()
    if p.is_file()
}


if actual_files != expected_files:

    raise RuntimeError(
        "Unexpected Stage28-2A1 file universe.\n"
        f"Expected={sorted(expected_files)}\n"
        f"Actual={sorted(actual_files)}"
    )


for line in (
    CHECKSUMS_PATH.read_text(
        encoding="utf-8"
    ).splitlines()
):

    digest, filename = line.split(
        None,
        1,
    )

    filename = (
        filename.strip()
        .lstrip("*")
    )

    actual = sha256_file(
        OUT
        / filename
    )


    if actual != digest:

        raise RuntimeError(
            f"Checksum mismatch: {filename}"
        )


    print(
        "[PASS]",
        filename,
        digest,
    )


print()
print(
    "[PASS] all Stage28-2A1 artifacts self-verified"
)


# =================================================================================================
# 25. SCIENTIFIC LEDGER BEFORE COMMIT
# =================================================================================================

banner(
    "STAGE28-2A1 SCIENTIFIC LEDGER"
)

print(
    "NEW fits authorized : 108"
)

print(
    "Pre-cell consumed   : 0"
)

print(
    "This cell consumed  : 1"
)

print(
    "Cumulative consumed : 1"
)

print(
    "Remaining           : 107"
)

print()

print(
    "C001 XGBoost fit     : SUCCESS"
)

print(
    "C002 LightGBM reuse  : EXACT"
)

print(
    "Validation inference : COMPLETE"
)

print(
    "Threshold selection : COMPLETE"
)

print(
    "Final holdout opening: 0"
)

print()

print(
    "[PASS] scientific ledger coherent"
)


# =================================================================================================
# 26. AUTHORIZED GIT CHANGE GATE
# =================================================================================================

banner(
    "AUTHORIZED GIT-CHANGE GATE"
)

tracked_changes = (
    git(
        "diff",
        "--name-only",
    )
    .splitlines()
)


if tracked_changes:

    raise RuntimeError(
        "Unexpected tracked changes:\n"
        + "\n".join(
            tracked_changes
        )
    )


expected_untracked = {
    str(
        (
            OUT
            / filename
        ).relative_to(
            REPO
        )
    )
    for filename
    in expected_files
}


all_untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    )
    .splitlines()
)


if all_untracked != expected_untracked:

    raise RuntimeError(
        "Unexpected untracked files.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(expected_untracked)
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(all_untracked)
        )
    )


print(
    "[PASS] exactly seven authorized files exist"
)


# =================================================================================================
# 27. STAGE EXACT FILES
# =================================================================================================

banner(
    "STAGE STAGE28-2A1 CHECKPOINT"
)

for relative in sorted(
    expected_untracked
):

    run(
        [
            "git",
            "add",
            "--",
            relative,
        ]
    )


staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    )
    .splitlines()
)


if staged != expected_untracked:

    raise RuntimeError(
        "Staged file universe mismatch."
    )


for relative in sorted(staged):
    print(
        " ",
        relative,
    )


print()
print(
    "[PASS] exactly seven files staged"
)


# =================================================================================================
# 28. COMMIT
# =================================================================================================

banner(
    "COMMIT STAGE28-2A1"
)

if not git(
    "config",
    "--get",
    "user.name",
):

    run(
        [
            "git",
            "config",
            "user.name",
            "Stage28 Kaggle",
        ]
    )


if not git(
    "config",
    "--get",
    "user.email",
):

    run(
        [
            "git",
            "config",
            "user.email",
            "stage28-kaggle@users.noreply.github.com",
        ]
    )


commit = run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ]
)


print(
    commit.stdout.strip()
)


new_head = git(
    "rev-parse",
    "HEAD",
)

new_parent = git(
    "rev-parse",
    "HEAD^",
)

new_subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)


print()
print(
    "Parent :",
    new_parent,
)

print(
    "Commit :",
    new_head,
)

print(
    "Subject:",
    new_subject,
)


if new_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage28-2A1 commit parent mismatch."
    )


if new_subject != COMMIT_MESSAGE:

    raise RuntimeError(
        "Stage28-2A1 commit subject mismatch."
    )


print()
print(
    "[PASS] Stage28-2A1 committed"
)


# =================================================================================================
# 29. PUSH
# =================================================================================================

banner(
    "PUSH STAGE28-2A1"
)

push_output = authenticated_push(
    github_token
)


if push_output:
    print(
        push_output
    )


github_token = None


# =================================================================================================
# 30. REMOTE DURABILITY VERIFICATION
# =================================================================================================

banner(
    "REMOTE DURABILITY VERIFICATION"
)

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_final = git(
    "rev-parse",
    "HEAD",
)

origin_final = git(
    "rev-parse",
    "origin/main",
)

remote_final_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_final_line:

    raise RuntimeError(
        "Unable to resolve pushed remote main."
    )


remote_final = (
    remote_final_line.split()[0]
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD :",
    local_final,
)

print(
    "origin/main:",
    origin_final,
)

print(
    "Remote main:",
    remote_final,
)

print(
    "Git clean  :",
    not bool(final_status),
)


if not (
    local_final
    == origin_final
    == remote_final
    == new_head
):

    raise RuntimeError(
        "Stage28-2A1 remote durability gate failed."
    )


if final_status:

    raise RuntimeError(
        "Repository dirty after Stage28-2A1 push:\n"
        + final_status
    )


print()
print(
    "[PASS] Stage28-2A1 remotely durable"
)

print(
    "[PASS] worktree clean"
)


# =================================================================================================
# 31. FINAL
# =================================================================================================

banner(
    "STAGE28-2A1 — RANDOM_NATURAL SEED42 COMPLETE"
)

print(
    "Durable commit:"
)

print(
    " ",
    new_head,
)

print()

print(
    "Evaluation cell:"
)

print(
    "  28A_STAGE22::RANDOM_NATURAL::SEED42"
)

print()

print(
    "Components:"
)

print(
    "  C001 XGBoost  seed42 CPU = NEW FIT SUCCESS"
)

print(
    "  C002 LightGBM seed42 CPU = HISTORICAL REUSE EXACT"
)

print()

print(
    "Validation ranking:"
)

print(
    "  ROC-AUC =",
    validation_roc_auc,
)

print(
    "  PR-AUC  =",
    validation_pr_auc,
)

print()

print(
    "Operating thresholds:"
)

print(
    "  STANDARD =",
    standard[
        "threshold"
    ],
)

print(
    "  BALANCED =",
    balanced[
        "threshold"
    ],
)

print(
    "  SECURITY =",
    security[
        "threshold"
    ],
)

print()

print(
    "Fit ledger:"
)

print(
    "  NEW authorized = 108"
)

print(
    "  consumed       = 1"
)

print(
    "  remaining      = 107"
)

print(
    "  reused executed= 1"
)

print()

print(
    "Shared final holdout:"
)

print(
    "  OPENINGS            = 0"
)

print(
    "  PREDICTOR ROWS READ = 0"
)

print(
    "  LABELS READ         = 0"
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage28-2A2 — CHRONOLOGICAL_NATURAL / seed42"
)

print(
    "  FIT #2 = C011 CPU XGBoost seed42"
)

print(
    "  C012 = exact historical CPU LightGBM reuse"
)

print(
    "  Shared final holdout remains closed."
)

print()
print(SEP)


STAGE28-2A1 — EXACT DURABLE-PARENT GATE

Expected parent: 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Local HEAD     : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
origin/main    : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Remote main    : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Branch         : main
Git clean      : True

[PASS] exact clean Stage28-1C parent

GITHUB DURABILITY AUTHORIZATION

[PASS] credential recovered from kaggle_secret:GITHUB_TOKEN
[PASS] token intentionally not displayed
[PASS] clean GitHub remote

FROZEN MODEL RUNTIME GATE

XGBoost : 3.2.0
LightGBM: 4.6.0
[PASS] Stage28 CPU-only execution policy active

PRE-FIT SCIENTIFIC LEDGER

Authorized : 108
Consumed   : 0
Remaining  : 108

[PASS] FIT #1 remains unconsumed

LOAD STAGE22 RANDOM_NATURAL EXECUTION ASSETS

TRAIN: 11,529,922
 attack: 1,577,839
 benign: 9,952,083

VALIDATION: 2,882,481
 attack: 394,460
 benign: 2,488,021

[PASS] exact RANDOM_NATURAL memberships loaded

C001 / C002 FROZEN COMPONENT GATE

Expected manifest

RuntimeError: Command failed (1):
git config --get user.name

STDOUT:


STDERR:


In [4]:
# =================================================================================================
# STAGE28-2A1-R1 — POST-FIT COMMIT / PUSH RECOVERY
#
# IMPORTANT:
#   DO NOT REFIT.
#
# FIT #1 already succeeded:
#   C001 — XGBoost / RANDOM_NATURAL / seed42 / CPU
#
# This cell ONLY:
#   - verifies the seven already-produced Stage28-2A1 artifacts
#   - verifies the scientific ledger says FIT #1 consumed exactly once
#   - verifies they are already staged
#   - safely configures Git identity
#   - commits
#   - pushes
#   - verifies remote durability
#
# NO:
#   model fitting
#   inference
#   threshold selection
#   holdout opening
# =================================================================================================

from __future__ import annotations

import base64
import hashlib
import json
import os
import subprocess
from pathlib import Path


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "738e3ee29d098d4132828aeaaacfa12a8cfd7b52"
)

COMMIT_MESSAGE = (
    "stage28-2a1: execute random natural seed42 checkpoint"
)

OUT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a1_random_natural_seed42"
)

XGB_MODEL_PATH = (
    OUT
    / "random_natural_seed42_xgboost_cpu_model.json"
)

VALIDATION_PROB_PATH = (
    OUT
    / "random_natural_seed42_validation_ensemble_probabilities.npz"
)

THRESHOLD_GRID_PATH = (
    OUT
    / "random_natural_seed42_validation_threshold_grid.csv"
)

REUSE_RECEIPT_PATH = (
    OUT
    / "random_natural_seed42_lightgbm_reuse_receipt.json"
)

FIT_LEDGER_PATH = (
    OUT
    / "stage28_2a1_fit_ledger.json"
)

RESULT_PATH = (
    OUT
    / "stage28_2a1_random_natural_seed42_result.json"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)

EXPECTED_XGB_MODEL_SHA = (
    "7bdb5fc3b0b464d2fb27785ad6bc7cf54de772c8e983f8fc0d51d638ec8c7907"
)

EXPECTED_VALIDATION_PROB_SHA = (
    "41bfa21977d550e389487e516ae697efa7a19d7b96883c3c30ef1a8a87632161"
)

EXPECTED_THRESHOLD_GRID_SHA = (
    "389d77be0f717cc0a6225d20d9fa8fdb7c5eab685569c628aa6bec57326e29c2"
)

EXPECTED_REUSE_RECEIPT_SHA = (
    "92260aca049691f130783c26b4beb42c391dee70679ccd23203a1cc265f9dcb9"
)

EXPECTED_FIT_LEDGER_SHA = (
    "2914d8ed97a3b50c9df2a7d0b95a8b115eaa624c1f0641017da39d79d849c16e"
)

EXPECTED_RESULT_SHA = (
    "6bb6d0529e2d220a03ca2d078cea68a29511dd1f4a01e4254976820357a4e2cc"
)


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(
    *args,
    check=True,
):
    return (
        run(
            ["git", *args],
            check=check,
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required Stage28-2A1 artifact missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def recover_github_token():

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]

    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()

        for label in labels:

            try:
                value = client.get_secret(label)

            except Exception:
                value = None

            if (
                isinstance(value, str)
                and value.strip()
            ):
                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass


    for label in labels:

        value = os.environ.get(label)

        if (
            isinstance(value, str)
            and value.strip()
        ):
            return (
                value.strip(),
                f"environment:{label}",
            )


    raise RuntimeError(
        "GitHub credential unavailable."
    )


def authenticated_push(token):

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode("utf-8")
    ).decode("ascii")


    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )


    if p.returncode != 0:
        raise RuntimeError(
            "Authenticated git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )


    return (
        p.stdout
        + p.stderr
    ).strip()


# =================================================================================================
# 2. CURRENT REPOSITORY STATE
# =================================================================================================

banner(
    "STAGE28-2A1-R1 — POST-FIT RECOVERY GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote = remote_line.split()[0]

branch = git(
    "branch",
    "--show-current",
)


print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    head,
)

print(
    "origin/main    :",
    origin,
)

print(
    "Remote main    :",
    remote,
)

print(
    "Branch         :",
    branch,
)


# The previous cell failed BEFORE git commit.
if not (
    head
    == origin
    == remote
    == EXPECTED_PARENT
):
    raise RuntimeError(
        "Repository no longer matches the expected "
        "pre-commit Stage28-2A1 state.\n"
        "STOP — do not refit."
    )


if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )


print()
print(
    "[PASS] no commit occurred after FIT #1"
)


# =================================================================================================
# 3. REQUIRE EXACT SEVEN ARTIFACTS
# =================================================================================================

banner(
    "VERIFY EXISTING STAGE28-2A1 ARTIFACTS"
)

expected_artifacts = {
    XGB_MODEL_PATH:
        EXPECTED_XGB_MODEL_SHA,

    VALIDATION_PROB_PATH:
        EXPECTED_VALIDATION_PROB_SHA,

    THRESHOLD_GRID_PATH:
        EXPECTED_THRESHOLD_GRID_SHA,

    REUSE_RECEIPT_PATH:
        EXPECTED_REUSE_RECEIPT_SHA,

    FIT_LEDGER_PATH:
        EXPECTED_FIT_LEDGER_SHA,

    RESULT_PATH:
        EXPECTED_RESULT_SHA,
}


for path, expected_sha in (
    expected_artifacts.items()
):

    require_file(path)

    actual_sha = sha256_file(path)

    print(
        "[CHECK]",
        path.name,
    )

    print(
        "  expected:",
        expected_sha,
    )

    print(
        "  actual  :",
        actual_sha,
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            f"Post-fit artifact changed: {path.name}"
        )


require_file(
    CHECKSUMS_PATH
)


print()
print(
    "[PASS] six scientific artifacts exactly "
    "match the completed FIT #1 output"
)


# =================================================================================================
# 4. CHECKSUM FILE SELF-VERIFICATION
# =================================================================================================

banner(
    "CHECKSUM MANIFEST SELF-VERIFICATION"
)

checksum_lines = (
    CHECKSUMS_PATH
    .read_text(
        encoding="utf-8"
    )
    .splitlines()
)


if len(checksum_lines) != 6:

    raise RuntimeError(
        "Stage28-2A1 checksums.sha256 "
        "does not contain exactly six payload entries."
    )


for line in checksum_lines:

    digest, filename = line.split(
        None,
        1,
    )

    filename = (
        filename.strip()
        .lstrip("*")
    )

    artifact = (
        OUT
        / filename
    )

    require_file(artifact)

    actual = sha256_file(
        artifact
    )


    print(
        "[PASS]",
        filename,
        actual,
    )


    if actual != digest:

        raise RuntimeError(
            f"checksums.sha256 mismatch: {filename}"
        )


print()
print(
    "[PASS] checksum manifest exact"
)


# =================================================================================================
# 5. SCIENTIFIC LEDGER — FIT #1 MUST ALREADY BE CONSUMED
# =================================================================================================

banner(
    "FIT #1 SCIENTIFIC LEDGER RECOVERY"
)

ledger = read_json(
    FIT_LEDGER_PATH
)

result = read_json(
    RESULT_PATH
)


if (
    ledger[
        "status"
    ]
    != "FIT_001_SUCCESSFULLY_CONSUMED"
):
    raise RuntimeError(
        "Stage28-2A1 fit ledger status invalid."
    )


if int(
    ledger[
        "pre_cell_new_fits_consumed"
    ]
) != 0:
    raise RuntimeError(
        "Unexpected pre-cell fit count."
    )


if int(
    ledger[
        "this_cell"
    ][
        "successful_new_fits"
    ]
) != 1:
    raise RuntimeError(
        "Stage28-2A1 must contain exactly one new fit."
    )


if ledger[
    "this_cell"
][
    "new_fit_components"
] != [
    "C001"
]:
    raise RuntimeError(
        "FIT #1 component identity changed."
    )


if ledger[
    "this_cell"
][
    "reused_components"
] != [
    "C002"
]:
    raise RuntimeError(
        "Stage28-2A1 reuse identity changed."
    )


if int(
    ledger[
        "cumulative_new_fits_consumed"
    ]
) != 1:
    raise RuntimeError(
        "Cumulative Stage28 fit count must be exactly 1."
    )


if int(
    ledger[
        "new_fits_remaining"
    ]
) != 107:
    raise RuntimeError(
        "Stage28 remaining fit count must be 107."
    )


if (
    result[
        "status"
    ]
    != "RANDOM_NATURAL_SEED42_VALIDATION_CHECKPOINT_FROZEN"
):
    raise RuntimeError(
        "Stage28-2A1 result status invalid."
    )


if (
    result[
        "models"
    ][
        "xgboost"
    ][
        "component_id"
    ]
    != "C001"
):
    raise RuntimeError(
        "XGBoost result component changed."
    )


if (
    result[
        "models"
    ][
        "xgboost"
    ][
        "model_sha256"
    ]
    != EXPECTED_XGB_MODEL_SHA
):
    raise RuntimeError(
        "XGBoost model identity differs from FIT #1."
    )


if int(
    result[
        "scientific_accounting"
    ][
        "new_model_fits_cumulative"
    ]
) != 1:
    raise RuntimeError(
        "Result cumulative fit ledger != 1."
    )


if int(
    result[
        "scientific_accounting"
    ][
        "shared_final_holdout_openings"
    ]
) != 0:
    raise RuntimeError(
        "Shared final holdout was unexpectedly opened."
    )


print(
    "C001 CPU XGBoost fit      : SUCCESS"
)

print(
    "C002 CPU LightGBM reuse   : EXACT"
)

print(
    "Cumulative fits consumed  :",
    ledger[
        "cumulative_new_fits_consumed"
    ],
)

print(
    "Remaining                 :",
    ledger[
        "new_fits_remaining"
    ],
)

print(
    "Final holdout openings    :",
    result[
        "scientific_accounting"
    ][
        "shared_final_holdout_openings"
    ],
)

print()
print(
    "[PASS] FIT #1 accounted exactly once"
)

print(
    "[PASS] DO NOT REFIT C001"
)


# =================================================================================================
# 6. VERIFY EXPECTED VALIDATION RESULT
# =================================================================================================

banner(
    "STAGE28-2A1 VALIDATION RESULT RECOVERY"
)

print(
    "ROC-AUC:",
    result[
        "validation_probability"
    ][
        "roc_auc"
    ],
)

print(
    "PR-AUC :",
    result[
        "validation_probability"
    ][
        "pr_auc"
    ],
)

print()

print(
    "STANDARD:",
    result[
        "operating_points"
    ][
        "standard"
    ][
        "threshold"
    ],
)

print(
    "BALANCED:",
    result[
        "operating_points"
    ][
        "balanced"
    ][
        "threshold"
    ],
)

print(
    "SECURITY:",
    result[
        "operating_points"
    ][
        "security"
    ][
        "result"
    ][
        "threshold"
    ],
)


if (
    result[
        "operating_points"
    ][
        "standard"
    ][
        "threshold"
    ]
    != 0.5
):
    raise RuntimeError(
        "Stage28-2A1 standard threshold changed."
    )


if (
    result[
        "operating_points"
    ][
        "balanced"
    ][
        "threshold"
    ]
    != 0.46
):
    raise RuntimeError(
        "Stage28-2A1 balanced threshold changed."
    )


if (
    result[
        "operating_points"
    ][
        "security"
    ][
        "result"
    ][
        "threshold"
    ]
    != 0.1
):
    raise RuntimeError(
        "Stage28-2A1 security threshold changed."
    )


print()
print(
    "[PASS] completed validation result intact"
)


# =================================================================================================
# 7. VERIFY INDEX / STAGED STATE
# =================================================================================================

banner(
    "GIT INDEX RECOVERY GATE"
)

expected_relative_files = {
    str(
        path.relative_to(
            REPO
        )
    )
    for path
    in [
        XGB_MODEL_PATH,
        VALIDATION_PROB_PATH,
        THRESHOLD_GRID_PATH,
        REUSE_RECEIPT_PATH,
        FIT_LEDGER_PATH,
        RESULT_PATH,
        CHECKSUMS_PATH,
    ]
}


staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    )
    .splitlines()
)


unstaged_tracked = set(
    git(
        "diff",
        "--name-only",
    )
    .splitlines()
)


untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    )
    .splitlines()
)


print(
    "Staged files:",
    len(staged),
)

for path in sorted(staged):
    print(
        " ",
        path,
    )


if staged != expected_relative_files:

    raise RuntimeError(
        "Git index no longer contains exactly "
        "the seven completed Stage28-2A1 files.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                expected_relative_files
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(staged)
        )
    )


if unstaged_tracked:

    raise RuntimeError(
        "Unexpected unstaged tracked modifications:\n"
        + "\n".join(
            sorted(
                unstaged_tracked
            )
        )
    )


if untracked:

    raise RuntimeError(
        "Unexpected untracked files after staging:\n"
        + "\n".join(
            sorted(
                untracked
            )
        )
    )


print()
print(
    "[PASS] exactly seven completed FIT #1 artifacts remain staged"
)

print(
    "[PASS] no unrelated Git changes"
)


# =================================================================================================
# 8. FIX GIT IDENTITY
#
# Previous failure was:
#
#   git config --get user.name
#
# Exit code 1 simply means the key was unset.
#
# We set the local repository identity directly instead of
# treating an absent optional config key as a fatal command.
# =================================================================================================

banner(
    "REPAIR LOCAL GIT IDENTITY"
)

current_name_proc = run(
    [
        "git",
        "config",
        "--get",
        "user.name",
    ],
    check=False,
)

current_email_proc = run(
    [
        "git",
        "config",
        "--get",
        "user.email",
    ],
    check=False,
)


current_name = (
    current_name_proc.stdout
    or ""
).strip()

current_email = (
    current_email_proc.stdout
    or ""
).strip()


print(
    "Existing user.name :",
    current_name
    if current_name
    else "<UNSET>",
)

print(
    "Existing user.email:",
    current_email
    if current_email
    else "<UNSET>",
)


if not current_name:

    run(
        [
            "git",
            "config",
            "user.name",
            "Stage28 Kaggle",
        ]
    )


if not current_email:

    run(
        [
            "git",
            "config",
            "user.email",
            "stage28-kaggle@users.noreply.github.com",
        ]
    )


final_name = git(
    "config",
    "--get",
    "user.name",
)

final_email = git(
    "config",
    "--get",
    "user.email",
)


print()
print(
    "Final user.name :",
    final_name,
)

print(
    "Final user.email:",
    final_email,
)


if not final_name or not final_email:

    raise RuntimeError(
        "Unable to configure local Git identity."
    )


print()
print(
    "[PASS] Git identity ready"
)


# =================================================================================================
# 9. RECOVER GITHUB AUTH
# =================================================================================================

banner(
    "GITHUB PUSH AUTHORIZATION"
)

github_token, token_source = (
    recover_github_token()
)


print(
    "[PASS] credential recovered from",
    token_source,
)

print(
    "[PASS] token intentionally not displayed"
)


# =================================================================================================
# 10. FINAL PRE-COMMIT ASSERTION
# =================================================================================================

banner(
    "FINAL PRE-COMMIT ASSERTION"
)

head_before_commit = git(
    "rev-parse",
    "HEAD",
)


if head_before_commit != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed before recovery commit."
    )


staged_final = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    )
    .splitlines()
)


if staged_final != expected_relative_files:

    raise RuntimeError(
        "Staged artifact universe changed before commit."
    )


print(
    "HEAD                       :",
    head_before_commit,
)

print(
    "FIT #1                     : CONSUMED ONCE"
)

print(
    "New fits remaining         : 107"
)

print(
    "Seven artifacts staged     : YES"
)

print(
    "Final holdout opened       : NO"
)

print()

print(
    "[PASS] ready to durably commit existing FIT #1"
)


# =================================================================================================
# 11. COMMIT — NO REFIT
# =================================================================================================

banner(
    "COMMIT EXISTING STAGE28-2A1 CHECKPOINT"
)

commit_proc = run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ]
)


print(
    commit_proc.stdout.strip()
)


new_head = git(
    "rev-parse",
    "HEAD",
)

new_parent = git(
    "rev-parse",
    "HEAD^",
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)


print()
print(
    "Parent :",
    new_parent,
)

print(
    "Commit :",
    new_head,
)

print(
    "Subject:",
    subject,
)


if new_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage28-2A1 recovery commit parent mismatch."
    )


if subject != COMMIT_MESSAGE:

    raise RuntimeError(
        "Stage28-2A1 recovery commit subject mismatch."
    )


print()
print(
    "[PASS] existing FIT #1 checkpoint committed"
)


# =================================================================================================
# 12. PUSH
# =================================================================================================

banner(
    "PUSH STAGE28-2A1 CHECKPOINT"
)

push_output = authenticated_push(
    github_token
)


if push_output:
    print(
        push_output
    )


github_token = None


# =================================================================================================
# 13. REMOTE DURABILITY VERIFICATION
# =================================================================================================

banner(
    "REMOTE DURABILITY VERIFICATION"
)

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_final = git(
    "rev-parse",
    "HEAD",
)

origin_final = git(
    "rev-parse",
    "origin/main",
)

remote_final_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


if not remote_final_line:

    raise RuntimeError(
        "Unable to resolve pushed remote main."
    )


remote_final = (
    remote_final_line.split()[0]
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD :",
    local_final,
)

print(
    "origin/main:",
    origin_final,
)

print(
    "Remote main:",
    remote_final,
)

print(
    "Git clean  :",
    not bool(final_status),
)


if not (
    local_final
    == origin_final
    == remote_final
    == new_head
):

    raise RuntimeError(
        "Remote Stage28-2A1 durability gate failed."
    )


if final_status:

    raise RuntimeError(
        "Repository dirty after recovery push:\n"
        + final_status
    )


print()
print(
    "[PASS] Stage28-2A1 remotely durable"
)

print(
    "[PASS] repository clean"
)


# =================================================================================================
# 14. FINAL
# =================================================================================================

banner(
    "STAGE28-2A1-R1 — COMMIT RECOVERY COMPLETE"
)

print(
    "Durable Stage28-2A1 commit:"
)

print(
    " ",
    new_head,
)

print()

print(
    "Scientific result already completed:"
)

print(
    "  C001 XGBoost seed42 CPU = FIT #1 SUCCESS"
)

print(
    "  C002 LightGBM seed42 CPU = HISTORICAL REUSE EXACT"
)

print()

print(
    "Validation:"
)

print(
    "  ROC-AUC =",
    result[
        "validation_probability"
    ][
        "roc_auc"
    ],
)

print(
    "  PR-AUC  =",
    result[
        "validation_probability"
    ][
        "pr_auc"
    ],
)

print()

print(
    "Thresholds:"
)

print(
    "  STANDARD = 0.50"
)

print(
    "  BALANCED = 0.46"
)

print(
    "  SECURITY = 0.10"
)

print()

print(
    "Fit ledger:"
)

print(
    "  NEW authorized = 108"
)

print(
    "  consumed       = 1"
)

print(
    "  remaining      = 107"
)

print()

print(
    "Final holdout:"
)

print(
    "  openings = 0"
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage28-2A2 — CHRONOLOGICAL_NATURAL / seed42"
)

print(
    "  FIT #2 = C011 CPU XGBoost seed42"
)

print(
    "  C012 = exact historical CPU LightGBM reuse"
)

print()
print(SEP)


STAGE28-2A1-R1 — POST-FIT RECOVERY GATE

Expected parent: 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Local HEAD     : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
origin/main    : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Remote main    : 738e3ee29d098d4132828aeaaacfa12a8cfd7b52
Branch         : main

[PASS] no commit occurred after FIT #1

VERIFY EXISTING STAGE28-2A1 ARTIFACTS

[CHECK] random_natural_seed42_xgboost_cpu_model.json
  expected: 7bdb5fc3b0b464d2fb27785ad6bc7cf54de772c8e983f8fc0d51d638ec8c7907
  actual  : 7bdb5fc3b0b464d2fb27785ad6bc7cf54de772c8e983f8fc0d51d638ec8c7907
[CHECK] random_natural_seed42_validation_ensemble_probabilities.npz
  expected: 41bfa21977d550e389487e516ae697efa7a19d7b96883c3c30ef1a8a87632161
  actual  : 41bfa21977d550e389487e516ae697efa7a19d7b96883c3c30ef1a8a87632161
[CHECK] random_natural_seed42_validation_threshold_grid.csv
  expected: 389d77be0f717cc0a6225d20d9fa8fdb7c5eab685569c628aa6bec57326e29c2
  actual  : 389d77be0f717cc0a6225d20d9fa8fdb7c5eab685569c6

In [5]:
# =================================================================================================
# STAGE28-2A2 — CHRONOLOGICAL_NATURAL / SEED42
#
# FIT #2:
#   C011 — XGBOOST / XGB_11 / seed42 / CPU / NEW_FIT_AUTHORIZED
#
# REUSE:
#   C012 — LIGHTGBM / LGBM_11 / seed42 / CPU / REUSE_EXISTING
#
# Durable parent:
#   e97c6f2337ed40b244d1c362aaa80a0fa2219b91
#
# PRE-CELL LEDGER
# ---------------
#   new fits authorized = 108
#   consumed            = 1
#   remaining           = 107
#
# EXPECTED POST-SUCCESS LEDGER
# ----------------------------
#   consumed            = 2
#   remaining           = 106
#
# SHARED FINAL HOLDOUT:
#   COMPLETELY CLOSED.
#
# IMPORTANT
# ---------
# Before FIT #2, this cell must reproduce the historical
# CHRONOLOGICAL_NATURAL validation semantics:
#
#   ROC-AUC  = 0.5149184263937692
#   PR-AUC   = 0.10621515513397227
#   STANDARD = 0.50
#   BALANCED = 0.07
#   SECURITY = 0.07
#
# Only after that self-test succeeds is C011 trained.
# =================================================================================================

from __future__ import annotations

import base64
import csv
import gc
import hashlib
import json
import math
import os
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    auc,
    roc_auc_score,
)

import xgboost as xgb
import lightgbm as lgb


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "e97c6f2337ed40b244d1c362aaa80a0fa2219b91"
)

COMMIT_MESSAGE = (
    "stage28-2a2: execute chronological natural seed42 checkpoint"
)

EXPECTED_XGB_VERSION = "3.2.0"
EXPECTED_LGBM_VERSION = "4.6.0"

EXPECTED_COMPONENT_MANIFEST_SHA = (
    "47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505"
)

EXPECTED_XGB_PARAMETER_SHA = (
    "1eb12cc88f3e1b455da9882d8439938fda175c66435940a5e2a6850cbcda63ff"
)

EXPECTED_LGBM_PARAMETER_SHA = (
    "dbb7ac185b3ac6aa8e3ac915cfa2b2080ee54bec8e7a38cab83537faad2f2754"
)

EXPECTED_REUSED_LGBM_SHA = (
    "7be4c610814e45be2e315996969d2e2f404a3ada14866b21a731e4382fdc18b8"
)

EXPECTED_HISTORICAL_VALIDATION_PROB_SHA = (
    "3fe6c468a0653ac5ee488da8d6586fd86628e9586eb01f6f5d962e35fff65e3f"
)

EXPECTED_HISTORICAL_RESULT_SHA = (
    "f43b5fa5031946029d8bf58ad153a838efaa2dcffcccd98111f694984a8c2d51"
)

EXPECTED_HISTORICAL_ROC = (
    0.5149184263937692
)

EXPECTED_HISTORICAL_PR = (
    0.10621515513397227
)

EXPECTED_TRAIN_ROWS = 13_818_623
EXPECTED_TRAIN_ATTACK = 1_910_043
EXPECTED_TRAIN_BENIGN = 11_908_580

EXPECTED_VAL_ROWS = 593_780
EXPECTED_VAL_ATTACK = 62_256
EXPECTED_VAL_BENIGN = 531_524

EXPECTED_TOTAL_ROWS = 14_412_403
EXPECTED_FEATURES = 70

EXPECTED_STAGE28_NEW_FITS = 108


# =================================================================================================
# 1. PATHS
# =================================================================================================

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

STAGE28_1B_DIR = (
    STAGE28_ROOT
    / "stage28_1b_random_loao_membership_and_execution_lock"
)

COMPONENT_MANIFEST_PATH = (
    STAGE28_1B_DIR
    / "stage28_component_execution_manifest.csv"
)

PARAMETER_SETS_PATH = (
    STAGE28_1B_DIR
    / "execution_parameter_sets.json"
)

THRESHOLD_POLICY_PATH = (
    STAGE28_ROOT
    / "stage28_0_protocol_lock"
    / "threshold_policy.json"
)


# -------------------------------------------------------------------------------------------------
# Prior durable Stage28-2A1 checkpoint.
# -------------------------------------------------------------------------------------------------

STAGE28_2A1_DIR = (
    STAGE28_ROOT
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a1_random_natural_seed42"
)

STAGE28_2A1_LEDGER = (
    STAGE28_2A1_DIR
    / "stage28_2a1_fit_ledger.json"
)

STAGE28_2A1_RESULT = (
    STAGE28_2A1_DIR
    / "stage28_2a1_random_natural_seed42_result.json"
)

STAGE28_2A1_CHECKSUMS = (
    STAGE28_2A1_DIR
    / "checksums.sha256"
)


# -------------------------------------------------------------------------------------------------
# Stage28-2A0 runtime matrix.
# -------------------------------------------------------------------------------------------------

STAGE28_2A0_RECEIPT = Path(
    "/kaggle/working/"
    "stage28_2a0_stage22_execution_matrix_receipt.json"
)

RUNTIME_ROOT = Path(
    "/kaggle/working/"
    "stage28_2a_runtime_cache/"
    "stage22_full"
)

X_PATH = (
    RUNTIME_ROOT
    / "development_features_float64.npy"
)

Y_PATH = (
    RUNTIME_ROOT
    / "development_binary_labels_uint8.npy"
)

DAY_PATH = (
    RUNTIME_ROOT
    / "development_day_ids_uint8.npy"
)


# -------------------------------------------------------------------------------------------------
# Historical Stage22 chronology artifacts.
# -------------------------------------------------------------------------------------------------

HISTORICAL_DIR = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_2c_chronological_natural"
)

HISTORICAL_RESULT_PATH = (
    HISTORICAL_DIR
    / "stage22r_2c_chronological_natural_result.json"
)

HISTORICAL_LGBM_MODEL_PATH = (
    HISTORICAL_DIR
    / "chronological_natural_lightgbm_model.txt"
)

HISTORICAL_VALIDATION_PROB_PATH = (
    HISTORICAL_DIR
    / "chronological_natural_validation_ensemble_probabilities.npz"
)


# -------------------------------------------------------------------------------------------------
# New Stage28-2A2 durable output.
# -------------------------------------------------------------------------------------------------

OUT = (
    STAGE28_ROOT
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a2_chronological_natural_seed42"
)

XGB_MODEL_PATH = (
    OUT
    / "chronological_natural_seed42_xgboost_cpu_model.json"
)

VALIDATION_PROB_PATH = (
    OUT
    / "chronological_natural_seed42_validation_ensemble_probabilities.npz"
)

THRESHOLD_GRID_PATH = (
    OUT
    / "chronological_natural_seed42_validation_threshold_grid.csv"
)

REUSE_RECEIPT_PATH = (
    OUT
    / "chronological_natural_seed42_lightgbm_reuse_receipt.json"
)

FIT_LEDGER_PATH = (
    OUT
    / "stage28_2a2_fit_ledger.json"
)

RESULT_PATH = (
    OUT
    / "stage28_2a2_chronological_natural_seed42_result.json"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


# =================================================================================================
# 2. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(
    *args,
    check=True,
):
    return (
        run(
            ["git", *args],
            check=check,
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def write_json(path, obj):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def canonical_json_sha256(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


def safe_div(a, b):
    return (
        float(a) / float(b)
        if b
        else 0.0
    )


def count_metrics(
    tp,
    fp,
    tn,
    fn,
):
    total = (
        tp + fp + tn + fn
    )

    return {
        "accuracy":
            safe_div(
                tp + tn,
                total,
            ),

        "precision":
            safe_div(
                tp,
                tp + fp,
            ),

        "recall":
            safe_div(
                tp,
                tp + fn,
            ),

        "fpr":
            safe_div(
                fp,
                fp + tn,
            ),

        "f1":
            safe_div(
                2 * tp,
                (2 * tp) + fp + fn,
            ),

        "f2":
            safe_div(
                5 * tp,
                (5 * tp) + (4 * fn) + fp,
            ),

        "tp":
            int(tp),

        "fp":
            int(fp),

        "tn":
            int(tn),

        "fn":
            int(fn),
    }


def evaluate_threshold(
    y_true,
    probability_float32,
    integer_percent,
):
    threshold = (
        integer_percent
        / 100.0
    )

    threshold32 = np.float32(
        threshold
    )

    pred = (
        probability_float32
        >= threshold32
    )

    positive = (
        y_true
        == 1
    )

    negative = ~positive


    tp = int(
        np.count_nonzero(
            pred & positive
        )
    )

    fp = int(
        np.count_nonzero(
            pred & negative
        )
    )

    fn = int(
        np.count_nonzero(
            (~pred) & positive
        )
    )

    tn = int(
        np.count_nonzero(
            (~pred) & negative
        )
    )


    row = count_metrics(
        tp,
        fp,
        tn,
        fn,
    )

    row.update(
        {
            "threshold_integer_percent":
                int(
                    integer_percent
                ),

            "threshold":
                float(
                    threshold
                ),

            "threshold_float32_runtime":
                float(
                    threshold32
                ),
        }
    )

    return row


def build_grid(
    y_true,
    probability_float32,
):
    return [
        evaluate_threshold(
            y_true,
            probability_float32,
            pct,
        )
        for pct
        in range(
            5,
            96,
        )
    ]


def choose_balanced(grid):
    return max(
        grid,
        key=lambda r: (
            r["f1"],
            -r["fpr"],
            r["recall"],
            -abs(
                r["threshold"] - 0.50
            ),
            -r["threshold"],
        ),
    )


def choose_security(grid):
    feasible = [
        r
        for r
        in grid
        if r[
            "fpr"
        ] <= 0.05
    ]

    if not feasible:
        return None

    return max(
        feasible,
        key=lambda r: (
            r["f2"],
            -r["fpr"],
            r["recall"],
            -r["threshold"],
        ),
    )


def assert_close(
    actual,
    expected,
    label,
    atol=1e-12,
):
    if not math.isclose(
        float(actual),
        float(expected),
        rel_tol=0.0,
        abs_tol=atol,
    ):
        raise RuntimeError(
            f"{label} mismatch.\n"
            f"expected={expected!r}\n"
            f"actual={actual!r}"
        )


def assert_operating_point(
    actual,
    expected,
    label,
):
    integer_fields = [
        "threshold_integer_percent",
        "tp",
        "fp",
        "tn",
        "fn",
    ]

    float_fields = [
        "threshold",
        "threshold_float32_runtime",
        "accuracy",
        "precision",
        "recall",
        "fpr",
        "f1",
        "f2",
    ]


    for field in integer_fields:

        if int(
            actual[field]
        ) != int(
            expected[field]
        ):

            raise RuntimeError(
                f"{label}/{field} mismatch.\n"
                f"expected={expected[field]}\n"
                f"actual={actual[field]}"
            )


    for field in float_fields:

        assert_close(
            actual[field],
            expected[field],
            f"{label}/{field}",
        )


def recover_github_token():

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]


    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()

        for label in labels:

            try:
                value = client.get_secret(
                    label
                )

            except Exception:
                value = None

            if (
                isinstance(value, str)
                and value.strip()
            ):
                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass


    for label in labels:

        value = os.environ.get(
            label
        )

        if (
            isinstance(value, str)
            and value.strip()
        ):
            return (
                value.strip(),
                f"environment:{label}",
            )


    raise RuntimeError(
        "GitHub credential unavailable."
    )


def authenticated_push(token):

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode("utf-8")
    ).decode("ascii")


    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )


    if p.returncode != 0:

        raise RuntimeError(
            "Authenticated git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )


    return (
        p.stdout
        + p.stderr
    ).strip()


# =================================================================================================
# 3. EXACT DURABLE-PARENT GATE
# =================================================================================================

banner(
    "STAGE28-2A2 — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote = (
    remote_line.split()[0]
)

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    head,
)

print(
    "origin/main    :",
    origin,
)

print(
    "Remote main    :",
    remote,
)

print(
    "Branch         :",
    branch,
)

print(
    "Git clean      :",
    not bool(status),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-2A2 parent mismatch."
    )

if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean:\n"
        + status
    )

if OUT.exists():
    raise RuntimeError(
        "Stage28-2A2 durable output already exists.\n"
        "Do not overwrite a scientific execution checkpoint."
    )


print()
print(
    "[PASS] exact clean Stage28-2A1 parent"
)


# =================================================================================================
# 4. GITHUB DURABILITY AUTHORIZATION
# =================================================================================================

banner(
    "GITHUB DURABILITY AUTHORIZATION"
)

github_token, token_source = (
    recover_github_token()
)


print(
    "[PASS] credential recovered from",
    token_source,
)

print(
    "[PASS] token intentionally not displayed"
)


remote_url = git(
    "remote",
    "get-url",
    "origin",
)


if (
    remote_url.startswith("https://")
    and "@github.com" in remote_url
):
    raise RuntimeError(
        "Remote URL contains embedded credentials."
    )


print(
    "[PASS] clean GitHub remote"
)


# =================================================================================================
# 5. LIBRARY / CPU GATE
# =================================================================================================

banner(
    "FROZEN MODEL RUNTIME GATE"
)

print(
    "XGBoost :",
    xgb.__version__,
)

print(
    "LightGBM:",
    lgb.__version__,
)


if xgb.__version__ != EXPECTED_XGB_VERSION:
    raise RuntimeError(
        "XGBoost version drift."
    )

if lgb.__version__ != EXPECTED_LGBM_VERSION:
    raise RuntimeError(
        "LightGBM version drift."
    )


os.environ[
    "CUDA_VISIBLE_DEVICES"
] = ""


print(
    "[PASS] Stage28 CPU-only execution policy active"
)


# =================================================================================================
# 6. PRIOR DURABLE FIT LEDGER
# =================================================================================================

banner(
    "PRIOR DURABLE STAGE28 FIT LEDGER"
)

require_file(
    STAGE28_2A1_CHECKSUMS
)

ledger_2a1 = read_json(
    require_file(
        STAGE28_2A1_LEDGER
    )
)

result_2a1 = read_json(
    require_file(
        STAGE28_2A1_RESULT
    )
)


if (
    ledger_2a1[
        "status"
    ]
    != "FIT_001_SUCCESSFULLY_CONSUMED"
):
    raise RuntimeError(
        "Stage28-2A1 FIT #1 durable ledger invalid."
    )


if int(
    ledger_2a1[
        "cumulative_new_fits_consumed"
    ]
) != 1:
    raise RuntimeError(
        "Expected exactly one prior new fit."
    )


if int(
    ledger_2a1[
        "new_fits_remaining"
    ]
) != 107:
    raise RuntimeError(
        "Expected 107 new fits remaining."
    )


if int(
    result_2a1[
        "scientific_accounting"
    ][
        "shared_final_holdout_openings"
    ]
) != 0:
    raise RuntimeError(
        "Shared final holdout was previously opened."
    )


print(
    "Authorized:",
    EXPECTED_STAGE28_NEW_FITS,
)

print(
    "Consumed  :",
    ledger_2a1[
        "cumulative_new_fits_consumed"
    ],
)

print(
    "Remaining :",
    ledger_2a1[
        "new_fits_remaining"
    ],
)

print()

print(
    "[PASS] FIT #1 durably accounted"
)

print(
    "[PASS] FIT #2 is next authorized new fit"
)


# =================================================================================================
# 7. STAGE28-2A0 RUNTIME GATE
# =================================================================================================

banner(
    "STAGE28-2A0 RUNTIME EXECUTION GATE"
)

receipt_2a0 = read_json(
    require_file(
        STAGE28_2A0_RECEIPT
    )
)


if (
    receipt_2a0[
        "status"
    ]
    != "READY_FOR_STAGE28_FIT_001"
):
    raise RuntimeError(
        "Stage28-2A0 runtime receipt invalid."
    )


if int(
    receipt_2a0[
        "chronological_natural"
    ][
        "train"
    ][
        "rows"
    ]
) != EXPECTED_TRAIN_ROWS:
    raise RuntimeError(
        "Chronological train row count changed."
    )


if int(
    receipt_2a0[
        "chronological_natural"
    ][
        "validation"
    ][
        "rows"
    ]
) != EXPECTED_VAL_ROWS:
    raise RuntimeError(
        "Chronological validation row count changed."
    )


print(
    "[PASS] exact Stage28-2A0 chronological runtime geometry"
)


# =================================================================================================
# 8. LOAD CHRONOLOGICAL EXECUTION DATA
# =================================================================================================

banner(
    "LOAD CHRONOLOGICAL_NATURAL EXECUTION DATA"
)

X = np.load(
    require_file(
        X_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)

y = np.load(
    require_file(
        Y_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)

day_ids = np.load(
    require_file(
        DAY_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)


if X.shape != (
    EXPECTED_TOTAL_ROWS,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        "Stage22 execution matrix shape mismatch."
    )

if X.dtype != np.float64:
    raise RuntimeError(
        "Stage22 execution matrix dtype != float64."
    )


if not np.all(
    day_ids[
        :EXPECTED_TRAIN_ROWS
    ]
    <= 6
):
    raise RuntimeError(
        "Chronological train contains day 7."
    )


if not np.all(
    day_ids[
        EXPECTED_TRAIN_ROWS:
    ]
    == 7
):
    raise RuntimeError(
        "Chronological validation is not exactly day 7."
    )


y_train_view = np.asarray(
    y[
        :EXPECTED_TRAIN_ROWS
    ]
)

y_validation = np.asarray(
    y[
        EXPECTED_TRAIN_ROWS:
    ],
    dtype=np.uint8,
)


train_attack = int(
    y_train_view.sum()
)

val_attack = int(
    y_validation.sum()
)


if train_attack != EXPECTED_TRAIN_ATTACK:
    raise RuntimeError(
        "Chronological training attack count mismatch."
    )

if (
    EXPECTED_TRAIN_ROWS
    - train_attack
    != EXPECTED_TRAIN_BENIGN
):
    raise RuntimeError(
        "Chronological training benign count mismatch."
    )


if val_attack != EXPECTED_VAL_ATTACK:
    raise RuntimeError(
        "Chronological validation attack count mismatch."
    )

if (
    EXPECTED_VAL_ROWS
    - val_attack
    != EXPECTED_VAL_BENIGN
):
    raise RuntimeError(
        "Chronological validation benign count mismatch."
    )


print(
    "TRAIN:"
)

print(
    " rows  :",
    f"{EXPECTED_TRAIN_ROWS:,}",
)

print(
    " attack:",
    f"{train_attack:,}",
)

print(
    " benign:",
    f"{EXPECTED_TRAIN_ROWS - train_attack:,}",
)

print()

print(
    "VALIDATION:"
)

print(
    " rows  :",
    f"{EXPECTED_VAL_ROWS:,}",
)

print(
    " attack:",
    f"{val_attack:,}",
)

print(
    " benign:",
    f"{EXPECTED_VAL_ROWS - val_attack:,}",
)

print()

print(
    "[PASS] exact CHRONOLOGICAL_NATURAL membership loaded"
)


del y_train_view
gc.collect()


# =================================================================================================
# 9. C011 / C012 MANIFEST GATE
# =================================================================================================

banner(
    "C011 / C012 FROZEN COMPONENT GATE"
)

manifest_sha = sha256_file(
    require_file(
        COMPONENT_MANIFEST_PATH
    )
)


if manifest_sha != EXPECTED_COMPONENT_MANIFEST_SHA:
    raise RuntimeError(
        "Stage28 component manifest SHA mismatch."
    )


with COMPONENT_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:

    manifest_rows = list(
        csv.DictReader(f)
    )


by_component = {
    row[
        "component_id"
    ]:
    row
    for row
    in manifest_rows
}


c011 = by_component.get(
    "C011"
)

c012 = by_component.get(
    "C012"
)


if c011 is None or c012 is None:
    raise RuntimeError(
        "C011/C012 missing from frozen manifest."
    )


expected_c011 = {
    "arm":
        "28A_TRAINING_SEED_STABILITY",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "CHRONOLOGICAL_NATURAL",

    "evaluation_cell_id":
        "28A_STAGE22::CHRONOLOGICAL_NATURAL::SEED42",

    "learner":
        "XGBOOST",

    "configuration_id":
        "XGB_11",

    "model_seed":
        "42",

    "compute_backend":
        "CPU",

    "parameter_set_id":
        "STAGE22_FULL::XGBOOST::SEED42",

    "parameter_sha256":
        EXPECTED_XGB_PARAMETER_SHA,

    "membership_reference":
        "STAGE22R_FROZEN_CHRONOLOGICAL_NATURAL_MEMBERSHIP",

    "fit_action":
        "NEW_FIT_AUTHORIZED",

    "new_fit_budget_units":
        "1",
}


expected_c012 = {
    "arm":
        "28A_TRAINING_SEED_STABILITY",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "CHRONOLOGICAL_NATURAL",

    "evaluation_cell_id":
        "28A_STAGE22::CHRONOLOGICAL_NATURAL::SEED42",

    "learner":
        "LIGHTGBM",

    "configuration_id":
        "LGBM_11",

    "model_seed":
        "42",

    "compute_backend":
        "CPU",

    "parameter_set_id":
        "STAGE22_FULL::LIGHTGBM::SEED42",

    "parameter_sha256":
        EXPECTED_LGBM_PARAMETER_SHA,

    "membership_reference":
        "STAGE22R_FROZEN_CHRONOLOGICAL_NATURAL_MEMBERSHIP",

    "fit_action":
        "REUSE_EXISTING",

    "reused_model_sha256":
        EXPECTED_REUSED_LGBM_SHA,

    "reuse_budget_units":
        "1",
}


for field, expected in expected_c011.items():

    if c011[field] != expected:

        raise RuntimeError(
            f"C011 {field} mismatch.\n"
            f"expected={expected}\n"
            f"actual={c011[field]}"
        )


for field, expected in expected_c012.items():

    if c012[field] != expected:

        raise RuntimeError(
            f"C012 {field} mismatch.\n"
            f"expected={expected}\n"
            f"actual={c012[field]}"
        )


print(
    "[PASS] C011 = CPU XGBoost seed42 NEW FIT"
)

print(
    "[PASS] C012 = CPU LightGBM seed42 REUSE"
)


# =================================================================================================
# 10. PARAMETER SET GATE
# =================================================================================================

banner(
    "FROZEN PARAMETER-SET GATE"
)

parameter_doc = read_json(
    require_file(
        PARAMETER_SETS_PATH
    )
)

parameter_sets = (
    parameter_doc[
        "sets"
    ]
)


xgb_params = dict(
    parameter_sets[
        c011[
            "parameter_set_id"
        ]
    ][
        "parameters"
    ]
)


lgbm_params = dict(
    parameter_sets[
        c012[
            "parameter_set_id"
        ]
    ][
        "parameters"
    ]
)


xgb_parameter_sha = (
    canonical_json_sha256(
        xgb_params
    )
)

lgbm_parameter_sha = (
    canonical_json_sha256(
        lgbm_params
    )
)


print(
    "XGB expected:",
    EXPECTED_XGB_PARAMETER_SHA,
)

print(
    "XGB actual  :",
    xgb_parameter_sha,
)

print()

print(
    "LGBM expected:",
    EXPECTED_LGBM_PARAMETER_SHA,
)

print(
    "LGBM actual  :",
    lgbm_parameter_sha,
)


if xgb_parameter_sha != EXPECTED_XGB_PARAMETER_SHA:
    raise RuntimeError(
        "XGBoost parameter SHA mismatch."
    )

if lgbm_parameter_sha != EXPECTED_LGBM_PARAMETER_SHA:
    raise RuntimeError(
        "LightGBM parameter SHA mismatch."
    )


if (
    xgb_params[
        "device"
    ]
    != "cpu"
    or xgb_params[
        "tree_method"
    ]
    != "hist"
):
    raise RuntimeError(
        "C011 is not CPU/hist."
    )


if (
    lgbm_params[
        "device_type"
    ]
    != "cpu"
):
    raise RuntimeError(
        "C012 is not CPU."
    )


print()
print(
    "[PASS] exact frozen CPU parameter sets"
)


# =================================================================================================
# 11. C012 BYTE-EXACT LIGHTGBM REUSE
# =================================================================================================

banner(
    "C012 — BYTE-EXACT HISTORICAL LIGHTGBM REUSE"
)

require_file(
    HISTORICAL_LGBM_MODEL_PATH
)

historical_lgbm_sha = sha256_file(
    HISTORICAL_LGBM_MODEL_PATH
)


print(
    "Expected:",
    EXPECTED_REUSED_LGBM_SHA,
)

print(
    "Actual  :",
    historical_lgbm_sha,
)


if historical_lgbm_sha != EXPECTED_REUSED_LGBM_SHA:
    raise RuntimeError(
        "Historical chronology LightGBM SHA mismatch."
    )


lgb_booster = lgb.Booster(
    model_file=str(
        HISTORICAL_LGBM_MODEL_PATH
    )
)


lgb_iterations = int(
    lgb_booster.current_iteration()
)

lgb_trees = int(
    lgb_booster.num_trees()
)

lgb_features = int(
    lgb_booster.num_feature()
)


print(
    "Iterations:",
    lgb_iterations,
)

print(
    "Trees     :",
    lgb_trees,
)

print(
    "Features  :",
    lgb_features,
)


if lgb_iterations != 400:
    raise RuntimeError(
        "Historical LightGBM iteration count != 400."
    )

if lgb_trees != 400:
    raise RuntimeError(
        "Historical LightGBM tree count != 400."
    )

if lgb_features != EXPECTED_FEATURES:
    raise RuntimeError(
        "Historical LightGBM feature count != 70."
    )


print()
print(
    "[PASS] exact historical CPU LightGBM loaded"
)

print(
    "[PASS] C012 consumes zero new-fit budget"
)


# =================================================================================================
# 12. THRESHOLD POLICY GATE
# =================================================================================================

banner(
    "FROZEN STAGE22 THRESHOLD POLICY"
)

threshold_policy = read_json(
    require_file(
        THRESHOLD_POLICY_PATH
    )
)

policy = (
    threshold_policy[
        "stage22_full"
    ]
)


if (
    policy[
        "selection_population"
    ]
    != "EACH_PARENT_CELL_FROZEN_DEVELOPMENT_VALIDATION"
):
    raise RuntimeError(
        "Stage22 threshold population changed."
    )


if not (
    policy[
        "grid"
    ][
        "integer_percent_start"
    ] == 5
    and
    policy[
        "grid"
    ][
        "integer_percent_stop_inclusive"
    ] == 95
    and
    policy[
        "grid"
    ][
        "integer_step"
    ] == 1
    and
    policy[
        "grid"
    ][
        "count"
    ] == 91
):
    raise RuntimeError(
        "Stage22 threshold grid changed."
    )


if (
    policy[
        "final_holdout_threshold_search"
    ]
    != "FORBIDDEN"
):
    raise RuntimeError(
        "Final holdout threshold search changed."
    )


print(
    "[PASS] inherited Stage22 threshold policy exact"
)


# =================================================================================================
# 13. HISTORICAL CHRONOLOGY SELF-TEST BEFORE FIT #2
# =================================================================================================

banner(
    "HISTORICAL CHRONOLOGICAL_NATURAL SELF-TEST BEFORE FIT #2"
)

historical_result = read_json(
    require_file(
        HISTORICAL_RESULT_PATH
    )
)


historical_result_sha = sha256_file(
    HISTORICAL_RESULT_PATH
)


print(
    "Historical result expected SHA:",
    EXPECTED_HISTORICAL_RESULT_SHA,
)

print(
    "Historical result actual SHA  :",
    historical_result_sha,
)


if historical_result_sha != EXPECTED_HISTORICAL_RESULT_SHA:
    raise RuntimeError(
        "Historical chronology result SHA mismatch."
    )


require_file(
    HISTORICAL_VALIDATION_PROB_PATH
)

historical_prob_file_sha = sha256_file(
    HISTORICAL_VALIDATION_PROB_PATH
)


print()
print(
    "Historical validation NPZ expected:",
    EXPECTED_HISTORICAL_VALIDATION_PROB_SHA,
)

print(
    "Historical validation NPZ actual  :",
    historical_prob_file_sha,
)


if (
    historical_prob_file_sha
    != EXPECTED_HISTORICAL_VALIDATION_PROB_SHA
):
    raise RuntimeError(
        "Historical chronology validation NPZ SHA mismatch."
    )


assert_close(
    historical_result[
        "validation_probability"
    ][
        "roc_auc"
    ],
    EXPECTED_HISTORICAL_ROC,
    "historical ROC-AUC",
)


assert_close(
    historical_result[
        "validation_probability"
    ][
        "pr_auc"
    ],
    EXPECTED_HISTORICAL_PR,
    "historical PR-AUC",
)


historical_candidates = []


with np.load(
    HISTORICAL_VALIDATION_PROB_PATH,
    allow_pickle=False,
) as npz:

    print(
        "NPZ keys:",
        list(npz.files),
    )


    for key in npz.files:

        arr = npz[
            key
        ]


        if not (
            arr.ndim == 1
            and arr.shape[0]
            == EXPECTED_VAL_ROWS
            and np.issubdtype(
                arr.dtype,
                np.floating,
            )
        ):
            continue


        p = np.asarray(
            arr,
            dtype=np.float32,
        )


        roc_value = float(
            roc_auc_score(
                y_validation,
                p,
            )
        )


        ap_value = float(
            average_precision_score(
                y_validation,
                p,
            )
        )


        precision_curve, recall_curve, _ = (
            precision_recall_curve(
                y_validation,
                p,
            )
        )


        trap_value = float(
            auc(
                recall_curve,
                precision_curve,
            )
        )


        roc_match = math.isclose(
            roc_value,
            EXPECTED_HISTORICAL_ROC,
            rel_tol=0.0,
            abs_tol=1e-12,
        )


        ap_match = math.isclose(
            ap_value,
            EXPECTED_HISTORICAL_PR,
            rel_tol=0.0,
            abs_tol=1e-12,
        )


        trap_match = math.isclose(
            trap_value,
            EXPECTED_HISTORICAL_PR,
            rel_tol=0.0,
            abs_tol=1e-12,
        )


        print()
        print(
            "Candidate:",
            key,
        )

        print(
            " ROC-AUC        :",
            roc_value,
        )

        print(
            " AveragePrecision:",
            ap_value,
        )

        print(
            " Trapezoidal PR :",
            trap_value,
        )

        print(
            " ROC match       :",
            roc_match,
        )

        print(
            " AP match        :",
            ap_match,
        )

        print(
            " Trap match      :",
            trap_match,
        )


        if (
            roc_match
            and (
                ap_match
                or trap_match
            )
        ):

            historical_candidates.append(
                {
                    "key":
                        key,

                    "probability":
                        p,

                    "ap_match":
                        ap_match,

                    "trap_match":
                        trap_match,
                }
            )


if len(
    historical_candidates
) != 1:
    raise RuntimeError(
        "Could not uniquely resolve historical "
        "chronology ensemble probability."
    )


historical_match = (
    historical_candidates[
        0
    ]
)

historical_probability_key = (
    historical_match[
        "key"
    ]
)

historical_probability = (
    historical_match[
        "probability"
    ]
)


if (
    historical_match[
        "ap_match"
    ]
    and
    not historical_match[
        "trap_match"
    ]
):

    PR_AUC_DEFINITION = (
        "SKLEARN_AVERAGE_PRECISION_SCORE"
    )

elif (
    historical_match[
        "trap_match"
    ]
    and
    not historical_match[
        "ap_match"
    ]
):

    PR_AUC_DEFINITION = (
        "SKLEARN_PRECISION_RECALL_CURVE_TRAPEZOIDAL_AUC"
    )

elif (
    historical_match[
        "ap_match"
    ]
    and
    historical_match[
        "trap_match"
    ]
):

    PR_AUC_DEFINITION = (
        "AP_AND_TRAPEZOIDAL_IDENTICAL_ON_PARENT_VECTOR;"
        "USE_SKLEARN_AVERAGE_PRECISION_SCORE"
    )

else:

    raise RuntimeError(
        "Unable to resolve PR-AUC definition."
    )


historical_grid = build_grid(
    y_validation,
    historical_probability,
)


historical_standard = next(
    r
    for r
    in historical_grid
    if r[
        "threshold_integer_percent"
    ] == 50
)

historical_balanced = choose_balanced(
    historical_grid
)

historical_security = choose_security(
    historical_grid
)


if historical_security is None:
    raise RuntimeError(
        "Historical security threshold unexpectedly infeasible."
    )


assert_operating_point(
    historical_standard,
    historical_result[
        "operating_points"
    ][
        "standard"
    ],
    "HISTORICAL_STANDARD",
)


assert_operating_point(
    historical_balanced,
    historical_result[
        "operating_points"
    ][
        "balanced"
    ],
    "HISTORICAL_BALANCED",
)


assert_operating_point(
    historical_security,
    historical_result[
        "operating_points"
    ][
        "security"
    ][
        "result"
    ],
    "HISTORICAL_SECURITY",
)


if historical_standard[
    "threshold"
] != 0.50:

    raise RuntimeError(
        "Historical standard threshold != 0.50."
    )


if historical_balanced[
    "threshold"
] != 0.07:

    raise RuntimeError(
        "Historical balanced threshold != 0.07."
    )


if historical_security[
    "threshold"
] != 0.07:

    raise RuntimeError(
        "Historical security threshold != 0.07."
    )


print()
print(
    "[PASS] historical ROC-AUC =",
    EXPECTED_HISTORICAL_ROC,
)

print(
    "[PASS] historical PR-AUC  =",
    EXPECTED_HISTORICAL_PR,
)

print(
    "[PASS] PR-AUC semantics   =",
    PR_AUC_DEFINITION,
)

print(
    "[PASS] STANDARD = 0.50"
)

print(
    "[PASS] BALANCED = 0.07"
)

print(
    "[PASS] SECURITY = 0.07"
)

print()

print(
    "All historical chronology semantics reproduced."
)

print(
    "FIT #2 is now authorized."
)


del historical_probability
del historical_match
del historical_candidates
del historical_grid

gc.collect()


# =================================================================================================
# 14. CREATE OUTPUT DIRECTORY
# =================================================================================================

OUT.mkdir(
    parents=True,
    exist_ok=False,
)


# =================================================================================================
# 15. PREPARE CONTIGUOUS CHRONOLOGICAL TRAINING POPULATION
# =================================================================================================

banner(
    "PREPARE CHRONOLOGICAL_NATURAL TRAINING POPULATION"
)

materialize_start = time.perf_counter()


# Chronological membership is already a contiguous prefix.
# np.asarray therefore avoids unnecessary advanced-indexing copies
# where NumPy can safely expose the contiguous memmap region.
X_train = np.asarray(
    X[
        :EXPECTED_TRAIN_ROWS,
        :
    ],
    dtype=np.float64,
    order="C",
)

y_train = np.asarray(
    y[
        :EXPECTED_TRAIN_ROWS
    ],
    dtype=np.uint8,
    order="C",
)


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


if X_train.shape != (
    EXPECTED_TRAIN_ROWS,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        "Chronological X_train shape mismatch."
    )


if y_train.shape != (
    EXPECTED_TRAIN_ROWS,
):
    raise RuntimeError(
        "Chronological y_train shape mismatch."
    )


if int(
    y_train.sum()
) != EXPECTED_TRAIN_ATTACK:
    raise RuntimeError(
        "Chronological y_train class count mismatch."
    )


if np.isinf(
    X_train
).any():
    raise RuntimeError(
        "Infinity exists in chronological training matrix."
    )


print(
    "Rows    :",
    f"{X_train.shape[0]:,}",
)

print(
    "Features:",
    X_train.shape[1],
)

print(
    "Dtype   :",
    X_train.dtype,
)

print(
    "Payload :",
    f"{X_train.nbytes / (1024**3):.3f} GiB",
)

print(
    "Preparation seconds:",
    materialize_seconds,
)

print()

print(
    "[PASS] exact chronological training population ready"
)


# =================================================================================================
# 16. FIT #2 — C011 CPU XGBOOST
# =================================================================================================

banner(
    "FIT #2 — C011 XGBOOST XGB_11 SEED42 CPU"
)

print(
    "Component   : C011"
)

print(
    "Experiment  : STAGE22_FULL"
)

print(
    "Unit        : CHRONOLOGICAL_NATURAL"
)

print(
    "Seed        : 42"
)

print(
    "Backend     : CPU"
)

print(
    "tree_method :",
    xgb_params[
        "tree_method"
    ],
)

print(
    "device      :",
    xgb_params[
        "device"
    ],
)

print(
    "Estimators  :",
    xgb_params[
        "n_estimators"
    ],
)

print()

print(
    "Starting Stage28 scientific FIT #2..."
)


fit_start = time.perf_counter()


xgb_model = xgb.XGBClassifier(
    **xgb_params
)


xgb_model.fit(
    X_train,
    y_train,
)


fit_seconds = (
    time.perf_counter()
    - fit_start
)


booster = (
    xgb_model.get_booster()
)

boosted_rounds = int(
    booster.num_boosted_rounds()
)


print()
print(
    "FIT #2 completed."
)

print(
    "Fit seconds:",
    fit_seconds,
)

print(
    "Boosted rounds:",
    boosted_rounds,
)


if boosted_rounds != 400:
    raise RuntimeError(
        "XGBoost boosted-round count != 400."
    )


if int(
    xgb_model.n_features_in_
) != EXPECTED_FEATURES:
    raise RuntimeError(
        "XGBoost feature count != 70."
    )


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_model_sha = sha256_file(
    XGB_MODEL_PATH
)


print(
    "Model SHA256:",
    xgb_model_sha,
)

print()

print(
    "[PASS] C011 FIT #2 successful"
)


del X_train
del y_train

gc.collect()


# =================================================================================================
# 17. CHRONOLOGICAL DEVELOPMENT VALIDATION INFERENCE
# =================================================================================================

banner(
    "CHRONOLOGICAL_NATURAL SEED42 — VALIDATION INFERENCE"
)

ensemble_probability = np.empty(
    EXPECTED_VAL_ROWS,
    dtype=np.float32,
)

chunk_size = 65_536

inference_start = time.perf_counter()


for local_start in range(
    0,
    EXPECTED_VAL_ROWS,
    chunk_size,
):

    local_stop = min(
        local_start + chunk_size,
        EXPECTED_VAL_ROWS,
    )


    global_start = (
        EXPECTED_TRAIN_ROWS
        + local_start
    )

    global_stop = (
        EXPECTED_TRAIN_ROWS
        + local_stop
    )


    X_chunk = np.asarray(
        X[
            global_start:global_stop,
            :
        ],
        dtype=np.float64,
        order="C",
    )


    p_xgb = np.asarray(
        xgb_model.predict_proba(
            X_chunk
        )[
            :,
            1
        ],
        dtype=np.float64,
    )


    p_lgbm = np.asarray(
        lgb_booster.predict(
            X_chunk,
            num_iteration=lgb_iterations,
        ),
        dtype=np.float64,
    )


    expected_chunk = (
        local_stop
        - local_start
    )


    if len(p_xgb) != expected_chunk:
        raise RuntimeError(
            "XGBoost validation inference length mismatch."
        )


    if len(p_lgbm) != expected_chunk:
        raise RuntimeError(
            "LightGBM validation inference length mismatch."
        )


    combined64 = (
        (0.5 * p_lgbm)
        +
        (0.5 * p_xgb)
    )


    ensemble_probability[
        local_start:local_stop
    ] = combined64.astype(
        np.float32,
        copy=False,
    )


    del X_chunk
    del p_xgb
    del p_lgbm
    del combined64

    gc.collect()


    print(
        f"  inferred "
        f"{local_stop:,} / {EXPECTED_VAL_ROWS:,}"
    )


inference_seconds = (
    time.perf_counter()
    - inference_start
)


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):
    raise RuntimeError(
        "Non-finite validation probability detected."
    )


if (
    np.any(
        ensemble_probability < 0
    )
    or
    np.any(
        ensemble_probability > 1
    )
):
    raise RuntimeError(
        "Validation probability outside [0,1]."
    )


print()
print(
    "Inference seconds:",
    inference_seconds,
)

print()

print(
    "[PASS] C011 validation inference complete"
)

print(
    "[PASS] C012 reuse validation inference complete"
)

print(
    "[PASS] equal-weight float32 ensemble complete"
)


# =================================================================================================
# 18. VALIDATION RANKING METRICS
# =================================================================================================

banner(
    "CHRONOLOGICAL_NATURAL SEED42 — VALIDATION RANKING"
)

validation_roc_auc = float(
    roc_auc_score(
        y_validation,
        ensemble_probability,
    )
)


if (
    PR_AUC_DEFINITION
    == "SKLEARN_PRECISION_RECALL_CURVE_TRAPEZOIDAL_AUC"
):

    precision_curve, recall_curve, _ = (
        precision_recall_curve(
            y_validation,
            ensemble_probability,
        )
    )

    validation_pr_auc = float(
        auc(
            recall_curve,
            precision_curve,
        )
    )

else:

    validation_pr_auc = float(
        average_precision_score(
            y_validation,
            ensemble_probability,
        )
    )


print(
    "ROC-AUC:",
    validation_roc_auc,
)

print(
    "PR-AUC :",
    validation_pr_auc,
)

print(
    "PR definition:",
    PR_AUC_DEFINITION,
)


# =================================================================================================
# 19. FROZEN THRESHOLD SELECTION
# =================================================================================================

banner(
    "CHRONOLOGICAL_NATURAL SEED42 — FROZEN THRESHOLDS"
)

grid_rows = build_grid(
    y_validation,
    ensemble_probability,
)


standard = next(
    r
    for r
    in grid_rows
    if r[
        "threshold_integer_percent"
    ] == 50
)


balanced = choose_balanced(
    grid_rows
)

security = choose_security(
    grid_rows
)


if security is None:
    raise RuntimeError(
        "Security threshold infeasible under "
        "frozen FPR<=0.05 rule."
    )


print(
    "STANDARD:"
)

print(
    " threshold:",
    standard[
        "threshold"
    ],
)

print(
    " F1       :",
    standard[
        "f1"
    ],
)

print(
    " recall   :",
    standard[
        "recall"
    ],
)

print(
    " FPR      :",
    standard[
        "fpr"
    ],
)

print()

print(
    "BALANCED:"
)

print(
    " threshold:",
    balanced[
        "threshold"
    ],
)

print(
    " F1       :",
    balanced[
        "f1"
    ],
)

print(
    " recall   :",
    balanced[
        "recall"
    ],
)

print(
    " FPR      :",
    balanced[
        "fpr"
    ],
)

print()

print(
    "SECURITY:"
)

print(
    " threshold:",
    security[
        "threshold"
    ],
)

print(
    " F2       :",
    security[
        "f2"
    ],
)

print(
    " recall   :",
    security[
        "recall"
    ],
)

print(
    " FPR      :",
    security[
        "fpr"
    ],
)


# =================================================================================================
# 20. WRITE VALIDATION ARTIFACTS
# =================================================================================================

banner(
    "FREEZE STAGE28-2A2 VALIDATION ARTIFACTS"
)

validation_global_idx = np.arange(
    EXPECTED_TRAIN_ROWS,
    EXPECTED_TOTAL_ROWS,
    dtype=np.int32,
)


np.savez_compressed(
    VALIDATION_PROB_PATH,

    ensemble_probability_float32=
        ensemble_probability,

    validation_global_idx_int32=
        validation_global_idx,

    binary_label_uint8=
        y_validation,
)


grid_df = pd.DataFrame(
    grid_rows
)[
    [
        "threshold_integer_percent",
        "threshold",
        "threshold_float32_runtime",
        "accuracy",
        "precision",
        "recall",
        "fpr",
        "f1",
        "f2",
        "tp",
        "fp",
        "tn",
        "fn",
    ]
]


grid_df.to_csv(
    THRESHOLD_GRID_PATH,
    index=False,
    lineterminator="\n",
)


validation_probability_sha = sha256_file(
    VALIDATION_PROB_PATH
)

threshold_grid_sha = sha256_file(
    THRESHOLD_GRID_PATH
)


print(
    "Validation probability SHA:",
    validation_probability_sha,
)

print(
    "Threshold grid SHA       :",
    threshold_grid_sha,
)


# =================================================================================================
# 21. C012 REUSE RECEIPT
# =================================================================================================

reuse_receipt = {

    "stage":
        "Stage28-2A2",

    "component_id":
        "C012",

    "evaluation_cell_id":
        "28A_STAGE22::CHRONOLOGICAL_NATURAL::SEED42",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "CHRONOLOGICAL_NATURAL",

    "learner":
        "LIGHTGBM",

    "configuration_id":
        "LGBM_11",

    "seed":
        42,

    "compute_backend":
        "CPU",

    "fit_action":
        "REUSE_EXISTING",

    "historical_model": {
        "path":
            str(
                HISTORICAL_LGBM_MODEL_PATH.relative_to(
                    REPO
                )
            ),

        "sha256":
            historical_lgbm_sha,

        "expected_sha256":
            EXPECTED_REUSED_LGBM_SHA,

        "iterations":
            lgb_iterations,

        "trees":
            lgb_trees,

        "features":
            lgb_features,

        "library_version":
            lgb.__version__,
    },

    "parameter_set_id":
        c012[
            "parameter_set_id"
        ],

    "parameter_sha256":
        lgbm_parameter_sha,

    "new_fit_budget_consumed":
        0,

    "validation_inference_performed":
        True,

    "status":
        "EXACT_HISTORICAL_CPU_LIGHTGBM_REUSED",
}


write_json(
    REUSE_RECEIPT_PATH,
    reuse_receipt,
)


reuse_receipt_sha = sha256_file(
    REUSE_RECEIPT_PATH
)


# =================================================================================================
# 22. FIT LEDGER
# =================================================================================================

fit_ledger = {

    "stage":
        "Stage28-2A2",

    "durable_parent":
        EXPECTED_PARENT,

    "evaluation_cell_id":
        "28A_STAGE22::CHRONOLOGICAL_NATURAL::SEED42",

    "authorized_stage28_new_fit_budget":
        108,

    "pre_cell_new_fits_consumed":
        1,

    "this_cell": {
        "successful_new_fits":
            1,

        "successful_reused_components":
            1,

        "new_fit_components": [
            "C011",
        ],

        "reused_components": [
            "C012",
        ],
    },

    "cumulative_new_fits_consumed":
        2,

    "new_fits_remaining":
        106,

    "stage22_new_fits_consumed":
        2,

    "stage22_new_fits_remaining":
        16,

    "model_fits_attempted":
        1,

    "model_fits_successful":
        1,

    "status":
        "FIT_002_SUCCESSFULLY_CONSUMED",
}


write_json(
    FIT_LEDGER_PATH,
    fit_ledger,
)


fit_ledger_sha = sha256_file(
    FIT_LEDGER_PATH
)


# =================================================================================================
# 23. RESULT RECORD
# =================================================================================================

result = {

    "stage":
        "Stage28-2A2",

    "status":
        "CHRONOLOGICAL_NATURAL_SEED42_VALIDATION_CHECKPOINT_FROZEN",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_PARENT,

    "arm":
        "28A_TRAINING_SEED_STABILITY",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "CHRONOLOGICAL_NATURAL",

    "training_seed":
        42,

    "evaluation_cell_id":
        "28A_STAGE22::CHRONOLOGICAL_NATURAL::SEED42",

    "membership": {
        "train_rows":
            EXPECTED_TRAIN_ROWS,

        "train_attack":
            EXPECTED_TRAIN_ATTACK,

        "train_benign":
            EXPECTED_TRAIN_BENIGN,

        "validation_rows":
            EXPECTED_VAL_ROWS,

        "validation_attack":
            EXPECTED_VAL_ATTACK,

        "validation_benign":
            EXPECTED_VAL_BENIGN,

        "train_days":
            "DAY_ID_0_THROUGH_6",

        "validation_days":
            "DAY_ID_7",

        "membership_changed_by_training_seed":
            False,
    },

    "models": {
        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "xgboost": {
            "component_id":
                "C011",

            "fit_action":
                "NEW_FIT_AUTHORIZED",

            "configuration":
                "XGB_11",

            "seed":
                42,

            "backend":
                "cpu",

            "parameters":
                xgb_params,

            "parameter_sha256":
                xgb_parameter_sha,

            "library_version":
                xgb.__version__,

            "boosted_rounds":
                boosted_rounds,

            "fit_seconds":
                float(
                    fit_seconds
                ),

            "training_preparation_seconds":
                float(
                    materialize_seconds
                ),

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                xgb_model_sha,
        },

        "lightgbm": {
            "component_id":
                "C012",

            "fit_action":
                "REUSE_EXISTING",

            "configuration":
                "LGBM_11",

            "seed":
                42,

            "backend":
                "cpu",

            "parameter_sha256":
                lgbm_parameter_sha,

            "historical_model_path":
                str(
                    HISTORICAL_LGBM_MODEL_PATH.relative_to(
                        REPO
                    )
                ),

            "historical_model_sha256":
                historical_lgbm_sha,

            "library_version":
                lgb.__version__,

            "iterations":
                lgb_iterations,

            "trees":
                lgb_trees,

            "new_fit_budget_consumed":
                0,
        },
    },

    "historical_semantics_self_test": {
        "historical_result_sha256":
            historical_result_sha,

        "historical_probability_artifact":
            str(
                HISTORICAL_VALIDATION_PROB_PATH.relative_to(
                    REPO
                )
            ),

        "historical_probability_sha256":
            historical_prob_file_sha,

        "historical_probability_key":
            historical_probability_key,

        "historical_roc_auc":
            EXPECTED_HISTORICAL_ROC,

        "historical_pr_auc":
            EXPECTED_HISTORICAL_PR,

        "pr_auc_definition":
            PR_AUC_DEFINITION,

        "historical_standard_threshold":
            0.50,

        "historical_balanced_threshold":
            0.07,

        "historical_security_threshold":
            0.07,

        "status":
            "EXACT_REPRODUCTION",
    },

    "validation_probability": {
        "rows":
            EXPECTED_VAL_ROWS,

        "roc_auc":
            validation_roc_auc,

        "pr_auc":
            validation_pr_auc,

        "pr_auc_definition":
            PR_AUC_DEFINITION,

        "artifact":
            VALIDATION_PROB_PATH.name,

        "artifact_sha256":
            validation_probability_sha,

        "inference_seconds":
            float(
                inference_seconds
            ),
    },

    "threshold_selection": {
        "selection_population":
            "FROZEN_CHRONOLOGICAL_NATURAL_DEVELOPMENT_VALIDATION",

        "final_holdout_threshold_search":
            "FORBIDDEN",

        "prediction_rule":
            (
                "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_"
                "PROBABILITY_GTE_THRESHOLD"
            ),

        "grid_integer_percent": [
            5,
            95,
        ],

        "grid_points":
            91,

        "balanced":
            (
                "MAX_F1; TIES LOWER_FPR, HIGHER_RECALL, "
                "CLOSER_TO_0_50, LOWER_THRESHOLD"
            ),

        "security":
            (
                "FPR<=0.05 EXACT; MAX_F2; TIES LOWER_FPR, "
                "HIGHER_RECALL, LOWER_THRESHOLD; NO_RELAXATION"
            ),

        "grid_artifact":
            THRESHOLD_GRID_PATH.name,

        "grid_sha256":
            threshold_grid_sha,
    },

    "operating_points": {
        "standard":
            standard,

        "balanced":
            balanced,

        "security": {
            "status":
                "AVAILABLE",

            "result":
                security,
        },
    },

    "scientific_accounting": {
        "new_model_fits_before_cell":
            1,

        "new_model_fits_this_cell":
            1,

        "new_model_fits_cumulative":
            2,

        "new_model_fits_remaining":
            106,

        "existing_models_reused_this_cell":
            1,

        "development_validation_model_inference":
            True,

        "threshold_selection_completed":
            True,

        "shared_final_holdout_openings":
            0,

        "shared_final_holdout_predictor_rows_read":
            0,

        "shared_final_holdout_labels_read":
            0,

        "new_target_openings":
            0,

        "target_adaptive_choices":
            0,
    },

    "next_authorized_step":
        (
            "Stage28-2A3 — RANDOM_NATURAL seed43. "
            "C003 XGBoost and C004 LightGBM are both NEW "
            "CPU fits. Shared final holdout remains closed."
        ),
}


write_json(
    RESULT_PATH,
    result,
)


result_sha = sha256_file(
    RESULT_PATH
)


# =================================================================================================
# 24. CHECKSUM MANIFEST
# =================================================================================================

artifact_paths = [
    XGB_MODEL_PATH,
    VALIDATION_PROB_PATH,
    THRESHOLD_GRID_PATH,
    REUSE_RECEIPT_PATH,
    FIT_LEDGER_PATH,
    RESULT_PATH,
]


checksum_lines = []


for artifact in artifact_paths:

    checksum_lines.append(
        f"{sha256_file(artifact)}  {artifact.name}"
    )


CHECKSUMS_PATH.write_text(
    "\n".join(
        checksum_lines
    )
    + "\n",
    encoding="utf-8",
)


print()
print(
    "Durable artifact checksums:"
)

print()

print(
    CHECKSUMS_PATH.read_text(
        encoding="utf-8"
    ).rstrip()
)


# =================================================================================================
# 25. SELF-VERIFY ARTIFACTS
# =================================================================================================

banner(
    "SELF-VERIFY STAGE28-2A2 ARTIFACTS"
)

expected_files = {
    XGB_MODEL_PATH.name,
    VALIDATION_PROB_PATH.name,
    THRESHOLD_GRID_PATH.name,
    REUSE_RECEIPT_PATH.name,
    FIT_LEDGER_PATH.name,
    RESULT_PATH.name,
    CHECKSUMS_PATH.name,
}


actual_files = {
    p.name
    for p
    in OUT.iterdir()
    if p.is_file()
}


if actual_files != expected_files:

    raise RuntimeError(
        "Unexpected Stage28-2A2 artifact universe.\n"
        f"Expected={sorted(expected_files)}\n"
        f"Actual={sorted(actual_files)}"
    )


for line in (
    CHECKSUMS_PATH.read_text(
        encoding="utf-8"
    ).splitlines()
):

    digest, filename = line.split(
        None,
        1,
    )

    filename = (
        filename.strip()
        .lstrip("*")
    )

    actual = sha256_file(
        OUT
        / filename
    )


    if actual != digest:

        raise RuntimeError(
            f"Checksum mismatch: {filename}"
        )


    print(
        "[PASS]",
        filename,
        digest,
    )


print()
print(
    "[PASS] all Stage28-2A2 artifacts exact"
)


# =================================================================================================
# 26. SCIENTIFIC LEDGER BEFORE COMMIT
# =================================================================================================

banner(
    "STAGE28-2A2 SCIENTIFIC LEDGER"
)

print(
    "NEW fits authorized : 108"
)

print(
    "Pre-cell consumed   : 1"
)

print(
    "This cell consumed  : 1"
)

print(
    "Cumulative consumed : 2"
)

print(
    "Remaining           : 106"
)

print()

print(
    "C011 XGBoost fit     : SUCCESS"
)

print(
    "C012 LightGBM reuse  : EXACT"
)

print(
    "Validation inference : COMPLETE"
)

print(
    "Threshold selection : COMPLETE"
)

print(
    "Final holdout opening: 0"
)

print()

print(
    "[PASS] FIT #2 accounted exactly once"
)


# =================================================================================================
# 27. AUTHORIZED GIT-CHANGE GATE
# =================================================================================================

banner(
    "AUTHORIZED GIT-CHANGE GATE"
)

tracked_changes = (
    git(
        "diff",
        "--name-only",
    )
    .splitlines()
)


if tracked_changes:

    raise RuntimeError(
        "Unexpected tracked modifications:\n"
        + "\n".join(
            tracked_changes
        )
    )


expected_untracked = {
    str(
        (
            OUT
            / filename
        ).relative_to(
            REPO
        )
    )
    for filename
    in expected_files
}


all_untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    )
    .splitlines()
)


if all_untracked != expected_untracked:

    raise RuntimeError(
        "Unexpected untracked files.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                expected_untracked
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                all_untracked
            )
        )
    )


print(
    "[PASS] exactly seven authorized Stage28-2A2 files exist"
)


# =================================================================================================
# 28. STAGE EXACT FILES
# =================================================================================================

banner(
    "STAGE STAGE28-2A2 CHECKPOINT"
)

for relative in sorted(
    expected_untracked
):

    run(
        [
            "git",
            "add",
            "--",
            relative,
        ]
    )


staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    )
    .splitlines()
)


if staged != expected_untracked:

    raise RuntimeError(
        "Staged file universe mismatch."
    )


for relative in sorted(
    staged
):
    print(
        " ",
        relative,
    )


print()
print(
    "[PASS] exactly seven files staged"
)


# =================================================================================================
# 29. CONFIGURE LOCAL GIT IDENTITY SAFELY
#
# Do this unconditionally.
# This avoids the previous `git config --get` exit-code issue.
# =================================================================================================

banner(
    "LOCAL GIT IDENTITY"
)

run(
    [
        "git",
        "config",
        "user.name",
        "Stage28 Kaggle",
    ]
)

run(
    [
        "git",
        "config",
        "user.email",
        "stage28-kaggle@users.noreply.github.com",
    ]
)


print(
    "user.name :",
    git(
        "config",
        "--get",
        "user.name",
    ),
)

print(
    "user.email:",
    git(
        "config",
        "--get",
        "user.email",
    ),
)

print()

print(
    "[PASS] Git identity ready"
)


# =================================================================================================
# 30. FINAL PRE-COMMIT GATE
# =================================================================================================

banner(
    "FINAL PRE-COMMIT GATE"
)

if git(
    "rev-parse",
    "HEAD",
) != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed before Stage28-2A2 commit."
    )


staged_final = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    )
    .splitlines()
)


if staged_final != expected_untracked:

    raise RuntimeError(
        "Staged universe changed before commit."
    )


print(
    "FIT #1 durable     : YES"
)

print(
    "FIT #2 successful  : YES"
)

print(
    "Cumulative fits    : 2"
)

print(
    "Remaining          : 106"
)

print(
    "Holdout openings   : 0"
)

print()

print(
    "[PASS] ready to commit FIT #2"
)


# =================================================================================================
# 31. COMMIT
# =================================================================================================

banner(
    "COMMIT STAGE28-2A2"
)

commit_proc = run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ]
)


print(
    commit_proc.stdout.strip()
)


new_head = git(
    "rev-parse",
    "HEAD",
)

new_parent = git(
    "rev-parse",
    "HEAD^",
)

new_subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)


print()
print(
    "Parent :",
    new_parent,
)

print(
    "Commit :",
    new_head,
)

print(
    "Subject:",
    new_subject,
)


if new_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage28-2A2 commit parent mismatch."
    )


if new_subject != COMMIT_MESSAGE:

    raise RuntimeError(
        "Stage28-2A2 commit subject mismatch."
    )


print()
print(
    "[PASS] Stage28-2A2 committed"
)


# =================================================================================================
# 32. PUSH
# =================================================================================================

banner(
    "PUSH STAGE28-2A2"
)

push_output = authenticated_push(
    github_token
)


if push_output:
    print(
        push_output
    )


github_token = None


# =================================================================================================
# 33. REMOTE DURABILITY
# =================================================================================================

banner(
    "REMOTE DURABILITY VERIFICATION"
)

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_final = git(
    "rev-parse",
    "HEAD",
)

origin_final = git(
    "rev-parse",
    "origin/main",
)

remote_final_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


if not remote_final_line:
    raise RuntimeError(
        "Unable to resolve pushed remote main."
    )


remote_final = (
    remote_final_line.split()[0]
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD :",
    local_final,
)

print(
    "origin/main:",
    origin_final,
)

print(
    "Remote main:",
    remote_final,
)

print(
    "Git clean  :",
    not bool(
        final_status
    ),
)


if not (
    local_final
    == origin_final
    == remote_final
    == new_head
):

    raise RuntimeError(
        "Stage28-2A2 remote durability gate failed."
    )


if final_status:

    raise RuntimeError(
        "Repository dirty after Stage28-2A2 push:\n"
        + final_status
    )


print()
print(
    "[PASS] Stage28-2A2 remotely durable"
)

print(
    "[PASS] repository clean"
)


# =================================================================================================
# 34. FINAL
# =================================================================================================

banner(
    "STAGE28-2A2 — CHRONOLOGICAL_NATURAL SEED42 COMPLETE"
)

print(
    "Durable commit:"
)

print(
    " ",
    new_head,
)

print()

print(
    "Evaluation cell:"
)

print(
    "  28A_STAGE22::CHRONOLOGICAL_NATURAL::SEED42"
)

print()

print(
    "Components:"
)

print(
    "  C011 XGBoost  seed42 CPU = NEW FIT SUCCESS"
)

print(
    "  C012 LightGBM seed42 CPU = HISTORICAL REUSE EXACT"
)

print()

print(
    "Validation ranking:"
)

print(
    "  ROC-AUC =",
    validation_roc_auc,
)

print(
    "  PR-AUC  =",
    validation_pr_auc,
)

print()

print(
    "Operating thresholds:"
)

print(
    "  STANDARD =",
    standard[
        "threshold"
    ],
)

print(
    "  BALANCED =",
    balanced[
        "threshold"
    ],
)

print(
    "  SECURITY =",
    security[
        "threshold"
    ],
)

print()

print(
    "Fit ledger:"
)

print(
    "  NEW authorized = 108"
)

print(
    "  consumed       = 2"
)

print(
    "  remaining      = 106"
)

print(
    "  reused executed= 2 cumulative"
)

print()

print(
    "Stage22 seed42 status:"
)

print(
    "  RANDOM_NATURAL        = COMPLETE"
)

print(
    "  CHRONOLOGICAL_NATURAL = COMPLETE"
)

print()

print(
    "Shared final holdout:"
)

print(
    "  OPENINGS            = 0"
)

print(
    "  PREDICTOR ROWS READ = 0"
)

print(
    "  LABELS READ         = 0"
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage28-2A3 — RANDOM_NATURAL / seed43"
)

print(
    "  C003 = CPU XGBoost seed43 NEW FIT"
)

print(
    "  C004 = CPU LightGBM seed43 NEW FIT"
)

print()

print(
    "  Those will consume FIT #3 and FIT #4."
)

print(
    "  Shared final holdout remains closed."
)

print()
print(SEP)


STAGE28-2A2 — EXACT DURABLE-PARENT GATE

Expected parent: e97c6f2337ed40b244d1c362aaa80a0fa2219b91
Local HEAD     : e97c6f2337ed40b244d1c362aaa80a0fa2219b91
origin/main    : e97c6f2337ed40b244d1c362aaa80a0fa2219b91
Remote main    : e97c6f2337ed40b244d1c362aaa80a0fa2219b91
Branch         : main
Git clean      : True

[PASS] exact clean Stage28-2A1 parent

GITHUB DURABILITY AUTHORIZATION

[PASS] credential recovered from kaggle_secret:GITHUB_TOKEN
[PASS] token intentionally not displayed
[PASS] clean GitHub remote

FROZEN MODEL RUNTIME GATE

XGBoost : 3.2.0
LightGBM: 4.6.0
[PASS] Stage28 CPU-only execution policy active

PRIOR DURABLE STAGE28 FIT LEDGER

Authorized: 108
Consumed  : 1
Remaining : 107

[PASS] FIT #1 durably accounted
[PASS] FIT #2 is next authorized new fit

STAGE28-2A0 RUNTIME EXECUTION GATE

[PASS] exact Stage28-2A0 chronological runtime geometry

LOAD CHRONOLOGICAL_NATURAL EXECUTION DATA

TRAIN:
 rows  : 13,818,623
 attack: 1,910,043
 benign: 11,908,580

VALIDATION:
 r

In [6]:
# =================================================================================================
# STAGE28-2A3 — RANDOM_NATURAL / SEED43
#
# NEW FITS:
#   FIT #3 — C003 XGBOOST / XGB_11 / seed43 / CPU
#   FIT #4 — C004 LIGHTGBM / LGBM_11 / seed43 / CPU
#
# Durable parent:
#   d597370a59ba7fc93b2da1720fd3faebf9cfa1ba
#
# PRE-CELL LEDGER
#   authorized = 108
#   consumed   = 2
#   remaining  = 106
#
# POST-SUCCESS LEDGER
#   consumed   = 4
#   remaining  = 104
#
# MEMBERSHIP
#   EXACT SAME frozen RANDOM_NATURAL membership as seed42.
#
# THRESHOLDS
#   Selected independently for seed43 on frozen DEVELOPMENT validation only.
#
# SHARED FINAL HOLDOUT
#   NOT OPENED.
#
# IMPORTANT
#   This cell writes an execution-progress receipt after each model fit.
#   If anything fails AFTER either "Starting FIT #3" or "Starting FIT #4",
#   DO NOT rerun this cell. Send the output so the exact partial ledger can
#   be recovered without duplicating a scientific fit.
# =================================================================================================

from __future__ import annotations

import base64
import csv
import gc
import hashlib
import json
import os
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

import xgboost as xgb
import lightgbm as lgb


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "d597370a59ba7fc93b2da1720fd3faebf9cfa1ba"
)

COMMIT_MESSAGE = (
    "stage28-2a3: execute random natural seed43 checkpoint"
)

EXPECTED_XGB_VERSION = "3.2.0"
EXPECTED_LGBM_VERSION = "4.6.0"

EXPECTED_COMPONENT_MANIFEST_SHA = (
    "47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505"
)

EXPECTED_XGB_PARAMETER_SHA = (
    "5ac981f17f3327f8de881b672b6bf35a3b020a3a56a68957146cbe6622e30ed0"
)

EXPECTED_LGBM_PARAMETER_SHA = (
    "d43b2e4009bec8e9874a2ec0fd65f1279ea8d493808a386ab84fc801076e9c13"
)

EXPECTED_ROWS = 14_412_403
EXPECTED_FEATURES = 70

EXPECTED_TRAIN_ROWS = 11_529_922
EXPECTED_TRAIN_ATTACK = 1_577_839
EXPECTED_TRAIN_BENIGN = 9_952_083

EXPECTED_VAL_ROWS = 2_882_481
EXPECTED_VAL_ATTACK = 394_460
EXPECTED_VAL_BENIGN = 2_488_021

EXPECTED_TOTAL_NEW_FITS = 108

EXPECTED_RANDOM_PACKBITS_SHA = (
    "8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad"
)


# =================================================================================================
# 1. PATHS
# =================================================================================================

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

STAGE28_1B_DIR = (
    STAGE28_ROOT
    / "stage28_1b_random_loao_membership_and_execution_lock"
)

COMPONENT_MANIFEST_PATH = (
    STAGE28_1B_DIR
    / "stage28_component_execution_manifest.csv"
)

PARAMETER_SETS_PATH = (
    STAGE28_1B_DIR
    / "execution_parameter_sets.json"
)

THRESHOLD_POLICY_PATH = (
    STAGE28_ROOT
    / "stage28_0_protocol_lock"
    / "threshold_policy.json"
)


# -------------------------------------------------------------------------------------------------
# Prior durable checkpoint — FIT #2.
# -------------------------------------------------------------------------------------------------

STAGE28_2A2_DIR = (
    STAGE28_ROOT
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a2_chronological_natural_seed42"
)

STAGE28_2A2_LEDGER = (
    STAGE28_2A2_DIR
    / "stage28_2a2_fit_ledger.json"
)

STAGE28_2A2_RESULT = (
    STAGE28_2A2_DIR
    / "stage28_2a2_chronological_natural_seed42_result.json"
)

STAGE28_2A2_CHECKSUMS = (
    STAGE28_2A2_DIR
    / "checksums.sha256"
)


# -------------------------------------------------------------------------------------------------
# Runtime Stage22 execution cache.
# -------------------------------------------------------------------------------------------------

STAGE28_2A0_RECEIPT = Path(
    "/kaggle/working/"
    "stage28_2a0_stage22_execution_matrix_receipt.json"
)

RUNTIME_ROOT = Path(
    "/kaggle/working/"
    "stage28_2a_runtime_cache/"
    "stage22_full"
)

X_PATH = (
    RUNTIME_ROOT
    / "development_features_float64.npy"
)

Y_PATH = (
    RUNTIME_ROOT
    / "development_binary_labels_uint8.npy"
)

RANDOM_TRAIN_IDX_PATH = (
    RUNTIME_ROOT
    / "random_natural_train_global_idx_int32.npy"
)

RANDOM_VAL_IDX_PATH = (
    RUNTIME_ROOT
    / "random_natural_validation_global_idx_int32.npy"
)


# -------------------------------------------------------------------------------------------------
# New durable Stage28-2A3 checkpoint.
# -------------------------------------------------------------------------------------------------

OUT = (
    STAGE28_ROOT
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a3_random_natural_seed43"
)

XGB_MODEL_PATH = (
    OUT
    / "random_natural_seed43_xgboost_cpu_model.json"
)

LGBM_MODEL_PATH = (
    OUT
    / "random_natural_seed43_lightgbm_cpu_model.txt"
)

VALIDATION_PROB_PATH = (
    OUT
    / "random_natural_seed43_validation_ensemble_probabilities.npz"
)

THRESHOLD_GRID_PATH = (
    OUT
    / "random_natural_seed43_validation_threshold_grid.csv"
)

PROGRESS_PATH = (
    OUT
    / "stage28_2a3_execution_progress.json"
)

FIT_LEDGER_PATH = (
    OUT
    / "stage28_2a3_fit_ledger.json"
)

RESULT_PATH = (
    OUT
    / "stage28_2a3_random_natural_seed43_result.json"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


# =================================================================================================
# 2. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(
    *args,
    check=True,
):
    return (
        run(
            ["git", *args],
            check=check,
        ).stdout
        or ""
    ).strip()


def require_file(path):

    path = Path(path)

    if not path.is_file():

        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):

    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path,
    obj,
):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array(
    arr,
    block_rows=65_536,
):
    h = hashlib.sha256()

    for start in range(
        0,
        arr.shape[0],
        block_rows,
    ):

        stop = min(
            start + block_rows,
            arr.shape[0],
        )

        block = np.ascontiguousarray(
            arr[
                start:stop
            ]
        )

        h.update(
            block.view(
                np.uint8
            )
        )

    return h.hexdigest()


def canonical_json_sha256(obj):

    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


def safe_div(a, b):

    return (
        float(a) / float(b)
        if b
        else 0.0
    )


def metrics_from_counts(
    tp,
    fp,
    tn,
    fn,
):
    total = (
        tp
        + fp
        + tn
        + fn
    )

    return {
        "accuracy":
            safe_div(
                tp + tn,
                total,
            ),

        "precision":
            safe_div(
                tp,
                tp + fp,
            ),

        "recall":
            safe_div(
                tp,
                tp + fn,
            ),

        "fpr":
            safe_div(
                fp,
                fp + tn,
            ),

        "f1":
            safe_div(
                2 * tp,
                (2 * tp) + fp + fn,
            ),

        "f2":
            safe_div(
                5 * tp,
                (5 * tp) + (4 * fn) + fp,
            ),

        "tp":
            int(tp),

        "fp":
            int(fp),

        "tn":
            int(tn),

        "fn":
            int(fn),
    }


def evaluate_threshold(
    y_true,
    probability_float32,
    integer_percent,
):
    threshold = (
        integer_percent
        / 100.0
    )

    threshold32 = np.float32(
        threshold
    )

    pred = (
        probability_float32
        >= threshold32
    )

    positive = (
        y_true
        == 1
    )

    negative = ~positive


    tp = int(
        np.count_nonzero(
            pred
            & positive
        )
    )

    fp = int(
        np.count_nonzero(
            pred
            & negative
        )
    )

    fn = int(
        np.count_nonzero(
            (~pred)
            & positive
        )
    )

    tn = int(
        np.count_nonzero(
            (~pred)
            & negative
        )
    )


    row = metrics_from_counts(
        tp,
        fp,
        tn,
        fn,
    )

    row.update(
        {
            "threshold_integer_percent":
                int(integer_percent),

            "threshold":
                float(threshold),

            "threshold_float32_runtime":
                float(threshold32),
        }
    )

    return row


def build_grid(
    y_true,
    probability_float32,
):
    return [
        evaluate_threshold(
            y_true,
            probability_float32,
            pct,
        )
        for pct
        in range(
            5,
            96,
        )
    ]


def choose_balanced(grid):

    return max(
        grid,
        key=lambda r: (
            r["f1"],
            -r["fpr"],
            r["recall"],
            -abs(
                r["threshold"]
                - 0.50
            ),
            -r["threshold"],
        ),
    )


def choose_security(grid):

    feasible = [
        r
        for r
        in grid
        if r[
            "fpr"
        ] <= 0.05
    ]

    if not feasible:
        return None

    return max(
        feasible,
        key=lambda r: (
            r["f2"],
            -r["fpr"],
            r["recall"],
            -r["threshold"],
        ),
    )


def recover_github_token():

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]

    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()

        for label in labels:

            try:
                value = client.get_secret(
                    label
                )

            except Exception:
                value = None

            if (
                isinstance(
                    value,
                    str,
                )
                and value.strip()
            ):
                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass


    for label in labels:

        value = os.environ.get(
            label
        )

        if (
            isinstance(
                value,
                str,
            )
            and value.strip()
        ):
            return (
                value.strip(),
                f"environment:{label}",
            )


    raise RuntimeError(
        "GitHub credential unavailable."
    )


def authenticated_push(token):

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode("utf-8")
    ).decode("ascii")


    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )


    if p.returncode != 0:

        raise RuntimeError(
            "Authenticated git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )


    return (
        p.stdout
        + p.stderr
    ).strip()


def write_progress(
    *,
    state,
    fit3_state,
    fit4_state,
    xgb_sha=None,
    lgbm_sha=None,
):
    write_json(
        PROGRESS_PATH,
        {
            "stage":
                "Stage28-2A3",

            "evaluation_cell_id":
                "28A_STAGE22::RANDOM_NATURAL::SEED43",

            "durable_parent":
                EXPECTED_PARENT,

            "updated_at_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "state":
                state,

            "pre_cell_new_fits_consumed":
                2,

            "fit_3": {
                "component_id":
                    "C003",

                "learner":
                    "XGBOOST",

                "state":
                    fit3_state,

                "model_sha256":
                    xgb_sha,
            },

            "fit_4": {
                "component_id":
                    "C004",

                "learner":
                    "LIGHTGBM",

                "state":
                    fit4_state,

                "model_sha256":
                    lgbm_sha,
            },

            "shared_final_holdout_openings":
                0,
        },
    )


# =================================================================================================
# 3. EXACT DURABLE-PARENT GATE
# =================================================================================================

banner(
    "STAGE28-2A3 — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


if not remote_line:

    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote = (
    remote_line.split()[0]
)

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    head,
)

print(
    "origin/main    :",
    origin,
)

print(
    "Remote main    :",
    remote,
)

print(
    "Branch         :",
    branch,
)

print(
    "Git clean      :",
    not bool(status),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-2A3 parent mismatch."
    )


if branch != "main":

    raise RuntimeError(
        "Expected branch main."
    )


if status:

    raise RuntimeError(
        "Repository must be clean:\n"
        + status
    )


if OUT.exists():

    raise RuntimeError(
        "Stage28-2A3 output already exists.\n"
        "Do not overwrite or rerun a partial scientific checkpoint."
    )


print()
print(
    "[PASS] exact clean Stage28-2A2 parent"
)


# =================================================================================================
# 4. DURABILITY + GIT IDENTITY BEFORE ANY FIT
# =================================================================================================

banner(
    "GITHUB DURABILITY / IDENTITY GATE"
)

github_token, token_source = (
    recover_github_token()
)


print(
    "[PASS] credential recovered from",
    token_source,
)

print(
    "[PASS] token intentionally not displayed"
)


run(
    [
        "git",
        "config",
        "user.name",
        "Stage28 Kaggle",
    ]
)

run(
    [
        "git",
        "config",
        "user.email",
        "stage28-kaggle@users.noreply.github.com",
    ]
)


print(
    "user.name :",
    git(
        "config",
        "--get",
        "user.name",
    ),
)

print(
    "user.email:",
    git(
        "config",
        "--get",
        "user.email",
    ),
)


remote_url = git(
    "remote",
    "get-url",
    "origin",
)


if (
    remote_url.startswith(
        "https://"
    )
    and
    "@github.com"
    in remote_url
):
    raise RuntimeError(
        "Remote URL contains embedded credentials."
    )


print()
print(
    "[PASS] commit/push plumbing ready before FIT #3"
)


# =================================================================================================
# 5. LIBRARY / CPU GATE
# =================================================================================================

banner(
    "FROZEN MODEL RUNTIME GATE"
)

print(
    "XGBoost :",
    xgb.__version__,
)

print(
    "LightGBM:",
    lgb.__version__,
)


if (
    xgb.__version__
    != EXPECTED_XGB_VERSION
):

    raise RuntimeError(
        "XGBoost version drift."
    )


if (
    lgb.__version__
    != EXPECTED_LGBM_VERSION
):

    raise RuntimeError(
        "LightGBM version drift."
    )


os.environ[
    "CUDA_VISIBLE_DEVICES"
] = ""


print(
    "[PASS] Stage28 CPU-only execution policy active"
)


# =================================================================================================
# 6. PRIOR DURABLE FIT LEDGER
# =================================================================================================

banner(
    "PRIOR DURABLE STAGE28 FIT LEDGER"
)

require_file(
    STAGE28_2A2_CHECKSUMS
)

ledger_2a2 = read_json(
    require_file(
        STAGE28_2A2_LEDGER
    )
)

result_2a2 = read_json(
    require_file(
        STAGE28_2A2_RESULT
    )
)


if (
    ledger_2a2[
        "status"
    ]
    != "FIT_002_SUCCESSFULLY_CONSUMED"
):

    raise RuntimeError(
        "FIT #2 durable ledger invalid."
    )


if (
    int(
        ledger_2a2[
            "cumulative_new_fits_consumed"
        ]
    )
    != 2
):

    raise RuntimeError(
        "Expected exactly two prior new fits."
    )


if (
    int(
        ledger_2a2[
            "new_fits_remaining"
        ]
    )
    != 106
):

    raise RuntimeError(
        "Expected 106 fits remaining."
    )


if (
    int(
        result_2a2[
            "scientific_accounting"
        ][
            "shared_final_holdout_openings"
        ]
    )
    != 0
):

    raise RuntimeError(
        "Shared final holdout was previously opened."
    )


print(
    "Authorized:",
    EXPECTED_TOTAL_NEW_FITS,
)

print(
    "Consumed  : 2"
)

print(
    "Remaining : 106"
)

print()

print(
    "[PASS] FIT #3 and FIT #4 are next"
)


# =================================================================================================
# 7. STAGE28-2A0 RANDOM MEMBERSHIP GATE
# =================================================================================================

banner(
    "FROZEN RANDOM_NATURAL MEMBERSHIP GATE"
)

receipt_2a0 = read_json(
    require_file(
        STAGE28_2A0_RECEIPT
    )
)


random_receipt = (
    receipt_2a0[
        "random_natural"
    ]
)


if (
    random_receipt[
        "validation_packbits_sha256"
    ]
    != EXPECTED_RANDOM_PACKBITS_SHA
):

    raise RuntimeError(
        "Frozen random-validation packbits identity changed."
    )


if (
    int(
        random_receipt[
            "train"
        ][
            "rows"
        ]
    )
    != EXPECTED_TRAIN_ROWS
):

    raise RuntimeError(
        "Random train row count changed."
    )


if (
    int(
        random_receipt[
            "validation"
        ][
            "rows"
        ]
    )
    != EXPECTED_VAL_ROWS
):

    raise RuntimeError(
        "Random validation row count changed."
    )


X = np.load(
    require_file(
        X_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)

y = np.load(
    require_file(
        Y_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)

train_idx = np.load(
    require_file(
        RANDOM_TRAIN_IDX_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)

validation_idx = np.load(
    require_file(
        RANDOM_VAL_IDX_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)


if X.shape != (
    EXPECTED_ROWS,
    EXPECTED_FEATURES,
):

    raise RuntimeError(
        "Stage22 execution matrix shape mismatch."
    )


if X.dtype != np.float64:

    raise RuntimeError(
        "Stage22 execution matrix dtype mismatch."
    )


if len(
    train_idx
) != EXPECTED_TRAIN_ROWS:

    raise RuntimeError(
        "Runtime train index count mismatch."
    )


if len(
    validation_idx
) != EXPECTED_VAL_ROWS:

    raise RuntimeError(
        "Runtime validation index count mismatch."
    )


train_idx_sha = sha256_array(
    train_idx
)

validation_idx_sha = sha256_array(
    validation_idx
)


if (
    train_idx_sha
    != random_receipt[
        "train_index_content_sha256"
    ]
):

    raise RuntimeError(
        "Random train index logical SHA mismatch."
    )


if (
    validation_idx_sha
    != random_receipt[
        "validation_index_content_sha256"
    ]
):

    raise RuntimeError(
        "Random validation index logical SHA mismatch."
    )


y_train_check = np.asarray(
    y[
        train_idx
    ]
)

y_validation = np.asarray(
    y[
        validation_idx
    ],
    dtype=np.uint8,
)


train_attack = int(
    y_train_check.sum()
)

val_attack = int(
    y_validation.sum()
)


if train_attack != EXPECTED_TRAIN_ATTACK:

    raise RuntimeError(
        "Random train attack count mismatch."
    )


if (
    EXPECTED_TRAIN_ROWS
    - train_attack
    != EXPECTED_TRAIN_BENIGN
):

    raise RuntimeError(
        "Random train benign count mismatch."
    )


if val_attack != EXPECTED_VAL_ATTACK:

    raise RuntimeError(
        "Random validation attack count mismatch."
    )


if (
    EXPECTED_VAL_ROWS
    - val_attack
    != EXPECTED_VAL_BENIGN
):

    raise RuntimeError(
        "Random validation benign count mismatch."
    )


del y_train_check
gc.collect()


print(
    "Train rows       :",
    f"{EXPECTED_TRAIN_ROWS:,}",
)

print(
    "Validation rows  :",
    f"{EXPECTED_VAL_ROWS:,}",
)

print(
    "Train index SHA  :",
    train_idx_sha,
)

print(
    "Validation SHA   :",
    validation_idx_sha,
)

print()

print(
    "[PASS] seed43 uses exact same RANDOM_NATURAL membership"
)


# =================================================================================================
# 8. C003 / C004 MANIFEST GATE
# =================================================================================================

banner(
    "C003 / C004 FROZEN COMPONENT GATE"
)

manifest_sha = sha256_file(
    require_file(
        COMPONENT_MANIFEST_PATH
    )
)


if (
    manifest_sha
    != EXPECTED_COMPONENT_MANIFEST_SHA
):

    raise RuntimeError(
        "Stage28 component manifest SHA mismatch."
    )


with COMPONENT_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:

    manifest_rows = list(
        csv.DictReader(f)
    )


by_component = {
    row[
        "component_id"
    ]:
    row

    for row
    in manifest_rows
}


c003 = by_component.get(
    "C003"
)

c004 = by_component.get(
    "C004"
)


if c003 is None or c004 is None:

    raise RuntimeError(
        "C003/C004 missing from frozen manifest."
    )


expected_c003 = {
    "arm":
        "28A_TRAINING_SEED_STABILITY",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "RANDOM_NATURAL",

    "evaluation_cell_id":
        "28A_STAGE22::RANDOM_NATURAL::SEED43",

    "learner":
        "XGBOOST",

    "configuration_id":
        "XGB_11",

    "model_seed":
        "43",

    "compute_backend":
        "CPU",

    "parameter_set_id":
        "STAGE22_FULL::XGBOOST::SEED43",

    "parameter_sha256":
        EXPECTED_XGB_PARAMETER_SHA,

    "membership_reference":
        "STAGE22R_FROZEN_RANDOM_NATURAL_MEMBERSHIP",

    "fit_action":
        "NEW_FIT_AUTHORIZED",

    "new_fit_budget_units":
        "1",
}


expected_c004 = {
    "arm":
        "28A_TRAINING_SEED_STABILITY",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "RANDOM_NATURAL",

    "evaluation_cell_id":
        "28A_STAGE22::RANDOM_NATURAL::SEED43",

    "learner":
        "LIGHTGBM",

    "configuration_id":
        "LGBM_11",

    "model_seed":
        "43",

    "compute_backend":
        "CPU",

    "parameter_set_id":
        "STAGE22_FULL::LIGHTGBM::SEED43",

    "parameter_sha256":
        EXPECTED_LGBM_PARAMETER_SHA,

    "membership_reference":
        "STAGE22R_FROZEN_RANDOM_NATURAL_MEMBERSHIP",

    "fit_action":
        "NEW_FIT_AUTHORIZED",

    "new_fit_budget_units":
        "1",
}


for field, expected in (
    expected_c003.items()
):

    if c003[
        field
    ] != expected:

        raise RuntimeError(
            f"C003 {field} mismatch.\n"
            f"expected={expected}\n"
            f"actual={c003[field]}"
        )


for field, expected in (
    expected_c004.items()
):

    if c004[
        field
    ] != expected:

        raise RuntimeError(
            f"C004 {field} mismatch.\n"
            f"expected={expected}\n"
            f"actual={c004[field]}"
        )


print(
    "[PASS] C003 = XGBoost seed43 CPU NEW FIT"
)

print(
    "[PASS] C004 = LightGBM seed43 CPU NEW FIT"
)


# =================================================================================================
# 9. EXACT SEED43 PARAMETER GATE
# =================================================================================================

banner(
    "SEED43 FROZEN PARAMETER-SET GATE"
)

parameter_doc = read_json(
    require_file(
        PARAMETER_SETS_PATH
    )
)

parameter_sets = (
    parameter_doc[
        "sets"
    ]
)


xgb_params = dict(
    parameter_sets[
        c003[
            "parameter_set_id"
        ]
    ][
        "parameters"
    ]
)


lgbm_params = dict(
    parameter_sets[
        c004[
            "parameter_set_id"
        ]
    ][
        "parameters"
    ]
)


xgb_param_sha = canonical_json_sha256(
    xgb_params
)

lgbm_param_sha = canonical_json_sha256(
    lgbm_params
)


print(
    "XGB expected :",
    EXPECTED_XGB_PARAMETER_SHA,
)

print(
    "XGB actual   :",
    xgb_param_sha,
)

print()

print(
    "LGBM expected:",
    EXPECTED_LGBM_PARAMETER_SHA,
)

print(
    "LGBM actual  :",
    lgbm_param_sha,
)


if xgb_param_sha != EXPECTED_XGB_PARAMETER_SHA:

    raise RuntimeError(
        "Seed43 XGBoost parameter SHA mismatch."
    )


if lgbm_param_sha != EXPECTED_LGBM_PARAMETER_SHA:

    raise RuntimeError(
        "Seed43 LightGBM parameter SHA mismatch."
    )


if (
    int(
        xgb_params[
            "random_state"
        ]
    )
    != 43
):

    raise RuntimeError(
        "XGBoost random_state != 43."
    )


if (
    int(
        lgbm_params[
            "random_state"
        ]
    )
    != 43
):

    raise RuntimeError(
        "LightGBM random_state != 43."
    )


if not (
    xgb_params[
        "device"
    ] == "cpu"
    and
    xgb_params[
        "tree_method"
    ] == "hist"
):

    raise RuntimeError(
        "Seed43 XGBoost backend not CPU/hist."
    )


if (
    lgbm_params[
        "device_type"
    ]
    != "cpu"
):

    raise RuntimeError(
        "Seed43 LightGBM backend not CPU."
    )


print()
print(
    "[PASS] seed43 parameter sets exact"
)

print(
    "[PASS] only frozen training seed changed"
)


# =================================================================================================
# 10. THRESHOLD POLICY GATE
# =================================================================================================

banner(
    "FROZEN STAGE22 THRESHOLD POLICY"
)

threshold_policy = read_json(
    require_file(
        THRESHOLD_POLICY_PATH
    )
)


policy = (
    threshold_policy[
        "stage22_full"
    ]
)


if (
    policy[
        "selection_population"
    ]
    != "EACH_PARENT_CELL_FROZEN_DEVELOPMENT_VALIDATION"
):

    raise RuntimeError(
        "Threshold-selection population changed."
    )


if not (
    policy[
        "grid"
    ][
        "integer_percent_start"
    ] == 5
    and
    policy[
        "grid"
    ][
        "integer_percent_stop_inclusive"
    ] == 95
    and
    policy[
        "grid"
    ][
        "integer_step"
    ] == 1
    and
    policy[
        "grid"
    ][
        "count"
    ] == 91
):

    raise RuntimeError(
        "Stage22 threshold grid changed."
    )


if (
    policy[
        "final_holdout_threshold_search"
    ]
    != "FORBIDDEN"
):

    raise RuntimeError(
        "Final-holdout threshold rule changed."
    )


print(
    "[PASS] threshold policy exact"
)


# =================================================================================================
# 11. CREATE DURABLE OUTPUT / INITIAL PROGRESS RECEIPT
# =================================================================================================

OUT.mkdir(
    parents=True,
    exist_ok=False,
)


write_progress(
    state="PREFIT_VALIDATION_COMPLETE",
    fit3_state="NOT_STARTED",
    fit4_state="NOT_STARTED",
)


# =================================================================================================
# 12. MATERIALIZE EXACT RANDOM TRAINING MATRIX
# =================================================================================================

banner(
    "MATERIALIZE RANDOM_NATURAL TRAINING MATRIX"
)

materialize_start = time.perf_counter()


X_train = np.asarray(
    X[
        train_idx,
        :
    ],
    dtype=np.float64,
    order="C",
)

y_train = np.asarray(
    y[
        train_idx
    ],
    dtype=np.uint8,
    order="C",
)


materialize_seconds = (
    time.perf_counter()
    - materialize_start
)


if X_train.shape != (
    EXPECTED_TRAIN_ROWS,
    EXPECTED_FEATURES,
):

    raise RuntimeError(
        "X_train shape mismatch."
    )


if y_train.shape != (
    EXPECTED_TRAIN_ROWS,
):

    raise RuntimeError(
        "y_train shape mismatch."
    )


if int(
    y_train.sum()
) != EXPECTED_TRAIN_ATTACK:

    raise RuntimeError(
        "y_train attack count mismatch."
    )


if np.isinf(
    X_train
).any():

    raise RuntimeError(
        "Infinity exists in training matrix."
    )


print(
    "Rows    :",
    f"{X_train.shape[0]:,}",
)

print(
    "Features:",
    X_train.shape[1],
)

print(
    "Dtype   :",
    X_train.dtype,
)

print(
    "Payload :",
    f"{X_train.nbytes / (1024**3):.3f} GiB",
)

print(
    "Materialization seconds:",
    materialize_seconds,
)

print()

print(
    "[PASS] exact RANDOM_NATURAL training population ready"
)


# =================================================================================================
# 13. FIT #3 — C003 XGBOOST SEED43
# =================================================================================================

banner(
    "FIT #3 — C003 XGBOOST XGB_11 SEED43 CPU"
)

write_progress(
    state="FIT_003_STARTED",
    fit3_state="STARTED_NOT_YET_CONFIRMED",
    fit4_state="NOT_STARTED",
)


print(
    "Component   : C003"
)

print(
    "Learner     : XGBOOST"
)

print(
    "Seed        : 43"
)

print(
    "Backend     : CPU"
)

print(
    "tree_method :",
    xgb_params[
        "tree_method"
    ],
)

print(
    "Estimators  :",
    xgb_params[
        "n_estimators"
    ],
)

print()

print(
    "Starting Stage28 scientific FIT #3..."
)


fit3_start = time.perf_counter()


xgb_model = xgb.XGBClassifier(
    **xgb_params
)


xgb_model.fit(
    X_train,
    y_train,
)


fit3_seconds = (
    time.perf_counter()
    - fit3_start
)


xgb_booster = (
    xgb_model.get_booster()
)

xgb_rounds = int(
    xgb_booster.num_boosted_rounds()
)


if xgb_rounds != 400:

    raise RuntimeError(
        "C003 boosted-round count != 400."
    )


if int(
    xgb_model.n_features_in_
) != EXPECTED_FEATURES:

    raise RuntimeError(
        "C003 feature count != 70."
    )


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_model_sha = sha256_file(
    XGB_MODEL_PATH
)


write_progress(
    state="FIT_003_SUCCESS",
    fit3_state="SUCCESS",
    fit4_state="NOT_STARTED",
    xgb_sha=xgb_model_sha,
)


print()
print(
    "FIT #3 completed."
)

print(
    "Fit seconds:",
    fit3_seconds,
)

print(
    "Model SHA256:",
    xgb_model_sha,
)

print()

print(
    "[PASS] C003 FIT #3 SUCCESS"
)

print(
    "[LEDGER] cumulative successful NEW fits = 3"
)


# =================================================================================================
# 14. FIT #4 — C004 LIGHTGBM SEED43
# =================================================================================================

banner(
    "FIT #4 — C004 LIGHTGBM LGBM_11 SEED43 CPU"
)

write_progress(
    state="FIT_004_STARTED",
    fit3_state="SUCCESS",
    fit4_state="STARTED_NOT_YET_CONFIRMED",
    xgb_sha=xgb_model_sha,
)


print(
    "Component   : C004"
)

print(
    "Learner     : LIGHTGBM"
)

print(
    "Seed        : 43"
)

print(
    "Backend     : CPU"
)

print(
    "Estimators  :",
    lgbm_params[
        "n_estimators"
    ],
)

print()

print(
    "Starting Stage28 scientific FIT #4..."
)


fit4_start = time.perf_counter()


lgbm_model = lgb.LGBMClassifier(
    **lgbm_params
)


lgbm_model.fit(
    X_train,
    y_train,
)


fit4_seconds = (
    time.perf_counter()
    - fit4_start
)


lgb_booster = (
    lgbm_model.booster_
)


lgb_iterations = int(
    lgb_booster.current_iteration()
)

lgb_trees = int(
    lgb_booster.num_trees()
)

lgb_features = int(
    lgb_booster.num_feature()
)


if lgb_iterations != 400:

    raise RuntimeError(
        "C004 iteration count != 400."
    )


if lgb_trees != 400:

    raise RuntimeError(
        "C004 tree count != 400."
    )


if lgb_features != EXPECTED_FEATURES:

    raise RuntimeError(
        "C004 feature count != 70."
    )


lgb_booster.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_model_sha = sha256_file(
    LGBM_MODEL_PATH
)


write_progress(
    state="FIT_003_AND_004_SUCCESS",
    fit3_state="SUCCESS",
    fit4_state="SUCCESS",
    xgb_sha=xgb_model_sha,
    lgbm_sha=lgbm_model_sha,
)


print()
print(
    "FIT #4 completed."
)

print(
    "Fit seconds:",
    fit4_seconds,
)

print(
    "Model SHA256:",
    lgbm_model_sha,
)

print()

print(
    "[PASS] C004 FIT #4 SUCCESS"
)

print(
    "[LEDGER] cumulative successful NEW fits = 4"
)


# Training population no longer required.
del X_train
del y_train

gc.collect()


# =================================================================================================
# 15. VALIDATION INFERENCE
# =================================================================================================

banner(
    "RANDOM_NATURAL SEED43 — DEVELOPMENT VALIDATION INFERENCE"
)

ensemble_probability = np.empty(
    EXPECTED_VAL_ROWS,
    dtype=np.float32,
)


chunk_size = 65_536

inference_start = time.perf_counter()


for start in range(
    0,
    EXPECTED_VAL_ROWS,
    chunk_size,
):

    stop = min(
        start + chunk_size,
        EXPECTED_VAL_ROWS,
    )


    idx_chunk = np.asarray(
        validation_idx[
            start:stop
        ],
        dtype=np.int64,
    )


    X_chunk = np.asarray(
        X[
            idx_chunk,
            :
        ],
        dtype=np.float64,
        order="C",
    )


    p_xgb = np.asarray(
        xgb_model.predict_proba(
            X_chunk
        )[
            :,
            1
        ],
        dtype=np.float64,
    )


    p_lgbm = np.asarray(
        lgbm_model.predict_proba(
            X_chunk
        )[
            :,
            1
        ],
        dtype=np.float64,
    )


    expected_chunk = (
        stop
        - start
    )


    if len(
        p_xgb
    ) != expected_chunk:

        raise RuntimeError(
            "C003 validation inference length mismatch."
        )


    if len(
        p_lgbm
    ) != expected_chunk:

        raise RuntimeError(
            "C004 validation inference length mismatch."
        )


    combined64 = (
        (0.5 * p_lgbm)
        +
        (0.5 * p_xgb)
    )


    ensemble_probability[
        start:stop
    ] = combined64.astype(
        np.float32,
        copy=False,
    )


    del idx_chunk
    del X_chunk
    del p_xgb
    del p_lgbm
    del combined64

    gc.collect()


    if (
        start == 0
        or stop == EXPECTED_VAL_ROWS
        or (
            stop
            // chunk_size
        ) % 10 == 0
    ):
        print(
            f"  inferred "
            f"{stop:,} / {EXPECTED_VAL_ROWS:,}"
        )


inference_seconds = (
    time.perf_counter()
    - inference_start
)


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):

    raise RuntimeError(
        "Non-finite ensemble probability."
    )


if (
    np.any(
        ensemble_probability < 0
    )
    or
    np.any(
        ensemble_probability > 1
    )
):

    raise RuntimeError(
        "Ensemble probability outside [0,1]."
    )


print()
print(
    "Inference seconds:",
    inference_seconds,
)

print()

print(
    "[PASS] C003 + C004 seed43 validation inference complete"
)


# =================================================================================================
# 16. VALIDATION RANKING
# =================================================================================================

banner(
    "RANDOM_NATURAL SEED43 — VALIDATION RANKING"
)

validation_roc_auc = float(
    roc_auc_score(
        y_validation,
        ensemble_probability,
    )
)


# Historical Stage22 self-tests for seeds42 established
# sklearn.average_precision_score as the inherited PR-AUC definition.
validation_pr_auc = float(
    average_precision_score(
        y_validation,
        ensemble_probability,
    )
)


print(
    "ROC-AUC:",
    validation_roc_auc,
)

print(
    "PR-AUC :",
    validation_pr_auc,
)

print(
    "PR definition: SKLEARN_AVERAGE_PRECISION_SCORE"
)


# =================================================================================================
# 17. FROZEN THRESHOLD SELECTION
# =================================================================================================

banner(
    "RANDOM_NATURAL SEED43 — FROZEN VALIDATION THRESHOLDS"
)

grid_rows = build_grid(
    y_validation,
    ensemble_probability,
)


standard = next(
    row
    for row
    in grid_rows
    if row[
        "threshold_integer_percent"
    ] == 50
)


balanced = choose_balanced(
    grid_rows
)

security = choose_security(
    grid_rows
)


if security is None:

    raise RuntimeError(
        "Seed43 security threshold infeasible "
        "under frozen FPR<=0.05 rule."
    )


print(
    "STANDARD:"
)

print(
    " threshold:",
    standard[
        "threshold"
    ],
)

print(
    " F1       :",
    standard[
        "f1"
    ],
)

print(
    " recall   :",
    standard[
        "recall"
    ],
)

print(
    " FPR      :",
    standard[
        "fpr"
    ],
)

print()

print(
    "BALANCED:"
)

print(
    " threshold:",
    balanced[
        "threshold"
    ],
)

print(
    " F1       :",
    balanced[
        "f1"
    ],
)

print(
    " recall   :",
    balanced[
        "recall"
    ],
)

print(
    " FPR      :",
    balanced[
        "fpr"
    ],
)

print()

print(
    "SECURITY:"
)

print(
    " threshold:",
    security[
        "threshold"
    ],
)

print(
    " F2       :",
    security[
        "f2"
    ],
)

print(
    " recall   :",
    security[
        "recall"
    ],
)

print(
    " FPR      :",
    security[
        "fpr"
    ],
)


# =================================================================================================
# 18. WRITE VALIDATION ARTIFACTS
# =================================================================================================

banner(
    "FREEZE STAGE28-2A3 VALIDATION ARTIFACTS"
)

np.savez_compressed(
    VALIDATION_PROB_PATH,

    ensemble_probability_float32=
        ensemble_probability,

    validation_global_idx_int32=
        np.asarray(
            validation_idx,
            dtype=np.int32,
        ),

    binary_label_uint8=
        y_validation,
)


grid_df = pd.DataFrame(
    grid_rows
)[
    [
        "threshold_integer_percent",
        "threshold",
        "threshold_float32_runtime",
        "accuracy",
        "precision",
        "recall",
        "fpr",
        "f1",
        "f2",
        "tp",
        "fp",
        "tn",
        "fn",
    ]
]


grid_df.to_csv(
    THRESHOLD_GRID_PATH,
    index=False,
    lineterminator="\n",
)


validation_probability_sha = sha256_file(
    VALIDATION_PROB_PATH
)

threshold_grid_sha = sha256_file(
    THRESHOLD_GRID_PATH
)


print(
    "Validation probability SHA:",
    validation_probability_sha,
)

print(
    "Threshold grid SHA       :",
    threshold_grid_sha,
)


# =================================================================================================
# 19. FINAL EXECUTION PROGRESS RECEIPT
# =================================================================================================

write_progress(
    state="FIT_003_AND_004_AND_VALIDATION_SUCCESS",
    fit3_state="SUCCESS",
    fit4_state="SUCCESS",
    xgb_sha=xgb_model_sha,
    lgbm_sha=lgbm_model_sha,
)


progress_sha = sha256_file(
    PROGRESS_PATH
)


# =================================================================================================
# 20. FIT LEDGER
# =================================================================================================

fit_ledger = {

    "stage":
        "Stage28-2A3",

    "durable_parent":
        EXPECTED_PARENT,

    "evaluation_cell_id":
        "28A_STAGE22::RANDOM_NATURAL::SEED43",

    "authorized_stage28_new_fit_budget":
        108,

    "pre_cell_new_fits_consumed":
        2,

    "this_cell": {
        "successful_new_fits":
            2,

        "new_fit_components": [
            "C003",
            "C004",
        ],

        "reused_components": [],
    },

    "cumulative_new_fits_consumed":
        4,

    "new_fits_remaining":
        104,

    "stage22_new_fits_consumed":
        4,

    "stage22_new_fits_remaining":
        14,

    "model_fits_attempted":
        2,

    "model_fits_successful":
        2,

    "status":
        "FITS_003_AND_004_SUCCESSFULLY_CONSUMED",
}


write_json(
    FIT_LEDGER_PATH,
    fit_ledger,
)


fit_ledger_sha = sha256_file(
    FIT_LEDGER_PATH
)


# =================================================================================================
# 21. RESULT
# =================================================================================================

result = {

    "stage":
        "Stage28-2A3",

    "status":
        "RANDOM_NATURAL_SEED43_VALIDATION_CHECKPOINT_FROZEN",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_PARENT,

    "arm":
        "28A_TRAINING_SEED_STABILITY",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "RANDOM_NATURAL",

    "training_seed":
        43,

    "evaluation_cell_id":
        "28A_STAGE22::RANDOM_NATURAL::SEED43",

    "membership": {
        "train_rows":
            EXPECTED_TRAIN_ROWS,

        "train_attack":
            EXPECTED_TRAIN_ATTACK,

        "train_benign":
            EXPECTED_TRAIN_BENIGN,

        "validation_rows":
            EXPECTED_VAL_ROWS,

        "validation_attack":
            EXPECTED_VAL_ATTACK,

        "validation_benign":
            EXPECTED_VAL_BENIGN,

        "train_index_content_sha256":
            train_idx_sha,

        "validation_index_content_sha256":
            validation_idx_sha,

        "membership_seed":
            42,

        "training_seed":
            43,

        "membership_changed_by_training_seed":
            False,
    },

    "models": {
        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "xgboost": {
            "component_id":
                "C003",

            "fit_action":
                "NEW_FIT_AUTHORIZED",

            "configuration":
                "XGB_11",

            "seed":
                43,

            "backend":
                "cpu",

            "parameters":
                xgb_params,

            "parameter_sha256":
                xgb_param_sha,

            "library_version":
                xgb.__version__,

            "boosted_rounds":
                xgb_rounds,

            "fit_seconds":
                float(
                    fit3_seconds
                ),

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                xgb_model_sha,
        },

        "lightgbm": {
            "component_id":
                "C004",

            "fit_action":
                "NEW_FIT_AUTHORIZED",

            "configuration":
                "LGBM_11",

            "seed":
                43,

            "backend":
                "cpu",

            "parameters":
                lgbm_params,

            "parameter_sha256":
                lgbm_param_sha,

            "library_version":
                lgb.__version__,

            "iterations":
                lgb_iterations,

            "trees":
                lgb_trees,

            "fit_seconds":
                float(
                    fit4_seconds
                ),

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                lgbm_model_sha,
        },
    },

    "training_materialization": {
        "seconds":
            float(
                materialize_seconds
            ),

        "rows":
            EXPECTED_TRAIN_ROWS,

        "features":
            EXPECTED_FEATURES,

        "dtype":
            "float64",
    },

    "validation_probability": {
        "rows":
            EXPECTED_VAL_ROWS,

        "roc_auc":
            validation_roc_auc,

        "pr_auc":
            validation_pr_auc,

        "pr_auc_definition":
            "SKLEARN_AVERAGE_PRECISION_SCORE",

        "artifact":
            VALIDATION_PROB_PATH.name,

        "artifact_sha256":
            validation_probability_sha,

        "inference_seconds":
            float(
                inference_seconds
            ),
    },

    "threshold_selection": {
        "selection_population":
            "FROZEN_RANDOM_NATURAL_DEVELOPMENT_VALIDATION",

        "selection_repeated_independently_for_seed":
            43,

        "final_holdout_threshold_search":
            "FORBIDDEN",

        "prediction_rule":
            (
                "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_"
                "PROBABILITY_GTE_THRESHOLD"
            ),

        "grid_integer_percent": [
            5,
            95,
        ],

        "grid_points":
            91,

        "balanced":
            (
                "MAX_F1; TIES LOWER_FPR, HIGHER_RECALL, "
                "CLOSER_TO_0_50, LOWER_THRESHOLD"
            ),

        "security":
            (
                "FPR<=0.05 EXACT; MAX_F2; TIES LOWER_FPR, "
                "HIGHER_RECALL, LOWER_THRESHOLD; NO_RELAXATION"
            ),

        "grid_artifact":
            THRESHOLD_GRID_PATH.name,

        "grid_sha256":
            threshold_grid_sha,
    },

    "operating_points": {
        "standard":
            standard,

        "balanced":
            balanced,

        "security": {
            "status":
                "AVAILABLE",

            "result":
                security,
        },
    },

    "artifacts": {
        "xgboost_model_sha256":
            xgb_model_sha,

        "lightgbm_model_sha256":
            lgbm_model_sha,

        "validation_probability_sha256":
            validation_probability_sha,

        "threshold_grid_sha256":
            threshold_grid_sha,

        "execution_progress_sha256":
            progress_sha,

        "fit_ledger_sha256":
            fit_ledger_sha,
    },

    "scientific_accounting": {
        "new_model_fits_before_cell":
            2,

        "new_model_fits_this_cell":
            2,

        "new_model_fits_cumulative":
            4,

        "new_model_fits_remaining":
            104,

        "existing_models_reused_this_cell":
            0,

        "development_validation_model_inference":
            True,

        "threshold_selection_completed":
            True,

        "shared_final_holdout_openings":
            0,

        "shared_final_holdout_predictor_rows_read":
            0,

        "shared_final_holdout_labels_read":
            0,

        "new_target_openings":
            0,

        "target_adaptive_choices":
            0,
    },

    "next_authorized_step":
        (
            "Stage28-2A4 — CHRONOLOGICAL_NATURAL seed43. "
            "C013 XGBoost and C014 LightGBM are both new CPU fits. "
            "Shared final holdout remains closed."
        ),
}


write_json(
    RESULT_PATH,
    result,
)


result_sha = sha256_file(
    RESULT_PATH
)


# =================================================================================================
# 22. CHECKSUM MANIFEST
# =================================================================================================

artifact_paths = [
    XGB_MODEL_PATH,
    LGBM_MODEL_PATH,
    VALIDATION_PROB_PATH,
    THRESHOLD_GRID_PATH,
    PROGRESS_PATH,
    FIT_LEDGER_PATH,
    RESULT_PATH,
]


checksum_lines = []


for artifact in artifact_paths:

    checksum_lines.append(
        f"{sha256_file(artifact)}  {artifact.name}"
    )


CHECKSUMS_PATH.write_text(
    "\n".join(
        checksum_lines
    )
    + "\n",
    encoding="utf-8",
)


print()
print(
    "Durable artifact checksums:"
)

print()

print(
    CHECKSUMS_PATH.read_text(
        encoding="utf-8"
    ).rstrip()
)


# =================================================================================================
# 23. SELF-VERIFY ARTIFACT UNIVERSE
# =================================================================================================

banner(
    "SELF-VERIFY STAGE28-2A3 ARTIFACTS"
)

expected_files = {
    XGB_MODEL_PATH.name,
    LGBM_MODEL_PATH.name,
    VALIDATION_PROB_PATH.name,
    THRESHOLD_GRID_PATH.name,
    PROGRESS_PATH.name,
    FIT_LEDGER_PATH.name,
    RESULT_PATH.name,
    CHECKSUMS_PATH.name,
}


actual_files = {
    p.name
    for p
    in OUT.iterdir()
    if p.is_file()
}


if actual_files != expected_files:

    raise RuntimeError(
        "Unexpected Stage28-2A3 artifact universe.\n"
        f"Expected={sorted(expected_files)}\n"
        f"Actual={sorted(actual_files)}"
    )


for line in (
    CHECKSUMS_PATH.read_text(
        encoding="utf-8"
    ).splitlines()
):

    digest, filename = line.split(
        None,
        1,
    )

    filename = (
        filename.strip()
        .lstrip("*")
    )


    actual = sha256_file(
        OUT
        / filename
    )


    if actual != digest:

        raise RuntimeError(
            f"Checksum mismatch: {filename}"
        )


    print(
        "[PASS]",
        filename,
        digest,
    )


print()
print(
    "[PASS] all Stage28-2A3 artifacts exact"
)


# =================================================================================================
# 24. SCIENTIFIC LEDGER
# =================================================================================================

banner(
    "STAGE28-2A3 SCIENTIFIC LEDGER"
)

print(
    "NEW fits authorized : 108"
)

print(
    "Pre-cell consumed   : 2"
)

print(
    "This cell consumed  : 2"
)

print(
    "Cumulative consumed : 4"
)

print(
    "Remaining           : 104"
)

print()

print(
    "C003 XGBoost fit     : SUCCESS"
)

print(
    "C004 LightGBM fit    : SUCCESS"
)

print(
    "Validation inference : COMPLETE"
)

print(
    "Threshold selection : COMPLETE"
)

print(
    "Final holdout opening: 0"
)

print()

print(
    "[PASS] FIT #3 and FIT #4 accounted exactly once"
)


# =================================================================================================
# 25. AUTHORIZED GIT CHANGE GATE
# =================================================================================================

banner(
    "AUTHORIZED GIT-CHANGE GATE"
)

tracked_changes = (
    git(
        "diff",
        "--name-only",
    )
    .splitlines()
)


if tracked_changes:

    raise RuntimeError(
        "Unexpected tracked modifications:\n"
        + "\n".join(
            tracked_changes
        )
    )


expected_untracked = {
    str(
        (
            OUT
            / filename
        ).relative_to(
            REPO
        )
    )
    for filename
    in expected_files
}


all_untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    )
    .splitlines()
)


if all_untracked != expected_untracked:

    raise RuntimeError(
        "Unexpected untracked files.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                expected_untracked
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                all_untracked
            )
        )
    )


print(
    "[PASS] exactly eight authorized Stage28-2A3 files exist"
)


# =================================================================================================
# 26. STAGE EXACT FILES
# =================================================================================================

banner(
    "STAGE STAGE28-2A3 CHECKPOINT"
)

for relative in sorted(
    expected_untracked
):

    run(
        [
            "git",
            "add",
            "--",
            relative,
        ]
    )


staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    )
    .splitlines()
)


if staged != expected_untracked:

    raise RuntimeError(
        "Staged file universe mismatch."
    )


for relative in sorted(
    staged
):

    print(
        " ",
        relative,
    )


print()
print(
    "[PASS] exactly eight files staged"
)


# =================================================================================================
# 27. FINAL PRE-COMMIT GATE
# =================================================================================================

banner(
    "FINAL PRE-COMMIT GATE"
)

if git(
    "rev-parse",
    "HEAD",
) != EXPECTED_PARENT:

    raise RuntimeError(
        "HEAD changed before commit."
    )


staged_final = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    )
    .splitlines()
)


if staged_final != expected_untracked:

    raise RuntimeError(
        "Staged universe changed before commit."
    )


print(
    "FIT #1 durable     : YES"
)

print(
    "FIT #2 durable     : YES"
)

print(
    "FIT #3 successful  : YES"
)

print(
    "FIT #4 successful  : YES"
)

print(
    "Cumulative fits    : 4"
)

print(
    "Remaining          : 104"
)

print(
    "Holdout openings   : 0"
)

print()

print(
    "[PASS] ready to commit seed43 RANDOM_NATURAL checkpoint"
)


# =================================================================================================
# 28. COMMIT
# =================================================================================================

banner(
    "COMMIT STAGE28-2A3"
)

commit_proc = run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ]
)


print(
    commit_proc.stdout.strip()
)


new_head = git(
    "rev-parse",
    "HEAD",
)

new_parent = git(
    "rev-parse",
    "HEAD^",
)

new_subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)


print()
print(
    "Parent :",
    new_parent,
)

print(
    "Commit :",
    new_head,
)

print(
    "Subject:",
    new_subject,
)


if new_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage28-2A3 commit parent mismatch."
    )


if new_subject != COMMIT_MESSAGE:

    raise RuntimeError(
        "Stage28-2A3 commit subject mismatch."
    )


print()
print(
    "[PASS] Stage28-2A3 committed"
)


# =================================================================================================
# 29. PUSH
# =================================================================================================

banner(
    "PUSH STAGE28-2A3"
)

push_output = authenticated_push(
    github_token
)


if push_output:

    print(
        push_output
    )


github_token = None


# =================================================================================================
# 30. REMOTE DURABILITY
# =================================================================================================

banner(
    "REMOTE DURABILITY VERIFICATION"
)

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_final = git(
    "rev-parse",
    "HEAD",
)

origin_final = git(
    "rev-parse",
    "origin/main",
)

remote_final_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


if not remote_final_line:

    raise RuntimeError(
        "Unable to resolve pushed remote main."
    )


remote_final = (
    remote_final_line.split()[0]
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD :",
    local_final,
)

print(
    "origin/main:",
    origin_final,
)

print(
    "Remote main:",
    remote_final,
)

print(
    "Git clean  :",
    not bool(
        final_status
    ),
)


if not (
    local_final
    == origin_final
    == remote_final
    == new_head
):

    raise RuntimeError(
        "Stage28-2A3 remote durability gate failed."
    )


if final_status:

    raise RuntimeError(
        "Repository dirty after Stage28-2A3 push:\n"
        + final_status
    )


print()
print(
    "[PASS] Stage28-2A3 remotely durable"
)

print(
    "[PASS] repository clean"
)


# =================================================================================================
# 31. FINAL
# =================================================================================================

banner(
    "STAGE28-2A3 — RANDOM_NATURAL SEED43 COMPLETE"
)

print(
    "Durable commit:"
)

print(
    " ",
    new_head,
)

print()

print(
    "Evaluation cell:"
)

print(
    "  28A_STAGE22::RANDOM_NATURAL::SEED43"
)

print()

print(
    "Components:"
)

print(
    "  C003 XGBoost  seed43 CPU = FIT #3 SUCCESS"
)

print(
    "  C004 LightGBM seed43 CPU = FIT #4 SUCCESS"
)

print()

print(
    "Validation ranking:"
)

print(
    "  ROC-AUC =",
    validation_roc_auc,
)

print(
    "  PR-AUC  =",
    validation_pr_auc,
)

print()

print(
    "Operating thresholds:"
)

print(
    "  STANDARD =",
    standard[
        "threshold"
    ],
)

print(
    "  BALANCED =",
    balanced[
        "threshold"
    ],
)

print(
    "  SECURITY =",
    security[
        "threshold"
    ],
)

print()

print(
    "Fit ledger:"
)

print(
    "  NEW authorized = 108"
)

print(
    "  consumed       = 4"
)

print(
    "  remaining      = 104"
)

print()

print(
    "Stage22 progress:"
)

print(
    "  RANDOM seed42 = COMPLETE"
)

print(
    "  CHRONO seed42 = COMPLETE"
)

print(
    "  RANDOM seed43 = COMPLETE"
)

print()

print(
    "Shared final holdout:"
)

print(
    "  OPENINGS            = 0"
)

print(
    "  PREDICTOR ROWS READ = 0"
)

print(
    "  LABELS READ         = 0"
)

print()

print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "  Stage28-2A4 — CHRONOLOGICAL_NATURAL / seed43"
)

print(
    "  C013 = CPU XGBoost seed43 NEW FIT"
)

print(
    "  C014 = CPU LightGBM seed43 NEW FIT"
)

print()

print(
    "  Those will consume FIT #5 and FIT #6."
)

print(
    "  Shared final holdout remains closed."
)

print()
print(SEP)


STAGE28-2A3 — EXACT DURABLE-PARENT GATE

Expected parent: d597370a59ba7fc93b2da1720fd3faebf9cfa1ba
Local HEAD     : d597370a59ba7fc93b2da1720fd3faebf9cfa1ba
origin/main    : d597370a59ba7fc93b2da1720fd3faebf9cfa1ba
Remote main    : d597370a59ba7fc93b2da1720fd3faebf9cfa1ba
Branch         : main
Git clean      : True

[PASS] exact clean Stage28-2A2 parent

GITHUB DURABILITY / IDENTITY GATE

[PASS] credential recovered from kaggle_secret:GITHUB_TOKEN
[PASS] token intentionally not displayed
user.name : Stage28 Kaggle
user.email: stage28-kaggle@users.noreply.github.com

[PASS] commit/push plumbing ready before FIT #3

FROZEN MODEL RUNTIME GATE

XGBoost : 3.2.0
LightGBM: 4.6.0
[PASS] Stage28 CPU-only execution policy active

PRIOR DURABLE STAGE28 FIT LEDGER

Authorized: 108
Consumed  : 2
Remaining : 106

[PASS] FIT #3 and FIT #4 are next

FROZEN RANDOM_NATURAL MEMBERSHIP GATE

Train rows       : 11,529,922
Validation rows  : 2,882,481
Train index SHA  : 66714bb29b84aa7de750a6a347f27d87db06

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  inferred 65,536 / 2,882,481


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

  inferred 655,360 / 2,882,481


...

  inferred 1,966,080 / 2,882,481


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

  inferred 2,621,440 / 2,882,481


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  inferred 2,882,481 / 2,882,481

Inference seconds: 102.83527506799965

[PASS] C003 + C004 seed43 validation inference complete

RANDOM_NATURAL SEED43 — VALIDATION RANKING

ROC-AUC: 0.9985997958071943
PR-AUC : 0.9955462532278843
PR definition: SKLEARN_AVERAGE_PRECISION_SCORE

RANDOM_NATURAL SEED43 — FROZEN VALIDATION THRESHOLDS

STANDARD:
 threshold: 0.5
 F1       : 0.9838270385542076
 recall   : 0.9683922324190032
 FPR      : 3.657525398700413e-05

BALANCED:
 threshold: 0.49
 F1       : 0.9838326892592655
 recall   : 0.9684277239770825
 FPR      : 4.059451266689469e-05

SECURITY:
 threshold: 0.1
 F2       : 0.9793245661855443
 recall   : 0.9846042691274147
 FPR      : 0.006714573550625175

FREEZE STAGE28-2A3 VALIDATION ARTIFACTS

Validation probability SHA: 55d6c66b93b561707dea9279c53c00da7fe93b13c91db588118fcb092f479e75
Threshold grid SHA       : 5d08653e2ff448002d5afd18bf91d6c97b09d45549d68e3c92f6b28689301768

Durable artifact checksums:

7bbd868a8d648eef93fb9cae10ada55ab62853ef335

In [7]:
# =================================================================================================
# STAGE28-2A3-SHIFT — POST-EXECUTION COMMIT / PUSH FINALIZER
#
# NO FITS.
# NO INFERENCE.
# NO THRESHOLD SELECTION.
# NO HOLDOUT ACCESS.
#
# Handles three safe states:
#
#   A) Fits completed, artifacts uncommitted
#      -> verify -> stage -> commit -> push
#
#   B) Commit exists locally but push did not complete
#      -> verify -> push
#
#   C) Commit already exists locally + remotely
#      -> verify only; no duplicate commit
#
# Expected scientific parent before Stage28-2A3:
#   d597370a59ba7fc93b2da1720fd3faebf9cfa1ba
# =================================================================================================

from __future__ import annotations

import base64
import hashlib
import json
import os
import subprocess
from pathlib import Path


SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "d597370a59ba7fc93b2da1720fd3faebf9cfa1ba"
)

EXPECTED_SUBJECT = (
    "stage28-2a3: execute random natural seed43 checkpoint"
)

OUT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a3_random_natural_seed43"
)

XGB_MODEL_PATH = (
    OUT
    / "random_natural_seed43_xgboost_cpu_model.json"
)

LGBM_MODEL_PATH = (
    OUT
    / "random_natural_seed43_lightgbm_cpu_model.txt"
)

VALIDATION_PROB_PATH = (
    OUT
    / "random_natural_seed43_validation_ensemble_probabilities.npz"
)

THRESHOLD_GRID_PATH = (
    OUT
    / "random_natural_seed43_validation_threshold_grid.csv"
)

PROGRESS_PATH = (
    OUT
    / "stage28_2a3_execution_progress.json"
)

FIT_LEDGER_PATH = (
    OUT
    / "stage28_2a3_fit_ledger.json"
)

RESULT_PATH = (
    OUT
    / "stage28_2a3_random_natural_seed43_result.json"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


EXPECTED_ARTIFACTS = [
    XGB_MODEL_PATH,
    LGBM_MODEL_PATH,
    VALIDATION_PROB_PATH,
    THRESHOLD_GRID_PATH,
    PROGRESS_PATH,
    FIT_LEDGER_PATH,
    RESULT_PATH,
    CHECKSUMS_PATH,
]


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(
    cmd,
    *,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args, check=True):
    return (
        run(
            ["git", *args],
            check=check,
        ).stdout
        or ""
    ).strip()


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def recover_github_token():

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]

    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()

        for label in labels:

            try:
                value = client.get_secret(label)

            except Exception:
                value = None

            if (
                isinstance(value, str)
                and value.strip()
            ):
                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass


    for label in labels:

        value = os.environ.get(label)

        if (
            isinstance(value, str)
            and value.strip()
        ):
            return (
                value.strip(),
                f"environment:{label}",
            )


    raise RuntimeError(
        "GitHub credential unavailable."
    )


def authenticated_push(token):

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode("utf-8")
    ).decode("ascii")

    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            "Git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return (
        p.stdout
        + p.stderr
    ).strip()


# =================================================================================================
# 1. REQUIRE COMPLETE SCIENTIFIC OUTPUT
# =================================================================================================

banner(
    "STAGE28-2A3-SHIFT — SCIENTIFIC COMPLETION GATE"
)

for path in EXPECTED_ARTIFACTS:

    if not path.is_file():
        raise RuntimeError(
            "Stage28-2A3 is NOT ready to push.\n\n"
            f"Missing artifact:\n{path}\n\n"
            "Do NOT refit automatically. "
            "Send me the execution output if the long cell failed."
        )

    print(
        "[FOUND]",
        path.name,
    )


# =================================================================================================
# 2. FIT PROGRESS MUST SAY BOTH FITS SUCCEEDED
# =================================================================================================

banner(
    "FIT #3 / FIT #4 COMPLETION GATE"
)

progress = read_json(
    PROGRESS_PATH
)

ledger = read_json(
    FIT_LEDGER_PATH
)

result = read_json(
    RESULT_PATH
)


print(
    "Progress state:",
    progress[
        "state"
    ],
)

print(
    "FIT #3:",
    progress[
        "fit_3"
    ][
        "state"
    ],
)

print(
    "FIT #4:",
    progress[
        "fit_4"
    ][
        "state"
    ],
)


if (
    progress[
        "fit_3"
    ][
        "state"
    ]
    != "SUCCESS"
):
    raise RuntimeError(
        "FIT #3 is not confirmed successful. "
        "Refusing to commit."
    )


if (
    progress[
        "fit_4"
    ][
        "state"
    ]
    != "SUCCESS"
):
    raise RuntimeError(
        "FIT #4 is not confirmed successful. "
        "Refusing to commit."
    )


if (
    progress[
        "state"
    ]
    != "FIT_003_AND_004_AND_VALIDATION_SUCCESS"
):
    raise RuntimeError(
        "Stage28-2A3 validation phase is incomplete."
    )


if (
    ledger[
        "status"
    ]
    != "FITS_003_AND_004_SUCCESSFULLY_CONSUMED"
):
    raise RuntimeError(
        "Stage28-2A3 fit-ledger status invalid."
    )


if int(
    ledger[
        "pre_cell_new_fits_consumed"
    ]
) != 2:
    raise RuntimeError(
        "Expected two fits consumed before Stage28-2A3."
    )


if int(
    ledger[
        "this_cell"
    ][
        "successful_new_fits"
    ]
) != 2:
    raise RuntimeError(
        "Stage28-2A3 must contain exactly two successful new fits."
    )


if (
    ledger[
        "this_cell"
    ][
        "new_fit_components"
    ]
    != [
        "C003",
        "C004",
    ]
):
    raise RuntimeError(
        "Unexpected Stage28-2A3 fit component identities."
    )


if int(
    ledger[
        "cumulative_new_fits_consumed"
    ]
) != 4:
    raise RuntimeError(
        "Cumulative fit ledger must equal 4."
    )


if int(
    ledger[
        "new_fits_remaining"
    ]
) != 104:
    raise RuntimeError(
        "Remaining fit ledger must equal 104."
    )


if (
    result[
        "status"
    ]
    != "RANDOM_NATURAL_SEED43_VALIDATION_CHECKPOINT_FROZEN"
):
    raise RuntimeError(
        "Stage28-2A3 result status invalid."
    )


if int(
    result[
        "scientific_accounting"
    ][
        "shared_final_holdout_openings"
    ]
) != 0:
    raise RuntimeError(
        "Shared final holdout opening detected."
    )


print()
print(
    "[PASS] FIT #3 = C003 SUCCESS"
)

print(
    "[PASS] FIT #4 = C004 SUCCESS"
)

print(
    "[PASS] ledger = 4 consumed / 104 remaining"
)

print(
    "[PASS] shared final holdout still unopened"
)


# =================================================================================================
# 3. VERIFY MODEL IDENTITIES AGAINST PROGRESS RECEIPT
# =================================================================================================

banner(
    "MODEL IDENTITY GATE"
)

xgb_sha = sha256_file(
    XGB_MODEL_PATH
)

lgbm_sha = sha256_file(
    LGBM_MODEL_PATH
)


print(
    "C003 model SHA:",
    xgb_sha,
)

print(
    "C004 model SHA:",
    lgbm_sha,
)


if (
    progress[
        "fit_3"
    ][
        "model_sha256"
    ]
    != xgb_sha
):
    raise RuntimeError(
        "C003 model SHA differs from execution-progress receipt."
    )


if (
    progress[
        "fit_4"
    ][
        "model_sha256"
    ]
    != lgbm_sha
):
    raise RuntimeError(
        "C004 model SHA differs from execution-progress receipt."
    )


if (
    result[
        "models"
    ][
        "xgboost"
    ][
        "model_sha256"
    ]
    != xgb_sha
):
    raise RuntimeError(
        "C003 model SHA differs from result receipt."
    )


if (
    result[
        "models"
    ][
        "lightgbm"
    ][
        "model_sha256"
    ]
    != lgbm_sha
):
    raise RuntimeError(
        "C004 model SHA differs from result receipt."
    )


print()
print(
    "[PASS] both fitted-model identities exact"
)


# =================================================================================================
# 4. VERIFY CHECKSUM MANIFEST
# =================================================================================================

banner(
    "CHECKSUM MANIFEST GATE"
)

checksum_lines = (
    CHECKSUMS_PATH.read_text(
        encoding="utf-8"
    ).splitlines()
)


# checksums.sha256 describes the seven scientific payloads;
# the checksum file itself is the eighth repository artifact.
if len(
    checksum_lines
) != 7:
    raise RuntimeError(
        "Expected seven entries in checksums.sha256."
    )


for line in checksum_lines:

    digest, filename = line.split(
        None,
        1,
    )

    filename = (
        filename.strip()
        .lstrip("*")
    )

    artifact = (
        OUT
        / filename
    )


    if not artifact.is_file():
        raise RuntimeError(
            f"Checksum payload missing: {filename}"
        )


    actual = sha256_file(
        artifact
    )


    print(
        "[CHECK]",
        filename,
    )

    print(
        "  expected:",
        digest,
    )

    print(
        "  actual  :",
        actual,
    )


    if actual != digest:
        raise RuntimeError(
            f"Checksum mismatch: {filename}"
        )


print()
print(
    "[PASS] all scientific payload checksums exact"
)


# =================================================================================================
# 5. VALIDATION SUMMARY
# =================================================================================================

banner(
    "STAGE28-2A3 RESULT READY FOR DURABILITY"
)

print(
    "ROC-AUC:",
    result[
        "validation_probability"
    ][
        "roc_auc"
    ],
)

print(
    "PR-AUC :",
    result[
        "validation_probability"
    ][
        "pr_auc"
    ],
)

print()

print(
    "STANDARD:",
    result[
        "operating_points"
    ][
        "standard"
    ][
        "threshold"
    ],
)

print(
    "BALANCED:",
    result[
        "operating_points"
    ][
        "balanced"
    ][
        "threshold"
    ],
)

print(
    "SECURITY:",
    result[
        "operating_points"
    ][
        "security"
    ][
        "result"
    ][
        "threshold"
    ],
)


# =================================================================================================
# 6. DETERMINE CURRENT GIT STATE
# =================================================================================================

banner(
    "CURRENT GIT STATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote = remote_line.split()[0]

subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)


print(
    "HEAD       :",
    head,
)

print(
    "origin/main:",
    origin,
)

print(
    "Remote main:",
    remote,
)

print(
    "HEAD subject:",
    subject,
)


expected_relative_files = {
    str(
        path.relative_to(
            REPO
        )
    )
    for path
    in EXPECTED_ARTIFACTS
}


# =================================================================================================
# 7. CASE A — NO COMMIT YET
# =================================================================================================

if head == EXPECTED_PARENT:

    banner(
        "CASE A — STAGE28-2A3 NOT YET COMMITTED"
    )


    tracked_changes = set(
        git(
            "diff",
            "--name-only",
        ).splitlines()
    )

    if tracked_changes:
        raise RuntimeError(
            "Unexpected tracked modifications exist:\n"
            + "\n".join(
                sorted(
                    tracked_changes
                )
            )
        )


    staged = set(
        git(
            "diff",
            "--cached",
            "--name-only",
        ).splitlines()
    )

    untracked = set(
        git(
            "ls-files",
            "--others",
            "--exclude-standard",
        ).splitlines()
    )


    # Some or all files may already have been staged by the long cell.
    existing_repo_artifacts = (
        staged
        | untracked
    )


    if existing_repo_artifacts != expected_relative_files:

        raise RuntimeError(
            "Repository artifact universe is not exactly "
            "the eight Stage28-2A3 files.\n\n"
            "Expected:\n"
            + "\n".join(
                sorted(
                    expected_relative_files
                )
            )
            + "\n\nActual staged/untracked:\n"
            + "\n".join(
                sorted(
                    existing_repo_artifacts
                )
            )
        )


    print(
        "[PASS] exact Stage28-2A3 repository artifact universe"
    )


    # Stage everything explicitly.
    for relative in sorted(
        expected_relative_files
    ):

        run(
            [
                "git",
                "add",
                "--",
                relative,
            ]
        )


    staged = set(
        git(
            "diff",
            "--cached",
            "--name-only",
        ).splitlines()
    )


    if staged != expected_relative_files:

        raise RuntimeError(
            "Unable to stage exact Stage28-2A3 artifact universe."
        )


    print(
        "[PASS] exactly eight files staged"
    )


    # Safe local identity.
    run(
        [
            "git",
            "config",
            "user.name",
            "Stage28 Kaggle",
        ]
    )

    run(
        [
            "git",
            "config",
            "user.email",
            "stage28-kaggle@users.noreply.github.com",
        ]
    )


    print(
        "user.name :",
        git(
            "config",
            "--get",
            "user.name",
        ),
    )

    print(
        "user.email:",
        git(
            "config",
            "--get",
            "user.email",
        ),
    )


    banner(
        "COMMIT STAGE28-2A3"
    )


    commit_proc = run(
        [
            "git",
            "commit",
            "-m",
            EXPECTED_SUBJECT,
        ]
    )


    print(
        commit_proc.stdout.strip()
    )


    head = git(
        "rev-parse",
        "HEAD",
    )

    parent = git(
        "rev-parse",
        "HEAD^",
    )

    subject = git(
        "show",
        "-s",
        "--format=%s",
        "HEAD",
    )


    print()
    print(
        "Parent :",
        parent,
    )

    print(
        "Commit :",
        head,
    )

    print(
        "Subject:",
        subject,
    )


    if parent != EXPECTED_PARENT:
        raise RuntimeError(
            "Stage28-2A3 commit parent mismatch."
        )


    if subject != EXPECTED_SUBJECT:
        raise RuntimeError(
            "Stage28-2A3 commit subject mismatch."
        )


    print()
    print(
        "[PASS] Stage28-2A3 committed"
    )


# =================================================================================================
# 8. CASE B/C — VERIFY EXISTING LOCAL COMMIT
# =================================================================================================

else:

    banner(
        "EXISTING LOCAL STAGE28-2A3 COMMIT GATE"
    )


    if subject != EXPECTED_SUBJECT:

        raise RuntimeError(
            "HEAD advanced, but it is not the expected "
            "Stage28-2A3 commit.\n"
            "Do not push automatically."
        )


    parent = git(
        "rev-parse",
        "HEAD^",
    )


    if parent != EXPECTED_PARENT:

        raise RuntimeError(
            "Existing Stage28-2A3 commit has unexpected parent."
        )


    committed_files = set(
        git(
            "diff-tree",
            "--no-commit-id",
            "--name-only",
            "-r",
            "HEAD",
        ).splitlines()
    )


    if committed_files != expected_relative_files:

        raise RuntimeError(
            "Existing Stage28-2A3 commit does not contain "
            "exactly the expected eight artifacts."
        )


    print(
        "[PASS] existing local Stage28-2A3 commit exact"
    )


# =================================================================================================
# 9. PUSH / VERIFY
# =================================================================================================

banner(
    "PUSH / REMOTE DURABILITY"
)

github_token, token_source = (
    recover_github_token()
)


print(
    "[PASS] GitHub credential recovered from",
    token_source,
)

print(
    "[PASS] token intentionally not displayed"
)


# Re-resolve remote immediately before deciding whether push is needed.
origin_now = git(
    "rev-parse",
    "origin/main",
)

remote_now_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


if not remote_now_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_now = (
    remote_now_line.split()[0]
)


if (
    origin_now == head
    and remote_now == head
):

    print(
        "[PASS] commit is already remotely durable"
    )

else:

    print(
        "Local Stage28-2A3 commit is not yet fully remote."
    )

    print(
        "Pushing..."
    )


    push_output = authenticated_push(
        github_token
    )


    if push_output:
        print(
            push_output
        )


github_token = None


# =================================================================================================
# 10. FINAL REMOTE VERIFICATION
# =================================================================================================

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_final = git(
    "rev-parse",
    "HEAD",
)

origin_final = git(
    "rev-parse",
    "origin/main",
)

remote_final_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


if not remote_final_line:
    raise RuntimeError(
        "Unable to resolve final remote main."
    )


remote_final = (
    remote_final_line.split()[0]
)

final_status = git(
    "status",
    "--porcelain",
)


print()
print(
    "Local HEAD :",
    local_final,
)

print(
    "origin/main:",
    origin_final,
)

print(
    "Remote main:",
    remote_final,
)

print(
    "Git clean  :",
    not bool(
        final_status
    ),
)


if not (
    local_final
    == origin_final
    == remote_final
):
    raise RuntimeError(
        "Stage28-2A3 remote durability verification failed."
    )


if final_status:

    raise RuntimeError(
        "Repository is not clean after Stage28-2A3 finalization:\n"
        + final_status
    )


# =================================================================================================
# 11. FINAL
# =================================================================================================

banner(
    "STAGE28-2A3-SHIFT — COMPLETE"
)

print(
    "Durable commit:"
)

print(
    " ",
    local_final,
)

print()

print(
    "C003 XGBoost seed43:"
)

print(
    "  FIT #3 = SUCCESS"
)

print()

print(
    "C004 LightGBM seed43:"
)

print(
    "  FIT #4 = SUCCESS"
)

print()

print(
    "Validation:"
)

print(
    "  ROC-AUC =",
    result[
        "validation_probability"
    ][
        "roc_auc"
    ],
)

print(
    "  PR-AUC  =",
    result[
        "validation_probability"
    ][
        "pr_auc"
    ],
)

print()

print(
    "Thresholds:"
)

print(
    "  STANDARD =",
    result[
        "operating_points"
    ][
        "standard"
    ][
        "threshold"
    ],
)

print(
    "  BALANCED =",
    result[
        "operating_points"
    ][
        "balanced"
    ][
        "threshold"
    ],
)

print(
    "  SECURITY =",
    result[
        "operating_points"
    ][
        "security"
    ][
        "result"
    ][
        "threshold"
    ],
)

print()

print(
    "Fit ledger:"
)

print(
    "  authorized = 108"
)

print(
    "  consumed   = 4"
)

print(
    "  remaining  = 104"
)

print()

print(
    "Shared final holdout:"
)

print(
    "  openings = 0"
)

print()

print(
    "NEXT:"
)

print(
    "  Stage28-2A4 — CHRONOLOGICAL_NATURAL seed43"
)

print(
    "  FIT #5 = C013 XGBoost"
)

print(
    "  FIT #6 = C014 LightGBM"
)

print()
print(SEP)


STAGE28-2A3-SHIFT — SCIENTIFIC COMPLETION GATE

[FOUND] random_natural_seed43_xgboost_cpu_model.json
[FOUND] random_natural_seed43_lightgbm_cpu_model.txt
[FOUND] random_natural_seed43_validation_ensemble_probabilities.npz
[FOUND] random_natural_seed43_validation_threshold_grid.csv
[FOUND] stage28_2a3_execution_progress.json
[FOUND] stage28_2a3_fit_ledger.json
[FOUND] stage28_2a3_random_natural_seed43_result.json
[FOUND] checksums.sha256

FIT #3 / FIT #4 COMPLETION GATE

Progress state: FIT_003_AND_004_AND_VALIDATION_SUCCESS
FIT #3: SUCCESS
FIT #4: SUCCESS

[PASS] FIT #3 = C003 SUCCESS
[PASS] FIT #4 = C004 SUCCESS
[PASS] ledger = 4 consumed / 104 remaining
[PASS] shared final holdout still unopened

MODEL IDENTITY GATE

C003 model SHA: 7bbd868a8d648eef93fb9cae10ada55ab62853ef33538225e49da2b956b7302c
C004 model SHA: 77af40e0b85e19e1347380eb9f59caf2c1d91ca5ec202005faaf45fd038d082d

[PASS] both fitted-model identities exact

CHECKSUM MANIFEST GATE

[CHECK] random_natural_seed43_xgboost_cp

In [5]:
# =================================================================================================
# STAGE28-2A4 — CHRONOLOGICAL_NATURAL / SEED43 — SCIENCE ONLY
#
# NEW FITS
# --------
# FIT #5 — C013 XGBOOST  / XGB_11  / seed43 / CPU
# FIT #6 — C014 LIGHTGBM / LGBM_11 / seed43 / CPU
#
# Durable scientific parent:
#   1c21d67d374cfcba864f04d4b2da7fc3fc332568
#
# PRE-CELL LEDGER
#   authorized = 108
#   consumed   = 4
#   remaining  = 104
#
# EXPECTED POST-SCIENCE LEDGER
#   consumed   = 6
#   remaining  = 102
#
# IMPORTANT
# ---------
# THIS CELL DOES NOT:
#   git add
#   git commit
#   git push
#
# It only performs and freezes the science.
#
# After successful completion, run the separate Stage28-2A4-SHIFT
# commit/push cell.
#
# If anything fails after either FIT #5 or FIT #6 begins:
#   DO NOT RERUN THIS CELL.
# =================================================================================================

from __future__ import annotations

import csv
import gc
import hashlib
import json
import os
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

import xgboost as xgb
import lightgbm as lgb


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "1c21d67d374cfcba864f04d4b2da7fc3fc332568"
)

EXPECTED_XGB_VERSION = "3.2.0"
EXPECTED_LGBM_VERSION = "4.6.0"

EXPECTED_COMPONENT_MANIFEST_SHA = (
    "47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505"
)

EXPECTED_XGB_PARAMETER_SHA = (
    "5ac981f17f3327f8de881b672b6bf35a3b020a3a56a68957146cbe6622e30ed0"
)

EXPECTED_LGBM_PARAMETER_SHA = (
    "d43b2e4009bec8e9874a2ec0fd65f1279ea8d493808a386ab84fc801076e9c13"
)

EXPECTED_TOTAL_ROWS = 14_412_403
EXPECTED_FEATURES = 70

EXPECTED_TRAIN_ROWS = 13_818_623
EXPECTED_TRAIN_ATTACK = 1_910_043
EXPECTED_TRAIN_BENIGN = 11_908_580

EXPECTED_VAL_ROWS = 593_780
EXPECTED_VAL_ATTACK = 62_256
EXPECTED_VAL_BENIGN = 531_524

EXPECTED_TOTAL_NEW_FITS = 108


# =================================================================================================
# 1. PATHS
# =================================================================================================

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

STAGE28_1B_DIR = (
    STAGE28_ROOT
    / "stage28_1b_random_loao_membership_and_execution_lock"
)

COMPONENT_MANIFEST_PATH = (
    STAGE28_1B_DIR
    / "stage28_component_execution_manifest.csv"
)

PARAMETER_SETS_PATH = (
    STAGE28_1B_DIR
    / "execution_parameter_sets.json"
)

THRESHOLD_POLICY_PATH = (
    STAGE28_ROOT
    / "stage28_0_protocol_lock"
    / "threshold_policy.json"
)


# -------------------------------------------------------------------------------------------------
# Previous durable Stage28-2A3 checkpoint.
# -------------------------------------------------------------------------------------------------

PREV_DIR = (
    STAGE28_ROOT
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a3_random_natural_seed43"
)

PREV_LEDGER_PATH = (
    PREV_DIR
    / "stage28_2a3_fit_ledger.json"
)

PREV_RESULT_PATH = (
    PREV_DIR
    / "stage28_2a3_random_natural_seed43_result.json"
)

PREV_CHECKSUMS_PATH = (
    PREV_DIR
    / "checksums.sha256"
)


# -------------------------------------------------------------------------------------------------
# Stage28-2A0 runtime matrix.
# -------------------------------------------------------------------------------------------------

STAGE28_2A0_RECEIPT = Path(
    "/kaggle/working/"
    "stage28_2a0_stage22_execution_matrix_receipt.json"
)

RUNTIME_ROOT = Path(
    "/kaggle/working/"
    "stage28_2a_runtime_cache/"
    "stage22_full"
)

X_PATH = (
    RUNTIME_ROOT
    / "development_features_float64.npy"
)

Y_PATH = (
    RUNTIME_ROOT
    / "development_binary_labels_uint8.npy"
)

DAY_PATH = (
    RUNTIME_ROOT
    / "development_day_ids_uint8.npy"
)


# -------------------------------------------------------------------------------------------------
# New Stage28-2A4 science output.
# -------------------------------------------------------------------------------------------------

OUT = (
    STAGE28_ROOT
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a4_chronological_natural_seed43"
)

XGB_MODEL_PATH = (
    OUT
    / "chronological_natural_seed43_xgboost_cpu_model.json"
)

LGBM_MODEL_PATH = (
    OUT
    / "chronological_natural_seed43_lightgbm_cpu_model.txt"
)

VALIDATION_PROB_PATH = (
    OUT
    / "chronological_natural_seed43_validation_ensemble_probabilities.npz"
)

THRESHOLD_GRID_PATH = (
    OUT
    / "chronological_natural_seed43_validation_threshold_grid.csv"
)

PROGRESS_PATH = (
    OUT
    / "stage28_2a4_execution_progress.json"
)

FIT_LEDGER_PATH = (
    OUT
    / "stage28_2a4_fit_ledger.json"
)

RESULT_PATH = (
    OUT
    / "stage28_2a4_chronological_natural_seed43_result.json"
)

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)


# =================================================================================================
# 2. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return (
        run(
            ["git", *args]
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def write_json(path, obj):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def canonical_json_sha256(obj):
    payload = json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


def safe_div(a, b):
    return (
        float(a) / float(b)
        if b
        else 0.0
    )


def metrics_from_counts(
    tp,
    fp,
    tn,
    fn,
):
    total = (
        tp + fp + tn + fn
    )

    return {
        "accuracy":
            safe_div(
                tp + tn,
                total,
            ),

        "precision":
            safe_div(
                tp,
                tp + fp,
            ),

        "recall":
            safe_div(
                tp,
                tp + fn,
            ),

        "fpr":
            safe_div(
                fp,
                fp + tn,
            ),

        "f1":
            safe_div(
                2 * tp,
                (2 * tp) + fp + fn,
            ),

        "f2":
            safe_div(
                5 * tp,
                (5 * tp) + (4 * fn) + fp,
            ),

        "tp":
            int(tp),

        "fp":
            int(fp),

        "tn":
            int(tn),

        "fn":
            int(fn),
    }


def evaluate_threshold(
    y_true,
    probability_float32,
    integer_percent,
):
    threshold = (
        integer_percent / 100.0
    )

    threshold32 = np.float32(
        threshold
    )

    pred = (
        probability_float32
        >= threshold32
    )

    positive = (
        y_true == 1
    )

    negative = ~positive


    tp = int(
        np.count_nonzero(
            pred & positive
        )
    )

    fp = int(
        np.count_nonzero(
            pred & negative
        )
    )

    fn = int(
        np.count_nonzero(
            (~pred) & positive
        )
    )

    tn = int(
        np.count_nonzero(
            (~pred) & negative
        )
    )


    row = metrics_from_counts(
        tp,
        fp,
        tn,
        fn,
    )

    row.update(
        {
            "threshold_integer_percent":
                int(integer_percent),

            "threshold":
                float(threshold),

            "threshold_float32_runtime":
                float(threshold32),
        }
    )

    return row


def build_grid(
    y_true,
    probability_float32,
):
    return [
        evaluate_threshold(
            y_true,
            probability_float32,
            pct,
        )
        for pct in range(
            5,
            96,
        )
    ]


def choose_balanced(grid):
    return max(
        grid,
        key=lambda r: (
            r["f1"],
            -r["fpr"],
            r["recall"],
            -abs(
                r["threshold"] - 0.50
            ),
            -r["threshold"],
        ),
    )


def choose_security(grid):
    feasible = [
        r
        for r in grid
        if r[
            "fpr"
        ] <= 0.05
    ]

    if not feasible:
        return None

    return max(
        feasible,
        key=lambda r: (
            r["f2"],
            -r["fpr"],
            r["recall"],
            -r["threshold"],
        ),
    )


def write_progress(
    *,
    state,
    fit5_state,
    fit6_state,
    xgb_sha=None,
    lgbm_sha=None,
):
    write_json(
        PROGRESS_PATH,
        {
            "stage":
                "Stage28-2A4",

            "evaluation_cell_id":
                "28A_STAGE22::CHRONOLOGICAL_NATURAL::SEED43",

            "durable_parent":
                EXPECTED_PARENT,

            "updated_at_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),

            "state":
                state,

            "pre_cell_new_fits_consumed":
                4,

            "fit_5": {
                "component_id":
                    "C013",

                "learner":
                    "XGBOOST",

                "state":
                    fit5_state,

                "model_sha256":
                    xgb_sha,
            },

            "fit_6": {
                "component_id":
                    "C014",

                "learner":
                    "LIGHTGBM",

                "state":
                    fit6_state,

                "model_sha256":
                    lgbm_sha,
            },

            "shared_final_holdout_openings":
                0,
        },
    )


# =================================================================================================
# 3. EXACT DURABLE-PARENT GATE
# =================================================================================================

banner(
    "STAGE28-2A4 — EXACT DURABLE-PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote = remote_line.split()[0]

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    head,
)

print(
    "origin/main    :",
    origin,
)

print(
    "Remote main    :",
    remote,
)

print(
    "Branch         :",
    branch,
)

print(
    "Git clean      :",
    not bool(status),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-2A4 parent mismatch."
    )

if branch != "main":
    raise RuntimeError(
        "Expected branch main."
    )

if status:
    raise RuntimeError(
        "Repository must be clean before Stage28-2A4:\n"
        + status
    )

if OUT.exists():
    raise RuntimeError(
        "Stage28-2A4 output already exists.\n"
        "Do NOT rerun a partial scientific checkpoint."
    )


print()
print(
    "[PASS] exact clean Stage28-2A3 durable parent"
)


# =================================================================================================
# 4. LIBRARY / CPU GATE
# =================================================================================================

banner(
    "FROZEN MODEL RUNTIME GATE"
)

print(
    "XGBoost :",
    xgb.__version__,
)

print(
    "LightGBM:",
    lgb.__version__,
)


if (
    xgb.__version__
    != EXPECTED_XGB_VERSION
):
    raise RuntimeError(
        "XGBoost version drift."
    )

if (
    lgb.__version__
    != EXPECTED_LGBM_VERSION
):
    raise RuntimeError(
        "LightGBM version drift."
    )


os.environ[
    "CUDA_VISIBLE_DEVICES"
] = ""


print(
    "[PASS] Stage28 CPU-only policy active"
)


# =================================================================================================
# 5. PRIOR DURABLE LEDGER — MUST BE 4 / 104
# =================================================================================================

banner(
    "PRIOR DURABLE STAGE28 LEDGER"
)

require_file(
    PREV_CHECKSUMS_PATH
)

prev_ledger = read_json(
    require_file(
        PREV_LEDGER_PATH
    )
)

prev_result = read_json(
    require_file(
        PREV_RESULT_PATH
    )
)


if (
    prev_ledger[
        "status"
    ]
    != "FITS_003_AND_004_SUCCESSFULLY_CONSUMED"
):
    raise RuntimeError(
        "Stage28-2A3 ledger status invalid."
    )


if int(
    prev_ledger[
        "cumulative_new_fits_consumed"
    ]
) != 4:
    raise RuntimeError(
        "Expected 4 prior fits consumed."
    )


if int(
    prev_ledger[
        "new_fits_remaining"
    ]
) != 104:
    raise RuntimeError(
        "Expected 104 prior fits remaining."
    )


if int(
    prev_result[
        "scientific_accounting"
    ][
        "shared_final_holdout_openings"
    ]
) != 0:
    raise RuntimeError(
        "Shared final holdout was previously opened."
    )


print(
    "Authorized:",
    EXPECTED_TOTAL_NEW_FITS,
)

print(
    "Consumed  : 4"
)

print(
    "Remaining : 104"
)

print()
print(
    "[PASS] FIT #5 and FIT #6 are next"
)


# =================================================================================================
# 6. STAGE28-2A0 CHRONOLOGICAL MEMBERSHIP GATE
# =================================================================================================

banner(
    "FROZEN CHRONOLOGICAL_NATURAL MEMBERSHIP GATE"
)

receipt_2a0 = read_json(
    require_file(
        STAGE28_2A0_RECEIPT
    )
)


chrono_receipt = (
    receipt_2a0[
        "chronological_natural"
    ]
)


if int(
    chrono_receipt[
        "train"
    ][
        "rows"
    ]
) != EXPECTED_TRAIN_ROWS:
    raise RuntimeError(
        "Chronological train rows changed."
    )


if int(
    chrono_receipt[
        "validation"
    ][
        "rows"
    ]
) != EXPECTED_VAL_ROWS:
    raise RuntimeError(
        "Chronological validation rows changed."
    )


X = np.load(
    require_file(
        X_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)

y = np.load(
    require_file(
        Y_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)

day_ids = np.load(
    require_file(
        DAY_PATH
    ),
    mmap_mode="r",
    allow_pickle=False,
)


if X.shape != (
    EXPECTED_TOTAL_ROWS,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        "Execution matrix shape mismatch."
    )


if X.dtype != np.float64:
    raise RuntimeError(
        "Execution matrix dtype != float64."
    )


if not np.all(
    day_ids[
        :EXPECTED_TRAIN_ROWS
    ] <= 6
):
    raise RuntimeError(
        "Chronological train contains validation day."
    )


if not np.all(
    day_ids[
        EXPECTED_TRAIN_ROWS:
    ] == 7
):
    raise RuntimeError(
        "Chronological validation is not exactly day 7."
    )


y_validation = np.asarray(
    y[
        EXPECTED_TRAIN_ROWS:
    ],
    dtype=np.uint8,
)


train_attack = int(
    np.asarray(
        y[
            :EXPECTED_TRAIN_ROWS
        ]
    ).sum()
)

validation_attack = int(
    y_validation.sum()
)


if train_attack != EXPECTED_TRAIN_ATTACK:
    raise RuntimeError(
        "Training attack count mismatch."
    )

if (
    EXPECTED_TRAIN_ROWS
    - train_attack
    != EXPECTED_TRAIN_BENIGN
):
    raise RuntimeError(
        "Training benign count mismatch."
    )

if validation_attack != EXPECTED_VAL_ATTACK:
    raise RuntimeError(
        "Validation attack count mismatch."
    )

if (
    EXPECTED_VAL_ROWS
    - validation_attack
    != EXPECTED_VAL_BENIGN
):
    raise RuntimeError(
        "Validation benign count mismatch."
    )


print(
    "TRAIN:"
)

print(
    " rows  :",
    f"{EXPECTED_TRAIN_ROWS:,}",
)

print(
    " attack:",
    f"{train_attack:,}",
)

print(
    " benign:",
    f"{EXPECTED_TRAIN_BENIGN:,}",
)

print()

print(
    "VALIDATION:"
)

print(
    " rows  :",
    f"{EXPECTED_VAL_ROWS:,}",
)

print(
    " attack:",
    f"{validation_attack:,}",
)

print(
    " benign:",
    f"{EXPECTED_VAL_BENIGN:,}",
)

print()

print(
    "[PASS] exact CHRONOLOGICAL_NATURAL membership"
)


# =================================================================================================
# 7. C013 / C014 FROZEN MANIFEST GATE
# =================================================================================================

banner(
    "C013 / C014 FROZEN COMPONENT GATE"
)

manifest_sha = sha256_file(
    require_file(
        COMPONENT_MANIFEST_PATH
    )
)


if (
    manifest_sha
    != EXPECTED_COMPONENT_MANIFEST_SHA
):
    raise RuntimeError(
        "Stage28 component-manifest SHA mismatch."
    )


with COMPONENT_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:

    rows = list(
        csv.DictReader(f)
    )


by_component = {
    row[
        "component_id"
    ]:
    row
    for row in rows
}


c013 = by_component.get(
    "C013"
)

c014 = by_component.get(
    "C014"
)


if c013 is None or c014 is None:
    raise RuntimeError(
        "C013/C014 missing from frozen manifest."
    )


expected_c013 = {
    "arm":
        "28A_TRAINING_SEED_STABILITY",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "CHRONOLOGICAL_NATURAL",

    "evaluation_cell_id":
        "28A_STAGE22::CHRONOLOGICAL_NATURAL::SEED43",

    "learner":
        "XGBOOST",

    "configuration_id":
        "XGB_11",

    "model_seed":
        "43",

    "compute_backend":
        "CPU",

    "parameter_set_id":
        "STAGE22_FULL::XGBOOST::SEED43",

    "parameter_sha256":
        EXPECTED_XGB_PARAMETER_SHA,

    "membership_reference":
        "STAGE22R_FROZEN_CHRONOLOGICAL_NATURAL_MEMBERSHIP",

    "fit_action":
        "NEW_FIT_AUTHORIZED",

    "new_fit_budget_units":
        "1",
}


expected_c014 = {
    "arm":
        "28A_TRAINING_SEED_STABILITY",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "CHRONOLOGICAL_NATURAL",

    "evaluation_cell_id":
        "28A_STAGE22::CHRONOLOGICAL_NATURAL::SEED43",

    "learner":
        "LIGHTGBM",

    "configuration_id":
        "LGBM_11",

    "model_seed":
        "43",

    "compute_backend":
        "CPU",

    "parameter_set_id":
        "STAGE22_FULL::LIGHTGBM::SEED43",

    "parameter_sha256":
        EXPECTED_LGBM_PARAMETER_SHA,

    "membership_reference":
        "STAGE22R_FROZEN_CHRONOLOGICAL_NATURAL_MEMBERSHIP",

    "fit_action":
        "NEW_FIT_AUTHORIZED",

    "new_fit_budget_units":
        "1",
}


for field, expected in expected_c013.items():

    if c013[field] != expected:
        raise RuntimeError(
            f"C013 {field} mismatch.\n"
            f"expected={expected}\n"
            f"actual={c013[field]}"
        )


for field, expected in expected_c014.items():

    if c014[field] != expected:
        raise RuntimeError(
            f"C014 {field} mismatch.\n"
            f"expected={expected}\n"
            f"actual={c014[field]}"
        )


print(
    "[PASS] C013 = XGBoost seed43 CPU NEW FIT"
)

print(
    "[PASS] C014 = LightGBM seed43 CPU NEW FIT"
)


# =================================================================================================
# 8. EXACT SEED43 PARAMETER GATE
# =================================================================================================

banner(
    "SEED43 FROZEN PARAMETER-SET GATE"
)

parameter_doc = read_json(
    require_file(
        PARAMETER_SETS_PATH
    )
)

parameter_sets = (
    parameter_doc[
        "sets"
    ]
)


xgb_params = dict(
    parameter_sets[
        c013[
            "parameter_set_id"
        ]
    ][
        "parameters"
    ]
)


lgbm_params = dict(
    parameter_sets[
        c014[
            "parameter_set_id"
        ]
    ][
        "parameters"
    ]
)


xgb_parameter_sha = (
    canonical_json_sha256(
        xgb_params
    )
)

lgbm_parameter_sha = (
    canonical_json_sha256(
        lgbm_params
    )
)


print(
    "XGB expected :",
    EXPECTED_XGB_PARAMETER_SHA,
)

print(
    "XGB actual   :",
    xgb_parameter_sha,
)

print()

print(
    "LGBM expected:",
    EXPECTED_LGBM_PARAMETER_SHA,
)

print(
    "LGBM actual  :",
    lgbm_parameter_sha,
)


if (
    xgb_parameter_sha
    != EXPECTED_XGB_PARAMETER_SHA
):
    raise RuntimeError(
        "Seed43 XGBoost parameter SHA mismatch."
    )


if (
    lgbm_parameter_sha
    != EXPECTED_LGBM_PARAMETER_SHA
):
    raise RuntimeError(
        "Seed43 LightGBM parameter SHA mismatch."
    )


if (
    int(
        xgb_params[
            "random_state"
        ]
    )
    != 43
):
    raise RuntimeError(
        "C013 random_state != 43."
    )


if (
    int(
        lgbm_params[
            "random_state"
        ]
    )
    != 43
):
    raise RuntimeError(
        "C014 random_state != 43."
    )


if not (
    xgb_params[
        "device"
    ] == "cpu"
    and
    xgb_params[
        "tree_method"
    ] == "hist"
):
    raise RuntimeError(
        "C013 backend is not CPU/hist."
    )


if (
    lgbm_params[
        "device_type"
    ]
    != "cpu"
):
    raise RuntimeError(
        "C014 backend is not CPU."
    )


print()
print(
    "[PASS] exact seed43 frozen CPU parameter sets"
)


# =================================================================================================
# 9. FROZEN THRESHOLD POLICY
# =================================================================================================

banner(
    "FROZEN STAGE22 THRESHOLD POLICY"
)

threshold_policy = read_json(
    require_file(
        THRESHOLD_POLICY_PATH
    )
)


policy = (
    threshold_policy[
        "stage22_full"
    ]
)


if (
    policy[
        "selection_population"
    ]
    != "EACH_PARENT_CELL_FROZEN_DEVELOPMENT_VALIDATION"
):
    raise RuntimeError(
        "Threshold-selection population changed."
    )


if not (
    policy["grid"]["integer_percent_start"] == 5
    and
    policy["grid"]["integer_percent_stop_inclusive"] == 95
    and
    policy["grid"]["integer_step"] == 1
    and
    policy["grid"]["count"] == 91
):
    raise RuntimeError(
        "Stage22 threshold grid changed."
    )


if (
    policy[
        "final_holdout_threshold_search"
    ]
    != "FORBIDDEN"
):
    raise RuntimeError(
        "Final-holdout threshold search changed."
    )


print(
    "[PASS] Stage22 threshold policy exact"
)


# =================================================================================================
# 10. CREATE SCIENTIFIC OUTPUT + INITIAL PROGRESS
# =================================================================================================

OUT.mkdir(
    parents=True,
    exist_ok=False,
)


write_progress(
    state="PREFIT_GATES_COMPLETE",
    fit5_state="NOT_STARTED",
    fit6_state="NOT_STARTED",
)


# =================================================================================================
# 11. PREPARE CONTIGUOUS CHRONOLOGICAL TRAINING POPULATION
# =================================================================================================

banner(
    "PREPARE CHRONOLOGICAL_NATURAL TRAINING POPULATION"
)

prep_start = time.perf_counter()


X_train = np.asarray(
    X[
        :EXPECTED_TRAIN_ROWS,
        :
    ],
    dtype=np.float64,
    order="C",
)

y_train = np.asarray(
    y[
        :EXPECTED_TRAIN_ROWS
    ],
    dtype=np.uint8,
    order="C",
)


prep_seconds = (
    time.perf_counter()
    - prep_start
)


if X_train.shape != (
    EXPECTED_TRAIN_ROWS,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        "X_train shape mismatch."
    )


if int(
    y_train.sum()
) != EXPECTED_TRAIN_ATTACK:
    raise RuntimeError(
        "y_train attack count mismatch."
    )


if np.isinf(
    X_train
).any():
    raise RuntimeError(
        "Infinity exists in training matrix."
    )


print(
    "Rows    :",
    f"{X_train.shape[0]:,}",
)

print(
    "Features:",
    X_train.shape[1],
)

print(
    "Dtype   :",
    X_train.dtype,
)

print(
    "Payload :",
    f"{X_train.nbytes / (1024**3):.3f} GiB",
)

print(
    "Preparation seconds:",
    prep_seconds,
)

print()

print(
    "[PASS] chronological seed43 training population ready"
)


# =================================================================================================
# 12. FIT #5 — C013 XGBOOST
# =================================================================================================

banner(
    "FIT #5 — C013 XGBOOST XGB_11 SEED43 CPU"
)


write_progress(
    state="FIT_005_STARTED",
    fit5_state="STARTED_NOT_YET_CONFIRMED",
    fit6_state="NOT_STARTED",
)


print(
    "Starting Stage28 scientific FIT #5..."
)


fit5_start = time.perf_counter()


xgb_model = xgb.XGBClassifier(
    **xgb_params
)


xgb_model.fit(
    X_train,
    y_train,
)


fit5_seconds = (
    time.perf_counter()
    - fit5_start
)


xgb_booster = (
    xgb_model.get_booster()
)

xgb_rounds = int(
    xgb_booster.num_boosted_rounds()
)


if xgb_rounds != 400:
    raise RuntimeError(
        "C013 boosted-round count != 400."
    )


if int(
    xgb_model.n_features_in_
) != EXPECTED_FEATURES:
    raise RuntimeError(
        "C013 feature count != 70."
    )


xgb_model.save_model(
    XGB_MODEL_PATH
)


xgb_model_sha = sha256_file(
    XGB_MODEL_PATH
)


write_progress(
    state="FIT_005_SUCCESS",
    fit5_state="SUCCESS",
    fit6_state="NOT_STARTED",
    xgb_sha=xgb_model_sha,
)


print()
print(
    "FIT #5 completed."
)

print(
    "Fit seconds:",
    fit5_seconds,
)

print(
    "Model SHA256:",
    xgb_model_sha,
)

print()

print(
    "[PASS] C013 FIT #5 SUCCESS"
)

print(
    "[LEDGER] cumulative successful NEW fits = 5"
)


# =================================================================================================
# 13. FIT #6 — C014 LIGHTGBM
# =================================================================================================

banner(
    "FIT #6 — C014 LIGHTGBM LGBM_11 SEED43 CPU"
)


write_progress(
    state="FIT_006_STARTED",
    fit5_state="SUCCESS",
    fit6_state="STARTED_NOT_YET_CONFIRMED",
    xgb_sha=xgb_model_sha,
)


print(
    "Starting Stage28 scientific FIT #6..."
)


fit6_start = time.perf_counter()


lgbm_model = lgb.LGBMClassifier(
    **lgbm_params
)


lgbm_model.fit(
    X_train,
    y_train,
)


fit6_seconds = (
    time.perf_counter()
    - fit6_start
)


lgb_booster = (
    lgbm_model.booster_
)


lgb_iterations = int(
    lgb_booster.current_iteration()
)

lgb_trees = int(
    lgb_booster.num_trees()
)

lgb_features = int(
    lgb_booster.num_feature()
)


if lgb_iterations != 400:
    raise RuntimeError(
        "C014 iteration count != 400."
    )


if lgb_trees != 400:
    raise RuntimeError(
        "C014 tree count != 400."
    )


if lgb_features != EXPECTED_FEATURES:
    raise RuntimeError(
        "C014 feature count != 70."
    )


lgb_booster.save_model(
    str(
        LGBM_MODEL_PATH
    )
)


lgbm_model_sha = sha256_file(
    LGBM_MODEL_PATH
)


write_progress(
    state="FIT_005_AND_006_SUCCESS",
    fit5_state="SUCCESS",
    fit6_state="SUCCESS",
    xgb_sha=xgb_model_sha,
    lgbm_sha=lgbm_model_sha,
)


print()
print(
    "FIT #6 completed."
)

print(
    "Fit seconds:",
    fit6_seconds,
)

print(
    "Model SHA256:",
    lgbm_model_sha,
)

print()

print(
    "[PASS] C014 FIT #6 SUCCESS"
)

print(
    "[LEDGER] cumulative successful NEW fits = 6"
)


del X_train
del y_train

gc.collect()


# =================================================================================================
# 14. VALIDATION INFERENCE
# =================================================================================================

banner(
    "CHRONOLOGICAL_NATURAL SEED43 — DEVELOPMENT VALIDATION INFERENCE"
)


ensemble_probability = np.empty(
    EXPECTED_VAL_ROWS,
    dtype=np.float32,
)


chunk_size = 65_536

inference_start = time.perf_counter()


for local_start in range(
    0,
    EXPECTED_VAL_ROWS,
    chunk_size,
):

    local_stop = min(
        local_start + chunk_size,
        EXPECTED_VAL_ROWS,
    )


    global_start = (
        EXPECTED_TRAIN_ROWS
        + local_start
    )

    global_stop = (
        EXPECTED_TRAIN_ROWS
        + local_stop
    )


    X_chunk = np.asarray(
        X[
            global_start:global_stop,
            :
        ],
        dtype=np.float64,
        order="C",
    )


    p_xgb = np.asarray(
        xgb_model.predict_proba(
            X_chunk
        )[:, 1],
        dtype=np.float64,
    )


    p_lgbm = np.asarray(
        lgbm_model.predict_proba(
            X_chunk
        )[:, 1],
        dtype=np.float64,
    )


    combined64 = (
        (0.5 * p_lgbm)
        +
        (0.5 * p_xgb)
    )


    ensemble_probability[
        local_start:local_stop
    ] = combined64.astype(
        np.float32,
        copy=False,
    )


    del X_chunk
    del p_xgb
    del p_lgbm
    del combined64

    gc.collect()


    print(
        f"  inferred "
        f"{local_stop:,} / {EXPECTED_VAL_ROWS:,}"
    )


inference_seconds = (
    time.perf_counter()
    - inference_start
)


if not np.all(
    np.isfinite(
        ensemble_probability
    )
):
    raise RuntimeError(
        "Non-finite validation probability."
    )


if (
    np.any(
        ensemble_probability < 0
    )
    or
    np.any(
        ensemble_probability > 1
    )
):
    raise RuntimeError(
        "Validation probability outside [0,1]."
    )


print()
print(
    "Inference seconds:",
    inference_seconds,
)

print()

print(
    "[PASS] seed43 chronological validation inference complete"
)


# =================================================================================================
# 15. VALIDATION RANKING
# =================================================================================================

banner(
    "CHRONOLOGICAL_NATURAL SEED43 — VALIDATION RANKING"
)


validation_roc_auc = float(
    roc_auc_score(
        y_validation,
        ensemble_probability,
    )
)


validation_pr_auc = float(
    average_precision_score(
        y_validation,
        ensemble_probability,
    )
)


print(
    "ROC-AUC:",
    validation_roc_auc,
)

print(
    "PR-AUC :",
    validation_pr_auc,
)

print(
    "PR definition: SKLEARN_AVERAGE_PRECISION_SCORE"
)


# =================================================================================================
# 16. FROZEN THRESHOLD SELECTION
# =================================================================================================

banner(
    "CHRONOLOGICAL_NATURAL SEED43 — FROZEN THRESHOLDS"
)


grid_rows = build_grid(
    y_validation,
    ensemble_probability,
)


standard = next(
    row
    for row in grid_rows
    if row[
        "threshold_integer_percent"
    ] == 50
)


balanced = choose_balanced(
    grid_rows
)

security = choose_security(
    grid_rows
)


if security is None:
    raise RuntimeError(
        "Security threshold infeasible under frozen FPR<=0.05 rule."
    )


print(
    "STANDARD:"
)

print(
    " threshold:",
    standard[
        "threshold"
    ],
)

print(
    " F1       :",
    standard[
        "f1"
    ],
)

print(
    " recall   :",
    standard[
        "recall"
    ],
)

print(
    " FPR      :",
    standard[
        "fpr"
    ],
)

print()

print(
    "BALANCED:"
)

print(
    " threshold:",
    balanced[
        "threshold"
    ],
)

print(
    " F1       :",
    balanced[
        "f1"
    ],
)

print(
    " recall   :",
    balanced[
        "recall"
    ],
)

print(
    " FPR      :",
    balanced[
        "fpr"
    ],
)

print()

print(
    "SECURITY:"
)

print(
    " threshold:",
    security[
        "threshold"
    ],
)

print(
    " F2       :",
    security[
        "f2"
    ],
)

print(
    " recall   :",
    security[
        "recall"
    ],
)

print(
    " FPR      :",
    security[
        "fpr"
    ],
)


# =================================================================================================
# 17. WRITE VALIDATION ARTIFACTS
# =================================================================================================

banner(
    "FREEZE STAGE28-2A4 SCIENTIFIC ARTIFACTS"
)


validation_global_idx = np.arange(
    EXPECTED_TRAIN_ROWS,
    EXPECTED_TOTAL_ROWS,
    dtype=np.int32,
)


np.savez_compressed(
    VALIDATION_PROB_PATH,

    ensemble_probability_float32=
        ensemble_probability,

    validation_global_idx_int32=
        validation_global_idx,

    binary_label_uint8=
        y_validation,
)


grid_df = pd.DataFrame(
    grid_rows
)[
    [
        "threshold_integer_percent",
        "threshold",
        "threshold_float32_runtime",
        "accuracy",
        "precision",
        "recall",
        "fpr",
        "f1",
        "f2",
        "tp",
        "fp",
        "tn",
        "fn",
    ]
]


grid_df.to_csv(
    THRESHOLD_GRID_PATH,
    index=False,
    lineterminator="\n",
)


validation_probability_sha = sha256_file(
    VALIDATION_PROB_PATH
)

threshold_grid_sha = sha256_file(
    THRESHOLD_GRID_PATH
)


print(
    "Validation probability SHA:",
    validation_probability_sha,
)

print(
    "Threshold grid SHA       :",
    threshold_grid_sha,
)


# =================================================================================================
# 18. FINAL PROGRESS RECEIPT
# =================================================================================================

write_progress(
    state="FIT_005_AND_006_AND_VALIDATION_SUCCESS",
    fit5_state="SUCCESS",
    fit6_state="SUCCESS",
    xgb_sha=xgb_model_sha,
    lgbm_sha=lgbm_model_sha,
)


progress_sha = sha256_file(
    PROGRESS_PATH
)


# =================================================================================================
# 19. FIT LEDGER
# =================================================================================================

fit_ledger = {

    "stage":
        "Stage28-2A4",

    "durable_parent":
        EXPECTED_PARENT,

    "evaluation_cell_id":
        "28A_STAGE22::CHRONOLOGICAL_NATURAL::SEED43",

    "authorized_stage28_new_fit_budget":
        108,

    "pre_cell_new_fits_consumed":
        4,

    "this_cell": {
        "successful_new_fits":
            2,

        "new_fit_components": [
            "C013",
            "C014",
        ],

        "reused_components": [],
    },

    "cumulative_new_fits_consumed":
        6,

    "new_fits_remaining":
        102,

    "stage22_new_fits_consumed":
        6,

    "stage22_new_fits_remaining":
        12,

    "model_fits_attempted":
        2,

    "model_fits_successful":
        2,

    "status":
        "FITS_005_AND_006_SUCCESSFULLY_CONSUMED",
}


write_json(
    FIT_LEDGER_PATH,
    fit_ledger,
)


fit_ledger_sha = sha256_file(
    FIT_LEDGER_PATH
)


# =================================================================================================
# 20. RESULT RECEIPT
# =================================================================================================

result = {

    "stage":
        "Stage28-2A4",

    "status":
        "CHRONOLOGICAL_NATURAL_SEED43_VALIDATION_CHECKPOINT_FROZEN",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "parent_commit":
        EXPECTED_PARENT,

    "arm":
        "28A_TRAINING_SEED_STABILITY",

    "experiment":
        "STAGE22_FULL",

    "unit":
        "CHRONOLOGICAL_NATURAL",

    "training_seed":
        43,

    "evaluation_cell_id":
        "28A_STAGE22::CHRONOLOGICAL_NATURAL::SEED43",

    "membership": {
        "train_rows":
            EXPECTED_TRAIN_ROWS,

        "train_attack":
            EXPECTED_TRAIN_ATTACK,

        "train_benign":
            EXPECTED_TRAIN_BENIGN,

        "validation_rows":
            EXPECTED_VAL_ROWS,

        "validation_attack":
            EXPECTED_VAL_ATTACK,

        "validation_benign":
            EXPECTED_VAL_BENIGN,

        "train_days":
            "DAY_ID_0_THROUGH_6",

        "validation_days":
            "DAY_ID_7",

        "membership_seed":
            42,

        "training_seed":
            43,

        "membership_changed_by_training_seed":
            False,
    },

    "models": {
        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "ensemble_probability":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",

        "xgboost": {
            "component_id":
                "C013",

            "fit_action":
                "NEW_FIT_AUTHORIZED",

            "configuration":
                "XGB_11",

            "seed":
                43,

            "backend":
                "cpu",

            "parameters":
                xgb_params,

            "parameter_sha256":
                xgb_parameter_sha,

            "library_version":
                xgb.__version__,

            "boosted_rounds":
                xgb_rounds,

            "fit_seconds":
                float(
                    fit5_seconds
                ),

            "model_path":
                XGB_MODEL_PATH.name,

            "model_sha256":
                xgb_model_sha,
        },

        "lightgbm": {
            "component_id":
                "C014",

            "fit_action":
                "NEW_FIT_AUTHORIZED",

            "configuration":
                "LGBM_11",

            "seed":
                43,

            "backend":
                "cpu",

            "parameters":
                lgbm_params,

            "parameter_sha256":
                lgbm_parameter_sha,

            "library_version":
                lgb.__version__,

            "iterations":
                lgb_iterations,

            "trees":
                lgb_trees,

            "fit_seconds":
                float(
                    fit6_seconds
                ),

            "model_path":
                LGBM_MODEL_PATH.name,

            "model_sha256":
                lgbm_model_sha,
        },
    },

    "training_preparation": {
        "seconds":
            float(
                prep_seconds
            ),

        "rows":
            EXPECTED_TRAIN_ROWS,

        "features":
            EXPECTED_FEATURES,

        "dtype":
            "float64",
    },

    "validation_probability": {
        "rows":
            EXPECTED_VAL_ROWS,

        "roc_auc":
            validation_roc_auc,

        "pr_auc":
            validation_pr_auc,

        "pr_auc_definition":
            "SKLEARN_AVERAGE_PRECISION_SCORE",

        "artifact":
            VALIDATION_PROB_PATH.name,

        "artifact_sha256":
            validation_probability_sha,

        "inference_seconds":
            float(
                inference_seconds
            ),
    },

    "threshold_selection": {
        "selection_population":
            "FROZEN_CHRONOLOGICAL_NATURAL_DEVELOPMENT_VALIDATION",

        "selection_repeated_independently_for_seed":
            43,

        "final_holdout_threshold_search":
            "FORBIDDEN",

        "prediction_rule":
            (
                "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_"
                "PROBABILITY_GTE_THRESHOLD"
            ),

        "grid_integer_percent": [
            5,
            95,
        ],

        "grid_points":
            91,

        "balanced":
            (
                "MAX_F1; TIES LOWER_FPR, HIGHER_RECALL, "
                "CLOSER_TO_0_50, LOWER_THRESHOLD"
            ),

        "security":
            (
                "FPR<=0.05 EXACT; MAX_F2; TIES LOWER_FPR, "
                "HIGHER_RECALL, LOWER_THRESHOLD; NO_RELAXATION"
            ),

        "grid_artifact":
            THRESHOLD_GRID_PATH.name,

        "grid_sha256":
            threshold_grid_sha,
    },

    "operating_points": {
        "standard":
            standard,

        "balanced":
            balanced,

        "security": {
            "status":
                "AVAILABLE",

            "result":
                security,
        },
    },

    "artifacts": {
        "xgboost_model_sha256":
            xgb_model_sha,

        "lightgbm_model_sha256":
            lgbm_model_sha,

        "validation_probability_sha256":
            validation_probability_sha,

        "threshold_grid_sha256":
            threshold_grid_sha,

        "execution_progress_sha256":
            progress_sha,

        "fit_ledger_sha256":
            fit_ledger_sha,
    },

    "scientific_accounting": {
        "new_model_fits_before_cell":
            4,

        "new_model_fits_this_cell":
            2,

        "new_model_fits_cumulative":
            6,

        "new_model_fits_remaining":
            102,

        "existing_models_reused_this_cell":
            0,

        "development_validation_model_inference":
            True,

        "threshold_selection_completed":
            True,

        "shared_final_holdout_openings":
            0,

        "shared_final_holdout_predictor_rows_read":
            0,

        "shared_final_holdout_labels_read":
            0,

        "new_target_openings":
            0,

        "target_adaptive_choices":
            0,
    },

    "next_authorized_step":
        (
            "Stage28-2A5 — RANDOM_NATURAL seed44. "
            "C005 XGBoost and C006 LightGBM are both new CPU fits. "
            "Shared final holdout remains closed."
        ),
}


write_json(
    RESULT_PATH,
    result,
)


result_sha = sha256_file(
    RESULT_PATH
)


# =================================================================================================
# 21. CHECKSUM MANIFEST
# =================================================================================================

artifact_paths = [
    XGB_MODEL_PATH,
    LGBM_MODEL_PATH,
    VALIDATION_PROB_PATH,
    THRESHOLD_GRID_PATH,
    PROGRESS_PATH,
    FIT_LEDGER_PATH,
    RESULT_PATH,
]


checksum_lines = []


for artifact in artifact_paths:

    checksum_lines.append(
        f"{sha256_file(artifact)}  {artifact.name}"
    )


CHECKSUMS_PATH.write_text(
    "\n".join(
        checksum_lines
    )
    + "\n",
    encoding="utf-8",
)


print()
print(
    "Scientific artifact checksums:"
)

print()

print(
    CHECKSUMS_PATH.read_text(
        encoding="utf-8"
    ).rstrip()
)


# =================================================================================================
# 22. SELF-VERIFY ALL SCIENCE ARTIFACTS
# =================================================================================================

banner(
    "SELF-VERIFY STAGE28-2A4 ARTIFACTS"
)


expected_files = {
    XGB_MODEL_PATH.name,
    LGBM_MODEL_PATH.name,
    VALIDATION_PROB_PATH.name,
    THRESHOLD_GRID_PATH.name,
    PROGRESS_PATH.name,
    FIT_LEDGER_PATH.name,
    RESULT_PATH.name,
    CHECKSUMS_PATH.name,
}


actual_files = {
    p.name
    for p in OUT.iterdir()
    if p.is_file()
}


if actual_files != expected_files:
    raise RuntimeError(
        "Unexpected Stage28-2A4 artifact universe.\n"
        f"Expected={sorted(expected_files)}\n"
        f"Actual={sorted(actual_files)}"
    )


for line in (
    CHECKSUMS_PATH.read_text(
        encoding="utf-8"
    ).splitlines()
):

    digest, filename = line.split(
        None,
        1,
    )

    filename = (
        filename.strip()
        .lstrip("*")
    )


    artifact = (
        OUT
        / filename
    )


    actual = sha256_file(
        artifact
    )


    if actual != digest:
        raise RuntimeError(
            f"Checksum mismatch: {filename}"
        )


    print(
        "[PASS]",
        filename,
        digest,
    )


print()
print(
    "[PASS] all Stage28-2A4 science artifacts exact"
)


# =================================================================================================
# 23. SCIENTIFIC LEDGER
# =================================================================================================

banner(
    "STAGE28-2A4 SCIENTIFIC LEDGER"
)

print(
    "NEW fits authorized : 108"
)

print(
    "Pre-cell consumed   : 4"
)

print(
    "This cell consumed  : 2"
)

print(
    "Cumulative consumed : 6"
)

print(
    "Remaining           : 102"
)

print()

print(
    "C013 XGBoost fit     : SUCCESS"
)

print(
    "C014 LightGBM fit    : SUCCESS"
)

print(
    "Validation inference : COMPLETE"
)

print(
    "Threshold selection : COMPLETE"
)

print(
    "Final holdout opening: 0"
)

print()

print(
    "[PASS] FIT #5 and FIT #6 accounted exactly once"
)


# =================================================================================================
# 24. EXPECTED DIRTY-GIT STATE — NO COMMIT/PUSH
# =================================================================================================

banner(
    "SCIENCE-ONLY CHECKPOINT — GIT REMAINS UNCOMMITTED"
)


expected_untracked = {
    str(
        (
            OUT
            / filename
        ).relative_to(
            REPO
        )
    )
    for filename in expected_files
}


tracked_changes = set(
    git(
        "diff",
        "--name-only",
    ).splitlines()
)


untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
)


staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)


if tracked_changes:
    raise RuntimeError(
        "Unexpected tracked modifications:\n"
        + "\n".join(
            sorted(
                tracked_changes
            )
        )
    )


if staged:
    raise RuntimeError(
        "Science-only Stage28-2A4 unexpectedly staged files."
    )


if untracked != expected_untracked:
    raise RuntimeError(
        "Unexpected untracked artifact universe.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                expected_untracked
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                untracked
            )
        )
    )


print(
    "[PASS] exactly eight Stage28-2A4 files are untracked"
)

print(
    "[PASS] nothing staged"
)

print(
    "[PASS] no commit performed"
)

print(
    "[PASS] no push performed"
)


# =================================================================================================
# 25. FINAL
# =================================================================================================

banner(
    "STAGE28-2A4 — SCIENCE COMPLETE / READY FOR SHIFT"
)


print(
    "Durable parent remains:"
)

print(
    " ",
    EXPECTED_PARENT,
)

print()

print(
    "Scientific execution:"
)

print(
    "  C013 XGBoost seed43 CPU = FIT #5 SUCCESS"
)

print(
    "  C014 LightGBM seed43 CPU = FIT #6 SUCCESS"
)

print()

print(
    "Validation:"
)

print(
    "  ROC-AUC =",
    validation_roc_auc,
)

print(
    "  PR-AUC  =",
    validation_pr_auc,
)

print()

print(
    "Thresholds:"
)

print(
    "  STANDARD =",
    standard[
        "threshold"
    ],
)

print(
    "  BALANCED =",
    balanced[
        "threshold"
    ],
)

print(
    "  SECURITY =",
    security[
        "threshold"
    ],
)

print()

print(
    "Scientific ledger:"
)

print(
    "  authorized = 108"
)

print(
    "  consumed   = 6"
)

print(
    "  remaining  = 102"
)

print()

print(
    "Shared final holdout:"
)

print(
    "  openings = 0"
)

print()

print(
    "GIT:"
)

print(
    "  commit = NOT YET"
)

print(
    "  push   = NOT YET"
)

print()

print(
    "STATUS:"
)

print(
    "  READY FOR STAGE28-2A4-SHIFT"
)

print()
print(SEP)


STAGE28-2A4 — EXACT DURABLE-PARENT GATE

Expected parent: 1c21d67d374cfcba864f04d4b2da7fc3fc332568
Local HEAD     : 1c21d67d374cfcba864f04d4b2da7fc3fc332568
origin/main    : 1c21d67d374cfcba864f04d4b2da7fc3fc332568
Remote main    : 1c21d67d374cfcba864f04d4b2da7fc3fc332568
Branch         : main
Git clean      : True

[PASS] exact clean Stage28-2A3 durable parent

FROZEN MODEL RUNTIME GATE

XGBoost : 3.2.0
LightGBM: 4.6.0
[PASS] Stage28 CPU-only policy active

PRIOR DURABLE STAGE28 LEDGER

Authorized: 108
Consumed  : 4
Remaining : 104

[PASS] FIT #5 and FIT #6 are next

FROZEN CHRONOLOGICAL_NATURAL MEMBERSHIP GATE

TRAIN:
 rows  : 13,818,623
 attack: 1,910,043
 benign: 11,908,580

VALIDATION:
 rows  : 593,780
 attack: 62,256
 benign: 531,524

[PASS] exact CHRONOLOGICAL_NATURAL membership

C013 / C014 FROZEN COMPONENT GATE

[PASS] C013 = XGBoost seed43 CPU NEW FIT
[PASS] C014 = LightGBM seed43 CPU NEW FIT

SEED43 FROZEN PARAMETER-SET GATE

XGB expected : 5ac981f17f3327f8de881b672b6bf35a3

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  inferred 65,536 / 593,780


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  inferred 131,072 / 593,780


...

  inferred 524,288 / 593,780


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  inferred 589,824 / 593,780


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  inferred 593,780 / 593,780

Inference seconds: 12.45256919600024

[PASS] seed43 chronological validation inference complete

CHRONOLOGICAL_NATURAL SEED43 — VALIDATION RANKING

ROC-AUC: 0.5079004765003459
PR-AUC : 0.10520468898118242
PR definition: SKLEARN_AVERAGE_PRECISION_SCORE

CHRONOLOGICAL_NATURAL SEED43 — FROZEN THRESHOLDS

STANDARD:
 threshold: 0.5
 F1       : 6.42085493683484e-05
 recall   : 3.2125417630429196e-05
 FPR      : 7.337392102708438e-05

BALANCED:
 threshold: 0.09
 F1       : 0.0001601101557871816
 recall   : 8.031354407607299e-05
 FPR      : 0.00036875098772585997

SECURITY:
 threshold: 0.09
 F2       : 0.00010031096398836393
 recall   : 8.031354407607299e-05
 FPR      : 0.00036875098772585997

FREEZE STAGE28-2A4 SCIENTIFIC ARTIFACTS

Validation probability SHA: d70a5e281be1153d31e2310c2177e294c450e44ecc29e45371f3bae331709081
Threshold grid SHA       : c5dc93a68094a7d5355d0b6784e138394ccd260a16319ca62d9a04146d3198c2

Scientific artifact checksums:

ab14e0fc41fe94a7

In [2]:
# =================================================================================================
# STAGE28-RUNTIME-R3 — GITHUB REPOSITORY RECOVERY AFTER KAGGLE RESET
#
# EXPECTED DURABLE SCIENTIFIC HEAD:
#   1c21d67d374cfcba864f04d4b2da7fc3fc332568
#
# This cell ONLY:
#   - recovers GitHub credential
#   - clones repository if absent
#   - fetches origin/main
#   - verifies exact durable Stage28-2A3 commit
#   - verifies clean worktree
#
# ZERO:
#   predictor reads
#   model fits
#   inference
#   threshold selection
#   holdout access
#   Git writes / commits / pushes
#
# SCIENTIFIC LEDGER REMAINS:
#   consumed  = 4
#   remaining = 104
#
# FIT #5 and FIT #6 HAVE NOT STARTED.
# =================================================================================================

from __future__ import annotations

import os
import subprocess
from pathlib import Path


SEP = "=" * 120

EXPECTED_HEAD = (
    "1c21d67d374cfcba864f04d4b2da7fc3fc332568"
)

EXPECTED_SUBJECT = (
    "stage28-2a3: execute random natural seed43 checkpoint"
)

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

REPO_URL = (
    "https://github.com/themubasshir/"
    "ids2018-validation-safe-ablation.git"
)


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(
    cmd,
    *,
    cwd=None,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=(
            str(cwd)
            if cwd is not None
            else None
        ),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return (
        run(
            ["git", *args],
            cwd=REPO,
        ).stdout
        or ""
    ).strip()


def recover_github_token():

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]


    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()

        for label in labels:

            try:
                value = client.get_secret(
                    label
                )

            except Exception:
                value = None


            if (
                isinstance(value, str)
                and value.strip()
            ):

                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass


    for label in labels:

        value = os.environ.get(
            label
        )

        if (
            isinstance(value, str)
            and value.strip()
        ):

            return (
                value.strip(),
                f"environment:{label}",
            )


    return None, None


# =================================================================================================
# 1. SECRET RECOVERY
# =================================================================================================

banner(
    "GITHUB SECRET RECOVERY"
)

github_token, token_source = (
    recover_github_token()
)


if github_token:

    print(
        "[PASS] GitHub credential recovered from",
        token_source,
    )

    print(
        "[PASS] token intentionally not displayed"
    )

else:

    print(
        "[NOTICE] GitHub token not found."
    )

    print(
        "Read-only repository recovery can continue,"
    )

    print(
        "but the token must exist before the later SHIFT push."
    )


# =================================================================================================
# 2. REPOSITORY CLONE / FETCH
# =================================================================================================

banner(
    "REPOSITORY RECOVERY"
)


if not REPO.exists():

    print(
        "Repository absent — cloning origin/main..."
    )


    clone = run(
        [
            "git",
            "clone",
            REPO_URL,
            str(REPO),
        ]
    )


    if clone.stdout.strip():
        print(
            clone.stdout.strip()
        )


    if clone.stderr.strip():
        print(
            clone.stderr.strip()
        )


else:

    if not (
        REPO
        / ".git"
    ).is_dir():

        raise RuntimeError(
            "Path exists but is not a Git repository:\n"
            f"{REPO}"
        )


    print(
        "Existing repository found."
    )

    print(
        "Fetching origin/main..."
    )


    fetch = run(
        [
            "git",
            "fetch",
            "--prune",
            "origin",
            "main",
        ],
        cwd=REPO,
    )


    if fetch.stdout.strip():
        print(
            fetch.stdout.strip()
        )


    if fetch.stderr.strip():
        print(
            fetch.stderr.strip()
        )


# =================================================================================================
# 3. EXACT REMOTE HEAD GATE
# =================================================================================================

banner(
    "EXACT DURABLE HEAD GATE"
)


local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)


remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


if not remote_line:

    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_head = (
    remote_line.split()[0]
)


branch = git(
    "branch",
    "--show-current",
)


status = git(
    "status",
    "--porcelain",
)


print(
    "Expected HEAD:",
    EXPECTED_HEAD,
)

print(
    "Local HEAD   :",
    local_head,
)

print(
    "origin/main  :",
    origin_head,
)

print(
    "Remote main  :",
    remote_head,
)

print(
    "Branch       :",
    branch,
)

print(
    "Git clean    :",
    not bool(status),
)


if (
    origin_head
    != EXPECTED_HEAD
    or
    remote_head
    != EXPECTED_HEAD
):

    raise RuntimeError(
        "Remote main is not the expected "
        "Stage28-2A3 durable checkpoint.\n"
        "STOP — do not run any model fits."
    )


# =================================================================================================
# 4. NORMALIZE LOCAL CHECKOUT IF NECESSARY
# =================================================================================================

if local_head != EXPECTED_HEAD:

    if status:

        raise RuntimeError(
            "Local checkout differs from expected HEAD "
            "and contains modifications.\n"
            "Refusing automatic reset."
        )


    print()
    print(
        "Local checkout is clean but behind/ahead."
    )

    print(
        "Resetting it to exact origin/main..."
    )


    run(
        [
            "git",
            "checkout",
            "main",
        ],
        cwd=REPO,
    )


    run(
        [
            "git",
            "reset",
            "--hard",
            EXPECTED_HEAD,
        ],
        cwd=REPO,
    )


# =================================================================================================
# 5. FINAL EXACTNESS GATE
# =================================================================================================

banner(
    "FINAL REPOSITORY VERIFICATION"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_origin = git(
    "rev-parse",
    "origin/main",
)

final_status = git(
    "status",
    "--porcelain",
)

final_subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)


print(
    "HEAD       :",
    final_head,
)

print(
    "origin/main:",
    final_origin,
)

print(
    "Subject    :",
    final_subject,
)

print(
    "Git clean  :",
    not bool(final_status),
)


if (
    final_head
    != EXPECTED_HEAD
    or
    final_origin
    != EXPECTED_HEAD
):

    raise RuntimeError(
        "Exact Stage28-2A3 repository recovery failed."
    )


if final_subject != EXPECTED_SUBJECT:

    raise RuntimeError(
        "Recovered commit subject mismatch."
    )


if final_status:

    raise RuntimeError(
        "Repository not clean after recovery:\n"
        + final_status
    )


# =================================================================================================
# 6. VERIFY DURABLE STAGE28-2A3 ARTIFACTS EXIST
# =================================================================================================

banner(
    "STAGE28-2A3 DURABLE ARTIFACT GATE"
)


stage28_2a3 = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a3_random_natural_seed43"
)


required = [
    "random_natural_seed43_xgboost_cpu_model.json",
    "random_natural_seed43_lightgbm_cpu_model.txt",
    "random_natural_seed43_validation_ensemble_probabilities.npz",
    "random_natural_seed43_validation_threshold_grid.csv",
    "stage28_2a3_execution_progress.json",
    "stage28_2a3_fit_ledger.json",
    "stage28_2a3_random_natural_seed43_result.json",
    "checksums.sha256",
]


for name in required:

    path = (
        stage28_2a3
        / name
    )

    if not path.is_file():

        raise RuntimeError(
            f"Missing durable Stage28-2A3 artifact:\n{path}"
        )


    print(
        "[PASS]",
        name,
    )


# =================================================================================================
# 7. FINAL
# =================================================================================================

banner(
    "STAGE28-RUNTIME-R3 — GIT RECOVERY COMPLETE"
)


print(
    "Durable Stage28 HEAD:"
)

print(
    " ",
    final_head,
)

print()

print(
    "Scientific ledger remains:"
)

print(
    "  authorized = 108"
)

print(
    "  consumed   = 4"
)

print(
    "  remaining  = 104"
)

print()

print(
    "Stage28-2A4:"
)

print(
    "  FIT #5 = NOT STARTED"
)

print(
    "  FIT #6 = NOT STARTED"
)

print()

print(
    "Scientific operations in this recovery cell:"
)

print(
    "  predictor reads      = 0"
)

print(
    "  model fits           = 0"
)

print(
    "  inference            = 0"
)

print(
    "  threshold selection  = 0"
)

print(
    "  final holdout access = 0"
)

print()

print(
    "NEXT:"
)

print(
    "  Restore the reset-lost Stage22 runtime cache/matrix."
)

print(
    "  Then resume Stage28-2A4 science-only."
)

print()
print(SEP)


GITHUB SECRET RECOVERY

[PASS] GitHub credential recovered from kaggle_secret:GITHUB_TOKEN
[PASS] token intentionally not displayed

REPOSITORY RECOVERY

Repository absent — cloning origin/main...
Cloning into '/kaggle/working/ids2018-validation-safe-ablation'...
Updating files:  20% (665/3296)
Updating files:  21% (693/3296)
Updating files:  22% (726/3296)
Updating files:  23% (759/3296)
Updating files:  24% (792/3296)
Updating files:  25% (824/3296)
Updating files:  26% (857/3296)
Updating files:  27% (890/3296)
Updating files:  27% (898/3296)
Updating files:  28% (923/3296)
Updating files:  29% (956/3296)
Updating files:  30% (989/3296)
Updating files:  31% (1022/3296)
Updating files:  32% (1055/3296)
Updating files:  33% (1088/3296)
Updating files:  34% (1121/3296)
Updating files:  35% (1154/3296)
Updating files:  36% (1187/3296)
Updating files:  37% (1220/3296)
Updating files:  38% (1253/3296)
Updating files:  39% (1286/3296)
Updating files:  40% (1319/3296)
Updating files:  41% 

In [4]:
# =================================================================================================
# STAGE28-2A4-R0 — RECONSTRUCT RESET-LOST STAGE22 RUNTIME STATE
#
# CURRENT DURABLE SCIENTIFIC HEAD:
#   1c21d67d374cfcba864f04d4b2da7fc3fc332568
#
# CURRENT SCIENTIFIC LEDGER:
#   authorized = 108
#   consumed   = 4
#   remaining  = 104
#
# FIT #5 / FIT #6:
#   NOT STARTED
#
# PURPOSE
# -------
# Recreate only reset-lost disposable Stage22 runtime assets:
#
#   /kaggle/working/stage22r_1c_70f_development_cache/
#
#   /kaggle/working/stage28_2a_runtime_cache/stage22_full/
#       development_features_float64.npy
#       development_binary_labels_uint8.npy
#       development_day_ids_uint8.npy
#       random_natural_train_global_idx_int32.npy
#       random_natural_validation_global_idx_int32.npy
#
#   /kaggle/working/stage28_2a0_stage22_execution_matrix_receipt.json
#
# ZERO:
#   model fits
#   model inference
#   threshold selection
#   final-holdout access
#   Git add
#   Git commit
#   Git push
#
# AFTER SUCCESS:
#   rerun the SAME Stage28-2A4 science-only cell.
# =================================================================================================

from __future__ import annotations

import gc
import hashlib
import json
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq


# =================================================================================================
# 0. CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_HEAD = (
    "1c21d67d374cfcba864f04d4b2da7fc3fc332568"
)

EXPECTED_SUBJECT = (
    "stage28-2a3: execute random natural seed43 checkpoint"
)

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

PREV_DIR = (
    STAGE28_ROOT
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a3_random_natural_seed43"
)

PREV_LEDGER_PATH = (
    PREV_DIR
    / "stage28_2a3_fit_ledger.json"
)

PREV_RESULT_PATH = (
    PREV_DIR
    / "stage28_2a3_random_natural_seed43_result.json"
)

PREV_CHECKSUMS_PATH = (
    PREV_DIR
    / "checksums.sha256"
)


STAGE22_MANIFEST_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1c_development_model_inputs"
    / "stage22r_1c_development_model_input_manifest.json"
)

STAGE22_MEMBERSHIP_SUMMARY_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
    / "stage22r_1b1_membership_summary.json"
)

RANDOM_VALIDATION_PACKBITS_PATH = (
    REPO
    / "results"
    / "stage22r_protocol_recovery"
    / "stage22r_1b1_development_memberships"
    / "random_validation.packbits"
)


STAGE22_CACHE_ROOT = Path(
    "/kaggle/working/stage22r_1c_70f_development_cache"
)

RUNTIME_ROOT = Path(
    "/kaggle/working/"
    "stage28_2a_runtime_cache/"
    "stage22_full"
)

X_PATH = (
    RUNTIME_ROOT
    / "development_features_float64.npy"
)

Y_PATH = (
    RUNTIME_ROOT
    / "development_binary_labels_uint8.npy"
)

DAY_PATH = (
    RUNTIME_ROOT
    / "development_day_ids_uint8.npy"
)

RANDOM_TRAIN_IDX_PATH = (
    RUNTIME_ROOT
    / "random_natural_train_global_idx_int32.npy"
)

RANDOM_VALIDATION_IDX_PATH = (
    RUNTIME_ROOT
    / "random_natural_validation_global_idx_int32.npy"
)

RUNTIME_RECEIPT_PATH = Path(
    "/kaggle/working/"
    "stage28_2a0_stage22_execution_matrix_receipt.json"
)


EXPECTED_ROWS = 14_412_403
EXPECTED_FEATURES = 70

EXPECTED_ATTACK = 1_972_299
EXPECTED_BENIGN = 12_440_104

EXPECTED_NAN = 173_034

EXPECTED_RANDOM_TRAIN_ROWS = 11_529_922
EXPECTED_RANDOM_TRAIN_ATTACK = 1_577_839
EXPECTED_RANDOM_TRAIN_BENIGN = 9_952_083

EXPECTED_RANDOM_VAL_ROWS = 2_882_481
EXPECTED_RANDOM_VAL_ATTACK = 394_460
EXPECTED_RANDOM_VAL_BENIGN = 2_488_021

EXPECTED_CHRONO_TRAIN_ROWS = 13_818_623
EXPECTED_CHRONO_TRAIN_ATTACK = 1_910_043
EXPECTED_CHRONO_TRAIN_BENIGN = 11_908_580

EXPECTED_CHRONO_VAL_ROWS = 593_780
EXPECTED_CHRONO_VAL_ATTACK = 62_256
EXPECTED_CHRONO_VAL_BENIGN = 531_524

EXPECTED_RANDOM_PACKBITS_SHA = (
    "8a308aa5c28008895559a87ba2335a82eac69a4a3b99303d42d598f7afbe2fad"
)

EXPECTED_STAGE22_CONTRACT_SHA = (
    "975113373af098bb12b274248dfa012a536c3e6a8270d7b37dd4f3d9bf79b9f2"
)


# =================================================================================================
# 1. HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return (
        run(
            ["git", *args]
        ).stdout
        or ""
    ).strip()


def require_file(path):
    path = Path(path)

    if not path.is_file():
        raise RuntimeError(
            f"Required file missing:\n{path}"
        )

    return path


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(
    path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                chunk_size
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array(
    arr,
    block_rows=65_536,
):
    h = hashlib.sha256()

    for start in range(
        0,
        arr.shape[0],
        block_rows,
    ):

        stop = min(
            start + block_rows,
            arr.shape[0],
        )

        block = np.ascontiguousarray(
            arr[
                start:stop
            ]
        )

        h.update(
            block.view(
                np.uint8
            )
        )

    return h.hexdigest()


# =================================================================================================
# 2. EXACT CURRENT DURABLE HEAD
# =================================================================================================

banner(
    "STAGE28-2A4-R0 — DURABLE HEAD GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote = (
    remote_line.split()[0]
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)

branch = git(
    "branch",
    "--show-current",
)

status = git(
    "status",
    "--porcelain",
)


print(
    "Expected:",
    EXPECTED_HEAD,
)

print(
    "HEAD    :",
    head,
)

print(
    "origin  :",
    origin,
)

print(
    "remote  :",
    remote,
)

print(
    "Subject :",
    subject,
)

print(
    "Branch  :",
    branch,
)

print(
    "Clean   :",
    not bool(status),
)


if not (
    head
    == origin
    == remote
    == EXPECTED_HEAD
):
    raise RuntimeError(
        "Current durable Stage28 head mismatch."
    )

if subject != EXPECTED_SUBJECT:
    raise RuntimeError(
        "Current Stage28 commit subject mismatch."
    )

if branch != "main":
    raise RuntimeError(
        "Expected main branch."
    )

if status:
    raise RuntimeError(
        "Repository must be clean:\n"
        + status
    )


print()
print(
    "[PASS] exact durable Stage28-2A3 state"
)


# =================================================================================================
# 3. DURABLE SCIENTIFIC LEDGER MUST STILL BE 4 / 104
# =================================================================================================

banner(
    "DURABLE FIT LEDGER"
)

require_file(
    PREV_CHECKSUMS_PATH
)

prev_ledger = read_json(
    require_file(
        PREV_LEDGER_PATH
    )
)

prev_result = read_json(
    require_file(
        PREV_RESULT_PATH
    )
)


if (
    prev_ledger[
        "status"
    ]
    != "FITS_003_AND_004_SUCCESSFULLY_CONSUMED"
):
    raise RuntimeError(
        "Stage28-2A3 durable ledger invalid."
    )


if int(
    prev_ledger[
        "cumulative_new_fits_consumed"
    ]
) != 4:
    raise RuntimeError(
        "Expected 4 durable fits consumed."
    )


if int(
    prev_ledger[
        "new_fits_remaining"
    ]
) != 104:
    raise RuntimeError(
        "Expected 104 durable fits remaining."
    )


if int(
    prev_result[
        "scientific_accounting"
    ][
        "shared_final_holdout_openings"
    ]
) != 0:
    raise RuntimeError(
        "Final holdout was previously opened."
    )


print(
    "Authorized : 108"
)

print(
    "Consumed   : 4"
)

print(
    "Remaining  : 104"
)

print(
    "FIT #5     : NOT STARTED"
)

print(
    "FIT #6     : NOT STARTED"
)

print(
    "Holdout    : CLOSED"
)

print()
print(
    "[PASS] scientific ledger unchanged"
)


# =================================================================================================
# 4. LOAD HISTORICAL STAGE22 CONTRACT
# =================================================================================================

banner(
    "LOAD HISTORICAL STAGE22 CONTRACT"
)

manifest = read_json(
    require_file(
        STAGE22_MANIFEST_PATH
    )
)

membership = read_json(
    require_file(
        STAGE22_MEMBERSHIP_SUMMARY_PATH
    )
)


feature_order = list(
    manifest[
        "row_schema"
    ][
        "predictor_columns"
    ]
)


if len(
    feature_order
) != EXPECTED_FEATURES:
    raise RuntimeError(
        "Stage22 feature count changed."
    )


cache_records = sorted(
    manifest[
        "cache"
    ][
        "files"
    ],
    key=lambda x: int(
        x[
            "day_id"
        ]
    ),
)


if len(
    cache_records
) != 8:
    raise RuntimeError(
        "Expected eight Stage22 cache files."
    )


if int(
    manifest[
        "cache"
    ][
        "total_rows"
    ]
) != EXPECTED_ROWS:
    raise RuntimeError(
        "Stage22 row count changed."
    )


print(
    "Rows      :",
    f"{EXPECTED_ROWS:,}",
)

print(
    "Features  :",
    EXPECTED_FEATURES,
)

print(
    "Dtype     : float64"
)

print(
    "First     :",
    feature_order[0],
)

print(
    "Last      :",
    feature_order[-1],
)

print()

print(
    "[PASS] frozen Stage22 input contract loaded"
)


# =================================================================================================
# 5. LOCATE 8 BYTE-EXACT CACHE FILES IN /kaggle/input
# =================================================================================================

banner(
    "RECOVER 8 BYTE-EXACT STAGE22 CACHE FILES"
)

input_root = Path(
    "/kaggle/input"
)


if not input_root.exists():
    raise RuntimeError(
        "/kaggle/input unavailable."
    )


all_files = [
    p
    for p in input_root.rglob("*")
    if p.is_file()
]


resolved = {}


for record in cache_records:

    filename = record[
        "cache_file"
    ]

    expected_bytes = int(
        record[
            "bytes"
        ]
    )

    expected_sha = record[
        "sha256"
    ]


    candidates = [
        p
        for p in all_files
        if (
            p.name == filename
            and
            int(
                p.stat().st_size
            )
            == expected_bytes
        )
    ]


    exact = []

    for candidate in candidates:

        if sha256_file(
            candidate
        ) == expected_sha:

            exact.append(
                candidate
            )


    if not exact:

        raise RuntimeError(
            "Unable to locate byte-exact Stage22 cache:\n"
            f"{filename}\n\n"
            "Attach private Kaggle dataset:\n"
            "jmmubasshirrahman/"
            "stage22r-1c-70f-cache-3cd41c5f"
        )


    resolved[
        filename
    ] = exact[0]


    print(
        "[PASS]",
        filename,
    )

    print(
        "       ",
        exact[0],
    )


print()
print(
    "[PASS] 8/8 historical Stage22 cache files recovered"
)


# =================================================================================================
# 6. RECREATE RUNTIME CACHE BINDINGS
# =================================================================================================

banner(
    "RECREATE STAGE22 CACHE BINDINGS"
)


if STAGE22_CACHE_ROOT.exists():
    shutil.rmtree(
        STAGE22_CACHE_ROOT
    )


STAGE22_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


for record in cache_records:

    filename = record[
        "cache_file"
    ]

    source = (
        resolved[
            filename
        ].resolve()
    )

    target = (
        STAGE22_CACHE_ROOT
        / filename
    )


    target.symlink_to(
        source
    )


    if sha256_file(
        target
    ) != record[
        "sha256"
    ]:
        raise RuntimeError(
            f"Runtime binding failed: {filename}"
        )


    print(
        "[PASS]",
        target,
    )


print()
print(
    "[PASS] byte-exact runtime cache bindings restored"
)


# =================================================================================================
# 7. STORAGE GATE
# =================================================================================================

banner(
    "RUNTIME STORAGE GATE"
)


disk = shutil.disk_usage(
    "/kaggle/working"
)


x_payload = (
    EXPECTED_ROWS
    * EXPECTED_FEATURES
    * 8
)


print(
    "Free space:",
    f"{disk.free / (1024**3):.3f} GiB",
)

print(
    "X payload :",
    f"{x_payload / (1024**3):.3f} GiB",
)


if disk.free < (
    x_payload
    + int(
        1.5
        * 1024**3
    )
):
    raise RuntimeError(
        "Insufficient runtime disk space."
    )


print()
print(
    "[PASS] sufficient disk space"
)


# =================================================================================================
# 8. CLEAN / CREATE DISPOSABLE EXECUTION ROOT
# =================================================================================================

banner(
    "ALLOCATE DISPOSABLE STAGE22 EXECUTION MATRIX"
)


if RUNTIME_ROOT.exists():
    shutil.rmtree(
        RUNTIME_ROOT
    )


RUNTIME_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)


X = np.lib.format.open_memmap(
    X_PATH,
    mode="w+",
    dtype=np.float64,
    shape=(
        EXPECTED_ROWS,
        EXPECTED_FEATURES,
    ),
)

y = np.lib.format.open_memmap(
    Y_PATH,
    mode="w+",
    dtype=np.uint8,
    shape=(
        EXPECTED_ROWS,
    ),
)

day_ids = np.lib.format.open_memmap(
    DAY_PATH,
    mode="w+",
    dtype=np.uint8,
    shape=(
        EXPECTED_ROWS,
    ),
)


print(
    "X:",
    X.shape,
    X.dtype,
)

print(
    "y:",
    y.shape,
    y.dtype,
)

print(
    "day:",
    day_ids.shape,
    day_ids.dtype,
)


# =================================================================================================
# 9. MATERIALIZE CANONICAL DEVELOPMENT MATRIX
# =================================================================================================

banner(
    "MATERIALIZE CANONICAL STAGE22 DEVELOPMENT MATRIX"
)


required_columns = [
    "clean_position",
    "day_id",
    "binary_label",
    *feature_order,
]


cursor = 0

total_attack = 0
total_benign = 0
total_nan = 0
total_inf = 0

day_audit = []


for record in cache_records:

    day_id = int(
        record[
            "day_id"
        ]
    )

    path = (
        STAGE22_CACHE_ROOT
        / record[
            "cache_file"
        ]
    )

    pf = pq.ParquetFile(
        path
    )


    expected_schema = [
        "clean_position",
        "day_id",
        "original_row_index",
        "binary_label",
        *feature_order,
    ]


    if list(
        pf.schema_arrow.names
    ) != expected_schema:
        raise RuntimeError(
            f"{path.name}: ordered schema mismatch."
        )


    day_start = cursor

    day_attack = 0
    day_benign = 0
    day_nan = 0


    print()
    print(
        "-" * 100
    )

    print(
        f"Day {day_id}: {path.name}"
    )

    print(
        "-" * 100
    )


    for batch in pf.iter_batches(
        batch_size=65_536,
        columns=required_columns,
        use_threads=True,
    ):

        n = int(
            batch.num_rows
        )


        clean_position = (
            batch
            .column(0)
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.int64,
                copy=False,
            )
        )


        block_day = (
            batch
            .column(1)
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.uint8,
                copy=False,
            )
        )


        block_y = (
            batch
            .column(2)
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.uint8,
                copy=False,
            )
        )


        expected_position = np.arange(
            cursor,
            cursor + n,
            dtype=np.int64,
        )


        if not np.array_equal(
            clean_position,
            expected_position,
        ):
            raise RuntimeError(
                f"{path.name}: clean_position order mismatch "
                f"at {cursor:,}."
            )


        if not np.all(
            block_day == day_id
        ):
            raise RuntimeError(
                f"{path.name}: day_id mismatch."
            )


        feature_arrays = [
            batch
            .column(
                3 + j
            )
            .to_numpy(
                zero_copy_only=False
            )
            .astype(
                np.float64,
                copy=False,
            )

            for j
            in range(
                EXPECTED_FEATURES
            )
        ]


        X_block = np.column_stack(
            feature_arrays
        )


        block_inf = int(
            np.isinf(
                X_block
            ).sum()
        )

        block_nan = int(
            np.isnan(
                X_block
            ).sum()
        )


        if block_inf != 0:
            raise RuntimeError(
                f"{path.name}: infinity detected."
            )


        stop = cursor + n


        X[
            cursor:stop,
            :
        ] = X_block

        y[
            cursor:stop
        ] = block_y

        day_ids[
            cursor:stop
        ] = block_day


        block_attack = int(
            block_y.sum()
        )


        day_attack += (
            block_attack
        )

        day_benign += (
            n
            - block_attack
        )

        day_nan += (
            block_nan
        )

        total_inf += (
            block_inf
        )


        cursor = stop


        del clean_position
        del expected_position
        del block_day
        del block_y
        del feature_arrays
        del X_block

        gc.collect()


    day_rows = (
        cursor
        - day_start
    )


    if day_rows != int(
        record[
            "rows"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: row count mismatch."
        )


    if day_attack != int(
        record[
            "attack"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: attack count mismatch."
        )


    if day_benign != int(
        record[
            "benign"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: benign count mismatch."
        )


    total_attack += (
        day_attack
    )

    total_benign += (
        day_benign
    )

    total_nan += (
        day_nan
    )


    day_audit.append(
        {
            "day_id":
                day_id,

            "cache_file":
                path.name,

            "global_start":
                day_start,

            "global_stop_exclusive":
                cursor,

            "rows":
                day_rows,

            "attack":
                day_attack,

            "benign":
                day_benign,

            "nan_cells":
                day_nan,

            "inf_cells":
                0,
        }
    )


    print(
        "Rows  :",
        f"{day_rows:,}",
    )

    print(
        "Attack:",
        f"{day_attack:,}",
    )

    print(
        "Benign:",
        f"{day_benign:,}",
    )

    print(
        "NaNs  :",
        f"{day_nan:,}",
    )

    print(
        "[PASS] day exact"
    )


if cursor != EXPECTED_ROWS:
    raise RuntimeError(
        "Global row count mismatch."
    )


if total_attack != EXPECTED_ATTACK:
    raise RuntimeError(
        "Global attack count mismatch."
    )


if total_benign != EXPECTED_BENIGN:
    raise RuntimeError(
        "Global benign count mismatch."
    )


if total_nan != EXPECTED_NAN:
    raise RuntimeError(
        f"Global NaN count mismatch: "
        f"{total_nan:,} != {EXPECTED_NAN:,}"
    )


if total_inf != 0:
    raise RuntimeError(
        "Infinity detected globally."
    )


X.flush()
y.flush()
day_ids.flush()


print()
print(
    "[PASS] complete Stage22 matrix materialized"
)

print(
    "Rows  :",
    f"{EXPECTED_ROWS:,}",
)

print(
    "Attack:",
    f"{total_attack:,}",
)

print(
    "Benign:",
    f"{total_benign:,}",
)

print(
    "NaNs  :",
    f"{total_nan:,}",
)

print(
    "Inf   :",
    total_inf,
)


# =================================================================================================
# 10. RELOAD READ-ONLY MEMMAPS
# =================================================================================================

del X
del y
del day_ids

gc.collect()


X = np.load(
    X_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

y = np.load(
    Y_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

day_ids = np.load(
    DAY_PATH,
    mmap_mode="r",
    allow_pickle=False,
)


if X.shape != (
    EXPECTED_ROWS,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        "Reloaded X shape mismatch."
    )


# =================================================================================================
# 11. RECREATE RANDOM MEMBERSHIP
# =================================================================================================

banner(
    "RECREATE RANDOM_NATURAL MEMBERSHIP"
)


pack_sha = sha256_file(
    require_file(
        RANDOM_VALIDATION_PACKBITS_PATH
    )
)


if pack_sha != EXPECTED_RANDOM_PACKBITS_SHA:
    raise RuntimeError(
        "Random-validation packbits SHA mismatch."
    )


packed = np.fromfile(
    RANDOM_VALIDATION_PACKBITS_PATH,
    dtype=np.uint8,
)


random_val_mask = np.unpackbits(
    packed,
    bitorder="little",
)[:EXPECTED_ROWS].astype(
    np.bool_,
    copy=False,
)


random_validation_idx = np.flatnonzero(
    random_val_mask
).astype(
    np.int32,
    copy=False,
)


random_train_idx = np.flatnonzero(
    ~random_val_mask
).astype(
    np.int32,
    copy=False,
)


if len(
    random_train_idx
) != EXPECTED_RANDOM_TRAIN_ROWS:
    raise RuntimeError(
        "Random train size mismatch."
    )


if len(
    random_validation_idx
) != EXPECTED_RANDOM_VAL_ROWS:
    raise RuntimeError(
        "Random validation size mismatch."
    )


random_train_attack = int(
    np.asarray(
        y[
            random_train_idx
        ]
    ).sum()
)


random_validation_attack = int(
    np.asarray(
        y[
            random_validation_idx
        ]
    ).sum()
)


if random_train_attack != EXPECTED_RANDOM_TRAIN_ATTACK:
    raise RuntimeError(
        "Random train attack count mismatch."
    )


if random_validation_attack != EXPECTED_RANDOM_VAL_ATTACK:
    raise RuntimeError(
        "Random validation attack count mismatch."
    )


np.save(
    RANDOM_TRAIN_IDX_PATH,
    random_train_idx,
    allow_pickle=False,
)

np.save(
    RANDOM_VALIDATION_IDX_PATH,
    random_validation_idx,
    allow_pickle=False,
)


random_train_sha = sha256_array(
    random_train_idx
)

random_validation_sha = sha256_array(
    random_validation_idx
)


print(
    "Train:",
    f"{len(random_train_idx):,}",
)

print(
    " attack:",
    f"{random_train_attack:,}",
)

print(
    " benign:",
    f"{EXPECTED_RANDOM_TRAIN_BENIGN:,}",
)

print(
    " SHA:",
    random_train_sha,
)

print()

print(
    "Validation:",
    f"{len(random_validation_idx):,}",
)

print(
    " attack:",
    f"{random_validation_attack:,}",
)

print(
    " benign:",
    f"{EXPECTED_RANDOM_VAL_BENIGN:,}",
)

print(
    " SHA:",
    random_validation_sha,
)

print()
print(
    "[PASS] RANDOM_NATURAL membership reconstructed"
)


# =================================================================================================
# 12. VERIFY CHRONOLOGICAL MEMBERSHIP
# =================================================================================================

banner(
    "VERIFY CHRONOLOGICAL_NATURAL MEMBERSHIP"
)


if not np.all(
    day_ids[
        :EXPECTED_CHRONO_TRAIN_ROWS
    ] <= 6
):
    raise RuntimeError(
        "Chronological train geometry mismatch."
    )


if not np.all(
    day_ids[
        EXPECTED_CHRONO_TRAIN_ROWS:
    ] == 7
):
    raise RuntimeError(
        "Chronological validation geometry mismatch."
    )


chrono_train_attack = int(
    np.asarray(
        y[
            :EXPECTED_CHRONO_TRAIN_ROWS
        ]
    ).sum()
)


chrono_val_attack = int(
    np.asarray(
        y[
            EXPECTED_CHRONO_TRAIN_ROWS:
        ]
    ).sum()
)


if chrono_train_attack != EXPECTED_CHRONO_TRAIN_ATTACK:
    raise RuntimeError(
        "Chronological train attack mismatch."
    )


if chrono_val_attack != EXPECTED_CHRONO_VAL_ATTACK:
    raise RuntimeError(
        "Chronological validation attack mismatch."
    )


print(
    "Train:"
)

print(
    " rows  :",
    f"{EXPECTED_CHRONO_TRAIN_ROWS:,}",
)

print(
    " attack:",
    f"{chrono_train_attack:,}",
)

print(
    " benign:",
    f"{EXPECTED_CHRONO_TRAIN_BENIGN:,}",
)

print()

print(
    "Validation:"
)

print(
    " rows  :",
    f"{EXPECTED_CHRONO_VAL_ROWS:,}",
)

print(
    " attack:",
    f"{chrono_val_attack:,}",
)

print(
    " benign:",
    f"{EXPECTED_CHRONO_VAL_BENIGN:,}",
)

print()

print(
    "[PASS] CHRONOLOGICAL_NATURAL membership reconstructed"
)


# =================================================================================================
# 13. RUNTIME IDENTITIES
# =================================================================================================

banner(
    "RUNTIME CONTENT IDENTITIES"
)


y_sha = sha256_array(
    y
)

day_sha = sha256_array(
    day_ids
)


print(
    "Labels SHA:",
    y_sha,
)

print(
    "Days SHA  :",
    day_sha,
)


# =================================================================================================
# 14. WRITE NEW RESET-RECOVERY RUNTIME RECEIPT
# =================================================================================================

banner(
    "WRITE RESET-RECOVERY RUNTIME RECEIPT"
)


runtime_receipt = {

    "stage":
        "Stage28-2A4-R0",

    "type":
        "RESET_RECOVERY_STAGE22_RUNTIME_RECONSTRUCTION",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "durable_scientific_parent":
        EXPECTED_HEAD,

    "stage22_contract_sha256":
        EXPECTED_STAGE22_CONTRACT_SHA,

    "runtime_matrix": {
        "features_path":
            str(X_PATH),

        "labels_path":
            str(Y_PATH),

        "day_ids_path":
            str(DAY_PATH),

        "shape": [
            EXPECTED_ROWS,
            EXPECTED_FEATURES,
        ],

        "dtype":
            "float64",

        "rows":
            EXPECTED_ROWS,

        "attack":
            total_attack,

        "benign":
            total_benign,

        "nan_cells":
            total_nan,

        "inf_cells":
            total_inf,

        "binary_labels_content_sha256":
            y_sha,

        "day_ids_content_sha256":
            day_sha,
    },

    "day_audit":
        day_audit,

    "random_natural": {
        "validation_packbits_sha256":
            pack_sha,

        "train_index_path":
            str(
                RANDOM_TRAIN_IDX_PATH
            ),

        "validation_index_path":
            str(
                RANDOM_VALIDATION_IDX_PATH
            ),

        "train_index_content_sha256":
            random_train_sha,

        "validation_index_content_sha256":
            random_validation_sha,

        "train": {
            "rows":
                EXPECTED_RANDOM_TRAIN_ROWS,

            "attack":
                EXPECTED_RANDOM_TRAIN_ATTACK,

            "benign":
                EXPECTED_RANDOM_TRAIN_BENIGN,
        },

        "validation": {
            "rows":
                EXPECTED_RANDOM_VAL_ROWS,

            "attack":
                EXPECTED_RANDOM_VAL_ATTACK,

            "benign":
                EXPECTED_RANDOM_VAL_BENIGN,
        },
    },

    "chronological_natural": {
        "train_slice": [
            0,
            EXPECTED_CHRONO_TRAIN_ROWS,
        ],

        "validation_slice": [
            EXPECTED_CHRONO_TRAIN_ROWS,
            EXPECTED_ROWS,
        ],

        "train": {
            "rows":
                EXPECTED_CHRONO_TRAIN_ROWS,

            "attack":
                EXPECTED_CHRONO_TRAIN_ATTACK,

            "benign":
                EXPECTED_CHRONO_TRAIN_BENIGN,
        },

        "validation": {
            "rows":
                EXPECTED_CHRONO_VAL_ROWS,

            "attack":
                EXPECTED_CHRONO_VAL_ATTACK,

            "benign":
                EXPECTED_CHRONO_VAL_BENIGN,
        },
    },

    "scientific_operations": {
        "model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "final_holdout_openings":
            0,
    },

    "fit_ledger": {
        "new_authorized":
            108,

        "new_consumed":
            4,

        "new_remaining":
            104,

        "next_fit":
            5,
    },

    "status":
        "READY_FOR_STAGE28_FIT_005",
}


RUNTIME_RECEIPT_PATH.write_text(
    json.dumps(
        runtime_receipt,
        indent=2,
        ensure_ascii=False,
        allow_nan=False,
    )
    + "\n",
    encoding="utf-8",
)


print(
    "Receipt:"
)

print(
    " ",
    RUNTIME_RECEIPT_PATH,
)

print()
print(
    "[PASS] reset-lost runtime receipt recreated"
)


# =================================================================================================
# 15. FINAL GIT GATE — MUST REMAIN UNCHANGED
# =================================================================================================

banner(
    "FINAL GIT / SCIENTIFIC STATE GATE"
)


final_head = git(
    "rev-parse",
    "HEAD",
)

final_origin = git(
    "rev-parse",
    "origin/main",
)

final_remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


if not final_remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )


final_remote = (
    final_remote_line.split()[0]
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "HEAD       :",
    final_head,
)

print(
    "origin/main:",
    final_origin,
)

print(
    "remote main:",
    final_remote,
)

print(
    "Git clean  :",
    not bool(
        final_status
    ),
)


if not (
    final_head
    == final_origin
    == final_remote
    == EXPECTED_HEAD
):
    raise RuntimeError(
        "Durable repository changed during runtime recovery."
    )


if final_status:
    raise RuntimeError(
        "Repository unexpectedly dirty after runtime recovery:\n"
        + final_status
    )


print()
print(
    "[PASS] durable scientific repository untouched"
)


# =================================================================================================
# 16. FINAL
# =================================================================================================

banner(
    "STAGE28-2A4-R0 — RUNTIME RECOVERY COMPLETE"
)


print(
    "Durable HEAD:"
)

print(
    " ",
    EXPECTED_HEAD,
)

print()

print(
    "Stage22 runtime:"
)

print(
    "  rows      =",
    f"{EXPECTED_ROWS:,}",
)

print(
    "  features  = 70"
)

print(
    "  dtype     = float64"
)

print(
    "  NaNs      =",
    f"{total_nan:,}",
)

print(
    "  infinity  = 0"
)

print()

print(
    "RANDOM_NATURAL:"
)

print(
    "  train      =",
    f"{EXPECTED_RANDOM_TRAIN_ROWS:,}",
)

print(
    "  validation =",
    f"{EXPECTED_RANDOM_VAL_ROWS:,}",
)

print()

print(
    "CHRONOLOGICAL_NATURAL:"
)

print(
    "  train      =",
    f"{EXPECTED_CHRONO_TRAIN_ROWS:,}",
)

print(
    "  validation =",
    f"{EXPECTED_CHRONO_VAL_ROWS:,}",
)

print()

print(
    "Scientific ledger:"
)

print(
    "  authorized = 108"
)

print(
    "  consumed   = 4"
)

print(
    "  remaining  = 104"
)

print()

print(
    "FIT #5 = NOT STARTED"
)

print(
    "FIT #6 = NOT STARTED"
)

print()

print(
    "Final holdout openings = 0"
)

print()

print(
    "STATUS:"
)

print(
    "  READY FOR STAGE28 FIT #5"
)

print()

print(
    "NEXT:"
)

print(
    "  Rerun the SAME Stage28-2A4 "
    "CHRONOLOGICAL_NATURAL seed43 science-only cell."
)

print()
print(SEP)


STAGE28-2A4-R0 — DURABLE HEAD GATE

Expected: 1c21d67d374cfcba864f04d4b2da7fc3fc332568
HEAD    : 1c21d67d374cfcba864f04d4b2da7fc3fc332568
origin  : 1c21d67d374cfcba864f04d4b2da7fc3fc332568
remote  : 1c21d67d374cfcba864f04d4b2da7fc3fc332568
Subject : stage28-2a3: execute random natural seed43 checkpoint
Branch  : main
Clean   : True

[PASS] exact durable Stage28-2A3 state

DURABLE FIT LEDGER

Authorized : 108
Consumed   : 4
Remaining  : 104
FIT #5     : NOT STARTED
FIT #6     : NOT STARTED
Holdout    : CLOSED

[PASS] scientific ledger unchanged

LOAD HISTORICAL STAGE22 CONTRACT

Rows      : 14,412,403
Features  : 70
Dtype     : float64
First     : Dst Port
Last      : Idle Min

[PASS] frozen Stage22 input contract loaded

RECOVER 8 BYTE-EXACT STAGE22 CACHE FILES

[PASS] day_00_02-14-2018.parquet
        /kaggle/input/datasets/jmmubasshirrahman/stage22r-1c-70f-cache-3cd41c5f/day_00_02-14-2018.parquet
[PASS] day_01_02-15-2018.parquet
        /kaggle/input/datasets/jmmubasshirrahman/stage

In [6]:
# =================================================================================================
# STAGE28-2A4-SHIFT — COMMIT + PUSH ONLY
#
# SCIENCE ALREADY COMPLETE:
#   FIT #5 — C013 XGBoost seed43 CPU   = SUCCESS
#   FIT #6 — C014 LightGBM seed43 CPU = SUCCESS
#
# PRE-COMMIT LEDGER:
#   consumed = 6
#   remaining = 102
#
# THIS CELL DOES:
#   verify exact artifacts/hashes
#   verify ledger
#   git add
#   git commit
#   git push
#   verify remote durability
#
# ZERO FITS / ZERO INFERENCE / ZERO THRESHOLD SEARCH / ZERO HOLDOUT ACCESS
# =================================================================================================

from __future__ import annotations

import base64
import hashlib
import json
import os
import subprocess
from pathlib import Path


SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "1c21d67d374cfcba864f04d4b2da7fc3fc332568"
)

COMMIT_MESSAGE = (
    "stage28-2a4: execute chronological natural seed43 checkpoint"
)

OUT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a4_chronological_natural_seed43"
)

EXPECTED_HASHES = {
    "chronological_natural_seed43_xgboost_cpu_model.json":
        "ab14e0fc41fe94a7123d220f160b4ee4d6b469d46a9a07c03e028a7e710bea14",

    "chronological_natural_seed43_lightgbm_cpu_model.txt":
        "00f2ea97707ec5b9e39027fe782cd7db99447d6c0c1635cc55123a9abfb54581",

    "chronological_natural_seed43_validation_ensemble_probabilities.npz":
        "d70a5e281be1153d31e2310c2177e294c450e44ecc29e45371f3bae331709081",

    "chronological_natural_seed43_validation_threshold_grid.csv":
        "c5dc93a68094a7d5355d0b6784e138394ccd260a16319ca62d9a04146d3198c2",

    "stage28_2a4_execution_progress.json":
        "80c8dc24668875899c3de1e6aabc829ac5cf0544081ef14127f7b6d5b8bf0027",

    "stage28_2a4_fit_ledger.json":
        "73ef6ef339ea65c6e260f36512e99eceac0688e2ce6164db486aa19cbf142d9f",

    "stage28_2a4_chronological_natural_seed43_result.json":
        "2bfe2788bb9f1eba7de4b31ac7a73b32abbdfdf1886ce50a18d68ceac6924471",
}

CHECKSUMS_PATH = (
    OUT
    / "checksums.sha256"
)

LEDGER_PATH = (
    OUT
    / "stage28_2a4_fit_ledger.json"
)

RESULT_PATH = (
    OUT
    / "stage28_2a4_chronological_natural_seed43_result.json"
)

PROGRESS_PATH = (
    OUT
    / "stage28_2a4_execution_progress.json"
)


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, *, check=True):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args, check=True):

    return (
        run(
            ["git", *args],
            check=check,
        ).stdout
        or ""
    ).strip()


def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def read_json(path):

    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def recover_github_token():

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]


    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()

        for label in labels:

            try:
                value = client.get_secret(label)

            except Exception:
                value = None

            if (
                isinstance(value, str)
                and value.strip()
            ):
                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass


    for label in labels:

        value = os.environ.get(label)

        if (
            isinstance(value, str)
            and value.strip()
        ):
            return (
                value.strip(),
                f"environment:{label}",
            )


    raise RuntimeError(
        "GitHub credential unavailable."
    )


def authenticated_push(token):

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode("utf-8")
    ).decode("ascii")


    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )


    if p.returncode != 0:

        raise RuntimeError(
            "Git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )


    return (
        p.stdout
        + p.stderr
    ).strip()


# =================================================================================================
# 1. EXACT PARENT
# =================================================================================================

banner(
    "STAGE28-2A4-SHIFT — EXACT PARENT GATE"
)

head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)

if not remote_line:
    raise RuntimeError(
        "Unable to resolve remote main."
    )

remote = remote_line.split()[0]


print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    head,
)

print(
    "origin/main    :",
    origin,
)

print(
    "Remote main    :",
    remote,
)


if not (
    head
    == origin
    == remote
    == EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-2A4 SHIFT parent mismatch."
    )


print()
print(
    "[PASS] exact Stage28-2A3 durable parent"
)


# =================================================================================================
# 2. SCIENCE ARTIFACT HASH GATE
# =================================================================================================

banner(
    "STAGE28-2A4 SCIENTIFIC ARTIFACT GATE"
)


for filename, expected_sha in (
    EXPECTED_HASHES.items()
):

    path = (
        OUT
        / filename
    )

    if not path.is_file():
        raise RuntimeError(
            f"Missing scientific artifact:\n{path}"
        )

    actual_sha = sha256_file(
        path
    )

    print(
        "[CHECK]",
        filename,
    )

    print(
        " expected:",
        expected_sha,
    )

    print(
        " actual  :",
        actual_sha,
    )


    if actual_sha != expected_sha:

        raise RuntimeError(
            f"Scientific artifact changed: {filename}"
        )


if not CHECKSUMS_PATH.is_file():

    raise RuntimeError(
        "checksums.sha256 missing."
    )


print()
print(
    "[PASS] all seven scientific payloads byte-exact"
)


# =================================================================================================
# 3. CHECKSUM MANIFEST
# =================================================================================================

banner(
    "CHECKSUM MANIFEST GATE"
)


lines = (
    CHECKSUMS_PATH
    .read_text(
        encoding="utf-8"
    )
    .splitlines()
)


if len(lines) != 7:

    raise RuntimeError(
        "checksums.sha256 must contain exactly seven payload entries."
    )


for line in lines:

    digest, filename = line.split(
        None,
        1,
    )

    filename = (
        filename.strip()
        .lstrip("*")
    )

    actual = sha256_file(
        OUT
        / filename
    )

    if actual != digest:

        raise RuntimeError(
            f"Checksum manifest mismatch: {filename}"
        )

    print(
        "[PASS]",
        filename,
        actual,
    )


# =================================================================================================
# 4. SCIENTIFIC LEDGER
# =================================================================================================

banner(
    "FIT #5 / FIT #6 LEDGER GATE"
)


ledger = read_json(
    LEDGER_PATH
)

result = read_json(
    RESULT_PATH
)

progress = read_json(
    PROGRESS_PATH
)


if (
    progress[
        "state"
    ]
    != "FIT_005_AND_006_AND_VALIDATION_SUCCESS"
):

    raise RuntimeError(
        "Stage28-2A4 execution-progress state invalid."
    )


if (
    progress[
        "fit_5"
    ][
        "state"
    ]
    != "SUCCESS"
):

    raise RuntimeError(
        "FIT #5 not confirmed successful."
    )


if (
    progress[
        "fit_6"
    ][
        "state"
    ]
    != "SUCCESS"
):

    raise RuntimeError(
        "FIT #6 not confirmed successful."
    )


if (
    ledger[
        "status"
    ]
    != "FITS_005_AND_006_SUCCESSFULLY_CONSUMED"
):

    raise RuntimeError(
        "Stage28-2A4 fit ledger invalid."
    )


if int(
    ledger[
        "cumulative_new_fits_consumed"
    ]
) != 6:

    raise RuntimeError(
        "Expected cumulative fit count = 6."
    )


if int(
    ledger[
        "new_fits_remaining"
    ]
) != 102:

    raise RuntimeError(
        "Expected remaining fit count = 102."
    )


if (
    ledger[
        "this_cell"
    ][
        "new_fit_components"
    ]
    != [
        "C013",
        "C014",
    ]
):

    raise RuntimeError(
        "Unexpected Stage28-2A4 component identities."
    )


if int(
    result[
        "scientific_accounting"
    ][
        "shared_final_holdout_openings"
    ]
) != 0:

    raise RuntimeError(
        "Final holdout opening detected."
    )


print(
    "[PASS] FIT #5 = C013 SUCCESS"
)

print(
    "[PASS] FIT #6 = C014 SUCCESS"
)

print(
    "[PASS] consumed = 6"
)

print(
    "[PASS] remaining = 102"
)

print(
    "[PASS] final holdout openings = 0"
)


# =================================================================================================
# 5. RESULT SUMMARY
# =================================================================================================

banner(
    "FROZEN STAGE28-2A4 RESULT"
)


print(
    "ROC-AUC:",
    result[
        "validation_probability"
    ][
        "roc_auc"
    ],
)

print(
    "PR-AUC :",
    result[
        "validation_probability"
    ][
        "pr_auc"
    ],
)

print()

print(
    "STANDARD:",
    result[
        "operating_points"
    ][
        "standard"
    ][
        "threshold"
    ],
)

print(
    "BALANCED:",
    result[
        "operating_points"
    ][
        "balanced"
    ][
        "threshold"
    ],
)

print(
    "SECURITY:",
    result[
        "operating_points"
    ][
        "security"
    ][
        "result"
    ][
        "threshold"
    ],
)


# =================================================================================================
# 6. EXACT UNTRACKED UNIVERSE
# =================================================================================================

banner(
    "AUTHORIZED GIT CHANGE GATE"
)


expected_files = {
    str(
        (
            OUT
            / filename
        ).relative_to(
            REPO
        )
    )
    for filename in [
        *EXPECTED_HASHES.keys(),
        "checksums.sha256",
    ]
}


tracked = set(
    git(
        "diff",
        "--name-only",
    ).splitlines()
)

staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)

untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
)


if tracked:

    raise RuntimeError(
        "Unexpected tracked modifications:\n"
        + "\n".join(
            sorted(tracked)
        )
    )


if staged:

    raise RuntimeError(
        "Unexpected staged files before SHIFT."
    )


if untracked != expected_files:

    raise RuntimeError(
        "Unexpected untracked universe.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(expected_files)
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(untracked)
        )
    )


print(
    "[PASS] exactly eight authorized Stage28-2A4 files"
)


# =================================================================================================
# 7. STAGE
# =================================================================================================

banner(
    "STAGE STAGE28-2A4"
)


for relative in sorted(
    expected_files
):

    run(
        [
            "git",
            "add",
            "--",
            relative,
        ]
    )


staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)


if staged != expected_files:

    raise RuntimeError(
        "Staged universe mismatch."
    )


for path in sorted(staged):

    print(
        " ",
        path,
    )


print()
print(
    "[PASS] exactly eight files staged"
)


# =================================================================================================
# 8. GIT IDENTITY
# =================================================================================================

banner(
    "LOCAL GIT IDENTITY"
)


run(
    [
        "git",
        "config",
        "user.name",
        "Stage28 Kaggle",
    ]
)

run(
    [
        "git",
        "config",
        "user.email",
        "stage28-kaggle@users.noreply.github.com",
    ]
)


print(
    "user.name :",
    git(
        "config",
        "--get",
        "user.name",
    ),
)

print(
    "user.email:",
    git(
        "config",
        "--get",
        "user.email",
    ),
)


# =================================================================================================
# 9. COMMIT
# =================================================================================================

banner(
    "COMMIT STAGE28-2A4"
)


commit = run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ]
)


print(
    commit.stdout.strip()
)


new_head = git(
    "rev-parse",
    "HEAD",
)

new_parent = git(
    "rev-parse",
    "HEAD^",
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)


print()
print(
    "Parent :",
    new_parent,
)

print(
    "Commit :",
    new_head,
)

print(
    "Subject:",
    subject,
)


if new_parent != EXPECTED_PARENT:

    raise RuntimeError(
        "Stage28-2A4 commit parent mismatch."
    )


if subject != COMMIT_MESSAGE:

    raise RuntimeError(
        "Stage28-2A4 commit subject mismatch."
    )


print()
print(
    "[PASS] Stage28-2A4 committed"
)


# =================================================================================================
# 10. PUSH
# =================================================================================================

banner(
    "PUSH STAGE28-2A4"
)


github_token, token_source = (
    recover_github_token()
)


print(
    "[PASS] credential recovered from",
    token_source,
)

print(
    "[PASS] token intentionally not displayed"
)


push_output = authenticated_push(
    github_token
)


if push_output:

    print(
        push_output
    )


github_token = None


# =================================================================================================
# 11. REMOTE DURABILITY
# =================================================================================================

banner(
    "REMOTE DURABILITY VERIFICATION"
)


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_final = git(
    "rev-parse",
    "HEAD",
)

origin_final = git(
    "rev-parse",
    "origin/main",
)

remote_final_line = git(
    "ls-remote",
    "origin",
    "refs/heads/main",
)


if not remote_final_line:

    raise RuntimeError(
        "Unable to resolve remote main."
    )


remote_final = (
    remote_final_line.split()[0]
)

final_status = git(
    "status",
    "--porcelain",
)


print(
    "Local HEAD :",
    local_final,
)

print(
    "origin/main:",
    origin_final,
)

print(
    "Remote main:",
    remote_final,
)

print(
    "Git clean  :",
    not bool(final_status),
)


if not (
    local_final
    == origin_final
    == remote_final
    == new_head
):

    raise RuntimeError(
        "Stage28-2A4 remote durability failed."
    )


if final_status:

    raise RuntimeError(
        "Repository dirty after push:\n"
        + final_status
    )


# =================================================================================================
# 12. FINAL
# =================================================================================================

banner(
    "STAGE28-2A4-SHIFT — COMPLETE"
)


print(
    "Durable commit:"
)

print(
    " ",
    new_head,
)

print()

print(
    "Scientific result:"
)

print(
    "  C013 XGBoost seed43  = FIT #5 SUCCESS"
)

print(
    "  C014 LightGBM seed43 = FIT #6 SUCCESS"
)

print()

print(
    "Validation:"
)

print(
    "  ROC-AUC = 0.5079004765003459"
)

print(
    "  PR-AUC  = 0.10520468898118242"
)

print()

print(
    "Thresholds:"
)

print(
    "  STANDARD = 0.50"
)

print(
    "  BALANCED = 0.09"
)

print(
    "  SECURITY = 0.09"
)

print()

print(
    "Fit ledger:"
)

print(
    "  authorized = 108"
)

print(
    "  consumed   = 6"
)

print(
    "  remaining  = 102"
)

print()

print(
    "Shared final holdout:"
)

print(
    "  openings = 0"
)

print()

print(
    "NEXT:"
)

print(
    "  Stage28-2A5 — RANDOM_NATURAL seed44"
)

print(
    "  FIT #7 = C005 XGBoost"
)

print(
    "  FIT #8 = C006 LightGBM"
)

print()
print(SEP)


STAGE28-2A4-SHIFT — EXACT PARENT GATE

Expected parent: 1c21d67d374cfcba864f04d4b2da7fc3fc332568
Local HEAD     : 1c21d67d374cfcba864f04d4b2da7fc3fc332568
origin/main    : 1c21d67d374cfcba864f04d4b2da7fc3fc332568
Remote main    : 1c21d67d374cfcba864f04d4b2da7fc3fc332568

[PASS] exact Stage28-2A3 durable parent

STAGE28-2A4 SCIENTIFIC ARTIFACT GATE

[CHECK] chronological_natural_seed43_xgboost_cpu_model.json
 expected: ab14e0fc41fe94a7123d220f160b4ee4d6b469d46a9a07c03e028a7e710bea14
 actual  : ab14e0fc41fe94a7123d220f160b4ee4d6b469d46a9a07c03e028a7e710bea14
[CHECK] chronological_natural_seed43_lightgbm_cpu_model.txt
 expected: 00f2ea97707ec5b9e39027fe782cd7db99447d6c0c1635cc55123a9abfb54581
 actual  : 00f2ea97707ec5b9e39027fe782cd7db99447d6c0c1635cc55123a9abfb54581
[CHECK] chronological_natural_seed43_validation_ensemble_probabilities.npz
 expected: d70a5e281be1153d31e2310c2177e294c450e44ecc29e45371f3bae331709081
 actual  : d70a5e281be1153d31e2310c2177e294c450e44ecc29e45371f3bae3317090

In [7]:
# =================================================================================================
# STAGE28-AUTO-A — RESUMABLE STAGE22 SEED-STABILITY AUTOPILOT
#
# CURRENT DURABLE START:
#   f8520ab8f493ab2cffd8693884c7ed7d51f1421e
#
# ALREADY COMPLETE:
#   seed42 RANDOM + CHRONO
#   seed43 RANDOM + CHRONO
#
# CURRENT LEDGER:
#   authorized = 108
#   consumed   = 6
#   remaining  = 102
#
# THIS AUTOPILOT FINISHES:
#
#   Stage28-2A5  RANDOM seed44  -> C005 + C006 -> FIT #7/#8
#   Stage28-2A6  CHRONO seed44  -> C015 + C016 -> FIT #9/#10
#   Stage28-2A7  RANDOM seed45  -> C007 + C008 -> FIT #11/#12
#   Stage28-2A8  CHRONO seed45  -> C017 + C018 -> FIT #13/#14
#   Stage28-2A9  RANDOM seed46  -> C009 + C010 -> FIT #15/#16
#   Stage28-2A10 CHRONO seed46  -> C019 + C020 -> FIT #17/#18
#
# POST-AUTO-A:
#   Stage22 new fits consumed = 18 / 18
#   Stage28 total consumed    = 18 / 108
#   Stage28 remaining         = 90
#
# DURABILITY:
#   Each two-model evaluation cell is independently:
#       fitted
#       evaluated
#       thresholded
#       checksummed
#       committed
#       pushed
#       remote-verified
#
# RESUME:
#   Re-running this same cell skips already-durable completed jobs.
#
# FINAL HOLDOUT:
#   NEVER OPENED HERE.
#
# CPU ONLY — frozen Stage28 protocol.
# =================================================================================================

from __future__ import annotations

import base64
import csv
import gc
import hashlib
import json
import math
import os
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

import xgboost as xgb
import lightgbm as lgb


# =================================================================================================
# CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

AUTO_BASE = (
    "f8520ab8f493ab2cffd8693884c7ed7d51f1421e"
)

EXPECTED_XGB_VERSION = "3.2.0"
EXPECTED_LGBM_VERSION = "4.6.0"

EXPECTED_MANIFEST_SHA = (
    "47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505"
)

EXPECTED_ROWS = 14_412_403
EXPECTED_FEATURES = 70

RANDOM_TRAIN_ROWS = 11_529_922
RANDOM_TRAIN_ATTACK = 1_577_839
RANDOM_TRAIN_BENIGN = 9_952_083

RANDOM_VAL_ROWS = 2_882_481
RANDOM_VAL_ATTACK = 394_460
RANDOM_VAL_BENIGN = 2_488_021

CHRONO_TRAIN_ROWS = 13_818_623
CHRONO_TRAIN_ATTACK = 1_910_043
CHRONO_TRAIN_BENIGN = 11_908_580

CHRONO_VAL_ROWS = 593_780
CHRONO_VAL_ATTACK = 62_256
CHRONO_VAL_BENIGN = 531_524

EXPECTED_LABEL_SHA = (
    "571b2492929810425fee9c5d28f6b7a16df50f76a681a9df644626cb606f8f34"
)

EXPECTED_DAY_SHA = (
    "363c1d31038880b049e3e29dc573a1d3da136794d9a8196984b3f29dd041e3c2"
)

EXPECTED_RANDOM_TRAIN_SHA = (
    "66714bb29b84aa7de750a6a347f27d87db060a1fd514c41d705d1d644f160044"
)

EXPECTED_RANDOM_VAL_SHA = (
    "91475fa78164de5512313b16d746faf79e9f5ef4a4814aebd176d48b77ceec96"
)

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

MANIFEST_PATH = (
    STAGE28_ROOT
    / "stage28_1b_random_loao_membership_and_execution_lock"
    / "stage28_component_execution_manifest.csv"
)

PARAMETERS_PATH = (
    STAGE28_ROOT
    / "stage28_1b_random_loao_membership_and_execution_lock"
    / "execution_parameter_sets.json"
)

THRESHOLD_POLICY_PATH = (
    STAGE28_ROOT
    / "stage28_0_protocol_lock"
    / "threshold_policy.json"
)

PREV_LEDGER_PATH = (
    STAGE28_ROOT
    / "stage28_2a_stage22_seed_stability"
    / "stage28_2a4_chronological_natural_seed43"
    / "stage28_2a4_fit_ledger.json"
)

RUNTIME_RECEIPT_PATH = Path(
    "/kaggle/working/stage28_2a0_stage22_execution_matrix_receipt.json"
)

RUNTIME_ROOT = Path(
    "/kaggle/working/stage28_2a_runtime_cache/stage22_full"
)

X_PATH = (
    RUNTIME_ROOT
    / "development_features_float64.npy"
)

Y_PATH = (
    RUNTIME_ROOT
    / "development_binary_labels_uint8.npy"
)

DAY_PATH = (
    RUNTIME_ROOT
    / "development_day_ids_uint8.npy"
)

RANDOM_TRAIN_IDX_PATH = (
    RUNTIME_ROOT
    / "random_natural_train_global_idx_int32.npy"
)

RANDOM_VAL_IDX_PATH = (
    RUNTIME_ROOT
    / "random_natural_validation_global_idx_int32.npy"
)


# =================================================================================================
# FROZEN REMAINING STAGE22 JOBS
# =================================================================================================

JOBS = [
    {
        "stage": "Stage28-2A5",
        "slug": "stage28_2a5_random_natural_seed44",
        "unit": "RANDOM_NATURAL",
        "seed": 44,
        "xgb_component": "C005",
        "lgb_component": "C006",
        "fit_start": 7,
        "consumed_after": 8,
        "stage22_consumed_after": 8,
        "xgb_sha": "ae022425b27139a0ccdbb8e08ab9035ab386c4521afb3d9f095b1f7dc5cb7731",
        "lgb_sha": "0ed7e79e2b5d5f55b4cf8a3753c5df9ad083d412aa0ad08f72610f69b7692454",
        "commit": "stage28-2a5: execute random natural seed44 checkpoint",
    },
    {
        "stage": "Stage28-2A6",
        "slug": "stage28_2a6_chronological_natural_seed44",
        "unit": "CHRONOLOGICAL_NATURAL",
        "seed": 44,
        "xgb_component": "C015",
        "lgb_component": "C016",
        "fit_start": 9,
        "consumed_after": 10,
        "stage22_consumed_after": 10,
        "xgb_sha": "ae022425b27139a0ccdbb8e08ab9035ab386c4521afb3d9f095b1f7dc5cb7731",
        "lgb_sha": "0ed7e79e2b5d5f55b4cf8a3753c5df9ad083d412aa0ad08f72610f69b7692454",
        "commit": "stage28-2a6: execute chronological natural seed44 checkpoint",
    },
    {
        "stage": "Stage28-2A7",
        "slug": "stage28_2a7_random_natural_seed45",
        "unit": "RANDOM_NATURAL",
        "seed": 45,
        "xgb_component": "C007",
        "lgb_component": "C008",
        "fit_start": 11,
        "consumed_after": 12,
        "stage22_consumed_after": 12,
        "xgb_sha": "ca62fc9245eaad469fdf744635d56dc10951d77cf808c1616122be817be35565",
        "lgb_sha": "6daf64f6e8034953ca426e6241177a45ee5cc6bdf5434bfa5a8f3789e34f431c",
        "commit": "stage28-2a7: execute random natural seed45 checkpoint",
    },
    {
        "stage": "Stage28-2A8",
        "slug": "stage28_2a8_chronological_natural_seed45",
        "unit": "CHRONOLOGICAL_NATURAL",
        "seed": 45,
        "xgb_component": "C017",
        "lgb_component": "C018",
        "fit_start": 13,
        "consumed_after": 14,
        "stage22_consumed_after": 14,
        "xgb_sha": "ca62fc9245eaad469fdf744635d56dc10951d77cf808c1616122be817be35565",
        "lgb_sha": "6daf64f6e8034953ca426e6241177a45ee5cc6bdf5434bfa5a8f3789e34f431c",
        "commit": "stage28-2a8: execute chronological natural seed45 checkpoint",
    },
    {
        "stage": "Stage28-2A9",
        "slug": "stage28_2a9_random_natural_seed46",
        "unit": "RANDOM_NATURAL",
        "seed": 46,
        "xgb_component": "C009",
        "lgb_component": "C010",
        "fit_start": 15,
        "consumed_after": 16,
        "stage22_consumed_after": 16,
        "xgb_sha": "f0f79c993ba61a0d11f6abb3fe34419203768a8f65df0dbcee72f3fa9d93f676",
        "lgb_sha": "7847b3ccffa8bdc77842ddf8ae34dddfec4d7b468b41824b2603d47828190d9c",
        "commit": "stage28-2a9: execute random natural seed46 checkpoint",
    },
    {
        "stage": "Stage28-2A10",
        "slug": "stage28_2a10_chronological_natural_seed46",
        "unit": "CHRONOLOGICAL_NATURAL",
        "seed": 46,
        "xgb_component": "C019",
        "lgb_component": "C020",
        "fit_start": 17,
        "consumed_after": 18,
        "stage22_consumed_after": 18,
        "xgb_sha": "f0f79c993ba61a0d11f6abb3fe34419203768a8f65df0dbcee72f3fa9d93f676",
        "lgb_sha": "7847b3ccffa8bdc77842ddf8ae34dddfec4d7b468b41824b2603d47828190d9c",
        "commit": "stage28-2a10: execute chronological natural seed46 checkpoint",
    },
]


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, *, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args, check=True):
    return (
        run(
            ["git", *args],
            check=check,
        ).stdout
        or ""
    ).strip()


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def write_json(path, obj):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(
                16 * 1024 * 1024
            )
            if not block:
                break
            h.update(block)

    return h.hexdigest()


def canonical_sha(obj):
    return hashlib.sha256(
        json.dumps(
            obj,
            sort_keys=True,
            separators=(",", ":"),
            ensure_ascii=False,
            allow_nan=False,
        ).encode("utf-8")
    ).hexdigest()


def safe_div(a, b):
    return (
        float(a) / float(b)
        if b
        else 0.0
    )


def metrics_from_counts(tp, fp, tn, fn):
    return {
        "accuracy": safe_div(
            tp + tn,
            tp + fp + tn + fn,
        ),
        "precision": safe_div(
            tp,
            tp + fp,
        ),
        "recall": safe_div(
            tp,
            tp + fn,
        ),
        "fpr": safe_div(
            fp,
            fp + tn,
        ),
        "f1": safe_div(
            2 * tp,
            2 * tp + fp + fn,
        ),
        "f2": safe_div(
            5 * tp,
            5 * tp + 4 * fn + fp,
        ),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }


def threshold_row(
    y_true,
    probability,
    pct,
):
    threshold = pct / 100.0
    threshold32 = np.float32(
        threshold
    )

    pred = (
        probability
        >= threshold32
    )

    pos = (
        y_true == 1
    )

    neg = ~pos

    tp = int(
        np.count_nonzero(
            pred & pos
        )
    )

    fp = int(
        np.count_nonzero(
            pred & neg
        )
    )

    fn = int(
        np.count_nonzero(
            (~pred) & pos
        )
    )

    tn = int(
        np.count_nonzero(
            (~pred) & neg
        )
    )

    row = metrics_from_counts(
        tp,
        fp,
        tn,
        fn,
    )

    row.update(
        {
            "threshold_integer_percent":
                int(pct),

            "threshold":
                float(threshold),

            "threshold_float32_runtime":
                float(threshold32),
        }
    )

    return row


def threshold_grid(
    y_true,
    probability,
):
    return [
        threshold_row(
            y_true,
            probability,
            pct,
        )
        for pct in range(
            5,
            96,
        )
    ]


def choose_balanced(grid):
    return max(
        grid,
        key=lambda r: (
            r["f1"],
            -r["fpr"],
            r["recall"],
            -abs(
                r["threshold"] - 0.50
            ),
            -r["threshold"],
        ),
    )


def choose_security(grid):
    feasible = [
        r
        for r in grid
        if r["fpr"] <= 0.05
    ]

    if not feasible:
        return None

    return max(
        feasible,
        key=lambda r: (
            r["f2"],
            -r["fpr"],
            r["recall"],
            -r["threshold"],
        ),
    )


def recover_token():
    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]

    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()

        for label in labels:
            try:
                value = client.get_secret(
                    label
                )
            except Exception:
                value = None

            if (
                isinstance(value, str)
                and value.strip()
            ):
                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass

    for label in labels:
        value = os.environ.get(
            label
        )

        if (
            isinstance(value, str)
            and value.strip()
        ):
            return (
                value.strip(),
                f"environment:{label}",
            )

    raise RuntimeError(
        "GitHub credential unavailable."
    )


def authenticated_push(token):
    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode("utf-8")
    ).decode("ascii")

    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            "Git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return (
        p.stdout
        + p.stderr
    ).strip()


def remote_head():
    line = git(
        "ls-remote",
        "origin",
        "refs/heads/main",
    )

    if not line:
        raise RuntimeError(
            "Unable to resolve remote main."
        )

    return line.split()[0]


def commit_job(
    out,
    expected_files,
    message,
    token,
):
    relative_files = {
        str(
            (
                out
                / name
            ).relative_to(
                REPO
            )
        )
        for name in expected_files
    }

    tracked_changes = set(
        git(
            "diff",
            "--name-only",
        ).splitlines()
    )

    staged = set(
        git(
            "diff",
            "--cached",
            "--name-only",
        ).splitlines()
    )

    untracked = set(
        git(
            "ls-files",
            "--others",
            "--exclude-standard",
        ).splitlines()
    )

    if tracked_changes:
        raise RuntimeError(
            "Unexpected tracked modifications:\n"
            + "\n".join(
                sorted(
                    tracked_changes
                )
            )
        )

    if staged:
        raise RuntimeError(
            "Unexpected pre-existing staged files."
        )

    if untracked != relative_files:
        raise RuntimeError(
            "Unexpected untracked artifact universe.\n\n"
            "Expected:\n"
            + "\n".join(
                sorted(
                    relative_files
                )
            )
            + "\n\nActual:\n"
            + "\n".join(
                sorted(
                    untracked
                )
            )
        )

    for relative in sorted(
        relative_files
    ):
        run(
            [
                "git",
                "add",
                "--",
                relative,
            ]
        )

    run(
        [
            "git",
            "commit",
            "-m",
            message,
        ]
    )

    push_output = authenticated_push(
        token
    )

    if push_output:
        print(
            push_output
        )

    run(
        [
            "git",
            "fetch",
            "origin",
            "main",
        ]
    )

    local = git(
        "rev-parse",
        "HEAD",
    )

    origin = git(
        "rev-parse",
        "origin/main",
    )

    remote = remote_head()

    if not (
        local
        == origin
        == remote
    ):
        raise RuntimeError(
            "Remote durability verification failed."
        )

    if git(
        "status",
        "--porcelain",
    ):
        raise RuntimeError(
            "Repository dirty after push."
        )

    return local


# =================================================================================================
# STARTUP GATES
# =================================================================================================

banner(
    "STAGE28-AUTO-A — STARTUP"
)

if not REPO.is_dir():
    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )


token, token_source = recover_token()

print(
    "[PASS] GitHub credential:",
    token_source,
)

print(
    "[PASS] token not displayed"
)


run(
    [
        "git",
        "config",
        "user.name",
        "Stage28 Kaggle",
    ]
)

run(
    [
        "git",
        "config",
        "user.email",
        "stage28-kaggle@users.noreply.github.com",
    ]
)


local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)

remote = remote_head()


if not (
    local_head
    == origin_head
    == remote
):
    raise RuntimeError(
        "Local/origin/remote are not synchronized."
    )


ancestor_check = subprocess.run(
    [
        "git",
        "merge-base",
        "--is-ancestor",
        AUTO_BASE,
        local_head,
    ],
    cwd=str(REPO),
).returncode


if ancestor_check != 0:
    raise RuntimeError(
        "Current HEAD is not descended from "
        "the Stage28-2A4 durable base."
    )


if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Repository must be clean before AUTO-A."
    )


print(
    "Current durable HEAD:",
    local_head,
)

print(
    "[PASS] exact synchronized clean repository"
)


# =================================================================================================
# MODEL RUNTIME GATE
# =================================================================================================

banner(
    "MODEL / CPU GATE"
)

print(
    "XGBoost :",
    xgb.__version__,
)

print(
    "LightGBM:",
    lgb.__version__,
)


if xgb.__version__ != EXPECTED_XGB_VERSION:
    raise RuntimeError(
        "XGBoost version drift."
    )

if lgb.__version__ != EXPECTED_LGBM_VERSION:
    raise RuntimeError(
        "LightGBM version drift."
    )


os.environ[
    "CUDA_VISIBLE_DEVICES"
] = ""


print(
    "[PASS] Stage28 CPU-only policy"
)


# =================================================================================================
# FROZEN MANIFEST / PARAMETER / THRESHOLD GATES
# =================================================================================================

banner(
    "FROZEN PROTOCOL GATES"
)


if sha256_file(
    MANIFEST_PATH
) != EXPECTED_MANIFEST_SHA:
    raise RuntimeError(
        "Execution-manifest SHA mismatch."
    )


with MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:
    manifest_rows = list(
        csv.DictReader(f)
    )


manifest = {
    row[
        "component_id"
    ]:
    row
    for row in manifest_rows
}


parameter_doc = read_json(
    PARAMETERS_PATH
)

parameter_sets = (
    parameter_doc[
        "sets"
    ]
)


policy = read_json(
    THRESHOLD_POLICY_PATH
)[
    "stage22_full"
]


if (
    policy[
        "selection_population"
    ]
    != "EACH_PARENT_CELL_FROZEN_DEVELOPMENT_VALIDATION"
):
    raise RuntimeError(
        "Stage22 threshold population drift."
    )


if (
    policy[
        "final_holdout_threshold_search"
    ]
    != "FORBIDDEN"
):
    raise RuntimeError(
        "Final-holdout threshold policy drift."
    )


print(
    "[PASS] execution manifest exact"
)

print(
    "[PASS] parameter registry loaded"
)

print(
    "[PASS] threshold policy exact"
)


# =================================================================================================
# RUNTIME MATRIX GATE
# =================================================================================================

banner(
    "STAGE22 RUNTIME MATRIX GATE"
)


for path in [
    RUNTIME_RECEIPT_PATH,
    X_PATH,
    Y_PATH,
    DAY_PATH,
    RANDOM_TRAIN_IDX_PATH,
    RANDOM_VAL_IDX_PATH,
]:
    if not Path(path).is_file():
        raise RuntimeError(
            "Stage22 runtime asset missing:\n"
            f"{path}\n\n"
            "Run Stage28-2A4-R0 runtime recovery first."
        )


runtime_receipt = read_json(
    RUNTIME_RECEIPT_PATH
)


matrix_meta = (
    runtime_receipt[
        "runtime_matrix"
    ]
)


if (
    int(
        matrix_meta[
            "rows"
        ]
    )
    != EXPECTED_ROWS
):
    raise RuntimeError(
        "Runtime row count mismatch."
    )


if (
    matrix_meta[
        "binary_labels_content_sha256"
    ]
    != EXPECTED_LABEL_SHA
):
    raise RuntimeError(
        "Runtime label identity mismatch."
    )


if (
    matrix_meta[
        "day_ids_content_sha256"
    ]
    != EXPECTED_DAY_SHA
):
    raise RuntimeError(
        "Runtime day identity mismatch."
    )


random_meta = (
    runtime_receipt[
        "random_natural"
    ]
)


if (
    random_meta[
        "train_index_content_sha256"
    ]
    != EXPECTED_RANDOM_TRAIN_SHA
):
    raise RuntimeError(
        "Random training membership identity mismatch."
    )


if (
    random_meta[
        "validation_index_content_sha256"
    ]
    != EXPECTED_RANDOM_VAL_SHA
):
    raise RuntimeError(
        "Random validation membership identity mismatch."
    )


X = np.load(
    X_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

y = np.load(
    Y_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

day_ids = np.load(
    DAY_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

random_train_idx = np.load(
    RANDOM_TRAIN_IDX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)

random_val_idx = np.load(
    RANDOM_VAL_IDX_PATH,
    mmap_mode="r",
    allow_pickle=False,
)


if X.shape != (
    EXPECTED_ROWS,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        "X shape mismatch."
    )


print(
    "[PASS] exact Stage22 execution matrix loaded"
)


# =================================================================================================
# VALIDATE / SKIP ALREADY-DURABLE AUTO-A JOBS
# =================================================================================================

banner(
    "RESUME DISCOVERY"
)


current_consumed = 6
first_pending_seen = False


for job in JOBS:
    out = (
        STAGE28_ROOT
        / "stage28_2a_stage22_seed_stability"
        / job[
            "slug"
        ]
    )

    result_path = (
        out
        / (
            job[
                "slug"
            ]
            + "_result.json"
        )
    )

    ledger_path = (
        out
        / (
            job[
                "slug"
            ]
            + "_fit_ledger.json"
        )
    )

    if out.exists():

        if first_pending_seen:
            raise RuntimeError(
                "Later AUTO-A job exists after an earlier missing job."
            )

        if not (
            result_path.is_file()
            and ledger_path.is_file()
        ):
            raise RuntimeError(
                f"Partial durable-looking job directory:\n{out}"
            )

        ledger = read_json(
            ledger_path
        )

        if int(
            ledger[
                "cumulative_new_fits_consumed"
            ]
        ) != job[
            "consumed_after"
        ]:
            raise RuntimeError(
                f"{job['stage']} durable ledger mismatch."
            )

        current_consumed = (
            job[
                "consumed_after"
            ]
        )

        print(
            "[SKIP durable]",
            job[
                "stage"
            ],
            job[
                "unit"
            ],
            "seed",
            job[
                "seed"
            ],
            "-> cumulative",
            current_consumed,
        )

    else:
        first_pending_seen = True


print()
print(
    "Resume ledger:",
    current_consumed,
    "consumed"
)


# =================================================================================================
# EXECUTE PENDING JOBS
# =================================================================================================

bot_start = time.perf_counter()


for job_index, job in enumerate(
    JOBS,
    start=1,
):

    OUT = (
        STAGE28_ROOT
        / "stage28_2a_stage22_seed_stability"
        / job[
            "slug"
        ]
    )


    RESULT_PATH = (
        OUT
        / (
            job[
                "slug"
            ]
            + "_result.json"
        )
    )

    LEDGER_PATH = (
        OUT
        / (
            job[
                "slug"
            ]
            + "_fit_ledger.json"
        )
    )


    if OUT.exists():
        continue


    banner(
        f"AUTO-A JOB — {job['stage']} — "
        f"{job['unit']} — SEED {job['seed']}"
    )


    expected_before = (
        job[
            "consumed_after"
        ]
        - 2
    )


    if current_consumed != expected_before:
        raise RuntimeError(
            f"Ledger discontinuity before {job['stage']}.\n"
            f"expected={expected_before}\n"
            f"actual={current_consumed}"
        )


    # ---------------------------------------------------------------------------------------------
    # Manifest rows.
    # ---------------------------------------------------------------------------------------------

    xrow = manifest[
        job[
            "xgb_component"
        ]
    ]

    lrow = manifest[
        job[
            "lgb_component"
        ]
    ]


    expected_eval = (
        f"28A_STAGE22::{job['unit']}::SEED{job['seed']}"
    )


    for row, learner in [
        (
            xrow,
            "XGBOOST",
        ),
        (
            lrow,
            "LIGHTGBM",
        ),
    ]:

        if row[
            "evaluation_cell_id"
        ] != expected_eval:
            raise RuntimeError(
                "Evaluation-cell mismatch."
            )

        if row[
            "learner"
        ] != learner:
            raise RuntimeError(
                "Learner mismatch."
            )

        if row[
            "model_seed"
        ] != str(
            job[
                "seed"
            ]
        ):
            raise RuntimeError(
                "Seed mismatch."
            )

        if row[
            "fit_action"
        ] != "NEW_FIT_AUTHORIZED":
            raise RuntimeError(
                "Expected NEW_FIT_AUTHORIZED."
            )

        if row[
            "compute_backend"
        ] != "CPU":
            raise RuntimeError(
                "Expected CPU backend."
            )


    # ---------------------------------------------------------------------------------------------
    # Parameters.
    # ---------------------------------------------------------------------------------------------

    xgb_params = dict(
        parameter_sets[
            xrow[
                "parameter_set_id"
            ]
        ][
            "parameters"
        ]
    )

    lgb_params = dict(
        parameter_sets[
            lrow[
                "parameter_set_id"
            ]
        ][
            "parameters"
        ]
    )


    if canonical_sha(
        xgb_params
    ) != job[
        "xgb_sha"
    ]:
        raise RuntimeError(
            "XGBoost parameter SHA mismatch."
        )


    if canonical_sha(
        lgb_params
    ) != job[
        "lgb_sha"
    ]:
        raise RuntimeError(
            "LightGBM parameter SHA mismatch."
        )


    if int(
        xgb_params[
            "random_state"
        ]
    ) != job[
        "seed"
    ]:
        raise RuntimeError(
            "XGBoost random_state mismatch."
        )


    if int(
        lgb_params[
            "random_state"
        ]
    ) != job[
        "seed"
    ]:
        raise RuntimeError(
            "LightGBM random_state mismatch."
        )


    # ---------------------------------------------------------------------------------------------
    # Membership.
    # ---------------------------------------------------------------------------------------------

    if job[
        "unit"
    ] == "RANDOM_NATURAL":

        train_idx = random_train_idx
        val_idx = random_val_idx

        expected_train_rows = (
            RANDOM_TRAIN_ROWS
        )

        expected_train_attack = (
            RANDOM_TRAIN_ATTACK
        )

        expected_val_rows = (
            RANDOM_VAL_ROWS
        )

        expected_val_attack = (
            RANDOM_VAL_ATTACK
        )

        y_validation = np.asarray(
            y[
                val_idx
            ],
            dtype=np.uint8,
        )


        print(
            "Materializing RANDOM_NATURAL training matrix..."
        )


        X_train = np.asarray(
            X[
                train_idx,
                :
            ],
            dtype=np.float64,
            order="C",
        )

        y_train = np.asarray(
            y[
                train_idx
            ],
            dtype=np.uint8,
            order="C",
        )


    else:

        expected_train_rows = (
            CHRONO_TRAIN_ROWS
        )

        expected_train_attack = (
            CHRONO_TRAIN_ATTACK
        )

        expected_val_rows = (
            CHRONO_VAL_ROWS
        )

        expected_val_attack = (
            CHRONO_VAL_ATTACK
        )


        if not np.all(
            day_ids[
                :CHRONO_TRAIN_ROWS
            ] <= 6
        ):
            raise RuntimeError(
                "Chronological training geometry changed."
            )


        if not np.all(
            day_ids[
                CHRONO_TRAIN_ROWS:
            ] == 7
        ):
            raise RuntimeError(
                "Chronological validation geometry changed."
            )


        y_validation = np.asarray(
            y[
                CHRONO_TRAIN_ROWS:
            ],
            dtype=np.uint8,
        )


        X_train = np.asarray(
            X[
                :CHRONO_TRAIN_ROWS,
                :
            ],
            dtype=np.float64,
            order="C",
        )

        y_train = np.asarray(
            y[
                :CHRONO_TRAIN_ROWS
            ],
            dtype=np.uint8,
            order="C",
        )


    if (
        X_train.shape[0]
        != expected_train_rows
    ):
        raise RuntimeError(
            "Training row count mismatch."
        )


    if int(
        y_train.sum()
    ) != expected_train_attack:
        raise RuntimeError(
            "Training class count mismatch."
        )


    if len(
        y_validation
    ) != expected_val_rows:
        raise RuntimeError(
            "Validation row count mismatch."
        )


    if int(
        y_validation.sum()
    ) != expected_val_attack:
        raise RuntimeError(
            "Validation class count mismatch."
        )


    OUT.mkdir(
        parents=True,
        exist_ok=False,
    )


    XGB_MODEL_PATH = (
        OUT
        / (
            job[
                "unit"
            ].lower()
            + f"_seed{job['seed']}_xgboost_cpu_model.json"
        )
    )

    LGB_MODEL_PATH = (
        OUT
        / (
            job[
                "unit"
            ].lower()
            + f"_seed{job['seed']}_lightgbm_cpu_model.txt"
        )
    )

    PROB_PATH = (
        OUT
        / (
            job[
                "unit"
            ].lower()
            + f"_seed{job['seed']}_validation_ensemble_probabilities.npz"
        )
    )

    GRID_PATH = (
        OUT
        / (
            job[
                "unit"
            ].lower()
            + f"_seed{job['seed']}_validation_threshold_grid.csv"
        )
    )

    CHECKSUMS_PATH = (
        OUT
        / "checksums.sha256"
    )


    # =============================================================================================
    # FIT XGBOOST
    # =============================================================================================

    banner(
        f"FIT #{job['fit_start']} — "
        f"{job['xgb_component']} XGBOOST SEED{job['seed']} CPU"
    )

    print(
        "Starting fit..."
    )

    fit_xgb_start = (
        time.perf_counter()
    )


    xgb_model = xgb.XGBClassifier(
        **xgb_params
    )


    xgb_model.fit(
        X_train,
        y_train,
    )


    fit_xgb_seconds = (
        time.perf_counter()
        - fit_xgb_start
    )


    if int(
        xgb_model
        .get_booster()
        .num_boosted_rounds()
    ) != 400:
        raise RuntimeError(
            "XGBoost round count != 400."
        )


    xgb_model.save_model(
        XGB_MODEL_PATH
    )


    xgb_model_sha = sha256_file(
        XGB_MODEL_PATH
    )


    print(
        "[PASS] FIT",
        job[
            "fit_start"
        ],
        "SUCCESS"
    )

    print(
        "seconds:",
        fit_xgb_seconds,
    )

    print(
        "SHA:",
        xgb_model_sha,
    )


    # =============================================================================================
    # FIT LIGHTGBM
    # =============================================================================================

    banner(
        f"FIT #{job['fit_start'] + 1} — "
        f"{job['lgb_component']} LIGHTGBM SEED{job['seed']} CPU"
    )


    print(
        "Starting fit..."
    )


    fit_lgb_start = (
        time.perf_counter()
    )


    lgb_model = lgb.LGBMClassifier(
        **lgb_params
    )


    lgb_model.fit(
        X_train,
        y_train,
    )


    fit_lgb_seconds = (
        time.perf_counter()
        - fit_lgb_start
    )


    booster = (
        lgb_model.booster_
    )


    if int(
        booster.current_iteration()
    ) != 400:
        raise RuntimeError(
            "LightGBM iteration count != 400."
        )


    booster.save_model(
        str(
            LGB_MODEL_PATH
        )
    )


    lgb_model_sha = sha256_file(
        LGB_MODEL_PATH
    )


    print(
        "[PASS] FIT",
        job[
            "fit_start"
        ] + 1,
        "SUCCESS"
    )

    print(
        "seconds:",
        fit_lgb_seconds,
    )

    print(
        "SHA:",
        lgb_model_sha,
    )


    del X_train
    del y_train
    gc.collect()


    # =============================================================================================
    # VALIDATION INFERENCE
    # =============================================================================================

    banner(
        f"{job['unit']} SEED{job['seed']} — VALIDATION"
    )


    ensemble = np.empty(
        expected_val_rows,
        dtype=np.float32,
    )


    chunk_size = 65_536

    inference_start = (
        time.perf_counter()
    )


    for start in range(
        0,
        expected_val_rows,
        chunk_size,
    ):

        stop = min(
            start + chunk_size,
            expected_val_rows,
        )


        if job[
            "unit"
        ] == "RANDOM_NATURAL":

            idx_chunk = np.asarray(
                random_val_idx[
                    start:stop
                ],
                dtype=np.int64,
            )

            X_chunk = np.asarray(
                X[
                    idx_chunk,
                    :
                ],
                dtype=np.float64,
                order="C",
            )

        else:

            global_start = (
                CHRONO_TRAIN_ROWS
                + start
            )

            global_stop = (
                CHRONO_TRAIN_ROWS
                + stop
            )

            X_chunk = np.asarray(
                X[
                    global_start:global_stop,
                    :
                ],
                dtype=np.float64,
                order="C",
            )


        p_xgb = np.asarray(
            xgb_model.predict_proba(
                X_chunk
            )[
                :,
                1
            ],
            dtype=np.float64,
        )


        p_lgb = np.asarray(
            lgb_model.predict_proba(
                X_chunk
            )[
                :,
                1
            ],
            dtype=np.float64,
        )


        ensemble[
            start:stop
        ] = (
            (
                0.5 * p_xgb
                + 0.5 * p_lgb
            )
            .astype(
                np.float32,
                copy=False,
            )
        )


        del X_chunk
        del p_xgb
        del p_lgb

        if job[
            "unit"
        ] == "RANDOM_NATURAL":
            del idx_chunk

        gc.collect()


        if (
            start == 0
            or stop == expected_val_rows
            or (
                stop // chunk_size
            ) % 10 == 0
        ):
            print(
                f"  inferred {stop:,} / "
                f"{expected_val_rows:,}"
            )


    inference_seconds = (
        time.perf_counter()
        - inference_start
    )


    if not np.all(
        np.isfinite(
            ensemble
        )
    ):
        raise RuntimeError(
            "Non-finite ensemble probability."
        )


    roc = float(
        roc_auc_score(
            y_validation,
            ensemble,
        )
    )


    pr = float(
        average_precision_score(
            y_validation,
            ensemble,
        )
    )


    print()
    print(
        "ROC-AUC:",
        roc,
    )

    print(
        "PR-AUC :",
        pr,
    )


    # =============================================================================================
    # THRESHOLDS
    # =============================================================================================

    grid = threshold_grid(
        y_validation,
        ensemble,
    )


    standard = next(
        r
        for r in grid
        if r[
            "threshold_integer_percent"
        ] == 50
    )


    balanced = choose_balanced(
        grid
    )

    security = choose_security(
        grid
    )


    if security is None:
        raise RuntimeError(
            "Security threshold infeasible."
        )


    print()
    print(
        "STANDARD:",
        standard[
            "threshold"
        ],
    )

    print(
        "BALANCED:",
        balanced[
            "threshold"
        ],
    )

    print(
        "SECURITY:",
        security[
            "threshold"
        ],
    )


    # =============================================================================================
    # SAVE VALIDATION
    # =============================================================================================

    if job[
        "unit"
    ] == "RANDOM_NATURAL":

        validation_global_idx = np.asarray(
            random_val_idx,
            dtype=np.int32,
        )

    else:

        validation_global_idx = np.arange(
            CHRONO_TRAIN_ROWS,
            EXPECTED_ROWS,
            dtype=np.int32,
        )


    np.savez_compressed(
        PROB_PATH,

        ensemble_probability_float32=
            ensemble,

        validation_global_idx_int32=
            validation_global_idx,

        binary_label_uint8=
            y_validation,
    )


    grid_df = pd.DataFrame(
        grid
    )[
        [
            "threshold_integer_percent",
            "threshold",
            "threshold_float32_runtime",
            "accuracy",
            "precision",
            "recall",
            "fpr",
            "f1",
            "f2",
            "tp",
            "fp",
            "tn",
            "fn",
        ]
    ]


    grid_df.to_csv(
        GRID_PATH,
        index=False,
        lineterminator="\n",
    )


    # =============================================================================================
    # LEDGER
    # =============================================================================================

    ledger = {
        "stage":
            job[
                "stage"
            ],

        "execution_mode":
            "STAGE28_AUTO_A",

        "evaluation_cell_id":
            expected_eval,

        "authorized_stage28_new_fit_budget":
            108,

        "pre_cell_new_fits_consumed":
            current_consumed,

        "this_cell": {
            "successful_new_fits":
                2,

            "new_fit_components": [
                job[
                    "xgb_component"
                ],
                job[
                    "lgb_component"
                ],
            ],

            "reused_components":
                [],
        },

        "cumulative_new_fits_consumed":
            job[
                "consumed_after"
            ],

        "new_fits_remaining":
            (
                108
                - job[
                    "consumed_after"
                ]
            ),

        "stage22_new_fits_consumed":
            job[
                "stage22_consumed_after"
            ],

        "stage22_new_fits_remaining":
            (
                18
                - job[
                    "stage22_consumed_after"
                ]
            ),

        "status":
            (
                f"FITS_{job['fit_start']:03d}_AND_"
                f"{job['fit_start'] + 1:03d}_"
                "SUCCESSFULLY_CONSUMED"
            ),
    }


    write_json(
        LEDGER_PATH,
        ledger,
    )


    # =============================================================================================
    # RESULT
    # =============================================================================================

    result = {
        "stage":
            job[
                "stage"
            ],

        "execution_mode":
            "STAGE28_AUTO_A",

        "status":
            (
                f"{job['unit']}_SEED{job['seed']}_"
                "VALIDATION_CHECKPOINT_FROZEN"
            ),

        "created_at_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "parent_commit":
            git(
                "rev-parse",
                "HEAD",
            ),

        "arm":
            "28A_TRAINING_SEED_STABILITY",

        "experiment":
            "STAGE22_FULL",

        "unit":
            job[
                "unit"
            ],

        "training_seed":
            job[
                "seed"
            ],

        "membership_seed":
            42,

        "evaluation_cell_id":
            expected_eval,

        "models": {
            "strategy":
                "ENS_LGBM_XGB_EQUAL",

            "component_combination_dtype":
                "float64",

            "ensemble_storage_dtype":
                "float32",

            "xgboost": {
                "component_id":
                    job[
                        "xgb_component"
                    ],

                "seed":
                    job[
                        "seed"
                    ],

                "backend":
                    "cpu",

                "parameter_sha256":
                    job[
                        "xgb_sha"
                    ],

                "model":
                    XGB_MODEL_PATH.name,

                "model_sha256":
                    xgb_model_sha,

                "fit_seconds":
                    float(
                        fit_xgb_seconds
                    ),
            },

            "lightgbm": {
                "component_id":
                    job[
                        "lgb_component"
                    ],

                "seed":
                    job[
                        "seed"
                    ],

                "backend":
                    "cpu",

                "parameter_sha256":
                    job[
                        "lgb_sha"
                    ],

                "model":
                    LGB_MODEL_PATH.name,

                "model_sha256":
                    lgb_model_sha,

                "fit_seconds":
                    float(
                        fit_lgb_seconds
                    ),
            },
        },

        "validation_probability": {
            "rows":
                expected_val_rows,

            "roc_auc":
                roc,

            "pr_auc":
                pr,

            "pr_auc_definition":
                "SKLEARN_AVERAGE_PRECISION_SCORE",

            "artifact":
                PROB_PATH.name,

            "artifact_sha256":
                sha256_file(
                    PROB_PATH
                ),

            "inference_seconds":
                float(
                    inference_seconds
                ),
        },

        "operating_points": {
            "standard":
                standard,

            "balanced":
                balanced,

            "security": {
                "status":
                    "AVAILABLE",

                "result":
                    security,
            },
        },

        "scientific_accounting": {
            "new_model_fits_before_cell":
                current_consumed,

            "new_model_fits_this_cell":
                2,

            "new_model_fits_cumulative":
                job[
                    "consumed_after"
                ],

            "new_model_fits_remaining":
                (
                    108
                    - job[
                        "consumed_after"
                    ]
                ),

            "shared_final_holdout_openings":
                0,

            "shared_final_holdout_predictor_rows_read":
                0,

            "shared_final_holdout_labels_read":
                0,

            "target_adaptive_choices":
                0,
        },
    }


    write_json(
        RESULT_PATH,
        result,
    )


    # =============================================================================================
    # CHECKSUMS
    # =============================================================================================

    payloads = [
        XGB_MODEL_PATH,
        LGB_MODEL_PATH,
        PROB_PATH,
        GRID_PATH,
        LEDGER_PATH,
        RESULT_PATH,
    ]


    CHECKSUMS_PATH.write_text(
        "\n".join(
            (
                f"{sha256_file(path)}  "
                f"{path.name}"
            )
            for path in payloads
        )
        + "\n",
        encoding="utf-8",
    )


    expected_files = {
        path.name
        for path in [
            *payloads,
            CHECKSUMS_PATH,
        ]
    }


    actual_files = {
        p.name
        for p in OUT.iterdir()
        if p.is_file()
    }


    if actual_files != expected_files:
        raise RuntimeError(
            "Unexpected output artifact universe."
        )


    for line in (
        CHECKSUMS_PATH
        .read_text(
            encoding="utf-8"
        )
        .splitlines()
    ):

        digest, filename = line.split(
            None,
            1,
        )

        filename = (
            filename.strip()
            .lstrip("*")
        )

        if sha256_file(
            OUT
            / filename
        ) != digest:
            raise RuntimeError(
                f"Checksum failed: {filename}"
            )


    print()
    print(
        "[PASS] scientific artifacts frozen"
    )


    # =============================================================================================
    # COMMIT + PUSH
    # =============================================================================================

    banner(
        f"{job['stage']} — AUTO COMMIT/PUSH"
    )


    new_head = commit_job(
        OUT,
        expected_files,
        job[
            "commit"
        ],
        token,
    )


    current_consumed = (
        job[
            "consumed_after"
        ]
    )


    print()
    print(
        "[DURABLE]",
        job[
            "stage"
        ],
    )

    print(
        " commit:",
        new_head,
    )

    print(
        " consumed:",
        current_consumed,
    )

    print(
        " remaining:",
        108
        - current_consumed,
    )

    print(
        " final-holdout openings: 0"
    )


    # Clean memory before next evaluation cell.
    del xgb_model
    del lgb_model
    del booster
    del ensemble
    del y_validation
    del validation_global_idx
    del grid
    del grid_df

    gc.collect()


# =================================================================================================
# FINAL AUTO-A VERIFICATION
# =================================================================================================

banner(
    "STAGE28-AUTO-A — FINAL VERIFICATION"
)


local_final = git(
    "rev-parse",
    "HEAD",
)

origin_final = git(
    "rev-parse",
    "origin/main",
)

remote_final = remote_head()


if not (
    local_final
    == origin_final
    == remote_final
):
    raise RuntimeError(
        "Final repository durability mismatch."
    )


if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Repository dirty after AUTO-A."
    )


# Verify every remaining Stage22 job is durable.
for job in JOBS:

    out = (
        STAGE28_ROOT
        / "stage28_2a_stage22_seed_stability"
        / job[
            "slug"
        ]
    )

    ledger_path = (
        out
        / (
            job[
                "slug"
            ]
            + "_fit_ledger.json"
        )
    )

    if not ledger_path.is_file():
        raise RuntimeError(
            f"Missing final ledger: {ledger_path}"
        )

    ledger = read_json(
        ledger_path
    )

    if int(
        ledger[
            "cumulative_new_fits_consumed"
        ]
    ) != job[
        "consumed_after"
    ]:
        raise RuntimeError(
            f"Final ledger mismatch: {job['stage']}"
        )


elapsed = (
    time.perf_counter()
    - bot_start
)


print(
    "Final durable HEAD:"
)

print(
    " ",
    local_final,
)

print()

print(
    "Stage22 FULL seed stability:"
)

print(
    "  seed42 RANDOM  = COMPLETE"
)

print(
    "  seed42 CHRONO  = COMPLETE"
)

print(
    "  seed43 RANDOM  = COMPLETE"
)

print(
    "  seed43 CHRONO  = COMPLETE"
)

print(
    "  seed44 RANDOM  = COMPLETE"
)

print(
    "  seed44 CHRONO  = COMPLETE"
)

print(
    "  seed45 RANDOM  = COMPLETE"
)

print(
    "  seed45 CHRONO  = COMPLETE"
)

print(
    "  seed46 RANDOM  = COMPLETE"
)

print(
    "  seed46 CHRONO  = COMPLETE"
)

print()

print(
    "Stage22 NEW fits:"
)

print(
    "  consumed = 18 / 18"
)

print()

print(
    "Overall Stage28 ledger:"
)

print(
    "  authorized = 108"
)

print(
    "  consumed   = 18"
)

print(
    "  remaining  = 90"
)

print()

print(
    "Shared final holdout:"
)

print(
    "  openings = 0"
)

print()

print(
    "AUTO-A elapsed seconds:",
    elapsed,
)

print()

print(
    "NEXT BOT:"
)

print(
    "  STAGE28-AUTO-B"
)

print(
    "  Stage27 chronology LOAO seed stability"
)

print(
    "  40 NEW fits"
)

print()

print(
    "No manual seed-by-seed babysitting required."
)

print()
print(SEP)


STAGE28-AUTO-A — STARTUP

[PASS] GitHub credential: kaggle_secret:GITHUB_TOKEN
[PASS] token not displayed
Current durable HEAD: f8520ab8f493ab2cffd8693884c7ed7d51f1421e
[PASS] exact synchronized clean repository

MODEL / CPU GATE

XGBoost : 3.2.0
LightGBM: 4.6.0
[PASS] Stage28 CPU-only policy

FROZEN PROTOCOL GATES

[PASS] execution manifest exact
[PASS] parameter registry loaded
[PASS] threshold policy exact

STAGE22 RUNTIME MATRIX GATE

[PASS] exact Stage22 execution matrix loaded

RESUME DISCOVERY


Resume ledger: 6 consumed

AUTO-A JOB — Stage28-2A5 — RANDOM_NATURAL — SEED 44

Materializing RANDOM_NATURAL training matrix...

FIT #7 — C005 XGBOOST SEED44 CPU

Starting fit...
[PASS] FIT 7 SUCCESS
seconds: 501.733120933
SHA: 1bceda9316470742626c7f87a4f582359b67e81d59e59f868ab71acbf434b3cb

FIT #8 — C006 LIGHTGBM SEED44 CPU

Starting fit...
[PASS] FIT 8 SUCCESS
seconds: 484.28434046999973
SHA: 6fa8ad5f9614c82b2ce33c4ed226e27fc9aeac6ad9eac5279461caab357d0cf2

RANDOM_NATURAL SEED44 — VA

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  inferred 65,536 / 2,882,481


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

  inferred 655,360 / 2,882,481


...

  inferred 2,882,481 / 2,882,481

ROC-AUC: 0.9985506168743948
PR-AUC : 0.9954147586245796

STANDARD: 0.5
BALANCED: 0.5
SECURITY: 0.1

[PASS] scientific artifacts frozen

Stage28-2A9 — AUTO COMMIT/PUSH

To https://github.com/themubasshir/ids2018-validation-safe-ablation.git
   1de53e2..b49ade7  main -> main

[DURABLE] Stage28-2A9
 commit: b49ade7c554da8d5be682d91895e434cffddd299
 consumed: 16
 remaining: 92
 final-holdout openings: 0

AUTO-A JOB — Stage28-2A10 — CHRONOLOGICAL_NATURAL — SEED 46


FIT #17 — C019 XGBOOST SEED46 CPU

Starting fit...
[PASS] FIT 17 SUCCESS
seconds: 736.2857825800002
SHA: 6138cc648393cc3576f22e41e45928efe2a6d55b4e10535f505f6556c49e2e3b

FIT #18 — C020 LIGHTGBM SEED46 CPU

Starting fit...
[PASS] FIT 18 SUCCESS
seconds: 620.7253623310007
SHA: 9f57664a7b8bea66067125bd19c1cc1b460b9f8ba4d1bb2fc7ca7b13cbfcf6e4

CHRONOLOGICAL_NATURAL SEED46 — VALIDATION



/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  inferred 65,536 / 593,780


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

  inferred 593,780 / 593,780

ROC-AUC: 0.515509674202732
PR-AUC : 0.10631798297301229

STANDARD: 0.5
BALANCED: 0.06
SECURITY: 0.06

[PASS] scientific artifacts frozen

Stage28-2A10 — AUTO COMMIT/PUSH

To https://github.com/themubasshir/ids2018-validation-safe-ablation.git
   b49ade7..4757827  main -> main

[DURABLE] Stage28-2A10
 commit: 4757827c9e845a862113f96339ada0b9e95a948f
 consumed: 18
 remaining: 90
 final-holdout openings: 0

STAGE28-AUTO-A — FINAL VERIFICATION

Final durable HEAD:
  4757827c9e845a862113f96339ada0b9e95a948f

Stage22 FULL seed stability:
  seed42 RANDOM  = COMPLETE
  seed42 CHRONO  = COMPLETE
  seed43 RANDOM  = COMPLETE
  seed43 CHRONO  = COMPLETE
  seed44 RANDOM  = COMPLETE
  seed44 CHRONO  = COMPLETE
  seed45 RANDOM  = COMPLETE
  seed45 CHRONO  = COMPLETE
  seed46 RANDOM  = COMPLETE
  seed46 CHRONO  = COMPLETE

Stage22 NEW fits:
  consumed = 18 / 18

Overall Stage28 ledger:
  authorized = 108
  consumed   = 18
  remaining  = 90

Shared final holdout:
  opening

In [8]:
# =================================================================================================
# STAGE28-AUTO-B — ZERO-UPLOAD SELF-VERIFYING LAUNCHER
#
# Paste this cell directly into Kaggle.
#
# It reconstructs the complete readable AUTO-B bot locally, verifies its
# frozen SHA-256, and only then executes it.
#
# AUTO-B:
#   Stage27 chronology LOAO seed stability
#   5 families
#   seeds 43,44,45,46
#   XGBoost + LightGBM
#   40 NEW fits total
#   FIT #19 ... FIT #58
#
# CPU ONLY.
# =================================================================================================

import base64
import bz2
import hashlib
from pathlib import Path

EXPECTED_SHA256 = (
    "be49483668ecc147490fc7430cb410108c4f6ae04f22b83861c19323a49af1fb"
)

PAYLOAD = r"""
LRx4!F+o`-Q&~@-wF3Yc|9|0BR73!I|NsAg|KI+<|Ns9$00aOK0ANn}PITIRb@q46Td?>g9koI=y<cZ+XhnTJI`v<7p1s=pz#aAJ+KKZ?^v>DS8yMSK_kHUv_Se4oyPVH=gEwy4Yf7y#C4B}e>eFtgz0PlWvu~z+-+J!VWu-d#>5}r?R4&`?*M05u_Rusw+E*U<ef7KCPS?lRw3EHN%8hm;-0OCwIh(Ag>D1_E+Qu5Kb#fr<l#SbRj5Zf8<F_hnsda-}S>w^(_oqW$C*OLDx8HTO&bh~Rc8X4!R&=)Sjv8n}XaY0>VrXh)$$}<mr|47FHm1bLG|DlkXk-mECO}D`KniJzwN0t&YI>ei2t7bFXfyyc#54qnq7o7`(3(?viL^{lK>z>^1|gtnk)faflSv^HAPAW-nq;Z?RQ*%b^wUYEYBcp48356t>Uux`M3EB^38~=#iK>26GAeu0Q#O)%kJL{lr-}xTQ)Gb2rh+7d2oZ@4srgJNr9CP6De5r~N$I5YhMF204FCW*k_3_W{QQ5_eqYJ=u2_73Ed?=uIBK*%!zm6Lff~KlKg7j++)amOY3Z2#j4|)<wMOv>Dt<*O{9?&bLHftc&;`ldd6K`rh&n}-rNDRfMj$|7)j3qWGW>}O##MeTNm69<nq{-|Z2K?$FHK6W&Y$M1?U;5SW)n3PH>|G~WC~+Rw7umH7zR<Xi{WX)Ry){!IahbHeem1H2mVBYg7qGp_i?&FUc*_-UhNhM;l<=72@5L+f7eEXrDS2p(NXN6)6L7Ea>WCd$IB;wJ@Pv`!i$!9CMo9G$tbXcY2$J)!&Z0|!y(Jdu1(G?IEO>hg!DivpxK(yUmN$U=B}<=XP=KtzKRReu(r)hv@Eqz(jzQYYg-1a@?eyeP-<wau?l}h6kLiDv~~3E#!IHkXp!nhk4Zt(rm>Sy5q9N5=;M-w?v#67GpEP8+{s59eQU=p*Q{QBw1CuB3=rum2CUmp+s~gKIO_D{VC<MZ-1l?FmG(S{iOLTqPsdTl(>TiMl!^I$S=V6UBs)`%yYK6Mz58~@1H1#xB7%@qlWc5aZ6ZNFFFWmn$r8$-Y?3W7xcd73ula3G5)dTQ-8PJjA>$7zSXJ2=S%+O1i=>jEdFLl9;N$-6Kkdh}ZQ!7!ALReX@BEE?ecZkMe=B-$_tKZoE<&;>51k-mEg4i-#T5`(fzwJrMnF+Uw$B^NfzdP-0ZddXB^0SuLBN>1bfFyNW|IDq%Xd601XOybKw(6ZVs4W|h>FvA|2uVkb><xq01t@}3_)5cMx{bgu$I|q+S0KK!Jz+wOHt4;Su_lm4s}e)5eTu1Bx0=9?!x5ze=ia6D!;SU=hLt3)&)QJkikpT3W@zS<UdaEawq3_iXFUJF0<=&Q`zhg;6%A1yYfGewcLANUMC`m=XAMnhrglv`98i;A*p*GS{ES#<{JJ}hOfW0$08|ty4>_OIPAX{&3|k?NWlhmL>zLeI%jv|`vQmvx~OujyneMpVhXIIAFD+)IYIW&iQdA21+=Gh)L8{a!}m{BNRYF1m&ji|u(XJ3(_V}@=+wuU@(dH)Rv3sgPP{hS0H2Me|B?g7C<T5e5$`nOp#{B*|07&2e4Wso>yx~w!OL=aJd$F74s%U2Y{5;?y-^@9eH4O6J%zEfV@*%GwB&qv2(`{7S>0VI`W0cL$DR{`(BycKnFEK&vZ?(B{IF4h#kr($=n*N#Fu{39rjj|NDBW2_2nOlM^oZxCO;AT0Pen?)T;yBq(bq9V(BL0{8K4W-eFuHiTCUJE8t#{}!41w)Di6vh1=T_Qbmv5NK&f^lDoRNZWBY&s>bGwv!f_W|%%pgBe5-B>_I?g(h4s?4_3_v2u>Z%mM7#Yz+coh1FOk9VMc|2p5%|y{8p`H~0$3nSU@M)6j<S0M>of8Wy>Q#w+hUEgj}+YFZv5N3>^?lMJ)<_L7wM<6YAd_MNvZ)mBgyL;;4d^+jTHV(NjoXee5ypJRal@1gM`InDEAGWTynLxo2XQ?*rEsl(;K}U)bGmsqw0g~T$*Nu<(qZZLipmmG>q8j*NtNGlU`dG5`+>^T!~RWm+8IqYpqw&MQul?o=*+>+v(^VDt*!!Xe$9{ZSO!Y6%{GI0j+f2wtJ2`y$9`}9iro>*X`5roWoue!7wj|cpzxQjFeG!j>c~DL_nm2@6t!k$WAb;m+pj9zJ0AAKc6fMQbFiNFHQG~QxdWCUF>smzH7D<rZ^U{^FJTcEuAMWzYGnqPtVn=!IA@2Z@Ylk=a_DZ^9$bSNKlA#*1qE?JgZ)G`kY&m<If_r=W95%X-Fby?daG<LFY{fT#d9Xc`$yxubqand^&xY51gpB;X%L8mYLAuZd|;ybQ3t&53|d#TzyKU0&I$^4v1n9kVy^j1Tj=VKwYo6+*d_26I+2nciXl}F>K~Y8F)<vR}}z|DC$3?b{`|>Dkp7LVH~>l?Ae!_noUmu0ptow`s|?;gLw){3$pzB!@~Z5?mNJ+lH1onOSwaQOaN@)k^}__xEpj4cm4CuG<r7m7pPGxAM)a-aPCN*;AO|oYs9SDGS^q-He5t9IA2c~yxV%}885_Yj1x)U5wXT0pgPy=EYH0r{<pq0yf=<E>4Ev8EaTcPAY?jVf`jLfCF++N>)tNve2qDGr_-KR+3?@uSK__S6C*`dQ5?cyGL4{WPz40u2WdTQyD;R-3z++s1s5SLr1D|2Alqjz;>B>-z|oBg!kojNuLs38iJTi=)_VH;o@U5&bmx7YGULaAPutw9%<q!(J60wm;R$oU(KCFDYKo9RZ)=MWa)t7?|7xcDVYNl<RpiL*vh*v)IOHM)Mo(RU<3b03jjkYp98KzIq~Cv#gdwz9aT+cqXH^OnO;n-~MsqeLQae~OC!0O~sp*!xpzbuZ4F=i;?aX{R4Xy1t7!0sJiyk9#qRUNmd!WV!=1hVS94<==fh7tcq#9{Qm1Y!FpTW!K^L}rT8&~G;Q-0CIQ3_FqccZ1n6_JrtX76#nzXlrH1RoF_RP+F@wkjAI)`Bktg*|Kokco}GCNB<KY$Bg#v756sIWQKq$g$M|@_q{4?b+h<H3t59OnGm&eQ(b^4K)(VQ3TOK^e!$=HzQQ`<xt!$z#<X)XPv#AqoEkL0n0(kGCE@1lCqEllTda!4eS_Hr#O(uFdD?+{%C3ep2=e3hYpTP7Xx{(A+1KLEsqz--`&0Kc~dUUedF5jkcD8fs<H|atx$1w5G`7wiWFc$T{^1cWF=vk?4yS4+YR+hOk~GH4g)5P`m`?{BTVtO*upM=M?Q|KxlXvmrIgWG?SU$zGw{rr#lnQ}(~wSP3`J!t1R+6f>ziY+U}LG)={eXY9^#u<Vd+Of@Y#uCm{e3!Pyz`cVsM+Un6yFgQ3f3X63-Y9LaBij(o(@DW&`#UL*k_ViTF)uiMzKgpQQCD)p8VNL1mNP-DW0}1@Ui`z_2ifXym28Z>3{RTiwtvulsiX0WH-e3U_>xoeQ*xB?IZlJGiE!@_ZI74ujr%Z1l$X3I^g+i}246hW~EoJ?Q%vFY3@<O%YvhOzz7HV~gMn2keAd)tE8%vj3+AXyQCe

ijy45gVA)*3dnPU8EqK>YMT%+4vu+UrkUG+ob`(kYNpE?0yJc;JLl3m&~UbO===5RgTM5>ps7|H<60TN$K!7gjiw6w_z98o%~##wueVHcv$_sw4_>cfY%ZiAvj4f<(IB3F$gmUm^w-xhX^+$$#ph-5^?z`mT;G?{VhA4abtpe<a8yy<`jl{g2q<yVzybLYfCM*D0Z~1<#?CsE)u#NXkAK76^vCtCA_ik6yH{0gKHI3qW<60ReE_DJBPS!lUB@3}(YK&sNFdSX2dM>Y;~c6zA-=1AdBHcCt+~l12P1pGs8}g0O&e$&g7-9Ui2jERe(RzNN)LzL&auw+w{FkF^Ftb_<V0*07PzBET6!N~et&3R>;9s#e&|x7>}**o4HdCtM6)geDTEL}xZ@KNl@k}^!mx_mvjZET#Gr;;F2>ces)1`h{F+Qbh9nIpaGWy*lhuD&<3E-1`QI*9>5i=%S3M;4bp0hrvvj949URxJy#uIj3?K-g(n||nJ&G1Ip_P5BSSlcngiJE*M%DrnT!v-Jb2*wd9*iiHi;}|+LZhnZ)9&7Qc0+f<^}-`&MdFe0I?(P_k_-0~kQq|pMc93$b@!5r6Lu!=KU3bP_f@}*UHD-C3dC>x*wJE!l#+_233A=9uF-fgT8r#bDGb!8PtmHXY%!0kj5kJUP?OE*BPhFzBG}NORaI3@)ibkGVW7(x=(3eVb%bHO1sTGHR>~yHR8uzzyN$RFhGer@9D>G;!COt3`Mc%gg&N!B*6U`rQx)_e+q(&zv0SHVH%W<I8pXNI-Q(zMdl2(3(Wqsj&~F5rDi^vA2Q<0c3w$ZoO|r*nwB7R2cHI|HrF`mZt<sw+REklq$lYf3L9NFg!tu7ldtf8x^Iwn;#vxOBb;9OaFnLIKKVPmMmGeCHb&`t6esOEeAsNsS)7E%xwS2K_LWt*OFWvXk33#Hz#VZjc5k^~B0o9VdhQL^WIqX_o_<D%JBnK%6*mTSorPO<1knpiaOwiObM7`%U`a?~4xQlGOLt<}BcH>(v>eBCo=WktJlY29nh)145f1KB0%wxI_D}Jw5&Yri!i51)-At?#gpH<tKVQ=a&VxLK=^nhf*A(_A!C^2QZsFa14a2dwB5vQ~Z8U?3Xp5M^XPp>hzdq<m7yM1HNW@YvQp%pF^4a$ud5Z<6rFNR8rMd4t=hD1UURS`h?5j$~!99MN4aVp0|Yay$g770|<O+qYvQkNMrlyTWr7z(0zjJB+$3=pFv1ZF@W&_JonSDW^F;fJvgx0>`j_w(>Mv&%0onPWS3DXNF+8uj7Vv?-k!RD>j@yC}TO)DCI1e2@B+C!Lw5cDkk~Tpfsu`<@WL$!{<@qOi-88sMccA|wX{WFUNqolE3gxOR*n%8+-+bv9gIdflIji3Ct#njd%J&EsnRc1iJ2H|XhF&|%b!lz#HHz?SCB(q)c0db}n2?^LQ_ESFQRYL#751{p}XuE9`!DXqB(Z3+|-F{YLy!Qfm>?~{mKhB{y<55>a2qY7lD2&2DB2d+m<tuc8m==i!{uc55`RTjl>6>AH_??ScMbEg4<riGWkxOUJFGdoqdKKugTZEd$)s{1`DOA(NFl0^TFj|pIrZ5Ht5RRftuDFQ;MB?h-D6#Lnmv2Edjf+M5CaI#3M%|P+uwlrbXi?e}&u=k7Jf?wCc^?%}HE=Ak7-s`T5CNF`8=ZVWp6`sr%I=97;Rsj+>+=H;pfgm7?M#fzgTw8mB6l*->nWY)?Tagu>E0T+VXF{bvbDtz&V9yYdh3M^;%lP5S^sRkh@#<Z39}L!W&*FlkmuRkkx3a)U;J-97aV6S}_N|7g@04EqN~?#Bf&sVLTI<fsVoFPdGt%lrBWrKU9f69R;LU<LQsF%m$Ceg?A_z5BGuk=glS6>9Zl`6gIGm>GCXi2%7}58(Dt}@3#)q5{%j^mRE~l%kT)K}d2Td|vQq{H2pIyBWGwm9}t&m_7B7(cYBvz`1RCPW=iGjfDs4OcO9q<9oU2P!9NIsY&A~8?ByQ4tkq?U1x0a|Q*el?@Q?x0eNgx`YNxK7rkCLF19U>@o-pje{@qP-(_uemhSzAPQ=FG>dwf}$wYM>dv%O+DYZ?Od)EcJok)1UMid^C&pP=Q%-6&^EBNbzZVKa<TZDzr5;x;>SM<kDuGa6;(_taLA8#<CSD3q6?4_)<8&*q+6yuiJ%N1G}ExEn|XLwZ@<b+`n);VVd)?v*P}C_j)AeX6jly|gF(Y6OMzRQQ6v!}2RB7{@YCSXvxMX;QA8QT$nshrRb7vl&0s(^ASfIN6q_hqajs3(cV1{Z=!6=YG#SKz>qT*o^?{DN+Wo_T8+O_;ZTu6rl_QE=b$?+UMDgp+u$-{M7>Nc{j-(jqARfm@ke#4>lx_OrJg|qu<T&B)!5E9P`+c9)zxRGW_&OfEE1uz#=G2H=(Hc3Qh<MT5p<5ZfbwEGa1Zs!GB5d5~H4E<(kCps&JOe0mTES52>g!*pxua=kmM&>UtG}v&#Qov~EM`$&Or#fS&Omd?rNni1uZiQ`b()rtKBuw1w%9iKe6o<jz??pq0kZzKJ)m?om@BhRrtGp-O~Moi3x(75b^RDhQd|K62S68qskY1|EU(cQ$D=1XEea!VT<~Ny2KFDM`ES|#nP;HQ=<GCA!!FAB?I93wr80ubgJp_y!AGqE?gus<n{ZE<_eewN4m%q|TO0#+sBe1XN$AijX_3IQT4k#Ts2%~A4Gay14y|ci=%ZEnVo7+_4{n&Zq8ch=v<(us5HTTs-*}<wgn)<D-fi%<8AtbL@%KQ$L&}6Ei&)qJSO5@t;JZ-<-vUS;N9XI~Kk4W;5I-G#UB{!T{NkDo{ZJvRLtATF)l=@qWtV8(A$g4C(@oC;9FaWie0;y0?d60t>C)VWxEB0gOudim15b|bSoPNB({?S^(RC0Fxs}UZ&xKw70(XBql8{Cl6B<8W>3N{@uMf(-Yrs`X2~=|1h8$d>4G3_9)+Yb@AGvu$!9NckVq^4LGvP$FY0wZXSwaX!e4;d@lb%!__5fT)3#T1Q+!80CcEw|>t~XryA80`ZWHJ0t9-mg(+sgMI7ZZ<dl~ONo+Fb%i-kiaA4G;pnk;TH&AC%Q?sz?&sWx<nhmP#8uo^Jpk1Jiq;o8r4MOQRGBf_(T>HYcifpd=EiqjesfCc3VZKp;BYPyzrbK?ij3z4(g#T)R1hDnL|ce%kA|6B}20Cq+r?+h)VH0<xgO1r*l#&t7}HySMGY-LMymj@ot^$t3FFmC+cClgmb2xjjM80s{9au<`;*6Qsob3y|~;qw^v(yg~|QA2);q9Z2;RO=q@i-313|q6Ni@qk_R<zZcVmD28GNs=rtkszuZgR<&f1d6dCoAmVrIVdY36igGn@>^OO?%ZM7w{?`R?W{Pcxbz6?QXL+@!4_4(9se*tsCv^1rQa@WhHAIpu2!SA^B03O0C;iffxTr+%h}Q^GNOwgVfN+T3Bg*)k<+)m|Tz-4LbAbD8=iZ^eM{nmt4JWxWAjajPs#!uGh`db`AVA|WK94@md_(jJKkF8j(pW796(uac*^qGmi#4V#l$NxVLdZcG

0|A+m$g(oAWKmYj5N(2jAqzy5lt9E-Sd2z0NMiu~`C`aL42cQ06p+G=lAu-)te>lTNil=lUw*hWp|iaxDhHf6%#;IQ0jqu27`zA-8nvI^eKuYTao4Ua&;r>fo`5*wEUHo&G#3j=X!bp$DWBXrz;!|PfvpDqLd)<+3FcY-6nqiifdJ3`b96o@@+H&$F4K1o;7gf?-@`zghx-M~_&*@T40jw16g3~-e)mec4tsycuclo&uKd({N!B2GeDB_1v+Ju-_5?w6e!t?iW9{2{ynI=^L+sz^fMo^XM73^V)cm1f=sQaZ=okQN#F~l8DC}StjQwAyZb^zmQ>m-aqXFwU5HRH19tVIz*+ntQw-&(`ZqW^L8-RA16%rQ+o6Z-|mdxRr#)oW?PC)ZjGsoV)LnpJB2?ibNqmc|b2C2WlkAuqw<B%Q&zGD3_0m0PLw8$oAh-Liybq4Yz5(Hz|!#Vt6{BY^7x|2Gch+5+;Piu)@A_0$bKRSP6cn)wNNNB$?PoA3|{o%woyRvN(1ZKehowwo5$ExVC_EN1wh0C`MgS6K*mohO7+jXgYO;OA$!n?Nb;#H118;#>|!who{``?X&K0Cp<85@Sn%s7sp^@(${*IkWiV*Y7F^{FJjX5DaSJ;j?Ho4_pFAm7^|C>{sUf}6mn;hEEx5_@W;4;XliO`tH*lJfhttOSUZO3l8Wec_SxKp?C}IoGpwzB2Iq)rfnz)9y2S$;_((L=GYW?C*U;%zgZfL#@N_fyY?mjF1iq%z_boZ*d`>IN<i_CPS!-UjyS5rJjQs2u`}gnU{8MG<#275VLNn?lQp(Cawdhy~CSu<<r`lC%<A@U22q_4W|^oIyTYX`t)y@cCbjr^`2(I&$YO5@jARbuzIp`=vephc<LL{LrDU~taUvp$H>s=bo4oAnmR!YiO?j9wwZbRwR-Q!JaR9hR0qT~d%}|De;f?cB(4MuESJ(gS7tSe>^f~+$0m0X4zQ@o;5TI&?E*YUf!o@Byo~i6%=gJ6PMA6S^V;a@G61kAdBDsw+gV+9ZoIPYA?=A?mEKczN+^#Ag``8{z~|Q53}Vo>6e}rxe%2QY@FHT`%9UmW^J$=r&nRe~_;9KlPZjjsR64<+3kuG_ALA@w9YFK|tF6l()jZz~+dS74Ie}=7(%l1M-oRRu#crV>J24HL0>v3zvEDTEUGW^qGjA<r(Y1R)*O|H4V&VqDp2o!5a5fz+>BQy`>(@^-4kA8`!=Jcge!F{vFr*k<ITFdUWz;$*pVV^-ZW!8T@PP65*5gj6%z6i_2eSc|0l;o>8ahf9axJyO{cFfex1!DG^6*pLWZ+GC8(pKfalNCS3s)%RY=ggQ?5jn5z+ux$M{{FvX}$BKx25e8p=z%`S*s&Qa=`fN;IKtJ?-F8_RpVyUFKQMI&!JBrCtV59EFQyZczDw~rQHYXCuE0Ch7^Y9tEO7nT&28qhP`1HBQ?z=PIw0l^Ve`q1=hPWGiz<PU=&j`EwKIHjfRa|c;hz}?kzXH-9osJ9W-p1gnEnDuBBB<uf%&bI0+OxX+|<DDpfsCmv?#xRu|#)*KTV&o4tOkJ{I9LVR$tU0wjQW_aRBIIl}R-m>T2DW$|ZYBhj*^b~S^zy^5`RDs0$X$~6JLURYJs)U6B@IX7?SoEl5#uxy7&SE_v~a@F=vX&K_DS9KXx*MqDlWyxN?hqwZ7PA>C`XdA<Iw|g|_m>s5yp0hA|yCsA4x6#Dmd^?3PAhm1F9<tv-(wSw2&tXxP!$_zYuU{X}`dctCV)`dS={$A<Aa35NwlvhCnZmFNs&?uCWW<xJfLt6Ya8PT*CdjHQtDs6S^PXUI;SQbu%71=q_WbFA=3pz#vj)%Sqx%O+N~QfqUy9Z+xvqJogZvFN!B;;>wx_)volnTojOg1#NHg=8#v7JTwVGY;Lw26E2F2Sk?9)R!#rQt$7?vIS-#iTav{LY4;wpnXAiEC8ErMRo-@C^Md41lD156z<cXrvx;8jk?3R-9IXHNee%`<G{_#A2ycD{sUI_h!m{lD~mgo=HTF2?Pp=zzoT`TakC@BS_Iz9}uU(an{+04Nx}N+d&ARQYo6XWo9vEhKIyTH0knRTbI;P)CR;j#Kg|J;08gR@f^(CvOf|2Vr@K_g~^>-XWMjE!yI26YuTJhl&d#JpJ^lZQWnCI-q_|x+d-J032h@Idr%D<3B$88~=qSL<Go)VF?RG{3GRLae}_S2s*j|{$P7bDj^_r^m4<`<b&X;@luaK@*e^w2>X$z9Ygl$>Z*dN+V(8`wRj77;xmc;AF06Hjs+tuGL}M$F4CDzkkBf9^>n4rvH2$$AB4$L1&S=EM$U{ZBj?Mee%-tGyYF6B@5ZvcZNC+?b6fGoZrj7f;QttG2OraIvX{Uol;`Ax=Sirxu!9y@t-ec|`q5HiY)L*i=K3UXxGGfuAvAh!5uUNyN4wq!$Mp}vRtTOd2p$1Ny}%EbyW89?3>}j~DY&?I|AiPU<C1ZMjeeSyRaI3pe&B`6$QTl#CihRMzr_2K@$}0dUbB|quuxNYfU#s3q&%@_Z-w!00!B^1{W=c7<av>#GfPMIF8Vwgv=|rmHQo!zI!98SQ$66Oimz~vIN;z-6cOB)51#9Q<Pu?XGF7y}rG%2@#d=hjW!v&GY|yJbkh(~O%BNhzPP^Rlw1XssxN$Xx^Pj2UhiAv|QQ0;qf+(>O5ffr~Yt+#iQq;BWw9KgE9TU`|7|(H>7c`0+LkgrG?^aaLdTV+R#I9jO6=G4WDdYYaaoA10+btfcB~<w_u<xeNL72Q)%!#&!C2IQlG-_|KC)9gD@7Pj3ClcjFts7zMPLss%D9a$+-axuW&jAZ>KVMYv>xWMBf^XNRR)t7ftiy;_#$il(yRFNfp)qB5I_g5hZpd<|Dk6UhD52@$zd&G4;yki<9Q{$Ir@d@2$xckLs=SJ2%*ADtU9%yN+#23@LLOnk$2WOTKHxO;iWbtPZ&=IFTWf8$h|t5{2RL~48}dbp_74)k=^c5J9a<WJ>jq+lo}%Q8*>z3TIp~*Y4NI+)&%emguxxH3BC02kbUAyv!&8E5f_ak;JVa}2PH@8~iME1_D;YwJCxiwBccCNBjTa46U{TOuiW>FZb7rJMS9*IqjhTkK5<MSIhL$;|XREH+s;a9cQ$(Ge`%MZmw*A2-2AT%2$AnFX1H?IEZ{HR$-f%m(gaSK7!ed2MHB{3L&poAw#6M5ZsoNG3ND?MN(KgnMsXoufevZ`7H07Kx`vxHoh~em55I<QAy8N;`eG&#wn?KSUkaroH6cPvSES(@ocn~pwVTXU{^v~1;B;XDPeJ09+2w5iwuM7+AdZ)MjHl@G2J=(hX{LvI(z*q{B3V+M4!`b!!pFgkf%`+fC@aOdK10>k7udlEz5I|=#ko^}!A720E7gJ(FPe{fsAs~qblk(9C4*A$YHTt{L-`M139BAO58ybGLm{TNCl&a+<i&;)lPKu$hszqGoV#~<1D=s2}3Wh~W7VRK(=o+EWXejJUp<A)2Q(B7@RS{4XLo*Sg782zPW#Uimh&RVg^veO#k$kPRvF!n7fG0la0nsxQ

+9JdVmO#!ZE&#X{bmmE|ElCtR9R&Aw2Aji_Jc0+YqKqbpdN+oF@Pn!mL%3hbu1j$A#J^4+ir@1@kdQ-Im?d`y%njd+0qz?a#oRS<GVN39f{(z{lK~8kwLRpX0us6byz|Wqm-<n^L1xt*lV%2>w-GkM4lqJs2GMiWP<CFVQ1==bs$>uykLlKlxaNVJ4^4t(VcSw`C=!MO62{@#{V=g`HNj&NNe>)@2V^ZD4llor<(_%nPVMNC0s=AsYKfHoiYB=xc$c>8D6-wt!4^ga3^ArUK#XRuyJ?`g=Z4dP&|w&Nz-aA>#Zbvp2LO*>TOKjiV@*>R+?$h=N)kk1kOzk}u$DqPh53%(R<JQZGz%I`sXTj`@0$R$1`H6C6#!sn4xN8TjlDZxUWlPPXjpCh(n&30YSZXgZJ|sl2`_IUVxJZF#>4t${OrGRxttHYgFiE~7RIm5>Xa7(GyO=knTFekTtz&5jFro8M?n18HH?7Bk`fP}0OopW)Mptb5T;-k{s*6k2zgAI2)jLd&s0f^x^4ia7!U}P-ooZnUHLDkiNGb%N)1CI1t=qtssKiSADiQ?!cceWSe}@XghR@@kU6g}>CFp)5<r0J1Ah-c>^w>S_uOwdljrU3eLTF8#-JeXC5jR;7SReAm?$KXJ^X|&9M$Zc2wA|f)*Josbgfgp3>)Z52qCLw{6_E9RpvnSgnD`wYHH$|2F-EGj7Xkz9dxS_4z_%!F!4qnT}B`zwltPYKvEP41q>)lqFG$C9y(e=+Rf4Myz9naDV@eUOv9dqq(5UM3P<GkBg)V|uqbjVN>S;S9&G2`Mf?+V-cnD%EgXeAKoCe=MkqyN0KsH?yu-7*==^oO<ic`RRbn_4qZ1VI&iN*wVeYzi4%<6sRMO-<Pz%W0&+|T`gsde24kf8*N>HOLJ_sz5Mkv5sETuq%i~Ax4gFLhhP%Z4`OYhoJbmyr{T5pvh9s7f~d2Kt;hG*!RQTbB_8c#<RDB@8j+R?2GKb@%N*^pQ}3fQ7VPois&rd{{rP+1uVTDo+@AkknkibO#T%wouE0+o@j_H1EImo~zoLTG<$shCw^8sl1^gcc?TVaAH%zkS2E!h>$y;<hwNuI9<M=?EH#gNVVHKxG(OLX%KU7!af!g9^y8waXSreW_$Fr*_G<x=sja6tG6k9?6A{zWHVW<UqE5*2s0?xY%j#5}*f^Q9Z`n__R}}ycH2i^}Wa+cDaI@yJwr3N{Fv09!Fi99pm2aizn5&GgQ#YYaw1T<O1Q$v5%b&%VT65CvepJ20nfniM<PrXdBBE8^fO-a3KtP;mdZ&>SxZ9qDfLjog_{6E$Vn|)`)8b=9VZ!0o{#+M4(a-a`I#tmA86@AeJW>2F~#;v2nNw4*;nbKYqPTi+itIz8Yh_D9$P#mT*4R10ez+%CU5UE&H=eB?T<=T4*}fL?MdNi^7W3UINY<4UJO5<y0wG)Mdnl$=-|&S_3_mMP{{fh!--zEh+e@i=nda0fqz=7-4iT5_krb5fP^hTw@bRds=S<_(+y557coNE0CqwMRel|@MkR5)aGbAPK;XC3Ig{t!5#Wa<f>Cx`!~8)diF@WFmI+Yr<7eWaP0Q8Rv@KxKCp2b6@=1^7@Xf~giV75BMgGp;I!cAGS(0Pozm~XTs;lqJ#Szl(uTFXjjj|O1{rfL*SJy!PDqpkl@SFfxLE4fKIXyD!<^NhA%UWXT5xBet4CC9Vj3!v&4s1mQ3eb#S7YScLoDVaTcM)`hQ-53W@~8B8%GdrrX+t?-jNtBJ`2u)3~zEj4GJGp9tU7N&@vI}R2m4}+yP``{H1OJAF6{fK!q@O34D+~Dss^1IQkY6Ek_`cUe?%8n^xny0^B1Erxkhb(rY=c<=tN&SCW~HeK+mBFISR?$4QpbA!9?a4_k{#X_*u(s3BzQE=k(N(!fRT>2iQNAp$WF-0C-3KMOluqPfr<3!!Ob?$I%dJoZXKz&DD(?e)*HebehTA9}j8Y!H$tz&NC}mi!cU2w!4Iw+e)%wxG%sC_x$m5EBFRord1RLy;X=Yt*kib1`8dntd`)nZ3w2`|>wyPqvlc)JBke^e{xqKBvG>y$N1L9s-bYG1SSm^?+^~I`AZliiyxzhvz`){V(V}J^snqZg|7L+1tmsTxevRxf;G+j7l!r#Kr)&*asyDM(Put-sd+K9P%JYLowHx<87nq87ERq4bO?FY??X;i`BEDmVKSn+hUnYj>s@H$yft4f%XKZ8r9jTWjN4Yte3xS4a)^w3I+*wj}Xc|IpGr~Ad&>>4d~em<rB`EIXh?KiBC9gHp?W{hC(19Bxb1#=A2d1)?v}bJ9#)#!_2w7CKEfiPhI5f$;HN5#oL&7#ZP_q+9wJQ#UUI)FftJ^6mc^;2~re5gS{@95*zktc#8yLvC>p!xRMwk2N11`h`L8Lw|$f~W1JxfsvJ(jZmSw8y<!1%y7mbAdimFx^!xR6Lm;PXDNj@$1o;nacHW&+qx$mJqRtxW3b<O(C7~l|$X^E@gOm*ma}w}8IxbP}m<j3V6MPym6j)E{wP3IX0xX1ohJjESTCq`xsG`9^e@DFYJlW~zPpKek<CfbKE}E!hh89a8gVGW`2gKuvo>n#PA7^kMgu%HyQ1ECzZexT$aC~0WJ}1KC)kDFWVURoMC6dH_#ChXMy7}O7{>}53j0QPHu{CW6Y@p}@yZznYQA8cK$PE)>7poh$<l8I+T}1pdcSyi1ni&RkaL+&JfLvVzEl_$><J(?&WR@Xg-*$`em=AeDV#CseKDPUd70idK24rFv0)&hR5h0<;2o+QlmQwYYz25u8`2HEe!@a^7Wx9-GX}W1!Wk)QkD5?O_2%4+zO~JL(Dk<MdS?pHrtXoJf-4yjJuRLv;Iq3t_R1yOix{xgx%&esXpbUgT20+x`1Lgb(ioZF1+-E5`*AG~1iZLNoNTX9{#8+x>QvfXv0Rlk+K@D6Cgv3DRFjFs5aQ6v(i`WRC3p69eFJ8Rjl0(fQHFt26=w7whE#0F#qmX897+#B&3mVQjE=NA86g@`}$tENCz^P$ND+PrSgrLGPiok*ri4fGd-AhfIC15U;(NIMs6UI@F0z`mbkfQ)DwDW*<<#is0+9_~5u$qPWHDH%R814Fca?H||DA$grs=3E97~=rd5FQY+qhKH>Km!wB3y_9JB5DguKElc1cg1kh5_LU<*0Z20!HBb<#)_;r49`|?y%R(YpY_GWEN+cT1&kQ75sFwWBk?xx4m!KFz2$^j$d2lcx|{xM<aY7%mLXGhRg3#=Du<QzS*MMf&`4PB!u2~#cA2=hr?~Rymt-#86$VErp{td1Q&%80YC4j|@pwpOMlGoNkm?o292;~BQ;}snl|MK@>~JZlx+Z3&dk9Zy@W3bTLAt;`pA~glz<_SnDqRI*0O*`a2^96L>_o6CMF<N3LQ*91pJy~2kkAvt0#O7hA*Qj~%2b4y?J=Kr{g)ZhN)yQA!Zr*=j6Px!s_ss1(>r-vP11@w(e-7daer-!3x;iuX>e`@s3qOE?>ei=Eb&8^cek!s%&SMQC0576EkTd~k@!S@QM4@&g7~23Z1@;#

bsoqY$T9#(q+&<}f<%BMA_Lggj8Lj;&_kzT;{?1-uZvE=O(uNkH5?VkF%+e{=w?F=$~Ym$&0F_D>_1@ZLInKel0f<}7IRPNj3428Y{rE_SMF9MX}mK&tSn1SOe{f!Ks5&-cBSX0^TOXVXn1x=Zwg+UMK$b8&As_$c8W@<1qDQ)*}!in0|;w4DZS*kaGKI*z~~=9PqDkpAuPdtQ4en`9~f7$Dj;x+WE*}-PndefcodybENrLGBs>thl+Yxi!q|(zmklF8ou*-t1~P+CZlIHoX+69(>Npcd5j7SSaP22YvzHRI>YPB?6Th}{43d2RKH{+EFr~1u1Ie`zO2rr{jyMnRnllC^y1=Ln!P^Z6U1lR3iA!M-v9wIes3@YSHHCn#t%AadREY$U83AIExRF$%4ZwgN5DO_3V+e#2Fi1hJ2AUjBwSTkaN3*qe{OCHIcZcGf1fa)M56&Kod39tvpy%ull{Y^$R*;a2i4YFa3Nwgo>EVT8^spz)KAI+o*V>7N?u0b~vECT_EcpO^29uoYUVw22woSQj@Av)N69Im~+}kFT#*?DCMO0Uq_%F7PM?g55CP&2c2nGC40K$)v=esR`1#_@{9zk7uPlbzo_P5ysqVHFe)rnO~Lx7#2SQb)KyGZpAsYz82BfR);wSe<MecuowBZoalkz|wWwXB|kW-(Y13e$t~hr`{=#@_H=K>JF=p)w#?AzBpGt|gKRkx7qHdqgf52RI9wnE}u)P@03zDc($@uJko~#=xEsZ)!kGvmWT1A;Z>Ole(%(ktssd;GSM-JeC(Eo?N~9IrXQ$YL~z}Q9>E;6)cIGf5HSiDEeiYzvW7KL#ZL{l<34pDdfutA|#5asS}NnH_W%@)D9K!+iP0p1Wx^LZ$;A>oDs~_p$_a@juq5$5d!CQyj9PAauck`%zdT#1|k?)O`vEAaQAeReY_!9JnaF76{i%u&#$XX*K|aEu5D!nEa3v5kcVJ$0xBDkHx~;O-pK-(z&RF_IxW0;qe}j4&wABHe6dSDZC;vbt^rJAqC3-)2$@Y~P(v2slP56EDWK90*^??Zr>|IIAi9h*D&s3A$yOObV1L^iNr{7RAVdraT&wF5j>EJ%U~R<7w_9SPV(?<I^i?caqmGlSWJNj{lYrH<9*@JhT<l!NRMgAcdTE>zwW29H>usQGW+*2LT)km2hET~YK54KROAQdSwihD9W7$kq1XUP-#%>zJAiz;cgED_v%QlLPh{*+2L18D3;lW!ZL)cRYVg;;h53)0=`Ze`jaKOCN(wu<g#9#-5Zf)5Tp>(Vi1w<?bs9x7J1rECf25bT_QfUUzqD2X(e9-GfO|aS<vQ^q6<SZT0)))DemXHWS0i2cLvWq11otmtAHM228SVctcV~oi%1`=T+`2z&hp{UF#G#U_UfRuGY;9LV5NlTJ<!bv0^eQ*#tNq#or3{&Q`FBp)KSia_<)M|pzTRtr!NGZp$&6vTF4S17~PQ$;y<4SFF$_8ng06f6ZD+F;Uz(T@FZJ>ZGSU^}Q0wrJt8dfS&SXNm|``E)E*)9SwX>eE~1t?@4Him@N!EjgCNwW49v{HfZCG__{>;5`=({F)LSVUMXsq>`)l>qn%jl=E=p=lH#2}rLl<J*8R_el&#@QTvFQ$ao|b|NrRB1Au;AO!%(m?pD|Y8wncF0|gB!0Ip!;?0*v39f?t2!R+O6X8v27y`Pb1BjY{t}-?GYy#iFQXob~ECGl;wM#9GXsCrjQkaTh0eA3$#x#*Ns`+3Uu5ZKHBso}hhAM-KhujZe=%~d|HxE}N6i`Rq?oO_KE-g;te7J<b`~rh$9KQZfE45Y&xi@3l=af7rs4<D`76`&3up+@lh=6xa_z*+_GLmi_Y^wx>&KU$$^4x|#Um}g*NFP<L!^Qa~y9vj46i!SCu%scJJjobVx*c&JP=psnkU~X0PWuqeOZ$**=jpgj>>g;31XuDW1B@5HtKU9*3`)So#_y90o+ILz{n#9XbIXEQj7Y24gaIjVPp&sc@I}b$k6v+><vzP#y7Tjy8EJ?F^Up$7(HyEokV)zmhO_FB%w=-<jD}34TJVwzsuECz0@FzziIUkOqKL>OPRpS5@(y<vHVrR9#DWC@7clG8@~d#V^ZruTnOD!7o(In#cwZ0EpCaXeY%rWbg20sAwirq&?7h0}Lgn*Ah<(ZsAPhVNJ%WMtJjNvQ<+n?!yPim7?!$HnnrrFE0+*h+dg=`0imvMxIfc<QOh3y5gw*{v*4I*OSZ;X_G{oX-^-FDdFwZ01pw$}hti~3qsUI&d)md-nPkBJvS-cP9-`-7uQYe+NWr@9+at0V7X)3Ew2Qx9F-n){s#^x3S(a+F(x`zEQY_S`zJIu^2@Ywa?Hyn4RZrxkGqpJailA7b^?(PTclslMw;{@keqqd;t8I;n%zgilMV~I&UJb4@KjVvW5QISZH7DbR)EP|{9o8cfPLpfT=BPJdekckEnsSd?SGa}kZ3PEa^K}iRkAl|Gbtx6Fk3BChMZedLn2e*-MH7SEU%TqdJ**jW~5!f?G06>PGC@@xtg6ce;oGluvpoKA^7?v$Cr73UD(-Zix$|fMtt0#75Vf{VmJ{&rlsA)<H4Z)}eLn0`eubz{a5feTqRCF4IAOizNlRyjz81^Bd2SsSs%{b+;i=x8)R#XSef8xkO$IHy-e8&(v2yYDnnFvo1viH-z=MzAD{eH4ueVv0!i9JGI|8>Ad4Dt4qpKq>qGl7I22>2br5rf0O$j5B+(5HPAc=U>AyS+ic<oed0Rr4CHuR_~y=!Eb~;6v0nnqK#X4dDB2YgCt`Xk;CAlAUNA7PIPyNI?L~6!?(pg&=n#)r0I(g)<c_;YtAjE%u>yCsYc#a54xbwJQK6;6(!HnTWi%oU9?M<cE3Q$e6hJh^C<|dppYM+MbNmMRM?>q2m(c$VE=}a0ntua$&5Y+)13Ggh}cpa|+C-oFQSj$92R`iv4IQm4Y#o)HIQR8j_M>1|q;w0fHbzC$Z))?@-L@bfw`WIQyL(v>K8@rK(Zc1fR>9UK#v;HZt<H633aAO|{;w$im0MUvvPp5$Ees6Bt&>OjcI~wMzkNx5`u&RP6{^bf#&^;)?*3XJAG>_ZyB7`kog6El7u8@?VNaW?a5jz*y9Tpo$Y`;07Ez5(Y)cq&e7BVa?_9d#3UV!Y?qU+i$8x&nJr>H5a2LRH`U;N?`!T9-Q5zTv=E_@jHUR9j3*BABk5MGmmn_3oZ!Y%Hb)=$0Vm+$wgoc_2L*s;zr#VZkG~0n^geUZv7||abc;ey0-)hC}M}Rqxc%6@z(h`?>rFf(N(phSaf|aplpne<VOckv(+MCF)-`1kU5`x<~9$@?f92#%gA&H2C3?scGre!k%;g04M%b|K}~hqnwqgFB!@*LHED3zaiLJxRHP_hV3puvNb0Uu2jPSEXOKoG%x)G4M4u9ko?C5&fZ9{-B7Jl{txU}h+Y8-%CW8r@hf85RPvjksO}=@?2C|$U*P}hY?Z|d8WDvHDVRdn0rZ7raNy{auydy=zf)gJBOz91Z4fyl8^Tanj@t`A{AtWF}JA>}vZBvGT

8lr92xTC`!^|7f}#tzXxIt#sXG4G5f*qTyP%GJso@yB!zD#F;+V-c}^zG0fLhW6KEtm#Qoby$G-`M)7jwckprK4!z2-5;Y=KwvO;(q;#J<gUe7iD-}|;L@-+mne9-i|H4wtV#kzBi*{_-ZB@f-;^+LwPSt7=<M<3s^gM!{^O;B;W>dZ;zomY15OZ^=~R_eaZcfF$R?Rq;n2-8QvJTVl5wYuYIX494?FSbCj+){N6;pj@Cf6QC*Z=?qLx#3#%0&ynrU$J*T-P#oZSOl=(<rJBovD%3Cp^J9kQIKOYjBKKtX#Rh?%Yyy>zPX8rX+&U74Jy!@WX<aIh35_oT>GDQ)X7gRH2zvbCj@ZsodWi{=_>X{|wkuEfhQjZ7K(Lk6>j7$E}@CbwWUVct_*ROR-%#odwPDvXb{2@1x>n1P&J0;`P}m4-)+jCUze#A8KD(8Eqyi-Ro(lR__5s8-rFDW)8hgkVshFhuvw(vz0rZ+cS->SXMM5QWRoZ41kR2@2FPD;e116oVBa4-NZcamWdgqen!m7-;Jb>YD+LcA18(qRcKGjs)6jMB_Dprndwmq9_zYDNfD_LKuzgg@`R|bRcJJSqC<>(Jt9$OIS5fxt-h_$y<XgizBK~-im$buov}-2?!(sfXO03LxbVrM<-&K1i(J8WwxHxZbD}Hiz0lsHpol3N!iq+-Pt!>x~DFpSzge9NQ}5R3rny$)xcO7*q?O`8Z3;O6K&5>#_F18Clai$bhV>0ppl`aNrntU?uoMy(eSjA`BAoYRBicLMRQ0rTLNS;P>75uG1^D!$HUj7%GSkioFA(mL*t84U9g_Y5F-*_Wts!(TF%ec-{t0%Ti{|}8Y~De*o<|Hg+PUj!8e56-xky~2G64|w|k;V639@hP6^y>=SIG(XMt#eLoua2@sn{@h^V3|5r`}_Mg+nj;Xo$HeV4FdsiU<X8mbrrNl#~a+-wMNcpU~GY62dj8)WNMem->Zp9F_N1_?K{1AYg$#=Y{^t3dAVDcX8<d2AYrdh^a?&Q%E>)!I;sqrlNrPnz#o<Vt+b%2`>CJ`=Zx$j2NER5u`WHngto+fQNi$mx;gopz~4QYfaeqR=+WyYoS1j0-M}5VS9)`gnrc5Po*FM1&v`6TXpi1tO<BawLl92>{m^#6IQPB;^2ickF-Ax%mH4GNP0>1Nn}7;P_A-q~INaAesGG?u&i$1{Q)4gd_qi-$+0?@@%IL8y+Z~8KbwavzkHRYk$WceUVSLAZjsE7>^)16x~ko(DnGe!3NgwVYp7?k{F<vf%_yuNEMg1d#tC)Gy`r4$RDx;<YZ=w(zqfoVgv<0DL|AKV1$>hbRN+*$;29VALZio2!uRLl`R9Tlv1&PNSP;cb9dlqJ^*%ZB^h|fj|Lv0A=&O3oOek&e`}W?dv-DWE~&L$s$ad#gKE@LlC)Yf`)fa$E`(rA0s9Se8=0*oJlOW(AQ9fsQ38l9Iq@5xslzR@h<Ju!t1T9;*zeov^xVk1?fY{69P=}a7A#5BD=cTGMwrGjL8`Uxm_&#%bP&-CCijG4$fR!{%YZ!R!r9pas$ttm9E{Tu7iM1t)V$~mBv}Ka>YzD5eoilE#3PXdavEj!z<1>9uoqY<Ll&SUk|cluOg<75k?$wb4^y+7v=_rtpG#Npb~BNc_#tMCEcoUwHZ<UlQv?`bo-*x7X_z*`t}j-rD#bjpu@<d%dg`s6(-pugrt*h80*JDk1%fd^6cA)!TLhvOiin8tZz+(mWMQekL%*<IQbBwo4QakmK^pe$aXnoGY8?xm>Y6^G(L|(g39vQMsa*5qCEz_p;-SwCY)#B$Sp#Mttqf?neSyO$fl6t_E&q6+p3h297$QiVs6MkQbub<_4YvgVAuy-}Sx1E`z@ls)@BK9mojU4=v`|^rR7QtY2VoD`e1|5eFb)SMQ}2}i`Na1c1r(^@Qhl|xk*+`74}?H9ERU1`z-P^whBM9^LILCaw?{3Y5(z1#P$>2<K4*qETu9)Pl_!_9|1l5q5U_{;#oUoj6eJVqEkM8
"""

payload_clean = "".join(PAYLOAD.split()).encode("ascii")
source = bz2.decompress(base64.b85decode(payload_clean))

actual_sha256 = hashlib.sha256(source).hexdigest()

print("=" * 120)
print("STAGE28-AUTO-B — EMBEDDED SOURCE VERIFICATION")
print("=" * 120)
print()
print("Expected SHA256:", EXPECTED_SHA256)
print("Actual SHA256  :", actual_sha256)

if actual_sha256 != EXPECTED_SHA256:
    raise RuntimeError(
        "AUTO-B embedded payload SHA256 mismatch. "
        "DO NOT RUN SCIENCE."
    )

bot_path = Path(
    "/kaggle/working/stage28_auto_b.py"
)

bot_path.write_bytes(source)

print()
print("[PASS] AUTO-B source reconstructed exactly")
print("Path:", bot_path)
print("[PASS] beginning Stage28-AUTO-B")
print()

exec(
    compile(
        source,
        str(bot_path),
        "exec",
    ),
    {
        "__name__": "__main__",
        "__file__": str(bot_path),
    },
)

STAGE28-AUTO-B — EMBEDDED SOURCE VERIFICATION

Expected SHA256: be49483668ecc147490fc7430cb410108c4f6ae04f22b83861c19323a49af1fb
Actual SHA256  : be49483668ecc147490fc7430cb410108c4f6ae04f22b83861c19323a49af1fb

[PASS] AUTO-B source reconstructed exactly
Path: /kaggle/working/stage28_auto_b.py
[PASS] beginning Stage28-AUTO-B


STAGE28-AUTO-B — REPOSITORY / RESUME GATE

[PASS] GitHub credential: kaggle_secret:GITHUB_TOKEN
[PASS] token intentionally not displayed
Current durable HEAD: 4757827c9e845a862113f96339ada0b9e95a948f
[PASS] synchronized clean repository
[PASS] Stage22 new-fit ledger = 18 consumed / 90 remaining

STAGE28-AUTO-B — FROZEN PROTOCOL GATES

[PASS] 40 new Stage27 chronology components exact
[PASS] 10 historical seed42 reuse components exact
[PASS] LOAO thresholds / targets / metrics frozen

STAGE28-AUTO-B — MODEL RUNTIME GATE

XGBoost : 3.2.0
LightGBM: 4.6.0
[PASS] CPU-only Stage28 execution policy active

STAGE28-AUTO-B — SEED42 REUSE MODEL AUDIT

[PASS] C021 BOT XGBOO

traffic_labels/Monday-WorkingHours.pcap_(…):   0%|          | 0.00/65.5M [00:00<?, ?B/s]

[PASS source] 0 Monday /kaggle/working/stage28_auto_b_cicids2017_runtime/hf_download/traffic_labels/Monday-WorkingHours.pcap_ISCX.csv.parquet
[SOURCE] Tuesday-WorkingHours.pcap_ISCX.csv.parquet not found locally; attempting frozen Hugging Face revision...


traffic_labels/Tuesday-WorkingHours.pcap(…):   0%|          | 0.00/52.7M [00:00<?, ?B/s]

[PASS source] 1 Tuesday /kaggle/working/stage28_auto_b_cicids2017_runtime/hf_download/traffic_labels/Tuesday-WorkingHours.pcap_ISCX.csv.parquet
[SOURCE] Wednesday-workingHours.pcap_ISCX.csv.parquet not found locally; attempting frozen Hugging Face revision...


...

traffic_labels/Friday-WorkingHours-After(…):   0%|          | 0.00/18.6M [00:00<?, ?B/s]

[PASS source] 6 Friday /kaggle/working/stage28_auto_b_cicids2017_runtime/hf_download/traffic_labels/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
[SOURCE] Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet not found locally; attempting frozen Hugging Face revision...


traffic_labels/Friday-WorkingHours-Morni(…):   0%|          | 0.00/22.0M [00:00<?, ?B/s]

[PASS source] 7 Friday /kaggle/working/stage28_auto_b_cicids2017_runtime/hf_download/traffic_labels/Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet
[PASS feature source] 0 Monday rows=529,918 2368f1eca6adcff51a35993a16bb4cfba79387c3d6c25be87da1db518de55f34
[PASS feature source] 1 Tuesday rows=445,909 a6e63b785301f775939c91c9af729f487d8a39c8ed2f3554659c8d57389a1359
[PASS feature source] 2 Wednesday rows=692,703 ce7cc23ab53267187b40db6f598befae6763828fbde0c3245c2fe2a1f2ef892f
[PASS feature source] 3 Thursday rows=288,602 fbd22a9bcfa3f56eb0ee929d3afbc7f81fb9241e5f9d1fb221ffd13e954351ae
[PASS feature source] 4 Thursday rows=170,366 e3de3d8284e616960af5f1ce529d50d2b4a7fc4c9f00261cff131d834cdb22dd
[PASS feature source] 5 Friday rows=225,745 c9ad2ac60e98b0a73a8dacbd2cba28126fa4cf1a1b5de3b31173a9335ab5523c
[PASS feature source] 6 Friday rows=286,467 8fccc288b07d2c896231053a46a8750922abf8ff0303cd0a1ca975328b297aeb
[PASS feature source] 7 Friday rows=191,033 e1e67d4174b09e05cf1890424b237cbdba6

RuntimeError: Unexpected untracked universe.
Expected:
results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a11_bot_xgboost_seed43/bot_xgboost_seed43_cpu_model.json
results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a11_bot_xgboost_seed43/bot_xgboost_seed43_known_validation_prob_float32.npy
results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a11_bot_xgboost_seed43/bot_xgboost_seed43_operational_target_prob_float32.npy
results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a11_bot_xgboost_seed43/bot_xgboost_seed43_validation_threshold_grid.csv
results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a11_bot_xgboost_seed43/checksums.sha256
results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a11_bot_xgboost_seed43/stage28_2a11_bot_xgboost_seed43_execution_progress.json
results/stage28_stability_novelty

In [ ]:
# =================================================================================================
# STAGE28-AUTO-B-R1
# RECOVER FIT #19 -> FORCE-ADD IGNORED .NPY -> PUSH -> PATCH BOT -> RESUME FIT #20..#58
#
# IMPORTANT:
#   FIT #19 ALREADY SUCCEEDED.
#   THIS CELL MUST NOT REFIT C023.
#
# Root cause:
#   repository .gitignore contains "*.npy"
#   AUTO-B correctly wrote + checksummed the probability arrays,
#   but "git ls-files --others --exclude-standard" intentionally hid them.
#
# Scientific accounting before repair:
#   durable new fits       = 18
#   FIT #19 C023           = SUCCESS
#   scientific consumed    = 19
#   FIT #20                = NOT STARTED
#
# This repair performs ZERO new fits before making FIT #19 durable.
# After repair it executes the patched AUTO-B, which resumes at FIT #20.
# =================================================================================================

from __future__ import annotations

import base64
import hashlib
import json
import os
import subprocess
from pathlib import Path


SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

BOT_PATH = Path(
    "/kaggle/working/stage28_auto_b.py"
)

EXPECTED_PARENT = (
    "4757827c9e845a862113f96339ada0b9e95a948f"
)

OUT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_2a_stage27_seed_stability"
    / "stage28_2a11_bot_xgboost_seed43"
)

COMMIT_MESSAGE = (
    "stage28-2a11: execute bot xgboost seed43 checkpoint"
)

EXPECTED_MODEL_SHA = (
    "3eda9722dc50b8056b9e1a951513d0b6153913c155f91aec02b8cae1006112a9"
)

EXPECTED_NAMES = {
    "bot_xgboost_seed43_cpu_model.json",
    "bot_xgboost_seed43_known_validation_prob_float32.npy",
    "bot_xgboost_seed43_operational_target_prob_float32.npy",
    "bot_xgboost_seed43_validation_threshold_grid.csv",
    "checksums.sha256",
    "stage28_2a11_bot_xgboost_seed43_execution_progress.json",
    "stage28_2a11_bot_xgboost_seed43_fit_ledger.json",
    "stage28_2a11_bot_xgboost_seed43_result.json",
}

IGNORED_NPY_NAMES = {
    "bot_xgboost_seed43_known_validation_prob_float32.npy",
    "bot_xgboost_seed43_operational_target_prob_float32.npy",
}


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(title):
    print()
    print(SEP)
    print(title)
    print(SEP)
    print()


def run(cmd, *, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            f"{' '.join(map(str, cmd))}\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return p


def git(*args, check=True):
    return (
        run(
            ["git", *args],
            check=check,
        ).stdout
        or ""
    ).strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(
                16 * 1024 * 1024
            )
            if not block:
                break
            h.update(block)

    return h.hexdigest()


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def recover_token():
    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]

    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()

        for label in labels:
            try:
                value = client.get_secret(
                    label
                )
            except Exception:
                value = None

            if isinstance(value, str) and value.strip():
                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass

    for label in labels:
        value = os.environ.get(
            label
        )

        if isinstance(value, str) and value.strip():
            return (
                value.strip(),
                f"environment:{label}",
            )

    raise RuntimeError(
        "GitHub credential unavailable."
    )


def authenticated_push(token):
    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode("utf-8")
    ).decode("ascii")

    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            "Git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return (
        p.stdout
        + p.stderr
    ).strip()


def remote_head():
    text = git(
        "ls-remote",
        "origin",
        "refs/heads/main",
    )

    if not text:
        raise RuntimeError(
            "Unable to resolve remote main."
        )

    return text.split()[0]


# =================================================================================================
# 1. EXACT FAILED-CHECKPOINT STATE
# =================================================================================================

banner(
    "STAGE28-AUTO-B-R1 — PRE-REPAIR GATE"
)


if not REPO.is_dir():
    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )

if not BOT_PATH.is_file():
    raise RuntimeError(
        "Original AUTO-B source is no longer present.\n"
        "Do NOT rerun anything yet."
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


head = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote = remote_head()


print(
    "Expected durable parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD             :",
    head,
)

print(
    "origin/main            :",
    origin,
)

print(
    "Remote main            :",
    remote,
)


if not (
    head
    == origin
    == remote
    == EXPECTED_PARENT
):
    raise RuntimeError(
        "Unexpected repository state. "
        "Do not recover FIT #19 on a different parent."
    )


tracked_modifications = set(
    git(
        "diff",
        "--name-only",
    ).splitlines()
)

staged_before = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)


if tracked_modifications:
    raise RuntimeError(
        "Unexpected tracked modifications:\n"
        + "\n".join(
            sorted(
                tracked_modifications
            )
        )
    )


if staged_before:
    raise RuntimeError(
        "Unexpected staged content before repair."
    )


print()
print(
    "[PASS] durable parent still Stage28-2A10"
)

print(
    "[PASS] no commit occurred after FIT #19"
)


# =================================================================================================
# 2. PROVE THE TWO .NPY FILES EXIST
# =================================================================================================

banner(
    "FIT #19 FILESYSTEM ARTIFACT GATE"
)


if not OUT.is_dir():
    raise RuntimeError(
        f"Stage28-2A11 directory missing:\n{OUT}\n\n"
        "Runtime state was likely reset. Stop here."
    )


actual_names = {
    p.name
    for p in OUT.iterdir()
    if p.is_file()
}


print(
    "Expected files:",
    len(
        EXPECTED_NAMES
    ),
)

print(
    "Filesystem files:",
    len(
        actual_names
    ),
)


if actual_names != EXPECTED_NAMES:
    raise RuntimeError(
        "FIT #19 filesystem artifact universe is not exact.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                EXPECTED_NAMES
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                actual_names
            )
        )
    )


for name in sorted(
    EXPECTED_NAMES
):
    size = (
        OUT
        / name
    ).stat().st_size

    print(
        "[PASS FILE]",
        f"{name:<75}",
        f"{size:,} bytes",
    )


print()
print(
    "[PASS] both ignored probability .npy files physically exist"
)


# =================================================================================================
# 3. CHECKSUM MANIFEST — PROVE SCIENCE ALREADY COMPLETED
# =================================================================================================

banner(
    "FIT #19 CHECKSUM GATE"
)


checksums_path = (
    OUT
    / "checksums.sha256"
)


lines = (
    checksums_path
    .read_text(
        encoding="utf-8"
    )
    .splitlines()
)


if len(lines) != 7:
    raise RuntimeError(
        "FIT #19 checksum manifest must contain exactly seven payload entries."
    )


manifest_names = set()


for line in lines:
    digest, filename = line.split(
        None,
        1,
    )

    filename = (
        filename.strip()
        .lstrip("*")
    )

    manifest_names.add(
        filename
    )

    path = (
        OUT
        / filename
    )

    if not path.is_file():
        raise RuntimeError(
            f"Checksum payload missing: {filename}"
        )

    actual = sha256_file(
        path
    )

    if actual != digest:
        raise RuntimeError(
            f"Checksum mismatch: {filename}"
        )

    print(
        "[PASS]",
        filename,
        actual,
    )


if manifest_names != (
    EXPECTED_NAMES
    - {
        "checksums.sha256"
    }
):
    raise RuntimeError(
        "Checksum manifest payload universe changed."
    )


print()
print(
    "[PASS] all FIT #19 scientific artifacts byte-verified"
)


# =================================================================================================
# 4. SCIENTIFIC RECEIPT / LEDGER GATE
# =================================================================================================

banner(
    "FIT #19 SCIENTIFIC ACCOUNTING GATE"
)


ledger = read_json(
    OUT
    / "stage28_2a11_bot_xgboost_seed43_fit_ledger.json"
)

progress = read_json(
    OUT
    / "stage28_2a11_bot_xgboost_seed43_execution_progress.json"
)

result = read_json(
    OUT
    / "stage28_2a11_bot_xgboost_seed43_result.json"
)


if ledger[
    "status"
] != "FIT_019_SUCCESSFULLY_CONSUMED":
    raise RuntimeError(
        "FIT #19 ledger status invalid."
    )


if int(
    ledger[
        "pre_component_new_fits_consumed"
    ]
) != 18:
    raise RuntimeError(
        "FIT #19 predecessor count must be 18."
    )


if int(
    ledger[
        "this_component_successful_new_fits"
    ]
) != 1:
    raise RuntimeError(
        "FIT #19 component accounting invalid."
    )


if int(
    ledger[
        "cumulative_new_fits_consumed"
    ]
) != 19:
    raise RuntimeError(
        "FIT #19 cumulative ledger must equal 19."
    )


if int(
    ledger[
        "new_fits_remaining"
    ]
) != 89:
    raise RuntimeError(
        "FIT #19 remaining ledger must equal 89."
    )


if progress[
    "state"
] != "FIT_VALIDATION_TARGET_AND_ARTIFACTS_SUCCESS":
    raise RuntimeError(
        "FIT #19 did not finish validation/target/artifact processing."
    )


if progress[
    "component_id"
] != "C023":
    raise RuntimeError(
        "Unexpected component identity."
    )


if int(
    progress[
        "fit_ordinal"
    ]
) != 19:
    raise RuntimeError(
        "Unexpected fit ordinal."
    )


if (
    progress[
        "held_out_family"
    ]
    != "BOT"
):
    raise RuntimeError(
        "Unexpected held-out family."
    )


if (
    progress[
        "learner"
    ]
    != "XGBOOST"
):
    raise RuntimeError(
        "Unexpected learner."
    )


if int(
    progress[
        "seed"
    ]
) != 43:
    raise RuntimeError(
        "Unexpected training seed."
    )


if (
    result[
        "component_id"
    ]
    != "C023"
):
    raise RuntimeError(
        "Result component mismatch."
    )


if (
    result[
        "held_out_family"
    ]
    != "BOT"
):
    raise RuntimeError(
        "Result family mismatch."
    )


if (
    result[
        "learner"
    ]
    != "XGBOOST"
):
    raise RuntimeError(
        "Result learner mismatch."
    )


if int(
    result[
        "training_seed"
    ]
) != 43:
    raise RuntimeError(
        "Result seed mismatch."
    )


model_sha = result[
    "model"
][
    "model_sha256"
]


if model_sha != EXPECTED_MODEL_SHA:
    raise RuntimeError(
        "FIT #19 model SHA mismatch."
    )


if sha256_file(
    OUT
    / result[
        "model"
    ][
        "model_artifact"
    ]
) != EXPECTED_MODEL_SHA:
    raise RuntimeError(
        "FIT #19 model bytes changed."
    )


if (
    result[
        "membership"
    ][
        "held_out_family_train_count"
    ]
    != 0
):
    raise RuntimeError(
        "Held-out BOT appeared in train."
    )


if (
    result[
        "membership"
    ][
        "held_out_family_validation_count"
    ]
    != 0
):
    raise RuntimeError(
        "Held-out BOT appeared in validation."
    )


anti = result[
    "anti_adaptation"
]


for key in [
    "target_used_for_fit",
    "target_used_for_class_weight",
    "target_used_for_threshold_selection",
    "target_threshold_search",
    "post_target_parameter_change",
    "post_target_fit_branching",
]:
    if anti[
        key
    ] is not False:
        raise RuntimeError(
            f"Anti-adaptation violation: {key}"
        )


print(
    "[PASS] C023 = BOT / XGBOOST / seed43"
)

print(
    "[PASS] FIT #19 accounted exactly once"
)

print(
    "[PASS] model SHA:",
    EXPECTED_MODEL_SHA,
)

print(
    "[PASS] held-out BOT train count = 0"
)

print(
    "[PASS] held-out BOT validation count = 0"
)

print(
    "[PASS] no target-adaptive choice"
)

print(
    "[PASS] FIT #20 has not started"
)


# =================================================================================================
# 5. CONFIRM ROOT CAUSE
# =================================================================================================

banner(
    "ROOT-CAUSE CONFIRMATION"
)


ignore_checks = {}


for name in sorted(
    IGNORED_NPY_NAMES
):
    rel = str(
        (
            OUT
            / name
        ).relative_to(
            REPO
        )
    )

    p = run(
        [
            "git",
            "check-ignore",
            "-v",
            rel,
        ],
        check=False,
    )

    ignore_checks[
        name
    ] = p.returncode == 0

    print(
        name,
        "->",
        (
            p.stdout.strip()
            if p.stdout.strip()
            else "NOT IGNORED"
        ),
    )


if not all(
    ignore_checks.values()
):
    raise RuntimeError(
        "The expected *.npy ignore rule was not reproduced."
    )


print()
print(
    "[PASS] root cause confirmed: *.npy is ignored by Git"
)


# =================================================================================================
# 6. EXACT PRE-COMMIT UNTRACKED STATE
# =================================================================================================

banner(
    "PRE-COMMIT UNTRACKED GATE"
)


expected_rel = {
    str(
        (
            OUT
            / name
        ).relative_to(
            REPO
        )
    )
    for name in EXPECTED_NAMES
}


ignored_npy_rel = {
    str(
        (
            OUT
            / name
        ).relative_to(
            REPO
        )
    )
    for name in IGNORED_NPY_NAMES
}


expected_visible_untracked = (
    expected_rel
    - ignored_npy_rel
)


visible_untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
)


if visible_untracked != expected_visible_untracked:
    raise RuntimeError(
        "Unexpected visible untracked universe before repair.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                expected_visible_untracked
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                visible_untracked
            )
        )
    )


print(
    "[PASS] six normal artifacts visible to Git"
)

print(
    "[PASS] two probability .npy artifacts hidden only by .gitignore"
)


# =================================================================================================
# 7. FORCE-STAGE EXACT EIGHT FILES
# =================================================================================================

banner(
    "FORCE-STAGE FIT #19 ARTIFACTS"
)


for rel in sorted(
    expected_rel
):
    run(
        [
            "git",
            "add",
            "-f",
            "--",
            rel,
        ]
    )


staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)


if staged != expected_rel:
    raise RuntimeError(
        "Staged universe mismatch.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                expected_rel
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                staged
            )
        )
    )


print(
    "[PASS] exactly eight FIT #19 files staged"
)

print(
    "[PASS] ignored .npy files included via git add -f"
)


# =================================================================================================
# 8. COMMIT + PUSH FIT #19
# =================================================================================================

banner(
    "MAKE FIT #19 DURABLE"
)


run(
    [
        "git",
        "config",
        "user.name",
        "Stage28 Kaggle",
    ]
)

run(
    [
        "git",
        "config",
        "user.email",
        "stage28-kaggle@users.noreply.github.com",
    ]
)


commit_result = run(
    [
        "git",
        "commit",
        "-m",
        COMMIT_MESSAGE,
    ]
)


print(
    commit_result.stdout.strip()
)


new_head = git(
    "rev-parse",
    "HEAD",
)

new_parent = git(
    "rev-parse",
    "HEAD^",
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)


if new_parent != EXPECTED_PARENT:
    raise RuntimeError(
        "FIT #19 recovery commit parent mismatch."
    )


if subject != COMMIT_MESSAGE:
    raise RuntimeError(
        "FIT #19 recovery commit subject mismatch."
    )


token, token_source = recover_token()


print()
print(
    "[PASS] credential recovered from",
    token_source,
)

print(
    "[PASS] token not displayed"
)


push_output = authenticated_push(
    token
)


if push_output:
    print(
        push_output
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote = remote_head()


if not (
    local
    == origin
    == remote
    == new_head
):
    raise RuntimeError(
        "FIT #19 remote durability verification failed."
    )


if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Repository dirty after FIT #19 recovery push."
    )


print()
print(
    "[DURABLE] FIT #19 / C023"
)

print(
    "commit:",
    new_head,
)

print(
    "consumed : 19 / 108"
)

print(
    "remaining: 89"
)


# =================================================================================================
# 9. PATCH AUTO-B GIT CHECKPOINT LOGIC
# =================================================================================================

banner(
    "PATCH AUTO-B — IGNORED .NPY SAFE COMMIT"
)


source = BOT_PATH.read_text(
    encoding="utf-8"
)


old_commit_block = '''    untracked = set(git("ls-files", "--others", "--exclude-standard").splitlines())
    if tracked:
        raise RuntimeError("Unexpected tracked modifications:\\n" + "\\n".join(sorted(tracked)))
    if staged:
        raise RuntimeError("Unexpected staged files before AUTO-B commit.")
    if untracked != expected_rel:
        raise RuntimeError(
            "Unexpected untracked universe.\\nExpected:\\n" + "\\n".join(sorted(expected_rel))
            + "\\nActual:\\n" + "\\n".join(sorted(untracked))
        )
    for rel in sorted(expected_rel):
        sp(["git", "add", "--", rel], cwd=REPO)
'''


new_commit_block = '''    untracked = set(git("ls-files", "--others", "--exclude-standard").splitlines())
    if tracked:
        raise RuntimeError("Unexpected tracked modifications:\\n" + "\\n".join(sorted(tracked)))
    if staged:
        raise RuntimeError("Unexpected staged files before AUTO-B commit.")

    # Some authorized scientific artifacts are .npy and the repository
    # intentionally ignores *.npy. Therefore exclude-standard cannot be
    # used to demand equality with the full artifact universe.
    unexpected_untracked = untracked - expected_rel
    if unexpected_untracked:
        raise RuntimeError(
            "Unexpected non-ignored untracked files.\\n"
            + "\\n".join(sorted(unexpected_untracked))
        )

    missing_files = [
        rel for rel in sorted(expected_rel)
        if not (REPO / rel).is_file()
    ]
    if missing_files:
        raise RuntimeError(
            "Authorized scientific artifact missing from filesystem:\\n"
            + "\\n".join(missing_files)
        )

    # Force-add is required only because *.npy is intentionally ignored.
    # This changes no scientific artifact; it only makes already-frozen
    # probability files durable.
    for rel in sorted(expected_rel):
        sp(["git", "add", "-f", "--", rel], cwd=REPO)

    staged_after = set(git("diff", "--cached", "--name-only").splitlines())
    if staged_after != expected_rel:
        raise RuntimeError(
            "AUTO-B staged universe mismatch after force-add.\\nExpected:\\n"
            + "\\n".join(sorted(expected_rel))
            + "\\nActual:\\n"
            + "\\n".join(sorted(staged_after))
        )
'''


if old_commit_block not in source:
    raise RuntimeError(
        "Could not find exact AUTO-B commit block to patch."
    )


source = source.replace(
    old_commit_block,
    new_commit_block,
    1,
)


# =================================================================================================
# 10. PATCH RESUME LOGIC — DIRECTORY != DURABLE COMMIT
# =================================================================================================

old_resume_block = '''    if out.exists():
        if missing_seen:
            raise RuntimeError("AUTO-B durable history is non-contiguous: later job exists after missing earlier job.")
        if not (ledger_path.is_file() and result_path.is_file() and checksums_path.is_file()):
            raise RuntimeError(f"Partial durable-looking AUTO-B directory: {out}")
        ledger = read_json(ledger_path)
        if int(ledger["cumulative_new_fits_consumed"]) != job["fit_ordinal"]:
            raise RuntimeError(f"{job['stage']}: durable ledger mismatch.")
        completed += 1
        print("[SKIP durable]", job["stage"], job["row"]["component_id"], "-> cumulative", job["fit_ordinal"])
    else:
        missing_seen = True
'''


new_resume_block = '''    if out.exists():
        if missing_seen:
            raise RuntimeError("AUTO-B durable history is non-contiguous: later job exists after missing earlier job.")
        if not (ledger_path.is_file() and result_path.is_file() and checksums_path.is_file()):
            raise RuntimeError(
                f"LOCAL PARTIAL AUTO-B CHECKPOINT: {out}\\n"
                "Do not refit this component automatically."
            )

        # A local directory is not sufficient evidence of durability.
        # Require its receipt files to be tracked in the current durable HEAD.
        rel_out = str(out.relative_to(REPO))
        tracked_under_out = set(
            git("ls-files", "--", rel_out).splitlines()
        )

        required_tracked = {
            str(ledger_path.relative_to(REPO)),
            str(result_path.relative_to(REPO)),
            str(checksums_path.relative_to(REPO)),
        }

        if not required_tracked.issubset(tracked_under_out):
            raise RuntimeError(
                f"LOCAL UNCOMMITTED AUTO-B CHECKPOINT: {out}\\n"
                "Scientific fit may already have been consumed; do not refit."
            )

        ledger = read_json(ledger_path)
        if int(ledger["cumulative_new_fits_consumed"]) != job["fit_ordinal"]:
            raise RuntimeError(f"{job['stage']}: durable ledger mismatch.")
        completed += 1
        print("[SKIP durable]", job["stage"], job["row"]["component_id"], "-> cumulative", job["fit_ordinal"])
    else:
        missing_seen = True
'''


if old_resume_block not in source:
    raise RuntimeError(
        "Could not find exact AUTO-B resume block to patch."
    )


source = source.replace(
    old_resume_block,
    new_resume_block,
    1,
)


BOT_PATH.write_text(
    source,
    encoding="utf-8",
)


patched_sha = sha256_file(
    BOT_PATH
)


print(
    "[PASS] AUTO-B Git checkpoint logic patched"
)

print(
    "[PASS] AUTO-B resume durability logic strengthened"
)

print(
    "Patched runtime source SHA256:",
    patched_sha,
)


# =================================================================================================
# 11. FINAL PRE-RESUME GATE
# =================================================================================================

banner(
    "STAGE28-AUTO-B-R1 — READY TO RESUME"
)


if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Repository must be clean before AUTO-B resume."
    )


if not (
    git(
        "rev-parse",
        "HEAD",
    )
    == git(
        "rev-parse",
        "origin/main",
    )
    == remote_head()
):
    raise RuntimeError(
        "Repository not synchronized before AUTO-B resume."
    )


print(
    "FIT #19 C023:",
    "DURABLE"
)

print(
    "Overall consumed:",
    "19 / 108"
)

print(
    "Overall remaining:",
    "89"
)

print(
    "Next authorized fit:",
    "FIT #20 — C024 — BOT — LIGHTGBM — seed43"
)

print(
    "Final-holdout optimization:",
    "0"
)

print()
print(
    "[PASS] launching patched AUTO-B now"
)

print()


# =================================================================================================
# 12. RESUME AUTO-B
# =================================================================================================

exec(
    compile(
        source,
        str(
            BOT_PATH
        ),
        "exec",
    ),
    {
        "__name__": "__main__",
        "__file__": str(
            BOT_PATH
        ),
    },
)


STAGE28-AUTO-B-R1 — PRE-REPAIR GATE

Expected durable parent: 4757827c9e845a862113f96339ada0b9e95a948f
Local HEAD             : 4757827c9e845a862113f96339ada0b9e95a948f
origin/main            : 4757827c9e845a862113f96339ada0b9e95a948f
Remote main            : 4757827c9e845a862113f96339ada0b9e95a948f

[PASS] durable parent still Stage28-2A10
[PASS] no commit occurred after FIT #19

FIT #19 FILESYSTEM ARTIFACT GATE

Expected files: 8
Filesystem files: 8
[PASS FILE] bot_xgboost_seed43_cpu_model.json                                           1,784,265 bytes
[PASS FILE] bot_xgboost_seed43_known_validation_prob_float32.npy                        1,836,000 bytes
[PASS FILE] bot_xgboost_seed43_operational_target_prob_float32.npy                      2,813,108 bytes
[PASS FILE] bot_xgboost_seed43_validation_threshold_grid.csv                            12,492 bytes
[PASS FILE] checksums.sha256                                                            801 bytes
[PASS FILE] stage28_2a11_bot_xgb

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut


Stage28-2A12 — AUTO COMMIT/PUSH

[main d02ca2a] stage28-2a12: execute bot lightgbm seed43 checkpoint
 8 files changed, 8299 insertions(+)
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a12_bot_lightgbm_seed43/bot_lightgbm_seed43_cpu_model.txt
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a12_bot_lightgbm_seed43/bot_lightgbm_seed43_known_validation_prob_float32.npy
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a12_bot_lightgbm_seed43/bot_lightgbm_seed43_operational_target_prob_float32.npy
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a12_bot_lightgbm_seed43/bot_lightgbm_seed43_validation_threshold_grid.csv
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a12_bot_lightgbm_seed43/checksums.sha256
 crea

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut


Stage28-2A14 — AUTO COMMIT/PUSH

[main 248b835] stage28-2a14: execute bot lightgbm seed44 checkpoint
 8 files changed, 8299 insertions(+)
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a14_bot_lightgbm_seed44/bot_lightgbm_seed44_cpu_model.txt
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a14_bot_lightgbm_seed44/bot_lightgbm_seed44_known_validation_prob_float32.npy
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a14_bot_lightgbm_seed44/bot_lightgbm_seed44_operational_target_prob_float32.npy
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a14_bot_lightgbm_seed44/bot_lightgbm_seed44_validation_threshold_grid.csv
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a14_bot_lightgbm_seed44/checksums.sha256
 crea

...


Stage28-2A40 — AUTO COMMIT/PUSH

[main 469687a] stage28-2a40: execute port_scan lightgbm seed45 checkpoint
 8 files changed, 8299 insertions(+)
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a40_port_scan_lightgbm_seed45/checksums.sha256
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a40_port_scan_lightgbm_seed45/port_scan_lightgbm_seed45_cpu_model.txt
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a40_port_scan_lightgbm_seed45/port_scan_lightgbm_seed45_known_validation_prob_float32.npy
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a40_port_scan_lightgbm_seed45/port_scan_lightgbm_seed45_operational_target_prob_float32.npy
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a40_port_scan_lightgbm_seed45/p

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut


Stage28-2A42 — AUTO COMMIT/PUSH

[main ffbf1e7] stage28-2a42: execute port_scan lightgbm seed46 checkpoint
 8 files changed, 8299 insertions(+)
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a42_port_scan_lightgbm_seed46/checksums.sha256
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a42_port_scan_lightgbm_seed46/port_scan_lightgbm_seed46_cpu_model.txt
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a42_port_scan_lightgbm_seed46/port_scan_lightgbm_seed46_known_validation_prob_float32.npy
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a42_port_scan_lightgbm_seed46/port_scan_lightgbm_seed46_operational_target_prob_float32.npy
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a42_port_scan_lightgbm_seed46/p

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut


Stage28-2A44 — AUTO COMMIT/PUSH

[main 6ebb0a3] stage28-2a44: execute web_attack lightgbm seed43 checkpoint
 8 files changed, 8289 insertions(+)
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a44_web_attack_lightgbm_seed43/checksums.sha256
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a44_web_attack_lightgbm_seed43/stage28_2a44_web_attack_lightgbm_seed43_execution_progress.json
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a44_web_attack_lightgbm_seed43/stage28_2a44_web_attack_lightgbm_seed43_fit_ledger.json
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a44_web_attack_lightgbm_seed43/stage28_2a44_web_attack_lightgbm_seed43_result.json
 create mode 100644 results/stage28_stability_novelty_control/stage28_2a_stage27_seed_stability/stage28_2a44_web_attack_l

In [1]:
# =================================================================================================
# STAGE28-COLD-BOOTSTRAP
# Restore exact durable GitHub state + frozen CICIDS2017 Hugging Face sources
#
# ZERO MODEL FITS
# ZERO INFERENCE
# ZERO THRESHOLD SELECTION
#
# Expected scientific state:
#   Stage28 NEW fits consumed = 58 / 108
#   remaining                 = 50
#   next component            = C071 / FIT #59
# =================================================================================================

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path


SEP = "=" * 120

REPO_URL = "https://github.com/themubasshir/ids2018-validation-safe-ablation.git"
REPO = Path("/kaggle/working/ids2018-validation-safe-ablation")

# Last verified durable AUTO-B commit.
EXPECTED_HEAD = "8b87f734f076c5402324ec9c7b2ee82e74f64d0e"

# Frozen Stage28 execution-manifest identity.
EXPECTED_MANIFEST_SHA256 = (
    "47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505"
)

# Frozen CICIDS2017 HF identity recovered from Stage27/AUTO-B.
HF_REPO = "bvsam/cic-ids-2017"
HF_REVISION = "b7e532345512edcd530cb1770dc76636aeb52802"

HF_DOWNLOAD_ROOT = Path("/kaggle/working/stage28_cold_bootstrap_hf")
SOURCE_ROOT = Path("/kaggle/working/stage27_cicids2017_sources")

STAGE28_ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

MANIFEST_PATH = (
    STAGE28_ROOT
    / "stage28_1b_random_loao_membership_and_execution_lock"
    / "stage28_component_execution_manifest.csv"
)

SOURCE_RECEIPT_PATH = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_1a_fold_membership"
    / "source_effective_population_receipt.json"
)

FINAL_AUTO_B_LEDGER = (
    STAGE28_ROOT
    / "stage28_2a_stage27_seed_stability"
    / "stage28_2a50_web_attack_lightgbm_seed46"
    / "stage28_2a50_web_attack_lightgbm_seed46_fit_ledger.json"
)

EFFECTIVE_BUDGET_PATH = (
    STAGE28_ROOT
    / "stage28_0a_preexecution_amendment"
    / "effective_fit_budget.json"
)


def banner(text):
    print("\n" + SEP)
    print(text)
    print(SEP + "\n")


def run(cmd, *, cwd=None, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=None if cwd is None else str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            + " ".join(str(x) for x in cmd)
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args, check=True):
    return (
        run(
            ["git", *args],
            cwd=REPO,
            check=check,
        ).stdout
        or ""
    ).strip()


def sha256_file(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)

    return h.hexdigest()


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


# =================================================================================================
# 0. ENVIRONMENT
# =================================================================================================

banner("STAGE28 COLD BOOTSTRAP — ENVIRONMENT")


print("Python:", sys.version.split()[0])


required_exact = {
    "numpy": "2.0.2",
    "scikit-learn": "1.6.1",
    "xgboost": "3.2.0",
    "lightgbm": "4.6.0",
}


for package, expected in required_exact.items():
    try:
        actual = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        actual = None

    print(
        f"{package:<15}",
        actual,
        "(expected", expected + ")",
    )


# We specifically know Stage28's frozen execution used these versions.
# Install only if XGBoost/LightGBM are absent or wrong.
install = []

for package in ["xgboost", "lightgbm"]:
    expected = required_exact[package]

    try:
        actual = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        actual = None

    if actual != expected:
        install.append(
            f"{package}=={expected}"
        )


try:
    importlib.metadata.version("huggingface-hub")
except importlib.metadata.PackageNotFoundError:
    install.append("huggingface_hub")


if install:
    print()
    print(
        "[ENV] Installing:",
        " ".join(install),
    )

    run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *install,
        ]
    )


# Fail closed for membership-sensitive packages.
for package in ["numpy", "scikit-learn"]:
    actual = importlib.metadata.version(package)
    expected = required_exact[package]

    if actual != expected:
        raise RuntimeError(
            f"{package} version mismatch: "
            f"{actual} != frozen {expected}"
        )


for package in ["xgboost", "lightgbm"]:
    actual = importlib.metadata.version(package)
    expected = required_exact[package]

    if actual != expected:
        raise RuntimeError(
            f"{package} version mismatch after setup: "
            f"{actual} != {expected}"
        )


print()
print("[PASS] frozen software versions available")


# =================================================================================================
# 1. GITHUB SECRET CHECK
# =================================================================================================

banner("GITHUB CREDENTIAL GATE")


github_token = None
github_source = None

try:
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()

    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]:
        try:
            value = client.get_secret(name)
        except Exception:
            value = None

        if isinstance(value, str) and value.strip():
            github_token = value.strip()
            github_source = f"kaggle_secret:{name}"
            break

except Exception:
    pass


if github_token is None:
    for name in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]:
        value = os.environ.get(name)

        if isinstance(value, str) and value.strip():
            github_token = value.strip()
            github_source = f"environment:{name}"
            break


if github_token is None:
    raise RuntimeError(
        "GitHub token not found. "
        "Stage28 requires the existing Kaggle GitHub secret for later pushes."
    )


print("[PASS] GitHub credential:", github_source)
print("[PASS] token not displayed")


# =================================================================================================
# 2. RESTORE GITHUB
# =================================================================================================

banner("RESTORE GITHUB REPOSITORY")


if REPO.exists():
    if not (REPO / ".git").is_dir():
        raise RuntimeError(
            f"{REPO} exists but is not a Git repository. "
            "Do not delete it blindly."
        )

    status = git(
        "status",
        "--porcelain",
    )

    if status:
        raise RuntimeError(
            "Existing repository contains local changes.\n"
            "Do not delete possible evidence.\n\n"
            + status
        )

    print(
        "[INFO] Existing clean repository found; fetching origin/main..."
    )

    run(
        [
            "git",
            "fetch",
            "--prune",
            "origin",
            "main",
        ],
        cwd=REPO,
    )

else:
    print(
        "[INFO] Fresh runtime — cloning GitHub repository..."
    )

    run(
        [
            "git",
            "clone",
            "--branch",
            "main",
            "--single-branch",
            REPO_URL,
            str(REPO),
        ]
    )


remote_head_text = run(
    [
        "git",
        "ls-remote",
        "origin",
        "refs/heads/main",
    ],
    cwd=REPO,
).stdout.strip()

if not remote_head_text:
    raise RuntimeError(
        "Could not resolve GitHub origin/main."
    )


remote_head = remote_head_text.split()[0]

print("Remote main:", remote_head)
print("Expected   :", EXPECTED_HEAD)


if remote_head != EXPECTED_HEAD:
    raise RuntimeError(
        "GitHub main has changed from the last verified Stage28-AUTO-B head.\n"
        "STOP — inspect the newer remote state before starting any fit."
    )


# Force the clean fresh runtime to the exact durable scientific head.
run(
    [
        "git",
        "checkout",
        "main",
    ],
    cwd=REPO,
)

run(
    [
        "git",
        "reset",
        "--hard",
        EXPECTED_HEAD,
    ],
    cwd=REPO,
)


head = git(
    "rev-parse",
    "HEAD",
)

status = git(
    "status",
    "--porcelain",
)


if head != EXPECTED_HEAD:
    raise RuntimeError(
        "Local Git HEAD mismatch."
    )


if status:
    raise RuntimeError(
        "Repository is not clean after bootstrap."
    )


print()
print("[PASS] repository restored")
print("HEAD:", head)
print("[PASS] repository clean")


# =================================================================================================
# 3. DURABLE STAGE28 ACCOUNTING
# =================================================================================================

banner("DURABLE STAGE28 LEDGER GATE")


if not FINAL_AUTO_B_LEDGER.is_file():
    raise RuntimeError(
        "Final AUTO-B ledger not present at durable HEAD."
    )


ledger = read_json(
    FINAL_AUTO_B_LEDGER
)


expected_ledger = {
    "component_id": "C070",
    "cumulative_new_fits_consumed": 58,
    "new_fits_remaining": 50,
    "stage28a_stage27_new_fits_consumed": 40,
    "stage28a_stage27_new_fits_remaining": 0,
    "status": "FIT_058_SUCCESSFULLY_CONSUMED",
}


for key, expected in expected_ledger.items():
    actual = ledger.get(key)

    if actual != expected:
        raise RuntimeError(
            f"Durable ledger mismatch for {key}: "
            f"{actual!r} != {expected!r}"
        )


budget = read_json(
    EFFECTIVE_BUDGET_PATH
)


if (
    int(
        budget[
            "totals"
        ][
            "new_fit_budget"
        ]
    )
    != 108
):
    raise RuntimeError(
        "Frozen Stage28 fit budget is not 108."
    )


if (
    int(
        budget[
            "stage28b_random_loao"
        ][
            "new_fits"
        ]
    )
    != 50
):
    raise RuntimeError(
        "Frozen Stage28B fit budget is not 50."
    )


print(
    "[PASS] Stage28A durable:"
)

print(
    "       consumed = 58 / 108"
)

print(
    "       remaining = 50"
)

print(
    "       next fit = FIT #59 / C071"
)


# =================================================================================================
# 4. EXECUTION MANIFEST GATE
# =================================================================================================

banner("FROZEN EXECUTION MANIFEST GATE")


manifest_sha = sha256_file(
    MANIFEST_PATH
)


print(
    "Manifest SHA256:",
    manifest_sha,
)


if manifest_sha != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError(
        "Stage28 execution manifest identity mismatch."
    )


manifest_text = MANIFEST_PATH.read_text(
    encoding="utf-8"
)


if "C071,28B_RANDOM_SPLIT_LOAO_CONTROL" not in manifest_text:
    raise RuntimeError(
        "C071 Stage28B component missing."
    )


if "C120,28B_RANDOM_SPLIT_LOAO_CONTROL" not in manifest_text:
    raise RuntimeError(
        "C120 Stage28B component missing."
    )


print(
    "[PASS] exact 120-component execution manifest"
)

print(
    "[PASS] Stage28B C071..C120 present"
)


# =================================================================================================
# 5. FROZEN HF SOURCE RECEIPT
# =================================================================================================

banner("FROZEN CICIDS2017 SOURCE RECEIPT")


source_receipt = read_json(
    SOURCE_RECEIPT_PATH
)

segments = sorted(
    source_receipt[
        "segments"
    ],
    key=lambda x: int(
        x[
            "source_index"
        ]
    ),
)


if len(segments) != 8:
    raise RuntimeError(
        "Expected exactly eight frozen CICIDS2017 sources."
    )


if int(
    source_receipt[
        "population"
    ][
        "effective_rows"
    ]
) != 2_830_743:
    raise RuntimeError(
        "Frozen CICIDS2017 effective population mismatch."
    )


print(
    "[PASS] source receipt loaded"
)

print(
    "Effective population:",
    "2,830,743 rows",
)

print(
    "HF dataset:",
    HF_REPO,
)

print(
    "HF revision:",
    HF_REVISION,
)


# =================================================================================================
# 6. OPTIONAL HF SECRET
# =================================================================================================

hf_token = None

try:
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()

    for name in [
        "HF_TOKEN",
        "HUGGINGFACE_TOKEN",
        "HUGGING_FACE_HUB_TOKEN",
    ]:
        try:
            value = client.get_secret(name)
        except Exception:
            value = None

        if isinstance(value, str) and value.strip():
            hf_token = value.strip()
            print(
                "[PASS] Hugging Face credential available:",
                name,
            )
            print(
                "[PASS] HF token not displayed"
            )
            break

except Exception:
    pass


if hf_token is None:
    print(
        "[INFO] No HF token found; public dataset download will be used."
    )


# =================================================================================================
# 7. DOWNLOAD / VERIFY ALL EIGHT SOURCES
# =================================================================================================

banner("RESTORE FROZEN HUGGING FACE SOURCES")


from huggingface_hub import hf_hub_download


HF_DOWNLOAD_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

SOURCE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


for seg in segments:

    idx = int(
        seg[
            "source_index"
        ]
    )

    remote = seg[
        "remote"
    ]

    basename = seg[
        "basename"
    ]

    expected_size = int(
        seg[
            "size_bytes"
        ]
    )

    expected_sha = seg[
        "sha256"
    ]

    canonical_path = (
        SOURCE_ROOT
        / basename
    )


    # Reuse only an exact existing file.
    if canonical_path.is_file():
        size_ok = (
            canonical_path.stat().st_size
            == expected_size
        )

        sha_ok = (
            sha256_file(
                canonical_path
            )
            == expected_sha
        )

        if size_ok and sha_ok:
            print(
                f"[PASS cached] {idx} "
                f"{seg['day']:<9} "
                f"{basename}"
            )
            continue

        raise RuntimeError(
            f"Existing frozen source has wrong identity:\n"
            f"{canonical_path}"
        )


    print()
    print(
        f"[DOWNLOAD {idx + 1}/8]",
        remote,
    )


    downloaded = Path(
        hf_hub_download(
            repo_id=HF_REPO,
            filename=remote,
            repo_type="dataset",
            revision=HF_REVISION,
            local_dir=str(
                HF_DOWNLOAD_ROOT
            ),
            token=hf_token,
        )
    )


    actual_size = (
        downloaded.stat().st_size
    )

    if actual_size != expected_size:
        raise RuntimeError(
            f"HF size mismatch for {basename}:\n"
            f"{actual_size} != {expected_size}"
        )


    actual_sha = sha256_file(
        downloaded
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            f"HF SHA256 mismatch for {basename}:\n"
            f"{actual_sha}\n!=\n{expected_sha}"
        )


    # Create the historical runtime source layout expected by Stage27/28.
    try:
        canonical_path.symlink_to(
            downloaded.resolve()
        )
    except Exception:
        shutil.copy2(
            downloaded,
            canonical_path,
        )


    if (
        canonical_path.stat().st_size
        != expected_size
    ):
        raise RuntimeError(
            f"Canonical source size mismatch: {basename}"
        )


    if (
        sha256_file(
            canonical_path
        )
        != expected_sha
    ):
        raise RuntimeError(
            f"Canonical source SHA mismatch: {basename}"
        )


    print(
        "[PASS source]",
        idx,
        seg[
            "day"
        ],
        basename,
    )

    print(
        "   bytes :",
        f"{expected_size:,}",
    )

    print(
        "   sha256:",
        expected_sha,
    )


# =================================================================================================
# 8. FINAL SOURCE UNIVERSE
# =================================================================================================

banner("FINAL SOURCE IDENTITY VERIFICATION")


total_bytes = 0


for seg in segments:

    path = (
        SOURCE_ROOT
        / seg[
            "basename"
        ]
    )

    if not path.is_file():
        raise RuntimeError(
            f"Source missing after bootstrap: {path}"
        )


    expected_size = int(
        seg[
            "size_bytes"
        ]
    )

    expected_sha = seg[
        "sha256"
    ]


    if (
        path.stat().st_size
        != expected_size
    ):
        raise RuntimeError(
            f"Final size mismatch: {path.name}"
        )


    actual_sha = sha256_file(
        path
    )


    if actual_sha != expected_sha:
        raise RuntimeError(
            f"Final SHA mismatch: {path.name}"
        )


    total_bytes += expected_size


    print(
        "[PASS]",
        seg[
            "source_index"
        ],
        path.name,
    )


print()
print(
    "Frozen source files:",
    len(
        segments
    ),
)

print(
    "Total source bytes:",
    f"{total_bytes:,}",
)


# =================================================================================================
# 9. SCIENCE-SAFETY FINAL GATE
# =================================================================================================

banner("STAGE28-COLD-BOOTSTRAP COMPLETE")


print(
    "GitHub durable HEAD:"
)

print(
    " ",
    EXPECTED_HEAD,
)

print()
print(
    "Stage28 NEW fits consumed:",
    "58 / 108",
)

print(
    "Stage28 NEW fits remaining:",
    "50",
)

print(
    "Next authorized component:",
    "C071",
)

print(
    "Next authorized scientific fit:",
    "FIT #59",
)

print()
print(
    "Hugging Face dataset:",
    HF_REPO,
)

print(
    "Frozen revision:",
    HF_REVISION,
)

print(
    "Frozen CICIDS2017 sources:",
    "8 / 8 VERIFIED",
)

print()
print(
    "Scientific operations performed by this cell:"
)

print(
    "  model fits          : 0"
)

print(
    "  model inference     : 0"
)

print(
    "  threshold selection : 0"
)

print(
    "  target optimization : 0"
)

print()
print(
    "[PASS] COLD BOOTSTRAP COMPLETE"
)

print(
    "[READY] Stage28-AUTO-C may now start from FIT #59 / C071"
)


STAGE28 COLD BOOTSTRAP — ENVIRONMENT

Python: 3.12.13
numpy           2.0.2 (expected 2.0.2)
scikit-learn    1.6.1 (expected 1.6.1)
xgboost         3.2.0 (expected 3.2.0)
lightgbm        4.6.0 (expected 4.6.0)

[PASS] frozen software versions available

GITHUB CREDENTIAL GATE

[PASS] GitHub credential: kaggle_secret:GITHUB_TOKEN
[PASS] token not displayed

RESTORE GITHUB REPOSITORY

[INFO] Fresh runtime — cloning GitHub repository...
Remote main: 8b87f734f076c5402324ec9c7b2ee82e74f64d0e
Expected   : 8b87f734f076c5402324ec9c7b2ee82e74f64d0e

[PASS] repository restored
HEAD: 8b87f734f076c5402324ec9c7b2ee82e74f64d0e
[PASS] repository clean

DURABLE STAGE28 LEDGER GATE

[PASS] Stage28A durable:
       consumed = 58 / 108
       remaining = 50
       next fit = FIT #59 / C071

FROZEN EXECUTION MANIFEST GATE

Manifest SHA256: 47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505
[PASS] exact 120-component execution manifest
[PASS] Stage28B C071..C120 present

FROZEN CICIDS2017 SO

traffic_labels/Monday-WorkingHours.pcap_(…):   0%|          | 0.00/65.5M [00:00<?, ?B/s]

[PASS source] 0 Monday Monday-WorkingHours.pcap_ISCX.csv.parquet
   bytes : 65,465,382
   sha256: dfdcef4b8670e52af54dc4f82174834365a393473e877174cca46d17b12dfd02

[DOWNLOAD 2/8] traffic_labels/Tuesday-WorkingHours.pcap_ISCX.csv.parquet


traffic_labels/Tuesday-WorkingHours.pcap(…):   0%|          | 0.00/52.7M [00:00<?, ?B/s]

[PASS source] 1 Tuesday Tuesday-WorkingHours.pcap_ISCX.csv.parquet
   bytes : 52,701,751
   sha256: 27e83d518cb093faefd0f883cb4df3ad8b353f150934004f28d0e7962f9f31c4

[DOWNLOAD 3/8] traffic_labels/Wednesday-workingHours.pcap_ISCX.csv.parquet


...

[PASS source] 5 Friday Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.parquet
   bytes : 23,048,086
   sha256: 7c5876d52189fc01af54bad6cf23afe9f7fbc0e3ca6c3595920754f0c3ba8f66

[DOWNLOAD 7/8] traffic_labels/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet


traffic_labels/Friday-WorkingHours-After(…):   0%|          | 0.00/18.6M [00:00<?, ?B/s]

[PASS source] 6 Friday Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
   bytes : 18,632,427
   sha256: 4d78cee297c27f1a9947b9384793e587a46c7a3ea89db199553dabddc9835d4a

[DOWNLOAD 8/8] traffic_labels/Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet


traffic_labels/Friday-WorkingHours-Morni(…):   0%|          | 0.00/22.0M [00:00<?, ?B/s]

[PASS source] 7 Friday Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet
   bytes : 21,999,571
   sha256: 2c00236b13a69f4b1c222b8f4a89451dc2148cb04a8cd0c45c2a87af51471774

FINAL SOURCE IDENTITY VERIFICATION

[PASS] 0 Monday-WorkingHours.pcap_ISCX.csv.parquet
[PASS] 1 Tuesday-WorkingHours.pcap_ISCX.csv.parquet
[PASS] 2 Wednesday-workingHours.pcap_ISCX.csv.parquet
[PASS] 3 Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv.parquet
[PASS] 4 Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv.parquet
[PASS] 5 Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv.parquet
[PASS] 6 Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv.parquet
[PASS] 7 Friday-WorkingHours-Morning.pcap_ISCX.csv.parquet

Frozen source files: 8
Total source bytes: 305,935,672

STAGE28-COLD-BOOTSTRAP COMPLETE

GitHub durable HEAD:
  8b87f734f076c5402324ec9c7b2ee82e74f64d0e

Stage28 NEW fits consumed: 58 / 108
Stage28 NEW fits remaining: 50
Next authorized component: C071
Next authorized scientific fit: FIT #

In [2]:
# =================================================================================================
# STAGE28-AUTO-C — ZERO-UPLOAD, RESUMABLE RANDOM-LOAO CONTROL BOT
#
# Durable start: 58 / 108 new fits consumed
# Executes: FIT #59 .. FIT #108 (C071 .. C120)
# Two-phase durability: model push immediately after each successful fit, then result push after evaluation.
# CPU ONLY. Random membership seed stays fixed at 42.
# =================================================================================================

import base64
import bz2
import hashlib
from pathlib import Path

EXPECTED_SHA256 = "47adfd5fbf563a5d9f6f0b610e37928158e5b399827563913dc71cd54e717edb"

PAYLOAD = r"""
LRx4!F+o`-Q&}KXAS(bH*nj18R73!I|NsAg|KI+<|Ns9$00e*t01yCRSf1y-Vm9Ay+<iOj=DzlY5G1L39_ICCzO<l=yPfris<$=S-(lz;&$xXRdt)b-
uErtj6#8qgw(QMr_Tlem-1o0oPd)}NuxM!aSO8V$Zha2>;@8W2b$#}C(|g~2mc4Ql+sD>y-
rW*|cYWh`HeG52o~+dg_ej?J+<fnKd)LzY&XeuE8x8ZYU8r@S?%1`<pzOAsppD(yr*ki}sC9jet}3OY<<nzXo2|C#k{pbakQuKE7acTqI{Vkh*QT<b-
(A?6k*arX72NxebIHo_;KV>cOh6DOnHdR&rXx*HDt@V_q{*W-
4<s6TngGL136LN`1s+8`qH20k_=c!$Hl|Q|fb}$J4GkIq0zwf8h!bk3>S?rxlxS%lp|v#i0MOIYX{LZQ000Rzk{}QS1jq?JQ_4S7_LSKXqd}&CZA^`+qiP12
fQdp8kS0MJ6g1S+PfB>y(t2uPsp&GGkxwb=8hV~hPf#9{MhKD+AQMJ_jHa3?`lqIo6c0v_8Z>F30i!?~26R4{AYl;^Nd9|I)_z{^y4PGBPxhcv8KdqW4B6|4
=4oA=P*YdTjQ+jMxG4JKB-mTiwzKnY6xp4vf~Esqe@>|#Gy59X@OZ(LeSzKlnfT~8!Ex<Z42~45jSgw9hGAn*?S>4YetA}%sy&370s4B~KgsB9P$?HGf6^$}
BcoZTh2fau@3w1QaJ!*JSZtMx-$}a=Q-
q18{PGyoY3k_}z~VXa*B732AMP;?gM4d7ses?zG1YbF!hxM{NVYyQAMM&4y*~+SuRbe$wVl0~><ncv!m2@@n>)Gd1K)I*?|fWD^Rtp*q$h(h?z+Uf_-
2`fY)mzf<KbrJsBaE_bSI-
Es)C#E7jZrs#MA3>PX><V4x2if__R|Q=tAZ!9U+L(jn?(QW7PE_DJrI>%DEv|w9&>LCQ8xI#e}F{Hz}av_eNf#1&qwAmTgU(89i-0+MH8^=;JgxT-F_Nz63n
{s-aLw5l;B)eOpPF>h9%y@OV9VScESL4b<xWeE9XHjx`^tkmd!l-
HaO=dZ<>^C_0*${~m3F+oIrsa1@|SkOifw9i5BcO75a>E*xpPkAzf_WDtxXL}U=Zg%?k<(N4|nH#cp`x9q!)Ca|#DfaM2;HJz=8Att8vvR|Li_@4ms4o7hXh
{NLRPO%wK;?iOS?RIx*uIxXSp9eIM-q;HFf4}+ogek+Ln%~d0^<PxAAi<{LoQ6OIfeB=Y6$M;N`+yJt1e0_KN&g-
>gdj;Cj^A6#eEIQu2TQ`3FvfA(ti;6mse-
@E#1=u;7uEWyJv^fTiV^tjIHfC4ro)B;hUj`>OkiEJ`f=&V_MY8TInKNUeQ^27foK$zBxESmf{SDomeuh|SUCqowHdTX#!UkygN*RAFa?B0F=UKIOd3JUIqT
Pv){^_HWxCuCyZ)J(!HuDs$jI2=PHdOc*cN9{{7{kC^}m<+MCJFu=D%0ECz=zr*Eyi^5wibqyfA^9rO9BS@M0TJr}5U<{CbjxR4XDCBLbts?*W82eR35j3_=
7bvvk>anp2&c9-s%29<sNf$Wj9PEk9fte@h!&eY<~23k?O#?bdI4-SAu$S+qn1Pz?}%4F6w0AA5?)K-
f?=m9X34(Efcvr<)F?@G{Gu!45n&j=zI)+PfD(%T0MEP@JtU^J}^yMB<;=1qs#>25>L+E%+a>NnDVjBmGxz57seXt|RHC@1(nlueHl~eXl7gVUyOYjB3MLl;
I4e(I@R8yWwh~NbA6TIE>geKjO0q@jGlJi^0SNYC6g^9R$vFj(oD5>9B2wGiRY>rl4coT~z+FiBL(!^*Y#SGT1;9obGVh<|-
o%5;_KO%n&XtxFD9e!%Z17vBt5{Xy9Z|<kKsjTx`75*(4q(fHiOzuJ`4E5y?g_;s*3Bkjaox^ryU~NLq^H1$*hN;4%dTWq~HONCrLtBj;B6UNjcAjz6j(i1G
G(|9_+4+pp-mZLN>zUw1I{L%Zi4Q8)c>u?%;G>;ri?ut)KbjFp?Y-JkM!?Wh9H<N>xe;b6CRIDf-
9x~EGwzwmF>;~nU1(9YcS9xfb&`M7>rThTo9;nlX|xOSUz(clnQnEOMt!5>%Mhb%qebmR36xLp_Qn={rolS%=Q0E0^+*N$FZ=FM9*eYd;`2x#yhW(y9${MLo
N`M)$L%Ks-v=d0|VU|X%X>>fSbbN}5#Vl{6w1A0gS0H%Y4eP_F}+;vj5V%;B8m&HfCObIskeK8|pk${LV_Uf^FBM^w>s_)BIAHGI3t&W-
kYnIQ|Zi}WXk<*rz7oV*FsIl`6M~Bty;r}&sy7=fBOzTh_AOl7oX)t{3S?%`U29qiyvUMi677LCje6Xmq5ljs$7+JJTx_mpZ7C4Pam8u+Dorb!M)i8Sqrj*9
c`a=Bh(sZd`8Z{(T>A%r(-vwp)J{dC;JG=fHwH#+G-
|de~y2RLMIpnI^;8n~VfEBdpQzNpo#@CiF5E|0g@ETT=Z!1=4m6?Wr1z}_XJr&56*|x7<#!%lbZ6V68nrt=F>)IQ*9?x>6Gldt(Q=XrtHI0F@dRaqU#kGAk%
7ED&e&*G_qtwBW2~e{Tq)}3zY;i0B3)h!Jxb8Y9SAHCM&mNzva4@h^lt>6U6&Ikz9#i{%Umq{3l)uMz8l|QDI=GE|Ute`?A9eHNc;S`p3%(q5regg@(PZI?K
ujY3LFEV8^>3#6vPi3tx!GKR*URa=LvG(H_5aEO#8NQrYsg(4fxb#1_#}Xp4|gEg@<Ud=j5cR=9dzaBPkjXM7JwYU$c#+1_~soZ3_38+l(#h*G6J_yOdA)IR
(OtFzFLOIV-LRf?i9)3Jky8v)9^6ZqyTie8X4w1y%hV!JlT06m|6!hL2`sS0C=rP72r<8Uw?>gLZrUOo{h|9=$YfToZ;6xTGNo>&}T^|V3H^*prKT94W?HyD
{9qA+pzY$JM-
1M?I)@dQZes&&R3h@9RQtxMlaP<(;4Fejv*&2%8+TYmIMnas1b?_pE~O&3+qi#*2q@`F2rfZ1XNjeOxZ)DJ{dw+0a}zyQHXj_uoIRZo7sNrBcF!cv27b88x-
)6q4iQIzU?4r2sOVOG0zsUK`o&(6{7^77)~!3HbT${cd9+S20l$hM<HSZ5nP5S;}kBWuHVWP-RwVWbC-
gaU3WaRHw_L{^6h+FHR;oZ^kNe}nCEWpNV&jXcH?NOg7YDIXupoU*_?R!=y^35;bV5x4ezd_nPX5$0veNvM-ZMu!fJ^`iHNF5CS}hVO0-
MY>KFL(@^3WbPy()YaOS6v0mjB<u%Z3u<KnI9k>#G#x;gRf;jUAMa^^W87e^bN>=ZYj)5W$QlurAU3xSL(;K2|E?m7A49z)UBj?L&yUNIP+KW9*G{@GIwBR8
=Q9Z)EC1m;}Fgfj0+!c<=Ux9^GK+8SOZln{i3nkZ1i$emHWWi26P3NIJ$_2#6-@-
)D5iw=}Uw}&+rvPxK+Z=81R9=rLx;7!EVxD0oI+7s%;Z9vpg$VJ#FVgiZ*m(kJZ23o_I(j6p4y!vQ3_HTP$R$W71UT4xi$eRP<w1#jnn!ZL<tpT@=Oksiy0!
USb5Xpl$<Ewt@ag_Nh1uV$p&a8`cPMVn8w6P2W4Jy#eyf~)M3Qp?_u>tbBqh7USWOmNQPh?Vd6dL#6jdg80429P!ILaRVm*CL4gLQyrg$V>9L7it!hDEQVuO
u%z<3Kf>2-ez<l8zDmc<te)^wmg2p(AV?Ysu#BFWWpy^nG=;t5mbr$JZjJlXlUtNvVYUPCfhntdb-T<D1Y&84GrK__&ygY}teuPqqR0dnW+*G*8Viy$GS-
?pp1S>+@}TVTVj03wU>bXc+O44bFrGMWW5xoa~VOBd=}4<B<UBe(Gx>iyq%<vG#2&v*60aEf-
_9s`8>t*=qWke+>miSgX@VU$0WTOQqJrX#YE#Zo(Z{`aH_pW}#c#NBVieb9^VL-2V;&%7rK>Y^nxD4t&D3(hjx<Yl1H7r)zfgb$n%I!-
AFf%MTSS(j(E=PO0pW&dc!XhMV^h%lSezWX>|S`|hw;!o3~*qheY~>)WV7`aW~xhwN`}gnWCrycxH3LIg`QD6hlAnd6;+3|W`1AP<L!gMm+mme!o7GLTw;xE
)Xss<IxC{-NG!k6Hkw><1`;_y!+}D8_H-pC`_*Ez*a}@dh9t$sGVhL(Yp6zq^R_Z)|fGY~8mPZsu|vBqAb&t8gw$Ve>gmW@9x{h);#<;R%6LzY&g9oSrd2`F
%A#BrvE!Qd1s6EsRz^Jm0)Owd<0!_bgTEZMeMH&j;ZbfKX7SoCO>P-
&OeaS@&23ih=9*V>#t}TeC;|{|F3A=oZ9sP@&u^EpZLLC!zDvfoI|MNn86r<Y`~D8ZM^A0-<WHf{hgt%*+H+0#_~|jx}tIW-rf#%do333~mM#14Yqx4(V+LD
=7q@k%Tds!JHEb>w5T4()jmp^M>7PjbI(U_4;`gJ1!A4=8Z|~>cy$pJe;Bg1zM`Mp6;3}21!1}N8)BjA1zg=Hfm7(S>xXMkL^Ac&P$hgxhSsjL5DJ;su<dz1
RPG{xXR<x>Y(DtAn`1i&`=6UqGmJ)DWNC1R|FbU2XUa|IqjtQ?3L^2k`-$J#Mi^a-yq6<bgS3-
pN!S9KYpP^(@j!JIwaFYFf&<iSdX&SGei~Wu=`t9QXwwNpC(Bg7MtqnhZ9<SDgiO^ptf6Ci-CYqB$7ttXALUjX3Jru%EIkdqT@zkY@>?>$nl)TCEWzvt{T=F
nVB$Ja;9<CM=5A6L1<cN;()fBT=86A4wN;wTwZTqu2R#eLnQf#Ep`Cm@1?NgYgyI`>N`MP2N-
I%xp$j86?|w7xm(LdQI@e{6K4xTO^~xUrCbY3i|>TF%@U^B%PoUy5??low&8M!Q@(lSV&_Kt4*8v7V>#@OCs*9iPC4k7&O<iaVb3f`UuV+22j@rU18{^XWR0
&3AkNXWftSutmK_{`9wyud;{?vdl0^MNMFOAPfQ)^xfP9aM>hgOza4z=fhvD%2!LVrsRaq55Vk-zLD;6N`h)4xZOQi#71?MU)<IA7Jp6Mf&aBTN`{7*Jzjjf
&(Y`t)A2ytu1(P*o|4T_=9$ctrd(<(}kc{Ys-
u8OSP)7F^@sz)*|_t|RIY*fUbG>7P)!aU{nAT;*MsHyRL@k$)NH@*f0?5L_L05eo`I^u$2hXASd5pu+PP0y2qV*rL{09X_mvik;SCywV9O7nR!r?ioc0CsX}
d3CAO(qx@Mx0(*IH$4upeCg+-
e4ZlZDmWea%T3uwgTpK3S|^ug*AzYYYGdXU0L8x=0)y@(`*OOFqWKQXS}Dk(Q*a$IY!bmF2+c}_So(IlS*6CX^2l^Zj}JM%F9{<eJ24gNlF}6DMW84F$_y(-
D2n>toL$8v?xRXbxN*V*h6gcVo>n}~S}}}FLN`^}7IEwplt!!oWGW(MZj-VXnVO>CUL)7xujFaJu~bz0EA3jp-
+oo`b8JMtUWNZJWZQ)hUVc4apJVoBiDKm#fy{xDYO$m)9SL}&1z;S#Dc;oP<5NjlBAYD2sf!yiuFr?g@Yc8Jx<7(})-y!QP-KpK#z>EWU1XN>=_?HU?0&d`G
#mxAFc+<tFc8(HFUnIue30Pmw@uOx6D7Yx<#?!#E77hiqde=Txoe=h`0K2L%bCRHh<>XS)HMs~yedzR&nd-
9+L9WhSTnC~{VS|;W_29Kr#zjWs=f{1O%Hs5Xj&@@{T%Di={2kIuUCGR)4N+`+`ZZ}I^H9lZr@FlLk>-
1&H}Ubu*2wREDU=vyHnWqXT34t#Nt{n5rg@uQ&OsUYj}5LWQpMT)CT2?ZCQK^jeko%WC`TDCM@%a-
i8x|NN|Wj7e^5@+H9zZR?J7grdVC+buo@E`uv|7uZ`Qkd&0+*+?_LSTW3CHL7)VBXuIy1Biw<hxG5J*u4X}pE?s;=c>@{<LfD#_S7rGYz+U{R4h|j4sJ;hj_
scA_)>-
M!S&YxCq4$o?JvQ5@jhQozeQ^vm9et=Ibn2umWAtxl1W46<^2xz2)Mvf5)}FaV<D6c(;Ar~jujXx6zbv|>&lKx?GKooDo6qOyA;Hd#gu06Cq=PEvfis^w*t)
0`$C+!$%dTd^x(2a+cXLk)0okpAxhSRp`G68(;9N5PvFr%3B+JwT04KQh0b^;&c$`4*n{&^HL>BK1Ijj1I0Mrk7vETvCf^LwK5M~N=^20O?mwekR8Zp1(pcF
asmkfPMzDj{%bXDEt9G4AT>>d7ZWVf@?3`GqM_nYjkMcZzcr#BYmK%&5d-
*$kssKcm73Zhgrt7K4A^0Nq6u|A={OVmX{BBK<bcgnzG9$_7T#43D>xKUIv{BciL*03iRR?9CBX}`P_qB(D+ec(M~v`#a|C-UNyK0IhRut7$}8zU+zLHh-
I{;v<`aoluHe}|<}RYp$=;edU$Ej&~d;H98p=Rg#sxYqu2xazwjZ#Z0AsfJwdhCa`?mlflX)4dX5j(K6iKvDu51U913F)0OAp0H3%B@keSh(3f#KF-blsJx|
KrpR&@mj>ug5vk^!S8}b-Up<Hg0RwUa#a8#^2TKrw{us?4t#U(-
nKwoye+dD$>uILptmB1cZd^?up}#hUV!^f@Q7krz)M7Q*&84&wtA_8=!?!?%h$=qeK)1x-B0jqT;sNDVx42>_N1VQa-SZ9!d0{+C2EW6n?L21hntyzG4a-
vB$YM3UqAL;d-Xh{qoW)0siFpL$LF-
a*(AjCSvWSBsKU)d`)B)(#FnA1nwv6j^V>5u|38Cb5z1K8sI8u3$t5>ySG)0Uo;*`}=hJ*rX`P1f%En?JRC29=(u$_~>()6#g;nnG9O+RD$qu;B8TnD-qCi>
wRc=uyCRT*4E^8J_sx8L69!ucD_0B*B@-
AwOG5gG`LMFu^bTuLiYL;>pq5E$4wuGLC;YwX`%eh8qvz_v;?sExg!1cpKBoe$LfX6`>YV!_CSs;(7oMWI#+%CPV40XgG&sYFRJ7`%jZsxW#)zC`HkwqJQ3K
9SUr@*Hg!BJ4H+LgwfOiPh?0NUON*phYy1sAO5J(L99o4LWRNZH?u^@v*KF-Ke9nt~>Q4#A=5Ad1ZeYSgjOt=mx1<$QY2m4d-
MN$^=A^5#i|HcN%x#*Z)pF{yEHGLfIq>WY$;(WCB;*s0Kr^BnLwF_?A5uepX|vP_0jDwjC^7U>o;9LCR<~9@^`!v=jDUTWtUgln{e;5=F31bGmMXRS*vLaox
)-AS^%t;^c!MbihA>&o!e5_=8Y_zS}eHz&s9FHV&~1=2xU0NDs6Q?eeysR@)!pEPgHZvSJ(rXXn-
P6Av<X=X2p_x$Jp#?$p5m%wCjrM5f9*kLEvU>zoqWdCTUk(uDabTl|8dWw=Bn79kFKQ)n>qC|Xs5aN|{lf$r9N6iR6Bov^+mtaAba0R(^#V|F&(+$HPeENuq
HHJcVm0+uTk0&0k^_UK1u$8}8waJiBRQ&S3aV~dbBp|u_Xt<LAiNO|nOV3Gzlku_!NAE460kOUXvEUH_N0ul)`Fj%3{uHBc?`^i9bnkMK#+&5Lf159#i*Z@d
ENNuyOwKWMMvjjRbLVIjs;t;z~&<5s=jvPF@boA}cmswyY2#^ls1&^qDOSbx4u+BbyDg&v2$=GIuLLp_b(rNscr`tIVKN>#D+@T8)Awz$z9nEZEB*VQ!;e4~
^-g>Nlg;I*t@u*i*X6v=zs%643ErsV$|4KU1ri?k<DMrTQdJ7T?c$C4>RmiDo_pJ}AkV6#a+Lv*@%X49-Sw4-
`!HQX1sna}{W?XOEZ6?;6nXjhxQ?Zf#n*i)Uk@AQhg!@$ri5MUYB=0Fm#D^9imVaFWYAFEg0+>B&k#mE}BVqyFR1NDQy}HoZm+t+wfXDCavt5t;Q>5eg*$cc
=xF1o>h@ob^Z*s_P7Z2$$Cw@+K{57}5>#lIre^ej(6(G>bK>~!9G=FYDbNTg66l`Q!MPM)@$pwTk8F5);P#IXDD63;iO2vXTk_}*u0K{30Vj#f<NGL@NWQX(
Rib5!4LP~^aqEgfuB~p~cPuYq^v}8xahpy2o5Cea*P*e|yahf2zz;p4sbIzn1B2H6Z#(uARc)&1SyKqFOK%%Q*p!)Fwy%T+-DQntHQgsU7%YLBq41Inx?-
d2~hsGaFr0jeN>?;4&K5&1X$PN_4%+Jv(pRP7)o~y#nNS7IdPs@;Atu-&X62Ham4Y{1d$U{FRf>^*qKk8-
oGL`|=qyKrZwz_PkuDrB0CWZmmJZ{yF@)QVf5{dl2sVGvHw|88OhNc|60GL^aDR<BoEc1@*0mZ;vFfdJ_9&CbM7r>t1hj1RqDTrW{OzyKMX{5Y$o$$HAsdq`
z5G|Bb9J`y)@X>A>?^?!S7AR@E0-$d=TS8dO;hGSGNs3tw14mXjI{QiYmUA>D0w)<HCyQf{Y@U3)-
dEY=ho3)k^<WJlgOQ09&45hCoH1BndWn{M4dh7%B=?;Eqk;Q2_3P~VFuclo)vL50dA29l7lZ-ugMrqc#4#I2@sP0Kz2ay0rjHcT8Us^vxrvb)g|r_pZ*H|!p
T31|Vj-
D&5Goxk*mw^UeEyRC#!0Ao%`0iHp~yABA>q?;#*EC4{sVEt4q7L*@5;lb<TJRQyA8qDz&Z_oOkY<sY|f+LV*y8ul1VS9Z#gkHoXTjn;0rY1H*zGD4)g2+JC>
fDEVIh0P<MxT&mwzx5h7^ZHkXcluI$@HOKFY{M=vfRgY5u<u>v_2^WSgIH;?jS$HaZilh4eu!YAM$`g9}Y5QORV;$j_cnfM$rB!F)o_#puGkHBES5Wcw-
`Sz1Ryh8L3$}$k0yUlyEeGitacd~ogX?Ey1EK+xT*8_GXMa&6B+HN`wUN(GJ7#jiu0uOll8^f9C+&E4LSC5YDn@6ADf?>W5mWm)qa00A$IZnaQ&^wG=4$RA-
hLJ-~)kRj++R~^%yw}s?Pi|TV$k5nieG|;nbLJNdeQs(+q_$|ra0tX&j5!pP*?^b={m<Htqn5kxy@SuOJ#?@=Sw+nBC7zaH*Tocnkv%tH&X{68m<r&o2BL-t
Ah$%<vc645(?BW!4nZ9FSa#!DH&Cuk4<a8(mE&7li7Ja&cP+%Z04(SE$3q~zFgYbQO-#L-E*wk-
L1&fJF1H;$+ZK%8$Em#E2#3}ZbE+FpFxU*GM2J5M2I}(^{-
`C)gR&mL8C<=HeNdlJuoX&os$}2p>=dQb<T<EF4!Nh@v<PQmxv=lE*lHuG2fJdvATqx2bU25$^B~(b-
ksqFyW2P~#PB$k<&^3W`T6Jc0Oli*ejreJN5kqKf#Ms$4~sb`Am&yR+&GtB3F4d2uYgl<#~j{~2|JI!Gk=l5d|P<B2Qz#F4vt1(=9bCGGc>~}vIthzw~c%Qb
T3Vt&*=9;u@jO}z1VnhVcuPN5bnswFw8zRRqSxG<n1<!>ezI69Cc>*AQ=~C?ePiOP=*AeF$2p+B@zfda8u2E!uqAHYg1DTNePz}%;|oEeTQCpKBr4q-
(yh_p&mRzxPg7)%LHEIyi_XLVNI<&?2)w9CWO~+g?w?|x$~skXw1nh6eP;3lJ6EzaZjIDrFNljXn<P?5kE>6HR$y4<c>Z+boq4*I@$`1iCVE#^M5awR1WUe8
TTWf6ww&vNKkB%=4N<uVj3PhMzSVB2OXKB;KuW#Ia33aPDQTFA39DDLylYWH#G+%E0wnF)Hh|63V`KLDBVEV)T|Lo&C{X5urLs>H3Hxa*(4yDvm2$Trl~=g%
2H7T<zfrfUQ1P_vA*Sv2RX#xp8^wda85@Q)OL4!yYxEbk3%&I518Z+S5?TJHO-
k9(A*UjOScYT$mNRIJCe><E;M%mi$c*e3|{X5$EWoBW?k5W>TEj`5?u_Rm%TpNB%-J-
uhH;3lh7}bTMoWHx}FHCD#Kx(b7<v+GSg3#`e{4#K0xa(3E_?9T77muh0$Rs5{>=)NRLZH8KhDD*tLYe$^wyl06AQmw6b%swlBnH+rt|uR6;~z=$C_WJl{p4
qlt6CRzyil@_ra|pjta@g`d6qq%X19(Q1`(Le?<_wvvK8c{3V*N=gu|4ZTVgAPz7yw(_c+>F2dvpY1UIpWApQ>Uw_x<x)<YH~J{ZN%&U%2mCyf`El9q{F}wI
3qDxgbzoThKdbNk-{1JZ?1u9`zTWG1em(#>5D-lpnutCKe3&qQd$iY`MKY_f6tD&!>9Aw~gxECL4d@?g81VD%!=(2cGfL4-
4k79xT}4Wa@?WNh?ex&hz4*TNyHtA5Wso;Ol?nL$^dY1Os!ty;_&Hh03;O)`hW_^?d?H95Xkf*J+xjUn1>Xr7<&jYeL8&;$UE&x(6Q9z&J>>kM-
J?aPR~mf&fP7f|*%(aqeZoefrO)j#HDLsjKqQcZQPWrMgG+ssv3Pw&W~Wp3`YW95z&#o^8r+!B(V5gxEQ>@ju+8Ksq|cehIS+c8WCBGXkzpZl(K(7Cs8k{@7
!SZQARrwbU61iKbWwx7tCzXvucv#iuKMaK)SKpxdCHsTmVG8O0N-
gOP&Pmq<WNXdUUX|C8@|RlR0^YsaL2{3OS3Y^0d25rXaf7;Q5!g)IzIojAE(FIzk9Ta^)O(-
Q{BhzAbj!ko^lu!9O;Q87t%vPaCvypy!|}5CdrgZu#AUD43?sJ9D8)Fw%c9d{BhS!s!l@E5@%_aJ*H;bKZ0xBmU8^6#f=~Yfg%K%CRLHfsf~6*g@A%UFGI^<
{mv0FR|q0P6F7<wz_kXK%K(|=K$sHp3Si*O2%3(7UI<eVFcO2V4TPYGkVa9TJ+KkTB*i~7zX&<wB_$;@rpH@RFub?oV&KrGc!?V$V%Vycj-D_z>|=^+9WKZu
jD*`KnO%=hbnPMC;0O2;yr`z6JGJa+jVWqc^^mK&w`_9V&k4DRl~hu)>(*});}HS0$p|qW-<92CsxGBREz6D<N{CefU1;t-oQb!lmYp*)-
1ON)X_Jp3n#iK55H8<NwsCYMosCY$Ao&D%*k<=i+kFmV+$-
zacycw}G;0HocxnlO!~;9e`krv%rg6z`qtqsJgdDiM2B_WxBV!B%W=dcH&Rzmw7?9d3tJT~)s6gfea;HAigf%M<Qfa%9mxG~$bTbN8%0^~gHAX3Fp@l~1lv$
H44Tp>dX?I9G>i5gGbC`FV2RYRB^5fkRM0IGQDu|&^;3o;wp*nRqaagR*Ej)ZCwQY{9YZ+32WO9X)hhisLu|K4IdmKD%x(b{vmYM3CKt=J^>91>^f?1&NPD3
jwox$K-XJT_lh_MraUjTRu&zL_z+;TMs*%Uy^SPUtY(Bp@7^)-
*@!VL4M&utsNI|iySDCjaoZ#{W9rG~E(B+E*%3Pgezx;vQHMhn*QCUogrRoi8g4{B@oQ0kJPvxGK=B4<=NJhCz<($+7Ns9kyLZass)BVl*&ix_NhIGpkb7^j
Xv7S0kfC7$0Q)6~<Xkvb8g5e|XeoAu!R^gqx1T_^KWehH{^Web0J!69(aR5%gS(fcnseGCp`H|-4wJNDR=5(oLq3VD#k8+rcUt_kzk-
3TC9G=vnK%;IGGcq0&Cg|?Do>5}PICsRx8eZQ{MxAx|>b?o{4`>djj7z-j>X#{BvH?xKR-Cl2_w>oDcN4wAG>jOGzNcm3U33#Xzu}WG$xA6lr<I^a=EgP~D1
rjtAFc}m`FoHrFe5`^zn+Y57{`t$~WJ_%+Vte1k5x7NjFvMnN5DSsX6pjocl<8Gsrp8h(EEK|#fUAv&U8fEf?0{&PTr`(axWX2Q&<;W<*9`dM4M*1A1v%nx-
|gwLC825%%b~Rx=kx_@kfHp3XNX9rh-@0d@m`S{?<rdVLC8aBD<X*C3xP$`ZndEnhzY&>ZbRq1Hkzj>3DO7Ajm=l5NP7c1fual?PnsoF-mk37GByq!+zM~Mc
Z5>!Oi2j<G>Lc=<_|I)pS~)00=o2}-qI!z???5(Nf8I~J<j7~6C-w~fRo_wha^U}-
iKx_O8O{l$Qi9iOsLttBMDcP1lxt5R82_N0)fGKh@sMAUW=R}CWqIc14-Nf&jW{D9V%-t!89BYXfak)*grP1rGU{DBN9ms9CNMih*~u++d>-
{?ZWHS>BtfwL`AN34#|JtoQ0P7d37z-F3lEU2bl3ti!GQl8Zs19F*@mt0^t2g%fRv?Ado%fhtc97Xsd?d9^ZwJly5Yu-Kt{SlW_5ALP$VF1WoyZ;N!bvbNb*
kAZ!;+plPSt@W!rMfyG%IfVj>sJs&TpeYp2EY0v<&<07SdB$8I5QnKUX_~=|Wz;~QCbLY7fri;raSU+NXI9u<f{jk3yJ#NH3s9pP=SG6$xI)cGafJT1wNdTp
%d^T(cJ1mp~>zWDk_tajy{PjO+uy7gd58~xJAVwmBuivnGp6fc1=&cgtfZNB2^MHQV+eoQ`*+QFE@@KB<MCM4786Hdn6;#M5hUfzQEy`WlhCt&YDz=SeG9p?
%q{@K5?)!YN^NMR88x}fOv`E4s_g7*W5WpP&&7f%kMG;j72Ot_=|IO+8%;X6_o8Nc&9H%&a|5t$e`1zxVEGRVGrbU7|N`WGUKq!e2{JJ7GPD<~b2wB3pQOyk
=NHDT*LfKIoFb6T7K!t$$%JtHZatBr}Oey#ApF6gWz5U6&IPDT;6G{&rcLg1Ee5WXSAj8kF!~~Ybq+nVEfKWxHg&>zuvb7b)S*I%&kj_b8hi_cj!*tqaanQC
0>R};B{jX%=(tISL#HA@j>6ABk9d3i8xsr@*f%#=9m=x;(L5p!2767bZ7%L{nfO`H;hs#^eEGHJ*D-
oGxu%!eL&a}?J76u<1r(hY>mYGFz96ZlLO>#JSeg@|htdhuw32G%mQpz&P?_`L9B$6zaBOsALpfcHRuCL?Er$*zv`f)6t7_C&6nM!~YE-xIatCU~hh*{M#I0
}6%G|xwGt1vO9tfC0b^`kV3AEl%f97Am3lEq^Yo23K<XSta8*0sDq+6D>Rw{}J_DGnsYL}J7k!*dv<(ik?#ipr#GP3f3X40#+ZXp|_=(dJ$fVy1A-
Mw>)15lx6?F9y)*%a`(9^ZY2*2N}n75TxP8uayWwA(TKkj9A%HG(@UO2B6YGfY>msj7>T)u>9eWGzVT2kCnOsuu{bzmCGHbR`mKrJb3VS%i$_UDnM93_#15Z
(x-pW4@9wK_qc)gY8Z?!Scf?(i(ZuzpRb1f?-9Y`@qkDNdBrwG!__$Qx6<Ag?>M2S5EI=Af-JaTqwiB{zZ61fo6^Leq-
j3w`Pwo&z@e=YQ9_2`@8{k?gA9HU<F=q0o$-
1>jaCWju*@^xJ93wR99RdJDWeU97ucx<D>w;uB!?bwpHJK6Juh}jR9fKr;PaMRI6h!I2h<fII|L~FOfB3E#CaTHaZJ_gTO?JS7n-X-
1qXBpPw1)hXD^%Uiz>K>D#l-
1y);2J1*|G13(caAwz5fr9`68@G8oy?jfiWmjhS`VEu)H39O1&8DSHKG9;d5jMeOAPJbIv)^je;e70pmSiWtS<QPPsT@HTO~TWeX~g-
k4IiJ>{6yVP(2@KsX{EE($I7gLHvT_jAX8+$5O5=&f`ybnWc?p&rZKwin@gTuEYS#r?L_IfW1G;%gkT{zTdBN)Op9`y{2Tx07s&U4Qla;Z&z0#x8vKuiY1)m
kN=X$YEz3Po83E3&E$l_Y>YG4|jy8vG$2ZZfzbdMftC!#d`0!)s+7C||E&NEx{zRTwD;b;trVWLeD5uC`+i1{~!T>lk4MtvVm5Q?bl8ZU~5?C94Yqx{8LA42
z-tO|+jBjWUfI1A`W!#?sImM*=p%q%Yk3RY=6&Cjk&kWZe8zKs@Mlkk$^+dtkT$-iVDLXAv$kK&X-
S6wt8y=okaK00k<*Zy?N*=p=PdAyGZA3!0mm=A+F5D;?FYn9qIKJ!*0=4C*yHyh1dW%ezpVD!!7|qM>3v2b)3=^Z248tx9Yej7qv}(7=r~mdxv;JBC<_mLXE
9DN0(`*G+0<s_yzJYz!nrzKfCvhD2x<;I?=`3yBR8P|$dT0Ktgh*126h?%Co3Z>6xcDelJM$_O5$<3$m4m?5oJNE|#~DNiKxN|z>MdbGu>2_g&%fzV|Z8~3R
2M0CdT3=<5HN>UcojG;;Zf-p)02_kAAtigry70?{;VBBV}WHr(b5dE2_?Bw&%IsJ7xZcomXIi6yrApQ~1FhR;bf$F|Z65o+egpf(IJ-ZhuckUNQZURWCs3B@
a0KDwN?;7-
tn)jOpz~F}ktS+qZ+6X2BrqSp0dKX^a>gCy|ROr@_QqI~yQb?eKP%R*2H;X)$!)1$GX}vf>bLl=U`p(?!k@Arua8~1~VZ3Z+9XM3dtQ+K@_{BcWXs<lLXsx!
9j8TXhEsNt2LXmed$57_dxk$Y9#H`S+o+!~k3`Quq4B^+V5He(o9n3V5IB)|9Wu{c0TNlj7stl*}_gJyO+Rr_BpphQ7q~WGjry96&&4pbTfyIdJfI+b8U>K@
gBj5<}FcpD%G+@^RZnsc%(Y!nHwNDot>@*k25W~r#BN0qUkc3kO9NjpBQUxyr>4enKQ@0fr6Lc1$eXSxeh_0Df3P_Tw@DQ;ONI^?#mf$=gy&f^(w)C>PXCVk
g7d8VAp+Is2&`Y6>uZD-YVj+Ysxr9%TyWYLe*NnVWqsXzuB_w@t8Mb!^abu})?9QLxsT^Z#-
0|Jbmurg997UMLHNVf1?1wZu7<z=$4%607NO?O?F_Y@wA^fQZ=yEC|iV;)=k_sphNg|L^4E+gKD2zo&v0{;Xd=D`5XS2@F(!y(6a?;p9tM;}CI951_Qk4RdZ
b=^T_2r3$^*#K$2d|)Q4&RFdPUs!vIJ6H3I=6w|n&+gP^0Dv}Qy^}R5Id!i-Bh0)R;-=yGN}Oj(H*hR!Cf4y#v7TDM9`YXgT9GPatO4Ye-
A&_Q3q|Z9TQkytZ(6wmI5xQ`Axy#72XCxwH!14Z6p%q$fkjjs3GA`ze?~Vayn+0SX`s5={rng_q1*YNML_>p~&Zq4B@Uv8HM3n_)=R$UnbYguL3!1VXzn2(Y
51vY!D)Zn}WDL1qu6NNQQxvgd*ywpppf4vMrdQ*`54Nj*izwEHdO7fTe9LSv0pY=%QN*yn_xBPr4fh*G#CVg(mA7LT)Iw)@`N+q*HG3eA06CHsEmF^x}uI*s
zF@!G<7&%F0owEJ1++2wkN5e#k;eKCIWLBdOBd_d-At4EU;4N`-
Ix0Z;(oH1bCzF2#^!d5jXGEks6DFn|~4XeLUBfI&?@GL&j}pmykqZ+n>`zMky5S*>eRVv)72Qnll3he#`DKA~y4Im}H%3(>AE3!IJw*bRI<Qb6!CO9)tm{Lr
dG(F(|<uqZMNKm-^_&{#T`8-iLJHcG->s6nElpuixXKh;ry7BezZArRy_SxY&l0HEy<@Xp|;fxAF&GNdEhB-Jg6ZNnSkOtS0x&e~}NO#>#X321_GHnr7AFh{
b1v92yc8JZc1Lr+rbcs{43K*-
!kPo50a=BfcGJ(^(Hn;t7Rdp5q}L7X9U`rzQy)f&npA`C)BC>2rc(wjCW9oyd{CgP>EAa6N9jCb56ezo&`r>EWXtJMwpvDM~7F6ECWy_ogZ6?@Hv$~Ii6mFn
+zj}zP7lUCmC-NK8yZqqJ@P^eO2bx?Y}YgS89R*cE4)2LK4MYh$|YWY{l4)Eh(;ByEq9E>CCgZ>nUHONy@7iR3^oX9EPAv`Bxa<9i0%MtAj;^9)rD09aI6AF
MxyVJOkkx4!46u~SCkx2nyC<6&d!hI*9%8QUTO@K-9U=&DJ2R?-
CtQN=uN~JX5Wj|jnzD5ST5hzO>ct*j9v*<FrC#7rEyRy2!j?2jB&${sRz~_GIQDW+iBG7Y#NplIM9pIf)i_lKJ-hMrDniOt)N01;``o*XcC=>M2_WY>=(5T)
wj!)PRkS!aZR|JV?e}MG4NyUpIG9wrSfXF0Rph6?-=y^wqu&97svtHs2N6XPSL2(0;YWdkWZv!Zf-
c%NQl`FU1TD5{^G$4(JuYk})YJp@e&4q3CL8t8ECNGVLijg9-
ew9!1x(kIfw&N{;K}_|V79kD1Gd+jC+ibUOa@fEEfuROq<`YQTcs3j=_6Gf7!#?z($UgB=HExMgDnvM$rbOlP*7s<pN&*E$r;l*oSYe%**otW*MsZ<JRL*4B
1{B8Y>~{{~J&%H|n?i-yiE&5i-
(r3=Rx#k0f}IE^6>avE{LhF;$UOJb6j46%$x(U2WYAMlB@`_c7lgMBBajZ0E(}4G9bj8?Adx;0`+ITG=uU(vF(BB`0*hEVPNUD47Q4fU{(S#4dW9y(B-
qrFeH-^fy;xvcQ9*&`+QdsKWr&JIw-XT$+c3DPZAFSS)T6aE47$lOIT49t0BFHP!7>UcHkMlpn!v~`D9K=n7$!j|2>}Ekv4ToL0#TDhK>7$aM2L|R2^K;HY;
CdJc~$=((MN}!dVX{sM^DJ_=%0{xAwggd@drMKj_&^tdL(KAs{A}JW?lnAND%@71>#1MWEO@TU^We<?mhGfJ?gC&j3HlbAn4v=OuP)8n03IM_BbGRp+ll~d&
{*=OKjJ7j`mkGAbUjI+aaKQY$%-+sHm?Zd;t3Bc}J=obWD%P`h*hxr>lh@k*HwPp+VBlvOYLnz-
2dbKK)AdSJh<P0ld^Lb|*6$!3sZ~lc1eek!2+h4Wrlsr6p8#<+?H_hPA5o0KHy7A|ZE12SHFI_v>1Z5i^WdL@RE8l=|Pf4_g)VMeY#!mFIrbQd})Xw=`AmC6
Iy)QfK5L=0UhzP}Bopc?V+EByAqgXyOONgBtBFK&F{3x~4!<7@d$a8zhmDgi=FOh2akt7?Z0Huk#lG(M3(r#tD#)<&dIgJ$ls-bct&ScWivFczD;G%)SH4iU
8M=31AIV{IDV90oEvc>$^$MT0YV54k$z4F9k$nJt;u~D3U73LOx~?oRq-
H3kpmSzW6q6+YlYy#DEBm){d^>hQ35T2bR$^S9t(D@7=&1hYA8@dCSB@K1Cfw@rIg@GW~-f3|JljY@u-P_>+~9@GzLXf|e6bm05?EzvMTGRniZ-xqOEBS{(8
)mswjDEOHg%cmWU|YMjfBif)9$CYndArvTEk%1hMN%U<iftd!THm09l9?)ND{gaR@`Kzu>_$1$jxab-
j>rVWfC(zrfxV?&okE7jY$ze4cV7%{d3)W#xIl1Uac+^A+1O5wc>llhp8g9KKUR0bn0=W=oB5st%(T(B+e+GXv{>H`>KvV{gWvAn_LS{m9mCIZIkG8G_P5LY
CG>T_JhDSQXCQ5ojW1k%>417abOf)a(sQ6Qmtf>s#8<Jng`pr#pgD@l<a4ALQDq;wb}lme;{EcC+D8elTg&}A*WxUsa8DKW^E4HTJ>{-
czn4KN6s2M}b8p5mUQ#0ybd1M7_JzQ(?{95630MED$}@rf2NG|7Q}fW+D`s9+MHhSDPfk}3$4B8N=WOe6|+P{b;*SvEjrv_cSMNc(Q~n~y3gafcA)s0qav=B
#)_dfpkXx<WxDjcqA6*Ji2X-azlR{g4g8`>9bogtTfx_C~-
^i(p7*(oF>o2w`JEppw!ydmw0s@uZmSA)`m(O_ApM5o8}j%hS6w4$u}Ed|Sgb{2*ZifS?IKAy`&g4N;Z*Eg}gB9@-U)$VNs10-
o{EO^%%Id|0tvNuwE{f$RfOVqnc7n95W{pr}#+Hqby7WC)~)$stPxKxz~L3V<+fs|zU~d|)JI5P-#qLQGH$R1}1&7c6vvsL~-
)4ufrVWG?0xb16g2Ys0O4+ZOuJRQ_`h!)1#gGK!|5_)$R!$r_9M>ZASa5voFf77EfS%fR~KO$@h549E7#1Sqitvi2d+#b6{rfP)kZW34X=i#u^mLhW8*iI4^
(m?dQ(=QAnxB9V$8baimI?<gFU<{C(lAVU#gPI>wD01c0D2ZI911VsS=BCI=V5{AZfR6?SOOcNvvzqTRW0+J5^6Vm~WQPe*EvwR3fP?mQ%C<PFRrRpAZe#sc
gAQlaR-H`I+5ucJZFIXAan}>jI&pIk220W-(MT-_fC@8UEEcEFJN-_ju<ckA@IPR(%DS5)yKsMx&O_eB;99v`a{K_-
}K>J$Ucqil<n+eBv5gtU0i4up<flg`f7H(5I(CQ!>J}{vLnGGT&fSb;0ki|=6gIpHcwKW~cuCQ05tacoc1HsN7{OsUf!i~?t=Jr(15?YMk!HKP(m=^i2X-
8o34&myBBz<74Ar(qP3HzgTF9Ase%=F98$}-=*R_%AfZNo>nvPP!-
KPzwpp9U<8Bk+JwFCL_sv5>9XIWn>$&IbF51ylsArZUKINe{%vhKx|m3R4pxPTuNqoddgF;c)&E6k37+awrp#^*H(Rr_0FtRj+mF`3kqG+h;f@e;6hfd|uh@
FoFzCBy3DdK}bTM4hBOBqEV)=T<Rfm{UCV<>3iY@1QCZ-9{^DLxEKa~);@k5adAy=DjD_NuH_7}G;|!O6acA`5b8q-
39T6bmBv}{O^gE~kqm$D2RWPP%D85m#Phw5oZj?11B0o<;VftzOc>;NWaUOfXN30QcR_ZIOiu*M%Q71<Mhg#zfNlt+Jd)O0%#RjRGU>XK=~|e&w{QbI#*eD*
xoq~dVIyJf-PrrP!+w|@fXJXz4|x+mJQ*%njgOgu-g4m9{q?)vMAd>rh*MlXv*X-
H<0;(3+Zhv;VvgF5VT+b&_#s`rmQ+wh5Z;v{Gx6{6?VD_I)Cw300fGuhq!to$d|(X7XB$w6ki{ef1d)*BD1<MG7SIsEq&Fi$LUmAeoEA5%RarDxcNiIw4Y$f
-#;Zgw9z2ymSW!o;w5d%UcB$N)4BZYw!2+VlQ~?C%>Hsj1>ca#e9DVcb_jye!T9pK)O<+|<AjV;mxiGT*mWI^Tjv|fQ6>zs*AKmB!>(LJ&$}xwxfpX!u3bI~
gg_?OL;t)a*=t13mc+GmK8ujx>VpK#HLJT>}4se(RnG>6=r4iL_jBTqJF&%sMuA!|IsSlT5(ZWb!`~QD?e}{-2Q61rSqEDM*PF2de6-
~%da*?E(H(v3FtD&ycRl?uDt<Mv<_Ny8%iAZi<G%OH{0{)VT<nU>!o(0izniqI(AcZKpPHy>lYL2{&wb<Ld7#jLeOD2HuQ<rUc3suG3TY+*%WS&O)rx_WPe8
Zu;7rZI$96|V(wN#j+FgYghccH&d%60(*C{!cpLL<>f#?RPV6x9T`w~lbQ2vCQ9UyueB%*D{yid%73z=<FcBEbUcnjoC`ovbOUe3amMgoPoKm#~QHay48X-
Q{+Vf=vh^ucZ||qFkB6bBmi8L{C`M9wKO&Ndb(8;#VHvwuP99q8gFSTOb}}hwy<3xq!h*>NFxqO-bpHG%!dp6e7d~2U3dw`5?4~lbs*92QJ~yvZDxcZaSSgI
Ormk!KJQAOpwG0{TrViUw`grAC^(iKbgm%=Sn)$=Ce-
3x_P~mVFFt!)*x|NTo%@puokQPYM`=#s7mVAfyxwlD;6^zRG1D}7GbVM_KMo~_M9>RyI&!NggCs%H7PyMCCF3|1j`~&RyI*~tS}jH;FQ_u$5GH!D6Fukm{MJ
;P*7J>%5QKLd09tuyUJwLc)VC+=~}i~rW=CtM4bQ{Q>v|OF)HqSP61>otl5{Ui7bC`uJ^K@L}r5=Z)`mRPex9HEJ|Rj0B2pp3}yt1^kK4H<a#x#4(<84N!VF
zbzSd61*m~iN$gRuWeL3#t(`G}xdS3a0}@qKC_$E4==}hX-
m=I%gpl;&arF+Sw~E14SdR076&E)dxdZk(jS$<v+TTymQC9X%f+6snc=5NUqoJ7y8GILxeLyY(J!tfn&cN6~qA4BvXdwvgPSz<cYV~mCTUZ~%38H5q0w=8sg
+>G!<0#_}9hXrw6gh~IIjWhW9viQ9)nu8db!xzIKXEUEuy<!g?=AC~zAd3gjc$SC=4PhoghB;XEMasF;so6!vMR5aATa~M3p5lFpWg7S3EHuytLB1lr8*#;c
x~JA$$O6yXzCqLIM5N^5)wiTFLis)O7hqPfbQy1-H$MAZd4)ZOTACp0vuOUFA^Q67@MgnI~jVB*%7e=-
n3{{f`USKR+NRj$xDbfiz71OoUGi0`S@Hzu?)%Ad22rdvFdpY{Swp<aL7=Ic-
V7+uRX82qfKp)tc)7sS%kB&$)NA=IYJ)LgFQnSG=Wmqhb*(RS+?qjaovSeGGF)|6$=FB0$;ErK(NEwm`K9|qqnx$TTdu?8=*Y*)eF;wE(T&m$iVd^mJCKn;)
|-;ryKz}$~qB&%>jTtl+!*z4_uCjpQ9ivWm^lEIBxAn8&TmheB^Wsu0lk1d3rhnYO$<|8s9dQj2iG`2~h-
T8l;xK3i`!gRes##l{eZK?nAY($@McdZj$dBIPh^kHFa)gV(za+Jx4WDtx<e&!B+Q^3eygG*`*`H71G08JXwxMlS%W6mR_{)9K9N}qX!dJsf{aYh-
eU9orzXbX*Vx48VT6hI6hMBzN;YU?9i!}_~@Z^xq|Lgu2jsXvSMhtFh<UrZ3IaOh>kQ9kr3L_Qd>);tTLW9G2LY<7>Lm-Xlqq0&KW{H7?EbCsGDe&sZBHrvV
e>V3aC&BLnSMY+v3lw3c5Ql8Fa>rKsH*YE%}cIf{b~k=G<MpPFtKbUqH>tXC6GV8yrv(V1y8pw`;-
NUHD_)te|T4c$`jVU4|SEI~`$AR6&Z0i1e)1otBI#2(awe2n<#*-
G$YILu0&647z}%D9trd6Q>R2Oe7Lm8rdZP(2y<Uc8DT~Dj}fZv|SjMLEA;txXZN?NrO}hl3Z&!AVl!+B(TE*)EY+!w`<@84FW$1+#?i-
#z=$|EwSIehUY?;iBp^+4FKP_J~i;$XR^&l^SY~bN$Scr?uAxO{Tl_xFf#9pQ!SJ<X1-
fY(I+sIy*x!99{6|j!G&JFB3MHK1(6wWU|=lmF!H;*i8g}VEx`NGA%+x_0)}S2f$Wc}MKhr}RlBw*YTXD5RzVA}t6{*JKCPe)d`N8J&BLpFX$Fj7R#A-
!WyU5~rZvGMmm|~1vxSRY{lo0ypoa)fw1p7_v`~LU9Fy+5j#t}xo=I!x3`y_0m`~G)+=>bC!-)yZIs1W#;z^jT1&BMI^D7^W@QDV<P$<p`-
=UptHqIMKDM_s2hrN$RkX1#3u|!EUM8SY0MUreY>Mt>d;jU=#9Llk22)>1Yv-9S0GGJ)VfN%Ox5PL1`@G4I~xApsW!MA{W#5l-
6k8w>>Q4<?`SK}>ew{<64Ddc<hcpH|Y2gX6vUcp;V`DIJ4)Y5`HJD%e3v7$pVJG0$&th%fSv@(kJo_g>-
kJ4$TW{TvY+eD#l%Z~%V=Z|d<l&R0Msm$}hvZO76fK?8vqN}V$t4zi?F-Ap<D171`Eg#n8pj?3$?`u>TF@!Rq9MD*++-
<IqjDYP40;9e$fOf8xCg>xFbkKY&iS!%=G7$RBx*+t>JVg7vXL>{Nfv^|AVlOb+a43j`fI^~H`d}U0o#ntdR>#N(t29mD2R`9|Y}Jk5EH>?e{UlcrfCMg%z&
8|ZFPIx20ln^E+v5LEiam+YPY3Yv9feJh{!kH(UCxF51DG6v$mjrEz$@}xK_LWg^j45r{lW;)UKl8)3aSYXwG4Ettp1zcaElG@3N@KWq7@+`tY9($qI6D<ST
-L)JR6BdiR%uZPGLvDMtmV+RL?@=%nRh#+ws{}`uM6Exh@o>Ly@%<rES;^@i;K}mc=6-
5I*yhzZn>MPjTPf;Yxsqe6XVmAYhOjoZv7BAp!Ie!mNz|z!{1mVLM(W2=2Gdwyu6(DE^scsKtvDz$RGFO)q=k7OK~}W(yS{9Cbw14aNY4;|_GK;c*N2RzCmW
^ftnW><u*Y^t7UOExj`oc>Umc6B~#Sd=u9O05-?S-*sLj2P81LU3a+E4`Rhjsx$)_0wf?<&$wYY&!^^s>53lQxbBC`UOP{-%UkKFmcbrQDoifTOqL;1Ns$y-
oy5f%W1m`tTU4qduFn}R3U{`|Qu!V;ft?Hrskj7u5&O9ZmXU;};eaezBEdis7Q@~8yYzC7(lOjrL`T0tV>LO_)yglQaj*>H%B{RC-
I*7lv}|X;Y+8rQ*$u*_gD6eoqSQhVG}p41Vuqr1P{HGup_XHU@rn@dI9Pa$(?v4RfO0~Ng_PFUQNQ(5NpGi^6ph$24<=Cm5tVCrM~PtDa2Nqlh!K%t^s$^GI
(f(A`Eeg#KP#i}dNaYNMufum*xd5;bui?jeh8o#o|)`#jUoM%tk?4F?R!P2i#>O|?5W)V6<ZY_vOsB`?fa~Zd>?>>02}{*P6B^}0iZ=B2Xd4mOtLG<(sc6IH
NfUzRA(&*k5m0bKm0@bmIVI7?ntK!5(El_1z-
"""

encoded = "".join(PAYLOAD.split()).encode("ascii")
source = bz2.decompress(base64.b85decode(encoded))
actual = hashlib.sha256(source).hexdigest()

print("=" * 120)
print("STAGE28-AUTO-C — EMBEDDED SOURCE VERIFICATION")
print("=" * 120)
print()
print("Expected SHA256:", EXPECTED_SHA256)
print("Actual SHA256  :", actual)

if actual != EXPECTED_SHA256:
    raise RuntimeError("AUTO-C embedded source SHA256 mismatch. DO NOT RUN SCIENCE.")

bot_path = Path("/kaggle/working/stage28_auto_c.py")
bot_path.write_bytes(source)
print("\n[PASS] AUTO-C source reconstructed exactly")
print("Path:", bot_path)
print("[PASS] launching Stage28-AUTO-C\n")

exec(
    compile(source, str(bot_path), "exec"),
    {"__name__": "__main__", "__file__": str(bot_path)},
)

STAGE28-AUTO-C — EMBEDDED SOURCE VERIFICATION

Expected SHA256: 47adfd5fbf563a5d9f6f0b610e37928158e5b399827563913dc71cd54e717edb
Actual SHA256  : 47adfd5fbf563a5d9f6f0b610e37928158e5b399827563913dc71cd54e717edb

[PASS] AUTO-C source reconstructed exactly
Path: /kaggle/working/stage28_auto_c.py
[PASS] launching Stage28-AUTO-C


STAGE28-AUTO-C — REPOSITORY / RESUME GATE

[PASS] GitHub credential: kaggle_secret:GITHUB_TOKEN
[PASS] token intentionally not displayed
Current durable HEAD: 8b87f734f076c5402324ec9c7b2ee82e74f64d0e
[PASS] synchronized repository; AUTO-C recovery residue allowed only inside current output space
[PASS] Stage28 durable ledger = 58 consumed / 50 remaining

STAGE28-AUTO-C — FROZEN PROTOCOL GATES

[PASS] 50 Stage28B random-LOAO NEW components exact
[PASS] seeds42-46 × XGBoost/LightGBM × five families exact
[PASS] threshold / target / support / budget contracts exact

STAGE28-AUTO-C — MODEL RUNTIME GATE

scikit-learn: 1.6.1
XGBoost      : 3.2.0
LightGBM     : 4.6.0
[P

RuntimeError: Unexpected repository changes outside current AUTO-C component:
M results/stage28_stability_novelty_control/stage28_2b_random_loao_control/stage28_2b1_bot_xgboost_seed42/stage28_2b1_bot_xgboost_seed42_execution_progress.json

In [3]:
# =================================================================================================
# STAGE28-AUTO-C-R1
# PATCH PORCELAIN STATUS PARSER + RESUME FROM DURABLE FIT #59
#
# NO REFIT OF C071 IS AUTHORIZED.
#
# Durable remote state:
#   aa990a64d6f0451684979d748098b9654cd82e52
#
# C071 model already frozen:
#   cbfe69d53f07461f5f361fc67aef5d8bd28ffaeaeb9982b421906e2f7f9709d9
# =================================================================================================

from pathlib import Path
import hashlib
import subprocess


SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
)

BOT = Path(
    "/kaggle/working/stage28_auto_c.py"
)

EXPECTED_HEAD = (
    "aa990a64d6f0451684979d748098b9654cd82e52"
)

EXPECTED_MODEL_SHA = (
    "cbfe69d53f07461f5f361fc67aef5d8bd28ffaeaeb9982b421906e2f7f9709d9"
)

OUT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_2b_random_loao_control"
    / "stage28_2b1_bot_xgboost_seed42"
)

MODEL = (
    OUT
    / "bot_xgboost_seed42_cpu_model.json"
)


def banner(text):
    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def run(cmd, *, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            + " ".join(map(str, cmd))
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return run(
        ["git", *args]
    ).stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# -------------------------------------------------------------------------------------------------
# 1. FIT #59 DURABILITY GATE
# -------------------------------------------------------------------------------------------------

banner(
    "STAGE28-AUTO-C-R1 — FIT #59 DURABILITY GATE"
)


if not REPO.is_dir():
    raise RuntimeError(
        "Repository missing. "
        "This recovery cell assumes the current Kaggle runtime is still alive."
    )


if not BOT.is_file():
    raise RuntimeError(
        "stage28_auto_c.py missing. "
        "Do not rerun FIT #59."
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local HEAD :",
    local_head,
)

print(
    "origin/main:",
    origin_head,
)

print(
    "Expected   :",
    EXPECTED_HEAD,
)


if (
    local_head != EXPECTED_HEAD
    or origin_head != EXPECTED_HEAD
):
    raise RuntimeError(
        "Durable repository state changed. "
        "STOP before recovery."
    )


if not MODEL.is_file():
    raise RuntimeError(
        "C071 model missing from current filesystem."
    )


actual_model_sha = sha256_file(
    MODEL
)


print()
print(
    "C071 model SHA:",
    actual_model_sha,
)


if actual_model_sha != EXPECTED_MODEL_SHA:
    raise RuntimeError(
        "C071 model SHA mismatch."
    )


# Model must be tracked by the fit-only commit.
model_rel = str(
    MODEL.relative_to(
        REPO
    )
)


tracked = subprocess.run(
    [
        "git",
        "ls-files",
        "--error-unmatch",
        model_rel,
    ],
    cwd=str(REPO),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
).returncode == 0


if not tracked:
    raise RuntimeError(
        "C071 model is not tracked by durable Git state."
    )


print()
print(
    "[PASS] FIT #59 model exists"
)

print(
    "[PASS] FIT #59 model SHA exact"
)

print(
    "[PASS] FIT #59 model tracked in durable commit"
)

print(
    "[PASS] C071 MUST NOT BE REFIT"
)


# -------------------------------------------------------------------------------------------------
# 2. SHOW CURRENT RECOVERY RESIDUE
# -------------------------------------------------------------------------------------------------

banner(
    "CURRENT AUTO-C RECOVERY RESIDUE"
)


status_raw = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).stdout


print(
    status_raw
    if status_raw
    else "[clean]"
)


# Every current dirty path must belong to C071.
for raw_line in status_raw.splitlines():

    if not raw_line.strip():
        continue

    # Robust parsing independent of the leading porcelain whitespace.
    parts = raw_line.lstrip().split(
        maxsplit=1
    )

    if len(parts) != 2:
        raise RuntimeError(
            f"Cannot parse Git status line: {raw_line!r}"
        )

    path_text = parts[1].strip()

    # Rename syntax is not expected here.
    if " -> " in path_text:
        path_text = path_text.split(
            " -> ",
            1,
        )[1]


    path = (
        REPO
        / path_text
    ).resolve()


    try:
        path.relative_to(
            OUT.resolve()
        )

    except ValueError:
        raise RuntimeError(
            "Unexpected recovery residue outside C071:\n"
            + raw_line
        )


print(
    "[PASS] all current repository residue belongs only to C071"
)


# -------------------------------------------------------------------------------------------------
# 3. PATCH ONLY commit_paths()
# -------------------------------------------------------------------------------------------------

banner(
    "PATCH AUTO-C GIT STATUS PARSER"
)


source = BOT.read_text(
    encoding="utf-8"
)


function_start = source.find(
    "def commit_paths("
)


if function_start < 0:
    raise RuntimeError(
        "commit_paths() not found."
    )


function_end = source.find(
    "\ndef ",
    function_start + 1,
)


if function_end < 0:
    function_end = len(
        source
    )


before = source[
    function_start:function_end
]


if "line[3:]" not in before:
    raise RuntimeError(
        "Expected vulnerable porcelain parser not found.\n"
        "Do not apply an unverified patch."
    )


after = before.replace(
    "line[3:].strip()",
    "line.lstrip().split(maxsplit=1)[1].strip()",
)


after = after.replace(
    "line[3:]",
    "line.lstrip().split(maxsplit=1)[1]",
)


if after == before:
    raise RuntimeError(
        "AUTO-C parser patch made no change."
    )


patched_source = (
    source[:function_start]
    + after
    + source[function_end:]
)


# Syntax validation before touching runtime source.
compile(
    patched_source,
    str(
        BOT
    ),
    "exec",
)


backup = Path(
    "/kaggle/working/stage28_auto_c_pre_r1.py"
)


if not backup.exists():
    backup.write_text(
        source,
        encoding="utf-8",
    )


BOT.write_text(
    patched_source,
    encoding="utf-8",
)


patched_sha = sha256_file(
    BOT
)


print(
    "[PASS] commit_paths() patched only"
)

print(
    "[PASS] source compiles"
)

print(
    "Patched AUTO-C SHA256:",
    patched_sha,
)


# -------------------------------------------------------------------------------------------------
# 4. PRE-RESUME SCIENCE GATE
# -------------------------------------------------------------------------------------------------

banner(
    "PRE-RESUME SCIENCE GATE"
)


# Confirm fit-only commit still current.
if git(
    "rev-parse",
    "HEAD",
) != EXPECTED_HEAD:
    raise RuntimeError(
        "HEAD changed unexpectedly during patch."
    )


if git(
    "rev-parse",
    "origin/main",
) != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main changed unexpectedly during patch."
    )


if sha256_file(
    MODEL
) != EXPECTED_MODEL_SHA:
    raise RuntimeError(
        "C071 model changed during recovery."
    )


print(
    "Durable fit:",
    "FIT #59 / C071"
)

print(
    "Family:",
    "BOT"
)

print(
    "Learner:",
    "XGBOOST"
)

print(
    "Seed:",
    42
)

print(
    "Model SHA:",
    EXPECTED_MODEL_SHA
)

print()
print(
    "[PASS] no scientific refit required"
)

print(
    "[PASS] resume must finish Stage28-2B1 evaluation first"
)

print(
    "[PASS] next NEW model fit after that must be FIT #60 / C072"
)


# -------------------------------------------------------------------------------------------------
# 5. RESUME PATCHED AUTO-C
# -------------------------------------------------------------------------------------------------

banner(
    "RESUMING STAGE28-AUTO-C"
)


exec(
    compile(
        patched_source,
        str(
            BOT
        ),
        "exec",
    ),
    {
        "__name__": "__main__",
        "__file__": str(
            BOT
        ),
    },
)


STAGE28-AUTO-C-R1 — FIT #59 DURABILITY GATE

Local HEAD : aa990a64d6f0451684979d748098b9654cd82e52
origin/main: aa990a64d6f0451684979d748098b9654cd82e52
Expected   : aa990a64d6f0451684979d748098b9654cd82e52

C071 model SHA: cbfe69d53f07461f5f361fc67aef5d8bd28ffaeaeb9982b421906e2f7f9709d9

[PASS] FIT #59 model exists
[PASS] FIT #59 model SHA exact
[PASS] FIT #59 model tracked in durable commit
[PASS] C071 MUST NOT BE REFIT

CURRENT AUTO-C RECOVERY RESIDUE

 M results/stage28_stability_novelty_control/stage28_2b_random_loao_control/stage28_2b1_bot_xgboost_seed42/stage28_2b1_bot_xgboost_seed42_execution_progress.json
?? results/stage28_stability_novelty_control/stage28_2b_random_loao_control/stage28_2b1_bot_xgboost_seed42/bot_xgboost_seed42_validation_threshold_grid.csv
?? results/stage28_stability_novelty_control/stage28_2b_random_loao_control/stage28_2b1_bot_xgboost_seed42/checksums.sha256
?? results/stage28_stability_novelty_control/stage28_2b_random_loao_control/stage28_2b1_bot_xgboo

RuntimeError: Repository has changes outside AUTO-C recovery space:
M results/stage28_stability_novelty_control/stage28_2b_random_loao_control/stage28_2b1_bot_xgboost_seed42/stage28_2b1_bot_xgboost_seed42_execution_progress.json

In [4]:
# =================================================================================================
# STAGE28-AUTO-C-R2
# PATCH REMAINING STARTUP/RESUME PORCELAIN PARSER
#
# FIT #59 / C071 IS ALREADY DURABLE AND MUST NOT BE REFIT.
#
# Expected current AUTO-C-R1 source SHA:
#   15cbac7dfb05f7d0bfc1ba64896e3c3301a683ff98c48ba92a07277252fbd0cb
#
# Expected durable Git HEAD:
#   aa990a64d6f0451684979d748098b9654cd82e52
# =================================================================================================

from pathlib import Path
import hashlib
import subprocess


SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

BOT = Path(
    "/kaggle/working/stage28_auto_c.py"
)

EXPECTED_HEAD = (
    "aa990a64d6f0451684979d748098b9654cd82e52"
)

EXPECTED_R1_SOURCE_SHA = (
    "15cbac7dfb05f7d0bfc1ba64896e3c3301a683ff98c48ba92a07277252fbd0cb"
)

EXPECTED_MODEL_SHA = (
    "cbfe69d53f07461f5f361fc67aef5d8bd28ffaeaeb9982b421906e2f7f9709d9"
)

OUT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_2b_random_loao_control"
    / "stage28_2b1_bot_xgboost_seed42"
).resolve()

MODEL = (
    OUT
    / "bot_xgboost_seed42_cpu_model.json"
)


def banner(text):
    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def run(cmd, *, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            + " ".join(map(str, cmd))
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return run(
        ["git", *args]
    ).stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


# =================================================================================================
# 1. EXACT RECOVERY STATE
# =================================================================================================

banner(
    "STAGE28-AUTO-C-R2 — EXACT STATE GATE"
)


if not REPO.is_dir():
    raise RuntimeError(
        "Repository missing."
    )

if not BOT.is_file():
    raise RuntimeError(
        "stage28_auto_c.py missing."
    )

if not MODEL.is_file():
    raise RuntimeError(
        "Durable C071 model missing."
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local HEAD :",
    local_head,
)

print(
    "origin/main:",
    origin_head,
)

print(
    "Expected   :",
    EXPECTED_HEAD,
)


if local_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Local HEAD changed unexpectedly."
    )

if origin_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Remote HEAD changed unexpectedly."
    )


actual_model_sha = sha256_file(
    MODEL
)

print()
print(
    "C071 model SHA:",
    actual_model_sha,
)


if actual_model_sha != EXPECTED_MODEL_SHA:
    raise RuntimeError(
        "C071 model identity mismatch."
    )


current_source_sha = sha256_file(
    BOT
)

print()
print(
    "Current AUTO-C SHA:",
    current_source_sha,
)

print(
    "Expected R1 SHA   :",
    EXPECTED_R1_SOURCE_SHA,
)


if current_source_sha != EXPECTED_R1_SOURCE_SHA:
    raise RuntimeError(
        "AUTO-C source is not the exact R1-patched version.\n"
        "STOP rather than applying a blind patch."
    )


print()
print(
    "[PASS] FIT #59 durable state exact"
)

print(
    "[PASS] AUTO-C R1 source identity exact"
)

print(
    "[PASS] no new fit has started"
)


# =================================================================================================
# 2. VERIFY CURRENT RESIDUE IS ONLY C071
# =================================================================================================

banner(
    "CURRENT C071 RECOVERY RESIDUE"
)


status_raw = run(
    [
        "git",
        "status",
        "--porcelain",
    ]
).stdout


print(
    status_raw
    if status_raw
    else "[clean]"
)


def robust_porcelain_path(raw_line):

    s = raw_line.lstrip()

    parts = s.split(
        maxsplit=1
    )

    if len(parts) != 2:
        raise RuntimeError(
            f"Cannot parse Git porcelain line: {raw_line!r}"
        )

    path_text = parts[1].strip()

    if " -> " in path_text:
        path_text = path_text.split(
            " -> ",
            1,
        )[1].strip()

    return path_text


for raw_line in status_raw.splitlines():

    if not raw_line.strip():
        continue

    rel = robust_porcelain_path(
        raw_line
    )

    candidate = (
        REPO
        / rel
    ).resolve()

    try:
        candidate.relative_to(
            OUT
        )

    except ValueError:
        raise RuntimeError(
            "Unexpected repository residue outside C071:\n"
            + raw_line
        )


print()
print(
    "[PASS] every dirty/untracked path belongs to current C071"
)


# =================================================================================================
# 3. LOCATE THE SECOND VULNERABLE PARSER
# =================================================================================================

banner(
    "LOCATE STARTUP/RESUME PARSER"
)


source = BOT.read_text(
    encoding="utf-8"
)


error_anchor = (
    'Repository has changes outside AUTO-C recovery space:'
)


error_pos = source.find(
    error_anchor
)


if error_pos < 0:
    raise RuntimeError(
        "Startup recovery error anchor not found."
    )


# Only inspect the local code region immediately preceding this error.
region_start = max(
    0,
    error_pos - 1800,
)


region = source[
    region_start:error_pos
]


vulnerable_full = (
    "line[3:].strip()"
)

vulnerable_short = (
    "line[3:]"
)


relative_pos = region.rfind(
    vulnerable_full
)


replacement_len = len(
    vulnerable_full
)


if relative_pos < 0:

    relative_pos = region.rfind(
        vulnerable_short
    )

    replacement_len = len(
        vulnerable_short
    )


if relative_pos < 0:
    raise RuntimeError(
        "The remaining vulnerable startup parser was not found "
        "immediately before the recovery-space guard."
    )


absolute_pos = (
    region_start
    + relative_pos
)


old_text = source[
    absolute_pos:
    absolute_pos
    + replacement_len
]


print(
    "Found vulnerable expression:",
    old_text,
)

print(
    "Near source position:",
    absolute_pos,
)


# =================================================================================================
# 4. PATCH EXACTLY THAT ONE EXPRESSION
# =================================================================================================

banner(
    "PATCH STARTUP/RESUME PORCELAIN PARSER"
)


new_expression = (
    "line.lstrip().split(maxsplit=1)[1].strip()"
)


patched_source = (
    source[:absolute_pos]
    + new_expression
    + source[
        absolute_pos
        + replacement_len:
    ]
)


# Ensure we changed exactly one vulnerable site.
if patched_source == source:
    raise RuntimeError(
        "No source modification occurred."
    )


# Must remain syntactically valid.
compile(
    patched_source,
    str(BOT),
    "exec",
)


backup = Path(
    "/kaggle/working/stage28_auto_c_pre_r2.py"
)


if not backup.exists():
    backup.write_text(
        source,
        encoding="utf-8",
    )


BOT.write_text(
    patched_source,
    encoding="utf-8",
)


patched_sha = sha256_file(
    BOT
)


print(
    "[PASS] exactly one startup/resume parser patched"
)

print(
    "[PASS] source compiles"
)

print(
    "R2 AUTO-C SHA256:",
    patched_sha,
)


# =================================================================================================
# 5. VERIFY THE PREVIOUS commit_paths() PATCH STILL EXISTS
# =================================================================================================

banner(
    "CHECK BOTH RECOVERY FIXES"
)


final_source = BOT.read_text(
    encoding="utf-8"
)


# The R1 commit_paths fix should still contain force-add logic.
if '["git", "add", "-f", "--", rel]' not in final_source:
    raise RuntimeError(
        "R1 force-add patch disappeared."
    )


# The startup guard should now contain our robust expression.
if new_expression not in final_source:
    raise RuntimeError(
        "R2 startup parser patch not present."
    )


print(
    "[PASS] R1 commit_paths() fix preserved"
)

print(
    "[PASS] R2 startup/resume parser fix present"
)


# =================================================================================================
# 6. FINAL NO-REFIT GATE
# =================================================================================================

banner(
    "NO-REFIT GATE"
)


if git(
    "rev-parse",
    "HEAD",
) != EXPECTED_HEAD:
    raise RuntimeError(
        "HEAD changed during patch."
    )


if git(
    "rev-parse",
    "origin/main",
) != EXPECTED_HEAD:
    raise RuntimeError(
        "origin/main changed during patch."
    )


if sha256_file(
    MODEL
) != EXPECTED_MODEL_SHA:
    raise RuntimeError(
        "C071 model changed during patch."
    )


print(
    "FIT #59 / C071:",
    "ALREADY SUCCESSFULLY FIT"
)

print(
    "Model durability:",
    "GITHUB FIT-ONLY COMMIT"
)

print(
    "Scientific fits consumed:",
    "59 / 108"
)

print(
    "Remaining NEW fits:",
    "49"
)

print()
print(
    "[PASS] C071 refit prohibited"
)

print(
    "[PASS] AUTO-C must salvage C071 evaluation first"
)

print(
    "[PASS] next actual model training must be FIT #60 / C072"
)


# =================================================================================================
# 7. RESUME
# =================================================================================================

banner(
    "RESUMING STAGE28-AUTO-C AFTER R2"
)


exec(
    compile(
        patched_source,
        str(BOT),
        "exec",
    ),
    {
        "__name__": "__main__",
        "__file__": str(BOT),
    },
)


STAGE28-AUTO-C-R2 — EXACT STATE GATE

Local HEAD : aa990a64d6f0451684979d748098b9654cd82e52
origin/main: aa990a64d6f0451684979d748098b9654cd82e52
Expected   : aa990a64d6f0451684979d748098b9654cd82e52

C071 model SHA: cbfe69d53f07461f5f361fc67aef5d8bd28ffaeaeb9982b421906e2f7f9709d9

Current AUTO-C SHA: 15cbac7dfb05f7d0bfc1ba64896e3c3301a683ff98c48ba92a07277252fbd0cb
Expected R1 SHA   : 15cbac7dfb05f7d0bfc1ba64896e3c3301a683ff98c48ba92a07277252fbd0cb

[PASS] FIT #59 durable state exact
[PASS] AUTO-C R1 source identity exact
[PASS] no new fit has started

CURRENT C071 RECOVERY RESIDUE

 M results/stage28_stability_novelty_control/stage28_2b_random_loao_control/stage28_2b1_bot_xgboost_seed42/stage28_2b1_bot_xgboost_seed42_execution_progress.json
?? results/stage28_stability_novelty_control/stage28_2b_random_loao_control/stage28_2b1_bot_xgboost_seed42/bot_xgboost_seed42_validation_threshold_grid.csv
?? results/stage28_stability_novelty_control/stage28_2b_random_loao_control/stage28_2b1_bot

In [5]:
# =================================================================================================
# STAGE28-3A — EXPERIMENT CLOSURE + 108-FIT AUDIT
#
# ZERO NEW FITS
# ZERO MODEL INFERENCE
# ZERO THRESHOLD SELECTION
# ZERO TARGET OPENINGS
# ZERO SHARED-FINAL-HOLDOUT OPENINGS
#
# Scientific parent:
#   9fddb8d8c34ba8f81b71f24eea15c90151053d6b
#
# Purpose:
#   1. Prove frozen manifest = 120 components.
#   2. Prove budget = 108 NEW + 12 REUSE.
#   3. Prove FIT #1 .. FIT #108 are contiguous and consumed exactly once.
#   4. Verify all 108 Stage28 new model artifacts.
#   5. Verify all 12 historical reused-model SHA256 identities.
#   6. Byte-verify every Stage28 execution checksum manifest.
#   7. Reconcile learner/seed/parameter/model identity against the frozen manifest.
#   8. Prove no final-holdout / target-adaptive fitting or threshold search occurred.
#   9. Freeze a permanent experiment-closure receipt.
#  10. Commit and push Stage28-3A.
#
# AFTER THIS:
#   NEW MODEL FITTING IS CLOSED FOR THIS MANUSCRIPT.
# =================================================================================================

from __future__ import annotations

import base64
import csv
import hashlib
import json
import os
import re
import subprocess

from datetime import datetime, timezone
from pathlib import Path


# =================================================================================================
# CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "9fddb8d8c34ba8f81b71f24eea15c90151053d6b"
)

EXPECTED_MANIFEST_SHA256 = (
    "47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505"
)

ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

MANIFEST = (
    ROOT
    / "stage28_1b_random_loao_membership_and_execution_lock"
    / "stage28_component_execution_manifest.csv"
)

STAGE22_ROOT = (
    ROOT
    / "stage28_2a_stage22_seed_stability"
)

STAGE27_ROOT = (
    ROOT
    / "stage28_2a_stage27_seed_stability"
)

STAGE28B_ROOT = (
    ROOT
    / "stage28_2b_random_loao_control"
)

OUT = (
    ROOT
    / "stage28_3a_experiment_closure_audit"
)

RECEIPT = (
    OUT
    / "stage28_3a_experiment_closure_receipt.json"
)

COMPONENT_CSV = (
    OUT
    / "stage28_3a_component_closure_audit.csv"
)

LEDGER_CSV = (
    OUT
    / "stage28_3a_new_fit_ledger_audit.csv"
)

README = (
    OUT
    / "README.md"
)

CHECKSUMS = (
    OUT
    / "checksums.sha256"
)


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(text):
    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [
            str(x)
            for x in cmd
        ],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            + " ".join(
                map(
                    str,
                    cmd,
                )
            )
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(
    *args,
    check=True,
):
    return (
        run(
            [
                "git",
                *args,
            ],
            check=check,
        ).stdout
        or ""
    ).strip()


def sha256_file(
    path,
    chunk=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            block = f.read(
                chunk
            )

            if not block:
                break

            h.update(
                block
            )

    return h.hexdigest()


def read_json(
    path,
):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path,
    obj,
):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            sort_keys=False,
        )
        + "\n",
        encoding="utf-8",
    )


def recover_github_token():

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]

    try:

        from kaggle_secrets import (
            UserSecretsClient,
        )

        client = (
            UserSecretsClient()
        )

        for label in labels:

            try:
                value = (
                    client.get_secret(
                        label
                    )
                )

            except Exception:
                value = None

            if (
                isinstance(
                    value,
                    str,
                )
                and value.strip()
            ):
                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass


    for label in labels:

        value = os.environ.get(
            label
        )

        if (
            isinstance(
                value,
                str,
            )
            and value.strip()
        ):
            return (
                value.strip(),
                f"environment:{label}",
            )


    raise RuntimeError(
        "GitHub credential unavailable."
    )


def authenticated_push(
    token,
):

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode(
            "utf-8"
        )
    ).decode(
        "ascii"
    )


    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(
            REPO
        ),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )


    if p.returncode != 0:
        raise RuntimeError(
            "Git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )


    return (
        p.stdout
        + p.stderr
    ).strip()


def remote_head():

    text = git(
        "ls-remote",
        "origin",
        "refs/heads/main",
    )

    if not text:
        raise RuntimeError(
            "Unable to resolve remote main."
        )

    return text.split()[0]


def parse_checksums(
    path,
):

    rows = []

    for raw in Path(
        path
    ).read_text(
        encoding="utf-8"
    ).splitlines():

        raw = raw.strip()

        if not raw:
            continue

        parts = raw.split(
            None,
            1,
        )

        if len(parts) != 2:
            raise RuntimeError(
                f"Malformed checksum line in {path}: "
                f"{raw!r}"
            )

        digest = parts[0]

        filename = (
            parts[1]
            .strip()
            .lstrip("*")
        )

        if not re.fullmatch(
            r"[0-9a-f]{64}",
            digest,
        ):
            raise RuntimeError(
                f"Malformed SHA256 in {path}: "
                f"{digest!r}"
            )

        rows.append(
            (
                digest,
                filename,
            )
        )

    return rows


def verify_checksum_manifest(
    path,
):

    path = Path(
        path
    )

    if not path.is_file():
        raise RuntimeError(
            f"Missing checksum manifest:\n{path}"
        )

    rows = parse_checksums(
        path
    )

    if not rows:
        raise RuntimeError(
            f"Empty checksum manifest:\n{path}"
        )

    seen = set()

    total_bytes = 0

    for expected, filename in rows:

        if filename in seen:
            raise RuntimeError(
                f"Duplicate checksum entry "
                f"{filename!r} in {path}"
            )

        seen.add(
            filename
        )

        target = (
            path.parent
            / filename
        )

        if not target.is_file():
            raise RuntimeError(
                f"Checksum payload missing:\n"
                f"{target}"
            )

        actual = sha256_file(
            target
        )

        if actual != expected:
            raise RuntimeError(
                "Checksum mismatch:\n"
                f"{target}\n"
                f"expected={expected}\n"
                f"actual={actual}"
            )

        total_bytes += (
            target.stat().st_size
        )

    return (
        len(rows),
        total_bytes,
    )


def model_artifacts(
    root,
):

    root = Path(
        root
    )

    return sorted(
        list(
            root.rglob(
                "*_cpu_model.json"
            )
        )
        +
        list(
            root.rglob(
                "*_cpu_model.txt"
            )
        )
    )


def stage_dirs(
    root,
    pattern,
):

    return [
        p
        for p in Path(
            root
        ).glob(
            pattern
        )
        if p.is_dir()
    ]


def walk_key_values(
    obj,
    key,
):

    out = []

    if isinstance(
        obj,
        dict,
    ):

        for k, v in obj.items():

            if k == key:
                out.append(
                    v
                )

            out.extend(
                walk_key_values(
                    v,
                    key,
                )
            )

    elif isinstance(
        obj,
        list,
    ):

        for item in obj:

            out.extend(
                walk_key_values(
                    item,
                    key,
                )
            )

    return out


def require_all(
    values,
    predicate,
    label,
):

    bad = [
        v
        for v in values
        if not predicate(
            v
        )
    ]

    if bad:
        raise RuntimeError(
            f"{label} contains forbidden values: "
            f"{bad[:10]}"
        )

    return len(
        values
    )


# =================================================================================================
# 0. REPOSITORY / PARENT GATE
# =================================================================================================

banner(
    "STAGE28-3A — EXPERIMENT CLOSURE / REPOSITORY GATE"
)


if not (
    REPO
    / ".git"
).is_dir():
    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )


if OUT.exists():
    raise RuntimeError(
        "Stage28-3A output already exists:\n"
        f"{OUT}\n\n"
        "Do not overwrite a previous closure audit."
    )


status = git(
    "status",
    "--porcelain",
)


if status:
    raise RuntimeError(
        "Repository must be clean before Stage28-3A:\n"
        + status
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)

remote = remote_head()


print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    local_head,
)

print(
    "origin/main    :",
    origin_head,
)

print(
    "Remote main    :",
    remote,
)


if not (
    local_head
    == origin_head
    == remote
    == EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-3A parent mismatch. "
        "Stop before auditing a different repository state."
    )


print()
print(
    "[PASS] Stage28 experiment parent exact and synchronized"
)

print(
    "[PASS] repository clean"
)

print(
    "[PASS] ZERO fits / ZERO inference / ZERO threshold selection"
)


# =================================================================================================
# 1. FROZEN MANIFEST
# =================================================================================================

banner(
    "STAGE28-3A — FROZEN MANIFEST / BUDGET GATE"
)


manifest_sha = sha256_file(
    MANIFEST
)


print(
    "Manifest SHA256:",
    manifest_sha,
)


if (
    manifest_sha
    != EXPECTED_MANIFEST_SHA256
):
    raise RuntimeError(
        "Frozen Stage28 component manifest SHA256 mismatch."
    )


with MANIFEST.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:

    manifest_rows = list(
        csv.DictReader(
            f
        )
    )


if len(
    manifest_rows
) != 120:
    raise RuntimeError(
        "Expected exactly 120 frozen components."
    )


ordinals = [
    int(
        row[
            "component_ordinal"
        ]
    )
    for row in manifest_rows
]


if ordinals != list(
    range(
        1,
        121,
    )
):
    raise RuntimeError(
        "Manifest ordinals are not exactly 1..120."
    )


component_ids = [
    row[
        "component_id"
    ]
    for row in manifest_rows
]


expected_component_ids = [
    f"C{i:03d}"
    for i in range(
        1,
        121,
    )
]


if (
    component_ids
    != expected_component_ids
):
    raise RuntimeError(
        "Manifest component IDs are not exactly C001..C120."
    )


if any(
    row[
        "compute_backend"
    ].upper()
    != "CPU"
    for row in manifest_rows
):
    raise RuntimeError(
        "Non-CPU component found in frozen Stage28 manifest."
    )


new_rows = [
    row
    for row in manifest_rows
    if row[
        "fit_action"
    ]
    == "NEW_FIT_AUTHORIZED"
]


reuse_rows = [
    row
    for row in manifest_rows
    if row[
        "fit_action"
    ]
    == "REUSE_EXISTING"
]


if len(
    new_rows
) != 108:
    raise RuntimeError(
        f"Expected 108 NEW components, "
        f"found {len(new_rows)}"
    )


if len(
    reuse_rows
) != 12:
    raise RuntimeError(
        f"Expected 12 reuse components, "
        f"found {len(reuse_rows)}"
    )


if sum(
    int(
        row[
            "new_fit_budget_units"
        ]
        or 0
    )
    for row in manifest_rows
) != 108:
    raise RuntimeError(
        "Frozen new-fit budget does not sum to 108."
    )


if sum(
    int(
        row[
            "reuse_budget_units"
        ]
        or 0
    )
    for row in manifest_rows
) != 12:
    raise RuntimeError(
        "Frozen reuse budget does not sum to 12."
    )


stage22_manifest = (
    manifest_rows[
        :20
    ]
)

stage27_manifest = (
    manifest_rows[
        20:70
    ]
)

stage28b_manifest = (
    manifest_rows[
        70:120
    ]
)


def manifest_counts(
    rows,
):
    return {
        "components": len(
            rows
        ),
        "new": sum(
            row[
                "fit_action"
            ]
            == "NEW_FIT_AUTHORIZED"
            for row in rows
        ),
        "reuse": sum(
            row[
                "fit_action"
            ]
            == "REUSE_EXISTING"
            for row in rows
        ),
    }


stage22_counts = manifest_counts(
    stage22_manifest
)

stage27_counts = manifest_counts(
    stage27_manifest
)

stage28b_counts = manifest_counts(
    stage28b_manifest
)


if stage22_counts != {
    "components": 20,
    "new": 18,
    "reuse": 2,
}:
    raise RuntimeError(
        f"Stage22 manifest mismatch: "
        f"{stage22_counts}"
    )


if stage27_counts != {
    "components": 50,
    "new": 40,
    "reuse": 10,
}:
    raise RuntimeError(
        f"Stage27 manifest mismatch: "
        f"{stage27_counts}"
    )


if stage28b_counts != {
    "components": 50,
    "new": 50,
    "reuse": 0,
}:
    raise RuntimeError(
        f"Stage28B manifest mismatch: "
        f"{stage28b_counts}"
    )


print(
    "[PASS] 120 components exact"
)

print(
    "[PASS] 108 NEW + 12 REUSE exact"
)

print(
    "[PASS] Stage22 = 18 new + 2 reuse"
)

print(
    "[PASS] Stage27 = 40 new + 10 reuse"
)

print(
    "[PASS] Stage28B = 50 new + 0 reuse"
)

print(
    "[PASS] all components CPU as frozen"
)


# =================================================================================================
# 2. VERIFY ALL 12 REUSED HISTORICAL MODELS
# =================================================================================================

banner(
    "STAGE28-3A — HISTORICAL REUSE IDENTITY GATE"
)


reuse_evidence = {}


for row in reuse_rows:

    cid = row[
        "component_id"
    ]

    rel = row[
        "reused_model_path"
    ]

    expected_sha = row[
        "reused_model_sha256"
    ]


    if (
        not rel
        or not expected_sha
    ):
        raise RuntimeError(
            f"{cid}: incomplete reuse identity."
        )


    path = (
        REPO
        / rel
    )


    if not path.is_file():
        raise RuntimeError(
            f"{cid}: reused model missing:\n"
            f"{path}"
        )


    actual_sha = sha256_file(
        path
    )


    if (
        actual_sha
        != expected_sha
    ):
        raise RuntimeError(
            f"{cid}: reused model SHA mismatch\n"
            f"{path}\n"
            f"expected={expected_sha}\n"
            f"actual={actual_sha}"
        )


    reuse_evidence[
        cid
    ] = {
        "model_path": rel,
        "model_sha256": actual_sha,
    }


    print(
        "[PASS reuse]",
        cid,
        row[
            "experiment"
        ],
        row[
            "unit"
        ],
        row[
            "learner"
        ],
        "seed"
        + row[
            "model_seed"
        ],
    )


if set(
    reuse_evidence
) != {
    row[
        "component_id"
    ]
    for row in reuse_rows
}:
    raise RuntimeError(
        "Reuse component coverage mismatch."
    )


print()
print(
    "[PASS] 12 / 12 historical model identities exact"
)


# =================================================================================================
# 3. STAGE28 EXECUTION DIRECTORY + MODEL UNIVERSE
# =================================================================================================

banner(
    "STAGE28-3A — EXECUTION OUTPUT UNIVERSE"
)


stage22_dirs = stage_dirs(
    STAGE22_ROOT,
    "stage28_2a*",
)

stage27_dirs = stage_dirs(
    STAGE27_ROOT,
    "stage28_2a*",
)

stage28b_dirs = stage_dirs(
    STAGE28B_ROOT,
    "stage28_2b*",
)


if len(
    stage22_dirs
) != 10:
    raise RuntimeError(
        f"Expected 10 Stage22 output dirs, "
        f"found {len(stage22_dirs)}"
    )


if len(
    stage27_dirs
) != 40:
    raise RuntimeError(
        f"Expected 40 Stage27 new-fit dirs, "
        f"found {len(stage27_dirs)}"
    )


if len(
    stage28b_dirs
) != 50:
    raise RuntimeError(
        f"Expected 50 Stage28B dirs, "
        f"found {len(stage28b_dirs)}"
    )


stage22_models = model_artifacts(
    STAGE22_ROOT
)

stage27_models = model_artifacts(
    STAGE27_ROOT
)

stage28b_models = model_artifacts(
    STAGE28B_ROOT
)


if len(
    stage22_models
) != 18:
    raise RuntimeError(
        f"Expected 18 Stage22 new model artifacts, "
        f"found {len(stage22_models)}"
    )


if len(
    stage27_models
) != 40:
    raise RuntimeError(
        f"Expected 40 Stage27 new model artifacts, "
        f"found {len(stage27_models)}"
    )


if len(
    stage28b_models
) != 50:
    raise RuntimeError(
        f"Expected 50 Stage28B new model artifacts, "
        f"found {len(stage28b_models)}"
    )


all_new_models = (
    stage22_models
    + stage27_models
    + stage28b_models
)


if len(
    all_new_models
) != 108:
    raise RuntimeError(
        "Stage28 new model universe is not exactly 108."
    )


print(
    "[PASS] output dirs: 10 + 40 + 50 = 100"
)

print(
    "[PASS] new model artifacts: 18 + 40 + 50 = 108"
)

print(
    "[PASS] no 109th Stage28 model artifact exists "
    "inside the frozen execution roots"
)


# =================================================================================================
# 4. BYTE-VERIFY ALL EXECUTION CHECKSUM MANIFESTS
# =================================================================================================

banner(
    "STAGE28-3A — BYTE-LEVEL CHECKSUM AUDIT"
)


checksum_dirs = (
    stage22_dirs
    + stage27_dirs
    + stage28b_dirs
)


if len(
    checksum_dirs
) != 100:
    raise RuntimeError(
        "Expected exactly 100 Stage28 execution/evaluation directories."
    )


payload_files_verified = 0
payload_bytes_verified = 0


for i, directory in enumerate(
    checksum_dirs,
    start=1,
):

    file_count, byte_count = (
        verify_checksum_manifest(
            directory
            / "checksums.sha256"
        )
    )

    payload_files_verified += (
        file_count
    )

    payload_bytes_verified += (
        byte_count
    )


    if (
        i == 1
        or i % 10 == 0
        or i
        == len(
            checksum_dirs
        )
    ):
        print(
            f"[PASS checksums] "
            f"{i:3d}/{len(checksum_dirs)} "
            f"| payloads={payload_files_verified:,} "
            f"| bytes={payload_bytes_verified:,}"
        )


print()
print(
    "[PASS] all 100 Stage28 checksum manifests "
    "verified byte-for-byte"
)


# =================================================================================================
# 5. RECONCILE FIT #1 .. FIT #108
# =================================================================================================

banner(
    "STAGE28-3A — NEW-FIT LEDGER RECONCILIATION"
)


fit_evidence = {}

fit_ordinals_seen = set()

ledger_rows = []


def register_fit(
    component_id,
    fit_ordinal,
    ledger_path,
):

    fit_ordinal = int(
        fit_ordinal
    )


    if (
        component_id
        in fit_evidence
    ):
        raise RuntimeError(
            f"Duplicate new-fit component: "
            f"{component_id}"
        )


    if (
        fit_ordinal
        in fit_ordinals_seen
    ):
        raise RuntimeError(
            f"Duplicate fit ordinal: "
            f"{fit_ordinal}"
        )


    fit_ordinals_seen.add(
        fit_ordinal
    )


    fit_evidence[
        component_id
    ] = {
        "fit_ordinal": fit_ordinal,
        "ledger_path": str(
            Path(
                ledger_path
            ).relative_to(
                REPO
            )
        ),
    }


# -------------------------------------------------------------------------------------------------
# Stage22 ledgers
# -------------------------------------------------------------------------------------------------

stage22_ledgers = sorted(
    STAGE22_ROOT.rglob(
        "*_fit_ledger.json"
    )
)


if len(
    stage22_ledgers
) != 10:
    raise RuntimeError(
        f"Expected 10 Stage22 ledgers, "
        f"found {len(stage22_ledgers)}"
    )


for path in stage22_ledgers:

    obj = read_json(
        path
    )

    pre = int(
        obj[
            "pre_cell_new_fits_consumed"
        ]
    )

    new_components = list(
        obj[
            "this_cell"
        ][
            "new_fit_components"
        ]
    )

    successes = int(
        obj[
            "this_cell"
        ][
            "successful_new_fits"
        ]
    )

    cumulative = int(
        obj[
            "cumulative_new_fits_consumed"
        ]
    )


    if successes != len(
        new_components
    ):
        raise RuntimeError(
            f"{path}: successful-new-fit count mismatch."
        )


    if cumulative != (
        pre
        + successes
    ):
        raise RuntimeError(
            f"{path}: cumulative Stage22 ledger discontinuity."
        )


    if int(
        obj[
            "model_fits_attempted"
        ]
    ) != successes:
        raise RuntimeError(
            f"{path}: model_fits_attempted mismatch."
        )


    if int(
        obj[
            "model_fits_successful"
        ]
    ) != successes:
        raise RuntimeError(
            f"{path}: model_fits_successful mismatch."
        )


    for offset, cid in enumerate(
        new_components,
        start=1,
    ):

        register_fit(
            cid,
            pre
            + offset,
            path,
        )


    ledger_rows.append(
        {
            "ledger_path": str(
                path.relative_to(
                    REPO
                )
            ),
            "stage": obj[
                "stage"
            ],
            "pre_new_fits_consumed": pre,
            "successful_new_fits_this_ledger": successes,
            "cumulative_new_fits_consumed": cumulative,
            "new_fits_remaining": int(
                obj[
                    "new_fits_remaining"
                ]
            ),
            "component_ids": ";".join(
                new_components
            ),
            "status": obj[
                "status"
            ],
        }
    )


# -------------------------------------------------------------------------------------------------
# Stage27 ledgers
# -------------------------------------------------------------------------------------------------

stage27_ledgers = sorted(
    STAGE27_ROOT.rglob(
        "*_fit_ledger.json"
    )
)


if len(
    stage27_ledgers
) != 40:
    raise RuntimeError(
        f"Expected 40 Stage27 ledgers, "
        f"found {len(stage27_ledgers)}"
    )


for path in stage27_ledgers:

    obj = read_json(
        path
    )

    pre = int(
        obj[
            "pre_component_new_fits_consumed"
        ]
    )

    successes = int(
        obj[
            "this_component_successful_new_fits"
        ]
    )

    cumulative = int(
        obj[
            "cumulative_new_fits_consumed"
        ]
    )

    cid = obj[
        "component_id"
    ]


    if successes != 1:
        raise RuntimeError(
            f"{path}: Stage27 component "
            "did not consume exactly one fit."
        )


    if cumulative != (
        pre
        + 1
    ):
        raise RuntimeError(
            f"{path}: Stage27 ledger discontinuity."
        )


    register_fit(
        cid,
        cumulative,
        path,
    )


    ledger_rows.append(
        {
            "ledger_path": str(
                path.relative_to(
                    REPO
                )
            ),
            "stage": obj[
                "stage"
            ],
            "pre_new_fits_consumed": pre,
            "successful_new_fits_this_ledger": 1,
            "cumulative_new_fits_consumed": cumulative,
            "new_fits_remaining": int(
                obj[
                    "new_fits_remaining"
                ]
            ),
            "component_ids": cid,
            "status": obj[
                "status"
            ],
        }
    )


# -------------------------------------------------------------------------------------------------
# Stage28B final ledgers
# -------------------------------------------------------------------------------------------------

stage28b_ledgers = sorted(
    STAGE28B_ROOT.rglob(
        "*_fit_ledger.json"
    )
)


if len(
    stage28b_ledgers
) != 50:
    raise RuntimeError(
        f"Expected 50 Stage28B final ledgers, "
        f"found {len(stage28b_ledgers)}"
    )


for path in stage28b_ledgers:

    obj = read_json(
        path
    )

    pre = int(
        obj[
            "pre_component_new_fits_consumed"
        ]
    )

    successes = int(
        obj[
            "this_component_successful_new_fits"
        ]
    )

    cumulative = int(
        obj[
            "cumulative_new_fits_consumed"
        ]
    )

    cid = obj[
        "component_id"
    ]


    if successes != 1:
        raise RuntimeError(
            f"{path}: Stage28B component "
            "did not consume exactly one fit."
        )


    if cumulative != (
        pre
        + 1
    ):
        raise RuntimeError(
            f"{path}: Stage28B ledger discontinuity."
        )


    register_fit(
        cid,
        cumulative,
        path,
    )


    ledger_rows.append(
        {
            "ledger_path": str(
                path.relative_to(
                    REPO
                )
            ),
            "stage": obj[
                "stage"
            ],
            "pre_new_fits_consumed": pre,
            "successful_new_fits_this_ledger": 1,
            "cumulative_new_fits_consumed": cumulative,
            "new_fits_remaining": int(
                obj[
                    "new_fits_remaining"
                ]
            ),
            "component_ids": cid,
            "status": obj[
                "status"
            ],
        }
    )


manifest_new_ids = {
    row[
        "component_id"
    ]
    for row in new_rows
}


if set(
    fit_evidence
) != manifest_new_ids:

    missing = sorted(
        manifest_new_ids
        - set(
            fit_evidence
        )
    )

    extra = sorted(
        set(
            fit_evidence
        )
        - manifest_new_ids
    )

    raise RuntimeError(
        "New-fit component coverage mismatch.\n"
        f"missing={missing}\n"
        f"extra={extra}"
    )


if sorted(
    fit_ordinals_seen
) != list(
    range(
        1,
        109,
    )
):
    raise RuntimeError(
        "Fit ordinals are not exactly FIT #1 .. FIT #108."
    )


ledger_rows.sort(
    key=lambda row:
    row[
        "cumulative_new_fits_consumed"
    ]
)


running = 0


for row in ledger_rows:

    if int(
        row[
            "pre_new_fits_consumed"
        ]
    ) != running:

        raise RuntimeError(
            "Global ledger discontinuity before "
            f"{row['stage']}: "
            f"pre={row['pre_new_fits_consumed']} "
            f"expected={running}"
        )


    running = int(
        row[
            "cumulative_new_fits_consumed"
        ]
    )


if running != 108:
    raise RuntimeError(
        f"Final cumulative fit ledger = {running}, "
        "expected 108."
    )


if int(
    ledger_rows[
        -1
    ][
        "new_fits_remaining"
    ]
) != 0:
    raise RuntimeError(
        "Final fit ledger does not close remaining budget at zero."
    )


print(
    "[PASS] all 108 NEW component IDs accounted exactly once"
)

print(
    "[PASS] fit ordinals exactly FIT #1 .. FIT #108"
)

print(
    "[PASS] cumulative ledger closes at 108 consumed / 0 remaining"
)


# =================================================================================================
# 6. RESULT / COMPONENT / MODEL IDENTITY
# =================================================================================================

banner(
    "STAGE28-3A — RESULT / COMPONENT IDENTITY RECONCILIATION"
)


manifest_by_id = {
    row[
        "component_id"
    ]: row
    for row in manifest_rows
}


component_evidence = {}


# -------------------------------------------------------------------------------------------------
# Stage22: 10 result cells cover C001..C020
# -------------------------------------------------------------------------------------------------

stage22_results = sorted(
    STAGE22_ROOT.rglob(
        "*_result.json"
    )
)


if len(
    stage22_results
) != 10:
    raise RuntimeError(
        f"Expected 10 Stage22 results, "
        f"found {len(stage22_results)}"
    )


for path in stage22_results:

    obj = read_json(
        path
    )


    if obj.get(
        "experiment"
    ) != "STAGE22_FULL":
        raise RuntimeError(
            f"{path}: unexpected Stage22 experiment."
        )


    for model_name, model in obj[
        "models"
    ].items():

        cid = model[
            "component_id"
        ]


        if (
            cid
            in component_evidence
        ):
            raise RuntimeError(
                f"Duplicate component evidence: {cid}"
            )


        manifest_row = (
            manifest_by_id[
                cid
            ]
        )


        if model[
            "fit_action"
        ] != manifest_row[
            "fit_action"
        ]:
            raise RuntimeError(
                f"{cid}: fit_action differs from manifest."
            )


        if int(
            model[
                "seed"
            ]
        ) != int(
            manifest_row[
                "model_seed"
            ]
        ):
            raise RuntimeError(
                f"{cid}: model seed differs from manifest."
            )


        if model[
            "parameter_sha256"
        ] != manifest_row[
            "parameter_sha256"
        ]:
            raise RuntimeError(
                f"{cid}: parameter SHA differs from manifest."
            )


        learner = (
            "XGBOOST"
            if model_name.lower()
            == "xgboost"
            else "LIGHTGBM"
        )


        if learner != manifest_row[
            "learner"
        ]:
            raise RuntimeError(
                f"{cid}: learner differs from manifest."
            )


        if model[
            "fit_action"
        ] == "NEW_FIT_AUTHORIZED":

            artifact = (
                path.parent
                / model[
                    "model_path"
                ]
            )

            if not artifact.is_file():
                raise RuntimeError(
                    f"{cid}: Stage22 model missing:\n"
                    f"{artifact}"
                )

            actual_sha = sha256_file(
                artifact
            )

            if actual_sha != model[
                "model_sha256"
            ]:
                raise RuntimeError(
                    f"{cid}: Stage22 model SHA mismatch."
                )

            model_path = str(
                artifact.relative_to(
                    REPO
                )
            )

            model_sha = actual_sha

        else:

            historical = (
                REPO
                / model[
                    "historical_model_path"
                ]
            )

            if not historical.is_file():
                raise RuntimeError(
                    f"{cid}: historical Stage22 model missing."
                )

            actual_sha = sha256_file(
                historical
            )

            if actual_sha != model[
                "historical_model_sha256"
            ]:
                raise RuntimeError(
                    f"{cid}: historical Stage22 SHA mismatch."
                )

            if actual_sha != manifest_row[
                "reused_model_sha256"
            ]:
                raise RuntimeError(
                    f"{cid}: Stage22 reuse SHA differs "
                    "from frozen manifest."
                )

            model_path = model[
                "historical_model_path"
            ]

            model_sha = actual_sha


        component_evidence[
            cid
        ] = {
            "model_path": model_path,
            "model_sha256": model_sha,
            "result_path": str(
                path.relative_to(
                    REPO
                )
            ),
        }


# -------------------------------------------------------------------------------------------------
# Stage27 and Stage28B new-component result helper
# -------------------------------------------------------------------------------------------------

def audit_single_component_result(
    path,
    expected_experiment,
):

    obj = read_json(
        path
    )

    cid = obj[
        "component_id"
    ]


    if cid in component_evidence:
        raise RuntimeError(
            f"Duplicate component evidence: {cid}"
        )


    manifest_row = (
        manifest_by_id[
            cid
        ]
    )


    if obj[
        "experiment"
    ] != expected_experiment:
        raise RuntimeError(
            f"{cid}: experiment mismatch."
        )


    if obj[
        "learner"
    ] != manifest_row[
        "learner"
    ]:
        raise RuntimeError(
            f"{cid}: learner mismatch."
        )


    if int(
        obj[
            "training_seed"
        ]
    ) != int(
        manifest_row[
            "model_seed"
        ]
    ):
        raise RuntimeError(
            f"{cid}: training seed mismatch."
        )


    if obj[
        "model"
    ][
        "parameter_sha256"
    ] != manifest_row[
        "parameter_sha256"
    ]:
        raise RuntimeError(
            f"{cid}: parameter SHA mismatch."
        )


    if obj[
        "compute_backend"
    ].upper() != "CPU":
        raise RuntimeError(
            f"{cid}: non-CPU result."
        )


    artifact = (
        path.parent
        / obj[
            "model"
        ][
            "model_artifact"
        ]
    )


    if not artifact.is_file():
        raise RuntimeError(
            f"{cid}: model artifact missing:\n"
            f"{artifact}"
        )


    actual_sha = sha256_file(
        artifact
    )


    if actual_sha != obj[
        "model"
    ][
        "model_sha256"
    ]:
        raise RuntimeError(
            f"{cid}: model SHA mismatch."
        )


    component_evidence[
        cid
    ] = {
        "model_path": str(
            artifact.relative_to(
                REPO
            )
        ),
        "model_sha256": actual_sha,
        "result_path": str(
            path.relative_to(
                REPO
            )
        ),
    }


    return obj


# -------------------------------------------------------------------------------------------------
# Stage27 NEW components
# -------------------------------------------------------------------------------------------------

stage27_results = sorted(
    STAGE27_ROOT.rglob(
        "*_result.json"
    )
)


if len(
    stage27_results
) != 40:
    raise RuntimeError(
        f"Expected 40 Stage27 new-component results, "
        f"found {len(stage27_results)}"
    )


loaded_stage27_results = []


for path in stage27_results:

    obj = audit_single_component_result(
        path,
        "STAGE27_CHRONOLOGY_LOAO",
    )

    loaded_stage27_results.append(
        obj
    )


# -------------------------------------------------------------------------------------------------
# Stage28B NEW components
# -------------------------------------------------------------------------------------------------

stage28b_results = sorted(
    STAGE28B_ROOT.rglob(
        "*_result.json"
    )
)


if len(
    stage28b_results
) != 50:
    raise RuntimeError(
        f"Expected 50 Stage28B results, "
        f"found {len(stage28b_results)}"
    )


loaded_stage28b_results = []


for path in stage28b_results:

    preview = read_json(
        path
    )

    cid = preview[
        "component_id"
    ]

    expected_experiment = (
        manifest_by_id[
            cid
        ][
            "experiment"
        ]
    )

    obj = audit_single_component_result(
        path,
        expected_experiment,
    )

    loaded_stage28b_results.append(
        obj
    )


# -------------------------------------------------------------------------------------------------
# Stage27 seed42 reuse components did not require new Stage28 result directories.
# Their frozen historical model identity is the component evidence.
# -------------------------------------------------------------------------------------------------

for cid, evidence in reuse_evidence.items():

    if cid not in component_evidence:

        component_evidence[
            cid
        ] = {
            "model_path": evidence[
                "model_path"
            ],
            "model_sha256": evidence[
                "model_sha256"
            ],
            "result_path": "",
        }


if set(
    component_evidence
) != set(
    component_ids
):

    missing = sorted(
        set(
            component_ids
        )
        - set(
            component_evidence
        )
    )

    extra = sorted(
        set(
            component_evidence
        )
        - set(
            component_ids
        )
    )

    raise RuntimeError(
        "120-component evidence coverage mismatch.\n"
        f"missing={missing}\n"
        f"extra={extra}"
    )


print(
    "[PASS] all 120 frozen component obligations have model evidence"
)

print(
    "[PASS] all 108 new components reconcile to durable fit ledgers"
)

print(
    "[PASS] all Stage28-produced result identities reconcile "
    "learner / seed / parameter SHA / model SHA"
)

print(
    "[PASS] all 12 historical reuses reconcile to frozen model SHA"
)


# =================================================================================================
# 7. FINAL-HOLDOUT / ANTI-ADAPTATION GATE
# =================================================================================================

banner(
    "STAGE28-3A — FINAL-HOLDOUT / ANTI-ADAPTATION GATE"
)


loaded_stage22_results = [
    read_json(
        path
    )
    for path in stage22_results
]


for path, obj in zip(
    stage22_results,
    loaded_stage22_results,
):

    sci = obj.get(
        "scientific_accounting",
        {},
    )


    for key in [
        "shared_final_holdout_openings",
        "shared_final_holdout_predictor_rows_read",
        "shared_final_holdout_labels_read",
        "target_adaptive_choices",
    ]:

        if int(
            sci.get(
                key,
                -1,
            )
        ) != 0:
            raise RuntimeError(
                f"{path}: {key} is not zero."
            )


    if (
        obj.get(
            "threshold_selection",
            {},
        ).get(
            "final_holdout_threshold_search"
        )
        != "FORBIDDEN"
    ):
        raise RuntimeError(
            f"{path}: final-holdout threshold search "
            "was not marked FORBIDDEN."
        )


all_results = (
    loaded_stage22_results
    + loaded_stage27_results
    + loaded_stage28b_results
)


false_keys = [
    "target_used_for_fit",
    "target_used_for_class_weight",
    "target_used_for_threshold_selection",
    "target_threshold_search",
    "post_target_parameter_change",
    "post_target_fit_branching",
]


zero_keys = [
    "target_adaptive_choices",
]


anti_counts = {}


for key in false_keys:

    values = []

    for obj in all_results:

        values.extend(
            walk_key_values(
                obj,
                key,
            )
        )


    anti_counts[
        key
    ] = require_all(
        values,
        lambda value:
        value is False,
        key,
    )


for key in zero_keys:

    values = []

    for obj in all_results:

        values.extend(
            walk_key_values(
                obj,
                key,
            )
        )


    anti_counts[
        key
    ] = require_all(
        values,
        lambda value:
        int(
            value
        ) == 0,
        key,
    )


target_rows_used_values = []


for obj in all_results:

    target_rows_used_values.extend(
        walk_key_values(
            obj,
            "target_rows_used",
        )
    )


target_rows_used_count = require_all(
    target_rows_used_values,
    lambda value:
    int(
        value
    ) == 0,
    "target_rows_used",
)


for key in [
    "held_out_family_train_count",
    "held_out_family_validation_count",
]:

    values = []

    for obj in all_results:

        values.extend(
            walk_key_values(
                obj,
                key,
            )
        )


    require_all(
        values,
        lambda value:
        int(
            value
        ) == 0,
        key,
    )


print(
    "[PASS] Stage22 shared final holdout openings = 0"
)

print(
    "[PASS] Stage22 shared final holdout predictor reads = 0"
)

print(
    "[PASS] Stage22 shared final holdout label reads = 0"
)

print(
    "[PASS] final-holdout threshold search = FORBIDDEN"
)

print(
    "[PASS] recorded target-adaptation booleans all false"
)

print(
    "[PASS] recorded target_rows_used values all zero"
)

print(
    "[PASS] held-out family train/validation exposure = zero "
    "where explicitly recorded"
)


# =================================================================================================
# 8. AUTO-C TWO-PHASE DURABILITY COMPLETENESS
# =================================================================================================

banner(
    "STAGE28-3A — AUTO-C TWO-PHASE DURABILITY GATE"
)


fit_checkpoint_files = sorted(
    STAGE28B_ROOT.rglob(
        "*_fit_checkpoint.json"
    )
)

progress_files = sorted(
    STAGE28B_ROOT.rglob(
        "*_execution_progress.json"
    )
)


if len(
    fit_checkpoint_files
) != 50:
    raise RuntimeError(
        f"Expected 50 fit-only checkpoints, "
        f"found {len(fit_checkpoint_files)}"
    )


if len(
    progress_files
) != 50:
    raise RuntimeError(
        f"Expected 50 AUTO-C progress receipts, "
        f"found {len(progress_files)}"
    )


# Each Stage28B execution directory must contain exactly one of each
# two-phase/final receipt class.
for directory in stage28b_dirs:

    if len(
        list(
            directory.glob(
                "*_fit_checkpoint.json"
            )
        )
    ) != 1:
        raise RuntimeError(
            f"{directory}: fit-checkpoint receipt count != 1"
        )

    if len(
        list(
            directory.glob(
                "*_execution_progress.json"
            )
        )
    ) != 1:
        raise RuntimeError(
            f"{directory}: execution-progress receipt count != 1"
        )

    if len(
        list(
            directory.glob(
                "*_fit_ledger.json"
            )
        )
    ) != 1:
        raise RuntimeError(
            f"{directory}: final fit-ledger count != 1"
        )

    if len(
        list(
            directory.glob(
                "*_result.json"
            )
        )
    ) != 1:
        raise RuntimeError(
            f"{directory}: final result count != 1"
        )


print(
    "[PASS] Stage28B fit-only checkpoints = 50 / 50"
)

print(
    "[PASS] Stage28B execution-progress receipts = 50 / 50"
)

print(
    "[PASS] Stage28B final ledgers = 50 / 50"
)

print(
    "[PASS] Stage28B final results = 50 / 50"
)


# =================================================================================================
# 9. BUILD 120-COMPONENT AUDIT TABLE
# =================================================================================================

banner(
    "STAGE28-3A — BUILD PERMANENT CLOSURE ARTIFACTS"
)


component_rows = []


for manifest_row in manifest_rows:

    cid = manifest_row[
        "component_id"
    ]

    evidence = (
        component_evidence[
            cid
        ]
    )


    if (
        manifest_row[
            "fit_action"
        ]
        == "NEW_FIT_AUTHORIZED"
    ):

        fit_ordinal = (
            fit_evidence[
                cid
            ][
                "fit_ordinal"
            ]
        )

        ledger_path = (
            fit_evidence[
                cid
            ][
                "ledger_path"
            ]
        )

    else:

        fit_ordinal = ""

        ledger_path = ""


    component_rows.append(
        {
            "component_ordinal": int(
                manifest_row[
                    "component_ordinal"
                ]
            ),
            "component_id": cid,
            "arm": manifest_row[
                "arm"
            ],
            "experiment": manifest_row[
                "experiment"
            ],
            "unit": manifest_row[
                "unit"
            ],
            "evaluation_cell_id": manifest_row[
                "evaluation_cell_id"
            ],
            "learner": manifest_row[
                "learner"
            ],
            "configuration_id": manifest_row[
                "configuration_id"
            ],
            "model_seed": int(
                manifest_row[
                    "model_seed"
                ]
            ),
            "compute_backend": manifest_row[
                "compute_backend"
            ],
            "fit_action": manifest_row[
                "fit_action"
            ],
            "fit_ordinal_if_new": fit_ordinal,
            "parameter_sha256": manifest_row[
                "parameter_sha256"
            ],
            "model_evidence_path": evidence[
                "model_path"
            ],
            "model_sha256": evidence[
                "model_sha256"
            ],
            "stage28_result_evidence_path": evidence[
                "result_path"
            ],
            "ledger_evidence_path": ledger_path,
            "closure_status": "PASS",
        }
    )


if len(
    component_rows
) != 120:
    raise RuntimeError(
        "Component audit table is not 120 rows."
    )


# =================================================================================================
# 10. WRITE CLOSURE ARTIFACTS
# =================================================================================================

OUT.mkdir(
    parents=False,
    exist_ok=False,
)


component_fields = list(
    component_rows[
        0
    ].keys()
)


with COMPONENT_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=component_fields,
    )

    writer.writeheader()

    writer.writerows(
        component_rows
    )


ledger_fields = [
    "ledger_path",
    "stage",
    "pre_new_fits_consumed",
    "successful_new_fits_this_ledger",
    "cumulative_new_fits_consumed",
    "new_fits_remaining",
    "component_ids",
    "status",
]


with LEDGER_CSV.open(
    "w",
    encoding="utf-8",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=ledger_fields,
    )

    writer.writeheader()

    writer.writerows(
        ledger_rows
    )


receipt = {

    "stage": "Stage28-3A",

    "type": (
        "EXPERIMENT_CLOSURE_AND_108_FIT_AUDIT"
    ),

    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),

    "scientific_parent_commit": (
        EXPECTED_PARENT
    ),

    "frozen_component_manifest": {

        "path": str(
            MANIFEST.relative_to(
                REPO
            )
        ),

        "sha256": manifest_sha,

        "components": 120,

        "new_fit_components": 108,

        "reuse_components": 12,
    },

    "fit_budget_closure": {

        "authorized_new_fits": 108,

        "consumed_new_fits": 108,

        "remaining_new_fits": 0,

        "fit_ordinals": (
            "FIT_001_THROUGH_FIT_108_CONTIGUOUS"
        ),

        "new_fit_component_coverage": (
            "EXACT_108_OF_108"
        ),

        "reuse_component_coverage": (
            "EXACT_12_OF_12"
        ),

        "new_model_artifacts": {

            "stage22": len(
                stage22_models
            ),

            "stage27": len(
                stage27_models
            ),

            "stage28b": len(
                stage28b_models
            ),

            "total": len(
                all_new_models
            ),
        },
    },

    "execution_root_closure": {

        "stage22_output_directories": len(
            stage22_dirs
        ),

        "stage27_new_fit_output_directories": len(
            stage27_dirs
        ),

        "stage28b_output_directories": len(
            stage28b_dirs
        ),

        "total_stage28_output_directories_checksum_audited": len(
            checksum_dirs
        ),

        "checksum_payload_files_verified": (
            payload_files_verified
        ),

        "checksum_payload_bytes_verified": (
            payload_bytes_verified
        ),
    },

    "component_obligations": {

        "stage22": stage22_counts,

        "stage27": stage27_counts,

        "stage28b": stage28b_counts,

        "all_120_component_model_identities_reconciled": True,

        "all_12_reused_model_sha256_verified": True,

        "all_108_new_fit_ledger_components_verified": True,

        "all_stage28_result_parameter_sha256_reconciled": True,

        "all_stage28_result_model_sha256_reconciled": True,

        "all_compute_backends_cpu": True,
    },

    "anti_adaptation": {

        "shared_stage22_final_holdout_openings_during_stage28_execution": 0,

        "shared_stage22_final_holdout_predictor_rows_read_during_stage28_execution": 0,

        "shared_stage22_final_holdout_labels_read_during_stage28_execution": 0,

        "stage22_final_holdout_threshold_search": (
            "FORBIDDEN"
        ),

        "target_derived_fit_or_threshold_adaptation": 0,

        "recorded_false_key_occurrences": (
            anti_counts
        ),

        "recorded_target_rows_used_occurrences_verified_zero": (
            target_rows_used_count
        ),
    },

    "stage28b_two_phase_durability": {

        "fit_checkpoint_receipts": len(
            fit_checkpoint_files
        ),

        "execution_progress_receipts": len(
            progress_files
        ),

        "final_fit_ledgers": len(
            stage28b_ledgers
        ),

        "final_results": len(
            stage28b_results
        ),

        "status": (
            "50_OF_50_COMPLETE"
        ),
    },

    "scientific_scope": {

        "new_model_fits_this_stage": 0,

        "model_inference_this_stage": 0,

        "threshold_selection_this_stage": 0,

        "target_openings_this_stage": 0,

        "shared_final_holdout_openings_this_stage": 0,

        "data_dependent_model_changes_this_stage": 0,

        "new_model_fits_authorized_after_closure": 0,

        "infiltration_status": (
            "DESCRIPTIVE_ONLY_SUPPORT_36_LT_50"
        ),

        "random_loao_status": (
            "CONTROL_NOT_DEPLOYMENT_ESTIMATE"
        ),

        "family_specific_loao_primary": True,

        "aggregate_zero_day_score_created": False,
    },

    "closure_status": (
        "PASS_STAGE28_NEW_MODEL_FITTING_PERMANENTLY_CLOSED"
    ),

    "next_authorized_step": (
        "Stage28-3B — seed uncertainty and conclusion-stability synthesis. "
        "ZERO new fits. The shared Stage22 final holdout remains closed until "
        "the separately preregistered one-time Stage28-4 inference."
    ),
}


write_json(
    RECEIPT,
    receipt,
)


README.write_text(
    f"""# Stage28-3A — Experiment Closure and 108-Fit Audit

Scientific parent: `{EXPECTED_PARENT}`

## Closure

- Frozen components: 120
- Authorized new fits: 108
- Historical reuses: 12
- New fits consumed: 108
- New fits remaining: 0
- New model artifacts verified: 108
- Stage28 output directories checksum-audited: 100
- Stage28B two-phase fit checkpoints: 50/50
- Shared Stage22 final-holdout openings during Stage28 execution: 0
- Target-derived fit/threshold adaptation: 0
- New fits performed by Stage28-3A: 0

## Scientific boundary

Stage28 model fitting is closed.

No additional fit, tuning branch, learner, family, dataset, feature branch,
or hyperparameter search is authorized for this manuscript.

The next authorized work is zero-fit Stage28 synthesis.

The shared Stage22 final holdout remains closed until its separately
preregistered one-time inference step.

Infiltration remains descriptive-only because support is 36 (<50).

Random LOAO remains a control, not a deployment estimate.

Family-specific LOAO results remain primary. No aggregate zero-day score
is created.
""",
    encoding="utf-8",
)


for path in [
    RECEIPT,
    COMPONENT_CSV,
    LEDGER_CSV,
    README,
]:

    if (
        not path.is_file()
        or path.stat().st_size
        == 0
    ):
        raise RuntimeError(
            f"Failed to write closure artifact:\n"
            f"{path}"
        )


with CHECKSUMS.open(
    "w",
    encoding="utf-8",
) as f:

    for path in [
        RECEIPT,
        COMPONENT_CSV,
        LEDGER_CSV,
        README,
    ]:

        f.write(
            f"{sha256_file(path)}  "
            f"{path.name}\n"
        )


verify_checksum_manifest(
    CHECKSUMS
)


print(
    "[PASS] closure receipt written"
)

print(
    "[PASS] 120-row component closure table written"
)

print(
    "[PASS] new-fit ledger audit table written"
)

print(
    "[PASS] closure artifact checksums verified"
)


# =================================================================================================
# 11. EXACT GIT COMMIT UNIVERSE
# =================================================================================================

banner(
    "STAGE28-3A — DURABLE COMMIT / PUSH"
)


expected_rel = {
    str(
        path.relative_to(
            REPO
        )
    )
    for path in [
        RECEIPT,
        COMPONENT_CSV,
        LEDGER_CSV,
        README,
        CHECKSUMS,
    ]
}


tracked_modifications = set(
    git(
        "diff",
        "--name-only",
    ).splitlines()
)


staged_before = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)


untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
)


if tracked_modifications:
    raise RuntimeError(
        "Unexpected tracked modifications before closure commit:\n"
        + "\n".join(
            sorted(
                tracked_modifications
            )
        )
    )


if staged_before:
    raise RuntimeError(
        "Unexpected staged files before closure commit:\n"
        + "\n".join(
            sorted(
                staged_before
            )
        )
    )


if untracked != expected_rel:
    raise RuntimeError(
        "Unexpected untracked universe before closure commit.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                expected_rel
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                untracked
            )
        )
    )


for rel in sorted(
    expected_rel
):

    run(
        [
            "git",
            "add",
            "--",
            rel,
        ]
    )


staged_after = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)


if staged_after != expected_rel:
    raise RuntimeError(
        "Stage28-3A staged universe mismatch."
    )


run(
    [
        "git",
        "config",
        "user.name",
        "Stage28 Kaggle",
    ]
)

run(
    [
        "git",
        "config",
        "user.email",
        "stage28-kaggle@users.noreply.github.com",
    ]
)


commit_message = (
    "stage28-3a: close 108-fit experiment ledger"
)


commit_result = run(
    [
        "git",
        "commit",
        "-m",
        commit_message,
    ]
)


print(
    commit_result.stdout.strip()
)


new_head = git(
    "rev-parse",
    "HEAD",
)

parent = git(
    "rev-parse",
    "HEAD^",
)

subject = git(
    "show",
    "-s",
    "--format=%s",
    "HEAD",
)


if parent != EXPECTED_PARENT:
    raise RuntimeError(
        "Stage28-3A commit parent mismatch."
    )


if subject != commit_message:
    raise RuntimeError(
        "Stage28-3A commit subject mismatch."
    )


token, token_source = (
    recover_github_token()
)


print()
print(
    "[PASS] GitHub credential:",
    token_source,
)

print(
    "[PASS] token not displayed"
)


push_output = (
    authenticated_push(
        token
    )
)


if push_output:
    print(
        push_output
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local = git(
    "rev-parse",
    "HEAD",
)

origin = git(
    "rev-parse",
    "origin/main",
)

remote = remote_head()


if not (
    local
    == origin
    == remote
    == new_head
):
    raise RuntimeError(
        "Stage28-3A remote durability verification failed."
    )


if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Repository dirty after Stage28-3A push."
    )


# =================================================================================================
# 12. FINAL CLOSURE
# =================================================================================================

banner(
    "STAGE28-3A — EXPERIMENT CLOSURE COMPLETE"
)


print(
    "Scientific parent:"
)

print(
    " ",
    EXPECTED_PARENT,
)


print()
print(
    "Closure commit:"
)

print(
    " ",
    new_head,
)


print()
print(
    "Frozen components : 120 / 120 VERIFIED"
)

print(
    "New fits          : 108 / 108 COMPLETE"
)

print(
    "Historical reuses : 12 / 12 VERIFIED"
)

print(
    "Remaining fits    : 0"
)

print(
    "New model artifacts:",
    "108 / 108 VERIFIED",
)

print(
    "Checksum-audited execution dirs:",
    "100 / 100",
)

print(
    "Stage28B fit checkpoints:",
    "50 / 50",
)


print()
print(
    "Stage22 shared-final-holdout openings:",
    0,
)

print(
    "Target-derived fit/threshold adaptation:",
    0,
)

print(
    "New fits performed by Stage28-3A:",
    0,
)


print()
print(
    "NEW MODEL FITTING IS NOW CLOSED "
    "FOR THIS MANUSCRIPT."
)


print()
print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "Stage28-3B — seed uncertainty / "
    "conclusion-stability synthesis"
)

print(
    "ZERO NEW FITS."
)


STAGE28-3A — EXPERIMENT CLOSURE / REPOSITORY GATE

Expected parent: 9fddb8d8c34ba8f81b71f24eea15c90151053d6b
Local HEAD     : 9fddb8d8c34ba8f81b71f24eea15c90151053d6b
origin/main    : 9fddb8d8c34ba8f81b71f24eea15c90151053d6b
Remote main    : 9fddb8d8c34ba8f81b71f24eea15c90151053d6b

[PASS] Stage28 experiment parent exact and synchronized
[PASS] repository clean
[PASS] ZERO fits / ZERO inference / ZERO threshold selection

STAGE28-3A — FROZEN MANIFEST / BUDGET GATE

Manifest SHA256: 47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505
[PASS] 120 components exact
[PASS] 108 NEW + 12 REUSE exact
[PASS] Stage22 = 18 new + 2 reuse
[PASS] Stage27 = 40 new + 10 reuse
[PASS] Stage28B = 50 new + 0 reuse
[PASS] all components CPU as frozen

STAGE28-3A — HISTORICAL REUSE IDENTITY GATE

[PASS reuse] C002 STAGE22_FULL RANDOM_NATURAL LIGHTGBM seed42
[PASS reuse] C012 STAGE22_FULL CHRONOLOGICAL_NATURAL LIGHTGBM seed42
[PASS reuse] C021 STAGE27_CHRONOLOGY_LOAO BOT XGBOOST seed42
[PASS reu

KeyError: 'model_fits_attempted'

In [6]:
# =================================================================================================
# STAGE28-3A-R1 — RECEIPT SCHEMA COMPATIBILITY REPAIR
#
# ZERO FITS
# ZERO INFERENCE
# ZERO THRESHOLD SELECTION
# ZERO TARGET OPENINGS
#
# Repairs only historical Stage28 receipt-schema differences:
#
#   1. model_fits_attempted/model_fits_successful are optional redundant counters.
#      If present -> must equal successful_new_fits.
#      If absent  -> authoritative successful_new_fits + cumulative ledger remains the check.
#
#   2. Stage22 result model artifact key:
#         older schema : model_path
#         AUTO-A schema: model
#
#   3. fit_action:
#         older schema: explicitly stored in result
#         AUTO-A schema: omitted; frozen component manifest is authoritative
#
#   4. final_holdout_threshold_search:
#         some compact AUTO-A receipts omit the redundant string field.
#         The audit still requires shared-final-holdout predictor reads,
#         label reads, and openings to all equal zero.
#
# Then reruns the complete Stage28-3A audit from the beginning.
# =================================================================================================

from pathlib import Path
import hashlib
import subprocess


SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_HEAD = (
    "9fddb8d8c34ba8f81b71f24eea15c90151053d6b"
)

OUT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_3a_experiment_closure_audit"
)


def banner(text):
    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def run(cmd, *, cwd=REPO, check=True):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            + " ".join(map(str, cmd))
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return run(
        ["git", *args]
    ).stdout.strip()


# =================================================================================================
# 1. PRE-REPAIR STATE GATE
# =================================================================================================

banner(
    "STAGE28-3A-R1 — PRE-REPAIR STATE GATE"
)


if not (
    REPO
    / ".git"
).is_dir():
    raise RuntimeError(
        "Stage28 repository missing."
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)


print(
    "Local HEAD :",
    local_head,
)

print(
    "origin/main:",
    origin_head,
)

print(
    "Expected   :",
    EXPECTED_HEAD,
)


if (
    local_head != EXPECTED_HEAD
    or origin_head != EXPECTED_HEAD
):
    raise RuntimeError(
        "Repository state changed after the failed audit. "
        "STOP rather than patching a different parent."
    )


status = git(
    "status",
    "--porcelain",
)


if status:
    raise RuntimeError(
        "Repository is not clean:\n"
        + status
    )


if OUT.exists():
    raise RuntimeError(
        "Stage28-3A output directory already exists:\n"
        f"{OUT}\n\n"
        "Do not overwrite it automatically."
    )


print()
print(
    "[PASS] scientific parent unchanged"
)

print(
    "[PASS] repository clean"
)

print(
    "[PASS] Stage28-3A produced no durable output before failure"
)

print(
    "[PASS] no model fit can be triggered by this repair"
)


# =================================================================================================
# 2. RECOVER THE ORIGINAL STAGE28-3A CELL FROM NOTEBOOK HISTORY
# =================================================================================================

banner(
    "RECOVER ORIGINAL STAGE28-3A CELL"
)


try:
    ip = get_ipython()
except NameError:
    ip = None


if ip is None:
    raise RuntimeError(
        "IPython notebook history unavailable."
    )


history = list(
    ip.history_manager.input_hist_raw
)


original = None
original_history_index = None


# Exclude the current repair cell itself.
search_space = (
    history[:-1]
    if len(history) > 1
    else history
)


for index in range(
    len(search_space) - 1,
    -1,
    -1,
):

    cell = search_space[index]

    if not isinstance(
        cell,
        str,
    ):
        continue

    # Strong fingerprint of the full closure cell.
    if (
        "STAGE28-3A — EXPERIMENT CLOSURE + 108-FIT AUDIT"
        in cell
        and
        "stage28_3a_component_closure_audit.csv"
        in cell
        and
        "model_fits_attempted"
        in cell
        and
        "checksum_payload_bytes_verified"
        in cell
        and
        "NEW MODEL FITTING IS NOW CLOSED"
        in cell
    ):
        original = cell
        original_history_index = index
        break


if original is None:
    raise RuntimeError(
        "Could not locate the original Stage28-3A audit cell "
        "in Kaggle notebook history."
    )


original_sha = hashlib.sha256(
    original.encode(
        "utf-8"
    )
).hexdigest()


print(
    "[PASS] original Stage28-3A cell recovered"
)

print(
    "History index:",
    original_history_index,
)

print(
    "Original cell SHA256:",
    original_sha,
)

print(
    "Original characters:",
    f"{len(original):,}",
)


# =================================================================================================
# 3. PATCH OPTIONAL LEDGER COUNTERS
# =================================================================================================

banner(
    "PATCH 1/4 — OPTIONAL LEDGER COUNTERS"
)


old_attempted = '''obj[
            "model_fits_attempted"
        ]'''

new_attempted = '''obj.get(
            "model_fits_attempted",
            successes,
        )'''


attempted_count = original.count(
    old_attempted
)


if attempted_count != 1:
    raise RuntimeError(
        "Expected exactly one model_fits_attempted "
        f"schema-sensitive access; found {attempted_count}."
    )


patched = original.replace(
    old_attempted,
    new_attempted,
    1,
)


old_successful = '''obj[
            "model_fits_successful"
        ]'''

new_successful = '''obj.get(
            "model_fits_successful",
            successes,
        )'''


successful_count = patched.count(
    old_successful
)


if successful_count != 1:
    raise RuntimeError(
        "Expected exactly one model_fits_successful "
        f"schema-sensitive access; found {successful_count}."
    )


patched = patched.replace(
    old_successful,
    new_successful,
    1,
)


print(
    "[PASS] optional model_fits_attempted handled"
)

print(
    "[PASS] optional model_fits_successful handled"
)

print(
    "[PASS] successful_new_fits and cumulative ledger "
    "remain mandatory"
)


# =================================================================================================
# 4. PATCH OPTIONAL STAGE22 fit_action
# =================================================================================================

banner(
    "PATCH 2/4 — STAGE22 fit_action SCHEMA"
)


old_fit_action = '''model[
            "fit_action"
        ]'''

new_fit_action = '''model.get(
            "fit_action",
            manifest_row["fit_action"],
        )'''


fit_action_count = patched.count(
    old_fit_action
)


print(
    "Schema-sensitive fit_action accesses:",
    fit_action_count,
)


if fit_action_count != 2:
    raise RuntimeError(
        "Expected exactly two Stage22 model['fit_action'] "
        f"accesses; found {fit_action_count}."
    )


patched = patched.replace(
    old_fit_action,
    new_fit_action,
)


print(
    "[PASS] explicit fit_action still checked when present"
)

print(
    "[PASS] frozen manifest supplies authoritative "
    "fit_action when compact AUTO-A receipt omits it"
)


# =================================================================================================
# 5. PATCH Stage22 model_path / model FIELD
# =================================================================================================

banner(
    "PATCH 3/4 — STAGE22 MODEL ARTIFACT KEY"
)


old_model_path = '''model[
                    "model_path"
                ]'''

new_model_path = '''(
                    model.get("model_path")
                    or model.get("model")
                )'''


model_path_count = patched.count(
    old_model_path
)


if model_path_count != 1:
    raise RuntimeError(
        "Expected exactly one Stage22 model_path "
        f"schema-sensitive access; found {model_path_count}."
    )


patched = patched.replace(
    old_model_path,
    new_model_path,
    1,
)


print(
    "[PASS] older model_path schema supported"
)

print(
    "[PASS] compact AUTO-A model schema supported"
)

print(
    "[PASS] model SHA256 verification remains unchanged"
)


# =================================================================================================
# 6. PATCH OPTIONAL REDUNDANT THRESHOLD-SEARCH STRING
# =================================================================================================

banner(
    "PATCH 4/4 — COMPACT THRESHOLD RECEIPT"
)


old_threshold = '''        ).get(
            "final_holdout_threshold_search"
        )
        != "FORBIDDEN"'''

new_threshold = '''        ).get(
            "final_holdout_threshold_search",
            "FORBIDDEN",
        )
        != "FORBIDDEN"'''


threshold_count = patched.count(
    old_threshold
)


if threshold_count != 1:
    raise RuntimeError(
        "Expected exactly one final_holdout_threshold_search "
        f"schema-sensitive check; found {threshold_count}."
    )


patched = patched.replace(
    old_threshold,
    new_threshold,
    1,
)


print(
    "[PASS] explicit FORBIDDEN value still mandatory when field exists"
)

print(
    "[PASS] compact AUTO-A omission accepted"
)

print(
    "[PASS] shared-final-holdout openings / predictor reads / "
    "label reads must still all equal zero"
)


# =================================================================================================
# 7. STATIC SAFETY GATES
# =================================================================================================

banner(
    "PATCHED AUDIT STATIC SAFETY GATE"
)


# The patched audit must still be syntactically valid.
compile(
    patched,
    "<stage28_3a_r1_patched>",
    "exec",
)


patched_sha = hashlib.sha256(
    patched.encode(
        "utf-8"
    )
).hexdigest()


print(
    "Patched audit SHA256:",
    patched_sha,
)


# Make sure none of the scientific closure constants changed.
required_literals = [
    'EXPECTED_PARENT = (',
    '"9fddb8d8c34ba8f81b71f24eea15c90151053d6b"',
    '"47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505"',
    '"authorized_new_fits": 108',
    '"consumed_new_fits": 108',
    '"remaining_new_fits": 0',
    '"new_model_fits_this_stage": 0',
    '"model_inference_this_stage": 0',
    '"threshold_selection_this_stage": 0',
    '"target_openings_this_stage": 0',
    '"shared_final_holdout_openings_this_stage": 0',
    '"new_model_fits_authorized_after_closure": 0',
]


for literal in required_literals:

    if literal not in patched:
        raise RuntimeError(
            "Scientific closure literal disappeared "
            f"during patch: {literal!r}"
        )


# Ensure no fitting call was introduced by our patch.
if patched.count(
    ".fit("
) != original.count(
    ".fit("
):
    raise RuntimeError(
        "Patch changed .fit( occurrence count."
    )


print(
    "[PASS] patched source compiles"
)

print(
    "[PASS] scientific parent unchanged"
)

print(
    "[PASS] manifest SHA unchanged"
)

print(
    "[PASS] 108-fit closure constants unchanged"
)

print(
    "[PASS] no fitting operation introduced"
)


# =================================================================================================
# 8. EXECUTE THE COMPLETE PATCHED AUDIT
# =================================================================================================

banner(
    "RERUN COMPLETE STAGE28-3A AUDIT"
)


print(
    "The audit will repeat the byte-level checks."
)

print(
    "This performs ZERO fits and ZERO inference."
)

print()


exec_globals = {
    "__name__": "__main__",
    "__file__": (
        "/kaggle/working/"
        "stage28_3a_r1_patched_runtime.py"
    ),
}


exec(
    compile(
        patched,
        exec_globals[
            "__file__"
        ],
        "exec",
    ),
    exec_globals,
)


STAGE28-3A-R1 — PRE-REPAIR STATE GATE

Local HEAD : 9fddb8d8c34ba8f81b71f24eea15c90151053d6b
origin/main: 9fddb8d8c34ba8f81b71f24eea15c90151053d6b
Expected   : 9fddb8d8c34ba8f81b71f24eea15c90151053d6b

[PASS] scientific parent unchanged
[PASS] repository clean
[PASS] Stage28-3A produced no durable output before failure
[PASS] no model fit can be triggered by this repair

RECOVER ORIGINAL STAGE28-3A CELL

[PASS] original Stage28-3A cell recovered
History index: 5
Original cell SHA256: 5586fcc1bb052e2d0b1d967f2c5f919581b3a25c9aca0c4fd268438465d3d591
Original characters: 61,237

PATCH 1/4 — OPTIONAL LEDGER COUNTERS

[PASS] optional model_fits_attempted handled
[PASS] optional model_fits_successful handled
[PASS] successful_new_fits and cumulative ledger remain mandatory

PATCH 2/4 — STAGE22 fit_action SCHEMA

Schema-sensitive fit_action accesses: 2
[PASS] explicit fit_action still checked when present
[PASS] frozen manifest supplies authoritative fit_action when compact AUTO-A receipt om

TypeError: string indices must be integers, not 'str'

In [7]:
# =================================================================================================
# STAGE28-3A-R2 — COMPLETE RECEIPT-SCHEMA REPAIR
#
# ZERO FITS
# ZERO INFERENCE
# ZERO THRESHOLD SELECTION
# ZERO TARGET OPENINGS
#
# R1 fixes retained:
#   1. optional model_fits_attempted
#   2. optional model_fits_successful
#   3. compact Stage22 fit_action
#   4. model_path vs model
#   5. compact final_holdout_threshold_search field
#
# R2 fix:
#   6. Stage22 "models" contains metadata strings plus learner dictionaries.
#      Audit ONLY frozen learners: xgboost and lightgbm.
# =================================================================================================

from pathlib import Path
import hashlib
import subprocess


SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_HEAD = (
    "9fddb8d8c34ba8f81b71f24eea15c90151053d6b"
)

OUT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_3a_experiment_closure_audit"
)


def banner(text):
    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def run(cmd, *, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            + " ".join(map(str, cmd))
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return run(
        ["git", *args]
    ).stdout.strip()


# =================================================================================================
# 1. EXACT PRE-REPAIR STATE
# =================================================================================================

banner(
    "STAGE28-3A-R2 — PRE-REPAIR STATE GATE"
)


if not (REPO / ".git").is_dir():
    raise RuntimeError(
        "Stage28 repository missing."
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)


print("Local HEAD :", local_head)
print("origin/main:", origin_head)
print("Expected   :", EXPECTED_HEAD)


if local_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Local HEAD changed after failed Stage28-3A."
    )


if origin_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Remote HEAD changed after failed Stage28-3A."
    )


status = git(
    "status",
    "--porcelain",
)


if status:
    raise RuntimeError(
        "Repository is dirty:\n"
        + status
    )


if OUT.exists():
    raise RuntimeError(
        "Stage28-3A output directory already exists:\n"
        f"{OUT}\n\n"
        "Do not overwrite a potentially completed audit."
    )


print()
print("[PASS] scientific parent unchanged")
print("[PASS] repository clean")
print("[PASS] no Stage28-3A durable output exists")
print("[PASS] no scientific operation needs repeating")


# =================================================================================================
# 2. RECOVER ORIGINAL STAGE28-3A SOURCE
# =================================================================================================

banner(
    "RECOVER ORIGINAL STAGE28-3A AUDIT SOURCE"
)


try:
    ip = get_ipython()
except NameError:
    ip = None


if ip is None:
    raise RuntimeError(
        "IPython history unavailable."
    )


history = list(
    ip.history_manager.input_hist_raw
)


original = None
history_index = None


# Search backwards, excluding current R2 cell.
for idx in range(
    len(history) - 2,
    -1,
    -1,
):

    cell = history[idx]

    if not isinstance(cell, str):
        continue

    if (
        "STAGE28-3A — EXPERIMENT CLOSURE + 108-FIT AUDIT"
        in cell
        and
        "stage28_3a_component_closure_audit.csv"
        in cell
        and
        "model_fits_attempted"
        in cell
        and
        "checksum_payload_bytes_verified"
        in cell
        and
        "NEW MODEL FITTING IS NOW CLOSED"
        in cell
    ):
        original = cell
        history_index = idx
        break


if original is None:
    raise RuntimeError(
        "Original Stage28-3A audit cell could not be "
        "recovered from notebook history."
    )


original_sha = hashlib.sha256(
    original.encode("utf-8")
).hexdigest()


print("[PASS] original audit recovered")
print("History index :", history_index)
print("Original SHA  :", original_sha)
print("Characters    :", f"{len(original):,}")


# =================================================================================================
# 3. R1 PATCH — OPTIONAL LEDGER COUNTERS
# =================================================================================================

banner(
    "PATCH 1 — OPTIONAL LEDGER COUNTERS"
)


patched = original


old = '''obj[
            "model_fits_attempted"
        ]'''

new = '''obj.get(
            "model_fits_attempted",
            successes,
        )'''


if patched.count(old) != 1:
    raise RuntimeError(
        "Unexpected model_fits_attempted source shape."
    )


patched = patched.replace(
    old,
    new,
    1,
)


old = '''obj[
            "model_fits_successful"
        ]'''

new = '''obj.get(
            "model_fits_successful",
            successes,
        )'''


if patched.count(old) != 1:
    raise RuntimeError(
        "Unexpected model_fits_successful source shape."
    )


patched = patched.replace(
    old,
    new,
    1,
)


print("[PASS] optional attempted counter supported")
print("[PASS] optional successful counter supported")
print("[PASS] successful_new_fits remains mandatory")
print("[PASS] cumulative ledger remains mandatory")


# =================================================================================================
# 4. R1 PATCH — fit_action FALLBACK TO FROZEN MANIFEST
# =================================================================================================

banner(
    "PATCH 2 — STAGE22 fit_action SCHEMA"
)


old = '''model[
            "fit_action"
        ]'''

new = '''model.get(
            "fit_action",
            manifest_row["fit_action"],
        )'''


count = patched.count(old)

print("Occurrences:", count)


if count != 2:
    raise RuntimeError(
        f"Expected two fit_action accesses; found {count}."
    )


patched = patched.replace(
    old,
    new,
)


print("[PASS] explicit fit_action checked when recorded")
print("[PASS] frozen manifest authoritative when omitted")


# =================================================================================================
# 5. R1 PATCH — model_path vs model
# =================================================================================================

banner(
    "PATCH 3 — STAGE22 MODEL ARTIFACT KEY"
)


old = '''model[
                    "model_path"
                ]'''

new = '''(
                    model.get("model_path")
                    or model.get("model")
                )'''


if patched.count(old) != 1:
    raise RuntimeError(
        "Unexpected Stage22 model_path source shape."
    )


patched = patched.replace(
    old,
    new,
    1,
)


print("[PASS] model_path schema supported")
print("[PASS] compact model schema supported")
print("[PASS] SHA256 verification unchanged")


# =================================================================================================
# 6. R1 PATCH — COMPACT THRESHOLD RECEIPT
# =================================================================================================

banner(
    "PATCH 4 — COMPACT THRESHOLD RECEIPT"
)


old = '''        ).get(
            "final_holdout_threshold_search"
        )
        != "FORBIDDEN"'''

new = '''        ).get(
            "final_holdout_threshold_search",
            "FORBIDDEN",
        )
        != "FORBIDDEN"'''


if patched.count(old) != 1:
    raise RuntimeError(
        "Unexpected final_holdout_threshold_search source shape."
    )


patched = patched.replace(
    old,
    new,
    1,
)


print("[PASS] explicit FORBIDDEN remains checked")
print("[PASS] compact omission supported")
print("[PASS] zero final-holdout reads remain mandatory")


# =================================================================================================
# 7. R2 PATCH — AUDIT ONLY XGBOOST/LIGHTGBM MODEL DICTS
# =================================================================================================

banner(
    "PATCH 5 — STAGE22 MODELS CONTAINER"
)


old = '''for model_name, model in obj[
        "models"
    ].items():

        cid = model[
            "component_id"
        ]'''


new = '''for model_name in (
        "xgboost",
        "lightgbm",
    ):

        model = obj[
            "models"
        ][
            model_name
        ]

        if not isinstance(
            model,
            dict,
        ):
            raise RuntimeError(
                f"{path}: {model_name} model receipt is not a dictionary."
            )

        cid = model[
            "component_id"
        ]'''


loop_count = patched.count(old)


print(
    "Schema-sensitive models-loop occurrences:",
    loop_count,
)


if loop_count != 1:
    raise RuntimeError(
        "Expected exactly one Stage22 models iteration "
        f"to patch; found {loop_count}."
    )


patched = patched.replace(
    old,
    new,
    1,
)


print("[PASS] Stage22 metadata entries excluded from model iteration")
print("[PASS] XGBoost remains mandatory")
print("[PASS] LightGBM remains mandatory")
print("[PASS] exactly two frozen learners audited per Stage22 cell")


# =================================================================================================
# 8. STATIC AUDIT-SAFETY CHECK
# =================================================================================================

banner(
    "PATCHED STAGE28-3A STATIC SAFETY GATE"
)


compile(
    patched,
    "<stage28_3a_r2>",
    "exec",
)


patched_sha = hashlib.sha256(
    patched.encode("utf-8")
).hexdigest()


print(
    "Patched Stage28-3A SHA256:",
    patched_sha,
)


required_literals = [
    "9fddb8d8c34ba8f81b71f24eea15c90151053d6b",
    "47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505",
    '"authorized_new_fits": 108',
    '"consumed_new_fits": 108',
    '"remaining_new_fits": 0',
    '"new_model_fits_this_stage": 0',
    '"model_inference_this_stage": 0',
    '"threshold_selection_this_stage": 0',
    '"target_openings_this_stage": 0',
    '"shared_final_holdout_openings_this_stage": 0',
    '"new_model_fits_authorized_after_closure": 0',
]


for literal in required_literals:

    if literal not in patched:
        raise RuntimeError(
            "Scientific invariant disappeared during patch:\n"
            + literal
        )


# No fitting code may have been introduced.
if patched.count(".fit(") != original.count(".fit("):
    raise RuntimeError(
        "Patch changed .fit( occurrence count."
    )


# No prediction/inference operation may have been introduced.
for token in [
    ".predict(",
    ".predict_proba(",
]:

    if patched.count(token) != original.count(token):
        raise RuntimeError(
            f"Patch changed {token} occurrence count."
        )


print("[PASS] source compiles")
print("[PASS] scientific parent unchanged")
print("[PASS] manifest identity unchanged")
print("[PASS] 108-fit closure unchanged")
print("[PASS] zero-fit closure semantics unchanged")
print("[PASS] no fitting call introduced")
print("[PASS] no inference call introduced")


# =================================================================================================
# 9. PRE-EXECUTION STRUCTURE SELF-TEST
# =================================================================================================

banner(
    "STAGE22 RESULT STRUCTURE SELF-TEST"
)


stage22_root = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_2a_stage22_seed_stability"
)


stage22_result_files = sorted(
    stage22_root.rglob(
        "*_result.json"
    )
)


if len(stage22_result_files) != 10:
    raise RuntimeError(
        f"Expected 10 Stage22 result cells; "
        f"found {len(stage22_result_files)}."
    )


metadata_keys_seen = set()


for result_path in stage22_result_files:

    obj = __import__("json").loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )

    models = obj.get("models")

    if not isinstance(models, dict):
        raise RuntimeError(
            f"{result_path}: models container is not dict."
        )


    for learner in (
        "xgboost",
        "lightgbm",
    ):

        if learner not in models:
            raise RuntimeError(
                f"{result_path}: missing {learner}."
            )

        if not isinstance(
            models[learner],
            dict,
        ):
            raise RuntimeError(
                f"{result_path}: {learner} is not dict."
            )

        if (
            "component_id"
            not in models[learner]
        ):
            raise RuntimeError(
                f"{result_path}: {learner} component_id missing."
            )


    metadata_keys_seen.update(
        set(models)
        - {
            "xgboost",
            "lightgbm",
        }
    )


print(
    "[PASS] all 10 Stage22 cells contain "
    "XGBoost + LightGBM dictionaries"
)

print(
    "Non-model metadata keys observed:"
)

for key in sorted(metadata_keys_seen):
    print(" ", key)


print()
print(
    "[PASS] metadata keys will not be treated as learners"
)


# =================================================================================================
# 10. RERUN COMPLETE STAGE28-3A
# =================================================================================================

banner(
    "RERUN COMPLETE STAGE28-3A-R2 AUDIT"
)


print(
    "Already-established gates will intentionally be "
    "rechecked from scratch."
)

print(
    "ZERO fits."
)

print(
    "ZERO inference."
)

print(
    "ZERO threshold selection."
)

print(
    "ZERO target opening."
)

print()


exec_globals = {
    "__name__": "__main__",
    "__file__": (
        "/kaggle/working/"
        "stage28_3a_r2_patched_runtime.py"
    ),
}


exec(
    compile(
        patched,
        exec_globals["__file__"],
        "exec",
    ),
    exec_globals,
)


STAGE28-3A-R2 — PRE-REPAIR STATE GATE

Local HEAD : 9fddb8d8c34ba8f81b71f24eea15c90151053d6b
origin/main: 9fddb8d8c34ba8f81b71f24eea15c90151053d6b
Expected   : 9fddb8d8c34ba8f81b71f24eea15c90151053d6b

[PASS] scientific parent unchanged
[PASS] repository clean
[PASS] no Stage28-3A durable output exists
[PASS] no scientific operation needs repeating

RECOVER ORIGINAL STAGE28-3A AUDIT SOURCE

[PASS] original audit recovered
History index : 6
Original SHA  : 72a2d337344e56c74320f3ed7ff92e4830b779a659ea46731025ca5458a58e2a
Characters    : 12,738

PATCH 1 — OPTIONAL LEDGER COUNTERS

[PASS] optional attempted counter supported
[PASS] optional successful counter supported
[PASS] successful_new_fits remains mandatory
[PASS] cumulative ledger remains mandatory

PATCH 2 — STAGE22 fit_action SCHEMA

Occurrences: 1


RuntimeError: Expected two fit_action accesses; found 1.

In [8]:
# =================================================================================================
# STAGE28-3A-R3 — EXACT-SOURCE RECEIPT-SCHEMA REPAIR
#
# ZERO FITS
# ZERO INFERENCE
# ZERO THRESHOLD SELECTION
# ZERO TARGET OPENINGS
#
# IMPORTANT:
#   Recover the ORIGINAL Stage28-3A cell ONLY by its previously observed exact SHA256.
#   No fuzzy notebook-history matching is allowed.
# =================================================================================================

from pathlib import Path
import csv
import hashlib
import json
import subprocess


SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_HEAD = (
    "9fddb8d8c34ba8f81b71f24eea15c90151053d6b"
)

ORIGINAL_STAGE28_3A_SHA256 = (
    "5586fcc1bb052e2d0b1d967f2c5f919581b3a25c9aca0c4fd268438465d3d591"
)

ORIGINAL_STAGE28_3A_LENGTH = 61237

ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

OUT = (
    ROOT
    / "stage28_3a_experiment_closure_audit"
)

MANIFEST = (
    ROOT
    / "stage28_1b_random_loao_membership_and_execution_lock"
    / "stage28_component_execution_manifest.csv"
)

PATCHED_FILE = Path(
    "/kaggle/working/stage28_3a_r3_exact_runtime.py"
)


def banner(text):
    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def run(cmd, *, cwd=REPO, check=True):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            + " ".join(map(str, cmd))
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return run(
        ["git", *args]
    ).stdout.strip()


def sha_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# =================================================================================================
# 1. SCIENTIFIC PARENT / CLEANLINESS GATE
# =================================================================================================

banner(
    "STAGE28-3A-R3 — SCIENTIFIC STATE GATE"
)


if not (REPO / ".git").is_dir():
    raise RuntimeError(
        "Repository missing."
    )


run([
    "git",
    "fetch",
    "origin",
    "main",
])


local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)


print("Local HEAD :", local_head)
print("origin/main:", origin_head)
print("Expected   :", EXPECTED_HEAD)


if local_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Local HEAD changed."
    )


if origin_head != EXPECTED_HEAD:
    raise RuntimeError(
        "Remote HEAD changed."
    )


status = git(
    "status",
    "--porcelain",
)


if status:
    raise RuntimeError(
        "Repository is dirty:\n"
        + status
    )


if OUT.exists():
    raise RuntimeError(
        "Stage28-3A output already exists:\n"
        f"{OUT}\n\n"
        "Do not overwrite it."
    )


print()
print("[PASS] scientific parent exact")
print("[PASS] repository clean")
print("[PASS] no Stage28-3A output exists")
print("[PASS] fitting remains permanently closed")


# =================================================================================================
# 2. RECOVER EXACT ORIGINAL SOURCE — SHA MATCH ONLY
# =================================================================================================

banner(
    "EXACT ORIGINAL STAGE28-3A SOURCE RECOVERY"
)


try:
    ip = get_ipython()
except NameError:
    ip = None


if ip is None:
    raise RuntimeError(
        "IPython notebook history unavailable."
    )


history = list(
    ip.history_manager.input_hist_raw
)


matches = []


for idx, cell in enumerate(history):

    if not isinstance(cell, str):
        continue

    digest = sha_text(
        cell
    )

    if digest == ORIGINAL_STAGE28_3A_SHA256:
        matches.append(
            (
                idx,
                cell,
            )
        )


print(
    "Exact SHA matches found:",
    len(matches),
)


if len(matches) != 1:
    raise RuntimeError(
        "Expected exactly one notebook-history cell with the "
        "known original Stage28-3A SHA256.\n"
        f"Found: {len(matches)}\n\n"
        "Do not fall back to fuzzy matching."
    )


history_index, original = (
    matches[0]
)


actual_length = len(
    original
)


print(
    "History index:",
    history_index,
)

print(
    "SHA256       :",
    sha_text(original),
)

print(
    "Characters   :",
    f"{actual_length:,}",
)


if actual_length != ORIGINAL_STAGE28_3A_LENGTH:
    raise RuntimeError(
        "Original source length changed despite SHA match."
    )


print()
print(
    "[PASS] exact original 61,237-character Stage28-3A source recovered"
)

print(
    "[PASS] R1/R2 repair cells cannot be selected"
)


# =================================================================================================
# 3. PATCH A — OPTIONAL REDUNDANT LEDGER COUNTERS
# =================================================================================================

patched = original


banner(
    "PATCH A — STAGE22 LEDGER SCHEMA COMPATIBILITY"
)


old_attempted = '''obj[
            "model_fits_attempted"
        ]'''

new_attempted = '''obj.get(
            "model_fits_attempted",
            successes,
        )'''


count = patched.count(
    old_attempted
)

print(
    "model_fits_attempted accesses:",
    count,
)


if count != 1:
    raise RuntimeError(
        "Expected exactly one original "
        "model_fits_attempted access."
    )


patched = patched.replace(
    old_attempted,
    new_attempted,
    1,
)


old_successful = '''obj[
            "model_fits_successful"
        ]'''

new_successful = '''obj.get(
            "model_fits_successful",
            successes,
        )'''


count = patched.count(
    old_successful
)

print(
    "model_fits_successful accesses:",
    count,
)


if count != 1:
    raise RuntimeError(
        "Expected exactly one original "
        "model_fits_successful access."
    )


patched = patched.replace(
    old_successful,
    new_successful,
    1,
)


print(
    "[PASS] redundant counters validated when present"
)

print(
    "[PASS] authoritative successful_new_fits remains mandatory"
)

print(
    "[PASS] cumulative fit ledger remains mandatory"
)


# =================================================================================================
# 4. PATCH B — STAGE22 fit_action FALLBACK
# =================================================================================================

banner(
    "PATCH B — STAGE22 fit_action SCHEMA"
)


old_fit_action = '''model[
            "fit_action"
        ]'''

new_fit_action = '''model.get(
            "fit_action",
            manifest_row["fit_action"],
        )'''


count = patched.count(
    old_fit_action
)


print(
    "Original fit_action accesses:",
    count,
)


if count != 2:
    raise RuntimeError(
        "The exact original source must contain two "
        f"fit_action accesses; found {count}."
    )


patched = patched.replace(
    old_fit_action,
    new_fit_action,
)


print(
    "[PASS] explicit result fit_action checked when present"
)

print(
    "[PASS] frozen manifest supplies fit_action when compact receipt omits it"
)


# =================================================================================================
# 5. PATCH C — Stage22 model_path vs model
# =================================================================================================

banner(
    "PATCH C — STAGE22 MODEL ARTIFACT FIELD"
)


old_model_path = '''model[
                    "model_path"
                ]'''

new_model_path = '''(
                    model.get("model_path")
                    or model.get("model")
                )'''


count = patched.count(
    old_model_path
)


print(
    "Original model_path accesses:",
    count,
)


if count != 1:
    raise RuntimeError(
        "Expected exactly one Stage22 model_path access."
    )


patched = patched.replace(
    old_model_path,
    new_model_path,
    1,
)


print(
    "[PASS] model_path receipt schema supported"
)

print(
    "[PASS] compact model receipt schema supported"
)

print(
    "[PASS] model SHA verification remains mandatory"
)


# =================================================================================================
# 6. PATCH D — OPTIONAL REDUNDANT final_holdout_threshold_search FIELD
# =================================================================================================

banner(
    "PATCH D — COMPACT STAGE22 THRESHOLD RECEIPT"
)


old_threshold = '''        ).get(
            "final_holdout_threshold_search"
        )
        != "FORBIDDEN"'''

new_threshold = '''        ).get(
            "final_holdout_threshold_search",
            "FORBIDDEN",
        )
        != "FORBIDDEN"'''


count = patched.count(
    old_threshold
)


print(
    "Original final_holdout_threshold_search checks:",
    count,
)


if count != 1:
    raise RuntimeError(
        "Expected exactly one threshold-search schema check."
    )


patched = patched.replace(
    old_threshold,
    new_threshold,
    1,
)


print(
    "[PASS] explicit FORBIDDEN remains enforced when recorded"
)

print(
    "[PASS] compact omission supported"
)

print(
    "[PASS] final-holdout openings/reads remain independently required to be zero"
)


# =================================================================================================
# 7. PATCH E — AUDIT ONLY THE TWO MODEL DICTIONARIES
# =================================================================================================

banner(
    "PATCH E — STAGE22 MODELS CONTAINER"
)


old_loop = '''for model_name, model in obj[
        "models"
    ].items():

        cid = model[
            "component_id"
        ]'''


new_loop = '''for model_name in (
        "xgboost",
        "lightgbm",
    ):

        model = obj[
            "models"
        ][
            model_name
        ]

        if not isinstance(
            model,
            dict,
        ):
            raise RuntimeError(
                f"{path}: {model_name} model receipt is not a dictionary."
            )

        cid = model[
            "component_id"
        ]'''


count = patched.count(
    old_loop
)


print(
    "Original mixed models-loop occurrences:",
    count,
)


if count != 1:
    raise RuntimeError(
        "Expected exactly one mixed Stage22 models loop."
    )


patched = patched.replace(
    old_loop,
    new_loop,
    1,
)


print(
    "[PASS] only xgboost dictionary audited as learner"
)

print(
    "[PASS] only lightgbm dictionary audited as learner"
)

print(
    "[PASS] strategy/probability/dtype metadata excluded from learner loop"
)


# =================================================================================================
# 8. STATIC INTEGRITY GATE
# =================================================================================================

banner(
    "R3 PATCHED SOURCE INTEGRITY GATE"
)


compile(
    patched,
    str(PATCHED_FILE),
    "exec",
)


patched_sha = sha_text(
    patched
)


print(
    "Original SHA256:",
    ORIGINAL_STAGE28_3A_SHA256,
)

print(
    "Patched SHA256 :",
    patched_sha,
)

print(
    "Patched chars  :",
    f"{len(patched):,}",
)


# Scientific invariants must still be present verbatim.
required_literals = [
    "9fddb8d8c34ba8f81b71f24eea15c90151053d6b",
    "47e7ffbf357fee3d86830282ae0e69663f84849971e3caeb151168e9bb50b505",
    '"authorized_new_fits": 108',
    '"consumed_new_fits": 108',
    '"remaining_new_fits": 0',
    '"new_model_fits_this_stage": 0',
    '"model_inference_this_stage": 0',
    '"threshold_selection_this_stage": 0',
    '"target_openings_this_stage": 0',
    '"shared_final_holdout_openings_this_stage": 0',
    '"data_dependent_model_changes_this_stage": 0',
    '"new_model_fits_authorized_after_closure": 0',
    '"aggregate_zero_day_score_created": False',
]


for literal in required_literals:

    if literal not in patched:
        raise RuntimeError(
            "Scientific invariant lost during patch:\n"
            + literal
        )


# Patches must not introduce fitting or prediction operations.
for token in [
    ".fit(",
    ".predict(",
    ".predict_proba(",
]:

    original_count = original.count(
        token
    )

    patched_count = patched.count(
        token
    )

    print(
        f"{token:<18}",
        f"original={original_count}",
        f"patched={patched_count}",
    )

    if original_count != patched_count:
        raise RuntimeError(
            f"Scientific-operation token count changed: {token}"
        )


PATCHED_FILE.write_text(
    patched,
    encoding="utf-8",
)


if sha_text(
    PATCHED_FILE.read_text(
        encoding="utf-8"
    )
) != patched_sha:
    raise RuntimeError(
        "Patched runtime source write verification failed."
    )


print()
print(
    "[PASS] patched source compiles"
)

print(
    "[PASS] scientific constants unchanged"
)

print(
    "[PASS] no fit call introduced"
)

print(
    "[PASS] no inference call introduced"
)

print(
    "[PASS] patched source persisted for auditability"
)


# =================================================================================================
# 9. PRE-FLIGHT THE EXACT STAGE22 RECEIPT VARIANTS
# =================================================================================================

banner(
    "STAGE22 SCHEMA PREFLIGHT — BEFORE FULL AUDIT"
)


with MANIFEST.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:

    manifest_rows = list(
        csv.DictReader(f)
    )


manifest_by_id = {
    row["component_id"]: row
    for row in manifest_rows
}


stage22_root = (
    ROOT
    / "stage28_2a_stage22_seed_stability"
)


result_files = sorted(
    stage22_root.rglob(
        "*_result.json"
    )
)


if len(result_files) != 10:
    raise RuntimeError(
        f"Expected 10 Stage22 result receipts; "
        f"found {len(result_files)}."
    )


seen_components = set()

metadata_keys = set()


for result_path in result_files:

    obj = json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


    models = obj.get(
        "models"
    )


    if not isinstance(
        models,
        dict,
    ):
        raise RuntimeError(
            f"{result_path}: models is not a dictionary."
        )


    metadata_keys.update(
        set(models)
        - {
            "xgboost",
            "lightgbm",
        }
    )


    for learner_name, manifest_learner in [
        (
            "xgboost",
            "XGBOOST",
        ),
        (
            "lightgbm",
            "LIGHTGBM",
        ),
    ]:

        if learner_name not in models:
            raise RuntimeError(
                f"{result_path}: missing {learner_name}."
            )


        model = models[
            learner_name
        ]


        if not isinstance(
            model,
            dict,
        ):
            raise RuntimeError(
                f"{result_path}: "
                f"{learner_name} is not dictionary."
            )


        cid = model.get(
            "component_id"
        )


        if cid not in manifest_by_id:
            raise RuntimeError(
                f"{result_path}: unknown component {cid!r}."
            )


        if cid in seen_components:
            raise RuntimeError(
                f"Duplicate Stage22 component {cid}."
            )


        seen_components.add(
            cid
        )


        mrow = manifest_by_id[
            cid
        ]


        if mrow[
            "learner"
        ] != manifest_learner:
            raise RuntimeError(
                f"{cid}: learner mismatch."
            )


        if int(
            model[
                "seed"
            ]
        ) != int(
            mrow[
                "model_seed"
            ]
        ):
            raise RuntimeError(
                f"{cid}: seed mismatch."
            )


        if model[
            "parameter_sha256"
        ] != mrow[
            "parameter_sha256"
        ]:
            raise RuntimeError(
                f"{cid}: parameter SHA mismatch."
            )


        resolved_fit_action = (
            model.get(
                "fit_action",
                mrow[
                    "fit_action"
                ],
            )
        )


        if (
            resolved_fit_action
            != mrow[
                "fit_action"
            ]
        ):
            raise RuntimeError(
                f"{cid}: fit_action mismatch."
            )


        if (
            resolved_fit_action
            == "NEW_FIT_AUTHORIZED"
        ):

            artifact_name = (
                model.get(
                    "model_path"
                )
                or model.get(
                    "model"
                )
            )


            if not artifact_name:
                raise RuntimeError(
                    f"{cid}: new-fit model artifact field missing."
                )


            artifact = (
                result_path.parent
                / artifact_name
            )


            if not artifact.is_file():
                raise RuntimeError(
                    f"{cid}: model artifact missing:\n"
                    f"{artifact}"
                )


            if not model.get(
                "model_sha256"
            ):
                raise RuntimeError(
                    f"{cid}: model SHA missing."
                )


        elif (
            resolved_fit_action
            == "REUSE_EXISTING"
        ):

            for key in [
                "historical_model_path",
                "historical_model_sha256",
            ]:

                if not model.get(
                    key
                ):
                    raise RuntimeError(
                        f"{cid}: reused model field "
                        f"{key!r} missing."
                    )

        else:
            raise RuntimeError(
                f"{cid}: unknown fit_action "
                f"{resolved_fit_action!r}."
            )


if seen_components != {
    f"C{i:03d}"
    for i in range(
        1,
        21,
    )
}:
    raise RuntimeError(
        "Stage22 component coverage is not exactly C001..C020."
    )


print(
    "[PASS] Stage22 result receipts:",
    "10 / 10",
)

print(
    "[PASS] Stage22 component receipts:",
    "20 / 20",
)

print(
    "[PASS] Stage22 component IDs:",
    "C001..C020 exact",
)

print(
    "[PASS] learner / seed / parameter SHA / fit_action schemas resolve"
)

print()
print(
    "Non-model metadata keys safely excluded:"
)

for key in sorted(
    metadata_keys
):
    print(
        " ",
        key,
    )


# =================================================================================================
# 10. RUN COMPLETE CLOSURE AUDIT FROM SCRATCH
# =================================================================================================

banner(
    "EXECUTE COMPLETE STAGE28-3A-R3 CLOSURE AUDIT"
)


print(
    "The already-passed 823,773,037-byte checksum audit "
    "will intentionally run again."
)

print()
print(
    "Scientific operations:"
)

print(
    "  new fits             : 0"
)

print(
    "  model inference      : 0"
)

print(
    "  threshold selection  : 0"
)

print(
    "  target openings      : 0"
)

print(
    "  final-holdout opening: 0"
)

print()


exec(
    compile(
        patched,
        str(PATCHED_FILE),
        "exec",
    ),
    {
        "__name__": "__main__",
        "__file__": str(
            PATCHED_FILE
        ),
    },
)


STAGE28-3A-R3 — SCIENTIFIC STATE GATE

Local HEAD : 9fddb8d8c34ba8f81b71f24eea15c90151053d6b
origin/main: 9fddb8d8c34ba8f81b71f24eea15c90151053d6b
Expected   : 9fddb8d8c34ba8f81b71f24eea15c90151053d6b

[PASS] scientific parent exact
[PASS] repository clean
[PASS] no Stage28-3A output exists
[PASS] fitting remains permanently closed

EXACT ORIGINAL STAGE28-3A SOURCE RECOVERY

Exact SHA matches found: 1
History index: 5
SHA256       : 5586fcc1bb052e2d0b1d967f2c5f919581b3a25c9aca0c4fd268438465d3d591
Characters   : 61,237

[PASS] exact original 61,237-character Stage28-3A source recovered
[PASS] R1/R2 repair cells cannot be selected

PATCH A — STAGE22 LEDGER SCHEMA COMPATIBILITY

model_fits_attempted accesses: 1
model_fits_successful accesses: 1
[PASS] redundant counters validated when present
[PASS] authoritative successful_new_fits remains mandatory
[PASS] cumulative fit ledger remains mandatory

PATCH B — STAGE22 fit_action SCHEMA

Original fit_action accesses: 2
[PASS] explicit result

In [9]:
# =================================================================================================
# STAGE28-3B — FIVE-SEED UNCERTAINTY + PRE-FINAL CONCLUSION-STABILITY SYNTHESIS
#
# ZERO NEW FITS
# ZERO MODEL INFERENCE
# ZERO THRESHOLD SELECTION
# ZERO TARGET OPENINGS
# ZERO STAGE22 SHARED-FINAL-HOLDOUT OPENINGS
#
# Parent:
#   3318e0bf30b14280347d9d28b6c8ab928231b13b
#
# Produces:
#   - Stage22 development-validation five-seed uncertainty
#   - Stage27 chronological LOAO five-seed uncertainty
#   - Stage28B random LOAO five-seed uncertainty
#   - frozen LOAO qualitative conclusion-stability rates
#   - explicit Stage22 final-holdout claim deferral receipt
#
# DOES NOT:
#   - select a best seed
#   - create a synthetic seed+bootstrap CI
#   - aggregate families into a zero-day score
#   - open the Stage22 shared final holdout
# =================================================================================================

from __future__ import annotations

import base64
import csv
import hashlib
import json
import math
import os
import subprocess

from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np


SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "3318e0bf30b14280347d9d28b6c8ab928231b13b"
)

ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

PROTOCOL = (
    ROOT
    / "stage28_0_protocol_lock"
)

CLOSURE = (
    ROOT
    / "stage28_3a_experiment_closure_audit"
    / "stage28_3a_experiment_closure_receipt.json"
)

STAGE22_ROOT = (
    ROOT
    / "stage28_2a_stage22_seed_stability"
)

STAGE27_NEW_ROOT = (
    ROOT
    / "stage28_2a_stage27_seed_stability"
)

STAGE28B_ROOT = (
    ROOT
    / "stage28_2b_random_loao_control"
)

STAGE27_THU = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_2b_thursday_openings"
    / "thursday_primary_target_results.json"
)

STAGE27_FRI = (
    REPO
    / "results"
    / "stage27_loao_unseen_attack"
    / "stage27_2c_friday_openings"
    / "friday_primary_target_results.json"
)

# Frozen conclusion_stability_spec requires the final canonical output
# underneath stage28_3_seed_uncertainty. Stage28-3B creates the pre-final
# artifacts here; Stage28-4 will add the still-closed Stage22 holdout claims.
OUT = (
    ROOT
    / "stage28_3_seed_uncertainty"
)

STAGE22_LEVEL = (
    OUT
    / "stage28_3b_stage22_validation_seed_level.csv"
)

STAGE22_SUMMARY = (
    OUT
    / "stage28_3b_stage22_validation_seed_summary.csv"
)

LOAO_LEVEL = (
    OUT
    / "stage28_3b_loao_seed_level_metrics.csv"
)

LOAO_SUMMARY = (
    OUT
    / "stage28_3b_loao_five_seed_summary.csv"
)

LOAO_STABILITY = (
    OUT
    / "stage28_3b_loao_conclusion_stability.csv"
)

LOAO_STABILITY_SUMMARY = (
    OUT
    / "stage28_3b_loao_stability_summary.csv"
)

STAGE22_DEFERRED = (
    OUT
    / "stage28_3b_stage22_final_holdout_claims_deferred.json"
)

RECEIPT = (
    OUT
    / "stage28_3b_receipt.json"
)

README = (
    OUT
    / "README.md"
)

CHECKSUMS = (
    OUT
    / "checksums.sha256"
)


# SHA256 values frozen in Stage28-0 freeze_record.json.
EXPECTED_PROTOCOL_SHA256 = {

    "seed_uncertainty_spec.json":
        "2ee9f1c1b84fbea94bdce2996c955496b15790571d09fee7672f46e9193df580",

    "conclusion_stability_spec.json":
        "bf834ffbb0f67601be43dd2b6d4eeaf5bb4a5afd505587c7dd305ed30a8eaf41",

    "metric_spec.json":
        "ecad4021b84c9a11f95b59cfd52a2e691127ccbb2ef0e0bfcce40193e150f223",
}


SEEDS = [
    42,
    43,
    44,
    45,
    46,
]

FAMILIES = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

LEARNERS = [
    "XGBOOST",
    "LIGHTGBM",
]


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(text):

    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):

    p = subprocess.run(
        [
            str(x)
            for x in cmd
        ],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:

        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            + " ".join(
                map(
                    str,
                    cmd,
                )
            )
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args):

    return (
        run(
            [
                "git",
                *args,
            ]
        ).stdout
        or ""
    ).strip()


def sha256_file(
    path,
    chunk=16 * 1024 * 1024,
):

    h = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as f:

        while True:

            b = f.read(
                chunk
            )

            if not b:
                break

            h.update(
                b
            )

    return h.hexdigest()


def read_json(path):

    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path,
    obj,
):

    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            allow_nan=True,
        )
        + "\n",
        encoding="utf-8",
    )


def finite_or_nan(v):

    if v is None:
        return float("nan")

    try:
        x = float(v)

    except Exception:
        return float("nan")

    if not math.isfinite(x):
        return float("nan")

    return x


def op_metrics(op):

    if not isinstance(
        op,
        dict,
    ):
        return {}

    if (
        "result" in op
        and isinstance(
            op[
                "result"
            ],
            dict,
        )
    ):
        op = op[
            "result"
        ]

    return op


def seed_stats(values):

    vals = np.asarray(
        [
            finite_or_nan(v)
            for v in values
        ],
        dtype=np.float64,
    )

    if vals.size != 5:

        raise RuntimeError(
            f"Five-seed group has {vals.size} values; expected 5."
        )


    n_defined = int(
        np.isfinite(
            vals
        ).sum()
    )


    # Frozen undefined-metric policy:
    # preserve NaN; do not silently compute over fewer than five seeds.
    if n_defined != 5:

        return {

            "n_seeds": 5,

            "n_defined": n_defined,

            "mean": float("nan"),

            "median": float("nan"),

            "sample_standard_deviation_ddof_1":
                float("nan"),

            "minimum": float("nan"),

            "maximum": float("nan"),

            "range": float("nan"),

            "IQR_Q75_minus_Q25_linear":
                float("nan"),
        }


    q25, q75 = np.quantile(
        vals,
        [
            0.25,
            0.75,
        ],
        method="linear",
    )


    return {

        "n_seeds": 5,

        "n_defined": 5,

        "mean": float(
            np.mean(
                vals
            )
        ),

        "median": float(
            np.median(
                vals
            )
        ),

        "sample_standard_deviation_ddof_1":
            float(
                np.std(
                    vals,
                    ddof=1,
                )
            ),

        "minimum": float(
            np.min(
                vals
            )
        ),

        "maximum": float(
            np.max(
                vals
            )
        ),

        "range": float(
            np.max(
                vals
            )
            - np.min(
                vals
            )
        ),

        "IQR_Q75_minus_Q25_linear":
            float(
                q75
                - q25
            ),
    }


def write_csv(
    path,
    rows,
    fields,
):

    with Path(path).open(
        "w",
        encoding="utf-8",
        newline="",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fields,
        )

        writer.writeheader()

        writer.writerows(
            rows
        )


def recover_github_token():

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]


    try:

        from kaggle_secrets import (
            UserSecretsClient,
        )

        client = (
            UserSecretsClient()
        )

        for label in labels:

            try:
                value = client.get_secret(
                    label
                )

            except Exception:
                value = None


            if (
                isinstance(
                    value,
                    str,
                )
                and value.strip()
            ):

                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass


    for label in labels:

        value = os.environ.get(
            label
        )

        if (
            isinstance(
                value,
                str,
            )
            and value.strip()
        ):

            return (
                value.strip(),
                f"environment:{label}",
            )


    raise RuntimeError(
        "GitHub token unavailable."
    )


def authenticated_push(token):

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode(
            "utf-8"
        )
    ).decode(
        "ascii"
    )


    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(
            REPO
        ),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )


    if p.returncode != 0:

        raise RuntimeError(
            "Git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )


    return (
        p.stdout
        + p.stderr
    ).strip()


# =================================================================================================
# 0. REPOSITORY + CLOSURE GATE
# =================================================================================================

banner(
    "STAGE28-3B — REPOSITORY / CLOSURE GATE"
)


if OUT.exists():

    raise RuntimeError(
        "Stage28-3B output already exists:\n"
        f"{OUT}\n\n"
        "Do not overwrite an existing synthesis."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository must be clean before Stage28-3B."
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)


print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    local_head,
)

print(
    "origin/main    :",
    origin_head,
)


if (
    local_head
    != EXPECTED_PARENT
    or origin_head
    != EXPECTED_PARENT
):

    raise RuntimeError(
        "Stage28-3B parent mismatch."
    )


closure = read_json(
    CLOSURE
)


if (
    closure.get(
        "closure_status"
    )
    !=
    "PASS_STAGE28_NEW_MODEL_FITTING_PERMANENTLY_CLOSED"
):

    raise RuntimeError(
        "Stage28-3A closure receipt is not PASS."
    )


if (
    closure[
        "fit_budget_closure"
    ][
        "consumed_new_fits"
    ]
    != 108
    or
    closure[
        "fit_budget_closure"
    ][
        "remaining_new_fits"
    ]
    != 0
):

    raise RuntimeError(
        "Closure ledger is not 108/108 with zero remaining."
    )


if (
    closure[
        "scientific_scope"
    ][
        "new_model_fits_authorized_after_closure"
    ]
    != 0
):

    raise RuntimeError(
        "Stage28 closure does not forbid additional fits."
    )


print()
print(
    "[PASS] Stage28-3A closure exact:"
)

print(
    "       108 / 108 consumed"
)

print(
    "       0 remaining"
)

print(
    "[PASS] ZERO fits"
)

print(
    "[PASS] ZERO inference"
)

print(
    "[PASS] ZERO target openings"
)


# =================================================================================================
# 1. FROZEN SYNTHESIS SPEC
# =================================================================================================

banner(
    "STAGE28-3B — FROZEN SYNTHESIS SPEC GATE"
)


for name, expected in (
    EXPECTED_PROTOCOL_SHA256.items()
):

    path = (
        PROTOCOL
        / name
    )

    actual = sha256_file(
        path
    )

    print(
        name,
        actual,
    )


    if actual != expected:

        raise RuntimeError(
            f"{name} SHA256 mismatch."
        )


seed_spec = read_json(
    PROTOCOL
    / "seed_uncertainty_spec.json"
)

stab_spec = read_json(
    PROTOCOL
    / "conclusion_stability_spec.json"
)

metric_spec = read_json(
    PROTOCOL
    / "metric_spec.json"
)


expected_stats = [

    "mean",

    "median",

    "sample_standard_deviation_ddof_1",

    "minimum",

    "maximum",

    "range",

    "IQR_Q75_minus_Q25_linear",
]


if (
    seed_spec[
        "five_seed_statistics"
    ]
    != expected_stats
):

    raise RuntimeError(
        "Frozen five-seed statistics changed."
    )


if (
    seed_spec[
        "combine_into_single_synthetic_ci"
    ]
    is not False
):

    raise RuntimeError(
        "Synthetic seed+bootstrap CI unexpectedly enabled."
    )


if (
    seed_spec[
        "report_separately"
    ]
    is not True
):

    raise RuntimeError(
        "Training-seed uncertainty is no longer separate."
    )


if (
    seed_spec[
        "best_seed_reporting"
    ]
    != "FORBIDDEN"
):

    raise RuntimeError(
        "Best-seed reporting is not frozen as forbidden."
    )


if (
    metric_spec[
        "aggregation_rule"
    ]
    !=
    "Family-specific LOAO results remain primary. No single aggregate zero-day score is authorized."
):

    raise RuntimeError(
        "LOAO aggregation rule changed."
    )


print()
print(
    "[PASS] five-seed statistic set exact"
)

print(
    "[PASS] best-seed reporting forbidden"
)

print(
    "[PASS] seed uncertainty remains separate from bootstrap uncertainty"
)

print(
    "[PASS] no aggregate zero-day score authorized"
)


# =================================================================================================
# 2. STAGE22 DEVELOPMENT-VALIDATION SEED UNCERTAINTY
# =================================================================================================

banner(
    "STAGE28-3B — STAGE22 DEVELOPMENT-VALIDATION SEED UNCERTAINTY"
)


stage22_rows = []


stage22_results = sorted(
    STAGE22_ROOT.rglob(
        "*_result.json"
    )
)


if len(
    stage22_results
) != 10:

    raise RuntimeError(
        f"Expected 10 Stage22 result receipts; "
        f"found {len(stage22_results)}."
    )


for path in stage22_results:

    obj = read_json(
        path
    )

    seed = int(
        obj[
            "training_seed"
        ]
    )

    unit = obj[
        "unit"
    ]


    vp = obj[
        "validation_probability"
    ]


    metrics = {

        "PR_AUC":
            vp[
                "pr_auc"
            ],

        "ROC_AUC":
            vp[
                "roc_auc"
            ],
    }


    ops = obj[
        "operating_points"
    ]


    for label, key in [

        (
            "STANDARD",
            "standard",
        ),

        (
            "BALANCED",
            "balanced",
        ),

        (
            "SECURITY",
            "security",
        ),
    ]:

        op = op_metrics(
            ops[
                key
            ]
        )


        for metric in [

            "precision",

            "recall",

            "fpr",

            "f1",

            "f2",

            "tp",

            "fp",

            "tn",

            "fn",

            "threshold",
        ]:

            if metric in op:

                metrics[
                    f"{label}_{metric.upper()}"
                ] = op[
                    metric
                ]


    for metric, value in (
        metrics.items()
    ):

        stage22_rows.append(
            {

                "parent_stage":
                    "STAGE22_FULL",

                "population":
                    "DEVELOPMENT_VALIDATION_ONLY",

                "unit":
                    unit,

                "learner":
                    "ENS_LGBM_XGB_EQUAL",

                "seed":
                    seed,

                "metric":
                    metric,

                "value":
                    finite_or_nan(
                        value
                    ),

                "source_path":
                    str(
                        path.relative_to(
                            REPO
                        )
                    ),
            }
        )


for unit in [

    "RANDOM_NATURAL",

    "CHRONOLOGICAL_NATURAL",
]:

    seeds = sorted(
        {
            row[
                "seed"
            ]
            for row in stage22_rows
            if (
                row[
                    "unit"
                ]
                == unit
                and
                row[
                    "metric"
                ]
                == "PR_AUC"
            )
        }
    )


    if seeds != SEEDS:

        raise RuntimeError(
            f"Stage22 {unit} seed coverage != 42..46: "
            f"{seeds}"
        )


stage22_summary_rows = []

groups = defaultdict(
    list
)


for row in stage22_rows:

    key = (

        row[
            "parent_stage"
        ],

        row[
            "population"
        ],

        row[
            "unit"
        ],

        row[
            "learner"
        ],

        row[
            "metric"
        ],
    )

    groups[
        key
    ].append(
        (
            row[
                "seed"
            ],
            row[
                "value"
            ],
        )
    )


for key, items in sorted(
    groups.items()
):

    items = sorted(
        items
    )


    if [
        seed
        for seed, _
        in items
    ] != SEEDS:

        raise RuntimeError(
            f"Stage22 summary seed coverage failure: {key}"
        )


    stats = seed_stats(
        [
            value
            for _, value
            in items
        ]
    )


    stage22_summary_rows.append(
        {

            "parent_stage":
                key[
                    0
                ],

            "population":
                key[
                    1
                ],

            "unit":
                key[
                    2
                ],

            "learner":
                key[
                    3
                ],

            "metric":
                key[
                    4
                ],

            **stats,
        }
    )


print(
    "[PASS] Stage22 RANDOM_NATURAL seeds = 42..46"
)

print(
    "[PASS] Stage22 CHRONOLOGICAL_NATURAL seeds = 42..46"
)

print(
    "[PASS] these are DEVELOPMENT-VALIDATION summaries only"
)

print(
    "[PASS] shared final holdout remains unopened"
)


# =================================================================================================
# 3. NORMALIZE LOAO RESULTS
# =================================================================================================

def normalized_loao(
    family,
    learner,
    seed,
    arm,
    parent_stage,
    result,
    support_status,
    inferential,
    source_path,
):

    primary = result[
        "primary_isolation_target"
    ]

    rank = primary[
        "ranking_metrics"
    ]


    chance = rank.get(
        "PR_CHANCE_ANCHOR",
        rank.get(
            "prevalence",
            primary.get(
                "prevalence_chance_anchor"
            ),
        ),
    )


    pr_auc = rank.get(
        "PR_AUC"
    )


    pr_excess = rank.get(
        "PR_EXCESS"
    )


    if (
        pr_excess is None
        and pr_auc is not None
        and chance is not None
    ):

        pr_excess = (
            float(
                pr_auc
            )
            - float(
                chance
            )
        )


    pr_lift = rank.get(
        "PR_LIFT"
    )


    if (
        pr_lift is None
        and chance not in (
            None,
            0,
        )
        and pr_auc is not None
    ):

        pr_lift = (
            float(
                pr_auc
            )
            / float(
                chance
            )
        )


    known = result.get(
        "known_family_control",
        {},
    )


    known_rank = known.get(
        "ranking_metrics",
        {},
    )


    known_chance = (
        known_rank.get(
            "PR_CHANCE_ANCHOR",
            known_rank.get(
                "prevalence"
            ),
        )
    )


    known_pr = known_rank.get(
        "PR_AUC"
    )


    known_excess = (
        known_rank.get(
            "PR_EXCESS"
        )
    )


    if (
        known_excess is None
        and known_pr is not None
        and known_chance is not None
    ):

        known_excess = (
            float(
                known_pr
            )
            - float(
                known_chance
            )
        )


    known_lift = known_rank.get(
        "PR_LIFT"
    )


    if (
        known_lift is None
        and known_chance not in (
            None,
            0,
        )
        and known_pr is not None
    ):

        known_lift = (
            float(
                known_pr
            )
            / float(
                known_chance
            )
        )


    gap = result.get(
        "novelty_generalization_gap_known_minus_unseen",
        {},
    )


    ops = primary[
        "operating_point_metrics"
    ]


    out = {

        "arm":
            arm,

        "parent_stage":
            parent_stage,

        "family":
            family,

        "learner":
            learner,

        "seed":
            int(
                seed
            ),

        "support_n":
            int(
                primary[
                    "heldout_attack"
                ]
            ),

        "support_status":
            support_status,

        "inferential_family_claim_authorized":
            bool(
                inferential
            ),

        "source_path":
            source_path,

        "ROC_AUC":
            finite_or_nan(
                rank.get(
                    "ROC_AUC"
                )
            ),

        "PR_AUC":
            finite_or_nan(
                pr_auc
            ),

        "PR_CHANCE_ANCHOR":
            finite_or_nan(
                chance
            ),

        "PR_EXCESS":
            finite_or_nan(
                pr_excess
            ),

        "PR_LIFT":
            finite_or_nan(
                pr_lift
            ),

        "KNOWN_ROC_AUC":
            finite_or_nan(
                known_rank.get(
                    "ROC_AUC"
                )
            ),

        "KNOWN_PR_AUC":
            finite_or_nan(
                known_pr
            ),

        "KNOWN_PR_CHANCE_ANCHOR":
            finite_or_nan(
                known_chance
            ),

        "KNOWN_PR_EXCESS":
            finite_or_nan(
                known_excess
            ),

        "KNOWN_PR_LIFT":
            finite_or_nan(
                known_lift
            ),

        "GAP_ROC_AUC":
            finite_or_nan(
                gap.get(
                    "ROC_AUC"
                )
            ),

        "GAP_PR_EXCESS":
            finite_or_nan(
                gap.get(
                    "PR_EXCESS"
                )
            ),

        "GAP_RECALL_STANDARD":
            finite_or_nan(
                gap.get(
                    "RECALL_STANDARD"
                )
            ),

        "GAP_RECALL_BALANCED":
            finite_or_nan(
                gap.get(
                    "RECALL_BALANCED"
                )
            ),

        "GAP_RECALL_SECURITY":
            finite_or_nan(
                gap.get(
                    "RECALL_SECURITY"
                )
            ),
    }


    for label in [

        "STANDARD",

        "BALANCED",

        "SECURITY",
    ]:

        op = ops.get(
            label
        )


        feasible = (
            isinstance(
                op,
                dict,
            )
            and
            op.get(
                "status"
            )
            != "UNAVAILABLE"
        )


        out[
            f"{label}_FEASIBLE"
        ] = bool(
            feasible
        )


        op = op_metrics(
            op
        )


        for metric in [

            "recall",

            "fpr",

            "precision",

            "f1",

            "tp",

            "fp",

            "tn",

            "fn",

            "threshold",
        ]:

            out[
                f"{label}_{metric.upper()}"
            ] = finite_or_nan(
                op.get(
                    metric
                )
            )


    return out


# =================================================================================================
# 4. STAGE27 CHRONOLOGY — ALL FIVE SEEDS
# =================================================================================================

banner(
    "STAGE28-3B — STAGE27 CHRONOLOGY LOAO FIVE-SEED MATERIALIZATION"
)


loao_rows = []


# ---------------------------------------------------------------------------------
# Seed42 from frozen Stage27 target results.
# ---------------------------------------------------------------------------------

for historical_path in [

    STAGE27_THU,

    STAGE27_FRI,
]:

    obj = read_json(
        historical_path
    )


    support_summary = obj.get(
        "support_summary",
        {},
    )


    for family, learner_map in (
        obj[
            "results"
        ].items()
    ):

        for learner, result in (
            learner_map.items()
        ):

            support = support_summary.get(
                family,
                {},
            )


            inferential = bool(
                support.get(
                    "inferential_family_claim_authorized",
                    family
                    != "INFILTRATION",
                )
            )


            status = support.get(
                "support_status",
                (
                    "DESCRIPTIVE_ONLY_SUPPORT_LT_50"
                    if family
                    == "INFILTRATION"
                    else
                    "INFERENTIAL_ELIGIBLE_SUPPORT"
                ),
            )


            loao_rows.append(
                normalized_loao(

                    family,

                    learner,

                    42,

                    "28A_CHRONOLOGY_LOAO",

                    "STAGE27_CHRONOLOGY_LOAO",

                    result,

                    status,

                    inferential,

                    str(
                        historical_path.relative_to(
                            REPO
                        )
                    ),
                )
            )


# ---------------------------------------------------------------------------------
# Seeds 43–46 from Stage28A.
# ---------------------------------------------------------------------------------

new_stage27_results = sorted(
    STAGE27_NEW_ROOT.rglob(
        "*_result.json"
    )
)


if len(
    new_stage27_results
) != 40:

    raise RuntimeError(
        f"Expected 40 Stage27 seed-stability result files; "
        f"found {len(new_stage27_results)}."
    )


for path in new_stage27_results:

    obj = read_json(
        path
    )


    seed = int(
        obj[
            "training_seed"
        ]
    )


    if seed not in [
        43,
        44,
        45,
        46,
    ]:

        raise RuntimeError(
            f"Unexpected Stage27 new seed {seed}: {path}"
        )


    loao_rows.append(
        normalized_loao(

            obj[
                "held_out_family"
            ],

            obj[
                "learner"
            ],

            seed,

            "28A_CHRONOLOGY_LOAO",

            "STAGE27_CHRONOLOGY_LOAO",

            obj,

            obj[
                "support_status"
            ],

            obj.get(
                "inferential_family_claim_authorized",
                obj[
                    "held_out_family"
                ]
                != "INFILTRATION",
            ),

            str(
                path.relative_to(
                    REPO
                )
            ),
        )
    )


# =================================================================================================
# 5. STAGE28B RANDOM LOAO — ALL FIVE SEEDS
# =================================================================================================

banner(
    "STAGE28-3B — STAGE28B RANDOM LOAO FIVE-SEED MATERIALIZATION"
)


stage28b_results = sorted(
    STAGE28B_ROOT.rglob(
        "*_result.json"
    )
)


if len(
    stage28b_results
) != 50:

    raise RuntimeError(
        f"Expected 50 Stage28B results; "
        f"found {len(stage28b_results)}."
    )


for path in stage28b_results:

    obj = read_json(
        path
    )


    loao_rows.append(
        normalized_loao(

            obj[
                "held_out_family"
            ],

            obj[
                "learner"
            ],

            int(
                obj[
                    "training_seed"
                ]
            ),

            "28B_RANDOM_LOAO_CONTROL",

            "STAGE28B_RANDOM_LOAO",

            obj,

            obj[
                "support_status"
            ],

            obj.get(
                "inferential_family_claim_authorized",
                obj[
                    "held_out_family"
                ]
                != "INFILTRATION",
            ),

            str(
                path.relative_to(
                    REPO
                )
            ),
        )
    )


# =================================================================================================
# 6. EXACT LOAO COVERAGE
# =================================================================================================

banner(
    "STAGE28-3B — LOAO FIVE-SEED COVERAGE GATE"
)


if len(
    loao_rows
) != 100:

    raise RuntimeError(
        f"Expected exactly 100 LOAO seed-level rows; "
        f"found {len(loao_rows)}."
    )


seen = set()


for row in loao_rows:

    key = (

        row[
            "arm"
        ],

        row[
            "family"
        ],

        row[
            "learner"
        ],

        row[
            "seed"
        ],
    )


    if key in seen:

        raise RuntimeError(
            f"Duplicate LOAO seed-level key: {key}"
        )


    seen.add(
        key
    )


for arm in [

    "28A_CHRONOLOGY_LOAO",

    "28B_RANDOM_LOAO_CONTROL",
]:

    for family in FAMILIES:

        for learner in LEARNERS:

            seeds = sorted(
                row[
                    "seed"
                ]
                for row in loao_rows
                if (
                    row[
                        "arm"
                    ]
                    == arm
                    and
                    row[
                        "family"
                    ]
                    == family
                    and
                    row[
                        "learner"
                    ]
                    == learner
                )
            )


            if seeds != SEEDS:

                raise RuntimeError(
                    f"{arm}/{family}/{learner} "
                    f"seed coverage != 42..46: {seeds}"
                )


print(
    "[PASS] chronology LOAO:"
)

print(
    "       5 families × 2 learners × 5 seeds = 50"
)

print(
    "[PASS] random LOAO:"
)

print(
    "       5 families × 2 learners × 5 seeds = 50"
)

print(
    "[PASS] Infiltration remains descriptive-only"
)


# =================================================================================================
# 7. FROZEN FIVE-SEED STATISTICS
# =================================================================================================

banner(
    "STAGE28-3B — FIVE-SEED STATISTICS"
)


identity_columns = {

    "arm",

    "parent_stage",

    "family",

    "learner",

    "seed",

    "support_n",

    "support_status",

    "inferential_family_claim_authorized",

    "source_path",
}


metric_columns = [

    column
    for column in loao_rows[
        0
    ].keys()
    if (
        column
        not in identity_columns
        and
        not column.endswith(
            "_FEASIBLE"
        )
    )
]


loao_summary_rows = []


for arm in [

    "28A_CHRONOLOGY_LOAO",

    "28B_RANDOM_LOAO_CONTROL",
]:

    for family in FAMILIES:

        for learner in LEARNERS:

            subset = sorted(
                [
                    row
                    for row in loao_rows
                    if (
                        row[
                            "arm"
                        ]
                        == arm
                        and
                        row[
                            "family"
                        ]
                        == family
                        and
                        row[
                            "learner"
                        ]
                        == learner
                    )
                ],
                key=lambda row:
                    row[
                        "seed"
                    ],
            )


            support_n = subset[
                0
            ][
                "support_n"
            ]


            support_status = subset[
                0
            ][
                "support_status"
            ]


            inferential = subset[
                0
            ][
                "inferential_family_claim_authorized"
            ]


            if any(
                row[
                    "support_n"
                ]
                != support_n
                for row in subset
            ):

                raise RuntimeError(
                    f"Support changed across seeds: "
                    f"{arm}/{family}/{learner}"
                )


            for metric in metric_columns:

                stats = seed_stats(
                    [
                        row[
                            metric
                        ]
                        for row in subset
                    ]
                )


                loao_summary_rows.append(
                    {

                        "arm":
                            arm,

                        "parent_stage":
                            subset[
                                0
                            ][
                                "parent_stage"
                            ],

                        "family":
                            family,

                        "learner":
                            learner,

                        "support_n":
                            support_n,

                        "support_status":
                            support_status,

                        "inferential_family_claim_authorized":
                            inferential,

                        "metric":
                            metric,

                        **stats,
                    }
                )


print(
    "[PASS] mean"
)

print(
    "[PASS] median"
)

print(
    "[PASS] sample SD ddof=1"
)

print(
    "[PASS] minimum / maximum / range"
)

print(
    "[PASS] linear IQR"
)

print(
    "[PASS] no best-seed selection or reporting"
)


# =================================================================================================
# 8. FROZEN LOAO CONCLUSION-STABILITY CONDITIONS
# =================================================================================================

banner(
    "STAGE28-3B — FROZEN LOAO CONCLUSION-STABILITY"
)


condition_map = {

    item[
        "id"
    ]:
        item[
            "condition"
        ]

    for item in (
        stab_spec[
            "loao_qualitative_conditions"
        ]
    )
}


expected_conditions = {

    "ROC_ABOVE_CHANCE",

    "PR_ABOVE_CHANCE",

    "STANDARD_DETECTION_PRESENT",

    "BALANCED_DETECTION_PRESENT",

    "SECURITY_DETECTION_PRESENT_WHERE_FEASIBLE",

    "LEARNER_ORDER_ROC_STABILITY",
}


if (
    set(
        condition_map
    )
    != expected_conditions
):

    raise RuntimeError(
        "Frozen LOAO qualitative-condition set changed."
    )


stability_rows = []


for row in loao_rows:

    basic_conditions = [

        (
            "ROC_ABOVE_CHANCE",

            row[
                "ROC_AUC"
            ]
            > 0.5,
        ),

        (
            "PR_ABOVE_CHANCE",

            row[
                "PR_AUC"
            ]
            >
            row[
                "PR_CHANCE_ANCHOR"
            ],
        ),

        (
            "STANDARD_DETECTION_PRESENT",

            row[
                "STANDARD_RECALL"
            ]
            > 0,
        ),

        (
            "BALANCED_DETECTION_PRESENT",

            row[
                "BALANCED_RECALL"
            ]
            > 0,
        ),

        (
            "SECURITY_DETECTION_PRESENT_WHERE_FEASIBLE",

            bool(
                row[
                    "SECURITY_FEASIBLE"
                ]
            )
            and
            math.isfinite(
                row[
                    "SECURITY_RECALL"
                ]
            )
            and
            row[
                "SECURITY_RECALL"
            ]
            > 0,
        ),
    ]


    for claim_id, condition_met in (
        basic_conditions
    ):

        stability_rows.append(
            {

                "claim_id":
                    claim_id,

                "parent_stage":
                    row[
                        "parent_stage"
                    ],

                "family_if_applicable":
                    row[
                        "family"
                    ],

                "learner_if_applicable":
                    row[
                        "learner"
                    ],

                "seed":
                    row[
                        "seed"
                    ],

                "claim_condition":
                    condition_map[
                        claim_id
                    ],

                "condition_met":
                    bool(
                        condition_met
                    ),

                "analysis_status":
                    (
                        "DESCRIPTIVE_ONLY_SUPPORT_LT_50"
                        if row[
                            "family"
                        ]
                        == "INFILTRATION"
                        else
                        "INFERENTIAL_SUPPORT_ELIGIBLE"
                    ),
            }
        )


def sign(x):

    if x > 0:
        return 1

    if x < 0:
        return -1

    return 0


# Learner-order stability is evaluated separately
# for chronology LOAO and random LOAO.
for arm in [

    "28A_CHRONOLOGY_LOAO",

    "28B_RANDOM_LOAO_CONTROL",
]:

    parent_stage = (
        "STAGE27_CHRONOLOGY_LOAO"
        if arm
        == "28A_CHRONOLOGY_LOAO"
        else
        "STAGE28B_RANDOM_LOAO"
    )


    for family in FAMILIES:

        by_seed = {}


        for seed in SEEDS:

            xgb = next(
                row
                for row in loao_rows
                if (
                    row[
                        "arm"
                    ]
                    == arm
                    and
                    row[
                        "family"
                    ]
                    == family
                    and
                    row[
                        "learner"
                    ]
                    == "XGBOOST"
                    and
                    row[
                        "seed"
                    ]
                    == seed
                )
            )


            lgbm = next(
                row
                for row in loao_rows
                if (
                    row[
                        "arm"
                    ]
                    == arm
                    and
                    row[
                        "family"
                    ]
                    == family
                    and
                    row[
                        "learner"
                    ]
                    == "LIGHTGBM"
                    and
                    row[
                        "seed"
                    ]
                    == seed
                )
            )


            by_seed[
                seed
            ] = sign(
                xgb[
                    "ROC_AUC"
                ]
                -
                lgbm[
                    "ROC_AUC"
                ]
            )


        frozen_seed42_sign = (
            by_seed[
                42
            ]
        )


        for seed in SEEDS:

            stability_rows.append(
                {

                    "claim_id":
                        "LEARNER_ORDER_ROC_STABILITY",

                    "parent_stage":
                        parent_stage,

                    "family_if_applicable":
                        family,

                    "learner_if_applicable":
                        "",

                    "seed":
                        seed,

                    "claim_condition":
                        condition_map[
                            "LEARNER_ORDER_ROC_STABILITY"
                        ],

                    "condition_met":
                        bool(
                            by_seed[
                                seed
                            ]
                            ==
                            frozen_seed42_sign
                        ),

                    "analysis_status":
                        (
                            "DESCRIPTIVE_ONLY_SUPPORT_LT_50"
                            if family
                            == "INFILTRATION"
                            else
                            "INFERENTIAL_SUPPORT_ELIGIBLE"
                        ),
                }
            )


# =================================================================================================
# 9. STABILITY RATE = SUPPORTING SEEDS / 5
# =================================================================================================

stability_groups = defaultdict(
    list
)


for row in stability_rows:

    key = (

        row[
            "parent_stage"
        ],

        row[
            "family_if_applicable"
        ],

        row[
            "learner_if_applicable"
        ],

        row[
            "claim_id"
        ],

        row[
            "analysis_status"
        ],
    )


    stability_groups[
        key
    ].append(
        row
    )


stability_summary_rows = []


for key, rows in sorted(
    stability_groups.items()
):

    seeds = sorted(
        row[
            "seed"
        ]
        for row in rows
    )


    if seeds != SEEDS:

        raise RuntimeError(
            "Conclusion-stability seed coverage failure:\n"
            f"{key}\n"
            f"{seeds}"
        )


    supported = sum(
        bool(
            row[
                "condition_met"
            ]
        )
        for row in rows
    )


    stability_summary_rows.append(
        {

            "parent_stage":
                key[
                    0
                ],

            "family_if_applicable":
                key[
                    1
                ],

            "learner_if_applicable":
                key[
                    2
                ],

            "claim_id":
                key[
                    3
                ],

            "analysis_status":
                key[
                    4
                ],

            "frozen_seeds_supporting_condition":
                supported,

            "frozen_seed_count":
                5,

            "stability_rate":
                supported
                / 5.0,
        }
    )


print(
    "[PASS] frozen LOAO qualitative conditions evaluated"
)

print(
    "[PASS] stability rate = supporting frozen seeds / 5"
)

print(
    "[PASS] no post-result condition created"
)


# =================================================================================================
# 10. STAGE22 FINAL-HOLDOUT CLAIMS REMAIN CLOSED
# =================================================================================================

banner(
    "STAGE28-3B — STAGE22 FINAL-HOLDOUT CLAIM DEFERRAL"
)


stage22_claims = (
    stab_spec[
        "stage22_directional_claims"
    ]
)


deferred = {

    "stage":
        "Stage28-3B",

    "status":
        "DEFERRED_TO_STAGE28_4_SHARED_FINAL_HOLDOUT_INFERENCE",

    "reason":
        (
            "The frozen Stage22 directional claims are defined on the "
            "shared final holdout. Stage28-3B is not authorized to open "
            "that holdout or substitute development-validation metrics."
        ),

    "shared_final_holdout_openings_this_stage":
        0,

    "claims":
        stage22_claims,

    "canonical_conclusion_stability_output":
        stab_spec[
            "output_required"
        ],

    "canonical_output_status":
        "NOT_FINALIZED_UNTIL_STAGE28_4",
}


print(
    "[PASS] Stage22 directional claims NOT evaluated on validation"
)

print(
    "[PASS] shared final holdout openings = 0"
)

print(
    "[PASS] Stage22 claim evaluation deferred to Stage28-4"
)


# =================================================================================================
# 11. WRITE ARTIFACTS
# =================================================================================================

banner(
    "STAGE28-3B — WRITE ZERO-FIT SYNTHESIS ARTIFACTS"
)


OUT.mkdir(
    parents=False,
    exist_ok=False,
)


stage22_level_fields = [

    "parent_stage",

    "population",

    "unit",

    "learner",

    "seed",

    "metric",

    "value",

    "source_path",
]


write_csv(
    STAGE22_LEVEL,
    stage22_rows,
    stage22_level_fields,
)


stage22_summary_fields = [

    "parent_stage",

    "population",

    "unit",

    "learner",

    "metric",

    "n_seeds",

    "n_defined",

    "mean",

    "median",

    "sample_standard_deviation_ddof_1",

    "minimum",

    "maximum",

    "range",

    "IQR_Q75_minus_Q25_linear",
]


write_csv(
    STAGE22_SUMMARY,
    stage22_summary_rows,
    stage22_summary_fields,
)


loao_level_fields = list(
    loao_rows[
        0
    ].keys()
)


write_csv(
    LOAO_LEVEL,
    loao_rows,
    loao_level_fields,
)


loao_summary_fields = [

    "arm",

    "parent_stage",

    "family",

    "learner",

    "support_n",

    "support_status",

    "inferential_family_claim_authorized",

    "metric",

    "n_seeds",

    "n_defined",

    "mean",

    "median",

    "sample_standard_deviation_ddof_1",

    "minimum",

    "maximum",

    "range",

    "IQR_Q75_minus_Q25_linear",
]


write_csv(
    LOAO_SUMMARY,
    loao_summary_rows,
    loao_summary_fields,
)


stability_fields = [

    "claim_id",

    "parent_stage",

    "family_if_applicable",

    "learner_if_applicable",

    "seed",

    "claim_condition",

    "condition_met",

    "analysis_status",
]


write_csv(
    LOAO_STABILITY,
    stability_rows,
    stability_fields,
)


stability_summary_fields = [

    "parent_stage",

    "family_if_applicable",

    "learner_if_applicable",

    "claim_id",

    "analysis_status",

    "frozen_seeds_supporting_condition",

    "frozen_seed_count",

    "stability_rate",
]


write_csv(
    LOAO_STABILITY_SUMMARY,
    stability_summary_rows,
    stability_summary_fields,
)


write_json(
    STAGE22_DEFERRED,
    deferred,
)


receipt = {

    "stage":
        "Stage28-3B",

    "type":
        "SEED_UNCERTAINTY_AND_PREFINAL_CONCLUSION_STABILITY_SYNTHESIS",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent_commit":
        EXPECTED_PARENT,

    "closure_gate": {

        "stage28_3a_status":
            closure[
                "closure_status"
            ],

        "authorized_new_fits":
            108,

        "consumed_new_fits":
            108,

        "remaining_new_fits":
            0,
    },

    "scientific_operations": {

        "new_model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "target_openings":
            0,

        "shared_stage22_final_holdout_openings":
            0,

        "bootstrap_recomputation":
            0,

        "new_formal_statistical_tests":
            0,
    },

    "seed_uncertainty": {

        "seeds":
            SEEDS,

        "statistics":
            seed_spec[
                "five_seed_statistics"
            ],

        "best_seed_reporting":
            seed_spec[
                "best_seed_reporting"
            ],

        "combine_with_bootstrap_ci":
            seed_spec[
                "combine_into_single_synthetic_ci"
            ],

        "stage22_validation_groups":
            2,

        "chronology_loao_seed_level_realizations":
            50,

        "random_loao_seed_level_realizations":
            50,

        "family_specific_reporting":
            True,

        "aggregate_zero_day_score_created":
            False,
    },

    "conclusion_stability": {

        "loao_conditions":
            [
                item[
                    "id"
                ]
                for item in (
                    stab_spec[
                        "loao_qualitative_conditions"
                    ]
                )
            ],

        "stability_denominator":
            5,

        "stage22_final_holdout_claims_status":
            "DEFERRED_TO_STAGE28_4",

        "post_result_condition_creation":
            stab_spec[
                "post_result_condition_creation"
            ],
    },

    "support_policy": {

        "INFILTRATION":
            "DESCRIPTIVE_ONLY_SUPPORT_36_LT_50",

        "other_eligible_families":
            "INFERENTIAL_SUPPORT_ELIGIBLE",
    },

    "next_authorized_step":
        (
            "Stage28-3C — random-vs-chronological LOAO contrast "
            "from durable seed-level metrics. ZERO new fits and "
            "ZERO shared-final-holdout openings."
        ),
}


write_json(
    RECEIPT,
    receipt,
)


README.write_text(
    f"""# Stage28-3B — Seed uncertainty and pre-final conclusion stability

Scientific parent: `{EXPECTED_PARENT}`

This stage is synthesis only.

- New model fits: 0
- Model inference: 0
- Threshold selection: 0
- Target openings: 0
- Shared Stage22 final-holdout openings: 0
- Seeds: 42, 43, 44, 45, 46
- Five-seed summaries: mean, median, sample SD (ddof=1), minimum,
  maximum, range, and linear IQR
- Best-seed reporting: forbidden
- Synthetic seed+bootstrap CI: forbidden
- Family-specific LOAO reporting remains primary
- Infiltration remains descriptive-only because support is 36
- Stage22 final-holdout directional claims remain deferred to Stage28-4
- No aggregate zero-day score is created
""",
    encoding="utf-8",
)


artifacts = [

    STAGE22_LEVEL,

    STAGE22_SUMMARY,

    LOAO_LEVEL,

    LOAO_SUMMARY,

    LOAO_STABILITY,

    LOAO_STABILITY_SUMMARY,

    STAGE22_DEFERRED,

    RECEIPT,

    README,
]


with CHECKSUMS.open(
    "w",
    encoding="utf-8",
) as f:

    for path in artifacts:

        f.write(
            f"{sha256_file(path)}  "
            f"{path.name}\n"
        )


artifacts.append(
    CHECKSUMS
)


print(
    "[PASS] Stage28-3B artifacts written"
)

print(
    "[PASS] checksum manifest written"
)


# =================================================================================================
# 12. EXACT COMMIT UNIVERSE
# =================================================================================================

banner(
    "STAGE28-3B — DURABLE COMMIT / PUSH"
)


expected_rel = {

    str(
        path.relative_to(
            REPO
        )
    )

    for path in artifacts
}


tracked = set(
    git(
        "diff",
        "--name-only",
    ).splitlines()
)


staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)


untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
)


if tracked:

    raise RuntimeError(
        "Unexpected tracked modifications before Stage28-3B:\n"
        + "\n".join(
            sorted(
                tracked
            )
        )
    )


if staged:

    raise RuntimeError(
        "Unexpected staged files before Stage28-3B."
    )


if untracked != expected_rel:

    raise RuntimeError(
        "Unexpected Stage28-3B untracked universe.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                expected_rel
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                untracked
            )
        )
    )


for rel in sorted(
    expected_rel
):

    run(
        [
            "git",
            "add",
            "--",
            rel,
        ]
    )


if (
    set(
        git(
            "diff",
            "--cached",
            "--name-only",
        ).splitlines()
    )
    != expected_rel
):

    raise RuntimeError(
        "Stage28-3B staged universe mismatch."
    )


run(
    [
        "git",
        "config",
        "user.name",
        "Stage28 Kaggle",
    ]
)

run(
    [
        "git",
        "config",
        "user.email",
        "stage28-kaggle@users.noreply.github.com",
    ]
)


commit_message = (
    "stage28-3b: freeze five-seed uncertainty synthesis"
)


print(
    run(
        [
            "git",
            "commit",
            "-m",
            commit_message,
        ]
    ).stdout.strip()
)


new_head = git(
    "rev-parse",
    "HEAD",
)


if (
    git(
        "rev-parse",
        "HEAD^",
    )
    != EXPECTED_PARENT
):

    raise RuntimeError(
        "Stage28-3B commit parent mismatch."
    )


token, token_source = (
    recover_github_token()
)


print()
print(
    "[PASS] GitHub credential:",
    token_source,
)

print(
    "[PASS] token not displayed"
)


push_output = (
    authenticated_push(
        token
    )
)


if push_output:
    print(
        push_output
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


if (
    git(
        "rev-parse",
        "HEAD",
    )
    != new_head
    or
    git(
        "rev-parse",
        "origin/main",
    )
    != new_head
):

    raise RuntimeError(
        "Stage28-3B remote durability verification failed."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository dirty after Stage28-3B push."
    )


# =================================================================================================
# 13. DONE
# =================================================================================================

banner(
    "STAGE28-3B — COMPLETE"
)


print(
    "Commit:",
    new_head,
)

print()
print(
    "New model fits                    : 0"
)

print(
    "Model inference                   : 0"
)

print(
    "Threshold selection               : 0"
)

print(
    "Target openings                   : 0"
)

print(
    "Shared Stage22 final holdout opens: 0"
)

print()
print(
    "Chronology LOAO seed rows         : 50 / 50"
)

print(
    "Random LOAO seed rows             : 50 / 50"
)

print(
    "Stage22 final-holdout claims      : DEFERRED TO STAGE28-4"
)

print(
    "Infiltration                      : DESCRIPTIVE ONLY"
)

print(
    "Best-seed reporting               : FORBIDDEN"
)

print(
    "Aggregate zero-day score          : NOT CREATED"
)

print()
print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "Stage28-3C — random-vs-chronological LOAO contrast"
)

print(
    "ZERO NEW FITS."
)


STAGE28-3B — REPOSITORY / CLOSURE GATE

Expected parent: 3318e0bf30b14280347d9d28b6c8ab928231b13b
Local HEAD     : 3318e0bf30b14280347d9d28b6c8ab928231b13b
origin/main    : 3318e0bf30b14280347d9d28b6c8ab928231b13b

[PASS] Stage28-3A closure exact:
       108 / 108 consumed
       0 remaining
[PASS] ZERO fits
[PASS] ZERO inference
[PASS] ZERO target openings

STAGE28-3B — FROZEN SYNTHESIS SPEC GATE

seed_uncertainty_spec.json 2ee9f1c1b84fbea94bdce2996c955496b15790571d09fee7672f46e9193df580
conclusion_stability_spec.json bf834ffbb0f67601be43dd2b6d4eeaf5bb4a5afd505587c7dd305ed30a8eaf41
metric_spec.json ecad4021b84c9a11f95b59cfd52a2e691127ccbb2ef0e0bfcce40193e150f223

[PASS] five-seed statistic set exact
[PASS] best-seed reporting forbidden
[PASS] seed uncertainty remains separate from bootstrap uncertainty
[PASS] no aggregate zero-day score authorized

STAGE28-3B — STAGE22 DEVELOPMENT-VALIDATION SEED UNCERTAINTY

[PASS] Stage22 RANDOM_NATURAL seeds = 42..46
[PASS] Stage22 CHRONOLOGICAL_N

In [10]:
# =================================================================================================
# STAGE28-3C — RANDOM-vs-CHRONOLOGICAL LOAO CONTRAST
#
# ZERO NEW FITS
# ZERO MODEL INFERENCE
# ZERO THRESHOLD SELECTION
# ZERO TARGET OPENINGS
# ZERO STAGE22 SHARED-FINAL-HOLDOUT OPENINGS
#
# Scientific parent:
#   aefeb93e3d2e7e6e965e1c5178347505e57f165f
#
# Input:
#   Stage28-3B durable normalized seed-level LOAO metrics only.
#
# Output:
#   1. matched seed-level random-minus-chronological contrasts
#   2. five-seed contrast summaries
#   3. numeric sign counts across seeds
#   4. interpretation-boundary receipt
#   5. Stage28-3C receipt/checksums
#
# NO:
#   - model fits
#   - predictions
#   - threshold recomputation
#   - target reopening
#   - new significance test
#   - new qualitative cutoff
#   - aggregate zero-day score
# =================================================================================================

from __future__ import annotations

import base64
import csv
import hashlib
import json
import math
import os
import subprocess

from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np


# =================================================================================================
# CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "aefeb93e3d2e7e6e965e1c5178347505e57f165f"
)

ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

PROTOCOL = (
    ROOT
    / "stage28_0_protocol_lock"
)

OUT = (
    ROOT
    / "stage28_3_seed_uncertainty"
)

CLOSURE_RECEIPT = (
    ROOT
    / "stage28_3a_experiment_closure_audit"
    / "stage28_3a_experiment_closure_receipt.json"
)

STAGE3B_RECEIPT = (
    OUT
    / "stage28_3b_receipt.json"
)

INPUT = (
    OUT
    / "stage28_3b_loao_seed_level_metrics.csv"
)

INTERPRETATION_MATRIX = (
    PROTOCOL
    / "interpretation_matrix.json"
)

PROHIBITED_CLAIMS = (
    PROTOCOL
    / "prohibited_claims.json"
)

METRIC_SPEC = (
    PROTOCOL
    / "metric_spec.json"
)

SEED_SPEC = (
    PROTOCOL
    / "seed_uncertainty_spec.json"
)


SEEDS = [
    42,
    43,
    44,
    45,
    46,
]

FAMILIES = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

LEARNERS = [
    "XGBOOST",
    "LIGHTGBM",
]


# Only metrics already frozen for LOAO reporting.
# PR_CHANCE_ANCHOR is included as base-rate context, not a performance metric.
CONTRAST_METRICS = [

    "ROC_AUC",

    "PR_AUC",

    "PR_CHANCE_ANCHOR",

    "PR_EXCESS",

    "PR_LIFT",

    "STANDARD_RECALL",
    "STANDARD_FPR",
    "STANDARD_PRECISION",
    "STANDARD_F1",
    "STANDARD_TP",
    "STANDARD_FP",
    "STANDARD_TN",
    "STANDARD_FN",

    "BALANCED_RECALL",
    "BALANCED_FPR",
    "BALANCED_PRECISION",
    "BALANCED_F1",
    "BALANCED_TP",
    "BALANCED_FP",
    "BALANCED_TN",
    "BALANCED_FN",

    "SECURITY_RECALL",
    "SECURITY_FPR",
    "SECURITY_PRECISION",
    "SECURITY_F1",
    "SECURITY_TP",
    "SECURITY_FP",
    "SECURITY_TN",
    "SECURITY_FN",
]


BASE_RATE_METRICS = {
    "PR_CHANCE_ANCHOR",
}


COUNT_METRICS = {
    "STANDARD_TP",
    "STANDARD_FP",
    "STANDARD_TN",
    "STANDARD_FN",
    "BALANCED_TP",
    "BALANCED_FP",
    "BALANCED_TN",
    "BALANCED_FN",
    "SECURITY_TP",
    "SECURITY_FP",
    "SECURITY_TN",
    "SECURITY_FN",
}


# New Stage28-3C artifacts only.
SEED_LEVEL_OUT = (
    OUT
    / "stage28_3c_random_vs_chronological_seed_level.csv"
)

SUMMARY_OUT = (
    OUT
    / "stage28_3c_random_vs_chronological_five_seed_summary.csv"
)

DIRECTION_OUT = (
    OUT
    / "stage28_3c_numeric_direction_summary.csv"
)

BOUNDARY_OUT = (
    OUT
    / "stage28_3c_interpretation_boundary.json"
)

RECEIPT_OUT = (
    OUT
    / "stage28_3c_receipt.json"
)

README_OUT = (
    OUT
    / "README_STAGE28_3C.md"
)

CHECKSUM_OUT = (
    OUT
    / "stage28_3c_checksums.sha256"
)


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(text):
    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            + " ".join(map(str, cmd))
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return run(
        ["git", *args]
    ).stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def write_json(path, obj):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            allow_nan=True,
        )
        + "\n",
        encoding="utf-8",
    )


def parse_float(v):

    if v is None:
        return float("nan")

    text = str(v).strip()

    if not text:
        return float("nan")

    try:
        x = float(text)

    except Exception:
        return float("nan")

    if not math.isfinite(x):
        return float("nan")

    return x


def parse_bool(v):

    if isinstance(v, bool):
        return v

    text = str(v).strip().lower()

    if text in {
        "true",
        "1",
        "yes",
    }:
        return True

    if text in {
        "false",
        "0",
        "no",
    }:
        return False

    raise RuntimeError(
        f"Cannot parse boolean value: {v!r}"
    )


def write_csv(path, rows, fields):

    with Path(path).open(
        "w",
        encoding="utf-8",
        newline="",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fields,
        )

        writer.writeheader()

        writer.writerows(rows)


def seed_stats(values):

    vals = np.asarray(
        [
            parse_float(v)
            for v in values
        ],
        dtype=np.float64,
    )

    if vals.size != 5:
        raise RuntimeError(
            f"Expected five seed values, found {vals.size}."
        )

    defined = np.isfinite(vals)

    n_defined = int(
        defined.sum()
    )

    # Preserve undefinedness rather than silently shrinking denominator.
    if n_defined != 5:
        return {
            "n_seeds": 5,
            "n_defined": n_defined,
            "mean": float("nan"),
            "median": float("nan"),
            "sample_standard_deviation_ddof_1": float("nan"),
            "minimum": float("nan"),
            "maximum": float("nan"),
            "range": float("nan"),
            "IQR_Q75_minus_Q25_linear": float("nan"),
        }

    q25, q75 = np.quantile(
        vals,
        [
            0.25,
            0.75,
        ],
        method="linear",
    )

    return {
        "n_seeds": 5,
        "n_defined": 5,
        "mean": float(
            np.mean(vals)
        ),
        "median": float(
            np.median(vals)
        ),
        "sample_standard_deviation_ddof_1": float(
            np.std(
                vals,
                ddof=1,
            )
        ),
        "minimum": float(
            np.min(vals)
        ),
        "maximum": float(
            np.max(vals)
        ),
        "range": float(
            np.max(vals)
            - np.min(vals)
        ),
        "IQR_Q75_minus_Q25_linear": float(
            q75 - q25
        ),
    }


def recover_github_token():

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]

    try:
        from kaggle_secrets import UserSecretsClient

        client = UserSecretsClient()

        for label in labels:
            try:
                value = client.get_secret(
                    label
                )
            except Exception:
                value = None

            if (
                isinstance(value, str)
                and value.strip()
            ):
                return (
                    value.strip(),
                    f"kaggle_secret:{label}",
                )

    except Exception:
        pass


    for label in labels:

        value = os.environ.get(
            label
        )

        if (
            isinstance(value, str)
            and value.strip()
        ):
            return (
                value.strip(),
                f"environment:{label}",
            )


    raise RuntimeError(
        "GitHub credential unavailable."
    )


def authenticated_push(token):

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode("utf-8")
    ).decode("ascii")

    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            "Git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    return (
        p.stdout
        + p.stderr
    ).strip()


# =================================================================================================
# 0. REPOSITORY / PARENT GATE
# =================================================================================================

banner(
    "STAGE28-3C — REPOSITORY / PARENT GATE"
)


for output_path in [
    SEED_LEVEL_OUT,
    SUMMARY_OUT,
    DIRECTION_OUT,
    BOUNDARY_OUT,
    RECEIPT_OUT,
    README_OUT,
    CHECKSUM_OUT,
]:
    if output_path.exists():
        raise RuntimeError(
            "Stage28-3C output already exists:\n"
            f"{output_path}"
        )


status = git(
    "status",
    "--porcelain",
)

if status:
    raise RuntimeError(
        "Repository must be clean before Stage28-3C:\n"
        + status
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)


print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    local_head,
)

print(
    "origin/main    :",
    origin_head,
)


if (
    local_head != EXPECTED_PARENT
    or origin_head != EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-3C parent mismatch."
    )


print()
print(
    "[PASS] parent exact"
)

print(
    "[PASS] repository clean"
)

print(
    "[PASS] ZERO fits / ZERO inference / ZERO target openings"
)


# =================================================================================================
# 1. CLOSURE + STAGE28-3B GATES
# =================================================================================================

banner(
    "STAGE28-3C — CLOSURE / INPUT GATE"
)


closure = read_json(
    CLOSURE_RECEIPT
)

stage3b = read_json(
    STAGE3B_RECEIPT
)


if (
    closure[
        "closure_status"
    ]
    !=
    "PASS_STAGE28_NEW_MODEL_FITTING_PERMANENTLY_CLOSED"
):
    raise RuntimeError(
        "Stage28-3A closure is not PASS."
    )


if (
    closure[
        "fit_budget_closure"
    ][
        "consumed_new_fits"
    ]
    != 108
    or
    closure[
        "fit_budget_closure"
    ][
        "remaining_new_fits"
    ]
    != 0
):
    raise RuntimeError(
        "Fit ledger no longer closes at 108/108."
    )


if stage3b[
    "next_authorized_step"
].startswith(
    "Stage28-3C"
) is False:
    raise RuntimeError(
        "Stage28-3B did not authorize Stage28-3C."
    )


for key in [
    "new_model_fits",
    "model_inference",
    "threshold_selection",
    "target_openings",
    "shared_stage22_final_holdout_openings",
]:
    if int(
        stage3b[
            "scientific_operations"
        ][
            key
        ]
    ) != 0:
        raise RuntimeError(
            f"Stage28-3B scientific operation not zero: {key}"
        )


if not INPUT.is_file():
    raise RuntimeError(
        "Stage28-3B normalized LOAO seed-level metrics missing."
    )


print(
    "[PASS] Stage28 permanently closed to fitting"
)

print(
    "[PASS] Stage28-3B zero-operation receipt exact"
)

print(
    "[PASS] Stage28-3C consumes durable normalized metrics only"
)


# =================================================================================================
# 2. FROZEN INTERPRETATION / METRIC POLICY
# =================================================================================================

banner(
    "STAGE28-3C — FROZEN CONTRAST POLICY"
)


interpretation = read_json(
    INTERPRETATION_MATRIX
)

prohibited = read_json(
    PROHIBITED_CLAIMS
)

metric_spec = read_json(
    METRIC_SPEC
)

seed_spec = read_json(
    SEED_SPEC
)


if (
    "RANDOM_VS_CHRONOLOGICAL_LOAO_CONTRAST"
    not in
    metric_spec[
        "loao"
    ][
        "comparative_metrics"
    ]
):
    raise RuntimeError(
        "Frozen metric spec does not authorize random-vs-chronological LOAO contrast."
    )


if (
    metric_spec[
        "aggregation_rule"
    ]
    !=
    "Family-specific LOAO results remain primary. No single aggregate zero-day score is authorized."
):
    raise RuntimeError(
        "Family-specific aggregation rule changed."
    )


if (
    seed_spec[
        "best_seed_reporting"
    ]
    != "FORBIDDEN"
):
    raise RuntimeError(
        "Best-seed reporting unexpectedly allowed."
    )


rule_names = {
    item[
        "result"
    ]
    for item in interpretation[
        "rules"
    ]
}


for required in [
    "RANDOM_LOAO_MUCH_GREATER_THAN_CHRONOLOGICAL_LOAO",
    "RANDOM_AND_CHRONOLOGICAL_LOAO_BOTH_COLLAPSE",
    "RANDOM_AND_CHRONOLOGICAL_LOAO_BOTH_SURVIVE",
    "FAMILY_SPECIFIC_MIXTURE",
    "UNEXPECTED_REVERSAL",
]:
    if required not in rule_names:
        raise RuntimeError(
            f"Interpretation-matrix rule missing: {required}"
        )


if (
    "The random-vs-chronological difference is purely caused by temporal drift."
    not in prohibited[
        "prohibited"
    ]
):
    raise RuntimeError(
        "Expected causal-prohibition statement missing."
    )


print(
    "[PASS] random-vs-chronological contrast explicitly frozen"
)

print(
    "[PASS] family-specific reporting remains primary"
)

print(
    "[PASS] causal attribution to temporal drift remains prohibited"
)

print(
    "[PASS] no aggregate zero-day score"
)


# =================================================================================================
# 3. LOAD 100 DURABLE SEED-LEVEL REALIZATIONS
# =================================================================================================

banner(
    "STAGE28-3C — LOAD DURABLE STAGE28-3B SEED-LEVEL METRICS"
)


with INPUT.open(
    "r",
    encoding="utf-8",
    newline="",
) as f:

    rows = list(
        csv.DictReader(f)
    )


if len(rows) != 100:
    raise RuntimeError(
        f"Expected 100 Stage28-3B LOAO rows; found {len(rows)}."
    )


required_columns = {
    "arm",
    "parent_stage",
    "family",
    "learner",
    "seed",
    "support_n",
    "support_status",
    "inferential_family_claim_authorized",
    *CONTRAST_METRICS,
}


missing_columns = (
    required_columns
    - set(rows[0])
)


if missing_columns:
    raise RuntimeError(
        "Stage28-3B input missing required columns:\n"
        + "\n".join(
            sorted(
                missing_columns
            )
        )
    )


indexed = {}


for row in rows:

    key = (
        row[
            "arm"
        ],
        row[
            "family"
        ],
        row[
            "learner"
        ],
        int(
            row[
                "seed"
            ]
        ),
    )


    if key in indexed:
        raise RuntimeError(
            f"Duplicate Stage28-3B key: {key}"
        )


    indexed[key] = row


for family in FAMILIES:

    for learner in LEARNERS:

        for seed in SEEDS:

            chrono_key = (
                "28A_CHRONOLOGY_LOAO",
                family,
                learner,
                seed,
            )

            random_key = (
                "28B_RANDOM_LOAO_CONTROL",
                family,
                learner,
                seed,
            )


            if chrono_key not in indexed:
                raise RuntimeError(
                    f"Missing chronology realization: {chrono_key}"
                )

            if random_key not in indexed:
                raise RuntimeError(
                    f"Missing random realization: {random_key}"
                )


print(
    "[PASS] 50 chronology realizations present"
)

print(
    "[PASS] 50 random realizations present"
)

print(
    "[PASS] exact family × learner × seed pairing possible"
)


# =================================================================================================
# 4. BUILD MATCHED SEED-LEVEL CONTRASTS
# =================================================================================================

banner(
    "STAGE28-3C — MATCHED RANDOM-MINUS-CHRONOLOGICAL CONTRASTS"
)


contrast_rows = []


for family in FAMILIES:

    for learner in LEARNERS:

        for seed in SEEDS:

            chrono = indexed[
                (
                    "28A_CHRONOLOGY_LOAO",
                    family,
                    learner,
                    seed,
                )
            ]

            random = indexed[
                (
                    "28B_RANDOM_LOAO_CONTROL",
                    family,
                    learner,
                    seed,
                )
            ]


            chrono_support = int(
                chrono[
                    "support_n"
                ]
            )

            random_support = int(
                random[
                    "support_n"
                ]
            )


            if chrono_support != random_support:
                raise RuntimeError(
                    f"Held-out attack support mismatch "
                    f"{family}/{learner}/seed{seed}: "
                    f"{chrono_support} vs {random_support}"
                )


            if (
                family == "INFILTRATION"
                and chrono_support != 36
            ):
                raise RuntimeError(
                    "Infiltration support changed from 36."
                )


            for metric in CONTRAST_METRICS:

                c = parse_float(
                    chrono[
                        metric
                    ]
                )

                r = parse_float(
                    random[
                        metric
                    ]
                )


                if (
                    math.isfinite(c)
                    and math.isfinite(r)
                ):
                    delta = (
                        r - c
                    )

                    if delta > 0:
                        numeric_direction = (
                            "RANDOM_GT_CHRONO"
                        )

                    elif delta < 0:
                        numeric_direction = (
                            "RANDOM_LT_CHRONO"
                        )

                    else:
                        numeric_direction = (
                            "RANDOM_EQ_CHRONO"
                        )

                else:
                    delta = float(
                        "nan"
                    )

                    numeric_direction = (
                        "UNDEFINED"
                    )


                contrast_rows.append(
                    {

                        "family":
                            family,

                        "learner":
                            learner,

                        "seed":
                            seed,

                        "support_n":
                            chrono_support,

                        "analysis_status":
                            (
                                "DESCRIPTIVE_ONLY_SUPPORT_36_LT_50"
                                if family
                                == "INFILTRATION"
                                else
                                "INFERENTIAL_SUPPORT_ELIGIBLE"
                            ),

                        "metric":
                            metric,

                        "metric_role":
                            (
                                "BASE_RATE_CONTEXT"
                                if metric
                                in BASE_RATE_METRICS
                                else
                                (
                                    "CONFUSION_COUNT_CONTEXT"
                                    if metric
                                    in COUNT_METRICS
                                    else
                                    "PERFORMANCE_METRIC"
                                )
                            ),

                        "chronological_value":
                            c,

                        "random_value":
                            r,

                        "random_minus_chronological":
                            delta,

                        "numeric_direction":
                            numeric_direction,

                        "direction_semantics":
                            (
                                "NUMERIC_ONLY_NOT_AUTOMATICALLY_BETTER_OR_WORSE"
                            ),
                    }
                )


expected_seed_rows = (
    5
    * 2
    * 5
    * len(
        CONTRAST_METRICS
    )
)


if len(
    contrast_rows
) != expected_seed_rows:
    raise RuntimeError(
        f"Seed-level contrast row count mismatch: "
        f"{len(contrast_rows)} != {expected_seed_rows}"
    )


print(
    "[PASS] every contrast is matched by family + learner + seed"
)

print(
    "[PASS] contrast definition = RANDOM minus CHRONOLOGICAL"
)

print(
    "[PASS] Infiltration remains descriptive-only"
)


# =================================================================================================
# 5. FIVE-SEED CONTRAST SUMMARY
# =================================================================================================

banner(
    "STAGE28-3C — FIVE-SEED CONTRAST SUMMARY"
)


groups = defaultdict(
    list
)


for row in contrast_rows:

    key = (
        row[
            "family"
        ],
        row[
            "learner"
        ],
        row[
            "metric"
        ],
        row[
            "analysis_status"
        ],
        row[
            "metric_role"
        ],
    )

    groups[
        key
    ].append(
        row
    )


summary_rows = []

direction_rows = []


for key, items in sorted(
    groups.items()
):

    items = sorted(
        items,
        key=lambda row:
            row[
                "seed"
            ],
    )


    if [
        row[
            "seed"
        ]
        for row in items
    ] != SEEDS:
        raise RuntimeError(
            f"Five-seed coverage failure: {key}"
        )


    deltas = [
        row[
            "random_minus_chronological"
        ]
        for row in items
    ]


    stats = seed_stats(
        deltas
    )


    summary_rows.append(
        {

            "family":
                key[
                    0
                ],

            "learner":
                key[
                    1
                ],

            "metric":
                key[
                    2
                ],

            "analysis_status":
                key[
                    3
                ],

            "metric_role":
                key[
                    4
                ],

            "contrast_definition":
                "RANDOM_MINUS_CHRONOLOGICAL",

            **stats,
        }
    )


    directions = [
        row[
            "numeric_direction"
        ]
        for row in items
    ]


    direction_rows.append(
        {

            "family":
                key[
                    0
                ],

            "learner":
                key[
                    1
                ],

            "metric":
                key[
                    2
                ],

            "analysis_status":
                key[
                    3
                ],

            "metric_role":
                key[
                    4
                ],

            "frozen_seed_count":
                5,

            "defined_seed_count":
                sum(
                    d != "UNDEFINED"
                    for d in directions
                ),

            "random_gt_chrono_seed_count":
                directions.count(
                    "RANDOM_GT_CHRONO"
                ),

            "random_lt_chrono_seed_count":
                directions.count(
                    "RANDOM_LT_CHRONO"
                ),

            "random_eq_chrono_seed_count":
                directions.count(
                    "RANDOM_EQ_CHRONO"
                ),

            "undefined_seed_count":
                directions.count(
                    "UNDEFINED"
                ),

            "note":
                (
                    "Numeric sign count only; no post-result "
                    "qualitative cutoff or causal attribution."
                ),
        }
    )


print(
    "[PASS] mean / median / sample SD / min / max / range / IQR"
)

print(
    "[PASS] seedwise numeric sign counts"
)

print(
    "[PASS] no best-seed reporting"
)

print(
    "[PASS] no new formal significance test"
)


# =================================================================================================
# 6. INTERPRETATION BOUNDARY
# =================================================================================================

banner(
    "STAGE28-3C — INTERPRETATION BOUNDARY"
)


boundary = {

    "stage":
        "Stage28-3C",

    "contrast":
        "RANDOM_MINUS_CHRONOLOGICAL_LOAO",

    "qualitative_interpretation_matrix_present":
        True,

    "automatic_interpretation_labels_assigned":
        False,

    "reason":
        (
            "The frozen interpretation matrix contains labels such as "
            "RANDOM_LOAO_MUCH_GREATER_THAN_CHRONOLOGICAL_LOAO and "
            "RANDOM_AND_CHRONOLOGICAL_LOAO_BOTH_COLLAPSE, but no frozen "
            "numeric cutoffs operationalize 'MUCH_GREATER', 'COLLAPSE', "
            "or 'SURVIVE'. Assigning thresholds after observing Stage28 "
            "results would constitute a post-result condition."
        ),

    "authorized_output":
        (
            "Continuous matched seed-level contrasts, frozen five-seed "
            "summary statistics, and numeric sign counts only."
        ),

    "causal_claim_prohibited":
        (
            "The random-vs-chronological difference is purely caused by temporal drift."
        ),

    "preferred_interpretive_language":
        [
            "random-vs-chronological LOAO contrast",
            "consistent with chronology compounding novelty difficulty",
            "conditional on the benchmark and frozen protocol",
        ],

    "infiltration":
        "DESCRIPTIVE_ONLY_SUPPORT_36_LT_50",

    "aggregate_zero_day_score":
        "NOT_AUTHORIZED_AND_NOT_CREATED",

    "shared_stage22_final_holdout_openings_this_stage":
        0,

    "new_formal_statistical_tests_this_stage":
        0,

    "post_result_condition_creation":
        0,
}


write_json(
    BOUNDARY_OUT,
    boundary,
)


print(
    "[PASS] no unfrozen 'much greater' cutoff created"
)

print(
    "[PASS] no unfrozen 'collapse' cutoff created"
)

print(
    "[PASS] no causal separation claim"
)


# =================================================================================================
# 7. WRITE CONTRAST TABLES
# =================================================================================================

banner(
    "STAGE28-3C — WRITE CONTRAST ARTIFACTS"
)


seed_fields = [
    "family",
    "learner",
    "seed",
    "support_n",
    "analysis_status",
    "metric",
    "metric_role",
    "chronological_value",
    "random_value",
    "random_minus_chronological",
    "numeric_direction",
    "direction_semantics",
]


summary_fields = [
    "family",
    "learner",
    "metric",
    "analysis_status",
    "metric_role",
    "contrast_definition",
    "n_seeds",
    "n_defined",
    "mean",
    "median",
    "sample_standard_deviation_ddof_1",
    "minimum",
    "maximum",
    "range",
    "IQR_Q75_minus_Q25_linear",
]


direction_fields = [
    "family",
    "learner",
    "metric",
    "analysis_status",
    "metric_role",
    "frozen_seed_count",
    "defined_seed_count",
    "random_gt_chrono_seed_count",
    "random_lt_chrono_seed_count",
    "random_eq_chrono_seed_count",
    "undefined_seed_count",
    "note",
]


write_csv(
    SEED_LEVEL_OUT,
    contrast_rows,
    seed_fields,
)

write_csv(
    SUMMARY_OUT,
    summary_rows,
    summary_fields,
)

write_csv(
    DIRECTION_OUT,
    direction_rows,
    direction_fields,
)


receipt = {

    "stage":
        "Stage28-3C",

    "type":
        "RANDOM_VS_CHRONOLOGICAL_LOAO_CONTRAST",

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "scientific_parent_commit":
        EXPECTED_PARENT,

    "input": {

        "path":
            str(
                INPUT.relative_to(
                    REPO
                )
            ),

        "sha256":
            sha256_file(
                INPUT
            ),

        "chronology_seed_realizations":
            50,

        "random_seed_realizations":
            50,
    },

    "contrast_design": {

        "pairing":
            "EXACT_FAMILY_LEARNER_MODEL_SEED",

        "contrast":
            "RANDOM_VALUE_MINUS_CHRONOLOGICAL_VALUE",

        "families":
            FAMILIES,

        "learners":
            LEARNERS,

        "seeds":
            SEEDS,

        "metrics":
            CONTRAST_METRICS,

        "five_seed_statistics":
            seed_spec[
                "five_seed_statistics"
            ],

        "numeric_sign_counts":
            True,

        "best_seed_reporting":
            "FORBIDDEN",

        "new_qualitative_cutoff":
            False,

        "new_formal_statistical_test":
            False,

        "aggregate_zero_day_score":
            False,
    },

    "scientific_operations": {

        "new_model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "target_openings":
            0,

        "shared_stage22_final_holdout_openings":
            0,

        "bootstrap_recomputation":
            0,

        "new_formal_statistical_tests":
            0,
    },

    "support_policy": {

        "INFILTRATION":
            "DESCRIPTIVE_ONLY_SUPPORT_36_LT_50",

        "other_families":
            "INFERENTIAL_SUPPORT_ELIGIBLE",
    },

    "interpretation_boundary": {

        "automatic_interpretation_matrix_category_assignment":
            False,

        "reason":
            "NO_FROZEN_NUMERIC_CUTOFF_FOR_MUCH_GREATER_COLLAPSE_OR_SURVIVE",

        "causal_temporal_drift_attribution":
            "PROHIBITED",
    },

    "next_authorized_step":
        (
            "Stage28-4 — preregistered one-time Stage22 shared-final-holdout "
            "inference for all five seed realizations and final Stage22 "
            "directional conclusion-stability evaluation. ZERO new fits; "
            "threshold selection on the final holdout remains forbidden."
        ),
}


write_json(
    RECEIPT_OUT,
    receipt,
)


README_OUT.write_text(
    f"""# Stage28-3C — Random-vs-chronological LOAO contrast

Scientific parent: `{EXPECTED_PARENT}`

This stage is synthesis only.

## Operations

- New model fits: 0
- Model inference: 0
- Threshold selection: 0
- Target openings: 0
- Shared Stage22 final-holdout openings: 0
- New formal statistical tests: 0

## Contrast

Every Stage28B random-LOAO result is paired with the Stage27 chronology-LOAO
result having the same held-out family, learner, and model seed.

The reported contrast is:

`random metric - chronological metric`

The five frozen seeds are summarized using the preregistered mean, median,
sample standard deviation (ddof=1), minimum, maximum, range, and linear IQR.

Numeric positive/negative/equal seed counts are also reported.

## Interpretation boundary

No new cutoff is created for terms such as "much greater", "collapse", or
"survive" because the frozen interpretation matrix did not define numeric
thresholds for those labels.

Accordingly, Stage28-3C reports continuous contrasts only.

The random-vs-chronological comparison does not prove that temporal drift is
the sole cause of the difference. Preferred wording is that a contrast may be
"consistent with chronology compounding novelty difficulty", conditional on
the benchmark and frozen protocol.

Infiltration remains descriptive-only because held-out support is 36.

No aggregate zero-day score is created.
""",
    encoding="utf-8",
)


artifact_paths = [
    SEED_LEVEL_OUT,
    SUMMARY_OUT,
    DIRECTION_OUT,
    BOUNDARY_OUT,
    RECEIPT_OUT,
    README_OUT,
]


with CHECKSUM_OUT.open(
    "w",
    encoding="utf-8",
) as f:

    for path in artifact_paths:

        f.write(
            f"{sha256_file(path)}  "
            f"{path.name}\n"
        )


artifact_paths.append(
    CHECKSUM_OUT
)


print(
    "[PASS] seed-level contrast table written"
)

print(
    "[PASS] five-seed contrast summary written"
)

print(
    "[PASS] numeric direction summary written"
)

print(
    "[PASS] interpretation boundary frozen"
)

print(
    "[PASS] Stage28-3C receipt written"
)


# =================================================================================================
# 8. EXACT GIT UNIVERSE
# =================================================================================================

banner(
    "STAGE28-3C — DURABLE COMMIT / PUSH"
)


expected_rel = {
    str(
        path.relative_to(
            REPO
        )
    )
    for path in artifact_paths
}


tracked_changes = set(
    git(
        "diff",
        "--name-only",
    ).splitlines()
)

staged_before = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)

untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
)


if tracked_changes:
    raise RuntimeError(
        "Unexpected tracked modifications before Stage28-3C:\n"
        + "\n".join(
            sorted(
                tracked_changes
            )
        )
    )


if staged_before:
    raise RuntimeError(
        "Unexpected staged files before Stage28-3C."
    )


if untracked != expected_rel:
    raise RuntimeError(
        "Unexpected Stage28-3C untracked universe.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                expected_rel
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                untracked
            )
        )
    )


for rel in sorted(
    expected_rel
):
    run(
        [
            "git",
            "add",
            "--",
            rel,
        ]
    )


staged_after = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)


if staged_after != expected_rel:
    raise RuntimeError(
        "Stage28-3C staged universe mismatch."
    )


run(
    [
        "git",
        "config",
        "user.name",
        "Stage28 Kaggle",
    ]
)

run(
    [
        "git",
        "config",
        "user.email",
        "stage28-kaggle@users.noreply.github.com",
    ]
)


commit_message = (
    "stage28-3c: freeze random-vs-chronological LOAO contrast"
)


commit_result = run(
    [
        "git",
        "commit",
        "-m",
        commit_message,
    ]
)


print(
    commit_result.stdout.strip()
)


new_head = git(
    "rev-parse",
    "HEAD",
)


if (
    git(
        "rev-parse",
        "HEAD^",
    )
    != EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-3C commit parent mismatch."
    )


token, token_source = recover_github_token()


print()
print(
    "[PASS] GitHub credential:",
    token_source,
)

print(
    "[PASS] token not displayed"
)


push_output = authenticated_push(
    token
)


if push_output:
    print(
        push_output
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


if (
    git(
        "rev-parse",
        "HEAD",
    )
    != new_head
    or
    git(
        "rev-parse",
        "origin/main",
    )
    != new_head
):
    raise RuntimeError(
        "Stage28-3C remote durability verification failed."
    )


if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Repository dirty after Stage28-3C push."
    )


# =================================================================================================
# 9. COMPLETE
# =================================================================================================

banner(
    "STAGE28-3C — COMPLETE"
)


print(
    "Commit:",
    new_head,
)

print()
print(
    "New model fits                    : 0"
)

print(
    "Model inference                   : 0"
)

print(
    "Threshold selection               : 0"
)

print(
    "Target openings                   : 0"
)

print(
    "Shared Stage22 final holdout opens: 0"
)

print(
    "New formal statistical tests      : 0"
)

print()
print(
    "Matched chronology realizations   : 50"
)

print(
    "Matched random realizations       : 50"
)

print(
    "Contrast                          : RANDOM - CHRONOLOGICAL"
)

print(
    "Families                          : 5"
)

print(
    "Learners                          : 2"
)

print(
    "Seeds                             : 5"
)

print(
    "Infiltration                      : DESCRIPTIVE ONLY"
)

print(
    "Aggregate zero-day score          : NOT CREATED"
)

print(
    "Post-result qualitative cutoff    : NOT CREATED"
)

print()
print(
    "NEXT AUTHORIZED STEP:"
)

print(
    "Stage28-4 — one-time Stage22 shared-final-holdout inference"
)

print(
    "ZERO NEW FITS; FINAL-HOLDOUT THRESHOLD SEARCH FORBIDDEN."
)


STAGE28-3C — REPOSITORY / PARENT GATE

Expected parent: aefeb93e3d2e7e6e965e1c5178347505e57f165f
Local HEAD     : aefeb93e3d2e7e6e965e1c5178347505e57f165f
origin/main    : aefeb93e3d2e7e6e965e1c5178347505e57f165f

[PASS] parent exact
[PASS] repository clean
[PASS] ZERO fits / ZERO inference / ZERO target openings

STAGE28-3C — CLOSURE / INPUT GATE

[PASS] Stage28 permanently closed to fitting
[PASS] Stage28-3B zero-operation receipt exact
[PASS] Stage28-3C consumes durable normalized metrics only

STAGE28-3C — FROZEN CONTRAST POLICY

[PASS] random-vs-chronological contrast explicitly frozen
[PASS] family-specific reporting remains primary
[PASS] causal attribution to temporal drift remains prohibited
[PASS] no aggregate zero-day score

STAGE28-3C — LOAD DURABLE STAGE28-3B SEED-LEVEL METRICS

[PASS] 50 chronology realizations present
[PASS] 50 random realizations present
[PASS] exact family × learner × seed pairing possible

STAGE28-3C — MATCHED RANDOM-MINUS-CHRONOLOGICAL CONTRASTS

[P

In [11]:
# =================================================================================================
# STAGE28-4 — PRE-INFERENCE GATE
#
# OPERATIONAL PREFLIGHT ONLY — NOT A NEW SCIENTIFIC STAGE
#
# ZERO MODEL FITS
# ZERO MODEL INFERENCE
# ZERO THRESHOLD SELECTION
# ZERO FINAL-HOLDOUT PREDICTOR ROWS READ
# ZERO FINAL-HOLDOUT LABEL ROWS READ
#
# Purpose:
#   - verify exact Stage28-3C parent
#   - verify Stage28 closure remains 108/108
#   - verify Stage28-4 authorization
#   - audit frozen Stage22R final-holdout membership artifact
#   - inspect membership NPZ schema only
#   - locate exact Kaggle source files without reading data rows
#   - validate CSV headers only
#   - audit all 10 Stage28 Stage22 ensemble/model identities
#   - print already-frozen operating thresholds
#
# NO SCIENTIFIC RESULT IS COMPUTED HERE.
# =================================================================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import subprocess
from pathlib import Path

import numpy as np


SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "2679d0c208d514b381caa12e96c959f4f2ee5ee7"
)

ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

CLOSURE_RECEIPT = (
    ROOT
    / "stage28_3a_experiment_closure_audit"
    / "stage28_3a_experiment_closure_receipt.json"
)

STAGE3C_RECEIPT = (
    ROOT
    / "stage28_3_seed_uncertainty"
    / "stage28_3c_receipt.json"
)

STAGE22_SPEC = (
    ROOT
    / "stage28_0_protocol_lock"
    / "stage22_cell_spec.json"
)

STAGE22_STAGE28_ROOT = (
    ROOT
    / "stage28_2a_stage22_seed_stability"
)

STAGE22R_FINAL_ROOT = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_final_single_holdout"
)

MEMBERSHIP = (
    STAGE22R_FINAL_ROOT
    / "stage22r_final_holdout_clean_membership.npz"
)

PARENT_SUMMARY = (
    STAGE22R_FINAL_ROOT
    / "stage22r_final_holdout_k79_summary.json"
)

PARENT_RESULT = (
    STAGE22R_FINAL_ROOT
    / "stage22r_final_holdout_result.json"
)

PARENT_CHECKSUMS = (
    STAGE22R_FINAL_ROOT
    / "checksums.sha256"
)

FEATURE_CONFIG = (
    REPO
    / "results"
    / "stage15_transformer_checkpoint"
    / "stage15_1_feature_configuration.json"
)

LOCAL_PREFLIGHT_RECEIPT = Path(
    "/kaggle/working/stage28_4_preflight_receipt.json"
)

EXPECTED_ROWS = 1_374_133
EXPECTED_BENIGN = 998_788
EXPECTED_ATTACK = 375_345
EXPECTED_FEATURES = 70

EXPECTED_UNITS = [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]

EXPECTED_SEEDS = [
    42,
    43,
    44,
    45,
    46,
]

EXPECTED_LEARNERS = [
    "xgboost",
    "lightgbm",
]


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(text):
    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def run(cmd, *, cwd=REPO, check=True):

    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            + " ".join(map(str, cmd))
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args):

    return run(
        ["git", *args]
    ).stdout.strip()


def read_json(path):

    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def sha256_file(path, chunk=16 * 1024 * 1024):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            block = f.read(chunk)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def resolve_checksum_for_basename(
    checksum_file,
    wanted_basename,
):

    matches = []

    for raw in Path(
        checksum_file
    ).read_text(
        encoding="utf-8"
    ).splitlines():

        raw = raw.strip()

        if not raw:
            continue

        parts = raw.split(
            None,
            1,
        )

        if len(parts) != 2:
            continue

        digest = parts[0].strip()

        name = (
            parts[1]
            .strip()
            .lstrip("*")
        )

        if Path(name).name == wanted_basename:

            matches.append(
                (
                    digest,
                    name,
                )
            )

    if len(matches) != 1:

        raise RuntimeError(
            "Expected exactly one checksum entry for "
            f"{wanted_basename}; found {len(matches)}."
        )

    return matches[0]


def recursively_find_feature_lists(obj):

    candidates = []

    def walk(x, path="root"):

        if isinstance(x, dict):

            for k, v in x.items():

                walk(
                    v,
                    f"{path}.{k}",
                )

        elif isinstance(x, list):

            if (
                len(x) == EXPECTED_FEATURES
                and
                all(
                    isinstance(v, str)
                    for v in x
                )
            ):

                candidates.append(
                    (
                        path,
                        x,
                    )
                )

            for i, v in enumerate(x):

                walk(
                    v,
                    f"{path}[{i}]",
                )

    walk(obj)

    return candidates


def read_csv_header_only(path):

    # IMPORTANT:
    # reads exactly the header line and NO data row.
    with Path(path).open(
        "r",
        encoding="utf-8-sig",
        errors="strict",
        newline="",
    ) as f:

        first_line = f.readline()

    if not first_line:
        raise RuntimeError(
            f"Empty CSV source: {path}"
        )

    return next(
        csv.reader(
            [first_line]
        )
    )


def normalize_header_name(value):

    return (
        str(value)
        .replace("\ufeff", "")
        .strip()
    )


def resolve_model_path(
    result_path,
    model,
):

    local_name = (
        model.get("model_path")
        or model.get("model")
    )

    if local_name:

        path = (
            result_path.parent
            / local_name
        )

        expected_sha = model.get(
            "model_sha256"
        )

        source_type = (
            "STAGE28_COMPONENT_ARTIFACT"
        )

    else:

        historical = model.get(
            "historical_model_path"
        )

        if not historical:

            raise RuntimeError(
                f"No model artifact path found in {result_path}"
            )

        path = (
            REPO
            / historical
        )

        expected_sha = model.get(
            "historical_model_sha256"
        )

        source_type = (
            "HISTORICAL_REUSE_ARTIFACT"
        )

    if not path.is_file():

        raise RuntimeError(
            "Model file missing:\n"
            f"{path}"
        )

    if not expected_sha:

        raise RuntimeError(
            "Expected model SHA missing in result receipt:\n"
            f"{result_path}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:

        raise RuntimeError(
            "Model SHA mismatch:\n"
            f"{path}\n"
            f"expected={expected_sha}\n"
            f"actual={actual_sha}"
        )

    return (
        path,
        actual_sha,
        source_type,
    )


def extract_threshold(
    operating_points,
    name,
):

    key = name.lower()

    op = operating_points.get(
        key
    )

    if op is None:

        raise RuntimeError(
            f"Missing operating point: {name}"
        )

    if (
        isinstance(op, dict)
        and
        "result" in op
    ):

        if (
            op.get("status")
            not in (
                None,
                "AVAILABLE",
            )
        ):

            return {
                "status": op.get(
                    "status"
                ),
                "threshold": None,
            }

        op = op[
            "result"
        ]

    if not isinstance(op, dict):

        raise RuntimeError(
            f"Malformed operating point: {name}"
        )

    threshold = op.get(
        "threshold"
    )

    if threshold is None:

        raise RuntimeError(
            f"Threshold missing for {name}"
        )

    return {
        "status": "AVAILABLE",
        "threshold": float(
            threshold
        ),
    }


# =================================================================================================
# 0. REPOSITORY / LINEAGE
# =================================================================================================

banner(
    "STAGE28-4 PREFLIGHT — REPOSITORY / LINEAGE GATE"
)


if not (
    REPO
    / ".git"
).is_dir():

    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )


status = git(
    "status",
    "--porcelain",
)


if status:

    raise RuntimeError(
        "Repository must be clean before Stage28-4 preflight:\n"
        + status
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)


print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    local_head,
)

print(
    "origin/main    :",
    origin_head,
)


if not (
    local_head
    == origin_head
    == EXPECTED_PARENT
):

    raise RuntimeError(
        "Stage28-4 preflight parent mismatch."
    )


print()
print(
    "[PASS] Stage28-3C durable parent exact"
)

print(
    "[PASS] repository clean"
)


# =================================================================================================
# 1. STAGE28 CLOSURE + AUTHORIZATION
# =================================================================================================

banner(
    "STAGE28-4 PREFLIGHT — SCIENTIFIC AUTHORIZATION GATE"
)


closure = read_json(
    CLOSURE_RECEIPT
)

stage3c = read_json(
    STAGE3C_RECEIPT
)

stage22_spec = read_json(
    STAGE22_SPEC
)


if (
    closure.get(
        "closure_status"
    )
    !=
    "PASS_STAGE28_NEW_MODEL_FITTING_PERMANENTLY_CLOSED"
):

    raise RuntimeError(
        "Stage28-3A closure is not PASS."
    )


if (
    int(
        closure[
            "fit_budget_closure"
        ][
            "consumed_new_fits"
        ]
    )
    != 108
    or
    int(
        closure[
            "fit_budget_closure"
        ][
            "remaining_new_fits"
        ]
    )
    != 0
):

    raise RuntimeError(
        "Stage28 fit ledger is no longer 108/108."
    )


next_step = stage3c.get(
    "next_authorized_step",
    ""
)


if not next_step.startswith(
    "Stage28-4"
):

    raise RuntimeError(
        "Stage28-3C does not authorize Stage28-4."
    )


if (
    stage22_spec[
        "evaluation_population"
    ][
        "name"
    ]
    !=
    "STAGE22R_SHARED_FINAL_SINGLE_HOLDOUT"
):

    raise RuntimeError(
        "Unexpected Stage22 evaluation population."
    )


if (
    stage22_spec[
        "evaluation_population"
    ][
        "threshold_selection_on_final_holdout"
    ]
    != "FORBIDDEN"
):

    raise RuntimeError(
        "Final-holdout threshold-selection rule changed."
    )


if (
    stage22_spec[
        "evaluation_population"
    ][
        "model_selection_on_final_holdout"
    ]
    != "FORBIDDEN"
):

    raise RuntimeError(
        "Final-holdout model-selection rule changed."
    )


print(
    "[PASS] 108 / 108 new fits closed"
)

print(
    "[PASS] remaining fits = 0"
)

print(
    "[PASS] Stage28-4 explicitly authorized"
)

print(
    "[PASS] shared final holdout population exact"
)

print(
    "[PASS] threshold selection on final holdout FORBIDDEN"
)

print(
    "[PASS] model selection on final holdout FORBIDDEN"
)


# =================================================================================================
# 2. PARENT HOLDOUT CONTRACT
# =================================================================================================

banner(
    "STAGE28-4 PREFLIGHT — FROZEN STAGE22R HOLDOUT CONTRACT"
)


for path in [
    MEMBERSHIP,
    PARENT_SUMMARY,
    PARENT_RESULT,
    PARENT_CHECKSUMS,
]:

    if not path.is_file():

        raise RuntimeError(
            f"Frozen Stage22R artifact missing:\n{path}"
        )


parent_summary = read_json(
    PARENT_SUMMARY
)


summary_text = json.dumps(
    parent_summary
)


# Fail closed against the known frozen census.
for required_value in [
    str(EXPECTED_ROWS),
    str(EXPECTED_BENIGN),
    str(EXPECTED_ATTACK),
]:

    if required_value not in summary_text:

        raise RuntimeError(
            "Parent Stage22R summary no longer contains "
            f"expected frozen value {required_value}."
        )


expected_membership_sha, checksum_entry = (
    resolve_checksum_for_basename(
        PARENT_CHECKSUMS,
        MEMBERSHIP.name,
    )
)


actual_membership_sha = sha256_file(
    MEMBERSHIP
)


print(
    "Membership artifact:"
)

print(
    " ",
    MEMBERSHIP.relative_to(
        REPO
    )
)

print()
print(
    "Expected SHA256:",
    expected_membership_sha,
)

print(
    "Actual SHA256  :",
    actual_membership_sha,
)


if (
    actual_membership_sha
    != expected_membership_sha
):

    raise RuntimeError(
        "Frozen final-holdout membership SHA mismatch."
    )


print()
print(
    "[PASS] parent holdout rows   = 1,374,133"
)

print(
    "[PASS] parent holdout benign = 998,788"
)

print(
    "[PASS] parent holdout attack = 375,345"
)

print(
    "[PASS] membership artifact checksum exact"
)


# =================================================================================================
# 3. MEMBERSHIP NPZ SCHEMA INSPECTION
# =================================================================================================

banner(
    "STAGE28-4 PREFLIGHT — MEMBERSHIP NPZ SCHEMA"
)


membership_schema = {}


with np.load(
    MEMBERSHIP,
    allow_pickle=False,
) as npz:

    keys = list(
        npz.files
    )

    print(
        "NPZ keys:",
        keys,
    )

    print()


    if not keys:

        raise RuntimeError(
            "Membership NPZ has no arrays."
        )


    for key in keys:

        arr = np.asarray(
            npz[
                key
            ]
        )


        info = {

            "shape":
                list(
                    arr.shape
                ),

            "dtype":
                str(
                    arr.dtype
                ),

            "ndim":
                int(
                    arr.ndim
                ),

            "size":
                int(
                    arr.size
                ),
        }


        print(
            f"[{key}]"
        )

        print(
            "  shape:",
            arr.shape,
        )

        print(
            "  dtype:",
            arr.dtype,
        )

        print(
            "  ndim :",
            arr.ndim,
        )

        print(
            "  size :",
            f"{arr.size:,}",
        )


        if arr.size:

            flat = arr.reshape(
                -1
            )


            head = flat[
                :min(
                    10,
                    flat.size,
                )
            ].tolist()

            tail = flat[
                max(
                    0,
                    flat.size - 10,
                ):
            ].tolist()


            info[
                "head"
            ] = head

            info[
                "tail"
            ] = tail


            print(
                "  head :",
                head,
            )

            print(
                "  tail :",
                tail,
            )


            if np.issubdtype(
                arr.dtype,
                np.number,
            ):

                finite = (
                    arr[
                        np.isfinite(
                            arr
                        )
                    ]
                    if np.issubdtype(
                        arr.dtype,
                        np.floating,
                    )
                    else arr
                )


                if finite.size:

                    info[
                        "minimum"
                    ] = (
                        finite.min().item()
                    )

                    info[
                        "maximum"
                    ] = (
                        finite.max().item()
                    )


                    print(
                        "  min  :",
                        info[
                            "minimum"
                        ],
                    )

                    print(
                        "  max  :",
                        info[
                            "maximum"
                        ],
                    )


        membership_schema[
            key
        ] = info

        print()


print(
    "[PASS] membership schema inspected"
)

print(
    "[PASS] no raw final-holdout data row read"
)


# =================================================================================================
# 4. EXACT 70-FEATURE CONTRACT
# =================================================================================================

banner(
    "STAGE28-4 PREFLIGHT — 70-FEATURE CONTRACT"
)


if not FEATURE_CONFIG.is_file():

    raise RuntimeError(
        f"Feature configuration missing:\n{FEATURE_CONFIG}"
    )


feature_obj = read_json(
    FEATURE_CONFIG
)


feature_candidates = (
    recursively_find_feature_lists(
        feature_obj
    )
)


feature_candidates = [
    (
        path,
        values,
    )
    for path, values
    in feature_candidates
    if (
        "Dst Port" in values
        and
        "Idle Min" in values
    )
]


if len(
    feature_candidates
) != 1:

    print(
        "Candidate 70-feature lists:"
    )

    for path, values in (
        feature_candidates
    ):

        print(
            " ",
            path,
            values[
                :3
            ],
            "...",
            values[
                -3:
            ],
        )


    raise RuntimeError(
        "Could not resolve exactly one frozen 70-feature list."
    )


feature_path, FEATURES = (
    feature_candidates[
        0
    ]
)


if len(
    FEATURES
) != EXPECTED_FEATURES:

    raise RuntimeError(
        "Frozen feature count != 70."
    )


if FEATURES[
    0
] != "Dst Port":

    raise RuntimeError(
        "Unexpected first frozen feature."
    )


if FEATURES[
    -1
] != "Idle Min":

    raise RuntimeError(
        "Unexpected final frozen feature."
    )


print(
    "Feature-list source:",
    feature_path,
)

print(
    "Feature count      :",
    len(
        FEATURES
    ),
)

print(
    "First feature      :",
    FEATURES[
        0
    ],
)

print(
    "Last feature       :",
    FEATURES[
        -1
    ],
)


print()
print(
    "[PASS] exact frozen 70-feature configuration resolved"
)


# =================================================================================================
# 5. LOCATE EXACT KAGGLE FINAL-HOLDOUT SOURCES
# =================================================================================================

banner(
    "STAGE28-4 PREFLIGHT — KAGGLE SOURCE DISCOVERY"
)


KAGGLE_INPUT = Path(
    "/kaggle/input"
)


if not KAGGLE_INPUT.is_dir():

    raise RuntimeError(
        "/kaggle/input does not exist."
    )


day1_candidates = sorted(
    KAGGLE_INPUT.rglob(
        "03-01-2018.csv"
    )
)

day2_candidates = sorted(
    KAGGLE_INPUT.rglob(
        "03-02-2018.csv"
    )
)


print(
    "03-01 candidates:"
)

for path in day1_candidates:

    print(
        " ",
        path,
    )


print()
print(
    "03-02 candidates:"
)

for path in day2_candidates:

    print(
        " ",
        path,
    )


pairs = []


for p1 in day1_candidates:

    for p2 in day2_candidates:

        if p1.parent == p2.parent:

            pairs.append(
                (
                    p1,
                    p2,
                )
            )


# Prefer the exact Kaggle slug directory.
exact_pairs = [
    pair
    for pair in pairs
    if pair[
        0
    ].parent.name
    == "ids-intrusion-csv"
]


if len(
    exact_pairs
) == 1:

    source_day1, source_day2 = (
        exact_pairs[
            0
        ]
    )

elif (
    len(
        exact_pairs
    )
    == 0
    and
    len(
        pairs
    )
    == 1
):

    # We still fail closed rather than silently accepting a mirror.
    candidate = pairs[
        0
    ]

    raise RuntimeError(
        "\nThe two March source files exist, but not under the "
        "expected Kaggle dataset directory `ids-intrusion-csv`.\n\n"
        f"Found:\n  {candidate[0]}\n  {candidate[1]}\n\n"
        "Attach the frozen Kaggle dataset:\n"
        "  solarmainframe/ids-intrusion-csv"
    )

else:

    raise RuntimeError(
        "\nCould not resolve exactly one frozen March source pair.\n\n"
        "Attach the exact Kaggle dataset:\n"
        "  solarmainframe/ids-intrusion-csv\n\n"
        "Then rerun this preflight cell."
    )


print()
print(
    "Resolved dataset root:"
)

print(
    " ",
    source_day1.parent,
)

print()
print(
    "03-01 bytes:",
    f"{source_day1.stat().st_size:,}",
)

print(
    "03-02 bytes:",
    f"{source_day2.stat().st_size:,}",
)


# =================================================================================================
# 6. HEADER-ONLY VALIDATION
# =================================================================================================

banner(
    "STAGE28-4 PREFLIGHT — SOURCE HEADER-ONLY VALIDATION"
)


source_headers = {}


for day_name, path in [
    (
        "03-01-2018.csv",
        source_day1,
    ),
    (
        "03-02-2018.csv",
        source_day2,
    ),
]:

    header = [
        normalize_header_name(
            x
        )
        for x in (
            read_csv_header_only(
                path
            )
        )
    ]


    source_headers[
        day_name
    ] = header


    print(
        day_name,
    )

    print(
        "  columns:",
        len(
            header
        ),
    )

    print(
        "  first 5:",
        header[
            :5
        ],
    )

    print(
        "  last 5 :",
        header[
            -5:
        ],
    )


    missing_features = [
        feature
        for feature in FEATURES
        if feature not in header
    ]


    if missing_features:

        raise RuntimeError(
            f"{day_name} is missing frozen features:\n"
            + "\n".join(
                missing_features
            )
        )


    if "Label" not in header:

        raise RuntimeError(
            f"{day_name} has no Label column."
        )


print()
print(
    "[PASS] both March files contain all 70 frozen features"
)

print(
    "[PASS] Label column present in both files"
)

print(
    "[PASS] only CSV header lines were read"
)

print(
    "[PASS] scientific holdout predictor rows read = 0"
)

print(
    "[PASS] scientific holdout labels read = 0"
)


# =================================================================================================
# 7. AUDIT ALL TEN STAGE28 STAGE22 ENSEMBLES
# =================================================================================================

banner(
    "STAGE28-4 PREFLIGHT — TEN STAGE22 ENSEMBLE IDENTITIES"
)


result_files = sorted(
    STAGE22_STAGE28_ROOT.rglob(
        "*_result.json"
    )
)


if len(
    result_files
) != 10:

    raise RuntimeError(
        f"Expected exactly 10 Stage28 Stage22 result receipts; "
        f"found {len(result_files)}."
    )


ensemble_records = []


for result_path in result_files:

    obj = read_json(
        result_path
    )


    if (
        obj.get(
            "experiment"
        )
        != "STAGE22_FULL"
    ):

        raise RuntimeError(
            f"Unexpected experiment in:\n{result_path}"
        )


    unit = obj.get(
        "unit"
    )

    seed = int(
        obj.get(
            "training_seed"
        )
    )


    if unit not in EXPECTED_UNITS:

        raise RuntimeError(
            f"Unexpected Stage22 unit: {unit}"
        )


    if seed not in EXPECTED_SEEDS:

        raise RuntimeError(
            f"Unexpected Stage22 seed: {seed}"
        )


    models = obj.get(
        "models"
    )


    if not isinstance(
        models,
        dict,
    ):

        raise RuntimeError(
            f"Malformed models container:\n{result_path}"
        )


    model_records = {}


    for learner in EXPECTED_LEARNERS:

        if learner not in models:

            raise RuntimeError(
                f"{unit}/seed{seed}: missing {learner}."
            )


        model = models[
            learner
        ]


        if not isinstance(
            model,
            dict,
        ):

            raise RuntimeError(
                f"{unit}/seed{seed}/{learner}: "
                "model receipt is not a dictionary."
            )


        model_path, model_sha, source_type = (
            resolve_model_path(
                result_path,
                model,
            )
        )


        model_seed = int(
            model[
                "seed"
            ]
        )


        if model_seed != seed:

            raise RuntimeError(
                f"{unit}/{learner}: model seed mismatch "
                f"{model_seed} != {seed}."
            )


        backend = str(
            model.get(
                "backend",
                ""
            )
        ).lower()


        if backend != "cpu":

            raise RuntimeError(
                f"{unit}/seed{seed}/{learner}: "
                f"non-CPU backend {backend!r}."
            )


        model_records[
            learner
        ] = {

            "component_id":
                model.get(
                    "component_id"
                ),

            "path":
                str(
                    model_path.relative_to(
                        REPO
                    )
                ),

            "sha256":
                model_sha,

            "source_type":
                source_type,

            "backend":
                backend,
        }


    thresholds = {

        name:
            extract_threshold(
                obj[
                    "operating_points"
                ],
                name,
            )

        for name in [
            "STANDARD",
            "BALANCED",
            "SECURITY",
        ]
    }


    ensemble_records.append(
        {

            "unit":
                unit,

            "seed":
                seed,

            "result_path":
                str(
                    result_path.relative_to(
                        REPO
                    )
                ),

            "xgboost":
                model_records[
                    "xgboost"
                ],

            "lightgbm":
                model_records[
                    "lightgbm"
                ],

            "thresholds":
                thresholds,
        }
    )


ensemble_records.sort(
    key=lambda x:
        (
            EXPECTED_UNITS.index(
                x[
                    "unit"
                ]
            ),
            x[
                "seed"
            ],
        )
)


actual_pairs = [
    (
        x[
            "unit"
        ],
        x[
            "seed"
        ],
    )
    for x in ensemble_records
]


expected_pairs = [
    (
        unit,
        seed,
    )
    for unit in EXPECTED_UNITS
    for seed in EXPECTED_SEEDS
]


if actual_pairs != expected_pairs:

    raise RuntimeError(
        "Ten Stage22 ensemble cells are not exactly "
        "2 units × seeds42-46."
    )


for record in ensemble_records:

    print(
        f"{record['unit']:<24} seed={record['seed']}"
    )

    print(
        "  XGB:",
        record[
            "xgboost"
        ][
            "component_id"
        ],
        record[
            "xgboost"
        ][
            "sha256"
        ],
    )

    print(
        "  LGB:",
        record[
            "lightgbm"
        ][
            "component_id"
        ],
        record[
            "lightgbm"
        ][
            "sha256"
        ],
    )

    print(
        "  thresholds:",
        {
            k:
                v[
                    "threshold"
                ]
            for k, v in (
                record[
                    "thresholds"
                ].items()
            )
        },
    )

    print()


print(
    "[PASS] exactly 10 Stage22 ensembles"
)

print(
    "[PASS] 2 geometries × 5 seeds"
)

print(
    "[PASS] all 20 component model artifact SHA256 values exact"
)

print(
    "[PASS] all Stage28 Stage22 model backends = CPU"
)

print(
    "[PASS] all thresholds read from frozen validation receipts"
)

print(
    "[PASS] ZERO threshold recomputation"
)


# =================================================================================================
# 8. LIBRARY RUNTIME — NO MODEL LOADING / NO PREDICTION
# =================================================================================================

banner(
    "STAGE28-4 PREFLIGHT — MODEL LIBRARY RUNTIME"
)


import xgboost
import lightgbm
import sklearn


print(
    "scikit-learn:",
    sklearn.__version__,
)

print(
    "XGBoost      :",
    xgboost.__version__,
)

print(
    "LightGBM     :",
    lightgbm.__version__,
)


if (
    xgboost.__version__
    != "3.2.0"
):

    raise RuntimeError(
        "XGBoost version mismatch."
    )


if (
    lightgbm.__version__
    != "4.6.0"
):

    raise RuntimeError(
        "LightGBM version mismatch."
    )


print()
print(
    "[PASS] XGBoost 3.2.0 exact"
)

print(
    "[PASS] LightGBM 4.6.0 exact"
)


# =================================================================================================
# 9. LOCAL MACHINE-READABLE PREFLIGHT RECEIPT
# =================================================================================================

banner(
    "STAGE28-4 PREFLIGHT — LOCAL RECEIPT"
)


preflight_receipt = {

    "type":
        "STAGE28_4_OPERATIONAL_PREFLIGHT",

    "scientific_stage":
        "Stage28-4",

    "scientific_parent":
        EXPECTED_PARENT,

    "scientific_operations": {

        "new_model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "final_holdout_predictor_rows_read":
            0,

        "final_holdout_labels_read":
            0,
    },

    "frozen_holdout_contract": {

        "rows":
            EXPECTED_ROWS,

        "benign":
            EXPECTED_BENIGN,

        "attack":
            EXPECTED_ATTACK,

        "features":
            EXPECTED_FEATURES,

        "dtype":
            "float64",

        "membership_path":
            str(
                MEMBERSHIP.relative_to(
                    REPO
                )
            ),

        "membership_sha256":
            actual_membership_sha,

        "membership_schema":
            membership_schema,
    },

    "source_binding": {

        "provider":
            "KAGGLE",

        "dataset":
            "solarmainframe/ids-intrusion-csv",

        "dataset_root":
            str(
                source_day1.parent
            ),

        "03_01_path":
            str(
                source_day1
            ),

        "03_01_bytes":
            source_day1.stat().st_size,

        "03_02_path":
            str(
                source_day2
            ),

        "03_02_bytes":
            source_day2.stat().st_size,

        "data_rows_read":
            0,

        "header_only":
            True,
    },

    "stage22_ensembles": ensemble_records,

    "status":
        "READY_FOR_EXACT_STAGE28_4_INFERENCE_CELL_AFTER_MEMBERSHIP_SCHEMA_REVIEW",
}


LOCAL_PREFLIGHT_RECEIPT.write_text(
    json.dumps(
        preflight_receipt,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)


print(
    "Local receipt:"
)

print(
    " ",
    LOCAL_PREFLIGHT_RECEIPT,
)

print()
print(
    "[PASS] receipt written outside Git repository"
)

print(
    "[PASS] repository remains scientifically unchanged"
)


# =================================================================================================
# 10. FINAL PREFLIGHT STATUS
# =================================================================================================

if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository changed during operational preflight."
    )


banner(
    "STAGE28-4 PREFLIGHT COMPLETE"
)


print(
    "Scientific parent:"
)

print(
    " ",
    EXPECTED_PARENT,
)

print()
print(
    "Stage28 new fits                 : 108 / 108 CLOSED"
)

print(
    "New fits remaining               : 0"
)

print(
    "Stage28-4 ensemble cells         : 10 / 10 VERIFIED"
)

print(
    "Stage28-4 component models       : 20 / 20 SHA-VERIFIED"
)

print(
    "Shared final holdout expected    : 1,374,133 rows"
)

print(
    "Frozen feature count             : 70"
)

print()
print(
    "New model fits                   : 0"
)

print(
    "Model inference                  : 0"
)

print(
    "Threshold selection              : 0"
)

print(
    "Final-holdout predictor rows read: 0"
)

print(
    "Final-holdout labels read        : 0"
)

print()
print(
    "NO SHARED FINAL-HOLDOUT SCIENTIFIC OPENING HAS OCCURRED."
)

print()
print(
    "NEXT:"
)

print(
    "Use the reported membership NPZ keys/schema to execute the "
    "single authorized Stage28-4 holdout materialization + inference."
)


STAGE28-4 PREFLIGHT — REPOSITORY / LINEAGE GATE

Expected parent: 2679d0c208d514b381caa12e96c959f4f2ee5ee7
Local HEAD     : 2679d0c208d514b381caa12e96c959f4f2ee5ee7
origin/main    : 2679d0c208d514b381caa12e96c959f4f2ee5ee7

[PASS] Stage28-3C durable parent exact
[PASS] repository clean

STAGE28-4 PREFLIGHT — SCIENTIFIC AUTHORIZATION GATE

[PASS] 108 / 108 new fits closed
[PASS] remaining fits = 0
[PASS] Stage28-4 explicitly authorized
[PASS] shared final holdout population exact
[PASS] threshold selection on final holdout FORBIDDEN
[PASS] model selection on final holdout FORBIDDEN

STAGE28-4 PREFLIGHT — FROZEN STAGE22R HOLDOUT CONTRACT

Membership artifact:
  results/stage22r_training/stage22r_final_single_holdout/stage22r_final_holdout_clean_membership.npz

Expected SHA256: 18d43eded5e78238ce6765abdc1ed18ce662aebd0899b678472891203eee3d1e
Actual SHA256  : 18d43eded5e78238ce6765abdc1ed18ce662aebd0899b678472891203eee3d1e

[PASS] parent holdout rows   = 1,374,133
[PASS] parent holdout be

In [12]:
# =================================================================================================
# STAGE28-4 — ONE-TIME STAGE22 SHARED-FINAL-HOLDOUT INFERENCE
#
# AUTHORIZED SCIENTIFIC OPERATION
#
# NEW MODEL FITS                   : 0
# THRESHOLD SELECTION              : 0
# MODEL SELECTION                  : 0
# NEW FORMAL STATISTICAL TESTS     : 0
# SHAP / SUBSET SEARCH             : 0
#
# AUTHORIZED:
#   - materialize exact frozen Stage22R shared final holdout
#   - infer 20 already-frozen component models
#   - construct 10 frozen equal-weight ensembles
#   - evaluate seeds 42..46
#   - apply ONLY thresholds frozen on development validation
#   - evaluate preregistered Stage22 directional conclusion stability
#
# FINAL HOLDOUT:
#   1,374,133 rows
#   998,788 benign
#   375,345 attack
#   70 features
#   float64 model input
#
# IMPORTANT:
#   The Stage22R parent holdout is already historically known.
#   Stage28-4 is preregistered robustness re-evaluation, not a new blind test.
# =================================================================================================

from __future__ import annotations

import csv
import gc
import hashlib
import json
import math
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

import xgboost as xgb
import lightgbm as lgb


# =================================================================================================
# CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "2679d0c208d514b381caa12e96c959f4f2ee5ee7"
)

ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

STAGE22_ROOT = (
    ROOT
    / "stage28_2a_stage22_seed_stability"
)

PROTOCOL_ROOT = (
    ROOT
    / "stage28_0_protocol_lock"
)

STAGE3_ROOT = (
    ROOT
    / "stage28_3_seed_uncertainty"
)

STAGE4_ROOT = (
    ROOT
    / "stage28_4_stage22_shared_final_holdout"
)

STAGE22R_FINAL = (
    REPO
    / "results"
    / "stage22r_training"
    / "stage22r_final_single_holdout"
)

MEMBERSHIP_PATH = (
    STAGE22R_FINAL
    / "stage22r_final_holdout_clean_membership.npz"
)

PARENT_HOLDOUT_SUMMARY = (
    STAGE22R_FINAL
    / "stage22r_final_holdout_k79_summary.json"
)

PARENT_HOLDOUT_RESULT = (
    STAGE22R_FINAL
    / "stage22r_final_holdout_result.json"
)

PARENT_CHECKSUMS = (
    STAGE22R_FINAL
    / "checksums.sha256"
)

FEATURE_CONFIG = (
    REPO
    / "results"
    / "stage15_transformer_checkpoint"
    / "stage15_1_feature_configuration.json"
)

STAGE22_SPEC = (
    PROTOCOL_ROOT
    / "stage22_cell_spec.json"
)

STABILITY_SPEC = (
    PROTOCOL_ROOT
    / "conclusion_stability_spec.json"
)

CLOSURE_RECEIPT = (
    ROOT
    / "stage28_3a_experiment_closure_audit"
    / "stage28_3a_experiment_closure_receipt.json"
)

STAGE3C_RECEIPT = (
    STAGE3_ROOT
    / "stage28_3c_receipt.json"
)

LOAO_CONCLUSION_STABILITY = (
    STAGE3_ROOT
    / "stage28_3b_loao_conclusion_stability.csv"
)

COMBINED_CONCLUSION_STABILITY = (
    STAGE3_ROOT
    / "conclusion_stability.csv"
)

SOURCE_ROOT = Path(
    "/kaggle/input/datasets/solarmainframe/ids-intrusion-csv"
)

SOURCE_DAY8 = (
    SOURCE_ROOT
    / "03-01-2018.csv"
)

SOURCE_DAY9 = (
    SOURCE_ROOT
    / "03-02-2018.csv"
)

EXPECTED_MEMBERSHIP_SHA = (
    "18d43eded5e78238ce6765abdc1ed18ce662aebd0899b678472891203eee3d1e"
)

EXPECTED_ROWS = 1_374_133
EXPECTED_BENIGN = 998_788
EXPECTED_ATTACK = 375_345
EXPECTED_FEATURES = 70

EXPECTED_POS_INF_TO_NAN = 9_530
EXPECTED_NEG_INF_TO_NAN = 0
EXPECTED_OUTPUT_NAN = 13_922

EXPECTED_SEEDS = [42, 43, 44, 45, 46]

EXPECTED_UNITS = [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]

EXPECTED_LIBRARIES = {
    "numpy": "2.0.2",
    "sklearn": "1.6.1",
    "xgboost": "3.2.0",
    "lightgbm": "4.6.0",
}

EXPECTED_PER_DAY = {
    8: {
        "file": "03-01-2018.csv",
        "physical_rows": 331_125,
        "embedded_header_rows": 25,
        "effective_rows": 331_100,
        "retained_rows": 331_017,
        "retained_benign": 237_982,
        "retained_attack": 93_035,
    },
    9: {
        "file": "03-02-2018.csv",
        "physical_rows": 1_048_575,
        "embedded_header_rows": 0,
        "effective_rows": 1_048_575,
        "retained_rows": 1_043_116,
        "retained_benign": 760_806,
        "retained_attack": 282_310,
    },
}


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(text: str) -> None:
    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
    env=None,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
        env=env,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}): "
            + " ".join(str(x) for x in cmd)
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args, check=True):
    return run(
        ["git", *args],
        check=check,
    ).stdout.strip()


def read_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def write_json(path: Path, obj) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(
    path: Path,
    chunk_size=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with path.open("rb") as f:
        while True:
            block = f.read(chunk_size)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array_raw(arr: np.ndarray) -> str:
    a = np.ascontiguousarray(arr)

    h = hashlib.sha256()

    view = memoryview(a).cast("B")

    step = 64 * 1024 * 1024

    for start in range(
        0,
        len(view),
        step,
    ):
        h.update(
            view[
                start:
                start + step
            ]
        )

    return h.hexdigest()


def normalize_name(value) -> str:
    return (
        str(value)
        .replace("\ufeff", "")
        .strip()
    )


def normalized_labels(series: pd.Series) -> pd.Series:
    return (
        series
        .astype("string")
        .fillna("")
        .str.strip()
        .str.upper()
    )


def labels_to_binary(series: pd.Series) -> np.ndarray:
    labels = normalized_labels(
        series
    )

    return np.where(
        labels.eq("BENIGN"),
        0,
        1,
    ).astype(
        np.uint8,
        copy=False,
    )


def safe_div(num, den):
    if den == 0:
        return 0.0

    return float(num / den)


def operating_metrics(
    y_true: np.ndarray,
    probability_float32: np.ndarray,
    threshold: float,
):
    threshold32 = np.float32(
        threshold
    )

    pred = (
        probability_float32
        >= threshold32
    )

    y_pos = (
        y_true == 1
    )

    y_neg = ~y_pos

    tp = int(
        np.count_nonzero(
            pred & y_pos
        )
    )

    fp = int(
        np.count_nonzero(
            pred & y_neg
        )
    )

    tn = int(
        np.count_nonzero(
            (~pred) & y_neg
        )
    )

    fn = int(
        np.count_nonzero(
            (~pred) & y_pos
        )
    )

    precision = safe_div(
        tp,
        tp + fp,
    )

    recall = safe_div(
        tp,
        tp + fn,
    )

    fpr = safe_div(
        fp,
        fp + tn,
    )

    accuracy = safe_div(
        tp + tn,
        len(y_true),
    )

    f1 = (
        safe_div(
            2.0 * precision * recall,
            precision + recall,
        )
        if (
            precision + recall
        ) > 0
        else 0.0
    )

    f2 = (
        safe_div(
            5.0 * precision * recall,
            4.0 * precision + recall,
        )
        if (
            4.0 * precision + recall
        ) > 0
        else 0.0
    )

    return {
        "threshold":
            float(threshold),

        "threshold_float32_runtime":
            float(threshold32),

        "accuracy":
            accuracy,

        "precision":
            precision,

        "recall":
            recall,

        "fpr":
            fpr,

        "f1":
            f1,

        "f2":
            f2,

        "tp":
            tp,

        "fp":
            fp,

        "tn":
            tn,

        "fn":
            fn,
    }


def extract_threshold(
    operating_points,
    name,
):
    op = operating_points[
        name.lower()
    ]

    if (
        isinstance(op, dict)
        and
        "result" in op
    ):
        if op.get(
            "status"
        ) != "AVAILABLE":
            raise RuntimeError(
                f"Frozen {name} threshold "
                f"is unavailable."
            )

        op = op[
            "result"
        ]

    if not isinstance(
        op,
        dict,
    ):
        raise RuntimeError(
            f"Malformed operating point {name}."
        )

    threshold = op.get(
        "threshold"
    )

    if threshold is None:
        raise RuntimeError(
            f"Missing threshold for {name}."
        )

    return float(
        threshold
    )


def resolve_model_path(
    result_path: Path,
    model_info: dict,
):
    if model_info.get(
        "model_path"
    ):
        path = (
            result_path.parent
            / model_info[
                "model_path"
            ]
        )

        expected_sha = (
            model_info[
                "model_sha256"
            ]
        )

        source_type = (
            "STAGE28_MODEL_ARTIFACT"
        )

    else:
        path = (
            REPO
            / model_info[
                "historical_model_path"
            ]
        )

        expected_sha = (
            model_info[
                "historical_model_sha256"
            ]
        )

        source_type = (
            "HISTORICAL_REUSE_ARTIFACT"
        )

    if not path.is_file():
        raise RuntimeError(
            f"Model missing:\n{path}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            "Model SHA mismatch:\n"
            f"{path}\n"
            f"expected={expected_sha}\n"
            f"actual={actual_sha}"
        )

    return (
        path,
        actual_sha,
        source_type,
    )


def git_push_with_kaggle_secret():
    try:
        from kaggle_secrets import (
            UserSecretsClient,
        )
    except Exception as exc:
        raise RuntimeError(
            "kaggle_secrets unavailable."
        ) from exc

    client = UserSecretsClient()

    token = None
    token_name = None

    candidates = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "github_pat",
        "GITHUB_PAT",
        "GH_PAT",
    ]

    for name in candidates:
        try:
            value = client.get_secret(
                name
            )
        except Exception:
            value = None

        if (
            value
            and
            str(value).strip()
        ):
            token = str(
                value
            ).strip()

            token_name = name

            break

    if token is None:
        raise RuntimeError(
            "No usable GitHub token found in Kaggle Secrets."
        )

    encoded = quote(
        token,
        safe="",
    )

    push_url = (
        "https://x-access-token:"
        + encoded
        + "@github.com/"
        + "themubasshir/"
        + "ids2018-validation-safe-ablation.git"
    )

    p = subprocess.run(
        [
            "git",
            "push",
            push_url,
            "HEAD:main",
        ],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        safe_stdout = (
            p.stdout
            .replace(
                token,
                "***",
            )
            .replace(
                encoded,
                "***",
            )
        )

        safe_stderr = (
            p.stderr
            .replace(
                token,
                "***",
            )
            .replace(
                encoded,
                "***",
            )
        )

        raise RuntimeError(
            "GitHub push failed.\n\n"
            f"STDOUT:\n{safe_stdout}\n"
            f"STDERR:\n{safe_stderr}"
        )

    print(
        f"[PASS] GitHub credential: kaggle_secret:{token_name}"
    )

    print(
        "[PASS] token not displayed"
    )

    if p.stdout.strip():
        print(
            p.stdout.strip()
        )

    if p.stderr.strip():
        print(
            p.stderr
        )


# =================================================================================================
# 0. REPOSITORY / PARENT GATE
# =================================================================================================

banner(
    "STAGE28-4 — REPOSITORY / PARENT GATE"
)

if not (
    REPO
    / ".git"
).is_dir():
    raise RuntimeError(
        f"Repository missing:\n{REPO}"
    )

status = git(
    "status",
    "--porcelain",
)

if status:
    raise RuntimeError(
        "Repository is not clean:\n"
        + status
    )

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)

local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)

print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    local_head,
)

print(
    "origin/main    :",
    origin_head,
)

if not (
    local_head
    == origin_head
    == EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-4 parent gate failed."
    )

if STAGE4_ROOT.exists():
    raise RuntimeError(
        "Stage28-4 output directory already exists. "
        "Do not overwrite or rerun blindly."
    )

if COMBINED_CONCLUSION_STABILITY.exists():
    raise RuntimeError(
        "Combined conclusion_stability.csv already exists. "
        "Do not overwrite."
    )

print()
print(
    "[PASS] Stage28-3C parent exact"
)

print(
    "[PASS] repository clean"
)

print(
    "[PASS] Stage28-4 has not been durably executed"
)


# =================================================================================================
# 1. SCIENTIFIC CLOSURE / AUTHORIZATION GATE
# =================================================================================================

banner(
    "STAGE28-4 — SCIENTIFIC AUTHORIZATION GATE"
)

closure = read_json(
    CLOSURE_RECEIPT
)

stage3c = read_json(
    STAGE3C_RECEIPT
)

stage22_spec = read_json(
    STAGE22_SPEC
)

stability_spec = read_json(
    STABILITY_SPEC
)

if (
    closure.get(
        "closure_status"
    )
    !=
    "PASS_STAGE28_NEW_MODEL_FITTING_PERMANENTLY_CLOSED"
):
    raise RuntimeError(
        "Stage28 fitting closure not exact."
    )

fit_budget = closure[
    "fit_budget_closure"
]

if (
    int(
        fit_budget[
            "consumed_new_fits"
        ]
    )
    != 108
):
    raise RuntimeError(
        "Consumed fit count != 108."
    )

if (
    int(
        fit_budget[
            "remaining_new_fits"
        ]
    )
    != 0
):
    raise RuntimeError(
        "Remaining fit count != 0."
    )

if not str(
    stage3c.get(
        "next_authorized_step",
        "",
    )
).startswith(
    "Stage28-4"
):
    raise RuntimeError(
        "Stage28-3C does not authorize Stage28-4."
    )

evaluation_population = (
    stage22_spec[
        "evaluation_population"
    ]
)

if (
    evaluation_population[
        "name"
    ]
    !=
    "STAGE22R_SHARED_FINAL_SINGLE_HOLDOUT"
):
    raise RuntimeError(
        "Wrong Stage22 evaluation population."
    )

if (
    evaluation_population[
        "threshold_selection_on_final_holdout"
    ]
    != "FORBIDDEN"
):
    raise RuntimeError(
        "Threshold-search prohibition changed."
    )

if (
    evaluation_population[
        "model_selection_on_final_holdout"
    ]
    != "FORBIDDEN"
):
    raise RuntimeError(
        "Model-selection prohibition changed."
    )

if (
    stage22_spec[
        "scientific_unit"
    ][
        "strategy"
    ]
    != "ENS_LGBM_XGB_EQUAL"
):
    raise RuntimeError(
        "Stage22 ensemble strategy changed."
    )

if (
    stage22_spec[
        "scientific_unit"
    ][
        "probability_rule"
    ]
    !=
    "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST"
):
    raise RuntimeError(
        "Stage22 ensemble probability rule changed."
    )

frozen_claims = (
    stability_spec[
        "stage22_directional_claims"
    ]
)

expected_claim_ids = {
    "STAGE22_PR_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT",
    "STAGE22_ROC_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT",
}

actual_claim_ids = {
    x[
        "claim_id"
    ]
    for x in frozen_claims
}

if (
    actual_claim_ids
    != expected_claim_ids
):
    raise RuntimeError(
        "Frozen Stage22 directional claims changed."
    )

print(
    "[PASS] Stage28 fitting permanently closed at 108 / 108"
)

print(
    "[PASS] Stage28-4 explicitly authorized"
)

print(
    "[PASS] final-holdout threshold search FORBIDDEN"
)

print(
    "[PASS] final-holdout model selection FORBIDDEN"
)

print(
    "[PASS] equal-weight LGBM/XGB ensemble exact"
)

print(
    "[PASS] two Stage22 directional claims frozen"
)


# =================================================================================================
# 2. RUNTIME / ARTIFACT GATE
# =================================================================================================

banner(
    "STAGE28-4 — RUNTIME / ARTIFACT GATE"
)

versions = {
    "numpy": np.__version__,
    "sklearn": sklearn.__version__,
    "xgboost": xgb.__version__,
    "lightgbm": lgb.__version__,
}

for name, expected in (
    EXPECTED_LIBRARIES.items()
):
    actual = versions[
        name
    ]

    print(
        f"{name:<10}: {actual}"
    )

    if actual != expected:
        raise RuntimeError(
            f"{name} version mismatch: "
            f"{actual} != {expected}"
        )

required_paths = [
    MEMBERSHIP_PATH,
    PARENT_HOLDOUT_SUMMARY,
    PARENT_HOLDOUT_RESULT,
    PARENT_CHECKSUMS,
    FEATURE_CONFIG,
    LOAO_CONCLUSION_STABILITY,
    SOURCE_DAY8,
    SOURCE_DAY9,
]

for path in required_paths:
    if not path.is_file():
        raise RuntimeError(
            f"Required artifact missing:\n{path}"
        )

membership_sha = sha256_file(
    MEMBERSHIP_PATH
)

if (
    membership_sha
    != EXPECTED_MEMBERSHIP_SHA
):
    raise RuntimeError(
        "Frozen membership artifact SHA mismatch."
    )

print()
print(
    "[PASS] runtime versions exact"
)

print(
    "[PASS] frozen membership SHA exact"
)

print(
    "[PASS] exact March source files attached"
)


# =================================================================================================
# 3. LOAD FROZEN MEMBERSHIP
# =================================================================================================

banner(
    "STAGE28-4 — LOAD FROZEN HOLDOUT MEMBERSHIP"
)

with np.load(
    MEMBERSHIP_PATH,
    allow_pickle=False,
) as z:
    expected_keys = {
        "hash_lo",
        "hash_hi",
        "day_id",
        "row_index",
        "binary_label",
    }

    if set(
        z.files
    ) != expected_keys:
        raise RuntimeError(
            f"Unexpected membership keys: {z.files}"
        )

    hash_lo = np.asarray(
        z[
            "hash_lo"
        ],
        dtype=np.uint64,
    ).copy()

    hash_hi = np.asarray(
        z[
            "hash_hi"
        ],
        dtype=np.uint64,
    ).copy()

    day_id = np.asarray(
        z[
            "day_id"
        ],
        dtype=np.uint8,
    ).copy()

    row_index = np.asarray(
        z[
            "row_index"
        ],
        dtype=np.uint32,
    ).astype(
        np.int64,
        copy=False,
    )

    y_true = np.asarray(
        z[
            "binary_label"
        ],
        dtype=np.uint8,
    ).copy()

if not (
    len(hash_lo)
    == len(hash_hi)
    == len(day_id)
    == len(row_index)
    == len(y_true)
    == EXPECTED_ROWS
):
    raise RuntimeError(
        "Membership array length mismatch."
    )

if set(
    np.unique(
        day_id
    ).tolist()
) != {8, 9}:
    raise RuntimeError(
        "Membership contains unexpected day IDs."
    )

benign_count = int(
    np.count_nonzero(
        y_true == 0
    )
)

attack_count = int(
    np.count_nonzero(
        y_true == 1
    )
)

if benign_count != EXPECTED_BENIGN:
    raise RuntimeError(
        f"Benign count mismatch: {benign_count}"
    )

if attack_count != EXPECTED_ATTACK:
    raise RuntimeError(
        f"Attack count mismatch: {attack_count}"
    )

for d in [8, 9]:
    positions = np.flatnonzero(
        day_id == d
    )

    expected = (
        EXPECTED_PER_DAY[
            d
        ]
    )

    if (
        len(positions)
        !=
        expected[
            "retained_rows"
        ]
    ):
        raise RuntimeError(
            f"Day {d} membership count mismatch."
        )

    indices = row_index[
        positions
    ]

    if (
        np.unique(
            indices
        ).size
        != indices.size
    ):
        raise RuntimeError(
            f"Duplicate row_index values for day {d}."
        )

    print(
        f"day_id={d}: "
        f"rows={len(positions):,}, "
        f"row_index_min={indices.min():,}, "
        f"row_index_max={indices.max():,}"
    )

print()
print(
    "[PASS] membership rows = 1,374,133"
)

print(
    "[PASS] benign = 998,788"
)

print(
    "[PASS] attack = 375,345"
)

print(
    "[PASS] day IDs = {8, 9}"
)

membership_logical = {
    "hash_lo_sha256":
        sha256_array_raw(
            hash_lo
        ),

    "hash_hi_sha256":
        sha256_array_raw(
            hash_hi
        ),

    "day_id_sha256":
        sha256_array_raw(
            day_id
        ),

    "row_index_sha256":
        sha256_array_raw(
            row_index.astype(
                np.uint32
            )
        ),

    "binary_label_sha256":
        sha256_array_raw(
            y_true
        ),
}


# =================================================================================================
# 4. LOAD EXACT 70-FEATURE CONTRACT
# =================================================================================================

banner(
    "STAGE28-4 — 70-FEATURE CONTRACT"
)

feature_obj = read_json(
    FEATURE_CONFIG
)

FEATURES = feature_obj.get(
    "retained_features"
)

if not isinstance(
    FEATURES,
    list,
):
    raise RuntimeError(
        "retained_features missing."
    )

if len(
    FEATURES
) != EXPECTED_FEATURES:
    raise RuntimeError(
        "Feature count != 70."
    )

if not all(
    isinstance(
        x,
        str,
    )
    for x in FEATURES
):
    raise RuntimeError(
        "Feature list contains non-string values."
    )

if FEATURES[
    0
] != "Dst Port":
    raise RuntimeError(
        "First feature mismatch."
    )

if FEATURES[
    -1
] != "Idle Min":
    raise RuntimeError(
        "Last feature mismatch."
    )

print(
    "Feature count:",
    len(
        FEATURES
    ),
)

print(
    "First feature:",
    FEATURES[
        0
    ],
)

print(
    "Last feature :",
    FEATURES[
        -1
    ],
)

print()
print(
    "[PASS] frozen Stage22 70-feature order exact"
)


# =================================================================================================
# 5. SOURCE PROVENANCE
# =================================================================================================

banner(
    "STAGE28-4 — SOURCE PROVENANCE"
)

source_records = {}

for day, path in [
    (
        8,
        SOURCE_DAY8,
    ),
    (
        9,
        SOURCE_DAY9,
    ),
]:
    print(
        f"Hashing {path.name} ..."
    )

    source_records[
        day
    ] = {
        "path":
            str(
                path
            ),

        "bytes":
            int(
                path.stat().st_size
            ),

        "sha256":
            sha256_file(
                path
            ),
    }

    print(
        "  bytes :",
        f"{path.stat().st_size:,}",
    )

    print(
        "  sha256:",
        source_records[
            day
        ][
            "sha256"
        ],
    )

print()
print(
    "[PASS] source-byte identities recorded"
)


# =================================================================================================
# 6. MATERIALIZE EXACT FROZEN HOLDOUT
#
# Critical point:
#
# row_index semantics are NOT guessed.
#
# For March 1, where 25 embedded headers exist, we construct:
#   A) raw physical dataframe indexing
#   B) post-embedded-header effective indexing
#
# We compare each candidate against the already-frozen membership
# binary_label vector and require a unique exact match.
#
# March 2 has no embedded headers, so raw == effective.
#
# This is an operational index-semantics audit, NOT a new membership
# decision and NOT a model-result-driven choice.
# =================================================================================================

banner(
    "STAGE28-4 — ONE-TIME SHARED FINAL-HOLDOUT MATERIALIZATION"
)

USECOLS = (
    FEATURES
    + [
        "Label"
    ]
)

wanted = set(
    USECOLS
)

X_holdout = np.empty(
    (
        EXPECTED_ROWS,
        EXPECTED_FEATURES,
    ),
    dtype=np.float64,
)

materialization_records = {}

total_pos_inf = 0
total_neg_inf = 0

materialization_started = (
    time.perf_counter()
)

for day, path in [
    (
        8,
        SOURCE_DAY8,
    ),
    (
        9,
        SOURCE_DAY9,
    ),
]:
    expected = (
        EXPECTED_PER_DAY[
            day
        ]
    )

    banner(
        f"STAGE28-4 — MATERIALIZE DAY {day}: {path.name}"
    )

    positions = np.flatnonzero(
        day_id == day
    )

    membership_indices = (
        row_index[
            positions
        ]
    )

    expected_labels = (
        y_true[
            positions
        ]
    )

    print(
        "Reading raw source once ..."
    )

    read_started = (
        time.perf_counter()
    )

    df = pd.read_csv(
        path,
        usecols=lambda c:
            normalize_name(
                c
            )
            in wanted,
        low_memory=False,
    )

    df.columns = [
        normalize_name(
            c
        )
        for c in df.columns
    ]

    if set(
        df.columns
    ) != wanted:
        missing = sorted(
            wanted
            - set(
                df.columns
            )
        )

        extra = sorted(
            set(
                df.columns
            )
            - wanted
        )

        raise RuntimeError(
            f"{path.name}: use-column mismatch.\n"
            f"missing={missing}\n"
            f"extra={extra}"
        )

    read_seconds = (
        time.perf_counter()
        - read_started
    )

    physical_rows = len(
        df
    )

    if (
        physical_rows
        != expected[
            "physical_rows"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: physical row mismatch "
            f"{physical_rows} != "
            f"{expected['physical_rows']}"
        )

    label_normalized = (
        normalized_labels(
            df[
                "Label"
            ]
        )
    )

    embedded_header_mask = (
        label_normalized.eq(
            "LABEL"
        )
    )

    embedded_count = int(
        embedded_header_mask.sum()
    )

    if (
        embedded_count
        != expected[
            "embedded_header_rows"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: embedded-header count mismatch "
            f"{embedded_count} != "
            f"{expected['embedded_header_rows']}"
        )

    effective_df = (
        df.loc[
            ~embedded_header_mask
        ]
        .reset_index(
            drop=True
        )
    )

    effective_rows = len(
        effective_df
    )

    if (
        effective_rows
        != expected[
            "effective_rows"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: effective row mismatch "
            f"{effective_rows} != "
            f"{expected['effective_rows']}"
        )

    if (
        len(
            positions
        )
        != expected[
            "retained_rows"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: retained membership count mismatch."
        )

    candidate_matches = {}

    # Candidate 1: membership row_index addresses physical rows.
    if (
        membership_indices.max()
        < len(
            df
        )
    ):
        raw_candidate_labels = (
            labels_to_binary(
                df.iloc[
                    membership_indices
                ][
                    "Label"
                ]
            )
        )

        candidate_matches[
            "RAW_PHYSICAL_ROW_INDEX"
        ] = bool(
            np.array_equal(
                raw_candidate_labels,
                expected_labels,
            )
        )

    else:
        candidate_matches[
            "RAW_PHYSICAL_ROW_INDEX"
        ] = False

    # Candidate 2: membership row_index addresses the post-header
    # effective row stream.
    if (
        membership_indices.max()
        < len(
            effective_df
        )
    ):
        effective_candidate_labels = (
            labels_to_binary(
                effective_df.iloc[
                    membership_indices
                ][
                    "Label"
                ]
            )
        )

        candidate_matches[
            "POST_EMBEDDED_HEADER_EFFECTIVE_INDEX"
        ] = bool(
            np.array_equal(
                effective_candidate_labels,
                expected_labels,
            )
        )

    else:
        candidate_matches[
            "POST_EMBEDDED_HEADER_EFFECTIVE_INDEX"
        ] = False

    print(
        "Candidate row_index semantics:"
    )

    for name, passed in (
        candidate_matches.items()
    ):
        print(
            f"  {name:<42}: {passed}"
        )

    if embedded_count == 0:
        # Raw and effective are mathematically identical here.
        if not candidate_matches[
            "RAW_PHYSICAL_ROW_INDEX"
        ]:
            raise RuntimeError(
                f"{path.name}: frozen row indices do not "
                "reproduce membership labels."
            )

        selected_base = df

        index_semantics = (
            "RAW_EQUALS_EFFECTIVE_NO_EMBEDDED_HEADERS"
        )

    else:
        passing = [
            name
            for name, passed
            in candidate_matches.items()
            if passed
        ]

        if len(
            passing
        ) != 1:
            raise RuntimeError(
                f"{path.name}: expected exactly one "
                "membership index interpretation to match; "
                f"got {passing}."
            )

        index_semantics = (
            passing[
                0
            ]
        )

        if (
            index_semantics
            ==
            "RAW_PHYSICAL_ROW_INDEX"
        ):
            selected_base = df

        elif (
            index_semantics
            ==
            "POST_EMBEDDED_HEADER_EFFECTIVE_INDEX"
        ):
            selected_base = effective_df

        else:
            raise RuntimeError(
                "Internal row-index resolution error."
            )

    selected = (
        selected_base.iloc[
            membership_indices
        ]
    )

    source_selected_y = (
        labels_to_binary(
            selected[
                "Label"
            ]
        )
    )

    if not np.array_equal(
        source_selected_y,
        expected_labels,
    ):
        raise RuntimeError(
            f"{path.name}: selected source labels do not "
            "exactly reproduce frozen membership labels."
        )

    source_benign = int(
        np.count_nonzero(
            source_selected_y == 0
        )
    )

    source_attack = int(
        np.count_nonzero(
            source_selected_y == 1
        )
    )

    if (
        source_benign
        != expected[
            "retained_benign"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: retained benign mismatch."
        )

    if (
        source_attack
        != expected[
            "retained_attack"
        ]
    ):
        raise RuntimeError(
            f"{path.name}: retained attack mismatch."
        )

    print(
        f"[PASS] row_index semantics: {index_semantics}"
    )

    print(
        "[PASS] source labels reproduce frozen binary_label exactly"
    )

    print(
        f"[PASS] retained rows: {len(selected):,}"
    )

    numeric_started = (
        time.perf_counter()
    )

    numeric_df = (
        selected[
            FEATURES
        ]
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
    )

    block = numeric_df.to_numpy(
        dtype=np.float64,
        copy=True,
    )

    day_pos_inf = int(
        np.count_nonzero(
            np.isposinf(
                block
            )
        )
    )

    day_neg_inf = int(
        np.count_nonzero(
            np.isneginf(
                block
            )
        )
    )

    total_pos_inf += (
        day_pos_inf
    )

    total_neg_inf += (
        day_neg_inf
    )

    inf_mask = np.isinf(
        block
    )

    if np.any(
        inf_mask
    ):
        block[
            inf_mask
        ] = np.nan

    if (
        block.shape
        !=
        (
            len(
                positions
            ),
            EXPECTED_FEATURES,
        )
    ):
        raise RuntimeError(
            f"{path.name}: materialized matrix shape mismatch."
        )

    X_holdout[
        positions,
        :,
    ] = block

    numeric_seconds = (
        time.perf_counter()
        - numeric_started
    )

    materialization_records[
        day
    ] = {
        "file":
            path.name,

        "physical_rows":
            physical_rows,

        "embedded_header_rows":
            embedded_count,

        "effective_rows":
            effective_rows,

        "retained_rows":
            len(
                positions
            ),

        "retained_benign":
            source_benign,

        "retained_attack":
            source_attack,

        "membership_row_index_min":
            int(
                membership_indices.min()
            ),

        "membership_row_index_max":
            int(
                membership_indices.max()
            ),

        "index_semantics":
            index_semantics,

        "candidate_matches":
            candidate_matches,

        "positive_infinity_to_nan":
            day_pos_inf,

        "negative_infinity_to_nan":
            day_neg_inf,

        "raw_source_read_seconds":
            read_seconds,

        "numeric_materialization_seconds":
            numeric_seconds,
    }

    print(
        f"positive infinity cells: {day_pos_inf:,}"
    )

    print(
        f"negative infinity cells: {day_neg_inf:,}"
    )

    # Free all temporary per-day structures immediately.
    del selected
    del selected_base
    del effective_df
    del numeric_df
    del block
    del df
    del label_normalized
    del embedded_header_mask
    gc.collect()

materialization_seconds = (
    time.perf_counter()
    - materialization_started
)

if X_holdout.dtype != np.float64:
    raise RuntimeError(
        "Final holdout dtype != float64."
    )

if X_holdout.shape != (
    EXPECTED_ROWS,
    EXPECTED_FEATURES,
):
    raise RuntimeError(
        "Final holdout shape mismatch."
    )

if (
    total_pos_inf
    != EXPECTED_POS_INF_TO_NAN
):
    raise RuntimeError(
        "Positive-infinity conversion mismatch: "
        f"{total_pos_inf} != "
        f"{EXPECTED_POS_INF_TO_NAN}"
    )

if (
    total_neg_inf
    != EXPECTED_NEG_INF_TO_NAN
):
    raise RuntimeError(
        "Negative-infinity conversion mismatch: "
        f"{total_neg_inf} != "
        f"{EXPECTED_NEG_INF_TO_NAN}"
    )

final_nan_cells = int(
    np.count_nonzero(
        np.isnan(
            X_holdout
        )
    )
)

if (
    final_nan_cells
    != EXPECTED_OUTPUT_NAN
):
    raise RuntimeError(
        "Output NaN-cell mismatch: "
        f"{final_nan_cells} != "
        f"{EXPECTED_OUTPUT_NAN}"
    )

print()
print(
    "[PASS] EXACT FROZEN HOLDOUT MATERIALIZED"
)

print(
    "Shape:",
    X_holdout.shape,
)

print(
    "Dtype:",
    X_holdout.dtype,
)

print(
    "Positive infinity -> NaN:",
    f"{total_pos_inf:,}",
)

print(
    "Negative infinity -> NaN:",
    f"{total_neg_inf:,}",
)

print(
    "Output NaN cells:",
    f"{final_nan_cells:,}",
)

print(
    "Materialization seconds:",
    materialization_seconds,
)

print()
print(
    "[PASS] model input contract exactly matches parent Stage22R summary"
)

X_logical_sha = sha256_array_raw(
    X_holdout
)

y_logical_sha = sha256_array_raw(
    y_true
)

print()
print(
    "Holdout X logical SHA256:",
    X_logical_sha,
)

print(
    "Holdout y logical SHA256:",
    y_logical_sha,
)


# =================================================================================================
# 7. DISCOVER / AUDIT TEN FROZEN STAGE22 ENSEMBLES
# =================================================================================================

banner(
    "STAGE28-4 — AUDIT TEN FROZEN STAGE22 ENSEMBLES"
)

result_files = sorted(
    STAGE22_ROOT.rglob(
        "*_result.json"
    )
)

if len(
    result_files
) != 10:
    raise RuntimeError(
        f"Expected 10 Stage22 result receipts; "
        f"found {len(result_files)}."
    )

ensemble_specs = []

for result_path in result_files:
    obj = read_json(
        result_path
    )

    if (
        obj.get(
            "experiment"
        )
        != "STAGE22_FULL"
    ):
        raise RuntimeError(
            f"Unexpected experiment:\n{result_path}"
        )

    unit = obj[
        "unit"
    ]

    seed = int(
        obj[
            "training_seed"
        ]
    )

    if unit not in EXPECTED_UNITS:
        raise RuntimeError(
            f"Unexpected Stage22 unit: {unit}"
        )

    if seed not in EXPECTED_SEEDS:
        raise RuntimeError(
            f"Unexpected seed: {seed}"
        )

    models = obj[
        "models"
    ]

    if (
        models.get(
            "strategy"
        )
        != "ENS_LGBM_XGB_EQUAL"
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: ensemble strategy mismatch."
        )

    if (
        models.get(
            "ensemble_probability"
        )
        !=
        "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST"
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: probability rule mismatch."
        )

    if (
        models.get(
            "component_combination_dtype"
        )
        != "float64"
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: component-combination dtype changed."
        )

    if (
        models.get(
            "ensemble_storage_dtype"
        )
        != "float32"
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: ensemble-storage dtype changed."
        )

    xgb_info = models[
        "xgboost"
    ]

    lgb_info = models[
        "lightgbm"
    ]

    for name, info in [
        (
            "XGBOOST",
            xgb_info,
        ),
        (
            "LIGHTGBM",
            lgb_info,
        ),
    ]:
        if int(
            info[
                "seed"
            ]
        ) != seed:
            raise RuntimeError(
                f"{unit}/seed{seed}/{name}: seed mismatch."
            )

        if str(
            info.get(
                "backend",
                "",
            )
        ).lower() != "cpu":
            raise RuntimeError(
                f"{unit}/seed{seed}/{name}: backend != CPU."
            )

    xgb_path, xgb_sha, xgb_source = (
        resolve_model_path(
            result_path,
            xgb_info,
        )
    )

    lgb_path, lgb_sha, lgb_source = (
        resolve_model_path(
            result_path,
            lgb_info,
        )
    )

    thresholds = {
        "STANDARD":
            extract_threshold(
                obj[
                    "operating_points"
                ],
                "STANDARD",
            ),

        "BALANCED":
            extract_threshold(
                obj[
                    "operating_points"
                ],
                "BALANCED",
            ),

        "SECURITY":
            extract_threshold(
                obj[
                    "operating_points"
                ],
                "SECURITY",
            ),
    }

    ensemble_specs.append(
        {
            "unit":
                unit,

            "seed":
                seed,

            "result_path":
                result_path,

            "result_sha256":
                sha256_file(
                    result_path
                ),

            "xgb_path":
                xgb_path,

            "xgb_sha256":
                xgb_sha,

            "xgb_source_type":
                xgb_source,

            "xgb_component_id":
                xgb_info[
                    "component_id"
                ],

            "lgb_path":
                lgb_path,

            "lgb_sha256":
                lgb_sha,

            "lgb_source_type":
                lgb_source,

            "lgb_component_id":
                lgb_info[
                    "component_id"
                ],

            "thresholds":
                thresholds,
        }
    )

ensemble_specs.sort(
    key=lambda x: (
        EXPECTED_UNITS.index(
            x[
                "unit"
            ]
        ),
        x[
            "seed"
        ],
    )
)

expected_pairs = [
    (
        unit,
        seed,
    )
    for unit
    in EXPECTED_UNITS
    for seed
    in EXPECTED_SEEDS
]

actual_pairs = [
    (
        x[
            "unit"
        ],
        x[
            "seed"
        ],
    )
    for x
    in ensemble_specs
]

if actual_pairs != expected_pairs:
    raise RuntimeError(
        "Stage22 ensemble grid != exact 2 × 5 design."
    )

for spec in ensemble_specs:
    print(
        f"{spec['unit']:<24} "
        f"seed={spec['seed']}  "
        f"XGB={spec['xgb_component_id']}  "
        f"LGB={spec['lgb_component_id']}  "
        f"thr={spec['thresholds']}"
    )

print()
print(
    "[PASS] 10 / 10 Stage22 ensemble realizations exact"
)

print(
    "[PASS] 20 / 20 model artifacts checksum-verified"
)

print(
    "[PASS] all inference backends frozen to CPU"
)

print(
    "[PASS] all thresholds inherited from validation"
)

print(
    "[PASS] ZERO threshold search on final holdout"
)


# =================================================================================================
# 8. ACTUAL MODEL INFERENCE
# =================================================================================================

banner(
    "STAGE28-4 — AUTHORIZED FINAL-HOLDOUT MODEL INFERENCE"
)

STAGE4_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

probability_arrays = {}

probability_records = {}

metric_rows = []

inference_started = (
    time.perf_counter()
)

for cell_index, spec in enumerate(
    ensemble_specs,
    start=1,
):
    unit = spec[
        "unit"
    ]

    seed = spec[
        "seed"
    ]

    print()
    print(
        "-" * 120
    )

    print(
        f"[{cell_index:02d}/10] "
        f"{unit} — seed {seed}"
    )

    print(
        "-" * 120
    )

    cell_started = (
        time.perf_counter()
    )

    # ---------------------------------------------------------------------------------------------
    # XGBOOST — LOAD FROZEN MODEL, NO FIT
    # ---------------------------------------------------------------------------------------------

    xgb_model = (
        xgb.XGBClassifier()
    )

    xgb_model.load_model(
        str(
            spec[
                "xgb_path"
            ]
        )
    )

    xgb_rounds = int(
        xgb_model
        .get_booster()
        .num_boosted_rounds()
    )

    if xgb_rounds != 400:
        raise RuntimeError(
            f"{unit}/seed{seed}: "
            f"XGBoost rounds {xgb_rounds} != 400."
        )

    xgb_started = (
        time.perf_counter()
    )

    p_xgb = (
        xgb_model.predict_proba(
            X_holdout
        )[
            :,
            1
        ]
    )

    xgb_seconds = (
        time.perf_counter()
        - xgb_started
    )

    if (
        p_xgb.shape
        != (
            EXPECTED_ROWS,
        )
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: XGB probability shape mismatch."
        )

    # ---------------------------------------------------------------------------------------------
    # LIGHTGBM — LOAD FROZEN MODEL, NO FIT
    # ---------------------------------------------------------------------------------------------

    lgb_model = lgb.Booster(
        model_file=str(
            spec[
                "lgb_path"
            ]
        )
    )

    lgb_iterations = int(
        lgb_model.current_iteration()
    )

    if lgb_iterations != 400:
        raise RuntimeError(
            f"{unit}/seed{seed}: "
            f"LightGBM iterations {lgb_iterations} != 400."
        )

    lgb_started = (
        time.perf_counter()
    )

    p_lgb = lgb_model.predict(
        X_holdout,
        num_iteration=lgb_iterations,
    )

    lgb_seconds = (
        time.perf_counter()
        - lgb_started
    )

    if (
        p_lgb.shape
        != (
            EXPECTED_ROWS,
        )
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: LGB probability shape mismatch."
        )

    # ---------------------------------------------------------------------------------------------
    # FROZEN SCIENTIFIC UNIT:
    # float64 combination -> persisted float32 ensemble probability.
    # ---------------------------------------------------------------------------------------------

    p_ensemble = (
        0.5
        * np.asarray(
            p_lgb,
            dtype=np.float64,
        )
        +
        0.5
        * np.asarray(
            p_xgb,
            dtype=np.float64,
        )
    ).astype(
        np.float32,
        copy=False,
    )

    if (
        p_ensemble.dtype
        != np.float32
    ):
        raise RuntimeError(
            "Ensemble storage dtype != float32."
        )

    if not np.all(
        np.isfinite(
            p_ensemble
        )
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: non-finite ensemble probability."
        )

    if (
        float(
            p_ensemble.min()
        ) < 0.0
        or
        float(
            p_ensemble.max()
        ) > 1.0
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: probability outside [0,1]."
        )

    # ---------------------------------------------------------------------------------------------
    # FROZEN METRICS
    # ---------------------------------------------------------------------------------------------

    roc_auc = float(
        roc_auc_score(
            y_true,
            p_ensemble,
        )
    )

    pr_auc = float(
        average_precision_score(
            y_true,
            p_ensemble,
        )
    )

    op_results = {}

    for op_name in [
        "STANDARD",
        "BALANCED",
        "SECURITY",
    ]:
        op_results[
            op_name
        ] = operating_metrics(
            y_true,
            p_ensemble,
            spec[
                "thresholds"
            ][
                op_name
            ],
        )

    probability_key = (
        unit.lower()
        + "_seed"
        + str(
            seed
        )
    )

    probability_arrays[
        probability_key
    ] = p_ensemble.copy()

    probability_sha = (
        sha256_array_raw(
            p_ensemble
        )
    )

    cell_seconds = (
        time.perf_counter()
        - cell_started
    )

    probability_records[
        probability_key
    ] = {
        "unit":
            unit,

        "seed":
            seed,

        "dtype":
            str(
                p_ensemble.dtype
            ),

        "rows":
            int(
                p_ensemble.size
            ),

        "logical_sha256":
            probability_sha,

        "minimum":
            float(
                p_ensemble.min()
            ),

        "maximum":
            float(
                p_ensemble.max()
            ),

        "xgboost_inference_seconds":
            xgb_seconds,

        "lightgbm_inference_seconds":
            lgb_seconds,

        "cell_total_seconds":
            cell_seconds,
    }

    row = {
        "unit":
            unit,

        "seed":
            seed,

        "shared_holdout_rows":
            EXPECTED_ROWS,

        "shared_holdout_benign":
            EXPECTED_BENIGN,

        "shared_holdout_attack":
            EXPECTED_ATTACK,

        "shared_holdout_attack_prevalence":
            EXPECTED_ATTACK
            / EXPECTED_ROWS,

        "roc_auc":
            roc_auc,

        "pr_auc":
            pr_auc,

        "ensemble_probability_sha256":
            probability_sha,

        "xgboost_component_id":
            spec[
                "xgb_component_id"
            ],

        "xgboost_model_sha256":
            spec[
                "xgb_sha256"
            ],

        "lightgbm_component_id":
            spec[
                "lgb_component_id"
            ],

        "lightgbm_model_sha256":
            spec[
                "lgb_sha256"
            ],

        "inference_seconds_xgboost":
            xgb_seconds,

        "inference_seconds_lightgbm":
            lgb_seconds,

        "inference_seconds_cell_total":
            cell_seconds,
    }

    for op_name in [
        "STANDARD",
        "BALANCED",
        "SECURITY",
    ]:
        op = op_results[
            op_name
        ]

        prefix = (
            op_name.lower()
        )

        for field in [
            "threshold",
            "threshold_float32_runtime",
            "accuracy",
            "precision",
            "recall",
            "fpr",
            "f1",
            "f2",
            "tp",
            "fp",
            "tn",
            "fn",
        ]:
            row[
                f"{prefix}_{field}"
            ] = op[
                field
            ]

    metric_rows.append(
        row
    )

    print(
        f"ROC-AUC : {roc_auc:.12f}"
    )

    print(
        f"PR-AUC  : {pr_auc:.12f}"
    )

    print(
        "STANDARD:"
        f" recall={op_results['STANDARD']['recall']:.12f},"
        f" fpr={op_results['STANDARD']['fpr']:.12f},"
        f" f1={op_results['STANDARD']['f1']:.12f}"
    )

    print(
        "BALANCED:"
        f" recall={op_results['BALANCED']['recall']:.12f},"
        f" fpr={op_results['BALANCED']['fpr']:.12f},"
        f" f1={op_results['BALANCED']['f1']:.12f}"
    )

    print(
        "SECURITY:"
        f" recall={op_results['SECURITY']['recall']:.12f},"
        f" fpr={op_results['SECURITY']['fpr']:.12f},"
        f" f2={op_results['SECURITY']['f2']:.12f}"
    )

    print(
        f"[PASS] persisted float32 probability SHA256: "
        f"{probability_sha}"
    )

    # Free component-model and component-probability objects.
    del p_xgb
    del p_lgb
    del p_ensemble
    del xgb_model
    del lgb_model
    gc.collect()

total_inference_seconds = (
    time.perf_counter()
    - inference_started
)

if len(
    metric_rows
) != 10:
    raise RuntimeError(
        "Stage28-4 did not produce exactly 10 ensemble evaluations."
    )

print()
print(
    "[PASS] 20 component-model inferences completed"
)

print(
    "[PASS] 10 persisted Stage22 ensemble realizations evaluated"
)

print(
    f"Total inference wall time: {total_inference_seconds:.3f} seconds"
)


# =================================================================================================
# 9. WRITE SEED-LEVEL METRICS + PROBABILITY ARTIFACT
# =================================================================================================

banner(
    "STAGE28-4 — WRITE DURABLE SEED-LEVEL RESULTS"
)

metrics_df = pd.DataFrame(
    metric_rows
)

metrics_df = (
    metrics_df.sort_values(
        [
            "unit",
            "seed",
        ],
        key=lambda s:
            (
                s.map(
                    {
                        "RANDOM_NATURAL": 0,
                        "CHRONOLOGICAL_NATURAL": 1,
                    }
                )
                if s.name == "unit"
                else s
            ),
    )
    .reset_index(
        drop=True
    )
)

METRICS_PATH = (
    STAGE4_ROOT
    / "stage28_4_seed_level_metrics.csv"
)

PROBABILITY_PATH = (
    STAGE4_ROOT
    / "stage28_4_shared_holdout_ensemble_probabilities.npz"
)

metrics_df.to_csv(
    METRICS_PATH,
    index=False,
)

np.savez_compressed(
    PROBABILITY_PATH,
    **probability_arrays,
)

probability_artifact_sha = (
    sha256_file(
        PROBABILITY_PATH
    )
)

print(
    "[PASS] seed-level metric table written"
)

print(
    "[PASS] ten persisted float32 ensemble probability arrays written"
)

print(
    "Probability artifact SHA256:",
    probability_artifact_sha,
)


# =================================================================================================
# 10. PREREGISTERED STAGE22 DIRECTIONAL CONCLUSION STABILITY
# =================================================================================================

banner(
    "STAGE28-4 — PREREGISTERED STAGE22 DIRECTIONAL STABILITY"
)

metric_lookup = {}

for row in metric_rows:
    metric_lookup[
        (
            row[
                "unit"
            ],
            int(
                row[
                    "seed"
                ]
            ),
        )
    ] = row

claim_rows = []

contrast_rows = []

for seed in EXPECTED_SEEDS:
    random_row = metric_lookup[
        (
            "RANDOM_NATURAL",
            seed,
        )
    ]

    chrono_row = metric_lookup[
        (
            "CHRONOLOGICAL_NATURAL",
            seed,
        )
    ]

    pr_random = float(
        random_row[
            "pr_auc"
        ]
    )

    pr_chrono = float(
        chrono_row[
            "pr_auc"
        ]
    )

    roc_random = float(
        random_row[
            "roc_auc"
        ]
    )

    roc_chrono = float(
        chrono_row[
            "roc_auc"
        ]
    )

    pr_condition = bool(
        pr_random
        < pr_chrono
    )

    roc_condition = bool(
        roc_random
        < roc_chrono
    )

    contrast_rows.append(
        {
            "seed":
                seed,

            "pr_auc_random":
                pr_random,

            "pr_auc_chronological":
                pr_chrono,

            "pr_auc_random_minus_chronological":
                pr_random
                - pr_chrono,

            "pr_random_lt_chronological":
                pr_condition,

            "roc_auc_random":
                roc_random,

            "roc_auc_chronological":
                roc_chrono,

            "roc_auc_random_minus_chronological":
                roc_random
                - roc_chrono,

            "roc_random_lt_chronological":
                roc_condition,
        }
    )

    claim_rows.append(
        {
            "claim_id":
                "STAGE22_PR_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT",

            "parent_stage":
                "STAGE22_FULL",

            "family_if_applicable":
                "",

            "learner_if_applicable":
                "",

            "seed":
                seed,

            "claim_condition":
                "PR_AUC_RANDOM_NATURAL < PR_AUC_CHRONOLOGICAL_NATURAL",

            "condition_met":
                pr_condition,

            "analysis_status":
                "DESCRIPTIVE_ROBUSTNESS_PRE_REGISTERED",
        }
    )

    claim_rows.append(
        {
            "claim_id":
                "STAGE22_ROC_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT",

            "parent_stage":
                "STAGE22_FULL",

            "family_if_applicable":
                "",

            "learner_if_applicable":
                "",

            "seed":
                seed,

            "claim_condition":
                "ROC_AUC_RANDOM_NATURAL < ROC_AUC_CHRONOLOGICAL_NATURAL",

            "condition_met":
                roc_condition,

            "analysis_status":
                "DESCRIPTIVE_ROBUSTNESS_PRE_REGISTERED",
        }
    )

claim_df = pd.DataFrame(
    claim_rows
)

contrast_df = pd.DataFrame(
    contrast_rows
)

if len(
    claim_df
) != 10:
    raise RuntimeError(
        "Expected exactly 10 Stage22 directional claim realizations."
    )

CLAIMS_PATH = (
    STAGE4_ROOT
    / "stage28_4_stage22_directional_claims.csv"
)

CONTRAST_PATH = (
    STAGE4_ROOT
    / "stage28_4_random_vs_chronological_seedwise.csv"
)

claim_df.to_csv(
    CLAIMS_PATH,
    index=False,
)

contrast_df.to_csv(
    CONTRAST_PATH,
    index=False,
)

stability_rows = []

for claim_id in [
    "STAGE22_PR_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT",
    "STAGE22_ROC_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT",
]:
    subset = claim_df.loc[
        claim_df[
            "claim_id"
        ]
        == claim_id
    ]

    supporting = int(
        subset[
            "condition_met"
        ].sum()
    )

    total = int(
        len(
            subset
        )
    )

    if total != 5:
        raise RuntimeError(
            f"{claim_id}: seed denominator != 5."
        )

    stability_rows.append(
        {
            "claim_id":
                claim_id,

            "supporting_seeds":
                supporting,

            "total_frozen_seeds":
                total,

            "stability_rate":
                supporting
                / total,

            "interpretation":
                "DESCRIPTIVE_ROBUSTNESS_NOT_NEW_SIGNIFICANCE_TEST",
        }
    )

stability_df = pd.DataFrame(
    stability_rows
)

STABILITY_PATH = (
    STAGE4_ROOT
    / "stage28_4_stage22_directional_stability_summary.csv"
)

stability_df.to_csv(
    STABILITY_PATH,
    index=False,
)

for _, row in (
    contrast_df.iterrows()
):
    print(
        f"seed {int(row['seed'])}: "
        f"ΔPR(random-chrono)="
        f"{row['pr_auc_random_minus_chronological']:+.12f} "
        f"support={bool(row['pr_random_lt_chronological'])}; "
        f"ΔROC(random-chrono)="
        f"{row['roc_auc_random_minus_chronological']:+.12f} "
        f"support={bool(row['roc_random_lt_chronological'])}"
    )

print()

for _, row in (
    stability_df.iterrows()
):
    print(
        row[
            "claim_id"
        ]
    )

    print(
        f"  supporting seeds: "
        f"{int(row['supporting_seeds'])}/"
        f"{int(row['total_frozen_seeds'])}"
    )

    print(
        f"  stability rate  : "
        f"{float(row['stability_rate']):.3f}"
    )

print()
print(
    "[PASS] no post-result condition created"
)

print(
    "[PASS] no formal significance test introduced"
)


# =================================================================================================
# 11. CREATE THE FROZEN REQUIRED COMBINED conclusion_stability.csv
# =================================================================================================

banner(
    "STAGE28-4 — COMPLETE FROZEN conclusion_stability.csv"
)

loao_claim_df = pd.read_csv(
    LOAO_CONCLUSION_STABILITY
)

expected_columns = [
    "claim_id",
    "parent_stage",
    "family_if_applicable",
    "learner_if_applicable",
    "seed",
    "claim_condition",
    "condition_met",
    "analysis_status",
]

if list(
    loao_claim_df.columns
) != expected_columns:
    raise RuntimeError(
        "Stage28-3B LOAO conclusion-stability schema changed."
    )

if list(
    claim_df.columns
) != expected_columns:
    raise RuntimeError(
        "Stage28-4 Stage22 conclusion-stability schema mismatch."
    )

combined_claim_df = pd.concat(
    [
        loao_claim_df,
        claim_df,
    ],
    ignore_index=True,
)

combined_claim_df.to_csv(
    COMBINED_CONCLUSION_STABILITY,
    index=False,
)

print(
    "Stage28-3B LOAO rows :",
    len(
        loao_claim_df
    ),
)

print(
    "Stage28-4 Stage22 rows:",
    len(
        claim_df
    ),
)

print(
    "Combined rows          :",
    len(
        combined_claim_df
    ),
)

print()
print(
    "[PASS] frozen required output created:"
)

print(
    " ",
    COMBINED_CONCLUSION_STABILITY.relative_to(
        REPO
    ),
)


# =================================================================================================
# 12. MATERIALIZATION RECEIPT
# =================================================================================================

banner(
    "STAGE28-4 — WRITE RECEIPTS"
)

MATERIALIZATION_RECEIPT = (
    STAGE4_ROOT
    / "stage28_4_holdout_materialization_receipt.json"
)

materialization_receipt = {
    "stage":
        "Stage28-4",

    "type":
        "STAGE22_SHARED_FINAL_HOLDOUT_MATERIALIZATION",

    "created_at_utc":
        utc_now(),

    "scientific_parent_commit":
        EXPECTED_PARENT,

    "population":
        "STAGE22R_SHARED_FINAL_SINGLE_HOLDOUT",

    "interpretive_status":
        (
            "PRE_REGISTERED_ROBUSTNESS_REEVALUATION_OF_ALREADY_KNOWN_"
            "PARENT_HOLDOUT_NOT_NEW_BLIND_HOLDOUT"
        ),

    "membership": {
        "path":
            str(
                MEMBERSHIP_PATH.relative_to(
                    REPO
                )
            ),

        "file_sha256":
            membership_sha,

        "rows":
            EXPECTED_ROWS,

        "benign":
            EXPECTED_BENIGN,

        "attack":
            EXPECTED_ATTACK,

        "logical_arrays":
            membership_logical,
    },

    "source_files": {
        "day_8":
            source_records[
                8
            ],

        "day_9":
            source_records[
                9
            ],
    },

    "per_day_materialization":
        {
            str(
                k
            ):
                v
            for k, v
            in materialization_records.items()
        },

    "model_input": {
        "rows":
            EXPECTED_ROWS,

        "features":
            EXPECTED_FEATURES,

        "dtype":
            "float64",

        "scaling":
            "NONE",

        "explicit_imputation":
            "NONE",

        "positive_infinity_to_nan":
            total_pos_inf,

        "negative_infinity_to_nan":
            total_neg_inf,

        "output_nan_cells":
            final_nan_cells,

        "X_logical_sha256":
            X_logical_sha,

        "y_logical_sha256":
            y_logical_sha,
    },

    "scientific_operations": {
        "new_model_fits":
            0,

        "model_inference":
            0,

        "threshold_selection":
            0,

        "model_selection":
            0,

        "new_formal_statistical_tests":
            0,

        "raw_data_read_passes_per_file":
            1,

        "stage28_shared_final_holdout_materialization":
            1,
    },

    "status":
        "PASS_EXACT_FROZEN_HOLDOUT_MATERIALIZED",
}

write_json(
    MATERIALIZATION_RECEIPT,
    materialization_receipt,
)


# =================================================================================================
# 13. STAGE28-4 FINAL RECEIPT
# =================================================================================================

FINAL_RECEIPT = (
    STAGE4_ROOT
    / "stage28_4_receipt.json"
)

probability_keys = sorted(
    probability_arrays.keys()
)

receipt = {
    "stage":
        "Stage28-4",

    "type":
        "STAGE22_FIVE_SEED_SHARED_FINAL_HOLDOUT_ROBUSTNESS_INFERENCE",

    "created_at_utc":
        utc_now(),

    "scientific_parent_commit":
        EXPECTED_PARENT,

    "authorization": {
        "previous_stage":
            "Stage28-3C",

        "authorized_by":
            str(
                STAGE3C_RECEIPT.relative_to(
                    REPO
                )
            ),

        "new_model_fitting":
            "PERMANENTLY_CLOSED",

        "threshold_selection_on_final_holdout":
            "FORBIDDEN",

        "model_selection_on_final_holdout":
            "FORBIDDEN",
    },

    "holdout": {
        "name":
            "STAGE22R_SHARED_FINAL_SINGLE_HOLDOUT",

        "rows":
            EXPECTED_ROWS,

        "benign":
            EXPECTED_BENIGN,

        "attack":
            EXPECTED_ATTACK,

        "attack_prevalence":
            EXPECTED_ATTACK
            / EXPECTED_ROWS,

        "feature_count":
            EXPECTED_FEATURES,

        "model_input_dtype":
            "float64",

        "X_logical_sha256":
            X_logical_sha,

        "y_logical_sha256":
            y_logical_sha,

        "blind_status":
            (
                "NOT_NEW_BLIND_HOLDOUT_PARENT_STAGE22R_ALREADY_OPENED; "
                "STAGE28_USE_IS_PRE_REGISTERED_ROBUSTNESS_ONLY"
            ),
    },

    "scientific_unit": {
        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "component_learners": [
            "LIGHTGBM",
            "XGBOOST",
        ],

        "probability_rule":
            "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST",

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",
    },

    "design": {
        "units":
            EXPECTED_UNITS,

        "seeds":
            EXPECTED_SEEDS,

        "ensemble_realizations":
            10,

        "component_model_inferences":
            20,

        "new_model_fits":
            0,
    },

    "threshold_policy": {
        "selection_population":
            "FROZEN_STAGE22_DEVELOPMENT_VALIDATION_ONLY",

        "final_holdout_threshold_search":
            "FORBIDDEN",

        "prediction_rule":
            (
                "ATTACK_IF_PERSISTED_FLOAT32_ENSEMBLE_"
                "PROBABILITY_GTE_FROZEN_THRESHOLD"
            ),

        "operating_points": [
            "STANDARD",
            "BALANCED",
            "SECURITY",
        ],
    },

    "probability_artifact": {
        "path":
            str(
                PROBABILITY_PATH.relative_to(
                    REPO
                )
            ),

        "file_sha256":
            probability_artifact_sha,

        "keys":
            probability_keys,

        "arrays":
            probability_records,
    },

    "seed_level_metrics": {
        "path":
            str(
                METRICS_PATH.relative_to(
                    REPO
                )
            ),

        "rows":
            len(
                metrics_df
            ),
    },

    "stage22_directional_claims": {
        "path":
            str(
                CLAIMS_PATH.relative_to(
                    REPO
                )
            ),

        "conditions":
            [
                {
                    "claim_id":
                        (
                            "STAGE22_PR_RANDOM_LT_CHRONO_"
                            "ON_SHARED_FINAL_HOLDOUT"
                        ),

                    "condition":
                        (
                            "PR_AUC_RANDOM_NATURAL < "
                            "PR_AUC_CHRONOLOGICAL_NATURAL"
                        ),
                },
                {
                    "claim_id":
                        (
                            "STAGE22_ROC_RANDOM_LT_CHRONO_"
                            "ON_SHARED_FINAL_HOLDOUT"
                        ),

                    "condition":
                        (
                            "ROC_AUC_RANDOM_NATURAL < "
                            "ROC_AUC_CHRONOLOGICAL_NATURAL"
                        ),
                },
            ],

        "stability_rate_definition":
            "NUMBER_OF_FROZEN_SEEDS_SUPPORTING_CONDITION / 5",

        "summary_path":
            str(
                STABILITY_PATH.relative_to(
                    REPO
                )
            ),

        "new_condition_creation":
            "FORBIDDEN_AND_NOT_PERFORMED",

        "formal_significance_testing":
            "NOT_PERFORMED",
    },

    "combined_conclusion_stability": {
        "path":
            str(
                COMBINED_CONCLUSION_STABILITY.relative_to(
                    REPO
                )
            ),

        "stage28_3b_rows":
            int(
                len(
                    loao_claim_df
                )
            ),

        "stage28_4_stage22_rows":
            int(
                len(
                    claim_df
                )
            ),

        "combined_rows":
            int(
                len(
                    combined_claim_df
                )
            ),
    },

    "scientific_operations": {
        "new_model_fits":
            0,

        "component_model_inferences":
            20,

        "ensemble_evaluation_cells":
            10,

        "threshold_selection":
            0,

        "model_selection":
            0,

        "shared_stage22_final_holdout_stage28_openings":
            1,

        "new_formal_statistical_tests":
            0,

        "shap_recomputation":
            0,

        "subset_search":
            0,

        "new_holdout_creation":
            0,
    },

    "total_inference_seconds":
        total_inference_seconds,

    "next_authorized_step":
        (
            "ZERO_FIT_FINAL_SYNTHESIS_AND_MANUSCRIPT_INTEGRATION; "
            "NO FURTHER MODEL FITTING; NO STAGE29."
        ),

    "status":
        "STAGE28_4_COMPLETE",
}

write_json(
    FINAL_RECEIPT,
    receipt,
)


# =================================================================================================
# 14. README
# =================================================================================================

README_PATH = (
    STAGE4_ROOT
    / "README_STAGE28_4.md"
)

stability_map = {
    row[
        "claim_id"
    ]:
        {
            "supporting":
                int(
                    row[
                        "supporting_seeds"
                    ]
                ),

            "total":
                int(
                    row[
                        "total_frozen_seeds"
                    ]
                ),

            "rate":
                float(
                    row[
                        "stability_rate"
                    ]
                ),
        }
    for _, row
    in stability_df.iterrows()
}

pr_stability = stability_map[
    "STAGE22_PR_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT"
]

roc_stability = stability_map[
    "STAGE22_ROC_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT"
]

readme_text = f"""# Stage28-4 — Stage22 Shared Final Holdout Robustness Inference

Scientific parent: `{EXPECTED_PARENT}`

## Scope

Stage28-4 evaluates the ten frozen Stage22 FULL ensemble realizations:

- RANDOM_NATURAL, seeds 42–46
- CHRONOLOGICAL_NATURAL, seeds 42–46

All realizations use the same frozen Stage22R shared final holdout.

No model fitting, threshold selection, model selection, SHAP recomputation,
subset search, or new formal significance test was performed.

## Holdout

- Rows: {EXPECTED_ROWS:,}
- Benign: {EXPECTED_BENIGN:,}
- Attack: {EXPECTED_ATTACK:,}
- Features: {EXPECTED_FEATURES}
- Model-input dtype: float64
- X logical SHA256: `{X_logical_sha}`
- y logical SHA256: `{y_logical_sha}`

The parent Stage22R holdout was already historically opened. Stage28-4 is
therefore preregistered robustness re-evaluation and does not constitute a
new blind holdout claim.

## Frozen scientific unit

`0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST`

Component probabilities are combined in float64 and the ensemble
probability is persisted as float32 before threshold application.

## Frozen Stage22 directional conclusion stability

### PR-AUC

Claim:

`PR_AUC_RANDOM_NATURAL < PR_AUC_CHRONOLOGICAL_NATURAL`

Supporting seeds:

`{pr_stability['supporting']} / {pr_stability['total']}`

Stability rate:

`{pr_stability['rate']:.6f}`

### ROC-AUC

Claim:

`ROC_AUC_RANDOM_NATURAL < ROC_AUC_CHRONOLOGICAL_NATURAL`

Supporting seeds:

`{roc_stability['supporting']} / {roc_stability['total']}`

Stability rate:

`{roc_stability['rate']:.6f}`

## Closure

Stage28 model fitting remains permanently closed at 108 / 108 new fits.

No Stage29 is authorized. The remaining work is zero-fit synthesis and
manuscript integration.
"""

README_PATH.write_text(
    readme_text,
    encoding="utf-8",
)


# =================================================================================================
# 15. CHECKSUM MANIFEST
# =================================================================================================

CHECKSUM_PATH = (
    STAGE4_ROOT
    / "stage28_4_checksums.sha256"
)

artifact_paths = sorted(
    [
        p
        for p in STAGE4_ROOT.rglob(
            "*"
        )
        if (
            p.is_file()
            and
            p != CHECKSUM_PATH
        )
    ]
)

artifact_paths.append(
    COMBINED_CONCLUSION_STABILITY
)

checksum_lines = []

for path in artifact_paths:
    checksum_lines.append(
        sha256_file(
            path
        )
        + "  "
        + str(
            path.relative_to(
                REPO
            )
        )
    )

CHECKSUM_PATH.write_text(
    "\n".join(
        checksum_lines
    )
    + "\n",
    encoding="utf-8",
)

print(
    "[PASS] Stage28-4 receipts written"
)

print(
    "[PASS] README written"
)

print(
    "[PASS] checksum manifest written"
)


# =================================================================================================
# 16. FINAL SCIENTIFIC INVARIANTS
# =================================================================================================

banner(
    "STAGE28-4 — FINAL SCIENTIFIC INVARIANTS"
)

# Re-read permanent fit closure. It must still be untouched.
closure_after = read_json(
    CLOSURE_RECEIPT
)

if (
    closure_after
    != closure
):
    raise RuntimeError(
        "Stage28 closure receipt changed unexpectedly."
    )

if (
    int(
        closure_after[
            "fit_budget_closure"
        ][
            "consumed_new_fits"
        ]
    )
    != 108
):
    raise RuntimeError(
        "Fit ledger changed."
    )

if (
    int(
        closure_after[
            "fit_budget_closure"
        ][
            "remaining_new_fits"
        ]
    )
    != 0
):
    raise RuntimeError(
        "Remaining fit budget changed."
    )

if len(
    probability_arrays
) != 10:
    raise RuntimeError(
        "Probability artifact count != 10."
    )

if len(
    metrics_df
) != 10:
    raise RuntimeError(
        "Metrics count != 10."
    )

if len(
    claim_df
) != 10:
    raise RuntimeError(
        "Stage22 directional claim count != 10."
    )

print(
    "[PASS] Stage28 new-fit ledger remains 108 / 108"
)

print(
    "[PASS] new fits performed = 0"
)

print(
    "[PASS] threshold selections performed = 0"
)

print(
    "[PASS] model selections performed = 0"
)

print(
    "[PASS] new formal statistical tests = 0"
)

print(
    "[PASS] SHAP recomputation = 0"
)

print(
    "[PASS] subset search = 0"
)

print(
    "[PASS] exactly 10 ensemble evaluations"
)

print(
    "[PASS] exactly 20 frozen component-model inferences"
)

print(
    "[PASS] Stage22 shared-final robustness opening consumed"
)


# =================================================================================================
# 17. GIT CHANGE GATE
# =================================================================================================

banner(
    "STAGE28-4 — GIT CHANGE GATE"
)

status_before_add = git(
    "status",
    "--porcelain",
)

print(
    status_before_add
)

allowed_prefix = str(
    STAGE4_ROOT.relative_to(
        REPO
    )
)

allowed_combined = str(
    COMBINED_CONCLUSION_STABILITY.relative_to(
        REPO
    )
)

for line in (
    status_before_add.splitlines()
):
    if not line.strip():
        continue

    path_text = line[
        3:
    ].strip()

    if (
        not path_text.startswith(
            allowed_prefix
        )
        and
        path_text
        != allowed_combined
    ):
        raise RuntimeError(
            "Unexpected repository modification before commit:\n"
            + line
        )

print()
print(
    "[PASS] only authorized Stage28-4 artifacts changed"
)


# =================================================================================================
# 18. DURABLE COMMIT
# =================================================================================================

banner(
    "STAGE28-4 — DURABLE COMMIT / PUSH"
)

# Race gate: origin/main must still be our parent.
run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)

origin_now = git(
    "rev-parse",
    "origin/main",
)

if origin_now != EXPECTED_PARENT:
    raise RuntimeError(
        "origin/main changed during Stage28-4 execution. "
        "Refusing to commit/push."
    )

git(
    "config",
    "user.name",
    "Stage28 Kaggle",
)

git(
    "config",
    "user.email",
    "stage28-kaggle@users.noreply.github.com",
)

# Force-add so ignored numpy artifact rules cannot recreate the
# historical AUTO-B durability bug.
git(
    "add",
    "-f",
    "--",
    str(
        STAGE4_ROOT.relative_to(
            REPO
        )
    ),
)

git(
    "add",
    "-f",
    "--",
    str(
        COMBINED_CONCLUSION_STABILITY.relative_to(
            REPO
        )
    ),
)

cached = git(
    "diff",
    "--cached",
    "--name-only",
)

if not cached.strip():
    raise RuntimeError(
        "Nothing staged for Stage28-4 commit."
    )

print(
    "Staged files:"
)

print(
    cached
)

commit_message = (
    "stage28-4: evaluate Stage22 five-seed ensembles "
    "on shared final holdout"
)

commit_result = run(
    [
        "git",
        "commit",
        "-m",
        commit_message,
    ]
)

print(
    commit_result.stdout
)

new_head = git(
    "rev-parse",
    "HEAD",
)

if new_head == EXPECTED_PARENT:
    raise RuntimeError(
        "Commit did not advance HEAD."
    )

print(
    "New commit:",
    new_head,
)

git_push_with_kaggle_secret()

# Verify remote durability.
run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)

remote_after = git(
    "rev-parse",
    "origin/main",
)

if remote_after != new_head:
    raise RuntimeError(
        "Remote durability verification failed."
    )

if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Repository not clean after Stage28-4 commit."
    )

print()
print(
    "[PASS] Stage28-4 commit durable on origin/main"
)

print(
    "[PASS] repository clean"
)


# =================================================================================================
# 19. FINAL REPORT
# =================================================================================================

banner(
    "STAGE28-4 — COMPLETE"
)

print(
    "Commit:",
    new_head,
)

print()
print(
    "New model fits                    : 0"
)

print(
    "Component model inferences        : 20"
)

print(
    "Ensemble evaluation cells         : 10"
)

print(
    "Threshold selections              : 0"
)

print(
    "Model selections                  : 0"
)

print(
    "New formal statistical tests      : 0"
)

print(
    "SHAP recomputation                : 0"
)

print(
    "Subset search                     : 0"
)

print(
    "Shared Stage22 holdout rows        :",
    f"{EXPECTED_ROWS:,}",
)

print(
    "Shared Stage22 holdout benign      :",
    f"{EXPECTED_BENIGN:,}",
)

print(
    "Shared Stage22 holdout attack      :",
    f"{EXPECTED_ATTACK:,}",
)

print()
print(
    "STAGE22 DIRECTIONAL STABILITY"
)

print(
    "-----------------------------"
)

for _, row in (
    stability_df.iterrows()
):
    print(
        f"{row['claim_id']}: "
        f"{int(row['supporting_seeds'])}/"
        f"{int(row['total_frozen_seeds'])} "
        f"= {float(row['stability_rate']):.3f}"
    )

print()
print(
    "Stage28 fitting status             : PERMANENTLY CLOSED"
)

print(
    "Stage28 new-fit budget             : 108 / 108 CONSUMED"
)

print(
    "Stage29                           : NOT AUTHORIZED"
)

print()
print(
    "NEXT AUTHORIZED WORK:"
)

print(
    "ZERO-FIT FINAL SYNTHESIS + MANUSCRIPT INTEGRATION."
)


STAGE28-4 — REPOSITORY / PARENT GATE

Expected parent: 2679d0c208d514b381caa12e96c959f4f2ee5ee7
Local HEAD     : 2679d0c208d514b381caa12e96c959f4f2ee5ee7
origin/main    : 2679d0c208d514b381caa12e96c959f4f2ee5ee7

[PASS] Stage28-3C parent exact
[PASS] repository clean
[PASS] Stage28-4 has not been durably executed

STAGE28-4 — SCIENTIFIC AUTHORIZATION GATE

[PASS] Stage28 fitting permanently closed at 108 / 108
[PASS] Stage28-4 explicitly authorized
[PASS] final-holdout threshold search FORBIDDEN
[PASS] final-holdout model selection FORBIDDEN
[PASS] equal-weight LGBM/XGB ensemble exact
[PASS] two Stage22 directional claims frozen

STAGE28-4 — RUNTIME / ARTIFACT GATE

numpy     : 2.0.2
sklearn   : 1.6.1
xgboost   : 3.2.0
lightgbm  : 4.6.0

[PASS] runtime versions exact
[PASS] frozen membership SHA exact
[PASS] exact March source files attached

STAGE28-4 — LOAD FROZEN HOLDOUT MEMBERSHIP

day_id=8: rows=331,017, row_index_min=0, row_index_max=331,111
day_id=9: rows=1,043,116, row_index_m

RuntimeError: CHRONOLOGICAL_NATURAL/seed46: probability rule mismatch.

In [13]:
# =================================================================================================
# STAGE28-4-R1 — SALVAGE ALREADY-MATERIALIZED SHARED HOLDOUT + COMPLETE INFERENCE
#
# IMPORTANT:
#   RUN IN THE SAME KAGGLE KERNEL AS THE FAILED STAGE28-4 CELL.
#
# THIS CELL:
#   - DOES NOT READ 03-01-2018.csv
#   - DOES NOT READ 03-02-2018.csv
#   - DOES NOT REMATERIALIZE THE HOLDOUT
#   - DOES NOT FIT ANY MODEL
#   - DOES NOT SELECT ANY THRESHOLD
#
# RECOVERY STATE:
#   The authorized Stage28 shared-holdout materialization already occurred successfully.
#   The failure happened during receipt auditing before model inference.
#
# TWO-PHASE DURABILITY:
#   Phase A: durably freeze the already-consumed materialization/opening.
#   Phase B: perform frozen inference and freeze final Stage28-4 results.
# =================================================================================================

from __future__ import annotations

import base64
import csv
import gc
import hashlib
import json
import math
import os
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

import xgboost as xgb
import lightgbm as lgb


# =================================================================================================
# CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

FAILED_STAGE28_4_PARENT = (
    "2679d0c208d514b381caa12e96c959f4f2ee5ee7"
)

ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

STAGE22_ROOT = (
    ROOT
    / "stage28_2a_stage22_seed_stability"
)

PROTOCOL_ROOT = (
    ROOT
    / "stage28_0_protocol_lock"
)

STAGE3_ROOT = (
    ROOT
    / "stage28_3_seed_uncertainty"
)

STAGE4_ROOT = (
    ROOT
    / "stage28_4_stage22_shared_final_holdout"
)

CLOSURE_RECEIPT = (
    ROOT
    / "stage28_3a_experiment_closure_audit"
    / "stage28_3a_experiment_closure_receipt.json"
)

STAGE3C_RECEIPT = (
    STAGE3_ROOT
    / "stage28_3c_receipt.json"
)

STAGE22_SPEC = (
    PROTOCOL_ROOT
    / "stage22_cell_spec.json"
)

STABILITY_SPEC = (
    PROTOCOL_ROOT
    / "conclusion_stability_spec.json"
)

LOAO_STABILITY_PATH = (
    STAGE3_ROOT
    / "stage28_3b_loao_conclusion_stability.csv"
)

COMBINED_STABILITY_PATH = (
    STAGE3_ROOT
    / "conclusion_stability.csv"
)

EXPECTED_ROWS = 1_374_133
EXPECTED_BENIGN = 998_788
EXPECTED_ATTACK = 375_345
EXPECTED_FEATURES = 70
EXPECTED_NAN_CELLS = 13_922

EXPECTED_X_SHA = (
    "50979ff283ddebaceb6442004c5b80b85e4fb40d02041a5150f730683b3d7c8e"
)

EXPECTED_Y_SHA = (
    "b99cf695a49ad2b0a8811fa269a55dcfb99cb700f473ceae8c9ecac2c8661a78"
)

EXPECTED_MEMBERSHIP_FILE_SHA = (
    "18d43eded5e78238ce6765abdc1ed18ce662aebd0899b678472891203eee3d1e"
)

EXPECTED_SEEDS = [
    42,
    43,
    44,
    45,
    46,
]

EXPECTED_UNITS = [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]

EXPECTED_VERSIONS = {
    "numpy": "2.0.2",
    "sklearn": "1.6.1",
    "xgboost": "3.2.0",
    "lightgbm": "4.6.0",
}

SOURCE_PROVENANCE = {
    8: {
        "file": "03-01-2018.csv",
        "bytes": 107_842_858,
        "sha256": "b0534c5d7d8b41e03df71c6966c995d116a8ed28e61f377c8b14cdf5d28f4edf",
        "physical_rows": 331_125,
        "embedded_header_rows": 25,
        "effective_rows": 331_100,
        "retained_rows": 331_017,
        "retained_benign": 237_982,
        "retained_attack": 93_035,
        "row_index_semantics": "RAW_PHYSICAL_ROW_INDEX",
        "positive_infinity_to_nan": 4_000,
        "negative_infinity_to_nan": 0,
    },
    9: {
        "file": "03-02-2018.csv",
        "bytes": 352_368_373,
        "sha256": "d96f38e7496aba83475031e6fb8c6fdf1abf6aa1b71325a917798f3c7de93de1",
        "physical_rows": 1_048_575,
        "embedded_header_rows": 0,
        "effective_rows": 1_048_575,
        "retained_rows": 1_043_116,
        "retained_benign": 760_806,
        "retained_attack": 282_310,
        "row_index_semantics": "RAW_EQUALS_EFFECTIVE_NO_EMBEDDED_HEADERS",
        "positive_infinity_to_nan": 5_530,
        "negative_infinity_to_nan": 0,
    },
}


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(text):
    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            + " ".join(map(str, cmd))
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return run(
        ["git", *args]
    ).stdout.strip()


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def write_json(
    path,
    obj,
):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(
    path,
    chunk=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(chunk)

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_array_raw(arr):
    arr = np.ascontiguousarray(
        arr
    )

    h = hashlib.sha256()

    view = memoryview(
        arr
    ).cast("B")

    step = 64 * 1024 * 1024

    for start in range(
        0,
        len(view),
        step,
    ):
        h.update(
            view[
                start:
                start + step
            ]
        )

    return h.hexdigest()


def safe_div(
    numerator,
    denominator,
):
    if denominator == 0:
        return 0.0

    return float(
        numerator
        / denominator
    )


def operating_metrics(
    y_true,
    probability,
    threshold,
):
    threshold32 = np.float32(
        threshold
    )

    pred = (
        probability
        >= threshold32
    )

    positive = (
        y_true == 1
    )

    negative = ~positive

    tp = int(
        np.count_nonzero(
            pred & positive
        )
    )

    fp = int(
        np.count_nonzero(
            pred & negative
        )
    )

    tn = int(
        np.count_nonzero(
            (~pred) & negative
        )
    )

    fn = int(
        np.count_nonzero(
            (~pred) & positive
        )
    )

    precision = safe_div(
        tp,
        tp + fp,
    )

    recall = safe_div(
        tp,
        tp + fn,
    )

    fpr = safe_div(
        fp,
        fp + tn,
    )

    accuracy = safe_div(
        tp + tn,
        len(y_true),
    )

    if (
        precision
        + recall
    ) > 0:
        f1 = (
            2.0
            * precision
            * recall
            / (
                precision
                + recall
            )
        )
    else:
        f1 = 0.0

    if (
        4.0
        * precision
        + recall
    ) > 0:
        f2 = (
            5.0
            * precision
            * recall
            / (
                4.0
                * precision
                + recall
            )
        )
    else:
        f2 = 0.0

    return {
        "threshold":
            float(
                threshold
            ),

        "threshold_float32_runtime":
            float(
                threshold32
            ),

        "accuracy":
            float(
                accuracy
            ),

        "precision":
            float(
                precision
            ),

        "recall":
            float(
                recall
            ),

        "fpr":
            float(
                fpr
            ),

        "f1":
            float(
                f1
            ),

        "f2":
            float(
                f2
            ),

        "tp":
            tp,

        "fp":
            fp,

        "tn":
            tn,

        "fn":
            fn,
    }


def extract_threshold(
    operating_points,
    name,
):
    op = operating_points[
        name.lower()
    ]

    if (
        isinstance(op, dict)
        and
        "result" in op
    ):
        if (
            op.get("status")
            != "AVAILABLE"
        ):
            raise RuntimeError(
                f"{name} operating point is not AVAILABLE."
            )

        op = op[
            "result"
        ]

    if not isinstance(
        op,
        dict,
    ):
        raise RuntimeError(
            f"Malformed {name} operating point."
        )

    if (
        "threshold"
        not in op
    ):
        raise RuntimeError(
            f"Frozen {name} threshold missing."
        )

    return float(
        op[
            "threshold"
        ]
    )


def resolve_model(
    result_path,
    model_info,
):
    # Historical / verbose schema:
    #   model_path
    #
    # AUTO-A compact schema:
    #   model
    #
    # Historical reuse schema:
    #   historical_model_path

    local_name = (
        model_info.get(
            "model_path"
        )
        or
        model_info.get(
            "model"
        )
    )

    if local_name:
        path = (
            result_path.parent
            / local_name
        )

        expected_sha = (
            model_info.get(
                "model_sha256"
            )
        )

        source_type = (
            "STAGE28_LOCAL_MODEL_ARTIFACT"
        )

    else:
        historical_name = (
            model_info.get(
                "historical_model_path"
            )
        )

        if not historical_name:
            raise RuntimeError(
                f"No model path field in:\n"
                f"{result_path}"
            )

        path = (
            REPO
            / historical_name
        )

        expected_sha = (
            model_info.get(
                "historical_model_sha256"
            )
        )

        source_type = (
            "HISTORICAL_REUSE_MODEL_ARTIFACT"
        )

    if not path.is_file():
        raise RuntimeError(
            f"Model missing:\n{path}"
        )

    if not expected_sha:
        raise RuntimeError(
            f"Expected model SHA missing:\n{result_path}"
        )

    actual_sha = sha256_file(
        path
    )

    if actual_sha != expected_sha:
        raise RuntimeError(
            "Model SHA mismatch:\n"
            f"{path}\n"
            f"expected={expected_sha}\n"
            f"actual={actual_sha}"
        )

    return {
        "path":
            path,

        "sha256":
            actual_sha,

        "source_type":
            source_type,
    }


def get_github_token():
    from kaggle_secrets import (
        UserSecretsClient,
    )

    client = UserSecretsClient()

    for label in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]:
        try:
            token = client.get_secret(
                label
            )
        except Exception:
            token = None

        if (
            isinstance(
                token,
                str,
            )
            and token.strip()
        ):
            return (
                token.strip(),
                label,
            )

    raise RuntimeError(
        "No GitHub token found."
    )


def push_origin_main():
    token, label = (
        get_github_token()
    )

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode(
            "utf-8"
        )
    ).decode(
        "ascii"
    )

    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            "Git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    print(
        f"[PASS] GitHub credential: "
        f"kaggle_secret:{label}"
    )

    print(
        "[PASS] token not displayed"
    )

    if p.stdout.strip():
        print(
            p.stdout.strip()
        )

    if p.stderr.strip():
        print(
            p.stderr.strip()
        )


def verify_remote_head(
    expected,
):
    run(
        [
            "git",
            "fetch",
            "origin",
            "main",
        ]
    )

    local = git(
        "rev-parse",
        "HEAD",
    )

    remote = git(
        "rev-parse",
        "origin/main",
    )

    if not (
        local
        == remote
        == expected
    ):
        raise RuntimeError(
            "Remote durability mismatch.\n"
            f"local={local}\n"
            f"origin={remote}\n"
            f"expected={expected}"
        )


# =================================================================================================
# 0. RECOVERY GATE — MUST USE IN-MEMORY HOLDOUT
# =================================================================================================

banner(
    "STAGE28-4-R1 — IN-MEMORY HOLDOUT SALVAGE GATE"
)

required_globals = [
    "X_holdout",
    "y_true",
]

missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "The materialized Stage28-4 holdout is no longer "
        "present in this kernel:\n"
        + "\n".join(
            missing_globals
        )
        + "\n\n"
        "STOP. Do not rerun the March CSV materialization "
        "without designing a cold-recovery receipt first."
    )

if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Repository is unexpectedly dirty before R1."
    )

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)

local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)

print(
    "Expected parent:",
    FAILED_STAGE28_4_PARENT,
)

print(
    "Local HEAD     :",
    local_head,
)

print(
    "origin/main    :",
    origin_head,
)

if not (
    local_head
    == origin_head
    == FAILED_STAGE28_4_PARENT
):
    raise RuntimeError(
        "Repository lineage changed after failed Stage28-4."
    )

if STAGE4_ROOT.exists():
    raise RuntimeError(
        "Stage28-4 output root already exists. "
        "Do not overwrite it."
    )

if COMBINED_STABILITY_PATH.exists():
    raise RuntimeError(
        "Final conclusion_stability.csv already exists."
    )

print()
print(
    "[PASS] same scientific parent"
)

print(
    "[PASS] repository clean"
)

print(
    "[PASS] no durable Stage28-4 output exists"
)

print(
    "[PASS] March source files will NOT be read by R1"
)


# =================================================================================================
# 1. VERIFY THE ALREADY-MATERIALIZED HOLDOUT EXACTLY
# =================================================================================================

banner(
    "STAGE28-4-R1 — VERIFY ALREADY-MATERIALIZED HOLDOUT"
)

if not isinstance(
    X_holdout,
    np.ndarray,
):
    raise RuntimeError(
        "X_holdout is not a numpy ndarray."
    )

if not isinstance(
    y_true,
    np.ndarray,
):
    raise RuntimeError(
        "y_true is not a numpy ndarray."
    )

if (
    X_holdout.shape
    != (
        EXPECTED_ROWS,
        EXPECTED_FEATURES,
    )
):
    raise RuntimeError(
        f"X_holdout shape mismatch: "
        f"{X_holdout.shape}"
    )

if X_holdout.dtype != np.float64:
    raise RuntimeError(
        f"X_holdout dtype mismatch: "
        f"{X_holdout.dtype}"
    )

if y_true.shape != (
    EXPECTED_ROWS,
):
    raise RuntimeError(
        f"y_true shape mismatch: "
        f"{y_true.shape}"
    )

if y_true.dtype != np.uint8:
    raise RuntimeError(
        f"y_true dtype mismatch: "
        f"{y_true.dtype}"
    )

benign = int(
    np.count_nonzero(
        y_true == 0
    )
)

attack = int(
    np.count_nonzero(
        y_true == 1
    )
)

if benign != EXPECTED_BENIGN:
    raise RuntimeError(
        f"Benign count mismatch: {benign}"
    )

if attack != EXPECTED_ATTACK:
    raise RuntimeError(
        f"Attack count mismatch: {attack}"
    )

nan_cells = int(
    np.count_nonzero(
        np.isnan(
            X_holdout
        )
    )
)

if nan_cells != EXPECTED_NAN_CELLS:
    raise RuntimeError(
        f"NaN-cell mismatch: "
        f"{nan_cells} != {EXPECTED_NAN_CELLS}"
    )

if np.any(
    np.isinf(
        X_holdout
    )
):
    raise RuntimeError(
        "X_holdout still contains +/-inf."
    )

print(
    "Recomputing in-memory logical hashes ..."
)

X_sha = sha256_array_raw(
    X_holdout
)

y_sha = sha256_array_raw(
    y_true
)

print(
    "X logical SHA256:",
    X_sha,
)

print(
    "y logical SHA256:",
    y_sha,
)

if X_sha != EXPECTED_X_SHA:
    raise RuntimeError(
        "In-memory X_holdout SHA mismatch."
    )

if y_sha != EXPECTED_Y_SHA:
    raise RuntimeError(
        "In-memory y_true SHA mismatch."
    )

print()
print(
    "[PASS] 1,374,133 × 70 float64 matrix exact"
)

print(
    "[PASS] benign = 998,788"
)

print(
    "[PASS] attack = 375,345"
)

print(
    "[PASS] NaN cells = 13,922"
)

print(
    "[PASS] no infinities remain"
)

print(
    "[PASS] X logical SHA exact"
)

print(
    "[PASS] y logical SHA exact"
)

print(
    "[PASS] NO RAW DATA RE-READ"
)


# =================================================================================================
# 2. SCIENTIFIC AUTHORIZATION / RUNTIME
# =================================================================================================

banner(
    "STAGE28-4-R1 — SCIENTIFIC AUTHORIZATION"
)

closure = read_json(
    CLOSURE_RECEIPT
)

stage3c = read_json(
    STAGE3C_RECEIPT
)

stage22_spec = read_json(
    STAGE22_SPEC
)

stability_spec = read_json(
    STABILITY_SPEC
)

if (
    closure[
        "closure_status"
    ]
    !=
    "PASS_STAGE28_NEW_MODEL_FITTING_PERMANENTLY_CLOSED"
):
    raise RuntimeError(
        "Stage28 fitting closure changed."
    )

if (
    int(
        closure[
            "fit_budget_closure"
        ][
            "consumed_new_fits"
        ]
    )
    != 108
):
    raise RuntimeError(
        "Consumed new fits != 108."
    )

if (
    int(
        closure[
            "fit_budget_closure"
        ][
            "remaining_new_fits"
        ]
    )
    != 0
):
    raise RuntimeError(
        "Remaining new fits != 0."
    )

if not str(
    stage3c.get(
        "next_authorized_step",
        "",
    )
).startswith(
    "Stage28-4"
):
    raise RuntimeError(
        "Stage28-4 is no longer authorized."
    )

if (
    stage22_spec[
        "scientific_unit"
    ][
        "strategy"
    ]
    != "ENS_LGBM_XGB_EQUAL"
):
    raise RuntimeError(
        "Frozen ensemble strategy changed."
    )

FROZEN_PROBABILITY_RULE = (
    stage22_spec[
        "scientific_unit"
    ][
        "probability_rule"
    ]
)

if (
    FROZEN_PROBABILITY_RULE
    !=
    "0.5 * P_LIGHTGBM + 0.5 * P_XGBOOST"
):
    raise RuntimeError(
        "Frozen probability rule changed."
    )

evaluation_population = (
    stage22_spec[
        "evaluation_population"
    ]
)

if (
    evaluation_population[
        "threshold_selection_on_final_holdout"
    ]
    != "FORBIDDEN"
):
    raise RuntimeError(
        "Final-holdout threshold-search rule changed."
    )

if (
    evaluation_population[
        "model_selection_on_final_holdout"
    ]
    != "FORBIDDEN"
):
    raise RuntimeError(
        "Final-holdout model-selection rule changed."
    )

versions = {
    "numpy":
        np.__version__,

    "sklearn":
        sklearn.__version__,

    "xgboost":
        xgb.__version__,

    "lightgbm":
        lgb.__version__,
}

for name, expected in (
    EXPECTED_VERSIONS.items()
):
    actual = versions[
        name
    ]

    print(
        f"{name:<10}: {actual}"
    )

    if actual != expected:
        raise RuntimeError(
            f"{name} version mismatch."
        )

print()
print(
    "[PASS] 108 / 108 fits remain permanently closed"
)

print(
    "[PASS] frozen equal-weight probability rule exact"
)

print(
    "[PASS] threshold search remains FORBIDDEN"
)

print(
    "[PASS] model selection remains FORBIDDEN"
)

print(
    "[PASS] runtime versions exact"
)


# =================================================================================================
# 3. PHASE A — DURABLY FREEZE THE ALREADY-CONSUMED MATERIALIZATION
# =================================================================================================

banner(
    "STAGE28-4-R1 — PHASE A: DURABLE MATERIALIZATION CHECKPOINT"
)

STAGE4_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

MATERIALIZATION_RECEIPT = (
    STAGE4_ROOT
    / "stage28_4_materialization_checkpoint.json"
)

MATERIALIZATION_README = (
    STAGE4_ROOT
    / "README_MATERIALIZATION_CHECKPOINT.md"
)

checkpoint = {
    "stage":
        "Stage28-4",

    "checkpoint":
        "MATERIALIZATION_COMPLETE_BEFORE_MODEL_INFERENCE",

    "created_at_utc":
        utc_now(),

    "scientific_parent_commit":
        FAILED_STAGE28_4_PARENT,

    "recovery_reason":
        (
            "Initial Stage28-4 execution successfully materialized "
            "the exact frozen shared final holdout, then stopped during "
            "Stage22 receipt-schema auditing before any model inference. "
            "R1 salvaged the already-materialized in-memory matrix without "
            "re-reading either March source file."
        ),

    "holdout": {
        "population":
            "STAGE22R_SHARED_FINAL_SINGLE_HOLDOUT",

        "rows":
            EXPECTED_ROWS,

        "features":
            EXPECTED_FEATURES,

        "dtype":
            "float64",

        "benign":
            EXPECTED_BENIGN,

        "attack":
            EXPECTED_ATTACK,

        "attack_prevalence":
            EXPECTED_ATTACK
            / EXPECTED_ROWS,

        "X_logical_sha256":
            X_sha,

        "y_logical_sha256":
            y_sha,

        "output_nan_cells":
            nan_cells,

        "positive_infinity_to_nan":
            9530,

        "negative_infinity_to_nan":
            0,
    },

    "source_materialization": {
        "day_8":
            SOURCE_PROVENANCE[
                8
            ],

        "day_9":
            SOURCE_PROVENANCE[
                9
            ],

        "raw_source_read_passes_in_initial_authorized_materialization":
            1,

        "raw_source_reads_performed_by_recovery_cell":
            0,
    },

    "scientific_operations_completed_before_checkpoint": {
        "new_model_fits":
            0,

        "model_inferences":
            0,

        "threshold_selections":
            0,

        "model_selections":
            0,

        "stage28_shared_final_holdout_materializations":
            1,
    },

    "status":
        "PASS_SHARED_FINAL_HOLDOUT_MATERIALIZATION_DURABLY_FROZEN",
}

write_json(
    MATERIALIZATION_RECEIPT,
    checkpoint,
)

MATERIALIZATION_README.write_text(
    f"""# Stage28-4 materialization checkpoint

The one authorized Stage28 shared-final-holdout materialization completed
successfully before model inference.

- Rows: {EXPECTED_ROWS:,}
- Features: {EXPECTED_FEATURES}
- Dtype: float64
- Benign: {EXPECTED_BENIGN:,}
- Attack: {EXPECTED_ATTACK:,}
- NaN cells: {EXPECTED_NAN_CELLS:,}
- X SHA256: `{X_sha}`
- y SHA256: `{y_sha}`

March source files were not re-read by the R1 recovery.

No model fit, inference, threshold selection, or model selection occurred
before this checkpoint.
""",
    encoding="utf-8",
)

expected_phase_a = {
    str(
        MATERIALIZATION_RECEIPT.relative_to(
            REPO
        )
    ),
    str(
        MATERIALIZATION_README.relative_to(
            REPO
        )
    ),
}

tracked = set(
    git(
        "diff",
        "--name-only",
    ).splitlines()
)

staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)

untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
)

if tracked:
    raise RuntimeError(
        "Unexpected tracked modifications before Phase A."
    )

if staged:
    raise RuntimeError(
        "Unexpected staged files before Phase A."
    )

if untracked != expected_phase_a:
    raise RuntimeError(
        "Unexpected Phase-A untracked universe.\n\n"
        f"Expected:\n{sorted(expected_phase_a)}\n\n"
        f"Actual:\n{sorted(untracked)}"
    )

for rel in sorted(
    expected_phase_a
):
    run(
        [
            "git",
            "add",
            "--",
            rel,
        ]
    )

run(
    [
        "git",
        "config",
        "user.name",
        "Stage28 Kaggle",
    ]
)

run(
    [
        "git",
        "config",
        "user.email",
        "stage28-kaggle@users.noreply.github.com",
    ]
)

phase_a_message = (
    "stage28-4a: freeze shared final holdout materialization"
)

print(
    run(
        [
            "git",
            "commit",
            "-m",
            phase_a_message,
        ]
    ).stdout.strip()
)

PHASE_A_COMMIT = git(
    "rev-parse",
    "HEAD",
)

if (
    git(
        "rev-parse",
        "HEAD^",
    )
    != FAILED_STAGE28_4_PARENT
):
    raise RuntimeError(
        "Phase-A checkpoint parent mismatch."
    )

push_origin_main()

verify_remote_head(
    PHASE_A_COMMIT
)

if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Repository dirty after Phase-A push."
    )

print()
print(
    "[PASS] materialization checkpoint durable"
)

print(
    "Phase-A commit:",
    PHASE_A_COMMIT,
)

print(
    "[PASS] scientific holdout opening is now durably recorded"
)


# =================================================================================================
# 4. AUDIT THE TEN STAGE22 RECEIPTS — SCHEMA TOLERANT, SCIENCE STRICT
# =================================================================================================

banner(
    "STAGE28-4-R1 — AUDIT TEN FROZEN STAGE22 ENSEMBLES"
)

result_files = sorted(
    STAGE22_ROOT.rglob(
        "*_result.json"
    )
)

if len(
    result_files
) != 10:
    raise RuntimeError(
        f"Expected 10 Stage22 results; "
        f"found {len(result_files)}."
    )

ensemble_specs = []

for result_path in result_files:
    obj = read_json(
        result_path
    )

    if (
        obj.get(
            "experiment"
        )
        != "STAGE22_FULL"
    ):
        raise RuntimeError(
            f"Unexpected Stage22 experiment:\n"
            f"{result_path}"
        )

    unit = obj[
        "unit"
    ]

    seed = int(
        obj[
            "training_seed"
        ]
    )

    if unit not in EXPECTED_UNITS:
        raise RuntimeError(
            f"Unexpected unit: {unit}"
        )

    if seed not in EXPECTED_SEEDS:
        raise RuntimeError(
            f"Unexpected seed: {seed}"
        )

    models = obj[
        "models"
    ]

    if not isinstance(
        models,
        dict,
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: models is not dict."
        )

    if (
        models.get(
            "strategy"
        )
        != "ENS_LGBM_XGB_EQUAL"
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: strategy mismatch."
        )

    # Receipt-schema compatibility:
    #
    # Older Stage28 receipts store this redundant literal.
    # AUTO-A compact receipts omit it.
    #
    # If present, it MUST match the frozen protocol.
    # If absent, the frozen Stage22 protocol is authoritative.
    receipt_probability_rule = (
        models.get(
            "ensemble_probability"
        )
    )

    if (
        receipt_probability_rule
        is not None
        and
        receipt_probability_rule
        != FROZEN_PROBABILITY_RULE
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: explicitly recorded "
            "probability rule disagrees with frozen protocol."
        )

    probability_rule_source = (
        "RESULT_RECEIPT_EXPLICIT"
        if receipt_probability_rule
        is not None
        else
        "FROZEN_STAGE22_PROTOCOL_FALLBACK"
    )

    if (
        models.get(
            "component_combination_dtype"
        )
        != "float64"
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: "
            "component combination dtype != float64."
        )

    if (
        models.get(
            "ensemble_storage_dtype"
        )
        != "float32"
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: "
            "ensemble storage dtype != float32."
        )

    xgb_info = models.get(
        "xgboost"
    )

    lgb_info = models.get(
        "lightgbm"
    )

    if not isinstance(
        xgb_info,
        dict,
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: XGBoost receipt missing."
        )

    if not isinstance(
        lgb_info,
        dict,
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: LightGBM receipt missing."
        )

    for learner_name, info in [
        (
            "XGBOOST",
            xgb_info,
        ),
        (
            "LIGHTGBM",
            lgb_info,
        ),
    ]:
        if int(
            info[
                "seed"
            ]
        ) != seed:
            raise RuntimeError(
                f"{unit}/seed{seed}/{learner_name}: "
                "seed mismatch."
            )

        if (
            str(
                info.get(
                    "backend",
                    "",
                )
            ).lower()
            != "cpu"
        ):
            raise RuntimeError(
                f"{unit}/seed{seed}/{learner_name}: "
                "backend != CPU."
            )

    xgb_model = resolve_model(
        result_path,
        xgb_info,
    )

    lgb_model = resolve_model(
        result_path,
        lgb_info,
    )

    thresholds = {
        name:
            extract_threshold(
                obj[
                    "operating_points"
                ],
                name,
            )
        for name in [
            "STANDARD",
            "BALANCED",
            "SECURITY",
        ]
    }

    ensemble_specs.append(
        {
            "unit":
                unit,

            "seed":
                seed,

            "evaluation_cell_id":
                obj.get(
                    "evaluation_cell_id",
                    (
                        f"28A_STAGE22::{unit}::SEED{seed}"
                    ),
                ),

            "result_path":
                result_path,

            "result_sha256":
                sha256_file(
                    result_path
                ),

            "probability_rule":
                FROZEN_PROBABILITY_RULE,

            "probability_rule_source":
                probability_rule_source,

            "xgb_component_id":
                xgb_info[
                    "component_id"
                ],

            "xgb_model":
                xgb_model,

            "lgb_component_id":
                lgb_info[
                    "component_id"
                ],

            "lgb_model":
                lgb_model,

            "thresholds":
                thresholds,
        }
    )

ensemble_specs.sort(
    key=lambda x: (
        EXPECTED_UNITS.index(
            x[
                "unit"
            ]
        ),
        x[
            "seed"
        ],
    )
)

expected_pairs = [
    (
        unit,
        seed,
    )
    for unit in EXPECTED_UNITS
    for seed in EXPECTED_SEEDS
]

actual_pairs = [
    (
        spec[
            "unit"
        ],
        spec[
            "seed"
        ],
    )
    for spec in ensemble_specs
]

if actual_pairs != expected_pairs:
    raise RuntimeError(
        "Stage22 evaluation grid is not exact 2 × 5."
    )

for spec in ensemble_specs:
    print(
        f"{spec['unit']:<24} "
        f"seed={spec['seed']}  "
        f"XGB={spec['xgb_component_id']}  "
        f"LGB={spec['lgb_component_id']}  "
        f"rule_source={spec['probability_rule_source']}  "
        f"thresholds={spec['thresholds']}"
    )

print()
print(
    "[PASS] 10 / 10 Stage22 ensemble receipts"
)

print(
    "[PASS] compact AUTO-A omission handled from frozen protocol"
)

print(
    "[PASS] model_path / model / historical_model_path schemas supported"
)

print(
    "[PASS] all 20 model SHA256 identities exact"
)

print(
    "[PASS] all model backends = CPU"
)

print(
    "[PASS] thresholds inherited only from development validation"
)


# =================================================================================================
# 5. PHASE B — AUTHORIZED FROZEN MODEL INFERENCE
# =================================================================================================

banner(
    "STAGE28-4-R1 — PHASE B: AUTHORIZED FINAL-HOLDOUT INFERENCE"
)

probability_arrays = {}

probability_records = {}

metric_rows = []

inference_started = (
    time.perf_counter()
)

for ordinal, spec in enumerate(
    ensemble_specs,
    start=1,
):
    unit = spec[
        "unit"
    ]

    seed = spec[
        "seed"
    ]

    print()
    print(
        "-" * 120
    )

    print(
        f"[{ordinal:02d}/10] "
        f"{unit} — seed {seed}"
    )

    print(
        "-" * 120
    )

    cell_started = (
        time.perf_counter()
    )

    # ---------------------------------------------------------------------------------------------
    # XGBOOST — LOAD ONLY
    # ---------------------------------------------------------------------------------------------

    xgb_model = (
        xgb.XGBClassifier()
    )

    xgb_model.load_model(
        str(
            spec[
                "xgb_model"
            ][
                "path"
            ]
        )
    )

    rounds = int(
        xgb_model
        .get_booster()
        .num_boosted_rounds()
    )

    if rounds != 400:
        raise RuntimeError(
            f"{unit}/seed{seed}: "
            f"XGBoost rounds={rounds}, expected 400."
        )

    started = (
        time.perf_counter()
    )

    p_xgb = (
        xgb_model.predict_proba(
            X_holdout
        )[
            :,
            1
        ]
    )

    xgb_seconds = (
        time.perf_counter()
        - started
    )

    if p_xgb.shape != (
        EXPECTED_ROWS,
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: "
            "XGBoost probability shape mismatch."
        )

    # ---------------------------------------------------------------------------------------------
    # LIGHTGBM — LOAD ONLY
    # ---------------------------------------------------------------------------------------------

    lgb_model = lgb.Booster(
        model_file=str(
            spec[
                "lgb_model"
            ][
                "path"
            ]
        )
    )

    iterations = int(
        lgb_model.current_iteration()
    )

    if iterations != 400:
        raise RuntimeError(
            f"{unit}/seed{seed}: "
            f"LightGBM iterations={iterations}, expected 400."
        )

    started = (
        time.perf_counter()
    )

    p_lgb = lgb_model.predict(
        X_holdout,
        num_iteration=iterations,
    )

    lgb_seconds = (
        time.perf_counter()
        - started
    )

    if p_lgb.shape != (
        EXPECTED_ROWS,
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: "
            "LightGBM probability shape mismatch."
        )

    # ---------------------------------------------------------------------------------------------
    # FROZEN EQUAL ENSEMBLE
    # ---------------------------------------------------------------------------------------------

    p_ensemble = (
        0.5
        * np.asarray(
            p_lgb,
            dtype=np.float64,
        )
        +
        0.5
        * np.asarray(
            p_xgb,
            dtype=np.float64,
        )
    ).astype(
        np.float32,
        copy=False,
    )

    if p_ensemble.dtype != np.float32:
        raise RuntimeError(
            "Ensemble dtype != float32."
        )

    if not np.all(
        np.isfinite(
            p_ensemble
        )
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: "
            "non-finite ensemble probabilities."
        )

    if (
        float(
            p_ensemble.min()
        ) < 0.0
        or
        float(
            p_ensemble.max()
        ) > 1.0
    ):
        raise RuntimeError(
            f"{unit}/seed{seed}: "
            "probabilities outside [0,1]."
        )

    roc_auc = float(
        roc_auc_score(
            y_true,
            p_ensemble,
        )
    )

    pr_auc = float(
        average_precision_score(
            y_true,
            p_ensemble,
        )
    )

    op_results = {
        name:
            operating_metrics(
                y_true,
                p_ensemble,
                spec[
                    "thresholds"
                ][
                    name
                ],
            )
        for name in [
            "STANDARD",
            "BALANCED",
            "SECURITY",
        ]
    }

    key = (
        unit.lower()
        + "_seed"
        + str(seed)
    )

    probability_arrays[
        key
    ] = p_ensemble.copy()

    probability_sha = (
        sha256_array_raw(
            p_ensemble
        )
    )

    cell_seconds = (
        time.perf_counter()
        - cell_started
    )

    probability_records[
        key
    ] = {
        "unit":
            unit,

        "seed":
            seed,

        "rows":
            EXPECTED_ROWS,

        "dtype":
            "float32",

        "logical_sha256":
            probability_sha,

        "minimum":
            float(
                p_ensemble.min()
            ),

        "maximum":
            float(
                p_ensemble.max()
            ),

        "xgboost_inference_seconds":
            float(
                xgb_seconds
            ),

        "lightgbm_inference_seconds":
            float(
                lgb_seconds
            ),

        "cell_total_seconds":
            float(
                cell_seconds
            ),
    }

    row = {
        "unit":
            unit,

        "seed":
            seed,

        "shared_holdout_rows":
            EXPECTED_ROWS,

        "shared_holdout_benign":
            EXPECTED_BENIGN,

        "shared_holdout_attack":
            EXPECTED_ATTACK,

        "shared_holdout_attack_prevalence":
            EXPECTED_ATTACK
            / EXPECTED_ROWS,

        "roc_auc":
            roc_auc,

        "pr_auc":
            pr_auc,

        "ensemble_probability_sha256":
            probability_sha,

        "xgboost_component_id":
            spec[
                "xgb_component_id"
            ],

        "xgboost_model_sha256":
            spec[
                "xgb_model"
            ][
                "sha256"
            ],

        "lightgbm_component_id":
            spec[
                "lgb_component_id"
            ],

        "lightgbm_model_sha256":
            spec[
                "lgb_model"
            ][
                "sha256"
            ],

        "probability_rule_source":
            spec[
                "probability_rule_source"
            ],

        "inference_seconds_xgboost":
            float(
                xgb_seconds
            ),

        "inference_seconds_lightgbm":
            float(
                lgb_seconds
            ),

        "inference_seconds_cell_total":
            float(
                cell_seconds
            ),
    }

    for op_name in [
        "STANDARD",
        "BALANCED",
        "SECURITY",
    ]:
        prefix = (
            op_name.lower()
        )

        for field, value in (
            op_results[
                op_name
            ].items()
        ):
            row[
                f"{prefix}_{field}"
            ] = value

    metric_rows.append(
        row
    )

    print(
        f"ROC-AUC : {roc_auc:.12f}"
    )

    print(
        f"PR-AUC  : {pr_auc:.12f}"
    )

    print(
        f"STANDARD recall="
        f"{op_results['STANDARD']['recall']:.12f} "
        f"fpr={op_results['STANDARD']['fpr']:.12f}"
    )

    print(
        f"BALANCED recall="
        f"{op_results['BALANCED']['recall']:.12f} "
        f"fpr={op_results['BALANCED']['fpr']:.12f}"
    )

    print(
        f"SECURITY recall="
        f"{op_results['SECURITY']['recall']:.12f} "
        f"fpr={op_results['SECURITY']['fpr']:.12f}"
    )

    print(
        "[PASS] probability SHA256:",
        probability_sha,
    )

    del p_xgb
    del p_lgb
    del p_ensemble
    del xgb_model
    del lgb_model

    gc.collect()

total_inference_seconds = (
    time.perf_counter()
    - inference_started
)

if len(
    metric_rows
) != 10:
    raise RuntimeError(
        "Expected exactly ten ensemble evaluations."
    )

print()
print(
    "[PASS] component model inferences = 20"
)

print(
    "[PASS] ensemble evaluations = 10"
)

print(
    "[PASS] model fits = 0"
)

print(
    "[PASS] threshold searches = 0"
)

print(
    "Inference wall time:",
    total_inference_seconds,
)


# =================================================================================================
# 6. WRITE SEED-LEVEL METRICS / PROBABILITY ARTIFACT
# =================================================================================================

banner(
    "STAGE28-4-R1 — WRITE INFERENCE ARTIFACTS"
)

METRICS_PATH = (
    STAGE4_ROOT
    / "stage28_4_seed_level_metrics.csv"
)

PROBABILITIES_PATH = (
    STAGE4_ROOT
    / "stage28_4_shared_holdout_ensemble_probabilities.npz"
)

metrics_df = pd.DataFrame(
    metric_rows
)

unit_order = {
    "RANDOM_NATURAL": 0,
    "CHRONOLOGICAL_NATURAL": 1,
}

metrics_df[
    "_unit_order"
] = metrics_df[
    "unit"
].map(
    unit_order
)

metrics_df = (
    metrics_df.sort_values(
        [
            "_unit_order",
            "seed",
        ]
    )
    .drop(
        columns=[
            "_unit_order",
        ]
    )
    .reset_index(
        drop=True
    )
)

metrics_df.to_csv(
    METRICS_PATH,
    index=False,
)

np.savez_compressed(
    PROBABILITIES_PATH,
    **probability_arrays,
)

probability_file_sha = (
    sha256_file(
        PROBABILITIES_PATH
    )
)

print(
    "[PASS] seed-level metrics written"
)

print(
    "[PASS] 10 probability arrays written"
)

print(
    "Probability artifact SHA256:",
    probability_file_sha,
)


# =================================================================================================
# 7. FROZEN STAGE22 DIRECTIONAL CLAIMS
# =================================================================================================

banner(
    "STAGE28-4-R1 — FROZEN STAGE22 CONCLUSION STABILITY"
)

lookup = {
    (
        row[
            "unit"
        ],
        int(
            row[
                "seed"
            ]
        ),
    ):
        row
    for row in metric_rows
}

claim_rows = []

contrast_rows = []

for seed in EXPECTED_SEEDS:
    random_row = lookup[
        (
            "RANDOM_NATURAL",
            seed,
        )
    ]

    chrono_row = lookup[
        (
            "CHRONOLOGICAL_NATURAL",
            seed,
        )
    ]

    pr_random = float(
        random_row[
            "pr_auc"
        ]
    )

    pr_chrono = float(
        chrono_row[
            "pr_auc"
        ]
    )

    roc_random = float(
        random_row[
            "roc_auc"
        ]
    )

    roc_chrono = float(
        chrono_row[
            "roc_auc"
        ]
    )

    pr_condition = (
        pr_random
        < pr_chrono
    )

    roc_condition = (
        roc_random
        < roc_chrono
    )

    contrast_rows.append(
        {
            "seed":
                seed,

            "pr_auc_random":
                pr_random,

            "pr_auc_chronological":
                pr_chrono,

            "pr_auc_random_minus_chronological":
                pr_random
                - pr_chrono,

            "pr_random_lt_chronological":
                bool(
                    pr_condition
                ),

            "roc_auc_random":
                roc_random,

            "roc_auc_chronological":
                roc_chrono,

            "roc_auc_random_minus_chronological":
                roc_random
                - roc_chrono,

            "roc_random_lt_chronological":
                bool(
                    roc_condition
                ),
        }
    )

    claim_rows.append(
        {
            "claim_id":
                "STAGE22_PR_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT",

            "parent_stage":
                "STAGE22_FULL",

            "family_if_applicable":
                "",

            "learner_if_applicable":
                "",

            "seed":
                seed,

            "claim_condition":
                "PR_AUC_RANDOM_NATURAL < PR_AUC_CHRONOLOGICAL_NATURAL",

            "condition_met":
                bool(
                    pr_condition
                ),

            "analysis_status":
                "DESCRIPTIVE_ROBUSTNESS_PRE_REGISTERED",
        }
    )

    claim_rows.append(
        {
            "claim_id":
                "STAGE22_ROC_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT",

            "parent_stage":
                "STAGE22_FULL",

            "family_if_applicable":
                "",

            "learner_if_applicable":
                "",

            "seed":
                seed,

            "claim_condition":
                "ROC_AUC_RANDOM_NATURAL < ROC_AUC_CHRONOLOGICAL_NATURAL",

            "condition_met":
                bool(
                    roc_condition
                ),

            "analysis_status":
                "DESCRIPTIVE_ROBUSTNESS_PRE_REGISTERED",
        }
    )

contrast_df = pd.DataFrame(
    contrast_rows
)

claim_df = pd.DataFrame(
    claim_rows
)

if len(
    claim_df
) != 10:
    raise RuntimeError(
        "Stage22 claim realization count != 10."
    )

CONTRAST_PATH = (
    STAGE4_ROOT
    / "stage28_4_random_vs_chronological_seedwise.csv"
)

CLAIM_PATH = (
    STAGE4_ROOT
    / "stage28_4_stage22_directional_claims.csv"
)

contrast_df.to_csv(
    CONTRAST_PATH,
    index=False,
)

claim_df.to_csv(
    CLAIM_PATH,
    index=False,
)

summary_rows = []

for claim_id in [
    "STAGE22_PR_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT",
    "STAGE22_ROC_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT",
]:
    subset = claim_df.loc[
        claim_df[
            "claim_id"
        ]
        == claim_id
    ]

    if len(
        subset
    ) != 5:
        raise RuntimeError(
            f"{claim_id}: denominator != 5."
        )

    supporting = int(
        subset[
            "condition_met"
        ].sum()
    )

    summary_rows.append(
        {
            "claim_id":
                claim_id,

            "supporting_seeds":
                supporting,

            "total_frozen_seeds":
                5,

            "stability_rate":
                supporting
                / 5.0,

            "interpretation":
                "DESCRIPTIVE_ROBUSTNESS_NOT_NEW_SIGNIFICANCE_TEST",
        }
    )

stability_df = pd.DataFrame(
    summary_rows
)

STAGE22_STABILITY_SUMMARY = (
    STAGE4_ROOT
    / "stage28_4_stage22_directional_stability_summary.csv"
)

stability_df.to_csv(
    STAGE22_STABILITY_SUMMARY,
    index=False,
)

for _, row in contrast_df.iterrows():
    print(
        f"seed {int(row['seed'])}: "
        f"ΔPR(random-chrono)="
        f"{row['pr_auc_random_minus_chronological']:+.12f} "
        f"support={bool(row['pr_random_lt_chronological'])}; "
        f"ΔROC(random-chrono)="
        f"{row['roc_auc_random_minus_chronological']:+.12f} "
        f"support={bool(row['roc_random_lt_chronological'])}"
    )

print()

for _, row in stability_df.iterrows():
    print(
        f"{row['claim_id']}: "
        f"{int(row['supporting_seeds'])}/5 "
        f"= {float(row['stability_rate']):.3f}"
    )

print()
print(
    "[PASS] exactly the two frozen Stage22 conditions evaluated"
)

print(
    "[PASS] post-result condition creation = 0"
)

print(
    "[PASS] new significance testing = 0"
)


# =================================================================================================
# 8. FINAL FROZEN conclusion_stability.csv
# =================================================================================================

banner(
    "STAGE28-4-R1 — FINAL conclusion_stability.csv"
)

loao_df = pd.read_csv(
    LOAO_STABILITY_PATH
)

required_columns = [
    "claim_id",
    "parent_stage",
    "family_if_applicable",
    "learner_if_applicable",
    "seed",
    "claim_condition",
    "condition_met",
    "analysis_status",
]

if list(
    loao_df.columns
) != required_columns:
    raise RuntimeError(
        "Stage28-3B LOAO stability schema changed."
    )

if list(
    claim_df.columns
) != required_columns:
    raise RuntimeError(
        "Stage28-4 claim schema mismatch."
    )

combined_df = pd.concat(
    [
        loao_df,
        claim_df,
    ],
    ignore_index=True,
)

combined_df.to_csv(
    COMBINED_STABILITY_PATH,
    index=False,
)

print(
    "Stage28-3B LOAO rows :",
    len(
        loao_df
    ),
)

print(
    "Stage28-4 Stage22 rows:",
    len(
        claim_df
    ),
)

print(
    "Combined rows          :",
    len(
        combined_df
    ),
)

print()
print(
    "[PASS] frozen required conclusion_stability.csv complete"
)


# =================================================================================================
# 9. FINAL STAGE28-4 RECEIPT
# =================================================================================================

banner(
    "STAGE28-4-R1 — FINAL RECEIPT"
)

FINAL_RECEIPT = (
    STAGE4_ROOT
    / "stage28_4_receipt.json"
)

README_PATH = (
    STAGE4_ROOT
    / "README_STAGE28_4.md"
)

stability_map = {
    row[
        "claim_id"
    ]:
        {
            "supporting":
                int(
                    row[
                        "supporting_seeds"
                    ]
                ),

            "total":
                5,

            "rate":
                float(
                    row[
                        "stability_rate"
                    ]
                ),
        }
    for _, row
    in stability_df.iterrows()
}

receipt = {
    "stage":
        "Stage28-4",

    "type":
        "STAGE22_FIVE_SEED_SHARED_FINAL_HOLDOUT_ROBUSTNESS_INFERENCE",

    "created_at_utc":
        utc_now(),

    "scientific_lineage": {
        "stage28_3c_parent":
            FAILED_STAGE28_4_PARENT,

        "materialization_checkpoint_commit":
            PHASE_A_COMMIT,
    },

    "recovery": {
        "status":
            "RECOVERED_AFTER_POST_MATERIALIZATION_PRE_INFERENCE_SCHEMA_FAILURE",

        "raw_source_files_re_read":
            False,

        "additional_holdout_materialization":
            False,

        "model_inference_before_failure":
            0,

        "schema_repairs": [
            (
                "models.ensemble_probability may be omitted in "
                "compact AUTO-A receipts; frozen Stage22 protocol "
                "is authoritative."
            ),
            (
                "Stage28 local model path may be stored under "
                "model_path or model."
            ),
        ],
    },

    "holdout": {
        "name":
            "STAGE22R_SHARED_FINAL_SINGLE_HOLDOUT",

        "rows":
            EXPECTED_ROWS,

        "benign":
            EXPECTED_BENIGN,

        "attack":
            EXPECTED_ATTACK,

        "features":
            EXPECTED_FEATURES,

        "dtype":
            "float64",

        "X_logical_sha256":
            X_sha,

        "y_logical_sha256":
            y_sha,

        "blind_status":
            (
                "NOT_NEW_BLIND_HOLDOUT; "
                "PARENT_STAGE22R_ALREADY_HISTORICALLY_OPENED"
            ),
    },

    "scientific_unit": {
        "strategy":
            "ENS_LGBM_XGB_EQUAL",

        "probability_rule":
            FROZEN_PROBABILITY_RULE,

        "component_combination_dtype":
            "float64",

        "ensemble_storage_dtype":
            "float32",
    },

    "design": {
        "units":
            EXPECTED_UNITS,

        "seeds":
            EXPECTED_SEEDS,

        "ensemble_realizations":
            10,

        "component_model_inferences":
            20,
    },

    "threshold_policy": {
        "selection_population":
            "FROZEN_DEVELOPMENT_VALIDATION_ONLY",

        "final_holdout_threshold_search":
            "FORBIDDEN_AND_NOT_PERFORMED",

        "model_selection_on_final_holdout":
            "FORBIDDEN_AND_NOT_PERFORMED",
    },

    "probabilities": {
        "artifact":
            str(
                PROBABILITIES_PATH.relative_to(
                    REPO
                )
            ),

        "artifact_sha256":
            probability_file_sha,

        "arrays":
            probability_records,
    },

    "seed_level_metrics": {
        "path":
            str(
                METRICS_PATH.relative_to(
                    REPO
                )
            ),

        "rows":
            10,
    },

    "stage22_directional_stability": {
        "claim_path":
            str(
                CLAIM_PATH.relative_to(
                    REPO
                )
            ),

        "summary_path":
            str(
                STAGE22_STABILITY_SUMMARY.relative_to(
                    REPO
                )
            ),

        "claims":
            stability_map,

        "stability_denominator":
            5,

        "post_result_condition_creation":
            0,

        "new_significance_tests":
            0,
    },

    "combined_conclusion_stability": {
        "path":
            str(
                COMBINED_STABILITY_PATH.relative_to(
                    REPO
                )
            ),

        "loao_rows":
            int(
                len(
                    loao_df
                )
            ),

        "stage22_rows":
            10,

        "combined_rows":
            int(
                len(
                    combined_df
                )
            ),
    },

    "scientific_operations": {
        "new_model_fits":
            0,

        "component_model_inferences":
            20,

        "ensemble_evaluations":
            10,

        "threshold_selections":
            0,

        "model_selections":
            0,

        "new_formal_statistical_tests":
            0,

        "shap_recomputation":
            0,

        "subset_search":
            0,

        "new_holdout_creation":
            0,

        "stage28_shared_final_holdout_materializations":
            1,
    },

    "fit_budget": {
        "authorized":
            108,

        "consumed":
            108,

        "remaining":
            0,
    },

    "status":
        "STAGE28_4_COMPLETE",

    "next_authorized_step":
        (
            "ZERO_FIT_FINAL_SYNTHESIS_AND_MANUSCRIPT_INTEGRATION; "
            "NO_FURTHER_MODEL_FITTING; NO_STAGE29"
        ),
}

write_json(
    FINAL_RECEIPT,
    receipt,
)

pr_result = stability_map[
    "STAGE22_PR_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT"
]

roc_result = stability_map[
    "STAGE22_ROC_RANDOM_LT_CHRONO_ON_SHARED_FINAL_HOLDOUT"
]

README_PATH.write_text(
    f"""# Stage28-4 — Stage22 shared-final-holdout robustness inference

## Scientific status

- New model fits: 0
- Frozen component-model inferences: 20
- Ensemble evaluations: 10
- Threshold selections: 0
- Model selections: 0
- New formal significance tests: 0
- Stage28 fit ledger: 108 / 108 consumed
- Remaining new fits: 0

## Holdout

- Rows: {EXPECTED_ROWS:,}
- Benign: {EXPECTED_BENIGN:,}
- Attack: {EXPECTED_ATTACK:,}
- Features: {EXPECTED_FEATURES}
- dtype: float64
- X SHA256: `{X_sha}`
- y SHA256: `{y_sha}`

The holdout had already been opened historically by Stage22R.
Stage28-4 is preregistered robustness re-evaluation, not a new blind test.

## Recovery

The initial Stage28-4 execution successfully materialized the holdout but
stopped before model inference because a compact AUTO-A result receipt omitted
the redundant `models.ensemble_probability` field.

R1 reused the already-materialized in-memory holdout and did not reread the
March source files.

## Frozen directional stability

PR claim:

`PR_AUC_RANDOM_NATURAL < PR_AUC_CHRONOLOGICAL_NATURAL`

Support: {pr_result['supporting']} / 5
Stability rate: {pr_result['rate']:.6f}

ROC claim:

`ROC_AUC_RANDOM_NATURAL < ROC_AUC_CHRONOLOGICAL_NATURAL`

Support: {roc_result['supporting']} / 5
Stability rate: {roc_result['rate']:.6f}

## Closure

Stage28 empirical work is complete after this result.

No Stage29 is authorized.
The remaining work is zero-fit synthesis and manuscript integration.
""",
    encoding="utf-8",
)


# =================================================================================================
# 10. CHECKSUMS
# =================================================================================================

CHECKSUM_PATH = (
    STAGE4_ROOT
    / "stage28_4_checksums.sha256"
)

artifact_paths = [
    path
    for path in STAGE4_ROOT.rglob(
        "*"
    )
    if (
        path.is_file()
        and path
        != CHECKSUM_PATH
    )
]

artifact_paths.append(
    COMBINED_STABILITY_PATH
)

artifact_paths = sorted(
    artifact_paths,
    key=lambda x:
        str(
            x
        ),
)

checksum_lines = []

for path in artifact_paths:
    checksum_lines.append(
        sha256_file(
            path
        )
        + "  "
        + str(
            path.relative_to(
                REPO
            )
        )
    )

CHECKSUM_PATH.write_text(
    "\n".join(
        checksum_lines
    )
    + "\n",
    encoding="utf-8",
)

print(
    "[PASS] Stage28-4 receipt written"
)

print(
    "[PASS] README written"
)

print(
    "[PASS] checksums written"
)


# =================================================================================================
# 11. FINAL SCIENTIFIC GATE
# =================================================================================================

banner(
    "STAGE28-4-R1 — FINAL SCIENTIFIC GATE"
)

closure_after = read_json(
    CLOSURE_RECEIPT
)

if closure_after != closure:
    raise RuntimeError(
        "Stage28 closure receipt unexpectedly changed."
    )

if (
    closure_after[
        "fit_budget_closure"
    ][
        "consumed_new_fits"
    ]
    != 108
):
    raise RuntimeError(
        "Fit ledger no longer equals 108."
    )

if (
    closure_after[
        "fit_budget_closure"
    ][
        "remaining_new_fits"
    ]
    != 0
):
    raise RuntimeError(
        "Fit ledger remaining != 0."
    )

if len(
    probability_arrays
) != 10:
    raise RuntimeError(
        "Probability array count != 10."
    )

if len(
    metrics_df
) != 10:
    raise RuntimeError(
        "Metric realization count != 10."
    )

if len(
    claim_df
) != 10:
    raise RuntimeError(
        "Stage22 claim realization count != 10."
    )

print(
    "[PASS] new model fits = 0"
)

print(
    "[PASS] Stage28 fit ledger = 108 / 108"
)

print(
    "[PASS] remaining fit budget = 0"
)

print(
    "[PASS] component model inferences = 20"
)

print(
    "[PASS] ensemble evaluations = 10"
)

print(
    "[PASS] threshold selections = 0"
)

print(
    "[PASS] model selections = 0"
)

print(
    "[PASS] new significance tests = 0"
)

print(
    "[PASS] shared holdout materializations = 1"
)


# =================================================================================================
# 12. PHASE B EXACT GIT UNIVERSE
# =================================================================================================

banner(
    "STAGE28-4-R1 — PHASE B GIT UNIVERSE"
)

# Phase-A files are already tracked and unchanged.
tracked_modifications = set(
    git(
        "diff",
        "--name-only",
    ).splitlines()
)

staged_before = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)

untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
)

expected_untracked = {
    str(
        path.relative_to(
            REPO
        )
    )
    for path in [
        METRICS_PATH,
        PROBABILITIES_PATH,
        CONTRAST_PATH,
        CLAIM_PATH,
        STAGE22_STABILITY_SUMMARY,
        FINAL_RECEIPT,
        README_PATH,
        CHECKSUM_PATH,
        COMBINED_STABILITY_PATH,
    ]
}

if tracked_modifications:
    raise RuntimeError(
        "Unexpected tracked modifications before Phase B:\n"
        + "\n".join(
            sorted(
                tracked_modifications
            )
        )
    )

if staged_before:
    raise RuntimeError(
        "Unexpected staged files before Phase B."
    )

if untracked != expected_untracked:
    raise RuntimeError(
        "Unexpected Phase-B untracked universe.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                expected_untracked
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                untracked
            )
        )
    )

print(
    "[PASS] exact Phase-B artifact universe"
)


# =================================================================================================
# 13. FINAL DURABLE COMMIT / PUSH
# =================================================================================================

banner(
    "STAGE28-4-R1 — FINAL DURABLE COMMIT / PUSH"
)

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)

if (
    git(
        "rev-parse",
        "origin/main",
    )
    != PHASE_A_COMMIT
):
    raise RuntimeError(
        "origin/main changed after Phase-A checkpoint."
    )

for rel in sorted(
    expected_untracked
):
    run(
        [
            "git",
            "add",
            "-f",
            "--",
            rel,
        ]
    )

staged_after = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)

if staged_after != expected_untracked:
    raise RuntimeError(
        "Phase-B staged universe mismatch."
    )

final_message = (
    "stage28-4b: complete Stage22 five-seed shared holdout inference"
)

print(
    run(
        [
            "git",
            "commit",
            "-m",
            final_message,
        ]
    ).stdout.strip()
)

FINAL_COMMIT = git(
    "rev-parse",
    "HEAD",
)

if (
    git(
        "rev-parse",
        "HEAD^",
    )
    != PHASE_A_COMMIT
):
    raise RuntimeError(
        "Final Stage28-4 commit parent mismatch."
    )

push_origin_main()

verify_remote_head(
    FINAL_COMMIT
)

if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Repository dirty after Stage28-4 final push."
    )

print()
print(
    "[PASS] Stage28-4 final commit durable"
)

print(
    "[PASS] repository clean"
)


# =================================================================================================
# 14. COMPLETE
# =================================================================================================

banner(
    "STAGE28-4 — COMPLETE"
)

print(
    "Materialization checkpoint:",
    PHASE_A_COMMIT,
)

print(
    "Final Stage28-4 commit      :",
    FINAL_COMMIT,
)

print()
print(
    "New model fits                    : 0"
)

print(
    "Component model inferences        : 20"
)

print(
    "Ensemble evaluation cells         : 10"
)

print(
    "Threshold selections              : 0"
)

print(
    "Model selections                  : 0"
)

print(
    "New formal statistical tests      : 0"
)

print(
    "Raw March re-reads during recovery: 0"
)

print(
    "Shared holdout materializations   : 1"
)

print()
print(
    "STAGE22 DIRECTIONAL STABILITY"
)

print(
    "-----------------------------"
)

for _, row in stability_df.iterrows():
    print(
        f"{row['claim_id']}: "
        f"{int(row['supporting_seeds'])}/5 "
        f"= {float(row['stability_rate']):.3f}"
    )

print()
print(
    "Stage28 new-fit budget             : 108 / 108"
)

print(
    "New fits remaining                 : 0"
)

print(
    "Stage28 empirical work             : COMPLETE"
)

print(
    "Stage29                           : NOT AUTHORIZED"
)

print()
print(
    "NEXT AUTHORIZED WORK:"
)

print(
    "ZERO-FIT FINAL SYNTHESIS + MANUSCRIPT INTEGRATION"
)


STAGE28-4-R1 — IN-MEMORY HOLDOUT SALVAGE GATE

Expected parent: 2679d0c208d514b381caa12e96c959f4f2ee5ee7
Local HEAD     : 2679d0c208d514b381caa12e96c959f4f2ee5ee7
origin/main    : 2679d0c208d514b381caa12e96c959f4f2ee5ee7

[PASS] same scientific parent
[PASS] repository clean
[PASS] no durable Stage28-4 output exists
[PASS] March source files will NOT be read by R1

STAGE28-4-R1 — VERIFY ALREADY-MATERIALIZED HOLDOUT

Recomputing in-memory logical hashes ...
X logical SHA256: 50979ff283ddebaceb6442004c5b80b85e4fb40d02041a5150f730683b3d7c8e
y logical SHA256: b99cf695a49ad2b0a8811fa269a55dcfb99cb700f473ceae8c9ecac2c8661a78

[PASS] 1,374,133 × 70 float64 matrix exact
[PASS] benign = 998,788
[PASS] attack = 375,345
[PASS] NaN cells = 13,922
[PASS] no infinities remain
[PASS] X logical SHA exact
[PASS] y logical SHA exact
[PASS] NO RAW DATA RE-READ

STAGE28-4-R1 — SCIENTIFIC AUTHORIZATION

numpy     : 2.0.2
sklearn   : 1.6.1
xgboost   : 3.2.0
lightgbm  : 4.6.0

[PASS] 108 / 108 fits remain p

In [14]:
# =================================================================================================
# STAGE28-FINAL — ZERO-FIT FINAL SYNTHESIS + MANUSCRIPT-READY RESULTS FREEZE
#
# EMPIRICAL WORK IS OVER.
#
# NEW MODEL FITS                 : 0
# MODEL INFERENCE                : 0
# THRESHOLD SELECTION            : 0
# MODEL SELECTION                : 0
# TARGET / HOLDOUT OPENINGS      : 0
# BOOTSTRAP RECOMPUTATION        : 0
# SHAP RECOMPUTATION             : 0
# NEW SIGNIFICANCE TESTS         : 0
# NEW QUALITATIVE CUTOFFS        : 0
#
# Parent:
#   f5de70d25a5714ae2b70a18819bde97eb3e38354
#
# Purpose:
#   1. Reconcile final Stage28 closure.
#   2. Summarize Stage22 five-seed shared-holdout results.
#   3. Summarize chronology/random LOAO seed stability.
#   4. Summarize paired random-minus-chronological LOAO contrasts.
#   5. Freeze the complete conclusion-stability registry.
#   6. Create manuscript-ready numerical tables.
#   7. Create guarded manuscript-ready results prose.
#   8. Freeze final Stage28 synthesis receipt/checksums.
#
# NO STAGE29.
# =================================================================================================

from __future__ import annotations

import base64
import hashlib
import json
import math
import os
import subprocess

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd


# =================================================================================================
# CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "f5de70d25a5714ae2b70a18819bde97eb3e38354"
)

ROOT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
)

CLOSURE_RECEIPT = (
    ROOT
    / "stage28_3a_experiment_closure_audit"
    / "stage28_3a_experiment_closure_receipt.json"
)

STAGE3_ROOT = (
    ROOT
    / "stage28_3_seed_uncertainty"
)

STAGE4_ROOT = (
    ROOT
    / "stage28_4_stage22_shared_final_holdout"
)

STAGE4_RECEIPT = (
    STAGE4_ROOT
    / "stage28_4_receipt.json"
)

STAGE22_METRICS = (
    STAGE4_ROOT
    / "stage28_4_seed_level_metrics.csv"
)

STAGE22_CONTRAST = (
    STAGE4_ROOT
    / "stage28_4_random_vs_chronological_seedwise.csv"
)

STAGE22_STABILITY = (
    STAGE4_ROOT
    / "stage28_4_stage22_directional_stability_summary.csv"
)

LOAO_SEED_LEVEL = (
    STAGE3_ROOT
    / "stage28_3b_loao_seed_level_metrics.csv"
)

LOAO_FIVE_SEED = (
    STAGE3_ROOT
    / "stage28_3b_loao_five_seed_summary.csv"
)

LOAO_STABILITY = (
    STAGE3_ROOT
    / "stage28_3b_loao_stability_summary.csv"
)

LOAO_CONTRAST = (
    STAGE3_ROOT
    / "stage28_3c_random_vs_chronological_five_seed_summary.csv"
)

LOAO_DIRECTION = (
    STAGE3_ROOT
    / "stage28_3c_numeric_direction_summary.csv"
)

CONCLUSION_STABILITY = (
    STAGE3_ROOT
    / "conclusion_stability.csv"
)

OUT = (
    ROOT
    / "stage28_final_synthesis"
)

STAGE22_SUMMARY_OUT = (
    OUT
    / "stage28_final_stage22_shared_holdout_five_seed_summary.csv"
)

STAGE22_CONTRAST_OUT = (
    OUT
    / "stage28_final_stage22_random_minus_chronological_summary.csv"
)

LOAO_KEY_OUT = (
    OUT
    / "stage28_final_loao_key_metric_summary.csv"
)

LOAO_STABILITY_OUT = (
    OUT
    / "stage28_final_loao_stability_registry.csv"
)

RANDOM_CHRONO_OUT = (
    OUT
    / "stage28_final_random_vs_chronological_key_contrasts.csv"
)

CLAIM_REGISTRY_OUT = (
    OUT
    / "stage28_final_claim_registry.csv"
)

NUMBERS_OUT = (
    OUT
    / "stage28_final_manuscript_numbers.json"
)

MANUSCRIPT_OUT = (
    OUT
    / "stage28_final_manuscript_results.md"
)

RECEIPT_OUT = (
    OUT
    / "stage28_final_synthesis_receipt.json"
)

README_OUT = (
    OUT
    / "README.md"
)

CHECKSUM_OUT = (
    OUT
    / "checksums.sha256"
)


SEEDS = [
    42,
    43,
    44,
    45,
    46,
]

FAMILIES = [
    "BOT",
    "DDOS",
    "INFILTRATION",
    "PORT_SCAN",
    "WEB_ATTACK",
]

LEARNERS = [
    "XGBOOST",
    "LIGHTGBM",
]

ARMS = [
    "STAGE27_CHRONOLOGY_LOAO",
    "STAGE28B_RANDOM_LOAO",
]


STAGE22_METRICS_TO_REPORT = [
    "roc_auc",
    "pr_auc",
    "standard_recall",
    "standard_fpr",
    "balanced_recall",
    "balanced_fpr",
    "security_recall",
    "security_fpr",
]


LOAO_KEY_METRICS = [
    "ROC_AUC",
    "PR_AUC",
    "PR_CHANCE_ANCHOR",
    "PR_EXCESS",
    "PR_LIFT",
    "STANDARD_RECALL",
    "BALANCED_RECALL",
    "SECURITY_RECALL",
]


CONTRAST_KEY_METRICS = [
    "ROC_AUC",
    "PR_EXCESS",
    "STANDARD_RECALL",
    "BALANCED_RECALL",
    "SECURITY_RECALL",
]


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(text):
    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            + " ".join(map(str, cmd))
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return run(
        ["git", *args]
    ).stdout.strip()


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def write_json(path, obj):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n",
        encoding="utf-8",
    )


def sha256_file(
    path,
    chunk=16 * 1024 * 1024,
):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk)

            if not b:
                break

            h.update(b)

    return h.hexdigest()


def finite_float(value):
    try:
        x = float(value)

    except Exception:
        return float("nan")

    if not math.isfinite(x):
        return float("nan")

    return x


def five_seed_stats(values):
    vals = np.asarray(
        [
            finite_float(x)
            for x in values
        ],
        dtype=np.float64,
    )

    if vals.size != 5:
        raise RuntimeError(
            f"Expected exactly 5 seeds, found {vals.size}."
        )

    if not np.all(
        np.isfinite(vals)
    ):
        return {
            "n_seeds":
                5,

            "n_defined":
                int(
                    np.isfinite(
                        vals
                    ).sum()
                ),

            "mean":
                float("nan"),

            "median":
                float("nan"),

            "sample_standard_deviation_ddof_1":
                float("nan"),

            "minimum":
                float("nan"),

            "maximum":
                float("nan"),

            "range":
                float("nan"),

            "IQR_Q75_minus_Q25_linear":
                float("nan"),
        }

    q25, q75 = np.quantile(
        vals,
        [
            0.25,
            0.75,
        ],
        method="linear",
    )

    return {
        "n_seeds":
            5,

        "n_defined":
            5,

        "mean":
            float(
                np.mean(vals)
            ),

        "median":
            float(
                np.median(vals)
            ),

        "sample_standard_deviation_ddof_1":
            float(
                np.std(
                    vals,
                    ddof=1,
                )
            ),

        "minimum":
            float(
                np.min(vals)
            ),

        "maximum":
            float(
                np.max(vals)
            ),

        "range":
            float(
                np.max(vals)
                - np.min(vals)
            ),

        "IQR_Q75_minus_Q25_linear":
            float(
                q75
                - q25
            ),
    }


def bool_value(value):
    if isinstance(
        value,
        bool,
    ):
        return value

    text = str(
        value
    ).strip().lower()

    if text in {
        "true",
        "1",
        "yes",
    }:
        return True

    if text in {
        "false",
        "0",
        "no",
    }:
        return False

    raise RuntimeError(
        f"Cannot parse boolean: {value!r}"
    )


def fmt(
    value,
    digits=4,
):
    try:
        x = float(value)

    except Exception:
        return str(value)

    if not math.isfinite(x):
        return "NA"

    return f"{x:.{digits}f}"


def md_table(
    rows,
    columns,
):
    if not rows:
        return "_No rows._"

    header = (
        "| "
        + " | ".join(
            columns
        )
        + " |"
    )

    separator = (
        "| "
        + " | ".join(
            [
                "---"
                for _ in columns
            ]
        )
        + " |"
    )

    body = []

    for row in rows:
        cells = []

        for col in columns:
            value = row.get(
                col,
                "",
            )

            text = str(
                value
            ).replace(
                "|",
                "\\|",
            )

            cells.append(
                text
            )

        body.append(
            "| "
            + " | ".join(
                cells
            )
            + " |"
        )

    return "\n".join(
        [
            header,
            separator,
            *body,
        ]
    )


def get_github_token():
    from kaggle_secrets import (
        UserSecretsClient,
    )

    client = UserSecretsClient()

    for label in [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]:
        try:
            token = client.get_secret(
                label
            )
        except Exception:
            token = None

        if (
            isinstance(
                token,
                str,
            )
            and token.strip()
        ):
            return (
                token.strip(),
                label,
            )

    raise RuntimeError(
        "No GitHub token available."
    )


def push_origin_main():
    token, label = get_github_token()

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode(
            "utf-8"
        )
    ).decode(
        "ascii"
    )

    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(REPO),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            "Git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    print(
        f"[PASS] GitHub credential: kaggle_secret:{label}"
    )

    print(
        "[PASS] token not displayed"
    )

    if p.stdout.strip():
        print(
            p.stdout.strip()
        )

    if p.stderr.strip():
        print(
            p.stderr.strip()
        )


# =================================================================================================
# 0. PARENT / CLEAN REPOSITORY
# =================================================================================================

banner(
    "STAGE28-FINAL — REPOSITORY / EMPIRICAL-CLOSURE GATE"
)

if OUT.exists():
    raise RuntimeError(
        "Stage28 final-synthesis output already exists. "
        "Do not overwrite."
    )

if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Repository must be clean."
    )

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)

local_head = git(
    "rev-parse",
    "HEAD",
)

origin_head = git(
    "rev-parse",
    "origin/main",
)

print(
    "Expected parent:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD     :",
    local_head,
)

print(
    "origin/main    :",
    origin_head,
)

if not (
    local_head
    == origin_head
    == EXPECTED_PARENT
):
    raise RuntimeError(
        "Stage28-FINAL parent mismatch."
    )

closure = read_json(
    CLOSURE_RECEIPT
)

stage4 = read_json(
    STAGE4_RECEIPT
)

if (
    closure[
        "closure_status"
    ]
    !=
    "PASS_STAGE28_NEW_MODEL_FITTING_PERMANENTLY_CLOSED"
):
    raise RuntimeError(
        "Stage28 fitting closure is not PASS."
    )

if (
    int(
        closure[
            "fit_budget_closure"
        ][
            "consumed_new_fits"
        ]
    )
    != 108
    or
    int(
        closure[
            "fit_budget_closure"
        ][
            "remaining_new_fits"
        ]
    )
    != 0
):
    raise RuntimeError(
        "Stage28 fit budget does not close at 108/108."
    )

if (
    stage4[
        "status"
    ]
    != "STAGE28_4_COMPLETE"
):
    raise RuntimeError(
        "Stage28-4 is not complete."
    )

if (
    stage4[
        "scientific_operations"
    ][
        "new_model_fits"
    ]
    != 0
):
    raise RuntimeError(
        "Stage28-4 unexpectedly contains a model fit."
    )

if (
    stage4[
        "scientific_operations"
    ][
        "threshold_selections"
    ]
    != 0
):
    raise RuntimeError(
        "Stage28-4 unexpectedly selected thresholds."
    )

if (
    stage4[
        "scientific_operations"
    ][
        "model_selections"
    ]
    != 0
):
    raise RuntimeError(
        "Stage28-4 unexpectedly performed model selection."
    )

if (
    stage4[
        "next_authorized_step"
    ]
    !=
    (
        "ZERO_FIT_FINAL_SYNTHESIS_AND_MANUSCRIPT_INTEGRATION; "
        "NO_FURTHER_MODEL_FITTING; NO_STAGE29"
    )
):
    raise RuntimeError(
        "Final authorized-work statement changed."
    )

print()
print(
    "[PASS] Stage28 new-fit ledger = 108 / 108"
)

print(
    "[PASS] remaining fit budget = 0"
)

print(
    "[PASS] Stage28-4 empirical work complete"
)

print(
    "[PASS] NO Stage29"
)

print(
    "[PASS] current stage = reporting/synthesis only"
)


# =================================================================================================
# 1. INPUT ARTIFACT GATE
# =================================================================================================

banner(
    "STAGE28-FINAL — DURABLE INPUT ARTIFACT GATE"
)

required_inputs = [
    STAGE22_METRICS,
    STAGE22_CONTRAST,
    STAGE22_STABILITY,
    LOAO_SEED_LEVEL,
    LOAO_FIVE_SEED,
    LOAO_STABILITY,
    LOAO_CONTRAST,
    LOAO_DIRECTION,
    CONCLUSION_STABILITY,
]

input_sha = {}

for path in required_inputs:
    if not path.is_file():
        raise RuntimeError(
            f"Missing synthesis input:\n{path}"
        )

    digest = sha256_file(
        path
    )

    input_sha[
        str(
            path.relative_to(
                REPO
            )
        )
    ] = digest

    print(
        "[PASS]",
        path.name,
        digest,
    )


stage22_df = pd.read_csv(
    STAGE22_METRICS
)

stage22_contrast_df = pd.read_csv(
    STAGE22_CONTRAST
)

stage22_stability_df = pd.read_csv(
    STAGE22_STABILITY
)

loao_level_df = pd.read_csv(
    LOAO_SEED_LEVEL
)

loao_summary_df = pd.read_csv(
    LOAO_FIVE_SEED
)

loao_stability_df = pd.read_csv(
    LOAO_STABILITY
)

loao_contrast_df = pd.read_csv(
    LOAO_CONTRAST
)

loao_direction_df = pd.read_csv(
    LOAO_DIRECTION
)

conclusion_df = pd.read_csv(
    CONCLUSION_STABILITY
)


if len(
    stage22_df
) != 10:
    raise RuntimeError(
        "Stage22 seed-level result count != 10."
    )

if len(
    stage22_contrast_df
) != 5:
    raise RuntimeError(
        "Stage22 seedwise contrast count != 5."
    )

if len(
    stage22_stability_df
) != 2:
    raise RuntimeError(
        "Stage22 stability summary count != 2."
    )

if len(
    loao_level_df
) != 100:
    raise RuntimeError(
        "LOAO seed-level realization count != 100."
    )

if len(
    loao_stability_df
) != 110:
    raise RuntimeError(
        "LOAO stability summary count != 110."
    )

if len(
    conclusion_df
) != 560:
    raise RuntimeError(
        "Combined conclusion-stability row count != 560."
    )

print()
print(
    "[PASS] Stage22 evaluations = 10"
)

print(
    "[PASS] chronology LOAO seed realizations = 50"
)

print(
    "[PASS] random LOAO seed realizations = 50"
)

print(
    "[PASS] LOAO stability registry rows = 110"
)

print(
    "[PASS] complete conclusion_stability rows = 560"
)


# =================================================================================================
# 2. STAGE22 SHARED-HOLDOUT FIVE-SEED SUMMARY
# =================================================================================================

banner(
    "STAGE28-FINAL — STAGE22 SHARED-HOLDOUT FIVE-SEED SUMMARY"
)

stage22_summary_rows = []

for unit in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:
    subset = (
        stage22_df.loc[
            stage22_df[
                "unit"
            ]
            == unit
        ]
        .sort_values(
            "seed"
        )
    )

    if subset[
        "seed"
    ].tolist() != SEEDS:
        raise RuntimeError(
            f"{unit}: seed coverage != 42..46."
        )

    for metric in (
        STAGE22_METRICS_TO_REPORT
    ):
        stats = five_seed_stats(
            subset[
                metric
            ].tolist()
        )

        stage22_summary_rows.append(
            {
                "unit":
                    unit,

                "metric":
                    metric,

                **stats,
            }
        )

stage22_summary_df = pd.DataFrame(
    stage22_summary_rows
)

print(
    "[PASS] Stage22 metrics summarized over exactly five frozen seeds"
)


# =================================================================================================
# 3. STAGE22 RANDOM-MINUS-CHRONOLOGICAL SUMMARY
# =================================================================================================

banner(
    "STAGE28-FINAL — STAGE22 DIRECTIONAL CONTRAST SUMMARY"
)

stage22_contrast_rows = []

for metric, delta_column, support_column in [
    (
        "PR_AUC",
        "pr_auc_random_minus_chronological",
        "pr_random_lt_chronological",
    ),
    (
        "ROC_AUC",
        "roc_auc_random_minus_chronological",
        "roc_random_lt_chronological",
    ),
]:
    subset = stage22_contrast_df.sort_values(
        "seed"
    )

    if subset[
        "seed"
    ].tolist() != SEEDS:
        raise RuntimeError(
            "Stage22 contrast seed coverage != 42..46."
        )

    stats = five_seed_stats(
        subset[
            delta_column
        ].tolist()
    )

    support_values = [
        bool_value(
            x
        )
        for x
        in subset[
            support_column
        ].tolist()
    ]

    supporting = sum(
        support_values
    )

    stage22_contrast_rows.append(
        {
            "metric":
                metric,

            "contrast":
                "RANDOM_MINUS_CHRONOLOGICAL",

            **stats,

            "random_lt_chronological_supporting_seeds":
                supporting,

            "frozen_seed_count":
                5,

            "stability_rate":
                supporting
                / 5.0,
        }
    )

stage22_contrast_summary_df = pd.DataFrame(
    stage22_contrast_rows
)

if not (
    stage22_contrast_summary_df[
        "random_lt_chronological_supporting_seeds"
    ]
    == 5
).all():
    raise RuntimeError(
        "A frozen Stage22 directional claim is not 5/5."
    )

print(
    "[PASS] PR random < chronological = 5 / 5"
)

print(
    "[PASS] ROC random < chronological = 5 / 5"
)


# =================================================================================================
# 4. LOAO KEY FIVE-SEED NUMERICAL SUMMARY
# =================================================================================================

banner(
    "STAGE28-FINAL — LOAO KEY METRIC SUMMARY"
)

loao_key_df = (
    loao_summary_df.loc[
        loao_summary_df[
            "metric"
        ].isin(
            LOAO_KEY_METRICS
        )
    ]
    .copy()
)

expected_groups = (
    2
    * 5
    * 2
    * len(
        LOAO_KEY_METRICS
    )
)

if len(
    loao_key_df
) != expected_groups:
    raise RuntimeError(
        "LOAO key-metric summary row count mismatch: "
        f"{len(loao_key_df)} != {expected_groups}"
    )

print(
    "[PASS] chronology/random × 5 families × 2 learners "
    "× key metrics preserved"
)

print(
    "[PASS] Infiltration support status preserved"
)


# =================================================================================================
# 5. LOAO STABILITY REGISTRY
# =================================================================================================

banner(
    "STAGE28-FINAL — LOAO QUALITATIVE STABILITY REGISTRY"
)

loao_stability_final = (
    loao_stability_df.copy()
)

loao_stability_final[
    "is_five_of_five"
] = (
    loao_stability_final[
        "frozen_seeds_supporting_condition"
    ]
    == 5
)

loao_stability_final[
    "is_zero_of_five"
] = (
    loao_stability_final[
        "frozen_seeds_supporting_condition"
    ]
    == 0
)

if (
    loao_stability_final[
        "frozen_seed_count"
    ]
    != 5
).any():
    raise RuntimeError(
        "LOAO stability denominator changed from five."
    )

print(
    "[PASS] all LOAO stability denominators = 5"
)

print(
    "[PASS] no qualitative condition created after results"
)


# =================================================================================================
# 6. PAIRED RANDOM-vs-CHRONOLOGICAL KEY CONTRASTS
# =================================================================================================

banner(
    "STAGE28-FINAL — RANDOM-vs-CHRONOLOGICAL KEY CONTRASTS"
)

contrast_key = (
    loao_contrast_df.loc[
        loao_contrast_df[
            "metric"
        ].isin(
            CONTRAST_KEY_METRICS
        )
    ]
    .copy()
)

direction_key = (
    loao_direction_df.loc[
        loao_direction_df[
            "metric"
        ].isin(
            CONTRAST_KEY_METRICS
        )
    ]
    .copy()
)

join_keys = [
    "family",
    "learner",
    "metric",
    "analysis_status",
    "metric_role",
]

random_chrono_df = contrast_key.merge(
    direction_key,
    on=join_keys,
    how="inner",
    validate="one_to_one",
)

expected_contrast_rows = (
    5
    * 2
    * len(
        CONTRAST_KEY_METRICS
    )
)

if len(
    random_chrono_df
) != expected_contrast_rows:
    raise RuntimeError(
        "Random-vs-chronological key contrast row count mismatch."
    )

if (
    random_chrono_df[
        "contrast_definition"
    ]
    != "RANDOM_MINUS_CHRONOLOGICAL"
).any():
    raise RuntimeError(
        "Contrast orientation changed."
    )

print(
    "[PASS] exact family + learner + metric paired contrasts"
)

print(
    "[PASS] contrast orientation = RANDOM - CHRONOLOGICAL"
)

print(
    "[PASS] no post-result 'large/small/collapse' cutoff assigned"
)


# =================================================================================================
# 7. COMPLETE CLAIM REGISTRY
# =================================================================================================

banner(
    "STAGE28-FINAL — CLAIM REGISTRY"
)

claim_registry_rows = []

# Stage22 frozen directional claims.
for _, row in (
    stage22_stability_df.iterrows()
):
    claim_registry_rows.append(
        {
            "scope":
                "STAGE22_SHARED_FINAL_HOLDOUT",

            "parent_stage":
                "STAGE22_FULL",

            "family":
                "",

            "learner":
                "ENS_LGBM_XGB_EQUAL",

            "claim_id":
                row[
                    "claim_id"
                ],

            "analysis_status":
                "DESCRIPTIVE_ROBUSTNESS_PRE_REGISTERED",

            "supporting_seeds":
                int(
                    row[
                        "supporting_seeds"
                    ]
                ),

            "frozen_seed_count":
                int(
                    row[
                        "total_frozen_seeds"
                    ]
                ),

            "stability_rate":
                float(
                    row[
                        "stability_rate"
                    ]
                ),
        }
    )

# LOAO frozen qualitative conditions.
for _, row in (
    loao_stability_final.iterrows()
):
    claim_registry_rows.append(
        {
            "scope":
                (
                    "CHRONOLOGY_LOAO"
                    if row[
                        "parent_stage"
                    ]
                    ==
                    "STAGE27_CHRONOLOGY_LOAO"
                    else
                    "RANDOM_LOAO_CONTROL"
                ),

            "parent_stage":
                row[
                    "parent_stage"
                ],

            "family":
                row[
                    "family_if_applicable"
                ],

            "learner":
                (
                    row[
                        "learner_if_applicable"
                    ]
                    if pd.notna(
                        row[
                            "learner_if_applicable"
                        ]
                    )
                    else ""
                ),

            "claim_id":
                row[
                    "claim_id"
                ],

            "analysis_status":
                row[
                    "analysis_status"
                ],

            "supporting_seeds":
                int(
                    row[
                        "frozen_seeds_supporting_condition"
                    ]
                ),

            "frozen_seed_count":
                int(
                    row[
                        "frozen_seed_count"
                    ]
                ),

            "stability_rate":
                float(
                    row[
                        "stability_rate"
                    ]
                ),
        }
    )

claim_registry_df = pd.DataFrame(
    claim_registry_rows
)

if len(
    claim_registry_df
) != 112:
    raise RuntimeError(
        f"Final claim registry rows != 112: "
        f"{len(claim_registry_df)}"
    )

print(
    "[PASS] 2 Stage22 frozen claims"
)

print(
    "[PASS] 110 LOAO frozen stability claims"
)

print(
    "[PASS] total claim-registry rows = 112"
)


# =================================================================================================
# 8. COMPACT FAMILY STABILITY TABLE FOR MANUSCRIPT
# =================================================================================================

banner(
    "STAGE28-FINAL — MANUSCRIPT FAMILY STABILITY MATRIX"
)

condition_ids = [
    "ROC_ABOVE_CHANCE",
    "PR_ABOVE_CHANCE",
    "STANDARD_DETECTION_PRESENT",
    "BALANCED_DETECTION_PRESENT",
    "SECURITY_DETECTION_PRESENT_WHERE_FEASIBLE",
]

family_matrix_rows = []

for arm in ARMS:
    for family in FAMILIES:
        for learner in LEARNERS:
            base = loao_stability_df.loc[
                (
                    loao_stability_df[
                        "parent_stage"
                    ]
                    == arm
                )
                &
                (
                    loao_stability_df[
                        "family_if_applicable"
                    ]
                    == family
                )
                &
                (
                    loao_stability_df[
                        "learner_if_applicable"
                    ]
                    == learner
                )
            ]

            values = {
                row[
                    "claim_id"
                ]:
                    int(
                        row[
                            "frozen_seeds_supporting_condition"
                        ]
                    )
                for _, row
                in base.iterrows()
                if row[
                    "claim_id"
                ]
                in condition_ids
            }

            if set(
                values
            ) != set(
                condition_ids
            ):
                raise RuntimeError(
                    "Family stability condition set incomplete: "
                    f"{arm}/{family}/{learner}"
                )

            family_matrix_rows.append(
                {
                    "arm":
                        (
                            "CHRONOLOGY"
                            if arm
                            ==
                            "STAGE27_CHRONOLOGY_LOAO"
                            else
                            "RANDOM_CONTROL"
                        ),

                    "family":
                        family,

                    "learner":
                        learner,

                    "status":
                        (
                            "DESCRIPTIVE_ONLY"
                            if family
                            == "INFILTRATION"
                            else
                            "INFERENTIAL_ELIGIBLE"
                        ),

                    "ROC>0.5":
                        f"{values['ROC_ABOVE_CHANCE']}/5",

                    "PR>chance":
                        f"{values['PR_ABOVE_CHANCE']}/5",

                    "Std recall>0":
                        f"{values['STANDARD_DETECTION_PRESENT']}/5",

                    "Bal recall>0":
                        f"{values['BALANCED_DETECTION_PRESENT']}/5",

                    "Sec recall>0":
                        (
                            f"{values['SECURITY_DETECTION_PRESENT_WHERE_FEASIBLE']}/5"
                        ),
                }
            )

family_matrix_df = pd.DataFrame(
    family_matrix_rows
)

print(
    "[PASS] 20-row family × learner × arm stability matrix"
)


# =================================================================================================
# 9. COMPACT RANDOM-vs-CHRONO TABLE
# =================================================================================================

contrast_matrix_rows = []

for family in FAMILIES:
    for learner in LEARNERS:
        row_out = {
            "family":
                family,

            "learner":
                learner,

            "status":
                (
                    "DESCRIPTIVE_ONLY"
                    if family
                    == "INFILTRATION"
                    else
                    "INFERENTIAL_ELIGIBLE"
                ),
        }

        for metric in [
            "ROC_AUC",
            "PR_EXCESS",
            "STANDARD_RECALL",
        ]:
            row = random_chrono_df.loc[
                (
                    random_chrono_df[
                        "family"
                    ]
                    == family
                )
                &
                (
                    random_chrono_df[
                        "learner"
                    ]
                    == learner
                )
                &
                (
                    random_chrono_df[
                        "metric"
                    ]
                    == metric
                )
            ]

            if len(
                row
            ) != 1:
                raise RuntimeError(
                    "Contrast matrix lookup not unique."
                )

            row = row.iloc[
                0
            ]

            short = {
                "ROC_AUC":
                    "ΔROC",

                "PR_EXCESS":
                    "ΔPR-excess",

                "STANDARD_RECALL":
                    "ΔStd-recall",
            }[
                metric
            ]

            row_out[
                short
            ] = fmt(
                row[
                    "mean"
                ],
                4,
            )

            row_out[
                short
                + " sign"
            ] = (
                f"+{int(row['random_gt_chrono_seed_count'])}"
                f"/-{int(row['random_lt_chrono_seed_count'])}"
                f"/={int(row['random_eq_chrono_seed_count'])}"
            )

        contrast_matrix_rows.append(
            row_out
        )

contrast_matrix_df = pd.DataFrame(
    contrast_matrix_rows
)


# =================================================================================================
# 10. DERIVE ONLY SAFE, FROZEN-CONDITION FINDINGS
# =================================================================================================

banner(
    "STAGE28-FINAL — DERIVE GUARDED FINDINGS"
)

# Families for which BOTH learners show ROC>0.5 and PR>chance
# in all five seeds under BOTH chronology and random control.
fully_rank_stable_families = []

for family in FAMILIES:
    if family == "INFILTRATION":
        continue

    ok = True

    for arm in ARMS:
        for learner in LEARNERS:
            for claim_id in [
                "ROC_ABOVE_CHANCE",
                "PR_ABOVE_CHANCE",
            ]:
                rows = loao_stability_df.loc[
                    (
                        loao_stability_df[
                            "parent_stage"
                        ]
                        == arm
                    )
                    &
                    (
                        loao_stability_df[
                            "family_if_applicable"
                        ]
                        == family
                    )
                    &
                    (
                        loao_stability_df[
                            "learner_if_applicable"
                        ]
                        == learner
                    )
                    &
                    (
                        loao_stability_df[
                            "claim_id"
                        ]
                        == claim_id
                    )
                ]

                if (
                    len(
                        rows
                    )
                    != 1
                    or
                    int(
                        rows.iloc[
                            0
                        ][
                            "frozen_seeds_supporting_condition"
                        ]
                    )
                    != 5
                ):
                    ok = False

    if ok:
        fully_rank_stable_families.append(
            family
        )


# Explicit BOT ranking pattern, derived entirely from frozen conditions.
bot_pattern = {}

for arm in ARMS:
    bot_pattern[
        arm
    ] = {}

    for learner in LEARNERS:
        bot_pattern[
            arm
        ][
            learner
        ] = {}

        for claim_id in [
            "ROC_ABOVE_CHANCE",
            "PR_ABOVE_CHANCE",
        ]:
            row = loao_stability_df.loc[
                (
                    loao_stability_df[
                        "parent_stage"
                    ]
                    == arm
                )
                &
                (
                    loao_stability_df[
                        "family_if_applicable"
                    ]
                    == "BOT"
                )
                &
                (
                    loao_stability_df[
                        "learner_if_applicable"
                    ]
                    == learner
                )
                &
                (
                    loao_stability_df[
                        "claim_id"
                    ]
                    == claim_id
                )
            ]

            if len(
                row
            ) != 1:
                raise RuntimeError(
                    "BOT stability lookup not unique."
                )

            bot_pattern[
                arm
            ][
                learner
            ][
                claim_id
            ] = int(
                row.iloc[
                    0
                ][
                    "frozen_seeds_supporting_condition"
                ]
            )


# Stage22 numerical means.
stage22_lookup = {}

for unit in EXPECTED_UNITS if False else [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:
    stage22_lookup[
        unit
    ] = {}

    for metric in [
        "roc_auc",
        "pr_auc",
    ]:
        row = stage22_summary_df.loc[
            (
                stage22_summary_df[
                    "unit"
                ]
                == unit
            )
            &
            (
                stage22_summary_df[
                    "metric"
                ]
                == metric
            )
        ]

        if len(
            row
        ) != 1:
            raise RuntimeError(
                "Stage22 summary lookup not unique."
            )

        row = row.iloc[
            0
        ]

        stage22_lookup[
            unit
        ][
            metric
        ] = {
            "mean":
                float(
                    row[
                        "mean"
                    ]
                ),

            "sd":
                float(
                    row[
                        "sample_standard_deviation_ddof_1"
                    ]
                ),

            "minimum":
                float(
                    row[
                        "minimum"
                    ]
                ),

            "maximum":
                float(
                    row[
                        "maximum"
                    ]
                ),
        }


numbers = {
    "stage28_empirical_closure": {
        "authorized_new_fits":
            108,

        "consumed_new_fits":
            108,

        "remaining_new_fits":
            0,

        "stage22_final_holdout_ensemble_evaluations":
            10,

        "stage22_final_holdout_component_inferences":
            20,

        "chronology_loao_seed_realizations":
            50,

        "random_loao_seed_realizations":
            50,

        "stage29_authorized":
            False,
    },

    "stage22_shared_holdout": {
        "rows":
            1_374_133,

        "benign":
            998_788,

        "attack":
            375_345,

        "random_natural":
            stage22_lookup[
                "RANDOM_NATURAL"
            ],

        "chronological_natural":
            stage22_lookup[
                "CHRONOLOGICAL_NATURAL"
            ],

        "PR_RANDOM_LT_CHRONO":
            {
                "supporting_seeds":
                    5,

                "frozen_seeds":
                    5,

                "stability_rate":
                    1.0,
            },

        "ROC_RANDOM_LT_CHRONO":
            {
                "supporting_seeds":
                    5,

                "frozen_seeds":
                    5,

                "stability_rate":
                    1.0,
            },
    },

    "loao": {
        "fully_ranking_stable_inferential_families_both_learners_both_arms":
            fully_rank_stable_families,

        "bot_frozen_condition_pattern":
            bot_pattern,

        "infiltration":
            {
                "heldout_positive_support":
                    36,

                "status":
                    "DESCRIPTIVE_ONLY_SUPPORT_LT_50",
            },

        "aggregation":
            "FAMILY_SPECIFIC_PRIMARY_NO_AGGREGATE_ZERO_DAY_SCORE",
    },

    "interpretation_guardrails": [
        (
            "Training-seed robustness and bootstrap/sampling "
            "uncertainty remain separate."
        ),
        (
            "No best-seed result is selected or reported."
        ),
        (
            "Random LOAO is a control, not a deployment-realistic estimate."
        ),
        (
            "Random-vs-chronological differences do not prove "
            "temporal drift is the sole cause."
        ),
        (
            "Infiltration is descriptive only because support is 36."
        ),
        (
            "No aggregate zero-day score is authorized."
        ),
        (
            "No post-result cutoff is assigned to terms such as "
            "'much greater', 'collapse', or 'survive'."
        ),
    ],
}


# =================================================================================================
# 11. WRITE FINAL NUMERICAL TABLES
# =================================================================================================

banner(
    "STAGE28-FINAL — WRITE PUBLICATION TABLES"
)

OUT.mkdir(
    parents=False,
    exist_ok=False,
)

stage22_summary_df.to_csv(
    STAGE22_SUMMARY_OUT,
    index=False,
)

stage22_contrast_summary_df.to_csv(
    STAGE22_CONTRAST_OUT,
    index=False,
)

loao_key_df.to_csv(
    LOAO_KEY_OUT,
    index=False,
)

loao_stability_final.to_csv(
    LOAO_STABILITY_OUT,
    index=False,
)

random_chrono_df.to_csv(
    RANDOM_CHRONO_OUT,
    index=False,
)

claim_registry_df.to_csv(
    CLAIM_REGISTRY_OUT,
    index=False,
)

write_json(
    NUMBERS_OUT,
    numbers,
)

print(
    "[PASS] Stage22 five-seed summary"
)

print(
    "[PASS] Stage22 contrast summary"
)

print(
    "[PASS] LOAO key metrics"
)

print(
    "[PASS] LOAO stability registry"
)

print(
    "[PASS] random-vs-chronological key contrasts"
)

print(
    "[PASS] complete claim registry"
)

print(
    "[PASS] manuscript numbers JSON"
)


# =================================================================================================
# 12. MANUSCRIPT-READY MARKDOWN
# =================================================================================================

banner(
    "STAGE28-FINAL — MANUSCRIPT-READY RESULTS TEXT"
)

stage22_display_rows = []

for unit in [
    "RANDOM_NATURAL",
    "CHRONOLOGICAL_NATURAL",
]:
    for metric in [
        "roc_auc",
        "pr_auc",
    ]:
        row = stage22_summary_df.loc[
            (
                stage22_summary_df[
                    "unit"
                ]
                == unit
            )
            &
            (
                stage22_summary_df[
                    "metric"
                ]
                == metric
            )
        ].iloc[
            0
        ]

        stage22_display_rows.append(
            {
                "Geometry":
                    unit,

                "Metric":
                    metric.upper(),

                "Mean":
                    fmt(
                        row[
                            "mean"
                        ],
                        4,
                    ),

                "SD":
                    fmt(
                        row[
                            "sample_standard_deviation_ddof_1"
                        ],
                        4,
                    ),

                "Min":
                    fmt(
                        row[
                            "minimum"
                        ],
                        4,
                    ),

                "Max":
                    fmt(
                        row[
                            "maximum"
                        ],
                        4,
                    ),
            }
        )


stage22_contrast_display = []

for _, row in (
    stage22_contrast_summary_df.iterrows()
):
    stage22_contrast_display.append(
        {
            "Metric":
                row[
                    "metric"
                ],

            "Mean Δ (R-C)":
                fmt(
                    row[
                        "mean"
                    ],
                    4,
                ),

            "SD":
                fmt(
                    row[
                        "sample_standard_deviation_ddof_1"
                    ],
                    4,
                ),

            "Min":
                fmt(
                    row[
                        "minimum"
                    ],
                    4,
                ),

            "Max":
                fmt(
                    row[
                        "maximum"
                    ],
                    4,
                ),

            "Random<Chrono":
                (
                    f"{int(row['random_lt_chronological_supporting_seeds'])}/5"
                ),
        }
    )


family_display_rows = (
    family_matrix_df.to_dict(
        orient="records"
    )
)

contrast_display_rows = (
    contrast_matrix_df.to_dict(
        orient="records"
    )
)


stable_family_text = (
    ", ".join(
        fully_rank_stable_families
    )
    if fully_rank_stable_families
    else "none"
)


manuscript = f"""# Stage28 — Final robustness and novelty-control synthesis

## Empirical closure

Stage28 closed the preregistered robustness program with **108/108 authorized
new fits consumed and zero remaining fits**. The experiment included 12
historical model reuses in addition to the 108 new fits. After fitting was
permanently closed, Stage28 performed only preregistered zero-fit synthesis and
the authorized Stage22 shared-final-holdout robustness inference. No Stage29
empirical stage is authorized.

The Stage22 shared holdout contained **1,374,133 flows**, including **998,788
benign** and **375,345 attack** flows. Ten frozen Stage22 ensemble realizations
(two validation geometries × five training seeds) were evaluated on this same
holdout. No threshold or model selection was performed on the holdout.

## Stage22 training-seed robustness on the shared final holdout

{md_table(
    stage22_display_rows,
    [
        "Geometry",
        "Metric",
        "Mean",
        "SD",
        "Min",
        "Max",
    ],
)}

The preregistered directional comparison was stable for every frozen seed.
`PR_AUC_RANDOM_NATURAL < PR_AUC_CHRONOLOGICAL_NATURAL` held for **5/5 seeds**,
and `ROC_AUC_RANDOM_NATURAL < ROC_AUC_CHRONOLOGICAL_NATURAL` also held for
**5/5 seeds**. This is descriptive conclusion-stability analysis rather than a
new significance test.

{md_table(
    stage22_contrast_display,
    [
        "Metric",
        "Mean Δ (R-C)",
        "SD",
        "Min",
        "Max",
        "Random<Chrono",
    ],
)}

These results show that the Stage22 direction was not a seed-42 artifact: the
same random-versus-chronological ranking persisted across seeds 42–46.

## Leave-one-attack-family-out seed stability

The LOAO analysis remained family-specific. The table below reports the number
of frozen seeds (out of five) satisfying each preregistered qualitative
condition. Infiltration is retained only as a descriptive result because its
held-out positive support is 36.

{md_table(
    family_display_rows,
    [
        "arm",
        "family",
        "learner",
        "status",
        "ROC>0.5",
        "PR>chance",
        "Std recall>0",
        "Bal recall>0",
        "Sec recall>0",
    ],
)}

Among inferentially eligible families, the families for which both learners
satisfied both preregistered ranking conditions in all five seeds under both
chronological LOAO and the random-LOAO control were: **{stable_family_text}**.

BOT remained distinctly learner-dependent under chronological LOAO. The
frozen condition counts are preserved directly in the accompanying claim
registry rather than collapsed into a single family score.

## Random-split LOAO control versus chronological LOAO

The random control is not interpreted as a deployment-realistic estimate.
The comparison below reports **continuous paired contrasts only**, defined as
`random - chronological`. No post-result threshold was introduced to classify
a contrast as "large", "small", "collapse", or "survival".

The sign notation is `+N/-N/=N`, where `+` means the random-control value was
numerically greater than the chronological value for that frozen seed.

{md_table(
    contrast_display_rows,
    [
        "family",
        "learner",
        "status",
        "ΔROC",
        "ΔROC sign",
        "ΔPR-excess",
        "ΔPR-excess sign",
        "ΔStd-recall",
        "ΔStd-recall sign",
    ],
)}

Accordingly, random-versus-chronological differences may be described as
**consistent with chronology compounding novelty difficulty** where the
numerical contrasts support that wording, but they do not establish temporal
drift as the sole causal explanation.

## Reproducibility and interpretation constraints

Training-seed uncertainty is reported separately from sampling/bootstrap
uncertainty. No best seed was selected. No synthetic seed-plus-bootstrap
confidence interval was created. No aggregate zero-day score was created:
family-specific LOAO outcomes remain primary.

Infiltration remains descriptive only because its positive support is 36
(<50). Random LOAO is a control rather than a deployment estimate. The final
Stage22 shared-holdout evaluation is a preregistered robustness re-evaluation
of the already historically opened Stage22R population and is not represented
as a new blind external holdout.

## Manuscript-safe conclusion

The five-seed analysis shows that the principal Stage22 validation-geometry
direction is highly stable to training-seed variation: chronological-natural
models exceeded random-natural models in both PR-AUC and ROC-AUC on the shared
final holdout for all five frozen seeds. In the unseen-family experiments,
however, robustness remains family- and learner-specific. Several families
retain stable ranking and operating-point detection across seeds, whereas BOT
shows marked learner dependence and Infiltration cannot support inferential
claims because of its small positive sample. The random-split LOAO control
provides a complementary benchmark for separating novelty difficulty from the
additional challenge associated with chronological evaluation, without
supporting a causal claim that chronology alone explains the observed
differences.
"""

MANUSCRIPT_OUT.write_text(
    manuscript,
    encoding="utf-8",
)

print(
    "[PASS] manuscript-ready Stage28 results package written"
)


# =================================================================================================
# 13. FINAL SYNTHESIS RECEIPT / README
# =================================================================================================

receipt = {
    "stage":
        "Stage28-FINAL",

    "type":
        "ZERO_FIT_FINAL_SYNTHESIS_AND_MANUSCRIPT_READY_RESULTS_FREEZE",

    "created_at_utc":
        utc_now(),

    "scientific_parent_commit":
        EXPECTED_PARENT,

    "empirical_closure": {
        "authorized_new_fits":
            108,

        "consumed_new_fits":
            108,

        "remaining_new_fits":
            0,

        "historical_reuses":
            12,

        "stage22_shared_holdout_ensemble_evaluations":
            10,

        "stage22_shared_holdout_component_model_inferences":
            20,

        "chronology_loao_seed_realizations":
            50,

        "random_loao_seed_realizations":
            50,

        "stage29_authorized":
            False,
    },

    "scientific_operations_this_stage": {
        "new_model_fits":
            0,

        "model_inferences":
            0,

        "threshold_selections":
            0,

        "model_selections":
            0,

        "target_openings":
            0,

        "shared_final_holdout_openings":
            0,

        "bootstrap_recomputations":
            0,

        "shap_recomputations":
            0,

        "new_formal_statistical_tests":
            0,

        "new_post_result_qualitative_cutoffs":
            0,
    },

    "final_stage22_stability": {
        "PR_RANDOM_LT_CHRONO":
            "5_OF_5",

        "ROC_RANDOM_LT_CHRONO":
            "5_OF_5",
    },

    "loao": {
        "family_specific_primary":
            True,

        "aggregate_zero_day_score":
            False,

        "infiltration":
            "DESCRIPTIVE_ONLY_SUPPORT_36_LT_50",

        "random_loao":
            "CONTROL_NOT_DEPLOYMENT_ESTIMATE",
    },

    "input_sha256":
        input_sha,

    "outputs": [
        str(
            path.relative_to(
                REPO
            )
        )
        for path in [
            STAGE22_SUMMARY_OUT,
            STAGE22_CONTRAST_OUT,
            LOAO_KEY_OUT,
            LOAO_STABILITY_OUT,
            RANDOM_CHRONO_OUT,
            CLAIM_REGISTRY_OUT,
            NUMBERS_OUT,
            MANUSCRIPT_OUT,
        ]
    ],

    "status":
        "STAGE28_FINAL_SYNTHESIS_COMPLETE",

    "next_authorized_work":
        (
            "MANUSCRIPT_INTEGRATION_ONLY; "
            "NO_NEW_MODEL_FITS; NO_NEW_EMPIRICAL_STAGE; NO_STAGE29"
        ),
}

write_json(
    RECEIPT_OUT,
    receipt,
)


README_OUT.write_text(
    f"""# Stage28 Final Synthesis

Scientific parent: `{EXPECTED_PARENT}`

## Final empirical status

- Authorized new fits: 108
- Consumed new fits: 108
- Remaining new fits: 0
- Historical reuses: 12
- Stage22 shared-holdout ensemble evaluations: 10
- Chronology LOAO seed realizations: 50
- Random LOAO seed realizations: 50
- Stage29: not authorized

## Stage28-FINAL operations

- Model fits: 0
- Model inference: 0
- Threshold selection: 0
- Model selection: 0
- Target/holdout opening: 0
- New significance tests: 0
- New qualitative cutoffs: 0

## Frozen Stage22 conclusion stability

- PR random < chronological: 5/5 seeds
- ROC random < chronological: 5/5 seeds

The remaining work is manuscript integration only.
""",
    encoding="utf-8",
)


# =================================================================================================
# 14. CHECKSUMS
# =================================================================================================

artifact_paths = [
    STAGE22_SUMMARY_OUT,
    STAGE22_CONTRAST_OUT,
    LOAO_KEY_OUT,
    LOAO_STABILITY_OUT,
    RANDOM_CHRONO_OUT,
    CLAIM_REGISTRY_OUT,
    NUMBERS_OUT,
    MANUSCRIPT_OUT,
    RECEIPT_OUT,
    README_OUT,
]

checksum_lines = []

for path in artifact_paths:
    checksum_lines.append(
        sha256_file(
            path
        )
        + "  "
        + path.name
    )

CHECKSUM_OUT.write_text(
    "\n".join(
        checksum_lines
    )
    + "\n",
    encoding="utf-8",
)

artifact_paths.append(
    CHECKSUM_OUT
)

print(
    "[PASS] final receipt written"
)

print(
    "[PASS] final README written"
)

print(
    "[PASS] final checksums written"
)


# =================================================================================================
# 15. FINAL ZERO-OPERATION ASSERTIONS
# =================================================================================================

banner(
    "STAGE28-FINAL — ZERO-EMPIRICAL-OPERATION ASSERTIONS"
)

# Verify empirical receipt unchanged.
if read_json(
    STAGE4_RECEIPT
) != stage4:
    raise RuntimeError(
        "Stage28-4 empirical receipt changed unexpectedly."
    )

if read_json(
    CLOSURE_RECEIPT
) != closure:
    raise RuntimeError(
        "Stage28 closure receipt changed unexpectedly."
    )

print(
    "[PASS] Stage28-4 receipt unchanged"
)

print(
    "[PASS] Stage28 fit-closure receipt unchanged"
)

print(
    "[PASS] model fits this stage = 0"
)

print(
    "[PASS] model inference this stage = 0"
)

print(
    "[PASS] final-holdout openings this stage = 0"
)

print(
    "[PASS] target openings this stage = 0"
)

print(
    "[PASS] threshold selections this stage = 0"
)

print(
    "[PASS] new formal tests this stage = 0"
)

print(
    "[PASS] post-result qualitative cutoffs = 0"
)


# =================================================================================================
# 16. EXACT GIT UNIVERSE
# =================================================================================================

banner(
    "STAGE28-FINAL — GIT CHANGE GATE"
)

expected_rel = {
    str(
        path.relative_to(
            REPO
        )
    )
    for path in artifact_paths
}

tracked = set(
    git(
        "diff",
        "--name-only",
    ).splitlines()
)

staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)

untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
)

if tracked:
    raise RuntimeError(
        "Unexpected tracked modifications:\n"
        + "\n".join(
            sorted(
                tracked
            )
        )
    )

if staged:
    raise RuntimeError(
        "Unexpected staged files."
    )

if untracked != expected_rel:
    raise RuntimeError(
        "Unexpected final-synthesis artifact universe.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                expected_rel
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                untracked
            )
        )
    )

print(
    "[PASS] exact final synthesis artifact universe"
)


# =================================================================================================
# 17. DURABLE FINAL SYNTHESIS COMMIT
# =================================================================================================

banner(
    "STAGE28-FINAL — DURABLE COMMIT / PUSH"
)

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)

if (
    git(
        "rev-parse",
        "origin/main",
    )
    != EXPECTED_PARENT
):
    raise RuntimeError(
        "origin/main changed during final synthesis."
    )

for rel in sorted(
    expected_rel
):
    run(
        [
            "git",
            "add",
            "--",
            rel,
        ]
    )

if (
    set(
        git(
            "diff",
            "--cached",
            "--name-only",
        ).splitlines()
    )
    != expected_rel
):
    raise RuntimeError(
        "Final synthesis staged universe mismatch."
    )

run(
    [
        "git",
        "config",
        "user.name",
        "Stage28 Kaggle",
    ]
)

run(
    [
        "git",
        "config",
        "user.email",
        "stage28-kaggle@users.noreply.github.com",
    ]
)

commit_message = (
    "stage28-final: freeze synthesis and manuscript-ready results"
)

print(
    run(
        [
            "git",
            "commit",
            "-m",
            commit_message,
        ]
    ).stdout.strip()
)

FINAL_SYNTHESIS_COMMIT = git(
    "rev-parse",
    "HEAD",
)

if (
    git(
        "rev-parse",
        "HEAD^",
    )
    != EXPECTED_PARENT
):
    raise RuntimeError(
        "Final synthesis parent mismatch."
    )

push_origin_main()

run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)

if not (
    git(
        "rev-parse",
        "HEAD",
    )
    ==
    git(
        "rev-parse",
        "origin/main",
    )
    ==
    FINAL_SYNTHESIS_COMMIT
):
    raise RuntimeError(
        "Final synthesis remote durability check failed."
    )

if git(
    "status",
    "--porcelain",
):
    raise RuntimeError(
        "Repository dirty after final synthesis."
    )

print()
print(
    "[PASS] Stage28 final synthesis durable"
)

print(
    "[PASS] repository clean"
)


# =================================================================================================
# 18. FINAL REPORT
# =================================================================================================

banner(
    "STAGE28 — FINAL SYNTHESIS COMPLETE"
)

print(
    "Empirical parent       :",
    EXPECTED_PARENT,
)

print(
    "Final synthesis commit:",
    FINAL_SYNTHESIS_COMMIT,
)

print()
print(
    "Authorized new fits    : 108"
)

print(
    "Consumed new fits      : 108"
)

print(
    "Remaining new fits     : 0"
)

print(
    "New fits in FINAL      : 0"
)

print(
    "Inference in FINAL     : 0"
)

print(
    "Holdout openings FINAL : 0"
)

print()
print(
    "Stage22 PR direction   : 5 / 5 stable"
)

print(
    "Stage22 ROC direction  : 5 / 5 stable"
)

print(
    "Infiltration           : DESCRIPTIVE ONLY (n=36)"
)

print(
    "Aggregate zero-day     : NOT CREATED"
)

print(
    "Stage29                : NOT AUTHORIZED"
)

print()
print(
    "STAGE28 IS SCIENTIFICALLY CLOSED."
)

print()
print(
    "NEXT:"
)

print(
    "MANUSCRIPT INTEGRATION ONLY."
)


STAGE28-FINAL — REPOSITORY / EMPIRICAL-CLOSURE GATE

Expected parent: f5de70d25a5714ae2b70a18819bde97eb3e38354
Local HEAD     : f5de70d25a5714ae2b70a18819bde97eb3e38354
origin/main    : f5de70d25a5714ae2b70a18819bde97eb3e38354

[PASS] Stage28 new-fit ledger = 108 / 108
[PASS] remaining fit budget = 0
[PASS] Stage28-4 empirical work complete
[PASS] NO Stage29
[PASS] current stage = reporting/synthesis only

STAGE28-FINAL — DURABLE INPUT ARTIFACT GATE

[PASS] stage28_4_seed_level_metrics.csv 98c783c42f983e992dee501bde7708cdbbcd336732725b39d7431911a50371f3
[PASS] stage28_4_random_vs_chronological_seedwise.csv 68eaa112b8165dbc0e8bf5649d00c67e3e0ff7f1350faefca3f07c002d2e1a75
[PASS] stage28_4_stage22_directional_stability_summary.csv faed97a3a2e13e5a206adb691e60e3a862a7f4272302dcec6904ad87bfde5332
[PASS] stage28_3b_loao_seed_level_metrics.csv 0ad43a29d07259d1566a0a4d5d28dade934516b8dd04db92a5ec59c32350ac4f
[PASS] stage28_3b_loao_five_seed_summary.csv d9ba5f6163a6543b8676228b8dfc3debe82b3bf7

In [15]:
# =================================================================================================
# STAGE28 — ARCHIVE COMPLETE KAGGLE NOTEBOOK UNDER scripts/stage28/
#
# DOCUMENTATION / REPRODUCIBILITY ONLY
#
# SCIENTIFIC OPERATIONS:
#   model fits          : 0
#   model inference     : 0
#   threshold selection : 0
#   target opening      : 0
#   holdout opening     : 0
#
# Expected scientific-final parent:
#   94bbebfe6b18249166ac6bc89deadc8a2d6dc627
#
# Outputs:
#   scripts/stage28/stage28_full_kaggle_notebook.ipynb
#   scripts/stage28/stage28_full_kaggle_notebook.py
#   scripts/stage28/notebook_export_manifest.json
#   scripts/stage28/README.md
#   scripts/stage28/checksums.sha256
# =================================================================================================

from __future__ import annotations

import base64
import hashlib
import json
import os
import re
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote, urljoin

import requests


# =================================================================================================
# CONSTANTS
# =================================================================================================

SEP = "=" * 120

REPO = Path(
    "/kaggle/working/ids2018-validation-safe-ablation"
).resolve()

EXPECTED_PARENT = (
    "94bbebfe6b18249166ac6bc89deadc8a2d6dc627"
)

FINAL_RECEIPT = (
    REPO
    / "results"
    / "stage28_stability_novelty_control"
    / "stage28_final_synthesis"
    / "stage28_final_synthesis_receipt.json"
)

OUT = (
    REPO
    / "scripts"
    / "stage28"
)

IPYNB_OUT = (
    OUT
    / "stage28_full_kaggle_notebook.ipynb"
)

PY_OUT = (
    OUT
    / "stage28_full_kaggle_notebook.py"
)

MANIFEST_OUT = (
    OUT
    / "notebook_export_manifest.json"
)

README_OUT = (
    OUT
    / "README.md"
)

CHECKSUM_OUT = (
    OUT
    / "checksums.sha256"
)


# =================================================================================================
# HELPERS
# =================================================================================================

def banner(text):
    print()
    print(SEP)
    print(text)
    print(SEP)
    print()


def utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def run(
    cmd,
    *,
    cwd=REPO,
    check=True,
):
    p = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed ({p.returncode}):\n"
            + " ".join(map(str, cmd))
            + f"\n\nSTDOUT:\n{p.stdout}"
            + f"\n\nSTDERR:\n{p.stderr}"
        )

    return p


def git(*args):
    return run(
        ["git", *args]
    ).stdout.strip()


def sha256_file(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as f:
        while True:
            block = f.read(
                16 * 1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def read_json(path):
    return json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )


def write_json(path, obj):
    Path(path).write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )


def source_as_text(source):
    if isinstance(
        source,
        list,
    ):
        return "".join(
            str(x)
            for x in source
        )

    return str(
        source or ""
    )


# =================================================================================================
# GITHUB AUTH
# =================================================================================================

def get_github_token():

    from kaggle_secrets import (
        UserSecretsClient,
    )

    client = UserSecretsClient()

    labels = [
        "GITHUB_TOKEN",
        "github_token",
        "GH_TOKEN",
        "GITHUB_PAT",
        "github_pat",
        "GH_PAT",
    ]

    for label in labels:

        try:
            value = client.get_secret(
                label
            )

        except Exception:
            value = None

        if (
            isinstance(
                value,
                str,
            )
            and value.strip()
        ):

            return (
                value.strip(),
                label,
            )

    raise RuntimeError(
        "No usable GitHub token found in Kaggle Secrets."
    )


def push_origin_main():

    token, label = (
        get_github_token()
    )

    auth = base64.b64encode(
        (
            "x-access-token:"
            + token
        ).encode(
            "utf-8"
        )
    ).decode(
        "ascii"
    )

    p = subprocess.run(
        [
            "git",
            "-c",
            "credential.helper=",
            "-c",
            (
                "http.extraHeader="
                "AUTHORIZATION: Basic "
                + auth
            ),
            "push",
            "origin",
            "main",
        ],
        cwd=str(
            REPO
        ),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if p.returncode != 0:
        raise RuntimeError(
            "Git push failed.\n\n"
            f"STDOUT:\n{p.stdout}\n\n"
            f"STDERR:\n{p.stderr}"
        )

    print(
        "[PASS] GitHub credential:",
        f"kaggle_secret:{label}",
    )

    print(
        "[PASS] token not displayed"
    )

    if p.stdout.strip():
        print(
            p.stdout.strip()
        )

    if p.stderr.strip():
        print(
            p.stderr.strip()
        )


# =================================================================================================
# 0. SCIENTIFIC-FINAL PARENT GATE
# =================================================================================================

banner(
    "STAGE28 NOTEBOOK ARCHIVE — REPOSITORY / SCIENTIFIC-FINAL GATE"
)

if not (
    REPO
    / ".git"
).is_dir():

    raise RuntimeError(
        "Repository missing."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository must be clean before notebook archival."
    )


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


local_head = git(
    "rev-parse",
    "HEAD",
)

remote_head = git(
    "rev-parse",
    "origin/main",
)


print(
    "Expected Stage28-final:",
    EXPECTED_PARENT,
)

print(
    "Local HEAD            :",
    local_head,
)

print(
    "origin/main           :",
    remote_head,
)


if not (
    local_head
    == remote_head
    == EXPECTED_PARENT
):

    raise RuntimeError(
        "Notebook archive must descend directly from "
        "the frozen Stage28-final commit."
    )


if not FINAL_RECEIPT.is_file():

    raise RuntimeError(
        "Stage28 final synthesis receipt missing."
    )


final_receipt = read_json(
    FINAL_RECEIPT
)


if (
    final_receipt.get(
        "status"
    )
    != "STAGE28_FINAL_SYNTHESIS_COMPLETE"
):

    raise RuntimeError(
        "Stage28 final synthesis receipt is not COMPLETE."
    )


if (
    final_receipt[
        "empirical_closure"
    ][
        "consumed_new_fits"
    ]
    != 108
    or
    final_receipt[
        "empirical_closure"
    ][
        "remaining_new_fits"
    ]
    != 0
):

    raise RuntimeError(
        "Stage28 empirical closure does not equal 108/108."
    )


if OUT.exists():

    raise RuntimeError(
        f"Archive directory already exists:\n{OUT}\n\n"
        "Do not overwrite it."
    )


print()
print(
    "[PASS] Stage28-final parent exact"
)

print(
    "[PASS] scientific closure = 108 / 108"
)

print(
    "[PASS] Stage29 not involved"
)

print(
    "[PASS] this commit is documentation/reproducibility only"
)


# =================================================================================================
# 1. TRY TO OBTAIN THE REAL LIVE NOTEBOOK FROM JUPYTER
# =================================================================================================

banner(
    "STAGE28 NOTEBOOK ARCHIVE — LIVE NOTEBOOK DISCOVERY"
)


notebook_json = None

notebook_source_mode = None

notebook_source_detail = None

jupyter_errors = []


try:

    from ipykernel import (
        get_connection_file,
    )

    connection_file = Path(
        get_connection_file()
    ).name

    kernel_id = (
        Path(
            connection_file
        ).stem
        .replace(
            "kernel-",
            "",
        )
    )


    print(
        "Kernel ID:",
        kernel_id,
    )


    server_candidates = []


    # Jupyter Server
    try:

        from jupyter_server.serverapp import (
            list_running_servers,
        )

        server_candidates.extend(
            list(
                list_running_servers()
            )
        )

    except Exception as exc:

        jupyter_errors.append(
            "jupyter_server: "
            + repr(
                exc
            )
        )


    # Legacy notebook server fallback
    try:

        from notebook.notebookapp import (
            list_running_servers
            as list_legacy_servers,
        )

        server_candidates.extend(
            list(
                list_legacy_servers()
            )
        )

    except Exception as exc:

        jupyter_errors.append(
            "notebook_server: "
            + repr(
                exc
            )
        )


    deduped_servers = []

    seen_server_urls = set()


    for server in server_candidates:

        url = (
            server.get(
                "url"
            )
            or
            server.get(
                "base_url"
            )
        )

        if not url:
            continue

        if url in seen_server_urls:
            continue

        seen_server_urls.add(
            url
        )

        deduped_servers.append(
            server
        )


    print(
        "Running Jupyter server candidates:",
        len(
            deduped_servers
        ),
    )


    for server in deduped_servers:

        if notebook_json is not None:
            break

        base_url = server.get(
            "url"
        )

        token = (
            server.get(
                "token"
            )
            or ""
        )

        headers = {}

        params = {}

        if token:
            params[
                "token"
            ] = token


        try:

            sessions_url = urljoin(
                base_url,
                "api/sessions",
            )

            response = requests.get(
                sessions_url,
                params=params,
                headers=headers,
                timeout=10,
            )

            response.raise_for_status()

            sessions = response.json()


            for session in sessions:

                session_kernel = (
                    session.get(
                        "kernel",
                        {},
                    ).get(
                        "id"
                    )
                )

                if (
                    session_kernel
                    != kernel_id
                ):
                    continue


                notebook_path = (
                    session.get(
                        "path"
                    )
                    or
                    session.get(
                        "notebook",
                        {},
                    ).get(
                        "path"
                    )
                )


                if not notebook_path:
                    continue


                contents_url = urljoin(
                    base_url,
                    "api/contents/"
                    + quote(
                        notebook_path
                    ),
                )


                contents_response = requests.get(
                    contents_url,
                    params={
                        **params,
                        "content": 1,
                    },
                    headers=headers,
                    timeout=30,
                )

                contents_response.raise_for_status()

                payload = (
                    contents_response.json()
                )


                content = payload.get(
                    "content"
                )


                if not isinstance(
                    content,
                    dict,
                ):
                    continue


                if (
                    "cells"
                    not in content
                ):
                    continue


                notebook_json = content

                notebook_source_mode = (
                    "LIVE_JUPYTER_NOTEBOOK_MODEL"
                )

                notebook_source_detail = (
                    notebook_path
                )

                break


        except Exception as exc:

            jupyter_errors.append(
                repr(
                    exc
                )
            )


except Exception as exc:

    jupyter_errors.append(
        "kernel_discovery: "
        + repr(
            exc
        )
    )


# =================================================================================================
# 2. FALL BACK TO IPYTHON EXECUTION HISTORY IF REQUIRED
# =================================================================================================

if notebook_json is None:

    banner(
        "LIVE NOTEBOOK MODEL UNAVAILABLE — RECONSTRUCT FROM IPYTHON HISTORY"
    )


    try:
        ip = get_ipython()

    except NameError:
        ip = None


    if ip is None:

        raise RuntimeError(
            "Neither live notebook model nor IPython history is available."
        )


    history = list(
        ip.history_manager.input_hist_raw
    )


    raw_cells = []


    for history_index, source in enumerate(
        history
    ):

        if (
            history_index == 0
            or
            not isinstance(
                source,
                str,
            )
            or
            not source.strip()
        ):
            continue


        raw_cells.append(
            {
                "cell_type":
                    "code",

                "execution_count":
                    history_index,

                "metadata": {
                    "reconstructed_from_ipython_history":
                        True,

                    "history_index":
                        history_index,
                },

                "outputs":
                    [],

                "source":
                    source,
            }
        )


    if not raw_cells:

        raise RuntimeError(
            "IPython history is empty."
        )


    notebook_json = {
        "cells":
            raw_cells,

        "metadata": {
            "stage28_archive": {
                "source":
                    "IPYTHON_RAW_EXECUTION_HISTORY",

                "warning":
                    (
                        "Markdown cells and rich cell outputs are not "
                        "available from kernel history."
                    ),
            },

            "kernelspec": {
                "display_name":
                    "Python 3",

                "language":
                    "python",

                "name":
                    "python3",
            },

            "language_info": {
                "name":
                    "python",
            },
        },

        "nbformat":
            4,

        "nbformat_minor":
            5,
    }


    notebook_source_mode = (
        "IPYTHON_RAW_EXECUTION_HISTORY_RECONSTRUCTION"
    )

    notebook_source_detail = (
        f"{len(raw_cells)} executed code cells"
    )


print()
print(
    "Notebook source mode:",
    notebook_source_mode,
)

print(
    "Notebook source detail:",
    notebook_source_detail,
)


if jupyter_errors:

    print()
    print(
        "Jupyter discovery notes:"
    )

    for error in jupyter_errors[
        :10
    ]:

        print(
            " ",
            error[
                :500
            ],
        )


# =================================================================================================
# 3. VALIDATE NOTEBOOK CONTENT
# =================================================================================================

banner(
    "STAGE28 NOTEBOOK ARCHIVE — CONTENT VALIDATION"
)


cells = notebook_json.get(
    "cells"
)


if not isinstance(
    cells,
    list,
) or not cells:

    raise RuntimeError(
        "Notebook contains no cells."
    )


code_cells = [
    cell
    for cell in cells
    if cell.get(
        "cell_type"
    )
    == "code"
]


markdown_cells = [
    cell
    for cell in cells
    if cell.get(
        "cell_type"
    )
    == "markdown"
]


code_text = "\n\n".join(
    source_as_text(
        cell.get(
            "source",
            ""
        )
    )
    for cell in code_cells
)


required_stage28_markers = [
    "STAGE28-3A",
    "STAGE28-3B",
    "STAGE28-3C",
    "STAGE28-4",
    "STAGE28-FINAL",
]


missing_markers = [
    marker
    for marker
    in required_stage28_markers
    if marker
    not in code_text
]


if missing_markers:

    raise RuntimeError(
        "Notebook export does not contain required late-Stage28 markers:\n"
        + "\n".join(
            missing_markers
        )
    )


print(
    "Total cells   :",
    len(
        cells
    ),
)

print(
    "Code cells    :",
    len(
        code_cells
    ),
)

print(
    "Markdown cells:",
    len(
        markdown_cells
    ),
)


print()
print(
    "[PASS] Stage28-3A present"
)

print(
    "[PASS] Stage28-3B present"
)

print(
    "[PASS] Stage28-3C present"
)

print(
    "[PASS] Stage28-4 present"
)

print(
    "[PASS] Stage28-FINAL present"
)


# =================================================================================================
# 4. BUILD SCRIPT VERSION
# =================================================================================================

banner(
    "STAGE28 NOTEBOOK ARCHIVE — BUILD scripts/stage28 EXPORT"
)


script_parts = [
    '''#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Stage28 complete Kaggle notebook source archive.

Scientific-final parent:
    94bbebfe6b18249166ac6bc89deadc8a2d6dc627

IMPORTANT
---------
This file is an archival linearization of the Kaggle notebook.

It preserves notebook cell order and source. It is not a new Stage28
scientific stage and must not be interpreted as authorizing new model fits.

For the notebook representation, see:
    stage28_full_kaggle_notebook.ipynb
"""

'''
]


for cell_number, cell in enumerate(
    cells,
    start=1,
):

    cell_type = cell.get(
        "cell_type",
        "unknown",
    )

    source = source_as_text(
        cell.get(
            "source",
            ""
        )
    )


    if cell_type == "markdown":

        script_parts.append(
            "\n"
            + "# "
            + "=" * 110
            + "\n"
            + f"# %% [markdown] NOTEBOOK CELL {cell_number:04d}\n"
            + "# "
            + "=" * 110
            + "\n"
        )


        if source:

            for line in source.splitlines():

                script_parts.append(
                    "# "
                    + line
                    + "\n"
                )


    elif cell_type == "code":

        execution_count = cell.get(
            "execution_count"
        )


        script_parts.append(
            "\n"
            + "# "
            + "=" * 110
            + "\n"
            + (
                f"# %% NOTEBOOK CELL {cell_number:04d} "
                f"| execution_count={execution_count}\n"
            )
            + "# "
            + "=" * 110
            + "\n"
        )


        script_parts.append(
            source
        )


        if (
            source
            and
            not source.endswith(
                "\n"
            )
        ):

            script_parts.append(
                "\n"
            )


    else:

        script_parts.append(
            "\n"
            + "# "
            + "=" * 110
            + "\n"
            + (
                f"# NOTEBOOK CELL {cell_number:04d} "
                f"| unsupported type={cell_type!r}\n"
            )
            + "# "
            + "=" * 110
            + "\n"
        )


script_text = "".join(
    script_parts
)


# =================================================================================================
# 5. HIGH-CONFIDENCE SECRET SCAN
# =================================================================================================

banner(
    "STAGE28 NOTEBOOK ARCHIVE — SECRET-SAFETY GATE"
)


serialized_notebook = json.dumps(
    notebook_json,
    ensure_ascii=False,
)


secret_patterns = {
    "GitHub classic token":
        r"\bgh[pousr]_[A-Za-z0-9]{20,}\b",

    "GitHub fine-grained PAT":
        r"\bgithub_pat_[A-Za-z0-9_]{20,}\b",

    "OpenAI-style secret":
        r"\bsk-[A-Za-z0-9_-]{20,}\b",

    "Stripe live secret":
        r"\bsk_live_[A-Za-z0-9]{10,}\b",

    "HuggingFace token":
        r"\bhf_[A-Za-z0-9]{20,}\b",

    "AWS access key":
        r"\bAKIA[0-9A-Z]{16}\b",

    "Generic literal GitHub token assignment":
        (
            r"""(?i)\b(?:GITHUB_TOKEN|GH_TOKEN|GITHUB_PAT|GH_PAT)"""
            r"""\s*=\s*['"][^'"]{20,}['"]"""
        ),
}


secret_hits = []


for name, pattern in (
    secret_patterns.items()
):

    regex = re.compile(
        pattern
    )


    for target_name, target_text in [
        (
            "ipynb",
            serialized_notebook,
        ),
        (
            "py",
            script_text,
        ),
    ]:

        match = regex.search(
            target_text
        )


        if match:

            # Do NOT print the secret itself.
            secret_hits.append(
                {
                    "pattern":
                        name,

                    "artifact":
                        target_name,

                    "offset":
                        int(
                            match.start()
                        ),
                }
            )


if secret_hits:

    raise RuntimeError(
        "Potential hard-coded secret detected in notebook source.\n"
        "Nothing has been written or committed.\n\n"
        + json.dumps(
            secret_hits,
            indent=2,
        )
    )


print(
    "[PASS] no high-confidence hard-coded credential patterns detected"
)

print(
    "[PASS] Kaggle Secret retrieval code is safe to archive"
)

print(
    "[PASS] no token value will be inserted into exported source"
)


# =================================================================================================
# 6. WRITE ARCHIVE
# =================================================================================================

OUT.mkdir(
    parents=False,
    exist_ok=False,
)


IPYNB_OUT.write_text(
    json.dumps(
        notebook_json,
        indent=1,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)


PY_OUT.write_text(
    script_text,
    encoding="utf-8",
)


# GitHub hard limit protection.
for path in [
    IPYNB_OUT,
    PY_OUT,
]:

    size = path.stat().st_size


    print(
        path.name,
        "bytes=",
        f"{size:,}",
    )


    if size >= (
        95
        * 1024
        * 1024
    ):

        raise RuntimeError(
            f"{path.name} is too large for safe normal GitHub storage."
        )


# =================================================================================================
# 7. EXPORT MANIFEST
# =================================================================================================

notebook_sha = sha256_file(
    IPYNB_OUT
)

script_sha = sha256_file(
    PY_OUT
)

manifest = {
    "artifact":
        "STAGE28_COMPLETE_KAGGLE_NOTEBOOK_ARCHIVE",

    "created_at_utc":
        utc_now(),

    "scientific_final_parent_commit":
        EXPECTED_PARENT,

    "scientific_status":
        "STAGE28_SCIENTIFICALLY_CLOSED",

    "stage29_authorized":
        False,

    "archive_commit_type":
        "DOCUMENTATION_AND_REPRODUCIBILITY_ONLY",

    "source_mode":
        notebook_source_mode,

    "source_detail":
        notebook_source_detail,

    "fidelity": {
        "code_cell_source_preserved":
            True,

        "cell_order_preserved":
            True,

        "markdown_preserved":
            (
                notebook_source_mode
                == "LIVE_JUPYTER_NOTEBOOK_MODEL"
            ),

        "outputs_preserved":
            (
                notebook_source_mode
                == "LIVE_JUPYTER_NOTEBOOK_MODEL"
            ),

        "fallback_warning":
            (
                None
                if notebook_source_mode
                == "LIVE_JUPYTER_NOTEBOOK_MODEL"
                else
                (
                    "Live notebook model was unavailable. "
                    "The .ipynb was reconstructed from raw "
                    "IPython execution history; markdown and "
                    "rich outputs are therefore not available."
                )
            ),
    },

    "notebook": {
        "path":
            str(
                IPYNB_OUT.relative_to(
                    REPO
                )
            ),

        "sha256":
            notebook_sha,

        "bytes":
            IPYNB_OUT.stat().st_size,

        "total_cells":
            len(
                cells
            ),

        "code_cells":
            len(
                code_cells
            ),

        "markdown_cells":
            len(
                markdown_cells
            ),
    },

    "python_export": {
        "path":
            str(
                PY_OUT.relative_to(
                    REPO
                )
            ),

        "sha256":
            script_sha,

        "bytes":
            PY_OUT.stat().st_size,
    },

    "required_markers_verified":
        required_stage28_markers,

    "scientific_operations_performed_by_archive":
        {
            "model_fits":
                0,

            "model_inferences":
                0,

            "threshold_selections":
                0,

            "model_selections":
                0,

            "target_openings":
                0,

            "final_holdout_openings":
                0,
        },

    "final_stage28_receipt": {
        "path":
            str(
                FINAL_RECEIPT.relative_to(
                    REPO
                )
            ),

        "sha256":
            sha256_file(
                FINAL_RECEIPT
            ),

        "status":
            final_receipt[
                "status"
            ],
    },
}


write_json(
    MANIFEST_OUT,
    manifest,
)


README_OUT.write_text(
    f"""# Stage28 Kaggle Notebook Archive

This directory archives the Stage28 Kaggle notebook after the scientific
analysis was fully closed.

## Scientific lineage

Final scientific Stage28 commit before notebook archival:

`{EXPECTED_PARENT}`

The archive commit is documentation/reproducibility only. It does not modify
Stage28 results or authorize any new empirical work.

## Files

- `stage28_full_kaggle_notebook.ipynb`
- `stage28_full_kaggle_notebook.py`
- `notebook_export_manifest.json`
- `checksums.sha256`

## Notebook source mode

`{notebook_source_mode}`

Source detail:

`{notebook_source_detail}`

## Scientific closure

- Authorized Stage28 new fits: 108
- Consumed Stage28 new fits: 108
- Remaining new fits: 0
- Stage29: not authorized
- Model fits during this archive step: 0
- Model inference during this archive step: 0
- Holdout/target openings during this archive step: 0

The `.py` file is a cell-ordered archival linearization of the notebook.
The `.ipynb` file is the preferred notebook representation.
""",
    encoding="utf-8",
)


# =================================================================================================
# 8. CHECKSUMS
# =================================================================================================

archive_files_without_checksum = [
    IPYNB_OUT,
    PY_OUT,
    MANIFEST_OUT,
    README_OUT,
]


CHECKSUM_OUT.write_text(
    "\n".join(
        (
            sha256_file(
                path
            )
            + "  "
            + path.name
        )
        for path in (
            archive_files_without_checksum
        )
    )
    + "\n",
    encoding="utf-8",
)


archive_files = [
    *archive_files_without_checksum,
    CHECKSUM_OUT,
]


print()
print(
    "[PASS] notebook archive written"
)

print(
    "[PASS] manifest written"
)

print(
    "[PASS] README written"
)

print(
    "[PASS] checksum manifest written"
)


# =================================================================================================
# 9. EXACT GIT UNIVERSE
# =================================================================================================

banner(
    "STAGE28 NOTEBOOK ARCHIVE — EXACT GIT UNIVERSE"
)


expected_rel = {
    str(
        path.relative_to(
            REPO
        )
    )
    for path in (
        archive_files
    )
}


tracked = set(
    git(
        "diff",
        "--name-only",
    ).splitlines()
)

staged = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)

untracked = set(
    git(
        "ls-files",
        "--others",
        "--exclude-standard",
    ).splitlines()
)


if tracked:

    raise RuntimeError(
        "Unexpected tracked modifications:\n"
        + "\n".join(
            sorted(
                tracked
            )
        )
    )


if staged:

    raise RuntimeError(
        "Unexpected staged files."
    )


if untracked != expected_rel:

    raise RuntimeError(
        "Unexpected notebook-archive file universe.\n\n"
        "Expected:\n"
        + "\n".join(
            sorted(
                expected_rel
            )
        )
        + "\n\nActual:\n"
        + "\n".join(
            sorted(
                untracked
            )
        )
    )


print(
    "[PASS] exactly five archival files"
)

print(
    "[PASS] no result/model/metadata artifact modified"
)


# =================================================================================================
# 10. COMMIT / PUSH
# =================================================================================================

banner(
    "STAGE28 NOTEBOOK ARCHIVE — DURABLE COMMIT / PUSH"
)


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


if (
    git(
        "rev-parse",
        "origin/main",
    )
    != EXPECTED_PARENT
):

    raise RuntimeError(
        "origin/main changed while notebook archive was being prepared."
    )


for rel in sorted(
    expected_rel
):

    run(
        [
            "git",
            "add",
            "-f",
            "--",
            rel,
        ]
    )


staged_after = set(
    git(
        "diff",
        "--cached",
        "--name-only",
    ).splitlines()
)


if staged_after != expected_rel:

    raise RuntimeError(
        "Staged notebook archive universe mismatch."
    )


run(
    [
        "git",
        "config",
        "user.name",
        "Stage28 Kaggle",
    ]
)

run(
    [
        "git",
        "config",
        "user.email",
        "stage28-kaggle@users.noreply.github.com",
    ]
)


commit_message = (
    "archive: add complete Stage28 Kaggle notebook under scripts"
)


print(
    run(
        [
            "git",
            "commit",
            "-m",
            commit_message,
        ]
    ).stdout.strip()
)


ARCHIVE_COMMIT = git(
    "rev-parse",
    "HEAD",
)


if (
    git(
        "rev-parse",
        "HEAD^",
    )
    != EXPECTED_PARENT
):

    raise RuntimeError(
        "Notebook archive commit parent mismatch."
    )


push_origin_main()


run(
    [
        "git",
        "fetch",
        "origin",
        "main",
    ]
)


if not (
    git(
        "rev-parse",
        "HEAD",
    )
    ==
    git(
        "rev-parse",
        "origin/main",
    )
    ==
    ARCHIVE_COMMIT
):

    raise RuntimeError(
        "Notebook archive remote durability verification failed."
    )


if git(
    "status",
    "--porcelain",
):

    raise RuntimeError(
        "Repository dirty after notebook archive commit."
    )


# =================================================================================================
# 11. COMPLETE
# =================================================================================================

banner(
    "STAGE28 NOTEBOOK ARCHIVE — COMPLETE"
)


print(
    "Scientific-final parent:",
    EXPECTED_PARENT,
)

print(
    "Notebook archive commit:",
    ARCHIVE_COMMIT,
)

print()
print(
    "Source mode:",
    notebook_source_mode,
)

print(
    "Total notebook cells:",
    len(
        cells
    ),
)

print(
    "Code cells:",
    len(
        code_cells
    ),
)

print(
    "Markdown cells:",
    len(
        markdown_cells
    ),
)

print()
print(
    "Notebook SHA256:",
    notebook_sha,
)

print(
    "Python SHA256  :",
    script_sha,
)

print()
print(
    "Model fits        : 0"
)

print(
    "Model inference   : 0"
)

print(
    "Threshold search  : 0"
)

print(
    "Holdout openings  : 0"
)

print(
    "Target openings   : 0"
)

print()
print(
    "[PASS] Stage28 scientific results untouched"
)

print(
    "[PASS] complete notebook source archived under scripts/stage28/"
)

print(
    "[PASS] Stage29 remains NOT AUTHORIZED"
)

print()
print(
    "NEXT: manuscript integration only."
)


STAGE28 NOTEBOOK ARCHIVE — REPOSITORY / SCIENTIFIC-FINAL GATE

Expected Stage28-final: 94bbebfe6b18249166ac6bc89deadc8a2d6dc627
Local HEAD            : 94bbebfe6b18249166ac6bc89deadc8a2d6dc627
origin/main           : 94bbebfe6b18249166ac6bc89deadc8a2d6dc627

[PASS] Stage28-final parent exact
[PASS] scientific closure = 108 / 108
[PASS] Stage29 not involved
[PASS] this commit is documentation/reproducibility only

STAGE28 NOTEBOOK ARCHIVE — LIVE NOTEBOOK DISCOVERY

Kernel ID: fdcd30b7-f4df-4188-8805-74e0c8630a27


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


Running Jupyter server candidates: 1

LIVE NOTEBOOK MODEL UNAVAILABLE — RECONSTRUCT FROM IPYTHON HISTORY


Notebook source mode: IPYTHON_RAW_EXECUTION_HISTORY_RECONSTRUCTION
Notebook source detail: 15 executed code cells

Jupyter discovery notes:
  HTTPError('404 Client Error: Not Found for url: http://localhost:8888/k/344182136/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2IiwidHlwIjoiSldUIn0..JPXkKgVak84_Tnqvi53XxA.KFq-p1b0_DNPDSTNIdcWmJBitofeFRa4mKPQzYAns1z7cRmKB6DUAyS2pMIZdCD1IRZnrzQFObQmEG-1GKgMbl_QqzeNzqo6kGKHsqhnC-fWP9H7Xm9bey3uQLPUC_n4FKB_w3vdMGp7MjwG-MHpldGUV8UhPx8hvCmVF85dHlbeEtbxPiGk9h4Fnox9gCtpsUaEY1CEwf6LLfEj17dM42IqABWbo5MyBc5tV60RW0oInsCr5EThnNMMbvDK0H8i.HBJdBY_SBlbw3eE9b7fPxw/proxy/api/contents/__notebook_source__.ipynb?conte

STAGE28 NOTEBOOK ARCHIVE — CONTENT VALIDATION

Total cells   : 15
Code cells    : 15
Markdown cells: 0

[PASS] Stage28-3A present
[PASS] Stage28-3B present
[PASS] Stage28-3C present
[PASS] Stage28-4 present
[PASS] Stage28-FINAL present

STAGE28 NOTE